In [ ]:
from __future__ import annotations

import contextlib
import csv
from decimal import Decimal, localcontext
from fractions import Fraction
import hashlib
import io
import json
import math
import os
from pathlib import Path
import shutil
import statistics
import subprocess
import sys
import tempfile
from typing import Any

import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
from IPython.display import display

print('Python', sys.version.split()[0], '| Gurobi', '.'.join(map(str, gp.gurobi.version())))


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'solution_transfer_20261018'
METHODS = ['LEAN-GPT-4.1', 'LEAN-gpt-oss-20b']





REPLAY_METHODS = set()

EXPECTED_IDS = {f'OR-{n:03}' for n in range(1, 102)}
CHECK_KINDS = ['constraint', 'bound', 'integrality', 'binary_domain']
TOLERANCES = {
    'diagnostic_1e-12': Fraction('1e-12'),
    'absolute_1e-9': Fraction('1e-9'),
    'absolute_1e-6': Fraction('1e-6'),
    'historical_scaled': None,
}
assert METHODS and len(set(METHODS)) == len(METHODS)
assert set(REPLAY_METHODS) <= set(METHODS)
assert (PROJECT_ROOT / 'Test_Dataset/Large-scale-or/Large-scale-or-101.csv').is_file(), (
    'Open the notebook from the root of the LEAN repository.')


In [ ]:
COHORTS = json.loads(r"""
{
 "LEAN-GPT-4.1": {
  "relative_directory": "outputs/ap_matrix_fixed_cache_v2/automatic",
  "source_commit": "af1c3738342d2704129843749784d4acdc6d6031",
  "results_sha256": "7a30ec96b3247cdbe8eeb3ec096006a349dc1e1fa939e4eedf8cc7acae7f3690",
  "incumbent_protocol": "published_saved_solution"
 },
 "LEAN-gpt-oss-20b": {
  "relative_directory": "outputs/final_101_V4_oss20b/automatic",
  "source_commit": "17f09b84c08e6b83ecc0f614d14a2caeaa311fc0",
  "results_sha256": "5e7e15901384d7cea49a4c8180d9678f498ee866dd053bac063d11368b55fb49",
  "incumbent_protocol": "frozen_fixed_code_replay"
 }
}
""")


In [ ]:
DATA_SHA256 = json.loads(r"""
{
 "Test_Dataset/Large-scale-or/Large-scale-or-101.csv": "4574d324f111202b96c2791b588f21b4a2a509e6e0635cb0d8bf9a1fdf887f49",
 "Test_Dataset/Large-scale-or/TP_testing/TP1/customer_demand.csv": "5499656a0e8e937cb80210810116f39b670b04c1e0883d781151e75e1c10defc",
 "Test_Dataset/Large-scale-or/TP_testing/TP1/supply_capacity.csv": "f8f169fca61e02f19bff5da273652ba016368ea526f6d0abead02007c23ee950",
 "Test_Dataset/Large-scale-or/TP_testing/TP1/transportation_costs.csv": "d15d2c1db282122b170697a44c1a87961d3266f8d803948d470218c22c652275",
 "Test_Dataset/Large-scale-or/TP_testing/TP2/customer_demand.csv": "fe1b7a4ce4ffee22c4a587ad73d63b0a0d998dcb269b70bcbf136303df61896c",
 "Test_Dataset/Large-scale-or/TP_testing/TP2/supply_capacity.csv": "ef72030251e7df43df8ced4f7c244856382b2b861f87107525ffec7a5c792382",
 "Test_Dataset/Large-scale-or/TP_testing/TP2/transportation_costs.csv": "4d2713827c424fb5373c8954abaed7f04708c5b7a4bdc4383fcc8631dd082689",
 "Test_Dataset/Large-scale-or/TP_testing/TP3/customer_demand.csv": "8e3d8d2c378a0e0548d15dd005b36d73c42d047c7052d9caab4c6bc0bc798587",
 "Test_Dataset/Large-scale-or/TP_testing/TP3/supply_capacity.csv": "4ac61a580b82bc701375e863fa695c1c95a005010d926bf8cdc35d8f68c7088a",
 "Test_Dataset/Large-scale-or/TP_testing/TP3/transportation_costs.csv": "ae79d34e4daa13feb2d66559f492e9954589a5c52230a61cace9fc08ec8a17a3",
 "Test_Dataset/Large-scale-or/TP_testing/TP4/customer_demand.csv": "389d029a2912ef374cff24d8240b60e77e4532316e2b015e1940911f36e941e5",
 "Test_Dataset/Large-scale-or/TP_testing/TP4/supply_capacity.csv": "c4049a14f8f8ee520589fc1e41219427652b9a3cfb3149f4b0646a5e9fc9a248",
 "Test_Dataset/Large-scale-or/TP_testing/TP4/transportation_costs.csv": "58655183760198106bb4c8dac8357b5f5bf29497b81788a5e8e85bf780912091",
 "Test_Dataset/Large-scale-or/TP_testing/TP5/customer_demand.csv": "f5e4cb6ecf3218b45aec7ff1ac4b5a6705e096733b32a280b40c395c39b9ca23",
 "Test_Dataset/Large-scale-or/TP_testing/TP5/supply_capacity.csv": "59c3133d4e8e3e06b02c2c55c21e58315f41ccfe76c5c1e76bc4771a74688d0f",
 "Test_Dataset/Large-scale-or/TP_testing/TP5/transportation_costs.csv": "0f94a7bd4b4ca29fd4cee9a17bb88e26d071e275103d70dfc1b6927e944a34bd",
 "Test_Dataset/Large-scale-or/TP_testing/TP6/customer_demand.csv": "641d516cde0d5cf1e21d3195a84221718bf0bb38ffcfb116b4a471fc0607a5de",
 "Test_Dataset/Large-scale-or/TP_testing/TP6/supply_capacity.csv": "76e9dff477f1743a0d66a1d0abf80e05f942d6a0a3039492d83eaab3b83f4844",
 "Test_Dataset/Large-scale-or/TP_testing/TP6/transportation_costs.csv": "7c4f1e6a2b1efc9bac54b0a1fb1a377e2e6eaf1be6c97ce266b41b0ce5625d16",
 "Test_Dataset/Large-scale-or/TP_testing/TP7/customer_demand.csv": "dc47c704349c8cc5b57c3e969f966f6941a787f41a361a2a076d83d549f55209",
 "Test_Dataset/Large-scale-or/TP_testing/TP7/supply_capacity.csv": "a979ed7c81c89a279aa6c934cfb4237b08544954ed0f62196d4362f988740eed",
 "Test_Dataset/Large-scale-or/TP_testing/TP7/transportation_costs.csv": "47e377319f1d321307a4f0b972bde6f82fb3571298ed672c7ba1af0312f52f74",
 "Test_Dataset/Large-scale-or/TP_testing/TP8/customer_demand.csv": "bc9f6c01a98786697a4add9b4bd46026aaa4733ebf95cafb42b3d8761aff4f50",
 "Test_Dataset/Large-scale-or/TP_testing/TP8/supply_capacity.csv": "341fa6b12f085076276f4a5c444d2abda8de8122f3693ed271ac19d443575efe",
 "Test_Dataset/Large-scale-or/TP_testing/TP8/transportation_costs.csv": "7912f35b974272abd5f88573ed2b43ad6badb90b2a098eb9ecdb59bdf612fd3c",
 "Test_Dataset/Large-scale-or/TP_testing/TP9/customer_demand.csv": "33a5052d984ae6bdf6f8212c2b8b2f364e552dc0c4b87f59bb31cfe81870f877",
 "Test_Dataset/Large-scale-or/TP_testing/TP9/supply_capacity.csv": "38529367dcdda1f85260f9ca463f213477e28d05c1a1afd1eb36b5ed1166cd7d",
 "Test_Dataset/Large-scale-or/TP_testing/TP9/transportation_costs.csv": "bc2cf7cca51d17ccad835aa34e008dbf6f40fde0d3e07884e6a8d0cb78cbda1f",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM1/MobileSalesDataset.csv": "091b6e833fce948797100177d28c06714b4aeb9836bcd0c66f78407f2931bc8a",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM2/OnlineRetailSalesDataset.csv": "080279712e7e543d375862556b8f8be47c8461ca18210b381a921ca02bc9169b",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM3/OnlineSalesDataset.csv": "58b733e4d1fcb2c3364a5f3213c48b397251f48946327e608d9e31d922897f74",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM4/OnlineSalesinUSA.csv": "ab9f7da9229fdf20bed21c961486c34e573d1b59d113d117dab76e154286fd3e",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM5/PizzaSalesDataset.csv": "943a4628e70a4271a46923293851cdc654dfc8f209d9437eb84d0ee1bd698492",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM6/RestaurantSalesreport.csv": "a086c472bc608c023fce018c8ea32237cc892db2023f7c2aad925bf293db4ba6",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM7/RetailSalesDataset.csv": "ffcea5fa0c066a16a069a68b0b09da4b5b6e7017df160b6f7af6b4347786e959",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM8/RetailStoreSalesTransactions(ScannerData).csv": "5ae5bae71639c142c44284dca4cb4bcd956a4f83c22ae684c5ad21b636f8d902",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM9/Salesdata.csv": "b120b37ea174d07288096c411f3f418090d43914bb073dbdc57694e485d2dcda",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM10/SalesDataAnalysis.csv": "ad0f584338b2e87704c16d806abfba492df7258eba5b7548cb2568a347c3e28f",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM11/SalesDatainBusinesses.csv": "c249e8c3eaaa01a661bdccec6c5ff28d6b0f2e591cea86189f35ee4570a9fc5f",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM12/Salesofsummerclothes.csv": "ad535d5e07e6d95c9806e866454aa617993bfca2b563d07d5a4e63a45f1b17d5",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM13/Salesorders.csv": "8caf12f4dfbe49778ea70b0202abe941ef161482b35b6ad53b21c77be263727a",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM14/SalesStoreoverview.csv": "dacce5b81b38dcb84638a419fd1eefa67f0c0155961cc181507be47405208f0c",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM15/SampleSalesData.csv": "42a7b837b14eee0f30c0d6d77d1c40f38533f26d283e85585952d9aaec23955a",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM16/SmartphoneRetailOutletSalesData.csv": "a73870273fe085d8d48af748013b85c78373dab56309637f59f8cad4e00ccd6b",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM17/SupermarketSales.csv": "697698bc34e0c9c5a1edacc8a51f013309b0af6763e15606160970c6a060629c",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM18/SupermartGrocerySales-RetailAnalyticsDataset.csv": "eba917a7d7addb6e9e8990dd56433b98e72e24da2b4ba94cb458cdef65719756",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM19/WomenClothingEcommerceSalesData.csv": "b67f871b94f2a8d4da1632e82dd1bfed1ec335ab7f2bdf300d874f4d67f74842",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM20/ZARASales.csv": "785fc304e8147a2db575f7eb457eabce6e2688c57b19fcd2f7be7201324f3405",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM21/BigMartSales.csv": "2c7017f632b5d326776214dcd1e8483a467137baa917b60f5ee0099f7aa1a1e3",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM22/DairyGoodsSalesDataset.csv": "00a197b026d710e5c00433f10182b37968a11e642315379125d2c2300e2950fd",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM23/DifferentStoreSales.csv": "772722170921e15da09e40aba9c4c48d8bf13943b9b5fc20af39a49a794fcc66",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM24/EuropeSalesRecords.csv": "f55f7281a09e5fe6ec794864b4007b8c781d1d11a732363f4534f37be76848ba",
 "Test_Dataset/Large-scale-or/NRM_testing/NRM25/Frenchbakerydailysales.csv": "d11ca52230f2f22a11ad494b1d90ea64c403eb405d7e88e45fcd1649e623e967",
 "Test_Dataset/Large-scale-or/RA_testing/RA1/BakerySales1/capacity.csv": "a7c5f189f0e4ecbc20b65c1bfe3e373db8b09e86cd489b28f1c0ef2477cbc1bf",
 "Test_Dataset/Large-scale-or/RA_testing/RA1/BakerySales1/products.csv": "decf9883b14104043c8b3c38cdb035d513fea3d08973f7eebd9f8115ca3fa0b7",
 "Test_Dataset/Large-scale-or/RA_testing/RA2/CarSales1/capacity.csv": "6e20b18f49ea9eac6ec56922cc490b85823ebd0eed6b945d75bd140ee0736eae",
 "Test_Dataset/Large-scale-or/RA_testing/RA2/CarSales1/products.csv": "cb6d87d80fa376b6ad1297aedb03fb9d2cb985b335f49ff5265ee9cc853f98d0",
 "Test_Dataset/Large-scale-or/RA_testing/RA3/CarSales2/capacity.csv": "d5f91a46877a5e87fa389563e18807c4445cf03d67ac8977a9812f9766cfda68",
 "Test_Dataset/Large-scale-or/RA_testing/RA3/CarSales2/products.csv": "b962c38ebc2e7b60da0a298de2df9408b5b9dca9ac3f6510a1d400cd08030628",
 "Test_Dataset/Large-scale-or/RA_testing/RA4/NewCarSalesInNorway1/capacity.csv": "5b034b564fba4cf7a4483bba53a6755f66ef113bd253932eb151fa9c9fbe833e",
 "Test_Dataset/Large-scale-or/RA_testing/RA4/NewCarSalesInNorway1/products.csv": "0d302bcd3dcb4a964f2914dd9db16070c17cc00172655c96350ee8f655d33196",
 "Test_Dataset/Large-scale-or/RA_testing/RA5/NewCarSalesInNorway2/capacity.csv": "0b2c362c76a4ee61be32016c6c8a66b6543f36cb77dce06eef1de73eec20a71f",
 "Test_Dataset/Large-scale-or/RA_testing/RA5/NewCarSalesInNorway2/products.csv": "0d302bcd3dcb4a964f2914dd9db16070c17cc00172655c96350ee8f655d33196",
 "Test_Dataset/Large-scale-or/RA_testing/RA6/NYCPropertySales1/capacity.csv": "81a62b5cce59145bb5193df5fd68bf32e876ecaea683355801c4a369daa6c153",
 "Test_Dataset/Large-scale-or/RA_testing/RA6/NYCPropertySales1/products.csv": "17194b5272b83b15e56cdd79f6c388e7c232ad2c7830f933d0fddc0b6df38b56",
 "Test_Dataset/Large-scale-or/RA_testing/RA7/NYCPropertySales2/capacity.csv": "5587cc518877ecfd5f06b0d03f9c68fc8723c2d0e26e9b8a9eceba766efb631d",
 "Test_Dataset/Large-scale-or/RA_testing/RA7/NYCPropertySales2/products.csv": "fdb52c2bf73f5d5e3de0e2503c288c7bd9a810f0235e581d657dd06e35942da6",
 "Test_Dataset/Large-scale-or/RA_testing/RA8/PharmaSalesData1/capacity.csv": "d6711c7cae88ee9168d9850aabbdce5cf26fde234cf76719fc86ffdccd1fa527",
 "Test_Dataset/Large-scale-or/RA_testing/RA8/PharmaSalesData1/products.csv": "04a35e5f71a15d1e774c8f9b1ba10f2751f60e095af5d60674da66a55bd72f1f",
 "Test_Dataset/Large-scale-or/RA_testing/RA9/PharmaSalesData2/capacity.csv": "fa12c4d31d2bd3b5a770c032d2e45992fa6039837362cba061b8c34f9f832f74",
 "Test_Dataset/Large-scale-or/RA_testing/RA9/PharmaSalesData2/products.csv": "0b586c45cf1a32658fc9cf4e474d7147bedc53b1f3ac998da123870e9d60758f",
 "Test_Dataset/Large-scale-or/RA_testing/RA10/SalesOfASupermarket1/capacity.csv": "e3858f52a1f37d5a8ed2478bb833612e82ec0388241b5398a614105a069a0b7e",
 "Test_Dataset/Large-scale-or/RA_testing/RA10/SalesOfASupermarket1/products.csv": "40bcc32492cc448f30a658700aa97a9e053a0a6fb06cd2fc94cdc8ffa620b9b2",
 "Test_Dataset/Large-scale-or/RA_testing/RA11/SupermarketSalesData1/capacity.csv": "5b437ca0a6fc363975a7670b6884f35ea8eaecdf6353652d7bf892ff485a182f",
 "Test_Dataset/Large-scale-or/RA_testing/RA11/SupermarketSalesData1/products.csv": "5eef4f77759f35c4161542d77c02725336e1403d326c7380795c0e986e9ffeaf",
 "Test_Dataset/Large-scale-or/RA_testing/RA12/SupermarketSalesData2/capacity.csv": "aabe26fed61769837013f24a2a6f846ed53a8a842b75ba1711dd18cf612c66c6",
 "Test_Dataset/Large-scale-or/RA_testing/RA12/SupermarketSalesData2/products.csv": "6214469728f168bacec685f3951cd9e544c9622bfd642bab81cae703494ef5e9",
 "Test_Dataset/Large-scale-or/RA_testing/RA13/VideoGameSales1/capacity.csv": "8b422a8e656dac50824672fef50a40988405f077c255f0e43e245baa56bf0a47",
 "Test_Dataset/Large-scale-or/RA_testing/RA13/VideoGameSales1/products.csv": "642182426b08987f13b1a30332bf170e8e42042ee8accc64e48fddd5a91340ca",
 "Test_Dataset/Large-scale-or/RA_testing/RA14/AmazonProductsSalesDataset2023/capacity.csv": "bb47bd52d7b613eda4a67c1bebb9a1f786c9b85038ad70a8bf06f2e2b688e0dd",
 "Test_Dataset/Large-scale-or/RA_testing/RA14/AmazonProductsSalesDataset2023/products.csv": "a9f3eb0a19da26b09a6d339bb3334a4fb6a1099ea5de82b364ffb4b400965a38",
 "Test_Dataset/Large-scale-or/RA_testing/RA15/RetailSalesAnalysis1/capacity.csv": "032c92e34b4bff880048b0a681b39ecde1aac4394bbae2eb82badb6d300383a6",
 "Test_Dataset/Large-scale-or/RA_testing/RA15/RetailSalesAnalysis1/products.csv": "d52b806a7e3a75d27314b1b00fa1e5d1f0a1e4d2d1d81a72058696604a75f20d",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture1/RetailSalesAnalysis/capacity.csv": "032c92e34b4bff880048b0a681b39ecde1aac4394bbae2eb82badb6d300383a6",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture1/RetailSalesAnalysis/products.csv": "d52b806a7e3a75d27314b1b00fa1e5d1f0a1e4d2d1d81a72058696604a75f20d",
 "Test_Dataset/Large-scale-or/RA_testing/RA17/CoffeeChainSalesAnalysis1/capacity.csv": "10b89c2fd24d9b7513915fcdb9dfcc435378184cb75dba81d18ec39df936e4da",
 "Test_Dataset/Large-scale-or/RA_testing/RA17/CoffeeChainSalesAnalysis1/products.csv": "2957906447e56e06f2fe259ddd0598df5b3ea3945e3bab13afdb5aa382d211d0",
 "Test_Dataset/Large-scale-or/RA_testing/RA18/BooksSalesAndRatings1/capacity.csv": "cbef27788783ac65a658a6630826c04435b33985af2b5feda528ec981eed019b",
 "Test_Dataset/Large-scale-or/RA_testing/RA18/BooksSalesAndRatings1/products.csv": "a39b65cb5615a233b130186e5ad0a8ca49080fb1bedbee8e492d25103223edff",
 "Test_Dataset/Large-scale-or/RA_testing/RA19/BigMartSalesData1/capacity.csv": "1ccea65da55f787178a86efef6366b7f5c6908727aa22da8a304b6217fd561d0",
 "Test_Dataset/Large-scale-or/RA_testing/RA19/BigMartSalesData1/products.csv": "8cfbf989364a66aee34280fd0a7e86ef95a2362301218ef69be5a77640b3e008",
 "Test_Dataset/Large-scale-or/RA_testing/RA20/BigMartSalesData2/capacity.csv": "a1f6a1f7fc0cea95e69f1a028300e8d75d0d6e0afe8d47953d8177d980eda3be",
 "Test_Dataset/Large-scale-or/RA_testing/RA20/BigMartSalesData2/products.csv": "a560cc91ca1f73321af1e80f678dde661df1117ae67088fb283a7cfdc4e4631d",
 "Test_Dataset/Large-scale-or/RA_testing/RA21/BoatSales1/capacity.csv": "739c15a341822fd704496b387684695638fd7c035f8c65cb6800b739be479548",
 "Test_Dataset/Large-scale-or/RA_testing/RA21/BoatSales1/products.csv": "648584dfb51b73ee43d6fbebf40b1d123bc59caffab17a78b96a680e927e1bc4",
 "Test_Dataset/Large-scale-or/RA_testing/RA22/BoatSales2/capacity.csv": "e49a6ff41156e305723de5aca1edcab6adc4946d197c241ceca8b7c28673dc03",
 "Test_Dataset/Large-scale-or/RA_testing/RA22/BoatSales2/products.csv": "1ab5603c1dcf8f6944ae6ef8f8dfc498a0d2b4a3492c6c0862aeca0a319b2723",
 "Test_Dataset/Large-scale-or/RA_testing/RA16/VideoGameSalesWithRatings1/capacity.csv": "affae625129d2747eaa518f5651a909c9f1325e0085005109ace84254de762ed",
 "Test_Dataset/Large-scale-or/RA_testing/RA16/VideoGameSalesWithRatings1/products.csv": "8d994d7e2a347458fe784d023db8d9cf795b4732f125a13f195400be5dc253c2",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP1/demand.csv": "40207a7fdf31ec6fdf6d8479c6c30067344e172f87b5e96a77266414a6e941e0",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP1/fixed_cost.csv": "92df27b6c9871fc064e749e1d3ed8b658e11b0bf23bae8754fa746aa82312bc9",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP1/transportation_costs.csv": "86408e602473a9642c8c7e0e4e8ec7d686f51adf698b3677ce770b9fdbd30a88",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP2/demand.csv": "8a667ef0c2783b6c04394b871705b3a86541927c370c01a89ebe6a4b1d72b281",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP2/fixed_cost.csv": "740abb6c0c85be936766e14a8a6b734c787d4fa0239fa3b4c15f577153e3e1a7",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP2/transportation_costs.csv": "1282b81bb52b6434832f29748b4e0ed9bafbc4794d385c87dc9178ddca4757e7",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP3/demand.csv": "a55cd981f4c5415b8d6f2076de09b17ccc80e9130fa1ce46ce42008dda320f6c",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP3/fixed_cost.csv": "0778a8d6a66a93b63c4d5b7526b0a584d3be7c6444a674bb57b5b7cc986c6e3e",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP3/transportation_costs.csv": "b1640937d5dc429ed28a237ec38f469b63fb6c6a8a100a441ebd9a71bf4c9481",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP5/demand.csv": "4ed478a31585ed0f8098f469615ae748c6b24692a8e597ecda2c7b4a4f51c802",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP5/fixed_cost.csv": "916b87211250108d2e0f32b643a99667e1d04e8a891f51ab5c18c5f51ce2d254",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP5/transportation_costs.csv": "ad6c9353bb673f0b5931ee2b1df20c00b2080736dbe112a82af965e11f9569c0",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP6/demand.csv": "8fbfdf11ea3723ad548dfcfbc455412a03055744c86c6e6214dbc371729534fb",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP6/fixed_cost.csv": "b741371b4938c2478495ae9bfe536e0df0f5be3a093f6ce9593f042bac64b89a",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP6/transportation_costs.csv": "1f9c3dee57b89cbd7d57956b7bb05ea5aec870368d673a865fa8b66c428bf60f",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP7/demand.csv": "06dd80a13fcf1e7176f4c2761b82b698e1dfcf011b7dde5799ea236a463bd14c",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP7/fixed_cost.csv": "de421fc63e9194e67dc06af7f8d51d83111e3837dcb043f694c5f926e73a278a",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP7/transportation_costs.csv": "08993dd8d77f1990d4887bac7c8f61de5e03afe8719fa95ff15827de5278bac6",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP8/demand.csv": "ec9b8fa194d8d55bdce213f19f055f81a12d4a5ae84da844820c273da37de9e7",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP8/fixed_cost.csv": "e377f19d39860785a32e1f394328e1ed658836d7277d059c7f4363fdc8194ca0",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP8/transportation_costs.csv": "572981048c3a604af854fb49cc71442ae1bf2237055f7ce4ec14250058d01566",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP9/demand.csv": "16ad9778dff705c7de7c4780c7e953c87f62497e5bd4402c182ba190fbbe5941",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP9/fixed_cost.csv": "33c3e53c24f98fa551a5c8394a820db48938edd44b442dbc12f2bf39443f3217",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP9/transportation_costs.csv": "21613a8e9b9fbe531f86bf7ab764d4c1e4d7f1d9742a4c1aa584f672f3c88616",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP10/demand.csv": "4bb1eb6d09329d99391279e86b50c439ff70b3dc16cb959065a6e7422c0e3e72",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP10/fixed_cost.csv": "b046b351d9b64f3e2cca6800097af5b61308e79b75c06bd175bc2c2679ba2170",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP10/transportation_costs.csv": "86ef1b1c5bc7da597220a32598e080d00c3a1edd70d09b48ca97861304862b8b",
 "Test_Dataset/Large-scale-or/AP_testing/AP2/manager_project_costs.csv": "8eb5bbaa573801aeef079026a61ab48f069750af43dc11c36113ad5df6673ee5",
 "Test_Dataset/Large-scale-or/AP_testing/AP3/manager_project_costs.csv": "4fdfa931f220fd28b949e50bba1be1cfc333020a76455f01059e108a3812eefd",
 "Test_Dataset/Large-scale-or/AP_testing/AP4/manager_project_costs.csv": "279314ecbe2c09bf2f895033cc5d586d78f5c8a9c9ac19701c9c86286cfe9a22",
 "Test_Dataset/Large-scale-or/AP_testing/AP5/manager_project_costs.csv": "be1de4df46ef32f13fe8813171c9ac6a788442e8d9a4a6a923b7dab7f1a1a0f3",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture2/41.csv": "ac6d39cb56e43da01b2a120af793ac98eb2cddd26a9d8af9b9fbea1d88c471b2",
 "Test_Dataset/Large-scale-or/Others_example/Others1/42.csv": "377b0bacab66e4d1fd49b1b0649d35c9d3f5b1b7aba584bab568da2b07abc711",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture3/43.csv": "72379f5520f929974f0ccbfd804a0d936e6124f76f9bdf83f50ed340baa513a2",
 "Test_Dataset/Large-scale-or/Others_example/Others2/44.csv": "c0f3e39ceb3b0890b514a8101e9f20f24fe22ed93c2fbb227122cb557631306c",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture4/project.csv": "6f922e537ec513c2889bbb9bbdbc89ac30c344a1d8bd28729f91599fd9435727",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP4/cost.csv": "15bfef9f5b175352c174c8454f0670586356248ff8b8b062de7a116db2ca643c",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP4/warehouse.csv": "e580021ce3cbb416ee1355595f5901c8a18b3309b27f89ff6b65f43a637aa7e7",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP4/demand.csv": "31bea2c0460f809eeb71099e6e5fcb1ad1c914fe9e3a5d5454a30a24ec004bdc",
 "Test_Dataset/Large-scale-or/Others_example/Others3/value.csv": "1c75d4b0cefcfe10ce1dc91fc3e0086b7809ea29333bdd7b986541b52fe9b407",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture5/energy.csv": "51ce25a4b37890137be448c95dbdfcac876c4207052c6a7cf906a56184abb219",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP11/facility_costs.csv": "438c922486a934afc89711b4168e8efa70157daa8bd3f1c12edc0621bebc8291",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP11/shipping_costs.csv": "22e52580a770cceb5b5c11854c30f5b9fcfbafd5d76862a61cba986622149ea6",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP11/demand_requirements.csv": "4d1304af2c062c28454beb69ffd9fa8dd62ba7fda9022555c2314747065d5359",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture6/parameters.csv": "b8edc2a683bf767727bd5c58ae48aa7147d06caa77827ca5b3c9972ec8d6442a",
 "Test_Dataset/Large-scale-or/Others_example/Others4/cost.csv": "4bb18894a791c334d702df07f1d21842f63499a97603f377ade3e8423c0f5c77",
 "Test_Dataset/Large-scale-or/Others_example/Others5/DistanceMatrix.csv": "14f025b663b876e4e19476e2492cd6124c7542f80bd88f15e05511c7c0b77499",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture7/15.csv": "72db476adbe42cc7f36e7e9624efd6b2185d5fb5f9d2f766f194b043b3542015",
 "Test_Dataset/Large-scale-or/Others_example/Others6/18.csv": "f2605fd69187beaa9b5a9c9c7cf8837f6eb1c910404ffdf6eb2033487014042d",
 "Test_Dataset/Large-scale-or/Others_example/Others7/20.csv": "ca2beb1786679ff7a3dfa4f0825c5358f473c042c7e93c85085069ca808cd330",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture8/30-1.csv": "128665bb0c5ba0190b8c83be81ca8ca4932b099fb1c63e298a00bd331819b4a3",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture8/30-2.csv": "e4be0a2bb45608859026e6997d1807c9c9bee071dfa591d8bf199f92450bc7a7",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture9/36-1.csv": "23db8b6295bf887e6d7358a90610ab93fe879b37622acc3aa409cdc566a1f46d",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture9/36-2.csv": "ac8c7b3bd87d1c6fbfa41d18f33e622a5b9f66efd0ac5c627358efa5c172f006",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture9/36-3.csv": "d81967b90f184d62f0972cebda63a2d5101c180dd033b5e7be5997644345e770",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP12/cost.csv": "cd85765800b1faf8bcc06995cb533c7e65ab13709c5abec0f34a47b356ef6a72",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP12/demand.csv": "55ceabde8a8201593949dfb3adca8cd1c81c42c2b990567f034a6a3d52895039",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP13/expanded_customer_service_costs.csv": "a46e4934b67eb043f35c820777ae41907dfc9c023c98c603c088b19379b2300d",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP13/service_centers_fixed_costs.csv": "48f890d274766a82ace6b8e4462af7c5b1a14470e989323144334a982bc85a54",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture10/factory_products_100.csv": "e1aa0e6b716a1fb60101d2fc7c785cf05acc40205535f34e948fefd9bbcc6e3d",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture10/resources_capacities.csv": "c0e94eb4e5eb446fd37b71641f3eff69c681aec8211d62be0955edfedaaf58a5",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture11/courses_42.csv": "000fb30f81d96db2a130b4ba308839597d79c656417f880c01b916e946c4a998",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture12/expanded_cost_matrix.csv": "b306fbd1796980bbe0ea31b89cf78bdde95aafcc585ff8359a933bcb9388cd73",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture12/expanded_destinations.csv": "0a83587726cdb710084d35f77f355cd91d4ba2e52bed3e3f3f6a9b37c6df8c0a",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture12/expanded_sources.csv": "fe5efc318907f3e9802adec2da8a2a12e2579dfe360b1ffe87b23d5466338322",
 "Test_Dataset/Large-scale-or/AP_testing/AP1/cost_12x12.csv": "aa3f2ed7ddb5720955695d1a1bda294dba2c9a5b83204189a2adaf8a16c57053",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture13/workstation_times.csv": "ea7d6e9f80214360197a627c1d082e76de5b5052b158f4ea6e5920cadcc3487c",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture14/resource_limits.csv": "6d018fd46384aa0db74c4f66b15261362c60837070464e14eb57f09313dfce53",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture14/product_resources.csv": "e34650f8c91879d30ae0a31ef0294a927bcb3bf62e66545060cff0be5b26c0d2",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture15/school_capacity.csv": "d19ab57215a092571eb1998c55c925f04078d3445d8068b60172b71e66363681",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture15/neighborhoods_population.csv": "abc5baee5025d1362e03297b15f5f6c3436f50ba218e9d7ac42f0045a9e01ba4",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture15/distance.csv": "ea9243808e680555ab432441ca1152cb162e5753900d573e1c891448bd93816b",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture16/OptionCharacteristics.csv": "8d5b080b391f45b9cd7ad096a3a137bae0425752501aeebf5609257a60c44f76",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture16/Option_AssetReferenceMatrix.csv": "3306cc2e10f21d60fd534e46147816edad4edeaafb7a111e9b0335d3707b702b",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture17/work_days.csv": "a933be3291673fbd19260290a7e2f3735ab84338107c82663d7007b548392519",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP14/PotentialWarehouses_Costs.csv": "0ea3f3850f5d13ae6f6f7fa66505cdd54003c3588eb7d6cf9c92bdf4999ce015",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP14/Stores_Demands.csv": "aa0a2bfbaecc84e1f3e65ee812dee4d5a765febc69b9f68baee32c1f5f880717",
 "Test_Dataset/Large-scale-or/UFLP_testing/UFLP14/TransportationCost.csv": "10baa2e9faf7b332664455ca7c4828b34138ce3ee05bd278979321fdbf25c826",
 "Test_Dataset/Large-scale-or/Others_example/Others8/processing_time_unit.csv": "91405471ec3bfea22ba3380d7e11dcfa02e67d76be7555e9b2ee9811ed8a9d95",
 "Test_Dataset/Large-scale-or/Others_example/Others8/unit_price.csv": "b7f5f0100a8e0cc8b5e03b0daed18ae6746d6b7cd2ffe959019f38bd8c607ce2",
 "Test_Dataset/Large-scale-or/Others_example/Others8/total_working_hours.csv": "717d0cb255300196a34c8eb776f809c3bce2885a667770558a24670becd17f5e",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture18/device_time.csv": "1b75e2e512d8ebf60abef46c67cea6edb892f3df74bc260cf889b6e2b09251b1",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture18/monthly_device_capacity.csv": "5242f2683630d1634b8ed7fbe760f974a4e765315d4129b5c44499e7b7d040e0",
 "Test_Dataset/Large-scale-or/Mixture_testing/Mixture18/unit_product_profits.csv": "62ffc58156fdc5ff6f3176f58566cfc224878eaa1c3ced55c871ed53371dc4d1"
}
""")


In [ ]:
REFERENCE_CERTIFICATES = json.loads(r"""
{
 "OR-001": {
  "problem_id": "OR-001",
  "status": 2,
  "solution_count": 1,
  "objective": 172752.82458839758,
  "objective_bound": 172752.82458839758,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 172752.82458839758,
  "historical_reference_objective_verified": true
 },
 "OR-002": {
  "problem_id": "OR-002",
  "status": 2,
  "solution_count": 1,
  "objective": 6464420.6996351,
  "objective_bound": 6464420.6996351,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 6464420.6996351,
  "historical_reference_objective_verified": true
 },
 "OR-003": {
  "problem_id": "OR-003",
  "status": 2,
  "solution_count": 1,
  "objective": 36778.51094448115,
  "objective_bound": 36778.51094448115,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 36778.51094448115,
  "historical_reference_objective_verified": true
 },
 "OR-004": {
  "problem_id": "OR-004",
  "status": 2,
  "solution_count": 1,
  "objective": 913520.1122932937,
  "objective_bound": 913520.1122932937,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 913520.1122932937,
  "historical_reference_objective_verified": true
 },
 "OR-005": {
  "problem_id": "OR-005",
  "status": 2,
  "solution_count": 1,
  "objective": 2710.9341837593297,
  "objective_bound": 2710.9341837593297,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 2710.9341837593297,
  "historical_reference_objective_verified": true
 },
 "OR-006": {
  "problem_id": "OR-006",
  "status": 2,
  "solution_count": 1,
  "objective": 6340.972286592267,
  "objective_bound": 6340.972286592267,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 6340.972286592267,
  "historical_reference_objective_verified": true
 },
 "OR-007": {
  "problem_id": "OR-007",
  "status": 2,
  "solution_count": 1,
  "objective": 10291.277006657501,
  "objective_bound": 10291.277006657501,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 10291.277006657501,
  "historical_reference_objective_verified": true
 },
 "OR-008": {
  "problem_id": "OR-008",
  "status": 2,
  "solution_count": 1,
  "objective": 660.0,
  "objective_bound": 660.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 660.0,
  "historical_reference_objective_verified": true
 },
 "OR-009": {
  "problem_id": "OR-009",
  "status": 2,
  "solution_count": 1,
  "objective": 238781.51192549666,
  "objective_bound": 238781.51192549666,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 238781.51192549666,
  "historical_reference_objective_verified": true
 },
 "OR-010": {
  "problem_id": "OR-010",
  "status": 2,
  "solution_count": 1,
  "objective": 4003445.2099999995,
  "objective_bound": 4003445.2099999995,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 4003445.2100000004,
  "historical_reference_objective_verified": true
 },
 "OR-011": {
  "problem_id": "OR-011",
  "status": 2,
  "solution_count": 1,
  "objective": 3552260.54,
  "objective_bound": 3552260.54,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 3552260.54,
  "historical_reference_objective_verified": true
 },
 "OR-012": {
  "problem_id": "OR-012",
  "status": 2,
  "solution_count": 1,
  "objective": 83748.37000000001,
  "objective_bound": 83748.37000000001,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 83748.37,
  "historical_reference_objective_verified": true
 },
 "OR-013": {
  "problem_id": "OR-013",
  "status": 2,
  "solution_count": 1,
  "objective": 532.3,
  "objective_bound": 532.3,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 532.3,
  "historical_reference_objective_verified": true
 },
 "OR-014": {
  "problem_id": "OR-014",
  "status": 2,
  "solution_count": 1,
  "objective": 1794269.0,
  "objective_bound": 1794269.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1794269.0,
  "historical_reference_objective_verified": true
 },
 "OR-015": {
  "problem_id": "OR-015",
  "status": 2,
  "solution_count": 1,
  "objective": 29660.0,
  "objective_bound": 29660.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 29660.0,
  "historical_reference_objective_verified": true
 },
 "OR-016": {
  "problem_id": "OR-016",
  "status": 2,
  "solution_count": 1,
  "objective": 1568955.0,
  "objective_bound": 1568955.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1568955.0,
  "historical_reference_objective_verified": true
 },
 "OR-017": {
  "problem_id": "OR-017",
  "status": 2,
  "solution_count": 1,
  "objective": 2015.08,
  "objective_bound": 2015.08,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 2015.08,
  "historical_reference_objective_verified": true
 },
 "OR-018": {
  "problem_id": "OR-018",
  "status": 2,
  "solution_count": 1,
  "objective": 782819438.64,
  "objective_bound": 782819438.64,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 782819438.64,
  "historical_reference_objective_verified": true
 },
 "OR-019": {
  "problem_id": "OR-019",
  "status": 2,
  "solution_count": 1,
  "objective": 7123134.6899999995,
  "objective_bound": 7123134.6899999995,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 7123134.6899999995,
  "historical_reference_objective_verified": true
 },
 "OR-020": {
  "problem_id": "OR-020",
  "status": 2,
  "solution_count": 1,
  "objective": 11258129.67,
  "objective_bound": 11258129.67,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 11258129.67,
  "historical_reference_objective_verified": true
 },
 "OR-021": {
  "problem_id": "OR-021",
  "status": 2,
  "solution_count": 1,
  "objective": 6020981.48,
  "objective_bound": 6020981.48,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 6020981.48,
  "historical_reference_objective_verified": true
 },
 "OR-022": {
  "problem_id": "OR-022",
  "status": 2,
  "solution_count": 1,
  "objective": 4049812.4247,
  "objective_bound": 4049812.4247,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 4049812.4247,
  "historical_reference_objective_verified": true
 },
 "OR-023": {
  "problem_id": "OR-023",
  "status": 2,
  "solution_count": 1,
  "objective": 63628.0,
  "objective_bound": 63628.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 63628.0,
  "historical_reference_objective_verified": true
 },
 "OR-024": {
  "problem_id": "OR-024",
  "status": 2,
  "solution_count": 1,
  "objective": 1264095.5,
  "objective_bound": 1264095.5,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1264095.5,
  "historical_reference_objective_verified": true
 },
 "OR-025": {
  "problem_id": "OR-025",
  "status": 2,
  "solution_count": 1,
  "objective": 5486959.829999999,
  "objective_bound": 5486959.829999999,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 5486959.83,
  "historical_reference_objective_verified": true
 },
 "OR-026": {
  "problem_id": "OR-026",
  "status": 2,
  "solution_count": 1,
  "objective": 8851.75,
  "objective_bound": 8851.75,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 8851.75,
  "historical_reference_objective_verified": true
 },
 "OR-027": {
  "problem_id": "OR-027",
  "status": 2,
  "solution_count": 1,
  "objective": 784297054.11,
  "objective_bound": 784297054.11,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 784297054.1099999,
  "historical_reference_objective_verified": true
 },
 "OR-028": {
  "problem_id": "OR-028",
  "status": 2,
  "solution_count": 1,
  "objective": 285816.0,
  "objective_bound": 285816.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 285816.0,
  "historical_reference_objective_verified": true
 },
 "OR-029": {
  "problem_id": "OR-029",
  "status": 2,
  "solution_count": 1,
  "objective": 5090631.58,
  "objective_bound": 5090631.58,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 5090631.580000001,
  "historical_reference_objective_verified": true
 },
 "OR-030": {
  "problem_id": "OR-030",
  "status": 2,
  "solution_count": 1,
  "objective": 34856.759999999995,
  "objective_bound": 34856.759999999995,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 34856.76,
  "historical_reference_objective_verified": true
 },
 "OR-031": {
  "problem_id": "OR-031",
  "status": 2,
  "solution_count": 1,
  "objective": 67016511.28,
  "objective_bound": 67016511.28,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 67016511.28000001,
  "historical_reference_objective_verified": true
 },
 "OR-032": {
  "problem_id": "OR-032",
  "status": 2,
  "solution_count": 1,
  "objective": 1240406.25,
  "objective_bound": 1240406.25,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1240406.25,
  "historical_reference_objective_verified": true
 },
 "OR-033": {
  "problem_id": "OR-033",
  "status": 2,
  "solution_count": 1,
  "objective": 195506188.0,
  "objective_bound": 195506188.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 195506188.0,
  "historical_reference_objective_verified": true
 },
 "OR-034": {
  "problem_id": "OR-034",
  "status": 2,
  "solution_count": 1,
  "objective": 914600.2500000002,
  "objective_bound": 914600.2500000002,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 914600.25,
  "historical_reference_objective_verified": true
 },
 "OR-035": {
  "problem_id": "OR-035",
  "status": 2,
  "solution_count": 2,
  "objective": 150660.0,
  "objective_bound": 150660.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 150660.0,
  "historical_reference_objective_verified": true
 },
 "OR-036": {
  "problem_id": "OR-036",
  "status": 2,
  "solution_count": 2,
  "objective": 509925.0,
  "objective_bound": 509925.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 509925.0,
  "historical_reference_objective_verified": true
 },
 "OR-037": {
  "problem_id": "OR-037",
  "status": 2,
  "solution_count": 4,
  "objective": 722376.0,
  "objective_bound": 722376.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 722376.0,
  "historical_reference_objective_verified": true
 },
 "OR-038": {
  "problem_id": "OR-038",
  "status": 2,
  "solution_count": 1,
  "objective": 1314500.0,
  "objective_bound": 1314500.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1314500.0,
  "historical_reference_objective_verified": true
 },
 "OR-039": {
  "problem_id": "OR-039",
  "status": 2,
  "solution_count": 3,
  "objective": 429000.0,
  "objective_bound": 429000.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 429000.0,
  "historical_reference_objective_verified": true
 },
 "OR-040": {
  "problem_id": "OR-040",
  "status": 2,
  "solution_count": 2,
  "objective": 74260.0,
  "objective_bound": 74260.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 74260.0,
  "historical_reference_objective_verified": true
 },
 "OR-041": {
  "problem_id": "OR-041",
  "status": 2,
  "solution_count": 2,
  "objective": 3912.0,
  "objective_bound": 3912.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 3912.0,
  "historical_reference_objective_verified": true
 },
 "OR-042": {
  "problem_id": "OR-042",
  "status": 2,
  "solution_count": 2,
  "objective": 81918.0,
  "objective_bound": 81918.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 81918.0,
  "historical_reference_objective_verified": true
 },
 "OR-043": {
  "problem_id": "OR-043",
  "status": 2,
  "solution_count": 2,
  "objective": 13570.0,
  "objective_bound": 13570.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 13570.0,
  "historical_reference_objective_verified": true
 },
 "OR-044": {
  "problem_id": "OR-044",
  "status": 2,
  "solution_count": 2,
  "objective": 8000.0,
  "objective_bound": 8000.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 8000.0,
  "historical_reference_objective_verified": true
 },
 "OR-045": {
  "problem_id": "OR-045",
  "status": 2,
  "solution_count": 2,
  "objective": 885.0,
  "objective_bound": 885.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 885.0,
  "historical_reference_objective_verified": true
 },
 "OR-046": {
  "problem_id": "OR-046",
  "status": 2,
  "solution_count": 1,
  "objective": 212.0,
  "objective_bound": 212.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 212.0,
  "historical_reference_objective_verified": true
 },
 "OR-047": {
  "problem_id": "OR-047",
  "status": 2,
  "solution_count": 5,
  "objective": 5433.0,
  "objective_bound": 5433.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 5433.0,
  "historical_reference_objective_verified": true
 },
 "OR-048": {
  "problem_id": "OR-048",
  "status": 2,
  "solution_count": 1,
  "objective": 635052.0,
  "objective_bound": 635052.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 635052.0,
  "historical_reference_objective_verified": true
 },
 "OR-049": {
  "problem_id": "OR-049",
  "status": 2,
  "solution_count": 4,
  "objective": 85500.0,
  "objective_bound": 85500.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 85500.0,
  "historical_reference_objective_verified": true
 },
 "OR-050": {
  "problem_id": "OR-050",
  "status": 2,
  "solution_count": 4,
  "objective": 80250.0,
  "objective_bound": 80250.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 80250.0,
  "historical_reference_objective_verified": true
 },
 "OR-051": {
  "problem_id": "OR-051",
  "status": 2,
  "solution_count": 3,
  "objective": 594900.0,
  "objective_bound": 594900.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 594900.0,
  "historical_reference_objective_verified": true
 },
 "OR-052": {
  "problem_id": "OR-052",
  "status": 2,
  "solution_count": 3,
  "objective": 30240.0,
  "objective_bound": 30240.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 30240.0,
  "historical_reference_objective_verified": true
 },
 "OR-053": {
  "problem_id": "OR-053",
  "status": 2,
  "solution_count": 3,
  "objective": 41040.0,
  "objective_bound": 41040.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 41040.0,
  "historical_reference_objective_verified": true
 },
 "OR-054": {
  "problem_id": "OR-054",
  "status": 2,
  "solution_count": 3,
  "objective": 98670.0,
  "objective_bound": 98670.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 98670.0,
  "historical_reference_objective_verified": true
 },
 "OR-055": {
  "problem_id": "OR-055",
  "status": 2,
  "solution_count": 4,
  "objective": 27088640.0,
  "objective_bound": 27088640.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 27088640.0,
  "historical_reference_objective_verified": true
 },
 "OR-056": {
  "problem_id": "OR-056",
  "status": 2,
  "solution_count": 5,
  "objective": 55642671.0,
  "objective_bound": 55642671.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 55642671.0,
  "historical_reference_objective_verified": true
 },
 "OR-057": {
  "problem_id": "OR-057",
  "status": 2,
  "solution_count": 2,
  "objective": 12502.0,
  "objective_bound": 12502.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 12502.0,
  "historical_reference_objective_verified": true
 },
 "OR-058": {
  "problem_id": "OR-058",
  "status": 2,
  "solution_count": 2,
  "objective": 35950.649999999994,
  "objective_bound": 35950.64999999999,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 35950.649999999994,
  "historical_reference_objective_verified": true
 },
 "OR-059": {
  "problem_id": "OR-059",
  "status": 2,
  "solution_count": 1,
  "objective": 1862205008776.5698,
  "objective_bound": 1862205007990.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 4.223862681668492e-10,
  "historical_reference_objective": 1862205008776.57,
  "historical_reference_objective_verified": true
 },
 "OR-060": {
  "problem_id": "OR-060",
  "status": 2,
  "solution_count": 4,
  "objective": 8643.2,
  "objective_bound": 8643.2,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 8643.2,
  "historical_reference_objective_verified": true
 },
 "OR-061": {
  "problem_id": "OR-061",
  "status": 2,
  "solution_count": 3,
  "objective": 10199.960000000001,
  "objective_bound": 10199.960000000001,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 10199.960000000001,
  "historical_reference_objective_verified": true
 },
 "OR-062": {
  "problem_id": "OR-062",
  "status": 2,
  "solution_count": 2,
  "objective": 172612.46,
  "objective_bound": 172612.46,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": null,
  "historical_reference_objective_verified": null,
  "purpose": "reference_only_no_generated_incumbent_available"
 },
 "OR-063": {
  "problem_id": "OR-063",
  "status": 2,
  "solution_count": 2,
  "objective": 778095.44,
  "objective_bound": 778095.44,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 778095.44,
  "historical_reference_objective_verified": true
 },
 "OR-064": {
  "problem_id": "OR-064",
  "status": 2,
  "solution_count": 5,
  "objective": 18461.11,
  "objective_bound": 18461.109999999997,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 18461.11,
  "historical_reference_objective_verified": true
 },
 "OR-065": {
  "problem_id": "OR-065",
  "status": 2,
  "solution_count": 2,
  "objective": 317318.5,
  "objective_bound": 317318.5,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 317318.50000000006,
  "historical_reference_objective_verified": true
 },
 "OR-066": {
  "problem_id": "OR-066",
  "status": 2,
  "solution_count": 1,
  "objective": 11396.51,
  "objective_bound": 11396.51,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 11396.51,
  "historical_reference_objective_verified": true
 },
 "OR-067": {
  "problem_id": "OR-067",
  "status": 2,
  "solution_count": 1,
  "objective": 9000.0,
  "objective_bound": 9000.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 9000.0,
  "historical_reference_objective_verified": true
 },
 "OR-068": {
  "problem_id": "OR-068",
  "status": 2,
  "solution_count": 1,
  "objective": 9089.0,
  "objective_bound": 9089.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 9089.0,
  "historical_reference_objective_verified": true
 },
 "OR-069": {
  "problem_id": "OR-069",
  "status": 2,
  "solution_count": 1,
  "objective": 4903.0,
  "objective_bound": 4903.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 4903.0,
  "historical_reference_objective_verified": true
 },
 "OR-070": {
  "problem_id": "OR-070",
  "status": 2,
  "solution_count": 1,
  "objective": 8809.0,
  "objective_bound": 8809.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 8809.0,
  "historical_reference_objective_verified": true
 },
 "OR-071": {
  "problem_id": "OR-071",
  "status": 2,
  "solution_count": 1,
  "objective": 24877.394636015328,
  "objective_bound": 24877.394636015328,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 24877.394636015324,
  "historical_reference_objective_verified": true
 },
 "OR-072": {
  "problem_id": "OR-072",
  "status": 2,
  "solution_count": 1,
  "objective": 320.0,
  "objective_bound": 320.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 320.0,
  "historical_reference_objective_verified": true
 },
 "OR-073": {
  "problem_id": "OR-073",
  "status": 2,
  "solution_count": 1,
  "objective": 1146.5665024630537,
  "objective_bound": 1146.5665024630537,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1146.5665024630543,
  "historical_reference_objective_verified": true
 },
 "OR-074": {
  "problem_id": "OR-074",
  "status": 2,
  "solution_count": 2,
  "objective": 25.0,
  "objective_bound": 25.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 25.0,
  "historical_reference_objective_verified": true
 },
 "OR-075": {
  "problem_id": "OR-075",
  "status": 2,
  "solution_count": 4,
  "objective": 1514.0,
  "objective_bound": 1514.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1514.0,
  "historical_reference_objective_verified": true
 },
 "OR-076": {
  "problem_id": "OR-076",
  "status": 2,
  "solution_count": 2,
  "objective": 103090.0,
  "objective_bound": 103090.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 103090.0,
  "historical_reference_objective_verified": true
 },
 "OR-077": {
  "problem_id": "OR-077",
  "status": 2,
  "solution_count": 2,
  "objective": 133.0,
  "objective_bound": 133.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 133.0,
  "historical_reference_objective_verified": true
 },
 "OR-078": {
  "problem_id": "OR-078",
  "status": 2,
  "solution_count": 2,
  "objective": 152.76,
  "objective_bound": 152.76,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 152.76,
  "historical_reference_objective_verified": true
 },
 "OR-079": {
  "problem_id": "OR-079",
  "status": 2,
  "solution_count": 5,
  "objective": 2965.0,
  "objective_bound": 2965.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 2965.0,
  "historical_reference_objective_verified": true
 },
 "OR-080": {
  "problem_id": "OR-080",
  "status": 2,
  "solution_count": 7,
  "objective": 14300.0,
  "objective_bound": 14300.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 14300.0,
  "historical_reference_objective_verified": true
 },
 "OR-081": {
  "problem_id": "OR-081",
  "status": 2,
  "solution_count": 1,
  "objective": 3.910164341169253,
  "objective_bound": 3.910164341169253,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 3.910164341169253,
  "historical_reference_objective_verified": true
 },
 "OR-082": {
  "problem_id": "OR-082",
  "status": 2,
  "solution_count": 1,
  "objective": 199.0,
  "objective_bound": 199.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 199.0,
  "historical_reference_objective_verified": true
 },
 "OR-083": {
  "problem_id": "OR-083",
  "status": 2,
  "solution_count": 2,
  "objective": 34.0,
  "objective_bound": 34.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 34.0,
  "historical_reference_objective_verified": true
 },
 "OR-084": {
  "problem_id": "OR-084",
  "status": 2,
  "solution_count": 4,
  "objective": 22.85,
  "objective_bound": 22.85,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 22.85,
  "historical_reference_objective_verified": true
 },
 "OR-085": {
  "problem_id": "OR-085",
  "status": 2,
  "solution_count": 10,
  "objective": 419.0,
  "objective_bound": 419.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 419.0,
  "historical_reference_objective_verified": true
 },
 "OR-086": {
  "problem_id": "OR-086",
  "status": 2,
  "solution_count": 1,
  "objective": 4500.0,
  "objective_bound": 4500.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 4500.0,
  "historical_reference_objective_verified": true
 },
 "OR-087": {
  "problem_id": "OR-087",
  "status": 2,
  "solution_count": 2,
  "objective": 6243055.959999999,
  "objective_bound": 6243055.959999999,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 6243055.960000001,
  "historical_reference_objective_verified": true
 },
 "OR-088": {
  "problem_id": "OR-088",
  "status": 2,
  "solution_count": 7,
  "objective": 150992.5,
  "objective_bound": 150992.5,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 150992.5,
  "historical_reference_objective_verified": true
 },
 "OR-089": {
  "problem_id": "OR-089",
  "status": 2,
  "solution_count": 4,
  "objective": 1555.4999999999993,
  "objective_bound": 1555.4999999999993,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1555.5,
  "historical_reference_objective_verified": true
 },
 "OR-090": {
  "problem_id": "OR-090",
  "status": 2,
  "solution_count": 9,
  "objective": 108659.80000000002,
  "objective_bound": 108667.06586613023,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 6.686802414703586e-05,
  "historical_reference_objective": 108659.8,
  "historical_reference_objective_verified": true
 },
 "OR-091": {
  "problem_id": "OR-091",
  "status": 2,
  "solution_count": 2,
  "objective": 421.0,
  "objective_bound": 421.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 421.0,
  "historical_reference_objective_verified": true
 },
 "OR-092": {
  "problem_id": "OR-092",
  "status": 2,
  "solution_count": 1,
  "objective": 3148.9299999999994,
  "objective_bound": 3148.9299999999994,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 3148.9300000000003,
  "historical_reference_objective_verified": true
 },
 "OR-093": {
  "problem_id": "OR-093",
  "status": 2,
  "solution_count": 2,
  "objective": 1577.6,
  "objective_bound": 1577.6,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1577.6,
  "historical_reference_objective_verified": true
 },
 "OR-094": {
  "problem_id": "OR-094",
  "status": 2,
  "solution_count": 6,
  "objective": 0.599999999999909,
  "objective_bound": 0.599999999999909,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 0.599999999999909,
  "historical_reference_objective_verified": true
 },
 "OR-095": {
  "problem_id": "OR-095",
  "status": 2,
  "solution_count": 1,
  "objective": 1344480.0,
  "objective_bound": 1344480.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1344480.0,
  "historical_reference_objective_verified": true
 },
 "OR-096": {
  "problem_id": "OR-096",
  "status": 2,
  "solution_count": 3,
  "objective": 5044.0,
  "objective_bound": 5044.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 5044.0,
  "historical_reference_objective_verified": true
 },
 "OR-097": {
  "problem_id": "OR-097",
  "status": 2,
  "solution_count": 8,
  "objective": 6.0,
  "objective_bound": 6.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 6.0,
  "historical_reference_objective_verified": true
 },
 "OR-098": {
  "problem_id": "OR-098",
  "status": 2,
  "solution_count": 1,
  "objective": 0.0,
  "objective_bound": 0.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 0.0,
  "historical_reference_objective_verified": true
 },
 "OR-099": {
  "problem_id": "OR-099",
  "status": 2,
  "solution_count": 5,
  "objective": 10419.0,
  "objective_bound": 10419.0,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 10419.0,
  "historical_reference_objective_verified": true
 },
 "OR-100": {
  "problem_id": "OR-100",
  "status": 2,
  "solution_count": 1,
  "objective": 1382200.7222841426,
  "objective_bound": 1382200.7222841426,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 1382200.7222841426,
  "historical_reference_objective_verified": true
 },
 "OR-101": {
  "problem_id": "OR-101",
  "status": 2,
  "solution_count": 1,
  "objective": 25820.15355597092,
  "objective_bound": 25820.15355597092,
  "configured_mip_gap": 0.0001,
  "achieved_mip_gap": 0.0,
  "historical_reference_objective": 25820.15355597092,
  "historical_reference_objective_verified": true
 }
}
""")


In [ ]:
IDENTITY_ARTIFACTS = {}


In [ ]:
GPT41_IDENTITIES = json.loads(r"""
{
 "OR-001": {
  "source_code_sha256": "5dea27e38b524dd455e22096203942815c4e8fd91439928ab5afe5555a278a12",
  "original_solution_sha256": "b6d1e0d04a8fbbf0ccbf6d0f0246d7b37ae46aa0bcfc86c385fbae4a1694d871",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S1,C11]"
   ],
   [
    "C11",
    "x_vars[S1,C12]"
   ],
   [
    "C12",
    "x_vars[S1,C13]"
   ],
   [
    "C13",
    "x_vars[S1,C14]"
   ],
   [
    "C14",
    "x_vars[S1,C15]"
   ],
   [
    "C15",
    "x_vars[S1,C16]"
   ],
   [
    "C16",
    "x_vars[S1,C17]"
   ],
   [
    "C17",
    "x_vars[S1,C18]"
   ],
   [
    "C18",
    "x_vars[S2,C1]"
   ],
   [
    "C19",
    "x_vars[S2,C2]"
   ],
   [
    "C20",
    "x_vars[S2,C3]"
   ],
   [
    "C21",
    "x_vars[S2,C4]"
   ],
   [
    "C22",
    "x_vars[S2,C5]"
   ],
   [
    "C23",
    "x_vars[S2,C6]"
   ],
   [
    "C24",
    "x_vars[S2,C7]"
   ],
   [
    "C25",
    "x_vars[S2,C8]"
   ],
   [
    "C26",
    "x_vars[S2,C9]"
   ],
   [
    "C27",
    "x_vars[S2,C10]"
   ],
   [
    "C28",
    "x_vars[S2,C11]"
   ],
   [
    "C29",
    "x_vars[S2,C12]"
   ],
   [
    "C30",
    "x_vars[S2,C13]"
   ],
   [
    "C31",
    "x_vars[S2,C14]"
   ],
   [
    "C32",
    "x_vars[S2,C15]"
   ],
   [
    "C33",
    "x_vars[S2,C16]"
   ],
   [
    "C34",
    "x_vars[S2,C17]"
   ],
   [
    "C35",
    "x_vars[S2,C18]"
   ],
   [
    "C36",
    "x_vars[S3,C1]"
   ],
   [
    "C37",
    "x_vars[S3,C2]"
   ],
   [
    "C38",
    "x_vars[S3,C3]"
   ],
   [
    "C39",
    "x_vars[S3,C4]"
   ],
   [
    "C40",
    "x_vars[S3,C5]"
   ],
   [
    "C41",
    "x_vars[S3,C6]"
   ],
   [
    "C42",
    "x_vars[S3,C7]"
   ],
   [
    "C43",
    "x_vars[S3,C8]"
   ],
   [
    "C44",
    "x_vars[S3,C9]"
   ],
   [
    "C45",
    "x_vars[S3,C10]"
   ],
   [
    "C46",
    "x_vars[S3,C11]"
   ],
   [
    "C47",
    "x_vars[S3,C12]"
   ],
   [
    "C48",
    "x_vars[S3,C13]"
   ],
   [
    "C49",
    "x_vars[S3,C14]"
   ],
   [
    "C50",
    "x_vars[S3,C15]"
   ],
   [
    "C51",
    "x_vars[S3,C16]"
   ],
   [
    "C52",
    "x_vars[S3,C17]"
   ],
   [
    "C53",
    "x_vars[S3,C18]"
   ],
   [
    "C54",
    "x_vars[S4,C1]"
   ],
   [
    "C55",
    "x_vars[S4,C2]"
   ],
   [
    "C56",
    "x_vars[S4,C3]"
   ],
   [
    "C57",
    "x_vars[S4,C4]"
   ],
   [
    "C58",
    "x_vars[S4,C5]"
   ],
   [
    "C59",
    "x_vars[S4,C6]"
   ],
   [
    "C60",
    "x_vars[S4,C7]"
   ],
   [
    "C61",
    "x_vars[S4,C8]"
   ],
   [
    "C62",
    "x_vars[S4,C9]"
   ],
   [
    "C63",
    "x_vars[S4,C10]"
   ],
   [
    "C64",
    "x_vars[S4,C11]"
   ],
   [
    "C65",
    "x_vars[S4,C12]"
   ],
   [
    "C66",
    "x_vars[S4,C13]"
   ],
   [
    "C67",
    "x_vars[S4,C14]"
   ],
   [
    "C68",
    "x_vars[S4,C15]"
   ],
   [
    "C69",
    "x_vars[S4,C16]"
   ],
   [
    "C70",
    "x_vars[S4,C17]"
   ],
   [
    "C71",
    "x_vars[S4,C18]"
   ],
   [
    "C72",
    "x_vars[S5,C1]"
   ],
   [
    "C73",
    "x_vars[S5,C2]"
   ],
   [
    "C74",
    "x_vars[S5,C3]"
   ],
   [
    "C75",
    "x_vars[S5,C4]"
   ],
   [
    "C76",
    "x_vars[S5,C5]"
   ],
   [
    "C77",
    "x_vars[S5,C6]"
   ],
   [
    "C78",
    "x_vars[S5,C7]"
   ],
   [
    "C79",
    "x_vars[S5,C8]"
   ],
   [
    "C80",
    "x_vars[S5,C9]"
   ],
   [
    "C81",
    "x_vars[S5,C10]"
   ],
   [
    "C82",
    "x_vars[S5,C11]"
   ],
   [
    "C83",
    "x_vars[S5,C12]"
   ],
   [
    "C84",
    "x_vars[S5,C13]"
   ],
   [
    "C85",
    "x_vars[S5,C14]"
   ],
   [
    "C86",
    "x_vars[S5,C15]"
   ],
   [
    "C87",
    "x_vars[S5,C16]"
   ],
   [
    "C88",
    "x_vars[S5,C17]"
   ],
   [
    "C89",
    "x_vars[S5,C18]"
   ],
   [
    "C90",
    "x_vars[S6,C1]"
   ],
   [
    "C91",
    "x_vars[S6,C2]"
   ],
   [
    "C92",
    "x_vars[S6,C3]"
   ],
   [
    "C93",
    "x_vars[S6,C4]"
   ],
   [
    "C94",
    "x_vars[S6,C5]"
   ],
   [
    "C95",
    "x_vars[S6,C6]"
   ],
   [
    "C96",
    "x_vars[S6,C7]"
   ],
   [
    "C97",
    "x_vars[S6,C8]"
   ],
   [
    "C98",
    "x_vars[S6,C9]"
   ],
   [
    "C99",
    "x_vars[S6,C10]"
   ],
   [
    "C100",
    "x_vars[S6,C11]"
   ],
   [
    "C101",
    "x_vars[S6,C12]"
   ],
   [
    "C102",
    "x_vars[S6,C13]"
   ],
   [
    "C103",
    "x_vars[S6,C14]"
   ],
   [
    "C104",
    "x_vars[S6,C15]"
   ],
   [
    "C105",
    "x_vars[S6,C16]"
   ],
   [
    "C106",
    "x_vars[S6,C17]"
   ],
   [
    "C107",
    "x_vars[S6,C18]"
   ],
   [
    "C108",
    "x_vars[S7,C1]"
   ],
   [
    "C109",
    "x_vars[S7,C2]"
   ],
   [
    "C110",
    "x_vars[S7,C3]"
   ],
   [
    "C111",
    "x_vars[S7,C4]"
   ],
   [
    "C112",
    "x_vars[S7,C5]"
   ],
   [
    "C113",
    "x_vars[S7,C6]"
   ],
   [
    "C114",
    "x_vars[S7,C7]"
   ],
   [
    "C115",
    "x_vars[S7,C8]"
   ],
   [
    "C116",
    "x_vars[S7,C9]"
   ],
   [
    "C117",
    "x_vars[S7,C10]"
   ],
   [
    "C118",
    "x_vars[S7,C11]"
   ],
   [
    "C119",
    "x_vars[S7,C12]"
   ],
   [
    "C120",
    "x_vars[S7,C13]"
   ],
   [
    "C121",
    "x_vars[S7,C14]"
   ],
   [
    "C122",
    "x_vars[S7,C15]"
   ],
   [
    "C123",
    "x_vars[S7,C16]"
   ],
   [
    "C124",
    "x_vars[S7,C17]"
   ],
   [
    "C125",
    "x_vars[S7,C18]"
   ],
   [
    "C126",
    "x_vars[S8,C1]"
   ],
   [
    "C127",
    "x_vars[S8,C2]"
   ],
   [
    "C128",
    "x_vars[S8,C3]"
   ],
   [
    "C129",
    "x_vars[S8,C4]"
   ],
   [
    "C130",
    "x_vars[S8,C5]"
   ],
   [
    "C131",
    "x_vars[S8,C6]"
   ],
   [
    "C132",
    "x_vars[S8,C7]"
   ],
   [
    "C133",
    "x_vars[S8,C8]"
   ],
   [
    "C134",
    "x_vars[S8,C9]"
   ],
   [
    "C135",
    "x_vars[S8,C10]"
   ],
   [
    "C136",
    "x_vars[S8,C11]"
   ],
   [
    "C137",
    "x_vars[S8,C12]"
   ],
   [
    "C138",
    "x_vars[S8,C13]"
   ],
   [
    "C139",
    "x_vars[S8,C14]"
   ],
   [
    "C140",
    "x_vars[S8,C15]"
   ],
   [
    "C141",
    "x_vars[S8,C16]"
   ],
   [
    "C142",
    "x_vars[S8,C17]"
   ],
   [
    "C143",
    "x_vars[S8,C18]"
   ],
   [
    "C144",
    "x_vars[S9,C1]"
   ],
   [
    "C145",
    "x_vars[S9,C2]"
   ],
   [
    "C146",
    "x_vars[S9,C3]"
   ],
   [
    "C147",
    "x_vars[S9,C4]"
   ],
   [
    "C148",
    "x_vars[S9,C5]"
   ],
   [
    "C149",
    "x_vars[S9,C6]"
   ],
   [
    "C150",
    "x_vars[S9,C7]"
   ],
   [
    "C151",
    "x_vars[S9,C8]"
   ],
   [
    "C152",
    "x_vars[S9,C9]"
   ],
   [
    "C153",
    "x_vars[S9,C10]"
   ],
   [
    "C154",
    "x_vars[S9,C11]"
   ],
   [
    "C155",
    "x_vars[S9,C12]"
   ],
   [
    "C156",
    "x_vars[S9,C13]"
   ],
   [
    "C157",
    "x_vars[S9,C14]"
   ],
   [
    "C158",
    "x_vars[S9,C15]"
   ],
   [
    "C159",
    "x_vars[S9,C16]"
   ],
   [
    "C160",
    "x_vars[S9,C17]"
   ],
   [
    "C161",
    "x_vars[S9,C18]"
   ],
   [
    "C162",
    "x_vars[S10,C1]"
   ],
   [
    "C163",
    "x_vars[S10,C2]"
   ],
   [
    "C164",
    "x_vars[S10,C3]"
   ],
   [
    "C165",
    "x_vars[S10,C4]"
   ],
   [
    "C166",
    "x_vars[S10,C5]"
   ],
   [
    "C167",
    "x_vars[S10,C6]"
   ],
   [
    "C168",
    "x_vars[S10,C7]"
   ],
   [
    "C169",
    "x_vars[S10,C8]"
   ],
   [
    "C170",
    "x_vars[S10,C9]"
   ],
   [
    "C171",
    "x_vars[S10,C10]"
   ],
   [
    "C172",
    "x_vars[S10,C11]"
   ],
   [
    "C173",
    "x_vars[S10,C12]"
   ],
   [
    "C174",
    "x_vars[S10,C13]"
   ],
   [
    "C175",
    "x_vars[S10,C14]"
   ],
   [
    "C176",
    "x_vars[S10,C15]"
   ],
   [
    "C177",
    "x_vars[S10,C16]"
   ],
   [
    "C178",
    "x_vars[S10,C17]"
   ],
   [
    "C179",
    "x_vars[S10,C18]"
   ],
   [
    "C180",
    "x_vars[S11,C1]"
   ],
   [
    "C181",
    "x_vars[S11,C2]"
   ],
   [
    "C182",
    "x_vars[S11,C3]"
   ],
   [
    "C183",
    "x_vars[S11,C4]"
   ],
   [
    "C184",
    "x_vars[S11,C5]"
   ],
   [
    "C185",
    "x_vars[S11,C6]"
   ],
   [
    "C186",
    "x_vars[S11,C7]"
   ],
   [
    "C187",
    "x_vars[S11,C8]"
   ],
   [
    "C188",
    "x_vars[S11,C9]"
   ],
   [
    "C189",
    "x_vars[S11,C10]"
   ],
   [
    "C190",
    "x_vars[S11,C11]"
   ],
   [
    "C191",
    "x_vars[S11,C12]"
   ],
   [
    "C192",
    "x_vars[S11,C13]"
   ],
   [
    "C193",
    "x_vars[S11,C14]"
   ],
   [
    "C194",
    "x_vars[S11,C15]"
   ],
   [
    "C195",
    "x_vars[S11,C16]"
   ],
   [
    "C196",
    "x_vars[S11,C17]"
   ],
   [
    "C197",
    "x_vars[S11,C18]"
   ],
   [
    "C198",
    "x_vars[S12,C1]"
   ],
   [
    "C199",
    "x_vars[S12,C2]"
   ],
   [
    "C200",
    "x_vars[S12,C3]"
   ],
   [
    "C201",
    "x_vars[S12,C4]"
   ],
   [
    "C202",
    "x_vars[S12,C5]"
   ],
   [
    "C203",
    "x_vars[S12,C6]"
   ],
   [
    "C204",
    "x_vars[S12,C7]"
   ],
   [
    "C205",
    "x_vars[S12,C8]"
   ],
   [
    "C206",
    "x_vars[S12,C9]"
   ],
   [
    "C207",
    "x_vars[S12,C10]"
   ],
   [
    "C208",
    "x_vars[S12,C11]"
   ],
   [
    "C209",
    "x_vars[S12,C12]"
   ],
   [
    "C210",
    "x_vars[S12,C13]"
   ],
   [
    "C211",
    "x_vars[S12,C14]"
   ],
   [
    "C212",
    "x_vars[S12,C15]"
   ],
   [
    "C213",
    "x_vars[S12,C16]"
   ],
   [
    "C214",
    "x_vars[S12,C17]"
   ],
   [
    "C215",
    "x_vars[S12,C18]"
   ],
   [
    "C216",
    "x_vars[S13,C1]"
   ],
   [
    "C217",
    "x_vars[S13,C2]"
   ],
   [
    "C218",
    "x_vars[S13,C3]"
   ],
   [
    "C219",
    "x_vars[S13,C4]"
   ],
   [
    "C220",
    "x_vars[S13,C5]"
   ],
   [
    "C221",
    "x_vars[S13,C6]"
   ],
   [
    "C222",
    "x_vars[S13,C7]"
   ],
   [
    "C223",
    "x_vars[S13,C8]"
   ],
   [
    "C224",
    "x_vars[S13,C9]"
   ],
   [
    "C225",
    "x_vars[S13,C10]"
   ],
   [
    "C226",
    "x_vars[S13,C11]"
   ],
   [
    "C227",
    "x_vars[S13,C12]"
   ],
   [
    "C228",
    "x_vars[S13,C13]"
   ],
   [
    "C229",
    "x_vars[S13,C14]"
   ],
   [
    "C230",
    "x_vars[S13,C15]"
   ],
   [
    "C231",
    "x_vars[S13,C16]"
   ],
   [
    "C232",
    "x_vars[S13,C17]"
   ],
   [
    "C233",
    "x_vars[S13,C18]"
   ],
   [
    "C234",
    "x_vars[S14,C1]"
   ],
   [
    "C235",
    "x_vars[S14,C2]"
   ],
   [
    "C236",
    "x_vars[S14,C3]"
   ],
   [
    "C237",
    "x_vars[S14,C4]"
   ],
   [
    "C238",
    "x_vars[S14,C5]"
   ],
   [
    "C239",
    "x_vars[S14,C6]"
   ],
   [
    "C240",
    "x_vars[S14,C7]"
   ],
   [
    "C241",
    "x_vars[S14,C8]"
   ],
   [
    "C242",
    "x_vars[S14,C9]"
   ],
   [
    "C243",
    "x_vars[S14,C10]"
   ],
   [
    "C244",
    "x_vars[S14,C11]"
   ],
   [
    "C245",
    "x_vars[S14,C12]"
   ],
   [
    "C246",
    "x_vars[S14,C13]"
   ],
   [
    "C247",
    "x_vars[S14,C14]"
   ],
   [
    "C248",
    "x_vars[S14,C15]"
   ],
   [
    "C249",
    "x_vars[S14,C16]"
   ],
   [
    "C250",
    "x_vars[S14,C17]"
   ],
   [
    "C251",
    "x_vars[S14,C18]"
   ],
   [
    "C252",
    "x_vars[S15,C1]"
   ],
   [
    "C253",
    "x_vars[S15,C2]"
   ],
   [
    "C254",
    "x_vars[S15,C3]"
   ],
   [
    "C255",
    "x_vars[S15,C4]"
   ],
   [
    "C256",
    "x_vars[S15,C5]"
   ],
   [
    "C257",
    "x_vars[S15,C6]"
   ],
   [
    "C258",
    "x_vars[S15,C7]"
   ],
   [
    "C259",
    "x_vars[S15,C8]"
   ],
   [
    "C260",
    "x_vars[S15,C9]"
   ],
   [
    "C261",
    "x_vars[S15,C10]"
   ],
   [
    "C262",
    "x_vars[S15,C11]"
   ],
   [
    "C263",
    "x_vars[S15,C12]"
   ],
   [
    "C264",
    "x_vars[S15,C13]"
   ],
   [
    "C265",
    "x_vars[S15,C14]"
   ],
   [
    "C266",
    "x_vars[S15,C15]"
   ],
   [
    "C267",
    "x_vars[S15,C16]"
   ],
   [
    "C268",
    "x_vars[S15,C17]"
   ],
   [
    "C269",
    "x_vars[S15,C18]"
   ],
   [
    "C270",
    "x_vars[S16,C1]"
   ],
   [
    "C271",
    "x_vars[S16,C2]"
   ],
   [
    "C272",
    "x_vars[S16,C3]"
   ],
   [
    "C273",
    "x_vars[S16,C4]"
   ],
   [
    "C274",
    "x_vars[S16,C5]"
   ],
   [
    "C275",
    "x_vars[S16,C6]"
   ],
   [
    "C276",
    "x_vars[S16,C7]"
   ],
   [
    "C277",
    "x_vars[S16,C8]"
   ],
   [
    "C278",
    "x_vars[S16,C9]"
   ],
   [
    "C279",
    "x_vars[S16,C10]"
   ],
   [
    "C280",
    "x_vars[S16,C11]"
   ],
   [
    "C281",
    "x_vars[S16,C12]"
   ],
   [
    "C282",
    "x_vars[S16,C13]"
   ],
   [
    "C283",
    "x_vars[S16,C14]"
   ],
   [
    "C284",
    "x_vars[S16,C15]"
   ],
   [
    "C285",
    "x_vars[S16,C16]"
   ],
   [
    "C286",
    "x_vars[S16,C17]"
   ],
   [
    "C287",
    "x_vars[S16,C18]"
   ],
   [
    "C288",
    "x_vars[S17,C1]"
   ],
   [
    "C289",
    "x_vars[S17,C2]"
   ],
   [
    "C290",
    "x_vars[S17,C3]"
   ],
   [
    "C291",
    "x_vars[S17,C4]"
   ],
   [
    "C292",
    "x_vars[S17,C5]"
   ],
   [
    "C293",
    "x_vars[S17,C6]"
   ],
   [
    "C294",
    "x_vars[S17,C7]"
   ],
   [
    "C295",
    "x_vars[S17,C8]"
   ],
   [
    "C296",
    "x_vars[S17,C9]"
   ],
   [
    "C297",
    "x_vars[S17,C10]"
   ],
   [
    "C298",
    "x_vars[S17,C11]"
   ],
   [
    "C299",
    "x_vars[S17,C12]"
   ],
   [
    "C300",
    "x_vars[S17,C13]"
   ],
   [
    "C301",
    "x_vars[S17,C14]"
   ],
   [
    "C302",
    "x_vars[S17,C15]"
   ],
   [
    "C303",
    "x_vars[S17,C16]"
   ],
   [
    "C304",
    "x_vars[S17,C17]"
   ],
   [
    "C305",
    "x_vars[S17,C18]"
   ],
   [
    "C306",
    "x_vars[S18,C1]"
   ],
   [
    "C307",
    "x_vars[S18,C2]"
   ],
   [
    "C308",
    "x_vars[S18,C3]"
   ],
   [
    "C309",
    "x_vars[S18,C4]"
   ],
   [
    "C310",
    "x_vars[S18,C5]"
   ],
   [
    "C311",
    "x_vars[S18,C6]"
   ],
   [
    "C312",
    "x_vars[S18,C7]"
   ],
   [
    "C313",
    "x_vars[S18,C8]"
   ],
   [
    "C314",
    "x_vars[S18,C9]"
   ],
   [
    "C315",
    "x_vars[S18,C10]"
   ],
   [
    "C316",
    "x_vars[S18,C11]"
   ],
   [
    "C317",
    "x_vars[S18,C12]"
   ],
   [
    "C318",
    "x_vars[S18,C13]"
   ],
   [
    "C319",
    "x_vars[S18,C14]"
   ],
   [
    "C320",
    "x_vars[S18,C15]"
   ],
   [
    "C321",
    "x_vars[S18,C16]"
   ],
   [
    "C322",
    "x_vars[S18,C17]"
   ],
   [
    "C323",
    "x_vars[S18,C18]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_0_10": "x_vars[S1,C11]",
   "x_0_11": "x_vars[S1,C12]",
   "x_0_12": "x_vars[S1,C13]",
   "x_0_13": "x_vars[S1,C14]",
   "x_0_14": "x_vars[S1,C15]",
   "x_0_15": "x_vars[S1,C16]",
   "x_0_16": "x_vars[S1,C17]",
   "x_0_17": "x_vars[S1,C18]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_1_10": "x_vars[S2,C11]",
   "x_1_11": "x_vars[S2,C12]",
   "x_1_12": "x_vars[S2,C13]",
   "x_1_13": "x_vars[S2,C14]",
   "x_1_14": "x_vars[S2,C15]",
   "x_1_15": "x_vars[S2,C16]",
   "x_1_16": "x_vars[S2,C17]",
   "x_1_17": "x_vars[S2,C18]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_2_10": "x_vars[S3,C11]",
   "x_2_11": "x_vars[S3,C12]",
   "x_2_12": "x_vars[S3,C13]",
   "x_2_13": "x_vars[S3,C14]",
   "x_2_14": "x_vars[S3,C15]",
   "x_2_15": "x_vars[S3,C16]",
   "x_2_16": "x_vars[S3,C17]",
   "x_2_17": "x_vars[S3,C18]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_3_10": "x_vars[S4,C11]",
   "x_3_11": "x_vars[S4,C12]",
   "x_3_12": "x_vars[S4,C13]",
   "x_3_13": "x_vars[S4,C14]",
   "x_3_14": "x_vars[S4,C15]",
   "x_3_15": "x_vars[S4,C16]",
   "x_3_16": "x_vars[S4,C17]",
   "x_3_17": "x_vars[S4,C18]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_4_10": "x_vars[S5,C11]",
   "x_4_11": "x_vars[S5,C12]",
   "x_4_12": "x_vars[S5,C13]",
   "x_4_13": "x_vars[S5,C14]",
   "x_4_14": "x_vars[S5,C15]",
   "x_4_15": "x_vars[S5,C16]",
   "x_4_16": "x_vars[S5,C17]",
   "x_4_17": "x_vars[S5,C18]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_5_10": "x_vars[S6,C11]",
   "x_5_11": "x_vars[S6,C12]",
   "x_5_12": "x_vars[S6,C13]",
   "x_5_13": "x_vars[S6,C14]",
   "x_5_14": "x_vars[S6,C15]",
   "x_5_15": "x_vars[S6,C16]",
   "x_5_16": "x_vars[S6,C17]",
   "x_5_17": "x_vars[S6,C18]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_8": "x_vars[S7,C9]",
   "x_6_9": "x_vars[S7,C10]",
   "x_6_10": "x_vars[S7,C11]",
   "x_6_11": "x_vars[S7,C12]",
   "x_6_12": "x_vars[S7,C13]",
   "x_6_13": "x_vars[S7,C14]",
   "x_6_14": "x_vars[S7,C15]",
   "x_6_15": "x_vars[S7,C16]",
   "x_6_16": "x_vars[S7,C17]",
   "x_6_17": "x_vars[S7,C18]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_7_10": "x_vars[S8,C11]",
   "x_7_11": "x_vars[S8,C12]",
   "x_7_12": "x_vars[S8,C13]",
   "x_7_13": "x_vars[S8,C14]",
   "x_7_14": "x_vars[S8,C15]",
   "x_7_15": "x_vars[S8,C16]",
   "x_7_16": "x_vars[S8,C17]",
   "x_7_17": "x_vars[S8,C18]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_8_10": "x_vars[S9,C11]",
   "x_8_11": "x_vars[S9,C12]",
   "x_8_12": "x_vars[S9,C13]",
   "x_8_13": "x_vars[S9,C14]",
   "x_8_14": "x_vars[S9,C15]",
   "x_8_15": "x_vars[S9,C16]",
   "x_8_16": "x_vars[S9,C17]",
   "x_8_17": "x_vars[S9,C18]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]",
   "x_9_10": "x_vars[S10,C11]",
   "x_9_11": "x_vars[S10,C12]",
   "x_9_12": "x_vars[S10,C13]",
   "x_9_13": "x_vars[S10,C14]",
   "x_9_14": "x_vars[S10,C15]",
   "x_9_15": "x_vars[S10,C16]",
   "x_9_16": "x_vars[S10,C17]",
   "x_9_17": "x_vars[S10,C18]",
   "x_10_0": "x_vars[S11,C1]",
   "x_10_1": "x_vars[S11,C2]",
   "x_10_2": "x_vars[S11,C3]",
   "x_10_3": "x_vars[S11,C4]",
   "x_10_4": "x_vars[S11,C5]",
   "x_10_5": "x_vars[S11,C6]",
   "x_10_6": "x_vars[S11,C7]",
   "x_10_7": "x_vars[S11,C8]",
   "x_10_8": "x_vars[S11,C9]",
   "x_10_9": "x_vars[S11,C10]",
   "x_10_10": "x_vars[S11,C11]",
   "x_10_11": "x_vars[S11,C12]",
   "x_10_12": "x_vars[S11,C13]",
   "x_10_13": "x_vars[S11,C14]",
   "x_10_14": "x_vars[S11,C15]",
   "x_10_15": "x_vars[S11,C16]",
   "x_10_16": "x_vars[S11,C17]",
   "x_10_17": "x_vars[S11,C18]",
   "x_11_0": "x_vars[S12,C1]",
   "x_11_1": "x_vars[S12,C2]",
   "x_11_2": "x_vars[S12,C3]",
   "x_11_3": "x_vars[S12,C4]",
   "x_11_4": "x_vars[S12,C5]",
   "x_11_5": "x_vars[S12,C6]",
   "x_11_6": "x_vars[S12,C7]",
   "x_11_7": "x_vars[S12,C8]",
   "x_11_8": "x_vars[S12,C9]",
   "x_11_9": "x_vars[S12,C10]",
   "x_11_10": "x_vars[S12,C11]",
   "x_11_11": "x_vars[S12,C12]",
   "x_11_12": "x_vars[S12,C13]",
   "x_11_13": "x_vars[S12,C14]",
   "x_11_14": "x_vars[S12,C15]",
   "x_11_15": "x_vars[S12,C16]",
   "x_11_16": "x_vars[S12,C17]",
   "x_11_17": "x_vars[S12,C18]",
   "x_12_0": "x_vars[S13,C1]",
   "x_12_1": "x_vars[S13,C2]",
   "x_12_2": "x_vars[S13,C3]",
   "x_12_3": "x_vars[S13,C4]",
   "x_12_4": "x_vars[S13,C5]",
   "x_12_5": "x_vars[S13,C6]",
   "x_12_6": "x_vars[S13,C7]",
   "x_12_7": "x_vars[S13,C8]",
   "x_12_8": "x_vars[S13,C9]",
   "x_12_9": "x_vars[S13,C10]",
   "x_12_10": "x_vars[S13,C11]",
   "x_12_11": "x_vars[S13,C12]",
   "x_12_12": "x_vars[S13,C13]",
   "x_12_13": "x_vars[S13,C14]",
   "x_12_14": "x_vars[S13,C15]",
   "x_12_15": "x_vars[S13,C16]",
   "x_12_16": "x_vars[S13,C17]",
   "x_12_17": "x_vars[S13,C18]",
   "x_13_0": "x_vars[S14,C1]",
   "x_13_1": "x_vars[S14,C2]",
   "x_13_2": "x_vars[S14,C3]",
   "x_13_3": "x_vars[S14,C4]",
   "x_13_4": "x_vars[S14,C5]",
   "x_13_5": "x_vars[S14,C6]",
   "x_13_6": "x_vars[S14,C7]",
   "x_13_7": "x_vars[S14,C8]",
   "x_13_8": "x_vars[S14,C9]",
   "x_13_9": "x_vars[S14,C10]",
   "x_13_10": "x_vars[S14,C11]",
   "x_13_11": "x_vars[S14,C12]",
   "x_13_12": "x_vars[S14,C13]",
   "x_13_13": "x_vars[S14,C14]",
   "x_13_14": "x_vars[S14,C15]",
   "x_13_15": "x_vars[S14,C16]",
   "x_13_16": "x_vars[S14,C17]",
   "x_13_17": "x_vars[S14,C18]",
   "x_14_0": "x_vars[S15,C1]",
   "x_14_1": "x_vars[S15,C2]",
   "x_14_2": "x_vars[S15,C3]",
   "x_14_3": "x_vars[S15,C4]",
   "x_14_4": "x_vars[S15,C5]",
   "x_14_5": "x_vars[S15,C6]",
   "x_14_6": "x_vars[S15,C7]",
   "x_14_7": "x_vars[S15,C8]",
   "x_14_8": "x_vars[S15,C9]",
   "x_14_9": "x_vars[S15,C10]",
   "x_14_10": "x_vars[S15,C11]",
   "x_14_11": "x_vars[S15,C12]",
   "x_14_12": "x_vars[S15,C13]",
   "x_14_13": "x_vars[S15,C14]",
   "x_14_14": "x_vars[S15,C15]",
   "x_14_15": "x_vars[S15,C16]",
   "x_14_16": "x_vars[S15,C17]",
   "x_14_17": "x_vars[S15,C18]",
   "x_15_0": "x_vars[S16,C1]",
   "x_15_1": "x_vars[S16,C2]",
   "x_15_2": "x_vars[S16,C3]",
   "x_15_3": "x_vars[S16,C4]",
   "x_15_4": "x_vars[S16,C5]",
   "x_15_5": "x_vars[S16,C6]",
   "x_15_6": "x_vars[S16,C7]",
   "x_15_7": "x_vars[S16,C8]",
   "x_15_8": "x_vars[S16,C9]",
   "x_15_9": "x_vars[S16,C10]",
   "x_15_10": "x_vars[S16,C11]",
   "x_15_11": "x_vars[S16,C12]",
   "x_15_12": "x_vars[S16,C13]",
   "x_15_13": "x_vars[S16,C14]",
   "x_15_14": "x_vars[S16,C15]",
   "x_15_15": "x_vars[S16,C16]",
   "x_15_16": "x_vars[S16,C17]",
   "x_15_17": "x_vars[S16,C18]",
   "x_16_0": "x_vars[S17,C1]",
   "x_16_1": "x_vars[S17,C2]",
   "x_16_2": "x_vars[S17,C3]",
   "x_16_3": "x_vars[S17,C4]",
   "x_16_4": "x_vars[S17,C5]",
   "x_16_5": "x_vars[S17,C6]",
   "x_16_6": "x_vars[S17,C7]",
   "x_16_7": "x_vars[S17,C8]",
   "x_16_8": "x_vars[S17,C9]",
   "x_16_9": "x_vars[S17,C10]",
   "x_16_10": "x_vars[S17,C11]",
   "x_16_11": "x_vars[S17,C12]",
   "x_16_12": "x_vars[S17,C13]",
   "x_16_13": "x_vars[S17,C14]",
   "x_16_14": "x_vars[S17,C15]",
   "x_16_15": "x_vars[S17,C16]",
   "x_16_16": "x_vars[S17,C17]",
   "x_16_17": "x_vars[S17,C18]",
   "x_17_0": "x_vars[S18,C1]",
   "x_17_1": "x_vars[S18,C2]",
   "x_17_2": "x_vars[S18,C3]",
   "x_17_3": "x_vars[S18,C4]",
   "x_17_4": "x_vars[S18,C5]",
   "x_17_5": "x_vars[S18,C6]",
   "x_17_6": "x_vars[S18,C7]",
   "x_17_7": "x_vars[S18,C8]",
   "x_17_8": "x_vars[S18,C9]",
   "x_17_9": "x_vars[S18,C10]",
   "x_17_10": "x_vars[S18,C11]",
   "x_17_11": "x_vars[S18,C12]",
   "x_17_12": "x_vars[S18,C13]",
   "x_17_13": "x_vars[S18,C14]",
   "x_17_14": "x_vars[S18,C15]",
   "x_17_15": "x_vars[S18,C16]",
   "x_17_16": "x_vars[S18,C17]",
   "x_17_17": "x_vars[S18,C18]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-002": {
  "source_code_sha256": "1f90c79b2083e10e2390aba027f5be2c3ebc45b30e763c3a4fc929fb39c5c8f9",
  "original_solution_sha256": "b69dadf89947a0ef575782b3cf72fcd95b8609b6eb70a5015203d3669730a53c",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S1,C11]"
   ],
   [
    "C11",
    "x_vars[S1,C12]"
   ],
   [
    "C12",
    "x_vars[S2,C1]"
   ],
   [
    "C13",
    "x_vars[S2,C2]"
   ],
   [
    "C14",
    "x_vars[S2,C3]"
   ],
   [
    "C15",
    "x_vars[S2,C4]"
   ],
   [
    "C16",
    "x_vars[S2,C5]"
   ],
   [
    "C17",
    "x_vars[S2,C6]"
   ],
   [
    "C18",
    "x_vars[S2,C7]"
   ],
   [
    "C19",
    "x_vars[S2,C8]"
   ],
   [
    "C20",
    "x_vars[S2,C9]"
   ],
   [
    "C21",
    "x_vars[S2,C10]"
   ],
   [
    "C22",
    "x_vars[S2,C11]"
   ],
   [
    "C23",
    "x_vars[S2,C12]"
   ],
   [
    "C24",
    "x_vars[S3,C1]"
   ],
   [
    "C25",
    "x_vars[S3,C2]"
   ],
   [
    "C26",
    "x_vars[S3,C3]"
   ],
   [
    "C27",
    "x_vars[S3,C4]"
   ],
   [
    "C28",
    "x_vars[S3,C5]"
   ],
   [
    "C29",
    "x_vars[S3,C6]"
   ],
   [
    "C30",
    "x_vars[S3,C7]"
   ],
   [
    "C31",
    "x_vars[S3,C8]"
   ],
   [
    "C32",
    "x_vars[S3,C9]"
   ],
   [
    "C33",
    "x_vars[S3,C10]"
   ],
   [
    "C34",
    "x_vars[S3,C11]"
   ],
   [
    "C35",
    "x_vars[S3,C12]"
   ],
   [
    "C36",
    "x_vars[S4,C1]"
   ],
   [
    "C37",
    "x_vars[S4,C2]"
   ],
   [
    "C38",
    "x_vars[S4,C3]"
   ],
   [
    "C39",
    "x_vars[S4,C4]"
   ],
   [
    "C40",
    "x_vars[S4,C5]"
   ],
   [
    "C41",
    "x_vars[S4,C6]"
   ],
   [
    "C42",
    "x_vars[S4,C7]"
   ],
   [
    "C43",
    "x_vars[S4,C8]"
   ],
   [
    "C44",
    "x_vars[S4,C9]"
   ],
   [
    "C45",
    "x_vars[S4,C10]"
   ],
   [
    "C46",
    "x_vars[S4,C11]"
   ],
   [
    "C47",
    "x_vars[S4,C12]"
   ],
   [
    "C48",
    "x_vars[S5,C1]"
   ],
   [
    "C49",
    "x_vars[S5,C2]"
   ],
   [
    "C50",
    "x_vars[S5,C3]"
   ],
   [
    "C51",
    "x_vars[S5,C4]"
   ],
   [
    "C52",
    "x_vars[S5,C5]"
   ],
   [
    "C53",
    "x_vars[S5,C6]"
   ],
   [
    "C54",
    "x_vars[S5,C7]"
   ],
   [
    "C55",
    "x_vars[S5,C8]"
   ],
   [
    "C56",
    "x_vars[S5,C9]"
   ],
   [
    "C57",
    "x_vars[S5,C10]"
   ],
   [
    "C58",
    "x_vars[S5,C11]"
   ],
   [
    "C59",
    "x_vars[S5,C12]"
   ],
   [
    "C60",
    "x_vars[S6,C1]"
   ],
   [
    "C61",
    "x_vars[S6,C2]"
   ],
   [
    "C62",
    "x_vars[S6,C3]"
   ],
   [
    "C63",
    "x_vars[S6,C4]"
   ],
   [
    "C64",
    "x_vars[S6,C5]"
   ],
   [
    "C65",
    "x_vars[S6,C6]"
   ],
   [
    "C66",
    "x_vars[S6,C7]"
   ],
   [
    "C67",
    "x_vars[S6,C8]"
   ],
   [
    "C68",
    "x_vars[S6,C9]"
   ],
   [
    "C69",
    "x_vars[S6,C10]"
   ],
   [
    "C70",
    "x_vars[S6,C11]"
   ],
   [
    "C71",
    "x_vars[S6,C12]"
   ],
   [
    "C72",
    "x_vars[S7,C1]"
   ],
   [
    "C73",
    "x_vars[S7,C2]"
   ],
   [
    "C74",
    "x_vars[S7,C3]"
   ],
   [
    "C75",
    "x_vars[S7,C4]"
   ],
   [
    "C76",
    "x_vars[S7,C5]"
   ],
   [
    "C77",
    "x_vars[S7,C6]"
   ],
   [
    "C78",
    "x_vars[S7,C7]"
   ],
   [
    "C79",
    "x_vars[S7,C8]"
   ],
   [
    "C80",
    "x_vars[S7,C9]"
   ],
   [
    "C81",
    "x_vars[S7,C10]"
   ],
   [
    "C82",
    "x_vars[S7,C11]"
   ],
   [
    "C83",
    "x_vars[S7,C12]"
   ],
   [
    "C84",
    "x_vars[S8,C1]"
   ],
   [
    "C85",
    "x_vars[S8,C2]"
   ],
   [
    "C86",
    "x_vars[S8,C3]"
   ],
   [
    "C87",
    "x_vars[S8,C4]"
   ],
   [
    "C88",
    "x_vars[S8,C5]"
   ],
   [
    "C89",
    "x_vars[S8,C6]"
   ],
   [
    "C90",
    "x_vars[S8,C7]"
   ],
   [
    "C91",
    "x_vars[S8,C8]"
   ],
   [
    "C92",
    "x_vars[S8,C9]"
   ],
   [
    "C93",
    "x_vars[S8,C10]"
   ],
   [
    "C94",
    "x_vars[S8,C11]"
   ],
   [
    "C95",
    "x_vars[S8,C12]"
   ],
   [
    "C96",
    "x_vars[S9,C1]"
   ],
   [
    "C97",
    "x_vars[S9,C2]"
   ],
   [
    "C98",
    "x_vars[S9,C3]"
   ],
   [
    "C99",
    "x_vars[S9,C4]"
   ],
   [
    "C100",
    "x_vars[S9,C5]"
   ],
   [
    "C101",
    "x_vars[S9,C6]"
   ],
   [
    "C102",
    "x_vars[S9,C7]"
   ],
   [
    "C103",
    "x_vars[S9,C8]"
   ],
   [
    "C104",
    "x_vars[S9,C9]"
   ],
   [
    "C105",
    "x_vars[S9,C10]"
   ],
   [
    "C106",
    "x_vars[S9,C11]"
   ],
   [
    "C107",
    "x_vars[S9,C12]"
   ],
   [
    "C108",
    "x_vars[S10,C1]"
   ],
   [
    "C109",
    "x_vars[S10,C2]"
   ],
   [
    "C110",
    "x_vars[S10,C3]"
   ],
   [
    "C111",
    "x_vars[S10,C4]"
   ],
   [
    "C112",
    "x_vars[S10,C5]"
   ],
   [
    "C113",
    "x_vars[S10,C6]"
   ],
   [
    "C114",
    "x_vars[S10,C7]"
   ],
   [
    "C115",
    "x_vars[S10,C8]"
   ],
   [
    "C116",
    "x_vars[S10,C9]"
   ],
   [
    "C117",
    "x_vars[S10,C10]"
   ],
   [
    "C118",
    "x_vars[S10,C11]"
   ],
   [
    "C119",
    "x_vars[S10,C12]"
   ],
   [
    "C120",
    "x_vars[S11,C1]"
   ],
   [
    "C121",
    "x_vars[S11,C2]"
   ],
   [
    "C122",
    "x_vars[S11,C3]"
   ],
   [
    "C123",
    "x_vars[S11,C4]"
   ],
   [
    "C124",
    "x_vars[S11,C5]"
   ],
   [
    "C125",
    "x_vars[S11,C6]"
   ],
   [
    "C126",
    "x_vars[S11,C7]"
   ],
   [
    "C127",
    "x_vars[S11,C8]"
   ],
   [
    "C128",
    "x_vars[S11,C9]"
   ],
   [
    "C129",
    "x_vars[S11,C10]"
   ],
   [
    "C130",
    "x_vars[S11,C11]"
   ],
   [
    "C131",
    "x_vars[S11,C12]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_0_10": "x_vars[S1,C11]",
   "x_0_11": "x_vars[S1,C12]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_1_10": "x_vars[S2,C11]",
   "x_1_11": "x_vars[S2,C12]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_2_10": "x_vars[S3,C11]",
   "x_2_11": "x_vars[S3,C12]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_3_10": "x_vars[S4,C11]",
   "x_3_11": "x_vars[S4,C12]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_4_10": "x_vars[S5,C11]",
   "x_4_11": "x_vars[S5,C12]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_5_10": "x_vars[S6,C11]",
   "x_5_11": "x_vars[S6,C12]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_8": "x_vars[S7,C9]",
   "x_6_9": "x_vars[S7,C10]",
   "x_6_10": "x_vars[S7,C11]",
   "x_6_11": "x_vars[S7,C12]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_7_10": "x_vars[S8,C11]",
   "x_7_11": "x_vars[S8,C12]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_8_10": "x_vars[S9,C11]",
   "x_8_11": "x_vars[S9,C12]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]",
   "x_9_10": "x_vars[S10,C11]",
   "x_9_11": "x_vars[S10,C12]",
   "x_10_0": "x_vars[S11,C1]",
   "x_10_1": "x_vars[S11,C2]",
   "x_10_2": "x_vars[S11,C3]",
   "x_10_3": "x_vars[S11,C4]",
   "x_10_4": "x_vars[S11,C5]",
   "x_10_5": "x_vars[S11,C6]",
   "x_10_6": "x_vars[S11,C7]",
   "x_10_7": "x_vars[S11,C8]",
   "x_10_8": "x_vars[S11,C9]",
   "x_10_9": "x_vars[S11,C10]",
   "x_10_10": "x_vars[S11,C11]",
   "x_10_11": "x_vars[S11,C12]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-003": {
  "source_code_sha256": "c0d0c8ca0bac72135aec87f9a25b41912212aed968d1db7b8f7cd8b954e912fa",
  "original_solution_sha256": "8595ff8cc874b03b28a1a1da093c74144fe369f0871dd21a8de22e8489fb5031",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S2,C1]"
   ],
   [
    "C11",
    "x_vars[S2,C2]"
   ],
   [
    "C12",
    "x_vars[S2,C3]"
   ],
   [
    "C13",
    "x_vars[S2,C4]"
   ],
   [
    "C14",
    "x_vars[S2,C5]"
   ],
   [
    "C15",
    "x_vars[S2,C6]"
   ],
   [
    "C16",
    "x_vars[S2,C7]"
   ],
   [
    "C17",
    "x_vars[S2,C8]"
   ],
   [
    "C18",
    "x_vars[S2,C9]"
   ],
   [
    "C19",
    "x_vars[S2,C10]"
   ],
   [
    "C20",
    "x_vars[S3,C1]"
   ],
   [
    "C21",
    "x_vars[S3,C2]"
   ],
   [
    "C22",
    "x_vars[S3,C3]"
   ],
   [
    "C23",
    "x_vars[S3,C4]"
   ],
   [
    "C24",
    "x_vars[S3,C5]"
   ],
   [
    "C25",
    "x_vars[S3,C6]"
   ],
   [
    "C26",
    "x_vars[S3,C7]"
   ],
   [
    "C27",
    "x_vars[S3,C8]"
   ],
   [
    "C28",
    "x_vars[S3,C9]"
   ],
   [
    "C29",
    "x_vars[S3,C10]"
   ],
   [
    "C30",
    "x_vars[S4,C1]"
   ],
   [
    "C31",
    "x_vars[S4,C2]"
   ],
   [
    "C32",
    "x_vars[S4,C3]"
   ],
   [
    "C33",
    "x_vars[S4,C4]"
   ],
   [
    "C34",
    "x_vars[S4,C5]"
   ],
   [
    "C35",
    "x_vars[S4,C6]"
   ],
   [
    "C36",
    "x_vars[S4,C7]"
   ],
   [
    "C37",
    "x_vars[S4,C8]"
   ],
   [
    "C38",
    "x_vars[S4,C9]"
   ],
   [
    "C39",
    "x_vars[S4,C10]"
   ],
   [
    "C40",
    "x_vars[S5,C1]"
   ],
   [
    "C41",
    "x_vars[S5,C2]"
   ],
   [
    "C42",
    "x_vars[S5,C3]"
   ],
   [
    "C43",
    "x_vars[S5,C4]"
   ],
   [
    "C44",
    "x_vars[S5,C5]"
   ],
   [
    "C45",
    "x_vars[S5,C6]"
   ],
   [
    "C46",
    "x_vars[S5,C7]"
   ],
   [
    "C47",
    "x_vars[S5,C8]"
   ],
   [
    "C48",
    "x_vars[S5,C9]"
   ],
   [
    "C49",
    "x_vars[S5,C10]"
   ],
   [
    "C50",
    "x_vars[S6,C1]"
   ],
   [
    "C51",
    "x_vars[S6,C2]"
   ],
   [
    "C52",
    "x_vars[S6,C3]"
   ],
   [
    "C53",
    "x_vars[S6,C4]"
   ],
   [
    "C54",
    "x_vars[S6,C5]"
   ],
   [
    "C55",
    "x_vars[S6,C6]"
   ],
   [
    "C56",
    "x_vars[S6,C7]"
   ],
   [
    "C57",
    "x_vars[S6,C8]"
   ],
   [
    "C58",
    "x_vars[S6,C9]"
   ],
   [
    "C59",
    "x_vars[S6,C10]"
   ],
   [
    "C60",
    "x_vars[S7,C1]"
   ],
   [
    "C61",
    "x_vars[S7,C2]"
   ],
   [
    "C62",
    "x_vars[S7,C3]"
   ],
   [
    "C63",
    "x_vars[S7,C4]"
   ],
   [
    "C64",
    "x_vars[S7,C5]"
   ],
   [
    "C65",
    "x_vars[S7,C6]"
   ],
   [
    "C66",
    "x_vars[S7,C7]"
   ],
   [
    "C67",
    "x_vars[S7,C8]"
   ],
   [
    "C68",
    "x_vars[S7,C9]"
   ],
   [
    "C69",
    "x_vars[S7,C10]"
   ],
   [
    "C70",
    "x_vars[S8,C1]"
   ],
   [
    "C71",
    "x_vars[S8,C2]"
   ],
   [
    "C72",
    "x_vars[S8,C3]"
   ],
   [
    "C73",
    "x_vars[S8,C4]"
   ],
   [
    "C74",
    "x_vars[S8,C5]"
   ],
   [
    "C75",
    "x_vars[S8,C6]"
   ],
   [
    "C76",
    "x_vars[S8,C7]"
   ],
   [
    "C77",
    "x_vars[S8,C8]"
   ],
   [
    "C78",
    "x_vars[S8,C9]"
   ],
   [
    "C79",
    "x_vars[S8,C10]"
   ],
   [
    "C80",
    "x_vars[S9,C1]"
   ],
   [
    "C81",
    "x_vars[S9,C2]"
   ],
   [
    "C82",
    "x_vars[S9,C3]"
   ],
   [
    "C83",
    "x_vars[S9,C4]"
   ],
   [
    "C84",
    "x_vars[S9,C5]"
   ],
   [
    "C85",
    "x_vars[S9,C6]"
   ],
   [
    "C86",
    "x_vars[S9,C7]"
   ],
   [
    "C87",
    "x_vars[S9,C8]"
   ],
   [
    "C88",
    "x_vars[S9,C9]"
   ],
   [
    "C89",
    "x_vars[S9,C10]"
   ],
   [
    "C90",
    "x_vars[S10,C1]"
   ],
   [
    "C91",
    "x_vars[S10,C2]"
   ],
   [
    "C92",
    "x_vars[S10,C3]"
   ],
   [
    "C93",
    "x_vars[S10,C4]"
   ],
   [
    "C94",
    "x_vars[S10,C5]"
   ],
   [
    "C95",
    "x_vars[S10,C6]"
   ],
   [
    "C96",
    "x_vars[S10,C7]"
   ],
   [
    "C97",
    "x_vars[S10,C8]"
   ],
   [
    "C98",
    "x_vars[S10,C9]"
   ],
   [
    "C99",
    "x_vars[S10,C10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_8": "x_vars[S7,C9]",
   "x_6_9": "x_vars[S7,C10]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-004": {
  "source_code_sha256": "0d2519be77f43c4bb9627f8066d19735973f7c6e19ea753da3c7f009949c4df0",
  "original_solution_sha256": "38f1e81fb6054e108af14a4b7625e64c33a20f61dc0ecf751ec3e531ef66e660",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S1,C11]"
   ],
   [
    "C11",
    "x_vars[S1,C12]"
   ],
   [
    "C12",
    "x_vars[S2,C1]"
   ],
   [
    "C13",
    "x_vars[S2,C2]"
   ],
   [
    "C14",
    "x_vars[S2,C3]"
   ],
   [
    "C15",
    "x_vars[S2,C4]"
   ],
   [
    "C16",
    "x_vars[S2,C5]"
   ],
   [
    "C17",
    "x_vars[S2,C6]"
   ],
   [
    "C18",
    "x_vars[S2,C7]"
   ],
   [
    "C19",
    "x_vars[S2,C8]"
   ],
   [
    "C20",
    "x_vars[S2,C9]"
   ],
   [
    "C21",
    "x_vars[S2,C10]"
   ],
   [
    "C22",
    "x_vars[S2,C11]"
   ],
   [
    "C23",
    "x_vars[S2,C12]"
   ],
   [
    "C24",
    "x_vars[S3,C1]"
   ],
   [
    "C25",
    "x_vars[S3,C2]"
   ],
   [
    "C26",
    "x_vars[S3,C3]"
   ],
   [
    "C27",
    "x_vars[S3,C4]"
   ],
   [
    "C28",
    "x_vars[S3,C5]"
   ],
   [
    "C29",
    "x_vars[S3,C6]"
   ],
   [
    "C30",
    "x_vars[S3,C7]"
   ],
   [
    "C31",
    "x_vars[S3,C8]"
   ],
   [
    "C32",
    "x_vars[S3,C9]"
   ],
   [
    "C33",
    "x_vars[S3,C10]"
   ],
   [
    "C34",
    "x_vars[S3,C11]"
   ],
   [
    "C35",
    "x_vars[S3,C12]"
   ],
   [
    "C36",
    "x_vars[S4,C1]"
   ],
   [
    "C37",
    "x_vars[S4,C2]"
   ],
   [
    "C38",
    "x_vars[S4,C3]"
   ],
   [
    "C39",
    "x_vars[S4,C4]"
   ],
   [
    "C40",
    "x_vars[S4,C5]"
   ],
   [
    "C41",
    "x_vars[S4,C6]"
   ],
   [
    "C42",
    "x_vars[S4,C7]"
   ],
   [
    "C43",
    "x_vars[S4,C8]"
   ],
   [
    "C44",
    "x_vars[S4,C9]"
   ],
   [
    "C45",
    "x_vars[S4,C10]"
   ],
   [
    "C46",
    "x_vars[S4,C11]"
   ],
   [
    "C47",
    "x_vars[S4,C12]"
   ],
   [
    "C48",
    "x_vars[S5,C1]"
   ],
   [
    "C49",
    "x_vars[S5,C2]"
   ],
   [
    "C50",
    "x_vars[S5,C3]"
   ],
   [
    "C51",
    "x_vars[S5,C4]"
   ],
   [
    "C52",
    "x_vars[S5,C5]"
   ],
   [
    "C53",
    "x_vars[S5,C6]"
   ],
   [
    "C54",
    "x_vars[S5,C7]"
   ],
   [
    "C55",
    "x_vars[S5,C8]"
   ],
   [
    "C56",
    "x_vars[S5,C9]"
   ],
   [
    "C57",
    "x_vars[S5,C10]"
   ],
   [
    "C58",
    "x_vars[S5,C11]"
   ],
   [
    "C59",
    "x_vars[S5,C12]"
   ],
   [
    "C60",
    "x_vars[S6,C1]"
   ],
   [
    "C61",
    "x_vars[S6,C2]"
   ],
   [
    "C62",
    "x_vars[S6,C3]"
   ],
   [
    "C63",
    "x_vars[S6,C4]"
   ],
   [
    "C64",
    "x_vars[S6,C5]"
   ],
   [
    "C65",
    "x_vars[S6,C6]"
   ],
   [
    "C66",
    "x_vars[S6,C7]"
   ],
   [
    "C67",
    "x_vars[S6,C8]"
   ],
   [
    "C68",
    "x_vars[S6,C9]"
   ],
   [
    "C69",
    "x_vars[S6,C10]"
   ],
   [
    "C70",
    "x_vars[S6,C11]"
   ],
   [
    "C71",
    "x_vars[S6,C12]"
   ],
   [
    "C72",
    "x_vars[S7,C1]"
   ],
   [
    "C73",
    "x_vars[S7,C2]"
   ],
   [
    "C74",
    "x_vars[S7,C3]"
   ],
   [
    "C75",
    "x_vars[S7,C4]"
   ],
   [
    "C76",
    "x_vars[S7,C5]"
   ],
   [
    "C77",
    "x_vars[S7,C6]"
   ],
   [
    "C78",
    "x_vars[S7,C7]"
   ],
   [
    "C79",
    "x_vars[S7,C8]"
   ],
   [
    "C80",
    "x_vars[S7,C9]"
   ],
   [
    "C81",
    "x_vars[S7,C10]"
   ],
   [
    "C82",
    "x_vars[S7,C11]"
   ],
   [
    "C83",
    "x_vars[S7,C12]"
   ],
   [
    "C84",
    "x_vars[S8,C1]"
   ],
   [
    "C85",
    "x_vars[S8,C2]"
   ],
   [
    "C86",
    "x_vars[S8,C3]"
   ],
   [
    "C87",
    "x_vars[S8,C4]"
   ],
   [
    "C88",
    "x_vars[S8,C5]"
   ],
   [
    "C89",
    "x_vars[S8,C6]"
   ],
   [
    "C90",
    "x_vars[S8,C7]"
   ],
   [
    "C91",
    "x_vars[S8,C8]"
   ],
   [
    "C92",
    "x_vars[S8,C9]"
   ],
   [
    "C93",
    "x_vars[S8,C10]"
   ],
   [
    "C94",
    "x_vars[S8,C11]"
   ],
   [
    "C95",
    "x_vars[S8,C12]"
   ],
   [
    "C96",
    "x_vars[S9,C1]"
   ],
   [
    "C97",
    "x_vars[S9,C2]"
   ],
   [
    "C98",
    "x_vars[S9,C3]"
   ],
   [
    "C99",
    "x_vars[S9,C4]"
   ],
   [
    "C100",
    "x_vars[S9,C5]"
   ],
   [
    "C101",
    "x_vars[S9,C6]"
   ],
   [
    "C102",
    "x_vars[S9,C7]"
   ],
   [
    "C103",
    "x_vars[S9,C8]"
   ],
   [
    "C104",
    "x_vars[S9,C9]"
   ],
   [
    "C105",
    "x_vars[S9,C10]"
   ],
   [
    "C106",
    "x_vars[S9,C11]"
   ],
   [
    "C107",
    "x_vars[S9,C12]"
   ],
   [
    "C108",
    "x_vars[S10,C1]"
   ],
   [
    "C109",
    "x_vars[S10,C2]"
   ],
   [
    "C110",
    "x_vars[S10,C3]"
   ],
   [
    "C111",
    "x_vars[S10,C4]"
   ],
   [
    "C112",
    "x_vars[S10,C5]"
   ],
   [
    "C113",
    "x_vars[S10,C6]"
   ],
   [
    "C114",
    "x_vars[S10,C7]"
   ],
   [
    "C115",
    "x_vars[S10,C8]"
   ],
   [
    "C116",
    "x_vars[S10,C9]"
   ],
   [
    "C117",
    "x_vars[S10,C10]"
   ],
   [
    "C118",
    "x_vars[S10,C11]"
   ],
   [
    "C119",
    "x_vars[S10,C12]"
   ],
   [
    "C120",
    "x_vars[S11,C1]"
   ],
   [
    "C121",
    "x_vars[S11,C2]"
   ],
   [
    "C122",
    "x_vars[S11,C3]"
   ],
   [
    "C123",
    "x_vars[S11,C4]"
   ],
   [
    "C124",
    "x_vars[S11,C5]"
   ],
   [
    "C125",
    "x_vars[S11,C6]"
   ],
   [
    "C126",
    "x_vars[S11,C7]"
   ],
   [
    "C127",
    "x_vars[S11,C8]"
   ],
   [
    "C128",
    "x_vars[S11,C9]"
   ],
   [
    "C129",
    "x_vars[S11,C10]"
   ],
   [
    "C130",
    "x_vars[S11,C11]"
   ],
   [
    "C131",
    "x_vars[S11,C12]"
   ],
   [
    "C132",
    "x_vars[S12,C1]"
   ],
   [
    "C133",
    "x_vars[S12,C2]"
   ],
   [
    "C134",
    "x_vars[S12,C3]"
   ],
   [
    "C135",
    "x_vars[S12,C4]"
   ],
   [
    "C136",
    "x_vars[S12,C5]"
   ],
   [
    "C137",
    "x_vars[S12,C6]"
   ],
   [
    "C138",
    "x_vars[S12,C7]"
   ],
   [
    "C139",
    "x_vars[S12,C8]"
   ],
   [
    "C140",
    "x_vars[S12,C9]"
   ],
   [
    "C141",
    "x_vars[S12,C10]"
   ],
   [
    "C142",
    "x_vars[S12,C11]"
   ],
   [
    "C143",
    "x_vars[S12,C12]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_0_10": "x_vars[S1,C11]",
   "x_0_11": "x_vars[S1,C12]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_1_10": "x_vars[S2,C11]",
   "x_1_11": "x_vars[S2,C12]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_2_10": "x_vars[S3,C11]",
   "x_2_11": "x_vars[S3,C12]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_3_10": "x_vars[S4,C11]",
   "x_3_11": "x_vars[S4,C12]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_4_10": "x_vars[S5,C11]",
   "x_4_11": "x_vars[S5,C12]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_5_10": "x_vars[S6,C11]",
   "x_5_11": "x_vars[S6,C12]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_8": "x_vars[S7,C9]",
   "x_6_9": "x_vars[S7,C10]",
   "x_6_10": "x_vars[S7,C11]",
   "x_6_11": "x_vars[S7,C12]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_7_10": "x_vars[S8,C11]",
   "x_7_11": "x_vars[S8,C12]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_8_10": "x_vars[S9,C11]",
   "x_8_11": "x_vars[S9,C12]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]",
   "x_9_10": "x_vars[S10,C11]",
   "x_9_11": "x_vars[S10,C12]",
   "x_10_0": "x_vars[S11,C1]",
   "x_10_1": "x_vars[S11,C2]",
   "x_10_2": "x_vars[S11,C3]",
   "x_10_3": "x_vars[S11,C4]",
   "x_10_4": "x_vars[S11,C5]",
   "x_10_5": "x_vars[S11,C6]",
   "x_10_6": "x_vars[S11,C7]",
   "x_10_7": "x_vars[S11,C8]",
   "x_10_8": "x_vars[S11,C9]",
   "x_10_9": "x_vars[S11,C10]",
   "x_10_10": "x_vars[S11,C11]",
   "x_10_11": "x_vars[S11,C12]",
   "x_11_0": "x_vars[S12,C1]",
   "x_11_1": "x_vars[S12,C2]",
   "x_11_2": "x_vars[S12,C3]",
   "x_11_3": "x_vars[S12,C4]",
   "x_11_4": "x_vars[S12,C5]",
   "x_11_5": "x_vars[S12,C6]",
   "x_11_6": "x_vars[S12,C7]",
   "x_11_7": "x_vars[S12,C8]",
   "x_11_8": "x_vars[S12,C9]",
   "x_11_9": "x_vars[S12,C10]",
   "x_11_10": "x_vars[S12,C11]",
   "x_11_11": "x_vars[S12,C12]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-005": {
  "source_code_sha256": "a5c5e45f8f8fcda550a0c85c269e41c5624620733cf52464f65585959dd75d32",
  "original_solution_sha256": "989e223f54ab77751bee58028a934fde31a33086f26e206f1ba847b7b280a029",
  "bindings": [
   [
    "C0",
    "x_vars[supplier1,demand1]"
   ],
   [
    "C1",
    "x_vars[supplier1,demand2]"
   ],
   [
    "C2",
    "x_vars[supplier1,demand3]"
   ],
   [
    "C3",
    "x_vars[supplier1,demand4]"
   ],
   [
    "C4",
    "x_vars[supplier1,demand5]"
   ],
   [
    "C5",
    "x_vars[supplier1,demand6]"
   ],
   [
    "C6",
    "x_vars[supplier1,demand7]"
   ],
   [
    "C7",
    "x_vars[supplier1,demand8]"
   ],
   [
    "C8",
    "x_vars[supplier2,demand1]"
   ],
   [
    "C9",
    "x_vars[supplier2,demand2]"
   ],
   [
    "C10",
    "x_vars[supplier2,demand3]"
   ],
   [
    "C11",
    "x_vars[supplier2,demand4]"
   ],
   [
    "C12",
    "x_vars[supplier2,demand5]"
   ],
   [
    "C13",
    "x_vars[supplier2,demand6]"
   ],
   [
    "C14",
    "x_vars[supplier2,demand7]"
   ],
   [
    "C15",
    "x_vars[supplier2,demand8]"
   ],
   [
    "C16",
    "x_vars[supplier3,demand1]"
   ],
   [
    "C17",
    "x_vars[supplier3,demand2]"
   ],
   [
    "C18",
    "x_vars[supplier3,demand3]"
   ],
   [
    "C19",
    "x_vars[supplier3,demand4]"
   ],
   [
    "C20",
    "x_vars[supplier3,demand5]"
   ],
   [
    "C21",
    "x_vars[supplier3,demand6]"
   ],
   [
    "C22",
    "x_vars[supplier3,demand7]"
   ],
   [
    "C23",
    "x_vars[supplier3,demand8]"
   ],
   [
    "C24",
    "x_vars[supplier4,demand1]"
   ],
   [
    "C25",
    "x_vars[supplier4,demand2]"
   ],
   [
    "C26",
    "x_vars[supplier4,demand3]"
   ],
   [
    "C27",
    "x_vars[supplier4,demand4]"
   ],
   [
    "C28",
    "x_vars[supplier4,demand5]"
   ],
   [
    "C29",
    "x_vars[supplier4,demand6]"
   ],
   [
    "C30",
    "x_vars[supplier4,demand7]"
   ],
   [
    "C31",
    "x_vars[supplier4,demand8]"
   ],
   [
    "C32",
    "x_vars[supplier5,demand1]"
   ],
   [
    "C33",
    "x_vars[supplier5,demand2]"
   ],
   [
    "C34",
    "x_vars[supplier5,demand3]"
   ],
   [
    "C35",
    "x_vars[supplier5,demand4]"
   ],
   [
    "C36",
    "x_vars[supplier5,demand5]"
   ],
   [
    "C37",
    "x_vars[supplier5,demand6]"
   ],
   [
    "C38",
    "x_vars[supplier5,demand7]"
   ],
   [
    "C39",
    "x_vars[supplier5,demand8]"
   ],
   [
    "C40",
    "x_vars[supplier6,demand1]"
   ],
   [
    "C41",
    "x_vars[supplier6,demand2]"
   ],
   [
    "C42",
    "x_vars[supplier6,demand3]"
   ],
   [
    "C43",
    "x_vars[supplier6,demand4]"
   ],
   [
    "C44",
    "x_vars[supplier6,demand5]"
   ],
   [
    "C45",
    "x_vars[supplier6,demand6]"
   ],
   [
    "C46",
    "x_vars[supplier6,demand7]"
   ],
   [
    "C47",
    "x_vars[supplier6,demand8]"
   ],
   [
    "C48",
    "x_vars[supplier7,demand1]"
   ],
   [
    "C49",
    "x_vars[supplier7,demand2]"
   ],
   [
    "C50",
    "x_vars[supplier7,demand3]"
   ],
   [
    "C51",
    "x_vars[supplier7,demand4]"
   ],
   [
    "C52",
    "x_vars[supplier7,demand5]"
   ],
   [
    "C53",
    "x_vars[supplier7,demand6]"
   ],
   [
    "C54",
    "x_vars[supplier7,demand7]"
   ],
   [
    "C55",
    "x_vars[supplier7,demand8]"
   ],
   [
    "C56",
    "x_vars[supplier8,demand1]"
   ],
   [
    "C57",
    "x_vars[supplier8,demand2]"
   ],
   [
    "C58",
    "x_vars[supplier8,demand3]"
   ],
   [
    "C59",
    "x_vars[supplier8,demand4]"
   ],
   [
    "C60",
    "x_vars[supplier8,demand5]"
   ],
   [
    "C61",
    "x_vars[supplier8,demand6]"
   ],
   [
    "C62",
    "x_vars[supplier8,demand7]"
   ],
   [
    "C63",
    "x_vars[supplier8,demand8]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[supplier1,demand1]",
   "x_0_1": "x_vars[supplier1,demand2]",
   "x_0_2": "x_vars[supplier1,demand3]",
   "x_0_3": "x_vars[supplier1,demand4]",
   "x_0_4": "x_vars[supplier1,demand5]",
   "x_0_5": "x_vars[supplier1,demand6]",
   "x_0_6": "x_vars[supplier1,demand7]",
   "x_0_7": "x_vars[supplier1,demand8]",
   "x_1_0": "x_vars[supplier2,demand1]",
   "x_1_1": "x_vars[supplier2,demand2]",
   "x_1_2": "x_vars[supplier2,demand3]",
   "x_1_3": "x_vars[supplier2,demand4]",
   "x_1_4": "x_vars[supplier2,demand5]",
   "x_1_5": "x_vars[supplier2,demand6]",
   "x_1_6": "x_vars[supplier2,demand7]",
   "x_1_7": "x_vars[supplier2,demand8]",
   "x_2_0": "x_vars[supplier3,demand1]",
   "x_2_1": "x_vars[supplier3,demand2]",
   "x_2_2": "x_vars[supplier3,demand3]",
   "x_2_3": "x_vars[supplier3,demand4]",
   "x_2_4": "x_vars[supplier3,demand5]",
   "x_2_5": "x_vars[supplier3,demand6]",
   "x_2_6": "x_vars[supplier3,demand7]",
   "x_2_7": "x_vars[supplier3,demand8]",
   "x_3_0": "x_vars[supplier4,demand1]",
   "x_3_1": "x_vars[supplier4,demand2]",
   "x_3_2": "x_vars[supplier4,demand3]",
   "x_3_3": "x_vars[supplier4,demand4]",
   "x_3_4": "x_vars[supplier4,demand5]",
   "x_3_5": "x_vars[supplier4,demand6]",
   "x_3_6": "x_vars[supplier4,demand7]",
   "x_3_7": "x_vars[supplier4,demand8]",
   "x_4_0": "x_vars[supplier5,demand1]",
   "x_4_1": "x_vars[supplier5,demand2]",
   "x_4_2": "x_vars[supplier5,demand3]",
   "x_4_3": "x_vars[supplier5,demand4]",
   "x_4_4": "x_vars[supplier5,demand5]",
   "x_4_5": "x_vars[supplier5,demand6]",
   "x_4_6": "x_vars[supplier5,demand7]",
   "x_4_7": "x_vars[supplier5,demand8]",
   "x_5_0": "x_vars[supplier6,demand1]",
   "x_5_1": "x_vars[supplier6,demand2]",
   "x_5_2": "x_vars[supplier6,demand3]",
   "x_5_3": "x_vars[supplier6,demand4]",
   "x_5_4": "x_vars[supplier6,demand5]",
   "x_5_5": "x_vars[supplier6,demand6]",
   "x_5_6": "x_vars[supplier6,demand7]",
   "x_5_7": "x_vars[supplier6,demand8]",
   "x_6_0": "x_vars[supplier7,demand1]",
   "x_6_1": "x_vars[supplier7,demand2]",
   "x_6_2": "x_vars[supplier7,demand3]",
   "x_6_3": "x_vars[supplier7,demand4]",
   "x_6_4": "x_vars[supplier7,demand5]",
   "x_6_5": "x_vars[supplier7,demand6]",
   "x_6_6": "x_vars[supplier7,demand7]",
   "x_6_7": "x_vars[supplier7,demand8]",
   "x_7_0": "x_vars[supplier8,demand1]",
   "x_7_1": "x_vars[supplier8,demand2]",
   "x_7_2": "x_vars[supplier8,demand3]",
   "x_7_3": "x_vars[supplier8,demand4]",
   "x_7_4": "x_vars[supplier8,demand5]",
   "x_7_5": "x_vars[supplier8,demand6]",
   "x_7_6": "x_vars[supplier8,demand7]",
   "x_7_7": "x_vars[supplier8,demand8]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-006": {
  "source_code_sha256": "e8e519c8f667d5448aa1129b9f401247263a0b5f8b98d0bcdde9570238c662f4",
  "original_solution_sha256": "408badf617759d1d597d06caee6f8c313f246f59585721b7cce5056f9a2cb7d4",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S2,C1]"
   ],
   [
    "C11",
    "x_vars[S2,C2]"
   ],
   [
    "C12",
    "x_vars[S2,C3]"
   ],
   [
    "C13",
    "x_vars[S2,C4]"
   ],
   [
    "C14",
    "x_vars[S2,C5]"
   ],
   [
    "C15",
    "x_vars[S2,C6]"
   ],
   [
    "C16",
    "x_vars[S2,C7]"
   ],
   [
    "C17",
    "x_vars[S2,C8]"
   ],
   [
    "C18",
    "x_vars[S2,C9]"
   ],
   [
    "C19",
    "x_vars[S2,C10]"
   ],
   [
    "C20",
    "x_vars[S3,C1]"
   ],
   [
    "C21",
    "x_vars[S3,C2]"
   ],
   [
    "C22",
    "x_vars[S3,C3]"
   ],
   [
    "C23",
    "x_vars[S3,C4]"
   ],
   [
    "C24",
    "x_vars[S3,C5]"
   ],
   [
    "C25",
    "x_vars[S3,C6]"
   ],
   [
    "C26",
    "x_vars[S3,C7]"
   ],
   [
    "C27",
    "x_vars[S3,C8]"
   ],
   [
    "C28",
    "x_vars[S3,C9]"
   ],
   [
    "C29",
    "x_vars[S3,C10]"
   ],
   [
    "C30",
    "x_vars[S4,C1]"
   ],
   [
    "C31",
    "x_vars[S4,C2]"
   ],
   [
    "C32",
    "x_vars[S4,C3]"
   ],
   [
    "C33",
    "x_vars[S4,C4]"
   ],
   [
    "C34",
    "x_vars[S4,C5]"
   ],
   [
    "C35",
    "x_vars[S4,C6]"
   ],
   [
    "C36",
    "x_vars[S4,C7]"
   ],
   [
    "C37",
    "x_vars[S4,C8]"
   ],
   [
    "C38",
    "x_vars[S4,C9]"
   ],
   [
    "C39",
    "x_vars[S4,C10]"
   ],
   [
    "C40",
    "x_vars[S5,C1]"
   ],
   [
    "C41",
    "x_vars[S5,C2]"
   ],
   [
    "C42",
    "x_vars[S5,C3]"
   ],
   [
    "C43",
    "x_vars[S5,C4]"
   ],
   [
    "C44",
    "x_vars[S5,C5]"
   ],
   [
    "C45",
    "x_vars[S5,C6]"
   ],
   [
    "C46",
    "x_vars[S5,C7]"
   ],
   [
    "C47",
    "x_vars[S5,C8]"
   ],
   [
    "C48",
    "x_vars[S5,C9]"
   ],
   [
    "C49",
    "x_vars[S5,C10]"
   ],
   [
    "C50",
    "x_vars[S6,C1]"
   ],
   [
    "C51",
    "x_vars[S6,C2]"
   ],
   [
    "C52",
    "x_vars[S6,C3]"
   ],
   [
    "C53",
    "x_vars[S6,C4]"
   ],
   [
    "C54",
    "x_vars[S6,C5]"
   ],
   [
    "C55",
    "x_vars[S6,C6]"
   ],
   [
    "C56",
    "x_vars[S6,C7]"
   ],
   [
    "C57",
    "x_vars[S6,C8]"
   ],
   [
    "C58",
    "x_vars[S6,C9]"
   ],
   [
    "C59",
    "x_vars[S6,C10]"
   ],
   [
    "C60",
    "x_vars[S7,C1]"
   ],
   [
    "C61",
    "x_vars[S7,C2]"
   ],
   [
    "C62",
    "x_vars[S7,C3]"
   ],
   [
    "C63",
    "x_vars[S7,C4]"
   ],
   [
    "C64",
    "x_vars[S7,C5]"
   ],
   [
    "C65",
    "x_vars[S7,C6]"
   ],
   [
    "C66",
    "x_vars[S7,C7]"
   ],
   [
    "C67",
    "x_vars[S7,C8]"
   ],
   [
    "C68",
    "x_vars[S7,C9]"
   ],
   [
    "C69",
    "x_vars[S7,C10]"
   ],
   [
    "C70",
    "x_vars[S8,C1]"
   ],
   [
    "C71",
    "x_vars[S8,C2]"
   ],
   [
    "C72",
    "x_vars[S8,C3]"
   ],
   [
    "C73",
    "x_vars[S8,C4]"
   ],
   [
    "C74",
    "x_vars[S8,C5]"
   ],
   [
    "C75",
    "x_vars[S8,C6]"
   ],
   [
    "C76",
    "x_vars[S8,C7]"
   ],
   [
    "C77",
    "x_vars[S8,C8]"
   ],
   [
    "C78",
    "x_vars[S8,C9]"
   ],
   [
    "C79",
    "x_vars[S8,C10]"
   ],
   [
    "C80",
    "x_vars[S9,C1]"
   ],
   [
    "C81",
    "x_vars[S9,C2]"
   ],
   [
    "C82",
    "x_vars[S9,C3]"
   ],
   [
    "C83",
    "x_vars[S9,C4]"
   ],
   [
    "C84",
    "x_vars[S9,C5]"
   ],
   [
    "C85",
    "x_vars[S9,C6]"
   ],
   [
    "C86",
    "x_vars[S9,C7]"
   ],
   [
    "C87",
    "x_vars[S9,C8]"
   ],
   [
    "C88",
    "x_vars[S9,C9]"
   ],
   [
    "C89",
    "x_vars[S9,C10]"
   ],
   [
    "C90",
    "x_vars[S10,C1]"
   ],
   [
    "C91",
    "x_vars[S10,C2]"
   ],
   [
    "C92",
    "x_vars[S10,C3]"
   ],
   [
    "C93",
    "x_vars[S10,C4]"
   ],
   [
    "C94",
    "x_vars[S10,C5]"
   ],
   [
    "C95",
    "x_vars[S10,C6]"
   ],
   [
    "C96",
    "x_vars[S10,C7]"
   ],
   [
    "C97",
    "x_vars[S10,C8]"
   ],
   [
    "C98",
    "x_vars[S10,C9]"
   ],
   [
    "C99",
    "x_vars[S10,C10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_8": "x_vars[S1,C9]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_8": "x_vars[S2,C9]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_8": "x_vars[S7,C9]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_8": "x_vars[S9,C9]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_8": "x_vars[S10,C9]",
   "x_0_1": "x_vars[S1,C2]",
   "x_1_1": "x_vars[S2,C2]",
   "x_6_1": "x_vars[S7,C2]",
   "x_8_1": "x_vars[S9,C2]",
   "x_9_1": "x_vars[S10,C2]",
   "x_0_3": "x_vars[S1,C4]",
   "x_1_3": "x_vars[S2,C4]",
   "x_6_3": "x_vars[S7,C4]",
   "x_8_3": "x_vars[S9,C4]",
   "x_9_3": "x_vars[S10,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_1_4": "x_vars[S2,C5]",
   "x_6_4": "x_vars[S7,C5]",
   "x_8_4": "x_vars[S9,C5]",
   "x_9_4": "x_vars[S10,C5]",
   "x_0_6": "x_vars[S1,C7]",
   "x_1_6": "x_vars[S2,C7]",
   "x_6_6": "x_vars[S7,C7]",
   "x_8_6": "x_vars[S9,C7]",
   "x_9_6": "x_vars[S10,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_1_7": "x_vars[S2,C8]",
   "x_6_7": "x_vars[S7,C8]",
   "x_8_7": "x_vars[S9,C8]",
   "x_9_7": "x_vars[S10,C8]",
   "x_0_9": "x_vars[S1,C10]",
   "x_1_9": "x_vars[S2,C10]",
   "x_6_9": "x_vars[S7,C10]",
   "x_8_9": "x_vars[S9,C10]",
   "x_9_9": "x_vars[S10,C10]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-007": {
  "source_code_sha256": "8b1f81d0212ef48d67b44f34322c9605a989219e705491315e0ce9ea67b70a1f",
  "original_solution_sha256": "35f7e8418b62a6aa52e6a6a758ea99034d20855b05651a939bb6f779c8a0809e",
  "bindings": [
   [
    "C0",
    "x_vars[S1,D1]"
   ],
   [
    "C1",
    "x_vars[S1,D2]"
   ],
   [
    "C2",
    "x_vars[S1,D3]"
   ],
   [
    "C3",
    "x_vars[S1,D4]"
   ],
   [
    "C4",
    "x_vars[S1,D5]"
   ],
   [
    "C5",
    "x_vars[S2,D1]"
   ],
   [
    "C6",
    "x_vars[S2,D2]"
   ],
   [
    "C7",
    "x_vars[S2,D3]"
   ],
   [
    "C8",
    "x_vars[S2,D4]"
   ],
   [
    "C9",
    "x_vars[S2,D5]"
   ],
   [
    "C10",
    "x_vars[S3,D1]"
   ],
   [
    "C11",
    "x_vars[S3,D2]"
   ],
   [
    "C12",
    "x_vars[S3,D3]"
   ],
   [
    "C13",
    "x_vars[S3,D4]"
   ],
   [
    "C14",
    "x_vars[S3,D5]"
   ],
   [
    "C15",
    "x_vars[S4,D1]"
   ],
   [
    "C16",
    "x_vars[S4,D2]"
   ],
   [
    "C17",
    "x_vars[S4,D3]"
   ],
   [
    "C18",
    "x_vars[S4,D4]"
   ],
   [
    "C19",
    "x_vars[S4,D5]"
   ],
   [
    "C20",
    "x_vars[S5,D1]"
   ],
   [
    "C21",
    "x_vars[S5,D2]"
   ],
   [
    "C22",
    "x_vars[S5,D3]"
   ],
   [
    "C23",
    "x_vars[S5,D4]"
   ],
   [
    "C24",
    "x_vars[S5,D5]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,D1]",
   "x_0_1": "x_vars[S1,D2]",
   "x_0_2": "x_vars[S1,D3]",
   "x_0_3": "x_vars[S1,D4]",
   "x_0_4": "x_vars[S1,D5]",
   "x_1_0": "x_vars[S2,D1]",
   "x_1_1": "x_vars[S2,D2]",
   "x_1_2": "x_vars[S2,D3]",
   "x_1_3": "x_vars[S2,D4]",
   "x_1_4": "x_vars[S2,D5]",
   "x_2_0": "x_vars[S3,D1]",
   "x_2_1": "x_vars[S3,D2]",
   "x_2_2": "x_vars[S3,D3]",
   "x_2_3": "x_vars[S3,D4]",
   "x_2_4": "x_vars[S3,D5]",
   "x_3_0": "x_vars[S4,D1]",
   "x_3_1": "x_vars[S4,D2]",
   "x_3_2": "x_vars[S4,D3]",
   "x_3_3": "x_vars[S4,D4]",
   "x_3_4": "x_vars[S4,D5]",
   "x_4_0": "x_vars[S5,D1]",
   "x_4_1": "x_vars[S5,D2]",
   "x_4_2": "x_vars[S5,D3]",
   "x_4_3": "x_vars[S5,D4]",
   "x_4_4": "x_vars[S5,D5]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-008": {
  "source_code_sha256": "96804db2e68ef1003482abd7d0846f6f139facabaee7a08b99f1bbadafd9c7eb",
  "original_solution_sha256": "5d45a995db09f4196573b0de17bc12170a9cad541aca69ded95259fd83ef6909",
  "bindings": [
   [
    "C0",
    "x_vars[Supplier1,Customer1]"
   ],
   [
    "C1",
    "x_vars[Supplier1,Customer2]"
   ],
   [
    "C2",
    "x_vars[Supplier1,Customer3]"
   ],
   [
    "C3",
    "x_vars[Supplier1,Customer4]"
   ],
   [
    "C4",
    "x_vars[Supplier1,Customer5]"
   ],
   [
    "C5",
    "x_vars[Supplier1,Customer6]"
   ],
   [
    "C6",
    "x_vars[Supplier2,Customer1]"
   ],
   [
    "C7",
    "x_vars[Supplier2,Customer2]"
   ],
   [
    "C8",
    "x_vars[Supplier2,Customer3]"
   ],
   [
    "C9",
    "x_vars[Supplier2,Customer4]"
   ],
   [
    "C10",
    "x_vars[Supplier2,Customer5]"
   ],
   [
    "C11",
    "x_vars[Supplier2,Customer6]"
   ],
   [
    "C12",
    "x_vars[Supplier3,Customer1]"
   ],
   [
    "C13",
    "x_vars[Supplier3,Customer2]"
   ],
   [
    "C14",
    "x_vars[Supplier3,Customer3]"
   ],
   [
    "C15",
    "x_vars[Supplier3,Customer4]"
   ],
   [
    "C16",
    "x_vars[Supplier3,Customer5]"
   ],
   [
    "C17",
    "x_vars[Supplier3,Customer6]"
   ],
   [
    "C18",
    "x_vars[Supplier4,Customer1]"
   ],
   [
    "C19",
    "x_vars[Supplier4,Customer2]"
   ],
   [
    "C20",
    "x_vars[Supplier4,Customer3]"
   ],
   [
    "C21",
    "x_vars[Supplier4,Customer4]"
   ],
   [
    "C22",
    "x_vars[Supplier4,Customer5]"
   ],
   [
    "C23",
    "x_vars[Supplier4,Customer6]"
   ],
   [
    "C24",
    "x_vars[Supplier5,Customer1]"
   ],
   [
    "C25",
    "x_vars[Supplier5,Customer2]"
   ],
   [
    "C26",
    "x_vars[Supplier5,Customer3]"
   ],
   [
    "C27",
    "x_vars[Supplier5,Customer4]"
   ],
   [
    "C28",
    "x_vars[Supplier5,Customer5]"
   ],
   [
    "C29",
    "x_vars[Supplier5,Customer6]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Supplier1,Customer1]",
   "x_0_1": "x_vars[Supplier1,Customer2]",
   "x_0_2": "x_vars[Supplier1,Customer3]",
   "x_0_3": "x_vars[Supplier1,Customer4]",
   "x_0_4": "x_vars[Supplier1,Customer5]",
   "x_0_5": "x_vars[Supplier1,Customer6]",
   "x_1_0": "x_vars[Supplier2,Customer1]",
   "x_1_1": "x_vars[Supplier2,Customer2]",
   "x_1_2": "x_vars[Supplier2,Customer3]",
   "x_1_3": "x_vars[Supplier2,Customer4]",
   "x_1_4": "x_vars[Supplier2,Customer5]",
   "x_1_5": "x_vars[Supplier2,Customer6]",
   "x_2_0": "x_vars[Supplier3,Customer1]",
   "x_2_1": "x_vars[Supplier3,Customer2]",
   "x_2_2": "x_vars[Supplier3,Customer3]",
   "x_2_3": "x_vars[Supplier3,Customer4]",
   "x_2_4": "x_vars[Supplier3,Customer5]",
   "x_2_5": "x_vars[Supplier3,Customer6]",
   "x_3_0": "x_vars[Supplier4,Customer1]",
   "x_3_1": "x_vars[Supplier4,Customer2]",
   "x_3_2": "x_vars[Supplier4,Customer3]",
   "x_3_3": "x_vars[Supplier4,Customer4]",
   "x_3_4": "x_vars[Supplier4,Customer5]",
   "x_3_5": "x_vars[Supplier4,Customer6]",
   "x_4_0": "x_vars[Supplier5,Customer1]",
   "x_4_1": "x_vars[Supplier5,Customer2]",
   "x_4_2": "x_vars[Supplier5,Customer3]",
   "x_4_3": "x_vars[Supplier5,Customer4]",
   "x_4_4": "x_vars[Supplier5,Customer5]",
   "x_4_5": "x_vars[Supplier5,Customer6]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-009": {
  "source_code_sha256": "d4890b5f51209188242de9e4bc24dc639b45adae05ad1ea41d1d7b0e7ce7d3c4",
  "original_solution_sha256": "5109ec802a33962fb5da569d909759c713ff6a56cd702641611b3ef7840b820b",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S2,C1]"
   ],
   [
    "C5",
    "x_vars[S2,C2]"
   ],
   [
    "C6",
    "x_vars[S2,C3]"
   ],
   [
    "C7",
    "x_vars[S2,C4]"
   ],
   [
    "C8",
    "x_vars[S3,C1]"
   ],
   [
    "C9",
    "x_vars[S3,C2]"
   ],
   [
    "C10",
    "x_vars[S3,C3]"
   ],
   [
    "C11",
    "x_vars[S3,C4]"
   ],
   [
    "C12",
    "x_vars[S4,C1]"
   ],
   [
    "C13",
    "x_vars[S4,C2]"
   ],
   [
    "C14",
    "x_vars[S4,C3]"
   ],
   [
    "C15",
    "x_vars[S4,C4]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-010": {
  "source_code_sha256": "4ce6f0ec8adc60060f39c94e7e2cf27dd59db84da5392e8b01a7ad4a49e3e280",
  "original_solution_sha256": "56cffd5a63e43b70d8a7822bf4e1b5444ea8d5cd3cf8a7e3f257cb408a8780ce",
  "bindings": [
   [
    "C0",
    "x_vars[Adams Group_service]"
   ],
   [
    "C1",
    "x_vars[Anderson-Leach_against]"
   ],
   [
    "C2",
    "x_vars[Anderson-Valdez_somebody]"
   ],
   [
    "C3",
    "x_vars[Anderson-White_son]"
   ],
   [
    "C4",
    "x_vars[Andrews LLC_matter]"
   ],
   [
    "C5",
    "x_vars[Andrews PLC_enter]"
   ],
   [
    "C6",
    "x_vars[Andrews-Martin_build]"
   ],
   [
    "C7",
    "x_vars[Arias-Mendoza_life]"
   ],
   [
    "C8",
    "x_vars[Bennett and Sons_down]"
   ],
   [
    "C9",
    "x_vars[Bennett, Foster and Moreno_enter]"
   ],
   [
    "C10",
    "x_vars[Fernandez, Long and Nelson_member]"
   ],
   [
    "C11",
    "x_vars[Fernandez-Fischer_million]"
   ],
   [
    "C12",
    "x_vars[Ferrell Inc_prove]"
   ],
   [
    "C13",
    "x_vars[Fields, Christensen and Daniels_nor]"
   ],
   [
    "C14",
    "x_vars[Figueroa LLC_involve]"
   ],
   [
    "C15",
    "x_vars[Finley Group_itself]"
   ],
   [
    "C16",
    "x_vars[Fisher Ltd_speak]"
   ],
   [
    "C17",
    "x_vars[Fisher-Marshall_cover]"
   ],
   [
    "C18",
    "x_vars[Fuller-Torres_behavior]"
   ],
   [
    "C19",
    "x_vars[Fuller-Walters_radio]"
   ],
   [
    "C20",
    "x_vars[Gallagher-Campbell_dark]"
   ],
   [
    "C21",
    "x_vars[Gallagher-Kirby_talk]"
   ],
   [
    "C22",
    "x_vars[Gallagher-Parker_recognize]"
   ],
   [
    "C23",
    "x_vars[Gonzalez PLC_task]"
   ],
   [
    "C24",
    "x_vars[Gonzalez, Coleman and Le_heavy]"
   ],
   [
    "C25",
    "x_vars[Gonzalez, Lowe and Robinson_education]"
   ],
   [
    "C26",
    "x_vars[Gonzalez-Horn_light]"
   ],
   [
    "C27",
    "x_vars[Good, Davis and Smith_station]"
   ],
   [
    "C28",
    "x_vars[Goodman, Hughes and White_realize]"
   ],
   [
    "C29",
    "x_vars[Goodwin PLC_bad]"
   ],
   [
    "C30",
    "x_vars[Gordon PLC_detail]"
   ],
   [
    "C31",
    "x_vars[Graham LLC_stage]"
   ],
   [
    "C32",
    "x_vars[Graham Ltd_marriage]"
   ],
   [
    "C33",
    "x_vars[Graham-Swanson_message]"
   ],
   [
    "C34",
    "x_vars[Grant, Cross and Bennett_religious]"
   ],
   [
    "C35",
    "x_vars[Grant, Mcdonald and Watson_owner]"
   ],
   [
    "C36",
    "x_vars[Graves, Turner and Crawford_wait]"
   ],
   [
    "C37",
    "x_vars[Gray, Smith and Barnes_much]"
   ],
   [
    "C38",
    "x_vars[Green Inc_direction]"
   ],
   [
    "C39",
    "x_vars[Green-Rogers_could]"
   ],
   [
    "C40",
    "x_vars[Greene-Baxter_them]"
   ],
   [
    "C41",
    "x_vars[Greer and Sons_keep]"
   ],
   [
    "C42",
    "x_vars[Griffin, Boyle and Dawson_anything]"
   ],
   [
    "C43",
    "x_vars[Harper Inc_well]"
   ],
   [
    "C44",
    "x_vars[Harper Ltd_off]"
   ],
   [
    "C45",
    "x_vars[Harrington, Sosa and Mccarty_coach]"
   ],
   [
    "C46",
    "x_vars[Harris Group_different]"
   ],
   [
    "C47",
    "x_vars[Harris and Sons_audience]"
   ],
   [
    "C48",
    "x_vars[Harris and Sons_fear]"
   ],
   [
    "C49",
    "x_vars[Harris, Hamilton and Rose_contain]"
   ],
   [
    "C50",
    "x_vars[Harris, Stevens and Hall_answer]"
   ],
   [
    "C51",
    "x_vars[Harris-Bell_painting]"
   ],
   [
    "C52",
    "x_vars[Harris-Melton_often]"
   ],
   [
    "C53",
    "x_vars[Harris-Rogers_after]"
   ],
   [
    "C54",
    "x_vars[Hart Group_cold]"
   ],
   [
    "C55",
    "x_vars[Hutchinson, Roberts and Mcbride_among]"
   ],
   [
    "C56",
    "x_vars[Ibarra, Jackson and Potter_upon]"
   ],
   [
    "C57",
    "x_vars[Jackson Inc_anyone]"
   ],
   [
    "C58",
    "x_vars[Jackson LLC_report]"
   ],
   [
    "C59",
    "x_vars[Jackson, Collier and Barber_result]"
   ],
   [
    "C60",
    "x_vars[Jackson, White and Brown_expect]"
   ],
   [
    "C61",
    "x_vars[Jackson-Carroll_six]"
   ],
   [
    "C62",
    "x_vars[Lee-Jones_soldier]"
   ],
   [
    "C63",
    "x_vars[Lewis, Sanchez and Turner_point]"
   ],
   [
    "C64",
    "x_vars[Lin LLC_performance]"
   ],
   [
    "C65",
    "x_vars[Lindsey, Avila and Brown_candidate]"
   ],
   [
    "C66",
    "x_vars[Lindsey-Strickland_modern]"
   ],
   [
    "C67",
    "x_vars[Little-Perez_choice]"
   ],
   [
    "C68",
    "x_vars[Lloyd, Stone and Mcguire_national]"
   ],
   [
    "C69",
    "x_vars[Long, Hughes and Gallegos_receive]"
   ],
   [
    "C70",
    "x_vars[Lopez PLC_analysis]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Adams Group_service]",
   "x_2": "x_vars[Anderson-Leach_against]",
   "x_3": "x_vars[Anderson-Valdez_somebody]",
   "x_4": "x_vars[Anderson-White_son]",
   "x_5": "x_vars[Andrews LLC_matter]",
   "x_6": "x_vars[Andrews PLC_enter]",
   "x_7": "x_vars[Andrews-Martin_build]",
   "x_8": "x_vars[Arias-Mendoza_life]",
   "x_9": "x_vars[Bennett and Sons_down]",
   "x_10": "x_vars[Bennett, Foster and Moreno_enter]",
   "x_11": "x_vars[Fernandez, Long and Nelson_member]",
   "x_12": "x_vars[Fernandez-Fischer_million]",
   "x_13": "x_vars[Ferrell Inc_prove]",
   "x_14": "x_vars[Fields, Christensen and Daniels_nor]",
   "x_15": "x_vars[Figueroa LLC_involve]",
   "x_16": "x_vars[Finley Group_itself]",
   "x_17": "x_vars[Fisher Ltd_speak]",
   "x_18": "x_vars[Fisher-Marshall_cover]",
   "x_19": "x_vars[Fuller-Torres_behavior]",
   "x_20": "x_vars[Fuller-Walters_radio]",
   "x_21": "x_vars[Gallagher-Campbell_dark]",
   "x_22": "x_vars[Gallagher-Kirby_talk]",
   "x_23": "x_vars[Gallagher-Parker_recognize]",
   "x_24": "x_vars[Gonzalez PLC_task]",
   "x_25": "x_vars[Gonzalez, Coleman and Le_heavy]",
   "x_26": "x_vars[Gonzalez, Lowe and Robinson_education]",
   "x_27": "x_vars[Gonzalez-Horn_light]",
   "x_28": "x_vars[Good, Davis and Smith_station]",
   "x_29": "x_vars[Goodman, Hughes and White_realize]",
   "x_30": "x_vars[Goodwin PLC_bad]",
   "x_31": "x_vars[Gordon PLC_detail]",
   "x_32": "x_vars[Graham LLC_stage]",
   "x_33": "x_vars[Graham Ltd_marriage]",
   "x_34": "x_vars[Graham-Swanson_message]",
   "x_35": "x_vars[Grant, Cross and Bennett_religious]",
   "x_36": "x_vars[Grant, Mcdonald and Watson_owner]",
   "x_37": "x_vars[Graves, Turner and Crawford_wait]",
   "x_38": "x_vars[Gray, Smith and Barnes_much]",
   "x_39": "x_vars[Green Inc_direction]",
   "x_40": "x_vars[Green-Rogers_could]",
   "x_41": "x_vars[Greene-Baxter_them]",
   "x_42": "x_vars[Greer and Sons_keep]",
   "x_43": "x_vars[Griffin, Boyle and Dawson_anything]",
   "x_44": "x_vars[Harper Inc_well]",
   "x_45": "x_vars[Harper Ltd_off]",
   "x_46": "x_vars[Harrington, Sosa and Mccarty_coach]",
   "x_47": "x_vars[Harris Group_different]",
   "x_48": "x_vars[Harris and Sons_audience]",
   "x_49": "x_vars[Harris and Sons_fear]",
   "x_50": "x_vars[Harris, Hamilton and Rose_contain]",
   "x_51": "x_vars[Harris, Stevens and Hall_answer]",
   "x_52": "x_vars[Harris-Bell_painting]",
   "x_53": "x_vars[Harris-Melton_often]",
   "x_54": "x_vars[Harris-Rogers_after]",
   "x_55": "x_vars[Hart Group_cold]",
   "x_56": "x_vars[Hutchinson, Roberts and Mcbride_among]",
   "x_57": "x_vars[Ibarra, Jackson and Potter_upon]",
   "x_58": "x_vars[Jackson Inc_anyone]",
   "x_59": "x_vars[Jackson LLC_report]",
   "x_60": "x_vars[Jackson, Collier and Barber_result]",
   "x_61": "x_vars[Jackson, White and Brown_expect]",
   "x_62": "x_vars[Jackson-Carroll_six]",
   "x_63": "x_vars[Lee-Jones_soldier]",
   "x_64": "x_vars[Lewis, Sanchez and Turner_point]",
   "x_65": "x_vars[Lin LLC_performance]",
   "x_66": "x_vars[Lindsey, Avila and Brown_candidate]",
   "x_67": "x_vars[Lindsey-Strickland_modern]",
   "x_68": "x_vars[Little-Perez_choice]",
   "x_69": "x_vars[Lloyd, Stone and Mcguire_national]",
   "x_70": "x_vars[Long, Hughes and Gallegos_receive]",
   "x_71": "x_vars[Lopez PLC_analysis]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-011": {
  "source_code_sha256": "4719be4724cf6a92d9faa86442069fdccf15db0ce36be90900f470b23ef3dca0",
  "original_solution_sha256": "68203b0c38371b684628431f6920ddc720a25bd2b165f6fb4fdc65d32eebc533",
  "bindings": [
   [
    "C0",
    "x_vars[id999]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[id999]"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-012": {
  "source_code_sha256": "4ae8594a9a62d3a4b3621e9ad8a1556ffd38176270e3b6d5db7ca082db8df8e9",
  "original_solution_sha256": "ffe95067bb7d348f9095f922b44dc1c6aa9280ab237c8d02ff9ab509a52b858e",
  "bindings": [
   [
    "C0",
    "x_vars[1984 by George Orwell]"
   ],
   [
    "C1",
    "x_vars[Adidas 3-Stripes Shorts]"
   ],
   [
    "C2",
    "x_vars[Adidas Essential Track Pants]"
   ],
   [
    "C3",
    "x_vars[Adidas FIFA World Cup Football]"
   ],
   [
    "C4",
    "x_vars[Adidas Originals Superstar Sneakers]"
   ],
   [
    "C5",
    "x_vars[Adidas Originals Trefoil Hoodie]"
   ],
   [
    "C6",
    "x_vars[Adidas Ultraboost Running Shoes]"
   ],
   [
    "C7",
    "x_vars[Adidas Ultraboost Shoes]"
   ],
   [
    "C8",
    "x_vars[Amazon Echo Dot (4th Gen)]"
   ],
   [
    "C9",
    "x_vars[Amazon Echo Show 10]"
   ],
   [
    "C10",
    "x_vars[Amazon Fire TV Stick 4K]"
   ],
   [
    "C11",
    "x_vars[Anastasia Beverly Hills Brow Wiz]"
   ],
   [
    "C12",
    "x_vars[Anker PowerCore Portable Charger]"
   ],
   [
    "C13",
    "x_vars[Anova Precision Cooker]"
   ],
   [
    "C14",
    "x_vars[Anova Precision Oven]"
   ],
   [
    "C15",
    "x_vars[Apple AirPods Max]"
   ],
   [
    "C16",
    "x_vars[Apple AirPods Pro]"
   ],
   [
    "C17",
    "x_vars[Apple MacBook Air]"
   ],
   [
    "C18",
    "x_vars[Apple MacBook Pro 16-inch]"
   ],
   [
    "C19",
    "x_vars[Apple TV 4K]"
   ],
   [
    "C20",
    "x_vars[Apple Watch Series 8]"
   ],
   [
    "C21",
    "x_vars[Apple iPad Air]"
   ],
   [
    "C22",
    "x_vars[Atomic Habits by James Clear]"
   ],
   [
    "C23",
    "x_vars[Babolat Pure Drive Tennis Racket]"
   ],
   [
    "C24",
    "x_vars[Becoming by Michelle Obama]"
   ],
   [
    "C25",
    "x_vars[Biore UV Aqua Rich Watery Essence Sunscreen]"
   ],
   [
    "C26",
    "x_vars[Blueair Classic 480i]"
   ],
   [
    "C27",
    "x_vars[Bose QuietComfort 35 Headphones]"
   ],
   [
    "C28",
    "x_vars[Bose QuietComfort 35 II Wireless Headphones]"
   ],
   [
    "C29",
    "x_vars[Bose SoundLink Color Bluetooth Speaker II]"
   ],
   [
    "C30",
    "x_vars[Bose SoundLink Revolve+ Speaker]"
   ],
   [
    "C31",
    "x_vars[Bose SoundSport Wireless Earbuds]"
   ],
   [
    "C32",
    "x_vars[Bowflex SelectTech 1090 Adjustable Dumbbells]"
   ],
   [
    "C33",
    "x_vars[Bowflex SelectTech 552 Dumbbells]"
   ],
   [
    "C34",
    "x_vars[Breville Nespresso Creatista Plus]"
   ],
   [
    "C35",
    "x_vars[Breville Smart Coffee Grinder Pro]"
   ],
   [
    "C36",
    "x_vars[Breville Smart Grill]"
   ],
   [
    "C37",
    "x_vars[Breville Smart Oven]"
   ],
   [
    "C38",
    "x_vars[Calvin Klein Boxer Briefs]"
   ],
   [
    "C39",
    "x_vars[Canon EOS R5 Camera]"
   ],
   [
    "C40",
    "x_vars[Canon EOS Rebel T7i DSLR Camera]"
   ],
   [
    "C41",
    "x_vars[Caudalie Vinoperfect Radiance Serum]"
   ],
   [
    "C42",
    "x_vars[CeraVe Hydrating Facial Cleanser]"
   ],
   [
    "C43",
    "x_vars[Champion Reverse Weave Hoodie]"
   ],
   [
    "C44",
    "x_vars[Chanel No. 5 Perfume]"
   ],
   [
    "C45",
    "x_vars[Charlotte Tilbury Magic Cream]"
   ],
   [
    "C46",
    "x_vars[Clinique Dramatically Different Moisturizing Lotion]"
   ],
   [
    "C47",
    "x_vars[Clinique Moisture Surge]"
   ],
   [
    "C48",
    "x_vars[Columbia Fleece Jacket]"
   ],
   [
    "C49",
    "x_vars[Crock-Pot 6-Quart Slow Cooker]"
   ],
   [
    "C50",
    "x_vars[Cuisinart Coffee Center]"
   ],
   [
    "C51",
    "x_vars[Cuisinart Custom 14-Cup Food Processor]"
   ],
   [
    "C52",
    "x_vars[Cuisinart Griddler Deluxe]"
   ],
   [
    "C53",
    "x_vars[De'Longhi Magnifica Espresso Machine]"
   ],
   [
    "C54",
    "x_vars[Dr. Jart+ Cicapair Tiger Grass Color Correcting Treatment]"
   ],
   [
    "C55",
    "x_vars[Drunk Elephant C-Firma Day Serum]"
   ],
   [
    "C56",
    "x_vars[Dune by Frank Herbert]"
   ],
   [
    "C57",
    "x_vars[Dyson Pure Cool Link]"
   ],
   [
    "C58",
    "x_vars[Dyson Supersonic Hair Dryer]"
   ],
   [
    "C59",
    "x_vars[Dyson V11 Vacuum]"
   ],
   [
    "C60",
    "x_vars[Dyson V8 Absolute]"
   ],
   [
    "C61",
    "x_vars[Educated by Tara Westover]"
   ],
   [
    "C62",
    "x_vars[Estee Lauder Advanced Night Repair]"
   ],
   [
    "C63",
    "x_vars[Eufy RoboVac 11S]"
   ],
   [
    "C64",
    "x_vars[Fenty Beauty Killawatt Highlighter]"
   ],
   [
    "C65",
    "x_vars[First Aid Beauty Ultra Repair Cream]"
   ],
   [
    "C66",
    "x_vars[Fitbit Charge 5]"
   ],
   [
    "C67",
    "x_vars[Fitbit Inspire 2]"
   ],
   [
    "C68",
    "x_vars[Fitbit Luxe]"
   ],
   [
    "C69",
    "x_vars[Fitbit Versa 3]"
   ],
   [
    "C70",
    "x_vars[Forever 21 Graphic Tee]"
   ],
   [
    "C71",
    "x_vars[Fresh Sugar Lip Treatment]"
   ],
   [
    "C72",
    "x_vars[Gap 1969 Original Fit Jeans]"
   ],
   [
    "C73",
    "x_vars[Gap Crewneck Sweatshirt]"
   ],
   [
    "C74",
    "x_vars[Gap Essential Crewneck T-Shirt]"
   ],
   [
    "C75",
    "x_vars[Gap High Rise Skinny Jeans]"
   ],
   [
    "C76",
    "x_vars[Garmin Edge 530]"
   ],
   [
    "C77",
    "x_vars[Garmin Fenix 6X Pro]"
   ],
   [
    "C78",
    "x_vars[Garmin Forerunner 245]"
   ],
   [
    "C79",
    "x_vars[Garmin Forerunner 945]"
   ],
   [
    "C80",
    "x_vars[GlamGlow Supermud Clearing Treatment]"
   ],
   [
    "C81",
    "x_vars[Glossier Boy Brow]"
   ],
   [
    "C82",
    "x_vars[Glossier Cloud Paint]"
   ],
   [
    "C83",
    "x_vars[GoPro HERO10 Black]"
   ],
   [
    "C84",
    "x_vars[GoPro HERO9 Black]"
   ],
   [
    "C85",
    "x_vars[Gone Girl by Gillian Flynn]"
   ],
   [
    "C86",
    "x_vars[Google Nest Hub Max]"
   ],
   [
    "C87",
    "x_vars[Google Nest Wifi Router]"
   ],
   [
    "C88",
    "x_vars[Google Pixel 6 Pro]"
   ],
   [
    "C89",
    "x_vars[Google Pixelbook Go]"
   ],
   [
    "C90",
    "x_vars[H&M Slim Fit Jeans]"
   ],
   [
    "C91",
    "x_vars[HP Spectre x360 Laptop]"
   ],
   [
    "C92",
    "x_vars[Hamilton Beach FlexBrew Coffee Maker]"
   ],
   [
    "C93",
    "x_vars[Hanes ComfortSoft T-Shirt]"
   ],
   [
    "C94",
    "x_vars[Harry Potter and the Sorcerer's Stone]"
   ],
   [
    "C95",
    "x_vars[Hydro Flask Standard Mouth Water Bottle]"
   ],
   [
    "C96",
    "x_vars[Hydro Flask Wide Mouth Water Bottle]"
   ],
   [
    "C97",
    "x_vars[Hyperice Hypervolt Massager]"
   ],
   [
    "C98",
    "x_vars[Instant Pot Duo]"
   ],
   [
    "C99",
    "x_vars[Instant Pot Duo Crisp]"
   ],
   [
    "C100",
    "x_vars[Instant Pot Duo Evo Plus]"
   ],
   [
    "C101",
    "x_vars[Instant Pot Duo Nova]"
   ],
   [
    "C102",
    "x_vars[Instant Pot Ultra]"
   ],
   [
    "C103",
    "x_vars[Keurig K-Elite Coffee Maker]"
   ],
   [
    "C104",
    "x_vars[Keurig K-Mini Coffee Maker]"
   ],
   [
    "C105",
    "x_vars[Kiehl's Midnight Recovery Concentrate]"
   ],
   [
    "C106",
    "x_vars[Kindle Paperwhite]"
   ],
   [
    "C107",
    "x_vars[KitchenAid Artisan Stand Mixer]"
   ],
   [
    "C108",
    "x_vars[KitchenAid Stand Mixer]"
   ],
   [
    "C109",
    "x_vars[L'Occitane Shea Butter Hand Cream]"
   ],
   [
    "C110",
    "x_vars[L'Oreal Revitalift Serum]"
   ],
   [
    "C111",
    "x_vars[LG OLED TV]"
   ],
   [
    "C112",
    "x_vars[La Mer Cr_¨me de la Mer Moisturizer]"
   ],
   [
    "C113",
    "x_vars[Lancome La Vie Est Belle]"
   ],
   [
    "C114",
    "x_vars[Laneige Water Sleeping Mask]"
   ],
   [
    "C115",
    "x_vars[Levi's 501 Jeans]"
   ],
   [
    "C116",
    "x_vars[Levi's 511 Slim Fit Jeans]"
   ],
   [
    "C117",
    "x_vars[Levi's Sherpa Trucker Jacket]"
   ],
   [
    "C118",
    "x_vars[Levi's Trucker Jacket]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[1984 by George Orwell]",
   "x_2": "x_vars[Adidas 3-Stripes Shorts]",
   "x_3": "x_vars[Adidas Essential Track Pants]",
   "x_4": "x_vars[Adidas FIFA World Cup Football]",
   "x_5": "x_vars[Adidas Originals Superstar Sneakers]",
   "x_6": "x_vars[Adidas Originals Trefoil Hoodie]",
   "x_7": "x_vars[Adidas Ultraboost Running Shoes]",
   "x_8": "x_vars[Adidas Ultraboost Shoes]",
   "x_9": "x_vars[Amazon Echo Dot (4th Gen)]",
   "x_10": "x_vars[Amazon Echo Show 10]",
   "x_11": "x_vars[Amazon Fire TV Stick 4K]",
   "x_12": "x_vars[Anastasia Beverly Hills Brow Wiz]",
   "x_13": "x_vars[Anker PowerCore Portable Charger]",
   "x_14": "x_vars[Anova Precision Cooker]",
   "x_15": "x_vars[Anova Precision Oven]",
   "x_16": "x_vars[Apple AirPods Max]",
   "x_17": "x_vars[Apple AirPods Pro]",
   "x_18": "x_vars[Apple MacBook Air]",
   "x_19": "x_vars[Apple MacBook Pro 16-inch]",
   "x_20": "x_vars[Apple TV 4K]",
   "x_21": "x_vars[Apple Watch Series 8]",
   "x_22": "x_vars[Apple iPad Air]",
   "x_23": "x_vars[Atomic Habits by James Clear]",
   "x_24": "x_vars[Babolat Pure Drive Tennis Racket]",
   "x_25": "x_vars[Becoming by Michelle Obama]",
   "x_26": "x_vars[Biore UV Aqua Rich Watery Essence Sunscreen]",
   "x_27": "x_vars[Blueair Classic 480i]",
   "x_28": "x_vars[Bose QuietComfort 35 Headphones]",
   "x_29": "x_vars[Bose QuietComfort 35 II Wireless Headphones]",
   "x_30": "x_vars[Bose SoundLink Color Bluetooth Speaker II]",
   "x_31": "x_vars[Bose SoundLink Revolve+ Speaker]",
   "x_32": "x_vars[Bose SoundSport Wireless Earbuds]",
   "x_33": "x_vars[Bowflex SelectTech 1090 Adjustable Dumbbells]",
   "x_34": "x_vars[Bowflex SelectTech 552 Dumbbells]",
   "x_35": "x_vars[Breville Nespresso Creatista Plus]",
   "x_36": "x_vars[Breville Smart Coffee Grinder Pro]",
   "x_37": "x_vars[Breville Smart Grill]",
   "x_38": "x_vars[Breville Smart Oven]",
   "x_39": "x_vars[Calvin Klein Boxer Briefs]",
   "x_40": "x_vars[Canon EOS R5 Camera]",
   "x_41": "x_vars[Canon EOS Rebel T7i DSLR Camera]",
   "x_42": "x_vars[Caudalie Vinoperfect Radiance Serum]",
   "x_43": "x_vars[CeraVe Hydrating Facial Cleanser]",
   "x_44": "x_vars[Champion Reverse Weave Hoodie]",
   "x_45": "x_vars[Chanel No. 5 Perfume]",
   "x_46": "x_vars[Charlotte Tilbury Magic Cream]",
   "x_47": "x_vars[Clinique Dramatically Different Moisturizing Lotion]",
   "x_48": "x_vars[Clinique Moisture Surge]",
   "x_49": "x_vars[Columbia Fleece Jacket]",
   "x_50": "x_vars[Crock-Pot 6-Quart Slow Cooker]",
   "x_51": "x_vars[Cuisinart Coffee Center]",
   "x_52": "x_vars[Cuisinart Custom 14-Cup Food Processor]",
   "x_53": "x_vars[Cuisinart Griddler Deluxe]",
   "x_54": "x_vars[De'Longhi Magnifica Espresso Machine]",
   "x_55": "x_vars[Dr. Jart+ Cicapair Tiger Grass Color Correcting Treatment]",
   "x_56": "x_vars[Drunk Elephant C-Firma Day Serum]",
   "x_57": "x_vars[Dune by Frank Herbert]",
   "x_58": "x_vars[Dyson Pure Cool Link]",
   "x_59": "x_vars[Dyson Supersonic Hair Dryer]",
   "x_60": "x_vars[Dyson V11 Vacuum]",
   "x_61": "x_vars[Dyson V8 Absolute]",
   "x_62": "x_vars[Educated by Tara Westover]",
   "x_63": "x_vars[Estee Lauder Advanced Night Repair]",
   "x_64": "x_vars[Eufy RoboVac 11S]",
   "x_65": "x_vars[Fenty Beauty Killawatt Highlighter]",
   "x_66": "x_vars[First Aid Beauty Ultra Repair Cream]",
   "x_67": "x_vars[Fitbit Charge 5]",
   "x_68": "x_vars[Fitbit Inspire 2]",
   "x_69": "x_vars[Fitbit Luxe]",
   "x_70": "x_vars[Fitbit Versa 3]",
   "x_71": "x_vars[Forever 21 Graphic Tee]",
   "x_72": "x_vars[Fresh Sugar Lip Treatment]",
   "x_73": "x_vars[Gap 1969 Original Fit Jeans]",
   "x_74": "x_vars[Gap Crewneck Sweatshirt]",
   "x_75": "x_vars[Gap Essential Crewneck T-Shirt]",
   "x_76": "x_vars[Gap High Rise Skinny Jeans]",
   "x_77": "x_vars[Garmin Edge 530]",
   "x_78": "x_vars[Garmin Fenix 6X Pro]",
   "x_79": "x_vars[Garmin Forerunner 245]",
   "x_80": "x_vars[Garmin Forerunner 945]",
   "x_81": "x_vars[GlamGlow Supermud Clearing Treatment]",
   "x_82": "x_vars[Glossier Boy Brow]",
   "x_83": "x_vars[Glossier Cloud Paint]",
   "x_84": "x_vars[GoPro HERO10 Black]",
   "x_85": "x_vars[GoPro HERO9 Black]",
   "x_86": "x_vars[Gone Girl by Gillian Flynn]",
   "x_87": "x_vars[Google Nest Hub Max]",
   "x_88": "x_vars[Google Nest Wifi Router]",
   "x_89": "x_vars[Google Pixel 6 Pro]",
   "x_90": "x_vars[Google Pixelbook Go]",
   "x_91": "x_vars[H&M Slim Fit Jeans]",
   "x_92": "x_vars[HP Spectre x360 Laptop]",
   "x_93": "x_vars[Hamilton Beach FlexBrew Coffee Maker]",
   "x_94": "x_vars[Hanes ComfortSoft T-Shirt]",
   "x_95": "x_vars[Harry Potter and the Sorcerer's Stone]",
   "x_96": "x_vars[Hydro Flask Standard Mouth Water Bottle]",
   "x_97": "x_vars[Hydro Flask Wide Mouth Water Bottle]",
   "x_98": "x_vars[Hyperice Hypervolt Massager]",
   "x_99": "x_vars[Instant Pot Duo]",
   "x_100": "x_vars[Instant Pot Duo Crisp]",
   "x_101": "x_vars[Instant Pot Duo Evo Plus]",
   "x_102": "x_vars[Instant Pot Duo Nova]",
   "x_103": "x_vars[Instant Pot Ultra]",
   "x_104": "x_vars[Keurig K-Elite Coffee Maker]",
   "x_105": "x_vars[Keurig K-Mini Coffee Maker]",
   "x_106": "x_vars[Kiehl's Midnight Recovery Concentrate]",
   "x_107": "x_vars[Kindle Paperwhite]",
   "x_108": "x_vars[KitchenAid Artisan Stand Mixer]",
   "x_109": "x_vars[KitchenAid Stand Mixer]",
   "x_110": "x_vars[L'Occitane Shea Butter Hand Cream]",
   "x_111": "x_vars[L'Oreal Revitalift Serum]",
   "x_112": "x_vars[LG OLED TV]",
   "x_113": "x_vars[La Mer Cr_¨me de la Mer Moisturizer]",
   "x_114": "x_vars[Lancome La Vie Est Belle]",
   "x_115": "x_vars[Laneige Water Sleeping Mask]",
   "x_116": "x_vars[Levi's 501 Jeans]",
   "x_117": "x_vars[Levi's 511 Slim Fit Jeans]",
   "x_118": "x_vars[Levi's Sherpa Trucker Jacket]",
   "x_119": "x_vars[Levi's Trucker Jacket]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-013": {
  "source_code_sha256": "0b156960c2e1be592e319e91ca83deb033065e2d6521e1d39d9c50f028b7847b",
  "original_solution_sha256": "78ac7981f5eb5af57c28b181727bffaef0f9fcb3bc848f14a6137c55d5293446",
  "bindings": [
   [
    "C0",
    "x_vars[4U_Service_22]"
   ],
   [
    "C1",
    "x_vars[4U_Service_36]"
   ],
   [
    "C2",
    "x_vars[4U_Service_7]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[4U_Service_22]",
   "x_2": "x_vars[4U_Service_36]",
   "x_3": "x_vars[4U_Service_7]"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-014": {
  "source_code_sha256": "ae6dbbcf71c8284bdd7e3122b7338458faa0805b08b47149e1181101d3d2997c",
  "original_solution_sha256": "62176e138bbf4a4e705142dd1b3963137c511d7cb0c41dfb54ed2dce188be1cb",
  "bindings": [
   [
    "C0",
    "x_vars[bbq_ckn_l]"
   ],
   [
    "C1",
    "x_vars[bbq_ckn_m]"
   ],
   [
    "C2",
    "x_vars[bbq_ckn_s]"
   ],
   [
    "C3",
    "x_vars[big_meat_s]"
   ],
   [
    "C4",
    "x_vars[brie_carre_s]"
   ],
   [
    "C5",
    "x_vars[calabrese_l]"
   ],
   [
    "C6",
    "x_vars[calabrese_m]"
   ],
   [
    "C7",
    "x_vars[calabrese_s]"
   ],
   [
    "C8",
    "x_vars[cali_ckn_l]"
   ],
   [
    "C9",
    "x_vars[cali_ckn_m]"
   ],
   [
    "C10",
    "x_vars[cali_ckn_s]"
   ],
   [
    "C11",
    "x_vars[ckn_alfredo_l]"
   ],
   [
    "C12",
    "x_vars[ckn_alfredo_m]"
   ],
   [
    "C13",
    "x_vars[ckn_alfredo_s]"
   ],
   [
    "C14",
    "x_vars[ckn_pesto_l]"
   ],
   [
    "C15",
    "x_vars[ckn_pesto_m]"
   ],
   [
    "C16",
    "x_vars[ckn_pesto_s]"
   ],
   [
    "C17",
    "x_vars[classic_dlx_l]"
   ],
   [
    "C18",
    "x_vars[classic_dlx_m]"
   ],
   [
    "C19",
    "x_vars[classic_dlx_s]"
   ],
   [
    "C20",
    "x_vars[five_cheese_l]"
   ],
   [
    "C21",
    "x_vars[four_cheese_l]"
   ],
   [
    "C22",
    "x_vars[four_cheese_m]"
   ],
   [
    "C23",
    "x_vars[green_garden_l]"
   ],
   [
    "C24",
    "x_vars[green_garden_m]"
   ],
   [
    "C25",
    "x_vars[green_garden_s]"
   ],
   [
    "C26",
    "x_vars[hawaiian_l]"
   ],
   [
    "C27",
    "x_vars[hawaiian_m]"
   ],
   [
    "C28",
    "x_vars[hawaiian_s]"
   ],
   [
    "C29",
    "x_vars[ital_cpcllo_l]"
   ],
   [
    "C30",
    "x_vars[ital_cpcllo_m]"
   ],
   [
    "C31",
    "x_vars[ital_cpcllo_s]"
   ],
   [
    "C32",
    "x_vars[ital_supr_l]"
   ],
   [
    "C33",
    "x_vars[ital_supr_m]"
   ],
   [
    "C34",
    "x_vars[ital_supr_s]"
   ],
   [
    "C35",
    "x_vars[ital_veggie_l]"
   ],
   [
    "C36",
    "x_vars[ital_veggie_m]"
   ],
   [
    "C37",
    "x_vars[ital_veggie_s]"
   ],
   [
    "C38",
    "x_vars[mediterraneo_l]"
   ],
   [
    "C39",
    "x_vars[mediterraneo_m]"
   ],
   [
    "C40",
    "x_vars[mediterraneo_s]"
   ],
   [
    "C41",
    "x_vars[mexicana_l]"
   ],
   [
    "C42",
    "x_vars[mexicana_m]"
   ],
   [
    "C43",
    "x_vars[mexicana_s]"
   ],
   [
    "C44",
    "x_vars[napolitana_l]"
   ],
   [
    "C45",
    "x_vars[napolitana_m]"
   ],
   [
    "C46",
    "x_vars[napolitana_s]"
   ],
   [
    "C47",
    "x_vars[pep_msh_pep_l]"
   ],
   [
    "C48",
    "x_vars[pep_msh_pep_m]"
   ],
   [
    "C49",
    "x_vars[pep_msh_pep_s]"
   ],
   [
    "C50",
    "x_vars[pepperoni_l]"
   ],
   [
    "C51",
    "x_vars[pepperoni_m]"
   ],
   [
    "C52",
    "x_vars[pepperoni_s]"
   ],
   [
    "C53",
    "x_vars[peppr_salami_l]"
   ],
   [
    "C54",
    "x_vars[peppr_salami_m]"
   ],
   [
    "C55",
    "x_vars[peppr_salami_s]"
   ],
   [
    "C56",
    "x_vars[prsc_argla_l]"
   ],
   [
    "C57",
    "x_vars[prsc_argla_m]"
   ],
   [
    "C58",
    "x_vars[prsc_argla_s]"
   ],
   [
    "C59",
    "x_vars[sicilian_l]"
   ],
   [
    "C60",
    "x_vars[sicilian_m]"
   ],
   [
    "C61",
    "x_vars[sicilian_s]"
   ],
   [
    "C62",
    "x_vars[soppressata_l]"
   ],
   [
    "C63",
    "x_vars[soppressata_m]"
   ],
   [
    "C64",
    "x_vars[soppressata_s]"
   ],
   [
    "C65",
    "x_vars[southw_ckn_l]"
   ],
   [
    "C66",
    "x_vars[southw_ckn_m]"
   ],
   [
    "C67",
    "x_vars[southw_ckn_s]"
   ],
   [
    "C68",
    "x_vars[spicy_ital_l]"
   ],
   [
    "C69",
    "x_vars[spicy_ital_m]"
   ],
   [
    "C70",
    "x_vars[spicy_ital_s]"
   ],
   [
    "C71",
    "x_vars[spin_pesto_l]"
   ],
   [
    "C72",
    "x_vars[spin_pesto_m]"
   ],
   [
    "C73",
    "x_vars[spin_pesto_s]"
   ],
   [
    "C74",
    "x_vars[spinach_fet_l]"
   ],
   [
    "C75",
    "x_vars[spinach_fet_m]"
   ],
   [
    "C76",
    "x_vars[spinach_fet_s]"
   ],
   [
    "C77",
    "x_vars[spinach_supr_l]"
   ],
   [
    "C78",
    "x_vars[spinach_supr_m]"
   ],
   [
    "C79",
    "x_vars[spinach_supr_s]"
   ],
   [
    "C80",
    "x_vars[thai_ckn_l]"
   ],
   [
    "C81",
    "x_vars[thai_ckn_m]"
   ],
   [
    "C82",
    "x_vars[thai_ckn_s]"
   ],
   [
    "C83",
    "x_vars[the_greek_l]"
   ],
   [
    "C84",
    "x_vars[the_greek_m]"
   ],
   [
    "C85",
    "x_vars[the_greek_s]"
   ],
   [
    "C86",
    "x_vars[the_greek_xl]"
   ],
   [
    "C87",
    "x_vars[the_greek_xxl]"
   ],
   [
    "C88",
    "x_vars[veggie_veg_l]"
   ],
   [
    "C89",
    "x_vars[veggie_veg_m]"
   ],
   [
    "C90",
    "x_vars[veggie_veg_s]"
   ],
   [
    "C91",
    "x_vars[fried_duck_l]"
   ],
   [
    "C92",
    "x_vars[fried_duck_m]"
   ],
   [
    "C93",
    "x_vars[fried_duck_s]"
   ],
   [
    "C94",
    "x_vars[mapo_Tofu_l]"
   ],
   [
    "C95",
    "x_vars[mapo_Tofu_m]"
   ],
   [
    "C96",
    "x_vars[mapo_Tofu_s]"
   ],
   [
    "C97",
    "x_vars[sea_food_l]"
   ],
   [
    "C98",
    "x_vars[sea_food_m]"
   ],
   [
    "C99",
    "x_vars[sea_food_s]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[bbq_ckn_l]",
   "x_2": "x_vars[bbq_ckn_m]",
   "x_3": "x_vars[bbq_ckn_s]",
   "x_4": "x_vars[big_meat_s]",
   "x_5": "x_vars[brie_carre_s]",
   "x_6": "x_vars[calabrese_l]",
   "x_7": "x_vars[calabrese_m]",
   "x_8": "x_vars[calabrese_s]",
   "x_9": "x_vars[cali_ckn_l]",
   "x_10": "x_vars[cali_ckn_m]",
   "x_11": "x_vars[cali_ckn_s]",
   "x_12": "x_vars[ckn_alfredo_l]",
   "x_13": "x_vars[ckn_alfredo_m]",
   "x_14": "x_vars[ckn_alfredo_s]",
   "x_15": "x_vars[ckn_pesto_l]",
   "x_16": "x_vars[ckn_pesto_m]",
   "x_17": "x_vars[ckn_pesto_s]",
   "x_18": "x_vars[classic_dlx_l]",
   "x_19": "x_vars[classic_dlx_m]",
   "x_20": "x_vars[classic_dlx_s]",
   "x_21": "x_vars[five_cheese_l]",
   "x_22": "x_vars[four_cheese_l]",
   "x_23": "x_vars[four_cheese_m]",
   "x_24": "x_vars[green_garden_l]",
   "x_25": "x_vars[green_garden_m]",
   "x_26": "x_vars[green_garden_s]",
   "x_27": "x_vars[hawaiian_l]",
   "x_28": "x_vars[hawaiian_m]",
   "x_29": "x_vars[hawaiian_s]",
   "x_30": "x_vars[ital_cpcllo_l]",
   "x_31": "x_vars[ital_cpcllo_m]",
   "x_32": "x_vars[ital_cpcllo_s]",
   "x_33": "x_vars[ital_supr_l]",
   "x_34": "x_vars[ital_supr_m]",
   "x_35": "x_vars[ital_supr_s]",
   "x_36": "x_vars[ital_veggie_l]",
   "x_37": "x_vars[ital_veggie_m]",
   "x_38": "x_vars[ital_veggie_s]",
   "x_39": "x_vars[mediterraneo_l]",
   "x_40": "x_vars[mediterraneo_m]",
   "x_41": "x_vars[mediterraneo_s]",
   "x_42": "x_vars[mexicana_l]",
   "x_43": "x_vars[mexicana_m]",
   "x_44": "x_vars[mexicana_s]",
   "x_45": "x_vars[napolitana_l]",
   "x_46": "x_vars[napolitana_m]",
   "x_47": "x_vars[napolitana_s]",
   "x_48": "x_vars[pep_msh_pep_l]",
   "x_49": "x_vars[pep_msh_pep_m]",
   "x_50": "x_vars[pep_msh_pep_s]",
   "x_51": "x_vars[pepperoni_l]",
   "x_52": "x_vars[pepperoni_m]",
   "x_53": "x_vars[pepperoni_s]",
   "x_54": "x_vars[peppr_salami_l]",
   "x_55": "x_vars[peppr_salami_m]",
   "x_56": "x_vars[peppr_salami_s]",
   "x_57": "x_vars[prsc_argla_l]",
   "x_58": "x_vars[prsc_argla_m]",
   "x_59": "x_vars[prsc_argla_s]",
   "x_60": "x_vars[sicilian_l]",
   "x_61": "x_vars[sicilian_m]",
   "x_62": "x_vars[sicilian_s]",
   "x_63": "x_vars[soppressata_l]",
   "x_64": "x_vars[soppressata_m]",
   "x_65": "x_vars[soppressata_s]",
   "x_66": "x_vars[southw_ckn_l]",
   "x_67": "x_vars[southw_ckn_m]",
   "x_68": "x_vars[southw_ckn_s]",
   "x_69": "x_vars[spicy_ital_l]",
   "x_70": "x_vars[spicy_ital_m]",
   "x_71": "x_vars[spicy_ital_s]",
   "x_72": "x_vars[spin_pesto_l]",
   "x_73": "x_vars[spin_pesto_m]",
   "x_74": "x_vars[spin_pesto_s]",
   "x_75": "x_vars[spinach_fet_l]",
   "x_76": "x_vars[spinach_fet_m]",
   "x_77": "x_vars[spinach_fet_s]",
   "x_78": "x_vars[spinach_supr_l]",
   "x_79": "x_vars[spinach_supr_m]",
   "x_80": "x_vars[spinach_supr_s]",
   "x_81": "x_vars[thai_ckn_l]",
   "x_82": "x_vars[thai_ckn_m]",
   "x_83": "x_vars[thai_ckn_s]",
   "x_84": "x_vars[the_greek_l]",
   "x_85": "x_vars[the_greek_m]",
   "x_86": "x_vars[the_greek_s]",
   "x_87": "x_vars[the_greek_xl]",
   "x_88": "x_vars[the_greek_xxl]",
   "x_89": "x_vars[veggie_veg_l]",
   "x_90": "x_vars[veggie_veg_m]",
   "x_91": "x_vars[veggie_veg_s]",
   "x_92": "x_vars[fried_duck_l]",
   "x_93": "x_vars[fried_duck_m]",
   "x_94": "x_vars[fried_duck_s]",
   "x_95": "x_vars[mapo_Tofu_l]",
   "x_96": "x_vars[mapo_Tofu_m]",
   "x_97": "x_vars[mapo_Tofu_s]",
   "x_98": "x_vars[sea_food_l]",
   "x_99": "x_vars[sea_food_m]",
   "x_100": "x_vars[sea_food_s]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-015": {
  "source_code_sha256": "bef1f6d5476d6160dc02a61f3d2bef3e7bffb63478c52685d12b5d0009254a76",
  "original_solution_sha256": "58ef4a44824dc79dc007715fabf283521d701f225a3ee6bda32c27164bea70dc",
  "bindings": [
   [
    "C0",
    "x_vars[Aalopuri]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Aalopuri]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-016": {
  "source_code_sha256": "e21f0a94c951fefd03e1793555491a021fa892faac63e9ccf450306eccf6a163",
  "original_solution_sha256": "a9f0bd8386118d0927b04658047118395831172b45be9d00d6b7fdff5dfb1b08",
  "bindings": [
   [
    "C0",
    "x_vars[Beauty - 25]"
   ],
   [
    "C1",
    "x_vars[Beauty - 30]"
   ],
   [
    "C2",
    "x_vars[Beauty - 300]"
   ],
   [
    "C3",
    "x_vars[Beauty - 50]"
   ],
   [
    "C4",
    "x_vars[Beauty - 500]"
   ],
   [
    "C5",
    "x_vars[Clothing - 25]"
   ],
   [
    "C6",
    "x_vars[Clothing - 30]"
   ],
   [
    "C7",
    "x_vars[Clothing - 300]"
   ],
   [
    "C8",
    "x_vars[Clothing - 50]"
   ],
   [
    "C9",
    "x_vars[Clothing - 500]"
   ],
   [
    "C10",
    "x_vars[Electronics - 25]"
   ],
   [
    "C11",
    "x_vars[Electronics - 30]"
   ],
   [
    "C12",
    "x_vars[Electronics - 300]"
   ],
   [
    "C13",
    "x_vars[Electronics - 50]"
   ],
   [
    "C14",
    "x_vars[Electronics - 500]"
   ],
   [
    "C15",
    "x_vars[Home Goods - 25]"
   ],
   [
    "C16",
    "x_vars[Home Goods - 30]"
   ],
   [
    "C17",
    "x_vars[Home Goods - 50]"
   ],
   [
    "C18",
    "x_vars[Home Goods - 300]"
   ],
   [
    "C19",
    "x_vars[Home Goods - 500]"
   ],
   [
    "C20",
    "x_vars[Sports - 25]"
   ],
   [
    "C21",
    "x_vars[Sports - 30]"
   ],
   [
    "C22",
    "x_vars[Sports - 50]"
   ],
   [
    "C23",
    "x_vars[Sports - 300]"
   ],
   [
    "C24",
    "x_vars[Sports - 500]"
   ],
   [
    "C25",
    "x_vars[Furniture - 25]"
   ],
   [
    "C26",
    "x_vars[Furniture - 30]"
   ],
   [
    "C27",
    "x_vars[Furniture - 50]"
   ],
   [
    "C28",
    "x_vars[Furniture - 300]"
   ],
   [
    "C29",
    "x_vars[Furniture - 500]"
   ],
   [
    "C30",
    "x_vars[Toys - 25]"
   ],
   [
    "C31",
    "x_vars[Toys - 30]"
   ],
   [
    "C32",
    "x_vars[Toys - 50]"
   ],
   [
    "C33",
    "x_vars[Toys - 300]"
   ],
   [
    "C34",
    "x_vars[Toys - 500]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Beauty - 25]",
   "x_2": "x_vars[Beauty - 30]",
   "x_3": "x_vars[Beauty - 300]",
   "x_4": "x_vars[Beauty - 50]",
   "x_5": "x_vars[Beauty - 500]",
   "x_6": "x_vars[Clothing - 25]",
   "x_7": "x_vars[Clothing - 30]",
   "x_8": "x_vars[Clothing - 300]",
   "x_9": "x_vars[Clothing - 50]",
   "x_10": "x_vars[Clothing - 500]",
   "x_11": "x_vars[Electronics - 25]",
   "x_12": "x_vars[Electronics - 30]",
   "x_13": "x_vars[Electronics - 300]",
   "x_14": "x_vars[Electronics - 50]",
   "x_15": "x_vars[Electronics - 500]",
   "x_16": "x_vars[Home Goods - 25]",
   "x_17": "x_vars[Home Goods - 30]",
   "x_18": "x_vars[Home Goods - 50]",
   "x_19": "x_vars[Home Goods - 300]",
   "x_20": "x_vars[Home Goods - 500]",
   "x_21": "x_vars[Sports - 25]",
   "x_22": "x_vars[Sports - 30]",
   "x_23": "x_vars[Sports - 50]",
   "x_24": "x_vars[Sports - 300]",
   "x_25": "x_vars[Sports - 500]",
   "x_26": "x_vars[Furniture - 25]",
   "x_27": "x_vars[Furniture - 30]",
   "x_28": "x_vars[Furniture - 50]",
   "x_29": "x_vars[Furniture - 300]",
   "x_30": "x_vars[Furniture - 500]",
   "x_31": "x_vars[Toys - 25]",
   "x_32": "x_vars[Toys - 30]",
   "x_33": "x_vars[Toys - 50]",
   "x_34": "x_vars[Toys - 300]",
   "x_35": "x_vars[Toys - 500]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-017": {
  "source_code_sha256": "74b6557cfcd3be2518bf1e32ddac43e4884f2d3c6cd3a99aaa7cb699c9db3377",
  "original_solution_sha256": "1adb3ecb4c1e88130f315d3cea4a93a6a9c6e0aed6138a7e474088e6485d8aaa",
  "bindings": [
   [
    "C0",
    "x_vars[ZZ2AO]"
   ],
   [
    "C1",
    "x_vars[ZZDW7]"
   ],
   [
    "C2",
    "x_vars[ZZM1A]"
   ],
   [
    "C3",
    "x_vars[ZZNC5]"
   ],
   [
    "C4",
    "x_vars[ZZX6K]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[ZZ2AO]",
   "x_2": "x_vars[ZZDW7]",
   "x_3": "x_vars[ZZM1A]",
   "x_4": "x_vars[ZZNC5]",
   "x_5": "x_vars[ZZX6K]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-018": {
  "source_code_sha256": "748ca5f30fc14faeae6dad91d921a2a17b34fdf6198902d883880e8e055b912e",
  "original_solution_sha256": "055485cf17ec2c7dee00c799a4f10e17a7660386f1dec9c5add0a9ca664b7cbe",
  "bindings": [
   [
    "C0",
    "x_vars[Baby Food_255.28]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Baby Food_255.28]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-019": {
  "source_code_sha256": "3618fdd808910f25c0cc658b23fe82801661d8ef226af596b328663b9ec0f230",
  "original_solution_sha256": "071705e568cc90567aae65a7638d2c73c37f55cdc04aa3a33d254db5d4d1a1d8",
  "bindings": [
   [
    "C0",
    "x_vars[27in 4K Gaming Monitor]"
   ],
   [
    "C1",
    "x_vars[27in FHD Monitor]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[27in 4K Gaming Monitor]",
   "x_2": "x_vars[27in FHD Monitor]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-020": {
  "source_code_sha256": "151703e8aa42b078fdb1a7558da9483ef956b7b0168586c3b6530bae06916c71",
  "original_solution_sha256": "ec2a59bed57dae6bf5b79cfd83373bc59780941287823cb98bd8179ae4bab9df",
  "bindings": [
   [
    "C0",
    "x_vars[S10_1678]"
   ],
   [
    "C1",
    "x_vars[S10_1949]"
   ],
   [
    "C2",
    "x_vars[S10_2016]"
   ],
   [
    "C3",
    "x_vars[S10_4698]"
   ],
   [
    "C4",
    "x_vars[S10_4757]"
   ],
   [
    "C5",
    "x_vars[S10_4962]"
   ],
   [
    "C6",
    "x_vars[S12_1099]"
   ],
   [
    "C7",
    "x_vars[S12_1108]"
   ],
   [
    "C8",
    "x_vars[S12_1666]"
   ],
   [
    "C9",
    "x_vars[S12_2823]"
   ],
   [
    "C10",
    "x_vars[S12_3148]"
   ],
   [
    "C11",
    "x_vars[S12_3380]"
   ],
   [
    "C12",
    "x_vars[S12_3891]"
   ],
   [
    "C13",
    "x_vars[S12_3990]"
   ],
   [
    "C14",
    "x_vars[S12_4473]"
   ],
   [
    "C15",
    "x_vars[S12_4675]"
   ],
   [
    "C16",
    "x_vars[S18_1097]"
   ],
   [
    "C17",
    "x_vars[S18_1129]"
   ],
   [
    "C18",
    "x_vars[S18_1342]"
   ],
   [
    "C19",
    "x_vars[S18_1367]"
   ],
   [
    "C20",
    "x_vars[S18_1589]"
   ],
   [
    "C21",
    "x_vars[S18_1662]"
   ],
   [
    "C22",
    "x_vars[S18_1749]"
   ],
   [
    "C23",
    "x_vars[S18_1889]"
   ],
   [
    "C24",
    "x_vars[S18_1984]"
   ],
   [
    "C25",
    "x_vars[S18_2238]"
   ],
   [
    "C26",
    "x_vars[S18_2248]"
   ],
   [
    "C27",
    "x_vars[S18_2319]"
   ],
   [
    "C28",
    "x_vars[S18_2325]"
   ],
   [
    "C29",
    "x_vars[S18_2432]"
   ],
   [
    "C30",
    "x_vars[S18_2581]"
   ],
   [
    "C31",
    "x_vars[S18_2625]"
   ],
   [
    "C32",
    "x_vars[S18_2795]"
   ],
   [
    "C33",
    "x_vars[S18_2870]"
   ],
   [
    "C34",
    "x_vars[S18_2949]"
   ],
   [
    "C35",
    "x_vars[S18_2957]"
   ],
   [
    "C36",
    "x_vars[S18_3029]"
   ],
   [
    "C37",
    "x_vars[S18_3136]"
   ],
   [
    "C38",
    "x_vars[S18_3140]"
   ],
   [
    "C39",
    "x_vars[S18_3232]"
   ],
   [
    "C40",
    "x_vars[S18_3259]"
   ],
   [
    "C41",
    "x_vars[S18_3278]"
   ],
   [
    "C42",
    "x_vars[S18_3320]"
   ],
   [
    "C43",
    "x_vars[S18_3482]"
   ],
   [
    "C44",
    "x_vars[S18_3685]"
   ],
   [
    "C45",
    "x_vars[S18_3782]"
   ],
   [
    "C46",
    "x_vars[S18_3856]"
   ],
   [
    "C47",
    "x_vars[S18_4027]"
   ],
   [
    "C48",
    "x_vars[S18_4409]"
   ],
   [
    "C49",
    "x_vars[S18_4522]"
   ],
   [
    "C50",
    "x_vars[S18_4600]"
   ],
   [
    "C51",
    "x_vars[S18_4668]"
   ],
   [
    "C52",
    "x_vars[S18_4721]"
   ],
   [
    "C53",
    "x_vars[S18_4933]"
   ],
   [
    "C54",
    "x_vars[S24_1046]"
   ],
   [
    "C55",
    "x_vars[S24_1444]"
   ],
   [
    "C56",
    "x_vars[S24_1578]"
   ],
   [
    "C57",
    "x_vars[S24_1628]"
   ],
   [
    "C58",
    "x_vars[S24_1785]"
   ],
   [
    "C59",
    "x_vars[S24_1937]"
   ],
   [
    "C60",
    "x_vars[S24_2000]"
   ],
   [
    "C61",
    "x_vars[S24_2011]"
   ],
   [
    "C62",
    "x_vars[S24_2022]"
   ],
   [
    "C63",
    "x_vars[S24_2300]"
   ],
   [
    "C64",
    "x_vars[S24_2360]"
   ],
   [
    "C65",
    "x_vars[S24_2766]"
   ],
   [
    "C66",
    "x_vars[S24_2840]"
   ],
   [
    "C67",
    "x_vars[S24_2841]"
   ],
   [
    "C68",
    "x_vars[S24_2887]"
   ],
   [
    "C69",
    "x_vars[S24_2972]"
   ],
   [
    "C70",
    "x_vars[S24_3151]"
   ],
   [
    "C71",
    "x_vars[S24_3191]"
   ],
   [
    "C72",
    "x_vars[S24_3371]"
   ],
   [
    "C73",
    "x_vars[S24_3420]"
   ],
   [
    "C74",
    "x_vars[S24_3432]"
   ],
   [
    "C75",
    "x_vars[S24_3816]"
   ],
   [
    "C76",
    "x_vars[S24_3856]"
   ],
   [
    "C77",
    "x_vars[S24_3949]"
   ],
   [
    "C78",
    "x_vars[S24_3969]"
   ],
   [
    "C79",
    "x_vars[S24_4048]"
   ],
   [
    "C80",
    "x_vars[S24_4258]"
   ],
   [
    "C81",
    "x_vars[S24_4278]"
   ],
   [
    "C82",
    "x_vars[S24_4620]"
   ],
   [
    "C83",
    "x_vars[S32_1268]"
   ],
   [
    "C84",
    "x_vars[S32_1374]"
   ],
   [
    "C85",
    "x_vars[S32_2206]"
   ],
   [
    "C86",
    "x_vars[S32_2509]"
   ],
   [
    "C87",
    "x_vars[S32_3207]"
   ],
   [
    "C88",
    "x_vars[S32_3522]"
   ],
   [
    "C89",
    "x_vars[S32_4289]"
   ],
   [
    "C90",
    "x_vars[S32_4485]"
   ],
   [
    "C91",
    "x_vars[S50_1341]"
   ],
   [
    "C92",
    "x_vars[S50_1392]"
   ],
   [
    "C93",
    "x_vars[S50_1514]"
   ],
   [
    "C94",
    "x_vars[S50_4713]"
   ],
   [
    "C95",
    "x_vars[S700_1138]"
   ],
   [
    "C96",
    "x_vars[S700_1691]"
   ],
   [
    "C97",
    "x_vars[S700_1938]"
   ],
   [
    "C98",
    "x_vars[S700_2047]"
   ],
   [
    "C99",
    "x_vars[S700_2466]"
   ],
   [
    "C100",
    "x_vars[S700_2610]"
   ],
   [
    "C101",
    "x_vars[S700_2824]"
   ],
   [
    "C102",
    "x_vars[S700_2834]"
   ],
   [
    "C103",
    "x_vars[S700_3167]"
   ],
   [
    "C104",
    "x_vars[S700_3505]"
   ],
   [
    "C105",
    "x_vars[S700_3962]"
   ],
   [
    "C106",
    "x_vars[S700_4002]"
   ],
   [
    "C107",
    "x_vars[S72_1253]"
   ],
   [
    "C108",
    "x_vars[S72_3212]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[S10_1678]",
   "x_2": "x_vars[S10_1949]",
   "x_3": "x_vars[S10_2016]",
   "x_4": "x_vars[S10_4698]",
   "x_5": "x_vars[S10_4757]",
   "x_6": "x_vars[S10_4962]",
   "x_7": "x_vars[S12_1099]",
   "x_8": "x_vars[S12_1108]",
   "x_9": "x_vars[S12_1666]",
   "x_10": "x_vars[S12_2823]",
   "x_11": "x_vars[S12_3148]",
   "x_12": "x_vars[S12_3380]",
   "x_13": "x_vars[S12_3891]",
   "x_14": "x_vars[S12_3990]",
   "x_15": "x_vars[S12_4473]",
   "x_16": "x_vars[S12_4675]",
   "x_17": "x_vars[S18_1097]",
   "x_18": "x_vars[S18_1129]",
   "x_19": "x_vars[S18_1342]",
   "x_20": "x_vars[S18_1367]",
   "x_21": "x_vars[S18_1589]",
   "x_22": "x_vars[S18_1662]",
   "x_23": "x_vars[S18_1749]",
   "x_24": "x_vars[S18_1889]",
   "x_25": "x_vars[S18_1984]",
   "x_26": "x_vars[S18_2238]",
   "x_27": "x_vars[S18_2248]",
   "x_28": "x_vars[S18_2319]",
   "x_29": "x_vars[S18_2325]",
   "x_30": "x_vars[S18_2432]",
   "x_31": "x_vars[S18_2581]",
   "x_32": "x_vars[S18_2625]",
   "x_33": "x_vars[S18_2795]",
   "x_34": "x_vars[S18_2870]",
   "x_35": "x_vars[S18_2949]",
   "x_36": "x_vars[S18_2957]",
   "x_37": "x_vars[S18_3029]",
   "x_38": "x_vars[S18_3136]",
   "x_39": "x_vars[S18_3140]",
   "x_40": "x_vars[S18_3232]",
   "x_41": "x_vars[S18_3259]",
   "x_42": "x_vars[S18_3278]",
   "x_43": "x_vars[S18_3320]",
   "x_44": "x_vars[S18_3482]",
   "x_45": "x_vars[S18_3685]",
   "x_46": "x_vars[S18_3782]",
   "x_47": "x_vars[S18_3856]",
   "x_48": "x_vars[S18_4027]",
   "x_49": "x_vars[S18_4409]",
   "x_50": "x_vars[S18_4522]",
   "x_51": "x_vars[S18_4600]",
   "x_52": "x_vars[S18_4668]",
   "x_53": "x_vars[S18_4721]",
   "x_54": "x_vars[S18_4933]",
   "x_55": "x_vars[S24_1046]",
   "x_56": "x_vars[S24_1444]",
   "x_57": "x_vars[S24_1578]",
   "x_58": "x_vars[S24_1628]",
   "x_59": "x_vars[S24_1785]",
   "x_60": "x_vars[S24_1937]",
   "x_61": "x_vars[S24_2000]",
   "x_62": "x_vars[S24_2011]",
   "x_63": "x_vars[S24_2022]",
   "x_64": "x_vars[S24_2300]",
   "x_65": "x_vars[S24_2360]",
   "x_66": "x_vars[S24_2766]",
   "x_67": "x_vars[S24_2840]",
   "x_68": "x_vars[S24_2841]",
   "x_69": "x_vars[S24_2887]",
   "x_70": "x_vars[S24_2972]",
   "x_71": "x_vars[S24_3151]",
   "x_72": "x_vars[S24_3191]",
   "x_73": "x_vars[S24_3371]",
   "x_74": "x_vars[S24_3420]",
   "x_75": "x_vars[S24_3432]",
   "x_76": "x_vars[S24_3816]",
   "x_77": "x_vars[S24_3856]",
   "x_78": "x_vars[S24_3949]",
   "x_79": "x_vars[S24_3969]",
   "x_80": "x_vars[S24_4048]",
   "x_81": "x_vars[S24_4258]",
   "x_82": "x_vars[S24_4278]",
   "x_83": "x_vars[S24_4620]",
   "x_84": "x_vars[S32_1268]",
   "x_85": "x_vars[S32_1374]",
   "x_86": "x_vars[S32_2206]",
   "x_87": "x_vars[S32_2509]",
   "x_88": "x_vars[S32_3207]",
   "x_89": "x_vars[S32_3522]",
   "x_90": "x_vars[S32_4289]",
   "x_91": "x_vars[S32_4485]",
   "x_92": "x_vars[S50_1341]",
   "x_93": "x_vars[S50_1392]",
   "x_94": "x_vars[S50_1514]",
   "x_95": "x_vars[S50_4713]",
   "x_96": "x_vars[S700_1138]",
   "x_97": "x_vars[S700_1691]",
   "x_98": "x_vars[S700_1938]",
   "x_99": "x_vars[S700_2047]",
   "x_100": "x_vars[S700_2466]",
   "x_101": "x_vars[S700_2610]",
   "x_102": "x_vars[S700_2824]",
   "x_103": "x_vars[S700_2834]",
   "x_104": "x_vars[S700_3167]",
   "x_105": "x_vars[S700_3505]",
   "x_106": "x_vars[S700_3962]",
   "x_107": "x_vars[S700_4002]",
   "x_108": "x_vars[S72_1253]",
   "x_109": "x_vars[S72_3212]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-021": {
  "source_code_sha256": "7fadb7a67b19a65b959e751203e1149b9b7f6f6f33d450e4282df71c96abf926",
  "original_solution_sha256": "adbf0a6c3c119ba3f6846ed025969f53556bb0da6ebc832920cabda62c22537e",
  "bindings": [
   [
    "C0",
    "x_vars[ 3 Colors New Fashion Summer Ladies Casual Jumpsuit Long Suspender Overalls Bib Pants]"
   ],
   [
    "C1",
    "x_vars[ Men's Casual Workout Athletic Gym Jersey Shorts Elastic Waist Drawstring Summer Training Running Knee Length Shorts with Zipper Pocket]"
   ],
   [
    "C2",
    "x_vars[ New Fashion Autumn Summer Women's Long Sleeve V Neck Long Dress Floral Print Split Maxi Dress Holiday Party Beach Sundress Evening Dresses]"
   ],
   [
    "C3",
    "x_vars[ New The New Men's Stitching Design Jogging Sports Cropped Trousers]"
   ],
   [
    "C4",
    "x_vars[ Plus Size S-5XL Women Summer Tops Casual V-Neck Short Sleeve Shirts Ladies Cotton Loose T Shirt Candy Color Lady Pullovers Blouse]"
   ],
   [
    "C5",
    "x_vars[ Plus Size Women Halter Striped Wide Leg Pants Casual Jumpsuit Rompers Fashion Shorts ]"
   ],
   [
    "C6",
    "x_vars[ Spring/Summer Fashion Women \"honey\"Letter Print Sleeveless Shirt Sexy Embroidered Bodycon Vest Knitted Cotton Vest Casual Tank Top ]"
   ],
   [
    "C7",
    "x_vars[ Summer Students Style Bodycon Rompers Womens Slim Fit Jumpsuit Chest Zipper Contrast Color Short Sleeve V-neck Shorts Bodysuit]"
   ],
   [
    "C8",
    "x_vars[ Summer Women s Fashion Lace Up Tie Pants Plus Size Casual High Waist Short Pants(S-5XL)]"
   ],
   [
    "C9",
    "x_vars[ Women's Fashion Graphic \\Don't Flatter Yourself....\\Tee for Women Summer Casual Tee T Shirts for Girls]"
   ],
   [
    "C10",
    "x_vars[ XS - XXL Fashion Women's Summer Casual T - Shirt Love Gesture Printed Short Sleeve Blouse XS - XXL]"
   ],
   [
    "C11",
    "x_vars['Let That Shit Go' Women Graphic Tee Casual Cotton Short Sleeve T-shirts Bohemia Style Mandala Namaste Printed Top Blouse]"
   ],
   [
    "C12",
    "x_vars[(S-5XL) Women Fashion Summer Double-Layer Sports Shorts Quick-Drying Yoga Sports Leggings Fitness Shorts Plus Size]"
   ],
   [
    "C13",
    "x_vars[(S-7XL) Women Camisole Summer Printed Vest Floral Sleeveless Vest V-neck Shirt Plus Size]"
   ],
   [
    "C14",
    "x_vars[(US Size) Cotton Graphic Tees for Women / Girls: 4 Colors, Spring Summer Tee, Cute Funny Short Sleeve T-Shirt, Loose Blouse Tops]"
   ],
   [
    "C15",
    "x_vars[1 Pcs Swimming Chair/Bed Swimming Pool Seat Inflatable Lazy Bed Lounge Chair Air Mattress Floating Bed/Chair with Net Foldable for Swimming Pool/Beach Water Relaxation]"
   ],
   [
    "C16",
    "x_vars[10 Color Women Summer Shorts Lace Up Elastic Waistband Loose Panties Plus Size S-6XL]"
   ],
   [
    "C17",
    "x_vars[10/20/30ml Slimming Firming Anti-Cellulite Massage Cream Fat Burning Weight Losing Moisturizer Gel for Shaping Waist]"
   ],
   [
    "C18",
    "x_vars[10/30/50ml Hair Removal Spray Super Natural Painless Permanent Depilatory Cream]"
   ],
   [
    "C19",
    "x_vars[100%Cotton ZANZEA S-5XL NEW Vintage Women Strap Dungaree Jumpsuit Casual Loose Trousers Overalls Rompers Jumpsuit]"
   ],
   [
    "C20",
    "x_vars[100pcs Summer Disposable Sweat Pad Perspiration Absorbing Guard Underarm Armpit Sweat Pad Pure Antiperspirant Adhesive Underarm Pads]"
   ],
   [
    "C21",
    "x_vars[11 Colors V Neck Women Sleeveless Tops Halter Neck Plus Size Chiffon CropTops(S~5XL)]"
   ],
   [
    "C22",
    "x_vars[16 Color Fashion Women Casual Sleeveless Camisoles Loose T-shirts with Ziper Deep V-neck Blouses Ladies Chiffon Shirts Tank Tops ]"
   ],
   [
    "C23",
    "x_vars[170cm Inflatable Spray Water Cushion Summer Kids Play Water Mat Lawn Games Pad Sprinkler Play Toys Outdoor Tub Swiming Pool ]"
   ],
   [
    "C24",
    "x_vars[18 Styles Women Summer Sexy Printing Buttons Off Shoulder Sleeveless Dress Princess Dress Spaghetti Straps Dresses]"
   ],
   [
    "C25",
    "x_vars[1PCS Swimwear Monokini Swimsuit Backless Bodysuit Women Swimsuit]"
   ],
   [
    "C26",
    "x_vars[1Set Lace Bikini Diamond Swimsuit Crystal Women Swimwear Nude Bikinis Brazilian Rhinestone Beachwear Push Up Bikini]"
   ],
   [
    "C27",
    "x_vars[1pcs Men's Running Shorts 2 in 1 Sports Jogging Fitness Shorts Summer Training Quick Dry Short Gym Shorts Sport Pants]"
   ],
   [
    "C28",
    "x_vars[20 Pcs Fashion Comfortable Short Socks Candy Color Invisible Silicone Non-slip Socks Slippers Socks]"
   ],
   [
    "C29",
    "x_vars[20/10pcs Women Ankle Invisible No Show Nonslip Loafer Boat Liner Cotton Socks Comfortable Socks Women Shoes Accessories  (Application Size: 33-43)]"
   ],
   [
    "C30",
    "x_vars[2017 Women Ladies Summer Dress Sleeveless Casual Sexy Floral Print Beach Dress Fashion Spaghetti Strap Mini Short Dress]"
   ],
   [
    "C31",
    "x_vars[2018 6 Color Summer Womenâ__s New Fashion Sexy Cute Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Slim Bodycon Off The Shoulder Short Sleeve Blouse Casual Cotton Outdoor Tops Plus Size S-XXXXL]"
   ],
   [
    "C32",
    "x_vars[2018 Black Flora Flower Printed Short Dress Women]"
   ],
   [
    "C33",
    "x_vars[2018 Fashion Summer Dress Women Sexy Dresses V Neck Backless Lace Stitching Dress Beach Dresses White Dress]"
   ],
   [
    "C34",
    "x_vars[2018 Hot Selling Spring Summer Women Flared Causal Trousers Loose Pants Drawstring Elastic Waist Middle Waisted  Wide Leg Pants]"
   ],
   [
    "C35",
    "x_vars[2018 New Fashion Women Bikini Set Push-up Padded Bra Swimsuit -Swimwear]"
   ],
   [
    "C36",
    "x_vars[2018 New Fashion Women Casual Playsuit Ladies Jumpsuit Romper Summer Floral Playsuit Brand New (3 Colors)]"
   ],
   [
    "C37",
    "x_vars[2018 New Fashion Women's Tops Sexy Strappy Sleeveless Lace Crop Tops]"
   ],
   [
    "C38",
    "x_vars[2018 New Summer Women Sexy V-neck Bandage T-shirts Camouflage Printed Short Sleeve Plus size Tops Tee(S-4XL)]"
   ],
   [
    "C39",
    "x_vars[2018 New The new cross-strait national wind body bikini Swimsuit  Swimwear LBT]"
   ],
   [
    "C40",
    "x_vars[2018 New ZANZEA Beach Romper Women Jumpsuits Front Zipper Sleeveless Sexy Playsuits]"
   ],
   [
    "C41",
    "x_vars[2018 Plus Size Summer Women Fashion Sexy V-neck Lace Up Plaid Blouse Tops Irregular Short Sleeve Shirtï¼_S-5XLï¼_]"
   ],
   [
    "C42",
    "x_vars[2018 Summer Fashion Women Casual Camouflage Tank Top Sleeveless O-neck Vest]"
   ],
   [
    "C43",
    "x_vars[2018 Summer Fashion Women Tank Tops Sexy Women Sleeveless Crop Tops Casual Style Women Cotton Print Lace Stitching Irregular Blouse Topsï¼_S-5XL)]"
   ],
   [
    "C44",
    "x_vars[2018 Summer New Women Fashion Sexy Sleeveless Tank Top Floral Print Casual Women Cotton T-shirts Plus Size Women Vestï¼_S-5XLï¼_]"
   ],
   [
    "C45",
    "x_vars[2018 Summer New Women Long Skirts Solid Sexy Split Pencil Skirts]"
   ],
   [
    "C46",
    "x_vars[2018 Summer Women Print Top Fashion Women Casual Army Camo Camouflage Tank Top Sleeveless O-neck Slim T-Shirts Plus Size S-XXXXXL]"
   ],
   [
    "C47",
    "x_vars[2018 Summer Women Sexy Off Shoulder Shirt Casual Half Sleeve Top Ladies Fashion Printing Loose T-shirts Plus Size Cotton Blouse XS-5XL]"
   ],
   [
    "C48",
    "x_vars[2018 Summer Womenâ__s New Fashion Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Casual Sexy Cotton Tops]"
   ],
   [
    "C49",
    "x_vars[2018 T Shirts + Shorts Brand Clothing Tshirt Men Homme Letter Printed Basketball Running Sports Set T-shirt Suit Male]"
   ],
   [
    "C50",
    "x_vars[2018 Women Cool T-shirt Funny 3d Tshirt Print Two Cat Short Sleeve Summer Tops Tees Teen Graphic Tee Cute Shirt Funny Gifts]"
   ],
   [
    "C51",
    "x_vars[2018 Women Fashion Stretchy Camisole  Spaghetti Strap Long Tank Top Slip Summer Fashion Floral Mini Dress 7 Color SIZE XXL]"
   ],
   [
    "C52",
    "x_vars[2018 Women Fashion Summer Sleeveless Maxi Halter Dresses Solid Color Halter Maxi Dress with Halter Tie and Pockets]"
   ],
   [
    "C53",
    "x_vars[2018 Women Ladies Fashion Crocheted Lace Summer Dress Maxi Dress Long Dress]"
   ],
   [
    "C54",
    "x_vars[2018 Women Ladies Fashion Musical Note Asymmetric Tank Top]"
   ],
   [
    "C55",
    "x_vars[2018 Women Summer Casual Solid Color Loose Sleeveless Beach Tank Top A-line Pocket Dress Sexy Deep V-neck Short Club Party Mini Halter Midi Dresses Robe Ete Femme Knee Length Pleated Skirts Ladies Fashion Swing Cotton T-Shirt Dress Plus Size S-6XL]"
   ],
   [
    "C56",
    "x_vars[2018 Women fashion Summer Lace Patchwork Tank Tops Casual Sleeveless Tops Vest Blouse (S-5XL) Plus Size]"
   ],
   [
    "C57",
    "x_vars[2019 Fashion Family matching swimwear beachwear mommy and me swimsuit mother daughter father son clothes dresses high waist bikini look mum]"
   ],
   [
    "C58",
    "x_vars[2019 Fashion Women Summer V-Neck Sleeveless Collect Waist Boho Print Maxi Long Dress S-XXXXXL]"
   ],
   [
    "C59",
    "x_vars[2019 Men's Summer Cool Jogging Shorts]"
   ],
   [
    "C60",
    "x_vars[2019 NEW Men's Short Sleeve T-Shirt Fitness Round Neck Casual Men's Zippers Summer T-Shirt]"
   ],
   [
    "C61",
    "x_vars[2019 New Fashion Spring Summer Women's Long Pants Short Sleeves Off Shoulder Jumpsuit]"
   ],
   [
    "C62",
    "x_vars[2019 New Fashion Summer Women Casual Dress Round Neck Loose Big Swing Skirt Sleeveless Soild Color Beach dress]"
   ],
   [
    "C63",
    "x_vars[2019 New Fashion Women Blouse Tops Sleeveless Spaghetti Strap Criss Cross V-neck T Shirt Tops Summer Casual Blouse Shirts]"
   ],
   [
    "C64",
    "x_vars[2019 New Fashion Women Casual Shorts Suit Summer Tie-Dye Print Halter Bandage Sleeveless Backless Crop Top And Elastic Waist Shorts Pants Two Piece Set]"
   ],
   [
    "C65",
    "x_vars[2019 New Fashion Women Loose Round Neck Dress Short Sleeve Solid Color Casual Dress]"
   ],
   [
    "C66",
    "x_vars[2019 New Fashion Women Summer Funny Penguin print T-Shirts Casual Short Sleeve O-Neck Tops Loose Cotton Cute Cool Tee Plus Size S-5XL 4 Colors]"
   ],
   [
    "C67",
    "x_vars[2019 New Fashion Women's Slim Plus Size Maxi Dress Ink Printing Spaghetti Strap Dress]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[ 3 Colors New Fashion Summer Ladies Casual Jumpsuit Long Suspender Overalls Bib Pants]",
   "x_2": "x_vars[ Men's Casual Workout Athletic Gym Jersey Shorts Elastic Waist Drawstring Summer Training Running Knee Length Shorts with Zipper Pocket]",
   "x_3": "x_vars[ New Fashion Autumn Summer Women's Long Sleeve V Neck Long Dress Floral Print Split Maxi Dress Holiday Party Beach Sundress Evening Dresses]",
   "x_4": "x_vars[ New The New Men's Stitching Design Jogging Sports Cropped Trousers]",
   "x_5": "x_vars[ Plus Size S-5XL Women Summer Tops Casual V-Neck Short Sleeve Shirts Ladies Cotton Loose T Shirt Candy Color Lady Pullovers Blouse]",
   "x_6": "x_vars[ Plus Size Women Halter Striped Wide Leg Pants Casual Jumpsuit Rompers Fashion Shorts ]",
   "x_7": "x_vars[ Spring/Summer Fashion Women \"honey\"Letter Print Sleeveless Shirt Sexy Embroidered Bodycon Vest Knitted Cotton Vest Casual Tank Top ]",
   "x_8": "x_vars[ Summer Students Style Bodycon Rompers Womens Slim Fit Jumpsuit Chest Zipper Contrast Color Short Sleeve V-neck Shorts Bodysuit]",
   "x_9": "x_vars[ Summer Women s Fashion Lace Up Tie Pants Plus Size Casual High Waist Short Pants(S-5XL)]",
   "x_10": "x_vars[ Women's Fashion Graphic \\Don't Flatter Yourself....\\Tee for Women Summer Casual Tee T Shirts for Girls]",
   "x_11": "x_vars[ XS - XXL Fashion Women's Summer Casual T - Shirt Love Gesture Printed Short Sleeve Blouse XS - XXL]",
   "x_12": "x_vars['Let That Shit Go' Women Graphic Tee Casual Cotton Short Sleeve T-shirts Bohemia Style Mandala Namaste Printed Top Blouse]",
   "x_13": "x_vars[(S-5XL) Women Fashion Summer Double-Layer Sports Shorts Quick-Drying Yoga Sports Leggings Fitness Shorts Plus Size]",
   "x_14": "x_vars[(S-7XL) Women Camisole Summer Printed Vest Floral Sleeveless Vest V-neck Shirt Plus Size]",
   "x_15": "x_vars[(US Size) Cotton Graphic Tees for Women / Girls: 4 Colors, Spring Summer Tee, Cute Funny Short Sleeve T-Shirt, Loose Blouse Tops]",
   "x_16": "x_vars[1 Pcs Swimming Chair/Bed Swimming Pool Seat Inflatable Lazy Bed Lounge Chair Air Mattress Floating Bed/Chair with Net Foldable for Swimming Pool/Beach Water Relaxation]",
   "x_17": "x_vars[10 Color Women Summer Shorts Lace Up Elastic Waistband Loose Panties Plus Size S-6XL]",
   "x_18": "x_vars[10/20/30ml Slimming Firming Anti-Cellulite Massage Cream Fat Burning Weight Losing Moisturizer Gel for Shaping Waist]",
   "x_19": "x_vars[10/30/50ml Hair Removal Spray Super Natural Painless Permanent Depilatory Cream]",
   "x_20": "x_vars[100%Cotton ZANZEA S-5XL NEW Vintage Women Strap Dungaree Jumpsuit Casual Loose Trousers Overalls Rompers Jumpsuit]",
   "x_21": "x_vars[100pcs Summer Disposable Sweat Pad Perspiration Absorbing Guard Underarm Armpit Sweat Pad Pure Antiperspirant Adhesive Underarm Pads]",
   "x_22": "x_vars[11 Colors V Neck Women Sleeveless Tops Halter Neck Plus Size Chiffon CropTops(S~5XL)]",
   "x_23": "x_vars[16 Color Fashion Women Casual Sleeveless Camisoles Loose T-shirts with Ziper Deep V-neck Blouses Ladies Chiffon Shirts Tank Tops ]",
   "x_24": "x_vars[170cm Inflatable Spray Water Cushion Summer Kids Play Water Mat Lawn Games Pad Sprinkler Play Toys Outdoor Tub Swiming Pool ]",
   "x_25": "x_vars[18 Styles Women Summer Sexy Printing Buttons Off Shoulder Sleeveless Dress Princess Dress Spaghetti Straps Dresses]",
   "x_26": "x_vars[1PCS Swimwear Monokini Swimsuit Backless Bodysuit Women Swimsuit]",
   "x_27": "x_vars[1Set Lace Bikini Diamond Swimsuit Crystal Women Swimwear Nude Bikinis Brazilian Rhinestone Beachwear Push Up Bikini]",
   "x_28": "x_vars[1pcs Men's Running Shorts 2 in 1 Sports Jogging Fitness Shorts Summer Training Quick Dry Short Gym Shorts Sport Pants]",
   "x_29": "x_vars[20 Pcs Fashion Comfortable Short Socks Candy Color Invisible Silicone Non-slip Socks Slippers Socks]",
   "x_30": "x_vars[20/10pcs Women Ankle Invisible No Show Nonslip Loafer Boat Liner Cotton Socks Comfortable Socks Women Shoes Accessories  (Application Size: 33-43)]",
   "x_31": "x_vars[2017 Women Ladies Summer Dress Sleeveless Casual Sexy Floral Print Beach Dress Fashion Spaghetti Strap Mini Short Dress]",
   "x_32": "x_vars[2018 6 Color Summer Womenâ__s New Fashion Sexy Cute Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Slim Bodycon Off The Shoulder Short Sleeve Blouse Casual Cotton Outdoor Tops Plus Size S-XXXXL]",
   "x_33": "x_vars[2018 Black Flora Flower Printed Short Dress Women]",
   "x_34": "x_vars[2018 Fashion Summer Dress Women Sexy Dresses V Neck Backless Lace Stitching Dress Beach Dresses White Dress]",
   "x_35": "x_vars[2018 Hot Selling Spring Summer Women Flared Causal Trousers Loose Pants Drawstring Elastic Waist Middle Waisted  Wide Leg Pants]",
   "x_36": "x_vars[2018 New Fashion Women Bikini Set Push-up Padded Bra Swimsuit -Swimwear]",
   "x_37": "x_vars[2018 New Fashion Women Casual Playsuit Ladies Jumpsuit Romper Summer Floral Playsuit Brand New (3 Colors)]",
   "x_38": "x_vars[2018 New Fashion Women's Tops Sexy Strappy Sleeveless Lace Crop Tops]",
   "x_39": "x_vars[2018 New Summer Women Sexy V-neck Bandage T-shirts Camouflage Printed Short Sleeve Plus size Tops Tee(S-4XL)]",
   "x_40": "x_vars[2018 New The new cross-strait national wind body bikini Swimsuit  Swimwear LBT]",
   "x_41": "x_vars[2018 New ZANZEA Beach Romper Women Jumpsuits Front Zipper Sleeveless Sexy Playsuits]",
   "x_42": "x_vars[2018 Plus Size Summer Women Fashion Sexy V-neck Lace Up Plaid Blouse Tops Irregular Short Sleeve Shirtï¼_S-5XLï¼_]",
   "x_43": "x_vars[2018 Summer Fashion Women Casual Camouflage Tank Top Sleeveless O-neck Vest]",
   "x_44": "x_vars[2018 Summer Fashion Women Tank Tops Sexy Women Sleeveless Crop Tops Casual Style Women Cotton Print Lace Stitching Irregular Blouse Topsï¼_S-5XL)]",
   "x_45": "x_vars[2018 Summer New Women Fashion Sexy Sleeveless Tank Top Floral Print Casual Women Cotton T-shirts Plus Size Women Vestï¼_S-5XLï¼_]",
   "x_46": "x_vars[2018 Summer New Women Long Skirts Solid Sexy Split Pencil Skirts]",
   "x_47": "x_vars[2018 Summer Women Print Top Fashion Women Casual Army Camo Camouflage Tank Top Sleeveless O-neck Slim T-Shirts Plus Size S-XXXXXL]",
   "x_48": "x_vars[2018 Summer Women Sexy Off Shoulder Shirt Casual Half Sleeve Top Ladies Fashion Printing Loose T-shirts Plus Size Cotton Blouse XS-5XL]",
   "x_49": "x_vars[2018 Summer Womenâ__s New Fashion Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Casual Sexy Cotton Tops]",
   "x_50": "x_vars[2018 T Shirts + Shorts Brand Clothing Tshirt Men Homme Letter Printed Basketball Running Sports Set T-shirt Suit Male]",
   "x_51": "x_vars[2018 Women Cool T-shirt Funny 3d Tshirt Print Two Cat Short Sleeve Summer Tops Tees Teen Graphic Tee Cute Shirt Funny Gifts]",
   "x_52": "x_vars[2018 Women Fashion Stretchy Camisole  Spaghetti Strap Long Tank Top Slip Summer Fashion Floral Mini Dress 7 Color SIZE XXL]",
   "x_53": "x_vars[2018 Women Fashion Summer Sleeveless Maxi Halter Dresses Solid Color Halter Maxi Dress with Halter Tie and Pockets]",
   "x_54": "x_vars[2018 Women Ladies Fashion Crocheted Lace Summer Dress Maxi Dress Long Dress]",
   "x_55": "x_vars[2018 Women Ladies Fashion Musical Note Asymmetric Tank Top]",
   "x_56": "x_vars[2018 Women Summer Casual Solid Color Loose Sleeveless Beach Tank Top A-line Pocket Dress Sexy Deep V-neck Short Club Party Mini Halter Midi Dresses Robe Ete Femme Knee Length Pleated Skirts Ladies Fashion Swing Cotton T-Shirt Dress Plus Size S-6XL]",
   "x_57": "x_vars[2018 Women fashion Summer Lace Patchwork Tank Tops Casual Sleeveless Tops Vest Blouse (S-5XL) Plus Size]",
   "x_58": "x_vars[2019 Fashion Family matching swimwear beachwear mommy and me swimsuit mother daughter father son clothes dresses high waist bikini look mum]",
   "x_59": "x_vars[2019 Fashion Women Summer V-Neck Sleeveless Collect Waist Boho Print Maxi Long Dress S-XXXXXL]",
   "x_60": "x_vars[2019 Men's Summer Cool Jogging Shorts]",
   "x_61": "x_vars[2019 NEW Men's Short Sleeve T-Shirt Fitness Round Neck Casual Men's Zippers Summer T-Shirt]",
   "x_62": "x_vars[2019 New Fashion Spring Summer Women's Long Pants Short Sleeves Off Shoulder Jumpsuit]",
   "x_63": "x_vars[2019 New Fashion Summer Women Casual Dress Round Neck Loose Big Swing Skirt Sleeveless Soild Color Beach dress]",
   "x_64": "x_vars[2019 New Fashion Women Blouse Tops Sleeveless Spaghetti Strap Criss Cross V-neck T Shirt Tops Summer Casual Blouse Shirts]",
   "x_65": "x_vars[2019 New Fashion Women Casual Shorts Suit Summer Tie-Dye Print Halter Bandage Sleeveless Backless Crop Top And Elastic Waist Shorts Pants Two Piece Set]",
   "x_66": "x_vars[2019 New Fashion Women Loose Round Neck Dress Short Sleeve Solid Color Casual Dress]",
   "x_67": "x_vars[2019 New Fashion Women Summer Funny Penguin print T-Shirts Casual Short Sleeve O-Neck Tops Loose Cotton Cute Cool Tee Plus Size S-5XL 4 Colors]",
   "x_68": "x_vars[2019 New Fashion Women's Slim Plus Size Maxi Dress Ink Printing Spaghetti Strap Dress]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-022": {
  "source_code_sha256": "79cd440ac5a8fdfcbc7151a5be0da9ea0d80b55258030b36d0fc687185146700",
  "original_solution_sha256": "e83759268b35c4c0697e96d5065e61abf933c0b0d55ec07a8cfeec9bd1abe122",
  "bindings": [
   [
    "C0",
    "x_vars[27in 4K Gaming Monitor]"
   ],
   [
    "C1",
    "x_vars[27in FHD Monitor]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[27in 4K Gaming Monitor]",
   "x_2": "x_vars[27in FHD Monitor]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-023": {
  "source_code_sha256": "58deca8e32cea92763504563d02fc551a7611d940fe3e5519d87a163f1e81008",
  "original_solution_sha256": "857b7c8f4516bfd435af112fc052f95ad5f99355ede98eb706dee8cd9328b565",
  "bindings": [
   [
    "C0",
    "x_vars[ELE-SMA-10000463]"
   ],
   [
    "C1",
    "x_vars[ELE-SMA-10000487]"
   ],
   [
    "C2",
    "x_vars[ELE-SMA-10003333]"
   ],
   [
    "C3",
    "x_vars[ELE-SMA-10009012]"
   ],
   [
    "C4",
    "x_vars[ELE-SMA-10009999]"
   ],
   [
    "C5",
    "x_vars[ELE-SMA-10011234]"
   ],
   [
    "C6",
    "x_vars[ELE-SMA-10027456]"
   ],
   [
    "C7",
    "x_vars[ELE-SMA-10028567]"
   ],
   [
    "C8",
    "x_vars[ELE-SPE-10000484]"
   ],
   [
    "C9",
    "x_vars[ELE-SPE-10003030]"
   ],
   [
    "C10",
    "x_vars[ELE-SPE-10024123]"
   ],
   [
    "C11",
    "x_vars[ELE-SPE-10025234]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[ELE-SMA-10000463]",
   "x_2": "x_vars[ELE-SMA-10000487]",
   "x_3": "x_vars[ELE-SMA-10003333]",
   "x_4": "x_vars[ELE-SMA-10009012]",
   "x_5": "x_vars[ELE-SMA-10009999]",
   "x_6": "x_vars[ELE-SMA-10011234]",
   "x_7": "x_vars[ELE-SMA-10027456]",
   "x_8": "x_vars[ELE-SMA-10028567]",
   "x_9": "x_vars[ELE-SPE-10000484]",
   "x_10": "x_vars[ELE-SPE-10003030]",
   "x_11": "x_vars[ELE-SPE-10024123]",
   "x_12": "x_vars[ELE-SPE-10025234]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-024": {
  "source_code_sha256": "0a08955d6ede0a5926d92d76d2a639077262a61a9ca06832d1d03174750c6df2",
  "original_solution_sha256": "7d7acdf448c3687be7923bf582fa7e480bada5f0038ec1298fe4d05bf00de380",
  "bindings": [
   [
    "C0",
    "x_vars[S700_1138]"
   ],
   [
    "C1",
    "x_vars[S700_1691]"
   ],
   [
    "C2",
    "x_vars[S700_1938]"
   ],
   [
    "C3",
    "x_vars[S700_2047]"
   ],
   [
    "C4",
    "x_vars[S700_2466]"
   ],
   [
    "C5",
    "x_vars[S700_2610]"
   ],
   [
    "C6",
    "x_vars[S700_2824]"
   ],
   [
    "C7",
    "x_vars[S700_2834]"
   ],
   [
    "C8",
    "x_vars[S700_3167]"
   ],
   [
    "C9",
    "x_vars[S700_3505]"
   ],
   [
    "C10",
    "x_vars[S700_3962]"
   ],
   [
    "C11",
    "x_vars[S700_4002]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[S700_1138]",
   "x_2": "x_vars[S700_1691]",
   "x_3": "x_vars[S700_1938]",
   "x_4": "x_vars[S700_2047]",
   "x_5": "x_vars[S700_2466]",
   "x_6": "x_vars[S700_2610]",
   "x_7": "x_vars[S700_2824]",
   "x_8": "x_vars[S700_2834]",
   "x_9": "x_vars[S700_3167]",
   "x_10": "x_vars[S700_3505]",
   "x_11": "x_vars[S700_3962]",
   "x_12": "x_vars[S700_4002]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-025": {
  "source_code_sha256": "f5cc5924e305172fc0890811b0003edd55ff371c2e381190b0f6c34f5234cc6a",
  "original_solution_sha256": "bea986bfc281338dcb4cdd25bd07ead2ec842c1b016df01a8522411c797e9871",
  "bindings": [
   [
    "C0",
    "x_vars[TABLET_10084.74]"
   ],
   [
    "C1",
    "x_vars[TABLET_12211.86]"
   ],
   [
    "C2",
    "x_vars[TABLET_14669.5]"
   ],
   [
    "C3",
    "x_vars[TABLET_14745.76]"
   ],
   [
    "C4",
    "x_vars[TABLET_14754.24]"
   ],
   [
    "C5",
    "x_vars[TABLET_16448.3]"
   ],
   [
    "C6",
    "x_vars[TABLET_16448.31]"
   ],
   [
    "C7",
    "x_vars[TABLET_20143.22]"
   ],
   [
    "C8",
    "x_vars[TABLET_2042.38]"
   ],
   [
    "C9",
    "x_vars[TABLET_24915.25]"
   ],
   [
    "C10",
    "x_vars[TABLET_24915.26]"
   ],
   [
    "C11",
    "x_vars[TABLET_26448.3]"
   ],
   [
    "C12",
    "x_vars[TABLET_27042.38]"
   ],
   [
    "C13",
    "x_vars[TABLET_30000.0]"
   ],
   [
    "C14",
    "x_vars[TABLET_33397.46]"
   ],
   [
    "C15",
    "x_vars[TABLET_33398.3]"
   ],
   [
    "C16",
    "x_vars[TABLET_48567.8]"
   ],
   [
    "C17",
    "x_vars[TABLET_48644.07]"
   ],
   [
    "C18",
    "x_vars[TABLET_50262.72]"
   ],
   [
    "C19",
    "x_vars[TABLET_53736.44]"
   ],
   [
    "C20",
    "x_vars[TABLET_6957.62]"
   ],
   [
    "C21",
    "x_vars[TABLET_6957.63]"
   ],
   [
    "C22",
    "x_vars[TABLET_7550.84]"
   ],
   [
    "C23",
    "x_vars[TABLET_7550.85]"
   ],
   [
    "C24",
    "x_vars[TABLET_9584.74]"
   ],
   [
    "C25",
    "x_vars[TABLET_9661.02]"
   ],
   [
    "C26",
    "x_vars[TABLET_9669.5]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[TABLET_10084.74]",
   "x_2": "x_vars[TABLET_12211.86]",
   "x_3": "x_vars[TABLET_14669.5]",
   "x_4": "x_vars[TABLET_14745.76]",
   "x_5": "x_vars[TABLET_14754.24]",
   "x_6": "x_vars[TABLET_16448.3]",
   "x_7": "x_vars[TABLET_16448.31]",
   "x_8": "x_vars[TABLET_20143.22]",
   "x_9": "x_vars[TABLET_2042.38]",
   "x_10": "x_vars[TABLET_24915.25]",
   "x_11": "x_vars[TABLET_24915.26]",
   "x_12": "x_vars[TABLET_26448.3]",
   "x_13": "x_vars[TABLET_27042.38]",
   "x_14": "x_vars[TABLET_30000.0]",
   "x_15": "x_vars[TABLET_33397.46]",
   "x_16": "x_vars[TABLET_33398.3]",
   "x_17": "x_vars[TABLET_48567.8]",
   "x_18": "x_vars[TABLET_48644.07]",
   "x_19": "x_vars[TABLET_50262.72]",
   "x_20": "x_vars[TABLET_53736.44]",
   "x_21": "x_vars[TABLET_6957.62]",
   "x_22": "x_vars[TABLET_6957.63]",
   "x_23": "x_vars[TABLET_7550.84]",
   "x_24": "x_vars[TABLET_7550.85]",
   "x_25": "x_vars[TABLET_9584.74]",
   "x_26": "x_vars[TABLET_9661.02]",
   "x_27": "x_vars[TABLET_9669.5]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-026": {
  "source_code_sha256": "384b3ae402340587b30928823ca109a67fcdfa3ef34a6ab4cfb9489fa730a59c",
  "original_solution_sha256": "98299db46bc1cda46ec192ef54092dba5fd6595a39d763406f43818c2fabc865",
  "bindings": [
   [
    "C0",
    "x_vars[Fashion accessories_10.18]"
   ],
   [
    "C1",
    "x_vars[Fashion accessories_12.09]"
   ],
   [
    "C2",
    "x_vars[Fashion accessories_12.19]"
   ],
   [
    "C3",
    "x_vars[Fashion accessories_12.54]"
   ],
   [
    "C4",
    "x_vars[Fashion accessories_12.78]"
   ],
   [
    "C5",
    "x_vars[Fashion accessories_14.48]"
   ],
   [
    "C6",
    "x_vars[Fashion accessories_15.43]"
   ],
   [
    "C7",
    "x_vars[Fashion accessories_15.5]"
   ],
   [
    "C8",
    "x_vars[Fashion accessories_15.62]"
   ],
   [
    "C9",
    "x_vars[Fashion accessories_16.28]"
   ],
   [
    "C10",
    "x_vars[Fashion accessories_16.45]"
   ],
   [
    "C11",
    "x_vars[Fashion accessories_17.48]"
   ],
   [
    "C12",
    "x_vars[Fashion accessories_17.49]"
   ],
   [
    "C13",
    "x_vars[Fashion accessories_17.87]"
   ],
   [
    "C14",
    "x_vars[Fashion accessories_17.94]"
   ],
   [
    "C15",
    "x_vars[Fashion accessories_18.08]"
   ],
   [
    "C16",
    "x_vars[Fashion accessories_19.66]"
   ],
   [
    "C17",
    "x_vars[Fashion accessories_19.7]"
   ],
   [
    "C18",
    "x_vars[Fashion accessories_19.77]"
   ],
   [
    "C19",
    "x_vars[Fashion accessories_20.01]"
   ],
   [
    "C20",
    "x_vars[Fashion accessories_21.32]"
   ],
   [
    "C21",
    "x_vars[Fashion accessories_21.48]"
   ],
   [
    "C22",
    "x_vars[Fashion accessories_21.94]"
   ],
   [
    "C23",
    "x_vars[Fashion accessories_22.32]"
   ],
   [
    "C24",
    "x_vars[Fashion accessories_22.51]"
   ],
   [
    "C25",
    "x_vars[Fashion accessories_23.82]"
   ],
   [
    "C26",
    "x_vars[Fashion accessories_25.42]"
   ],
   [
    "C27",
    "x_vars[Fashion accessories_25.56]"
   ],
   [
    "C28",
    "x_vars[Fashion accessories_27.02]"
   ],
   [
    "C29",
    "x_vars[Fashion accessories_27.18]"
   ],
   [
    "C30",
    "x_vars[Fashion accessories_27.38]"
   ],
   [
    "C31",
    "x_vars[Fashion accessories_29.42]"
   ],
   [
    "C32",
    "x_vars[Fashion accessories_29.56]"
   ],
   [
    "C33",
    "x_vars[Fashion accessories_30.14]"
   ],
   [
    "C34",
    "x_vars[Fashion accessories_30.37]"
   ],
   [
    "C35",
    "x_vars[Fashion accessories_30.61]"
   ],
   [
    "C36",
    "x_vars[Fashion accessories_30.62]"
   ],
   [
    "C37",
    "x_vars[Fashion accessories_31.73]"
   ],
   [
    "C38",
    "x_vars[Fashion accessories_31.9]"
   ],
   [
    "C39",
    "x_vars[Fashion accessories_32.62]"
   ],
   [
    "C40",
    "x_vars[Fashion accessories_33.52]"
   ],
   [
    "C41",
    "x_vars[Fashion accessories_33.63]"
   ],
   [
    "C42",
    "x_vars[Fashion accessories_34.7]"
   ],
   [
    "C43",
    "x_vars[Fashion accessories_35.19]"
   ],
   [
    "C44",
    "x_vars[Fashion accessories_36.51]"
   ],
   [
    "C45",
    "x_vars[Fashion accessories_36.85]"
   ],
   [
    "C46",
    "x_vars[Fashion accessories_37.15]"
   ],
   [
    "C47",
    "x_vars[Fashion accessories_37.55]"
   ],
   [
    "C48",
    "x_vars[Fashion accessories_37.95]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Fashion accessories_10.18]",
   "x_2": "x_vars[Fashion accessories_12.09]",
   "x_3": "x_vars[Fashion accessories_12.19]",
   "x_4": "x_vars[Fashion accessories_12.54]",
   "x_5": "x_vars[Fashion accessories_12.78]",
   "x_6": "x_vars[Fashion accessories_14.48]",
   "x_7": "x_vars[Fashion accessories_15.43]",
   "x_8": "x_vars[Fashion accessories_15.5]",
   "x_9": "x_vars[Fashion accessories_15.62]",
   "x_10": "x_vars[Fashion accessories_16.28]",
   "x_11": "x_vars[Fashion accessories_16.45]",
   "x_12": "x_vars[Fashion accessories_17.48]",
   "x_13": "x_vars[Fashion accessories_17.49]",
   "x_14": "x_vars[Fashion accessories_17.87]",
   "x_15": "x_vars[Fashion accessories_17.94]",
   "x_16": "x_vars[Fashion accessories_18.08]",
   "x_17": "x_vars[Fashion accessories_19.66]",
   "x_18": "x_vars[Fashion accessories_19.7]",
   "x_19": "x_vars[Fashion accessories_19.77]",
   "x_20": "x_vars[Fashion accessories_20.01]",
   "x_21": "x_vars[Fashion accessories_21.32]",
   "x_22": "x_vars[Fashion accessories_21.48]",
   "x_23": "x_vars[Fashion accessories_21.94]",
   "x_24": "x_vars[Fashion accessories_22.32]",
   "x_25": "x_vars[Fashion accessories_22.51]",
   "x_26": "x_vars[Fashion accessories_23.82]",
   "x_27": "x_vars[Fashion accessories_25.42]",
   "x_28": "x_vars[Fashion accessories_25.56]",
   "x_29": "x_vars[Fashion accessories_27.02]",
   "x_30": "x_vars[Fashion accessories_27.18]",
   "x_31": "x_vars[Fashion accessories_27.38]",
   "x_32": "x_vars[Fashion accessories_29.42]",
   "x_33": "x_vars[Fashion accessories_29.56]",
   "x_34": "x_vars[Fashion accessories_30.14]",
   "x_35": "x_vars[Fashion accessories_30.37]",
   "x_36": "x_vars[Fashion accessories_30.61]",
   "x_37": "x_vars[Fashion accessories_30.62]",
   "x_38": "x_vars[Fashion accessories_31.73]",
   "x_39": "x_vars[Fashion accessories_31.9]",
   "x_40": "x_vars[Fashion accessories_32.62]",
   "x_41": "x_vars[Fashion accessories_33.52]",
   "x_42": "x_vars[Fashion accessories_33.63]",
   "x_43": "x_vars[Fashion accessories_34.7]",
   "x_44": "x_vars[Fashion accessories_35.19]",
   "x_45": "x_vars[Fashion accessories_36.51]",
   "x_46": "x_vars[Fashion accessories_36.85]",
   "x_47": "x_vars[Fashion accessories_37.15]",
   "x_48": "x_vars[Fashion accessories_37.55]",
   "x_49": "x_vars[Fashion accessories_37.95]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-027": {
  "source_code_sha256": "12060e75d5ab526795799ac165e828e7432ff1349941d659851d9221a70aaa73",
  "original_solution_sha256": "6f02b80d9ed8484982472ade0bbe3da295492220b93f445a34743a4fdd6c3fd0",
  "bindings": [
   [
    "C0",
    "x_vars[Organic Fruits]"
   ],
   [
    "C1",
    "x_vars[Organic Staples]"
   ],
   [
    "C2",
    "x_vars[Organic Vegetables]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Organic Fruits]",
   "x_2": "x_vars[Organic Staples]",
   "x_3": "x_vars[Organic Vegetables]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-028": {
  "source_code_sha256": "83f681bdb5d79a9340083e62fa381c69e169c01eb62df10e46005d097d0a1411",
  "original_solution_sha256": "c342ff4f837337896a13f2e71a8dd1634f4e6432b9786821ebf1ef1a84c3f22f",
  "bindings": [
   [
    "C0",
    "x_vars[sku_I27]"
   ],
   [
    "C1",
    "x_vars[sku_I499]"
   ],
   [
    "C2",
    "x_vars[sku_I719]"
   ],
   [
    "C3",
    "x_vars[sku_T18]"
   ],
   [
    "C4",
    "x_vars[sku_T29]"
   ],
   [
    "C5",
    "x_vars[sku_T39]"
   ],
   [
    "C6",
    "x_vars[sku_T499]"
   ],
   [
    "C7",
    "x_vars[sku_T9]"
   ],
   [
    "C8",
    "x_vars[sku_3081]"
   ],
   [
    "C9",
    "x_vars[sku_339]"
   ],
   [
    "C10",
    "x_vars[sku_3799]"
   ],
   [
    "C11",
    "x_vars[sku_439]"
   ],
   [
    "C12",
    "x_vars[sku_539]"
   ],
   [
    "C13",
    "x_vars[sku_61399]"
   ],
   [
    "C14",
    "x_vars[sku_628]"
   ],
   [
    "C15",
    "x_vars[sku_708]"
   ],
   [
    "C16",
    "x_vars[sku_77]"
   ],
   [
    "C17",
    "x_vars[sku_79]"
   ],
   [
    "C18",
    "x_vars[sku_799]"
   ],
   [
    "C19",
    "x_vars[sku_8499]"
   ],
   [
    "C20",
    "x_vars[sku_89]"
   ],
   [
    "C21",
    "x_vars[sku_897]"
   ],
   [
    "C22",
    "x_vars[sku_9699]"
   ],
   [
    "C23",
    "x_vars[sku_bobo]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[sku_I27]",
   "x_2": "x_vars[sku_I499]",
   "x_3": "x_vars[sku_I719]",
   "x_4": "x_vars[sku_T18]",
   "x_5": "x_vars[sku_T29]",
   "x_6": "x_vars[sku_T39]",
   "x_7": "x_vars[sku_T499]",
   "x_8": "x_vars[sku_T9]",
   "x_9": "x_vars[sku_3081]",
   "x_10": "x_vars[sku_339]",
   "x_11": "x_vars[sku_3799]",
   "x_12": "x_vars[sku_439]",
   "x_13": "x_vars[sku_539]",
   "x_14": "x_vars[sku_61399]",
   "x_15": "x_vars[sku_628]",
   "x_16": "x_vars[sku_708]",
   "x_17": "x_vars[sku_77]",
   "x_18": "x_vars[sku_79]",
   "x_19": "x_vars[sku_799]",
   "x_20": "x_vars[sku_8499]",
   "x_21": "x_vars[sku_89]",
   "x_22": "x_vars[sku_897]",
   "x_23": "x_vars[sku_9699]",
   "x_24": "x_vars[sku_bobo]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-029": {
  "source_code_sha256": "aa8d4ff041c6c22d154c9223fe1b384253babc9220e4a2f485dc55904d0ff207",
  "original_solution_sha256": "fbf70fd20dab297bb8104fa3e6880c9ec863d39164c4fa923526f21164bb9ba5",
  "bindings": [
   [
    "C0",
    "x_vars[FAUX FUR JEWEL SWEATER]"
   ],
   [
    "C1",
    "x_vars[FAUX LEATHER BOMBER JACKET]"
   ],
   [
    "C2",
    "x_vars[FAUX LEATHER BOXY FIT JACKET]"
   ],
   [
    "C3",
    "x_vars[FAUX LEATHER JACKET]"
   ],
   [
    "C4",
    "x_vars[FAUX LEATHER OVERSIZED JACKET LIMITED EDITION]"
   ],
   [
    "C5",
    "x_vars[FAUX LEATHER PUFFER JACKET]"
   ],
   [
    "C6",
    "x_vars[FAUX SHEARLING LINED SUEDE BOOTS]"
   ],
   [
    "C7",
    "x_vars[FAUX SHEARLING PLAID JACKET]"
   ],
   [
    "C8",
    "x_vars[FAUX SUEDE BOMBER JACKET]"
   ],
   [
    "C9",
    "x_vars[FAUX SUEDE JACKET]"
   ],
   [
    "C10",
    "x_vars[FAUX SUEDE OVERSHIRT]"
   ],
   [
    "C11",
    "x_vars[FAUX SUEDE PATCH JACKET]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[FAUX FUR JEWEL SWEATER]",
   "x_2": "x_vars[FAUX LEATHER BOMBER JACKET]",
   "x_3": "x_vars[FAUX LEATHER BOXY FIT JACKET]",
   "x_4": "x_vars[FAUX LEATHER JACKET]",
   "x_5": "x_vars[FAUX LEATHER OVERSIZED JACKET LIMITED EDITION]",
   "x_6": "x_vars[FAUX LEATHER PUFFER JACKET]",
   "x_7": "x_vars[FAUX SHEARLING LINED SUEDE BOOTS]",
   "x_8": "x_vars[FAUX SHEARLING PLAID JACKET]",
   "x_9": "x_vars[FAUX SUEDE BOMBER JACKET]",
   "x_10": "x_vars[FAUX SUEDE JACKET]",
   "x_11": "x_vars[FAUX SUEDE OVERSHIRT]",
   "x_12": "x_vars[FAUX SUEDE PATCH JACKET]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-030": {
  "source_code_sha256": "c197e197b87d5f830a00043936d51c1f70fb4acac20bd49a8f7f8f83c43b2ac8",
  "original_solution_sha256": "e2de0211b4d242a6080cee5137f39314222fa17c6d1c709dfd4b32df81c15aeb",
  "bindings": [
   [
    "C0",
    "x_vars[1163]"
   ],
   [
    "C1",
    "x_vars[1501]"
   ],
   [
    "C2",
    "x_vars[1576]"
   ],
   [
    "C3",
    "x_vars[1793]"
   ],
   [
    "C4",
    "x_vars[2438]"
   ],
   [
    "C5",
    "x_vars[4297]"
   ],
   [
    "C6",
    "x_vars[4806]"
   ],
   [
    "C7",
    "x_vars[4942]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[1163]",
   "x_2": "x_vars[1501]",
   "x_3": "x_vars[1576]",
   "x_4": "x_vars[1793]",
   "x_5": "x_vars[2438]",
   "x_6": "x_vars[4297]",
   "x_7": "x_vars[4806]",
   "x_8": "x_vars[4942]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-031": {
  "source_code_sha256": "9e76d9780ad0a43bea4239cf0584d0fd4b30bc4c8e0d08fa009cb0745867f8ac",
  "original_solution_sha256": "93b28149b40f21c890f3bde2c27821a82abcd65216ecf90b8e40455845517b81",
  "bindings": [
   [
    "C0",
    "x_vars[Butter_Amul]"
   ],
   [
    "C1",
    "x_vars[Butter_Mother Dairy]"
   ],
   [
    "C2",
    "x_vars[Butter_Parag Milk Foods]"
   ],
   [
    "C3",
    "x_vars[Butter_Warana]"
   ],
   [
    "C4",
    "x_vars[Buttermilk_Amul]"
   ],
   [
    "C5",
    "x_vars[Buttermilk_Mother Dairy]"
   ],
   [
    "C6",
    "x_vars[Buttermilk_Raj]"
   ],
   [
    "C7",
    "x_vars[Buttermilk_Sudha]"
   ],
   [
    "C8",
    "x_vars[Cheese_Amul]"
   ],
   [
    "C9",
    "x_vars[Cheese_Britannia Industries]"
   ],
   [
    "C10",
    "x_vars[Cheese_Dynamix Dairies]"
   ],
   [
    "C11",
    "x_vars[Cheese_Passion Cheese]"
   ],
   [
    "C12",
    "x_vars[Curd_Amul]"
   ],
   [
    "C13",
    "x_vars[Curd_Mother Dairy]"
   ],
   [
    "C14",
    "x_vars[Curd_Raj]"
   ],
   [
    "C15",
    "x_vars[Curd_Sudha]"
   ],
   [
    "C16",
    "x_vars[Ghee_Amul]"
   ],
   [
    "C17",
    "x_vars[Ghee_Mother Dairy]"
   ],
   [
    "C18",
    "x_vars[Ghee_Raj]"
   ],
   [
    "C19",
    "x_vars[Ghee_Sudha]"
   ],
   [
    "C20",
    "x_vars[Ice Cream_Amul]"
   ],
   [
    "C21",
    "x_vars[Ice Cream_Dodla Dairy]"
   ],
   [
    "C22",
    "x_vars[Ice Cream_Mother Dairy]"
   ],
   [
    "C23",
    "x_vars[Ice Cream_Palle2patnam]"
   ],
   [
    "C24",
    "x_vars[Lassi_Amul]"
   ],
   [
    "C25",
    "x_vars[Lassi_Mother Dairy]"
   ],
   [
    "C26",
    "x_vars[Lassi_Raj]"
   ],
   [
    "C27",
    "x_vars[Lassi_Sudha]"
   ],
   [
    "C28",
    "x_vars[Milk_Amul]"
   ],
   [
    "C29",
    "x_vars[Milk_Mother Dairy]"
   ],
   [
    "C30",
    "x_vars[Milk_Raj]"
   ],
   [
    "C31",
    "x_vars[Milk_Sudha]"
   ],
   [
    "C32",
    "x_vars[Paneer_Amul]"
   ],
   [
    "C33",
    "x_vars[Paneer_Mother Dairy]"
   ],
   [
    "C34",
    "x_vars[Paneer_Raj]"
   ],
   [
    "C35",
    "x_vars[Paneer_Sudha]"
   ],
   [
    "C36",
    "x_vars[Yogurt_Amul]"
   ],
   [
    "C37",
    "x_vars[Yogurt_Dodla Dairy]"
   ],
   [
    "C38",
    "x_vars[Yogurt_Mother Dairy]"
   ],
   [
    "C39",
    "x_vars[Yogurt_Palle2patnam]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Butter_Amul]",
   "x_2": "x_vars[Butter_Mother Dairy]",
   "x_3": "x_vars[Butter_Parag Milk Foods]",
   "x_4": "x_vars[Butter_Warana]",
   "x_5": "x_vars[Buttermilk_Amul]",
   "x_6": "x_vars[Buttermilk_Mother Dairy]",
   "x_7": "x_vars[Buttermilk_Raj]",
   "x_8": "x_vars[Buttermilk_Sudha]",
   "x_9": "x_vars[Cheese_Amul]",
   "x_10": "x_vars[Cheese_Britannia Industries]",
   "x_11": "x_vars[Cheese_Dynamix Dairies]",
   "x_12": "x_vars[Cheese_Passion Cheese]",
   "x_13": "x_vars[Curd_Amul]",
   "x_14": "x_vars[Curd_Mother Dairy]",
   "x_15": "x_vars[Curd_Raj]",
   "x_16": "x_vars[Curd_Sudha]",
   "x_17": "x_vars[Ghee_Amul]",
   "x_18": "x_vars[Ghee_Mother Dairy]",
   "x_19": "x_vars[Ghee_Raj]",
   "x_20": "x_vars[Ghee_Sudha]",
   "x_21": "x_vars[Ice Cream_Amul]",
   "x_22": "x_vars[Ice Cream_Dodla Dairy]",
   "x_23": "x_vars[Ice Cream_Mother Dairy]",
   "x_24": "x_vars[Ice Cream_Palle2patnam]",
   "x_25": "x_vars[Lassi_Amul]",
   "x_26": "x_vars[Lassi_Mother Dairy]",
   "x_27": "x_vars[Lassi_Raj]",
   "x_28": "x_vars[Lassi_Sudha]",
   "x_29": "x_vars[Milk_Amul]",
   "x_30": "x_vars[Milk_Mother Dairy]",
   "x_31": "x_vars[Milk_Raj]",
   "x_32": "x_vars[Milk_Sudha]",
   "x_33": "x_vars[Paneer_Amul]",
   "x_34": "x_vars[Paneer_Mother Dairy]",
   "x_35": "x_vars[Paneer_Raj]",
   "x_36": "x_vars[Paneer_Sudha]",
   "x_37": "x_vars[Yogurt_Amul]",
   "x_38": "x_vars[Yogurt_Dodla Dairy]",
   "x_39": "x_vars[Yogurt_Mother Dairy]",
   "x_40": "x_vars[Yogurt_Palle2patnam]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-032": {
  "source_code_sha256": "d7cb29419a8e0b39c1ddc7310046e1a3cc8c7d64fde5320c53eadf69a084cc43",
  "original_solution_sha256": "1b784364c94bf18cad5b7749fd9105fc0236256f7fe15effa75af73e97f94e3d",
  "bindings": [
   [
    "C0",
    "x_vars[Books_15.15]"
   ],
   [
    "C1",
    "x_vars[Books_30.3]"
   ],
   [
    "C2",
    "x_vars[Books_45.45]"
   ],
   [
    "C3",
    "x_vars[Books_60.6]"
   ],
   [
    "C4",
    "x_vars[Books_75.75]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Books_15.15]",
   "x_2": "x_vars[Books_30.3]",
   "x_3": "x_vars[Books_45.45]",
   "x_4": "x_vars[Books_60.6]",
   "x_5": "x_vars[Books_75.75]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-033": {
  "source_code_sha256": "55614195f29ad3ed6fab5fd3be4af5a572728fb29a3d3f00dc177fc7e0c9f977",
  "original_solution_sha256": "2821aefe9bfe2db49216fd701a4a17406bc1586d54ed2c33e130cc524de64cfa",
  "bindings": [
   [
    "C0",
    "x_vars[Baby Food_255.28]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[Baby Food_255.28]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-034": {
  "source_code_sha256": "73f053b4657f4aaf81e3b816cc7734189bbc5b61abd3be18c18daad516f3b715",
  "original_solution_sha256": "145176984d504aea75da64f4e7b42ab4811fa4fe2ddc58e0ad23b914c9fff8f9",
  "bindings": [
   [
    "C0",
    "x_vars[12 MACARON]"
   ],
   [
    "C1",
    "x_vars[ARMORICAIN]"
   ],
   [
    "C2",
    "x_vars[ARTICLE 295]"
   ],
   [
    "C3",
    "x_vars[BAGUETTE]"
   ],
   [
    "C4",
    "x_vars[BAGUETTE APERO]"
   ],
   [
    "C5",
    "x_vars[BAGUETTE GRAINE]"
   ],
   [
    "C6",
    "x_vars[BANETTE]"
   ],
   [
    "C7",
    "x_vars[BANETTINE]"
   ],
   [
    "C8",
    "x_vars[BOISSON 33CL]"
   ],
   [
    "C9",
    "x_vars[BOTTEREAU]"
   ],
   [
    "C10",
    "x_vars[BOULE 200G]"
   ],
   [
    "C11",
    "x_vars[BOULE 400G]"
   ],
   [
    "C12",
    "x_vars[BOULE POLKA]"
   ],
   [
    "C13",
    "x_vars[BRIOCHE]"
   ],
   [
    "C14",
    "x_vars[BRIOCHE DE NOEL]"
   ],
   [
    "C15",
    "x_vars[BRIOCHETTE]"
   ],
   [
    "C16",
    "x_vars[BROWNIES]"
   ],
   [
    "C17",
    "x_vars[BUCHE 4PERS]"
   ],
   [
    "C18",
    "x_vars[BUCHE 6PERS]"
   ],
   [
    "C19",
    "x_vars[BUCHE 8PERS]"
   ],
   [
    "C20",
    "x_vars[CAFE OU EAU]"
   ],
   [
    "C21",
    "x_vars[CAKE]"
   ],
   [
    "C22",
    "x_vars[CAMPAGNE]"
   ],
   [
    "C23",
    "x_vars[CARAMEL NOIX]"
   ],
   [
    "C24",
    "x_vars[CEREAL BAGUETTE]"
   ],
   [
    "C25",
    "x_vars[CHAUSSON AUX POMMES]"
   ],
   [
    "C26",
    "x_vars[CHOCOLAT]"
   ],
   [
    "C27",
    "x_vars[CHOU CHANTILLY]"
   ],
   [
    "C28",
    "x_vars[COMPLET]"
   ],
   [
    "C29",
    "x_vars[COOKIE]"
   ],
   [
    "C30",
    "x_vars[COUPE]"
   ],
   [
    "C31",
    "x_vars[CROISSANT]"
   ],
   [
    "C32",
    "x_vars[CROISSANT AMANDES]"
   ],
   [
    "C33",
    "x_vars[CRUMBLE]"
   ],
   [
    "C34",
    "x_vars[CRUMBLECARAMEL OU PISTAE]"
   ],
   [
    "C35",
    "x_vars[DELICETROPICAL]"
   ],
   [
    "C36",
    "x_vars[DEMI BAGUETTE]"
   ],
   [
    "C37",
    "x_vars[DEMI PAIN]"
   ],
   [
    "C38",
    "x_vars[DIVERS BOISSONS]"
   ],
   [
    "C39",
    "x_vars[DIVERS BOULANGERIE]"
   ],
   [
    "C40",
    "x_vars[DIVERS CONFISERIE]"
   ],
   [
    "C41",
    "x_vars[DIVERS PATISSERIE]"
   ],
   [
    "C42",
    "x_vars[DIVERS SANDWICHS]"
   ],
   [
    "C43",
    "x_vars[DIVERS VIENNOISERIE]"
   ],
   [
    "C44",
    "x_vars[DOUCEUR D HIVER]"
   ],
   [
    "C45",
    "x_vars[ECLAIR]"
   ],
   [
    "C46",
    "x_vars[ECLAIR FRAISE PISTACHE]"
   ],
   [
    "C47",
    "x_vars[ENTREMETS]"
   ],
   [
    "C48",
    "x_vars[FICELLE]"
   ],
   [
    "C49",
    "x_vars[FINANCIER]"
   ],
   [
    "C50",
    "x_vars[FINANCIER X5]"
   ],
   [
    "C51",
    "x_vars[FLAN]"
   ],
   [
    "C52",
    "x_vars[FLAN ABRICOT]"
   ],
   [
    "C53",
    "x_vars[FONDANT CHOCOLAT]"
   ],
   [
    "C54",
    "x_vars[FORMULE PATE]"
   ],
   [
    "C55",
    "x_vars[FORMULE PLAT PREPARE]"
   ],
   [
    "C56",
    "x_vars[FORMULE SANDWICH]"
   ],
   [
    "C57",
    "x_vars[FRAISIER]"
   ],
   [
    "C58",
    "x_vars[FRAMBOISIER]"
   ],
   [
    "C59",
    "x_vars[GACHE]"
   ],
   [
    "C60",
    "x_vars[GAL FRANGIPANE 4P]"
   ],
   [
    "C61",
    "x_vars[GAL FRANGIPANE 6P]"
   ],
   [
    "C62",
    "x_vars[GAL POIRE CHOCO 4P]"
   ],
   [
    "C63",
    "x_vars[GAL POIRE CHOCO 6P]"
   ],
   [
    "C64",
    "x_vars[GAL POMME 4P]"
   ],
   [
    "C65",
    "x_vars[GAL POMME 6P]"
   ],
   [
    "C66",
    "x_vars[GALETTE 8 PERS]"
   ],
   [
    "C67",
    "x_vars[GD FAR BRETON]"
   ],
   [
    "C68",
    "x_vars[GD KOUIGN AMANN]"
   ],
   [
    "C69",
    "x_vars[GD NANTAIS]"
   ],
   [
    "C70",
    "x_vars[GD PLATEAU SALE]"
   ],
   [
    "C71",
    "x_vars[GRAND FAR BRETON]"
   ],
   [
    "C72",
    "x_vars[GRANDE SUCETTE]"
   ],
   [
    "C73",
    "x_vars[GUERANDAIS]"
   ],
   [
    "C74",
    "x_vars[KOUIGN AMANN]"
   ],
   [
    "C75",
    "x_vars[MACARON]"
   ],
   [
    "C76",
    "x_vars[MERINGUE]"
   ],
   [
    "C77",
    "x_vars[MILLES FEUILLES]"
   ],
   [
    "C78",
    "x_vars[MOISSON]"
   ],
   [
    "C79",
    "x_vars[NANTAIS]"
   ],
   [
    "C80",
    "x_vars[NID DE POULE]"
   ],
   [
    "C81",
    "x_vars[NOIX JAPONAISE]"
   ],
   [
    "C82",
    "x_vars[PAILLE]"
   ],
   [
    "C83",
    "x_vars[PAIN]"
   ],
   [
    "C84",
    "x_vars[PAIN AU CHOCOLAT]"
   ],
   [
    "C85",
    "x_vars[PAIN AUX RAISINS]"
   ],
   [
    "C86",
    "x_vars[PAIN BANETTE]"
   ],
   [
    "C87",
    "x_vars[PAIN CHOCO AMANDES]"
   ],
   [
    "C88",
    "x_vars[PAIN DE MIE]"
   ],
   [
    "C89",
    "x_vars[PAIN GRAINES]"
   ],
   [
    "C90",
    "x_vars[PAIN NOIR]"
   ],
   [
    "C91",
    "x_vars[PAIN S/SEL]"
   ],
   [
    "C92",
    "x_vars[PAIN SUISSE PEPITO]"
   ],
   [
    "C93",
    "x_vars[PALET BRETON]"
   ],
   [
    "C94",
    "x_vars[PALMIER]"
   ],
   [
    "C95",
    "x_vars[PARIS BREST]"
   ],
   [
    "C96",
    "x_vars[PATES]"
   ],
   [
    "C97",
    "x_vars[PLAQUE TARTE 25P]"
   ],
   [
    "C98",
    "x_vars[PLAT]"
   ],
   [
    "C99",
    "x_vars[PLAT 6.50E]"
   ],
   [
    "C100",
    "x_vars[PLAT 7.00]"
   ],
   [
    "C101",
    "x_vars[PLAT 7.60E]"
   ],
   [
    "C102",
    "x_vars[PLAT 8.30E]"
   ],
   [
    "C103",
    "x_vars[PLATPREPARE5,50]"
   ],
   [
    "C104",
    "x_vars[PLATPREPARE6,00]"
   ],
   [
    "C105",
    "x_vars[PLATPREPARE6,50]"
   ],
   [
    "C106",
    "x_vars[PLATPREPARE7,00]"
   ],
   [
    "C107",
    "x_vars[PT NANTAIS]"
   ],
   [
    "C108",
    "x_vars[PT PLATEAU SALE]"
   ],
   [
    "C109",
    "x_vars[QUIM BREAD]"
   ],
   [
    "C110",
    "x_vars[REDUCTION SUCREES 12]"
   ],
   [
    "C111",
    "x_vars[REDUCTION SUCREES 24]"
   ],
   [
    "C112",
    "x_vars[RELIGIEUSE]"
   ],
   [
    "C113",
    "x_vars[ROYAL]"
   ],
   [
    "C114",
    "x_vars[ROYAL 4P]"
   ],
   [
    "C115",
    "x_vars[ROYAL 6P]"
   ],
   [
    "C116",
    "x_vars[SABLE F  P]"
   ],
   [
    "C117",
    "x_vars[SACHET DE CROUTON]"
   ],
   [
    "C118",
    "x_vars[SACHET DE VIENNOISERIE]"
   ],
   [
    "C119",
    "x_vars[SACHET VIENNOISERIE]"
   ],
   [
    "C120",
    "x_vars[SAND JB]"
   ],
   [
    "C121",
    "x_vars[SAND JB EMMENTAL]"
   ],
   [
    "C122",
    "x_vars[SANDWICH COMPLET]"
   ],
   [
    "C123",
    "x_vars[SAVARIN]"
   ],
   [
    "C124",
    "x_vars[SEIGLE]"
   ],
   [
    "C125",
    "x_vars[SPECIAL BREAD]"
   ],
   [
    "C126",
    "x_vars[SPECIAL BREAD KG]"
   ],
   [
    "C127",
    "x_vars[ST HONORE]"
   ],
   [
    "C128",
    "x_vars[SUCETTE]"
   ],
   [
    "C129",
    "x_vars[TARTE FINE]"
   ],
   [
    "C130",
    "x_vars[TARTE FRAISE 4PER]"
   ],
   [
    "C131",
    "x_vars[TARTE FRAISE 6P]"
   ],
   [
    "C132",
    "x_vars[TARTE FRUITS 4P]"
   ],
   [
    "C133",
    "x_vars[TARTE FRUITS 6P]"
   ],
   [
    "C134",
    "x_vars[TARTELETTE]"
   ],
   [
    "C135",
    "x_vars[TARTELETTE CHOC]"
   ],
   [
    "C136",
    "x_vars[TARTELETTE COCKTAIL]"
   ],
   [
    "C137",
    "x_vars[TARTELETTE FRAISE]"
   ],
   [
    "C138",
    "x_vars[THE]"
   ],
   [
    "C139",
    "x_vars[TRADITIONAL BAGUETTE]"
   ],
   [
    "C140",
    "x_vars[TRAITEUR]"
   ],
   [
    "C141",
    "x_vars[TRIANGLES]"
   ],
   [
    "C142",
    "x_vars[TROIS CHOCOLAT]"
   ],
   [
    "C143",
    "x_vars[TROPEZIENNE]"
   ],
   [
    "C144",
    "x_vars[TROPEZIENNE FRAMBOISE]"
   ],
   [
    "C145",
    "x_vars[TULIPE]"
   ],
   [
    "C146",
    "x_vars[VIENNOISE]"
   ],
   [
    "C147",
    "x_vars[VIK BREAD]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x_vars[12 MACARON]",
   "x_2": "x_vars[ARMORICAIN]",
   "x_3": "x_vars[ARTICLE 295]",
   "x_4": "x_vars[BAGUETTE]",
   "x_5": "x_vars[BAGUETTE APERO]",
   "x_6": "x_vars[BAGUETTE GRAINE]",
   "x_7": "x_vars[BANETTE]",
   "x_8": "x_vars[BANETTINE]",
   "x_9": "x_vars[BOISSON 33CL]",
   "x_10": "x_vars[BOTTEREAU]",
   "x_11": "x_vars[BOULE 200G]",
   "x_12": "x_vars[BOULE 400G]",
   "x_13": "x_vars[BOULE POLKA]",
   "x_14": "x_vars[BRIOCHE]",
   "x_15": "x_vars[BRIOCHE DE NOEL]",
   "x_16": "x_vars[BRIOCHETTE]",
   "x_17": "x_vars[BROWNIES]",
   "x_18": "x_vars[BUCHE 4PERS]",
   "x_19": "x_vars[BUCHE 6PERS]",
   "x_20": "x_vars[BUCHE 8PERS]",
   "x_21": "x_vars[CAFE OU EAU]",
   "x_22": "x_vars[CAKE]",
   "x_23": "x_vars[CAMPAGNE]",
   "x_24": "x_vars[CARAMEL NOIX]",
   "x_25": "x_vars[CEREAL BAGUETTE]",
   "x_26": "x_vars[CHAUSSON AUX POMMES]",
   "x_27": "x_vars[CHOCOLAT]",
   "x_28": "x_vars[CHOU CHANTILLY]",
   "x_29": "x_vars[COMPLET]",
   "x_30": "x_vars[COOKIE]",
   "x_31": "x_vars[COUPE]",
   "x_32": "x_vars[CROISSANT]",
   "x_33": "x_vars[CROISSANT AMANDES]",
   "x_34": "x_vars[CRUMBLE]",
   "x_35": "x_vars[CRUMBLECARAMEL OU PISTAE]",
   "x_36": "x_vars[DELICETROPICAL]",
   "x_37": "x_vars[DEMI BAGUETTE]",
   "x_38": "x_vars[DEMI PAIN]",
   "x_39": "x_vars[DIVERS BOISSONS]",
   "x_40": "x_vars[DIVERS BOULANGERIE]",
   "x_41": "x_vars[DIVERS CONFISERIE]",
   "x_42": "x_vars[DIVERS PATISSERIE]",
   "x_43": "x_vars[DIVERS SANDWICHS]",
   "x_44": "x_vars[DIVERS VIENNOISERIE]",
   "x_45": "x_vars[DOUCEUR D HIVER]",
   "x_46": "x_vars[ECLAIR]",
   "x_47": "x_vars[ECLAIR FRAISE PISTACHE]",
   "x_48": "x_vars[ENTREMETS]",
   "x_49": "x_vars[FICELLE]",
   "x_50": "x_vars[FINANCIER]",
   "x_51": "x_vars[FINANCIER X5]",
   "x_52": "x_vars[FLAN]",
   "x_53": "x_vars[FLAN ABRICOT]",
   "x_54": "x_vars[FONDANT CHOCOLAT]",
   "x_55": "x_vars[FORMULE PATE]",
   "x_56": "x_vars[FORMULE PLAT PREPARE]",
   "x_57": "x_vars[FORMULE SANDWICH]",
   "x_58": "x_vars[FRAISIER]",
   "x_59": "x_vars[FRAMBOISIER]",
   "x_60": "x_vars[GACHE]",
   "x_61": "x_vars[GAL FRANGIPANE 4P]",
   "x_62": "x_vars[GAL FRANGIPANE 6P]",
   "x_63": "x_vars[GAL POIRE CHOCO 4P]",
   "x_64": "x_vars[GAL POIRE CHOCO 6P]",
   "x_65": "x_vars[GAL POMME 4P]",
   "x_66": "x_vars[GAL POMME 6P]",
   "x_67": "x_vars[GALETTE 8 PERS]",
   "x_68": "x_vars[GD FAR BRETON]",
   "x_69": "x_vars[GD KOUIGN AMANN]",
   "x_70": "x_vars[GD NANTAIS]",
   "x_71": "x_vars[GD PLATEAU SALE]",
   "x_72": "x_vars[GRAND FAR BRETON]",
   "x_73": "x_vars[GRANDE SUCETTE]",
   "x_74": "x_vars[GUERANDAIS]",
   "x_75": "x_vars[KOUIGN AMANN]",
   "x_76": "x_vars[MACARON]",
   "x_77": "x_vars[MERINGUE]",
   "x_78": "x_vars[MILLES FEUILLES]",
   "x_79": "x_vars[MOISSON]",
   "x_80": "x_vars[NANTAIS]",
   "x_81": "x_vars[NID DE POULE]",
   "x_82": "x_vars[NOIX JAPONAISE]",
   "x_83": "x_vars[PAILLE]",
   "x_84": "x_vars[PAIN]",
   "x_85": "x_vars[PAIN AU CHOCOLAT]",
   "x_86": "x_vars[PAIN AUX RAISINS]",
   "x_87": "x_vars[PAIN BANETTE]",
   "x_88": "x_vars[PAIN CHOCO AMANDES]",
   "x_89": "x_vars[PAIN DE MIE]",
   "x_90": "x_vars[PAIN GRAINES]",
   "x_91": "x_vars[PAIN NOIR]",
   "x_92": "x_vars[PAIN S/SEL]",
   "x_93": "x_vars[PAIN SUISSE PEPITO]",
   "x_94": "x_vars[PALET BRETON]",
   "x_95": "x_vars[PALMIER]",
   "x_96": "x_vars[PARIS BREST]",
   "x_97": "x_vars[PATES]",
   "x_98": "x_vars[PLAQUE TARTE 25P]",
   "x_99": "x_vars[PLAT]",
   "x_100": "x_vars[PLAT 6.50E]",
   "x_101": "x_vars[PLAT 7.00]",
   "x_102": "x_vars[PLAT 7.60E]",
   "x_103": "x_vars[PLAT 8.30E]",
   "x_104": "x_vars[PLATPREPARE5,50]",
   "x_105": "x_vars[PLATPREPARE6,00]",
   "x_106": "x_vars[PLATPREPARE6,50]",
   "x_107": "x_vars[PLATPREPARE7,00]",
   "x_108": "x_vars[PT NANTAIS]",
   "x_109": "x_vars[PT PLATEAU SALE]",
   "x_110": "x_vars[QUIM BREAD]",
   "x_111": "x_vars[REDUCTION SUCREES 12]",
   "x_112": "x_vars[REDUCTION SUCREES 24]",
   "x_113": "x_vars[RELIGIEUSE]",
   "x_114": "x_vars[ROYAL]",
   "x_115": "x_vars[ROYAL 4P]",
   "x_116": "x_vars[ROYAL 6P]",
   "x_117": "x_vars[SABLE F  P]",
   "x_118": "x_vars[SACHET DE CROUTON]",
   "x_119": "x_vars[SACHET DE VIENNOISERIE]",
   "x_120": "x_vars[SACHET VIENNOISERIE]",
   "x_121": "x_vars[SAND JB]",
   "x_122": "x_vars[SAND JB EMMENTAL]",
   "x_123": "x_vars[SANDWICH COMPLET]",
   "x_124": "x_vars[SAVARIN]",
   "x_125": "x_vars[SEIGLE]",
   "x_126": "x_vars[SPECIAL BREAD]",
   "x_127": "x_vars[SPECIAL BREAD KG]",
   "x_128": "x_vars[ST HONORE]",
   "x_129": "x_vars[SUCETTE]",
   "x_130": "x_vars[TARTE FINE]",
   "x_131": "x_vars[TARTE FRAISE 4PER]",
   "x_132": "x_vars[TARTE FRAISE 6P]",
   "x_133": "x_vars[TARTE FRUITS 4P]",
   "x_134": "x_vars[TARTE FRUITS 6P]",
   "x_135": "x_vars[TARTELETTE]",
   "x_136": "x_vars[TARTELETTE CHOC]",
   "x_137": "x_vars[TARTELETTE COCKTAIL]",
   "x_138": "x_vars[TARTELETTE FRAISE]",
   "x_139": "x_vars[THE]",
   "x_140": "x_vars[TRADITIONAL BAGUETTE]",
   "x_141": "x_vars[TRAITEUR]",
   "x_142": "x_vars[TRIANGLES]",
   "x_143": "x_vars[TROIS CHOCOLAT]",
   "x_144": "x_vars[TROPEZIENNE]",
   "x_145": "x_vars[TROPEZIENNE FRAMBOISE]",
   "x_146": "x_vars[TULIPE]",
   "x_147": "x_vars[VIENNOISE]",
   "x_148": "x_vars[VIK BREAD]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-035": {
  "source_code_sha256": "aee4d55a7f3b49975dff8ca3d0eab60f722567ed4ce95f7aeb2df74c1d179084",
  "original_solution_sha256": "b077b5e16efbc30347743a558640144bcbe4a1446d5008e3425b98b9088bd16d",
  "bindings": [
   [
    "C0",
    "x_vars[Baguette]"
   ],
   [
    "C1",
    "x_vars[Croissant]"
   ],
   [
    "C2",
    "x_vars[Sourdough]"
   ],
   [
    "C3",
    "x_vars[Rye Bread]"
   ],
   [
    "C4",
    "x_vars[Brioche]"
   ],
   [
    "C5",
    "x_vars[Focaccia]"
   ],
   [
    "C6",
    "x_vars[Ciabatta]"
   ],
   [
    "C7",
    "x_vars[Pita]"
   ],
   [
    "C8",
    "x_vars[Bagel]"
   ],
   [
    "C9",
    "x_vars[English Muffin]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Baguette]",
   "x_1_0": "x_vars[Croissant]",
   "x_2_0": "x_vars[Sourdough]",
   "x_3_0": "x_vars[Rye Bread]",
   "x_4_0": "x_vars[Brioche]",
   "x_5_0": "x_vars[Focaccia]",
   "x_6_0": "x_vars[Ciabatta]",
   "x_7_0": "x_vars[Pita]",
   "x_8_0": "x_vars[Bagel]",
   "x_9_0": "x_vars[English Muffin]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-036": {
  "source_code_sha256": "a82ee0127ed68cbe14fdf12e23bea52c10b5621d4528e66736d54add6bd7ae1e",
  "original_solution_sha256": "c0bd95f1b2dba8a9d9976975ad9d8d84c31a63b4b6a521197d5f32239c9b2ef1",
  "bindings": [
   [
    "C0",
    "x_vars[Sedan]"
   ],
   [
    "C1",
    "x_vars[SUV]"
   ],
   [
    "C2",
    "x_vars[Truck]"
   ],
   [
    "C3",
    "x_vars[Convertible]"
   ],
   [
    "C4",
    "x_vars[Minivan]"
   ],
   [
    "C5",
    "x_vars[Coupe]"
   ],
   [
    "C6",
    "x_vars[Hatchback]"
   ],
   [
    "C7",
    "x_vars[Station Wagon]"
   ],
   [
    "C8",
    "x_vars[Electric Car]"
   ],
   [
    "C9",
    "x_vars[Hybrid Car]"
   ],
   [
    "C10",
    "x_vars[Luxury Sedan]"
   ],
   [
    "C11",
    "x_vars[Sports Car]"
   ],
   [
    "C12",
    "x_vars[Crossover]"
   ],
   [
    "C13",
    "x_vars[Diesel Truck]"
   ],
   [
    "C14",
    "x_vars[Compact SUV]"
   ],
   [
    "C15",
    "x_vars[Luxury SUV]"
   ],
   [
    "C16",
    "x_vars[Cargo Van]"
   ],
   [
    "C17",
    "x_vars[Pickup Truck]"
   ],
   [
    "C18",
    "x_vars[Roadster]"
   ],
   [
    "C19",
    "x_vars[Muscle Car]"
   ],
   [
    "C20",
    "x_vars[Off-road Vehicle]"
   ],
   [
    "C21",
    "x_vars[Camper Van]"
   ],
   [
    "C22",
    "x_vars[Compact Car]"
   ],
   [
    "C23",
    "x_vars[Motorcycle]"
   ],
   [
    "C24",
    "x_vars[Electric SUV]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Sedan]",
   "x_1_0": "x_vars[SUV]",
   "x_2_0": "x_vars[Truck]",
   "x_3_0": "x_vars[Convertible]",
   "x_4_0": "x_vars[Minivan]",
   "x_5_0": "x_vars[Coupe]",
   "x_6_0": "x_vars[Hatchback]",
   "x_7_0": "x_vars[Station Wagon]",
   "x_8_0": "x_vars[Electric Car]",
   "x_9_0": "x_vars[Hybrid Car]",
   "x_10_0": "x_vars[Luxury Sedan]",
   "x_11_0": "x_vars[Sports Car]",
   "x_12_0": "x_vars[Crossover]",
   "x_13_0": "x_vars[Diesel Truck]",
   "x_14_0": "x_vars[Compact SUV]",
   "x_15_0": "x_vars[Luxury SUV]",
   "x_16_0": "x_vars[Cargo Van]",
   "x_17_0": "x_vars[Pickup Truck]",
   "x_18_0": "x_vars[Roadster]",
   "x_19_0": "x_vars[Muscle Car]",
   "x_20_0": "x_vars[Off-road Vehicle]",
   "x_21_0": "x_vars[Camper Van]",
   "x_22_0": "x_vars[Compact Car]",
   "x_23_0": "x_vars[Motorcycle]",
   "x_24_0": "x_vars[Electric SUV]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-037": {
  "source_code_sha256": "112df4885ff519f58365a6d7de154009e6ec45746c5eeeb0c28f588749f6996b",
  "original_solution_sha256": "db6fd2358b41a29d98d26925c158d5d25e4e0bbed638ab55b89fc326eb5c9d58",
  "bindings": [
   [
    "C0",
    "x_vars[Sedan]"
   ],
   [
    "C1",
    "x_vars[SUV]"
   ],
   [
    "C2",
    "x_vars[Truck]"
   ],
   [
    "C3",
    "x_vars[Convertible]"
   ],
   [
    "C4",
    "x_vars[Minivan]"
   ],
   [
    "C5",
    "x_vars[Coupe]"
   ],
   [
    "C6",
    "x_vars[Hatchback]"
   ],
   [
    "C7",
    "x_vars[Station Wagon]"
   ],
   [
    "C8",
    "x_vars[Electric Car]"
   ],
   [
    "C9",
    "x_vars[Hybrid Car]"
   ],
   [
    "C10",
    "x_vars[Luxury Sedan]"
   ],
   [
    "C11",
    "x_vars[Sports Car]"
   ],
   [
    "C12",
    "x_vars[Crossover]"
   ],
   [
    "C13",
    "x_vars[Diesel Truck]"
   ],
   [
    "C14",
    "x_vars[Compact SUV]"
   ],
   [
    "C15",
    "x_vars[Luxury SUV]"
   ],
   [
    "C16",
    "x_vars[Cargo Van]"
   ],
   [
    "C17",
    "x_vars[Pickup Truck]"
   ],
   [
    "C18",
    "x_vars[Roadster]"
   ],
   [
    "C19",
    "x_vars[Muscle Car]"
   ],
   [
    "C20",
    "x_vars[Off-road Vehicle]"
   ],
   [
    "C21",
    "x_vars[Camper Van]"
   ],
   [
    "C22",
    "x_vars[Compact Car]"
   ],
   [
    "C23",
    "x_vars[Motorcycle]"
   ],
   [
    "C24",
    "x_vars[Electric SUV]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Sedan]",
   "x_1_0": "x_vars[SUV]",
   "x_2_0": "x_vars[Truck]",
   "x_3_0": "x_vars[Convertible]",
   "x_4_0": "x_vars[Minivan]",
   "x_5_0": "x_vars[Coupe]",
   "x_6_0": "x_vars[Hatchback]",
   "x_7_0": "x_vars[Station Wagon]",
   "x_8_0": "x_vars[Electric Car]",
   "x_9_0": "x_vars[Hybrid Car]",
   "x_10_0": "x_vars[Luxury Sedan]",
   "x_11_0": "x_vars[Sports Car]",
   "x_12_0": "x_vars[Crossover]",
   "x_13_0": "x_vars[Diesel Truck]",
   "x_14_0": "x_vars[Compact SUV]",
   "x_15_0": "x_vars[Luxury SUV]",
   "x_16_0": "x_vars[Cargo Van]",
   "x_17_0": "x_vars[Pickup Truck]",
   "x_18_0": "x_vars[Roadster]",
   "x_19_0": "x_vars[Muscle Car]",
   "x_20_0": "x_vars[Off-road Vehicle]",
   "x_21_0": "x_vars[Camper Van]",
   "x_22_0": "x_vars[Compact Car]",
   "x_23_0": "x_vars[Motorcycle]",
   "x_24_0": "x_vars[Electric SUV]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-038": {
  "source_code_sha256": "7c13d10148469ccc42da39330f2c35c5260fcffff4894a4d520b47f3a3da2e55",
  "original_solution_sha256": "d91e2401f20bd532a42c6b47d15b2d22a6d2483cc2165324083a670901940969",
  "bindings": [
   [
    "C0",
    "x_vars[1]"
   ],
   [
    "C1",
    "x_vars[2]"
   ],
   [
    "C2",
    "x_vars[3]"
   ],
   [
    "C3",
    "x_vars[4]"
   ],
   [
    "C4",
    "x_vars[5]"
   ],
   [
    "C5",
    "x_vars[6]"
   ],
   [
    "C6",
    "x_vars[7]"
   ],
   [
    "C7",
    "x_vars[8]"
   ],
   [
    "C8",
    "x_vars[9]"
   ],
   [
    "C9",
    "x_vars[10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1]",
   "x_1_0": "x_vars[2]",
   "x_2_0": "x_vars[3]",
   "x_3_0": "x_vars[4]",
   "x_4_0": "x_vars[5]",
   "x_5_0": "x_vars[6]",
   "x_6_0": "x_vars[7]",
   "x_7_0": "x_vars[8]",
   "x_8_0": "x_vars[9]",
   "x_9_0": "x_vars[10]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-039": {
  "source_code_sha256": "e92f9f1e92bd013597931ecd189946723d6d1526fa75a78b67f94d75c43283e1",
  "original_solution_sha256": "2c3b16c7c15bb291e2d1947e015a04a3ad7555fd579ee2bff5f046c27b39fc81",
  "bindings": [
   [
    "C0",
    "x_vars[Sedans,Warehouse 1]"
   ],
   [
    "C1",
    "x_vars[Sedans,Warehouse 2]"
   ],
   [
    "C2",
    "x_vars[Sedans,Warehouse 3]"
   ],
   [
    "C3",
    "x_vars[Sedans,Warehouse 4]"
   ],
   [
    "C4",
    "x_vars[Sedans,Warehouse 5]"
   ],
   [
    "C5",
    "x_vars[Sedans,Warehouse 6]"
   ],
   [
    "C6",
    "x_vars[Sedans,Warehouse 7]"
   ],
   [
    "C7",
    "x_vars[Sedans,Warehouse 8]"
   ],
   [
    "C8",
    "x_vars[Sedans,Warehouse 9]"
   ],
   [
    "C9",
    "x_vars[Sedans,Warehouse 10]"
   ],
   [
    "C10",
    "x_vars[SUVs,Warehouse 1]"
   ],
   [
    "C11",
    "x_vars[SUVs,Warehouse 2]"
   ],
   [
    "C12",
    "x_vars[SUVs,Warehouse 3]"
   ],
   [
    "C13",
    "x_vars[SUVs,Warehouse 4]"
   ],
   [
    "C14",
    "x_vars[SUVs,Warehouse 5]"
   ],
   [
    "C15",
    "x_vars[SUVs,Warehouse 6]"
   ],
   [
    "C16",
    "x_vars[SUVs,Warehouse 7]"
   ],
   [
    "C17",
    "x_vars[SUVs,Warehouse 8]"
   ],
   [
    "C18",
    "x_vars[SUVs,Warehouse 9]"
   ],
   [
    "C19",
    "x_vars[SUVs,Warehouse 10]"
   ],
   [
    "C20",
    "x_vars[Electric Vehicles,Warehouse 1]"
   ],
   [
    "C21",
    "x_vars[Electric Vehicles,Warehouse 2]"
   ],
   [
    "C22",
    "x_vars[Electric Vehicles,Warehouse 3]"
   ],
   [
    "C23",
    "x_vars[Electric Vehicles,Warehouse 4]"
   ],
   [
    "C24",
    "x_vars[Electric Vehicles,Warehouse 5]"
   ],
   [
    "C25",
    "x_vars[Electric Vehicles,Warehouse 6]"
   ],
   [
    "C26",
    "x_vars[Electric Vehicles,Warehouse 7]"
   ],
   [
    "C27",
    "x_vars[Electric Vehicles,Warehouse 8]"
   ],
   [
    "C28",
    "x_vars[Electric Vehicles,Warehouse 9]"
   ],
   [
    "C29",
    "x_vars[Electric Vehicles,Warehouse 10]"
   ],
   [
    "C30",
    "x_vars[Hybrid Vehicles,Warehouse 1]"
   ],
   [
    "C31",
    "x_vars[Hybrid Vehicles,Warehouse 2]"
   ],
   [
    "C32",
    "x_vars[Hybrid Vehicles,Warehouse 3]"
   ],
   [
    "C33",
    "x_vars[Hybrid Vehicles,Warehouse 4]"
   ],
   [
    "C34",
    "x_vars[Hybrid Vehicles,Warehouse 5]"
   ],
   [
    "C35",
    "x_vars[Hybrid Vehicles,Warehouse 6]"
   ],
   [
    "C36",
    "x_vars[Hybrid Vehicles,Warehouse 7]"
   ],
   [
    "C37",
    "x_vars[Hybrid Vehicles,Warehouse 8]"
   ],
   [
    "C38",
    "x_vars[Hybrid Vehicles,Warehouse 9]"
   ],
   [
    "C39",
    "x_vars[Hybrid Vehicles,Warehouse 10]"
   ],
   [
    "C40",
    "x_vars[Trucks,Warehouse 1]"
   ],
   [
    "C41",
    "x_vars[Trucks,Warehouse 2]"
   ],
   [
    "C42",
    "x_vars[Trucks,Warehouse 3]"
   ],
   [
    "C43",
    "x_vars[Trucks,Warehouse 4]"
   ],
   [
    "C44",
    "x_vars[Trucks,Warehouse 5]"
   ],
   [
    "C45",
    "x_vars[Trucks,Warehouse 6]"
   ],
   [
    "C46",
    "x_vars[Trucks,Warehouse 7]"
   ],
   [
    "C47",
    "x_vars[Trucks,Warehouse 8]"
   ],
   [
    "C48",
    "x_vars[Trucks,Warehouse 9]"
   ],
   [
    "C49",
    "x_vars[Trucks,Warehouse 10]"
   ],
   [
    "C50",
    "x_vars[Sports Cars,Warehouse 1]"
   ],
   [
    "C51",
    "x_vars[Sports Cars,Warehouse 2]"
   ],
   [
    "C52",
    "x_vars[Sports Cars,Warehouse 3]"
   ],
   [
    "C53",
    "x_vars[Sports Cars,Warehouse 4]"
   ],
   [
    "C54",
    "x_vars[Sports Cars,Warehouse 5]"
   ],
   [
    "C55",
    "x_vars[Sports Cars,Warehouse 6]"
   ],
   [
    "C56",
    "x_vars[Sports Cars,Warehouse 7]"
   ],
   [
    "C57",
    "x_vars[Sports Cars,Warehouse 8]"
   ],
   [
    "C58",
    "x_vars[Sports Cars,Warehouse 9]"
   ],
   [
    "C59",
    "x_vars[Sports Cars,Warehouse 10]"
   ],
   [
    "C60",
    "x_vars[Compact Cars,Warehouse 1]"
   ],
   [
    "C61",
    "x_vars[Compact Cars,Warehouse 2]"
   ],
   [
    "C62",
    "x_vars[Compact Cars,Warehouse 3]"
   ],
   [
    "C63",
    "x_vars[Compact Cars,Warehouse 4]"
   ],
   [
    "C64",
    "x_vars[Compact Cars,Warehouse 5]"
   ],
   [
    "C65",
    "x_vars[Compact Cars,Warehouse 6]"
   ],
   [
    "C66",
    "x_vars[Compact Cars,Warehouse 7]"
   ],
   [
    "C67",
    "x_vars[Compact Cars,Warehouse 8]"
   ],
   [
    "C68",
    "x_vars[Compact Cars,Warehouse 9]"
   ],
   [
    "C69",
    "x_vars[Compact Cars,Warehouse 10]"
   ],
   [
    "C70",
    "x_vars[Luxury Sedans,Warehouse 1]"
   ],
   [
    "C71",
    "x_vars[Luxury Sedans,Warehouse 2]"
   ],
   [
    "C72",
    "x_vars[Luxury Sedans,Warehouse 3]"
   ],
   [
    "C73",
    "x_vars[Luxury Sedans,Warehouse 4]"
   ],
   [
    "C74",
    "x_vars[Luxury Sedans,Warehouse 5]"
   ],
   [
    "C75",
    "x_vars[Luxury Sedans,Warehouse 6]"
   ],
   [
    "C76",
    "x_vars[Luxury Sedans,Warehouse 7]"
   ],
   [
    "C77",
    "x_vars[Luxury Sedans,Warehouse 8]"
   ],
   [
    "C78",
    "x_vars[Luxury Sedans,Warehouse 9]"
   ],
   [
    "C79",
    "x_vars[Luxury Sedans,Warehouse 10]"
   ],
   [
    "C80",
    "x_vars[Vans,Warehouse 1]"
   ],
   [
    "C81",
    "x_vars[Vans,Warehouse 2]"
   ],
   [
    "C82",
    "x_vars[Vans,Warehouse 3]"
   ],
   [
    "C83",
    "x_vars[Vans,Warehouse 4]"
   ],
   [
    "C84",
    "x_vars[Vans,Warehouse 5]"
   ],
   [
    "C85",
    "x_vars[Vans,Warehouse 6]"
   ],
   [
    "C86",
    "x_vars[Vans,Warehouse 7]"
   ],
   [
    "C87",
    "x_vars[Vans,Warehouse 8]"
   ],
   [
    "C88",
    "x_vars[Vans,Warehouse 9]"
   ],
   [
    "C89",
    "x_vars[Vans,Warehouse 10]"
   ],
   [
    "C90",
    "x_vars[Pickup Trucks,Warehouse 1]"
   ],
   [
    "C91",
    "x_vars[Pickup Trucks,Warehouse 2]"
   ],
   [
    "C92",
    "x_vars[Pickup Trucks,Warehouse 3]"
   ],
   [
    "C93",
    "x_vars[Pickup Trucks,Warehouse 4]"
   ],
   [
    "C94",
    "x_vars[Pickup Trucks,Warehouse 5]"
   ],
   [
    "C95",
    "x_vars[Pickup Trucks,Warehouse 6]"
   ],
   [
    "C96",
    "x_vars[Pickup Trucks,Warehouse 7]"
   ],
   [
    "C97",
    "x_vars[Pickup Trucks,Warehouse 8]"
   ],
   [
    "C98",
    "x_vars[Pickup Trucks,Warehouse 9]"
   ],
   [
    "C99",
    "x_vars[Pickup Trucks,Warehouse 10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Sedans,Warehouse 1]",
   "x_0_1": "x_vars[Sedans,Warehouse 2]",
   "x_0_2": "x_vars[Sedans,Warehouse 3]",
   "x_0_3": "x_vars[Sedans,Warehouse 4]",
   "x_0_4": "x_vars[Sedans,Warehouse 5]",
   "x_0_5": "x_vars[Sedans,Warehouse 6]",
   "x_0_6": "x_vars[Sedans,Warehouse 7]",
   "x_0_7": "x_vars[Sedans,Warehouse 8]",
   "x_0_8": "x_vars[Sedans,Warehouse 9]",
   "x_0_9": "x_vars[Sedans,Warehouse 10]",
   "x_1_0": "x_vars[SUVs,Warehouse 1]",
   "x_1_1": "x_vars[SUVs,Warehouse 2]",
   "x_1_2": "x_vars[SUVs,Warehouse 3]",
   "x_1_3": "x_vars[SUVs,Warehouse 4]",
   "x_1_4": "x_vars[SUVs,Warehouse 5]",
   "x_1_5": "x_vars[SUVs,Warehouse 6]",
   "x_1_6": "x_vars[SUVs,Warehouse 7]",
   "x_1_7": "x_vars[SUVs,Warehouse 8]",
   "x_1_8": "x_vars[SUVs,Warehouse 9]",
   "x_1_9": "x_vars[SUVs,Warehouse 10]",
   "x_2_0": "x_vars[Electric Vehicles,Warehouse 1]",
   "x_2_1": "x_vars[Electric Vehicles,Warehouse 2]",
   "x_2_2": "x_vars[Electric Vehicles,Warehouse 3]",
   "x_2_3": "x_vars[Electric Vehicles,Warehouse 4]",
   "x_2_4": "x_vars[Electric Vehicles,Warehouse 5]",
   "x_2_5": "x_vars[Electric Vehicles,Warehouse 6]",
   "x_2_6": "x_vars[Electric Vehicles,Warehouse 7]",
   "x_2_7": "x_vars[Electric Vehicles,Warehouse 8]",
   "x_2_8": "x_vars[Electric Vehicles,Warehouse 9]",
   "x_2_9": "x_vars[Electric Vehicles,Warehouse 10]",
   "x_3_0": "x_vars[Hybrid Vehicles,Warehouse 1]",
   "x_3_1": "x_vars[Hybrid Vehicles,Warehouse 2]",
   "x_3_2": "x_vars[Hybrid Vehicles,Warehouse 3]",
   "x_3_3": "x_vars[Hybrid Vehicles,Warehouse 4]",
   "x_3_4": "x_vars[Hybrid Vehicles,Warehouse 5]",
   "x_3_5": "x_vars[Hybrid Vehicles,Warehouse 6]",
   "x_3_6": "x_vars[Hybrid Vehicles,Warehouse 7]",
   "x_3_7": "x_vars[Hybrid Vehicles,Warehouse 8]",
   "x_3_8": "x_vars[Hybrid Vehicles,Warehouse 9]",
   "x_3_9": "x_vars[Hybrid Vehicles,Warehouse 10]",
   "x_4_0": "x_vars[Trucks,Warehouse 1]",
   "x_4_1": "x_vars[Trucks,Warehouse 2]",
   "x_4_2": "x_vars[Trucks,Warehouse 3]",
   "x_4_3": "x_vars[Trucks,Warehouse 4]",
   "x_4_4": "x_vars[Trucks,Warehouse 5]",
   "x_4_5": "x_vars[Trucks,Warehouse 6]",
   "x_4_6": "x_vars[Trucks,Warehouse 7]",
   "x_4_7": "x_vars[Trucks,Warehouse 8]",
   "x_4_8": "x_vars[Trucks,Warehouse 9]",
   "x_4_9": "x_vars[Trucks,Warehouse 10]",
   "x_5_0": "x_vars[Sports Cars,Warehouse 1]",
   "x_5_1": "x_vars[Sports Cars,Warehouse 2]",
   "x_5_2": "x_vars[Sports Cars,Warehouse 3]",
   "x_5_3": "x_vars[Sports Cars,Warehouse 4]",
   "x_5_4": "x_vars[Sports Cars,Warehouse 5]",
   "x_5_5": "x_vars[Sports Cars,Warehouse 6]",
   "x_5_6": "x_vars[Sports Cars,Warehouse 7]",
   "x_5_7": "x_vars[Sports Cars,Warehouse 8]",
   "x_5_8": "x_vars[Sports Cars,Warehouse 9]",
   "x_5_9": "x_vars[Sports Cars,Warehouse 10]",
   "x_6_0": "x_vars[Compact Cars,Warehouse 1]",
   "x_6_1": "x_vars[Compact Cars,Warehouse 2]",
   "x_6_2": "x_vars[Compact Cars,Warehouse 3]",
   "x_6_3": "x_vars[Compact Cars,Warehouse 4]",
   "x_6_4": "x_vars[Compact Cars,Warehouse 5]",
   "x_6_5": "x_vars[Compact Cars,Warehouse 6]",
   "x_6_6": "x_vars[Compact Cars,Warehouse 7]",
   "x_6_7": "x_vars[Compact Cars,Warehouse 8]",
   "x_6_8": "x_vars[Compact Cars,Warehouse 9]",
   "x_6_9": "x_vars[Compact Cars,Warehouse 10]",
   "x_7_0": "x_vars[Luxury Sedans,Warehouse 1]",
   "x_7_1": "x_vars[Luxury Sedans,Warehouse 2]",
   "x_7_2": "x_vars[Luxury Sedans,Warehouse 3]",
   "x_7_3": "x_vars[Luxury Sedans,Warehouse 4]",
   "x_7_4": "x_vars[Luxury Sedans,Warehouse 5]",
   "x_7_5": "x_vars[Luxury Sedans,Warehouse 6]",
   "x_7_6": "x_vars[Luxury Sedans,Warehouse 7]",
   "x_7_7": "x_vars[Luxury Sedans,Warehouse 8]",
   "x_7_8": "x_vars[Luxury Sedans,Warehouse 9]",
   "x_7_9": "x_vars[Luxury Sedans,Warehouse 10]",
   "x_8_0": "x_vars[Vans,Warehouse 1]",
   "x_8_1": "x_vars[Vans,Warehouse 2]",
   "x_8_2": "x_vars[Vans,Warehouse 3]",
   "x_8_3": "x_vars[Vans,Warehouse 4]",
   "x_8_4": "x_vars[Vans,Warehouse 5]",
   "x_8_5": "x_vars[Vans,Warehouse 6]",
   "x_8_6": "x_vars[Vans,Warehouse 7]",
   "x_8_7": "x_vars[Vans,Warehouse 8]",
   "x_8_8": "x_vars[Vans,Warehouse 9]",
   "x_8_9": "x_vars[Vans,Warehouse 10]",
   "x_9_0": "x_vars[Pickup Trucks,Warehouse 1]",
   "x_9_1": "x_vars[Pickup Trucks,Warehouse 2]",
   "x_9_2": "x_vars[Pickup Trucks,Warehouse 3]",
   "x_9_3": "x_vars[Pickup Trucks,Warehouse 4]",
   "x_9_4": "x_vars[Pickup Trucks,Warehouse 5]",
   "x_9_5": "x_vars[Pickup Trucks,Warehouse 6]",
   "x_9_6": "x_vars[Pickup Trucks,Warehouse 7]",
   "x_9_7": "x_vars[Pickup Trucks,Warehouse 8]",
   "x_9_8": "x_vars[Pickup Trucks,Warehouse 9]",
   "x_9_9": "x_vars[Pickup Trucks,Warehouse 10]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-040": {
  "source_code_sha256": "c5158d1378d0e7f096f0e260988db03b81d5dddc0e24d7af935b9be37668b956",
  "original_solution_sha256": "8e2d8c41e214b874c850caf35709ac7f67d1362246b12fd810ddc360af914459",
  "bindings": [
   [
    "C0",
    "x_vars[Queens]"
   ],
   [
    "C1",
    "x_vars[Brooklyn]"
   ],
   [
    "C2",
    "x_vars[Manhattan]"
   ],
   [
    "C3",
    "x_vars[Bronx]"
   ],
   [
    "C4",
    "x_vars[Staten Island]"
   ],
   [
    "C5",
    "x_vars[Harlem]"
   ],
   [
    "C6",
    "x_vars[Upper East Side]"
   ],
   [
    "C7",
    "x_vars[Lower Manhattan]"
   ],
   [
    "C8",
    "x_vars[Midtown]"
   ],
   [
    "C9",
    "x_vars[Long Island City]"
   ],
   [
    "C10",
    "x_vars[Williamsburg]"
   ],
   [
    "C11",
    "x_vars[Bushwick]"
   ],
   [
    "C12",
    "x_vars[Flatbush]"
   ],
   [
    "C13",
    "x_vars[Greenpoint]"
   ],
   [
    "C14",
    "x_vars[Park Slope]"
   ],
   [
    "C15",
    "x_vars[Astoria]"
   ],
   [
    "C16",
    "x_vars[Jackson Heights]"
   ],
   [
    "C17",
    "x_vars[Flushing]"
   ],
   [
    "C18",
    "x_vars[Sunnyside]"
   ],
   [
    "C19",
    "x_vars[Ditmars]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Queens]",
   "x_1_0": "x_vars[Brooklyn]",
   "x_2_0": "x_vars[Manhattan]",
   "x_3_0": "x_vars[Bronx]",
   "x_4_0": "x_vars[Staten Island]",
   "x_5_0": "x_vars[Harlem]",
   "x_6_0": "x_vars[Upper East Side]",
   "x_7_0": "x_vars[Lower Manhattan]",
   "x_8_0": "x_vars[Midtown]",
   "x_9_0": "x_vars[Long Island City]",
   "x_10_0": "x_vars[Williamsburg]",
   "x_11_0": "x_vars[Bushwick]",
   "x_12_0": "x_vars[Flatbush]",
   "x_13_0": "x_vars[Greenpoint]",
   "x_14_0": "x_vars[Park Slope]",
   "x_15_0": "x_vars[Astoria]",
   "x_16_0": "x_vars[Jackson Heights]",
   "x_17_0": "x_vars[Flushing]",
   "x_18_0": "x_vars[Sunnyside]",
   "x_19_0": "x_vars[Ditmars]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-041": {
  "source_code_sha256": "9db3cd4d0597b2e36d2a2b8126a28d64151a48b635c03cd366a770d49077eb38",
  "original_solution_sha256": "a4bdf58012c54ed1aaf2710641de7d7d7ee530d669cb76c72c6dcc3685f5bd85",
  "bindings": [
   [
    "C0",
    "x_vars[Queens]"
   ],
   [
    "C1",
    "x_vars[Brooklyn]"
   ],
   [
    "C2",
    "x_vars[Manhattan]"
   ],
   [
    "C3",
    "x_vars[Bronx]"
   ],
   [
    "C4",
    "x_vars[Staten Island]"
   ],
   [
    "C5",
    "x_vars[Harlem]"
   ],
   [
    "C6",
    "x_vars[Upper East Side]"
   ],
   [
    "C7",
    "x_vars[Lower Manhattan]"
   ],
   [
    "C8",
    "x_vars[Midtown]"
   ],
   [
    "C9",
    "x_vars[Long Island City]"
   ],
   [
    "C10",
    "x_vars[Williamsburg]"
   ],
   [
    "C11",
    "x_vars[Bushwick]"
   ],
   [
    "C12",
    "x_vars[Flatbush]"
   ],
   [
    "C13",
    "x_vars[Greenpoint]"
   ],
   [
    "C14",
    "x_vars[Park Slope]"
   ],
   [
    "C15",
    "x_vars[Astoria]"
   ],
   [
    "C16",
    "x_vars[Jackson Heights]"
   ],
   [
    "C17",
    "x_vars[Flushing]"
   ],
   [
    "C18",
    "x_vars[Sunnyside]"
   ],
   [
    "C19",
    "x_vars[Ditmars]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Queens]",
   "x_1_0": "x_vars[Brooklyn]",
   "x_2_0": "x_vars[Manhattan]",
   "x_3_0": "x_vars[Bronx]",
   "x_4_0": "x_vars[Staten Island]",
   "x_5_0": "x_vars[Harlem]",
   "x_6_0": "x_vars[Upper East Side]",
   "x_7_0": "x_vars[Lower Manhattan]",
   "x_8_0": "x_vars[Midtown]",
   "x_9_0": "x_vars[Long Island City]",
   "x_10_0": "x_vars[Williamsburg]",
   "x_11_0": "x_vars[Bushwick]",
   "x_12_0": "x_vars[Flatbush]",
   "x_13_0": "x_vars[Greenpoint]",
   "x_14_0": "x_vars[Park Slope]",
   "x_15_0": "x_vars[Astoria]",
   "x_16_0": "x_vars[Jackson Heights]",
   "x_17_0": "x_vars[Flushing]",
   "x_18_0": "x_vars[Sunnyside]",
   "x_19_0": "x_vars[Ditmars]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-042": {
  "source_code_sha256": "b4c6064022ccb6dffb9bd270deefffc92e3def4d2d2a0204fad44d0bc5ad7285",
  "original_solution_sha256": "e1410690e75d6bd907ab2e270b0c40b33929ecd14066b4710cc1031e9e6ecbc6",
  "bindings": [
   [
    "C0",
    "x_vars[NSAIDs]"
   ],
   [
    "C1",
    "x_vars[Antirheumatic Drugs]"
   ],
   [
    "C2",
    "x_vars[Acetic Acid Derivatives]"
   ],
   [
    "C3",
    "x_vars[Antibiotics]"
   ],
   [
    "C4",
    "x_vars[Antiviral Drugs]"
   ],
   [
    "C5",
    "x_vars[Antifungal Agents]"
   ],
   [
    "C6",
    "x_vars[Antidepressants]"
   ],
   [
    "C7",
    "x_vars[Antipsychotics]"
   ],
   [
    "C8",
    "x_vars[Antihistamines]"
   ],
   [
    "C9",
    "x_vars[Corticosteroids]"
   ],
   [
    "C10",
    "x_vars[Beta Blockers]"
   ],
   [
    "C11",
    "x_vars[Calcium Channel Blockers]"
   ],
   [
    "C12",
    "x_vars[ACE Inhibitors]"
   ],
   [
    "C13",
    "x_vars[Angiotensin II Receptor Blockers]"
   ],
   [
    "C14",
    "x_vars[Diuretics]"
   ],
   [
    "C15",
    "x_vars[Statins]"
   ],
   [
    "C16",
    "x_vars[Insulin]"
   ],
   [
    "C17",
    "x_vars[Anticoagulants]"
   ],
   [
    "C18",
    "x_vars[Antiepileptic Drugs]"
   ],
   [
    "C19",
    "x_vars[Antiemetics]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[NSAIDs]",
   "x_1_0": "x_vars[Antirheumatic Drugs]",
   "x_2_0": "x_vars[Acetic Acid Derivatives]",
   "x_3_0": "x_vars[Antibiotics]",
   "x_4_0": "x_vars[Antiviral Drugs]",
   "x_5_0": "x_vars[Antifungal Agents]",
   "x_6_0": "x_vars[Antidepressants]",
   "x_7_0": "x_vars[Antipsychotics]",
   "x_8_0": "x_vars[Antihistamines]",
   "x_9_0": "x_vars[Corticosteroids]",
   "x_10_0": "x_vars[Beta Blockers]",
   "x_11_0": "x_vars[Calcium Channel Blockers]",
   "x_12_0": "x_vars[ACE Inhibitors]",
   "x_13_0": "x_vars[Angiotensin II Receptor Blockers]",
   "x_14_0": "x_vars[Diuretics]",
   "x_15_0": "x_vars[Statins]",
   "x_16_0": "x_vars[Insulin]",
   "x_17_0": "x_vars[Anticoagulants]",
   "x_18_0": "x_vars[Antiepileptic Drugs]",
   "x_19_0": "x_vars[Antiemetics]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-043": {
  "source_code_sha256": "3f7c887823006aadc575bb8bcd88f3b49c9591c23f489def73912dca599e5ff1",
  "original_solution_sha256": "8c88ac0a4b8930b744f571f5e2f1727cc706e816f62bc63416286f852c99225d",
  "bindings": [
   [
    "C0",
    "x_vars[NSAIDs]"
   ],
   [
    "C1",
    "x_vars[Antirheumatic Drugs]"
   ],
   [
    "C2",
    "x_vars[Acetic Acid Derivatives]"
   ],
   [
    "C3",
    "x_vars[Antibiotics]"
   ],
   [
    "C4",
    "x_vars[Antiviral Drugs]"
   ],
   [
    "C5",
    "x_vars[Antifungal Agents]"
   ],
   [
    "C6",
    "x_vars[Antidepressants]"
   ],
   [
    "C7",
    "x_vars[Antipsychotics]"
   ],
   [
    "C8",
    "x_vars[Antihistamines]"
   ],
   [
    "C9",
    "x_vars[Corticosteroids]"
   ],
   [
    "C10",
    "x_vars[Beta Blockers]"
   ],
   [
    "C11",
    "x_vars[Calcium Channel Blockers]"
   ],
   [
    "C12",
    "x_vars[ACE Inhibitors]"
   ],
   [
    "C13",
    "x_vars[Angiotensin II Receptor Blockers]"
   ],
   [
    "C14",
    "x_vars[Diuretics]"
   ],
   [
    "C15",
    "x_vars[Statins]"
   ],
   [
    "C16",
    "x_vars[Insulin]"
   ],
   [
    "C17",
    "x_vars[Anticoagulants]"
   ],
   [
    "C18",
    "x_vars[Antiepileptic Drugs]"
   ],
   [
    "C19",
    "x_vars[Antiemetics]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[NSAIDs]",
   "x_1_0": "x_vars[Antirheumatic Drugs]",
   "x_2_0": "x_vars[Acetic Acid Derivatives]",
   "x_3_0": "x_vars[Antibiotics]",
   "x_4_0": "x_vars[Antiviral Drugs]",
   "x_5_0": "x_vars[Antifungal Agents]",
   "x_6_0": "x_vars[Antidepressants]",
   "x_7_0": "x_vars[Antipsychotics]",
   "x_8_0": "x_vars[Antihistamines]",
   "x_9_0": "x_vars[Corticosteroids]",
   "x_10_0": "x_vars[Beta Blockers]",
   "x_11_0": "x_vars[Calcium Channel Blockers]",
   "x_12_0": "x_vars[ACE Inhibitors]",
   "x_13_0": "x_vars[Angiotensin II Receptor Blockers]",
   "x_14_0": "x_vars[Diuretics]",
   "x_15_0": "x_vars[Statins]",
   "x_16_0": "x_vars[Insulin]",
   "x_17_0": "x_vars[Anticoagulants]",
   "x_18_0": "x_vars[Antiepileptic Drugs]",
   "x_19_0": "x_vars[Antiemetics]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-044": {
  "source_code_sha256": "31e6743b9f2362456c36a284c1a18b0a2579940fb7a33fa924914c6adbac722a",
  "original_solution_sha256": "e6dd56492f5eda5827745e102076a2166e1d9b1d6f5fceca0c0297bac094bac9",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[1,4]"
   ],
   [
    "C4",
    "x_vars[1,5]"
   ],
   [
    "C5",
    "x_vars[1,6]"
   ],
   [
    "C6",
    "x_vars[1,7]"
   ],
   [
    "C7",
    "x_vars[1,8]"
   ],
   [
    "C8",
    "x_vars[1,9]"
   ],
   [
    "C9",
    "x_vars[1,10]"
   ],
   [
    "C10",
    "x_vars[2,1]"
   ],
   [
    "C11",
    "x_vars[2,2]"
   ],
   [
    "C12",
    "x_vars[2,3]"
   ],
   [
    "C13",
    "x_vars[2,4]"
   ],
   [
    "C14",
    "x_vars[2,5]"
   ],
   [
    "C15",
    "x_vars[2,6]"
   ],
   [
    "C16",
    "x_vars[2,7]"
   ],
   [
    "C17",
    "x_vars[2,8]"
   ],
   [
    "C18",
    "x_vars[2,9]"
   ],
   [
    "C19",
    "x_vars[2,10]"
   ],
   [
    "C20",
    "x_vars[3,1]"
   ],
   [
    "C21",
    "x_vars[3,2]"
   ],
   [
    "C22",
    "x_vars[3,3]"
   ],
   [
    "C23",
    "x_vars[3,4]"
   ],
   [
    "C24",
    "x_vars[3,5]"
   ],
   [
    "C25",
    "x_vars[3,6]"
   ],
   [
    "C26",
    "x_vars[3,7]"
   ],
   [
    "C27",
    "x_vars[3,8]"
   ],
   [
    "C28",
    "x_vars[3,9]"
   ],
   [
    "C29",
    "x_vars[3,10]"
   ],
   [
    "C30",
    "x_vars[4,1]"
   ],
   [
    "C31",
    "x_vars[4,2]"
   ],
   [
    "C32",
    "x_vars[4,3]"
   ],
   [
    "C33",
    "x_vars[4,4]"
   ],
   [
    "C34",
    "x_vars[4,5]"
   ],
   [
    "C35",
    "x_vars[4,6]"
   ],
   [
    "C36",
    "x_vars[4,7]"
   ],
   [
    "C37",
    "x_vars[4,8]"
   ],
   [
    "C38",
    "x_vars[4,9]"
   ],
   [
    "C39",
    "x_vars[4,10]"
   ],
   [
    "C40",
    "x_vars[5,1]"
   ],
   [
    "C41",
    "x_vars[5,2]"
   ],
   [
    "C42",
    "x_vars[5,3]"
   ],
   [
    "C43",
    "x_vars[5,4]"
   ],
   [
    "C44",
    "x_vars[5,5]"
   ],
   [
    "C45",
    "x_vars[5,6]"
   ],
   [
    "C46",
    "x_vars[5,7]"
   ],
   [
    "C47",
    "x_vars[5,8]"
   ],
   [
    "C48",
    "x_vars[5,9]"
   ],
   [
    "C49",
    "x_vars[5,10]"
   ],
   [
    "C50",
    "x_vars[6,1]"
   ],
   [
    "C51",
    "x_vars[6,2]"
   ],
   [
    "C52",
    "x_vars[6,3]"
   ],
   [
    "C53",
    "x_vars[6,4]"
   ],
   [
    "C54",
    "x_vars[6,5]"
   ],
   [
    "C55",
    "x_vars[6,6]"
   ],
   [
    "C56",
    "x_vars[6,7]"
   ],
   [
    "C57",
    "x_vars[6,8]"
   ],
   [
    "C58",
    "x_vars[6,9]"
   ],
   [
    "C59",
    "x_vars[6,10]"
   ],
   [
    "C60",
    "x_vars[7,1]"
   ],
   [
    "C61",
    "x_vars[7,2]"
   ],
   [
    "C62",
    "x_vars[7,3]"
   ],
   [
    "C63",
    "x_vars[7,4]"
   ],
   [
    "C64",
    "x_vars[7,5]"
   ],
   [
    "C65",
    "x_vars[7,6]"
   ],
   [
    "C66",
    "x_vars[7,7]"
   ],
   [
    "C67",
    "x_vars[7,8]"
   ],
   [
    "C68",
    "x_vars[7,9]"
   ],
   [
    "C69",
    "x_vars[7,10]"
   ],
   [
    "C70",
    "x_vars[8,1]"
   ],
   [
    "C71",
    "x_vars[8,2]"
   ],
   [
    "C72",
    "x_vars[8,3]"
   ],
   [
    "C73",
    "x_vars[8,4]"
   ],
   [
    "C74",
    "x_vars[8,5]"
   ],
   [
    "C75",
    "x_vars[8,6]"
   ],
   [
    "C76",
    "x_vars[8,7]"
   ],
   [
    "C77",
    "x_vars[8,8]"
   ],
   [
    "C78",
    "x_vars[8,9]"
   ],
   [
    "C79",
    "x_vars[8,10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,1]",
   "x_0_1": "x_vars[2,1]",
   "x_0_2": "x_vars[3,1]",
   "x_0_3": "x_vars[4,1]",
   "x_0_4": "x_vars[5,1]",
   "x_0_5": "x_vars[6,1]",
   "x_0_6": "x_vars[7,1]",
   "x_0_7": "x_vars[8,1]",
   "x_1_0": "x_vars[1,2]",
   "x_1_1": "x_vars[2,2]",
   "x_1_2": "x_vars[3,2]",
   "x_1_3": "x_vars[4,2]",
   "x_1_4": "x_vars[5,2]",
   "x_1_5": "x_vars[6,2]",
   "x_1_6": "x_vars[7,2]",
   "x_1_7": "x_vars[8,2]",
   "x_2_0": "x_vars[1,3]",
   "x_2_1": "x_vars[2,3]",
   "x_2_2": "x_vars[3,3]",
   "x_2_3": "x_vars[4,3]",
   "x_2_4": "x_vars[5,3]",
   "x_2_5": "x_vars[6,3]",
   "x_2_6": "x_vars[7,3]",
   "x_2_7": "x_vars[8,3]",
   "x_3_0": "x_vars[1,4]",
   "x_3_1": "x_vars[2,4]",
   "x_3_2": "x_vars[3,4]",
   "x_3_3": "x_vars[4,4]",
   "x_3_4": "x_vars[5,4]",
   "x_3_5": "x_vars[6,4]",
   "x_3_6": "x_vars[7,4]",
   "x_3_7": "x_vars[8,4]",
   "x_4_0": "x_vars[1,5]",
   "x_4_1": "x_vars[2,5]",
   "x_4_2": "x_vars[3,5]",
   "x_4_3": "x_vars[4,5]",
   "x_4_4": "x_vars[5,5]",
   "x_4_5": "x_vars[6,5]",
   "x_4_6": "x_vars[7,5]",
   "x_4_7": "x_vars[8,5]",
   "x_5_0": "x_vars[1,6]",
   "x_5_1": "x_vars[2,6]",
   "x_5_2": "x_vars[3,6]",
   "x_5_3": "x_vars[4,6]",
   "x_5_4": "x_vars[5,6]",
   "x_5_5": "x_vars[6,6]",
   "x_5_6": "x_vars[7,6]",
   "x_5_7": "x_vars[8,6]",
   "x_6_0": "x_vars[1,7]",
   "x_6_1": "x_vars[2,7]",
   "x_6_2": "x_vars[3,7]",
   "x_6_3": "x_vars[4,7]",
   "x_6_4": "x_vars[5,7]",
   "x_6_5": "x_vars[6,7]",
   "x_6_6": "x_vars[7,7]",
   "x_6_7": "x_vars[8,7]",
   "x_7_0": "x_vars[1,8]",
   "x_7_1": "x_vars[2,8]",
   "x_7_2": "x_vars[3,8]",
   "x_7_3": "x_vars[4,8]",
   "x_7_4": "x_vars[5,8]",
   "x_7_5": "x_vars[6,8]",
   "x_7_6": "x_vars[7,8]",
   "x_7_7": "x_vars[8,8]",
   "x_8_0": "x_vars[1,9]",
   "x_8_1": "x_vars[2,9]",
   "x_8_2": "x_vars[3,9]",
   "x_8_3": "x_vars[4,9]",
   "x_8_4": "x_vars[5,9]",
   "x_8_5": "x_vars[6,9]",
   "x_8_6": "x_vars[7,9]",
   "x_8_7": "x_vars[8,9]",
   "x_9_0": "x_vars[1,10]",
   "x_9_1": "x_vars[2,10]",
   "x_9_2": "x_vars[3,10]",
   "x_9_3": "x_vars[4,10]",
   "x_9_4": "x_vars[5,10]",
   "x_9_5": "x_vars[6,10]",
   "x_9_6": "x_vars[7,10]",
   "x_9_7": "x_vars[8,10]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-045": {
  "source_code_sha256": "3789a81a04013749d5c596929ed8b3f3be420ac9ca7f03e8b32b3c8fb69426d4",
  "original_solution_sha256": "66a82d3222d1f4c64117422a0b64c6440b3775c98c5e351a4c80ae4b4888420c",
  "bindings": [
   [
    "C0",
    "x_vars[Spinach]"
   ],
   [
    "C1",
    "x_vars[Shiitake Mushrooms]"
   ],
   [
    "C2",
    "x_vars[Apples]"
   ],
   [
    "C3",
    "x_vars[Carrots]"
   ],
   [
    "C4",
    "x_vars[Basil]"
   ],
   [
    "C5",
    "x_vars[Potatoes]"
   ],
   [
    "C6",
    "x_vars[Green Beans]"
   ],
   [
    "C7",
    "x_vars[Blueberries]"
   ],
   [
    "C8",
    "x_vars[Oranges]"
   ],
   [
    "C9",
    "x_vars[Watermelons]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Spinach]",
   "x_1_0": "x_vars[Shiitake Mushrooms]",
   "x_2_0": "x_vars[Apples]",
   "x_3_0": "x_vars[Carrots]",
   "x_4_0": "x_vars[Basil]",
   "x_5_0": "x_vars[Potatoes]",
   "x_6_0": "x_vars[Green Beans]",
   "x_7_0": "x_vars[Blueberries]",
   "x_8_0": "x_vars[Oranges]",
   "x_9_0": "x_vars[Watermelons]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-046": {
  "source_code_sha256": "ddf26f4e16c4b289cf100704d9dc88943b5bd9288b7f2a213727da281aea2204",
  "original_solution_sha256": "d920562de6283a4e2a31bc0cf85f74ac501f1b025af2b0aa47247e72425f650c",
  "bindings": [
   [
    "C0",
    "x_vars[Spinach]"
   ],
   [
    "C1",
    "x_vars[Shiitake Mushrooms]"
   ],
   [
    "C2",
    "x_vars[Apples]"
   ],
   [
    "C3",
    "x_vars[Carrots]"
   ],
   [
    "C4",
    "x_vars[Basil]"
   ],
   [
    "C5",
    "x_vars[Potatoes]"
   ],
   [
    "C6",
    "x_vars[Green Beans]"
   ],
   [
    "C7",
    "x_vars[Blueberries]"
   ],
   [
    "C8",
    "x_vars[Oranges]"
   ],
   [
    "C9",
    "x_vars[Watermelons]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Spinach]",
   "x_1_0": "x_vars[Shiitake Mushrooms]",
   "x_2_0": "x_vars[Apples]",
   "x_3_0": "x_vars[Carrots]",
   "x_4_0": "x_vars[Basil]",
   "x_5_0": "x_vars[Potatoes]",
   "x_6_0": "x_vars[Green Beans]",
   "x_7_0": "x_vars[Blueberries]",
   "x_8_0": "x_vars[Oranges]",
   "x_9_0": "x_vars[Watermelons]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-047": {
  "source_code_sha256": "b30435285256c9ef9b696168741c1094442daf7017ac23105a887978ee99467d",
  "original_solution_sha256": "4c6caf5504ac4b216a3bf536e68d9bd7090d83099c026319d8b816b8ef107d34",
  "bindings": [
   [
    "C0",
    "x_vars[1,Racing]"
   ],
   [
    "C1",
    "x_vars[1,Sports]"
   ],
   [
    "C2",
    "x_vars[1,Action]"
   ],
   [
    "C3",
    "x_vars[1,Adventure]"
   ],
   [
    "C4",
    "x_vars[1,RPG]"
   ],
   [
    "C5",
    "x_vars[1,Shooter]"
   ],
   [
    "C6",
    "x_vars[1,Strategy]"
   ],
   [
    "C7",
    "x_vars[1,Simulation]"
   ],
   [
    "C8",
    "x_vars[1,Puzzle]"
   ],
   [
    "C9",
    "x_vars[1,Fighting]"
   ],
   [
    "C10",
    "x_vars[1,Platformer]"
   ],
   [
    "C11",
    "x_vars[1,Survival]"
   ],
   [
    "C12",
    "x_vars[1,Horror]"
   ],
   [
    "C13",
    "x_vars[1,Sandbox]"
   ],
   [
    "C14",
    "x_vars[1,MMO]"
   ],
   [
    "C15",
    "x_vars[2,Racing]"
   ],
   [
    "C16",
    "x_vars[2,Sports]"
   ],
   [
    "C17",
    "x_vars[2,Action]"
   ],
   [
    "C18",
    "x_vars[2,Adventure]"
   ],
   [
    "C19",
    "x_vars[2,RPG]"
   ],
   [
    "C20",
    "x_vars[2,Shooter]"
   ],
   [
    "C21",
    "x_vars[2,Strategy]"
   ],
   [
    "C22",
    "x_vars[2,Simulation]"
   ],
   [
    "C23",
    "x_vars[2,Puzzle]"
   ],
   [
    "C24",
    "x_vars[2,Fighting]"
   ],
   [
    "C25",
    "x_vars[2,Platformer]"
   ],
   [
    "C26",
    "x_vars[2,Survival]"
   ],
   [
    "C27",
    "x_vars[2,Horror]"
   ],
   [
    "C28",
    "x_vars[2,Sandbox]"
   ],
   [
    "C29",
    "x_vars[2,MMO]"
   ],
   [
    "C30",
    "x_vars[3,Racing]"
   ],
   [
    "C31",
    "x_vars[3,Sports]"
   ],
   [
    "C32",
    "x_vars[3,Action]"
   ],
   [
    "C33",
    "x_vars[3,Adventure]"
   ],
   [
    "C34",
    "x_vars[3,RPG]"
   ],
   [
    "C35",
    "x_vars[3,Shooter]"
   ],
   [
    "C36",
    "x_vars[3,Strategy]"
   ],
   [
    "C37",
    "x_vars[3,Simulation]"
   ],
   [
    "C38",
    "x_vars[3,Puzzle]"
   ],
   [
    "C39",
    "x_vars[3,Fighting]"
   ],
   [
    "C40",
    "x_vars[3,Platformer]"
   ],
   [
    "C41",
    "x_vars[3,Survival]"
   ],
   [
    "C42",
    "x_vars[3,Horror]"
   ],
   [
    "C43",
    "x_vars[3,Sandbox]"
   ],
   [
    "C44",
    "x_vars[3,MMO]"
   ],
   [
    "C45",
    "x_vars[4,Racing]"
   ],
   [
    "C46",
    "x_vars[4,Sports]"
   ],
   [
    "C47",
    "x_vars[4,Action]"
   ],
   [
    "C48",
    "x_vars[4,Adventure]"
   ],
   [
    "C49",
    "x_vars[4,RPG]"
   ],
   [
    "C50",
    "x_vars[4,Shooter]"
   ],
   [
    "C51",
    "x_vars[4,Strategy]"
   ],
   [
    "C52",
    "x_vars[4,Simulation]"
   ],
   [
    "C53",
    "x_vars[4,Puzzle]"
   ],
   [
    "C54",
    "x_vars[4,Fighting]"
   ],
   [
    "C55",
    "x_vars[4,Platformer]"
   ],
   [
    "C56",
    "x_vars[4,Survival]"
   ],
   [
    "C57",
    "x_vars[4,Horror]"
   ],
   [
    "C58",
    "x_vars[4,Sandbox]"
   ],
   [
    "C59",
    "x_vars[4,MMO]"
   ],
   [
    "C60",
    "x_vars[5,Racing]"
   ],
   [
    "C61",
    "x_vars[5,Sports]"
   ],
   [
    "C62",
    "x_vars[5,Action]"
   ],
   [
    "C63",
    "x_vars[5,Adventure]"
   ],
   [
    "C64",
    "x_vars[5,RPG]"
   ],
   [
    "C65",
    "x_vars[5,Shooter]"
   ],
   [
    "C66",
    "x_vars[5,Strategy]"
   ],
   [
    "C67",
    "x_vars[5,Simulation]"
   ],
   [
    "C68",
    "x_vars[5,Puzzle]"
   ],
   [
    "C69",
    "x_vars[5,Fighting]"
   ],
   [
    "C70",
    "x_vars[5,Platformer]"
   ],
   [
    "C71",
    "x_vars[5,Survival]"
   ],
   [
    "C72",
    "x_vars[5,Horror]"
   ],
   [
    "C73",
    "x_vars[5,Sandbox]"
   ],
   [
    "C74",
    "x_vars[5,MMO]"
   ],
   [
    "C75",
    "x_vars[6,Racing]"
   ],
   [
    "C76",
    "x_vars[6,Sports]"
   ],
   [
    "C77",
    "x_vars[6,Action]"
   ],
   [
    "C78",
    "x_vars[6,Adventure]"
   ],
   [
    "C79",
    "x_vars[6,RPG]"
   ],
   [
    "C80",
    "x_vars[6,Shooter]"
   ],
   [
    "C81",
    "x_vars[6,Strategy]"
   ],
   [
    "C82",
    "x_vars[6,Simulation]"
   ],
   [
    "C83",
    "x_vars[6,Puzzle]"
   ],
   [
    "C84",
    "x_vars[6,Fighting]"
   ],
   [
    "C85",
    "x_vars[6,Platformer]"
   ],
   [
    "C86",
    "x_vars[6,Survival]"
   ],
   [
    "C87",
    "x_vars[6,Horror]"
   ],
   [
    "C88",
    "x_vars[6,Sandbox]"
   ],
   [
    "C89",
    "x_vars[6,MMO]"
   ],
   [
    "C90",
    "x_vars[7,Racing]"
   ],
   [
    "C91",
    "x_vars[7,Sports]"
   ],
   [
    "C92",
    "x_vars[7,Action]"
   ],
   [
    "C93",
    "x_vars[7,Adventure]"
   ],
   [
    "C94",
    "x_vars[7,RPG]"
   ],
   [
    "C95",
    "x_vars[7,Shooter]"
   ],
   [
    "C96",
    "x_vars[7,Strategy]"
   ],
   [
    "C97",
    "x_vars[7,Simulation]"
   ],
   [
    "C98",
    "x_vars[7,Puzzle]"
   ],
   [
    "C99",
    "x_vars[7,Fighting]"
   ],
   [
    "C100",
    "x_vars[7,Platformer]"
   ],
   [
    "C101",
    "x_vars[7,Survival]"
   ],
   [
    "C102",
    "x_vars[7,Horror]"
   ],
   [
    "C103",
    "x_vars[7,Sandbox]"
   ],
   [
    "C104",
    "x_vars[7,MMO]"
   ],
   [
    "C105",
    "x_vars[8,Racing]"
   ],
   [
    "C106",
    "x_vars[8,Sports]"
   ],
   [
    "C107",
    "x_vars[8,Action]"
   ],
   [
    "C108",
    "x_vars[8,Adventure]"
   ],
   [
    "C109",
    "x_vars[8,RPG]"
   ],
   [
    "C110",
    "x_vars[8,Shooter]"
   ],
   [
    "C111",
    "x_vars[8,Strategy]"
   ],
   [
    "C112",
    "x_vars[8,Simulation]"
   ],
   [
    "C113",
    "x_vars[8,Puzzle]"
   ],
   [
    "C114",
    "x_vars[8,Fighting]"
   ],
   [
    "C115",
    "x_vars[8,Platformer]"
   ],
   [
    "C116",
    "x_vars[8,Survival]"
   ],
   [
    "C117",
    "x_vars[8,Horror]"
   ],
   [
    "C118",
    "x_vars[8,Sandbox]"
   ],
   [
    "C119",
    "x_vars[8,MMO]"
   ],
   [
    "C120",
    "x_vars[9,Racing]"
   ],
   [
    "C121",
    "x_vars[9,Sports]"
   ],
   [
    "C122",
    "x_vars[9,Action]"
   ],
   [
    "C123",
    "x_vars[9,Adventure]"
   ],
   [
    "C124",
    "x_vars[9,RPG]"
   ],
   [
    "C125",
    "x_vars[9,Shooter]"
   ],
   [
    "C126",
    "x_vars[9,Strategy]"
   ],
   [
    "C127",
    "x_vars[9,Simulation]"
   ],
   [
    "C128",
    "x_vars[9,Puzzle]"
   ],
   [
    "C129",
    "x_vars[9,Fighting]"
   ],
   [
    "C130",
    "x_vars[9,Platformer]"
   ],
   [
    "C131",
    "x_vars[9,Survival]"
   ],
   [
    "C132",
    "x_vars[9,Horror]"
   ],
   [
    "C133",
    "x_vars[9,Sandbox]"
   ],
   [
    "C134",
    "x_vars[9,MMO]"
   ],
   [
    "C135",
    "x_vars[10,Racing]"
   ],
   [
    "C136",
    "x_vars[10,Sports]"
   ],
   [
    "C137",
    "x_vars[10,Action]"
   ],
   [
    "C138",
    "x_vars[10,Adventure]"
   ],
   [
    "C139",
    "x_vars[10,RPG]"
   ],
   [
    "C140",
    "x_vars[10,Shooter]"
   ],
   [
    "C141",
    "x_vars[10,Strategy]"
   ],
   [
    "C142",
    "x_vars[10,Simulation]"
   ],
   [
    "C143",
    "x_vars[10,Puzzle]"
   ],
   [
    "C144",
    "x_vars[10,Fighting]"
   ],
   [
    "C145",
    "x_vars[10,Platformer]"
   ],
   [
    "C146",
    "x_vars[10,Survival]"
   ],
   [
    "C147",
    "x_vars[10,Horror]"
   ],
   [
    "C148",
    "x_vars[10,Sandbox]"
   ],
   [
    "C149",
    "x_vars[10,MMO]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Racing]",
   "x_0_1": "x_vars[2,Racing]",
   "x_0_2": "x_vars[3,Racing]",
   "x_0_3": "x_vars[4,Racing]",
   "x_0_4": "x_vars[5,Racing]",
   "x_0_5": "x_vars[6,Racing]",
   "x_0_6": "x_vars[7,Racing]",
   "x_0_7": "x_vars[8,Racing]",
   "x_0_8": "x_vars[9,Racing]",
   "x_0_9": "x_vars[10,Racing]",
   "x_1_0": "x_vars[1,Sports]",
   "x_1_1": "x_vars[2,Sports]",
   "x_1_2": "x_vars[3,Sports]",
   "x_1_3": "x_vars[4,Sports]",
   "x_1_4": "x_vars[5,Sports]",
   "x_1_5": "x_vars[6,Sports]",
   "x_1_6": "x_vars[7,Sports]",
   "x_1_7": "x_vars[8,Sports]",
   "x_1_8": "x_vars[9,Sports]",
   "x_1_9": "x_vars[10,Sports]",
   "x_2_0": "x_vars[1,Action]",
   "x_2_1": "x_vars[2,Action]",
   "x_2_2": "x_vars[3,Action]",
   "x_2_3": "x_vars[4,Action]",
   "x_2_4": "x_vars[5,Action]",
   "x_2_5": "x_vars[6,Action]",
   "x_2_6": "x_vars[7,Action]",
   "x_2_7": "x_vars[8,Action]",
   "x_2_8": "x_vars[9,Action]",
   "x_2_9": "x_vars[10,Action]",
   "x_3_0": "x_vars[1,Adventure]",
   "x_3_1": "x_vars[2,Adventure]",
   "x_3_2": "x_vars[3,Adventure]",
   "x_3_3": "x_vars[4,Adventure]",
   "x_3_4": "x_vars[5,Adventure]",
   "x_3_5": "x_vars[6,Adventure]",
   "x_3_6": "x_vars[7,Adventure]",
   "x_3_7": "x_vars[8,Adventure]",
   "x_3_8": "x_vars[9,Adventure]",
   "x_3_9": "x_vars[10,Adventure]",
   "x_4_0": "x_vars[1,RPG]",
   "x_4_1": "x_vars[2,RPG]",
   "x_4_2": "x_vars[3,RPG]",
   "x_4_3": "x_vars[4,RPG]",
   "x_4_4": "x_vars[5,RPG]",
   "x_4_5": "x_vars[6,RPG]",
   "x_4_6": "x_vars[7,RPG]",
   "x_4_7": "x_vars[8,RPG]",
   "x_4_8": "x_vars[9,RPG]",
   "x_4_9": "x_vars[10,RPG]",
   "x_5_0": "x_vars[1,Shooter]",
   "x_5_1": "x_vars[2,Shooter]",
   "x_5_2": "x_vars[3,Shooter]",
   "x_5_3": "x_vars[4,Shooter]",
   "x_5_4": "x_vars[5,Shooter]",
   "x_5_5": "x_vars[6,Shooter]",
   "x_5_6": "x_vars[7,Shooter]",
   "x_5_7": "x_vars[8,Shooter]",
   "x_5_8": "x_vars[9,Shooter]",
   "x_5_9": "x_vars[10,Shooter]",
   "x_6_0": "x_vars[1,Strategy]",
   "x_6_1": "x_vars[2,Strategy]",
   "x_6_2": "x_vars[3,Strategy]",
   "x_6_3": "x_vars[4,Strategy]",
   "x_6_4": "x_vars[5,Strategy]",
   "x_6_5": "x_vars[6,Strategy]",
   "x_6_6": "x_vars[7,Strategy]",
   "x_6_7": "x_vars[8,Strategy]",
   "x_6_8": "x_vars[9,Strategy]",
   "x_6_9": "x_vars[10,Strategy]",
   "x_7_0": "x_vars[1,Simulation]",
   "x_7_1": "x_vars[2,Simulation]",
   "x_7_2": "x_vars[3,Simulation]",
   "x_7_3": "x_vars[4,Simulation]",
   "x_7_4": "x_vars[5,Simulation]",
   "x_7_5": "x_vars[6,Simulation]",
   "x_7_6": "x_vars[7,Simulation]",
   "x_7_7": "x_vars[8,Simulation]",
   "x_7_8": "x_vars[9,Simulation]",
   "x_7_9": "x_vars[10,Simulation]",
   "x_8_0": "x_vars[1,Puzzle]",
   "x_8_1": "x_vars[2,Puzzle]",
   "x_8_2": "x_vars[3,Puzzle]",
   "x_8_3": "x_vars[4,Puzzle]",
   "x_8_4": "x_vars[5,Puzzle]",
   "x_8_5": "x_vars[6,Puzzle]",
   "x_8_6": "x_vars[7,Puzzle]",
   "x_8_7": "x_vars[8,Puzzle]",
   "x_8_8": "x_vars[9,Puzzle]",
   "x_8_9": "x_vars[10,Puzzle]",
   "x_9_0": "x_vars[1,Fighting]",
   "x_9_1": "x_vars[2,Fighting]",
   "x_9_2": "x_vars[3,Fighting]",
   "x_9_3": "x_vars[4,Fighting]",
   "x_9_4": "x_vars[5,Fighting]",
   "x_9_5": "x_vars[6,Fighting]",
   "x_9_6": "x_vars[7,Fighting]",
   "x_9_7": "x_vars[8,Fighting]",
   "x_9_8": "x_vars[9,Fighting]",
   "x_9_9": "x_vars[10,Fighting]",
   "x_10_0": "x_vars[1,Platformer]",
   "x_10_1": "x_vars[2,Platformer]",
   "x_10_2": "x_vars[3,Platformer]",
   "x_10_3": "x_vars[4,Platformer]",
   "x_10_4": "x_vars[5,Platformer]",
   "x_10_5": "x_vars[6,Platformer]",
   "x_10_6": "x_vars[7,Platformer]",
   "x_10_7": "x_vars[8,Platformer]",
   "x_10_8": "x_vars[9,Platformer]",
   "x_10_9": "x_vars[10,Platformer]",
   "x_11_0": "x_vars[1,Survival]",
   "x_11_1": "x_vars[2,Survival]",
   "x_11_2": "x_vars[3,Survival]",
   "x_11_3": "x_vars[4,Survival]",
   "x_11_4": "x_vars[5,Survival]",
   "x_11_5": "x_vars[6,Survival]",
   "x_11_6": "x_vars[7,Survival]",
   "x_11_7": "x_vars[8,Survival]",
   "x_11_8": "x_vars[9,Survival]",
   "x_11_9": "x_vars[10,Survival]",
   "x_12_0": "x_vars[1,Horror]",
   "x_12_1": "x_vars[2,Horror]",
   "x_12_2": "x_vars[3,Horror]",
   "x_12_3": "x_vars[4,Horror]",
   "x_12_4": "x_vars[5,Horror]",
   "x_12_5": "x_vars[6,Horror]",
   "x_12_6": "x_vars[7,Horror]",
   "x_12_7": "x_vars[8,Horror]",
   "x_12_8": "x_vars[9,Horror]",
   "x_12_9": "x_vars[10,Horror]",
   "x_13_0": "x_vars[1,Sandbox]",
   "x_13_1": "x_vars[2,Sandbox]",
   "x_13_2": "x_vars[3,Sandbox]",
   "x_13_3": "x_vars[4,Sandbox]",
   "x_13_4": "x_vars[5,Sandbox]",
   "x_13_5": "x_vars[6,Sandbox]",
   "x_13_6": "x_vars[7,Sandbox]",
   "x_13_7": "x_vars[8,Sandbox]",
   "x_13_8": "x_vars[9,Sandbox]",
   "x_13_9": "x_vars[10,Sandbox]",
   "x_14_0": "x_vars[1,MMO]",
   "x_14_1": "x_vars[2,MMO]",
   "x_14_2": "x_vars[3,MMO]",
   "x_14_3": "x_vars[4,MMO]",
   "x_14_4": "x_vars[5,MMO]",
   "x_14_5": "x_vars[6,MMO]",
   "x_14_6": "x_vars[7,MMO]",
   "x_14_7": "x_vars[8,MMO]",
   "x_14_8": "x_vars[9,MMO]",
   "x_14_9": "x_vars[10,MMO]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-048": {
  "source_code_sha256": "9fab04ca10893631e1f9c96dcb82ed4e9302150615b4e4f5a961549062ea28d4",
  "original_solution_sha256": "ce3b40ea1ba2404b7a327ea54b547984f6ec09b5d8edd15a1f03969efbb848f0",
  "bindings": [
   [
    "C0",
    "x_vars[1,Window Unit]"
   ],
   [
    "C1",
    "x_vars[1,Portable Unit]"
   ],
   [
    "C2",
    "x_vars[1,Split System]"
   ],
   [
    "C3",
    "x_vars[1,Ductless System]"
   ],
   [
    "C4",
    "x_vars[1,Central AC]"
   ],
   [
    "C5",
    "x_vars[1,Hybrid AC]"
   ],
   [
    "C6",
    "x_vars[1,Geothermal AC]"
   ],
   [
    "C7",
    "x_vars[1,Smart AC]"
   ],
   [
    "C8",
    "x_vars[1,Evaporative Cooler]"
   ],
   [
    "C9",
    "x_vars[1,Package Unit]"
   ],
   [
    "C10",
    "x_vars[2,Window Unit]"
   ],
   [
    "C11",
    "x_vars[2,Portable Unit]"
   ],
   [
    "C12",
    "x_vars[2,Split System]"
   ],
   [
    "C13",
    "x_vars[2,Ductless System]"
   ],
   [
    "C14",
    "x_vars[2,Central AC]"
   ],
   [
    "C15",
    "x_vars[2,Hybrid AC]"
   ],
   [
    "C16",
    "x_vars[2,Geothermal AC]"
   ],
   [
    "C17",
    "x_vars[2,Smart AC]"
   ],
   [
    "C18",
    "x_vars[2,Evaporative Cooler]"
   ],
   [
    "C19",
    "x_vars[2,Package Unit]"
   ],
   [
    "C20",
    "x_vars[3,Window Unit]"
   ],
   [
    "C21",
    "x_vars[3,Portable Unit]"
   ],
   [
    "C22",
    "x_vars[3,Split System]"
   ],
   [
    "C23",
    "x_vars[3,Ductless System]"
   ],
   [
    "C24",
    "x_vars[3,Central AC]"
   ],
   [
    "C25",
    "x_vars[3,Hybrid AC]"
   ],
   [
    "C26",
    "x_vars[3,Geothermal AC]"
   ],
   [
    "C27",
    "x_vars[3,Smart AC]"
   ],
   [
    "C28",
    "x_vars[3,Evaporative Cooler]"
   ],
   [
    "C29",
    "x_vars[3,Package Unit]"
   ],
   [
    "C30",
    "x_vars[4,Window Unit]"
   ],
   [
    "C31",
    "x_vars[4,Portable Unit]"
   ],
   [
    "C32",
    "x_vars[4,Split System]"
   ],
   [
    "C33",
    "x_vars[4,Ductless System]"
   ],
   [
    "C34",
    "x_vars[4,Central AC]"
   ],
   [
    "C35",
    "x_vars[4,Hybrid AC]"
   ],
   [
    "C36",
    "x_vars[4,Geothermal AC]"
   ],
   [
    "C37",
    "x_vars[4,Smart AC]"
   ],
   [
    "C38",
    "x_vars[4,Evaporative Cooler]"
   ],
   [
    "C39",
    "x_vars[4,Package Unit]"
   ],
   [
    "C40",
    "x_vars[5,Window Unit]"
   ],
   [
    "C41",
    "x_vars[5,Portable Unit]"
   ],
   [
    "C42",
    "x_vars[5,Split System]"
   ],
   [
    "C43",
    "x_vars[5,Ductless System]"
   ],
   [
    "C44",
    "x_vars[5,Central AC]"
   ],
   [
    "C45",
    "x_vars[5,Hybrid AC]"
   ],
   [
    "C46",
    "x_vars[5,Geothermal AC]"
   ],
   [
    "C47",
    "x_vars[5,Smart AC]"
   ],
   [
    "C48",
    "x_vars[5,Evaporative Cooler]"
   ],
   [
    "C49",
    "x_vars[5,Package Unit]"
   ],
   [
    "C50",
    "x_vars[6,Window Unit]"
   ],
   [
    "C51",
    "x_vars[6,Portable Unit]"
   ],
   [
    "C52",
    "x_vars[6,Split System]"
   ],
   [
    "C53",
    "x_vars[6,Ductless System]"
   ],
   [
    "C54",
    "x_vars[6,Central AC]"
   ],
   [
    "C55",
    "x_vars[6,Hybrid AC]"
   ],
   [
    "C56",
    "x_vars[6,Geothermal AC]"
   ],
   [
    "C57",
    "x_vars[6,Smart AC]"
   ],
   [
    "C58",
    "x_vars[6,Evaporative Cooler]"
   ],
   [
    "C59",
    "x_vars[6,Package Unit]"
   ],
   [
    "C60",
    "x_vars[7,Window Unit]"
   ],
   [
    "C61",
    "x_vars[7,Portable Unit]"
   ],
   [
    "C62",
    "x_vars[7,Split System]"
   ],
   [
    "C63",
    "x_vars[7,Ductless System]"
   ],
   [
    "C64",
    "x_vars[7,Central AC]"
   ],
   [
    "C65",
    "x_vars[7,Hybrid AC]"
   ],
   [
    "C66",
    "x_vars[7,Geothermal AC]"
   ],
   [
    "C67",
    "x_vars[7,Smart AC]"
   ],
   [
    "C68",
    "x_vars[7,Evaporative Cooler]"
   ],
   [
    "C69",
    "x_vars[7,Package Unit]"
   ],
   [
    "C70",
    "x_vars[8,Window Unit]"
   ],
   [
    "C71",
    "x_vars[8,Portable Unit]"
   ],
   [
    "C72",
    "x_vars[8,Split System]"
   ],
   [
    "C73",
    "x_vars[8,Ductless System]"
   ],
   [
    "C74",
    "x_vars[8,Central AC]"
   ],
   [
    "C75",
    "x_vars[8,Hybrid AC]"
   ],
   [
    "C76",
    "x_vars[8,Geothermal AC]"
   ],
   [
    "C77",
    "x_vars[8,Smart AC]"
   ],
   [
    "C78",
    "x_vars[8,Evaporative Cooler]"
   ],
   [
    "C79",
    "x_vars[8,Package Unit]"
   ],
   [
    "C80",
    "x_vars[9,Window Unit]"
   ],
   [
    "C81",
    "x_vars[9,Portable Unit]"
   ],
   [
    "C82",
    "x_vars[9,Split System]"
   ],
   [
    "C83",
    "x_vars[9,Ductless System]"
   ],
   [
    "C84",
    "x_vars[9,Central AC]"
   ],
   [
    "C85",
    "x_vars[9,Hybrid AC]"
   ],
   [
    "C86",
    "x_vars[9,Geothermal AC]"
   ],
   [
    "C87",
    "x_vars[9,Smart AC]"
   ],
   [
    "C88",
    "x_vars[9,Evaporative Cooler]"
   ],
   [
    "C89",
    "x_vars[9,Package Unit]"
   ],
   [
    "C90",
    "x_vars[10,Window Unit]"
   ],
   [
    "C91",
    "x_vars[10,Portable Unit]"
   ],
   [
    "C92",
    "x_vars[10,Split System]"
   ],
   [
    "C93",
    "x_vars[10,Ductless System]"
   ],
   [
    "C94",
    "x_vars[10,Central AC]"
   ],
   [
    "C95",
    "x_vars[10,Hybrid AC]"
   ],
   [
    "C96",
    "x_vars[10,Geothermal AC]"
   ],
   [
    "C97",
    "x_vars[10,Smart AC]"
   ],
   [
    "C98",
    "x_vars[10,Evaporative Cooler]"
   ],
   [
    "C99",
    "x_vars[10,Package Unit]"
   ],
   [
    "C100",
    "x_vars[11,Window Unit]"
   ],
   [
    "C101",
    "x_vars[11,Portable Unit]"
   ],
   [
    "C102",
    "x_vars[11,Split System]"
   ],
   [
    "C103",
    "x_vars[11,Ductless System]"
   ],
   [
    "C104",
    "x_vars[11,Central AC]"
   ],
   [
    "C105",
    "x_vars[11,Hybrid AC]"
   ],
   [
    "C106",
    "x_vars[11,Geothermal AC]"
   ],
   [
    "C107",
    "x_vars[11,Smart AC]"
   ],
   [
    "C108",
    "x_vars[11,Evaporative Cooler]"
   ],
   [
    "C109",
    "x_vars[11,Package Unit]"
   ],
   [
    "C110",
    "x_vars[12,Window Unit]"
   ],
   [
    "C111",
    "x_vars[12,Portable Unit]"
   ],
   [
    "C112",
    "x_vars[12,Split System]"
   ],
   [
    "C113",
    "x_vars[12,Ductless System]"
   ],
   [
    "C114",
    "x_vars[12,Central AC]"
   ],
   [
    "C115",
    "x_vars[12,Hybrid AC]"
   ],
   [
    "C116",
    "x_vars[12,Geothermal AC]"
   ],
   [
    "C117",
    "x_vars[12,Smart AC]"
   ],
   [
    "C118",
    "x_vars[12,Evaporative Cooler]"
   ],
   [
    "C119",
    "x_vars[12,Package Unit]"
   ],
   [
    "C120",
    "x_vars[13,Window Unit]"
   ],
   [
    "C121",
    "x_vars[13,Portable Unit]"
   ],
   [
    "C122",
    "x_vars[13,Split System]"
   ],
   [
    "C123",
    "x_vars[13,Ductless System]"
   ],
   [
    "C124",
    "x_vars[13,Central AC]"
   ],
   [
    "C125",
    "x_vars[13,Hybrid AC]"
   ],
   [
    "C126",
    "x_vars[13,Geothermal AC]"
   ],
   [
    "C127",
    "x_vars[13,Smart AC]"
   ],
   [
    "C128",
    "x_vars[13,Evaporative Cooler]"
   ],
   [
    "C129",
    "x_vars[13,Package Unit]"
   ],
   [
    "C130",
    "x_vars[14,Window Unit]"
   ],
   [
    "C131",
    "x_vars[14,Portable Unit]"
   ],
   [
    "C132",
    "x_vars[14,Split System]"
   ],
   [
    "C133",
    "x_vars[14,Ductless System]"
   ],
   [
    "C134",
    "x_vars[14,Central AC]"
   ],
   [
    "C135",
    "x_vars[14,Hybrid AC]"
   ],
   [
    "C136",
    "x_vars[14,Geothermal AC]"
   ],
   [
    "C137",
    "x_vars[14,Smart AC]"
   ],
   [
    "C138",
    "x_vars[14,Evaporative Cooler]"
   ],
   [
    "C139",
    "x_vars[14,Package Unit]"
   ],
   [
    "C140",
    "x_vars[15,Window Unit]"
   ],
   [
    "C141",
    "x_vars[15,Portable Unit]"
   ],
   [
    "C142",
    "x_vars[15,Split System]"
   ],
   [
    "C143",
    "x_vars[15,Ductless System]"
   ],
   [
    "C144",
    "x_vars[15,Central AC]"
   ],
   [
    "C145",
    "x_vars[15,Hybrid AC]"
   ],
   [
    "C146",
    "x_vars[15,Geothermal AC]"
   ],
   [
    "C147",
    "x_vars[15,Smart AC]"
   ],
   [
    "C148",
    "x_vars[15,Evaporative Cooler]"
   ],
   [
    "C149",
    "x_vars[15,Package Unit]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Window Unit]",
   "x_0_1": "x_vars[2,Window Unit]",
   "x_0_2": "x_vars[3,Window Unit]",
   "x_0_3": "x_vars[4,Window Unit]",
   "x_0_4": "x_vars[5,Window Unit]",
   "x_0_5": "x_vars[6,Window Unit]",
   "x_0_6": "x_vars[7,Window Unit]",
   "x_0_7": "x_vars[8,Window Unit]",
   "x_0_8": "x_vars[9,Window Unit]",
   "x_0_9": "x_vars[10,Window Unit]",
   "x_0_10": "x_vars[11,Window Unit]",
   "x_0_11": "x_vars[12,Window Unit]",
   "x_0_12": "x_vars[13,Window Unit]",
   "x_0_13": "x_vars[14,Window Unit]",
   "x_0_14": "x_vars[15,Window Unit]",
   "x_1_0": "x_vars[1,Portable Unit]",
   "x_1_1": "x_vars[2,Portable Unit]",
   "x_1_2": "x_vars[3,Portable Unit]",
   "x_1_3": "x_vars[4,Portable Unit]",
   "x_1_4": "x_vars[5,Portable Unit]",
   "x_1_5": "x_vars[6,Portable Unit]",
   "x_1_6": "x_vars[7,Portable Unit]",
   "x_1_7": "x_vars[8,Portable Unit]",
   "x_1_8": "x_vars[9,Portable Unit]",
   "x_1_9": "x_vars[10,Portable Unit]",
   "x_1_10": "x_vars[11,Portable Unit]",
   "x_1_11": "x_vars[12,Portable Unit]",
   "x_1_12": "x_vars[13,Portable Unit]",
   "x_1_13": "x_vars[14,Portable Unit]",
   "x_1_14": "x_vars[15,Portable Unit]",
   "x_2_0": "x_vars[1,Split System]",
   "x_2_1": "x_vars[2,Split System]",
   "x_2_2": "x_vars[3,Split System]",
   "x_2_3": "x_vars[4,Split System]",
   "x_2_4": "x_vars[5,Split System]",
   "x_2_5": "x_vars[6,Split System]",
   "x_2_6": "x_vars[7,Split System]",
   "x_2_7": "x_vars[8,Split System]",
   "x_2_8": "x_vars[9,Split System]",
   "x_2_9": "x_vars[10,Split System]",
   "x_2_10": "x_vars[11,Split System]",
   "x_2_11": "x_vars[12,Split System]",
   "x_2_12": "x_vars[13,Split System]",
   "x_2_13": "x_vars[14,Split System]",
   "x_2_14": "x_vars[15,Split System]",
   "x_3_0": "x_vars[1,Ductless System]",
   "x_3_1": "x_vars[2,Ductless System]",
   "x_3_2": "x_vars[3,Ductless System]",
   "x_3_3": "x_vars[4,Ductless System]",
   "x_3_4": "x_vars[5,Ductless System]",
   "x_3_5": "x_vars[6,Ductless System]",
   "x_3_6": "x_vars[7,Ductless System]",
   "x_3_7": "x_vars[8,Ductless System]",
   "x_3_8": "x_vars[9,Ductless System]",
   "x_3_9": "x_vars[10,Ductless System]",
   "x_3_10": "x_vars[11,Ductless System]",
   "x_3_11": "x_vars[12,Ductless System]",
   "x_3_12": "x_vars[13,Ductless System]",
   "x_3_13": "x_vars[14,Ductless System]",
   "x_3_14": "x_vars[15,Ductless System]",
   "x_4_0": "x_vars[1,Central AC]",
   "x_4_1": "x_vars[2,Central AC]",
   "x_4_2": "x_vars[3,Central AC]",
   "x_4_3": "x_vars[4,Central AC]",
   "x_4_4": "x_vars[5,Central AC]",
   "x_4_5": "x_vars[6,Central AC]",
   "x_4_6": "x_vars[7,Central AC]",
   "x_4_7": "x_vars[8,Central AC]",
   "x_4_8": "x_vars[9,Central AC]",
   "x_4_9": "x_vars[10,Central AC]",
   "x_4_10": "x_vars[11,Central AC]",
   "x_4_11": "x_vars[12,Central AC]",
   "x_4_12": "x_vars[13,Central AC]",
   "x_4_13": "x_vars[14,Central AC]",
   "x_4_14": "x_vars[15,Central AC]",
   "x_5_0": "x_vars[1,Hybrid AC]",
   "x_5_1": "x_vars[2,Hybrid AC]",
   "x_5_2": "x_vars[3,Hybrid AC]",
   "x_5_3": "x_vars[4,Hybrid AC]",
   "x_5_4": "x_vars[5,Hybrid AC]",
   "x_5_5": "x_vars[6,Hybrid AC]",
   "x_5_6": "x_vars[7,Hybrid AC]",
   "x_5_7": "x_vars[8,Hybrid AC]",
   "x_5_8": "x_vars[9,Hybrid AC]",
   "x_5_9": "x_vars[10,Hybrid AC]",
   "x_5_10": "x_vars[11,Hybrid AC]",
   "x_5_11": "x_vars[12,Hybrid AC]",
   "x_5_12": "x_vars[13,Hybrid AC]",
   "x_5_13": "x_vars[14,Hybrid AC]",
   "x_5_14": "x_vars[15,Hybrid AC]",
   "x_6_0": "x_vars[1,Geothermal AC]",
   "x_6_1": "x_vars[2,Geothermal AC]",
   "x_6_2": "x_vars[3,Geothermal AC]",
   "x_6_3": "x_vars[4,Geothermal AC]",
   "x_6_4": "x_vars[5,Geothermal AC]",
   "x_6_5": "x_vars[6,Geothermal AC]",
   "x_6_6": "x_vars[7,Geothermal AC]",
   "x_6_7": "x_vars[8,Geothermal AC]",
   "x_6_8": "x_vars[9,Geothermal AC]",
   "x_6_9": "x_vars[10,Geothermal AC]",
   "x_6_10": "x_vars[11,Geothermal AC]",
   "x_6_11": "x_vars[12,Geothermal AC]",
   "x_6_12": "x_vars[13,Geothermal AC]",
   "x_6_13": "x_vars[14,Geothermal AC]",
   "x_6_14": "x_vars[15,Geothermal AC]",
   "x_7_0": "x_vars[1,Smart AC]",
   "x_7_1": "x_vars[2,Smart AC]",
   "x_7_2": "x_vars[3,Smart AC]",
   "x_7_3": "x_vars[4,Smart AC]",
   "x_7_4": "x_vars[5,Smart AC]",
   "x_7_5": "x_vars[6,Smart AC]",
   "x_7_6": "x_vars[7,Smart AC]",
   "x_7_7": "x_vars[8,Smart AC]",
   "x_7_8": "x_vars[9,Smart AC]",
   "x_7_9": "x_vars[10,Smart AC]",
   "x_7_10": "x_vars[11,Smart AC]",
   "x_7_11": "x_vars[12,Smart AC]",
   "x_7_12": "x_vars[13,Smart AC]",
   "x_7_13": "x_vars[14,Smart AC]",
   "x_7_14": "x_vars[15,Smart AC]",
   "x_8_0": "x_vars[1,Evaporative Cooler]",
   "x_8_1": "x_vars[2,Evaporative Cooler]",
   "x_8_2": "x_vars[3,Evaporative Cooler]",
   "x_8_3": "x_vars[4,Evaporative Cooler]",
   "x_8_4": "x_vars[5,Evaporative Cooler]",
   "x_8_5": "x_vars[6,Evaporative Cooler]",
   "x_8_6": "x_vars[7,Evaporative Cooler]",
   "x_8_7": "x_vars[8,Evaporative Cooler]",
   "x_8_8": "x_vars[9,Evaporative Cooler]",
   "x_8_9": "x_vars[10,Evaporative Cooler]",
   "x_8_10": "x_vars[11,Evaporative Cooler]",
   "x_8_11": "x_vars[12,Evaporative Cooler]",
   "x_8_12": "x_vars[13,Evaporative Cooler]",
   "x_8_13": "x_vars[14,Evaporative Cooler]",
   "x_8_14": "x_vars[15,Evaporative Cooler]",
   "x_9_0": "x_vars[1,Package Unit]",
   "x_9_1": "x_vars[2,Package Unit]",
   "x_9_2": "x_vars[3,Package Unit]",
   "x_9_3": "x_vars[4,Package Unit]",
   "x_9_4": "x_vars[5,Package Unit]",
   "x_9_5": "x_vars[6,Package Unit]",
   "x_9_6": "x_vars[7,Package Unit]",
   "x_9_7": "x_vars[8,Package Unit]",
   "x_9_8": "x_vars[9,Package Unit]",
   "x_9_9": "x_vars[10,Package Unit]",
   "x_9_10": "x_vars[11,Package Unit]",
   "x_9_11": "x_vars[12,Package Unit]",
   "x_9_12": "x_vars[13,Package Unit]",
   "x_9_13": "x_vars[14,Package Unit]",
   "x_9_14": "x_vars[15,Package Unit]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-049": {
  "source_code_sha256": "008ef5dbbf431a7860d2d3f665d81ef826825b35636a6b9200426a3f66521631",
  "original_solution_sha256": "818d23acaffa9267df75706492cd632d5f60fe8b7f08eacec77592296254f75a",
  "bindings": [
   [
    "C0",
    "x_vars[1,Smartphone]"
   ],
   [
    "C1",
    "x_vars[1,Laptop]"
   ],
   [
    "C2",
    "x_vars[1,Headphones]"
   ],
   [
    "C3",
    "x_vars[1,Camera]"
   ],
   [
    "C4",
    "x_vars[1,Smartwatch]"
   ],
   [
    "C5",
    "x_vars[1,Tablet]"
   ],
   [
    "C6",
    "x_vars[1,Bluetooth Speaker]"
   ],
   [
    "C7",
    "x_vars[1,Keyboard]"
   ],
   [
    "C8",
    "x_vars[1,Mouse]"
   ],
   [
    "C9",
    "x_vars[1,Monitor]"
   ],
   [
    "C10",
    "x_vars[1,Printer]"
   ],
   [
    "C11",
    "x_vars[1,External Hard Drive]"
   ],
   [
    "C12",
    "x_vars[1,Router]"
   ],
   [
    "C13",
    "x_vars[1,Power Bank]"
   ],
   [
    "C14",
    "x_vars[1,Memory Card]"
   ],
   [
    "C15",
    "x_vars[1,USB Flash Drive]"
   ],
   [
    "C16",
    "x_vars[1,Smart Home Hub]"
   ],
   [
    "C17",
    "x_vars[1,Gaming Console]"
   ],
   [
    "C18",
    "x_vars[1,Fitness Tracker]"
   ],
   [
    "C19",
    "x_vars[1,E-Reader]"
   ],
   [
    "C20",
    "x_vars[2,Smartphone]"
   ],
   [
    "C21",
    "x_vars[2,Laptop]"
   ],
   [
    "C22",
    "x_vars[2,Headphones]"
   ],
   [
    "C23",
    "x_vars[2,Camera]"
   ],
   [
    "C24",
    "x_vars[2,Smartwatch]"
   ],
   [
    "C25",
    "x_vars[2,Tablet]"
   ],
   [
    "C26",
    "x_vars[2,Bluetooth Speaker]"
   ],
   [
    "C27",
    "x_vars[2,Keyboard]"
   ],
   [
    "C28",
    "x_vars[2,Mouse]"
   ],
   [
    "C29",
    "x_vars[2,Monitor]"
   ],
   [
    "C30",
    "x_vars[2,Printer]"
   ],
   [
    "C31",
    "x_vars[2,External Hard Drive]"
   ],
   [
    "C32",
    "x_vars[2,Router]"
   ],
   [
    "C33",
    "x_vars[2,Power Bank]"
   ],
   [
    "C34",
    "x_vars[2,Memory Card]"
   ],
   [
    "C35",
    "x_vars[2,USB Flash Drive]"
   ],
   [
    "C36",
    "x_vars[2,Smart Home Hub]"
   ],
   [
    "C37",
    "x_vars[2,Gaming Console]"
   ],
   [
    "C38",
    "x_vars[2,Fitness Tracker]"
   ],
   [
    "C39",
    "x_vars[2,E-Reader]"
   ],
   [
    "C40",
    "x_vars[3,Smartphone]"
   ],
   [
    "C41",
    "x_vars[3,Laptop]"
   ],
   [
    "C42",
    "x_vars[3,Headphones]"
   ],
   [
    "C43",
    "x_vars[3,Camera]"
   ],
   [
    "C44",
    "x_vars[3,Smartwatch]"
   ],
   [
    "C45",
    "x_vars[3,Tablet]"
   ],
   [
    "C46",
    "x_vars[3,Bluetooth Speaker]"
   ],
   [
    "C47",
    "x_vars[3,Keyboard]"
   ],
   [
    "C48",
    "x_vars[3,Mouse]"
   ],
   [
    "C49",
    "x_vars[3,Monitor]"
   ],
   [
    "C50",
    "x_vars[3,Printer]"
   ],
   [
    "C51",
    "x_vars[3,External Hard Drive]"
   ],
   [
    "C52",
    "x_vars[3,Router]"
   ],
   [
    "C53",
    "x_vars[3,Power Bank]"
   ],
   [
    "C54",
    "x_vars[3,Memory Card]"
   ],
   [
    "C55",
    "x_vars[3,USB Flash Drive]"
   ],
   [
    "C56",
    "x_vars[3,Smart Home Hub]"
   ],
   [
    "C57",
    "x_vars[3,Gaming Console]"
   ],
   [
    "C58",
    "x_vars[3,Fitness Tracker]"
   ],
   [
    "C59",
    "x_vars[3,E-Reader]"
   ],
   [
    "C60",
    "x_vars[4,Smartphone]"
   ],
   [
    "C61",
    "x_vars[4,Laptop]"
   ],
   [
    "C62",
    "x_vars[4,Headphones]"
   ],
   [
    "C63",
    "x_vars[4,Camera]"
   ],
   [
    "C64",
    "x_vars[4,Smartwatch]"
   ],
   [
    "C65",
    "x_vars[4,Tablet]"
   ],
   [
    "C66",
    "x_vars[4,Bluetooth Speaker]"
   ],
   [
    "C67",
    "x_vars[4,Keyboard]"
   ],
   [
    "C68",
    "x_vars[4,Mouse]"
   ],
   [
    "C69",
    "x_vars[4,Monitor]"
   ],
   [
    "C70",
    "x_vars[4,Printer]"
   ],
   [
    "C71",
    "x_vars[4,External Hard Drive]"
   ],
   [
    "C72",
    "x_vars[4,Router]"
   ],
   [
    "C73",
    "x_vars[4,Power Bank]"
   ],
   [
    "C74",
    "x_vars[4,Memory Card]"
   ],
   [
    "C75",
    "x_vars[4,USB Flash Drive]"
   ],
   [
    "C76",
    "x_vars[4,Smart Home Hub]"
   ],
   [
    "C77",
    "x_vars[4,Gaming Console]"
   ],
   [
    "C78",
    "x_vars[4,Fitness Tracker]"
   ],
   [
    "C79",
    "x_vars[4,E-Reader]"
   ],
   [
    "C80",
    "x_vars[5,Smartphone]"
   ],
   [
    "C81",
    "x_vars[5,Laptop]"
   ],
   [
    "C82",
    "x_vars[5,Headphones]"
   ],
   [
    "C83",
    "x_vars[5,Camera]"
   ],
   [
    "C84",
    "x_vars[5,Smartwatch]"
   ],
   [
    "C85",
    "x_vars[5,Tablet]"
   ],
   [
    "C86",
    "x_vars[5,Bluetooth Speaker]"
   ],
   [
    "C87",
    "x_vars[5,Keyboard]"
   ],
   [
    "C88",
    "x_vars[5,Mouse]"
   ],
   [
    "C89",
    "x_vars[5,Monitor]"
   ],
   [
    "C90",
    "x_vars[5,Printer]"
   ],
   [
    "C91",
    "x_vars[5,External Hard Drive]"
   ],
   [
    "C92",
    "x_vars[5,Router]"
   ],
   [
    "C93",
    "x_vars[5,Power Bank]"
   ],
   [
    "C94",
    "x_vars[5,Memory Card]"
   ],
   [
    "C95",
    "x_vars[5,USB Flash Drive]"
   ],
   [
    "C96",
    "x_vars[5,Smart Home Hub]"
   ],
   [
    "C97",
    "x_vars[5,Gaming Console]"
   ],
   [
    "C98",
    "x_vars[5,Fitness Tracker]"
   ],
   [
    "C99",
    "x_vars[5,E-Reader]"
   ],
   [
    "C100",
    "x_vars[6,Smartphone]"
   ],
   [
    "C101",
    "x_vars[6,Laptop]"
   ],
   [
    "C102",
    "x_vars[6,Headphones]"
   ],
   [
    "C103",
    "x_vars[6,Camera]"
   ],
   [
    "C104",
    "x_vars[6,Smartwatch]"
   ],
   [
    "C105",
    "x_vars[6,Tablet]"
   ],
   [
    "C106",
    "x_vars[6,Bluetooth Speaker]"
   ],
   [
    "C107",
    "x_vars[6,Keyboard]"
   ],
   [
    "C108",
    "x_vars[6,Mouse]"
   ],
   [
    "C109",
    "x_vars[6,Monitor]"
   ],
   [
    "C110",
    "x_vars[6,Printer]"
   ],
   [
    "C111",
    "x_vars[6,External Hard Drive]"
   ],
   [
    "C112",
    "x_vars[6,Router]"
   ],
   [
    "C113",
    "x_vars[6,Power Bank]"
   ],
   [
    "C114",
    "x_vars[6,Memory Card]"
   ],
   [
    "C115",
    "x_vars[6,USB Flash Drive]"
   ],
   [
    "C116",
    "x_vars[6,Smart Home Hub]"
   ],
   [
    "C117",
    "x_vars[6,Gaming Console]"
   ],
   [
    "C118",
    "x_vars[6,Fitness Tracker]"
   ],
   [
    "C119",
    "x_vars[6,E-Reader]"
   ],
   [
    "C120",
    "x_vars[7,Smartphone]"
   ],
   [
    "C121",
    "x_vars[7,Laptop]"
   ],
   [
    "C122",
    "x_vars[7,Headphones]"
   ],
   [
    "C123",
    "x_vars[7,Camera]"
   ],
   [
    "C124",
    "x_vars[7,Smartwatch]"
   ],
   [
    "C125",
    "x_vars[7,Tablet]"
   ],
   [
    "C126",
    "x_vars[7,Bluetooth Speaker]"
   ],
   [
    "C127",
    "x_vars[7,Keyboard]"
   ],
   [
    "C128",
    "x_vars[7,Mouse]"
   ],
   [
    "C129",
    "x_vars[7,Monitor]"
   ],
   [
    "C130",
    "x_vars[7,Printer]"
   ],
   [
    "C131",
    "x_vars[7,External Hard Drive]"
   ],
   [
    "C132",
    "x_vars[7,Router]"
   ],
   [
    "C133",
    "x_vars[7,Power Bank]"
   ],
   [
    "C134",
    "x_vars[7,Memory Card]"
   ],
   [
    "C135",
    "x_vars[7,USB Flash Drive]"
   ],
   [
    "C136",
    "x_vars[7,Smart Home Hub]"
   ],
   [
    "C137",
    "x_vars[7,Gaming Console]"
   ],
   [
    "C138",
    "x_vars[7,Fitness Tracker]"
   ],
   [
    "C139",
    "x_vars[7,E-Reader]"
   ],
   [
    "C140",
    "x_vars[8,Smartphone]"
   ],
   [
    "C141",
    "x_vars[8,Laptop]"
   ],
   [
    "C142",
    "x_vars[8,Headphones]"
   ],
   [
    "C143",
    "x_vars[8,Camera]"
   ],
   [
    "C144",
    "x_vars[8,Smartwatch]"
   ],
   [
    "C145",
    "x_vars[8,Tablet]"
   ],
   [
    "C146",
    "x_vars[8,Bluetooth Speaker]"
   ],
   [
    "C147",
    "x_vars[8,Keyboard]"
   ],
   [
    "C148",
    "x_vars[8,Mouse]"
   ],
   [
    "C149",
    "x_vars[8,Monitor]"
   ],
   [
    "C150",
    "x_vars[8,Printer]"
   ],
   [
    "C151",
    "x_vars[8,External Hard Drive]"
   ],
   [
    "C152",
    "x_vars[8,Router]"
   ],
   [
    "C153",
    "x_vars[8,Power Bank]"
   ],
   [
    "C154",
    "x_vars[8,Memory Card]"
   ],
   [
    "C155",
    "x_vars[8,USB Flash Drive]"
   ],
   [
    "C156",
    "x_vars[8,Smart Home Hub]"
   ],
   [
    "C157",
    "x_vars[8,Gaming Console]"
   ],
   [
    "C158",
    "x_vars[8,Fitness Tracker]"
   ],
   [
    "C159",
    "x_vars[8,E-Reader]"
   ],
   [
    "C160",
    "x_vars[9,Smartphone]"
   ],
   [
    "C161",
    "x_vars[9,Laptop]"
   ],
   [
    "C162",
    "x_vars[9,Headphones]"
   ],
   [
    "C163",
    "x_vars[9,Camera]"
   ],
   [
    "C164",
    "x_vars[9,Smartwatch]"
   ],
   [
    "C165",
    "x_vars[9,Tablet]"
   ],
   [
    "C166",
    "x_vars[9,Bluetooth Speaker]"
   ],
   [
    "C167",
    "x_vars[9,Keyboard]"
   ],
   [
    "C168",
    "x_vars[9,Mouse]"
   ],
   [
    "C169",
    "x_vars[9,Monitor]"
   ],
   [
    "C170",
    "x_vars[9,Printer]"
   ],
   [
    "C171",
    "x_vars[9,External Hard Drive]"
   ],
   [
    "C172",
    "x_vars[9,Router]"
   ],
   [
    "C173",
    "x_vars[9,Power Bank]"
   ],
   [
    "C174",
    "x_vars[9,Memory Card]"
   ],
   [
    "C175",
    "x_vars[9,USB Flash Drive]"
   ],
   [
    "C176",
    "x_vars[9,Smart Home Hub]"
   ],
   [
    "C177",
    "x_vars[9,Gaming Console]"
   ],
   [
    "C178",
    "x_vars[9,Fitness Tracker]"
   ],
   [
    "C179",
    "x_vars[9,E-Reader]"
   ],
   [
    "C180",
    "x_vars[10,Smartphone]"
   ],
   [
    "C181",
    "x_vars[10,Laptop]"
   ],
   [
    "C182",
    "x_vars[10,Headphones]"
   ],
   [
    "C183",
    "x_vars[10,Camera]"
   ],
   [
    "C184",
    "x_vars[10,Smartwatch]"
   ],
   [
    "C185",
    "x_vars[10,Tablet]"
   ],
   [
    "C186",
    "x_vars[10,Bluetooth Speaker]"
   ],
   [
    "C187",
    "x_vars[10,Keyboard]"
   ],
   [
    "C188",
    "x_vars[10,Mouse]"
   ],
   [
    "C189",
    "x_vars[10,Monitor]"
   ],
   [
    "C190",
    "x_vars[10,Printer]"
   ],
   [
    "C191",
    "x_vars[10,External Hard Drive]"
   ],
   [
    "C192",
    "x_vars[10,Router]"
   ],
   [
    "C193",
    "x_vars[10,Power Bank]"
   ],
   [
    "C194",
    "x_vars[10,Memory Card]"
   ],
   [
    "C195",
    "x_vars[10,USB Flash Drive]"
   ],
   [
    "C196",
    "x_vars[10,Smart Home Hub]"
   ],
   [
    "C197",
    "x_vars[10,Gaming Console]"
   ],
   [
    "C198",
    "x_vars[10,Fitness Tracker]"
   ],
   [
    "C199",
    "x_vars[10,E-Reader]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Smartphone]",
   "x_0_1": "x_vars[2,Smartphone]",
   "x_0_2": "x_vars[3,Smartphone]",
   "x_0_3": "x_vars[4,Smartphone]",
   "x_0_4": "x_vars[5,Smartphone]",
   "x_0_5": "x_vars[6,Smartphone]",
   "x_0_6": "x_vars[7,Smartphone]",
   "x_0_7": "x_vars[8,Smartphone]",
   "x_0_8": "x_vars[9,Smartphone]",
   "x_0_9": "x_vars[10,Smartphone]",
   "x_1_0": "x_vars[1,Laptop]",
   "x_1_1": "x_vars[2,Laptop]",
   "x_1_2": "x_vars[3,Laptop]",
   "x_1_3": "x_vars[4,Laptop]",
   "x_1_4": "x_vars[5,Laptop]",
   "x_1_5": "x_vars[6,Laptop]",
   "x_1_6": "x_vars[7,Laptop]",
   "x_1_7": "x_vars[8,Laptop]",
   "x_1_8": "x_vars[9,Laptop]",
   "x_1_9": "x_vars[10,Laptop]",
   "x_2_0": "x_vars[1,Headphones]",
   "x_2_1": "x_vars[2,Headphones]",
   "x_2_2": "x_vars[3,Headphones]",
   "x_2_3": "x_vars[4,Headphones]",
   "x_2_4": "x_vars[5,Headphones]",
   "x_2_5": "x_vars[6,Headphones]",
   "x_2_6": "x_vars[7,Headphones]",
   "x_2_7": "x_vars[8,Headphones]",
   "x_2_8": "x_vars[9,Headphones]",
   "x_2_9": "x_vars[10,Headphones]",
   "x_3_0": "x_vars[1,Camera]",
   "x_3_1": "x_vars[2,Camera]",
   "x_3_2": "x_vars[3,Camera]",
   "x_3_3": "x_vars[4,Camera]",
   "x_3_4": "x_vars[5,Camera]",
   "x_3_5": "x_vars[6,Camera]",
   "x_3_6": "x_vars[7,Camera]",
   "x_3_7": "x_vars[8,Camera]",
   "x_3_8": "x_vars[9,Camera]",
   "x_3_9": "x_vars[10,Camera]",
   "x_4_0": "x_vars[1,Smartwatch]",
   "x_4_1": "x_vars[2,Smartwatch]",
   "x_4_2": "x_vars[3,Smartwatch]",
   "x_4_3": "x_vars[4,Smartwatch]",
   "x_4_4": "x_vars[5,Smartwatch]",
   "x_4_5": "x_vars[6,Smartwatch]",
   "x_4_6": "x_vars[7,Smartwatch]",
   "x_4_7": "x_vars[8,Smartwatch]",
   "x_4_8": "x_vars[9,Smartwatch]",
   "x_4_9": "x_vars[10,Smartwatch]",
   "x_5_0": "x_vars[1,Tablet]",
   "x_5_1": "x_vars[2,Tablet]",
   "x_5_2": "x_vars[3,Tablet]",
   "x_5_3": "x_vars[4,Tablet]",
   "x_5_4": "x_vars[5,Tablet]",
   "x_5_5": "x_vars[6,Tablet]",
   "x_5_6": "x_vars[7,Tablet]",
   "x_5_7": "x_vars[8,Tablet]",
   "x_5_8": "x_vars[9,Tablet]",
   "x_5_9": "x_vars[10,Tablet]",
   "x_6_0": "x_vars[1,Bluetooth Speaker]",
   "x_6_1": "x_vars[2,Bluetooth Speaker]",
   "x_6_2": "x_vars[3,Bluetooth Speaker]",
   "x_6_3": "x_vars[4,Bluetooth Speaker]",
   "x_6_4": "x_vars[5,Bluetooth Speaker]",
   "x_6_5": "x_vars[6,Bluetooth Speaker]",
   "x_6_6": "x_vars[7,Bluetooth Speaker]",
   "x_6_7": "x_vars[8,Bluetooth Speaker]",
   "x_6_8": "x_vars[9,Bluetooth Speaker]",
   "x_6_9": "x_vars[10,Bluetooth Speaker]",
   "x_7_0": "x_vars[1,Keyboard]",
   "x_7_1": "x_vars[2,Keyboard]",
   "x_7_2": "x_vars[3,Keyboard]",
   "x_7_3": "x_vars[4,Keyboard]",
   "x_7_4": "x_vars[5,Keyboard]",
   "x_7_5": "x_vars[6,Keyboard]",
   "x_7_6": "x_vars[7,Keyboard]",
   "x_7_7": "x_vars[8,Keyboard]",
   "x_7_8": "x_vars[9,Keyboard]",
   "x_7_9": "x_vars[10,Keyboard]",
   "x_8_0": "x_vars[1,Mouse]",
   "x_8_1": "x_vars[2,Mouse]",
   "x_8_2": "x_vars[3,Mouse]",
   "x_8_3": "x_vars[4,Mouse]",
   "x_8_4": "x_vars[5,Mouse]",
   "x_8_5": "x_vars[6,Mouse]",
   "x_8_6": "x_vars[7,Mouse]",
   "x_8_7": "x_vars[8,Mouse]",
   "x_8_8": "x_vars[9,Mouse]",
   "x_8_9": "x_vars[10,Mouse]",
   "x_9_0": "x_vars[1,Monitor]",
   "x_9_1": "x_vars[2,Monitor]",
   "x_9_2": "x_vars[3,Monitor]",
   "x_9_3": "x_vars[4,Monitor]",
   "x_9_4": "x_vars[5,Monitor]",
   "x_9_5": "x_vars[6,Monitor]",
   "x_9_6": "x_vars[7,Monitor]",
   "x_9_7": "x_vars[8,Monitor]",
   "x_9_8": "x_vars[9,Monitor]",
   "x_9_9": "x_vars[10,Monitor]",
   "x_10_0": "x_vars[1,Printer]",
   "x_10_1": "x_vars[2,Printer]",
   "x_10_2": "x_vars[3,Printer]",
   "x_10_3": "x_vars[4,Printer]",
   "x_10_4": "x_vars[5,Printer]",
   "x_10_5": "x_vars[6,Printer]",
   "x_10_6": "x_vars[7,Printer]",
   "x_10_7": "x_vars[8,Printer]",
   "x_10_8": "x_vars[9,Printer]",
   "x_10_9": "x_vars[10,Printer]",
   "x_11_0": "x_vars[1,External Hard Drive]",
   "x_11_1": "x_vars[2,External Hard Drive]",
   "x_11_2": "x_vars[3,External Hard Drive]",
   "x_11_3": "x_vars[4,External Hard Drive]",
   "x_11_4": "x_vars[5,External Hard Drive]",
   "x_11_5": "x_vars[6,External Hard Drive]",
   "x_11_6": "x_vars[7,External Hard Drive]",
   "x_11_7": "x_vars[8,External Hard Drive]",
   "x_11_8": "x_vars[9,External Hard Drive]",
   "x_11_9": "x_vars[10,External Hard Drive]",
   "x_12_0": "x_vars[1,Router]",
   "x_12_1": "x_vars[2,Router]",
   "x_12_2": "x_vars[3,Router]",
   "x_12_3": "x_vars[4,Router]",
   "x_12_4": "x_vars[5,Router]",
   "x_12_5": "x_vars[6,Router]",
   "x_12_6": "x_vars[7,Router]",
   "x_12_7": "x_vars[8,Router]",
   "x_12_8": "x_vars[9,Router]",
   "x_12_9": "x_vars[10,Router]",
   "x_13_0": "x_vars[1,Power Bank]",
   "x_13_1": "x_vars[2,Power Bank]",
   "x_13_2": "x_vars[3,Power Bank]",
   "x_13_3": "x_vars[4,Power Bank]",
   "x_13_4": "x_vars[5,Power Bank]",
   "x_13_5": "x_vars[6,Power Bank]",
   "x_13_6": "x_vars[7,Power Bank]",
   "x_13_7": "x_vars[8,Power Bank]",
   "x_13_8": "x_vars[9,Power Bank]",
   "x_13_9": "x_vars[10,Power Bank]",
   "x_14_0": "x_vars[1,Memory Card]",
   "x_14_1": "x_vars[2,Memory Card]",
   "x_14_2": "x_vars[3,Memory Card]",
   "x_14_3": "x_vars[4,Memory Card]",
   "x_14_4": "x_vars[5,Memory Card]",
   "x_14_5": "x_vars[6,Memory Card]",
   "x_14_6": "x_vars[7,Memory Card]",
   "x_14_7": "x_vars[8,Memory Card]",
   "x_14_8": "x_vars[9,Memory Card]",
   "x_14_9": "x_vars[10,Memory Card]",
   "x_15_0": "x_vars[1,USB Flash Drive]",
   "x_15_1": "x_vars[2,USB Flash Drive]",
   "x_15_2": "x_vars[3,USB Flash Drive]",
   "x_15_3": "x_vars[4,USB Flash Drive]",
   "x_15_4": "x_vars[5,USB Flash Drive]",
   "x_15_5": "x_vars[6,USB Flash Drive]",
   "x_15_6": "x_vars[7,USB Flash Drive]",
   "x_15_7": "x_vars[8,USB Flash Drive]",
   "x_15_8": "x_vars[9,USB Flash Drive]",
   "x_15_9": "x_vars[10,USB Flash Drive]",
   "x_16_0": "x_vars[1,Smart Home Hub]",
   "x_16_1": "x_vars[2,Smart Home Hub]",
   "x_16_2": "x_vars[3,Smart Home Hub]",
   "x_16_3": "x_vars[4,Smart Home Hub]",
   "x_16_4": "x_vars[5,Smart Home Hub]",
   "x_16_5": "x_vars[6,Smart Home Hub]",
   "x_16_6": "x_vars[7,Smart Home Hub]",
   "x_16_7": "x_vars[8,Smart Home Hub]",
   "x_16_8": "x_vars[9,Smart Home Hub]",
   "x_16_9": "x_vars[10,Smart Home Hub]",
   "x_17_0": "x_vars[1,Gaming Console]",
   "x_17_1": "x_vars[2,Gaming Console]",
   "x_17_2": "x_vars[3,Gaming Console]",
   "x_17_3": "x_vars[4,Gaming Console]",
   "x_17_4": "x_vars[5,Gaming Console]",
   "x_17_5": "x_vars[6,Gaming Console]",
   "x_17_6": "x_vars[7,Gaming Console]",
   "x_17_7": "x_vars[8,Gaming Console]",
   "x_17_8": "x_vars[9,Gaming Console]",
   "x_17_9": "x_vars[10,Gaming Console]",
   "x_18_0": "x_vars[1,Fitness Tracker]",
   "x_18_1": "x_vars[2,Fitness Tracker]",
   "x_18_2": "x_vars[3,Fitness Tracker]",
   "x_18_3": "x_vars[4,Fitness Tracker]",
   "x_18_4": "x_vars[5,Fitness Tracker]",
   "x_18_5": "x_vars[6,Fitness Tracker]",
   "x_18_6": "x_vars[7,Fitness Tracker]",
   "x_18_7": "x_vars[8,Fitness Tracker]",
   "x_18_8": "x_vars[9,Fitness Tracker]",
   "x_18_9": "x_vars[10,Fitness Tracker]",
   "x_19_0": "x_vars[1,E-Reader]",
   "x_19_1": "x_vars[2,E-Reader]",
   "x_19_2": "x_vars[3,E-Reader]",
   "x_19_3": "x_vars[4,E-Reader]",
   "x_19_4": "x_vars[5,E-Reader]",
   "x_19_5": "x_vars[6,E-Reader]",
   "x_19_6": "x_vars[7,E-Reader]",
   "x_19_7": "x_vars[8,E-Reader]",
   "x_19_8": "x_vars[9,E-Reader]",
   "x_19_9": "x_vars[10,E-Reader]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-050": {
  "source_code_sha256": "6b90d8c6fd59f1d5820a9cba49f4eafc62f2738bf2b0ee0e32a0768dd98b5c65",
  "original_solution_sha256": "4458cfa2d051a53ebd5570ee89439918e3b04bacb4e394fdbfa4783439747b31",
  "bindings": [
   [
    "C0",
    "x_vars[1,Smartphone]"
   ],
   [
    "C1",
    "x_vars[1,Laptop]"
   ],
   [
    "C2",
    "x_vars[1,Headphones]"
   ],
   [
    "C3",
    "x_vars[1,Camera]"
   ],
   [
    "C4",
    "x_vars[1,Smartwatch]"
   ],
   [
    "C5",
    "x_vars[1,Tablet]"
   ],
   [
    "C6",
    "x_vars[1,Bluetooth Speaker]"
   ],
   [
    "C7",
    "x_vars[1,Keyboard]"
   ],
   [
    "C8",
    "x_vars[1,Mouse]"
   ],
   [
    "C9",
    "x_vars[1,Monitor]"
   ],
   [
    "C10",
    "x_vars[1,Printer]"
   ],
   [
    "C11",
    "x_vars[1,External Hard Drive]"
   ],
   [
    "C12",
    "x_vars[1,Router]"
   ],
   [
    "C13",
    "x_vars[1,Power Bank]"
   ],
   [
    "C14",
    "x_vars[1,Memory Card]"
   ],
   [
    "C15",
    "x_vars[1,USB Flash Drive]"
   ],
   [
    "C16",
    "x_vars[1,Smart Home Hub]"
   ],
   [
    "C17",
    "x_vars[1,Gaming Console]"
   ],
   [
    "C18",
    "x_vars[1,Fitness Tracker]"
   ],
   [
    "C19",
    "x_vars[1,E-Reader]"
   ],
   [
    "C20",
    "x_vars[2,Smartphone]"
   ],
   [
    "C21",
    "x_vars[2,Laptop]"
   ],
   [
    "C22",
    "x_vars[2,Headphones]"
   ],
   [
    "C23",
    "x_vars[2,Camera]"
   ],
   [
    "C24",
    "x_vars[2,Smartwatch]"
   ],
   [
    "C25",
    "x_vars[2,Tablet]"
   ],
   [
    "C26",
    "x_vars[2,Bluetooth Speaker]"
   ],
   [
    "C27",
    "x_vars[2,Keyboard]"
   ],
   [
    "C28",
    "x_vars[2,Mouse]"
   ],
   [
    "C29",
    "x_vars[2,Monitor]"
   ],
   [
    "C30",
    "x_vars[2,Printer]"
   ],
   [
    "C31",
    "x_vars[2,External Hard Drive]"
   ],
   [
    "C32",
    "x_vars[2,Router]"
   ],
   [
    "C33",
    "x_vars[2,Power Bank]"
   ],
   [
    "C34",
    "x_vars[2,Memory Card]"
   ],
   [
    "C35",
    "x_vars[2,USB Flash Drive]"
   ],
   [
    "C36",
    "x_vars[2,Smart Home Hub]"
   ],
   [
    "C37",
    "x_vars[2,Gaming Console]"
   ],
   [
    "C38",
    "x_vars[2,Fitness Tracker]"
   ],
   [
    "C39",
    "x_vars[2,E-Reader]"
   ],
   [
    "C40",
    "x_vars[3,Smartphone]"
   ],
   [
    "C41",
    "x_vars[3,Laptop]"
   ],
   [
    "C42",
    "x_vars[3,Headphones]"
   ],
   [
    "C43",
    "x_vars[3,Camera]"
   ],
   [
    "C44",
    "x_vars[3,Smartwatch]"
   ],
   [
    "C45",
    "x_vars[3,Tablet]"
   ],
   [
    "C46",
    "x_vars[3,Bluetooth Speaker]"
   ],
   [
    "C47",
    "x_vars[3,Keyboard]"
   ],
   [
    "C48",
    "x_vars[3,Mouse]"
   ],
   [
    "C49",
    "x_vars[3,Monitor]"
   ],
   [
    "C50",
    "x_vars[3,Printer]"
   ],
   [
    "C51",
    "x_vars[3,External Hard Drive]"
   ],
   [
    "C52",
    "x_vars[3,Router]"
   ],
   [
    "C53",
    "x_vars[3,Power Bank]"
   ],
   [
    "C54",
    "x_vars[3,Memory Card]"
   ],
   [
    "C55",
    "x_vars[3,USB Flash Drive]"
   ],
   [
    "C56",
    "x_vars[3,Smart Home Hub]"
   ],
   [
    "C57",
    "x_vars[3,Gaming Console]"
   ],
   [
    "C58",
    "x_vars[3,Fitness Tracker]"
   ],
   [
    "C59",
    "x_vars[3,E-Reader]"
   ],
   [
    "C60",
    "x_vars[4,Smartphone]"
   ],
   [
    "C61",
    "x_vars[4,Laptop]"
   ],
   [
    "C62",
    "x_vars[4,Headphones]"
   ],
   [
    "C63",
    "x_vars[4,Camera]"
   ],
   [
    "C64",
    "x_vars[4,Smartwatch]"
   ],
   [
    "C65",
    "x_vars[4,Tablet]"
   ],
   [
    "C66",
    "x_vars[4,Bluetooth Speaker]"
   ],
   [
    "C67",
    "x_vars[4,Keyboard]"
   ],
   [
    "C68",
    "x_vars[4,Mouse]"
   ],
   [
    "C69",
    "x_vars[4,Monitor]"
   ],
   [
    "C70",
    "x_vars[4,Printer]"
   ],
   [
    "C71",
    "x_vars[4,External Hard Drive]"
   ],
   [
    "C72",
    "x_vars[4,Router]"
   ],
   [
    "C73",
    "x_vars[4,Power Bank]"
   ],
   [
    "C74",
    "x_vars[4,Memory Card]"
   ],
   [
    "C75",
    "x_vars[4,USB Flash Drive]"
   ],
   [
    "C76",
    "x_vars[4,Smart Home Hub]"
   ],
   [
    "C77",
    "x_vars[4,Gaming Console]"
   ],
   [
    "C78",
    "x_vars[4,Fitness Tracker]"
   ],
   [
    "C79",
    "x_vars[4,E-Reader]"
   ],
   [
    "C80",
    "x_vars[5,Smartphone]"
   ],
   [
    "C81",
    "x_vars[5,Laptop]"
   ],
   [
    "C82",
    "x_vars[5,Headphones]"
   ],
   [
    "C83",
    "x_vars[5,Camera]"
   ],
   [
    "C84",
    "x_vars[5,Smartwatch]"
   ],
   [
    "C85",
    "x_vars[5,Tablet]"
   ],
   [
    "C86",
    "x_vars[5,Bluetooth Speaker]"
   ],
   [
    "C87",
    "x_vars[5,Keyboard]"
   ],
   [
    "C88",
    "x_vars[5,Mouse]"
   ],
   [
    "C89",
    "x_vars[5,Monitor]"
   ],
   [
    "C90",
    "x_vars[5,Printer]"
   ],
   [
    "C91",
    "x_vars[5,External Hard Drive]"
   ],
   [
    "C92",
    "x_vars[5,Router]"
   ],
   [
    "C93",
    "x_vars[5,Power Bank]"
   ],
   [
    "C94",
    "x_vars[5,Memory Card]"
   ],
   [
    "C95",
    "x_vars[5,USB Flash Drive]"
   ],
   [
    "C96",
    "x_vars[5,Smart Home Hub]"
   ],
   [
    "C97",
    "x_vars[5,Gaming Console]"
   ],
   [
    "C98",
    "x_vars[5,Fitness Tracker]"
   ],
   [
    "C99",
    "x_vars[5,E-Reader]"
   ],
   [
    "C100",
    "x_vars[6,Smartphone]"
   ],
   [
    "C101",
    "x_vars[6,Laptop]"
   ],
   [
    "C102",
    "x_vars[6,Headphones]"
   ],
   [
    "C103",
    "x_vars[6,Camera]"
   ],
   [
    "C104",
    "x_vars[6,Smartwatch]"
   ],
   [
    "C105",
    "x_vars[6,Tablet]"
   ],
   [
    "C106",
    "x_vars[6,Bluetooth Speaker]"
   ],
   [
    "C107",
    "x_vars[6,Keyboard]"
   ],
   [
    "C108",
    "x_vars[6,Mouse]"
   ],
   [
    "C109",
    "x_vars[6,Monitor]"
   ],
   [
    "C110",
    "x_vars[6,Printer]"
   ],
   [
    "C111",
    "x_vars[6,External Hard Drive]"
   ],
   [
    "C112",
    "x_vars[6,Router]"
   ],
   [
    "C113",
    "x_vars[6,Power Bank]"
   ],
   [
    "C114",
    "x_vars[6,Memory Card]"
   ],
   [
    "C115",
    "x_vars[6,USB Flash Drive]"
   ],
   [
    "C116",
    "x_vars[6,Smart Home Hub]"
   ],
   [
    "C117",
    "x_vars[6,Gaming Console]"
   ],
   [
    "C118",
    "x_vars[6,Fitness Tracker]"
   ],
   [
    "C119",
    "x_vars[6,E-Reader]"
   ],
   [
    "C120",
    "x_vars[7,Smartphone]"
   ],
   [
    "C121",
    "x_vars[7,Laptop]"
   ],
   [
    "C122",
    "x_vars[7,Headphones]"
   ],
   [
    "C123",
    "x_vars[7,Camera]"
   ],
   [
    "C124",
    "x_vars[7,Smartwatch]"
   ],
   [
    "C125",
    "x_vars[7,Tablet]"
   ],
   [
    "C126",
    "x_vars[7,Bluetooth Speaker]"
   ],
   [
    "C127",
    "x_vars[7,Keyboard]"
   ],
   [
    "C128",
    "x_vars[7,Mouse]"
   ],
   [
    "C129",
    "x_vars[7,Monitor]"
   ],
   [
    "C130",
    "x_vars[7,Printer]"
   ],
   [
    "C131",
    "x_vars[7,External Hard Drive]"
   ],
   [
    "C132",
    "x_vars[7,Router]"
   ],
   [
    "C133",
    "x_vars[7,Power Bank]"
   ],
   [
    "C134",
    "x_vars[7,Memory Card]"
   ],
   [
    "C135",
    "x_vars[7,USB Flash Drive]"
   ],
   [
    "C136",
    "x_vars[7,Smart Home Hub]"
   ],
   [
    "C137",
    "x_vars[7,Gaming Console]"
   ],
   [
    "C138",
    "x_vars[7,Fitness Tracker]"
   ],
   [
    "C139",
    "x_vars[7,E-Reader]"
   ],
   [
    "C140",
    "x_vars[8,Smartphone]"
   ],
   [
    "C141",
    "x_vars[8,Laptop]"
   ],
   [
    "C142",
    "x_vars[8,Headphones]"
   ],
   [
    "C143",
    "x_vars[8,Camera]"
   ],
   [
    "C144",
    "x_vars[8,Smartwatch]"
   ],
   [
    "C145",
    "x_vars[8,Tablet]"
   ],
   [
    "C146",
    "x_vars[8,Bluetooth Speaker]"
   ],
   [
    "C147",
    "x_vars[8,Keyboard]"
   ],
   [
    "C148",
    "x_vars[8,Mouse]"
   ],
   [
    "C149",
    "x_vars[8,Monitor]"
   ],
   [
    "C150",
    "x_vars[8,Printer]"
   ],
   [
    "C151",
    "x_vars[8,External Hard Drive]"
   ],
   [
    "C152",
    "x_vars[8,Router]"
   ],
   [
    "C153",
    "x_vars[8,Power Bank]"
   ],
   [
    "C154",
    "x_vars[8,Memory Card]"
   ],
   [
    "C155",
    "x_vars[8,USB Flash Drive]"
   ],
   [
    "C156",
    "x_vars[8,Smart Home Hub]"
   ],
   [
    "C157",
    "x_vars[8,Gaming Console]"
   ],
   [
    "C158",
    "x_vars[8,Fitness Tracker]"
   ],
   [
    "C159",
    "x_vars[8,E-Reader]"
   ],
   [
    "C160",
    "x_vars[9,Smartphone]"
   ],
   [
    "C161",
    "x_vars[9,Laptop]"
   ],
   [
    "C162",
    "x_vars[9,Headphones]"
   ],
   [
    "C163",
    "x_vars[9,Camera]"
   ],
   [
    "C164",
    "x_vars[9,Smartwatch]"
   ],
   [
    "C165",
    "x_vars[9,Tablet]"
   ],
   [
    "C166",
    "x_vars[9,Bluetooth Speaker]"
   ],
   [
    "C167",
    "x_vars[9,Keyboard]"
   ],
   [
    "C168",
    "x_vars[9,Mouse]"
   ],
   [
    "C169",
    "x_vars[9,Monitor]"
   ],
   [
    "C170",
    "x_vars[9,Printer]"
   ],
   [
    "C171",
    "x_vars[9,External Hard Drive]"
   ],
   [
    "C172",
    "x_vars[9,Router]"
   ],
   [
    "C173",
    "x_vars[9,Power Bank]"
   ],
   [
    "C174",
    "x_vars[9,Memory Card]"
   ],
   [
    "C175",
    "x_vars[9,USB Flash Drive]"
   ],
   [
    "C176",
    "x_vars[9,Smart Home Hub]"
   ],
   [
    "C177",
    "x_vars[9,Gaming Console]"
   ],
   [
    "C178",
    "x_vars[9,Fitness Tracker]"
   ],
   [
    "C179",
    "x_vars[9,E-Reader]"
   ],
   [
    "C180",
    "x_vars[10,Smartphone]"
   ],
   [
    "C181",
    "x_vars[10,Laptop]"
   ],
   [
    "C182",
    "x_vars[10,Headphones]"
   ],
   [
    "C183",
    "x_vars[10,Camera]"
   ],
   [
    "C184",
    "x_vars[10,Smartwatch]"
   ],
   [
    "C185",
    "x_vars[10,Tablet]"
   ],
   [
    "C186",
    "x_vars[10,Bluetooth Speaker]"
   ],
   [
    "C187",
    "x_vars[10,Keyboard]"
   ],
   [
    "C188",
    "x_vars[10,Mouse]"
   ],
   [
    "C189",
    "x_vars[10,Monitor]"
   ],
   [
    "C190",
    "x_vars[10,Printer]"
   ],
   [
    "C191",
    "x_vars[10,External Hard Drive]"
   ],
   [
    "C192",
    "x_vars[10,Router]"
   ],
   [
    "C193",
    "x_vars[10,Power Bank]"
   ],
   [
    "C194",
    "x_vars[10,Memory Card]"
   ],
   [
    "C195",
    "x_vars[10,USB Flash Drive]"
   ],
   [
    "C196",
    "x_vars[10,Smart Home Hub]"
   ],
   [
    "C197",
    "x_vars[10,Gaming Console]"
   ],
   [
    "C198",
    "x_vars[10,Fitness Tracker]"
   ],
   [
    "C199",
    "x_vars[10,E-Reader]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Smartphone]",
   "x_1_0": "x_vars[2,Smartphone]",
   "x_2_0": "x_vars[3,Smartphone]",
   "x_3_0": "x_vars[4,Smartphone]",
   "x_4_0": "x_vars[5,Smartphone]",
   "x_5_0": "x_vars[6,Smartphone]",
   "x_6_0": "x_vars[7,Smartphone]",
   "x_7_0": "x_vars[8,Smartphone]",
   "x_8_0": "x_vars[9,Smartphone]",
   "x_9_0": "x_vars[10,Smartphone]",
   "x_0_1": "x_vars[1,Laptop]",
   "x_1_1": "x_vars[2,Laptop]",
   "x_2_1": "x_vars[3,Laptop]",
   "x_3_1": "x_vars[4,Laptop]",
   "x_4_1": "x_vars[5,Laptop]",
   "x_5_1": "x_vars[6,Laptop]",
   "x_6_1": "x_vars[7,Laptop]",
   "x_7_1": "x_vars[8,Laptop]",
   "x_8_1": "x_vars[9,Laptop]",
   "x_9_1": "x_vars[10,Laptop]",
   "x_0_2": "x_vars[1,Headphones]",
   "x_1_2": "x_vars[2,Headphones]",
   "x_2_2": "x_vars[3,Headphones]",
   "x_3_2": "x_vars[4,Headphones]",
   "x_4_2": "x_vars[5,Headphones]",
   "x_5_2": "x_vars[6,Headphones]",
   "x_6_2": "x_vars[7,Headphones]",
   "x_7_2": "x_vars[8,Headphones]",
   "x_8_2": "x_vars[9,Headphones]",
   "x_9_2": "x_vars[10,Headphones]",
   "x_0_3": "x_vars[1,Camera]",
   "x_1_3": "x_vars[2,Camera]",
   "x_2_3": "x_vars[3,Camera]",
   "x_3_3": "x_vars[4,Camera]",
   "x_4_3": "x_vars[5,Camera]",
   "x_5_3": "x_vars[6,Camera]",
   "x_6_3": "x_vars[7,Camera]",
   "x_7_3": "x_vars[8,Camera]",
   "x_8_3": "x_vars[9,Camera]",
   "x_9_3": "x_vars[10,Camera]",
   "x_0_4": "x_vars[1,Smartwatch]",
   "x_1_4": "x_vars[2,Smartwatch]",
   "x_2_4": "x_vars[3,Smartwatch]",
   "x_3_4": "x_vars[4,Smartwatch]",
   "x_4_4": "x_vars[5,Smartwatch]",
   "x_5_4": "x_vars[6,Smartwatch]",
   "x_6_4": "x_vars[7,Smartwatch]",
   "x_7_4": "x_vars[8,Smartwatch]",
   "x_8_4": "x_vars[9,Smartwatch]",
   "x_9_4": "x_vars[10,Smartwatch]",
   "x_0_5": "x_vars[1,Tablet]",
   "x_1_5": "x_vars[2,Tablet]",
   "x_2_5": "x_vars[3,Tablet]",
   "x_3_5": "x_vars[4,Tablet]",
   "x_4_5": "x_vars[5,Tablet]",
   "x_5_5": "x_vars[6,Tablet]",
   "x_6_5": "x_vars[7,Tablet]",
   "x_7_5": "x_vars[8,Tablet]",
   "x_8_5": "x_vars[9,Tablet]",
   "x_9_5": "x_vars[10,Tablet]",
   "x_0_6": "x_vars[1,Bluetooth Speaker]",
   "x_1_6": "x_vars[2,Bluetooth Speaker]",
   "x_2_6": "x_vars[3,Bluetooth Speaker]",
   "x_3_6": "x_vars[4,Bluetooth Speaker]",
   "x_4_6": "x_vars[5,Bluetooth Speaker]",
   "x_5_6": "x_vars[6,Bluetooth Speaker]",
   "x_6_6": "x_vars[7,Bluetooth Speaker]",
   "x_7_6": "x_vars[8,Bluetooth Speaker]",
   "x_8_6": "x_vars[9,Bluetooth Speaker]",
   "x_9_6": "x_vars[10,Bluetooth Speaker]",
   "x_0_7": "x_vars[1,Keyboard]",
   "x_1_7": "x_vars[2,Keyboard]",
   "x_2_7": "x_vars[3,Keyboard]",
   "x_3_7": "x_vars[4,Keyboard]",
   "x_4_7": "x_vars[5,Keyboard]",
   "x_5_7": "x_vars[6,Keyboard]",
   "x_6_7": "x_vars[7,Keyboard]",
   "x_7_7": "x_vars[8,Keyboard]",
   "x_8_7": "x_vars[9,Keyboard]",
   "x_9_7": "x_vars[10,Keyboard]",
   "x_0_8": "x_vars[1,Mouse]",
   "x_1_8": "x_vars[2,Mouse]",
   "x_2_8": "x_vars[3,Mouse]",
   "x_3_8": "x_vars[4,Mouse]",
   "x_4_8": "x_vars[5,Mouse]",
   "x_5_8": "x_vars[6,Mouse]",
   "x_6_8": "x_vars[7,Mouse]",
   "x_7_8": "x_vars[8,Mouse]",
   "x_8_8": "x_vars[9,Mouse]",
   "x_9_8": "x_vars[10,Mouse]",
   "x_0_9": "x_vars[1,Monitor]",
   "x_1_9": "x_vars[2,Monitor]",
   "x_2_9": "x_vars[3,Monitor]",
   "x_3_9": "x_vars[4,Monitor]",
   "x_4_9": "x_vars[5,Monitor]",
   "x_5_9": "x_vars[6,Monitor]",
   "x_6_9": "x_vars[7,Monitor]",
   "x_7_9": "x_vars[8,Monitor]",
   "x_8_9": "x_vars[9,Monitor]",
   "x_9_9": "x_vars[10,Monitor]",
   "x_0_10": "x_vars[1,Printer]",
   "x_1_10": "x_vars[2,Printer]",
   "x_2_10": "x_vars[3,Printer]",
   "x_3_10": "x_vars[4,Printer]",
   "x_4_10": "x_vars[5,Printer]",
   "x_5_10": "x_vars[6,Printer]",
   "x_6_10": "x_vars[7,Printer]",
   "x_7_10": "x_vars[8,Printer]",
   "x_8_10": "x_vars[9,Printer]",
   "x_9_10": "x_vars[10,Printer]",
   "x_0_11": "x_vars[1,External Hard Drive]",
   "x_1_11": "x_vars[2,External Hard Drive]",
   "x_2_11": "x_vars[3,External Hard Drive]",
   "x_3_11": "x_vars[4,External Hard Drive]",
   "x_4_11": "x_vars[5,External Hard Drive]",
   "x_5_11": "x_vars[6,External Hard Drive]",
   "x_6_11": "x_vars[7,External Hard Drive]",
   "x_7_11": "x_vars[8,External Hard Drive]",
   "x_8_11": "x_vars[9,External Hard Drive]",
   "x_9_11": "x_vars[10,External Hard Drive]",
   "x_0_12": "x_vars[1,Router]",
   "x_1_12": "x_vars[2,Router]",
   "x_2_12": "x_vars[3,Router]",
   "x_3_12": "x_vars[4,Router]",
   "x_4_12": "x_vars[5,Router]",
   "x_5_12": "x_vars[6,Router]",
   "x_6_12": "x_vars[7,Router]",
   "x_7_12": "x_vars[8,Router]",
   "x_8_12": "x_vars[9,Router]",
   "x_9_12": "x_vars[10,Router]",
   "x_0_13": "x_vars[1,Power Bank]",
   "x_1_13": "x_vars[2,Power Bank]",
   "x_2_13": "x_vars[3,Power Bank]",
   "x_3_13": "x_vars[4,Power Bank]",
   "x_4_13": "x_vars[5,Power Bank]",
   "x_5_13": "x_vars[6,Power Bank]",
   "x_6_13": "x_vars[7,Power Bank]",
   "x_7_13": "x_vars[8,Power Bank]",
   "x_8_13": "x_vars[9,Power Bank]",
   "x_9_13": "x_vars[10,Power Bank]",
   "x_0_14": "x_vars[1,Memory Card]",
   "x_1_14": "x_vars[2,Memory Card]",
   "x_2_14": "x_vars[3,Memory Card]",
   "x_3_14": "x_vars[4,Memory Card]",
   "x_4_14": "x_vars[5,Memory Card]",
   "x_5_14": "x_vars[6,Memory Card]",
   "x_6_14": "x_vars[7,Memory Card]",
   "x_7_14": "x_vars[8,Memory Card]",
   "x_8_14": "x_vars[9,Memory Card]",
   "x_9_14": "x_vars[10,Memory Card]",
   "x_0_15": "x_vars[1,USB Flash Drive]",
   "x_1_15": "x_vars[2,USB Flash Drive]",
   "x_2_15": "x_vars[3,USB Flash Drive]",
   "x_3_15": "x_vars[4,USB Flash Drive]",
   "x_4_15": "x_vars[5,USB Flash Drive]",
   "x_5_15": "x_vars[6,USB Flash Drive]",
   "x_6_15": "x_vars[7,USB Flash Drive]",
   "x_7_15": "x_vars[8,USB Flash Drive]",
   "x_8_15": "x_vars[9,USB Flash Drive]",
   "x_9_15": "x_vars[10,USB Flash Drive]",
   "x_0_16": "x_vars[1,Smart Home Hub]",
   "x_1_16": "x_vars[2,Smart Home Hub]",
   "x_2_16": "x_vars[3,Smart Home Hub]",
   "x_3_16": "x_vars[4,Smart Home Hub]",
   "x_4_16": "x_vars[5,Smart Home Hub]",
   "x_5_16": "x_vars[6,Smart Home Hub]",
   "x_6_16": "x_vars[7,Smart Home Hub]",
   "x_7_16": "x_vars[8,Smart Home Hub]",
   "x_8_16": "x_vars[9,Smart Home Hub]",
   "x_9_16": "x_vars[10,Smart Home Hub]",
   "x_0_17": "x_vars[1,Gaming Console]",
   "x_1_17": "x_vars[2,Gaming Console]",
   "x_2_17": "x_vars[3,Gaming Console]",
   "x_3_17": "x_vars[4,Gaming Console]",
   "x_4_17": "x_vars[5,Gaming Console]",
   "x_5_17": "x_vars[6,Gaming Console]",
   "x_6_17": "x_vars[7,Gaming Console]",
   "x_7_17": "x_vars[8,Gaming Console]",
   "x_8_17": "x_vars[9,Gaming Console]",
   "x_9_17": "x_vars[10,Gaming Console]",
   "x_0_18": "x_vars[1,Fitness Tracker]",
   "x_1_18": "x_vars[2,Fitness Tracker]",
   "x_2_18": "x_vars[3,Fitness Tracker]",
   "x_3_18": "x_vars[4,Fitness Tracker]",
   "x_4_18": "x_vars[5,Fitness Tracker]",
   "x_5_18": "x_vars[6,Fitness Tracker]",
   "x_6_18": "x_vars[7,Fitness Tracker]",
   "x_7_18": "x_vars[8,Fitness Tracker]",
   "x_8_18": "x_vars[9,Fitness Tracker]",
   "x_9_18": "x_vars[10,Fitness Tracker]",
   "x_0_19": "x_vars[1,E-Reader]",
   "x_1_19": "x_vars[2,E-Reader]",
   "x_2_19": "x_vars[3,E-Reader]",
   "x_3_19": "x_vars[4,E-Reader]",
   "x_4_19": "x_vars[5,E-Reader]",
   "x_5_19": "x_vars[6,E-Reader]",
   "x_6_19": "x_vars[7,E-Reader]",
   "x_7_19": "x_vars[8,E-Reader]",
   "x_8_19": "x_vars[9,E-Reader]",
   "x_9_19": "x_vars[10,E-Reader]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-051": {
  "source_code_sha256": "221c0f9612edfa3df95a800b08de7f269582de0e3edb94e94278c3c84319eff1",
  "original_solution_sha256": "d409487192057476c48b668d5d60ef7ac7a9f285eaab11a48b88c135ec76b999",
  "bindings": [
   [
    "C0",
    "x_vars[1,Espresso Beans]"
   ],
   [
    "C1",
    "x_vars[1,Colombian Roast]"
   ],
   [
    "C2",
    "x_vars[1,Arabica Blend]"
   ],
   [
    "C3",
    "x_vars[1,French Roast]"
   ],
   [
    "C4",
    "x_vars[1,Italian Roast]"
   ],
   [
    "C5",
    "x_vars[1,House Blend]"
   ],
   [
    "C6",
    "x_vars[1,Sumatra Coffee]"
   ],
   [
    "C7",
    "x_vars[1,Mocha Java]"
   ],
   [
    "C8",
    "x_vars[1,Hazelnut Flavor]"
   ],
   [
    "C9",
    "x_vars[1,Caramel Blend]"
   ],
   [
    "C10",
    "x_vars[1,Vanilla Flavor]"
   ],
   [
    "C11",
    "x_vars[1,Cappuccino Mix]"
   ],
   [
    "C12",
    "x_vars[1,Pumpkin Spice]"
   ],
   [
    "C13",
    "x_vars[1,Decaf Roast]"
   ],
   [
    "C14",
    "x_vars[1,Organic Roast]"
   ],
   [
    "C15",
    "x_vars[1,Cold Brew]"
   ],
   [
    "C16",
    "x_vars[1,Peruvian Blend]"
   ],
   [
    "C17",
    "x_vars[1,Kenyan AA]"
   ],
   [
    "C18",
    "x_vars[2,Espresso Beans]"
   ],
   [
    "C19",
    "x_vars[2,Colombian Roast]"
   ],
   [
    "C20",
    "x_vars[2,Arabica Blend]"
   ],
   [
    "C21",
    "x_vars[2,French Roast]"
   ],
   [
    "C22",
    "x_vars[2,Italian Roast]"
   ],
   [
    "C23",
    "x_vars[2,House Blend]"
   ],
   [
    "C24",
    "x_vars[2,Sumatra Coffee]"
   ],
   [
    "C25",
    "x_vars[2,Mocha Java]"
   ],
   [
    "C26",
    "x_vars[2,Hazelnut Flavor]"
   ],
   [
    "C27",
    "x_vars[2,Caramel Blend]"
   ],
   [
    "C28",
    "x_vars[2,Vanilla Flavor]"
   ],
   [
    "C29",
    "x_vars[2,Cappuccino Mix]"
   ],
   [
    "C30",
    "x_vars[2,Pumpkin Spice]"
   ],
   [
    "C31",
    "x_vars[2,Decaf Roast]"
   ],
   [
    "C32",
    "x_vars[2,Organic Roast]"
   ],
   [
    "C33",
    "x_vars[2,Cold Brew]"
   ],
   [
    "C34",
    "x_vars[2,Peruvian Blend]"
   ],
   [
    "C35",
    "x_vars[2,Kenyan AA]"
   ],
   [
    "C36",
    "x_vars[3,Espresso Beans]"
   ],
   [
    "C37",
    "x_vars[3,Colombian Roast]"
   ],
   [
    "C38",
    "x_vars[3,Arabica Blend]"
   ],
   [
    "C39",
    "x_vars[3,French Roast]"
   ],
   [
    "C40",
    "x_vars[3,Italian Roast]"
   ],
   [
    "C41",
    "x_vars[3,House Blend]"
   ],
   [
    "C42",
    "x_vars[3,Sumatra Coffee]"
   ],
   [
    "C43",
    "x_vars[3,Mocha Java]"
   ],
   [
    "C44",
    "x_vars[3,Hazelnut Flavor]"
   ],
   [
    "C45",
    "x_vars[3,Caramel Blend]"
   ],
   [
    "C46",
    "x_vars[3,Vanilla Flavor]"
   ],
   [
    "C47",
    "x_vars[3,Cappuccino Mix]"
   ],
   [
    "C48",
    "x_vars[3,Pumpkin Spice]"
   ],
   [
    "C49",
    "x_vars[3,Decaf Roast]"
   ],
   [
    "C50",
    "x_vars[3,Organic Roast]"
   ],
   [
    "C51",
    "x_vars[3,Cold Brew]"
   ],
   [
    "C52",
    "x_vars[3,Peruvian Blend]"
   ],
   [
    "C53",
    "x_vars[3,Kenyan AA]"
   ],
   [
    "C54",
    "x_vars[4,Espresso Beans]"
   ],
   [
    "C55",
    "x_vars[4,Colombian Roast]"
   ],
   [
    "C56",
    "x_vars[4,Arabica Blend]"
   ],
   [
    "C57",
    "x_vars[4,French Roast]"
   ],
   [
    "C58",
    "x_vars[4,Italian Roast]"
   ],
   [
    "C59",
    "x_vars[4,House Blend]"
   ],
   [
    "C60",
    "x_vars[4,Sumatra Coffee]"
   ],
   [
    "C61",
    "x_vars[4,Mocha Java]"
   ],
   [
    "C62",
    "x_vars[4,Hazelnut Flavor]"
   ],
   [
    "C63",
    "x_vars[4,Caramel Blend]"
   ],
   [
    "C64",
    "x_vars[4,Vanilla Flavor]"
   ],
   [
    "C65",
    "x_vars[4,Cappuccino Mix]"
   ],
   [
    "C66",
    "x_vars[4,Pumpkin Spice]"
   ],
   [
    "C67",
    "x_vars[4,Decaf Roast]"
   ],
   [
    "C68",
    "x_vars[4,Organic Roast]"
   ],
   [
    "C69",
    "x_vars[4,Cold Brew]"
   ],
   [
    "C70",
    "x_vars[4,Peruvian Blend]"
   ],
   [
    "C71",
    "x_vars[4,Kenyan AA]"
   ],
   [
    "C72",
    "x_vars[5,Espresso Beans]"
   ],
   [
    "C73",
    "x_vars[5,Colombian Roast]"
   ],
   [
    "C74",
    "x_vars[5,Arabica Blend]"
   ],
   [
    "C75",
    "x_vars[5,French Roast]"
   ],
   [
    "C76",
    "x_vars[5,Italian Roast]"
   ],
   [
    "C77",
    "x_vars[5,House Blend]"
   ],
   [
    "C78",
    "x_vars[5,Sumatra Coffee]"
   ],
   [
    "C79",
    "x_vars[5,Mocha Java]"
   ],
   [
    "C80",
    "x_vars[5,Hazelnut Flavor]"
   ],
   [
    "C81",
    "x_vars[5,Caramel Blend]"
   ],
   [
    "C82",
    "x_vars[5,Vanilla Flavor]"
   ],
   [
    "C83",
    "x_vars[5,Cappuccino Mix]"
   ],
   [
    "C84",
    "x_vars[5,Pumpkin Spice]"
   ],
   [
    "C85",
    "x_vars[5,Decaf Roast]"
   ],
   [
    "C86",
    "x_vars[5,Organic Roast]"
   ],
   [
    "C87",
    "x_vars[5,Cold Brew]"
   ],
   [
    "C88",
    "x_vars[5,Peruvian Blend]"
   ],
   [
    "C89",
    "x_vars[5,Kenyan AA]"
   ],
   [
    "C90",
    "x_vars[6,Espresso Beans]"
   ],
   [
    "C91",
    "x_vars[6,Colombian Roast]"
   ],
   [
    "C92",
    "x_vars[6,Arabica Blend]"
   ],
   [
    "C93",
    "x_vars[6,French Roast]"
   ],
   [
    "C94",
    "x_vars[6,Italian Roast]"
   ],
   [
    "C95",
    "x_vars[6,House Blend]"
   ],
   [
    "C96",
    "x_vars[6,Sumatra Coffee]"
   ],
   [
    "C97",
    "x_vars[6,Mocha Java]"
   ],
   [
    "C98",
    "x_vars[6,Hazelnut Flavor]"
   ],
   [
    "C99",
    "x_vars[6,Caramel Blend]"
   ],
   [
    "C100",
    "x_vars[6,Vanilla Flavor]"
   ],
   [
    "C101",
    "x_vars[6,Cappuccino Mix]"
   ],
   [
    "C102",
    "x_vars[6,Pumpkin Spice]"
   ],
   [
    "C103",
    "x_vars[6,Decaf Roast]"
   ],
   [
    "C104",
    "x_vars[6,Organic Roast]"
   ],
   [
    "C105",
    "x_vars[6,Cold Brew]"
   ],
   [
    "C106",
    "x_vars[6,Peruvian Blend]"
   ],
   [
    "C107",
    "x_vars[6,Kenyan AA]"
   ],
   [
    "C108",
    "x_vars[7,Espresso Beans]"
   ],
   [
    "C109",
    "x_vars[7,Colombian Roast]"
   ],
   [
    "C110",
    "x_vars[7,Arabica Blend]"
   ],
   [
    "C111",
    "x_vars[7,French Roast]"
   ],
   [
    "C112",
    "x_vars[7,Italian Roast]"
   ],
   [
    "C113",
    "x_vars[7,House Blend]"
   ],
   [
    "C114",
    "x_vars[7,Sumatra Coffee]"
   ],
   [
    "C115",
    "x_vars[7,Mocha Java]"
   ],
   [
    "C116",
    "x_vars[7,Hazelnut Flavor]"
   ],
   [
    "C117",
    "x_vars[7,Caramel Blend]"
   ],
   [
    "C118",
    "x_vars[7,Vanilla Flavor]"
   ],
   [
    "C119",
    "x_vars[7,Cappuccino Mix]"
   ],
   [
    "C120",
    "x_vars[7,Pumpkin Spice]"
   ],
   [
    "C121",
    "x_vars[7,Decaf Roast]"
   ],
   [
    "C122",
    "x_vars[7,Organic Roast]"
   ],
   [
    "C123",
    "x_vars[7,Cold Brew]"
   ],
   [
    "C124",
    "x_vars[7,Peruvian Blend]"
   ],
   [
    "C125",
    "x_vars[7,Kenyan AA]"
   ],
   [
    "C126",
    "x_vars[8,Espresso Beans]"
   ],
   [
    "C127",
    "x_vars[8,Colombian Roast]"
   ],
   [
    "C128",
    "x_vars[8,Arabica Blend]"
   ],
   [
    "C129",
    "x_vars[8,French Roast]"
   ],
   [
    "C130",
    "x_vars[8,Italian Roast]"
   ],
   [
    "C131",
    "x_vars[8,House Blend]"
   ],
   [
    "C132",
    "x_vars[8,Sumatra Coffee]"
   ],
   [
    "C133",
    "x_vars[8,Mocha Java]"
   ],
   [
    "C134",
    "x_vars[8,Hazelnut Flavor]"
   ],
   [
    "C135",
    "x_vars[8,Caramel Blend]"
   ],
   [
    "C136",
    "x_vars[8,Vanilla Flavor]"
   ],
   [
    "C137",
    "x_vars[8,Cappuccino Mix]"
   ],
   [
    "C138",
    "x_vars[8,Pumpkin Spice]"
   ],
   [
    "C139",
    "x_vars[8,Decaf Roast]"
   ],
   [
    "C140",
    "x_vars[8,Organic Roast]"
   ],
   [
    "C141",
    "x_vars[8,Cold Brew]"
   ],
   [
    "C142",
    "x_vars[8,Peruvian Blend]"
   ],
   [
    "C143",
    "x_vars[8,Kenyan AA]"
   ],
   [
    "C144",
    "x_vars[9,Espresso Beans]"
   ],
   [
    "C145",
    "x_vars[9,Colombian Roast]"
   ],
   [
    "C146",
    "x_vars[9,Arabica Blend]"
   ],
   [
    "C147",
    "x_vars[9,French Roast]"
   ],
   [
    "C148",
    "x_vars[9,Italian Roast]"
   ],
   [
    "C149",
    "x_vars[9,House Blend]"
   ],
   [
    "C150",
    "x_vars[9,Sumatra Coffee]"
   ],
   [
    "C151",
    "x_vars[9,Mocha Java]"
   ],
   [
    "C152",
    "x_vars[9,Hazelnut Flavor]"
   ],
   [
    "C153",
    "x_vars[9,Caramel Blend]"
   ],
   [
    "C154",
    "x_vars[9,Vanilla Flavor]"
   ],
   [
    "C155",
    "x_vars[9,Cappuccino Mix]"
   ],
   [
    "C156",
    "x_vars[9,Pumpkin Spice]"
   ],
   [
    "C157",
    "x_vars[9,Decaf Roast]"
   ],
   [
    "C158",
    "x_vars[9,Organic Roast]"
   ],
   [
    "C159",
    "x_vars[9,Cold Brew]"
   ],
   [
    "C160",
    "x_vars[9,Peruvian Blend]"
   ],
   [
    "C161",
    "x_vars[9,Kenyan AA]"
   ],
   [
    "C162",
    "x_vars[10,Espresso Beans]"
   ],
   [
    "C163",
    "x_vars[10,Colombian Roast]"
   ],
   [
    "C164",
    "x_vars[10,Arabica Blend]"
   ],
   [
    "C165",
    "x_vars[10,French Roast]"
   ],
   [
    "C166",
    "x_vars[10,Italian Roast]"
   ],
   [
    "C167",
    "x_vars[10,House Blend]"
   ],
   [
    "C168",
    "x_vars[10,Sumatra Coffee]"
   ],
   [
    "C169",
    "x_vars[10,Mocha Java]"
   ],
   [
    "C170",
    "x_vars[10,Hazelnut Flavor]"
   ],
   [
    "C171",
    "x_vars[10,Caramel Blend]"
   ],
   [
    "C172",
    "x_vars[10,Vanilla Flavor]"
   ],
   [
    "C173",
    "x_vars[10,Cappuccino Mix]"
   ],
   [
    "C174",
    "x_vars[10,Pumpkin Spice]"
   ],
   [
    "C175",
    "x_vars[10,Decaf Roast]"
   ],
   [
    "C176",
    "x_vars[10,Organic Roast]"
   ],
   [
    "C177",
    "x_vars[10,Cold Brew]"
   ],
   [
    "C178",
    "x_vars[10,Peruvian Blend]"
   ],
   [
    "C179",
    "x_vars[10,Kenyan AA]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Espresso Beans]",
   "x_0_1": "x_vars[2,Espresso Beans]",
   "x_0_2": "x_vars[3,Espresso Beans]",
   "x_0_3": "x_vars[4,Espresso Beans]",
   "x_0_4": "x_vars[5,Espresso Beans]",
   "x_0_5": "x_vars[6,Espresso Beans]",
   "x_0_6": "x_vars[7,Espresso Beans]",
   "x_0_7": "x_vars[8,Espresso Beans]",
   "x_0_8": "x_vars[9,Espresso Beans]",
   "x_0_9": "x_vars[10,Espresso Beans]",
   "x_1_0": "x_vars[1,Colombian Roast]",
   "x_1_1": "x_vars[2,Colombian Roast]",
   "x_1_2": "x_vars[3,Colombian Roast]",
   "x_1_3": "x_vars[4,Colombian Roast]",
   "x_1_4": "x_vars[5,Colombian Roast]",
   "x_1_5": "x_vars[6,Colombian Roast]",
   "x_1_6": "x_vars[7,Colombian Roast]",
   "x_1_7": "x_vars[8,Colombian Roast]",
   "x_1_8": "x_vars[9,Colombian Roast]",
   "x_1_9": "x_vars[10,Colombian Roast]",
   "x_2_0": "x_vars[1,Arabica Blend]",
   "x_2_1": "x_vars[2,Arabica Blend]",
   "x_2_2": "x_vars[3,Arabica Blend]",
   "x_2_3": "x_vars[4,Arabica Blend]",
   "x_2_4": "x_vars[5,Arabica Blend]",
   "x_2_5": "x_vars[6,Arabica Blend]",
   "x_2_6": "x_vars[7,Arabica Blend]",
   "x_2_7": "x_vars[8,Arabica Blend]",
   "x_2_8": "x_vars[9,Arabica Blend]",
   "x_2_9": "x_vars[10,Arabica Blend]",
   "x_3_0": "x_vars[1,French Roast]",
   "x_3_1": "x_vars[2,French Roast]",
   "x_3_2": "x_vars[3,French Roast]",
   "x_3_3": "x_vars[4,French Roast]",
   "x_3_4": "x_vars[5,French Roast]",
   "x_3_5": "x_vars[6,French Roast]",
   "x_3_6": "x_vars[7,French Roast]",
   "x_3_7": "x_vars[8,French Roast]",
   "x_3_8": "x_vars[9,French Roast]",
   "x_3_9": "x_vars[10,French Roast]",
   "x_4_0": "x_vars[1,Italian Roast]",
   "x_4_1": "x_vars[2,Italian Roast]",
   "x_4_2": "x_vars[3,Italian Roast]",
   "x_4_3": "x_vars[4,Italian Roast]",
   "x_4_4": "x_vars[5,Italian Roast]",
   "x_4_5": "x_vars[6,Italian Roast]",
   "x_4_6": "x_vars[7,Italian Roast]",
   "x_4_7": "x_vars[8,Italian Roast]",
   "x_4_8": "x_vars[9,Italian Roast]",
   "x_4_9": "x_vars[10,Italian Roast]",
   "x_5_0": "x_vars[1,House Blend]",
   "x_5_1": "x_vars[2,House Blend]",
   "x_5_2": "x_vars[3,House Blend]",
   "x_5_3": "x_vars[4,House Blend]",
   "x_5_4": "x_vars[5,House Blend]",
   "x_5_5": "x_vars[6,House Blend]",
   "x_5_6": "x_vars[7,House Blend]",
   "x_5_7": "x_vars[8,House Blend]",
   "x_5_8": "x_vars[9,House Blend]",
   "x_5_9": "x_vars[10,House Blend]",
   "x_6_0": "x_vars[1,Sumatra Coffee]",
   "x_6_1": "x_vars[2,Sumatra Coffee]",
   "x_6_2": "x_vars[3,Sumatra Coffee]",
   "x_6_3": "x_vars[4,Sumatra Coffee]",
   "x_6_4": "x_vars[5,Sumatra Coffee]",
   "x_6_5": "x_vars[6,Sumatra Coffee]",
   "x_6_6": "x_vars[7,Sumatra Coffee]",
   "x_6_7": "x_vars[8,Sumatra Coffee]",
   "x_6_8": "x_vars[9,Sumatra Coffee]",
   "x_6_9": "x_vars[10,Sumatra Coffee]",
   "x_7_0": "x_vars[1,Mocha Java]",
   "x_7_1": "x_vars[2,Mocha Java]",
   "x_7_2": "x_vars[3,Mocha Java]",
   "x_7_3": "x_vars[4,Mocha Java]",
   "x_7_4": "x_vars[5,Mocha Java]",
   "x_7_5": "x_vars[6,Mocha Java]",
   "x_7_6": "x_vars[7,Mocha Java]",
   "x_7_7": "x_vars[8,Mocha Java]",
   "x_7_8": "x_vars[9,Mocha Java]",
   "x_7_9": "x_vars[10,Mocha Java]",
   "x_8_0": "x_vars[1,Hazelnut Flavor]",
   "x_8_1": "x_vars[2,Hazelnut Flavor]",
   "x_8_2": "x_vars[3,Hazelnut Flavor]",
   "x_8_3": "x_vars[4,Hazelnut Flavor]",
   "x_8_4": "x_vars[5,Hazelnut Flavor]",
   "x_8_5": "x_vars[6,Hazelnut Flavor]",
   "x_8_6": "x_vars[7,Hazelnut Flavor]",
   "x_8_7": "x_vars[8,Hazelnut Flavor]",
   "x_8_8": "x_vars[9,Hazelnut Flavor]",
   "x_8_9": "x_vars[10,Hazelnut Flavor]",
   "x_9_0": "x_vars[1,Caramel Blend]",
   "x_9_1": "x_vars[2,Caramel Blend]",
   "x_9_2": "x_vars[3,Caramel Blend]",
   "x_9_3": "x_vars[4,Caramel Blend]",
   "x_9_4": "x_vars[5,Caramel Blend]",
   "x_9_5": "x_vars[6,Caramel Blend]",
   "x_9_6": "x_vars[7,Caramel Blend]",
   "x_9_7": "x_vars[8,Caramel Blend]",
   "x_9_8": "x_vars[9,Caramel Blend]",
   "x_9_9": "x_vars[10,Caramel Blend]",
   "x_10_0": "x_vars[1,Vanilla Flavor]",
   "x_10_1": "x_vars[2,Vanilla Flavor]",
   "x_10_2": "x_vars[3,Vanilla Flavor]",
   "x_10_3": "x_vars[4,Vanilla Flavor]",
   "x_10_4": "x_vars[5,Vanilla Flavor]",
   "x_10_5": "x_vars[6,Vanilla Flavor]",
   "x_10_6": "x_vars[7,Vanilla Flavor]",
   "x_10_7": "x_vars[8,Vanilla Flavor]",
   "x_10_8": "x_vars[9,Vanilla Flavor]",
   "x_10_9": "x_vars[10,Vanilla Flavor]",
   "x_11_0": "x_vars[1,Cappuccino Mix]",
   "x_11_1": "x_vars[2,Cappuccino Mix]",
   "x_11_2": "x_vars[3,Cappuccino Mix]",
   "x_11_3": "x_vars[4,Cappuccino Mix]",
   "x_11_4": "x_vars[5,Cappuccino Mix]",
   "x_11_5": "x_vars[6,Cappuccino Mix]",
   "x_11_6": "x_vars[7,Cappuccino Mix]",
   "x_11_7": "x_vars[8,Cappuccino Mix]",
   "x_11_8": "x_vars[9,Cappuccino Mix]",
   "x_11_9": "x_vars[10,Cappuccino Mix]",
   "x_12_0": "x_vars[1,Pumpkin Spice]",
   "x_12_1": "x_vars[2,Pumpkin Spice]",
   "x_12_2": "x_vars[3,Pumpkin Spice]",
   "x_12_3": "x_vars[4,Pumpkin Spice]",
   "x_12_4": "x_vars[5,Pumpkin Spice]",
   "x_12_5": "x_vars[6,Pumpkin Spice]",
   "x_12_6": "x_vars[7,Pumpkin Spice]",
   "x_12_7": "x_vars[8,Pumpkin Spice]",
   "x_12_8": "x_vars[9,Pumpkin Spice]",
   "x_12_9": "x_vars[10,Pumpkin Spice]",
   "x_13_0": "x_vars[1,Decaf Roast]",
   "x_13_1": "x_vars[2,Decaf Roast]",
   "x_13_2": "x_vars[3,Decaf Roast]",
   "x_13_3": "x_vars[4,Decaf Roast]",
   "x_13_4": "x_vars[5,Decaf Roast]",
   "x_13_5": "x_vars[6,Decaf Roast]",
   "x_13_6": "x_vars[7,Decaf Roast]",
   "x_13_7": "x_vars[8,Decaf Roast]",
   "x_13_8": "x_vars[9,Decaf Roast]",
   "x_13_9": "x_vars[10,Decaf Roast]",
   "x_14_0": "x_vars[1,Organic Roast]",
   "x_14_1": "x_vars[2,Organic Roast]",
   "x_14_2": "x_vars[3,Organic Roast]",
   "x_14_3": "x_vars[4,Organic Roast]",
   "x_14_4": "x_vars[5,Organic Roast]",
   "x_14_5": "x_vars[6,Organic Roast]",
   "x_14_6": "x_vars[7,Organic Roast]",
   "x_14_7": "x_vars[8,Organic Roast]",
   "x_14_8": "x_vars[9,Organic Roast]",
   "x_14_9": "x_vars[10,Organic Roast]",
   "x_15_0": "x_vars[1,Cold Brew]",
   "x_15_1": "x_vars[2,Cold Brew]",
   "x_15_2": "x_vars[3,Cold Brew]",
   "x_15_3": "x_vars[4,Cold Brew]",
   "x_15_4": "x_vars[5,Cold Brew]",
   "x_15_5": "x_vars[6,Cold Brew]",
   "x_15_6": "x_vars[7,Cold Brew]",
   "x_15_7": "x_vars[8,Cold Brew]",
   "x_15_8": "x_vars[9,Cold Brew]",
   "x_15_9": "x_vars[10,Cold Brew]",
   "x_16_0": "x_vars[1,Peruvian Blend]",
   "x_16_1": "x_vars[2,Peruvian Blend]",
   "x_16_2": "x_vars[3,Peruvian Blend]",
   "x_16_3": "x_vars[4,Peruvian Blend]",
   "x_16_4": "x_vars[5,Peruvian Blend]",
   "x_16_5": "x_vars[6,Peruvian Blend]",
   "x_16_6": "x_vars[7,Peruvian Blend]",
   "x_16_7": "x_vars[8,Peruvian Blend]",
   "x_16_8": "x_vars[9,Peruvian Blend]",
   "x_16_9": "x_vars[10,Peruvian Blend]",
   "x_17_0": "x_vars[1,Kenyan AA]",
   "x_17_1": "x_vars[2,Kenyan AA]",
   "x_17_2": "x_vars[3,Kenyan AA]",
   "x_17_3": "x_vars[4,Kenyan AA]",
   "x_17_4": "x_vars[5,Kenyan AA]",
   "x_17_5": "x_vars[6,Kenyan AA]",
   "x_17_6": "x_vars[7,Kenyan AA]",
   "x_17_7": "x_vars[8,Kenyan AA]",
   "x_17_8": "x_vars[9,Kenyan AA]",
   "x_17_9": "x_vars[10,Kenyan AA]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-052": {
  "source_code_sha256": "73a40d2c8cf07dcc730905539446e46ce08503cd9d1e270f6bf0d4833dc50c35",
  "original_solution_sha256": "85dfeb28f05861c79314d7ffa365df698153e8a4c2b779e8a2b4e7a4336e5a16",
  "bindings": [
   [
    "C0",
    "x_vars[1,The Great Gatsby]"
   ],
   [
    "C1",
    "x_vars[1,To Kill a Mockingbird]"
   ],
   [
    "C2",
    "x_vars[1,1984]"
   ],
   [
    "C3",
    "x_vars[1,Pride and Prejudice]"
   ],
   [
    "C4",
    "x_vars[1,The Catcher in the Rye]"
   ],
   [
    "C5",
    "x_vars[1,Moby Dick]"
   ],
   [
    "C6",
    "x_vars[1,Jane Eyre]"
   ],
   [
    "C7",
    "x_vars[1,War and Peace]"
   ],
   [
    "C8",
    "x_vars[1,The Odyssey]"
   ],
   [
    "C9",
    "x_vars[1,Crime and Punishment]"
   ],
   [
    "C10",
    "x_vars[1,The Hobbit]"
   ],
   [
    "C11",
    "x_vars[1,Brave New World]"
   ],
   [
    "C12",
    "x_vars[1,Anna Karenina]"
   ],
   [
    "C13",
    "x_vars[1,Wuthering Heights]"
   ],
   [
    "C14",
    "x_vars[1,The Divine Comedy]"
   ],
   [
    "C15",
    "x_vars[1,The Iliad]"
   ],
   [
    "C16",
    "x_vars[1,Les Misérables]"
   ],
   [
    "C17",
    "x_vars[1,Dracula]"
   ],
   [
    "C18",
    "x_vars[1,Frankenstein]"
   ],
   [
    "C19",
    "x_vars[1,The Brothers Karamazov]"
   ],
   [
    "C20",
    "x_vars[1,Don Quixote]"
   ],
   [
    "C21",
    "x_vars[1,One Hundred Years of Solitude]"
   ],
   [
    "C22",
    "x_vars[1,Ulysses]"
   ],
   [
    "C23",
    "x_vars[1,The Alchemist]"
   ],
   [
    "C24",
    "x_vars[1,Meditations]"
   ],
   [
    "C25",
    "x_vars[2,The Great Gatsby]"
   ],
   [
    "C26",
    "x_vars[2,To Kill a Mockingbird]"
   ],
   [
    "C27",
    "x_vars[2,1984]"
   ],
   [
    "C28",
    "x_vars[2,Pride and Prejudice]"
   ],
   [
    "C29",
    "x_vars[2,The Catcher in the Rye]"
   ],
   [
    "C30",
    "x_vars[2,Moby Dick]"
   ],
   [
    "C31",
    "x_vars[2,Jane Eyre]"
   ],
   [
    "C32",
    "x_vars[2,War and Peace]"
   ],
   [
    "C33",
    "x_vars[2,The Odyssey]"
   ],
   [
    "C34",
    "x_vars[2,Crime and Punishment]"
   ],
   [
    "C35",
    "x_vars[2,The Hobbit]"
   ],
   [
    "C36",
    "x_vars[2,Brave New World]"
   ],
   [
    "C37",
    "x_vars[2,Anna Karenina]"
   ],
   [
    "C38",
    "x_vars[2,Wuthering Heights]"
   ],
   [
    "C39",
    "x_vars[2,The Divine Comedy]"
   ],
   [
    "C40",
    "x_vars[2,The Iliad]"
   ],
   [
    "C41",
    "x_vars[2,Les Misérables]"
   ],
   [
    "C42",
    "x_vars[2,Dracula]"
   ],
   [
    "C43",
    "x_vars[2,Frankenstein]"
   ],
   [
    "C44",
    "x_vars[2,The Brothers Karamazov]"
   ],
   [
    "C45",
    "x_vars[2,Don Quixote]"
   ],
   [
    "C46",
    "x_vars[2,One Hundred Years of Solitude]"
   ],
   [
    "C47",
    "x_vars[2,Ulysses]"
   ],
   [
    "C48",
    "x_vars[2,The Alchemist]"
   ],
   [
    "C49",
    "x_vars[2,Meditations]"
   ],
   [
    "C50",
    "x_vars[3,The Great Gatsby]"
   ],
   [
    "C51",
    "x_vars[3,To Kill a Mockingbird]"
   ],
   [
    "C52",
    "x_vars[3,1984]"
   ],
   [
    "C53",
    "x_vars[3,Pride and Prejudice]"
   ],
   [
    "C54",
    "x_vars[3,The Catcher in the Rye]"
   ],
   [
    "C55",
    "x_vars[3,Moby Dick]"
   ],
   [
    "C56",
    "x_vars[3,Jane Eyre]"
   ],
   [
    "C57",
    "x_vars[3,War and Peace]"
   ],
   [
    "C58",
    "x_vars[3,The Odyssey]"
   ],
   [
    "C59",
    "x_vars[3,Crime and Punishment]"
   ],
   [
    "C60",
    "x_vars[3,The Hobbit]"
   ],
   [
    "C61",
    "x_vars[3,Brave New World]"
   ],
   [
    "C62",
    "x_vars[3,Anna Karenina]"
   ],
   [
    "C63",
    "x_vars[3,Wuthering Heights]"
   ],
   [
    "C64",
    "x_vars[3,The Divine Comedy]"
   ],
   [
    "C65",
    "x_vars[3,The Iliad]"
   ],
   [
    "C66",
    "x_vars[3,Les Misérables]"
   ],
   [
    "C67",
    "x_vars[3,Dracula]"
   ],
   [
    "C68",
    "x_vars[3,Frankenstein]"
   ],
   [
    "C69",
    "x_vars[3,The Brothers Karamazov]"
   ],
   [
    "C70",
    "x_vars[3,Don Quixote]"
   ],
   [
    "C71",
    "x_vars[3,One Hundred Years of Solitude]"
   ],
   [
    "C72",
    "x_vars[3,Ulysses]"
   ],
   [
    "C73",
    "x_vars[3,The Alchemist]"
   ],
   [
    "C74",
    "x_vars[3,Meditations]"
   ],
   [
    "C75",
    "x_vars[4,The Great Gatsby]"
   ],
   [
    "C76",
    "x_vars[4,To Kill a Mockingbird]"
   ],
   [
    "C77",
    "x_vars[4,1984]"
   ],
   [
    "C78",
    "x_vars[4,Pride and Prejudice]"
   ],
   [
    "C79",
    "x_vars[4,The Catcher in the Rye]"
   ],
   [
    "C80",
    "x_vars[4,Moby Dick]"
   ],
   [
    "C81",
    "x_vars[4,Jane Eyre]"
   ],
   [
    "C82",
    "x_vars[4,War and Peace]"
   ],
   [
    "C83",
    "x_vars[4,The Odyssey]"
   ],
   [
    "C84",
    "x_vars[4,Crime and Punishment]"
   ],
   [
    "C85",
    "x_vars[4,The Hobbit]"
   ],
   [
    "C86",
    "x_vars[4,Brave New World]"
   ],
   [
    "C87",
    "x_vars[4,Anna Karenina]"
   ],
   [
    "C88",
    "x_vars[4,Wuthering Heights]"
   ],
   [
    "C89",
    "x_vars[4,The Divine Comedy]"
   ],
   [
    "C90",
    "x_vars[4,The Iliad]"
   ],
   [
    "C91",
    "x_vars[4,Les Misérables]"
   ],
   [
    "C92",
    "x_vars[4,Dracula]"
   ],
   [
    "C93",
    "x_vars[4,Frankenstein]"
   ],
   [
    "C94",
    "x_vars[4,The Brothers Karamazov]"
   ],
   [
    "C95",
    "x_vars[4,Don Quixote]"
   ],
   [
    "C96",
    "x_vars[4,One Hundred Years of Solitude]"
   ],
   [
    "C97",
    "x_vars[4,Ulysses]"
   ],
   [
    "C98",
    "x_vars[4,The Alchemist]"
   ],
   [
    "C99",
    "x_vars[4,Meditations]"
   ],
   [
    "C100",
    "x_vars[5,The Great Gatsby]"
   ],
   [
    "C101",
    "x_vars[5,To Kill a Mockingbird]"
   ],
   [
    "C102",
    "x_vars[5,1984]"
   ],
   [
    "C103",
    "x_vars[5,Pride and Prejudice]"
   ],
   [
    "C104",
    "x_vars[5,The Catcher in the Rye]"
   ],
   [
    "C105",
    "x_vars[5,Moby Dick]"
   ],
   [
    "C106",
    "x_vars[5,Jane Eyre]"
   ],
   [
    "C107",
    "x_vars[5,War and Peace]"
   ],
   [
    "C108",
    "x_vars[5,The Odyssey]"
   ],
   [
    "C109",
    "x_vars[5,Crime and Punishment]"
   ],
   [
    "C110",
    "x_vars[5,The Hobbit]"
   ],
   [
    "C111",
    "x_vars[5,Brave New World]"
   ],
   [
    "C112",
    "x_vars[5,Anna Karenina]"
   ],
   [
    "C113",
    "x_vars[5,Wuthering Heights]"
   ],
   [
    "C114",
    "x_vars[5,The Divine Comedy]"
   ],
   [
    "C115",
    "x_vars[5,The Iliad]"
   ],
   [
    "C116",
    "x_vars[5,Les Misérables]"
   ],
   [
    "C117",
    "x_vars[5,Dracula]"
   ],
   [
    "C118",
    "x_vars[5,Frankenstein]"
   ],
   [
    "C119",
    "x_vars[5,The Brothers Karamazov]"
   ],
   [
    "C120",
    "x_vars[5,Don Quixote]"
   ],
   [
    "C121",
    "x_vars[5,One Hundred Years of Solitude]"
   ],
   [
    "C122",
    "x_vars[5,Ulysses]"
   ],
   [
    "C123",
    "x_vars[5,The Alchemist]"
   ],
   [
    "C124",
    "x_vars[5,Meditations]"
   ],
   [
    "C125",
    "x_vars[6,The Great Gatsby]"
   ],
   [
    "C126",
    "x_vars[6,To Kill a Mockingbird]"
   ],
   [
    "C127",
    "x_vars[6,1984]"
   ],
   [
    "C128",
    "x_vars[6,Pride and Prejudice]"
   ],
   [
    "C129",
    "x_vars[6,The Catcher in the Rye]"
   ],
   [
    "C130",
    "x_vars[6,Moby Dick]"
   ],
   [
    "C131",
    "x_vars[6,Jane Eyre]"
   ],
   [
    "C132",
    "x_vars[6,War and Peace]"
   ],
   [
    "C133",
    "x_vars[6,The Odyssey]"
   ],
   [
    "C134",
    "x_vars[6,Crime and Punishment]"
   ],
   [
    "C135",
    "x_vars[6,The Hobbit]"
   ],
   [
    "C136",
    "x_vars[6,Brave New World]"
   ],
   [
    "C137",
    "x_vars[6,Anna Karenina]"
   ],
   [
    "C138",
    "x_vars[6,Wuthering Heights]"
   ],
   [
    "C139",
    "x_vars[6,The Divine Comedy]"
   ],
   [
    "C140",
    "x_vars[6,The Iliad]"
   ],
   [
    "C141",
    "x_vars[6,Les Misérables]"
   ],
   [
    "C142",
    "x_vars[6,Dracula]"
   ],
   [
    "C143",
    "x_vars[6,Frankenstein]"
   ],
   [
    "C144",
    "x_vars[6,The Brothers Karamazov]"
   ],
   [
    "C145",
    "x_vars[6,Don Quixote]"
   ],
   [
    "C146",
    "x_vars[6,One Hundred Years of Solitude]"
   ],
   [
    "C147",
    "x_vars[6,Ulysses]"
   ],
   [
    "C148",
    "x_vars[6,The Alchemist]"
   ],
   [
    "C149",
    "x_vars[6,Meditations]"
   ],
   [
    "C150",
    "x_vars[7,The Great Gatsby]"
   ],
   [
    "C151",
    "x_vars[7,To Kill a Mockingbird]"
   ],
   [
    "C152",
    "x_vars[7,1984]"
   ],
   [
    "C153",
    "x_vars[7,Pride and Prejudice]"
   ],
   [
    "C154",
    "x_vars[7,The Catcher in the Rye]"
   ],
   [
    "C155",
    "x_vars[7,Moby Dick]"
   ],
   [
    "C156",
    "x_vars[7,Jane Eyre]"
   ],
   [
    "C157",
    "x_vars[7,War and Peace]"
   ],
   [
    "C158",
    "x_vars[7,The Odyssey]"
   ],
   [
    "C159",
    "x_vars[7,Crime and Punishment]"
   ],
   [
    "C160",
    "x_vars[7,The Hobbit]"
   ],
   [
    "C161",
    "x_vars[7,Brave New World]"
   ],
   [
    "C162",
    "x_vars[7,Anna Karenina]"
   ],
   [
    "C163",
    "x_vars[7,Wuthering Heights]"
   ],
   [
    "C164",
    "x_vars[7,The Divine Comedy]"
   ],
   [
    "C165",
    "x_vars[7,The Iliad]"
   ],
   [
    "C166",
    "x_vars[7,Les Misérables]"
   ],
   [
    "C167",
    "x_vars[7,Dracula]"
   ],
   [
    "C168",
    "x_vars[7,Frankenstein]"
   ],
   [
    "C169",
    "x_vars[7,The Brothers Karamazov]"
   ],
   [
    "C170",
    "x_vars[7,Don Quixote]"
   ],
   [
    "C171",
    "x_vars[7,One Hundred Years of Solitude]"
   ],
   [
    "C172",
    "x_vars[7,Ulysses]"
   ],
   [
    "C173",
    "x_vars[7,The Alchemist]"
   ],
   [
    "C174",
    "x_vars[7,Meditations]"
   ],
   [
    "C175",
    "x_vars[8,The Great Gatsby]"
   ],
   [
    "C176",
    "x_vars[8,To Kill a Mockingbird]"
   ],
   [
    "C177",
    "x_vars[8,1984]"
   ],
   [
    "C178",
    "x_vars[8,Pride and Prejudice]"
   ],
   [
    "C179",
    "x_vars[8,The Catcher in the Rye]"
   ],
   [
    "C180",
    "x_vars[8,Moby Dick]"
   ],
   [
    "C181",
    "x_vars[8,Jane Eyre]"
   ],
   [
    "C182",
    "x_vars[8,War and Peace]"
   ],
   [
    "C183",
    "x_vars[8,The Odyssey]"
   ],
   [
    "C184",
    "x_vars[8,Crime and Punishment]"
   ],
   [
    "C185",
    "x_vars[8,The Hobbit]"
   ],
   [
    "C186",
    "x_vars[8,Brave New World]"
   ],
   [
    "C187",
    "x_vars[8,Anna Karenina]"
   ],
   [
    "C188",
    "x_vars[8,Wuthering Heights]"
   ],
   [
    "C189",
    "x_vars[8,The Divine Comedy]"
   ],
   [
    "C190",
    "x_vars[8,The Iliad]"
   ],
   [
    "C191",
    "x_vars[8,Les Misérables]"
   ],
   [
    "C192",
    "x_vars[8,Dracula]"
   ],
   [
    "C193",
    "x_vars[8,Frankenstein]"
   ],
   [
    "C194",
    "x_vars[8,The Brothers Karamazov]"
   ],
   [
    "C195",
    "x_vars[8,Don Quixote]"
   ],
   [
    "C196",
    "x_vars[8,One Hundred Years of Solitude]"
   ],
   [
    "C197",
    "x_vars[8,Ulysses]"
   ],
   [
    "C198",
    "x_vars[8,The Alchemist]"
   ],
   [
    "C199",
    "x_vars[8,Meditations]"
   ],
   [
    "C200",
    "x_vars[9,The Great Gatsby]"
   ],
   [
    "C201",
    "x_vars[9,To Kill a Mockingbird]"
   ],
   [
    "C202",
    "x_vars[9,1984]"
   ],
   [
    "C203",
    "x_vars[9,Pride and Prejudice]"
   ],
   [
    "C204",
    "x_vars[9,The Catcher in the Rye]"
   ],
   [
    "C205",
    "x_vars[9,Moby Dick]"
   ],
   [
    "C206",
    "x_vars[9,Jane Eyre]"
   ],
   [
    "C207",
    "x_vars[9,War and Peace]"
   ],
   [
    "C208",
    "x_vars[9,The Odyssey]"
   ],
   [
    "C209",
    "x_vars[9,Crime and Punishment]"
   ],
   [
    "C210",
    "x_vars[9,The Hobbit]"
   ],
   [
    "C211",
    "x_vars[9,Brave New World]"
   ],
   [
    "C212",
    "x_vars[9,Anna Karenina]"
   ],
   [
    "C213",
    "x_vars[9,Wuthering Heights]"
   ],
   [
    "C214",
    "x_vars[9,The Divine Comedy]"
   ],
   [
    "C215",
    "x_vars[9,The Iliad]"
   ],
   [
    "C216",
    "x_vars[9,Les Misérables]"
   ],
   [
    "C217",
    "x_vars[9,Dracula]"
   ],
   [
    "C218",
    "x_vars[9,Frankenstein]"
   ],
   [
    "C219",
    "x_vars[9,The Brothers Karamazov]"
   ],
   [
    "C220",
    "x_vars[9,Don Quixote]"
   ],
   [
    "C221",
    "x_vars[9,One Hundred Years of Solitude]"
   ],
   [
    "C222",
    "x_vars[9,Ulysses]"
   ],
   [
    "C223",
    "x_vars[9,The Alchemist]"
   ],
   [
    "C224",
    "x_vars[9,Meditations]"
   ],
   [
    "C225",
    "x_vars[10,The Great Gatsby]"
   ],
   [
    "C226",
    "x_vars[10,To Kill a Mockingbird]"
   ],
   [
    "C227",
    "x_vars[10,1984]"
   ],
   [
    "C228",
    "x_vars[10,Pride and Prejudice]"
   ],
   [
    "C229",
    "x_vars[10,The Catcher in the Rye]"
   ],
   [
    "C230",
    "x_vars[10,Moby Dick]"
   ],
   [
    "C231",
    "x_vars[10,Jane Eyre]"
   ],
   [
    "C232",
    "x_vars[10,War and Peace]"
   ],
   [
    "C233",
    "x_vars[10,The Odyssey]"
   ],
   [
    "C234",
    "x_vars[10,Crime and Punishment]"
   ],
   [
    "C235",
    "x_vars[10,The Hobbit]"
   ],
   [
    "C236",
    "x_vars[10,Brave New World]"
   ],
   [
    "C237",
    "x_vars[10,Anna Karenina]"
   ],
   [
    "C238",
    "x_vars[10,Wuthering Heights]"
   ],
   [
    "C239",
    "x_vars[10,The Divine Comedy]"
   ],
   [
    "C240",
    "x_vars[10,The Iliad]"
   ],
   [
    "C241",
    "x_vars[10,Les Misérables]"
   ],
   [
    "C242",
    "x_vars[10,Dracula]"
   ],
   [
    "C243",
    "x_vars[10,Frankenstein]"
   ],
   [
    "C244",
    "x_vars[10,The Brothers Karamazov]"
   ],
   [
    "C245",
    "x_vars[10,Don Quixote]"
   ],
   [
    "C246",
    "x_vars[10,One Hundred Years of Solitude]"
   ],
   [
    "C247",
    "x_vars[10,Ulysses]"
   ],
   [
    "C248",
    "x_vars[10,The Alchemist]"
   ],
   [
    "C249",
    "x_vars[10,Meditations]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,The Great Gatsby]",
   "x_0_1": "x_vars[2,The Great Gatsby]",
   "x_0_2": "x_vars[3,The Great Gatsby]",
   "x_0_3": "x_vars[4,The Great Gatsby]",
   "x_0_4": "x_vars[5,The Great Gatsby]",
   "x_0_5": "x_vars[6,The Great Gatsby]",
   "x_0_6": "x_vars[7,The Great Gatsby]",
   "x_0_7": "x_vars[8,The Great Gatsby]",
   "x_0_8": "x_vars[9,The Great Gatsby]",
   "x_0_9": "x_vars[10,The Great Gatsby]",
   "x_1_0": "x_vars[1,To Kill a Mockingbird]",
   "x_1_1": "x_vars[2,To Kill a Mockingbird]",
   "x_1_2": "x_vars[3,To Kill a Mockingbird]",
   "x_1_3": "x_vars[4,To Kill a Mockingbird]",
   "x_1_4": "x_vars[5,To Kill a Mockingbird]",
   "x_1_5": "x_vars[6,To Kill a Mockingbird]",
   "x_1_6": "x_vars[7,To Kill a Mockingbird]",
   "x_1_7": "x_vars[8,To Kill a Mockingbird]",
   "x_1_8": "x_vars[9,To Kill a Mockingbird]",
   "x_1_9": "x_vars[10,To Kill a Mockingbird]",
   "x_2_0": "x_vars[1,1984]",
   "x_2_1": "x_vars[2,1984]",
   "x_2_2": "x_vars[3,1984]",
   "x_2_3": "x_vars[4,1984]",
   "x_2_4": "x_vars[5,1984]",
   "x_2_5": "x_vars[6,1984]",
   "x_2_6": "x_vars[7,1984]",
   "x_2_7": "x_vars[8,1984]",
   "x_2_8": "x_vars[9,1984]",
   "x_2_9": "x_vars[10,1984]",
   "x_3_0": "x_vars[1,Pride and Prejudice]",
   "x_3_1": "x_vars[2,Pride and Prejudice]",
   "x_3_2": "x_vars[3,Pride and Prejudice]",
   "x_3_3": "x_vars[4,Pride and Prejudice]",
   "x_3_4": "x_vars[5,Pride and Prejudice]",
   "x_3_5": "x_vars[6,Pride and Prejudice]",
   "x_3_6": "x_vars[7,Pride and Prejudice]",
   "x_3_7": "x_vars[8,Pride and Prejudice]",
   "x_3_8": "x_vars[9,Pride and Prejudice]",
   "x_3_9": "x_vars[10,Pride and Prejudice]",
   "x_4_0": "x_vars[1,The Catcher in the Rye]",
   "x_4_1": "x_vars[2,The Catcher in the Rye]",
   "x_4_2": "x_vars[3,The Catcher in the Rye]",
   "x_4_3": "x_vars[4,The Catcher in the Rye]",
   "x_4_4": "x_vars[5,The Catcher in the Rye]",
   "x_4_5": "x_vars[6,The Catcher in the Rye]",
   "x_4_6": "x_vars[7,The Catcher in the Rye]",
   "x_4_7": "x_vars[8,The Catcher in the Rye]",
   "x_4_8": "x_vars[9,The Catcher in the Rye]",
   "x_4_9": "x_vars[10,The Catcher in the Rye]",
   "x_5_0": "x_vars[1,Moby Dick]",
   "x_5_1": "x_vars[2,Moby Dick]",
   "x_5_2": "x_vars[3,Moby Dick]",
   "x_5_3": "x_vars[4,Moby Dick]",
   "x_5_4": "x_vars[5,Moby Dick]",
   "x_5_5": "x_vars[6,Moby Dick]",
   "x_5_6": "x_vars[7,Moby Dick]",
   "x_5_7": "x_vars[8,Moby Dick]",
   "x_5_8": "x_vars[9,Moby Dick]",
   "x_5_9": "x_vars[10,Moby Dick]",
   "x_6_0": "x_vars[1,Jane Eyre]",
   "x_6_1": "x_vars[2,Jane Eyre]",
   "x_6_2": "x_vars[3,Jane Eyre]",
   "x_6_3": "x_vars[4,Jane Eyre]",
   "x_6_4": "x_vars[5,Jane Eyre]",
   "x_6_5": "x_vars[6,Jane Eyre]",
   "x_6_6": "x_vars[7,Jane Eyre]",
   "x_6_7": "x_vars[8,Jane Eyre]",
   "x_6_8": "x_vars[9,Jane Eyre]",
   "x_6_9": "x_vars[10,Jane Eyre]",
   "x_7_0": "x_vars[1,War and Peace]",
   "x_7_1": "x_vars[2,War and Peace]",
   "x_7_2": "x_vars[3,War and Peace]",
   "x_7_3": "x_vars[4,War and Peace]",
   "x_7_4": "x_vars[5,War and Peace]",
   "x_7_5": "x_vars[6,War and Peace]",
   "x_7_6": "x_vars[7,War and Peace]",
   "x_7_7": "x_vars[8,War and Peace]",
   "x_7_8": "x_vars[9,War and Peace]",
   "x_7_9": "x_vars[10,War and Peace]",
   "x_8_0": "x_vars[1,The Odyssey]",
   "x_8_1": "x_vars[2,The Odyssey]",
   "x_8_2": "x_vars[3,The Odyssey]",
   "x_8_3": "x_vars[4,The Odyssey]",
   "x_8_4": "x_vars[5,The Odyssey]",
   "x_8_5": "x_vars[6,The Odyssey]",
   "x_8_6": "x_vars[7,The Odyssey]",
   "x_8_7": "x_vars[8,The Odyssey]",
   "x_8_8": "x_vars[9,The Odyssey]",
   "x_8_9": "x_vars[10,The Odyssey]",
   "x_9_0": "x_vars[1,Crime and Punishment]",
   "x_9_1": "x_vars[2,Crime and Punishment]",
   "x_9_2": "x_vars[3,Crime and Punishment]",
   "x_9_3": "x_vars[4,Crime and Punishment]",
   "x_9_4": "x_vars[5,Crime and Punishment]",
   "x_9_5": "x_vars[6,Crime and Punishment]",
   "x_9_6": "x_vars[7,Crime and Punishment]",
   "x_9_7": "x_vars[8,Crime and Punishment]",
   "x_9_8": "x_vars[9,Crime and Punishment]",
   "x_9_9": "x_vars[10,Crime and Punishment]",
   "x_10_0": "x_vars[1,The Hobbit]",
   "x_10_1": "x_vars[2,The Hobbit]",
   "x_10_2": "x_vars[3,The Hobbit]",
   "x_10_3": "x_vars[4,The Hobbit]",
   "x_10_4": "x_vars[5,The Hobbit]",
   "x_10_5": "x_vars[6,The Hobbit]",
   "x_10_6": "x_vars[7,The Hobbit]",
   "x_10_7": "x_vars[8,The Hobbit]",
   "x_10_8": "x_vars[9,The Hobbit]",
   "x_10_9": "x_vars[10,The Hobbit]",
   "x_11_0": "x_vars[1,Brave New World]",
   "x_11_1": "x_vars[2,Brave New World]",
   "x_11_2": "x_vars[3,Brave New World]",
   "x_11_3": "x_vars[4,Brave New World]",
   "x_11_4": "x_vars[5,Brave New World]",
   "x_11_5": "x_vars[6,Brave New World]",
   "x_11_6": "x_vars[7,Brave New World]",
   "x_11_7": "x_vars[8,Brave New World]",
   "x_11_8": "x_vars[9,Brave New World]",
   "x_11_9": "x_vars[10,Brave New World]",
   "x_12_0": "x_vars[1,Anna Karenina]",
   "x_12_1": "x_vars[2,Anna Karenina]",
   "x_12_2": "x_vars[3,Anna Karenina]",
   "x_12_3": "x_vars[4,Anna Karenina]",
   "x_12_4": "x_vars[5,Anna Karenina]",
   "x_12_5": "x_vars[6,Anna Karenina]",
   "x_12_6": "x_vars[7,Anna Karenina]",
   "x_12_7": "x_vars[8,Anna Karenina]",
   "x_12_8": "x_vars[9,Anna Karenina]",
   "x_12_9": "x_vars[10,Anna Karenina]",
   "x_13_0": "x_vars[1,Wuthering Heights]",
   "x_13_1": "x_vars[2,Wuthering Heights]",
   "x_13_2": "x_vars[3,Wuthering Heights]",
   "x_13_3": "x_vars[4,Wuthering Heights]",
   "x_13_4": "x_vars[5,Wuthering Heights]",
   "x_13_5": "x_vars[6,Wuthering Heights]",
   "x_13_6": "x_vars[7,Wuthering Heights]",
   "x_13_7": "x_vars[8,Wuthering Heights]",
   "x_13_8": "x_vars[9,Wuthering Heights]",
   "x_13_9": "x_vars[10,Wuthering Heights]",
   "x_14_0": "x_vars[1,The Divine Comedy]",
   "x_14_1": "x_vars[2,The Divine Comedy]",
   "x_14_2": "x_vars[3,The Divine Comedy]",
   "x_14_3": "x_vars[4,The Divine Comedy]",
   "x_14_4": "x_vars[5,The Divine Comedy]",
   "x_14_5": "x_vars[6,The Divine Comedy]",
   "x_14_6": "x_vars[7,The Divine Comedy]",
   "x_14_7": "x_vars[8,The Divine Comedy]",
   "x_14_8": "x_vars[9,The Divine Comedy]",
   "x_14_9": "x_vars[10,The Divine Comedy]",
   "x_15_0": "x_vars[1,The Iliad]",
   "x_15_1": "x_vars[2,The Iliad]",
   "x_15_2": "x_vars[3,The Iliad]",
   "x_15_3": "x_vars[4,The Iliad]",
   "x_15_4": "x_vars[5,The Iliad]",
   "x_15_5": "x_vars[6,The Iliad]",
   "x_15_6": "x_vars[7,The Iliad]",
   "x_15_7": "x_vars[8,The Iliad]",
   "x_15_8": "x_vars[9,The Iliad]",
   "x_15_9": "x_vars[10,The Iliad]",
   "x_16_0": "x_vars[1,Les Misérables]",
   "x_16_1": "x_vars[2,Les Misérables]",
   "x_16_2": "x_vars[3,Les Misérables]",
   "x_16_3": "x_vars[4,Les Misérables]",
   "x_16_4": "x_vars[5,Les Misérables]",
   "x_16_5": "x_vars[6,Les Misérables]",
   "x_16_6": "x_vars[7,Les Misérables]",
   "x_16_7": "x_vars[8,Les Misérables]",
   "x_16_8": "x_vars[9,Les Misérables]",
   "x_16_9": "x_vars[10,Les Misérables]",
   "x_17_0": "x_vars[1,Dracula]",
   "x_17_1": "x_vars[2,Dracula]",
   "x_17_2": "x_vars[3,Dracula]",
   "x_17_3": "x_vars[4,Dracula]",
   "x_17_4": "x_vars[5,Dracula]",
   "x_17_5": "x_vars[6,Dracula]",
   "x_17_6": "x_vars[7,Dracula]",
   "x_17_7": "x_vars[8,Dracula]",
   "x_17_8": "x_vars[9,Dracula]",
   "x_17_9": "x_vars[10,Dracula]",
   "x_18_0": "x_vars[1,Frankenstein]",
   "x_18_1": "x_vars[2,Frankenstein]",
   "x_18_2": "x_vars[3,Frankenstein]",
   "x_18_3": "x_vars[4,Frankenstein]",
   "x_18_4": "x_vars[5,Frankenstein]",
   "x_18_5": "x_vars[6,Frankenstein]",
   "x_18_6": "x_vars[7,Frankenstein]",
   "x_18_7": "x_vars[8,Frankenstein]",
   "x_18_8": "x_vars[9,Frankenstein]",
   "x_18_9": "x_vars[10,Frankenstein]",
   "x_19_0": "x_vars[1,The Brothers Karamazov]",
   "x_19_1": "x_vars[2,The Brothers Karamazov]",
   "x_19_2": "x_vars[3,The Brothers Karamazov]",
   "x_19_3": "x_vars[4,The Brothers Karamazov]",
   "x_19_4": "x_vars[5,The Brothers Karamazov]",
   "x_19_5": "x_vars[6,The Brothers Karamazov]",
   "x_19_6": "x_vars[7,The Brothers Karamazov]",
   "x_19_7": "x_vars[8,The Brothers Karamazov]",
   "x_19_8": "x_vars[9,The Brothers Karamazov]",
   "x_19_9": "x_vars[10,The Brothers Karamazov]",
   "x_20_0": "x_vars[1,Don Quixote]",
   "x_20_1": "x_vars[2,Don Quixote]",
   "x_20_2": "x_vars[3,Don Quixote]",
   "x_20_3": "x_vars[4,Don Quixote]",
   "x_20_4": "x_vars[5,Don Quixote]",
   "x_20_5": "x_vars[6,Don Quixote]",
   "x_20_6": "x_vars[7,Don Quixote]",
   "x_20_7": "x_vars[8,Don Quixote]",
   "x_20_8": "x_vars[9,Don Quixote]",
   "x_20_9": "x_vars[10,Don Quixote]",
   "x_21_0": "x_vars[1,One Hundred Years of Solitude]",
   "x_21_1": "x_vars[2,One Hundred Years of Solitude]",
   "x_21_2": "x_vars[3,One Hundred Years of Solitude]",
   "x_21_3": "x_vars[4,One Hundred Years of Solitude]",
   "x_21_4": "x_vars[5,One Hundred Years of Solitude]",
   "x_21_5": "x_vars[6,One Hundred Years of Solitude]",
   "x_21_6": "x_vars[7,One Hundred Years of Solitude]",
   "x_21_7": "x_vars[8,One Hundred Years of Solitude]",
   "x_21_8": "x_vars[9,One Hundred Years of Solitude]",
   "x_21_9": "x_vars[10,One Hundred Years of Solitude]",
   "x_22_0": "x_vars[1,Ulysses]",
   "x_22_1": "x_vars[2,Ulysses]",
   "x_22_2": "x_vars[3,Ulysses]",
   "x_22_3": "x_vars[4,Ulysses]",
   "x_22_4": "x_vars[5,Ulysses]",
   "x_22_5": "x_vars[6,Ulysses]",
   "x_22_6": "x_vars[7,Ulysses]",
   "x_22_7": "x_vars[8,Ulysses]",
   "x_22_8": "x_vars[9,Ulysses]",
   "x_22_9": "x_vars[10,Ulysses]",
   "x_23_0": "x_vars[1,The Alchemist]",
   "x_23_1": "x_vars[2,The Alchemist]",
   "x_23_2": "x_vars[3,The Alchemist]",
   "x_23_3": "x_vars[4,The Alchemist]",
   "x_23_4": "x_vars[5,The Alchemist]",
   "x_23_5": "x_vars[6,The Alchemist]",
   "x_23_6": "x_vars[7,The Alchemist]",
   "x_23_7": "x_vars[8,The Alchemist]",
   "x_23_8": "x_vars[9,The Alchemist]",
   "x_23_9": "x_vars[10,The Alchemist]",
   "x_24_0": "x_vars[1,Meditations]",
   "x_24_1": "x_vars[2,Meditations]",
   "x_24_2": "x_vars[3,Meditations]",
   "x_24_3": "x_vars[4,Meditations]",
   "x_24_4": "x_vars[5,Meditations]",
   "x_24_5": "x_vars[6,Meditations]",
   "x_24_6": "x_vars[7,Meditations]",
   "x_24_7": "x_vars[8,Meditations]",
   "x_24_8": "x_vars[9,Meditations]",
   "x_24_9": "x_vars[10,Meditations]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-053": {
  "source_code_sha256": "5054e084e70052be69b65b32ae7fdf0d3e99e143765691261ebef29c315bc0f8",
  "original_solution_sha256": "85b8313391ec386654edd65ce2bbafc051bf7523976cc1e79a8eeafa9757ffe8",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[1,4]"
   ],
   [
    "C4",
    "x_vars[1,5]"
   ],
   [
    "C5",
    "x_vars[1,6]"
   ],
   [
    "C6",
    "x_vars[1,7]"
   ],
   [
    "C7",
    "x_vars[1,8]"
   ],
   [
    "C8",
    "x_vars[1,9]"
   ],
   [
    "C9",
    "x_vars[1,10]"
   ],
   [
    "C10",
    "x_vars[1,11]"
   ],
   [
    "C11",
    "x_vars[1,12]"
   ],
   [
    "C12",
    "x_vars[1,13]"
   ],
   [
    "C13",
    "x_vars[1,14]"
   ],
   [
    "C14",
    "x_vars[1,15]"
   ],
   [
    "C15",
    "x_vars[1,16]"
   ],
   [
    "C16",
    "x_vars[1,17]"
   ],
   [
    "C17",
    "x_vars[1,18]"
   ],
   [
    "C18",
    "x_vars[1,19]"
   ],
   [
    "C19",
    "x_vars[1,20]"
   ],
   [
    "C20",
    "x_vars[2,1]"
   ],
   [
    "C21",
    "x_vars[2,2]"
   ],
   [
    "C22",
    "x_vars[2,3]"
   ],
   [
    "C23",
    "x_vars[2,4]"
   ],
   [
    "C24",
    "x_vars[2,5]"
   ],
   [
    "C25",
    "x_vars[2,6]"
   ],
   [
    "C26",
    "x_vars[2,7]"
   ],
   [
    "C27",
    "x_vars[2,8]"
   ],
   [
    "C28",
    "x_vars[2,9]"
   ],
   [
    "C29",
    "x_vars[2,10]"
   ],
   [
    "C30",
    "x_vars[2,11]"
   ],
   [
    "C31",
    "x_vars[2,12]"
   ],
   [
    "C32",
    "x_vars[2,13]"
   ],
   [
    "C33",
    "x_vars[2,14]"
   ],
   [
    "C34",
    "x_vars[2,15]"
   ],
   [
    "C35",
    "x_vars[2,16]"
   ],
   [
    "C36",
    "x_vars[2,17]"
   ],
   [
    "C37",
    "x_vars[2,18]"
   ],
   [
    "C38",
    "x_vars[2,19]"
   ],
   [
    "C39",
    "x_vars[2,20]"
   ],
   [
    "C40",
    "x_vars[3,1]"
   ],
   [
    "C41",
    "x_vars[3,2]"
   ],
   [
    "C42",
    "x_vars[3,3]"
   ],
   [
    "C43",
    "x_vars[3,4]"
   ],
   [
    "C44",
    "x_vars[3,5]"
   ],
   [
    "C45",
    "x_vars[3,6]"
   ],
   [
    "C46",
    "x_vars[3,7]"
   ],
   [
    "C47",
    "x_vars[3,8]"
   ],
   [
    "C48",
    "x_vars[3,9]"
   ],
   [
    "C49",
    "x_vars[3,10]"
   ],
   [
    "C50",
    "x_vars[3,11]"
   ],
   [
    "C51",
    "x_vars[3,12]"
   ],
   [
    "C52",
    "x_vars[3,13]"
   ],
   [
    "C53",
    "x_vars[3,14]"
   ],
   [
    "C54",
    "x_vars[3,15]"
   ],
   [
    "C55",
    "x_vars[3,16]"
   ],
   [
    "C56",
    "x_vars[3,17]"
   ],
   [
    "C57",
    "x_vars[3,18]"
   ],
   [
    "C58",
    "x_vars[3,19]"
   ],
   [
    "C59",
    "x_vars[3,20]"
   ],
   [
    "C60",
    "x_vars[4,1]"
   ],
   [
    "C61",
    "x_vars[4,2]"
   ],
   [
    "C62",
    "x_vars[4,3]"
   ],
   [
    "C63",
    "x_vars[4,4]"
   ],
   [
    "C64",
    "x_vars[4,5]"
   ],
   [
    "C65",
    "x_vars[4,6]"
   ],
   [
    "C66",
    "x_vars[4,7]"
   ],
   [
    "C67",
    "x_vars[4,8]"
   ],
   [
    "C68",
    "x_vars[4,9]"
   ],
   [
    "C69",
    "x_vars[4,10]"
   ],
   [
    "C70",
    "x_vars[4,11]"
   ],
   [
    "C71",
    "x_vars[4,12]"
   ],
   [
    "C72",
    "x_vars[4,13]"
   ],
   [
    "C73",
    "x_vars[4,14]"
   ],
   [
    "C74",
    "x_vars[4,15]"
   ],
   [
    "C75",
    "x_vars[4,16]"
   ],
   [
    "C76",
    "x_vars[4,17]"
   ],
   [
    "C77",
    "x_vars[4,18]"
   ],
   [
    "C78",
    "x_vars[4,19]"
   ],
   [
    "C79",
    "x_vars[4,20]"
   ],
   [
    "C80",
    "x_vars[5,1]"
   ],
   [
    "C81",
    "x_vars[5,2]"
   ],
   [
    "C82",
    "x_vars[5,3]"
   ],
   [
    "C83",
    "x_vars[5,4]"
   ],
   [
    "C84",
    "x_vars[5,5]"
   ],
   [
    "C85",
    "x_vars[5,6]"
   ],
   [
    "C86",
    "x_vars[5,7]"
   ],
   [
    "C87",
    "x_vars[5,8]"
   ],
   [
    "C88",
    "x_vars[5,9]"
   ],
   [
    "C89",
    "x_vars[5,10]"
   ],
   [
    "C90",
    "x_vars[5,11]"
   ],
   [
    "C91",
    "x_vars[5,12]"
   ],
   [
    "C92",
    "x_vars[5,13]"
   ],
   [
    "C93",
    "x_vars[5,14]"
   ],
   [
    "C94",
    "x_vars[5,15]"
   ],
   [
    "C95",
    "x_vars[5,16]"
   ],
   [
    "C96",
    "x_vars[5,17]"
   ],
   [
    "C97",
    "x_vars[5,18]"
   ],
   [
    "C98",
    "x_vars[5,19]"
   ],
   [
    "C99",
    "x_vars[5,20]"
   ],
   [
    "C100",
    "x_vars[6,1]"
   ],
   [
    "C101",
    "x_vars[6,2]"
   ],
   [
    "C102",
    "x_vars[6,3]"
   ],
   [
    "C103",
    "x_vars[6,4]"
   ],
   [
    "C104",
    "x_vars[6,5]"
   ],
   [
    "C105",
    "x_vars[6,6]"
   ],
   [
    "C106",
    "x_vars[6,7]"
   ],
   [
    "C107",
    "x_vars[6,8]"
   ],
   [
    "C108",
    "x_vars[6,9]"
   ],
   [
    "C109",
    "x_vars[6,10]"
   ],
   [
    "C110",
    "x_vars[6,11]"
   ],
   [
    "C111",
    "x_vars[6,12]"
   ],
   [
    "C112",
    "x_vars[6,13]"
   ],
   [
    "C113",
    "x_vars[6,14]"
   ],
   [
    "C114",
    "x_vars[6,15]"
   ],
   [
    "C115",
    "x_vars[6,16]"
   ],
   [
    "C116",
    "x_vars[6,17]"
   ],
   [
    "C117",
    "x_vars[6,18]"
   ],
   [
    "C118",
    "x_vars[6,19]"
   ],
   [
    "C119",
    "x_vars[6,20]"
   ],
   [
    "C120",
    "x_vars[7,1]"
   ],
   [
    "C121",
    "x_vars[7,2]"
   ],
   [
    "C122",
    "x_vars[7,3]"
   ],
   [
    "C123",
    "x_vars[7,4]"
   ],
   [
    "C124",
    "x_vars[7,5]"
   ],
   [
    "C125",
    "x_vars[7,6]"
   ],
   [
    "C126",
    "x_vars[7,7]"
   ],
   [
    "C127",
    "x_vars[7,8]"
   ],
   [
    "C128",
    "x_vars[7,9]"
   ],
   [
    "C129",
    "x_vars[7,10]"
   ],
   [
    "C130",
    "x_vars[7,11]"
   ],
   [
    "C131",
    "x_vars[7,12]"
   ],
   [
    "C132",
    "x_vars[7,13]"
   ],
   [
    "C133",
    "x_vars[7,14]"
   ],
   [
    "C134",
    "x_vars[7,15]"
   ],
   [
    "C135",
    "x_vars[7,16]"
   ],
   [
    "C136",
    "x_vars[7,17]"
   ],
   [
    "C137",
    "x_vars[7,18]"
   ],
   [
    "C138",
    "x_vars[7,19]"
   ],
   [
    "C139",
    "x_vars[7,20]"
   ],
   [
    "C140",
    "x_vars[8,1]"
   ],
   [
    "C141",
    "x_vars[8,2]"
   ],
   [
    "C142",
    "x_vars[8,3]"
   ],
   [
    "C143",
    "x_vars[8,4]"
   ],
   [
    "C144",
    "x_vars[8,5]"
   ],
   [
    "C145",
    "x_vars[8,6]"
   ],
   [
    "C146",
    "x_vars[8,7]"
   ],
   [
    "C147",
    "x_vars[8,8]"
   ],
   [
    "C148",
    "x_vars[8,9]"
   ],
   [
    "C149",
    "x_vars[8,10]"
   ],
   [
    "C150",
    "x_vars[8,11]"
   ],
   [
    "C151",
    "x_vars[8,12]"
   ],
   [
    "C152",
    "x_vars[8,13]"
   ],
   [
    "C153",
    "x_vars[8,14]"
   ],
   [
    "C154",
    "x_vars[8,15]"
   ],
   [
    "C155",
    "x_vars[8,16]"
   ],
   [
    "C156",
    "x_vars[8,17]"
   ],
   [
    "C157",
    "x_vars[8,18]"
   ],
   [
    "C158",
    "x_vars[8,19]"
   ],
   [
    "C159",
    "x_vars[8,20]"
   ],
   [
    "C160",
    "x_vars[9,1]"
   ],
   [
    "C161",
    "x_vars[9,2]"
   ],
   [
    "C162",
    "x_vars[9,3]"
   ],
   [
    "C163",
    "x_vars[9,4]"
   ],
   [
    "C164",
    "x_vars[9,5]"
   ],
   [
    "C165",
    "x_vars[9,6]"
   ],
   [
    "C166",
    "x_vars[9,7]"
   ],
   [
    "C167",
    "x_vars[9,8]"
   ],
   [
    "C168",
    "x_vars[9,9]"
   ],
   [
    "C169",
    "x_vars[9,10]"
   ],
   [
    "C170",
    "x_vars[9,11]"
   ],
   [
    "C171",
    "x_vars[9,12]"
   ],
   [
    "C172",
    "x_vars[9,13]"
   ],
   [
    "C173",
    "x_vars[9,14]"
   ],
   [
    "C174",
    "x_vars[9,15]"
   ],
   [
    "C175",
    "x_vars[9,16]"
   ],
   [
    "C176",
    "x_vars[9,17]"
   ],
   [
    "C177",
    "x_vars[9,18]"
   ],
   [
    "C178",
    "x_vars[9,19]"
   ],
   [
    "C179",
    "x_vars[9,20]"
   ],
   [
    "C180",
    "x_vars[10,1]"
   ],
   [
    "C181",
    "x_vars[10,2]"
   ],
   [
    "C182",
    "x_vars[10,3]"
   ],
   [
    "C183",
    "x_vars[10,4]"
   ],
   [
    "C184",
    "x_vars[10,5]"
   ],
   [
    "C185",
    "x_vars[10,6]"
   ],
   [
    "C186",
    "x_vars[10,7]"
   ],
   [
    "C187",
    "x_vars[10,8]"
   ],
   [
    "C188",
    "x_vars[10,9]"
   ],
   [
    "C189",
    "x_vars[10,10]"
   ],
   [
    "C190",
    "x_vars[10,11]"
   ],
   [
    "C191",
    "x_vars[10,12]"
   ],
   [
    "C192",
    "x_vars[10,13]"
   ],
   [
    "C193",
    "x_vars[10,14]"
   ],
   [
    "C194",
    "x_vars[10,15]"
   ],
   [
    "C195",
    "x_vars[10,16]"
   ],
   [
    "C196",
    "x_vars[10,17]"
   ],
   [
    "C197",
    "x_vars[10,18]"
   ],
   [
    "C198",
    "x_vars[10,19]"
   ],
   [
    "C199",
    "x_vars[10,20]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,1]",
   "x_0_1": "x_vars[2,1]",
   "x_0_2": "x_vars[3,1]",
   "x_0_3": "x_vars[4,1]",
   "x_0_4": "x_vars[5,1]",
   "x_0_5": "x_vars[6,1]",
   "x_0_6": "x_vars[7,1]",
   "x_0_7": "x_vars[8,1]",
   "x_0_8": "x_vars[9,1]",
   "x_0_9": "x_vars[10,1]",
   "x_1_0": "x_vars[1,2]",
   "x_1_1": "x_vars[2,2]",
   "x_1_2": "x_vars[3,2]",
   "x_1_3": "x_vars[4,2]",
   "x_1_4": "x_vars[5,2]",
   "x_1_5": "x_vars[6,2]",
   "x_1_6": "x_vars[7,2]",
   "x_1_7": "x_vars[8,2]",
   "x_1_8": "x_vars[9,2]",
   "x_1_9": "x_vars[10,2]",
   "x_2_0": "x_vars[1,3]",
   "x_2_1": "x_vars[2,3]",
   "x_2_2": "x_vars[3,3]",
   "x_2_3": "x_vars[4,3]",
   "x_2_4": "x_vars[5,3]",
   "x_2_5": "x_vars[6,3]",
   "x_2_6": "x_vars[7,3]",
   "x_2_7": "x_vars[8,3]",
   "x_2_8": "x_vars[9,3]",
   "x_2_9": "x_vars[10,3]",
   "x_3_0": "x_vars[1,4]",
   "x_3_1": "x_vars[2,4]",
   "x_3_2": "x_vars[3,4]",
   "x_3_3": "x_vars[4,4]",
   "x_3_4": "x_vars[5,4]",
   "x_3_5": "x_vars[6,4]",
   "x_3_6": "x_vars[7,4]",
   "x_3_7": "x_vars[8,4]",
   "x_3_8": "x_vars[9,4]",
   "x_3_9": "x_vars[10,4]",
   "x_4_0": "x_vars[1,5]",
   "x_4_1": "x_vars[2,5]",
   "x_4_2": "x_vars[3,5]",
   "x_4_3": "x_vars[4,5]",
   "x_4_4": "x_vars[5,5]",
   "x_4_5": "x_vars[6,5]",
   "x_4_6": "x_vars[7,5]",
   "x_4_7": "x_vars[8,5]",
   "x_4_8": "x_vars[9,5]",
   "x_4_9": "x_vars[10,5]",
   "x_5_0": "x_vars[1,6]",
   "x_5_1": "x_vars[2,6]",
   "x_5_2": "x_vars[3,6]",
   "x_5_3": "x_vars[4,6]",
   "x_5_4": "x_vars[5,6]",
   "x_5_5": "x_vars[6,6]",
   "x_5_6": "x_vars[7,6]",
   "x_5_7": "x_vars[8,6]",
   "x_5_8": "x_vars[9,6]",
   "x_5_9": "x_vars[10,6]",
   "x_6_0": "x_vars[1,7]",
   "x_6_1": "x_vars[2,7]",
   "x_6_2": "x_vars[3,7]",
   "x_6_3": "x_vars[4,7]",
   "x_6_4": "x_vars[5,7]",
   "x_6_5": "x_vars[6,7]",
   "x_6_6": "x_vars[7,7]",
   "x_6_7": "x_vars[8,7]",
   "x_6_8": "x_vars[9,7]",
   "x_6_9": "x_vars[10,7]",
   "x_7_0": "x_vars[1,8]",
   "x_7_1": "x_vars[2,8]",
   "x_7_2": "x_vars[3,8]",
   "x_7_3": "x_vars[4,8]",
   "x_7_4": "x_vars[5,8]",
   "x_7_5": "x_vars[6,8]",
   "x_7_6": "x_vars[7,8]",
   "x_7_7": "x_vars[8,8]",
   "x_7_8": "x_vars[9,8]",
   "x_7_9": "x_vars[10,8]",
   "x_8_0": "x_vars[1,9]",
   "x_8_1": "x_vars[2,9]",
   "x_8_2": "x_vars[3,9]",
   "x_8_3": "x_vars[4,9]",
   "x_8_4": "x_vars[5,9]",
   "x_8_5": "x_vars[6,9]",
   "x_8_6": "x_vars[7,9]",
   "x_8_7": "x_vars[8,9]",
   "x_8_8": "x_vars[9,9]",
   "x_8_9": "x_vars[10,9]",
   "x_9_0": "x_vars[1,10]",
   "x_9_1": "x_vars[2,10]",
   "x_9_2": "x_vars[3,10]",
   "x_9_3": "x_vars[4,10]",
   "x_9_4": "x_vars[5,10]",
   "x_9_5": "x_vars[6,10]",
   "x_9_6": "x_vars[7,10]",
   "x_9_7": "x_vars[8,10]",
   "x_9_8": "x_vars[9,10]",
   "x_9_9": "x_vars[10,10]",
   "x_10_0": "x_vars[1,11]",
   "x_10_1": "x_vars[2,11]",
   "x_10_2": "x_vars[3,11]",
   "x_10_3": "x_vars[4,11]",
   "x_10_4": "x_vars[5,11]",
   "x_10_5": "x_vars[6,11]",
   "x_10_6": "x_vars[7,11]",
   "x_10_7": "x_vars[8,11]",
   "x_10_8": "x_vars[9,11]",
   "x_10_9": "x_vars[10,11]",
   "x_11_0": "x_vars[1,12]",
   "x_11_1": "x_vars[2,12]",
   "x_11_2": "x_vars[3,12]",
   "x_11_3": "x_vars[4,12]",
   "x_11_4": "x_vars[5,12]",
   "x_11_5": "x_vars[6,12]",
   "x_11_6": "x_vars[7,12]",
   "x_11_7": "x_vars[8,12]",
   "x_11_8": "x_vars[9,12]",
   "x_11_9": "x_vars[10,12]",
   "x_12_0": "x_vars[1,13]",
   "x_12_1": "x_vars[2,13]",
   "x_12_2": "x_vars[3,13]",
   "x_12_3": "x_vars[4,13]",
   "x_12_4": "x_vars[5,13]",
   "x_12_5": "x_vars[6,13]",
   "x_12_6": "x_vars[7,13]",
   "x_12_7": "x_vars[8,13]",
   "x_12_8": "x_vars[9,13]",
   "x_12_9": "x_vars[10,13]",
   "x_13_0": "x_vars[1,14]",
   "x_13_1": "x_vars[2,14]",
   "x_13_2": "x_vars[3,14]",
   "x_13_3": "x_vars[4,14]",
   "x_13_4": "x_vars[5,14]",
   "x_13_5": "x_vars[6,14]",
   "x_13_6": "x_vars[7,14]",
   "x_13_7": "x_vars[8,14]",
   "x_13_8": "x_vars[9,14]",
   "x_13_9": "x_vars[10,14]",
   "x_14_0": "x_vars[1,15]",
   "x_14_1": "x_vars[2,15]",
   "x_14_2": "x_vars[3,15]",
   "x_14_3": "x_vars[4,15]",
   "x_14_4": "x_vars[5,15]",
   "x_14_5": "x_vars[6,15]",
   "x_14_6": "x_vars[7,15]",
   "x_14_7": "x_vars[8,15]",
   "x_14_8": "x_vars[9,15]",
   "x_14_9": "x_vars[10,15]",
   "x_15_0": "x_vars[1,16]",
   "x_15_1": "x_vars[2,16]",
   "x_15_2": "x_vars[3,16]",
   "x_15_3": "x_vars[4,16]",
   "x_15_4": "x_vars[5,16]",
   "x_15_5": "x_vars[6,16]",
   "x_15_6": "x_vars[7,16]",
   "x_15_7": "x_vars[8,16]",
   "x_15_8": "x_vars[9,16]",
   "x_15_9": "x_vars[10,16]",
   "x_16_0": "x_vars[1,17]",
   "x_16_1": "x_vars[2,17]",
   "x_16_2": "x_vars[3,17]",
   "x_16_3": "x_vars[4,17]",
   "x_16_4": "x_vars[5,17]",
   "x_16_5": "x_vars[6,17]",
   "x_16_6": "x_vars[7,17]",
   "x_16_7": "x_vars[8,17]",
   "x_16_8": "x_vars[9,17]",
   "x_16_9": "x_vars[10,17]",
   "x_17_0": "x_vars[1,18]",
   "x_17_1": "x_vars[2,18]",
   "x_17_2": "x_vars[3,18]",
   "x_17_3": "x_vars[4,18]",
   "x_17_4": "x_vars[5,18]",
   "x_17_5": "x_vars[6,18]",
   "x_17_6": "x_vars[7,18]",
   "x_17_7": "x_vars[8,18]",
   "x_17_8": "x_vars[9,18]",
   "x_17_9": "x_vars[10,18]",
   "x_18_0": "x_vars[1,19]",
   "x_18_1": "x_vars[2,19]",
   "x_18_2": "x_vars[3,19]",
   "x_18_3": "x_vars[4,19]",
   "x_18_4": "x_vars[5,19]",
   "x_18_5": "x_vars[6,19]",
   "x_18_6": "x_vars[7,19]",
   "x_18_7": "x_vars[8,19]",
   "x_18_8": "x_vars[9,19]",
   "x_18_9": "x_vars[10,19]",
   "x_19_0": "x_vars[1,20]",
   "x_19_1": "x_vars[2,20]",
   "x_19_2": "x_vars[3,20]",
   "x_19_3": "x_vars[4,20]",
   "x_19_4": "x_vars[5,20]",
   "x_19_5": "x_vars[6,20]",
   "x_19_6": "x_vars[7,20]",
   "x_19_7": "x_vars[8,20]",
   "x_19_8": "x_vars[9,20]",
   "x_19_9": "x_vars[10,20]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-054": {
  "source_code_sha256": "81e0bbfd73b9af932b890e8e41d0c525a802c6ebca0eefb5a588dc1be5c5e240",
  "original_solution_sha256": "763cc8a10acb85fbf69cbc0d68bc505975588bdb7752fae71b76faf7e61c1ba9",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[1,4]"
   ],
   [
    "C4",
    "x_vars[1,5]"
   ],
   [
    "C5",
    "x_vars[1,6]"
   ],
   [
    "C6",
    "x_vars[1,7]"
   ],
   [
    "C7",
    "x_vars[1,8]"
   ],
   [
    "C8",
    "x_vars[1,9]"
   ],
   [
    "C9",
    "x_vars[1,10]"
   ],
   [
    "C10",
    "x_vars[1,11]"
   ],
   [
    "C11",
    "x_vars[1,12]"
   ],
   [
    "C12",
    "x_vars[1,13]"
   ],
   [
    "C13",
    "x_vars[1,14]"
   ],
   [
    "C14",
    "x_vars[1,15]"
   ],
   [
    "C15",
    "x_vars[1,16]"
   ],
   [
    "C16",
    "x_vars[1,17]"
   ],
   [
    "C17",
    "x_vars[1,18]"
   ],
   [
    "C18",
    "x_vars[1,19]"
   ],
   [
    "C19",
    "x_vars[1,20]"
   ],
   [
    "C20",
    "x_vars[2,1]"
   ],
   [
    "C21",
    "x_vars[2,2]"
   ],
   [
    "C22",
    "x_vars[2,3]"
   ],
   [
    "C23",
    "x_vars[2,4]"
   ],
   [
    "C24",
    "x_vars[2,5]"
   ],
   [
    "C25",
    "x_vars[2,6]"
   ],
   [
    "C26",
    "x_vars[2,7]"
   ],
   [
    "C27",
    "x_vars[2,8]"
   ],
   [
    "C28",
    "x_vars[2,9]"
   ],
   [
    "C29",
    "x_vars[2,10]"
   ],
   [
    "C30",
    "x_vars[2,11]"
   ],
   [
    "C31",
    "x_vars[2,12]"
   ],
   [
    "C32",
    "x_vars[2,13]"
   ],
   [
    "C33",
    "x_vars[2,14]"
   ],
   [
    "C34",
    "x_vars[2,15]"
   ],
   [
    "C35",
    "x_vars[2,16]"
   ],
   [
    "C36",
    "x_vars[2,17]"
   ],
   [
    "C37",
    "x_vars[2,18]"
   ],
   [
    "C38",
    "x_vars[2,19]"
   ],
   [
    "C39",
    "x_vars[2,20]"
   ],
   [
    "C40",
    "x_vars[3,1]"
   ],
   [
    "C41",
    "x_vars[3,2]"
   ],
   [
    "C42",
    "x_vars[3,3]"
   ],
   [
    "C43",
    "x_vars[3,4]"
   ],
   [
    "C44",
    "x_vars[3,5]"
   ],
   [
    "C45",
    "x_vars[3,6]"
   ],
   [
    "C46",
    "x_vars[3,7]"
   ],
   [
    "C47",
    "x_vars[3,8]"
   ],
   [
    "C48",
    "x_vars[3,9]"
   ],
   [
    "C49",
    "x_vars[3,10]"
   ],
   [
    "C50",
    "x_vars[3,11]"
   ],
   [
    "C51",
    "x_vars[3,12]"
   ],
   [
    "C52",
    "x_vars[3,13]"
   ],
   [
    "C53",
    "x_vars[3,14]"
   ],
   [
    "C54",
    "x_vars[3,15]"
   ],
   [
    "C55",
    "x_vars[3,16]"
   ],
   [
    "C56",
    "x_vars[3,17]"
   ],
   [
    "C57",
    "x_vars[3,18]"
   ],
   [
    "C58",
    "x_vars[3,19]"
   ],
   [
    "C59",
    "x_vars[3,20]"
   ],
   [
    "C60",
    "x_vars[4,1]"
   ],
   [
    "C61",
    "x_vars[4,2]"
   ],
   [
    "C62",
    "x_vars[4,3]"
   ],
   [
    "C63",
    "x_vars[4,4]"
   ],
   [
    "C64",
    "x_vars[4,5]"
   ],
   [
    "C65",
    "x_vars[4,6]"
   ],
   [
    "C66",
    "x_vars[4,7]"
   ],
   [
    "C67",
    "x_vars[4,8]"
   ],
   [
    "C68",
    "x_vars[4,9]"
   ],
   [
    "C69",
    "x_vars[4,10]"
   ],
   [
    "C70",
    "x_vars[4,11]"
   ],
   [
    "C71",
    "x_vars[4,12]"
   ],
   [
    "C72",
    "x_vars[4,13]"
   ],
   [
    "C73",
    "x_vars[4,14]"
   ],
   [
    "C74",
    "x_vars[4,15]"
   ],
   [
    "C75",
    "x_vars[4,16]"
   ],
   [
    "C76",
    "x_vars[4,17]"
   ],
   [
    "C77",
    "x_vars[4,18]"
   ],
   [
    "C78",
    "x_vars[4,19]"
   ],
   [
    "C79",
    "x_vars[4,20]"
   ],
   [
    "C80",
    "x_vars[5,1]"
   ],
   [
    "C81",
    "x_vars[5,2]"
   ],
   [
    "C82",
    "x_vars[5,3]"
   ],
   [
    "C83",
    "x_vars[5,4]"
   ],
   [
    "C84",
    "x_vars[5,5]"
   ],
   [
    "C85",
    "x_vars[5,6]"
   ],
   [
    "C86",
    "x_vars[5,7]"
   ],
   [
    "C87",
    "x_vars[5,8]"
   ],
   [
    "C88",
    "x_vars[5,9]"
   ],
   [
    "C89",
    "x_vars[5,10]"
   ],
   [
    "C90",
    "x_vars[5,11]"
   ],
   [
    "C91",
    "x_vars[5,12]"
   ],
   [
    "C92",
    "x_vars[5,13]"
   ],
   [
    "C93",
    "x_vars[5,14]"
   ],
   [
    "C94",
    "x_vars[5,15]"
   ],
   [
    "C95",
    "x_vars[5,16]"
   ],
   [
    "C96",
    "x_vars[5,17]"
   ],
   [
    "C97",
    "x_vars[5,18]"
   ],
   [
    "C98",
    "x_vars[5,19]"
   ],
   [
    "C99",
    "x_vars[5,20]"
   ],
   [
    "C100",
    "x_vars[6,1]"
   ],
   [
    "C101",
    "x_vars[6,2]"
   ],
   [
    "C102",
    "x_vars[6,3]"
   ],
   [
    "C103",
    "x_vars[6,4]"
   ],
   [
    "C104",
    "x_vars[6,5]"
   ],
   [
    "C105",
    "x_vars[6,6]"
   ],
   [
    "C106",
    "x_vars[6,7]"
   ],
   [
    "C107",
    "x_vars[6,8]"
   ],
   [
    "C108",
    "x_vars[6,9]"
   ],
   [
    "C109",
    "x_vars[6,10]"
   ],
   [
    "C110",
    "x_vars[6,11]"
   ],
   [
    "C111",
    "x_vars[6,12]"
   ],
   [
    "C112",
    "x_vars[6,13]"
   ],
   [
    "C113",
    "x_vars[6,14]"
   ],
   [
    "C114",
    "x_vars[6,15]"
   ],
   [
    "C115",
    "x_vars[6,16]"
   ],
   [
    "C116",
    "x_vars[6,17]"
   ],
   [
    "C117",
    "x_vars[6,18]"
   ],
   [
    "C118",
    "x_vars[6,19]"
   ],
   [
    "C119",
    "x_vars[6,20]"
   ],
   [
    "C120",
    "x_vars[7,1]"
   ],
   [
    "C121",
    "x_vars[7,2]"
   ],
   [
    "C122",
    "x_vars[7,3]"
   ],
   [
    "C123",
    "x_vars[7,4]"
   ],
   [
    "C124",
    "x_vars[7,5]"
   ],
   [
    "C125",
    "x_vars[7,6]"
   ],
   [
    "C126",
    "x_vars[7,7]"
   ],
   [
    "C127",
    "x_vars[7,8]"
   ],
   [
    "C128",
    "x_vars[7,9]"
   ],
   [
    "C129",
    "x_vars[7,10]"
   ],
   [
    "C130",
    "x_vars[7,11]"
   ],
   [
    "C131",
    "x_vars[7,12]"
   ],
   [
    "C132",
    "x_vars[7,13]"
   ],
   [
    "C133",
    "x_vars[7,14]"
   ],
   [
    "C134",
    "x_vars[7,15]"
   ],
   [
    "C135",
    "x_vars[7,16]"
   ],
   [
    "C136",
    "x_vars[7,17]"
   ],
   [
    "C137",
    "x_vars[7,18]"
   ],
   [
    "C138",
    "x_vars[7,19]"
   ],
   [
    "C139",
    "x_vars[7,20]"
   ],
   [
    "C140",
    "x_vars[8,1]"
   ],
   [
    "C141",
    "x_vars[8,2]"
   ],
   [
    "C142",
    "x_vars[8,3]"
   ],
   [
    "C143",
    "x_vars[8,4]"
   ],
   [
    "C144",
    "x_vars[8,5]"
   ],
   [
    "C145",
    "x_vars[8,6]"
   ],
   [
    "C146",
    "x_vars[8,7]"
   ],
   [
    "C147",
    "x_vars[8,8]"
   ],
   [
    "C148",
    "x_vars[8,9]"
   ],
   [
    "C149",
    "x_vars[8,10]"
   ],
   [
    "C150",
    "x_vars[8,11]"
   ],
   [
    "C151",
    "x_vars[8,12]"
   ],
   [
    "C152",
    "x_vars[8,13]"
   ],
   [
    "C153",
    "x_vars[8,14]"
   ],
   [
    "C154",
    "x_vars[8,15]"
   ],
   [
    "C155",
    "x_vars[8,16]"
   ],
   [
    "C156",
    "x_vars[8,17]"
   ],
   [
    "C157",
    "x_vars[8,18]"
   ],
   [
    "C158",
    "x_vars[8,19]"
   ],
   [
    "C159",
    "x_vars[8,20]"
   ],
   [
    "C160",
    "x_vars[9,1]"
   ],
   [
    "C161",
    "x_vars[9,2]"
   ],
   [
    "C162",
    "x_vars[9,3]"
   ],
   [
    "C163",
    "x_vars[9,4]"
   ],
   [
    "C164",
    "x_vars[9,5]"
   ],
   [
    "C165",
    "x_vars[9,6]"
   ],
   [
    "C166",
    "x_vars[9,7]"
   ],
   [
    "C167",
    "x_vars[9,8]"
   ],
   [
    "C168",
    "x_vars[9,9]"
   ],
   [
    "C169",
    "x_vars[9,10]"
   ],
   [
    "C170",
    "x_vars[9,11]"
   ],
   [
    "C171",
    "x_vars[9,12]"
   ],
   [
    "C172",
    "x_vars[9,13]"
   ],
   [
    "C173",
    "x_vars[9,14]"
   ],
   [
    "C174",
    "x_vars[9,15]"
   ],
   [
    "C175",
    "x_vars[9,16]"
   ],
   [
    "C176",
    "x_vars[9,17]"
   ],
   [
    "C177",
    "x_vars[9,18]"
   ],
   [
    "C178",
    "x_vars[9,19]"
   ],
   [
    "C179",
    "x_vars[9,20]"
   ],
   [
    "C180",
    "x_vars[10,1]"
   ],
   [
    "C181",
    "x_vars[10,2]"
   ],
   [
    "C182",
    "x_vars[10,3]"
   ],
   [
    "C183",
    "x_vars[10,4]"
   ],
   [
    "C184",
    "x_vars[10,5]"
   ],
   [
    "C185",
    "x_vars[10,6]"
   ],
   [
    "C186",
    "x_vars[10,7]"
   ],
   [
    "C187",
    "x_vars[10,8]"
   ],
   [
    "C188",
    "x_vars[10,9]"
   ],
   [
    "C189",
    "x_vars[10,10]"
   ],
   [
    "C190",
    "x_vars[10,11]"
   ],
   [
    "C191",
    "x_vars[10,12]"
   ],
   [
    "C192",
    "x_vars[10,13]"
   ],
   [
    "C193",
    "x_vars[10,14]"
   ],
   [
    "C194",
    "x_vars[10,15]"
   ],
   [
    "C195",
    "x_vars[10,16]"
   ],
   [
    "C196",
    "x_vars[10,17]"
   ],
   [
    "C197",
    "x_vars[10,18]"
   ],
   [
    "C198",
    "x_vars[10,19]"
   ],
   [
    "C199",
    "x_vars[10,20]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,1]",
   "x_0_1": "x_vars[2,1]",
   "x_0_2": "x_vars[3,1]",
   "x_0_3": "x_vars[4,1]",
   "x_0_4": "x_vars[5,1]",
   "x_0_5": "x_vars[6,1]",
   "x_0_6": "x_vars[7,1]",
   "x_0_7": "x_vars[8,1]",
   "x_0_8": "x_vars[9,1]",
   "x_0_9": "x_vars[10,1]",
   "x_1_0": "x_vars[1,2]",
   "x_1_1": "x_vars[2,2]",
   "x_1_2": "x_vars[3,2]",
   "x_1_3": "x_vars[4,2]",
   "x_1_4": "x_vars[5,2]",
   "x_1_5": "x_vars[6,2]",
   "x_1_6": "x_vars[7,2]",
   "x_1_7": "x_vars[8,2]",
   "x_1_8": "x_vars[9,2]",
   "x_1_9": "x_vars[10,2]",
   "x_2_0": "x_vars[1,3]",
   "x_2_1": "x_vars[2,3]",
   "x_2_2": "x_vars[3,3]",
   "x_2_3": "x_vars[4,3]",
   "x_2_4": "x_vars[5,3]",
   "x_2_5": "x_vars[6,3]",
   "x_2_6": "x_vars[7,3]",
   "x_2_7": "x_vars[8,3]",
   "x_2_8": "x_vars[9,3]",
   "x_2_9": "x_vars[10,3]",
   "x_3_0": "x_vars[1,4]",
   "x_3_1": "x_vars[2,4]",
   "x_3_2": "x_vars[3,4]",
   "x_3_3": "x_vars[4,4]",
   "x_3_4": "x_vars[5,4]",
   "x_3_5": "x_vars[6,4]",
   "x_3_6": "x_vars[7,4]",
   "x_3_7": "x_vars[8,4]",
   "x_3_8": "x_vars[9,4]",
   "x_3_9": "x_vars[10,4]",
   "x_4_0": "x_vars[1,5]",
   "x_4_1": "x_vars[2,5]",
   "x_4_2": "x_vars[3,5]",
   "x_4_3": "x_vars[4,5]",
   "x_4_4": "x_vars[5,5]",
   "x_4_5": "x_vars[6,5]",
   "x_4_6": "x_vars[7,5]",
   "x_4_7": "x_vars[8,5]",
   "x_4_8": "x_vars[9,5]",
   "x_4_9": "x_vars[10,5]",
   "x_5_0": "x_vars[1,6]",
   "x_5_1": "x_vars[2,6]",
   "x_5_2": "x_vars[3,6]",
   "x_5_3": "x_vars[4,6]",
   "x_5_4": "x_vars[5,6]",
   "x_5_5": "x_vars[6,6]",
   "x_5_6": "x_vars[7,6]",
   "x_5_7": "x_vars[8,6]",
   "x_5_8": "x_vars[9,6]",
   "x_5_9": "x_vars[10,6]",
   "x_6_0": "x_vars[1,7]",
   "x_6_1": "x_vars[2,7]",
   "x_6_2": "x_vars[3,7]",
   "x_6_3": "x_vars[4,7]",
   "x_6_4": "x_vars[5,7]",
   "x_6_5": "x_vars[6,7]",
   "x_6_6": "x_vars[7,7]",
   "x_6_7": "x_vars[8,7]",
   "x_6_8": "x_vars[9,7]",
   "x_6_9": "x_vars[10,7]",
   "x_7_0": "x_vars[1,8]",
   "x_7_1": "x_vars[2,8]",
   "x_7_2": "x_vars[3,8]",
   "x_7_3": "x_vars[4,8]",
   "x_7_4": "x_vars[5,8]",
   "x_7_5": "x_vars[6,8]",
   "x_7_6": "x_vars[7,8]",
   "x_7_7": "x_vars[8,8]",
   "x_7_8": "x_vars[9,8]",
   "x_7_9": "x_vars[10,8]",
   "x_8_0": "x_vars[1,9]",
   "x_8_1": "x_vars[2,9]",
   "x_8_2": "x_vars[3,9]",
   "x_8_3": "x_vars[4,9]",
   "x_8_4": "x_vars[5,9]",
   "x_8_5": "x_vars[6,9]",
   "x_8_6": "x_vars[7,9]",
   "x_8_7": "x_vars[8,9]",
   "x_8_8": "x_vars[9,9]",
   "x_8_9": "x_vars[10,9]",
   "x_9_0": "x_vars[1,10]",
   "x_9_1": "x_vars[2,10]",
   "x_9_2": "x_vars[3,10]",
   "x_9_3": "x_vars[4,10]",
   "x_9_4": "x_vars[5,10]",
   "x_9_5": "x_vars[6,10]",
   "x_9_6": "x_vars[7,10]",
   "x_9_7": "x_vars[8,10]",
   "x_9_8": "x_vars[9,10]",
   "x_9_9": "x_vars[10,10]",
   "x_10_0": "x_vars[1,11]",
   "x_10_1": "x_vars[2,11]",
   "x_10_2": "x_vars[3,11]",
   "x_10_3": "x_vars[4,11]",
   "x_10_4": "x_vars[5,11]",
   "x_10_5": "x_vars[6,11]",
   "x_10_6": "x_vars[7,11]",
   "x_10_7": "x_vars[8,11]",
   "x_10_8": "x_vars[9,11]",
   "x_10_9": "x_vars[10,11]",
   "x_11_0": "x_vars[1,12]",
   "x_11_1": "x_vars[2,12]",
   "x_11_2": "x_vars[3,12]",
   "x_11_3": "x_vars[4,12]",
   "x_11_4": "x_vars[5,12]",
   "x_11_5": "x_vars[6,12]",
   "x_11_6": "x_vars[7,12]",
   "x_11_7": "x_vars[8,12]",
   "x_11_8": "x_vars[9,12]",
   "x_11_9": "x_vars[10,12]",
   "x_12_0": "x_vars[1,13]",
   "x_12_1": "x_vars[2,13]",
   "x_12_2": "x_vars[3,13]",
   "x_12_3": "x_vars[4,13]",
   "x_12_4": "x_vars[5,13]",
   "x_12_5": "x_vars[6,13]",
   "x_12_6": "x_vars[7,13]",
   "x_12_7": "x_vars[8,13]",
   "x_12_8": "x_vars[9,13]",
   "x_12_9": "x_vars[10,13]",
   "x_13_0": "x_vars[1,14]",
   "x_13_1": "x_vars[2,14]",
   "x_13_2": "x_vars[3,14]",
   "x_13_3": "x_vars[4,14]",
   "x_13_4": "x_vars[5,14]",
   "x_13_5": "x_vars[6,14]",
   "x_13_6": "x_vars[7,14]",
   "x_13_7": "x_vars[8,14]",
   "x_13_8": "x_vars[9,14]",
   "x_13_9": "x_vars[10,14]",
   "x_14_0": "x_vars[1,15]",
   "x_14_1": "x_vars[2,15]",
   "x_14_2": "x_vars[3,15]",
   "x_14_3": "x_vars[4,15]",
   "x_14_4": "x_vars[5,15]",
   "x_14_5": "x_vars[6,15]",
   "x_14_6": "x_vars[7,15]",
   "x_14_7": "x_vars[8,15]",
   "x_14_8": "x_vars[9,15]",
   "x_14_9": "x_vars[10,15]",
   "x_15_0": "x_vars[1,16]",
   "x_15_1": "x_vars[2,16]",
   "x_15_2": "x_vars[3,16]",
   "x_15_3": "x_vars[4,16]",
   "x_15_4": "x_vars[5,16]",
   "x_15_5": "x_vars[6,16]",
   "x_15_6": "x_vars[7,16]",
   "x_15_7": "x_vars[8,16]",
   "x_15_8": "x_vars[9,16]",
   "x_15_9": "x_vars[10,16]",
   "x_16_0": "x_vars[1,17]",
   "x_16_1": "x_vars[2,17]",
   "x_16_2": "x_vars[3,17]",
   "x_16_3": "x_vars[4,17]",
   "x_16_4": "x_vars[5,17]",
   "x_16_5": "x_vars[6,17]",
   "x_16_6": "x_vars[7,17]",
   "x_16_7": "x_vars[8,17]",
   "x_16_8": "x_vars[9,17]",
   "x_16_9": "x_vars[10,17]",
   "x_17_0": "x_vars[1,18]",
   "x_17_1": "x_vars[2,18]",
   "x_17_2": "x_vars[3,18]",
   "x_17_3": "x_vars[4,18]",
   "x_17_4": "x_vars[5,18]",
   "x_17_5": "x_vars[6,18]",
   "x_17_6": "x_vars[7,18]",
   "x_17_7": "x_vars[8,18]",
   "x_17_8": "x_vars[9,18]",
   "x_17_9": "x_vars[10,18]",
   "x_18_0": "x_vars[1,19]",
   "x_18_1": "x_vars[2,19]",
   "x_18_2": "x_vars[3,19]",
   "x_18_3": "x_vars[4,19]",
   "x_18_4": "x_vars[5,19]",
   "x_18_5": "x_vars[6,19]",
   "x_18_6": "x_vars[7,19]",
   "x_18_7": "x_vars[8,19]",
   "x_18_8": "x_vars[9,19]",
   "x_18_9": "x_vars[10,19]",
   "x_19_0": "x_vars[1,20]",
   "x_19_1": "x_vars[2,20]",
   "x_19_2": "x_vars[3,20]",
   "x_19_3": "x_vars[4,20]",
   "x_19_4": "x_vars[5,20]",
   "x_19_5": "x_vars[6,20]",
   "x_19_6": "x_vars[7,20]",
   "x_19_7": "x_vars[8,20]",
   "x_19_8": "x_vars[9,20]",
   "x_19_9": "x_vars[10,20]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-055": {
  "source_code_sha256": "e4a0b8ce7cb1744ab2cfce6787b468cdef57273c21eac12506ecc49d0e5afc51",
  "original_solution_sha256": "78f1a854ceafe2eef30ae1d5063b9311ce898d0af26b10742e6b9875eeb2117a",
  "bindings": [
   [
    "C0",
    "x_vars[1,Speedboat]"
   ],
   [
    "C1",
    "x_vars[1,Fishing Boat]"
   ],
   [
    "C2",
    "x_vars[1,Catamaran]"
   ],
   [
    "C3",
    "x_vars[1,Yacht]"
   ],
   [
    "C4",
    "x_vars[1,Sailboat]"
   ],
   [
    "C5",
    "x_vars[1,Kayak]"
   ],
   [
    "C6",
    "x_vars[1,Canoe]"
   ],
   [
    "C7",
    "x_vars[1,Houseboat]"
   ],
   [
    "C8",
    "x_vars[1,Pontoon]"
   ],
   [
    "C9",
    "x_vars[1,Jet Ski]"
   ],
   [
    "C10",
    "x_vars[1,Rowboat]"
   ],
   [
    "C11",
    "x_vars[1,Hovercraft]"
   ],
   [
    "C12",
    "x_vars[1,Cabin Cruiser]"
   ],
   [
    "C13",
    "x_vars[1,Wakeboard Boat]"
   ],
   [
    "C14",
    "x_vars[1,Dinghy]"
   ],
   [
    "C15",
    "x_vars[1,Trawler]"
   ],
   [
    "C16",
    "x_vars[1,Paddle Boat]"
   ],
   [
    "C17",
    "x_vars[1,Submarine]"
   ],
   [
    "C18",
    "x_vars[1,RIB]"
   ],
   [
    "C19",
    "x_vars[1,Skiff]"
   ],
   [
    "C20",
    "x_vars[2,Speedboat]"
   ],
   [
    "C21",
    "x_vars[2,Fishing Boat]"
   ],
   [
    "C22",
    "x_vars[2,Catamaran]"
   ],
   [
    "C23",
    "x_vars[2,Yacht]"
   ],
   [
    "C24",
    "x_vars[2,Sailboat]"
   ],
   [
    "C25",
    "x_vars[2,Kayak]"
   ],
   [
    "C26",
    "x_vars[2,Canoe]"
   ],
   [
    "C27",
    "x_vars[2,Houseboat]"
   ],
   [
    "C28",
    "x_vars[2,Pontoon]"
   ],
   [
    "C29",
    "x_vars[2,Jet Ski]"
   ],
   [
    "C30",
    "x_vars[2,Rowboat]"
   ],
   [
    "C31",
    "x_vars[2,Hovercraft]"
   ],
   [
    "C32",
    "x_vars[2,Cabin Cruiser]"
   ],
   [
    "C33",
    "x_vars[2,Wakeboard Boat]"
   ],
   [
    "C34",
    "x_vars[2,Dinghy]"
   ],
   [
    "C35",
    "x_vars[2,Trawler]"
   ],
   [
    "C36",
    "x_vars[2,Paddle Boat]"
   ],
   [
    "C37",
    "x_vars[2,Submarine]"
   ],
   [
    "C38",
    "x_vars[2,RIB]"
   ],
   [
    "C39",
    "x_vars[2,Skiff]"
   ],
   [
    "C40",
    "x_vars[3,Speedboat]"
   ],
   [
    "C41",
    "x_vars[3,Fishing Boat]"
   ],
   [
    "C42",
    "x_vars[3,Catamaran]"
   ],
   [
    "C43",
    "x_vars[3,Yacht]"
   ],
   [
    "C44",
    "x_vars[3,Sailboat]"
   ],
   [
    "C45",
    "x_vars[3,Kayak]"
   ],
   [
    "C46",
    "x_vars[3,Canoe]"
   ],
   [
    "C47",
    "x_vars[3,Houseboat]"
   ],
   [
    "C48",
    "x_vars[3,Pontoon]"
   ],
   [
    "C49",
    "x_vars[3,Jet Ski]"
   ],
   [
    "C50",
    "x_vars[3,Rowboat]"
   ],
   [
    "C51",
    "x_vars[3,Hovercraft]"
   ],
   [
    "C52",
    "x_vars[3,Cabin Cruiser]"
   ],
   [
    "C53",
    "x_vars[3,Wakeboard Boat]"
   ],
   [
    "C54",
    "x_vars[3,Dinghy]"
   ],
   [
    "C55",
    "x_vars[3,Trawler]"
   ],
   [
    "C56",
    "x_vars[3,Paddle Boat]"
   ],
   [
    "C57",
    "x_vars[3,Submarine]"
   ],
   [
    "C58",
    "x_vars[3,RIB]"
   ],
   [
    "C59",
    "x_vars[3,Skiff]"
   ],
   [
    "C60",
    "x_vars[4,Speedboat]"
   ],
   [
    "C61",
    "x_vars[4,Fishing Boat]"
   ],
   [
    "C62",
    "x_vars[4,Catamaran]"
   ],
   [
    "C63",
    "x_vars[4,Yacht]"
   ],
   [
    "C64",
    "x_vars[4,Sailboat]"
   ],
   [
    "C65",
    "x_vars[4,Kayak]"
   ],
   [
    "C66",
    "x_vars[4,Canoe]"
   ],
   [
    "C67",
    "x_vars[4,Houseboat]"
   ],
   [
    "C68",
    "x_vars[4,Pontoon]"
   ],
   [
    "C69",
    "x_vars[4,Jet Ski]"
   ],
   [
    "C70",
    "x_vars[4,Rowboat]"
   ],
   [
    "C71",
    "x_vars[4,Hovercraft]"
   ],
   [
    "C72",
    "x_vars[4,Cabin Cruiser]"
   ],
   [
    "C73",
    "x_vars[4,Wakeboard Boat]"
   ],
   [
    "C74",
    "x_vars[4,Dinghy]"
   ],
   [
    "C75",
    "x_vars[4,Trawler]"
   ],
   [
    "C76",
    "x_vars[4,Paddle Boat]"
   ],
   [
    "C77",
    "x_vars[4,Submarine]"
   ],
   [
    "C78",
    "x_vars[4,RIB]"
   ],
   [
    "C79",
    "x_vars[4,Skiff]"
   ],
   [
    "C80",
    "x_vars[5,Speedboat]"
   ],
   [
    "C81",
    "x_vars[5,Fishing Boat]"
   ],
   [
    "C82",
    "x_vars[5,Catamaran]"
   ],
   [
    "C83",
    "x_vars[5,Yacht]"
   ],
   [
    "C84",
    "x_vars[5,Sailboat]"
   ],
   [
    "C85",
    "x_vars[5,Kayak]"
   ],
   [
    "C86",
    "x_vars[5,Canoe]"
   ],
   [
    "C87",
    "x_vars[5,Houseboat]"
   ],
   [
    "C88",
    "x_vars[5,Pontoon]"
   ],
   [
    "C89",
    "x_vars[5,Jet Ski]"
   ],
   [
    "C90",
    "x_vars[5,Rowboat]"
   ],
   [
    "C91",
    "x_vars[5,Hovercraft]"
   ],
   [
    "C92",
    "x_vars[5,Cabin Cruiser]"
   ],
   [
    "C93",
    "x_vars[5,Wakeboard Boat]"
   ],
   [
    "C94",
    "x_vars[5,Dinghy]"
   ],
   [
    "C95",
    "x_vars[5,Trawler]"
   ],
   [
    "C96",
    "x_vars[5,Paddle Boat]"
   ],
   [
    "C97",
    "x_vars[5,Submarine]"
   ],
   [
    "C98",
    "x_vars[5,RIB]"
   ],
   [
    "C99",
    "x_vars[5,Skiff]"
   ],
   [
    "C100",
    "x_vars[6,Speedboat]"
   ],
   [
    "C101",
    "x_vars[6,Fishing Boat]"
   ],
   [
    "C102",
    "x_vars[6,Catamaran]"
   ],
   [
    "C103",
    "x_vars[6,Yacht]"
   ],
   [
    "C104",
    "x_vars[6,Sailboat]"
   ],
   [
    "C105",
    "x_vars[6,Kayak]"
   ],
   [
    "C106",
    "x_vars[6,Canoe]"
   ],
   [
    "C107",
    "x_vars[6,Houseboat]"
   ],
   [
    "C108",
    "x_vars[6,Pontoon]"
   ],
   [
    "C109",
    "x_vars[6,Jet Ski]"
   ],
   [
    "C110",
    "x_vars[6,Rowboat]"
   ],
   [
    "C111",
    "x_vars[6,Hovercraft]"
   ],
   [
    "C112",
    "x_vars[6,Cabin Cruiser]"
   ],
   [
    "C113",
    "x_vars[6,Wakeboard Boat]"
   ],
   [
    "C114",
    "x_vars[6,Dinghy]"
   ],
   [
    "C115",
    "x_vars[6,Trawler]"
   ],
   [
    "C116",
    "x_vars[6,Paddle Boat]"
   ],
   [
    "C117",
    "x_vars[6,Submarine]"
   ],
   [
    "C118",
    "x_vars[6,RIB]"
   ],
   [
    "C119",
    "x_vars[6,Skiff]"
   ],
   [
    "C120",
    "x_vars[7,Speedboat]"
   ],
   [
    "C121",
    "x_vars[7,Fishing Boat]"
   ],
   [
    "C122",
    "x_vars[7,Catamaran]"
   ],
   [
    "C123",
    "x_vars[7,Yacht]"
   ],
   [
    "C124",
    "x_vars[7,Sailboat]"
   ],
   [
    "C125",
    "x_vars[7,Kayak]"
   ],
   [
    "C126",
    "x_vars[7,Canoe]"
   ],
   [
    "C127",
    "x_vars[7,Houseboat]"
   ],
   [
    "C128",
    "x_vars[7,Pontoon]"
   ],
   [
    "C129",
    "x_vars[7,Jet Ski]"
   ],
   [
    "C130",
    "x_vars[7,Rowboat]"
   ],
   [
    "C131",
    "x_vars[7,Hovercraft]"
   ],
   [
    "C132",
    "x_vars[7,Cabin Cruiser]"
   ],
   [
    "C133",
    "x_vars[7,Wakeboard Boat]"
   ],
   [
    "C134",
    "x_vars[7,Dinghy]"
   ],
   [
    "C135",
    "x_vars[7,Trawler]"
   ],
   [
    "C136",
    "x_vars[7,Paddle Boat]"
   ],
   [
    "C137",
    "x_vars[7,Submarine]"
   ],
   [
    "C138",
    "x_vars[7,RIB]"
   ],
   [
    "C139",
    "x_vars[7,Skiff]"
   ],
   [
    "C140",
    "x_vars[8,Speedboat]"
   ],
   [
    "C141",
    "x_vars[8,Fishing Boat]"
   ],
   [
    "C142",
    "x_vars[8,Catamaran]"
   ],
   [
    "C143",
    "x_vars[8,Yacht]"
   ],
   [
    "C144",
    "x_vars[8,Sailboat]"
   ],
   [
    "C145",
    "x_vars[8,Kayak]"
   ],
   [
    "C146",
    "x_vars[8,Canoe]"
   ],
   [
    "C147",
    "x_vars[8,Houseboat]"
   ],
   [
    "C148",
    "x_vars[8,Pontoon]"
   ],
   [
    "C149",
    "x_vars[8,Jet Ski]"
   ],
   [
    "C150",
    "x_vars[8,Rowboat]"
   ],
   [
    "C151",
    "x_vars[8,Hovercraft]"
   ],
   [
    "C152",
    "x_vars[8,Cabin Cruiser]"
   ],
   [
    "C153",
    "x_vars[8,Wakeboard Boat]"
   ],
   [
    "C154",
    "x_vars[8,Dinghy]"
   ],
   [
    "C155",
    "x_vars[8,Trawler]"
   ],
   [
    "C156",
    "x_vars[8,Paddle Boat]"
   ],
   [
    "C157",
    "x_vars[8,Submarine]"
   ],
   [
    "C158",
    "x_vars[8,RIB]"
   ],
   [
    "C159",
    "x_vars[8,Skiff]"
   ],
   [
    "C160",
    "x_vars[9,Speedboat]"
   ],
   [
    "C161",
    "x_vars[9,Fishing Boat]"
   ],
   [
    "C162",
    "x_vars[9,Catamaran]"
   ],
   [
    "C163",
    "x_vars[9,Yacht]"
   ],
   [
    "C164",
    "x_vars[9,Sailboat]"
   ],
   [
    "C165",
    "x_vars[9,Kayak]"
   ],
   [
    "C166",
    "x_vars[9,Canoe]"
   ],
   [
    "C167",
    "x_vars[9,Houseboat]"
   ],
   [
    "C168",
    "x_vars[9,Pontoon]"
   ],
   [
    "C169",
    "x_vars[9,Jet Ski]"
   ],
   [
    "C170",
    "x_vars[9,Rowboat]"
   ],
   [
    "C171",
    "x_vars[9,Hovercraft]"
   ],
   [
    "C172",
    "x_vars[9,Cabin Cruiser]"
   ],
   [
    "C173",
    "x_vars[9,Wakeboard Boat]"
   ],
   [
    "C174",
    "x_vars[9,Dinghy]"
   ],
   [
    "C175",
    "x_vars[9,Trawler]"
   ],
   [
    "C176",
    "x_vars[9,Paddle Boat]"
   ],
   [
    "C177",
    "x_vars[9,Submarine]"
   ],
   [
    "C178",
    "x_vars[9,RIB]"
   ],
   [
    "C179",
    "x_vars[9,Skiff]"
   ],
   [
    "C180",
    "x_vars[10,Speedboat]"
   ],
   [
    "C181",
    "x_vars[10,Fishing Boat]"
   ],
   [
    "C182",
    "x_vars[10,Catamaran]"
   ],
   [
    "C183",
    "x_vars[10,Yacht]"
   ],
   [
    "C184",
    "x_vars[10,Sailboat]"
   ],
   [
    "C185",
    "x_vars[10,Kayak]"
   ],
   [
    "C186",
    "x_vars[10,Canoe]"
   ],
   [
    "C187",
    "x_vars[10,Houseboat]"
   ],
   [
    "C188",
    "x_vars[10,Pontoon]"
   ],
   [
    "C189",
    "x_vars[10,Jet Ski]"
   ],
   [
    "C190",
    "x_vars[10,Rowboat]"
   ],
   [
    "C191",
    "x_vars[10,Hovercraft]"
   ],
   [
    "C192",
    "x_vars[10,Cabin Cruiser]"
   ],
   [
    "C193",
    "x_vars[10,Wakeboard Boat]"
   ],
   [
    "C194",
    "x_vars[10,Dinghy]"
   ],
   [
    "C195",
    "x_vars[10,Trawler]"
   ],
   [
    "C196",
    "x_vars[10,Paddle Boat]"
   ],
   [
    "C197",
    "x_vars[10,Submarine]"
   ],
   [
    "C198",
    "x_vars[10,RIB]"
   ],
   [
    "C199",
    "x_vars[10,Skiff]"
   ],
   [
    "C200",
    "x_vars[11,Speedboat]"
   ],
   [
    "C201",
    "x_vars[11,Fishing Boat]"
   ],
   [
    "C202",
    "x_vars[11,Catamaran]"
   ],
   [
    "C203",
    "x_vars[11,Yacht]"
   ],
   [
    "C204",
    "x_vars[11,Sailboat]"
   ],
   [
    "C205",
    "x_vars[11,Kayak]"
   ],
   [
    "C206",
    "x_vars[11,Canoe]"
   ],
   [
    "C207",
    "x_vars[11,Houseboat]"
   ],
   [
    "C208",
    "x_vars[11,Pontoon]"
   ],
   [
    "C209",
    "x_vars[11,Jet Ski]"
   ],
   [
    "C210",
    "x_vars[11,Rowboat]"
   ],
   [
    "C211",
    "x_vars[11,Hovercraft]"
   ],
   [
    "C212",
    "x_vars[11,Cabin Cruiser]"
   ],
   [
    "C213",
    "x_vars[11,Wakeboard Boat]"
   ],
   [
    "C214",
    "x_vars[11,Dinghy]"
   ],
   [
    "C215",
    "x_vars[11,Trawler]"
   ],
   [
    "C216",
    "x_vars[11,Paddle Boat]"
   ],
   [
    "C217",
    "x_vars[11,Submarine]"
   ],
   [
    "C218",
    "x_vars[11,RIB]"
   ],
   [
    "C219",
    "x_vars[11,Skiff]"
   ],
   [
    "C220",
    "x_vars[12,Speedboat]"
   ],
   [
    "C221",
    "x_vars[12,Fishing Boat]"
   ],
   [
    "C222",
    "x_vars[12,Catamaran]"
   ],
   [
    "C223",
    "x_vars[12,Yacht]"
   ],
   [
    "C224",
    "x_vars[12,Sailboat]"
   ],
   [
    "C225",
    "x_vars[12,Kayak]"
   ],
   [
    "C226",
    "x_vars[12,Canoe]"
   ],
   [
    "C227",
    "x_vars[12,Houseboat]"
   ],
   [
    "C228",
    "x_vars[12,Pontoon]"
   ],
   [
    "C229",
    "x_vars[12,Jet Ski]"
   ],
   [
    "C230",
    "x_vars[12,Rowboat]"
   ],
   [
    "C231",
    "x_vars[12,Hovercraft]"
   ],
   [
    "C232",
    "x_vars[12,Cabin Cruiser]"
   ],
   [
    "C233",
    "x_vars[12,Wakeboard Boat]"
   ],
   [
    "C234",
    "x_vars[12,Dinghy]"
   ],
   [
    "C235",
    "x_vars[12,Trawler]"
   ],
   [
    "C236",
    "x_vars[12,Paddle Boat]"
   ],
   [
    "C237",
    "x_vars[12,Submarine]"
   ],
   [
    "C238",
    "x_vars[12,RIB]"
   ],
   [
    "C239",
    "x_vars[12,Skiff]"
   ],
   [
    "C240",
    "x_vars[13,Speedboat]"
   ],
   [
    "C241",
    "x_vars[13,Fishing Boat]"
   ],
   [
    "C242",
    "x_vars[13,Catamaran]"
   ],
   [
    "C243",
    "x_vars[13,Yacht]"
   ],
   [
    "C244",
    "x_vars[13,Sailboat]"
   ],
   [
    "C245",
    "x_vars[13,Kayak]"
   ],
   [
    "C246",
    "x_vars[13,Canoe]"
   ],
   [
    "C247",
    "x_vars[13,Houseboat]"
   ],
   [
    "C248",
    "x_vars[13,Pontoon]"
   ],
   [
    "C249",
    "x_vars[13,Jet Ski]"
   ],
   [
    "C250",
    "x_vars[13,Rowboat]"
   ],
   [
    "C251",
    "x_vars[13,Hovercraft]"
   ],
   [
    "C252",
    "x_vars[13,Cabin Cruiser]"
   ],
   [
    "C253",
    "x_vars[13,Wakeboard Boat]"
   ],
   [
    "C254",
    "x_vars[13,Dinghy]"
   ],
   [
    "C255",
    "x_vars[13,Trawler]"
   ],
   [
    "C256",
    "x_vars[13,Paddle Boat]"
   ],
   [
    "C257",
    "x_vars[13,Submarine]"
   ],
   [
    "C258",
    "x_vars[13,RIB]"
   ],
   [
    "C259",
    "x_vars[13,Skiff]"
   ],
   [
    "C260",
    "x_vars[14,Speedboat]"
   ],
   [
    "C261",
    "x_vars[14,Fishing Boat]"
   ],
   [
    "C262",
    "x_vars[14,Catamaran]"
   ],
   [
    "C263",
    "x_vars[14,Yacht]"
   ],
   [
    "C264",
    "x_vars[14,Sailboat]"
   ],
   [
    "C265",
    "x_vars[14,Kayak]"
   ],
   [
    "C266",
    "x_vars[14,Canoe]"
   ],
   [
    "C267",
    "x_vars[14,Houseboat]"
   ],
   [
    "C268",
    "x_vars[14,Pontoon]"
   ],
   [
    "C269",
    "x_vars[14,Jet Ski]"
   ],
   [
    "C270",
    "x_vars[14,Rowboat]"
   ],
   [
    "C271",
    "x_vars[14,Hovercraft]"
   ],
   [
    "C272",
    "x_vars[14,Cabin Cruiser]"
   ],
   [
    "C273",
    "x_vars[14,Wakeboard Boat]"
   ],
   [
    "C274",
    "x_vars[14,Dinghy]"
   ],
   [
    "C275",
    "x_vars[14,Trawler]"
   ],
   [
    "C276",
    "x_vars[14,Paddle Boat]"
   ],
   [
    "C277",
    "x_vars[14,Submarine]"
   ],
   [
    "C278",
    "x_vars[14,RIB]"
   ],
   [
    "C279",
    "x_vars[14,Skiff]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Speedboat]",
   "x_0_1": "x_vars[2,Speedboat]",
   "x_0_2": "x_vars[3,Speedboat]",
   "x_0_3": "x_vars[4,Speedboat]",
   "x_0_4": "x_vars[5,Speedboat]",
   "x_0_5": "x_vars[6,Speedboat]",
   "x_0_6": "x_vars[7,Speedboat]",
   "x_0_7": "x_vars[8,Speedboat]",
   "x_0_8": "x_vars[9,Speedboat]",
   "x_0_9": "x_vars[10,Speedboat]",
   "x_0_10": "x_vars[11,Speedboat]",
   "x_0_11": "x_vars[12,Speedboat]",
   "x_0_12": "x_vars[13,Speedboat]",
   "x_0_13": "x_vars[14,Speedboat]",
   "x_1_0": "x_vars[1,Fishing Boat]",
   "x_1_1": "x_vars[2,Fishing Boat]",
   "x_1_2": "x_vars[3,Fishing Boat]",
   "x_1_3": "x_vars[4,Fishing Boat]",
   "x_1_4": "x_vars[5,Fishing Boat]",
   "x_1_5": "x_vars[6,Fishing Boat]",
   "x_1_6": "x_vars[7,Fishing Boat]",
   "x_1_7": "x_vars[8,Fishing Boat]",
   "x_1_8": "x_vars[9,Fishing Boat]",
   "x_1_9": "x_vars[10,Fishing Boat]",
   "x_1_10": "x_vars[11,Fishing Boat]",
   "x_1_11": "x_vars[12,Fishing Boat]",
   "x_1_12": "x_vars[13,Fishing Boat]",
   "x_1_13": "x_vars[14,Fishing Boat]",
   "x_2_0": "x_vars[1,Catamaran]",
   "x_2_1": "x_vars[2,Catamaran]",
   "x_2_2": "x_vars[3,Catamaran]",
   "x_2_3": "x_vars[4,Catamaran]",
   "x_2_4": "x_vars[5,Catamaran]",
   "x_2_5": "x_vars[6,Catamaran]",
   "x_2_6": "x_vars[7,Catamaran]",
   "x_2_7": "x_vars[8,Catamaran]",
   "x_2_8": "x_vars[9,Catamaran]",
   "x_2_9": "x_vars[10,Catamaran]",
   "x_2_10": "x_vars[11,Catamaran]",
   "x_2_11": "x_vars[12,Catamaran]",
   "x_2_12": "x_vars[13,Catamaran]",
   "x_2_13": "x_vars[14,Catamaran]",
   "x_3_0": "x_vars[1,Yacht]",
   "x_3_1": "x_vars[2,Yacht]",
   "x_3_2": "x_vars[3,Yacht]",
   "x_3_3": "x_vars[4,Yacht]",
   "x_3_4": "x_vars[5,Yacht]",
   "x_3_5": "x_vars[6,Yacht]",
   "x_3_6": "x_vars[7,Yacht]",
   "x_3_7": "x_vars[8,Yacht]",
   "x_3_8": "x_vars[9,Yacht]",
   "x_3_9": "x_vars[10,Yacht]",
   "x_3_10": "x_vars[11,Yacht]",
   "x_3_11": "x_vars[12,Yacht]",
   "x_3_12": "x_vars[13,Yacht]",
   "x_3_13": "x_vars[14,Yacht]",
   "x_4_0": "x_vars[1,Sailboat]",
   "x_4_1": "x_vars[2,Sailboat]",
   "x_4_2": "x_vars[3,Sailboat]",
   "x_4_3": "x_vars[4,Sailboat]",
   "x_4_4": "x_vars[5,Sailboat]",
   "x_4_5": "x_vars[6,Sailboat]",
   "x_4_6": "x_vars[7,Sailboat]",
   "x_4_7": "x_vars[8,Sailboat]",
   "x_4_8": "x_vars[9,Sailboat]",
   "x_4_9": "x_vars[10,Sailboat]",
   "x_4_10": "x_vars[11,Sailboat]",
   "x_4_11": "x_vars[12,Sailboat]",
   "x_4_12": "x_vars[13,Sailboat]",
   "x_4_13": "x_vars[14,Sailboat]",
   "x_5_0": "x_vars[1,Kayak]",
   "x_5_1": "x_vars[2,Kayak]",
   "x_5_2": "x_vars[3,Kayak]",
   "x_5_3": "x_vars[4,Kayak]",
   "x_5_4": "x_vars[5,Kayak]",
   "x_5_5": "x_vars[6,Kayak]",
   "x_5_6": "x_vars[7,Kayak]",
   "x_5_7": "x_vars[8,Kayak]",
   "x_5_8": "x_vars[9,Kayak]",
   "x_5_9": "x_vars[10,Kayak]",
   "x_5_10": "x_vars[11,Kayak]",
   "x_5_11": "x_vars[12,Kayak]",
   "x_5_12": "x_vars[13,Kayak]",
   "x_5_13": "x_vars[14,Kayak]",
   "x_6_0": "x_vars[1,Canoe]",
   "x_6_1": "x_vars[2,Canoe]",
   "x_6_2": "x_vars[3,Canoe]",
   "x_6_3": "x_vars[4,Canoe]",
   "x_6_4": "x_vars[5,Canoe]",
   "x_6_5": "x_vars[6,Canoe]",
   "x_6_6": "x_vars[7,Canoe]",
   "x_6_7": "x_vars[8,Canoe]",
   "x_6_8": "x_vars[9,Canoe]",
   "x_6_9": "x_vars[10,Canoe]",
   "x_6_10": "x_vars[11,Canoe]",
   "x_6_11": "x_vars[12,Canoe]",
   "x_6_12": "x_vars[13,Canoe]",
   "x_6_13": "x_vars[14,Canoe]",
   "x_7_0": "x_vars[1,Houseboat]",
   "x_7_1": "x_vars[2,Houseboat]",
   "x_7_2": "x_vars[3,Houseboat]",
   "x_7_3": "x_vars[4,Houseboat]",
   "x_7_4": "x_vars[5,Houseboat]",
   "x_7_5": "x_vars[6,Houseboat]",
   "x_7_6": "x_vars[7,Houseboat]",
   "x_7_7": "x_vars[8,Houseboat]",
   "x_7_8": "x_vars[9,Houseboat]",
   "x_7_9": "x_vars[10,Houseboat]",
   "x_7_10": "x_vars[11,Houseboat]",
   "x_7_11": "x_vars[12,Houseboat]",
   "x_7_12": "x_vars[13,Houseboat]",
   "x_7_13": "x_vars[14,Houseboat]",
   "x_8_0": "x_vars[1,Pontoon]",
   "x_8_1": "x_vars[2,Pontoon]",
   "x_8_2": "x_vars[3,Pontoon]",
   "x_8_3": "x_vars[4,Pontoon]",
   "x_8_4": "x_vars[5,Pontoon]",
   "x_8_5": "x_vars[6,Pontoon]",
   "x_8_6": "x_vars[7,Pontoon]",
   "x_8_7": "x_vars[8,Pontoon]",
   "x_8_8": "x_vars[9,Pontoon]",
   "x_8_9": "x_vars[10,Pontoon]",
   "x_8_10": "x_vars[11,Pontoon]",
   "x_8_11": "x_vars[12,Pontoon]",
   "x_8_12": "x_vars[13,Pontoon]",
   "x_8_13": "x_vars[14,Pontoon]",
   "x_9_0": "x_vars[1,Jet Ski]",
   "x_9_1": "x_vars[2,Jet Ski]",
   "x_9_2": "x_vars[3,Jet Ski]",
   "x_9_3": "x_vars[4,Jet Ski]",
   "x_9_4": "x_vars[5,Jet Ski]",
   "x_9_5": "x_vars[6,Jet Ski]",
   "x_9_6": "x_vars[7,Jet Ski]",
   "x_9_7": "x_vars[8,Jet Ski]",
   "x_9_8": "x_vars[9,Jet Ski]",
   "x_9_9": "x_vars[10,Jet Ski]",
   "x_9_10": "x_vars[11,Jet Ski]",
   "x_9_11": "x_vars[12,Jet Ski]",
   "x_9_12": "x_vars[13,Jet Ski]",
   "x_9_13": "x_vars[14,Jet Ski]",
   "x_10_0": "x_vars[1,Rowboat]",
   "x_10_1": "x_vars[2,Rowboat]",
   "x_10_2": "x_vars[3,Rowboat]",
   "x_10_3": "x_vars[4,Rowboat]",
   "x_10_4": "x_vars[5,Rowboat]",
   "x_10_5": "x_vars[6,Rowboat]",
   "x_10_6": "x_vars[7,Rowboat]",
   "x_10_7": "x_vars[8,Rowboat]",
   "x_10_8": "x_vars[9,Rowboat]",
   "x_10_9": "x_vars[10,Rowboat]",
   "x_10_10": "x_vars[11,Rowboat]",
   "x_10_11": "x_vars[12,Rowboat]",
   "x_10_12": "x_vars[13,Rowboat]",
   "x_10_13": "x_vars[14,Rowboat]",
   "x_11_0": "x_vars[1,Hovercraft]",
   "x_11_1": "x_vars[2,Hovercraft]",
   "x_11_2": "x_vars[3,Hovercraft]",
   "x_11_3": "x_vars[4,Hovercraft]",
   "x_11_4": "x_vars[5,Hovercraft]",
   "x_11_5": "x_vars[6,Hovercraft]",
   "x_11_6": "x_vars[7,Hovercraft]",
   "x_11_7": "x_vars[8,Hovercraft]",
   "x_11_8": "x_vars[9,Hovercraft]",
   "x_11_9": "x_vars[10,Hovercraft]",
   "x_11_10": "x_vars[11,Hovercraft]",
   "x_11_11": "x_vars[12,Hovercraft]",
   "x_11_12": "x_vars[13,Hovercraft]",
   "x_11_13": "x_vars[14,Hovercraft]",
   "x_12_0": "x_vars[1,Cabin Cruiser]",
   "x_12_1": "x_vars[2,Cabin Cruiser]",
   "x_12_2": "x_vars[3,Cabin Cruiser]",
   "x_12_3": "x_vars[4,Cabin Cruiser]",
   "x_12_4": "x_vars[5,Cabin Cruiser]",
   "x_12_5": "x_vars[6,Cabin Cruiser]",
   "x_12_6": "x_vars[7,Cabin Cruiser]",
   "x_12_7": "x_vars[8,Cabin Cruiser]",
   "x_12_8": "x_vars[9,Cabin Cruiser]",
   "x_12_9": "x_vars[10,Cabin Cruiser]",
   "x_12_10": "x_vars[11,Cabin Cruiser]",
   "x_12_11": "x_vars[12,Cabin Cruiser]",
   "x_12_12": "x_vars[13,Cabin Cruiser]",
   "x_12_13": "x_vars[14,Cabin Cruiser]",
   "x_13_0": "x_vars[1,Wakeboard Boat]",
   "x_13_1": "x_vars[2,Wakeboard Boat]",
   "x_13_2": "x_vars[3,Wakeboard Boat]",
   "x_13_3": "x_vars[4,Wakeboard Boat]",
   "x_13_4": "x_vars[5,Wakeboard Boat]",
   "x_13_5": "x_vars[6,Wakeboard Boat]",
   "x_13_6": "x_vars[7,Wakeboard Boat]",
   "x_13_7": "x_vars[8,Wakeboard Boat]",
   "x_13_8": "x_vars[9,Wakeboard Boat]",
   "x_13_9": "x_vars[10,Wakeboard Boat]",
   "x_13_10": "x_vars[11,Wakeboard Boat]",
   "x_13_11": "x_vars[12,Wakeboard Boat]",
   "x_13_12": "x_vars[13,Wakeboard Boat]",
   "x_13_13": "x_vars[14,Wakeboard Boat]",
   "x_14_0": "x_vars[1,Dinghy]",
   "x_14_1": "x_vars[2,Dinghy]",
   "x_14_2": "x_vars[3,Dinghy]",
   "x_14_3": "x_vars[4,Dinghy]",
   "x_14_4": "x_vars[5,Dinghy]",
   "x_14_5": "x_vars[6,Dinghy]",
   "x_14_6": "x_vars[7,Dinghy]",
   "x_14_7": "x_vars[8,Dinghy]",
   "x_14_8": "x_vars[9,Dinghy]",
   "x_14_9": "x_vars[10,Dinghy]",
   "x_14_10": "x_vars[11,Dinghy]",
   "x_14_11": "x_vars[12,Dinghy]",
   "x_14_12": "x_vars[13,Dinghy]",
   "x_14_13": "x_vars[14,Dinghy]",
   "x_15_0": "x_vars[1,Trawler]",
   "x_15_1": "x_vars[2,Trawler]",
   "x_15_2": "x_vars[3,Trawler]",
   "x_15_3": "x_vars[4,Trawler]",
   "x_15_4": "x_vars[5,Trawler]",
   "x_15_5": "x_vars[6,Trawler]",
   "x_15_6": "x_vars[7,Trawler]",
   "x_15_7": "x_vars[8,Trawler]",
   "x_15_8": "x_vars[9,Trawler]",
   "x_15_9": "x_vars[10,Trawler]",
   "x_15_10": "x_vars[11,Trawler]",
   "x_15_11": "x_vars[12,Trawler]",
   "x_15_12": "x_vars[13,Trawler]",
   "x_15_13": "x_vars[14,Trawler]",
   "x_16_0": "x_vars[1,Paddle Boat]",
   "x_16_1": "x_vars[2,Paddle Boat]",
   "x_16_2": "x_vars[3,Paddle Boat]",
   "x_16_3": "x_vars[4,Paddle Boat]",
   "x_16_4": "x_vars[5,Paddle Boat]",
   "x_16_5": "x_vars[6,Paddle Boat]",
   "x_16_6": "x_vars[7,Paddle Boat]",
   "x_16_7": "x_vars[8,Paddle Boat]",
   "x_16_8": "x_vars[9,Paddle Boat]",
   "x_16_9": "x_vars[10,Paddle Boat]",
   "x_16_10": "x_vars[11,Paddle Boat]",
   "x_16_11": "x_vars[12,Paddle Boat]",
   "x_16_12": "x_vars[13,Paddle Boat]",
   "x_16_13": "x_vars[14,Paddle Boat]",
   "x_17_0": "x_vars[1,Submarine]",
   "x_17_1": "x_vars[2,Submarine]",
   "x_17_2": "x_vars[3,Submarine]",
   "x_17_3": "x_vars[4,Submarine]",
   "x_17_4": "x_vars[5,Submarine]",
   "x_17_5": "x_vars[6,Submarine]",
   "x_17_6": "x_vars[7,Submarine]",
   "x_17_7": "x_vars[8,Submarine]",
   "x_17_8": "x_vars[9,Submarine]",
   "x_17_9": "x_vars[10,Submarine]",
   "x_17_10": "x_vars[11,Submarine]",
   "x_17_11": "x_vars[12,Submarine]",
   "x_17_12": "x_vars[13,Submarine]",
   "x_17_13": "x_vars[14,Submarine]",
   "x_18_0": "x_vars[1,RIB]",
   "x_18_1": "x_vars[2,RIB]",
   "x_18_2": "x_vars[3,RIB]",
   "x_18_3": "x_vars[4,RIB]",
   "x_18_4": "x_vars[5,RIB]",
   "x_18_5": "x_vars[6,RIB]",
   "x_18_6": "x_vars[7,RIB]",
   "x_18_7": "x_vars[8,RIB]",
   "x_18_8": "x_vars[9,RIB]",
   "x_18_9": "x_vars[10,RIB]",
   "x_18_10": "x_vars[11,RIB]",
   "x_18_11": "x_vars[12,RIB]",
   "x_18_12": "x_vars[13,RIB]",
   "x_18_13": "x_vars[14,RIB]",
   "x_19_0": "x_vars[1,Skiff]",
   "x_19_1": "x_vars[2,Skiff]",
   "x_19_2": "x_vars[3,Skiff]",
   "x_19_3": "x_vars[4,Skiff]",
   "x_19_4": "x_vars[5,Skiff]",
   "x_19_5": "x_vars[6,Skiff]",
   "x_19_6": "x_vars[7,Skiff]",
   "x_19_7": "x_vars[8,Skiff]",
   "x_19_8": "x_vars[9,Skiff]",
   "x_19_9": "x_vars[10,Skiff]",
   "x_19_10": "x_vars[11,Skiff]",
   "x_19_11": "x_vars[12,Skiff]",
   "x_19_12": "x_vars[13,Skiff]",
   "x_19_13": "x_vars[14,Skiff]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-056": {
  "source_code_sha256": "81fb344dd553bc3998bf445b285e67f14a36d4af23e2c26334211c257c11420c",
  "original_solution_sha256": "ec3f37c07a4743a464dfaab2feb60b9233d476a401f284ea59d71dd41cdbf9ac",
  "bindings": [
   [
    "C0",
    "x_vars[1,Speedboat]"
   ],
   [
    "C1",
    "x_vars[1,Fishing Boat]"
   ],
   [
    "C2",
    "x_vars[1,Catamaran]"
   ],
   [
    "C3",
    "x_vars[1,Yacht]"
   ],
   [
    "C4",
    "x_vars[1,Sailboat]"
   ],
   [
    "C5",
    "x_vars[1,Kayak]"
   ],
   [
    "C6",
    "x_vars[1,Canoe]"
   ],
   [
    "C7",
    "x_vars[1,Houseboat]"
   ],
   [
    "C8",
    "x_vars[1,Pontoon]"
   ],
   [
    "C9",
    "x_vars[1,Jet Ski]"
   ],
   [
    "C10",
    "x_vars[1,Rowboat]"
   ],
   [
    "C11",
    "x_vars[1,Hovercraft]"
   ],
   [
    "C12",
    "x_vars[1,Cabin Cruiser]"
   ],
   [
    "C13",
    "x_vars[1,Wakeboard Boat]"
   ],
   [
    "C14",
    "x_vars[1,Dinghy]"
   ],
   [
    "C15",
    "x_vars[1,Trawler]"
   ],
   [
    "C16",
    "x_vars[1,Paddle Boat]"
   ],
   [
    "C17",
    "x_vars[1,Submarine]"
   ],
   [
    "C18",
    "x_vars[1,RIB]"
   ],
   [
    "C19",
    "x_vars[1,Skiff]"
   ],
   [
    "C20",
    "x_vars[2,Speedboat]"
   ],
   [
    "C21",
    "x_vars[2,Fishing Boat]"
   ],
   [
    "C22",
    "x_vars[2,Catamaran]"
   ],
   [
    "C23",
    "x_vars[2,Yacht]"
   ],
   [
    "C24",
    "x_vars[2,Sailboat]"
   ],
   [
    "C25",
    "x_vars[2,Kayak]"
   ],
   [
    "C26",
    "x_vars[2,Canoe]"
   ],
   [
    "C27",
    "x_vars[2,Houseboat]"
   ],
   [
    "C28",
    "x_vars[2,Pontoon]"
   ],
   [
    "C29",
    "x_vars[2,Jet Ski]"
   ],
   [
    "C30",
    "x_vars[2,Rowboat]"
   ],
   [
    "C31",
    "x_vars[2,Hovercraft]"
   ],
   [
    "C32",
    "x_vars[2,Cabin Cruiser]"
   ],
   [
    "C33",
    "x_vars[2,Wakeboard Boat]"
   ],
   [
    "C34",
    "x_vars[2,Dinghy]"
   ],
   [
    "C35",
    "x_vars[2,Trawler]"
   ],
   [
    "C36",
    "x_vars[2,Paddle Boat]"
   ],
   [
    "C37",
    "x_vars[2,Submarine]"
   ],
   [
    "C38",
    "x_vars[2,RIB]"
   ],
   [
    "C39",
    "x_vars[2,Skiff]"
   ],
   [
    "C40",
    "x_vars[3,Speedboat]"
   ],
   [
    "C41",
    "x_vars[3,Fishing Boat]"
   ],
   [
    "C42",
    "x_vars[3,Catamaran]"
   ],
   [
    "C43",
    "x_vars[3,Yacht]"
   ],
   [
    "C44",
    "x_vars[3,Sailboat]"
   ],
   [
    "C45",
    "x_vars[3,Kayak]"
   ],
   [
    "C46",
    "x_vars[3,Canoe]"
   ],
   [
    "C47",
    "x_vars[3,Houseboat]"
   ],
   [
    "C48",
    "x_vars[3,Pontoon]"
   ],
   [
    "C49",
    "x_vars[3,Jet Ski]"
   ],
   [
    "C50",
    "x_vars[3,Rowboat]"
   ],
   [
    "C51",
    "x_vars[3,Hovercraft]"
   ],
   [
    "C52",
    "x_vars[3,Cabin Cruiser]"
   ],
   [
    "C53",
    "x_vars[3,Wakeboard Boat]"
   ],
   [
    "C54",
    "x_vars[3,Dinghy]"
   ],
   [
    "C55",
    "x_vars[3,Trawler]"
   ],
   [
    "C56",
    "x_vars[3,Paddle Boat]"
   ],
   [
    "C57",
    "x_vars[3,Submarine]"
   ],
   [
    "C58",
    "x_vars[3,RIB]"
   ],
   [
    "C59",
    "x_vars[3,Skiff]"
   ],
   [
    "C60",
    "x_vars[4,Speedboat]"
   ],
   [
    "C61",
    "x_vars[4,Fishing Boat]"
   ],
   [
    "C62",
    "x_vars[4,Catamaran]"
   ],
   [
    "C63",
    "x_vars[4,Yacht]"
   ],
   [
    "C64",
    "x_vars[4,Sailboat]"
   ],
   [
    "C65",
    "x_vars[4,Kayak]"
   ],
   [
    "C66",
    "x_vars[4,Canoe]"
   ],
   [
    "C67",
    "x_vars[4,Houseboat]"
   ],
   [
    "C68",
    "x_vars[4,Pontoon]"
   ],
   [
    "C69",
    "x_vars[4,Jet Ski]"
   ],
   [
    "C70",
    "x_vars[4,Rowboat]"
   ],
   [
    "C71",
    "x_vars[4,Hovercraft]"
   ],
   [
    "C72",
    "x_vars[4,Cabin Cruiser]"
   ],
   [
    "C73",
    "x_vars[4,Wakeboard Boat]"
   ],
   [
    "C74",
    "x_vars[4,Dinghy]"
   ],
   [
    "C75",
    "x_vars[4,Trawler]"
   ],
   [
    "C76",
    "x_vars[4,Paddle Boat]"
   ],
   [
    "C77",
    "x_vars[4,Submarine]"
   ],
   [
    "C78",
    "x_vars[4,RIB]"
   ],
   [
    "C79",
    "x_vars[4,Skiff]"
   ],
   [
    "C80",
    "x_vars[5,Speedboat]"
   ],
   [
    "C81",
    "x_vars[5,Fishing Boat]"
   ],
   [
    "C82",
    "x_vars[5,Catamaran]"
   ],
   [
    "C83",
    "x_vars[5,Yacht]"
   ],
   [
    "C84",
    "x_vars[5,Sailboat]"
   ],
   [
    "C85",
    "x_vars[5,Kayak]"
   ],
   [
    "C86",
    "x_vars[5,Canoe]"
   ],
   [
    "C87",
    "x_vars[5,Houseboat]"
   ],
   [
    "C88",
    "x_vars[5,Pontoon]"
   ],
   [
    "C89",
    "x_vars[5,Jet Ski]"
   ],
   [
    "C90",
    "x_vars[5,Rowboat]"
   ],
   [
    "C91",
    "x_vars[5,Hovercraft]"
   ],
   [
    "C92",
    "x_vars[5,Cabin Cruiser]"
   ],
   [
    "C93",
    "x_vars[5,Wakeboard Boat]"
   ],
   [
    "C94",
    "x_vars[5,Dinghy]"
   ],
   [
    "C95",
    "x_vars[5,Trawler]"
   ],
   [
    "C96",
    "x_vars[5,Paddle Boat]"
   ],
   [
    "C97",
    "x_vars[5,Submarine]"
   ],
   [
    "C98",
    "x_vars[5,RIB]"
   ],
   [
    "C99",
    "x_vars[5,Skiff]"
   ],
   [
    "C100",
    "x_vars[6,Speedboat]"
   ],
   [
    "C101",
    "x_vars[6,Fishing Boat]"
   ],
   [
    "C102",
    "x_vars[6,Catamaran]"
   ],
   [
    "C103",
    "x_vars[6,Yacht]"
   ],
   [
    "C104",
    "x_vars[6,Sailboat]"
   ],
   [
    "C105",
    "x_vars[6,Kayak]"
   ],
   [
    "C106",
    "x_vars[6,Canoe]"
   ],
   [
    "C107",
    "x_vars[6,Houseboat]"
   ],
   [
    "C108",
    "x_vars[6,Pontoon]"
   ],
   [
    "C109",
    "x_vars[6,Jet Ski]"
   ],
   [
    "C110",
    "x_vars[6,Rowboat]"
   ],
   [
    "C111",
    "x_vars[6,Hovercraft]"
   ],
   [
    "C112",
    "x_vars[6,Cabin Cruiser]"
   ],
   [
    "C113",
    "x_vars[6,Wakeboard Boat]"
   ],
   [
    "C114",
    "x_vars[6,Dinghy]"
   ],
   [
    "C115",
    "x_vars[6,Trawler]"
   ],
   [
    "C116",
    "x_vars[6,Paddle Boat]"
   ],
   [
    "C117",
    "x_vars[6,Submarine]"
   ],
   [
    "C118",
    "x_vars[6,RIB]"
   ],
   [
    "C119",
    "x_vars[6,Skiff]"
   ],
   [
    "C120",
    "x_vars[7,Speedboat]"
   ],
   [
    "C121",
    "x_vars[7,Fishing Boat]"
   ],
   [
    "C122",
    "x_vars[7,Catamaran]"
   ],
   [
    "C123",
    "x_vars[7,Yacht]"
   ],
   [
    "C124",
    "x_vars[7,Sailboat]"
   ],
   [
    "C125",
    "x_vars[7,Kayak]"
   ],
   [
    "C126",
    "x_vars[7,Canoe]"
   ],
   [
    "C127",
    "x_vars[7,Houseboat]"
   ],
   [
    "C128",
    "x_vars[7,Pontoon]"
   ],
   [
    "C129",
    "x_vars[7,Jet Ski]"
   ],
   [
    "C130",
    "x_vars[7,Rowboat]"
   ],
   [
    "C131",
    "x_vars[7,Hovercraft]"
   ],
   [
    "C132",
    "x_vars[7,Cabin Cruiser]"
   ],
   [
    "C133",
    "x_vars[7,Wakeboard Boat]"
   ],
   [
    "C134",
    "x_vars[7,Dinghy]"
   ],
   [
    "C135",
    "x_vars[7,Trawler]"
   ],
   [
    "C136",
    "x_vars[7,Paddle Boat]"
   ],
   [
    "C137",
    "x_vars[7,Submarine]"
   ],
   [
    "C138",
    "x_vars[7,RIB]"
   ],
   [
    "C139",
    "x_vars[7,Skiff]"
   ],
   [
    "C140",
    "x_vars[8,Speedboat]"
   ],
   [
    "C141",
    "x_vars[8,Fishing Boat]"
   ],
   [
    "C142",
    "x_vars[8,Catamaran]"
   ],
   [
    "C143",
    "x_vars[8,Yacht]"
   ],
   [
    "C144",
    "x_vars[8,Sailboat]"
   ],
   [
    "C145",
    "x_vars[8,Kayak]"
   ],
   [
    "C146",
    "x_vars[8,Canoe]"
   ],
   [
    "C147",
    "x_vars[8,Houseboat]"
   ],
   [
    "C148",
    "x_vars[8,Pontoon]"
   ],
   [
    "C149",
    "x_vars[8,Jet Ski]"
   ],
   [
    "C150",
    "x_vars[8,Rowboat]"
   ],
   [
    "C151",
    "x_vars[8,Hovercraft]"
   ],
   [
    "C152",
    "x_vars[8,Cabin Cruiser]"
   ],
   [
    "C153",
    "x_vars[8,Wakeboard Boat]"
   ],
   [
    "C154",
    "x_vars[8,Dinghy]"
   ],
   [
    "C155",
    "x_vars[8,Trawler]"
   ],
   [
    "C156",
    "x_vars[8,Paddle Boat]"
   ],
   [
    "C157",
    "x_vars[8,Submarine]"
   ],
   [
    "C158",
    "x_vars[8,RIB]"
   ],
   [
    "C159",
    "x_vars[8,Skiff]"
   ],
   [
    "C160",
    "x_vars[9,Speedboat]"
   ],
   [
    "C161",
    "x_vars[9,Fishing Boat]"
   ],
   [
    "C162",
    "x_vars[9,Catamaran]"
   ],
   [
    "C163",
    "x_vars[9,Yacht]"
   ],
   [
    "C164",
    "x_vars[9,Sailboat]"
   ],
   [
    "C165",
    "x_vars[9,Kayak]"
   ],
   [
    "C166",
    "x_vars[9,Canoe]"
   ],
   [
    "C167",
    "x_vars[9,Houseboat]"
   ],
   [
    "C168",
    "x_vars[9,Pontoon]"
   ],
   [
    "C169",
    "x_vars[9,Jet Ski]"
   ],
   [
    "C170",
    "x_vars[9,Rowboat]"
   ],
   [
    "C171",
    "x_vars[9,Hovercraft]"
   ],
   [
    "C172",
    "x_vars[9,Cabin Cruiser]"
   ],
   [
    "C173",
    "x_vars[9,Wakeboard Boat]"
   ],
   [
    "C174",
    "x_vars[9,Dinghy]"
   ],
   [
    "C175",
    "x_vars[9,Trawler]"
   ],
   [
    "C176",
    "x_vars[9,Paddle Boat]"
   ],
   [
    "C177",
    "x_vars[9,Submarine]"
   ],
   [
    "C178",
    "x_vars[9,RIB]"
   ],
   [
    "C179",
    "x_vars[9,Skiff]"
   ],
   [
    "C180",
    "x_vars[10,Speedboat]"
   ],
   [
    "C181",
    "x_vars[10,Fishing Boat]"
   ],
   [
    "C182",
    "x_vars[10,Catamaran]"
   ],
   [
    "C183",
    "x_vars[10,Yacht]"
   ],
   [
    "C184",
    "x_vars[10,Sailboat]"
   ],
   [
    "C185",
    "x_vars[10,Kayak]"
   ],
   [
    "C186",
    "x_vars[10,Canoe]"
   ],
   [
    "C187",
    "x_vars[10,Houseboat]"
   ],
   [
    "C188",
    "x_vars[10,Pontoon]"
   ],
   [
    "C189",
    "x_vars[10,Jet Ski]"
   ],
   [
    "C190",
    "x_vars[10,Rowboat]"
   ],
   [
    "C191",
    "x_vars[10,Hovercraft]"
   ],
   [
    "C192",
    "x_vars[10,Cabin Cruiser]"
   ],
   [
    "C193",
    "x_vars[10,Wakeboard Boat]"
   ],
   [
    "C194",
    "x_vars[10,Dinghy]"
   ],
   [
    "C195",
    "x_vars[10,Trawler]"
   ],
   [
    "C196",
    "x_vars[10,Paddle Boat]"
   ],
   [
    "C197",
    "x_vars[10,Submarine]"
   ],
   [
    "C198",
    "x_vars[10,RIB]"
   ],
   [
    "C199",
    "x_vars[10,Skiff]"
   ],
   [
    "C200",
    "x_vars[11,Speedboat]"
   ],
   [
    "C201",
    "x_vars[11,Fishing Boat]"
   ],
   [
    "C202",
    "x_vars[11,Catamaran]"
   ],
   [
    "C203",
    "x_vars[11,Yacht]"
   ],
   [
    "C204",
    "x_vars[11,Sailboat]"
   ],
   [
    "C205",
    "x_vars[11,Kayak]"
   ],
   [
    "C206",
    "x_vars[11,Canoe]"
   ],
   [
    "C207",
    "x_vars[11,Houseboat]"
   ],
   [
    "C208",
    "x_vars[11,Pontoon]"
   ],
   [
    "C209",
    "x_vars[11,Jet Ski]"
   ],
   [
    "C210",
    "x_vars[11,Rowboat]"
   ],
   [
    "C211",
    "x_vars[11,Hovercraft]"
   ],
   [
    "C212",
    "x_vars[11,Cabin Cruiser]"
   ],
   [
    "C213",
    "x_vars[11,Wakeboard Boat]"
   ],
   [
    "C214",
    "x_vars[11,Dinghy]"
   ],
   [
    "C215",
    "x_vars[11,Trawler]"
   ],
   [
    "C216",
    "x_vars[11,Paddle Boat]"
   ],
   [
    "C217",
    "x_vars[11,Submarine]"
   ],
   [
    "C218",
    "x_vars[11,RIB]"
   ],
   [
    "C219",
    "x_vars[11,Skiff]"
   ],
   [
    "C220",
    "x_vars[12,Speedboat]"
   ],
   [
    "C221",
    "x_vars[12,Fishing Boat]"
   ],
   [
    "C222",
    "x_vars[12,Catamaran]"
   ],
   [
    "C223",
    "x_vars[12,Yacht]"
   ],
   [
    "C224",
    "x_vars[12,Sailboat]"
   ],
   [
    "C225",
    "x_vars[12,Kayak]"
   ],
   [
    "C226",
    "x_vars[12,Canoe]"
   ],
   [
    "C227",
    "x_vars[12,Houseboat]"
   ],
   [
    "C228",
    "x_vars[12,Pontoon]"
   ],
   [
    "C229",
    "x_vars[12,Jet Ski]"
   ],
   [
    "C230",
    "x_vars[12,Rowboat]"
   ],
   [
    "C231",
    "x_vars[12,Hovercraft]"
   ],
   [
    "C232",
    "x_vars[12,Cabin Cruiser]"
   ],
   [
    "C233",
    "x_vars[12,Wakeboard Boat]"
   ],
   [
    "C234",
    "x_vars[12,Dinghy]"
   ],
   [
    "C235",
    "x_vars[12,Trawler]"
   ],
   [
    "C236",
    "x_vars[12,Paddle Boat]"
   ],
   [
    "C237",
    "x_vars[12,Submarine]"
   ],
   [
    "C238",
    "x_vars[12,RIB]"
   ],
   [
    "C239",
    "x_vars[12,Skiff]"
   ],
   [
    "C240",
    "x_vars[13,Speedboat]"
   ],
   [
    "C241",
    "x_vars[13,Fishing Boat]"
   ],
   [
    "C242",
    "x_vars[13,Catamaran]"
   ],
   [
    "C243",
    "x_vars[13,Yacht]"
   ],
   [
    "C244",
    "x_vars[13,Sailboat]"
   ],
   [
    "C245",
    "x_vars[13,Kayak]"
   ],
   [
    "C246",
    "x_vars[13,Canoe]"
   ],
   [
    "C247",
    "x_vars[13,Houseboat]"
   ],
   [
    "C248",
    "x_vars[13,Pontoon]"
   ],
   [
    "C249",
    "x_vars[13,Jet Ski]"
   ],
   [
    "C250",
    "x_vars[13,Rowboat]"
   ],
   [
    "C251",
    "x_vars[13,Hovercraft]"
   ],
   [
    "C252",
    "x_vars[13,Cabin Cruiser]"
   ],
   [
    "C253",
    "x_vars[13,Wakeboard Boat]"
   ],
   [
    "C254",
    "x_vars[13,Dinghy]"
   ],
   [
    "C255",
    "x_vars[13,Trawler]"
   ],
   [
    "C256",
    "x_vars[13,Paddle Boat]"
   ],
   [
    "C257",
    "x_vars[13,Submarine]"
   ],
   [
    "C258",
    "x_vars[13,RIB]"
   ],
   [
    "C259",
    "x_vars[13,Skiff]"
   ],
   [
    "C260",
    "x_vars[14,Speedboat]"
   ],
   [
    "C261",
    "x_vars[14,Fishing Boat]"
   ],
   [
    "C262",
    "x_vars[14,Catamaran]"
   ],
   [
    "C263",
    "x_vars[14,Yacht]"
   ],
   [
    "C264",
    "x_vars[14,Sailboat]"
   ],
   [
    "C265",
    "x_vars[14,Kayak]"
   ],
   [
    "C266",
    "x_vars[14,Canoe]"
   ],
   [
    "C267",
    "x_vars[14,Houseboat]"
   ],
   [
    "C268",
    "x_vars[14,Pontoon]"
   ],
   [
    "C269",
    "x_vars[14,Jet Ski]"
   ],
   [
    "C270",
    "x_vars[14,Rowboat]"
   ],
   [
    "C271",
    "x_vars[14,Hovercraft]"
   ],
   [
    "C272",
    "x_vars[14,Cabin Cruiser]"
   ],
   [
    "C273",
    "x_vars[14,Wakeboard Boat]"
   ],
   [
    "C274",
    "x_vars[14,Dinghy]"
   ],
   [
    "C275",
    "x_vars[14,Trawler]"
   ],
   [
    "C276",
    "x_vars[14,Paddle Boat]"
   ],
   [
    "C277",
    "x_vars[14,Submarine]"
   ],
   [
    "C278",
    "x_vars[14,RIB]"
   ],
   [
    "C279",
    "x_vars[14,Skiff]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Speedboat]",
   "x_0_1": "x_vars[2,Speedboat]",
   "x_0_2": "x_vars[3,Speedboat]",
   "x_0_3": "x_vars[4,Speedboat]",
   "x_0_4": "x_vars[5,Speedboat]",
   "x_0_5": "x_vars[6,Speedboat]",
   "x_0_6": "x_vars[7,Speedboat]",
   "x_0_7": "x_vars[8,Speedboat]",
   "x_0_8": "x_vars[9,Speedboat]",
   "x_0_9": "x_vars[10,Speedboat]",
   "x_0_10": "x_vars[11,Speedboat]",
   "x_0_11": "x_vars[12,Speedboat]",
   "x_0_12": "x_vars[13,Speedboat]",
   "x_0_13": "x_vars[14,Speedboat]",
   "x_1_0": "x_vars[1,Fishing Boat]",
   "x_1_1": "x_vars[2,Fishing Boat]",
   "x_1_2": "x_vars[3,Fishing Boat]",
   "x_1_3": "x_vars[4,Fishing Boat]",
   "x_1_4": "x_vars[5,Fishing Boat]",
   "x_1_5": "x_vars[6,Fishing Boat]",
   "x_1_6": "x_vars[7,Fishing Boat]",
   "x_1_7": "x_vars[8,Fishing Boat]",
   "x_1_8": "x_vars[9,Fishing Boat]",
   "x_1_9": "x_vars[10,Fishing Boat]",
   "x_1_10": "x_vars[11,Fishing Boat]",
   "x_1_11": "x_vars[12,Fishing Boat]",
   "x_1_12": "x_vars[13,Fishing Boat]",
   "x_1_13": "x_vars[14,Fishing Boat]",
   "x_2_0": "x_vars[1,Catamaran]",
   "x_2_1": "x_vars[2,Catamaran]",
   "x_2_2": "x_vars[3,Catamaran]",
   "x_2_3": "x_vars[4,Catamaran]",
   "x_2_4": "x_vars[5,Catamaran]",
   "x_2_5": "x_vars[6,Catamaran]",
   "x_2_6": "x_vars[7,Catamaran]",
   "x_2_7": "x_vars[8,Catamaran]",
   "x_2_8": "x_vars[9,Catamaran]",
   "x_2_9": "x_vars[10,Catamaran]",
   "x_2_10": "x_vars[11,Catamaran]",
   "x_2_11": "x_vars[12,Catamaran]",
   "x_2_12": "x_vars[13,Catamaran]",
   "x_2_13": "x_vars[14,Catamaran]",
   "x_3_0": "x_vars[1,Yacht]",
   "x_3_1": "x_vars[2,Yacht]",
   "x_3_2": "x_vars[3,Yacht]",
   "x_3_3": "x_vars[4,Yacht]",
   "x_3_4": "x_vars[5,Yacht]",
   "x_3_5": "x_vars[6,Yacht]",
   "x_3_6": "x_vars[7,Yacht]",
   "x_3_7": "x_vars[8,Yacht]",
   "x_3_8": "x_vars[9,Yacht]",
   "x_3_9": "x_vars[10,Yacht]",
   "x_3_10": "x_vars[11,Yacht]",
   "x_3_11": "x_vars[12,Yacht]",
   "x_3_12": "x_vars[13,Yacht]",
   "x_3_13": "x_vars[14,Yacht]",
   "x_4_0": "x_vars[1,Sailboat]",
   "x_4_1": "x_vars[2,Sailboat]",
   "x_4_2": "x_vars[3,Sailboat]",
   "x_4_3": "x_vars[4,Sailboat]",
   "x_4_4": "x_vars[5,Sailboat]",
   "x_4_5": "x_vars[6,Sailboat]",
   "x_4_6": "x_vars[7,Sailboat]",
   "x_4_7": "x_vars[8,Sailboat]",
   "x_4_8": "x_vars[9,Sailboat]",
   "x_4_9": "x_vars[10,Sailboat]",
   "x_4_10": "x_vars[11,Sailboat]",
   "x_4_11": "x_vars[12,Sailboat]",
   "x_4_12": "x_vars[13,Sailboat]",
   "x_4_13": "x_vars[14,Sailboat]",
   "x_5_0": "x_vars[1,Kayak]",
   "x_5_1": "x_vars[2,Kayak]",
   "x_5_2": "x_vars[3,Kayak]",
   "x_5_3": "x_vars[4,Kayak]",
   "x_5_4": "x_vars[5,Kayak]",
   "x_5_5": "x_vars[6,Kayak]",
   "x_5_6": "x_vars[7,Kayak]",
   "x_5_7": "x_vars[8,Kayak]",
   "x_5_8": "x_vars[9,Kayak]",
   "x_5_9": "x_vars[10,Kayak]",
   "x_5_10": "x_vars[11,Kayak]",
   "x_5_11": "x_vars[12,Kayak]",
   "x_5_12": "x_vars[13,Kayak]",
   "x_5_13": "x_vars[14,Kayak]",
   "x_6_0": "x_vars[1,Canoe]",
   "x_6_1": "x_vars[2,Canoe]",
   "x_6_2": "x_vars[3,Canoe]",
   "x_6_3": "x_vars[4,Canoe]",
   "x_6_4": "x_vars[5,Canoe]",
   "x_6_5": "x_vars[6,Canoe]",
   "x_6_6": "x_vars[7,Canoe]",
   "x_6_7": "x_vars[8,Canoe]",
   "x_6_8": "x_vars[9,Canoe]",
   "x_6_9": "x_vars[10,Canoe]",
   "x_6_10": "x_vars[11,Canoe]",
   "x_6_11": "x_vars[12,Canoe]",
   "x_6_12": "x_vars[13,Canoe]",
   "x_6_13": "x_vars[14,Canoe]",
   "x_7_0": "x_vars[1,Houseboat]",
   "x_7_1": "x_vars[2,Houseboat]",
   "x_7_2": "x_vars[3,Houseboat]",
   "x_7_3": "x_vars[4,Houseboat]",
   "x_7_4": "x_vars[5,Houseboat]",
   "x_7_5": "x_vars[6,Houseboat]",
   "x_7_6": "x_vars[7,Houseboat]",
   "x_7_7": "x_vars[8,Houseboat]",
   "x_7_8": "x_vars[9,Houseboat]",
   "x_7_9": "x_vars[10,Houseboat]",
   "x_7_10": "x_vars[11,Houseboat]",
   "x_7_11": "x_vars[12,Houseboat]",
   "x_7_12": "x_vars[13,Houseboat]",
   "x_7_13": "x_vars[14,Houseboat]",
   "x_8_0": "x_vars[1,Pontoon]",
   "x_8_1": "x_vars[2,Pontoon]",
   "x_8_2": "x_vars[3,Pontoon]",
   "x_8_3": "x_vars[4,Pontoon]",
   "x_8_4": "x_vars[5,Pontoon]",
   "x_8_5": "x_vars[6,Pontoon]",
   "x_8_6": "x_vars[7,Pontoon]",
   "x_8_7": "x_vars[8,Pontoon]",
   "x_8_8": "x_vars[9,Pontoon]",
   "x_8_9": "x_vars[10,Pontoon]",
   "x_8_10": "x_vars[11,Pontoon]",
   "x_8_11": "x_vars[12,Pontoon]",
   "x_8_12": "x_vars[13,Pontoon]",
   "x_8_13": "x_vars[14,Pontoon]",
   "x_9_0": "x_vars[1,Jet Ski]",
   "x_9_1": "x_vars[2,Jet Ski]",
   "x_9_2": "x_vars[3,Jet Ski]",
   "x_9_3": "x_vars[4,Jet Ski]",
   "x_9_4": "x_vars[5,Jet Ski]",
   "x_9_5": "x_vars[6,Jet Ski]",
   "x_9_6": "x_vars[7,Jet Ski]",
   "x_9_7": "x_vars[8,Jet Ski]",
   "x_9_8": "x_vars[9,Jet Ski]",
   "x_9_9": "x_vars[10,Jet Ski]",
   "x_9_10": "x_vars[11,Jet Ski]",
   "x_9_11": "x_vars[12,Jet Ski]",
   "x_9_12": "x_vars[13,Jet Ski]",
   "x_9_13": "x_vars[14,Jet Ski]",
   "x_10_0": "x_vars[1,Rowboat]",
   "x_10_1": "x_vars[2,Rowboat]",
   "x_10_2": "x_vars[3,Rowboat]",
   "x_10_3": "x_vars[4,Rowboat]",
   "x_10_4": "x_vars[5,Rowboat]",
   "x_10_5": "x_vars[6,Rowboat]",
   "x_10_6": "x_vars[7,Rowboat]",
   "x_10_7": "x_vars[8,Rowboat]",
   "x_10_8": "x_vars[9,Rowboat]",
   "x_10_9": "x_vars[10,Rowboat]",
   "x_10_10": "x_vars[11,Rowboat]",
   "x_10_11": "x_vars[12,Rowboat]",
   "x_10_12": "x_vars[13,Rowboat]",
   "x_10_13": "x_vars[14,Rowboat]",
   "x_11_0": "x_vars[1,Hovercraft]",
   "x_11_1": "x_vars[2,Hovercraft]",
   "x_11_2": "x_vars[3,Hovercraft]",
   "x_11_3": "x_vars[4,Hovercraft]",
   "x_11_4": "x_vars[5,Hovercraft]",
   "x_11_5": "x_vars[6,Hovercraft]",
   "x_11_6": "x_vars[7,Hovercraft]",
   "x_11_7": "x_vars[8,Hovercraft]",
   "x_11_8": "x_vars[9,Hovercraft]",
   "x_11_9": "x_vars[10,Hovercraft]",
   "x_11_10": "x_vars[11,Hovercraft]",
   "x_11_11": "x_vars[12,Hovercraft]",
   "x_11_12": "x_vars[13,Hovercraft]",
   "x_11_13": "x_vars[14,Hovercraft]",
   "x_12_0": "x_vars[1,Cabin Cruiser]",
   "x_12_1": "x_vars[2,Cabin Cruiser]",
   "x_12_2": "x_vars[3,Cabin Cruiser]",
   "x_12_3": "x_vars[4,Cabin Cruiser]",
   "x_12_4": "x_vars[5,Cabin Cruiser]",
   "x_12_5": "x_vars[6,Cabin Cruiser]",
   "x_12_6": "x_vars[7,Cabin Cruiser]",
   "x_12_7": "x_vars[8,Cabin Cruiser]",
   "x_12_8": "x_vars[9,Cabin Cruiser]",
   "x_12_9": "x_vars[10,Cabin Cruiser]",
   "x_12_10": "x_vars[11,Cabin Cruiser]",
   "x_12_11": "x_vars[12,Cabin Cruiser]",
   "x_12_12": "x_vars[13,Cabin Cruiser]",
   "x_12_13": "x_vars[14,Cabin Cruiser]",
   "x_13_0": "x_vars[1,Wakeboard Boat]",
   "x_13_1": "x_vars[2,Wakeboard Boat]",
   "x_13_2": "x_vars[3,Wakeboard Boat]",
   "x_13_3": "x_vars[4,Wakeboard Boat]",
   "x_13_4": "x_vars[5,Wakeboard Boat]",
   "x_13_5": "x_vars[6,Wakeboard Boat]",
   "x_13_6": "x_vars[7,Wakeboard Boat]",
   "x_13_7": "x_vars[8,Wakeboard Boat]",
   "x_13_8": "x_vars[9,Wakeboard Boat]",
   "x_13_9": "x_vars[10,Wakeboard Boat]",
   "x_13_10": "x_vars[11,Wakeboard Boat]",
   "x_13_11": "x_vars[12,Wakeboard Boat]",
   "x_13_12": "x_vars[13,Wakeboard Boat]",
   "x_13_13": "x_vars[14,Wakeboard Boat]",
   "x_14_0": "x_vars[1,Dinghy]",
   "x_14_1": "x_vars[2,Dinghy]",
   "x_14_2": "x_vars[3,Dinghy]",
   "x_14_3": "x_vars[4,Dinghy]",
   "x_14_4": "x_vars[5,Dinghy]",
   "x_14_5": "x_vars[6,Dinghy]",
   "x_14_6": "x_vars[7,Dinghy]",
   "x_14_7": "x_vars[8,Dinghy]",
   "x_14_8": "x_vars[9,Dinghy]",
   "x_14_9": "x_vars[10,Dinghy]",
   "x_14_10": "x_vars[11,Dinghy]",
   "x_14_11": "x_vars[12,Dinghy]",
   "x_14_12": "x_vars[13,Dinghy]",
   "x_14_13": "x_vars[14,Dinghy]",
   "x_15_0": "x_vars[1,Trawler]",
   "x_15_1": "x_vars[2,Trawler]",
   "x_15_2": "x_vars[3,Trawler]",
   "x_15_3": "x_vars[4,Trawler]",
   "x_15_4": "x_vars[5,Trawler]",
   "x_15_5": "x_vars[6,Trawler]",
   "x_15_6": "x_vars[7,Trawler]",
   "x_15_7": "x_vars[8,Trawler]",
   "x_15_8": "x_vars[9,Trawler]",
   "x_15_9": "x_vars[10,Trawler]",
   "x_15_10": "x_vars[11,Trawler]",
   "x_15_11": "x_vars[12,Trawler]",
   "x_15_12": "x_vars[13,Trawler]",
   "x_15_13": "x_vars[14,Trawler]",
   "x_16_0": "x_vars[1,Paddle Boat]",
   "x_16_1": "x_vars[2,Paddle Boat]",
   "x_16_2": "x_vars[3,Paddle Boat]",
   "x_16_3": "x_vars[4,Paddle Boat]",
   "x_16_4": "x_vars[5,Paddle Boat]",
   "x_16_5": "x_vars[6,Paddle Boat]",
   "x_16_6": "x_vars[7,Paddle Boat]",
   "x_16_7": "x_vars[8,Paddle Boat]",
   "x_16_8": "x_vars[9,Paddle Boat]",
   "x_16_9": "x_vars[10,Paddle Boat]",
   "x_16_10": "x_vars[11,Paddle Boat]",
   "x_16_11": "x_vars[12,Paddle Boat]",
   "x_16_12": "x_vars[13,Paddle Boat]",
   "x_16_13": "x_vars[14,Paddle Boat]",
   "x_17_0": "x_vars[1,Submarine]",
   "x_17_1": "x_vars[2,Submarine]",
   "x_17_2": "x_vars[3,Submarine]",
   "x_17_3": "x_vars[4,Submarine]",
   "x_17_4": "x_vars[5,Submarine]",
   "x_17_5": "x_vars[6,Submarine]",
   "x_17_6": "x_vars[7,Submarine]",
   "x_17_7": "x_vars[8,Submarine]",
   "x_17_8": "x_vars[9,Submarine]",
   "x_17_9": "x_vars[10,Submarine]",
   "x_17_10": "x_vars[11,Submarine]",
   "x_17_11": "x_vars[12,Submarine]",
   "x_17_12": "x_vars[13,Submarine]",
   "x_17_13": "x_vars[14,Submarine]",
   "x_18_0": "x_vars[1,RIB]",
   "x_18_1": "x_vars[2,RIB]",
   "x_18_2": "x_vars[3,RIB]",
   "x_18_3": "x_vars[4,RIB]",
   "x_18_4": "x_vars[5,RIB]",
   "x_18_5": "x_vars[6,RIB]",
   "x_18_6": "x_vars[7,RIB]",
   "x_18_7": "x_vars[8,RIB]",
   "x_18_8": "x_vars[9,RIB]",
   "x_18_9": "x_vars[10,RIB]",
   "x_18_10": "x_vars[11,RIB]",
   "x_18_11": "x_vars[12,RIB]",
   "x_18_12": "x_vars[13,RIB]",
   "x_18_13": "x_vars[14,RIB]",
   "x_19_0": "x_vars[1,Skiff]",
   "x_19_1": "x_vars[2,Skiff]",
   "x_19_2": "x_vars[3,Skiff]",
   "x_19_3": "x_vars[4,Skiff]",
   "x_19_4": "x_vars[5,Skiff]",
   "x_19_5": "x_vars[6,Skiff]",
   "x_19_6": "x_vars[7,Skiff]",
   "x_19_7": "x_vars[8,Skiff]",
   "x_19_8": "x_vars[9,Skiff]",
   "x_19_9": "x_vars[10,Skiff]",
   "x_19_10": "x_vars[11,Skiff]",
   "x_19_11": "x_vars[12,Skiff]",
   "x_19_12": "x_vars[13,Skiff]",
   "x_19_13": "x_vars[14,Skiff]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-057": {
  "source_code_sha256": "63e3a2fe8288cda6f5f77cd047f1e29bccbaafcdccdfb04fb9930de429fc0759",
  "original_solution_sha256": "2a181e28654bd02a714bfbb8e8ca9ce75d92efa42887dd7e48b5d52c1f84b74e",
  "bindings": [
   [
    "C0",
    "x_vars[1,Racing]"
   ],
   [
    "C1",
    "x_vars[1,Sports]"
   ],
   [
    "C2",
    "x_vars[1,Action]"
   ],
   [
    "C3",
    "x_vars[1,Adventure]"
   ],
   [
    "C4",
    "x_vars[1,RPG]"
   ],
   [
    "C5",
    "x_vars[1,Shooter]"
   ],
   [
    "C6",
    "x_vars[1,Strategy]"
   ],
   [
    "C7",
    "x_vars[1,Simulation]"
   ],
   [
    "C8",
    "x_vars[1,Puzzle]"
   ],
   [
    "C9",
    "x_vars[1,Fighting]"
   ],
   [
    "C10",
    "x_vars[1,Platformer]"
   ],
   [
    "C11",
    "x_vars[1,Survival]"
   ],
   [
    "C12",
    "x_vars[1,Horror]"
   ],
   [
    "C13",
    "x_vars[1,Sandbox]"
   ],
   [
    "C14",
    "x_vars[1,MMO]"
   ],
   [
    "C15",
    "x_vars[2,Racing]"
   ],
   [
    "C16",
    "x_vars[2,Sports]"
   ],
   [
    "C17",
    "x_vars[2,Action]"
   ],
   [
    "C18",
    "x_vars[2,Adventure]"
   ],
   [
    "C19",
    "x_vars[2,RPG]"
   ],
   [
    "C20",
    "x_vars[2,Shooter]"
   ],
   [
    "C21",
    "x_vars[2,Strategy]"
   ],
   [
    "C22",
    "x_vars[2,Simulation]"
   ],
   [
    "C23",
    "x_vars[2,Puzzle]"
   ],
   [
    "C24",
    "x_vars[2,Fighting]"
   ],
   [
    "C25",
    "x_vars[2,Platformer]"
   ],
   [
    "C26",
    "x_vars[2,Survival]"
   ],
   [
    "C27",
    "x_vars[2,Horror]"
   ],
   [
    "C28",
    "x_vars[2,Sandbox]"
   ],
   [
    "C29",
    "x_vars[2,MMO]"
   ],
   [
    "C30",
    "x_vars[3,Racing]"
   ],
   [
    "C31",
    "x_vars[3,Sports]"
   ],
   [
    "C32",
    "x_vars[3,Action]"
   ],
   [
    "C33",
    "x_vars[3,Adventure]"
   ],
   [
    "C34",
    "x_vars[3,RPG]"
   ],
   [
    "C35",
    "x_vars[3,Shooter]"
   ],
   [
    "C36",
    "x_vars[3,Strategy]"
   ],
   [
    "C37",
    "x_vars[3,Simulation]"
   ],
   [
    "C38",
    "x_vars[3,Puzzle]"
   ],
   [
    "C39",
    "x_vars[3,Fighting]"
   ],
   [
    "C40",
    "x_vars[3,Platformer]"
   ],
   [
    "C41",
    "x_vars[3,Survival]"
   ],
   [
    "C42",
    "x_vars[3,Horror]"
   ],
   [
    "C43",
    "x_vars[3,Sandbox]"
   ],
   [
    "C44",
    "x_vars[3,MMO]"
   ],
   [
    "C45",
    "x_vars[4,Racing]"
   ],
   [
    "C46",
    "x_vars[4,Sports]"
   ],
   [
    "C47",
    "x_vars[4,Action]"
   ],
   [
    "C48",
    "x_vars[4,Adventure]"
   ],
   [
    "C49",
    "x_vars[4,RPG]"
   ],
   [
    "C50",
    "x_vars[4,Shooter]"
   ],
   [
    "C51",
    "x_vars[4,Strategy]"
   ],
   [
    "C52",
    "x_vars[4,Simulation]"
   ],
   [
    "C53",
    "x_vars[4,Puzzle]"
   ],
   [
    "C54",
    "x_vars[4,Fighting]"
   ],
   [
    "C55",
    "x_vars[4,Platformer]"
   ],
   [
    "C56",
    "x_vars[4,Survival]"
   ],
   [
    "C57",
    "x_vars[4,Horror]"
   ],
   [
    "C58",
    "x_vars[4,Sandbox]"
   ],
   [
    "C59",
    "x_vars[4,MMO]"
   ],
   [
    "C60",
    "x_vars[5,Racing]"
   ],
   [
    "C61",
    "x_vars[5,Sports]"
   ],
   [
    "C62",
    "x_vars[5,Action]"
   ],
   [
    "C63",
    "x_vars[5,Adventure]"
   ],
   [
    "C64",
    "x_vars[5,RPG]"
   ],
   [
    "C65",
    "x_vars[5,Shooter]"
   ],
   [
    "C66",
    "x_vars[5,Strategy]"
   ],
   [
    "C67",
    "x_vars[5,Simulation]"
   ],
   [
    "C68",
    "x_vars[5,Puzzle]"
   ],
   [
    "C69",
    "x_vars[5,Fighting]"
   ],
   [
    "C70",
    "x_vars[5,Platformer]"
   ],
   [
    "C71",
    "x_vars[5,Survival]"
   ],
   [
    "C72",
    "x_vars[5,Horror]"
   ],
   [
    "C73",
    "x_vars[5,Sandbox]"
   ],
   [
    "C74",
    "x_vars[5,MMO]"
   ],
   [
    "C75",
    "x_vars[6,Racing]"
   ],
   [
    "C76",
    "x_vars[6,Sports]"
   ],
   [
    "C77",
    "x_vars[6,Action]"
   ],
   [
    "C78",
    "x_vars[6,Adventure]"
   ],
   [
    "C79",
    "x_vars[6,RPG]"
   ],
   [
    "C80",
    "x_vars[6,Shooter]"
   ],
   [
    "C81",
    "x_vars[6,Strategy]"
   ],
   [
    "C82",
    "x_vars[6,Simulation]"
   ],
   [
    "C83",
    "x_vars[6,Puzzle]"
   ],
   [
    "C84",
    "x_vars[6,Fighting]"
   ],
   [
    "C85",
    "x_vars[6,Platformer]"
   ],
   [
    "C86",
    "x_vars[6,Survival]"
   ],
   [
    "C87",
    "x_vars[6,Horror]"
   ],
   [
    "C88",
    "x_vars[6,Sandbox]"
   ],
   [
    "C89",
    "x_vars[6,MMO]"
   ],
   [
    "C90",
    "x_vars[7,Racing]"
   ],
   [
    "C91",
    "x_vars[7,Sports]"
   ],
   [
    "C92",
    "x_vars[7,Action]"
   ],
   [
    "C93",
    "x_vars[7,Adventure]"
   ],
   [
    "C94",
    "x_vars[7,RPG]"
   ],
   [
    "C95",
    "x_vars[7,Shooter]"
   ],
   [
    "C96",
    "x_vars[7,Strategy]"
   ],
   [
    "C97",
    "x_vars[7,Simulation]"
   ],
   [
    "C98",
    "x_vars[7,Puzzle]"
   ],
   [
    "C99",
    "x_vars[7,Fighting]"
   ],
   [
    "C100",
    "x_vars[7,Platformer]"
   ],
   [
    "C101",
    "x_vars[7,Survival]"
   ],
   [
    "C102",
    "x_vars[7,Horror]"
   ],
   [
    "C103",
    "x_vars[7,Sandbox]"
   ],
   [
    "C104",
    "x_vars[7,MMO]"
   ],
   [
    "C105",
    "x_vars[8,Racing]"
   ],
   [
    "C106",
    "x_vars[8,Sports]"
   ],
   [
    "C107",
    "x_vars[8,Action]"
   ],
   [
    "C108",
    "x_vars[8,Adventure]"
   ],
   [
    "C109",
    "x_vars[8,RPG]"
   ],
   [
    "C110",
    "x_vars[8,Shooter]"
   ],
   [
    "C111",
    "x_vars[8,Strategy]"
   ],
   [
    "C112",
    "x_vars[8,Simulation]"
   ],
   [
    "C113",
    "x_vars[8,Puzzle]"
   ],
   [
    "C114",
    "x_vars[8,Fighting]"
   ],
   [
    "C115",
    "x_vars[8,Platformer]"
   ],
   [
    "C116",
    "x_vars[8,Survival]"
   ],
   [
    "C117",
    "x_vars[8,Horror]"
   ],
   [
    "C118",
    "x_vars[8,Sandbox]"
   ],
   [
    "C119",
    "x_vars[8,MMO]"
   ],
   [
    "C120",
    "x_vars[9,Racing]"
   ],
   [
    "C121",
    "x_vars[9,Sports]"
   ],
   [
    "C122",
    "x_vars[9,Action]"
   ],
   [
    "C123",
    "x_vars[9,Adventure]"
   ],
   [
    "C124",
    "x_vars[9,RPG]"
   ],
   [
    "C125",
    "x_vars[9,Shooter]"
   ],
   [
    "C126",
    "x_vars[9,Strategy]"
   ],
   [
    "C127",
    "x_vars[9,Simulation]"
   ],
   [
    "C128",
    "x_vars[9,Puzzle]"
   ],
   [
    "C129",
    "x_vars[9,Fighting]"
   ],
   [
    "C130",
    "x_vars[9,Platformer]"
   ],
   [
    "C131",
    "x_vars[9,Survival]"
   ],
   [
    "C132",
    "x_vars[9,Horror]"
   ],
   [
    "C133",
    "x_vars[9,Sandbox]"
   ],
   [
    "C134",
    "x_vars[9,MMO]"
   ],
   [
    "C135",
    "x_vars[10,Racing]"
   ],
   [
    "C136",
    "x_vars[10,Sports]"
   ],
   [
    "C137",
    "x_vars[10,Action]"
   ],
   [
    "C138",
    "x_vars[10,Adventure]"
   ],
   [
    "C139",
    "x_vars[10,RPG]"
   ],
   [
    "C140",
    "x_vars[10,Shooter]"
   ],
   [
    "C141",
    "x_vars[10,Strategy]"
   ],
   [
    "C142",
    "x_vars[10,Simulation]"
   ],
   [
    "C143",
    "x_vars[10,Puzzle]"
   ],
   [
    "C144",
    "x_vars[10,Fighting]"
   ],
   [
    "C145",
    "x_vars[10,Platformer]"
   ],
   [
    "C146",
    "x_vars[10,Survival]"
   ],
   [
    "C147",
    "x_vars[10,Horror]"
   ],
   [
    "C148",
    "x_vars[10,Sandbox]"
   ],
   [
    "C149",
    "x_vars[10,MMO]"
   ],
   [
    "C150",
    "x_vars[11,Racing]"
   ],
   [
    "C151",
    "x_vars[11,Sports]"
   ],
   [
    "C152",
    "x_vars[11,Action]"
   ],
   [
    "C153",
    "x_vars[11,Adventure]"
   ],
   [
    "C154",
    "x_vars[11,RPG]"
   ],
   [
    "C155",
    "x_vars[11,Shooter]"
   ],
   [
    "C156",
    "x_vars[11,Strategy]"
   ],
   [
    "C157",
    "x_vars[11,Simulation]"
   ],
   [
    "C158",
    "x_vars[11,Puzzle]"
   ],
   [
    "C159",
    "x_vars[11,Fighting]"
   ],
   [
    "C160",
    "x_vars[11,Platformer]"
   ],
   [
    "C161",
    "x_vars[11,Survival]"
   ],
   [
    "C162",
    "x_vars[11,Horror]"
   ],
   [
    "C163",
    "x_vars[11,Sandbox]"
   ],
   [
    "C164",
    "x_vars[11,MMO]"
   ],
   [
    "C165",
    "x_vars[12,Racing]"
   ],
   [
    "C166",
    "x_vars[12,Sports]"
   ],
   [
    "C167",
    "x_vars[12,Action]"
   ],
   [
    "C168",
    "x_vars[12,Adventure]"
   ],
   [
    "C169",
    "x_vars[12,RPG]"
   ],
   [
    "C170",
    "x_vars[12,Shooter]"
   ],
   [
    "C171",
    "x_vars[12,Strategy]"
   ],
   [
    "C172",
    "x_vars[12,Simulation]"
   ],
   [
    "C173",
    "x_vars[12,Puzzle]"
   ],
   [
    "C174",
    "x_vars[12,Fighting]"
   ],
   [
    "C175",
    "x_vars[12,Platformer]"
   ],
   [
    "C176",
    "x_vars[12,Survival]"
   ],
   [
    "C177",
    "x_vars[12,Horror]"
   ],
   [
    "C178",
    "x_vars[12,Sandbox]"
   ],
   [
    "C179",
    "x_vars[12,MMO]"
   ],
   [
    "C180",
    "x_vars[13,Racing]"
   ],
   [
    "C181",
    "x_vars[13,Sports]"
   ],
   [
    "C182",
    "x_vars[13,Action]"
   ],
   [
    "C183",
    "x_vars[13,Adventure]"
   ],
   [
    "C184",
    "x_vars[13,RPG]"
   ],
   [
    "C185",
    "x_vars[13,Shooter]"
   ],
   [
    "C186",
    "x_vars[13,Strategy]"
   ],
   [
    "C187",
    "x_vars[13,Simulation]"
   ],
   [
    "C188",
    "x_vars[13,Puzzle]"
   ],
   [
    "C189",
    "x_vars[13,Fighting]"
   ],
   [
    "C190",
    "x_vars[13,Platformer]"
   ],
   [
    "C191",
    "x_vars[13,Survival]"
   ],
   [
    "C192",
    "x_vars[13,Horror]"
   ],
   [
    "C193",
    "x_vars[13,Sandbox]"
   ],
   [
    "C194",
    "x_vars[13,MMO]"
   ],
   [
    "C195",
    "x_vars[14,Racing]"
   ],
   [
    "C196",
    "x_vars[14,Sports]"
   ],
   [
    "C197",
    "x_vars[14,Action]"
   ],
   [
    "C198",
    "x_vars[14,Adventure]"
   ],
   [
    "C199",
    "x_vars[14,RPG]"
   ],
   [
    "C200",
    "x_vars[14,Shooter]"
   ],
   [
    "C201",
    "x_vars[14,Strategy]"
   ],
   [
    "C202",
    "x_vars[14,Simulation]"
   ],
   [
    "C203",
    "x_vars[14,Puzzle]"
   ],
   [
    "C204",
    "x_vars[14,Fighting]"
   ],
   [
    "C205",
    "x_vars[14,Platformer]"
   ],
   [
    "C206",
    "x_vars[14,Survival]"
   ],
   [
    "C207",
    "x_vars[14,Horror]"
   ],
   [
    "C208",
    "x_vars[14,Sandbox]"
   ],
   [
    "C209",
    "x_vars[14,MMO]"
   ],
   [
    "C210",
    "x_vars[15,Racing]"
   ],
   [
    "C211",
    "x_vars[15,Sports]"
   ],
   [
    "C212",
    "x_vars[15,Action]"
   ],
   [
    "C213",
    "x_vars[15,Adventure]"
   ],
   [
    "C214",
    "x_vars[15,RPG]"
   ],
   [
    "C215",
    "x_vars[15,Shooter]"
   ],
   [
    "C216",
    "x_vars[15,Strategy]"
   ],
   [
    "C217",
    "x_vars[15,Simulation]"
   ],
   [
    "C218",
    "x_vars[15,Puzzle]"
   ],
   [
    "C219",
    "x_vars[15,Fighting]"
   ],
   [
    "C220",
    "x_vars[15,Platformer]"
   ],
   [
    "C221",
    "x_vars[15,Survival]"
   ],
   [
    "C222",
    "x_vars[15,Horror]"
   ],
   [
    "C223",
    "x_vars[15,Sandbox]"
   ],
   [
    "C224",
    "x_vars[15,MMO]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[1,Racing]",
   "x_0_1": "x_vars[2,Racing]",
   "x_0_2": "x_vars[3,Racing]",
   "x_0_3": "x_vars[4,Racing]",
   "x_0_4": "x_vars[5,Racing]",
   "x_0_5": "x_vars[6,Racing]",
   "x_0_6": "x_vars[7,Racing]",
   "x_0_7": "x_vars[8,Racing]",
   "x_0_8": "x_vars[9,Racing]",
   "x_0_9": "x_vars[10,Racing]",
   "x_0_10": "x_vars[11,Racing]",
   "x_0_11": "x_vars[12,Racing]",
   "x_0_12": "x_vars[13,Racing]",
   "x_0_13": "x_vars[14,Racing]",
   "x_0_14": "x_vars[15,Racing]",
   "x_1_0": "x_vars[1,Sports]",
   "x_1_1": "x_vars[2,Sports]",
   "x_1_2": "x_vars[3,Sports]",
   "x_1_3": "x_vars[4,Sports]",
   "x_1_4": "x_vars[5,Sports]",
   "x_1_5": "x_vars[6,Sports]",
   "x_1_6": "x_vars[7,Sports]",
   "x_1_7": "x_vars[8,Sports]",
   "x_1_8": "x_vars[9,Sports]",
   "x_1_9": "x_vars[10,Sports]",
   "x_1_10": "x_vars[11,Sports]",
   "x_1_11": "x_vars[12,Sports]",
   "x_1_12": "x_vars[13,Sports]",
   "x_1_13": "x_vars[14,Sports]",
   "x_1_14": "x_vars[15,Sports]",
   "x_2_0": "x_vars[1,Action]",
   "x_2_1": "x_vars[2,Action]",
   "x_2_2": "x_vars[3,Action]",
   "x_2_3": "x_vars[4,Action]",
   "x_2_4": "x_vars[5,Action]",
   "x_2_5": "x_vars[6,Action]",
   "x_2_6": "x_vars[7,Action]",
   "x_2_7": "x_vars[8,Action]",
   "x_2_8": "x_vars[9,Action]",
   "x_2_9": "x_vars[10,Action]",
   "x_2_10": "x_vars[11,Action]",
   "x_2_11": "x_vars[12,Action]",
   "x_2_12": "x_vars[13,Action]",
   "x_2_13": "x_vars[14,Action]",
   "x_2_14": "x_vars[15,Action]",
   "x_3_0": "x_vars[1,Adventure]",
   "x_3_1": "x_vars[2,Adventure]",
   "x_3_2": "x_vars[3,Adventure]",
   "x_3_3": "x_vars[4,Adventure]",
   "x_3_4": "x_vars[5,Adventure]",
   "x_3_5": "x_vars[6,Adventure]",
   "x_3_6": "x_vars[7,Adventure]",
   "x_3_7": "x_vars[8,Adventure]",
   "x_3_8": "x_vars[9,Adventure]",
   "x_3_9": "x_vars[10,Adventure]",
   "x_3_10": "x_vars[11,Adventure]",
   "x_3_11": "x_vars[12,Adventure]",
   "x_3_12": "x_vars[13,Adventure]",
   "x_3_13": "x_vars[14,Adventure]",
   "x_3_14": "x_vars[15,Adventure]",
   "x_4_0": "x_vars[1,RPG]",
   "x_4_1": "x_vars[2,RPG]",
   "x_4_2": "x_vars[3,RPG]",
   "x_4_3": "x_vars[4,RPG]",
   "x_4_4": "x_vars[5,RPG]",
   "x_4_5": "x_vars[6,RPG]",
   "x_4_6": "x_vars[7,RPG]",
   "x_4_7": "x_vars[8,RPG]",
   "x_4_8": "x_vars[9,RPG]",
   "x_4_9": "x_vars[10,RPG]",
   "x_4_10": "x_vars[11,RPG]",
   "x_4_11": "x_vars[12,RPG]",
   "x_4_12": "x_vars[13,RPG]",
   "x_4_13": "x_vars[14,RPG]",
   "x_4_14": "x_vars[15,RPG]",
   "x_5_0": "x_vars[1,Shooter]",
   "x_5_1": "x_vars[2,Shooter]",
   "x_5_2": "x_vars[3,Shooter]",
   "x_5_3": "x_vars[4,Shooter]",
   "x_5_4": "x_vars[5,Shooter]",
   "x_5_5": "x_vars[6,Shooter]",
   "x_5_6": "x_vars[7,Shooter]",
   "x_5_7": "x_vars[8,Shooter]",
   "x_5_8": "x_vars[9,Shooter]",
   "x_5_9": "x_vars[10,Shooter]",
   "x_5_10": "x_vars[11,Shooter]",
   "x_5_11": "x_vars[12,Shooter]",
   "x_5_12": "x_vars[13,Shooter]",
   "x_5_13": "x_vars[14,Shooter]",
   "x_5_14": "x_vars[15,Shooter]",
   "x_6_0": "x_vars[1,Strategy]",
   "x_6_1": "x_vars[2,Strategy]",
   "x_6_2": "x_vars[3,Strategy]",
   "x_6_3": "x_vars[4,Strategy]",
   "x_6_4": "x_vars[5,Strategy]",
   "x_6_5": "x_vars[6,Strategy]",
   "x_6_6": "x_vars[7,Strategy]",
   "x_6_7": "x_vars[8,Strategy]",
   "x_6_8": "x_vars[9,Strategy]",
   "x_6_9": "x_vars[10,Strategy]",
   "x_6_10": "x_vars[11,Strategy]",
   "x_6_11": "x_vars[12,Strategy]",
   "x_6_12": "x_vars[13,Strategy]",
   "x_6_13": "x_vars[14,Strategy]",
   "x_6_14": "x_vars[15,Strategy]",
   "x_7_0": "x_vars[1,Simulation]",
   "x_7_1": "x_vars[2,Simulation]",
   "x_7_2": "x_vars[3,Simulation]",
   "x_7_3": "x_vars[4,Simulation]",
   "x_7_4": "x_vars[5,Simulation]",
   "x_7_5": "x_vars[6,Simulation]",
   "x_7_6": "x_vars[7,Simulation]",
   "x_7_7": "x_vars[8,Simulation]",
   "x_7_8": "x_vars[9,Simulation]",
   "x_7_9": "x_vars[10,Simulation]",
   "x_7_10": "x_vars[11,Simulation]",
   "x_7_11": "x_vars[12,Simulation]",
   "x_7_12": "x_vars[13,Simulation]",
   "x_7_13": "x_vars[14,Simulation]",
   "x_7_14": "x_vars[15,Simulation]",
   "x_8_0": "x_vars[1,Puzzle]",
   "x_8_1": "x_vars[2,Puzzle]",
   "x_8_2": "x_vars[3,Puzzle]",
   "x_8_3": "x_vars[4,Puzzle]",
   "x_8_4": "x_vars[5,Puzzle]",
   "x_8_5": "x_vars[6,Puzzle]",
   "x_8_6": "x_vars[7,Puzzle]",
   "x_8_7": "x_vars[8,Puzzle]",
   "x_8_8": "x_vars[9,Puzzle]",
   "x_8_9": "x_vars[10,Puzzle]",
   "x_8_10": "x_vars[11,Puzzle]",
   "x_8_11": "x_vars[12,Puzzle]",
   "x_8_12": "x_vars[13,Puzzle]",
   "x_8_13": "x_vars[14,Puzzle]",
   "x_8_14": "x_vars[15,Puzzle]",
   "x_9_0": "x_vars[1,Fighting]",
   "x_9_1": "x_vars[2,Fighting]",
   "x_9_2": "x_vars[3,Fighting]",
   "x_9_3": "x_vars[4,Fighting]",
   "x_9_4": "x_vars[5,Fighting]",
   "x_9_5": "x_vars[6,Fighting]",
   "x_9_6": "x_vars[7,Fighting]",
   "x_9_7": "x_vars[8,Fighting]",
   "x_9_8": "x_vars[9,Fighting]",
   "x_9_9": "x_vars[10,Fighting]",
   "x_9_10": "x_vars[11,Fighting]",
   "x_9_11": "x_vars[12,Fighting]",
   "x_9_12": "x_vars[13,Fighting]",
   "x_9_13": "x_vars[14,Fighting]",
   "x_9_14": "x_vars[15,Fighting]",
   "x_10_0": "x_vars[1,Platformer]",
   "x_10_1": "x_vars[2,Platformer]",
   "x_10_2": "x_vars[3,Platformer]",
   "x_10_3": "x_vars[4,Platformer]",
   "x_10_4": "x_vars[5,Platformer]",
   "x_10_5": "x_vars[6,Platformer]",
   "x_10_6": "x_vars[7,Platformer]",
   "x_10_7": "x_vars[8,Platformer]",
   "x_10_8": "x_vars[9,Platformer]",
   "x_10_9": "x_vars[10,Platformer]",
   "x_10_10": "x_vars[11,Platformer]",
   "x_10_11": "x_vars[12,Platformer]",
   "x_10_12": "x_vars[13,Platformer]",
   "x_10_13": "x_vars[14,Platformer]",
   "x_10_14": "x_vars[15,Platformer]",
   "x_11_0": "x_vars[1,Survival]",
   "x_11_1": "x_vars[2,Survival]",
   "x_11_2": "x_vars[3,Survival]",
   "x_11_3": "x_vars[4,Survival]",
   "x_11_4": "x_vars[5,Survival]",
   "x_11_5": "x_vars[6,Survival]",
   "x_11_6": "x_vars[7,Survival]",
   "x_11_7": "x_vars[8,Survival]",
   "x_11_8": "x_vars[9,Survival]",
   "x_11_9": "x_vars[10,Survival]",
   "x_11_10": "x_vars[11,Survival]",
   "x_11_11": "x_vars[12,Survival]",
   "x_11_12": "x_vars[13,Survival]",
   "x_11_13": "x_vars[14,Survival]",
   "x_11_14": "x_vars[15,Survival]",
   "x_12_0": "x_vars[1,Horror]",
   "x_12_1": "x_vars[2,Horror]",
   "x_12_2": "x_vars[3,Horror]",
   "x_12_3": "x_vars[4,Horror]",
   "x_12_4": "x_vars[5,Horror]",
   "x_12_5": "x_vars[6,Horror]",
   "x_12_6": "x_vars[7,Horror]",
   "x_12_7": "x_vars[8,Horror]",
   "x_12_8": "x_vars[9,Horror]",
   "x_12_9": "x_vars[10,Horror]",
   "x_12_10": "x_vars[11,Horror]",
   "x_12_11": "x_vars[12,Horror]",
   "x_12_12": "x_vars[13,Horror]",
   "x_12_13": "x_vars[14,Horror]",
   "x_12_14": "x_vars[15,Horror]",
   "x_13_0": "x_vars[1,Sandbox]",
   "x_13_1": "x_vars[2,Sandbox]",
   "x_13_2": "x_vars[3,Sandbox]",
   "x_13_3": "x_vars[4,Sandbox]",
   "x_13_4": "x_vars[5,Sandbox]",
   "x_13_5": "x_vars[6,Sandbox]",
   "x_13_6": "x_vars[7,Sandbox]",
   "x_13_7": "x_vars[8,Sandbox]",
   "x_13_8": "x_vars[9,Sandbox]",
   "x_13_9": "x_vars[10,Sandbox]",
   "x_13_10": "x_vars[11,Sandbox]",
   "x_13_11": "x_vars[12,Sandbox]",
   "x_13_12": "x_vars[13,Sandbox]",
   "x_13_13": "x_vars[14,Sandbox]",
   "x_13_14": "x_vars[15,Sandbox]",
   "x_14_0": "x_vars[1,MMO]",
   "x_14_1": "x_vars[2,MMO]",
   "x_14_2": "x_vars[3,MMO]",
   "x_14_3": "x_vars[4,MMO]",
   "x_14_4": "x_vars[5,MMO]",
   "x_14_5": "x_vars[6,MMO]",
   "x_14_6": "x_vars[7,MMO]",
   "x_14_7": "x_vars[8,MMO]",
   "x_14_8": "x_vars[9,MMO]",
   "x_14_9": "x_vars[10,MMO]",
   "x_14_10": "x_vars[11,MMO]",
   "x_14_11": "x_vars[12,MMO]",
   "x_14_12": "x_vars[13,MMO]",
   "x_14_13": "x_vars[14,MMO]",
   "x_14_14": "x_vars[15,MMO]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-058": {
  "source_code_sha256": "d924abab236ff14cc02f3029bb0b85e19b4387ccdfee0694e3ac876f0415cad5",
  "original_solution_sha256": "d9571a7b201f3122db5bb762c3bc642e8babe053e853c962fddf1209556316d3",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S2,C1]"
   ],
   [
    "C7",
    "x_vars[S2,C2]"
   ],
   [
    "C8",
    "x_vars[S2,C3]"
   ],
   [
    "C9",
    "x_vars[S2,C4]"
   ],
   [
    "C10",
    "x_vars[S2,C5]"
   ],
   [
    "C11",
    "x_vars[S2,C6]"
   ],
   [
    "C12",
    "x_vars[S3,C1]"
   ],
   [
    "C13",
    "x_vars[S3,C2]"
   ],
   [
    "C14",
    "x_vars[S3,C3]"
   ],
   [
    "C15",
    "x_vars[S3,C4]"
   ],
   [
    "C16",
    "x_vars[S3,C5]"
   ],
   [
    "C17",
    "x_vars[S3,C6]"
   ],
   [
    "C18",
    "x_vars[S4,C1]"
   ],
   [
    "C19",
    "x_vars[S4,C2]"
   ],
   [
    "C20",
    "x_vars[S4,C3]"
   ],
   [
    "C21",
    "x_vars[S4,C4]"
   ],
   [
    "C22",
    "x_vars[S4,C5]"
   ],
   [
    "C23",
    "x_vars[S4,C6]"
   ],
   [
    "C24",
    "x_vars[S5,C1]"
   ],
   [
    "C25",
    "x_vars[S5,C2]"
   ],
   [
    "C26",
    "x_vars[S5,C3]"
   ],
   [
    "C27",
    "x_vars[S5,C4]"
   ],
   [
    "C28",
    "x_vars[S5,C5]"
   ],
   [
    "C29",
    "x_vars[S5,C6]"
   ],
   [
    "C30",
    "x_vars[S6,C1]"
   ],
   [
    "C31",
    "x_vars[S6,C2]"
   ],
   [
    "C32",
    "x_vars[S6,C3]"
   ],
   [
    "C33",
    "x_vars[S6,C4]"
   ],
   [
    "C34",
    "x_vars[S6,C5]"
   ],
   [
    "C35",
    "x_vars[S6,C6]"
   ],
   [
    "C36",
    "y_vars[S1]"
   ],
   [
    "C37",
    "y_vars[S2]"
   ],
   [
    "C38",
    "y_vars[S3]"
   ],
   [
    "C39",
    "y_vars[S4]"
   ],
   [
    "C40",
    "y_vars[S5]"
   ],
   [
    "C41",
    "y_vars[S6]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]",
   "y_3": "y_vars[S4]",
   "y_4": "y_vars[S5]",
   "y_5": "y_vars[S6]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-059": {
  "source_code_sha256": "6d751ebc41dac7059b5570f781f9eb05c9fd42efc9cd9a43a4c6c75c76a4ccb8",
  "original_solution_sha256": "d73d6e47c6829a10dc707a0371cc253309b1fddf7a98aaa46d80a80ca88ea938",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S2,C1]"
   ],
   [
    "C10",
    "x_vars[S2,C2]"
   ],
   [
    "C11",
    "x_vars[S2,C3]"
   ],
   [
    "C12",
    "x_vars[S2,C4]"
   ],
   [
    "C13",
    "x_vars[S2,C5]"
   ],
   [
    "C14",
    "x_vars[S2,C6]"
   ],
   [
    "C15",
    "x_vars[S2,C7]"
   ],
   [
    "C16",
    "x_vars[S2,C8]"
   ],
   [
    "C17",
    "x_vars[S2,C9]"
   ],
   [
    "C18",
    "x_vars[S3,C1]"
   ],
   [
    "C19",
    "x_vars[S3,C2]"
   ],
   [
    "C20",
    "x_vars[S3,C3]"
   ],
   [
    "C21",
    "x_vars[S3,C4]"
   ],
   [
    "C22",
    "x_vars[S3,C5]"
   ],
   [
    "C23",
    "x_vars[S3,C6]"
   ],
   [
    "C24",
    "x_vars[S3,C7]"
   ],
   [
    "C25",
    "x_vars[S3,C8]"
   ],
   [
    "C26",
    "x_vars[S3,C9]"
   ],
   [
    "C27",
    "x_vars[S4,C1]"
   ],
   [
    "C28",
    "x_vars[S4,C2]"
   ],
   [
    "C29",
    "x_vars[S4,C3]"
   ],
   [
    "C30",
    "x_vars[S4,C4]"
   ],
   [
    "C31",
    "x_vars[S4,C5]"
   ],
   [
    "C32",
    "x_vars[S4,C6]"
   ],
   [
    "C33",
    "x_vars[S4,C7]"
   ],
   [
    "C34",
    "x_vars[S4,C8]"
   ],
   [
    "C35",
    "x_vars[S4,C9]"
   ],
   [
    "C36",
    "x_vars[S5,C1]"
   ],
   [
    "C37",
    "x_vars[S5,C2]"
   ],
   [
    "C38",
    "x_vars[S5,C3]"
   ],
   [
    "C39",
    "x_vars[S5,C4]"
   ],
   [
    "C40",
    "x_vars[S5,C5]"
   ],
   [
    "C41",
    "x_vars[S5,C6]"
   ],
   [
    "C42",
    "x_vars[S5,C7]"
   ],
   [
    "C43",
    "x_vars[S5,C8]"
   ],
   [
    "C44",
    "x_vars[S5,C9]"
   ],
   [
    "C45",
    "x_vars[S6,C1]"
   ],
   [
    "C46",
    "x_vars[S6,C2]"
   ],
   [
    "C47",
    "x_vars[S6,C3]"
   ],
   [
    "C48",
    "x_vars[S6,C4]"
   ],
   [
    "C49",
    "x_vars[S6,C5]"
   ],
   [
    "C50",
    "x_vars[S6,C6]"
   ],
   [
    "C51",
    "x_vars[S6,C7]"
   ],
   [
    "C52",
    "x_vars[S6,C8]"
   ],
   [
    "C53",
    "x_vars[S6,C9]"
   ],
   [
    "C54",
    "x_vars[S7,C1]"
   ],
   [
    "C55",
    "x_vars[S7,C2]"
   ],
   [
    "C56",
    "x_vars[S7,C3]"
   ],
   [
    "C57",
    "x_vars[S7,C4]"
   ],
   [
    "C58",
    "x_vars[S7,C5]"
   ],
   [
    "C59",
    "x_vars[S7,C6]"
   ],
   [
    "C60",
    "x_vars[S7,C7]"
   ],
   [
    "C61",
    "x_vars[S7,C8]"
   ],
   [
    "C62",
    "x_vars[S7,C9]"
   ],
   [
    "C63",
    "x_vars[S8,C1]"
   ],
   [
    "C64",
    "x_vars[S8,C2]"
   ],
   [
    "C65",
    "x_vars[S8,C3]"
   ],
   [
    "C66",
    "x_vars[S8,C4]"
   ],
   [
    "C67",
    "x_vars[S8,C5]"
   ],
   [
    "C68",
    "x_vars[S8,C6]"
   ],
   [
    "C69",
    "x_vars[S8,C7]"
   ],
   [
    "C70",
    "x_vars[S8,C8]"
   ],
   [
    "C71",
    "x_vars[S8,C9]"
   ],
   [
    "C72",
    "y_vars[S1]"
   ],
   [
    "C73",
    "y_vars[S2]"
   ],
   [
    "C74",
    "y_vars[S3]"
   ],
   [
    "C75",
    "y_vars[S4]"
   ],
   [
    "C76",
    "y_vars[S5]"
   ],
   [
    "C77",
    "y_vars[S6]"
   ],
   [
    "C78",
    "y_vars[S7]"
   ],
   [
    "C79",
    "y_vars[S8]"
   ]
  ],
  "primary_mapping": {
   "y[S1]": "y_vars[S1]",
   "y[S2]": "y_vars[S2]",
   "y[S3]": "y_vars[S3]",
   "y[S4]": "y_vars[S4]",
   "y[S5]": "y_vars[S5]",
   "y[S6]": "y_vars[S6]",
   "y[S7]": "y_vars[S7]",
   "y[S8]": "y_vars[S8]",
   "x[S1,C1]": "x_vars[S1,C1]",
   "x[S1,C9]": "x_vars[S1,C9]",
   "x[S2,C2]": "x_vars[S2,C2]",
   "x[S2,C9]": "x_vars[S2,C9]",
   "x[S3,C3]": "x_vars[S3,C3]",
   "x[S3,C9]": "x_vars[S3,C9]",
   "x[S4,C4]": "x_vars[S4,C4]",
   "x[S4,C9]": "x_vars[S4,C9]",
   "x[S5,C5]": "x_vars[S5,C5]",
   "x[S5,C9]": "x_vars[S5,C9]",
   "x[S6,C6]": "x_vars[S6,C6]",
   "x[S6,C9]": "x_vars[S6,C9]",
   "x[S7,C7]": "x_vars[S7,C7]",
   "x[S7,C9]": "x_vars[S7,C9]",
   "x[S8,C8]": "x_vars[S8,C8]",
   "x[S8,C9]": "x_vars[S8,C9]",
   "x[S1,C2]": "x_vars[S1,C2]",
   "x[S1,C3]": "x_vars[S1,C3]",
   "x[S1,C4]": "x_vars[S1,C4]",
   "x[S1,C5]": "x_vars[S1,C5]",
   "x[S1,C6]": "x_vars[S1,C6]",
   "x[S1,C7]": "x_vars[S1,C7]",
   "x[S1,C8]": "x_vars[S1,C8]",
   "x[S2,C1]": "x_vars[S2,C1]",
   "x[S2,C3]": "x_vars[S2,C3]",
   "x[S2,C4]": "x_vars[S2,C4]",
   "x[S2,C5]": "x_vars[S2,C5]",
   "x[S2,C6]": "x_vars[S2,C6]",
   "x[S2,C7]": "x_vars[S2,C7]",
   "x[S2,C8]": "x_vars[S2,C8]",
   "x[S3,C1]": "x_vars[S3,C1]",
   "x[S3,C2]": "x_vars[S3,C2]",
   "x[S3,C4]": "x_vars[S3,C4]",
   "x[S3,C5]": "x_vars[S3,C5]",
   "x[S3,C6]": "x_vars[S3,C6]",
   "x[S3,C7]": "x_vars[S3,C7]",
   "x[S3,C8]": "x_vars[S3,C8]",
   "x[S4,C1]": "x_vars[S4,C1]",
   "x[S4,C2]": "x_vars[S4,C2]",
   "x[S4,C3]": "x_vars[S4,C3]",
   "x[S4,C5]": "x_vars[S4,C5]",
   "x[S4,C6]": "x_vars[S4,C6]",
   "x[S4,C7]": "x_vars[S4,C7]",
   "x[S4,C8]": "x_vars[S4,C8]",
   "x[S5,C1]": "x_vars[S5,C1]",
   "x[S5,C2]": "x_vars[S5,C2]",
   "x[S5,C3]": "x_vars[S5,C3]",
   "x[S5,C4]": "x_vars[S5,C4]",
   "x[S5,C6]": "x_vars[S5,C6]",
   "x[S5,C7]": "x_vars[S5,C7]",
   "x[S5,C8]": "x_vars[S5,C8]",
   "x[S6,C1]": "x_vars[S6,C1]",
   "x[S6,C2]": "x_vars[S6,C2]",
   "x[S6,C3]": "x_vars[S6,C3]",
   "x[S6,C4]": "x_vars[S6,C4]",
   "x[S6,C5]": "x_vars[S6,C5]",
   "x[S6,C7]": "x_vars[S6,C7]",
   "x[S6,C8]": "x_vars[S6,C8]",
   "x[S7,C1]": "x_vars[S7,C1]",
   "x[S7,C2]": "x_vars[S7,C2]",
   "x[S7,C3]": "x_vars[S7,C3]",
   "x[S7,C4]": "x_vars[S7,C4]",
   "x[S7,C5]": "x_vars[S7,C5]",
   "x[S7,C6]": "x_vars[S7,C6]",
   "x[S7,C8]": "x_vars[S7,C8]",
   "x[S8,C1]": "x_vars[S8,C1]",
   "x[S8,C2]": "x_vars[S8,C2]",
   "x[S8,C3]": "x_vars[S8,C3]",
   "x[S8,C4]": "x_vars[S8,C4]",
   "x[S8,C5]": "x_vars[S8,C5]",
   "x[S8,C6]": "x_vars[S8,C6]",
   "x[S8,C7]": "x_vars[S8,C7]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-060": {
  "source_code_sha256": "212baa67ec97c85376da297b26dc576ba2426d0ae6f95f0addf4f6bd76eaa6d7",
  "original_solution_sha256": "334a6abd03ef3e20cb28ac19f741564c4af2d12d989ed5a2585613be90a35b29",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S1,C11]"
   ],
   [
    "C11",
    "x_vars[S1,C12]"
   ],
   [
    "C12",
    "x_vars[S2,C1]"
   ],
   [
    "C13",
    "x_vars[S2,C2]"
   ],
   [
    "C14",
    "x_vars[S2,C3]"
   ],
   [
    "C15",
    "x_vars[S2,C4]"
   ],
   [
    "C16",
    "x_vars[S2,C5]"
   ],
   [
    "C17",
    "x_vars[S2,C6]"
   ],
   [
    "C18",
    "x_vars[S2,C7]"
   ],
   [
    "C19",
    "x_vars[S2,C8]"
   ],
   [
    "C20",
    "x_vars[S2,C9]"
   ],
   [
    "C21",
    "x_vars[S2,C10]"
   ],
   [
    "C22",
    "x_vars[S2,C11]"
   ],
   [
    "C23",
    "x_vars[S2,C12]"
   ],
   [
    "C24",
    "x_vars[S3,C1]"
   ],
   [
    "C25",
    "x_vars[S3,C2]"
   ],
   [
    "C26",
    "x_vars[S3,C3]"
   ],
   [
    "C27",
    "x_vars[S3,C4]"
   ],
   [
    "C28",
    "x_vars[S3,C5]"
   ],
   [
    "C29",
    "x_vars[S3,C6]"
   ],
   [
    "C30",
    "x_vars[S3,C7]"
   ],
   [
    "C31",
    "x_vars[S3,C8]"
   ],
   [
    "C32",
    "x_vars[S3,C9]"
   ],
   [
    "C33",
    "x_vars[S3,C10]"
   ],
   [
    "C34",
    "x_vars[S3,C11]"
   ],
   [
    "C35",
    "x_vars[S3,C12]"
   ],
   [
    "C36",
    "x_vars[S4,C1]"
   ],
   [
    "C37",
    "x_vars[S4,C2]"
   ],
   [
    "C38",
    "x_vars[S4,C3]"
   ],
   [
    "C39",
    "x_vars[S4,C4]"
   ],
   [
    "C40",
    "x_vars[S4,C5]"
   ],
   [
    "C41",
    "x_vars[S4,C6]"
   ],
   [
    "C42",
    "x_vars[S4,C7]"
   ],
   [
    "C43",
    "x_vars[S4,C8]"
   ],
   [
    "C44",
    "x_vars[S4,C9]"
   ],
   [
    "C45",
    "x_vars[S4,C10]"
   ],
   [
    "C46",
    "x_vars[S4,C11]"
   ],
   [
    "C47",
    "x_vars[S4,C12]"
   ],
   [
    "C48",
    "x_vars[S5,C1]"
   ],
   [
    "C49",
    "x_vars[S5,C2]"
   ],
   [
    "C50",
    "x_vars[S5,C3]"
   ],
   [
    "C51",
    "x_vars[S5,C4]"
   ],
   [
    "C52",
    "x_vars[S5,C5]"
   ],
   [
    "C53",
    "x_vars[S5,C6]"
   ],
   [
    "C54",
    "x_vars[S5,C7]"
   ],
   [
    "C55",
    "x_vars[S5,C8]"
   ],
   [
    "C56",
    "x_vars[S5,C9]"
   ],
   [
    "C57",
    "x_vars[S5,C10]"
   ],
   [
    "C58",
    "x_vars[S5,C11]"
   ],
   [
    "C59",
    "x_vars[S5,C12]"
   ],
   [
    "C60",
    "x_vars[S6,C1]"
   ],
   [
    "C61",
    "x_vars[S6,C2]"
   ],
   [
    "C62",
    "x_vars[S6,C3]"
   ],
   [
    "C63",
    "x_vars[S6,C4]"
   ],
   [
    "C64",
    "x_vars[S6,C5]"
   ],
   [
    "C65",
    "x_vars[S6,C6]"
   ],
   [
    "C66",
    "x_vars[S6,C7]"
   ],
   [
    "C67",
    "x_vars[S6,C8]"
   ],
   [
    "C68",
    "x_vars[S6,C9]"
   ],
   [
    "C69",
    "x_vars[S6,C10]"
   ],
   [
    "C70",
    "x_vars[S6,C11]"
   ],
   [
    "C71",
    "x_vars[S6,C12]"
   ],
   [
    "C72",
    "x_vars[S7,C1]"
   ],
   [
    "C73",
    "x_vars[S7,C2]"
   ],
   [
    "C74",
    "x_vars[S7,C3]"
   ],
   [
    "C75",
    "x_vars[S7,C4]"
   ],
   [
    "C76",
    "x_vars[S7,C5]"
   ],
   [
    "C77",
    "x_vars[S7,C6]"
   ],
   [
    "C78",
    "x_vars[S7,C7]"
   ],
   [
    "C79",
    "x_vars[S7,C8]"
   ],
   [
    "C80",
    "x_vars[S7,C9]"
   ],
   [
    "C81",
    "x_vars[S7,C10]"
   ],
   [
    "C82",
    "x_vars[S7,C11]"
   ],
   [
    "C83",
    "x_vars[S7,C12]"
   ],
   [
    "C84",
    "x_vars[S8,C1]"
   ],
   [
    "C85",
    "x_vars[S8,C2]"
   ],
   [
    "C86",
    "x_vars[S8,C3]"
   ],
   [
    "C87",
    "x_vars[S8,C4]"
   ],
   [
    "C88",
    "x_vars[S8,C5]"
   ],
   [
    "C89",
    "x_vars[S8,C6]"
   ],
   [
    "C90",
    "x_vars[S8,C7]"
   ],
   [
    "C91",
    "x_vars[S8,C8]"
   ],
   [
    "C92",
    "x_vars[S8,C9]"
   ],
   [
    "C93",
    "x_vars[S8,C10]"
   ],
   [
    "C94",
    "x_vars[S8,C11]"
   ],
   [
    "C95",
    "x_vars[S8,C12]"
   ],
   [
    "C96",
    "x_vars[S9,C1]"
   ],
   [
    "C97",
    "x_vars[S9,C2]"
   ],
   [
    "C98",
    "x_vars[S9,C3]"
   ],
   [
    "C99",
    "x_vars[S9,C4]"
   ],
   [
    "C100",
    "x_vars[S9,C5]"
   ],
   [
    "C101",
    "x_vars[S9,C6]"
   ],
   [
    "C102",
    "x_vars[S9,C7]"
   ],
   [
    "C103",
    "x_vars[S9,C8]"
   ],
   [
    "C104",
    "x_vars[S9,C9]"
   ],
   [
    "C105",
    "x_vars[S9,C10]"
   ],
   [
    "C106",
    "x_vars[S9,C11]"
   ],
   [
    "C107",
    "x_vars[S9,C12]"
   ],
   [
    "C108",
    "x_vars[S10,C1]"
   ],
   [
    "C109",
    "x_vars[S10,C2]"
   ],
   [
    "C110",
    "x_vars[S10,C3]"
   ],
   [
    "C111",
    "x_vars[S10,C4]"
   ],
   [
    "C112",
    "x_vars[S10,C5]"
   ],
   [
    "C113",
    "x_vars[S10,C6]"
   ],
   [
    "C114",
    "x_vars[S10,C7]"
   ],
   [
    "C115",
    "x_vars[S10,C8]"
   ],
   [
    "C116",
    "x_vars[S10,C9]"
   ],
   [
    "C117",
    "x_vars[S10,C10]"
   ],
   [
    "C118",
    "x_vars[S10,C11]"
   ],
   [
    "C119",
    "x_vars[S10,C12]"
   ],
   [
    "C120",
    "x_vars[S11,C1]"
   ],
   [
    "C121",
    "x_vars[S11,C2]"
   ],
   [
    "C122",
    "x_vars[S11,C3]"
   ],
   [
    "C123",
    "x_vars[S11,C4]"
   ],
   [
    "C124",
    "x_vars[S11,C5]"
   ],
   [
    "C125",
    "x_vars[S11,C6]"
   ],
   [
    "C126",
    "x_vars[S11,C7]"
   ],
   [
    "C127",
    "x_vars[S11,C8]"
   ],
   [
    "C128",
    "x_vars[S11,C9]"
   ],
   [
    "C129",
    "x_vars[S11,C10]"
   ],
   [
    "C130",
    "x_vars[S11,C11]"
   ],
   [
    "C131",
    "x_vars[S11,C12]"
   ],
   [
    "C132",
    "x_vars[S12,C1]"
   ],
   [
    "C133",
    "x_vars[S12,C2]"
   ],
   [
    "C134",
    "x_vars[S12,C3]"
   ],
   [
    "C135",
    "x_vars[S12,C4]"
   ],
   [
    "C136",
    "x_vars[S12,C5]"
   ],
   [
    "C137",
    "x_vars[S12,C6]"
   ],
   [
    "C138",
    "x_vars[S12,C7]"
   ],
   [
    "C139",
    "x_vars[S12,C8]"
   ],
   [
    "C140",
    "x_vars[S12,C9]"
   ],
   [
    "C141",
    "x_vars[S12,C10]"
   ],
   [
    "C142",
    "x_vars[S12,C11]"
   ],
   [
    "C143",
    "x_vars[S12,C12]"
   ],
   [
    "C144",
    "y_vars[S1]"
   ],
   [
    "C145",
    "y_vars[S2]"
   ],
   [
    "C146",
    "y_vars[S3]"
   ],
   [
    "C147",
    "y_vars[S4]"
   ],
   [
    "C148",
    "y_vars[S5]"
   ],
   [
    "C149",
    "y_vars[S6]"
   ],
   [
    "C150",
    "y_vars[S7]"
   ],
   [
    "C151",
    "y_vars[S8]"
   ],
   [
    "C152",
    "y_vars[S9]"
   ],
   [
    "C153",
    "y_vars[S10]"
   ],
   [
    "C154",
    "y_vars[S11]"
   ],
   [
    "C155",
    "y_vars[S12]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_0_10": "x_vars[S1,C11]",
   "x_0_11": "x_vars[S1,C12]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_1_10": "x_vars[S2,C11]",
   "x_1_11": "x_vars[S2,C12]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_2_10": "x_vars[S3,C11]",
   "x_2_11": "x_vars[S3,C12]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_3_10": "x_vars[S4,C11]",
   "x_3_11": "x_vars[S4,C12]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_4_10": "x_vars[S5,C11]",
   "x_4_11": "x_vars[S5,C12]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_5_10": "x_vars[S6,C11]",
   "x_5_11": "x_vars[S6,C12]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_9": "x_vars[S7,C10]",
   "x_6_10": "x_vars[S7,C11]",
   "x_6_11": "x_vars[S7,C12]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_7_10": "x_vars[S8,C11]",
   "x_7_11": "x_vars[S8,C12]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_8_10": "x_vars[S9,C11]",
   "x_8_11": "x_vars[S9,C12]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]",
   "x_9_10": "x_vars[S10,C11]",
   "x_9_11": "x_vars[S10,C12]",
   "x_10_0": "x_vars[S11,C1]",
   "x_10_2": "x_vars[S11,C3]",
   "x_10_3": "x_vars[S11,C4]",
   "x_10_4": "x_vars[S11,C5]",
   "x_10_5": "x_vars[S11,C6]",
   "x_10_6": "x_vars[S11,C7]",
   "x_10_7": "x_vars[S11,C8]",
   "x_10_8": "x_vars[S11,C9]",
   "x_10_9": "x_vars[S11,C10]",
   "x_10_10": "x_vars[S11,C11]",
   "x_10_11": "x_vars[S11,C12]",
   "x_11_0": "x_vars[S12,C1]",
   "x_11_1": "x_vars[S12,C2]",
   "x_11_2": "x_vars[S12,C3]",
   "x_11_3": "x_vars[S12,C4]",
   "x_11_4": "x_vars[S12,C5]",
   "x_11_5": "x_vars[S12,C6]",
   "x_11_6": "x_vars[S12,C7]",
   "x_11_7": "x_vars[S12,C8]",
   "x_11_8": "x_vars[S12,C9]",
   "x_11_9": "x_vars[S12,C10]",
   "x_11_10": "x_vars[S12,C11]",
   "x_11_11": "x_vars[S12,C12]",
   "x_10_1": "x_vars[S11,C2]",
   "x_6_8": "x_vars[S7,C9]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]",
   "y_3": "y_vars[S4]",
   "y_4": "y_vars[S5]",
   "y_5": "y_vars[S6]",
   "y_6": "y_vars[S7]",
   "y_7": "y_vars[S8]",
   "y_8": "y_vars[S9]",
   "y_9": "y_vars[S10]",
   "y_10": "y_vars[S11]",
   "y_11": "y_vars[S12]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-061": {
  "source_code_sha256": "f070b38c2715b953fc8327d15d08d484560f0ca7f9791d77ff0aabcbeb49a02c",
  "original_solution_sha256": "a6f83c019b93e1892fcbd4e2e29ccaf3cb82fdca7020bcdb0c7205ab0df04f6b",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S2,C1]"
   ],
   [
    "C6",
    "x_vars[S2,C2]"
   ],
   [
    "C7",
    "x_vars[S2,C3]"
   ],
   [
    "C8",
    "x_vars[S2,C4]"
   ],
   [
    "C9",
    "x_vars[S2,C5]"
   ],
   [
    "C10",
    "x_vars[S3,C1]"
   ],
   [
    "C11",
    "x_vars[S3,C2]"
   ],
   [
    "C12",
    "x_vars[S3,C3]"
   ],
   [
    "C13",
    "x_vars[S3,C4]"
   ],
   [
    "C14",
    "x_vars[S3,C5]"
   ],
   [
    "C15",
    "x_vars[S4,C1]"
   ],
   [
    "C16",
    "x_vars[S4,C2]"
   ],
   [
    "C17",
    "x_vars[S4,C3]"
   ],
   [
    "C18",
    "x_vars[S4,C4]"
   ],
   [
    "C19",
    "x_vars[S4,C5]"
   ],
   [
    "C20",
    "x_vars[S5,C1]"
   ],
   [
    "C21",
    "x_vars[S5,C2]"
   ],
   [
    "C22",
    "x_vars[S5,C3]"
   ],
   [
    "C23",
    "x_vars[S5,C4]"
   ],
   [
    "C24",
    "x_vars[S5,C5]"
   ],
   [
    "C25",
    "y_vars[S1]"
   ],
   [
    "C26",
    "y_vars[S2]"
   ],
   [
    "C27",
    "y_vars[S3]"
   ],
   [
    "C28",
    "y_vars[S4]"
   ],
   [
    "C29",
    "y_vars[S5]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]",
   "y_3": "y_vars[S4]",
   "y_4": "y_vars[S5]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-062": {
  "source_code_sha256": "e10da2df4bcee4e4daf57b4be3b96fffa3280fa3e5387126b80e7e6d29ace5bf",
  "original_solution_sha256": "9fc0b60fb12e3e839b4b5f0d76c98060f636b4173bf5294b76ff2915ee0c3f34",
  "bindings": [
   [
    "C0",
    "x_vars[MOUNT AYR,CLARINDA]"
   ],
   [
    "C1",
    "x_vars[MOUNT AYR,FORT MADISON]"
   ],
   [
    "C2",
    "x_vars[MOUNT AYR,SIOUX CITY]"
   ],
   [
    "C3",
    "x_vars[MOUNT AYR,TOLEDO]"
   ],
   [
    "C4",
    "x_vars[MOUNT AYR,BANCROFT]"
   ],
   [
    "C5",
    "x_vars[WAUKEE,CLARINDA]"
   ],
   [
    "C6",
    "x_vars[WAUKEE,FORT MADISON]"
   ],
   [
    "C7",
    "x_vars[WAUKEE,SIOUX CITY]"
   ],
   [
    "C8",
    "x_vars[WAUKEE,TOLEDO]"
   ],
   [
    "C9",
    "x_vars[WAUKEE,BANCROFT]"
   ],
   [
    "C10",
    "x_vars[WAVERLY,CLARINDA]"
   ],
   [
    "C11",
    "x_vars[WAVERLY,FORT MADISON]"
   ],
   [
    "C12",
    "x_vars[WAVERLY,SIOUX CITY]"
   ],
   [
    "C13",
    "x_vars[WAVERLY,TOLEDO]"
   ],
   [
    "C14",
    "x_vars[WAVERLY,BANCROFT]"
   ],
   [
    "C15",
    "x_vars[PELLA,CLARINDA]"
   ],
   [
    "C16",
    "x_vars[PELLA,FORT MADISON]"
   ],
   [
    "C17",
    "x_vars[PELLA,SIOUX CITY]"
   ],
   [
    "C18",
    "x_vars[PELLA,TOLEDO]"
   ],
   [
    "C19",
    "x_vars[PELLA,BANCROFT]"
   ],
   [
    "C20",
    "x_vars[DES MOINES,CLARINDA]"
   ],
   [
    "C21",
    "x_vars[DES MOINES,FORT MADISON]"
   ],
   [
    "C22",
    "x_vars[DES MOINES,SIOUX CITY]"
   ],
   [
    "C23",
    "x_vars[DES MOINES,TOLEDO]"
   ],
   [
    "C24",
    "x_vars[DES MOINES,BANCROFT]"
   ],
   [
    "C25",
    "y_vars[MOUNT AYR]"
   ],
   [
    "C26",
    "y_vars[WAUKEE]"
   ],
   [
    "C27",
    "y_vars[WAVERLY]"
   ],
   [
    "C28",
    "y_vars[PELLA]"
   ],
   [
    "C29",
    "y_vars[DES MOINES]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[MOUNT AYR,CLARINDA]",
   "x_0_1": "x_vars[MOUNT AYR,FORT MADISON]",
   "x_0_2": "x_vars[MOUNT AYR,SIOUX CITY]",
   "x_0_3": "x_vars[MOUNT AYR,TOLEDO]",
   "x_0_4": "x_vars[MOUNT AYR,BANCROFT]",
   "x_1_0": "x_vars[WAUKEE,CLARINDA]",
   "x_1_1": "x_vars[WAUKEE,FORT MADISON]",
   "x_1_2": "x_vars[WAUKEE,SIOUX CITY]",
   "x_1_3": "x_vars[WAUKEE,TOLEDO]",
   "x_1_4": "x_vars[WAUKEE,BANCROFT]",
   "x_2_0": "x_vars[WAVERLY,CLARINDA]",
   "x_2_1": "x_vars[WAVERLY,FORT MADISON]",
   "x_2_2": "x_vars[WAVERLY,SIOUX CITY]",
   "x_2_3": "x_vars[WAVERLY,TOLEDO]",
   "x_2_4": "x_vars[WAVERLY,BANCROFT]",
   "x_3_0": "x_vars[PELLA,CLARINDA]",
   "x_3_1": "x_vars[PELLA,FORT MADISON]",
   "x_3_2": "x_vars[PELLA,SIOUX CITY]",
   "x_3_3": "x_vars[PELLA,TOLEDO]",
   "x_3_4": "x_vars[PELLA,BANCROFT]",
   "x_4_0": "x_vars[DES MOINES,CLARINDA]",
   "x_4_1": "x_vars[DES MOINES,FORT MADISON]",
   "x_4_2": "x_vars[DES MOINES,SIOUX CITY]",
   "x_4_3": "x_vars[DES MOINES,TOLEDO]",
   "x_4_4": "x_vars[DES MOINES,BANCROFT]",
   "y_0": "y_vars[MOUNT AYR]",
   "y_1": "y_vars[WAUKEE]",
   "y_2": "y_vars[WAVERLY]",
   "y_3": "y_vars[PELLA]",
   "y_4": "y_vars[DES MOINES]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-063": {
  "source_code_sha256": "392a8b1180565ca8931497efff6bf9504d61c0f57758a506fa9bce2ff8cc37c3",
  "original_solution_sha256": "a39be2de0325f735fde048339e8c8e345a246d4f85babdb82675cb794178b88b",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S2,C1]"
   ],
   [
    "C8",
    "x_vars[S2,C2]"
   ],
   [
    "C9",
    "x_vars[S2,C3]"
   ],
   [
    "C10",
    "x_vars[S2,C4]"
   ],
   [
    "C11",
    "x_vars[S2,C5]"
   ],
   [
    "C12",
    "x_vars[S2,C6]"
   ],
   [
    "C13",
    "x_vars[S2,C7]"
   ],
   [
    "C14",
    "x_vars[S3,C1]"
   ],
   [
    "C15",
    "x_vars[S3,C2]"
   ],
   [
    "C16",
    "x_vars[S3,C3]"
   ],
   [
    "C17",
    "x_vars[S3,C4]"
   ],
   [
    "C18",
    "x_vars[S3,C5]"
   ],
   [
    "C19",
    "x_vars[S3,C6]"
   ],
   [
    "C20",
    "x_vars[S3,C7]"
   ],
   [
    "C21",
    "x_vars[S4,C1]"
   ],
   [
    "C22",
    "x_vars[S4,C2]"
   ],
   [
    "C23",
    "x_vars[S4,C3]"
   ],
   [
    "C24",
    "x_vars[S4,C4]"
   ],
   [
    "C25",
    "x_vars[S4,C5]"
   ],
   [
    "C26",
    "x_vars[S4,C6]"
   ],
   [
    "C27",
    "x_vars[S4,C7]"
   ],
   [
    "C28",
    "x_vars[S5,C1]"
   ],
   [
    "C29",
    "x_vars[S5,C2]"
   ],
   [
    "C30",
    "x_vars[S5,C3]"
   ],
   [
    "C31",
    "x_vars[S5,C4]"
   ],
   [
    "C32",
    "x_vars[S5,C5]"
   ],
   [
    "C33",
    "x_vars[S5,C6]"
   ],
   [
    "C34",
    "x_vars[S5,C7]"
   ],
   [
    "C35",
    "x_vars[S6,C1]"
   ],
   [
    "C36",
    "x_vars[S6,C2]"
   ],
   [
    "C37",
    "x_vars[S6,C3]"
   ],
   [
    "C38",
    "x_vars[S6,C4]"
   ],
   [
    "C39",
    "x_vars[S6,C5]"
   ],
   [
    "C40",
    "x_vars[S6,C6]"
   ],
   [
    "C41",
    "x_vars[S6,C7]"
   ],
   [
    "C42",
    "x_vars[S7,C1]"
   ],
   [
    "C43",
    "x_vars[S7,C2]"
   ],
   [
    "C44",
    "x_vars[S7,C3]"
   ],
   [
    "C45",
    "x_vars[S7,C4]"
   ],
   [
    "C46",
    "x_vars[S7,C5]"
   ],
   [
    "C47",
    "x_vars[S7,C6]"
   ],
   [
    "C48",
    "x_vars[S7,C7]"
   ],
   [
    "C49",
    "y_vars[S1]"
   ],
   [
    "C50",
    "y_vars[S2]"
   ],
   [
    "C51",
    "y_vars[S3]"
   ],
   [
    "C52",
    "y_vars[S4]"
   ],
   [
    "C53",
    "y_vars[S5]"
   ],
   [
    "C54",
    "y_vars[S6]"
   ],
   [
    "C55",
    "y_vars[S7]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]",
   "y_3": "y_vars[S4]",
   "y_4": "y_vars[S5]",
   "y_5": "y_vars[S6]",
   "y_6": "y_vars[S7]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-064": {
  "source_code_sha256": "cf0983725953965d33dcc0abc336691e3cdf817e061ae244bb08c9506dcfc8de",
  "original_solution_sha256": "7f587853ca7491fac2946e4b22fb28f74fd7f469402e67365bc2ab96207a63ec",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S1,C4]"
   ],
   [
    "C4",
    "x_vars[S1,C5]"
   ],
   [
    "C5",
    "x_vars[S1,C6]"
   ],
   [
    "C6",
    "x_vars[S1,C7]"
   ],
   [
    "C7",
    "x_vars[S1,C8]"
   ],
   [
    "C8",
    "x_vars[S1,C9]"
   ],
   [
    "C9",
    "x_vars[S1,C10]"
   ],
   [
    "C10",
    "x_vars[S1,C11]"
   ],
   [
    "C11",
    "x_vars[S1,C12]"
   ],
   [
    "C12",
    "x_vars[S1,C13]"
   ],
   [
    "C13",
    "x_vars[S1,C14]"
   ],
   [
    "C14",
    "x_vars[S1,C15]"
   ],
   [
    "C15",
    "x_vars[S1,C16]"
   ],
   [
    "C16",
    "x_vars[S1,C17]"
   ],
   [
    "C17",
    "x_vars[S1,C18]"
   ],
   [
    "C18",
    "x_vars[S1,C19]"
   ],
   [
    "C19",
    "x_vars[S1,C20]"
   ],
   [
    "C20",
    "x_vars[S1,C21]"
   ],
   [
    "C21",
    "x_vars[S1,C22]"
   ],
   [
    "C22",
    "x_vars[S1,C23]"
   ],
   [
    "C23",
    "x_vars[S1,C24]"
   ],
   [
    "C24",
    "x_vars[S1,C25]"
   ],
   [
    "C25",
    "x_vars[S2,C1]"
   ],
   [
    "C26",
    "x_vars[S2,C2]"
   ],
   [
    "C27",
    "x_vars[S2,C3]"
   ],
   [
    "C28",
    "x_vars[S2,C4]"
   ],
   [
    "C29",
    "x_vars[S2,C5]"
   ],
   [
    "C30",
    "x_vars[S2,C6]"
   ],
   [
    "C31",
    "x_vars[S2,C7]"
   ],
   [
    "C32",
    "x_vars[S2,C8]"
   ],
   [
    "C33",
    "x_vars[S2,C9]"
   ],
   [
    "C34",
    "x_vars[S2,C10]"
   ],
   [
    "C35",
    "x_vars[S2,C11]"
   ],
   [
    "C36",
    "x_vars[S2,C12]"
   ],
   [
    "C37",
    "x_vars[S2,C13]"
   ],
   [
    "C38",
    "x_vars[S2,C14]"
   ],
   [
    "C39",
    "x_vars[S2,C15]"
   ],
   [
    "C40",
    "x_vars[S2,C16]"
   ],
   [
    "C41",
    "x_vars[S2,C17]"
   ],
   [
    "C42",
    "x_vars[S2,C18]"
   ],
   [
    "C43",
    "x_vars[S2,C19]"
   ],
   [
    "C44",
    "x_vars[S2,C20]"
   ],
   [
    "C45",
    "x_vars[S2,C21]"
   ],
   [
    "C46",
    "x_vars[S2,C22]"
   ],
   [
    "C47",
    "x_vars[S2,C23]"
   ],
   [
    "C48",
    "x_vars[S2,C24]"
   ],
   [
    "C49",
    "x_vars[S2,C25]"
   ],
   [
    "C50",
    "x_vars[S3,C1]"
   ],
   [
    "C51",
    "x_vars[S3,C2]"
   ],
   [
    "C52",
    "x_vars[S3,C3]"
   ],
   [
    "C53",
    "x_vars[S3,C4]"
   ],
   [
    "C54",
    "x_vars[S3,C5]"
   ],
   [
    "C55",
    "x_vars[S3,C6]"
   ],
   [
    "C56",
    "x_vars[S3,C7]"
   ],
   [
    "C57",
    "x_vars[S3,C8]"
   ],
   [
    "C58",
    "x_vars[S3,C9]"
   ],
   [
    "C59",
    "x_vars[S3,C10]"
   ],
   [
    "C60",
    "x_vars[S3,C11]"
   ],
   [
    "C61",
    "x_vars[S3,C12]"
   ],
   [
    "C62",
    "x_vars[S3,C13]"
   ],
   [
    "C63",
    "x_vars[S3,C14]"
   ],
   [
    "C64",
    "x_vars[S3,C15]"
   ],
   [
    "C65",
    "x_vars[S3,C16]"
   ],
   [
    "C66",
    "x_vars[S3,C17]"
   ],
   [
    "C67",
    "x_vars[S3,C18]"
   ],
   [
    "C68",
    "x_vars[S3,C19]"
   ],
   [
    "C69",
    "x_vars[S3,C20]"
   ],
   [
    "C70",
    "x_vars[S3,C21]"
   ],
   [
    "C71",
    "x_vars[S3,C22]"
   ],
   [
    "C72",
    "x_vars[S3,C23]"
   ],
   [
    "C73",
    "x_vars[S3,C24]"
   ],
   [
    "C74",
    "x_vars[S3,C25]"
   ],
   [
    "C75",
    "x_vars[S4,C1]"
   ],
   [
    "C76",
    "x_vars[S4,C2]"
   ],
   [
    "C77",
    "x_vars[S4,C3]"
   ],
   [
    "C78",
    "x_vars[S4,C4]"
   ],
   [
    "C79",
    "x_vars[S4,C5]"
   ],
   [
    "C80",
    "x_vars[S4,C6]"
   ],
   [
    "C81",
    "x_vars[S4,C7]"
   ],
   [
    "C82",
    "x_vars[S4,C8]"
   ],
   [
    "C83",
    "x_vars[S4,C9]"
   ],
   [
    "C84",
    "x_vars[S4,C10]"
   ],
   [
    "C85",
    "x_vars[S4,C11]"
   ],
   [
    "C86",
    "x_vars[S4,C12]"
   ],
   [
    "C87",
    "x_vars[S4,C13]"
   ],
   [
    "C88",
    "x_vars[S4,C14]"
   ],
   [
    "C89",
    "x_vars[S4,C15]"
   ],
   [
    "C90",
    "x_vars[S4,C16]"
   ],
   [
    "C91",
    "x_vars[S4,C17]"
   ],
   [
    "C92",
    "x_vars[S4,C18]"
   ],
   [
    "C93",
    "x_vars[S4,C19]"
   ],
   [
    "C94",
    "x_vars[S4,C20]"
   ],
   [
    "C95",
    "x_vars[S4,C21]"
   ],
   [
    "C96",
    "x_vars[S4,C22]"
   ],
   [
    "C97",
    "x_vars[S4,C23]"
   ],
   [
    "C98",
    "x_vars[S4,C24]"
   ],
   [
    "C99",
    "x_vars[S4,C25]"
   ],
   [
    "C100",
    "x_vars[S5,C1]"
   ],
   [
    "C101",
    "x_vars[S5,C2]"
   ],
   [
    "C102",
    "x_vars[S5,C3]"
   ],
   [
    "C103",
    "x_vars[S5,C4]"
   ],
   [
    "C104",
    "x_vars[S5,C5]"
   ],
   [
    "C105",
    "x_vars[S5,C6]"
   ],
   [
    "C106",
    "x_vars[S5,C7]"
   ],
   [
    "C107",
    "x_vars[S5,C8]"
   ],
   [
    "C108",
    "x_vars[S5,C9]"
   ],
   [
    "C109",
    "x_vars[S5,C10]"
   ],
   [
    "C110",
    "x_vars[S5,C11]"
   ],
   [
    "C111",
    "x_vars[S5,C12]"
   ],
   [
    "C112",
    "x_vars[S5,C13]"
   ],
   [
    "C113",
    "x_vars[S5,C14]"
   ],
   [
    "C114",
    "x_vars[S5,C15]"
   ],
   [
    "C115",
    "x_vars[S5,C16]"
   ],
   [
    "C116",
    "x_vars[S5,C17]"
   ],
   [
    "C117",
    "x_vars[S5,C18]"
   ],
   [
    "C118",
    "x_vars[S5,C19]"
   ],
   [
    "C119",
    "x_vars[S5,C20]"
   ],
   [
    "C120",
    "x_vars[S5,C21]"
   ],
   [
    "C121",
    "x_vars[S5,C22]"
   ],
   [
    "C122",
    "x_vars[S5,C23]"
   ],
   [
    "C123",
    "x_vars[S5,C24]"
   ],
   [
    "C124",
    "x_vars[S5,C25]"
   ],
   [
    "C125",
    "x_vars[S6,C1]"
   ],
   [
    "C126",
    "x_vars[S6,C2]"
   ],
   [
    "C127",
    "x_vars[S6,C3]"
   ],
   [
    "C128",
    "x_vars[S6,C4]"
   ],
   [
    "C129",
    "x_vars[S6,C5]"
   ],
   [
    "C130",
    "x_vars[S6,C6]"
   ],
   [
    "C131",
    "x_vars[S6,C7]"
   ],
   [
    "C132",
    "x_vars[S6,C8]"
   ],
   [
    "C133",
    "x_vars[S6,C9]"
   ],
   [
    "C134",
    "x_vars[S6,C10]"
   ],
   [
    "C135",
    "x_vars[S6,C11]"
   ],
   [
    "C136",
    "x_vars[S6,C12]"
   ],
   [
    "C137",
    "x_vars[S6,C13]"
   ],
   [
    "C138",
    "x_vars[S6,C14]"
   ],
   [
    "C139",
    "x_vars[S6,C15]"
   ],
   [
    "C140",
    "x_vars[S6,C16]"
   ],
   [
    "C141",
    "x_vars[S6,C17]"
   ],
   [
    "C142",
    "x_vars[S6,C18]"
   ],
   [
    "C143",
    "x_vars[S6,C19]"
   ],
   [
    "C144",
    "x_vars[S6,C20]"
   ],
   [
    "C145",
    "x_vars[S6,C21]"
   ],
   [
    "C146",
    "x_vars[S6,C22]"
   ],
   [
    "C147",
    "x_vars[S6,C23]"
   ],
   [
    "C148",
    "x_vars[S6,C24]"
   ],
   [
    "C149",
    "x_vars[S6,C25]"
   ],
   [
    "C150",
    "x_vars[S7,C1]"
   ],
   [
    "C151",
    "x_vars[S7,C2]"
   ],
   [
    "C152",
    "x_vars[S7,C3]"
   ],
   [
    "C153",
    "x_vars[S7,C4]"
   ],
   [
    "C154",
    "x_vars[S7,C5]"
   ],
   [
    "C155",
    "x_vars[S7,C6]"
   ],
   [
    "C156",
    "x_vars[S7,C7]"
   ],
   [
    "C157",
    "x_vars[S7,C8]"
   ],
   [
    "C158",
    "x_vars[S7,C9]"
   ],
   [
    "C159",
    "x_vars[S7,C10]"
   ],
   [
    "C160",
    "x_vars[S7,C11]"
   ],
   [
    "C161",
    "x_vars[S7,C12]"
   ],
   [
    "C162",
    "x_vars[S7,C13]"
   ],
   [
    "C163",
    "x_vars[S7,C14]"
   ],
   [
    "C164",
    "x_vars[S7,C15]"
   ],
   [
    "C165",
    "x_vars[S7,C16]"
   ],
   [
    "C166",
    "x_vars[S7,C17]"
   ],
   [
    "C167",
    "x_vars[S7,C18]"
   ],
   [
    "C168",
    "x_vars[S7,C19]"
   ],
   [
    "C169",
    "x_vars[S7,C20]"
   ],
   [
    "C170",
    "x_vars[S7,C21]"
   ],
   [
    "C171",
    "x_vars[S7,C22]"
   ],
   [
    "C172",
    "x_vars[S7,C23]"
   ],
   [
    "C173",
    "x_vars[S7,C24]"
   ],
   [
    "C174",
    "x_vars[S7,C25]"
   ],
   [
    "C175",
    "x_vars[S8,C1]"
   ],
   [
    "C176",
    "x_vars[S8,C2]"
   ],
   [
    "C177",
    "x_vars[S8,C3]"
   ],
   [
    "C178",
    "x_vars[S8,C4]"
   ],
   [
    "C179",
    "x_vars[S8,C5]"
   ],
   [
    "C180",
    "x_vars[S8,C6]"
   ],
   [
    "C181",
    "x_vars[S8,C7]"
   ],
   [
    "C182",
    "x_vars[S8,C8]"
   ],
   [
    "C183",
    "x_vars[S8,C9]"
   ],
   [
    "C184",
    "x_vars[S8,C10]"
   ],
   [
    "C185",
    "x_vars[S8,C11]"
   ],
   [
    "C186",
    "x_vars[S8,C12]"
   ],
   [
    "C187",
    "x_vars[S8,C13]"
   ],
   [
    "C188",
    "x_vars[S8,C14]"
   ],
   [
    "C189",
    "x_vars[S8,C15]"
   ],
   [
    "C190",
    "x_vars[S8,C16]"
   ],
   [
    "C191",
    "x_vars[S8,C17]"
   ],
   [
    "C192",
    "x_vars[S8,C18]"
   ],
   [
    "C193",
    "x_vars[S8,C19]"
   ],
   [
    "C194",
    "x_vars[S8,C20]"
   ],
   [
    "C195",
    "x_vars[S8,C21]"
   ],
   [
    "C196",
    "x_vars[S8,C22]"
   ],
   [
    "C197",
    "x_vars[S8,C23]"
   ],
   [
    "C198",
    "x_vars[S8,C24]"
   ],
   [
    "C199",
    "x_vars[S8,C25]"
   ],
   [
    "C200",
    "x_vars[S9,C1]"
   ],
   [
    "C201",
    "x_vars[S9,C2]"
   ],
   [
    "C202",
    "x_vars[S9,C3]"
   ],
   [
    "C203",
    "x_vars[S9,C4]"
   ],
   [
    "C204",
    "x_vars[S9,C5]"
   ],
   [
    "C205",
    "x_vars[S9,C6]"
   ],
   [
    "C206",
    "x_vars[S9,C7]"
   ],
   [
    "C207",
    "x_vars[S9,C8]"
   ],
   [
    "C208",
    "x_vars[S9,C9]"
   ],
   [
    "C209",
    "x_vars[S9,C10]"
   ],
   [
    "C210",
    "x_vars[S9,C11]"
   ],
   [
    "C211",
    "x_vars[S9,C12]"
   ],
   [
    "C212",
    "x_vars[S9,C13]"
   ],
   [
    "C213",
    "x_vars[S9,C14]"
   ],
   [
    "C214",
    "x_vars[S9,C15]"
   ],
   [
    "C215",
    "x_vars[S9,C16]"
   ],
   [
    "C216",
    "x_vars[S9,C17]"
   ],
   [
    "C217",
    "x_vars[S9,C18]"
   ],
   [
    "C218",
    "x_vars[S9,C19]"
   ],
   [
    "C219",
    "x_vars[S9,C20]"
   ],
   [
    "C220",
    "x_vars[S9,C21]"
   ],
   [
    "C221",
    "x_vars[S9,C22]"
   ],
   [
    "C222",
    "x_vars[S9,C23]"
   ],
   [
    "C223",
    "x_vars[S9,C24]"
   ],
   [
    "C224",
    "x_vars[S9,C25]"
   ],
   [
    "C225",
    "x_vars[S10,C1]"
   ],
   [
    "C226",
    "x_vars[S10,C2]"
   ],
   [
    "C227",
    "x_vars[S10,C3]"
   ],
   [
    "C228",
    "x_vars[S10,C4]"
   ],
   [
    "C229",
    "x_vars[S10,C5]"
   ],
   [
    "C230",
    "x_vars[S10,C6]"
   ],
   [
    "C231",
    "x_vars[S10,C7]"
   ],
   [
    "C232",
    "x_vars[S10,C8]"
   ],
   [
    "C233",
    "x_vars[S10,C9]"
   ],
   [
    "C234",
    "x_vars[S10,C10]"
   ],
   [
    "C235",
    "x_vars[S10,C11]"
   ],
   [
    "C236",
    "x_vars[S10,C12]"
   ],
   [
    "C237",
    "x_vars[S10,C13]"
   ],
   [
    "C238",
    "x_vars[S10,C14]"
   ],
   [
    "C239",
    "x_vars[S10,C15]"
   ],
   [
    "C240",
    "x_vars[S10,C16]"
   ],
   [
    "C241",
    "x_vars[S10,C17]"
   ],
   [
    "C242",
    "x_vars[S10,C18]"
   ],
   [
    "C243",
    "x_vars[S10,C19]"
   ],
   [
    "C244",
    "x_vars[S10,C20]"
   ],
   [
    "C245",
    "x_vars[S10,C21]"
   ],
   [
    "C246",
    "x_vars[S10,C22]"
   ],
   [
    "C247",
    "x_vars[S10,C23]"
   ],
   [
    "C248",
    "x_vars[S10,C24]"
   ],
   [
    "C249",
    "x_vars[S10,C25]"
   ],
   [
    "C250",
    "x_vars[S11,C1]"
   ],
   [
    "C251",
    "x_vars[S11,C2]"
   ],
   [
    "C252",
    "x_vars[S11,C3]"
   ],
   [
    "C253",
    "x_vars[S11,C4]"
   ],
   [
    "C254",
    "x_vars[S11,C5]"
   ],
   [
    "C255",
    "x_vars[S11,C6]"
   ],
   [
    "C256",
    "x_vars[S11,C7]"
   ],
   [
    "C257",
    "x_vars[S11,C8]"
   ],
   [
    "C258",
    "x_vars[S11,C9]"
   ],
   [
    "C259",
    "x_vars[S11,C10]"
   ],
   [
    "C260",
    "x_vars[S11,C11]"
   ],
   [
    "C261",
    "x_vars[S11,C12]"
   ],
   [
    "C262",
    "x_vars[S11,C13]"
   ],
   [
    "C263",
    "x_vars[S11,C14]"
   ],
   [
    "C264",
    "x_vars[S11,C15]"
   ],
   [
    "C265",
    "x_vars[S11,C16]"
   ],
   [
    "C266",
    "x_vars[S11,C17]"
   ],
   [
    "C267",
    "x_vars[S11,C18]"
   ],
   [
    "C268",
    "x_vars[S11,C19]"
   ],
   [
    "C269",
    "x_vars[S11,C20]"
   ],
   [
    "C270",
    "x_vars[S11,C21]"
   ],
   [
    "C271",
    "x_vars[S11,C22]"
   ],
   [
    "C272",
    "x_vars[S11,C23]"
   ],
   [
    "C273",
    "x_vars[S11,C24]"
   ],
   [
    "C274",
    "x_vars[S11,C25]"
   ],
   [
    "C275",
    "x_vars[S12,C1]"
   ],
   [
    "C276",
    "x_vars[S12,C2]"
   ],
   [
    "C277",
    "x_vars[S12,C3]"
   ],
   [
    "C278",
    "x_vars[S12,C4]"
   ],
   [
    "C279",
    "x_vars[S12,C5]"
   ],
   [
    "C280",
    "x_vars[S12,C6]"
   ],
   [
    "C281",
    "x_vars[S12,C7]"
   ],
   [
    "C282",
    "x_vars[S12,C8]"
   ],
   [
    "C283",
    "x_vars[S12,C9]"
   ],
   [
    "C284",
    "x_vars[S12,C10]"
   ],
   [
    "C285",
    "x_vars[S12,C11]"
   ],
   [
    "C286",
    "x_vars[S12,C12]"
   ],
   [
    "C287",
    "x_vars[S12,C13]"
   ],
   [
    "C288",
    "x_vars[S12,C14]"
   ],
   [
    "C289",
    "x_vars[S12,C15]"
   ],
   [
    "C290",
    "x_vars[S12,C16]"
   ],
   [
    "C291",
    "x_vars[S12,C17]"
   ],
   [
    "C292",
    "x_vars[S12,C18]"
   ],
   [
    "C293",
    "x_vars[S12,C19]"
   ],
   [
    "C294",
    "x_vars[S12,C20]"
   ],
   [
    "C295",
    "x_vars[S12,C21]"
   ],
   [
    "C296",
    "x_vars[S12,C22]"
   ],
   [
    "C297",
    "x_vars[S12,C23]"
   ],
   [
    "C298",
    "x_vars[S12,C24]"
   ],
   [
    "C299",
    "x_vars[S12,C25]"
   ],
   [
    "C300",
    "x_vars[S13,C1]"
   ],
   [
    "C301",
    "x_vars[S13,C2]"
   ],
   [
    "C302",
    "x_vars[S13,C3]"
   ],
   [
    "C303",
    "x_vars[S13,C4]"
   ],
   [
    "C304",
    "x_vars[S13,C5]"
   ],
   [
    "C305",
    "x_vars[S13,C6]"
   ],
   [
    "C306",
    "x_vars[S13,C7]"
   ],
   [
    "C307",
    "x_vars[S13,C8]"
   ],
   [
    "C308",
    "x_vars[S13,C9]"
   ],
   [
    "C309",
    "x_vars[S13,C10]"
   ],
   [
    "C310",
    "x_vars[S13,C11]"
   ],
   [
    "C311",
    "x_vars[S13,C12]"
   ],
   [
    "C312",
    "x_vars[S13,C13]"
   ],
   [
    "C313",
    "x_vars[S13,C14]"
   ],
   [
    "C314",
    "x_vars[S13,C15]"
   ],
   [
    "C315",
    "x_vars[S13,C16]"
   ],
   [
    "C316",
    "x_vars[S13,C17]"
   ],
   [
    "C317",
    "x_vars[S13,C18]"
   ],
   [
    "C318",
    "x_vars[S13,C19]"
   ],
   [
    "C319",
    "x_vars[S13,C20]"
   ],
   [
    "C320",
    "x_vars[S13,C21]"
   ],
   [
    "C321",
    "x_vars[S13,C22]"
   ],
   [
    "C322",
    "x_vars[S13,C23]"
   ],
   [
    "C323",
    "x_vars[S13,C24]"
   ],
   [
    "C324",
    "x_vars[S13,C25]"
   ],
   [
    "C325",
    "x_vars[S14,C1]"
   ],
   [
    "C326",
    "x_vars[S14,C2]"
   ],
   [
    "C327",
    "x_vars[S14,C3]"
   ],
   [
    "C328",
    "x_vars[S14,C4]"
   ],
   [
    "C329",
    "x_vars[S14,C5]"
   ],
   [
    "C330",
    "x_vars[S14,C6]"
   ],
   [
    "C331",
    "x_vars[S14,C7]"
   ],
   [
    "C332",
    "x_vars[S14,C8]"
   ],
   [
    "C333",
    "x_vars[S14,C9]"
   ],
   [
    "C334",
    "x_vars[S14,C10]"
   ],
   [
    "C335",
    "x_vars[S14,C11]"
   ],
   [
    "C336",
    "x_vars[S14,C12]"
   ],
   [
    "C337",
    "x_vars[S14,C13]"
   ],
   [
    "C338",
    "x_vars[S14,C14]"
   ],
   [
    "C339",
    "x_vars[S14,C15]"
   ],
   [
    "C340",
    "x_vars[S14,C16]"
   ],
   [
    "C341",
    "x_vars[S14,C17]"
   ],
   [
    "C342",
    "x_vars[S14,C18]"
   ],
   [
    "C343",
    "x_vars[S14,C19]"
   ],
   [
    "C344",
    "x_vars[S14,C20]"
   ],
   [
    "C345",
    "x_vars[S14,C21]"
   ],
   [
    "C346",
    "x_vars[S14,C22]"
   ],
   [
    "C347",
    "x_vars[S14,C23]"
   ],
   [
    "C348",
    "x_vars[S14,C24]"
   ],
   [
    "C349",
    "x_vars[S14,C25]"
   ],
   [
    "C350",
    "x_vars[S15,C1]"
   ],
   [
    "C351",
    "x_vars[S15,C2]"
   ],
   [
    "C352",
    "x_vars[S15,C3]"
   ],
   [
    "C353",
    "x_vars[S15,C4]"
   ],
   [
    "C354",
    "x_vars[S15,C5]"
   ],
   [
    "C355",
    "x_vars[S15,C6]"
   ],
   [
    "C356",
    "x_vars[S15,C7]"
   ],
   [
    "C357",
    "x_vars[S15,C8]"
   ],
   [
    "C358",
    "x_vars[S15,C9]"
   ],
   [
    "C359",
    "x_vars[S15,C10]"
   ],
   [
    "C360",
    "x_vars[S15,C11]"
   ],
   [
    "C361",
    "x_vars[S15,C12]"
   ],
   [
    "C362",
    "x_vars[S15,C13]"
   ],
   [
    "C363",
    "x_vars[S15,C14]"
   ],
   [
    "C364",
    "x_vars[S15,C15]"
   ],
   [
    "C365",
    "x_vars[S15,C16]"
   ],
   [
    "C366",
    "x_vars[S15,C17]"
   ],
   [
    "C367",
    "x_vars[S15,C18]"
   ],
   [
    "C368",
    "x_vars[S15,C19]"
   ],
   [
    "C369",
    "x_vars[S15,C20]"
   ],
   [
    "C370",
    "x_vars[S15,C21]"
   ],
   [
    "C371",
    "x_vars[S15,C22]"
   ],
   [
    "C372",
    "x_vars[S15,C23]"
   ],
   [
    "C373",
    "x_vars[S15,C24]"
   ],
   [
    "C374",
    "x_vars[S15,C25]"
   ],
   [
    "C375",
    "x_vars[S16,C1]"
   ],
   [
    "C376",
    "x_vars[S16,C2]"
   ],
   [
    "C377",
    "x_vars[S16,C3]"
   ],
   [
    "C378",
    "x_vars[S16,C4]"
   ],
   [
    "C379",
    "x_vars[S16,C5]"
   ],
   [
    "C380",
    "x_vars[S16,C6]"
   ],
   [
    "C381",
    "x_vars[S16,C7]"
   ],
   [
    "C382",
    "x_vars[S16,C8]"
   ],
   [
    "C383",
    "x_vars[S16,C9]"
   ],
   [
    "C384",
    "x_vars[S16,C10]"
   ],
   [
    "C385",
    "x_vars[S16,C11]"
   ],
   [
    "C386",
    "x_vars[S16,C12]"
   ],
   [
    "C387",
    "x_vars[S16,C13]"
   ],
   [
    "C388",
    "x_vars[S16,C14]"
   ],
   [
    "C389",
    "x_vars[S16,C15]"
   ],
   [
    "C390",
    "x_vars[S16,C16]"
   ],
   [
    "C391",
    "x_vars[S16,C17]"
   ],
   [
    "C392",
    "x_vars[S16,C18]"
   ],
   [
    "C393",
    "x_vars[S16,C19]"
   ],
   [
    "C394",
    "x_vars[S16,C20]"
   ],
   [
    "C395",
    "x_vars[S16,C21]"
   ],
   [
    "C396",
    "x_vars[S16,C22]"
   ],
   [
    "C397",
    "x_vars[S16,C23]"
   ],
   [
    "C398",
    "x_vars[S16,C24]"
   ],
   [
    "C399",
    "x_vars[S16,C25]"
   ],
   [
    "C400",
    "x_vars[S17,C1]"
   ],
   [
    "C401",
    "x_vars[S17,C2]"
   ],
   [
    "C402",
    "x_vars[S17,C3]"
   ],
   [
    "C403",
    "x_vars[S17,C4]"
   ],
   [
    "C404",
    "x_vars[S17,C5]"
   ],
   [
    "C405",
    "x_vars[S17,C6]"
   ],
   [
    "C406",
    "x_vars[S17,C7]"
   ],
   [
    "C407",
    "x_vars[S17,C8]"
   ],
   [
    "C408",
    "x_vars[S17,C9]"
   ],
   [
    "C409",
    "x_vars[S17,C10]"
   ],
   [
    "C410",
    "x_vars[S17,C11]"
   ],
   [
    "C411",
    "x_vars[S17,C12]"
   ],
   [
    "C412",
    "x_vars[S17,C13]"
   ],
   [
    "C413",
    "x_vars[S17,C14]"
   ],
   [
    "C414",
    "x_vars[S17,C15]"
   ],
   [
    "C415",
    "x_vars[S17,C16]"
   ],
   [
    "C416",
    "x_vars[S17,C17]"
   ],
   [
    "C417",
    "x_vars[S17,C18]"
   ],
   [
    "C418",
    "x_vars[S17,C19]"
   ],
   [
    "C419",
    "x_vars[S17,C20]"
   ],
   [
    "C420",
    "x_vars[S17,C21]"
   ],
   [
    "C421",
    "x_vars[S17,C22]"
   ],
   [
    "C422",
    "x_vars[S17,C23]"
   ],
   [
    "C423",
    "x_vars[S17,C24]"
   ],
   [
    "C424",
    "x_vars[S17,C25]"
   ],
   [
    "C425",
    "x_vars[S18,C1]"
   ],
   [
    "C426",
    "x_vars[S18,C2]"
   ],
   [
    "C427",
    "x_vars[S18,C3]"
   ],
   [
    "C428",
    "x_vars[S18,C4]"
   ],
   [
    "C429",
    "x_vars[S18,C5]"
   ],
   [
    "C430",
    "x_vars[S18,C6]"
   ],
   [
    "C431",
    "x_vars[S18,C7]"
   ],
   [
    "C432",
    "x_vars[S18,C8]"
   ],
   [
    "C433",
    "x_vars[S18,C9]"
   ],
   [
    "C434",
    "x_vars[S18,C10]"
   ],
   [
    "C435",
    "x_vars[S18,C11]"
   ],
   [
    "C436",
    "x_vars[S18,C12]"
   ],
   [
    "C437",
    "x_vars[S18,C13]"
   ],
   [
    "C438",
    "x_vars[S18,C14]"
   ],
   [
    "C439",
    "x_vars[S18,C15]"
   ],
   [
    "C440",
    "x_vars[S18,C16]"
   ],
   [
    "C441",
    "x_vars[S18,C17]"
   ],
   [
    "C442",
    "x_vars[S18,C18]"
   ],
   [
    "C443",
    "x_vars[S18,C19]"
   ],
   [
    "C444",
    "x_vars[S18,C20]"
   ],
   [
    "C445",
    "x_vars[S18,C21]"
   ],
   [
    "C446",
    "x_vars[S18,C22]"
   ],
   [
    "C447",
    "x_vars[S18,C23]"
   ],
   [
    "C448",
    "x_vars[S18,C24]"
   ],
   [
    "C449",
    "x_vars[S18,C25]"
   ],
   [
    "C450",
    "x_vars[S19,C1]"
   ],
   [
    "C451",
    "x_vars[S19,C2]"
   ],
   [
    "C452",
    "x_vars[S19,C3]"
   ],
   [
    "C453",
    "x_vars[S19,C4]"
   ],
   [
    "C454",
    "x_vars[S19,C5]"
   ],
   [
    "C455",
    "x_vars[S19,C6]"
   ],
   [
    "C456",
    "x_vars[S19,C7]"
   ],
   [
    "C457",
    "x_vars[S19,C8]"
   ],
   [
    "C458",
    "x_vars[S19,C9]"
   ],
   [
    "C459",
    "x_vars[S19,C10]"
   ],
   [
    "C460",
    "x_vars[S19,C11]"
   ],
   [
    "C461",
    "x_vars[S19,C12]"
   ],
   [
    "C462",
    "x_vars[S19,C13]"
   ],
   [
    "C463",
    "x_vars[S19,C14]"
   ],
   [
    "C464",
    "x_vars[S19,C15]"
   ],
   [
    "C465",
    "x_vars[S19,C16]"
   ],
   [
    "C466",
    "x_vars[S19,C17]"
   ],
   [
    "C467",
    "x_vars[S19,C18]"
   ],
   [
    "C468",
    "x_vars[S19,C19]"
   ],
   [
    "C469",
    "x_vars[S19,C20]"
   ],
   [
    "C470",
    "x_vars[S19,C21]"
   ],
   [
    "C471",
    "x_vars[S19,C22]"
   ],
   [
    "C472",
    "x_vars[S19,C23]"
   ],
   [
    "C473",
    "x_vars[S19,C24]"
   ],
   [
    "C474",
    "x_vars[S19,C25]"
   ],
   [
    "C475",
    "x_vars[S20,C1]"
   ],
   [
    "C476",
    "x_vars[S20,C2]"
   ],
   [
    "C477",
    "x_vars[S20,C3]"
   ],
   [
    "C478",
    "x_vars[S20,C4]"
   ],
   [
    "C479",
    "x_vars[S20,C5]"
   ],
   [
    "C480",
    "x_vars[S20,C6]"
   ],
   [
    "C481",
    "x_vars[S20,C7]"
   ],
   [
    "C482",
    "x_vars[S20,C8]"
   ],
   [
    "C483",
    "x_vars[S20,C9]"
   ],
   [
    "C484",
    "x_vars[S20,C10]"
   ],
   [
    "C485",
    "x_vars[S20,C11]"
   ],
   [
    "C486",
    "x_vars[S20,C12]"
   ],
   [
    "C487",
    "x_vars[S20,C13]"
   ],
   [
    "C488",
    "x_vars[S20,C14]"
   ],
   [
    "C489",
    "x_vars[S20,C15]"
   ],
   [
    "C490",
    "x_vars[S20,C16]"
   ],
   [
    "C491",
    "x_vars[S20,C17]"
   ],
   [
    "C492",
    "x_vars[S20,C18]"
   ],
   [
    "C493",
    "x_vars[S20,C19]"
   ],
   [
    "C494",
    "x_vars[S20,C20]"
   ],
   [
    "C495",
    "x_vars[S20,C21]"
   ],
   [
    "C496",
    "x_vars[S20,C22]"
   ],
   [
    "C497",
    "x_vars[S20,C23]"
   ],
   [
    "C498",
    "x_vars[S20,C24]"
   ],
   [
    "C499",
    "x_vars[S20,C25]"
   ],
   [
    "C500",
    "x_vars[S21,C1]"
   ],
   [
    "C501",
    "x_vars[S21,C2]"
   ],
   [
    "C502",
    "x_vars[S21,C3]"
   ],
   [
    "C503",
    "x_vars[S21,C4]"
   ],
   [
    "C504",
    "x_vars[S21,C5]"
   ],
   [
    "C505",
    "x_vars[S21,C6]"
   ],
   [
    "C506",
    "x_vars[S21,C7]"
   ],
   [
    "C507",
    "x_vars[S21,C8]"
   ],
   [
    "C508",
    "x_vars[S21,C9]"
   ],
   [
    "C509",
    "x_vars[S21,C10]"
   ],
   [
    "C510",
    "x_vars[S21,C11]"
   ],
   [
    "C511",
    "x_vars[S21,C12]"
   ],
   [
    "C512",
    "x_vars[S21,C13]"
   ],
   [
    "C513",
    "x_vars[S21,C14]"
   ],
   [
    "C514",
    "x_vars[S21,C15]"
   ],
   [
    "C515",
    "x_vars[S21,C16]"
   ],
   [
    "C516",
    "x_vars[S21,C17]"
   ],
   [
    "C517",
    "x_vars[S21,C18]"
   ],
   [
    "C518",
    "x_vars[S21,C19]"
   ],
   [
    "C519",
    "x_vars[S21,C20]"
   ],
   [
    "C520",
    "x_vars[S21,C21]"
   ],
   [
    "C521",
    "x_vars[S21,C22]"
   ],
   [
    "C522",
    "x_vars[S21,C23]"
   ],
   [
    "C523",
    "x_vars[S21,C24]"
   ],
   [
    "C524",
    "x_vars[S21,C25]"
   ],
   [
    "C525",
    "x_vars[S22,C1]"
   ],
   [
    "C526",
    "x_vars[S22,C2]"
   ],
   [
    "C527",
    "x_vars[S22,C3]"
   ],
   [
    "C528",
    "x_vars[S22,C4]"
   ],
   [
    "C529",
    "x_vars[S22,C5]"
   ],
   [
    "C530",
    "x_vars[S22,C6]"
   ],
   [
    "C531",
    "x_vars[S22,C7]"
   ],
   [
    "C532",
    "x_vars[S22,C8]"
   ],
   [
    "C533",
    "x_vars[S22,C9]"
   ],
   [
    "C534",
    "x_vars[S22,C10]"
   ],
   [
    "C535",
    "x_vars[S22,C11]"
   ],
   [
    "C536",
    "x_vars[S22,C12]"
   ],
   [
    "C537",
    "x_vars[S22,C13]"
   ],
   [
    "C538",
    "x_vars[S22,C14]"
   ],
   [
    "C539",
    "x_vars[S22,C15]"
   ],
   [
    "C540",
    "x_vars[S22,C16]"
   ],
   [
    "C541",
    "x_vars[S22,C17]"
   ],
   [
    "C542",
    "x_vars[S22,C18]"
   ],
   [
    "C543",
    "x_vars[S22,C19]"
   ],
   [
    "C544",
    "x_vars[S22,C20]"
   ],
   [
    "C545",
    "x_vars[S22,C21]"
   ],
   [
    "C546",
    "x_vars[S22,C22]"
   ],
   [
    "C547",
    "x_vars[S22,C23]"
   ],
   [
    "C548",
    "x_vars[S22,C24]"
   ],
   [
    "C549",
    "x_vars[S22,C25]"
   ],
   [
    "C550",
    "x_vars[S23,C1]"
   ],
   [
    "C551",
    "x_vars[S23,C2]"
   ],
   [
    "C552",
    "x_vars[S23,C3]"
   ],
   [
    "C553",
    "x_vars[S23,C4]"
   ],
   [
    "C554",
    "x_vars[S23,C5]"
   ],
   [
    "C555",
    "x_vars[S23,C6]"
   ],
   [
    "C556",
    "x_vars[S23,C7]"
   ],
   [
    "C557",
    "x_vars[S23,C8]"
   ],
   [
    "C558",
    "x_vars[S23,C9]"
   ],
   [
    "C559",
    "x_vars[S23,C10]"
   ],
   [
    "C560",
    "x_vars[S23,C11]"
   ],
   [
    "C561",
    "x_vars[S23,C12]"
   ],
   [
    "C562",
    "x_vars[S23,C13]"
   ],
   [
    "C563",
    "x_vars[S23,C14]"
   ],
   [
    "C564",
    "x_vars[S23,C15]"
   ],
   [
    "C565",
    "x_vars[S23,C16]"
   ],
   [
    "C566",
    "x_vars[S23,C17]"
   ],
   [
    "C567",
    "x_vars[S23,C18]"
   ],
   [
    "C568",
    "x_vars[S23,C19]"
   ],
   [
    "C569",
    "x_vars[S23,C20]"
   ],
   [
    "C570",
    "x_vars[S23,C21]"
   ],
   [
    "C571",
    "x_vars[S23,C22]"
   ],
   [
    "C572",
    "x_vars[S23,C23]"
   ],
   [
    "C573",
    "x_vars[S23,C24]"
   ],
   [
    "C574",
    "x_vars[S23,C25]"
   ],
   [
    "C575",
    "x_vars[S24,C1]"
   ],
   [
    "C576",
    "x_vars[S24,C2]"
   ],
   [
    "C577",
    "x_vars[S24,C3]"
   ],
   [
    "C578",
    "x_vars[S24,C4]"
   ],
   [
    "C579",
    "x_vars[S24,C5]"
   ],
   [
    "C580",
    "x_vars[S24,C6]"
   ],
   [
    "C581",
    "x_vars[S24,C7]"
   ],
   [
    "C582",
    "x_vars[S24,C8]"
   ],
   [
    "C583",
    "x_vars[S24,C9]"
   ],
   [
    "C584",
    "x_vars[S24,C10]"
   ],
   [
    "C585",
    "x_vars[S24,C11]"
   ],
   [
    "C586",
    "x_vars[S24,C12]"
   ],
   [
    "C587",
    "x_vars[S24,C13]"
   ],
   [
    "C588",
    "x_vars[S24,C14]"
   ],
   [
    "C589",
    "x_vars[S24,C15]"
   ],
   [
    "C590",
    "x_vars[S24,C16]"
   ],
   [
    "C591",
    "x_vars[S24,C17]"
   ],
   [
    "C592",
    "x_vars[S24,C18]"
   ],
   [
    "C593",
    "x_vars[S24,C19]"
   ],
   [
    "C594",
    "x_vars[S24,C20]"
   ],
   [
    "C595",
    "x_vars[S24,C21]"
   ],
   [
    "C596",
    "x_vars[S24,C22]"
   ],
   [
    "C597",
    "x_vars[S24,C23]"
   ],
   [
    "C598",
    "x_vars[S24,C24]"
   ],
   [
    "C599",
    "x_vars[S24,C25]"
   ],
   [
    "C600",
    "y_vars[S1]"
   ],
   [
    "C601",
    "y_vars[S2]"
   ],
   [
    "C602",
    "y_vars[S3]"
   ],
   [
    "C603",
    "y_vars[S4]"
   ],
   [
    "C604",
    "y_vars[S5]"
   ],
   [
    "C605",
    "y_vars[S6]"
   ],
   [
    "C606",
    "y_vars[S7]"
   ],
   [
    "C607",
    "y_vars[S8]"
   ],
   [
    "C608",
    "y_vars[S9]"
   ],
   [
    "C609",
    "y_vars[S10]"
   ],
   [
    "C610",
    "y_vars[S11]"
   ],
   [
    "C611",
    "y_vars[S12]"
   ],
   [
    "C612",
    "y_vars[S13]"
   ],
   [
    "C613",
    "y_vars[S14]"
   ],
   [
    "C614",
    "y_vars[S15]"
   ],
   [
    "C615",
    "y_vars[S16]"
   ],
   [
    "C616",
    "y_vars[S17]"
   ],
   [
    "C617",
    "y_vars[S18]"
   ],
   [
    "C618",
    "y_vars[S19]"
   ],
   [
    "C619",
    "y_vars[S20]"
   ],
   [
    "C620",
    "y_vars[S21]"
   ],
   [
    "C621",
    "y_vars[S22]"
   ],
   [
    "C622",
    "y_vars[S23]"
   ],
   [
    "C623",
    "y_vars[S24]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_0_3": "x_vars[S1,C4]",
   "x_0_4": "x_vars[S1,C5]",
   "x_0_5": "x_vars[S1,C6]",
   "x_0_6": "x_vars[S1,C7]",
   "x_0_7": "x_vars[S1,C8]",
   "x_0_8": "x_vars[S1,C9]",
   "x_0_9": "x_vars[S1,C10]",
   "x_0_10": "x_vars[S1,C11]",
   "x_0_11": "x_vars[S1,C12]",
   "x_0_12": "x_vars[S1,C13]",
   "x_0_13": "x_vars[S1,C14]",
   "x_0_14": "x_vars[S1,C15]",
   "x_0_15": "x_vars[S1,C16]",
   "x_0_16": "x_vars[S1,C17]",
   "x_0_17": "x_vars[S1,C18]",
   "x_0_18": "x_vars[S1,C19]",
   "x_0_19": "x_vars[S1,C20]",
   "x_0_20": "x_vars[S1,C21]",
   "x_0_21": "x_vars[S1,C22]",
   "x_0_22": "x_vars[S1,C23]",
   "x_0_23": "x_vars[S1,C24]",
   "x_0_24": "x_vars[S1,C25]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_1_3": "x_vars[S2,C4]",
   "x_1_4": "x_vars[S2,C5]",
   "x_1_5": "x_vars[S2,C6]",
   "x_1_6": "x_vars[S2,C7]",
   "x_1_7": "x_vars[S2,C8]",
   "x_1_8": "x_vars[S2,C9]",
   "x_1_9": "x_vars[S2,C10]",
   "x_1_10": "x_vars[S2,C11]",
   "x_1_11": "x_vars[S2,C12]",
   "x_1_12": "x_vars[S2,C13]",
   "x_1_13": "x_vars[S2,C14]",
   "x_1_14": "x_vars[S2,C15]",
   "x_1_15": "x_vars[S2,C16]",
   "x_1_16": "x_vars[S2,C17]",
   "x_1_17": "x_vars[S2,C18]",
   "x_1_18": "x_vars[S2,C19]",
   "x_1_19": "x_vars[S2,C20]",
   "x_1_21": "x_vars[S2,C22]",
   "x_1_22": "x_vars[S2,C23]",
   "x_1_23": "x_vars[S2,C24]",
   "x_1_24": "x_vars[S2,C25]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "x_2_3": "x_vars[S3,C4]",
   "x_2_4": "x_vars[S3,C5]",
   "x_2_5": "x_vars[S3,C6]",
   "x_2_6": "x_vars[S3,C7]",
   "x_2_7": "x_vars[S3,C8]",
   "x_2_8": "x_vars[S3,C9]",
   "x_2_9": "x_vars[S3,C10]",
   "x_2_10": "x_vars[S3,C11]",
   "x_2_11": "x_vars[S3,C12]",
   "x_2_12": "x_vars[S3,C13]",
   "x_2_13": "x_vars[S3,C14]",
   "x_2_14": "x_vars[S3,C15]",
   "x_2_15": "x_vars[S3,C16]",
   "x_2_16": "x_vars[S3,C17]",
   "x_2_17": "x_vars[S3,C18]",
   "x_2_18": "x_vars[S3,C19]",
   "x_2_19": "x_vars[S3,C20]",
   "x_2_21": "x_vars[S3,C22]",
   "x_2_22": "x_vars[S3,C23]",
   "x_2_23": "x_vars[S3,C24]",
   "x_2_24": "x_vars[S3,C25]",
   "x_3_0": "x_vars[S4,C1]",
   "x_3_1": "x_vars[S4,C2]",
   "x_3_2": "x_vars[S4,C3]",
   "x_3_3": "x_vars[S4,C4]",
   "x_3_4": "x_vars[S4,C5]",
   "x_3_5": "x_vars[S4,C6]",
   "x_3_6": "x_vars[S4,C7]",
   "x_3_7": "x_vars[S4,C8]",
   "x_3_8": "x_vars[S4,C9]",
   "x_3_9": "x_vars[S4,C10]",
   "x_3_10": "x_vars[S4,C11]",
   "x_3_11": "x_vars[S4,C12]",
   "x_3_12": "x_vars[S4,C13]",
   "x_3_13": "x_vars[S4,C14]",
   "x_3_14": "x_vars[S4,C15]",
   "x_3_15": "x_vars[S4,C16]",
   "x_3_16": "x_vars[S4,C17]",
   "x_3_17": "x_vars[S4,C18]",
   "x_3_18": "x_vars[S4,C19]",
   "x_3_19": "x_vars[S4,C20]",
   "x_3_20": "x_vars[S4,C21]",
   "x_3_21": "x_vars[S4,C22]",
   "x_3_22": "x_vars[S4,C23]",
   "x_3_23": "x_vars[S4,C24]",
   "x_3_24": "x_vars[S4,C25]",
   "x_4_0": "x_vars[S5,C1]",
   "x_4_1": "x_vars[S5,C2]",
   "x_4_2": "x_vars[S5,C3]",
   "x_4_3": "x_vars[S5,C4]",
   "x_4_4": "x_vars[S5,C5]",
   "x_4_5": "x_vars[S5,C6]",
   "x_4_6": "x_vars[S5,C7]",
   "x_4_7": "x_vars[S5,C8]",
   "x_4_8": "x_vars[S5,C9]",
   "x_4_9": "x_vars[S5,C10]",
   "x_4_10": "x_vars[S5,C11]",
   "x_4_11": "x_vars[S5,C12]",
   "x_4_12": "x_vars[S5,C13]",
   "x_4_13": "x_vars[S5,C14]",
   "x_4_14": "x_vars[S5,C15]",
   "x_4_15": "x_vars[S5,C16]",
   "x_4_16": "x_vars[S5,C17]",
   "x_4_17": "x_vars[S5,C18]",
   "x_4_18": "x_vars[S5,C19]",
   "x_4_19": "x_vars[S5,C20]",
   "x_4_20": "x_vars[S5,C21]",
   "x_4_21": "x_vars[S5,C22]",
   "x_4_22": "x_vars[S5,C23]",
   "x_4_23": "x_vars[S5,C24]",
   "x_4_24": "x_vars[S5,C25]",
   "x_5_0": "x_vars[S6,C1]",
   "x_5_1": "x_vars[S6,C2]",
   "x_5_2": "x_vars[S6,C3]",
   "x_5_3": "x_vars[S6,C4]",
   "x_5_4": "x_vars[S6,C5]",
   "x_5_5": "x_vars[S6,C6]",
   "x_5_6": "x_vars[S6,C7]",
   "x_5_7": "x_vars[S6,C8]",
   "x_5_8": "x_vars[S6,C9]",
   "x_5_9": "x_vars[S6,C10]",
   "x_5_10": "x_vars[S6,C11]",
   "x_5_11": "x_vars[S6,C12]",
   "x_5_12": "x_vars[S6,C13]",
   "x_5_13": "x_vars[S6,C14]",
   "x_5_14": "x_vars[S6,C15]",
   "x_5_15": "x_vars[S6,C16]",
   "x_5_16": "x_vars[S6,C17]",
   "x_5_17": "x_vars[S6,C18]",
   "x_5_18": "x_vars[S6,C19]",
   "x_5_19": "x_vars[S6,C20]",
   "x_5_20": "x_vars[S6,C21]",
   "x_5_21": "x_vars[S6,C22]",
   "x_5_22": "x_vars[S6,C23]",
   "x_5_23": "x_vars[S6,C24]",
   "x_5_24": "x_vars[S6,C25]",
   "x_6_0": "x_vars[S7,C1]",
   "x_6_1": "x_vars[S7,C2]",
   "x_6_2": "x_vars[S7,C3]",
   "x_6_3": "x_vars[S7,C4]",
   "x_6_4": "x_vars[S7,C5]",
   "x_6_5": "x_vars[S7,C6]",
   "x_6_6": "x_vars[S7,C7]",
   "x_6_7": "x_vars[S7,C8]",
   "x_6_9": "x_vars[S7,C10]",
   "x_6_10": "x_vars[S7,C11]",
   "x_6_11": "x_vars[S7,C12]",
   "x_6_12": "x_vars[S7,C13]",
   "x_6_13": "x_vars[S7,C14]",
   "x_6_14": "x_vars[S7,C15]",
   "x_6_15": "x_vars[S7,C16]",
   "x_6_16": "x_vars[S7,C17]",
   "x_6_17": "x_vars[S7,C18]",
   "x_6_18": "x_vars[S7,C19]",
   "x_6_19": "x_vars[S7,C20]",
   "x_6_20": "x_vars[S7,C21]",
   "x_6_21": "x_vars[S7,C22]",
   "x_6_22": "x_vars[S7,C23]",
   "x_6_23": "x_vars[S7,C24]",
   "x_6_24": "x_vars[S7,C25]",
   "x_7_0": "x_vars[S8,C1]",
   "x_7_1": "x_vars[S8,C2]",
   "x_7_2": "x_vars[S8,C3]",
   "x_7_3": "x_vars[S8,C4]",
   "x_7_4": "x_vars[S8,C5]",
   "x_7_5": "x_vars[S8,C6]",
   "x_7_6": "x_vars[S8,C7]",
   "x_7_7": "x_vars[S8,C8]",
   "x_7_8": "x_vars[S8,C9]",
   "x_7_9": "x_vars[S8,C10]",
   "x_7_10": "x_vars[S8,C11]",
   "x_7_11": "x_vars[S8,C12]",
   "x_7_12": "x_vars[S8,C13]",
   "x_7_13": "x_vars[S8,C14]",
   "x_7_14": "x_vars[S8,C15]",
   "x_7_15": "x_vars[S8,C16]",
   "x_7_16": "x_vars[S8,C17]",
   "x_7_17": "x_vars[S8,C18]",
   "x_7_18": "x_vars[S8,C19]",
   "x_7_19": "x_vars[S8,C20]",
   "x_7_21": "x_vars[S8,C22]",
   "x_7_22": "x_vars[S8,C23]",
   "x_7_23": "x_vars[S8,C24]",
   "x_7_24": "x_vars[S8,C25]",
   "x_8_0": "x_vars[S9,C1]",
   "x_8_1": "x_vars[S9,C2]",
   "x_8_2": "x_vars[S9,C3]",
   "x_8_3": "x_vars[S9,C4]",
   "x_8_4": "x_vars[S9,C5]",
   "x_8_5": "x_vars[S9,C6]",
   "x_8_6": "x_vars[S9,C7]",
   "x_8_7": "x_vars[S9,C8]",
   "x_8_8": "x_vars[S9,C9]",
   "x_8_9": "x_vars[S9,C10]",
   "x_8_10": "x_vars[S9,C11]",
   "x_8_11": "x_vars[S9,C12]",
   "x_8_12": "x_vars[S9,C13]",
   "x_8_13": "x_vars[S9,C14]",
   "x_8_14": "x_vars[S9,C15]",
   "x_8_15": "x_vars[S9,C16]",
   "x_8_16": "x_vars[S9,C17]",
   "x_8_17": "x_vars[S9,C18]",
   "x_8_18": "x_vars[S9,C19]",
   "x_8_19": "x_vars[S9,C20]",
   "x_8_20": "x_vars[S9,C21]",
   "x_8_21": "x_vars[S9,C22]",
   "x_8_22": "x_vars[S9,C23]",
   "x_8_23": "x_vars[S9,C24]",
   "x_8_24": "x_vars[S9,C25]",
   "x_9_0": "x_vars[S10,C1]",
   "x_9_1": "x_vars[S10,C2]",
   "x_9_2": "x_vars[S10,C3]",
   "x_9_3": "x_vars[S10,C4]",
   "x_9_4": "x_vars[S10,C5]",
   "x_9_5": "x_vars[S10,C6]",
   "x_9_6": "x_vars[S10,C7]",
   "x_9_7": "x_vars[S10,C8]",
   "x_9_8": "x_vars[S10,C9]",
   "x_9_9": "x_vars[S10,C10]",
   "x_9_10": "x_vars[S10,C11]",
   "x_9_11": "x_vars[S10,C12]",
   "x_9_12": "x_vars[S10,C13]",
   "x_9_13": "x_vars[S10,C14]",
   "x_9_14": "x_vars[S10,C15]",
   "x_9_15": "x_vars[S10,C16]",
   "x_9_16": "x_vars[S10,C17]",
   "x_9_17": "x_vars[S10,C18]",
   "x_9_18": "x_vars[S10,C19]",
   "x_9_19": "x_vars[S10,C20]",
   "x_9_20": "x_vars[S10,C21]",
   "x_9_21": "x_vars[S10,C22]",
   "x_9_22": "x_vars[S10,C23]",
   "x_9_23": "x_vars[S10,C24]",
   "x_9_24": "x_vars[S10,C25]",
   "x_10_0": "x_vars[S11,C1]",
   "x_10_2": "x_vars[S11,C3]",
   "x_10_3": "x_vars[S11,C4]",
   "x_10_4": "x_vars[S11,C5]",
   "x_10_5": "x_vars[S11,C6]",
   "x_10_6": "x_vars[S11,C7]",
   "x_10_7": "x_vars[S11,C8]",
   "x_10_8": "x_vars[S11,C9]",
   "x_10_9": "x_vars[S11,C10]",
   "x_10_10": "x_vars[S11,C11]",
   "x_10_11": "x_vars[S11,C12]",
   "x_10_12": "x_vars[S11,C13]",
   "x_10_13": "x_vars[S11,C14]",
   "x_10_14": "x_vars[S11,C15]",
   "x_10_15": "x_vars[S11,C16]",
   "x_10_16": "x_vars[S11,C17]",
   "x_10_17": "x_vars[S11,C18]",
   "x_10_18": "x_vars[S11,C19]",
   "x_10_19": "x_vars[S11,C20]",
   "x_10_20": "x_vars[S11,C21]",
   "x_10_21": "x_vars[S11,C22]",
   "x_10_22": "x_vars[S11,C23]",
   "x_10_23": "x_vars[S11,C24]",
   "x_10_24": "x_vars[S11,C25]",
   "x_11_0": "x_vars[S12,C1]",
   "x_11_1": "x_vars[S12,C2]",
   "x_11_2": "x_vars[S12,C3]",
   "x_11_3": "x_vars[S12,C4]",
   "x_11_4": "x_vars[S12,C5]",
   "x_11_5": "x_vars[S12,C6]",
   "x_11_6": "x_vars[S12,C7]",
   "x_11_7": "x_vars[S12,C8]",
   "x_11_8": "x_vars[S12,C9]",
   "x_11_9": "x_vars[S12,C10]",
   "x_11_10": "x_vars[S12,C11]",
   "x_11_11": "x_vars[S12,C12]",
   "x_11_12": "x_vars[S12,C13]",
   "x_11_13": "x_vars[S12,C14]",
   "x_11_14": "x_vars[S12,C15]",
   "x_11_15": "x_vars[S12,C16]",
   "x_11_16": "x_vars[S12,C17]",
   "x_11_17": "x_vars[S12,C18]",
   "x_11_18": "x_vars[S12,C19]",
   "x_11_19": "x_vars[S12,C20]",
   "x_11_20": "x_vars[S12,C21]",
   "x_11_21": "x_vars[S12,C22]",
   "x_11_22": "x_vars[S12,C23]",
   "x_11_23": "x_vars[S12,C24]",
   "x_11_24": "x_vars[S12,C25]",
   "x_12_0": "x_vars[S13,C1]",
   "x_12_1": "x_vars[S13,C2]",
   "x_12_2": "x_vars[S13,C3]",
   "x_12_3": "x_vars[S13,C4]",
   "x_12_4": "x_vars[S13,C5]",
   "x_12_5": "x_vars[S13,C6]",
   "x_12_6": "x_vars[S13,C7]",
   "x_12_7": "x_vars[S13,C8]",
   "x_12_8": "x_vars[S13,C9]",
   "x_12_9": "x_vars[S13,C10]",
   "x_12_10": "x_vars[S13,C11]",
   "x_12_11": "x_vars[S13,C12]",
   "x_12_12": "x_vars[S13,C13]",
   "x_12_13": "x_vars[S13,C14]",
   "x_12_14": "x_vars[S13,C15]",
   "x_12_15": "x_vars[S13,C16]",
   "x_12_16": "x_vars[S13,C17]",
   "x_12_17": "x_vars[S13,C18]",
   "x_12_18": "x_vars[S13,C19]",
   "x_12_19": "x_vars[S13,C20]",
   "x_12_20": "x_vars[S13,C21]",
   "x_12_21": "x_vars[S13,C22]",
   "x_12_22": "x_vars[S13,C23]",
   "x_12_23": "x_vars[S13,C24]",
   "x_12_24": "x_vars[S13,C25]",
   "x_13_0": "x_vars[S14,C1]",
   "x_13_1": "x_vars[S14,C2]",
   "x_13_2": "x_vars[S14,C3]",
   "x_13_3": "x_vars[S14,C4]",
   "x_13_4": "x_vars[S14,C5]",
   "x_13_5": "x_vars[S14,C6]",
   "x_13_6": "x_vars[S14,C7]",
   "x_13_7": "x_vars[S14,C8]",
   "x_13_8": "x_vars[S14,C9]",
   "x_13_9": "x_vars[S14,C10]",
   "x_13_10": "x_vars[S14,C11]",
   "x_13_11": "x_vars[S14,C12]",
   "x_13_12": "x_vars[S14,C13]",
   "x_13_13": "x_vars[S14,C14]",
   "x_13_14": "x_vars[S14,C15]",
   "x_13_15": "x_vars[S14,C16]",
   "x_13_16": "x_vars[S14,C17]",
   "x_13_17": "x_vars[S14,C18]",
   "x_13_18": "x_vars[S14,C19]",
   "x_13_19": "x_vars[S14,C20]",
   "x_13_20": "x_vars[S14,C21]",
   "x_13_21": "x_vars[S14,C22]",
   "x_13_22": "x_vars[S14,C23]",
   "x_13_23": "x_vars[S14,C24]",
   "x_13_24": "x_vars[S14,C25]",
   "x_14_0": "x_vars[S15,C1]",
   "x_14_1": "x_vars[S15,C2]",
   "x_14_2": "x_vars[S15,C3]",
   "x_14_3": "x_vars[S15,C4]",
   "x_14_4": "x_vars[S15,C5]",
   "x_14_5": "x_vars[S15,C6]",
   "x_14_6": "x_vars[S15,C7]",
   "x_14_7": "x_vars[S15,C8]",
   "x_14_8": "x_vars[S15,C9]",
   "x_14_9": "x_vars[S15,C10]",
   "x_14_10": "x_vars[S15,C11]",
   "x_14_11": "x_vars[S15,C12]",
   "x_14_12": "x_vars[S15,C13]",
   "x_14_13": "x_vars[S15,C14]",
   "x_14_14": "x_vars[S15,C15]",
   "x_14_15": "x_vars[S15,C16]",
   "x_14_16": "x_vars[S15,C17]",
   "x_14_17": "x_vars[S15,C18]",
   "x_14_18": "x_vars[S15,C19]",
   "x_14_19": "x_vars[S15,C20]",
   "x_14_20": "x_vars[S15,C21]",
   "x_14_21": "x_vars[S15,C22]",
   "x_14_22": "x_vars[S15,C23]",
   "x_14_23": "x_vars[S15,C24]",
   "x_14_24": "x_vars[S15,C25]",
   "x_15_0": "x_vars[S16,C1]",
   "x_15_1": "x_vars[S16,C2]",
   "x_15_2": "x_vars[S16,C3]",
   "x_15_3": "x_vars[S16,C4]",
   "x_15_4": "x_vars[S16,C5]",
   "x_15_5": "x_vars[S16,C6]",
   "x_15_6": "x_vars[S16,C7]",
   "x_15_7": "x_vars[S16,C8]",
   "x_15_8": "x_vars[S16,C9]",
   "x_15_9": "x_vars[S16,C10]",
   "x_15_10": "x_vars[S16,C11]",
   "x_15_11": "x_vars[S16,C12]",
   "x_15_12": "x_vars[S16,C13]",
   "x_15_13": "x_vars[S16,C14]",
   "x_15_14": "x_vars[S16,C15]",
   "x_15_15": "x_vars[S16,C16]",
   "x_15_16": "x_vars[S16,C17]",
   "x_15_17": "x_vars[S16,C18]",
   "x_15_18": "x_vars[S16,C19]",
   "x_15_19": "x_vars[S16,C20]",
   "x_15_20": "x_vars[S16,C21]",
   "x_15_21": "x_vars[S16,C22]",
   "x_15_22": "x_vars[S16,C23]",
   "x_15_23": "x_vars[S16,C24]",
   "x_15_24": "x_vars[S16,C25]",
   "x_16_0": "x_vars[S17,C1]",
   "x_16_1": "x_vars[S17,C2]",
   "x_16_2": "x_vars[S17,C3]",
   "x_16_3": "x_vars[S17,C4]",
   "x_16_4": "x_vars[S17,C5]",
   "x_16_5": "x_vars[S17,C6]",
   "x_16_6": "x_vars[S17,C7]",
   "x_16_7": "x_vars[S17,C8]",
   "x_16_8": "x_vars[S17,C9]",
   "x_16_9": "x_vars[S17,C10]",
   "x_16_10": "x_vars[S17,C11]",
   "x_16_11": "x_vars[S17,C12]",
   "x_16_12": "x_vars[S17,C13]",
   "x_16_13": "x_vars[S17,C14]",
   "x_16_14": "x_vars[S17,C15]",
   "x_16_15": "x_vars[S17,C16]",
   "x_16_16": "x_vars[S17,C17]",
   "x_16_17": "x_vars[S17,C18]",
   "x_16_18": "x_vars[S17,C19]",
   "x_16_19": "x_vars[S17,C20]",
   "x_16_20": "x_vars[S17,C21]",
   "x_16_21": "x_vars[S17,C22]",
   "x_16_22": "x_vars[S17,C23]",
   "x_16_23": "x_vars[S17,C24]",
   "x_16_24": "x_vars[S17,C25]",
   "x_17_0": "x_vars[S18,C1]",
   "x_17_1": "x_vars[S18,C2]",
   "x_17_2": "x_vars[S18,C3]",
   "x_17_3": "x_vars[S18,C4]",
   "x_17_4": "x_vars[S18,C5]",
   "x_17_5": "x_vars[S18,C6]",
   "x_17_6": "x_vars[S18,C7]",
   "x_17_7": "x_vars[S18,C8]",
   "x_17_8": "x_vars[S18,C9]",
   "x_17_9": "x_vars[S18,C10]",
   "x_17_10": "x_vars[S18,C11]",
   "x_17_11": "x_vars[S18,C12]",
   "x_17_12": "x_vars[S18,C13]",
   "x_17_13": "x_vars[S18,C14]",
   "x_17_14": "x_vars[S18,C15]",
   "x_17_15": "x_vars[S18,C16]",
   "x_17_16": "x_vars[S18,C17]",
   "x_17_17": "x_vars[S18,C18]",
   "x_17_18": "x_vars[S18,C19]",
   "x_17_19": "x_vars[S18,C20]",
   "x_17_20": "x_vars[S18,C21]",
   "x_17_21": "x_vars[S18,C22]",
   "x_17_22": "x_vars[S18,C23]",
   "x_17_23": "x_vars[S18,C24]",
   "x_17_24": "x_vars[S18,C25]",
   "x_18_0": "x_vars[S19,C1]",
   "x_18_1": "x_vars[S19,C2]",
   "x_18_2": "x_vars[S19,C3]",
   "x_18_3": "x_vars[S19,C4]",
   "x_18_4": "x_vars[S19,C5]",
   "x_18_5": "x_vars[S19,C6]",
   "x_18_6": "x_vars[S19,C7]",
   "x_18_7": "x_vars[S19,C8]",
   "x_18_8": "x_vars[S19,C9]",
   "x_18_9": "x_vars[S19,C10]",
   "x_18_10": "x_vars[S19,C11]",
   "x_18_11": "x_vars[S19,C12]",
   "x_18_12": "x_vars[S19,C13]",
   "x_18_13": "x_vars[S19,C14]",
   "x_18_14": "x_vars[S19,C15]",
   "x_18_15": "x_vars[S19,C16]",
   "x_18_16": "x_vars[S19,C17]",
   "x_18_17": "x_vars[S19,C18]",
   "x_18_18": "x_vars[S19,C19]",
   "x_18_19": "x_vars[S19,C20]",
   "x_18_20": "x_vars[S19,C21]",
   "x_18_21": "x_vars[S19,C22]",
   "x_18_22": "x_vars[S19,C23]",
   "x_18_23": "x_vars[S19,C24]",
   "x_18_24": "x_vars[S19,C25]",
   "x_19_0": "x_vars[S20,C1]",
   "x_19_1": "x_vars[S20,C2]",
   "x_19_2": "x_vars[S20,C3]",
   "x_19_3": "x_vars[S20,C4]",
   "x_19_4": "x_vars[S20,C5]",
   "x_19_5": "x_vars[S20,C6]",
   "x_19_6": "x_vars[S20,C7]",
   "x_19_7": "x_vars[S20,C8]",
   "x_19_8": "x_vars[S20,C9]",
   "x_19_9": "x_vars[S20,C10]",
   "x_19_10": "x_vars[S20,C11]",
   "x_19_11": "x_vars[S20,C12]",
   "x_19_12": "x_vars[S20,C13]",
   "x_19_13": "x_vars[S20,C14]",
   "x_19_14": "x_vars[S20,C15]",
   "x_19_15": "x_vars[S20,C16]",
   "x_19_16": "x_vars[S20,C17]",
   "x_19_17": "x_vars[S20,C18]",
   "x_19_18": "x_vars[S20,C19]",
   "x_19_19": "x_vars[S20,C20]",
   "x_19_20": "x_vars[S20,C21]",
   "x_19_21": "x_vars[S20,C22]",
   "x_19_22": "x_vars[S20,C23]",
   "x_19_23": "x_vars[S20,C24]",
   "x_19_24": "x_vars[S20,C25]",
   "x_20_0": "x_vars[S21,C1]",
   "x_20_1": "x_vars[S21,C2]",
   "x_20_2": "x_vars[S21,C3]",
   "x_20_3": "x_vars[S21,C4]",
   "x_20_4": "x_vars[S21,C5]",
   "x_20_5": "x_vars[S21,C6]",
   "x_20_6": "x_vars[S21,C7]",
   "x_20_7": "x_vars[S21,C8]",
   "x_20_8": "x_vars[S21,C9]",
   "x_20_9": "x_vars[S21,C10]",
   "x_20_10": "x_vars[S21,C11]",
   "x_20_11": "x_vars[S21,C12]",
   "x_20_12": "x_vars[S21,C13]",
   "x_20_13": "x_vars[S21,C14]",
   "x_20_14": "x_vars[S21,C15]",
   "x_20_15": "x_vars[S21,C16]",
   "x_20_16": "x_vars[S21,C17]",
   "x_20_17": "x_vars[S21,C18]",
   "x_20_18": "x_vars[S21,C19]",
   "x_20_19": "x_vars[S21,C20]",
   "x_20_20": "x_vars[S21,C21]",
   "x_20_21": "x_vars[S21,C22]",
   "x_20_22": "x_vars[S21,C23]",
   "x_20_23": "x_vars[S21,C24]",
   "x_20_24": "x_vars[S21,C25]",
   "x_21_0": "x_vars[S22,C1]",
   "x_21_1": "x_vars[S22,C2]",
   "x_21_2": "x_vars[S22,C3]",
   "x_21_3": "x_vars[S22,C4]",
   "x_21_4": "x_vars[S22,C5]",
   "x_21_5": "x_vars[S22,C6]",
   "x_21_6": "x_vars[S22,C7]",
   "x_21_7": "x_vars[S22,C8]",
   "x_21_8": "x_vars[S22,C9]",
   "x_21_9": "x_vars[S22,C10]",
   "x_21_10": "x_vars[S22,C11]",
   "x_21_11": "x_vars[S22,C12]",
   "x_21_12": "x_vars[S22,C13]",
   "x_21_13": "x_vars[S22,C14]",
   "x_21_14": "x_vars[S22,C15]",
   "x_21_15": "x_vars[S22,C16]",
   "x_21_16": "x_vars[S22,C17]",
   "x_21_17": "x_vars[S22,C18]",
   "x_21_18": "x_vars[S22,C19]",
   "x_21_19": "x_vars[S22,C20]",
   "x_21_20": "x_vars[S22,C21]",
   "x_21_21": "x_vars[S22,C22]",
   "x_21_22": "x_vars[S22,C23]",
   "x_21_23": "x_vars[S22,C24]",
   "x_21_24": "x_vars[S22,C25]",
   "x_22_0": "x_vars[S23,C1]",
   "x_22_1": "x_vars[S23,C2]",
   "x_22_2": "x_vars[S23,C3]",
   "x_22_3": "x_vars[S23,C4]",
   "x_22_4": "x_vars[S23,C5]",
   "x_22_5": "x_vars[S23,C6]",
   "x_22_6": "x_vars[S23,C7]",
   "x_22_7": "x_vars[S23,C8]",
   "x_22_8": "x_vars[S23,C9]",
   "x_22_9": "x_vars[S23,C10]",
   "x_22_10": "x_vars[S23,C11]",
   "x_22_11": "x_vars[S23,C12]",
   "x_22_12": "x_vars[S23,C13]",
   "x_22_13": "x_vars[S23,C14]",
   "x_22_14": "x_vars[S23,C15]",
   "x_22_15": "x_vars[S23,C16]",
   "x_22_16": "x_vars[S23,C17]",
   "x_22_17": "x_vars[S23,C18]",
   "x_22_18": "x_vars[S23,C19]",
   "x_22_19": "x_vars[S23,C20]",
   "x_22_20": "x_vars[S23,C21]",
   "x_22_21": "x_vars[S23,C22]",
   "x_22_22": "x_vars[S23,C23]",
   "x_22_23": "x_vars[S23,C24]",
   "x_22_24": "x_vars[S23,C25]",
   "x_23_0": "x_vars[S24,C1]",
   "x_23_1": "x_vars[S24,C2]",
   "x_23_2": "x_vars[S24,C3]",
   "x_23_3": "x_vars[S24,C4]",
   "x_23_4": "x_vars[S24,C5]",
   "x_23_5": "x_vars[S24,C6]",
   "x_23_6": "x_vars[S24,C7]",
   "x_23_7": "x_vars[S24,C8]",
   "x_23_8": "x_vars[S24,C9]",
   "x_23_9": "x_vars[S24,C10]",
   "x_23_10": "x_vars[S24,C11]",
   "x_23_11": "x_vars[S24,C12]",
   "x_23_12": "x_vars[S24,C13]",
   "x_23_13": "x_vars[S24,C14]",
   "x_23_14": "x_vars[S24,C15]",
   "x_23_15": "x_vars[S24,C16]",
   "x_23_16": "x_vars[S24,C17]",
   "x_23_17": "x_vars[S24,C18]",
   "x_23_18": "x_vars[S24,C19]",
   "x_23_19": "x_vars[S24,C20]",
   "x_23_20": "x_vars[S24,C21]",
   "x_23_21": "x_vars[S24,C22]",
   "x_23_22": "x_vars[S24,C23]",
   "x_23_23": "x_vars[S24,C24]",
   "x_23_24": "x_vars[S24,C25]",
   "x_10_1": "x_vars[S11,C2]",
   "x_6_8": "x_vars[S7,C9]",
   "x_1_20": "x_vars[S2,C21]",
   "x_2_20": "x_vars[S3,C21]",
   "x_7_20": "x_vars[S8,C21]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]",
   "y_3": "y_vars[S4]",
   "y_4": "y_vars[S5]",
   "y_5": "y_vars[S6]",
   "y_6": "y_vars[S7]",
   "y_7": "y_vars[S8]",
   "y_8": "y_vars[S9]",
   "y_9": "y_vars[S10]",
   "y_10": "y_vars[S11]",
   "y_11": "y_vars[S12]",
   "y_12": "y_vars[S13]",
   "y_13": "y_vars[S14]",
   "y_14": "y_vars[S15]",
   "y_15": "y_vars[S16]",
   "y_16": "y_vars[S17]",
   "y_17": "y_vars[S18]",
   "y_18": "y_vars[S19]",
   "y_19": "y_vars[S20]",
   "y_20": "y_vars[S21]",
   "y_21": "y_vars[S22]",
   "y_22": "y_vars[S23]",
   "y_23": "y_vars[S24]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-065": {
  "source_code_sha256": "fc7cba4f5e9e1e3434ff092bf4d9a5db0fce297efef4020cd0e7c9f0d1294b18",
  "original_solution_sha256": "9f10625464ea5be243cf19fd8acc413397e7ca0cdeb26a47524b048d53db8c12",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S1,C3]"
   ],
   [
    "C3",
    "x_vars[S2,C1]"
   ],
   [
    "C4",
    "x_vars[S2,C2]"
   ],
   [
    "C5",
    "x_vars[S2,C3]"
   ],
   [
    "C6",
    "x_vars[S3,C1]"
   ],
   [
    "C7",
    "x_vars[S3,C2]"
   ],
   [
    "C8",
    "x_vars[S3,C3]"
   ],
   [
    "C9",
    "y_vars[S1]"
   ],
   [
    "C10",
    "y_vars[S2]"
   ],
   [
    "C11",
    "y_vars[S3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_0_2": "x_vars[S1,C3]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "x_1_2": "x_vars[S2,C3]",
   "x_2_0": "x_vars[S3,C1]",
   "x_2_1": "x_vars[S3,C2]",
   "x_2_2": "x_vars[S3,C3]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]",
   "y_2": "y_vars[S3]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-066": {
  "source_code_sha256": "575ab02c9b2815b76b695a6992f132543dc85a77d69ad147eb66d9a079e36653",
  "original_solution_sha256": "feac0468c67fc04e0149f83bb4701b7245d0e0629e4616625b44ec9051a5abcb",
  "bindings": [
   [
    "C0",
    "x_vars[S1,C1]"
   ],
   [
    "C1",
    "x_vars[S1,C2]"
   ],
   [
    "C2",
    "x_vars[S2,C1]"
   ],
   [
    "C3",
    "x_vars[S2,C2]"
   ],
   [
    "C4",
    "y_vars[S1]"
   ],
   [
    "C5",
    "y_vars[S2]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[S1,C1]",
   "x_0_1": "x_vars[S1,C2]",
   "x_1_0": "x_vars[S2,C1]",
   "x_1_1": "x_vars[S2,C2]",
   "y_0": "y_vars[S1]",
   "y_1": "y_vars[S2]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-067": {
  "source_code_sha256": "8c17d2ef67a16f9b4fb59e40ed016ff61f77256b850e5ff4352e560a9b35d0a0",
  "original_solution_sha256": "80b7db5f84ce9becda4807c8f2c70bc69aab9561f86bc8bec9e138f6dbae405f",
  "bindings": [
   [
    "C0",
    "x_vars[MA,P1]"
   ],
   [
    "C1",
    "x_vars[MA,P2]"
   ],
   [
    "C2",
    "x_vars[MA,P3]"
   ],
   [
    "C3",
    "x_vars[MB,P1]"
   ],
   [
    "C4",
    "x_vars[MB,P2]"
   ],
   [
    "C5",
    "x_vars[MB,P3]"
   ],
   [
    "C6",
    "x_vars[MC,P1]"
   ],
   [
    "C7",
    "x_vars[MC,P2]"
   ],
   [
    "C8",
    "x_vars[MC,P3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[MA,P1]",
   "x_0_1": "x_vars[MA,P2]",
   "x_0_2": "x_vars[MA,P3]",
   "x_1_0": "x_vars[MB,P1]",
   "x_1_1": "x_vars[MB,P2]",
   "x_1_2": "x_vars[MB,P3]",
   "x_2_0": "x_vars[MC,P1]",
   "x_2_1": "x_vars[MC,P2]",
   "x_2_2": "x_vars[MC,P3]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-068": {
  "source_code_sha256": "125f8748df7cea328edae247f906aae879208d1aaba598a51392d2f0a8b9f195",
  "original_solution_sha256": "42bd7e848e91f292b559ceffbe2a5b282b53743efada860871afd630b4ef5fa3",
  "bindings": [
   [
    "C0",
    "x_vars[MA,P1]"
   ],
   [
    "C1",
    "x_vars[MA,P2]"
   ],
   [
    "C2",
    "x_vars[MA,P3]"
   ],
   [
    "C3",
    "x_vars[MA,P4]"
   ],
   [
    "C4",
    "x_vars[MA,P5]"
   ],
   [
    "C5",
    "x_vars[MA,P6]"
   ],
   [
    "C6",
    "x_vars[MB,P1]"
   ],
   [
    "C7",
    "x_vars[MB,P2]"
   ],
   [
    "C8",
    "x_vars[MB,P3]"
   ],
   [
    "C9",
    "x_vars[MB,P4]"
   ],
   [
    "C10",
    "x_vars[MB,P5]"
   ],
   [
    "C11",
    "x_vars[MB,P6]"
   ],
   [
    "C12",
    "x_vars[MC,P1]"
   ],
   [
    "C13",
    "x_vars[MC,P2]"
   ],
   [
    "C14",
    "x_vars[MC,P3]"
   ],
   [
    "C15",
    "x_vars[MC,P4]"
   ],
   [
    "C16",
    "x_vars[MC,P5]"
   ],
   [
    "C17",
    "x_vars[MC,P6]"
   ],
   [
    "C18",
    "x_vars[MD,P1]"
   ],
   [
    "C19",
    "x_vars[MD,P2]"
   ],
   [
    "C20",
    "x_vars[MD,P3]"
   ],
   [
    "C21",
    "x_vars[MD,P4]"
   ],
   [
    "C22",
    "x_vars[MD,P5]"
   ],
   [
    "C23",
    "x_vars[MD,P6]"
   ],
   [
    "C24",
    "x_vars[ME,P1]"
   ],
   [
    "C25",
    "x_vars[ME,P2]"
   ],
   [
    "C26",
    "x_vars[ME,P3]"
   ],
   [
    "C27",
    "x_vars[ME,P4]"
   ],
   [
    "C28",
    "x_vars[ME,P5]"
   ],
   [
    "C29",
    "x_vars[ME,P6]"
   ],
   [
    "C30",
    "x_vars[MF,P1]"
   ],
   [
    "C31",
    "x_vars[MF,P2]"
   ],
   [
    "C32",
    "x_vars[MF,P3]"
   ],
   [
    "C33",
    "x_vars[MF,P4]"
   ],
   [
    "C34",
    "x_vars[MF,P5]"
   ],
   [
    "C35",
    "x_vars[MF,P6]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[MA,P1]",
   "x_0_1": "x_vars[MA,P2]",
   "x_0_2": "x_vars[MA,P3]",
   "x_0_3": "x_vars[MA,P4]",
   "x_0_4": "x_vars[MA,P5]",
   "x_0_5": "x_vars[MA,P6]",
   "x_1_0": "x_vars[MB,P1]",
   "x_1_1": "x_vars[MB,P2]",
   "x_1_2": "x_vars[MB,P3]",
   "x_1_3": "x_vars[MB,P4]",
   "x_1_4": "x_vars[MB,P5]",
   "x_1_5": "x_vars[MB,P6]",
   "x_2_0": "x_vars[MC,P1]",
   "x_2_1": "x_vars[MC,P2]",
   "x_2_2": "x_vars[MC,P3]",
   "x_2_3": "x_vars[MC,P4]",
   "x_2_4": "x_vars[MC,P5]",
   "x_2_5": "x_vars[MC,P6]",
   "x_3_0": "x_vars[MD,P1]",
   "x_3_1": "x_vars[MD,P2]",
   "x_3_2": "x_vars[MD,P3]",
   "x_3_3": "x_vars[MD,P4]",
   "x_3_4": "x_vars[MD,P5]",
   "x_3_5": "x_vars[MD,P6]",
   "x_4_0": "x_vars[ME,P1]",
   "x_4_1": "x_vars[ME,P2]",
   "x_4_2": "x_vars[ME,P3]",
   "x_4_3": "x_vars[ME,P4]",
   "x_4_4": "x_vars[ME,P5]",
   "x_4_5": "x_vars[ME,P6]",
   "x_5_0": "x_vars[MF,P1]",
   "x_5_1": "x_vars[MF,P2]",
   "x_5_2": "x_vars[MF,P3]",
   "x_5_3": "x_vars[MF,P4]",
   "x_5_4": "x_vars[MF,P5]",
   "x_5_5": "x_vars[MF,P6]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-069": {
  "source_code_sha256": "d2d0010533cdb2628c59062429b5b71928fec25cae81109271977a846d83367a",
  "original_solution_sha256": "167914019b6a244d702ecd4d123caf4300e526dfc1991d0e0048c5d4fcc580b7",
  "bindings": [
   [
    "C0",
    "x_vars[Manager 1,Project 1]"
   ],
   [
    "C1",
    "x_vars[Manager 1,Project 2]"
   ],
   [
    "C2",
    "x_vars[Manager 1,Project 3]"
   ],
   [
    "C3",
    "x_vars[Manager 1,Project 4]"
   ],
   [
    "C4",
    "x_vars[Manager 1,Project 5]"
   ],
   [
    "C5",
    "x_vars[Manager 1,Project 6]"
   ],
   [
    "C6",
    "x_vars[Manager 1,Project 7]"
   ],
   [
    "C7",
    "x_vars[Manager 1,Project 8]"
   ],
   [
    "C8",
    "x_vars[Manager 1,Project 9]"
   ],
   [
    "C9",
    "x_vars[Manager 1,Project 10]"
   ],
   [
    "C10",
    "x_vars[Manager 1,Project 11]"
   ],
   [
    "C11",
    "x_vars[Manager 2,Project 1]"
   ],
   [
    "C12",
    "x_vars[Manager 2,Project 2]"
   ],
   [
    "C13",
    "x_vars[Manager 2,Project 3]"
   ],
   [
    "C14",
    "x_vars[Manager 2,Project 4]"
   ],
   [
    "C15",
    "x_vars[Manager 2,Project 5]"
   ],
   [
    "C16",
    "x_vars[Manager 2,Project 6]"
   ],
   [
    "C17",
    "x_vars[Manager 2,Project 7]"
   ],
   [
    "C18",
    "x_vars[Manager 2,Project 8]"
   ],
   [
    "C19",
    "x_vars[Manager 2,Project 9]"
   ],
   [
    "C20",
    "x_vars[Manager 2,Project 10]"
   ],
   [
    "C21",
    "x_vars[Manager 2,Project 11]"
   ],
   [
    "C22",
    "x_vars[Manager 3,Project 1]"
   ],
   [
    "C23",
    "x_vars[Manager 3,Project 2]"
   ],
   [
    "C24",
    "x_vars[Manager 3,Project 3]"
   ],
   [
    "C25",
    "x_vars[Manager 3,Project 4]"
   ],
   [
    "C26",
    "x_vars[Manager 3,Project 5]"
   ],
   [
    "C27",
    "x_vars[Manager 3,Project 6]"
   ],
   [
    "C28",
    "x_vars[Manager 3,Project 7]"
   ],
   [
    "C29",
    "x_vars[Manager 3,Project 8]"
   ],
   [
    "C30",
    "x_vars[Manager 3,Project 9]"
   ],
   [
    "C31",
    "x_vars[Manager 3,Project 10]"
   ],
   [
    "C32",
    "x_vars[Manager 3,Project 11]"
   ],
   [
    "C33",
    "x_vars[Manager 4,Project 1]"
   ],
   [
    "C34",
    "x_vars[Manager 4,Project 2]"
   ],
   [
    "C35",
    "x_vars[Manager 4,Project 3]"
   ],
   [
    "C36",
    "x_vars[Manager 4,Project 4]"
   ],
   [
    "C37",
    "x_vars[Manager 4,Project 5]"
   ],
   [
    "C38",
    "x_vars[Manager 4,Project 6]"
   ],
   [
    "C39",
    "x_vars[Manager 4,Project 7]"
   ],
   [
    "C40",
    "x_vars[Manager 4,Project 8]"
   ],
   [
    "C41",
    "x_vars[Manager 4,Project 9]"
   ],
   [
    "C42",
    "x_vars[Manager 4,Project 10]"
   ],
   [
    "C43",
    "x_vars[Manager 4,Project 11]"
   ],
   [
    "C44",
    "x_vars[Manager 5,Project 1]"
   ],
   [
    "C45",
    "x_vars[Manager 5,Project 2]"
   ],
   [
    "C46",
    "x_vars[Manager 5,Project 3]"
   ],
   [
    "C47",
    "x_vars[Manager 5,Project 4]"
   ],
   [
    "C48",
    "x_vars[Manager 5,Project 5]"
   ],
   [
    "C49",
    "x_vars[Manager 5,Project 6]"
   ],
   [
    "C50",
    "x_vars[Manager 5,Project 7]"
   ],
   [
    "C51",
    "x_vars[Manager 5,Project 8]"
   ],
   [
    "C52",
    "x_vars[Manager 5,Project 9]"
   ],
   [
    "C53",
    "x_vars[Manager 5,Project 10]"
   ],
   [
    "C54",
    "x_vars[Manager 5,Project 11]"
   ],
   [
    "C55",
    "x_vars[Manager 6,Project 1]"
   ],
   [
    "C56",
    "x_vars[Manager 6,Project 2]"
   ],
   [
    "C57",
    "x_vars[Manager 6,Project 3]"
   ],
   [
    "C58",
    "x_vars[Manager 6,Project 4]"
   ],
   [
    "C59",
    "x_vars[Manager 6,Project 5]"
   ],
   [
    "C60",
    "x_vars[Manager 6,Project 6]"
   ],
   [
    "C61",
    "x_vars[Manager 6,Project 7]"
   ],
   [
    "C62",
    "x_vars[Manager 6,Project 8]"
   ],
   [
    "C63",
    "x_vars[Manager 6,Project 9]"
   ],
   [
    "C64",
    "x_vars[Manager 6,Project 10]"
   ],
   [
    "C65",
    "x_vars[Manager 6,Project 11]"
   ],
   [
    "C66",
    "x_vars[Manager 7,Project 1]"
   ],
   [
    "C67",
    "x_vars[Manager 7,Project 2]"
   ],
   [
    "C68",
    "x_vars[Manager 7,Project 3]"
   ],
   [
    "C69",
    "x_vars[Manager 7,Project 4]"
   ],
   [
    "C70",
    "x_vars[Manager 7,Project 5]"
   ],
   [
    "C71",
    "x_vars[Manager 7,Project 6]"
   ],
   [
    "C72",
    "x_vars[Manager 7,Project 7]"
   ],
   [
    "C73",
    "x_vars[Manager 7,Project 8]"
   ],
   [
    "C74",
    "x_vars[Manager 7,Project 9]"
   ],
   [
    "C75",
    "x_vars[Manager 7,Project 10]"
   ],
   [
    "C76",
    "x_vars[Manager 7,Project 11]"
   ],
   [
    "C77",
    "x_vars[Manager 8,Project 1]"
   ],
   [
    "C78",
    "x_vars[Manager 8,Project 2]"
   ],
   [
    "C79",
    "x_vars[Manager 8,Project 3]"
   ],
   [
    "C80",
    "x_vars[Manager 8,Project 4]"
   ],
   [
    "C81",
    "x_vars[Manager 8,Project 5]"
   ],
   [
    "C82",
    "x_vars[Manager 8,Project 6]"
   ],
   [
    "C83",
    "x_vars[Manager 8,Project 7]"
   ],
   [
    "C84",
    "x_vars[Manager 8,Project 8]"
   ],
   [
    "C85",
    "x_vars[Manager 8,Project 9]"
   ],
   [
    "C86",
    "x_vars[Manager 8,Project 10]"
   ],
   [
    "C87",
    "x_vars[Manager 8,Project 11]"
   ],
   [
    "C88",
    "x_vars[Manager 9,Project 1]"
   ],
   [
    "C89",
    "x_vars[Manager 9,Project 2]"
   ],
   [
    "C90",
    "x_vars[Manager 9,Project 3]"
   ],
   [
    "C91",
    "x_vars[Manager 9,Project 4]"
   ],
   [
    "C92",
    "x_vars[Manager 9,Project 5]"
   ],
   [
    "C93",
    "x_vars[Manager 9,Project 6]"
   ],
   [
    "C94",
    "x_vars[Manager 9,Project 7]"
   ],
   [
    "C95",
    "x_vars[Manager 9,Project 8]"
   ],
   [
    "C96",
    "x_vars[Manager 9,Project 9]"
   ],
   [
    "C97",
    "x_vars[Manager 9,Project 10]"
   ],
   [
    "C98",
    "x_vars[Manager 9,Project 11]"
   ],
   [
    "C99",
    "x_vars[Manager 10,Project 1]"
   ],
   [
    "C100",
    "x_vars[Manager 10,Project 2]"
   ],
   [
    "C101",
    "x_vars[Manager 10,Project 3]"
   ],
   [
    "C102",
    "x_vars[Manager 10,Project 4]"
   ],
   [
    "C103",
    "x_vars[Manager 10,Project 5]"
   ],
   [
    "C104",
    "x_vars[Manager 10,Project 6]"
   ],
   [
    "C105",
    "x_vars[Manager 10,Project 7]"
   ],
   [
    "C106",
    "x_vars[Manager 10,Project 8]"
   ],
   [
    "C107",
    "x_vars[Manager 10,Project 9]"
   ],
   [
    "C108",
    "x_vars[Manager 10,Project 10]"
   ],
   [
    "C109",
    "x_vars[Manager 10,Project 11]"
   ],
   [
    "C110",
    "x_vars[Manager 11,Project 1]"
   ],
   [
    "C111",
    "x_vars[Manager 11,Project 2]"
   ],
   [
    "C112",
    "x_vars[Manager 11,Project 3]"
   ],
   [
    "C113",
    "x_vars[Manager 11,Project 4]"
   ],
   [
    "C114",
    "x_vars[Manager 11,Project 5]"
   ],
   [
    "C115",
    "x_vars[Manager 11,Project 6]"
   ],
   [
    "C116",
    "x_vars[Manager 11,Project 7]"
   ],
   [
    "C117",
    "x_vars[Manager 11,Project 8]"
   ],
   [
    "C118",
    "x_vars[Manager 11,Project 9]"
   ],
   [
    "C119",
    "x_vars[Manager 11,Project 10]"
   ],
   [
    "C120",
    "x_vars[Manager 11,Project 11]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Manager 1,Project 1]",
   "x_0_1": "x_vars[Manager 1,Project 2]",
   "x_0_2": "x_vars[Manager 1,Project 3]",
   "x_0_3": "x_vars[Manager 1,Project 4]",
   "x_0_4": "x_vars[Manager 1,Project 5]",
   "x_0_5": "x_vars[Manager 1,Project 6]",
   "x_0_6": "x_vars[Manager 1,Project 7]",
   "x_0_7": "x_vars[Manager 1,Project 8]",
   "x_0_8": "x_vars[Manager 1,Project 9]",
   "x_0_9": "x_vars[Manager 1,Project 10]",
   "x_0_10": "x_vars[Manager 1,Project 11]",
   "x_1_0": "x_vars[Manager 2,Project 1]",
   "x_1_1": "x_vars[Manager 2,Project 2]",
   "x_1_2": "x_vars[Manager 2,Project 3]",
   "x_1_3": "x_vars[Manager 2,Project 4]",
   "x_1_4": "x_vars[Manager 2,Project 5]",
   "x_1_5": "x_vars[Manager 2,Project 6]",
   "x_1_6": "x_vars[Manager 2,Project 7]",
   "x_1_7": "x_vars[Manager 2,Project 8]",
   "x_1_8": "x_vars[Manager 2,Project 9]",
   "x_1_9": "x_vars[Manager 2,Project 10]",
   "x_1_10": "x_vars[Manager 2,Project 11]",
   "x_2_0": "x_vars[Manager 3,Project 1]",
   "x_2_1": "x_vars[Manager 3,Project 2]",
   "x_2_2": "x_vars[Manager 3,Project 3]",
   "x_2_3": "x_vars[Manager 3,Project 4]",
   "x_2_4": "x_vars[Manager 3,Project 5]",
   "x_2_5": "x_vars[Manager 3,Project 6]",
   "x_2_6": "x_vars[Manager 3,Project 7]",
   "x_2_7": "x_vars[Manager 3,Project 8]",
   "x_2_8": "x_vars[Manager 3,Project 9]",
   "x_2_9": "x_vars[Manager 3,Project 10]",
   "x_2_10": "x_vars[Manager 3,Project 11]",
   "x_3_0": "x_vars[Manager 4,Project 1]",
   "x_3_1": "x_vars[Manager 4,Project 2]",
   "x_3_2": "x_vars[Manager 4,Project 3]",
   "x_3_3": "x_vars[Manager 4,Project 4]",
   "x_3_4": "x_vars[Manager 4,Project 5]",
   "x_3_5": "x_vars[Manager 4,Project 6]",
   "x_3_6": "x_vars[Manager 4,Project 7]",
   "x_3_7": "x_vars[Manager 4,Project 8]",
   "x_3_8": "x_vars[Manager 4,Project 9]",
   "x_3_9": "x_vars[Manager 4,Project 10]",
   "x_3_10": "x_vars[Manager 4,Project 11]",
   "x_4_0": "x_vars[Manager 5,Project 1]",
   "x_4_1": "x_vars[Manager 5,Project 2]",
   "x_4_2": "x_vars[Manager 5,Project 3]",
   "x_4_3": "x_vars[Manager 5,Project 4]",
   "x_4_4": "x_vars[Manager 5,Project 5]",
   "x_4_5": "x_vars[Manager 5,Project 6]",
   "x_4_6": "x_vars[Manager 5,Project 7]",
   "x_4_7": "x_vars[Manager 5,Project 8]",
   "x_4_8": "x_vars[Manager 5,Project 9]",
   "x_4_9": "x_vars[Manager 5,Project 10]",
   "x_4_10": "x_vars[Manager 5,Project 11]",
   "x_5_0": "x_vars[Manager 6,Project 1]",
   "x_5_1": "x_vars[Manager 6,Project 2]",
   "x_5_2": "x_vars[Manager 6,Project 3]",
   "x_5_3": "x_vars[Manager 6,Project 4]",
   "x_5_4": "x_vars[Manager 6,Project 5]",
   "x_5_5": "x_vars[Manager 6,Project 6]",
   "x_5_6": "x_vars[Manager 6,Project 7]",
   "x_5_7": "x_vars[Manager 6,Project 8]",
   "x_5_8": "x_vars[Manager 6,Project 9]",
   "x_5_9": "x_vars[Manager 6,Project 10]",
   "x_5_10": "x_vars[Manager 6,Project 11]",
   "x_6_0": "x_vars[Manager 7,Project 1]",
   "x_6_1": "x_vars[Manager 7,Project 2]",
   "x_6_2": "x_vars[Manager 7,Project 3]",
   "x_6_3": "x_vars[Manager 7,Project 4]",
   "x_6_4": "x_vars[Manager 7,Project 5]",
   "x_6_5": "x_vars[Manager 7,Project 6]",
   "x_6_6": "x_vars[Manager 7,Project 7]",
   "x_6_7": "x_vars[Manager 7,Project 8]",
   "x_6_8": "x_vars[Manager 7,Project 9]",
   "x_6_9": "x_vars[Manager 7,Project 10]",
   "x_6_10": "x_vars[Manager 7,Project 11]",
   "x_7_0": "x_vars[Manager 8,Project 1]",
   "x_7_1": "x_vars[Manager 8,Project 2]",
   "x_7_2": "x_vars[Manager 8,Project 3]",
   "x_7_3": "x_vars[Manager 8,Project 4]",
   "x_7_4": "x_vars[Manager 8,Project 5]",
   "x_7_5": "x_vars[Manager 8,Project 6]",
   "x_7_6": "x_vars[Manager 8,Project 7]",
   "x_7_7": "x_vars[Manager 8,Project 8]",
   "x_7_8": "x_vars[Manager 8,Project 9]",
   "x_7_9": "x_vars[Manager 8,Project 10]",
   "x_7_10": "x_vars[Manager 8,Project 11]",
   "x_8_0": "x_vars[Manager 9,Project 1]",
   "x_8_1": "x_vars[Manager 9,Project 2]",
   "x_8_2": "x_vars[Manager 9,Project 3]",
   "x_8_3": "x_vars[Manager 9,Project 4]",
   "x_8_4": "x_vars[Manager 9,Project 5]",
   "x_8_5": "x_vars[Manager 9,Project 6]",
   "x_8_6": "x_vars[Manager 9,Project 7]",
   "x_8_7": "x_vars[Manager 9,Project 8]",
   "x_8_8": "x_vars[Manager 9,Project 9]",
   "x_8_9": "x_vars[Manager 9,Project 10]",
   "x_8_10": "x_vars[Manager 9,Project 11]",
   "x_9_0": "x_vars[Manager 10,Project 1]",
   "x_9_1": "x_vars[Manager 10,Project 2]",
   "x_9_2": "x_vars[Manager 10,Project 3]",
   "x_9_3": "x_vars[Manager 10,Project 4]",
   "x_9_4": "x_vars[Manager 10,Project 5]",
   "x_9_5": "x_vars[Manager 10,Project 6]",
   "x_9_6": "x_vars[Manager 10,Project 7]",
   "x_9_7": "x_vars[Manager 10,Project 8]",
   "x_9_8": "x_vars[Manager 10,Project 9]",
   "x_9_9": "x_vars[Manager 10,Project 10]",
   "x_9_10": "x_vars[Manager 10,Project 11]",
   "x_10_0": "x_vars[Manager 11,Project 1]",
   "x_10_1": "x_vars[Manager 11,Project 2]",
   "x_10_2": "x_vars[Manager 11,Project 3]",
   "x_10_3": "x_vars[Manager 11,Project 4]",
   "x_10_4": "x_vars[Manager 11,Project 5]",
   "x_10_5": "x_vars[Manager 11,Project 6]",
   "x_10_6": "x_vars[Manager 11,Project 7]",
   "x_10_7": "x_vars[Manager 11,Project 8]",
   "x_10_8": "x_vars[Manager 11,Project 9]",
   "x_10_9": "x_vars[Manager 11,Project 10]",
   "x_10_10": "x_vars[Manager 11,Project 11]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-070": {
  "source_code_sha256": "3da613317286d4de7465cc95fc112ae1e67fe4844676efaf2a1c9b1f831a4da7",
  "original_solution_sha256": "810dcd637f4b50d39b4745b5791639fbba8c1b2c3186ab8bb472fc6112e3174d",
  "bindings": [
   [
    "C0",
    "x_vars[Manager 1,Project 1]"
   ],
   [
    "C1",
    "x_vars[Manager 1,Project 2]"
   ],
   [
    "C2",
    "x_vars[Manager 1,Project 3]"
   ],
   [
    "C3",
    "x_vars[Manager 1,Project 4]"
   ],
   [
    "C4",
    "x_vars[Manager 1,Project 5]"
   ],
   [
    "C5",
    "x_vars[Manager 1,Project 6]"
   ],
   [
    "C6",
    "x_vars[Manager 1,Project 7]"
   ],
   [
    "C7",
    "x_vars[Manager 2,Project 1]"
   ],
   [
    "C8",
    "x_vars[Manager 2,Project 2]"
   ],
   [
    "C9",
    "x_vars[Manager 2,Project 3]"
   ],
   [
    "C10",
    "x_vars[Manager 2,Project 4]"
   ],
   [
    "C11",
    "x_vars[Manager 2,Project 5]"
   ],
   [
    "C12",
    "x_vars[Manager 2,Project 6]"
   ],
   [
    "C13",
    "x_vars[Manager 2,Project 7]"
   ],
   [
    "C14",
    "x_vars[Manager 3,Project 1]"
   ],
   [
    "C15",
    "x_vars[Manager 3,Project 2]"
   ],
   [
    "C16",
    "x_vars[Manager 3,Project 3]"
   ],
   [
    "C17",
    "x_vars[Manager 3,Project 4]"
   ],
   [
    "C18",
    "x_vars[Manager 3,Project 5]"
   ],
   [
    "C19",
    "x_vars[Manager 3,Project 6]"
   ],
   [
    "C20",
    "x_vars[Manager 3,Project 7]"
   ],
   [
    "C21",
    "x_vars[Manager 4,Project 1]"
   ],
   [
    "C22",
    "x_vars[Manager 4,Project 2]"
   ],
   [
    "C23",
    "x_vars[Manager 4,Project 3]"
   ],
   [
    "C24",
    "x_vars[Manager 4,Project 4]"
   ],
   [
    "C25",
    "x_vars[Manager 4,Project 5]"
   ],
   [
    "C26",
    "x_vars[Manager 4,Project 6]"
   ],
   [
    "C27",
    "x_vars[Manager 4,Project 7]"
   ],
   [
    "C28",
    "x_vars[Manager 5,Project 1]"
   ],
   [
    "C29",
    "x_vars[Manager 5,Project 2]"
   ],
   [
    "C30",
    "x_vars[Manager 5,Project 3]"
   ],
   [
    "C31",
    "x_vars[Manager 5,Project 4]"
   ],
   [
    "C32",
    "x_vars[Manager 5,Project 5]"
   ],
   [
    "C33",
    "x_vars[Manager 5,Project 6]"
   ],
   [
    "C34",
    "x_vars[Manager 5,Project 7]"
   ],
   [
    "C35",
    "x_vars[Manager 6,Project 1]"
   ],
   [
    "C36",
    "x_vars[Manager 6,Project 2]"
   ],
   [
    "C37",
    "x_vars[Manager 6,Project 3]"
   ],
   [
    "C38",
    "x_vars[Manager 6,Project 4]"
   ],
   [
    "C39",
    "x_vars[Manager 6,Project 5]"
   ],
   [
    "C40",
    "x_vars[Manager 6,Project 6]"
   ],
   [
    "C41",
    "x_vars[Manager 6,Project 7]"
   ],
   [
    "C42",
    "x_vars[Manager 7,Project 1]"
   ],
   [
    "C43",
    "x_vars[Manager 7,Project 2]"
   ],
   [
    "C44",
    "x_vars[Manager 7,Project 3]"
   ],
   [
    "C45",
    "x_vars[Manager 7,Project 4]"
   ],
   [
    "C46",
    "x_vars[Manager 7,Project 5]"
   ],
   [
    "C47",
    "x_vars[Manager 7,Project 6]"
   ],
   [
    "C48",
    "x_vars[Manager 7,Project 7]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x_vars[Manager 1,Project 1]",
   "x_0_1": "x_vars[Manager 1,Project 2]",
   "x_0_2": "x_vars[Manager 1,Project 3]",
   "x_0_3": "x_vars[Manager 1,Project 4]",
   "x_0_4": "x_vars[Manager 1,Project 5]",
   "x_0_5": "x_vars[Manager 1,Project 6]",
   "x_0_6": "x_vars[Manager 1,Project 7]",
   "x_1_0": "x_vars[Manager 2,Project 1]",
   "x_1_1": "x_vars[Manager 2,Project 2]",
   "x_1_2": "x_vars[Manager 2,Project 3]",
   "x_1_3": "x_vars[Manager 2,Project 4]",
   "x_1_4": "x_vars[Manager 2,Project 5]",
   "x_1_5": "x_vars[Manager 2,Project 6]",
   "x_1_6": "x_vars[Manager 2,Project 7]",
   "x_2_0": "x_vars[Manager 3,Project 1]",
   "x_2_1": "x_vars[Manager 3,Project 2]",
   "x_2_2": "x_vars[Manager 3,Project 3]",
   "x_2_3": "x_vars[Manager 3,Project 4]",
   "x_2_4": "x_vars[Manager 3,Project 5]",
   "x_2_5": "x_vars[Manager 3,Project 6]",
   "x_2_6": "x_vars[Manager 3,Project 7]",
   "x_3_0": "x_vars[Manager 4,Project 1]",
   "x_3_1": "x_vars[Manager 4,Project 2]",
   "x_3_2": "x_vars[Manager 4,Project 3]",
   "x_3_3": "x_vars[Manager 4,Project 4]",
   "x_3_4": "x_vars[Manager 4,Project 5]",
   "x_3_5": "x_vars[Manager 4,Project 6]",
   "x_3_6": "x_vars[Manager 4,Project 7]",
   "x_4_0": "x_vars[Manager 5,Project 1]",
   "x_4_1": "x_vars[Manager 5,Project 2]",
   "x_4_2": "x_vars[Manager 5,Project 3]",
   "x_4_3": "x_vars[Manager 5,Project 4]",
   "x_4_4": "x_vars[Manager 5,Project 5]",
   "x_4_5": "x_vars[Manager 5,Project 6]",
   "x_4_6": "x_vars[Manager 5,Project 7]",
   "x_5_0": "x_vars[Manager 6,Project 1]",
   "x_5_1": "x_vars[Manager 6,Project 2]",
   "x_5_2": "x_vars[Manager 6,Project 3]",
   "x_5_3": "x_vars[Manager 6,Project 4]",
   "x_5_4": "x_vars[Manager 6,Project 5]",
   "x_5_5": "x_vars[Manager 6,Project 6]",
   "x_5_6": "x_vars[Manager 6,Project 7]",
   "x_6_0": "x_vars[Manager 7,Project 1]",
   "x_6_1": "x_vars[Manager 7,Project 2]",
   "x_6_2": "x_vars[Manager 7,Project 3]",
   "x_6_3": "x_vars[Manager 7,Project 4]",
   "x_6_4": "x_vars[Manager 7,Project 5]",
   "x_6_5": "x_vars[Manager 7,Project 6]",
   "x_6_6": "x_vars[Manager 7,Project 7]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-071": {
  "source_code_sha256": "12d645bbab01d777e14f4e53b74c5a861f1f9d228642b90653641e57da3c3aac",
  "original_solution_sha256": "bd2c67c5b4f0f21edf7640d760524a54c53fc5a9b6b33f19acb6495c0b9e16bb",
  "bindings": [
   [
    "C0",
    "x_vars[Classic Oxford Shirt]"
   ],
   [
    "C1",
    "x_vars[Cotton Crew Neck T-Shirt]"
   ],
   [
    "C2",
    "x_vars[French Terry Hoodie]"
   ],
   [
    "C3",
    "x_vars[Denim Snap Shirt]"
   ],
   [
    "C4",
    "x_vars[Jersey V-Neck T-Shirt]"
   ],
   [
    "C5",
    "x_vars[Unisex Jogger Pants]"
   ],
   [
    "C6",
    "x_vars[Flannel Plaid Shirt]"
   ],
   [
    "C7",
    "x_vars[Performance Polo Shirt]"
   ],
   [
    "C8",
    "x_vars[Long-sleeve Henley]"
   ],
   [
    "C9",
    "x_vars[Heavyweight Sweatshirt]"
   ],
   [
    "C10",
    "x_vars[Linen Blend Shirt]"
   ],
   [
    "C11",
    "x_vars[Graphic Print Tee]"
   ],
   [
    "C12",
    "x_vars[Quilted Vest]"
   ],
   [
    "C13",
    "x_vars[Chambray Work Shirt]"
   ],
   [
    "C14",
    "x_vars[Microfiber Sport Shirt]"
   ],
   [
    "C15",
    "x_vars[Wool Blend Peacoat]"
   ],
   [
    "C16",
    "x_vars[Seersucker Short-sleeve]"
   ],
   [
    "C17",
    "x_vars[Twill Utility Shirt]"
   ],
   [
    "C18",
    "x_vars[Basic Scoop Neck Tee]"
   ],
   [
    "C19",
    "x_vars[Waffle Knit Pullover]"
   ],
   [
    "C20",
    "x_vars[Button-Down Poplin Shirt]"
   ],
   [
    "C21",
    "x_vars[Slub Cotton T-Shirt]"
   ],
   [
    "C22",
    "x_vars[Drawstring Cargo Pants]"
   ],
   [
    "C23",
    "x_vars[Pima Cotton Dress Shirt]"
   ],
   [
    "C24",
    "x_vars[Pocket T-Shirt]"
   ],
   [
    "C25",
    "x_vars[Fleece-Lined Track Suit]"
   ],
   [
    "C26",
    "x_vars[Printed Camp Collar Shirt]"
   ],
   [
    "C27",
    "x_vars[Performance Dry-Fit Tee]"
   ],
   [
    "C28",
    "x_vars[Corduroy Shacket]"
   ],
   [
    "C29",
    "x_vars[Herringbone Casual Shirt]"
   ],
   [
    "C30",
    "x_vars[Organic Cotton Tee]"
   ],
   [
    "C31",
    "x_vars[Cashmere Blend Sweater]"
   ],
   [
    "C32",
    "x_vars[Striped Rugby Shirt]"
   ],
   [
    "C33",
    "x_vars[Raglan Sleeve Tee]"
   ],
   [
    "C34",
    "x_vars[Waterproof Rain Jacket]"
   ],
   [
    "C35",
    "x_vars[Micro-check Business Shirt]"
   ],
   [
    "C36",
    "x_vars[Heather Grey T-Shirt]"
   ],
   [
    "C37",
    "x_vars[Slim Fit Chinos]"
   ],
   [
    "C38",
    "x_vars[Band Collar Shirt]"
   ],
   [
    "C39",
    "x_vars[Triblend Jersey Tee]"
   ],
   [
    "C40",
    "x_vars[Sherpa Lined Pullover]"
   ],
   [
    "C41",
    "x_vars[Western Style Shirt]"
   ],
   [
    "C42",
    "x_vars[Recycled Poly T-Shirt]"
   ],
   [
    "C43",
    "x_vars[Insulated Puffer Jacket]"
   ],
   [
    "C44",
    "x_vars[End-on-End Fabric Shirt]"
   ],
   [
    "C45",
    "x_vars[Oversized Graphic Tee]"
   ],
   [
    "C46",
    "x_vars[Stretch Hybrid Shorts]"
   ],
   [
    "C47",
    "x_vars[Non-Iron Travel Shirt]"
   ],
   [
    "C48",
    "x_vars[Sun-Protective T-Shirt]"
   ],
   [
    "C49",
    "x_vars[Velvet Bomber Jacket]"
   ],
   [
    "C50",
    "x_vars[Madras Plaid Shirt]"
   ],
   [
    "C51",
    "x_vars[Tie-Dye T-Shirt]"
   ],
   [
    "C52",
    "x_vars[Lightweight Windbreaker]"
   ],
   [
    "C53",
    "x_vars[Pinpoint Oxford Shirt]"
   ],
   [
    "C54",
    "x_vars[Bamboo Viscose Tee]"
   ],
   [
    "C55",
    "x_vars[Tapered Denim Jeans]"
   ],
   [
    "C56",
    "x_vars[Contrast Collar Shirt]"
   ],
   [
    "C57",
    "x_vars[Mock Neck Long-sleeve]"
   ],
   [
    "C58",
    "x_vars[Down-filled Parka]"
   ],
   [
    "C59",
    "x_vars[Textured Grid Shirt]"
   ],
   [
    "C60",
    "x_vars[Heavy Cotton T-Shirt]"
   ],
   [
    "C61",
    "x_vars[Convertible Zip-Off Trousers]"
   ],
   [
    "C62",
    "x_vars[Tencel Blend Casual Shirt]"
   ],
   [
    "C63",
    "x_vars[Relaxed Fit Tee]"
   ],
   [
    "C64",
    "x_vars[Faux Leather Jacket]"
   ],
   [
    "C65",
    "x_vars[Speckled Slub Shirt]"
   ],
   [
    "C66",
    "x_vars[Embroidered T-Shirt]"
   ],
   [
    "C67",
    "x_vars[Knit Cardigan Sweater]"
   ],
   [
    "C68",
    "x_vars[Two-Pocket Utility Shirt]"
   ],
   [
    "C69",
    "x_vars[Performance Stretch Tee]"
   ],
   [
    "C70",
    "x_vars[Technical Shell Jacket]"
   ],
   [
    "C71",
    "x_vars[Semi-Spread Collar Shirt]"
   ],
   [
    "C72",
    "x_vars[Washed Pigment Dye Tee]"
   ],
   [
    "C73",
    "x_vars[Twill Blazer (Unlined)]"
   ],
   [
    "C74",
    "x_vars[Brushed Twill Shirt]"
   ],
   [
    "C75",
    "x_vars[Supima Cotton Tee]"
   ],
   [
    "C76",
    "x_vars[High-Waist Trousers]"
   ],
   [
    "C77",
    "x_vars[Printed Geometric Shirt]"
   ],
   [
    "C78",
    "x_vars[Minimalist Plain Tee]"
   ],
   [
    "C79",
    "x_vars[Reversible Quilted Jacket]"
   ],
   [
    "C80",
    "x_vars[Dobby Weave Shirt]"
   ],
   [
    "C81",
    "x_vars[Heavyweight Boxy Tee]"
   ],
   [
    "C82",
    "x_vars[Casual Wool Overcoat]"
   ],
   [
    "C83",
    "x_vars[Half-Zip Pullover Shirt]"
   ],
   [
    "C84",
    "x_vars[Drop Shoulder T-Shirt]"
   ],
   [
    "C85",
    "x_vars[Straight Leg Corduroys]"
   ],
   [
    "C86",
    "x_vars[Camp Collar Floral Shirt]"
   ],
   [
    "C87",
    "x_vars[Garment Dyed Tee]"
   ],
   [
    "C88",
    "x_vars[Suede Bomber Jacket]"
   ],
   [
    "C89",
    "x_vars[Melange Knit Shirt]"
   ],
   [
    "C90",
    "x_vars[Longline T-Shirt]"
   ],
   [
    "C91",
    "x_vars[Wide Leg Culottes]"
   ],
   [
    "C92",
    "x_vars[Tropical Print Shirt]"
   ],
   [
    "C93",
    "x_vars[Distressed Cotton Tee]"
   ],
   [
    "C94",
    "x_vars[Patchwork Denim Jacket]"
   ],
   [
    "C95",
    "x_vars[Micro-Dot Shirt]"
   ],
   [
    "C96",
    "x_vars[Muscle Fit T-Shirt]"
   ],
   [
    "C97",
    "x_vars[Woven Utility Vest]"
   ],
   [
    "C98",
    "x_vars[Silk Blend Dress Shirt]"
   ],
   [
    "C99",
    "x_vars[Heavyweight Jersey T-Shirt]"
   ],
   [
    "C100",
    "x_vars[Slim Fit Ankle Grazer]"
   ],
   [
    "C101",
    "x_vars[Lattice Weave Shirt]"
   ],
   [
    "C102",
    "x_vars[Boxy Cropped Tee]"
   ],
   [
    "C103",
    "x_vars[Reversible Bomber Jacket]"
   ],
   [
    "C104",
    "x_vars[Stretch Performance Shirt]"
   ],
   [
    "C105",
    "x_vars[Sublimation Print Tee]"
   ],
   [
    "C106",
    "x_vars[Brushed Cotton Trousers]"
   ],
   [
    "C107",
    "x_vars[Micro-Waffle Knit Shirt]"
   ],
   [
    "C108",
    "x_vars[Reflective Detail Tee]"
   ],
   [
    "C109",
    "x_vars[Shearling Lined Coat]"
   ],
   [
    "C110",
    "x_vars[Printed Patchwork Shirt]"
   ],
   [
    "C111",
    "x_vars[Washed Effect T-Shirt]"
   ],
   [
    "C112",
    "x_vars[Velour Track Pants]"
   ],
   [
    "C113",
    "x_vars[Fine Wale Corduroy Shirt]"
   ],
   [
    "C114",
    "x_vars[Organic Hemp T-Shirt]"
   ],
   [
    "C115",
    "x_vars[Leather Biker Jacket]"
   ],
   [
    "C116",
    "x_vars[Supersoft Modal Shirt]"
   ],
   [
    "C117",
    "x_vars[Deep V-Neck Tee]"
   ],
   [
    "C118",
    "x_vars[Wool Blend Trousers]"
   ],
   [
    "C119",
    "x_vars[Contrast Stitch Shirt]"
   ],
   [
    "C120",
    "x_vars[Embroidered Pocket Tee]"
   ],
   [
    "C121",
    "x_vars[Camo Print Hoodie]"
   ],
   [
    "C122",
    "x_vars[Silk-Touch Poplin Shirt]"
   ],
   [
    "C123",
    "x_vars[Slim Fit Pique Polo]"
   ],
   [
    "C124",
    "x_vars[Technical Cargo Vest]"
   ],
   [
    "C125",
    "x_vars[Cutaway Collar Shirt]"
   ],
   [
    "C126",
    "x_vars[Lycra Blend Active Tee]"
   ],
   [
    "C127",
    "x_vars[Heavy Knit Sweater]"
   ],
   [
    "C128",
    "x_vars[Textured Stripe Shirt]"
   ],
   [
    "C129",
    "x_vars[Bio-Washed T-Shirt]"
   ],
   [
    "C130",
    "x_vars[Mohair Blend Cardigan]"
   ],
   [
    "C131",
    "x_vars[Shantung Silk Shirt]"
   ],
   [
    "C132",
    "x_vars[Distressed Denim Tee]"
   ],
   [
    "C133",
    "x_vars[High-Tech Ski Jacket]"
   ],
   [
    "C134",
    "x_vars[Fine Poplin Check Shirt]"
   ],
   [
    "C135",
    "x_vars[Slim Fit Linen T-Shirt]"
   ],
   [
    "C136",
    "x_vars[Ripstop Utility Pants]"
   ],
   [
    "C137",
    "x_vars[Classic Barrel Cuff Shirt]"
   ],
   [
    "C138",
    "x_vars[Bamboo Fiber Tee]"
   ],
   [
    "C139",
    "x_vars[Shearling Trench Coat]"
   ],
   [
    "C140",
    "x_vars[Slim Fit Chambray]"
   ],
   [
    "C141",
    "x_vars[Printed Geometric Tee]"
   ],
   [
    "C142",
    "x_vars[Wool Cargo Trousers]"
   ],
   [
    "C143",
    "x_vars[Two-Tone Oxford Shirt]"
   ],
   [
    "C144",
    "x_vars[Slogan Print T-Shirt]"
   ],
   [
    "C145",
    "x_vars[Water-Resistant Shell]"
   ],
   [
    "C146",
    "x_vars[Grandad Collar Shirt]"
   ],
   [
    "C147",
    "x_vars[Embroidered Logo Tee]"
   ],
   [
    "C148",
    "x_vars[Casual Knit Blazer]"
   ],
   [
    "C149",
    "x_vars[Eco-Friendly Hemp Shirt]"
   ],
   [
    "C150",
    "x_vars[Cotton Slub Polo]"
   ],
   [
    "C151",
    "x_vars[Corduroy Carpenter Pants]"
   ],
   [
    "C152",
    "x_vars[Linen Safari Shirt]"
   ],
   [
    "C153",
    "x_vars[Ribbed Crew Neck Tee]"
   ],
   [
    "C154",
    "x_vars[Hybrid Hiking Trousers]"
   ],
   [
    "C155",
    "x_vars[Brushed Flannel Shirt]"
   ],
   [
    "C156",
    "x_vars[Microfiber V-Neck]"
   ],
   [
    "C157",
    "x_vars[Down-Alternative Vest]"
   ],
   [
    "C158",
    "x_vars[Checked Seersucker Shirt]"
   ],
   [
    "C159",
    "x_vars[Fitted Basic Tee]"
   ],
   [
    "C160",
    "x_vars[Heavyweight Denim Jacket]"
   ],
   [
    "C161",
    "x_vars[Woven Stripe Shirt]"
   ],
   [
    "C162",
    "x_vars[Tech Fleece Pullover]"
   ],
   [
    "C163",
    "x_vars[Pin Dot Print Shirt]"
   ],
   [
    "C164",
    "x_vars[Supersoft Modal T-Shirt]"
   ],
   [
    "C165",
    "x_vars[Water-Resistant Chinos]"
   ],
   [
    "C166",
    "x_vars[Tailored Fit Dress Shirt]"
   ],
   [
    "C167",
    "x_vars[Printed Art Graphic Tee]"
   ],
   [
    "C168",
    "x_vars[Cashmere Cable Knit]"
   ],
   [
    "C169",
    "x_vars[Plaid Flannel Overshirt]"
   ],
   [
    "C170",
    "x_vars[Moisture-Wicking Polo]"
   ],
   [
    "C171",
    "x_vars[Slim Tapered Jeans]"
   ],
   [
    "C172",
    "x_vars[Contrast Cuff Shirt]"
   ],
   [
    "C173",
    "x_vars[Sustainable Cotton Tee]"
   ],
   [
    "C174",
    "x_vars[Trench Coat (Lined)]"
   ],
   [
    "C175",
    "x_vars[Spread Collar Shirt]"
   ],
   [
    "C176",
    "x_vars[Printed Stripe Tee]"
   ],
   [
    "C177",
    "x_vars[Non-Iron Twill Shirt]"
   ],
   [
    "C178",
    "x_vars[Recycled Material Tee]"
   ],
   [
    "C179",
    "x_vars[Faux Fur Coat]"
   ],
   [
    "C180",
    "x_vars[Seersucker Popover Shirt]"
   ],
   [
    "C181",
    "x_vars[Graphic Back Print Tee]"
   ],
   [
    "C182",
    "x_vars[Corduroy Field Jacket]"
   ],
   [
    "C183",
    "x_vars[Linen/Cotton Blend Shirt]"
   ],
   [
    "C184",
    "x_vars[Lightweight Jersey Tee]"
   ],
   [
    "C185",
    "x_vars[High Performance Hiking Pants]"
   ],
   [
    "C186",
    "x_vars[Cuban Collar Shirt]"
   ],
   [
    "C187",
    "x_vars[Faded Pigment Tee]"
   ],
   [
    "C188",
    "x_vars[Canvas Chore Coat]"
   ],
   [
    "C189",
    "x_vars[Brushed Cotton Shirt]"
   ],
   [
    "C190",
    "x_vars[Ribbed Knit T-Shirt]"
   ],
   [
    "C191",
    "x_vars[Wide Leg Trousers]"
   ],
   [
    "C192",
    "x_vars[Patchwork Denim Shirt]"
   ],
   [
    "C193",
    "x_vars[Basic White T-Shirt]"
   ],
   [
    "C194",
    "x_vars[Mohair Knit Sweater]"
   ],
   [
    "C195",
    "x_vars[Contrast Stitch Polo]"
   ],
   [
    "C196",
    "x_vars[Tailored Wool Trousers]"
   ],
   [
    "C197",
    "x_vars[Heavyweight Denim Shirt]"
   ]
  ],
  "primary_mapping": {
   "x[Classic_Oxford_Shirt]": "x_vars[Classic Oxford Shirt]",
   "x[Cotton_Crew_Neck_T-Shirt]": "x_vars[Cotton Crew Neck T-Shirt]",
   "x[French_Terry_Hoodie]": "x_vars[French Terry Hoodie]",
   "x[Denim_Snap_Shirt]": "x_vars[Denim Snap Shirt]",
   "x[Jersey_V-Neck_T-Shirt]": "x_vars[Jersey V-Neck T-Shirt]",
   "x[Unisex_Jogger_Pants]": "x_vars[Unisex Jogger Pants]",
   "x[Flannel_Plaid_Shirt]": "x_vars[Flannel Plaid Shirt]",
   "x[Performance_Polo_Shirt]": "x_vars[Performance Polo Shirt]",
   "x[Long-sleeve_Henley]": "x_vars[Long-sleeve Henley]",
   "x[Heavyweight_Sweatshirt]": "x_vars[Heavyweight Sweatshirt]",
   "x[Linen_Blend_Shirt]": "x_vars[Linen Blend Shirt]",
   "x[Graphic_Print_Tee]": "x_vars[Graphic Print Tee]",
   "x[Quilted_Vest]": "x_vars[Quilted Vest]",
   "x[Chambray_Work_Shirt]": "x_vars[Chambray Work Shirt]",
   "x[Microfiber_Sport_Shirt]": "x_vars[Microfiber Sport Shirt]",
   "x[Wool_Blend_Peacoat]": "x_vars[Wool Blend Peacoat]",
   "x[Seersucker_Short-sleeve]": "x_vars[Seersucker Short-sleeve]",
   "x[Twill_Utility_Shirt]": "x_vars[Twill Utility Shirt]",
   "x[Basic_Scoop_Neck_Tee]": "x_vars[Basic Scoop Neck Tee]",
   "x[Waffle_Knit_Pullover]": "x_vars[Waffle Knit Pullover]",
   "x[Button-Down_Poplin_Shirt]": "x_vars[Button-Down Poplin Shirt]",
   "x[Slub_Cotton_T-Shirt]": "x_vars[Slub Cotton T-Shirt]",
   "x[Drawstring_Cargo_Pants]": "x_vars[Drawstring Cargo Pants]",
   "x[Pima_Cotton_Dress_Shirt]": "x_vars[Pima Cotton Dress Shirt]",
   "x[Pocket_T-Shirt]": "x_vars[Pocket T-Shirt]",
   "x[Fleece-Lined_Track_Suit]": "x_vars[Fleece-Lined Track Suit]",
   "x[Printed_Camp_Collar_Shirt]": "x_vars[Printed Camp Collar Shirt]",
   "x[Performance_Dry-Fit_Tee]": "x_vars[Performance Dry-Fit Tee]",
   "x[Corduroy_Shacket]": "x_vars[Corduroy Shacket]",
   "x[Herringbone_Casual_Shirt]": "x_vars[Herringbone Casual Shirt]",
   "x[Organic_Cotton_Tee]": "x_vars[Organic Cotton Tee]",
   "x[Cashmere_Blend_Sweater]": "x_vars[Cashmere Blend Sweater]",
   "x[Striped_Rugby_Shirt]": "x_vars[Striped Rugby Shirt]",
   "x[Raglan_Sleeve_Tee]": "x_vars[Raglan Sleeve Tee]",
   "x[Waterproof_Rain_Jacket]": "x_vars[Waterproof Rain Jacket]",
   "x[Micro-check_Business_Shirt]": "x_vars[Micro-check Business Shirt]",
   "x[Heather_Grey_T-Shirt]": "x_vars[Heather Grey T-Shirt]",
   "x[Slim_Fit_Chinos]": "x_vars[Slim Fit Chinos]",
   "x[Band_Collar_Shirt]": "x_vars[Band Collar Shirt]",
   "x[Triblend_Jersey_Tee]": "x_vars[Triblend Jersey Tee]",
   "x[Sherpa_Lined_Pullover]": "x_vars[Sherpa Lined Pullover]",
   "x[Western_Style_Shirt]": "x_vars[Western Style Shirt]",
   "x[Recycled_Poly_T-Shirt]": "x_vars[Recycled Poly T-Shirt]",
   "x[Insulated_Puffer_Jacket]": "x_vars[Insulated Puffer Jacket]",
   "x[End-on-End_Fabric_Shirt]": "x_vars[End-on-End Fabric Shirt]",
   "x[Oversized_Graphic_Tee]": "x_vars[Oversized Graphic Tee]",
   "x[Stretch_Hybrid_Shorts]": "x_vars[Stretch Hybrid Shorts]",
   "x[Non-Iron_Travel_Shirt]": "x_vars[Non-Iron Travel Shirt]",
   "x[Sun-Protective_T-Shirt]": "x_vars[Sun-Protective T-Shirt]",
   "x[Velvet_Bomber_Jacket]": "x_vars[Velvet Bomber Jacket]",
   "x[Madras_Plaid_Shirt]": "x_vars[Madras Plaid Shirt]",
   "x[Tie-Dye_T-Shirt]": "x_vars[Tie-Dye T-Shirt]",
   "x[Lightweight_Windbreaker]": "x_vars[Lightweight Windbreaker]",
   "x[Pinpoint_Oxford_Shirt]": "x_vars[Pinpoint Oxford Shirt]",
   "x[Bamboo_Viscose_Tee]": "x_vars[Bamboo Viscose Tee]",
   "x[Tapered_Denim_Jeans]": "x_vars[Tapered Denim Jeans]",
   "x[Contrast_Collar_Shirt]": "x_vars[Contrast Collar Shirt]",
   "x[Mock_Neck_Long-sleeve]": "x_vars[Mock Neck Long-sleeve]",
   "x[Down-filled_Parka]": "x_vars[Down-filled Parka]",
   "x[Textured_Grid_Shirt]": "x_vars[Textured Grid Shirt]",
   "x[Heavy_Cotton_T-Shirt]": "x_vars[Heavy Cotton T-Shirt]",
   "x[Convertible_Zip-Off_Trousers]": "x_vars[Convertible Zip-Off Trousers]",
   "x[Tencel_Blend_Casual_Shirt]": "x_vars[Tencel Blend Casual Shirt]",
   "x[Relaxed_Fit_Tee]": "x_vars[Relaxed Fit Tee]",
   "x[Faux_Leather_Jacket]": "x_vars[Faux Leather Jacket]",
   "x[Speckled_Slub_Shirt]": "x_vars[Speckled Slub Shirt]",
   "x[Embroidered_T-Shirt]": "x_vars[Embroidered T-Shirt]",
   "x[Knit_Cardigan_Sweater]": "x_vars[Knit Cardigan Sweater]",
   "x[Two-Pocket_Utility_Shirt]": "x_vars[Two-Pocket Utility Shirt]",
   "x[Performance_Stretch_Tee]": "x_vars[Performance Stretch Tee]",
   "x[Technical_Shell_Jacket]": "x_vars[Technical Shell Jacket]",
   "x[Semi-Spread_Collar_Shirt]": "x_vars[Semi-Spread Collar Shirt]",
   "x[Washed_Pigment_Dye_Tee]": "x_vars[Washed Pigment Dye Tee]",
   "x[Twill_Blazer_(Unlined)]": "x_vars[Twill Blazer (Unlined)]",
   "x[Brushed_Twill_Shirt]": "x_vars[Brushed Twill Shirt]",
   "x[Supima_Cotton_Tee]": "x_vars[Supima Cotton Tee]",
   "x[High-Waist_Trousers]": "x_vars[High-Waist Trousers]",
   "x[Printed_Geometric_Shirt]": "x_vars[Printed Geometric Shirt]",
   "x[Minimalist_Plain_Tee]": "x_vars[Minimalist Plain Tee]",
   "x[Reversible_Quilted_Jacket]": "x_vars[Reversible Quilted Jacket]",
   "x[Dobby_Weave_Shirt]": "x_vars[Dobby Weave Shirt]",
   "x[Heavyweight_Boxy_Tee]": "x_vars[Heavyweight Boxy Tee]",
   "x[Casual_Wool_Overcoat]": "x_vars[Casual Wool Overcoat]",
   "x[Half-Zip_Pullover_Shirt]": "x_vars[Half-Zip Pullover Shirt]",
   "x[Drop_Shoulder_T-Shirt]": "x_vars[Drop Shoulder T-Shirt]",
   "x[Straight_Leg_Corduroys]": "x_vars[Straight Leg Corduroys]",
   "x[Camp_Collar_Floral_Shirt]": "x_vars[Camp Collar Floral Shirt]",
   "x[Garment_Dyed_Tee]": "x_vars[Garment Dyed Tee]",
   "x[Suede_Bomber_Jacket]": "x_vars[Suede Bomber Jacket]",
   "x[Melange_Knit_Shirt]": "x_vars[Melange Knit Shirt]",
   "x[Longline_T-Shirt]": "x_vars[Longline T-Shirt]",
   "x[Wide_Leg_Culottes]": "x_vars[Wide Leg Culottes]",
   "x[Tropical_Print_Shirt]": "x_vars[Tropical Print Shirt]",
   "x[Distressed_Cotton_Tee]": "x_vars[Distressed Cotton Tee]",
   "x[Patchwork_Denim_Jacket]": "x_vars[Patchwork Denim Jacket]",
   "x[Micro-Dot_Shirt]": "x_vars[Micro-Dot Shirt]",
   "x[Muscle_Fit_T-Shirt]": "x_vars[Muscle Fit T-Shirt]",
   "x[Woven_Utility_Vest]": "x_vars[Woven Utility Vest]",
   "x[Silk_Blend_Dress_Shirt]": "x_vars[Silk Blend Dress Shirt]",
   "x[Heavyweight_Jersey_T-Shirt]": "x_vars[Heavyweight Jersey T-Shirt]",
   "x[Slim_Fit_Ankle_Grazer]": "x_vars[Slim Fit Ankle Grazer]",
   "x[Lattice_Weave_Shirt]": "x_vars[Lattice Weave Shirt]",
   "x[Boxy_Cropped_Tee]": "x_vars[Boxy Cropped Tee]",
   "x[Reversible_Bomber_Jacket]": "x_vars[Reversible Bomber Jacket]",
   "x[Stretch_Performance_Shirt]": "x_vars[Stretch Performance Shirt]",
   "x[Sublimation_Print_Tee]": "x_vars[Sublimation Print Tee]",
   "x[Brushed_Cotton_Trousers]": "x_vars[Brushed Cotton Trousers]",
   "x[Micro-Waffle_Knit_Shirt]": "x_vars[Micro-Waffle Knit Shirt]",
   "x[Reflective_Detail_Tee]": "x_vars[Reflective Detail Tee]",
   "x[Shearling_Lined_Coat]": "x_vars[Shearling Lined Coat]",
   "x[Printed_Patchwork_Shirt]": "x_vars[Printed Patchwork Shirt]",
   "x[Washed_Effect_T-Shirt]": "x_vars[Washed Effect T-Shirt]",
   "x[Velour_Track_Pants]": "x_vars[Velour Track Pants]",
   "x[Fine_Wale_Corduroy_Shirt]": "x_vars[Fine Wale Corduroy Shirt]",
   "x[Organic_Hemp_T-Shirt]": "x_vars[Organic Hemp T-Shirt]",
   "x[Leather_Biker_Jacket]": "x_vars[Leather Biker Jacket]",
   "x[Supersoft_Modal_Shirt]": "x_vars[Supersoft Modal Shirt]",
   "x[Deep_V-Neck_Tee]": "x_vars[Deep V-Neck Tee]",
   "x[Wool_Blend_Trousers]": "x_vars[Wool Blend Trousers]",
   "x[Contrast_Stitch_Shirt]": "x_vars[Contrast Stitch Shirt]",
   "x[Embroidered_Pocket_Tee]": "x_vars[Embroidered Pocket Tee]",
   "x[Camo_Print_Hoodie]": "x_vars[Camo Print Hoodie]",
   "x[Silk-Touch_Poplin_Shirt]": "x_vars[Silk-Touch Poplin Shirt]",
   "x[Slim_Fit_Pique_Polo]": "x_vars[Slim Fit Pique Polo]",
   "x[Technical_Cargo_Vest]": "x_vars[Technical Cargo Vest]",
   "x[Cutaway_Collar_Shirt]": "x_vars[Cutaway Collar Shirt]",
   "x[Lycra_Blend_Active_Tee]": "x_vars[Lycra Blend Active Tee]",
   "x[Heavy_Knit_Sweater]": "x_vars[Heavy Knit Sweater]",
   "x[Textured_Stripe_Shirt]": "x_vars[Textured Stripe Shirt]",
   "x[Bio-Washed_T-Shirt]": "x_vars[Bio-Washed T-Shirt]",
   "x[Mohair_Blend_Cardigan]": "x_vars[Mohair Blend Cardigan]",
   "x[Shantung_Silk_Shirt]": "x_vars[Shantung Silk Shirt]",
   "x[Distressed_Denim_Tee]": "x_vars[Distressed Denim Tee]",
   "x[High-Tech_Ski_Jacket]": "x_vars[High-Tech Ski Jacket]",
   "x[Fine_Poplin_Check_Shirt]": "x_vars[Fine Poplin Check Shirt]",
   "x[Slim_Fit_Linen_T-Shirt]": "x_vars[Slim Fit Linen T-Shirt]",
   "x[Ripstop_Utility_Pants]": "x_vars[Ripstop Utility Pants]",
   "x[Classic_Barrel_Cuff_Shirt]": "x_vars[Classic Barrel Cuff Shirt]",
   "x[Bamboo_Fiber_Tee]": "x_vars[Bamboo Fiber Tee]",
   "x[Shearling_Trench_Coat]": "x_vars[Shearling Trench Coat]",
   "x[Slim_Fit_Chambray]": "x_vars[Slim Fit Chambray]",
   "x[Printed_Geometric_Tee]": "x_vars[Printed Geometric Tee]",
   "x[Wool_Cargo_Trousers]": "x_vars[Wool Cargo Trousers]",
   "x[Two-Tone_Oxford_Shirt]": "x_vars[Two-Tone Oxford Shirt]",
   "x[Slogan_Print_T-Shirt]": "x_vars[Slogan Print T-Shirt]",
   "x[Water-Resistant_Shell]": "x_vars[Water-Resistant Shell]",
   "x[Grandad_Collar_Shirt]": "x_vars[Grandad Collar Shirt]",
   "x[Embroidered_Logo_Tee]": "x_vars[Embroidered Logo Tee]",
   "x[Casual_Knit_Blazer]": "x_vars[Casual Knit Blazer]",
   "x[Eco-Friendly_Hemp_Shirt]": "x_vars[Eco-Friendly Hemp Shirt]",
   "x[Cotton_Slub_Polo]": "x_vars[Cotton Slub Polo]",
   "x[Corduroy_Carpenter_Pants]": "x_vars[Corduroy Carpenter Pants]",
   "x[Linen_Safari_Shirt]": "x_vars[Linen Safari Shirt]",
   "x[Ribbed_Crew_Neck_Tee]": "x_vars[Ribbed Crew Neck Tee]",
   "x[Hybrid_Hiking_Trousers]": "x_vars[Hybrid Hiking Trousers]",
   "x[Brushed_Flannel_Shirt]": "x_vars[Brushed Flannel Shirt]",
   "x[Microfiber_V-Neck]": "x_vars[Microfiber V-Neck]",
   "x[Down-Alternative_Vest]": "x_vars[Down-Alternative Vest]",
   "x[Checked_Seersucker_Shirt]": "x_vars[Checked Seersucker Shirt]",
   "x[Fitted_Basic_Tee]": "x_vars[Fitted Basic Tee]",
   "x[Heavyweight_Denim_Jacket]": "x_vars[Heavyweight Denim Jacket]",
   "x[Woven_Stripe_Shirt]": "x_vars[Woven Stripe Shirt]",
   "x[Tech_Fleece_Pullover]": "x_vars[Tech Fleece Pullover]",
   "x[Pin_Dot_Print_Shirt]": "x_vars[Pin Dot Print Shirt]",
   "x[Supersoft_Modal_T-Shirt]": "x_vars[Supersoft Modal T-Shirt]",
   "x[Water-Resistant_Chinos]": "x_vars[Water-Resistant Chinos]",
   "x[Tailored_Fit_Dress_Shirt]": "x_vars[Tailored Fit Dress Shirt]",
   "x[Printed_Art_Graphic_Tee]": "x_vars[Printed Art Graphic Tee]",
   "x[Cashmere_Cable_Knit]": "x_vars[Cashmere Cable Knit]",
   "x[Plaid_Flannel_Overshirt]": "x_vars[Plaid Flannel Overshirt]",
   "x[Moisture-Wicking_Polo]": "x_vars[Moisture-Wicking Polo]",
   "x[Slim_Tapered_Jeans]": "x_vars[Slim Tapered Jeans]",
   "x[Contrast_Cuff_Shirt]": "x_vars[Contrast Cuff Shirt]",
   "x[Sustainable_Cotton_Tee]": "x_vars[Sustainable Cotton Tee]",
   "x[Trench_Coat_(Lined)]": "x_vars[Trench Coat (Lined)]",
   "x[Spread_Collar_Shirt]": "x_vars[Spread Collar Shirt]",
   "x[Printed_Stripe_Tee]": "x_vars[Printed Stripe Tee]",
   "x[Non-Iron_Twill_Shirt]": "x_vars[Non-Iron Twill Shirt]",
   "x[Recycled_Material_Tee]": "x_vars[Recycled Material Tee]",
   "x[Faux_Fur_Coat]": "x_vars[Faux Fur Coat]",
   "x[Seersucker_Popover_Shirt]": "x_vars[Seersucker Popover Shirt]",
   "x[Graphic_Back_Print_Tee]": "x_vars[Graphic Back Print Tee]",
   "x[Corduroy_Field_Jacket]": "x_vars[Corduroy Field Jacket]",
   "x[Linen/Cotton_Blend_Shirt]": "x_vars[Linen/Cotton Blend Shirt]",
   "x[Lightweight_Jersey_Tee]": "x_vars[Lightweight Jersey Tee]",
   "x[High_Performance_Hiking_Pants]": "x_vars[High Performance Hiking Pants]",
   "x[Cuban_Collar_Shirt]": "x_vars[Cuban Collar Shirt]",
   "x[Faded_Pigment_Tee]": "x_vars[Faded Pigment Tee]",
   "x[Canvas_Chore_Coat]": "x_vars[Canvas Chore Coat]",
   "x[Brushed_Cotton_Shirt]": "x_vars[Brushed Cotton Shirt]",
   "x[Ribbed_Knit_T-Shirt]": "x_vars[Ribbed Knit T-Shirt]",
   "x[Wide_Leg_Trousers]": "x_vars[Wide Leg Trousers]",
   "x[Patchwork_Denim_Shirt]": "x_vars[Patchwork Denim Shirt]",
   "x[Basic_White_T-Shirt]": "x_vars[Basic White T-Shirt]",
   "x[Mohair_Knit_Sweater]": "x_vars[Mohair Knit Sweater]",
   "x[Contrast_Stitch_Polo]": "x_vars[Contrast Stitch Polo]",
   "x[Tailored_Wool_Trousers]": "x_vars[Tailored Wool Trousers]",
   "x[Heavyweight_Denim_Shirt]": "x_vars[Heavyweight Denim Shirt]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-072": {
  "source_code_sha256": "972a3170510df3735060880ff1af7fd2b311ef229ef607033b389e5d1120df56",
  "original_solution_sha256": "6b9d488653a612575c58e5e3941364fb5f47c7b7a1cc8f9649a50a0372ff8594",
  "bindings": [
   [
    "C0",
    "x_vars[1]"
   ],
   [
    "C1",
    "x_vars[2]"
   ],
   [
    "C2",
    "x_vars[3]"
   ],
   [
    "C3",
    "x_vars[4]"
   ],
   [
    "C4",
    "x_vars[5]"
   ],
   [
    "C5",
    "x_vars[6]"
   ],
   [
    "C6",
    "x_vars[7]"
   ],
   [
    "C7",
    "x_vars[8]"
   ],
   [
    "C8",
    "x_vars[9]"
   ],
   [
    "C9",
    "x_vars[10]"
   ],
   [
    "C10",
    "x_vars[11]"
   ],
   [
    "C11",
    "x_vars[12]"
   ],
   [
    "C12",
    "x_vars[13]"
   ],
   [
    "C13",
    "x_vars[14]"
   ],
   [
    "C14",
    "x_vars[15]"
   ],
   [
    "C15",
    "x_vars[16]"
   ],
   [
    "C16",
    "x_vars[17]"
   ],
   [
    "C17",
    "x_vars[18]"
   ],
   [
    "C18",
    "x_vars[19]"
   ],
   [
    "C19",
    "x_vars[20]"
   ],
   [
    "C20",
    "x_vars[21]"
   ],
   [
    "C21",
    "x_vars[22]"
   ],
   [
    "C22",
    "x_vars[23]"
   ],
   [
    "C23",
    "x_vars[24]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x_vars[1]",
   "x[1]": "x_vars[2]",
   "x[2]": "x_vars[3]",
   "x[3]": "x_vars[4]",
   "x[4]": "x_vars[5]",
   "x[5]": "x_vars[6]",
   "x[6]": "x_vars[7]",
   "x[7]": "x_vars[8]",
   "x[8]": "x_vars[9]",
   "x[9]": "x_vars[10]",
   "x[10]": "x_vars[11]",
   "x[11]": "x_vars[12]",
   "x[12]": "x_vars[13]",
   "x[13]": "x_vars[14]",
   "x[14]": "x_vars[15]",
   "x[15]": "x_vars[16]",
   "x[16]": "x_vars[17]",
   "x[17]": "x_vars[18]",
   "x[18]": "x_vars[19]",
   "x[19]": "x_vars[20]",
   "x[20]": "x_vars[21]",
   "x[21]": "x_vars[22]",
   "x[22]": "x_vars[23]",
   "x[23]": "x_vars[24]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-073": {
  "source_code_sha256": "e5c5be6c1f6ab329e67243610a7c3036c0248147000ec8074b96e0ab0a67e16c",
  "original_solution_sha256": "0c319d4ad78c5c6f3124ac3159cfc97b0c236d834fee54f0fc96f2f74348c349",
  "bindings": [
   [
    "x_I_A2",
    "x_I_A2"
   ],
   [
    "x_I_A1",
    "x_I_A1"
   ],
   [
    "x_I_B1",
    "x_I_B1"
   ],
   [
    "x_I_B2",
    "x_I_B2"
   ],
   [
    "x_I_B3",
    "x_I_B3"
   ],
   [
    "x_II_A2",
    "x_II_A2"
   ],
   [
    "x_II_A1",
    "x_II_A1"
   ],
   [
    "x_II_B1",
    "x_II_B1"
   ],
   [
    "x_III_A2",
    "x_III_A2"
   ],
   [
    "x_III_B2",
    "x_III_B2"
   ]
  ],
  "primary_mapping": {
   "x[A1,I]": "x_I_A1",
   "x[A2,II]": "x_II_A2",
   "x[A2,III]": "x_III_A2",
   "x[B1,I]": "x_I_B1",
   "x[B2,III]": "x_III_B2",
   "x[B3,I]": "x_I_B3",
   "x[A1,II]": "x_II_A1",
   "x[A2,I]": "x_I_A2",
   "x[B1,II]": "x_II_B1",
   "x[B2,I]": "x_I_B2"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-074": {
  "source_code_sha256": "40fd636eeb343143881ae3ab30c729844d61ce2e6149b3127ff433d55f1ce4b2",
  "original_solution_sha256": "bdecbccb5c8bc776e31b1be37e3606d424d7f2fa947ba920f4190f53b33bfa51",
  "bindings": [
   [
    "C0",
    "x_vars[0]"
   ],
   [
    "C1",
    "x_vars[1]"
   ],
   [
    "C2",
    "x_vars[2]"
   ],
   [
    "C3",
    "x_vars[3]"
   ],
   [
    "C4",
    "x_vars[4]"
   ],
   [
    "C5",
    "x_vars[5]"
   ],
   [
    "C6",
    "x_vars[6]"
   ],
   [
    "C7",
    "x_vars[7]"
   ],
   [
    "C8",
    "x_vars[8]"
   ],
   [
    "C9",
    "x_vars[9]"
   ],
   [
    "C10",
    "x_vars[10]"
   ],
   [
    "C11",
    "x_vars[11]"
   ],
   [
    "C12",
    "x_vars[12]"
   ],
   [
    "C13",
    "x_vars[13]"
   ],
   [
    "C14",
    "x_vars[14]"
   ],
   [
    "C15",
    "x_vars[15]"
   ],
   [
    "C16",
    "x_vars[16]"
   ],
   [
    "C17",
    "x_vars[17]"
   ],
   [
    "C18",
    "x_vars[18]"
   ],
   [
    "C19",
    "x_vars[19]"
   ],
   [
    "C20",
    "x_vars[20]"
   ],
   [
    "C21",
    "x_vars[21]"
   ],
   [
    "C22",
    "x_vars[22]"
   ],
   [
    "C23",
    "x_vars[23]"
   ],
   [
    "C24",
    "x_vars[24]"
   ],
   [
    "C25",
    "x_vars[25]"
   ],
   [
    "C26",
    "x_vars[26]"
   ],
   [
    "C27",
    "x_vars[27]"
   ],
   [
    "C28",
    "x_vars[28]"
   ],
   [
    "C29",
    "x_vars[29]"
   ],
   [
    "C30",
    "x_vars[30]"
   ],
   [
    "C31",
    "x_vars[31]"
   ],
   [
    "C32",
    "x_vars[32]"
   ],
   [
    "C33",
    "x_vars[33]"
   ],
   [
    "C34",
    "x_vars[34]"
   ],
   [
    "C35",
    "x_vars[35]"
   ],
   [
    "C36",
    "x_vars[36]"
   ],
   [
    "C37",
    "x_vars[37]"
   ],
   [
    "C38",
    "x_vars[38]"
   ],
   [
    "C39",
    "x_vars[39]"
   ],
   [
    "C40",
    "x_vars[40]"
   ],
   [
    "C41",
    "x_vars[41]"
   ],
   [
    "C42",
    "x_vars[42]"
   ],
   [
    "C43",
    "x_vars[43]"
   ],
   [
    "C44",
    "x_vars[44]"
   ],
   [
    "C45",
    "x_vars[45]"
   ],
   [
    "C46",
    "x_vars[46]"
   ],
   [
    "C47",
    "x_vars[47]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x_vars[0]",
   "x[1]": "x_vars[1]",
   "x[2]": "x_vars[2]",
   "x[3]": "x_vars[3]",
   "x[4]": "x_vars[4]",
   "x[5]": "x_vars[5]",
   "x[6]": "x_vars[6]",
   "x[7]": "x_vars[7]",
   "x[8]": "x_vars[8]",
   "x[9]": "x_vars[9]",
   "x[10]": "x_vars[10]",
   "x[11]": "x_vars[11]",
   "x[12]": "x_vars[12]",
   "x[13]": "x_vars[13]",
   "x[14]": "x_vars[14]",
   "x[15]": "x_vars[15]",
   "x[16]": "x_vars[16]",
   "x[17]": "x_vars[17]",
   "x[18]": "x_vars[18]",
   "x[19]": "x_vars[19]",
   "x[20]": "x_vars[20]",
   "x[21]": "x_vars[21]",
   "x[22]": "x_vars[22]",
   "x[23]": "x_vars[23]",
   "x[24]": "x_vars[24]",
   "x[25]": "x_vars[25]",
   "x[26]": "x_vars[26]",
   "x[27]": "x_vars[27]",
   "x[28]": "x_vars[28]",
   "x[29]": "x_vars[29]",
   "x[30]": "x_vars[30]",
   "x[31]": "x_vars[31]",
   "x[32]": "x_vars[32]",
   "x[33]": "x_vars[33]",
   "x[34]": "x_vars[34]",
   "x[35]": "x_vars[35]",
   "x[36]": "x_vars[36]",
   "x[37]": "x_vars[37]",
   "x[38]": "x_vars[38]",
   "x[39]": "x_vars[39]",
   "x[40]": "x_vars[40]",
   "x[41]": "x_vars[41]",
   "x[42]": "x_vars[42]",
   "x[43]": "x_vars[43]",
   "x[44]": "x_vars[44]",
   "x[45]": "x_vars[45]",
   "x[46]": "x_vars[46]",
   "x[47]": "x_vars[47]"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-075": {
  "source_code_sha256": "dd2e4878261a9744a7b6584b146f18644cbc9d82d6cac898ef39a7dcc85b42e3",
  "original_solution_sha256": "f66d7578c64f979d4379b5f3211192585a5f4f68874a8edb433e6daef5bcc4cf",
  "bindings": [
   [
    "C0",
    "x_vars[1]"
   ],
   [
    "C1",
    "x_vars[2]"
   ],
   [
    "C2",
    "x_vars[3]"
   ],
   [
    "C3",
    "x_vars[4]"
   ],
   [
    "C4",
    "x_vars[5]"
   ],
   [
    "C5",
    "x_vars[6]"
   ],
   [
    "C6",
    "x_vars[7]"
   ],
   [
    "C7",
    "x_vars[8]"
   ],
   [
    "C8",
    "x_vars[9]"
   ],
   [
    "C9",
    "x_vars[10]"
   ],
   [
    "C10",
    "x_vars[11]"
   ],
   [
    "C11",
    "x_vars[12]"
   ],
   [
    "C12",
    "x_vars[13]"
   ],
   [
    "C13",
    "x_vars[14]"
   ],
   [
    "C14",
    "x_vars[15]"
   ],
   [
    "C15",
    "x_vars[16]"
   ],
   [
    "C16",
    "x_vars[17]"
   ],
   [
    "C17",
    "x_vars[18]"
   ],
   [
    "C18",
    "x_vars[19]"
   ],
   [
    "C19",
    "x_vars[20]"
   ],
   [
    "C20",
    "x_vars[21]"
   ],
   [
    "C21",
    "x_vars[22]"
   ],
   [
    "C22",
    "x_vars[23]"
   ],
   [
    "C23",
    "x_vars[24]"
   ],
   [
    "C24",
    "x_vars[25]"
   ],
   [
    "C25",
    "x_vars[26]"
   ],
   [
    "C26",
    "x_vars[27]"
   ],
   [
    "C27",
    "x_vars[28]"
   ],
   [
    "C28",
    "x_vars[29]"
   ],
   [
    "C29",
    "x_vars[30]"
   ],
   [
    "C30",
    "x_vars[31]"
   ],
   [
    "C31",
    "x_vars[32]"
   ],
   [
    "C32",
    "x_vars[33]"
   ],
   [
    "C33",
    "x_vars[34]"
   ],
   [
    "C34",
    "x_vars[35]"
   ],
   [
    "C35",
    "x_vars[36]"
   ],
   [
    "C36",
    "x_vars[37]"
   ],
   [
    "C37",
    "x_vars[38]"
   ],
   [
    "C38",
    "x_vars[39]"
   ],
   [
    "C39",
    "x_vars[40]"
   ],
   [
    "C40",
    "x_vars[41]"
   ],
   [
    "C41",
    "x_vars[42]"
   ],
   [
    "C42",
    "x_vars[43]"
   ],
   [
    "C43",
    "x_vars[44]"
   ],
   [
    "C44",
    "x_vars[45]"
   ],
   [
    "C45",
    "x_vars[46]"
   ],
   [
    "C46",
    "x_vars[47]"
   ],
   [
    "C47",
    "x_vars[48]"
   ],
   [
    "C48",
    "x_vars[49]"
   ],
   [
    "C49",
    "x_vars[50]"
   ],
   [
    "C50",
    "x_vars[51]"
   ],
   [
    "C51",
    "x_vars[52]"
   ],
   [
    "C52",
    "x_vars[53]"
   ],
   [
    "C53",
    "x_vars[54]"
   ],
   [
    "C54",
    "x_vars[55]"
   ],
   [
    "C55",
    "x_vars[56]"
   ],
   [
    "C56",
    "x_vars[57]"
   ],
   [
    "C57",
    "x_vars[58]"
   ],
   [
    "C58",
    "x_vars[59]"
   ],
   [
    "C59",
    "x_vars[60]"
   ],
   [
    "C60",
    "x_vars[61]"
   ],
   [
    "C61",
    "x_vars[62]"
   ],
   [
    "C62",
    "x_vars[63]"
   ],
   [
    "C63",
    "x_vars[64]"
   ],
   [
    "C64",
    "x_vars[65]"
   ],
   [
    "C65",
    "x_vars[66]"
   ],
   [
    "C66",
    "x_vars[67]"
   ],
   [
    "C67",
    "x_vars[68]"
   ],
   [
    "C68",
    "x_vars[69]"
   ],
   [
    "C69",
    "x_vars[70]"
   ],
   [
    "C70",
    "x_vars[71]"
   ],
   [
    "C71",
    "x_vars[72]"
   ],
   [
    "C72",
    "x_vars[73]"
   ],
   [
    "C73",
    "x_vars[74]"
   ],
   [
    "C74",
    "x_vars[75]"
   ],
   [
    "C75",
    "x_vars[76]"
   ],
   [
    "C76",
    "x_vars[77]"
   ],
   [
    "C77",
    "x_vars[78]"
   ],
   [
    "C78",
    "x_vars[79]"
   ],
   [
    "C79",
    "x_vars[80]"
   ],
   [
    "C80",
    "x_vars[81]"
   ],
   [
    "C81",
    "x_vars[82]"
   ],
   [
    "C82",
    "x_vars[83]"
   ],
   [
    "C83",
    "x_vars[84]"
   ],
   [
    "C84",
    "x_vars[85]"
   ],
   [
    "C85",
    "x_vars[86]"
   ],
   [
    "C86",
    "x_vars[87]"
   ],
   [
    "C87",
    "x_vars[88]"
   ],
   [
    "C88",
    "x_vars[89]"
   ],
   [
    "C89",
    "x_vars[90]"
   ],
   [
    "C90",
    "x_vars[91]"
   ],
   [
    "C91",
    "x_vars[92]"
   ],
   [
    "C92",
    "x_vars[93]"
   ],
   [
    "C93",
    "x_vars[94]"
   ],
   [
    "C94",
    "x_vars[95]"
   ],
   [
    "C95",
    "x_vars[96]"
   ],
   [
    "C96",
    "x_vars[97]"
   ],
   [
    "C97",
    "x_vars[98]"
   ],
   [
    "C98",
    "x_vars[99]"
   ],
   [
    "C99",
    "x_vars[100]"
   ],
   [
    "C100",
    "x_vars[101]"
   ],
   [
    "C101",
    "x_vars[102]"
   ],
   [
    "C102",
    "x_vars[103]"
   ],
   [
    "C103",
    "x_vars[104]"
   ],
   [
    "C104",
    "x_vars[105]"
   ],
   [
    "C105",
    "x_vars[106]"
   ],
   [
    "C106",
    "x_vars[107]"
   ],
   [
    "C107",
    "x_vars[108]"
   ],
   [
    "C108",
    "x_vars[109]"
   ],
   [
    "C109",
    "x_vars[110]"
   ]
  ],
  "primary_mapping": {
   "y[1]": "x_vars[1]",
   "y[2]": "x_vars[2]",
   "y[3]": "x_vars[3]",
   "y[4]": "x_vars[4]",
   "y[5]": "x_vars[5]",
   "y[6]": "x_vars[6]",
   "y[7]": "x_vars[7]",
   "y[8]": "x_vars[8]",
   "y[9]": "x_vars[9]",
   "y[10]": "x_vars[10]",
   "y[11]": "x_vars[11]",
   "y[12]": "x_vars[12]",
   "y[13]": "x_vars[13]",
   "y[14]": "x_vars[14]",
   "y[15]": "x_vars[15]",
   "y[16]": "x_vars[16]",
   "y[17]": "x_vars[17]",
   "y[18]": "x_vars[18]",
   "y[19]": "x_vars[19]",
   "y[20]": "x_vars[20]",
   "y[21]": "x_vars[21]",
   "y[22]": "x_vars[22]",
   "y[23]": "x_vars[23]",
   "y[24]": "x_vars[24]",
   "y[25]": "x_vars[25]",
   "y[26]": "x_vars[26]",
   "y[27]": "x_vars[27]",
   "y[28]": "x_vars[28]",
   "y[29]": "x_vars[29]",
   "y[30]": "x_vars[30]",
   "y[31]": "x_vars[31]",
   "y[32]": "x_vars[32]",
   "y[33]": "x_vars[33]",
   "y[34]": "x_vars[34]",
   "y[35]": "x_vars[35]",
   "y[36]": "x_vars[36]",
   "y[37]": "x_vars[37]",
   "y[38]": "x_vars[38]",
   "y[39]": "x_vars[39]",
   "y[40]": "x_vars[40]",
   "y[41]": "x_vars[41]",
   "y[42]": "x_vars[42]",
   "y[43]": "x_vars[43]",
   "y[44]": "x_vars[44]",
   "y[45]": "x_vars[45]",
   "y[46]": "x_vars[46]",
   "y[47]": "x_vars[47]",
   "y[48]": "x_vars[48]",
   "y[49]": "x_vars[49]",
   "y[50]": "x_vars[50]",
   "y[51]": "x_vars[51]",
   "y[52]": "x_vars[52]",
   "y[53]": "x_vars[53]",
   "y[54]": "x_vars[54]",
   "y[55]": "x_vars[55]",
   "y[56]": "x_vars[56]",
   "y[57]": "x_vars[57]",
   "y[58]": "x_vars[58]",
   "y[59]": "x_vars[59]",
   "y[60]": "x_vars[60]",
   "y[61]": "x_vars[61]",
   "y[62]": "x_vars[62]",
   "y[63]": "x_vars[63]",
   "y[64]": "x_vars[64]",
   "y[65]": "x_vars[65]",
   "y[66]": "x_vars[66]",
   "y[67]": "x_vars[67]",
   "y[68]": "x_vars[68]",
   "y[69]": "x_vars[69]",
   "y[70]": "x_vars[70]",
   "y[71]": "x_vars[71]",
   "y[72]": "x_vars[72]",
   "y[73]": "x_vars[73]",
   "y[74]": "x_vars[74]",
   "y[75]": "x_vars[75]",
   "y[76]": "x_vars[76]",
   "y[77]": "x_vars[77]",
   "y[78]": "x_vars[78]",
   "y[79]": "x_vars[79]",
   "y[80]": "x_vars[80]",
   "y[81]": "x_vars[81]",
   "y[82]": "x_vars[82]",
   "y[83]": "x_vars[83]",
   "y[84]": "x_vars[84]",
   "y[85]": "x_vars[85]",
   "y[86]": "x_vars[86]",
   "y[87]": "x_vars[87]",
   "y[88]": "x_vars[88]",
   "y[89]": "x_vars[89]",
   "y[90]": "x_vars[90]",
   "y[91]": "x_vars[91]",
   "y[92]": "x_vars[92]",
   "y[93]": "x_vars[93]",
   "y[94]": "x_vars[94]",
   "y[95]": "x_vars[95]",
   "y[96]": "x_vars[96]",
   "y[97]": "x_vars[97]",
   "y[98]": "x_vars[98]",
   "y[99]": "x_vars[99]",
   "y[100]": "x_vars[100]",
   "y[101]": "x_vars[101]",
   "y[102]": "x_vars[102]",
   "y[103]": "x_vars[103]",
   "y[104]": "x_vars[104]",
   "y[105]": "x_vars[105]",
   "y[106]": "x_vars[106]",
   "y[107]": "x_vars[107]",
   "y[108]": "x_vars[108]",
   "y[109]": "x_vars[109]",
   "y[110]": "x_vars[110]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-076": {
  "source_code_sha256": "8586b246c8b3735974414f9143dd6e45dd8894cc15fbbd79a8982f64c5faa447",
  "original_solution_sha256": "e6aa3a71e03ae9a888785a19494feb3d9c0eafeb3531f4229213a9738efe876f",
  "bindings": [
   [
    "C0",
    "x_vars[W1,C1]"
   ],
   [
    "C1",
    "x_vars[W1,C2]"
   ],
   [
    "C2",
    "x_vars[W1,C3]"
   ],
   [
    "C3",
    "x_vars[W1,C4]"
   ],
   [
    "C4",
    "x_vars[W1,C5]"
   ],
   [
    "C5",
    "x_vars[W1,C6]"
   ],
   [
    "C6",
    "x_vars[W1,C7]"
   ],
   [
    "C7",
    "x_vars[W1,C8]"
   ],
   [
    "C8",
    "x_vars[W1,C9]"
   ],
   [
    "C9",
    "x_vars[W1,C10]"
   ],
   [
    "C10",
    "x_vars[W1,C11]"
   ],
   [
    "C11",
    "x_vars[W1,C12]"
   ],
   [
    "C12",
    "x_vars[W1,C13]"
   ],
   [
    "C13",
    "x_vars[W1,C14]"
   ],
   [
    "C14",
    "x_vars[W1,C15]"
   ],
   [
    "C15",
    "x_vars[W1,C16]"
   ],
   [
    "C16",
    "x_vars[W1,C17]"
   ],
   [
    "C17",
    "x_vars[W1,C18]"
   ],
   [
    "C18",
    "x_vars[W1,C19]"
   ],
   [
    "C19",
    "x_vars[W1,C20]"
   ],
   [
    "C20",
    "x_vars[W2,C1]"
   ],
   [
    "C21",
    "x_vars[W2,C2]"
   ],
   [
    "C22",
    "x_vars[W2,C3]"
   ],
   [
    "C23",
    "x_vars[W2,C4]"
   ],
   [
    "C24",
    "x_vars[W2,C5]"
   ],
   [
    "C25",
    "x_vars[W2,C6]"
   ],
   [
    "C26",
    "x_vars[W2,C7]"
   ],
   [
    "C27",
    "x_vars[W2,C8]"
   ],
   [
    "C28",
    "x_vars[W2,C9]"
   ],
   [
    "C29",
    "x_vars[W2,C10]"
   ],
   [
    "C30",
    "x_vars[W2,C11]"
   ],
   [
    "C31",
    "x_vars[W2,C12]"
   ],
   [
    "C32",
    "x_vars[W2,C13]"
   ],
   [
    "C33",
    "x_vars[W2,C14]"
   ],
   [
    "C34",
    "x_vars[W2,C15]"
   ],
   [
    "C35",
    "x_vars[W2,C16]"
   ],
   [
    "C36",
    "x_vars[W2,C17]"
   ],
   [
    "C37",
    "x_vars[W2,C18]"
   ],
   [
    "C38",
    "x_vars[W2,C19]"
   ],
   [
    "C39",
    "x_vars[W2,C20]"
   ],
   [
    "C40",
    "x_vars[W3,C1]"
   ],
   [
    "C41",
    "x_vars[W3,C2]"
   ],
   [
    "C42",
    "x_vars[W3,C3]"
   ],
   [
    "C43",
    "x_vars[W3,C4]"
   ],
   [
    "C44",
    "x_vars[W3,C5]"
   ],
   [
    "C45",
    "x_vars[W3,C6]"
   ],
   [
    "C46",
    "x_vars[W3,C7]"
   ],
   [
    "C47",
    "x_vars[W3,C8]"
   ],
   [
    "C48",
    "x_vars[W3,C9]"
   ],
   [
    "C49",
    "x_vars[W3,C10]"
   ],
   [
    "C50",
    "x_vars[W3,C11]"
   ],
   [
    "C51",
    "x_vars[W3,C12]"
   ],
   [
    "C52",
    "x_vars[W3,C13]"
   ],
   [
    "C53",
    "x_vars[W3,C14]"
   ],
   [
    "C54",
    "x_vars[W3,C15]"
   ],
   [
    "C55",
    "x_vars[W3,C16]"
   ],
   [
    "C56",
    "x_vars[W3,C17]"
   ],
   [
    "C57",
    "x_vars[W3,C18]"
   ],
   [
    "C58",
    "x_vars[W3,C19]"
   ],
   [
    "C59",
    "x_vars[W3,C20]"
   ],
   [
    "C60",
    "x_vars[W4,C1]"
   ],
   [
    "C61",
    "x_vars[W4,C2]"
   ],
   [
    "C62",
    "x_vars[W4,C3]"
   ],
   [
    "C63",
    "x_vars[W4,C4]"
   ],
   [
    "C64",
    "x_vars[W4,C5]"
   ],
   [
    "C65",
    "x_vars[W4,C6]"
   ],
   [
    "C66",
    "x_vars[W4,C7]"
   ],
   [
    "C67",
    "x_vars[W4,C8]"
   ],
   [
    "C68",
    "x_vars[W4,C9]"
   ],
   [
    "C69",
    "x_vars[W4,C10]"
   ],
   [
    "C70",
    "x_vars[W4,C11]"
   ],
   [
    "C71",
    "x_vars[W4,C12]"
   ],
   [
    "C72",
    "x_vars[W4,C13]"
   ],
   [
    "C73",
    "x_vars[W4,C14]"
   ],
   [
    "C74",
    "x_vars[W4,C15]"
   ],
   [
    "C75",
    "x_vars[W4,C16]"
   ],
   [
    "C76",
    "x_vars[W4,C17]"
   ],
   [
    "C77",
    "x_vars[W4,C18]"
   ],
   [
    "C78",
    "x_vars[W4,C19]"
   ],
   [
    "C79",
    "x_vars[W4,C20]"
   ],
   [
    "C80",
    "x_vars[W5,C1]"
   ],
   [
    "C81",
    "x_vars[W5,C2]"
   ],
   [
    "C82",
    "x_vars[W5,C3]"
   ],
   [
    "C83",
    "x_vars[W5,C4]"
   ],
   [
    "C84",
    "x_vars[W5,C5]"
   ],
   [
    "C85",
    "x_vars[W5,C6]"
   ],
   [
    "C86",
    "x_vars[W5,C7]"
   ],
   [
    "C87",
    "x_vars[W5,C8]"
   ],
   [
    "C88",
    "x_vars[W5,C9]"
   ],
   [
    "C89",
    "x_vars[W5,C10]"
   ],
   [
    "C90",
    "x_vars[W5,C11]"
   ],
   [
    "C91",
    "x_vars[W5,C12]"
   ],
   [
    "C92",
    "x_vars[W5,C13]"
   ],
   [
    "C93",
    "x_vars[W5,C14]"
   ],
   [
    "C94",
    "x_vars[W5,C15]"
   ],
   [
    "C95",
    "x_vars[W5,C16]"
   ],
   [
    "C96",
    "x_vars[W5,C17]"
   ],
   [
    "C97",
    "x_vars[W5,C18]"
   ],
   [
    "C98",
    "x_vars[W5,C19]"
   ],
   [
    "C99",
    "x_vars[W5,C20]"
   ],
   [
    "C100",
    "x_vars[W6,C1]"
   ],
   [
    "C101",
    "x_vars[W6,C2]"
   ],
   [
    "C102",
    "x_vars[W6,C3]"
   ],
   [
    "C103",
    "x_vars[W6,C4]"
   ],
   [
    "C104",
    "x_vars[W6,C5]"
   ],
   [
    "C105",
    "x_vars[W6,C6]"
   ],
   [
    "C106",
    "x_vars[W6,C7]"
   ],
   [
    "C107",
    "x_vars[W6,C8]"
   ],
   [
    "C108",
    "x_vars[W6,C9]"
   ],
   [
    "C109",
    "x_vars[W6,C10]"
   ],
   [
    "C110",
    "x_vars[W6,C11]"
   ],
   [
    "C111",
    "x_vars[W6,C12]"
   ],
   [
    "C112",
    "x_vars[W6,C13]"
   ],
   [
    "C113",
    "x_vars[W6,C14]"
   ],
   [
    "C114",
    "x_vars[W6,C15]"
   ],
   [
    "C115",
    "x_vars[W6,C16]"
   ],
   [
    "C116",
    "x_vars[W6,C17]"
   ],
   [
    "C117",
    "x_vars[W6,C18]"
   ],
   [
    "C118",
    "x_vars[W6,C19]"
   ],
   [
    "C119",
    "x_vars[W6,C20]"
   ],
   [
    "C120",
    "x_vars[W7,C1]"
   ],
   [
    "C121",
    "x_vars[W7,C2]"
   ],
   [
    "C122",
    "x_vars[W7,C3]"
   ],
   [
    "C123",
    "x_vars[W7,C4]"
   ],
   [
    "C124",
    "x_vars[W7,C5]"
   ],
   [
    "C125",
    "x_vars[W7,C6]"
   ],
   [
    "C126",
    "x_vars[W7,C7]"
   ],
   [
    "C127",
    "x_vars[W7,C8]"
   ],
   [
    "C128",
    "x_vars[W7,C9]"
   ],
   [
    "C129",
    "x_vars[W7,C10]"
   ],
   [
    "C130",
    "x_vars[W7,C11]"
   ],
   [
    "C131",
    "x_vars[W7,C12]"
   ],
   [
    "C132",
    "x_vars[W7,C13]"
   ],
   [
    "C133",
    "x_vars[W7,C14]"
   ],
   [
    "C134",
    "x_vars[W7,C15]"
   ],
   [
    "C135",
    "x_vars[W7,C16]"
   ],
   [
    "C136",
    "x_vars[W7,C17]"
   ],
   [
    "C137",
    "x_vars[W7,C18]"
   ],
   [
    "C138",
    "x_vars[W7,C19]"
   ],
   [
    "C139",
    "x_vars[W7,C20]"
   ],
   [
    "C140",
    "x_vars[W8,C1]"
   ],
   [
    "C141",
    "x_vars[W8,C2]"
   ],
   [
    "C142",
    "x_vars[W8,C3]"
   ],
   [
    "C143",
    "x_vars[W8,C4]"
   ],
   [
    "C144",
    "x_vars[W8,C5]"
   ],
   [
    "C145",
    "x_vars[W8,C6]"
   ],
   [
    "C146",
    "x_vars[W8,C7]"
   ],
   [
    "C147",
    "x_vars[W8,C8]"
   ],
   [
    "C148",
    "x_vars[W8,C9]"
   ],
   [
    "C149",
    "x_vars[W8,C10]"
   ],
   [
    "C150",
    "x_vars[W8,C11]"
   ],
   [
    "C151",
    "x_vars[W8,C12]"
   ],
   [
    "C152",
    "x_vars[W8,C13]"
   ],
   [
    "C153",
    "x_vars[W8,C14]"
   ],
   [
    "C154",
    "x_vars[W8,C15]"
   ],
   [
    "C155",
    "x_vars[W8,C16]"
   ],
   [
    "C156",
    "x_vars[W8,C17]"
   ],
   [
    "C157",
    "x_vars[W8,C18]"
   ],
   [
    "C158",
    "x_vars[W8,C19]"
   ],
   [
    "C159",
    "x_vars[W8,C20]"
   ],
   [
    "C160",
    "x_vars[W9,C1]"
   ],
   [
    "C161",
    "x_vars[W9,C2]"
   ],
   [
    "C162",
    "x_vars[W9,C3]"
   ],
   [
    "C163",
    "x_vars[W9,C4]"
   ],
   [
    "C164",
    "x_vars[W9,C5]"
   ],
   [
    "C165",
    "x_vars[W9,C6]"
   ],
   [
    "C166",
    "x_vars[W9,C7]"
   ],
   [
    "C167",
    "x_vars[W9,C8]"
   ],
   [
    "C168",
    "x_vars[W9,C9]"
   ],
   [
    "C169",
    "x_vars[W9,C10]"
   ],
   [
    "C170",
    "x_vars[W9,C11]"
   ],
   [
    "C171",
    "x_vars[W9,C12]"
   ],
   [
    "C172",
    "x_vars[W9,C13]"
   ],
   [
    "C173",
    "x_vars[W9,C14]"
   ],
   [
    "C174",
    "x_vars[W9,C15]"
   ],
   [
    "C175",
    "x_vars[W9,C16]"
   ],
   [
    "C176",
    "x_vars[W9,C17]"
   ],
   [
    "C177",
    "x_vars[W9,C18]"
   ],
   [
    "C178",
    "x_vars[W9,C19]"
   ],
   [
    "C179",
    "x_vars[W9,C20]"
   ],
   [
    "C180",
    "x_vars[W10,C1]"
   ],
   [
    "C181",
    "x_vars[W10,C2]"
   ],
   [
    "C182",
    "x_vars[W10,C3]"
   ],
   [
    "C183",
    "x_vars[W10,C4]"
   ],
   [
    "C184",
    "x_vars[W10,C5]"
   ],
   [
    "C185",
    "x_vars[W10,C6]"
   ],
   [
    "C186",
    "x_vars[W10,C7]"
   ],
   [
    "C187",
    "x_vars[W10,C8]"
   ],
   [
    "C188",
    "x_vars[W10,C9]"
   ],
   [
    "C189",
    "x_vars[W10,C10]"
   ],
   [
    "C190",
    "x_vars[W10,C11]"
   ],
   [
    "C191",
    "x_vars[W10,C12]"
   ],
   [
    "C192",
    "x_vars[W10,C13]"
   ],
   [
    "C193",
    "x_vars[W10,C14]"
   ],
   [
    "C194",
    "x_vars[W10,C15]"
   ],
   [
    "C195",
    "x_vars[W10,C16]"
   ],
   [
    "C196",
    "x_vars[W10,C17]"
   ],
   [
    "C197",
    "x_vars[W10,C18]"
   ],
   [
    "C198",
    "x_vars[W10,C19]"
   ],
   [
    "C199",
    "x_vars[W10,C20]"
   ],
   [
    "C200",
    "y_vars[W1]"
   ],
   [
    "C201",
    "y_vars[W2]"
   ],
   [
    "C202",
    "y_vars[W3]"
   ],
   [
    "C203",
    "y_vars[W4]"
   ],
   [
    "C204",
    "y_vars[W5]"
   ],
   [
    "C205",
    "y_vars[W6]"
   ],
   [
    "C206",
    "y_vars[W7]"
   ],
   [
    "C207",
    "y_vars[W8]"
   ],
   [
    "C208",
    "y_vars[W9]"
   ],
   [
    "C209",
    "y_vars[W10]"
   ]
  ],
  "primary_mapping": {
   "y[W1]": "y_vars[W1]",
   "y[W2]": "y_vars[W2]",
   "y[W3]": "y_vars[W3]",
   "y[W4]": "y_vars[W4]",
   "y[W5]": "y_vars[W5]",
   "y[W6]": "y_vars[W6]",
   "y[W7]": "y_vars[W7]",
   "y[W8]": "y_vars[W8]",
   "y[W9]": "y_vars[W9]",
   "y[W10]": "y_vars[W10]",
   "s[W1,C1]": "x_vars[W1,C1]",
   "s[W1,C11]": "x_vars[W1,C11]",
   "s[W1,C12]": "x_vars[W1,C12]",
   "s[W1,C13]": "x_vars[W1,C13]",
   "s[W1,C14]": "x_vars[W1,C14]",
   "s[W1,C15]": "x_vars[W1,C15]",
   "s[W1,C16]": "x_vars[W1,C16]",
   "s[W1,C17]": "x_vars[W1,C17]",
   "s[W1,C18]": "x_vars[W1,C18]",
   "s[W1,C19]": "x_vars[W1,C19]",
   "s[W1,C20]": "x_vars[W1,C20]",
   "s[W2,C2]": "x_vars[W2,C2]",
   "s[W2,C11]": "x_vars[W2,C11]",
   "s[W2,C12]": "x_vars[W2,C12]",
   "s[W2,C13]": "x_vars[W2,C13]",
   "s[W2,C14]": "x_vars[W2,C14]",
   "s[W2,C15]": "x_vars[W2,C15]",
   "s[W2,C16]": "x_vars[W2,C16]",
   "s[W2,C17]": "x_vars[W2,C17]",
   "s[W2,C18]": "x_vars[W2,C18]",
   "s[W2,C19]": "x_vars[W2,C19]",
   "s[W2,C20]": "x_vars[W2,C20]",
   "s[W3,C3]": "x_vars[W3,C3]",
   "s[W3,C11]": "x_vars[W3,C11]",
   "s[W3,C12]": "x_vars[W3,C12]",
   "s[W3,C13]": "x_vars[W3,C13]",
   "s[W3,C14]": "x_vars[W3,C14]",
   "s[W3,C15]": "x_vars[W3,C15]",
   "s[W3,C16]": "x_vars[W3,C16]",
   "s[W3,C17]": "x_vars[W3,C17]",
   "s[W3,C18]": "x_vars[W3,C18]",
   "s[W3,C19]": "x_vars[W3,C19]",
   "s[W3,C20]": "x_vars[W3,C20]",
   "s[W4,C4]": "x_vars[W4,C4]",
   "s[W4,C11]": "x_vars[W4,C11]",
   "s[W4,C12]": "x_vars[W4,C12]",
   "s[W4,C13]": "x_vars[W4,C13]",
   "s[W4,C14]": "x_vars[W4,C14]",
   "s[W4,C15]": "x_vars[W4,C15]",
   "s[W4,C16]": "x_vars[W4,C16]",
   "s[W4,C17]": "x_vars[W4,C17]",
   "s[W4,C18]": "x_vars[W4,C18]",
   "s[W4,C19]": "x_vars[W4,C19]",
   "s[W4,C20]": "x_vars[W4,C20]",
   "s[W5,C5]": "x_vars[W5,C5]",
   "s[W5,C11]": "x_vars[W5,C11]",
   "s[W5,C12]": "x_vars[W5,C12]",
   "s[W5,C13]": "x_vars[W5,C13]",
   "s[W5,C14]": "x_vars[W5,C14]",
   "s[W5,C15]": "x_vars[W5,C15]",
   "s[W5,C16]": "x_vars[W5,C16]",
   "s[W5,C17]": "x_vars[W5,C17]",
   "s[W5,C18]": "x_vars[W5,C18]",
   "s[W5,C19]": "x_vars[W5,C19]",
   "s[W5,C20]": "x_vars[W5,C20]",
   "s[W6,C6]": "x_vars[W6,C6]",
   "s[W6,C11]": "x_vars[W6,C11]",
   "s[W6,C12]": "x_vars[W6,C12]",
   "s[W6,C13]": "x_vars[W6,C13]",
   "s[W6,C14]": "x_vars[W6,C14]",
   "s[W6,C15]": "x_vars[W6,C15]",
   "s[W6,C16]": "x_vars[W6,C16]",
   "s[W6,C17]": "x_vars[W6,C17]",
   "s[W6,C18]": "x_vars[W6,C18]",
   "s[W6,C19]": "x_vars[W6,C19]",
   "s[W6,C20]": "x_vars[W6,C20]",
   "s[W7,C7]": "x_vars[W7,C7]",
   "s[W7,C11]": "x_vars[W7,C11]",
   "s[W7,C12]": "x_vars[W7,C12]",
   "s[W7,C13]": "x_vars[W7,C13]",
   "s[W7,C14]": "x_vars[W7,C14]",
   "s[W7,C15]": "x_vars[W7,C15]",
   "s[W7,C16]": "x_vars[W7,C16]",
   "s[W7,C17]": "x_vars[W7,C17]",
   "s[W7,C18]": "x_vars[W7,C18]",
   "s[W7,C19]": "x_vars[W7,C19]",
   "s[W7,C20]": "x_vars[W7,C20]",
   "s[W8,C8]": "x_vars[W8,C8]",
   "s[W8,C11]": "x_vars[W8,C11]",
   "s[W8,C12]": "x_vars[W8,C12]",
   "s[W8,C13]": "x_vars[W8,C13]",
   "s[W8,C14]": "x_vars[W8,C14]",
   "s[W8,C15]": "x_vars[W8,C15]",
   "s[W8,C16]": "x_vars[W8,C16]",
   "s[W8,C17]": "x_vars[W8,C17]",
   "s[W8,C18]": "x_vars[W8,C18]",
   "s[W8,C19]": "x_vars[W8,C19]",
   "s[W8,C20]": "x_vars[W8,C20]",
   "s[W9,C9]": "x_vars[W9,C9]",
   "s[W9,C11]": "x_vars[W9,C11]",
   "s[W9,C12]": "x_vars[W9,C12]",
   "s[W9,C13]": "x_vars[W9,C13]",
   "s[W9,C14]": "x_vars[W9,C14]",
   "s[W9,C15]": "x_vars[W9,C15]",
   "s[W9,C16]": "x_vars[W9,C16]",
   "s[W9,C17]": "x_vars[W9,C17]",
   "s[W9,C18]": "x_vars[W9,C18]",
   "s[W9,C19]": "x_vars[W9,C19]",
   "s[W9,C20]": "x_vars[W9,C20]",
   "s[W10,C10]": "x_vars[W10,C10]",
   "s[W10,C11]": "x_vars[W10,C11]",
   "s[W10,C12]": "x_vars[W10,C12]",
   "s[W10,C13]": "x_vars[W10,C13]",
   "s[W10,C14]": "x_vars[W10,C14]",
   "s[W10,C15]": "x_vars[W10,C15]",
   "s[W10,C16]": "x_vars[W10,C16]",
   "s[W10,C17]": "x_vars[W10,C17]",
   "s[W10,C18]": "x_vars[W10,C18]",
   "s[W10,C19]": "x_vars[W10,C19]",
   "s[W10,C20]": "x_vars[W10,C20]",
   "s[W1,C2]": "x_vars[W1,C2]",
   "s[W1,C3]": "x_vars[W1,C3]",
   "s[W1,C4]": "x_vars[W1,C4]",
   "s[W1,C5]": "x_vars[W1,C5]",
   "s[W1,C6]": "x_vars[W1,C6]",
   "s[W1,C7]": "x_vars[W1,C7]",
   "s[W1,C8]": "x_vars[W1,C8]",
   "s[W1,C9]": "x_vars[W1,C9]",
   "s[W1,C10]": "x_vars[W1,C10]",
   "s[W2,C1]": "x_vars[W2,C1]",
   "s[W2,C3]": "x_vars[W2,C3]",
   "s[W2,C4]": "x_vars[W2,C4]",
   "s[W2,C5]": "x_vars[W2,C5]",
   "s[W2,C6]": "x_vars[W2,C6]",
   "s[W2,C7]": "x_vars[W2,C7]",
   "s[W2,C8]": "x_vars[W2,C8]",
   "s[W2,C9]": "x_vars[W2,C9]",
   "s[W2,C10]": "x_vars[W2,C10]",
   "s[W3,C1]": "x_vars[W3,C1]",
   "s[W3,C2]": "x_vars[W3,C2]",
   "s[W3,C4]": "x_vars[W3,C4]",
   "s[W3,C5]": "x_vars[W3,C5]",
   "s[W3,C6]": "x_vars[W3,C6]",
   "s[W3,C7]": "x_vars[W3,C7]",
   "s[W3,C8]": "x_vars[W3,C8]",
   "s[W3,C9]": "x_vars[W3,C9]",
   "s[W3,C10]": "x_vars[W3,C10]",
   "s[W4,C1]": "x_vars[W4,C1]",
   "s[W4,C2]": "x_vars[W4,C2]",
   "s[W4,C3]": "x_vars[W4,C3]",
   "s[W4,C5]": "x_vars[W4,C5]",
   "s[W4,C6]": "x_vars[W4,C6]",
   "s[W4,C7]": "x_vars[W4,C7]",
   "s[W4,C8]": "x_vars[W4,C8]",
   "s[W4,C9]": "x_vars[W4,C9]",
   "s[W4,C10]": "x_vars[W4,C10]",
   "s[W5,C1]": "x_vars[W5,C1]",
   "s[W5,C2]": "x_vars[W5,C2]",
   "s[W5,C3]": "x_vars[W5,C3]",
   "s[W5,C4]": "x_vars[W5,C4]",
   "s[W5,C6]": "x_vars[W5,C6]",
   "s[W5,C7]": "x_vars[W5,C7]",
   "s[W5,C8]": "x_vars[W5,C8]",
   "s[W5,C9]": "x_vars[W5,C9]",
   "s[W5,C10]": "x_vars[W5,C10]",
   "s[W6,C1]": "x_vars[W6,C1]",
   "s[W6,C2]": "x_vars[W6,C2]",
   "s[W6,C3]": "x_vars[W6,C3]",
   "s[W6,C4]": "x_vars[W6,C4]",
   "s[W6,C5]": "x_vars[W6,C5]",
   "s[W6,C7]": "x_vars[W6,C7]",
   "s[W6,C8]": "x_vars[W6,C8]",
   "s[W6,C9]": "x_vars[W6,C9]",
   "s[W6,C10]": "x_vars[W6,C10]",
   "s[W7,C1]": "x_vars[W7,C1]",
   "s[W7,C2]": "x_vars[W7,C2]",
   "s[W7,C3]": "x_vars[W7,C3]",
   "s[W7,C4]": "x_vars[W7,C4]",
   "s[W7,C5]": "x_vars[W7,C5]",
   "s[W7,C6]": "x_vars[W7,C6]",
   "s[W7,C8]": "x_vars[W7,C8]",
   "s[W7,C9]": "x_vars[W7,C9]",
   "s[W7,C10]": "x_vars[W7,C10]",
   "s[W8,C1]": "x_vars[W8,C1]",
   "s[W8,C2]": "x_vars[W8,C2]",
   "s[W8,C3]": "x_vars[W8,C3]",
   "s[W8,C4]": "x_vars[W8,C4]",
   "s[W8,C5]": "x_vars[W8,C5]",
   "s[W8,C6]": "x_vars[W8,C6]",
   "s[W8,C7]": "x_vars[W8,C7]",
   "s[W8,C9]": "x_vars[W8,C9]",
   "s[W8,C10]": "x_vars[W8,C10]",
   "s[W9,C1]": "x_vars[W9,C1]",
   "s[W9,C2]": "x_vars[W9,C2]",
   "s[W9,C3]": "x_vars[W9,C3]",
   "s[W9,C4]": "x_vars[W9,C4]",
   "s[W9,C5]": "x_vars[W9,C5]",
   "s[W9,C6]": "x_vars[W9,C6]",
   "s[W9,C7]": "x_vars[W9,C7]",
   "s[W9,C8]": "x_vars[W9,C8]",
   "s[W9,C10]": "x_vars[W9,C10]",
   "s[W10,C1]": "x_vars[W10,C1]",
   "s[W10,C2]": "x_vars[W10,C2]",
   "s[W10,C3]": "x_vars[W10,C3]",
   "s[W10,C4]": "x_vars[W10,C4]",
   "s[W10,C5]": "x_vars[W10,C5]",
   "s[W10,C6]": "x_vars[W10,C6]",
   "s[W10,C7]": "x_vars[W10,C7]",
   "s[W10,C8]": "x_vars[W10,C8]",
   "s[W10,C9]": "x_vars[W10,C9]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-077": {
  "source_code_sha256": "02c2d048b137a990def3a5b6aea55e97a0a13cc023b651a85ded2edbbad9cfa0",
  "original_solution_sha256": "2adcbede4949f9e7827335fe535c6007bd8de9da543c3e67286e38a1281ee354",
  "bindings": [
   [
    "C0",
    "x_vars[1]"
   ],
   [
    "C1",
    "x_vars[2]"
   ],
   [
    "C2",
    "x_vars[3]"
   ],
   [
    "C3",
    "x_vars[4]"
   ],
   [
    "C4",
    "x_vars[5]"
   ],
   [
    "C5",
    "x_vars[6]"
   ],
   [
    "C6",
    "x_vars[7]"
   ],
   [
    "C7",
    "x_vars[8]"
   ],
   [
    "C8",
    "x_vars[9]"
   ],
   [
    "C9",
    "x_vars[10]"
   ],
   [
    "C10",
    "x_vars[11]"
   ],
   [
    "C11",
    "x_vars[12]"
   ],
   [
    "C12",
    "x_vars[13]"
   ],
   [
    "C13",
    "x_vars[14]"
   ],
   [
    "C14",
    "x_vars[15]"
   ],
   [
    "C15",
    "x_vars[16]"
   ],
   [
    "C16",
    "x_vars[17]"
   ],
   [
    "C17",
    "x_vars[18]"
   ],
   [
    "C18",
    "x_vars[19]"
   ],
   [
    "C19",
    "x_vars[20]"
   ],
   [
    "C20",
    "x_vars[21]"
   ],
   [
    "C21",
    "x_vars[22]"
   ],
   [
    "C22",
    "x_vars[23]"
   ],
   [
    "C23",
    "x_vars[24]"
   ],
   [
    "C24",
    "x_vars[25]"
   ],
   [
    "C25",
    "x_vars[26]"
   ],
   [
    "C26",
    "x_vars[27]"
   ],
   [
    "C27",
    "x_vars[28]"
   ],
   [
    "C28",
    "x_vars[29]"
   ],
   [
    "C29",
    "x_vars[30]"
   ],
   [
    "C30",
    "x_vars[31]"
   ],
   [
    "C31",
    "x_vars[32]"
   ],
   [
    "C32",
    "x_vars[33]"
   ],
   [
    "C33",
    "x_vars[34]"
   ],
   [
    "C34",
    "x_vars[35]"
   ],
   [
    "C35",
    "x_vars[36]"
   ],
   [
    "C36",
    "x_vars[37]"
   ],
   [
    "C37",
    "x_vars[38]"
   ],
   [
    "C38",
    "x_vars[39]"
   ],
   [
    "C39",
    "x_vars[40]"
   ],
   [
    "C40",
    "x_vars[41]"
   ],
   [
    "C41",
    "x_vars[42]"
   ],
   [
    "C42",
    "x_vars[43]"
   ],
   [
    "C43",
    "x_vars[44]"
   ],
   [
    "C44",
    "x_vars[45]"
   ],
   [
    "C45",
    "x_vars[46]"
   ],
   [
    "C46",
    "x_vars[47]"
   ],
   [
    "C47",
    "x_vars[48]"
   ],
   [
    "C48",
    "x_vars[49]"
   ],
   [
    "C49",
    "x_vars[50]"
   ],
   [
    "C50",
    "x_vars[51]"
   ],
   [
    "C51",
    "x_vars[52]"
   ],
   [
    "C52",
    "x_vars[53]"
   ],
   [
    "C53",
    "x_vars[54]"
   ],
   [
    "C54",
    "x_vars[55]"
   ],
   [
    "C55",
    "x_vars[56]"
   ],
   [
    "C56",
    "x_vars[57]"
   ],
   [
    "C57",
    "x_vars[58]"
   ],
   [
    "C58",
    "x_vars[59]"
   ],
   [
    "C59",
    "x_vars[60]"
   ],
   [
    "C60",
    "x_vars[61]"
   ],
   [
    "C61",
    "x_vars[62]"
   ],
   [
    "C62",
    "x_vars[63]"
   ],
   [
    "C63",
    "x_vars[64]"
   ],
   [
    "C64",
    "x_vars[65]"
   ],
   [
    "C65",
    "x_vars[66]"
   ],
   [
    "C66",
    "x_vars[67]"
   ],
   [
    "C67",
    "x_vars[68]"
   ],
   [
    "C68",
    "x_vars[69]"
   ],
   [
    "C69",
    "x_vars[70]"
   ],
   [
    "C70",
    "x_vars[71]"
   ],
   [
    "C71",
    "x_vars[72]"
   ],
   [
    "C72",
    "x_vars[73]"
   ],
   [
    "C73",
    "x_vars[74]"
   ],
   [
    "C74",
    "x_vars[75]"
   ],
   [
    "C75",
    "x_vars[76]"
   ],
   [
    "C76",
    "x_vars[77]"
   ],
   [
    "C77",
    "x_vars[78]"
   ],
   [
    "C78",
    "x_vars[79]"
   ],
   [
    "C79",
    "x_vars[80]"
   ],
   [
    "C80",
    "x_vars[81]"
   ],
   [
    "C81",
    "x_vars[82]"
   ],
   [
    "C82",
    "x_vars[83]"
   ],
   [
    "C83",
    "x_vars[84]"
   ],
   [
    "C84",
    "x_vars[85]"
   ],
   [
    "C85",
    "x_vars[86]"
   ],
   [
    "C86",
    "x_vars[87]"
   ],
   [
    "C87",
    "x_vars[88]"
   ],
   [
    "C88",
    "x_vars[89]"
   ],
   [
    "C89",
    "x_vars[90]"
   ],
   [
    "C90",
    "x_vars[91]"
   ],
   [
    "C91",
    "x_vars[92]"
   ],
   [
    "C92",
    "x_vars[93]"
   ],
   [
    "C93",
    "x_vars[94]"
   ],
   [
    "C94",
    "x_vars[95]"
   ],
   [
    "C95",
    "x_vars[96]"
   ],
   [
    "C96",
    "x_vars[97]"
   ],
   [
    "C97",
    "x_vars[98]"
   ],
   [
    "C98",
    "x_vars[99]"
   ],
   [
    "C99",
    "x_vars[100]"
   ],
   [
    "C100",
    "x_vars[101]"
   ],
   [
    "C101",
    "x_vars[102]"
   ],
   [
    "C102",
    "x_vars[103]"
   ],
   [
    "C103",
    "x_vars[104]"
   ],
   [
    "C104",
    "x_vars[105]"
   ],
   [
    "C105",
    "x_vars[106]"
   ],
   [
    "C106",
    "x_vars[107]"
   ],
   [
    "C107",
    "x_vars[108]"
   ],
   [
    "C108",
    "x_vars[109]"
   ],
   [
    "C109",
    "x_vars[110]"
   ],
   [
    "C110",
    "x_vars[111]"
   ],
   [
    "C111",
    "x_vars[112]"
   ],
   [
    "C112",
    "x_vars[113]"
   ],
   [
    "C113",
    "x_vars[114]"
   ],
   [
    "C114",
    "x_vars[115]"
   ],
   [
    "C115",
    "x_vars[116]"
   ],
   [
    "C116",
    "x_vars[117]"
   ],
   [
    "C117",
    "x_vars[118]"
   ],
   [
    "C118",
    "x_vars[119]"
   ],
   [
    "C119",
    "x_vars[120]"
   ],
   [
    "C120",
    "x_vars[121]"
   ],
   [
    "C121",
    "x_vars[122]"
   ],
   [
    "C122",
    "x_vars[123]"
   ],
   [
    "C123",
    "x_vars[124]"
   ],
   [
    "C124",
    "x_vars[125]"
   ],
   [
    "C125",
    "x_vars[126]"
   ],
   [
    "C126",
    "x_vars[127]"
   ],
   [
    "C127",
    "x_vars[128]"
   ],
   [
    "C128",
    "x_vars[129]"
   ],
   [
    "C129",
    "x_vars[130]"
   ],
   [
    "C130",
    "x_vars[131]"
   ],
   [
    "C131",
    "x_vars[132]"
   ],
   [
    "C132",
    "x_vars[133]"
   ],
   [
    "C133",
    "x_vars[134]"
   ],
   [
    "C134",
    "x_vars[135]"
   ],
   [
    "C135",
    "x_vars[136]"
   ],
   [
    "C136",
    "x_vars[137]"
   ],
   [
    "C137",
    "x_vars[138]"
   ],
   [
    "C138",
    "x_vars[139]"
   ],
   [
    "C139",
    "x_vars[140]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x_vars[1]",
   "x[1]": "x_vars[2]",
   "x[2]": "x_vars[3]",
   "x[3]": "x_vars[4]",
   "x[4]": "x_vars[5]",
   "x[5]": "x_vars[6]",
   "x[6]": "x_vars[7]",
   "x[7]": "x_vars[8]",
   "x[8]": "x_vars[9]",
   "x[9]": "x_vars[10]",
   "x[10]": "x_vars[11]",
   "x[11]": "x_vars[12]",
   "x[12]": "x_vars[13]",
   "x[13]": "x_vars[14]",
   "x[14]": "x_vars[15]",
   "x[15]": "x_vars[16]",
   "x[16]": "x_vars[17]",
   "x[17]": "x_vars[18]",
   "x[18]": "x_vars[19]",
   "x[19]": "x_vars[20]",
   "x[20]": "x_vars[21]",
   "x[21]": "x_vars[22]",
   "x[22]": "x_vars[23]",
   "x[23]": "x_vars[24]",
   "x[24]": "x_vars[25]",
   "x[25]": "x_vars[26]",
   "x[26]": "x_vars[27]",
   "x[27]": "x_vars[28]",
   "x[28]": "x_vars[29]",
   "x[29]": "x_vars[30]",
   "x[30]": "x_vars[31]",
   "x[31]": "x_vars[32]",
   "x[32]": "x_vars[33]",
   "x[33]": "x_vars[34]",
   "x[34]": "x_vars[35]",
   "x[35]": "x_vars[36]",
   "x[36]": "x_vars[37]",
   "x[37]": "x_vars[38]",
   "x[38]": "x_vars[39]",
   "x[39]": "x_vars[40]",
   "x[40]": "x_vars[41]",
   "x[41]": "x_vars[42]",
   "x[42]": "x_vars[43]",
   "x[43]": "x_vars[44]",
   "x[44]": "x_vars[45]",
   "x[45]": "x_vars[46]",
   "x[46]": "x_vars[47]",
   "x[47]": "x_vars[48]",
   "x[48]": "x_vars[49]",
   "x[49]": "x_vars[50]",
   "x[50]": "x_vars[51]",
   "x[51]": "x_vars[52]",
   "x[52]": "x_vars[53]",
   "x[53]": "x_vars[54]",
   "x[54]": "x_vars[55]",
   "x[55]": "x_vars[56]",
   "x[56]": "x_vars[57]",
   "x[57]": "x_vars[58]",
   "x[58]": "x_vars[59]",
   "x[59]": "x_vars[60]",
   "x[60]": "x_vars[61]",
   "x[61]": "x_vars[62]",
   "x[62]": "x_vars[63]",
   "x[63]": "x_vars[64]",
   "x[64]": "x_vars[65]",
   "x[65]": "x_vars[66]",
   "x[66]": "x_vars[67]",
   "x[67]": "x_vars[68]",
   "x[68]": "x_vars[69]",
   "x[69]": "x_vars[70]",
   "x[70]": "x_vars[71]",
   "x[71]": "x_vars[72]",
   "x[72]": "x_vars[73]",
   "x[73]": "x_vars[74]",
   "x[74]": "x_vars[75]",
   "x[75]": "x_vars[76]",
   "x[76]": "x_vars[77]",
   "x[77]": "x_vars[78]",
   "x[78]": "x_vars[79]",
   "x[79]": "x_vars[80]",
   "x[80]": "x_vars[81]",
   "x[81]": "x_vars[82]",
   "x[82]": "x_vars[83]",
   "x[83]": "x_vars[84]",
   "x[84]": "x_vars[85]",
   "x[85]": "x_vars[86]",
   "x[86]": "x_vars[87]",
   "x[87]": "x_vars[88]",
   "x[88]": "x_vars[89]",
   "x[89]": "x_vars[90]",
   "x[90]": "x_vars[91]",
   "x[91]": "x_vars[92]",
   "x[92]": "x_vars[93]",
   "x[93]": "x_vars[94]",
   "x[94]": "x_vars[95]",
   "x[95]": "x_vars[96]",
   "x[96]": "x_vars[97]",
   "x[97]": "x_vars[98]",
   "x[98]": "x_vars[99]",
   "x[99]": "x_vars[100]",
   "x[100]": "x_vars[101]",
   "x[101]": "x_vars[102]",
   "x[102]": "x_vars[103]",
   "x[103]": "x_vars[104]",
   "x[104]": "x_vars[105]",
   "x[105]": "x_vars[106]",
   "x[106]": "x_vars[107]",
   "x[107]": "x_vars[108]",
   "x[108]": "x_vars[109]",
   "x[109]": "x_vars[110]",
   "x[110]": "x_vars[111]",
   "x[111]": "x_vars[112]",
   "x[112]": "x_vars[113]",
   "x[113]": "x_vars[114]",
   "x[114]": "x_vars[115]",
   "x[115]": "x_vars[116]",
   "x[116]": "x_vars[117]",
   "x[117]": "x_vars[118]",
   "x[118]": "x_vars[119]",
   "x[119]": "x_vars[120]",
   "x[120]": "x_vars[121]",
   "x[121]": "x_vars[122]",
   "x[122]": "x_vars[123]",
   "x[123]": "x_vars[124]",
   "x[124]": "x_vars[125]",
   "x[125]": "x_vars[126]",
   "x[126]": "x_vars[127]",
   "x[127]": "x_vars[128]",
   "x[128]": "x_vars[129]",
   "x[129]": "x_vars[130]",
   "x[130]": "x_vars[131]",
   "x[131]": "x_vars[132]",
   "x[132]": "x_vars[133]",
   "x[133]": "x_vars[134]",
   "x[134]": "x_vars[135]",
   "x[135]": "x_vars[136]",
   "x[136]": "x_vars[137]",
   "x[137]": "x_vars[138]",
   "x[138]": "x_vars[139]",
   "x[139]": "x_vars[140]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-078": {
  "source_code_sha256": "12e4ebd1bc5e63f1d47e031ec4ae2d700da01ef81c4e28012d223ec4a3112e2b",
  "original_solution_sha256": "51826467d314ad5a50f20f912475ba727b2ef2efbc573b5b2ed7dae16f4cd823",
  "bindings": [
   [
    "C0",
    "x_vars[coal_001]"
   ],
   [
    "C1",
    "x_vars[coal_002]"
   ],
   [
    "C2",
    "x_vars[coal_003]"
   ],
   [
    "C3",
    "x_vars[coal_004]"
   ],
   [
    "C4",
    "x_vars[coal_005]"
   ],
   [
    "C5",
    "x_vars[coal_006]"
   ],
   [
    "C6",
    "x_vars[coal_007]"
   ],
   [
    "C7",
    "x_vars[coal_008]"
   ],
   [
    "C8",
    "x_vars[coal_009]"
   ],
   [
    "C9",
    "x_vars[coal_010]"
   ],
   [
    "C10",
    "x_vars[coal_011]"
   ],
   [
    "C11",
    "x_vars[coal_012]"
   ],
   [
    "C12",
    "x_vars[coal_013]"
   ],
   [
    "C13",
    "x_vars[coal_014]"
   ],
   [
    "C14",
    "x_vars[coal_015]"
   ],
   [
    "C15",
    "x_vars[coal_043]"
   ],
   [
    "C16",
    "x_vars[coal_044]"
   ],
   [
    "C17",
    "x_vars[coal_045]"
   ],
   [
    "C18",
    "x_vars[coal_046]"
   ],
   [
    "C19",
    "x_vars[coal_047]"
   ],
   [
    "C20",
    "x_vars[coal_048]"
   ],
   [
    "C21",
    "x_vars[coal_049]"
   ],
   [
    "C22",
    "x_vars[coal_050]"
   ],
   [
    "C23",
    "x_vars[coal_051]"
   ],
   [
    "C24",
    "x_vars[coal_091]"
   ],
   [
    "C25",
    "x_vars[coal_092]"
   ],
   [
    "C26",
    "x_vars[coal_093]"
   ],
   [
    "C27",
    "x_vars[coal_094]"
   ],
   [
    "C28",
    "x_vars[coal_095]"
   ],
   [
    "C29",
    "x_vars[coal_096]"
   ],
   [
    "C30",
    "x_vars[coal_097]"
   ],
   [
    "C31",
    "x_vars[coal_098]"
   ],
   [
    "C32",
    "x_vars[coal_099]"
   ],
   [
    "C33",
    "x_vars[coal_100]"
   ],
   [
    "C34",
    "x_vars[gas_001]"
   ],
   [
    "C35",
    "x_vars[gas_002]"
   ],
   [
    "C36",
    "x_vars[gas_003]"
   ],
   [
    "C37",
    "x_vars[gas_004]"
   ],
   [
    "C38",
    "x_vars[gas_005]"
   ],
   [
    "C39",
    "x_vars[gas_006]"
   ],
   [
    "C40",
    "x_vars[gas_007]"
   ],
   [
    "C41",
    "x_vars[gas_008]"
   ],
   [
    "C42",
    "x_vars[gas_009]"
   ],
   [
    "C43",
    "x_vars[gas_010]"
   ],
   [
    "C44",
    "x_vars[gas_011]"
   ],
   [
    "C45",
    "x_vars[gas_012]"
   ],
   [
    "C46",
    "x_vars[gas_013]"
   ],
   [
    "C47",
    "x_vars[gas_014]"
   ],
   [
    "C48",
    "x_vars[gas_015]"
   ],
   [
    "C49",
    "x_vars[gas_016]"
   ],
   [
    "C50",
    "x_vars[gas_017]"
   ],
   [
    "C51",
    "x_vars[gas_018]"
   ],
   [
    "C52",
    "x_vars[gas_019]"
   ],
   [
    "C53",
    "x_vars[gas_020]"
   ],
   [
    "C54",
    "x_vars[gas_021]"
   ],
   [
    "C55",
    "x_vars[gas_022]"
   ],
   [
    "C56",
    "x_vars[gas_023]"
   ],
   [
    "C57",
    "x_vars[gas_024]"
   ],
   [
    "C58",
    "x_vars[gas_025]"
   ],
   [
    "C59",
    "x_vars[gas_026]"
   ],
   [
    "C60",
    "x_vars[gas_027]"
   ],
   [
    "C61",
    "x_vars[gas_028]"
   ],
   [
    "C62",
    "x_vars[gas_029]"
   ],
   [
    "C63",
    "x_vars[gas_030]"
   ],
   [
    "C64",
    "x_vars[gas_031]"
   ],
   [
    "C65",
    "x_vars[gas_032]"
   ],
   [
    "C66",
    "x_vars[gas_033]"
   ],
   [
    "C67",
    "x_vars[gas_034]"
   ],
   [
    "C68",
    "x_vars[gas_035]"
   ],
   [
    "C69",
    "x_vars[gas_036]"
   ],
   [
    "C70",
    "x_vars[gas_037]"
   ],
   [
    "C71",
    "x_vars[gas_038]"
   ],
   [
    "C72",
    "x_vars[gas_039]"
   ],
   [
    "C73",
    "x_vars[gas_040]"
   ],
   [
    "C74",
    "x_vars[gas_041]"
   ],
   [
    "C75",
    "x_vars[gas_042]"
   ],
   [
    "C76",
    "x_vars[gas_043]"
   ],
   [
    "C77",
    "x_vars[gas_044]"
   ],
   [
    "C78",
    "x_vars[gas_045]"
   ],
   [
    "C79",
    "x_vars[gas_062]"
   ],
   [
    "C80",
    "x_vars[gas_063]"
   ],
   [
    "C81",
    "x_vars[gas_064]"
   ],
   [
    "C82",
    "x_vars[gas_065]"
   ],
   [
    "C83",
    "x_vars[gas_066]"
   ],
   [
    "C84",
    "x_vars[gas_067]"
   ],
   [
    "C85",
    "x_vars[gas_096]"
   ],
   [
    "C86",
    "x_vars[gas_097]"
   ],
   [
    "C87",
    "x_vars[gas_098]"
   ],
   [
    "C88",
    "x_vars[gas_099]"
   ],
   [
    "C89",
    "x_vars[gas_100]"
   ],
   [
    "C90",
    "x_vars[renewables_001]"
   ],
   [
    "C91",
    "x_vars[renewables_002]"
   ],
   [
    "C92",
    "x_vars[renewables_003]"
   ],
   [
    "C93",
    "x_vars[renewables_004]"
   ],
   [
    "C94",
    "x_vars[renewables_005]"
   ],
   [
    "C95",
    "x_vars[renewables_021]"
   ],
   [
    "C96",
    "x_vars[renewables_022]"
   ],
   [
    "C97",
    "x_vars[renewables_023]"
   ],
   [
    "C98",
    "x_vars[renewables_024]"
   ],
   [
    "C99",
    "x_vars[renewables_025]"
   ],
   [
    "C100",
    "x_vars[renewables_026]"
   ],
   [
    "C101",
    "x_vars[renewables_027]"
   ],
   [
    "C102",
    "x_vars[renewables_028]"
   ],
   [
    "C103",
    "x_vars[renewables_029]"
   ],
   [
    "C104",
    "x_vars[renewables_030]"
   ],
   [
    "C105",
    "x_vars[renewables_031]"
   ],
   [
    "C106",
    "x_vars[renewables_032]"
   ],
   [
    "C107",
    "x_vars[renewables_033]"
   ],
   [
    "C108",
    "x_vars[renewables_034]"
   ],
   [
    "C109",
    "x_vars[renewables_035]"
   ],
   [
    "C110",
    "x_vars[renewables_036]"
   ],
   [
    "C111",
    "x_vars[renewables_037]"
   ],
   [
    "C112",
    "x_vars[renewables_038]"
   ],
   [
    "C113",
    "x_vars[renewables_039]"
   ],
   [
    "C114",
    "x_vars[renewables_040]"
   ],
   [
    "C115",
    "x_vars[renewables_041]"
   ],
   [
    "C116",
    "x_vars[renewables_042]"
   ],
   [
    "C117",
    "x_vars[renewables_043]"
   ],
   [
    "C118",
    "x_vars[renewables_044]"
   ],
   [
    "C119",
    "x_vars[renewables_045]"
   ],
   [
    "C120",
    "x_vars[renewables_046]"
   ],
   [
    "C121",
    "x_vars[renewables_047]"
   ],
   [
    "C122",
    "x_vars[renewables_048]"
   ],
   [
    "C123",
    "x_vars[renewables_049]"
   ],
   [
    "C124",
    "x_vars[renewables_050]"
   ],
   [
    "C125",
    "x_vars[renewables_051]"
   ],
   [
    "C126",
    "x_vars[renewables_052]"
   ],
   [
    "C127",
    "x_vars[renewables_053]"
   ],
   [
    "C128",
    "x_vars[renewables_054]"
   ],
   [
    "C129",
    "x_vars[renewables_055]"
   ],
   [
    "C130",
    "x_vars[renewables_056]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x_vars[coal_001]",
   "x[1]": "x_vars[coal_002]",
   "x[2]": "x_vars[coal_003]",
   "x[3]": "x_vars[coal_004]",
   "x[4]": "x_vars[coal_005]",
   "x[5]": "x_vars[coal_006]",
   "x[6]": "x_vars[coal_007]",
   "x[7]": "x_vars[coal_008]",
   "x[8]": "x_vars[coal_009]",
   "x[9]": "x_vars[coal_010]",
   "x[10]": "x_vars[coal_011]",
   "x[11]": "x_vars[coal_012]",
   "x[12]": "x_vars[coal_013]",
   "x[13]": "x_vars[coal_014]",
   "x[14]": "x_vars[coal_015]",
   "x[15]": "x_vars[coal_043]",
   "x[16]": "x_vars[coal_044]",
   "x[17]": "x_vars[coal_045]",
   "x[18]": "x_vars[coal_046]",
   "x[19]": "x_vars[coal_047]",
   "x[20]": "x_vars[coal_048]",
   "x[21]": "x_vars[coal_049]",
   "x[22]": "x_vars[coal_050]",
   "x[23]": "x_vars[coal_051]",
   "x[24]": "x_vars[coal_091]",
   "x[25]": "x_vars[coal_092]",
   "x[26]": "x_vars[coal_093]",
   "x[27]": "x_vars[coal_094]",
   "x[28]": "x_vars[coal_095]",
   "x[29]": "x_vars[coal_096]",
   "x[30]": "x_vars[coal_097]",
   "x[31]": "x_vars[coal_098]",
   "x[32]": "x_vars[coal_099]",
   "x[33]": "x_vars[coal_100]",
   "x[34]": "x_vars[gas_001]",
   "x[35]": "x_vars[gas_002]",
   "x[36]": "x_vars[gas_003]",
   "x[37]": "x_vars[gas_004]",
   "x[38]": "x_vars[gas_005]",
   "x[39]": "x_vars[gas_006]",
   "x[40]": "x_vars[gas_007]",
   "x[41]": "x_vars[gas_008]",
   "x[42]": "x_vars[gas_009]",
   "x[43]": "x_vars[gas_010]",
   "x[44]": "x_vars[gas_011]",
   "x[45]": "x_vars[gas_012]",
   "x[46]": "x_vars[gas_013]",
   "x[47]": "x_vars[gas_014]",
   "x[48]": "x_vars[gas_015]",
   "x[49]": "x_vars[gas_016]",
   "x[50]": "x_vars[gas_017]",
   "x[51]": "x_vars[gas_018]",
   "x[52]": "x_vars[gas_019]",
   "x[53]": "x_vars[gas_020]",
   "x[54]": "x_vars[gas_021]",
   "x[55]": "x_vars[gas_022]",
   "x[56]": "x_vars[gas_023]",
   "x[57]": "x_vars[gas_024]",
   "x[58]": "x_vars[gas_025]",
   "x[59]": "x_vars[gas_026]",
   "x[60]": "x_vars[gas_027]",
   "x[61]": "x_vars[gas_028]",
   "x[62]": "x_vars[gas_029]",
   "x[63]": "x_vars[gas_030]",
   "x[64]": "x_vars[gas_031]",
   "x[65]": "x_vars[gas_032]",
   "x[66]": "x_vars[gas_033]",
   "x[67]": "x_vars[gas_034]",
   "x[68]": "x_vars[gas_035]",
   "x[69]": "x_vars[gas_036]",
   "x[70]": "x_vars[gas_037]",
   "x[71]": "x_vars[gas_038]",
   "x[72]": "x_vars[gas_039]",
   "x[73]": "x_vars[gas_040]",
   "x[74]": "x_vars[gas_041]",
   "x[75]": "x_vars[gas_042]",
   "x[76]": "x_vars[gas_043]",
   "x[77]": "x_vars[gas_044]",
   "x[78]": "x_vars[gas_045]",
   "x[79]": "x_vars[gas_062]",
   "x[80]": "x_vars[gas_063]",
   "x[81]": "x_vars[gas_064]",
   "x[82]": "x_vars[gas_065]",
   "x[83]": "x_vars[gas_066]",
   "x[84]": "x_vars[gas_067]",
   "x[85]": "x_vars[gas_096]",
   "x[86]": "x_vars[gas_097]",
   "x[87]": "x_vars[gas_098]",
   "x[88]": "x_vars[gas_099]",
   "x[89]": "x_vars[gas_100]",
   "x[90]": "x_vars[renewables_001]",
   "x[91]": "x_vars[renewables_002]",
   "x[92]": "x_vars[renewables_003]",
   "x[93]": "x_vars[renewables_004]",
   "x[94]": "x_vars[renewables_005]",
   "x[95]": "x_vars[renewables_021]",
   "x[96]": "x_vars[renewables_022]",
   "x[97]": "x_vars[renewables_023]",
   "x[98]": "x_vars[renewables_024]",
   "x[99]": "x_vars[renewables_025]",
   "x[100]": "x_vars[renewables_026]",
   "x[101]": "x_vars[renewables_027]",
   "x[102]": "x_vars[renewables_028]",
   "x[103]": "x_vars[renewables_029]",
   "x[104]": "x_vars[renewables_030]",
   "x[105]": "x_vars[renewables_031]",
   "x[106]": "x_vars[renewables_032]",
   "x[107]": "x_vars[renewables_033]",
   "x[108]": "x_vars[renewables_034]",
   "x[109]": "x_vars[renewables_035]",
   "x[110]": "x_vars[renewables_036]",
   "x[111]": "x_vars[renewables_037]",
   "x[112]": "x_vars[renewables_038]",
   "x[113]": "x_vars[renewables_039]",
   "x[114]": "x_vars[renewables_040]",
   "x[115]": "x_vars[renewables_041]",
   "x[116]": "x_vars[renewables_042]",
   "x[117]": "x_vars[renewables_043]",
   "x[118]": "x_vars[renewables_044]",
   "x[119]": "x_vars[renewables_045]",
   "x[120]": "x_vars[renewables_046]",
   "x[121]": "x_vars[renewables_047]",
   "x[122]": "x_vars[renewables_048]",
   "x[123]": "x_vars[renewables_049]",
   "x[124]": "x_vars[renewables_050]",
   "x[125]": "x_vars[renewables_051]",
   "x[126]": "x_vars[renewables_052]",
   "x[127]": "x_vars[renewables_053]",
   "x[128]": "x_vars[renewables_054]",
   "x[129]": "x_vars[renewables_055]",
   "x[130]": "x_vars[renewables_056]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-079": {
  "source_code_sha256": "e2538f9026d3dae5f75e88fa32b8a1b274e73db1fc586526bc817975271c601b",
  "original_solution_sha256": "538c230448d7e17e03b916ac63f4a66f2cfa4b5605cc7c7f7274aa3085be61f5",
  "bindings": [
   [
    "C0",
    "x_vars[A1,B1]"
   ],
   [
    "C1",
    "x_vars[A1,B2]"
   ],
   [
    "C2",
    "x_vars[A1,B3]"
   ],
   [
    "C3",
    "x_vars[A1,B4]"
   ],
   [
    "C4",
    "x_vars[A1,B5]"
   ],
   [
    "C5",
    "x_vars[A1,B6]"
   ],
   [
    "C6",
    "x_vars[A1,B7]"
   ],
   [
    "C7",
    "x_vars[A1,B8]"
   ],
   [
    "C8",
    "x_vars[A2,B1]"
   ],
   [
    "C9",
    "x_vars[A2,B2]"
   ],
   [
    "C10",
    "x_vars[A2,B3]"
   ],
   [
    "C11",
    "x_vars[A2,B4]"
   ],
   [
    "C12",
    "x_vars[A2,B5]"
   ],
   [
    "C13",
    "x_vars[A2,B6]"
   ],
   [
    "C14",
    "x_vars[A2,B7]"
   ],
   [
    "C15",
    "x_vars[A2,B8]"
   ],
   [
    "C16",
    "x_vars[A3,B1]"
   ],
   [
    "C17",
    "x_vars[A3,B2]"
   ],
   [
    "C18",
    "x_vars[A3,B3]"
   ],
   [
    "C19",
    "x_vars[A3,B4]"
   ],
   [
    "C20",
    "x_vars[A3,B5]"
   ],
   [
    "C21",
    "x_vars[A3,B6]"
   ],
   [
    "C22",
    "x_vars[A3,B7]"
   ],
   [
    "C23",
    "x_vars[A3,B8]"
   ],
   [
    "C24",
    "x_vars[A4,B1]"
   ],
   [
    "C25",
    "x_vars[A4,B2]"
   ],
   [
    "C26",
    "x_vars[A4,B3]"
   ],
   [
    "C27",
    "x_vars[A4,B4]"
   ],
   [
    "C28",
    "x_vars[A4,B5]"
   ],
   [
    "C29",
    "x_vars[A4,B6]"
   ],
   [
    "C30",
    "x_vars[A4,B7]"
   ],
   [
    "C31",
    "x_vars[A4,B8]"
   ],
   [
    "C32",
    "x_vars[A5,B1]"
   ],
   [
    "C33",
    "x_vars[A5,B2]"
   ],
   [
    "C34",
    "x_vars[A5,B3]"
   ],
   [
    "C35",
    "x_vars[A5,B4]"
   ],
   [
    "C36",
    "x_vars[A5,B5]"
   ],
   [
    "C37",
    "x_vars[A5,B6]"
   ],
   [
    "C38",
    "x_vars[A5,B7]"
   ],
   [
    "C39",
    "x_vars[A5,B8]"
   ],
   [
    "C40",
    "x_vars[A6,B1]"
   ],
   [
    "C41",
    "x_vars[A6,B2]"
   ],
   [
    "C42",
    "x_vars[A6,B3]"
   ],
   [
    "C43",
    "x_vars[A6,B4]"
   ],
   [
    "C44",
    "x_vars[A6,B5]"
   ],
   [
    "C45",
    "x_vars[A6,B6]"
   ],
   [
    "C46",
    "x_vars[A6,B7]"
   ],
   [
    "C47",
    "x_vars[A6,B8]"
   ],
   [
    "C48",
    "x_vars[A7,B1]"
   ],
   [
    "C49",
    "x_vars[A7,B2]"
   ],
   [
    "C50",
    "x_vars[A7,B3]"
   ],
   [
    "C51",
    "x_vars[A7,B4]"
   ],
   [
    "C52",
    "x_vars[A7,B5]"
   ],
   [
    "C53",
    "x_vars[A7,B6]"
   ],
   [
    "C54",
    "x_vars[A7,B7]"
   ],
   [
    "C55",
    "x_vars[A7,B8]"
   ],
   [
    "C56",
    "x_vars[A8,B1]"
   ],
   [
    "C57",
    "x_vars[A8,B2]"
   ],
   [
    "C58",
    "x_vars[A8,B3]"
   ],
   [
    "C59",
    "x_vars[A8,B4]"
   ],
   [
    "C60",
    "x_vars[A8,B5]"
   ],
   [
    "C61",
    "x_vars[A8,B6]"
   ],
   [
    "C62",
    "x_vars[A8,B7]"
   ],
   [
    "C63",
    "x_vars[A8,B8]"
   ],
   [
    "C64",
    "x_vars[A9,B1]"
   ],
   [
    "C65",
    "x_vars[A9,B2]"
   ],
   [
    "C66",
    "x_vars[A9,B3]"
   ],
   [
    "C67",
    "x_vars[A9,B4]"
   ],
   [
    "C68",
    "x_vars[A9,B5]"
   ],
   [
    "C69",
    "x_vars[A9,B6]"
   ],
   [
    "C70",
    "x_vars[A9,B7]"
   ],
   [
    "C71",
    "x_vars[A9,B8]"
   ],
   [
    "C72",
    "x_vars[A10,B1]"
   ],
   [
    "C73",
    "x_vars[A10,B2]"
   ],
   [
    "C74",
    "x_vars[A10,B3]"
   ],
   [
    "C75",
    "x_vars[A10,B4]"
   ],
   [
    "C76",
    "x_vars[A10,B5]"
   ],
   [
    "C77",
    "x_vars[A10,B6]"
   ],
   [
    "C78",
    "x_vars[A10,B7]"
   ],
   [
    "C79",
    "x_vars[A10,B8]"
   ],
   [
    "C80",
    "x_vars[A11,B1]"
   ],
   [
    "C81",
    "x_vars[A11,B2]"
   ],
   [
    "C82",
    "x_vars[A11,B3]"
   ],
   [
    "C83",
    "x_vars[A11,B4]"
   ],
   [
    "C84",
    "x_vars[A11,B5]"
   ],
   [
    "C85",
    "x_vars[A11,B6]"
   ],
   [
    "C86",
    "x_vars[A11,B7]"
   ],
   [
    "C87",
    "x_vars[A11,B8]"
   ],
   [
    "C88",
    "x_vars[A12,B1]"
   ],
   [
    "C89",
    "x_vars[A12,B2]"
   ],
   [
    "C90",
    "x_vars[A12,B3]"
   ],
   [
    "C91",
    "x_vars[A12,B4]"
   ],
   [
    "C92",
    "x_vars[A12,B5]"
   ],
   [
    "C93",
    "x_vars[A12,B6]"
   ],
   [
    "C94",
    "x_vars[A12,B7]"
   ],
   [
    "C95",
    "x_vars[A12,B8]"
   ],
   [
    "C96",
    "x_vars[A13,B1]"
   ],
   [
    "C97",
    "x_vars[A13,B2]"
   ],
   [
    "C98",
    "x_vars[A13,B3]"
   ],
   [
    "C99",
    "x_vars[A13,B4]"
   ],
   [
    "C100",
    "x_vars[A13,B5]"
   ],
   [
    "C101",
    "x_vars[A13,B6]"
   ],
   [
    "C102",
    "x_vars[A13,B7]"
   ],
   [
    "C103",
    "x_vars[A13,B8]"
   ],
   [
    "C104",
    "x_vars[A14,B1]"
   ],
   [
    "C105",
    "x_vars[A14,B2]"
   ],
   [
    "C106",
    "x_vars[A14,B3]"
   ],
   [
    "C107",
    "x_vars[A14,B4]"
   ],
   [
    "C108",
    "x_vars[A14,B5]"
   ],
   [
    "C109",
    "x_vars[A14,B6]"
   ],
   [
    "C110",
    "x_vars[A14,B7]"
   ],
   [
    "C111",
    "x_vars[A14,B8]"
   ],
   [
    "C112",
    "x_vars[A15,B1]"
   ],
   [
    "C113",
    "x_vars[A15,B2]"
   ],
   [
    "C114",
    "x_vars[A15,B3]"
   ],
   [
    "C115",
    "x_vars[A15,B4]"
   ],
   [
    "C116",
    "x_vars[A15,B5]"
   ],
   [
    "C117",
    "x_vars[A15,B6]"
   ],
   [
    "C118",
    "x_vars[A15,B7]"
   ],
   [
    "C119",
    "x_vars[A15,B8]"
   ],
   [
    "C120",
    "y_vars[A1]"
   ],
   [
    "C121",
    "y_vars[A2]"
   ],
   [
    "C122",
    "y_vars[A3]"
   ],
   [
    "C123",
    "y_vars[A4]"
   ],
   [
    "C124",
    "y_vars[A5]"
   ],
   [
    "C125",
    "y_vars[A6]"
   ],
   [
    "C126",
    "y_vars[A7]"
   ],
   [
    "C127",
    "y_vars[A8]"
   ],
   [
    "C128",
    "y_vars[A9]"
   ],
   [
    "C129",
    "y_vars[A10]"
   ],
   [
    "C130",
    "y_vars[A11]"
   ],
   [
    "C131",
    "y_vars[A12]"
   ],
   [
    "C132",
    "y_vars[A13]"
   ],
   [
    "C133",
    "y_vars[A14]"
   ],
   [
    "C134",
    "y_vars[A15]"
   ]
  ],
  "primary_mapping": {
   "x[0,0]": "x_vars[A1,B1]",
   "x[0,1]": "x_vars[A1,B2]",
   "x[0,2]": "x_vars[A1,B3]",
   "x[0,3]": "x_vars[A1,B4]",
   "x[0,4]": "x_vars[A1,B5]",
   "x[0,5]": "x_vars[A1,B6]",
   "x[0,6]": "x_vars[A1,B7]",
   "x[0,7]": "x_vars[A1,B8]",
   "x[1,0]": "x_vars[A2,B1]",
   "x[1,1]": "x_vars[A2,B2]",
   "x[1,2]": "x_vars[A2,B3]",
   "x[1,3]": "x_vars[A2,B4]",
   "x[1,4]": "x_vars[A2,B5]",
   "x[1,5]": "x_vars[A2,B6]",
   "x[1,6]": "x_vars[A2,B7]",
   "x[1,7]": "x_vars[A2,B8]",
   "x[2,0]": "x_vars[A3,B1]",
   "x[2,1]": "x_vars[A3,B2]",
   "x[2,2]": "x_vars[A3,B3]",
   "x[2,3]": "x_vars[A3,B4]",
   "x[2,4]": "x_vars[A3,B5]",
   "x[2,5]": "x_vars[A3,B6]",
   "x[2,6]": "x_vars[A3,B7]",
   "x[2,7]": "x_vars[A3,B8]",
   "x[3,0]": "x_vars[A4,B1]",
   "x[3,1]": "x_vars[A4,B2]",
   "x[3,2]": "x_vars[A4,B3]",
   "x[3,3]": "x_vars[A4,B4]",
   "x[3,4]": "x_vars[A4,B5]",
   "x[3,5]": "x_vars[A4,B6]",
   "x[3,6]": "x_vars[A4,B7]",
   "x[3,7]": "x_vars[A4,B8]",
   "x[4,0]": "x_vars[A5,B1]",
   "x[4,1]": "x_vars[A5,B2]",
   "x[4,2]": "x_vars[A5,B3]",
   "x[4,3]": "x_vars[A5,B4]",
   "x[4,4]": "x_vars[A5,B5]",
   "x[4,5]": "x_vars[A5,B6]",
   "x[4,6]": "x_vars[A5,B7]",
   "x[4,7]": "x_vars[A5,B8]",
   "x[5,0]": "x_vars[A6,B1]",
   "x[5,1]": "x_vars[A6,B2]",
   "x[5,2]": "x_vars[A6,B3]",
   "x[5,3]": "x_vars[A6,B4]",
   "x[5,4]": "x_vars[A6,B5]",
   "x[5,5]": "x_vars[A6,B6]",
   "x[5,6]": "x_vars[A6,B7]",
   "x[5,7]": "x_vars[A6,B8]",
   "x[6,0]": "x_vars[A7,B1]",
   "x[6,1]": "x_vars[A7,B2]",
   "x[6,2]": "x_vars[A7,B3]",
   "x[6,3]": "x_vars[A7,B4]",
   "x[6,4]": "x_vars[A7,B5]",
   "x[6,5]": "x_vars[A7,B6]",
   "x[6,6]": "x_vars[A7,B7]",
   "x[6,7]": "x_vars[A7,B8]",
   "x[7,0]": "x_vars[A8,B1]",
   "x[7,1]": "x_vars[A8,B2]",
   "x[7,2]": "x_vars[A8,B3]",
   "x[7,3]": "x_vars[A8,B4]",
   "x[7,4]": "x_vars[A8,B5]",
   "x[7,5]": "x_vars[A8,B6]",
   "x[7,6]": "x_vars[A8,B7]",
   "x[7,7]": "x_vars[A8,B8]",
   "x[8,0]": "x_vars[A9,B1]",
   "x[8,1]": "x_vars[A9,B2]",
   "x[8,2]": "x_vars[A9,B3]",
   "x[8,3]": "x_vars[A9,B4]",
   "x[8,4]": "x_vars[A9,B5]",
   "x[8,5]": "x_vars[A9,B6]",
   "x[8,6]": "x_vars[A9,B7]",
   "x[8,7]": "x_vars[A9,B8]",
   "x[9,0]": "x_vars[A10,B1]",
   "x[9,1]": "x_vars[A10,B2]",
   "x[9,2]": "x_vars[A10,B3]",
   "x[9,3]": "x_vars[A10,B4]",
   "x[9,4]": "x_vars[A10,B5]",
   "x[9,5]": "x_vars[A10,B6]",
   "x[9,6]": "x_vars[A10,B7]",
   "x[9,7]": "x_vars[A10,B8]",
   "x[10,0]": "x_vars[A11,B1]",
   "x[10,1]": "x_vars[A11,B2]",
   "x[10,2]": "x_vars[A11,B3]",
   "x[10,3]": "x_vars[A11,B4]",
   "x[10,4]": "x_vars[A11,B5]",
   "x[10,5]": "x_vars[A11,B6]",
   "x[10,6]": "x_vars[A11,B7]",
   "x[10,7]": "x_vars[A11,B8]",
   "x[11,0]": "x_vars[A12,B1]",
   "x[11,1]": "x_vars[A12,B2]",
   "x[11,2]": "x_vars[A12,B3]",
   "x[11,3]": "x_vars[A12,B4]",
   "x[11,4]": "x_vars[A12,B5]",
   "x[11,5]": "x_vars[A12,B6]",
   "x[11,6]": "x_vars[A12,B7]",
   "x[11,7]": "x_vars[A12,B8]",
   "x[12,0]": "x_vars[A13,B1]",
   "x[12,1]": "x_vars[A13,B2]",
   "x[12,2]": "x_vars[A13,B3]",
   "x[12,3]": "x_vars[A13,B4]",
   "x[12,4]": "x_vars[A13,B5]",
   "x[12,5]": "x_vars[A13,B6]",
   "x[12,6]": "x_vars[A13,B7]",
   "x[12,7]": "x_vars[A13,B8]",
   "x[13,0]": "x_vars[A14,B1]",
   "x[13,1]": "x_vars[A14,B2]",
   "x[13,2]": "x_vars[A14,B3]",
   "x[13,3]": "x_vars[A14,B4]",
   "x[13,4]": "x_vars[A14,B5]",
   "x[13,5]": "x_vars[A14,B6]",
   "x[13,6]": "x_vars[A14,B7]",
   "x[13,7]": "x_vars[A14,B8]",
   "x[14,0]": "x_vars[A15,B1]",
   "x[14,1]": "x_vars[A15,B2]",
   "x[14,2]": "x_vars[A15,B3]",
   "x[14,3]": "x_vars[A15,B4]",
   "x[14,4]": "x_vars[A15,B5]",
   "x[14,5]": "x_vars[A15,B6]",
   "x[14,6]": "x_vars[A15,B7]",
   "x[14,7]": "x_vars[A15,B8]",
   "y[1]": "y_vars[A2]",
   "y[2]": "y_vars[A3]",
   "y[3]": "y_vars[A4]",
   "y[4]": "y_vars[A5]",
   "y[5]": "y_vars[A6]",
   "y[6]": "y_vars[A7]",
   "y[7]": "y_vars[A8]",
   "y[8]": "y_vars[A9]",
   "y[9]": "y_vars[A10]",
   "y[10]": "y_vars[A11]",
   "y[11]": "y_vars[A12]",
   "y[12]": "y_vars[A13]",
   "y[13]": "y_vars[A14]",
   "y[14]": "y_vars[A15]",
   "y[0]": "y_vars[A1]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-080": {
  "source_code_sha256": "e8db98f6f134db1cf4857758b4fc788c915e45387ef95a2f1ff849047d9bd13d",
  "original_solution_sha256": "00d8fd9923ac69218bd29f0a7546963723f227d2f32e3500ea1002a6c8d19242",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[1,4]"
   ],
   [
    "C4",
    "x_vars[2,1]"
   ],
   [
    "C5",
    "x_vars[2,2]"
   ],
   [
    "C6",
    "x_vars[2,3]"
   ],
   [
    "C7",
    "x_vars[2,4]"
   ],
   [
    "C8",
    "x_vars[3,1]"
   ],
   [
    "C9",
    "x_vars[3,2]"
   ],
   [
    "C10",
    "x_vars[3,3]"
   ],
   [
    "C11",
    "x_vars[3,4]"
   ],
   [
    "C12",
    "x_vars[4,1]"
   ],
   [
    "C13",
    "x_vars[4,2]"
   ],
   [
    "C14",
    "x_vars[4,3]"
   ],
   [
    "C15",
    "x_vars[4,4]"
   ],
   [
    "C16",
    "x_vars[5,1]"
   ],
   [
    "C17",
    "x_vars[5,2]"
   ],
   [
    "C18",
    "x_vars[5,3]"
   ],
   [
    "C19",
    "x_vars[5,4]"
   ],
   [
    "C20",
    "x_vars[6,1]"
   ],
   [
    "C21",
    "x_vars[6,2]"
   ],
   [
    "C22",
    "x_vars[6,3]"
   ],
   [
    "C23",
    "x_vars[6,4]"
   ],
   [
    "C24",
    "x_vars[7,1]"
   ],
   [
    "C25",
    "x_vars[7,2]"
   ],
   [
    "C26",
    "x_vars[7,3]"
   ],
   [
    "C27",
    "x_vars[7,4]"
   ],
   [
    "C28",
    "x_vars[8,1]"
   ],
   [
    "C29",
    "x_vars[8,2]"
   ],
   [
    "C30",
    "x_vars[8,3]"
   ],
   [
    "C31",
    "x_vars[8,4]"
   ],
   [
    "C32",
    "x_vars[9,1]"
   ],
   [
    "C33",
    "x_vars[9,2]"
   ],
   [
    "C34",
    "x_vars[9,3]"
   ],
   [
    "C35",
    "x_vars[9,4]"
   ],
   [
    "C36",
    "x_vars[10,1]"
   ],
   [
    "C37",
    "x_vars[10,2]"
   ],
   [
    "C38",
    "x_vars[10,3]"
   ],
   [
    "C39",
    "x_vars[10,4]"
   ],
   [
    "C40",
    "y_vars[1,1]"
   ],
   [
    "C41",
    "y_vars[1,2]"
   ],
   [
    "C42",
    "y_vars[1,3]"
   ],
   [
    "C43",
    "y_vars[1,4]"
   ],
   [
    "C44",
    "y_vars[2,1]"
   ],
   [
    "C45",
    "y_vars[2,2]"
   ],
   [
    "C46",
    "y_vars[2,3]"
   ],
   [
    "C47",
    "y_vars[2,4]"
   ],
   [
    "C48",
    "y_vars[3,1]"
   ],
   [
    "C49",
    "y_vars[3,2]"
   ],
   [
    "C50",
    "y_vars[3,3]"
   ],
   [
    "C51",
    "y_vars[3,4]"
   ],
   [
    "C52",
    "y_vars[4,1]"
   ],
   [
    "C53",
    "y_vars[4,2]"
   ],
   [
    "C54",
    "y_vars[4,3]"
   ],
   [
    "C55",
    "y_vars[4,4]"
   ],
   [
    "C56",
    "y_vars[5,1]"
   ],
   [
    "C57",
    "y_vars[5,2]"
   ],
   [
    "C58",
    "y_vars[5,3]"
   ],
   [
    "C59",
    "y_vars[5,4]"
   ],
   [
    "C60",
    "y_vars[6,1]"
   ],
   [
    "C61",
    "y_vars[6,2]"
   ],
   [
    "C62",
    "y_vars[6,3]"
   ],
   [
    "C63",
    "y_vars[6,4]"
   ],
   [
    "C64",
    "y_vars[7,1]"
   ],
   [
    "C65",
    "y_vars[7,2]"
   ],
   [
    "C66",
    "y_vars[7,3]"
   ],
   [
    "C67",
    "y_vars[7,4]"
   ],
   [
    "C68",
    "y_vars[8,1]"
   ],
   [
    "C69",
    "y_vars[8,2]"
   ],
   [
    "C70",
    "y_vars[8,3]"
   ],
   [
    "C71",
    "y_vars[8,4]"
   ],
   [
    "C72",
    "y_vars[9,1]"
   ],
   [
    "C73",
    "y_vars[9,2]"
   ],
   [
    "C74",
    "y_vars[9,3]"
   ],
   [
    "C75",
    "y_vars[9,4]"
   ],
   [
    "C76",
    "y_vars[10,1]"
   ],
   [
    "C77",
    "y_vars[10,2]"
   ],
   [
    "C78",
    "y_vars[10,3]"
   ],
   [
    "C79",
    "y_vars[10,4]"
   ],
   [
    "C80",
    "s_vars[1,1]"
   ],
   [
    "C81",
    "s_vars[1,2]"
   ],
   [
    "C82",
    "s_vars[1,3]"
   ],
   [
    "C83",
    "s_vars[1,4]"
   ],
   [
    "C84",
    "s_vars[2,1]"
   ],
   [
    "C85",
    "s_vars[2,2]"
   ],
   [
    "C86",
    "s_vars[2,3]"
   ],
   [
    "C87",
    "s_vars[2,4]"
   ],
   [
    "C88",
    "s_vars[3,1]"
   ],
   [
    "C89",
    "s_vars[3,2]"
   ],
   [
    "C90",
    "s_vars[3,3]"
   ],
   [
    "C91",
    "s_vars[3,4]"
   ],
   [
    "C92",
    "s_vars[4,1]"
   ],
   [
    "C93",
    "s_vars[4,2]"
   ],
   [
    "C94",
    "s_vars[4,3]"
   ],
   [
    "C95",
    "s_vars[4,4]"
   ],
   [
    "C96",
    "s_vars[5,1]"
   ],
   [
    "C97",
    "s_vars[5,2]"
   ],
   [
    "C98",
    "s_vars[5,3]"
   ],
   [
    "C99",
    "s_vars[5,4]"
   ],
   [
    "C100",
    "s_vars[6,1]"
   ],
   [
    "C101",
    "s_vars[6,2]"
   ],
   [
    "C102",
    "s_vars[6,3]"
   ],
   [
    "C103",
    "s_vars[6,4]"
   ],
   [
    "C104",
    "s_vars[7,1]"
   ],
   [
    "C105",
    "s_vars[7,2]"
   ],
   [
    "C106",
    "s_vars[7,3]"
   ],
   [
    "C107",
    "s_vars[7,4]"
   ],
   [
    "C108",
    "s_vars[8,1]"
   ],
   [
    "C109",
    "s_vars[8,2]"
   ],
   [
    "C110",
    "s_vars[8,3]"
   ],
   [
    "C111",
    "s_vars[8,4]"
   ],
   [
    "C112",
    "s_vars[9,1]"
   ],
   [
    "C113",
    "s_vars[9,2]"
   ],
   [
    "C114",
    "s_vars[9,3]"
   ],
   [
    "C115",
    "s_vars[9,4]"
   ],
   [
    "C116",
    "s_vars[10,1]"
   ],
   [
    "C117",
    "s_vars[10,2]"
   ],
   [
    "C118",
    "s_vars[10,3]"
   ],
   [
    "C119",
    "s_vars[10,4]"
   ]
  ],
  "primary_mapping": {
   "w[1,1]": "x_vars[1,1]",
   "w[1,2]": "x_vars[1,2]",
   "w[1,3]": "x_vars[1,3]",
   "w[1,4]": "x_vars[1,4]",
   "w[2,1]": "x_vars[2,1]",
   "w[2,2]": "x_vars[2,2]",
   "w[2,3]": "x_vars[2,3]",
   "w[2,4]": "x_vars[2,4]",
   "w[3,1]": "x_vars[3,1]",
   "w[3,2]": "x_vars[3,2]",
   "w[3,3]": "x_vars[3,3]",
   "w[3,4]": "x_vars[3,4]",
   "w[4,1]": "x_vars[4,1]",
   "w[4,2]": "x_vars[4,2]",
   "w[4,3]": "x_vars[4,3]",
   "w[4,4]": "x_vars[4,4]",
   "w[5,1]": "x_vars[5,1]",
   "w[5,2]": "x_vars[5,2]",
   "w[5,3]": "x_vars[5,3]",
   "w[5,4]": "x_vars[5,4]",
   "w[6,1]": "x_vars[6,1]",
   "w[6,2]": "x_vars[6,2]",
   "w[6,3]": "x_vars[6,3]",
   "w[6,4]": "x_vars[6,4]",
   "w[7,1]": "x_vars[7,1]",
   "w[7,2]": "x_vars[7,2]",
   "w[7,3]": "x_vars[7,3]",
   "w[7,4]": "x_vars[7,4]",
   "w[8,1]": "x_vars[8,1]",
   "w[8,2]": "x_vars[8,2]",
   "w[8,3]": "x_vars[8,3]",
   "w[8,4]": "x_vars[8,4]",
   "w[9,1]": "x_vars[9,1]",
   "w[9,2]": "x_vars[9,2]",
   "w[9,3]": "x_vars[9,3]",
   "w[9,4]": "x_vars[9,4]",
   "w[10,1]": "x_vars[10,1]",
   "w[10,2]": "x_vars[10,2]",
   "w[10,3]": "x_vars[10,3]",
   "w[10,4]": "x_vars[10,4]",
   "y[1,1]": "y_vars[1,1]",
   "y[1,2]": "y_vars[1,2]",
   "y[1,3]": "y_vars[1,3]",
   "y[1,4]": "y_vars[1,4]",
   "y[2,1]": "y_vars[2,1]",
   "y[2,2]": "y_vars[2,2]",
   "y[2,3]": "y_vars[2,3]",
   "y[2,4]": "y_vars[2,4]",
   "y[3,1]": "y_vars[3,1]",
   "y[3,2]": "y_vars[3,2]",
   "y[3,3]": "y_vars[3,3]",
   "y[3,4]": "y_vars[3,4]",
   "y[4,1]": "y_vars[4,1]",
   "y[4,2]": "y_vars[4,2]",
   "y[4,3]": "y_vars[4,3]",
   "y[4,4]": "y_vars[4,4]",
   "y[5,1]": "y_vars[5,1]",
   "y[5,2]": "y_vars[5,2]",
   "y[5,3]": "y_vars[5,3]",
   "y[5,4]": "y_vars[5,4]",
   "y[6,1]": "y_vars[6,1]",
   "y[6,2]": "y_vars[6,2]",
   "y[6,3]": "y_vars[6,3]",
   "y[6,4]": "y_vars[6,4]",
   "y[7,1]": "y_vars[7,1]",
   "y[7,2]": "y_vars[7,2]",
   "y[7,3]": "y_vars[7,3]",
   "y[7,4]": "y_vars[7,4]",
   "y[8,1]": "y_vars[8,1]",
   "y[8,2]": "y_vars[8,2]",
   "y[8,3]": "y_vars[8,3]",
   "y[8,4]": "y_vars[8,4]",
   "y[9,1]": "y_vars[9,1]",
   "y[9,2]": "y_vars[9,2]",
   "y[9,3]": "y_vars[9,3]",
   "y[9,4]": "y_vars[9,4]",
   "y[10,1]": "y_vars[10,1]",
   "y[10,2]": "y_vars[10,2]",
   "y[10,3]": "y_vars[10,3]",
   "y[10,4]": "y_vars[10,4]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-081": {
  "source_code_sha256": "5180a1b19c2b7018182f74d99f6d8039aaca61e32d39d0bc218b34106723a03b",
  "original_solution_sha256": "8b4e8159dab27db39243def6d49337e88a112004cf85a5071f04281927c7bbd2",
  "bindings": [
   [
    "C0",
    "x_vars[Oatmeal]"
   ],
   [
    "C1",
    "x_vars[Milk]"
   ],
   [
    "C2",
    "x_vars[Egg]"
   ],
   [
    "C3",
    "x_vars[Banana]"
   ],
   [
    "C4",
    "x_vars[Grain_001]"
   ],
   [
    "C5",
    "x_vars[Grain_002]"
   ],
   [
    "C6",
    "x_vars[Grain_003]"
   ],
   [
    "C7",
    "x_vars[Grain_004]"
   ],
   [
    "C8",
    "x_vars[Grain_005]"
   ],
   [
    "C9",
    "x_vars[Grain_006]"
   ],
   [
    "C10",
    "x_vars[Grain_007]"
   ],
   [
    "C11",
    "x_vars[Grain_008]"
   ],
   [
    "C12",
    "x_vars[Grain_009]"
   ],
   [
    "C13",
    "x_vars[Grain_010]"
   ],
   [
    "C14",
    "x_vars[Grain_011]"
   ],
   [
    "C15",
    "x_vars[Grain_012]"
   ],
   [
    "C16",
    "x_vars[Grain_013]"
   ],
   [
    "C17",
    "x_vars[Grain_014]"
   ],
   [
    "C18",
    "x_vars[Grain_015]"
   ],
   [
    "C19",
    "x_vars[Grain_016]"
   ],
   [
    "C20",
    "x_vars[Grain_017]"
   ],
   [
    "C21",
    "x_vars[Grain_018]"
   ],
   [
    "C22",
    "x_vars[Grain_019]"
   ],
   [
    "C23",
    "x_vars[Grain_020]"
   ],
   [
    "C24",
    "x_vars[Grain_021]"
   ],
   [
    "C25",
    "x_vars[Grain_022]"
   ],
   [
    "C26",
    "x_vars[Grain_023]"
   ],
   [
    "C27",
    "x_vars[Grain_024]"
   ],
   [
    "C28",
    "x_vars[Grain_025]"
   ],
   [
    "C29",
    "x_vars[Dairy_001]"
   ],
   [
    "C30",
    "x_vars[Dairy_002]"
   ],
   [
    "C31",
    "x_vars[Dairy_003]"
   ],
   [
    "C32",
    "x_vars[Dairy_004]"
   ],
   [
    "C33",
    "x_vars[Dairy_005]"
   ],
   [
    "C34",
    "x_vars[Dairy_006]"
   ],
   [
    "C35",
    "x_vars[Dairy_007]"
   ],
   [
    "C36",
    "x_vars[Dairy_008]"
   ],
   [
    "C37",
    "x_vars[Dairy_009]"
   ],
   [
    "C38",
    "x_vars[Dairy_010]"
   ],
   [
    "C39",
    "x_vars[Dairy_011]"
   ],
   [
    "C40",
    "x_vars[Dairy_012]"
   ],
   [
    "C41",
    "x_vars[Dairy_013]"
   ],
   [
    "C42",
    "x_vars[Dairy_014]"
   ],
   [
    "C43",
    "x_vars[Dairy_015]"
   ],
   [
    "C44",
    "x_vars[Dairy_016]"
   ],
   [
    "C45",
    "x_vars[Dairy_017]"
   ],
   [
    "C46",
    "x_vars[Dairy_018]"
   ],
   [
    "C47",
    "x_vars[Dairy_019]"
   ],
   [
    "C48",
    "x_vars[Dairy_020]"
   ],
   [
    "C49",
    "x_vars[Dairy_021]"
   ],
   [
    "C50",
    "x_vars[Dairy_022]"
   ],
   [
    "C51",
    "x_vars[Dairy_023]"
   ],
   [
    "C52",
    "x_vars[Dairy_024]"
   ],
   [
    "C53",
    "x_vars[Dairy_025]"
   ],
   [
    "C54",
    "x_vars[Meat_001]"
   ],
   [
    "C55",
    "x_vars[Meat_002]"
   ],
   [
    "C56",
    "x_vars[Meat_003]"
   ],
   [
    "C57",
    "x_vars[Meat_004]"
   ],
   [
    "C58",
    "x_vars[Meat_005]"
   ],
   [
    "C59",
    "x_vars[Meat_006]"
   ],
   [
    "C60",
    "x_vars[Meat_007]"
   ],
   [
    "C61",
    "x_vars[Meat_008]"
   ],
   [
    "C62",
    "x_vars[Meat_009]"
   ],
   [
    "C63",
    "x_vars[Meat_010]"
   ],
   [
    "C64",
    "x_vars[Meat_011]"
   ],
   [
    "C65",
    "x_vars[Meat_012]"
   ],
   [
    "C66",
    "x_vars[Meat_013]"
   ],
   [
    "C67",
    "x_vars[Meat_014]"
   ],
   [
    "C68",
    "x_vars[Meat_015]"
   ],
   [
    "C69",
    "x_vars[Meat_016]"
   ],
   [
    "C70",
    "x_vars[Meat_017]"
   ],
   [
    "C71",
    "x_vars[Meat_018]"
   ],
   [
    "C72",
    "x_vars[Meat_019]"
   ],
   [
    "C73",
    "x_vars[Meat_020]"
   ],
   [
    "C74",
    "x_vars[Meat_021]"
   ],
   [
    "C75",
    "x_vars[Meat_022]"
   ],
   [
    "C76",
    "x_vars[Meat_023]"
   ],
   [
    "C77",
    "x_vars[Meat_024]"
   ],
   [
    "C78",
    "x_vars[Meat_025]"
   ],
   [
    "C79",
    "x_vars[Fruit_001]"
   ],
   [
    "C80",
    "x_vars[Fruit_002]"
   ],
   [
    "C81",
    "x_vars[Fruit_003]"
   ],
   [
    "C82",
    "x_vars[Fruit_004]"
   ],
   [
    "C83",
    "x_vars[Fruit_005]"
   ],
   [
    "C84",
    "x_vars[Fruit_006]"
   ],
   [
    "C85",
    "x_vars[Fruit_007]"
   ],
   [
    "C86",
    "x_vars[Fruit_008]"
   ],
   [
    "C87",
    "x_vars[Fruit_009]"
   ],
   [
    "C88",
    "x_vars[Fruit_010]"
   ],
   [
    "C89",
    "x_vars[Fruit_011]"
   ],
   [
    "C90",
    "x_vars[Fruit_012]"
   ],
   [
    "C91",
    "x_vars[Fruit_013]"
   ],
   [
    "C92",
    "x_vars[Fruit_014]"
   ],
   [
    "C93",
    "x_vars[Fruit_015]"
   ],
   [
    "C94",
    "x_vars[Fruit_016]"
   ],
   [
    "C95",
    "x_vars[Fruit_017]"
   ],
   [
    "C96",
    "x_vars[Fruit_018]"
   ],
   [
    "C97",
    "x_vars[Fruit_019]"
   ],
   [
    "C98",
    "x_vars[Fruit_020]"
   ],
   [
    "C99",
    "x_vars[Fruit_021]"
   ],
   [
    "C100",
    "x_vars[Fruit_022]"
   ],
   [
    "C101",
    "x_vars[Fruit_023]"
   ],
   [
    "C102",
    "x_vars[Fruit_024]"
   ],
   [
    "C103",
    "x_vars[Fruit_025]"
   ],
   [
    "C104",
    "x_vars[Veg_001]"
   ],
   [
    "C105",
    "x_vars[Veg_002]"
   ],
   [
    "C106",
    "x_vars[Veg_003]"
   ],
   [
    "C107",
    "x_vars[Veg_004]"
   ],
   [
    "C108",
    "x_vars[Veg_005]"
   ],
   [
    "C109",
    "x_vars[Veg_006]"
   ],
   [
    "C110",
    "x_vars[Veg_007]"
   ],
   [
    "C111",
    "x_vars[Veg_008]"
   ],
   [
    "C112",
    "x_vars[Veg_009]"
   ],
   [
    "C113",
    "x_vars[Veg_010]"
   ],
   [
    "C114",
    "x_vars[Veg_011]"
   ],
   [
    "C115",
    "x_vars[Veg_012]"
   ],
   [
    "C116",
    "x_vars[Veg_013]"
   ],
   [
    "C117",
    "x_vars[Veg_014]"
   ],
   [
    "C118",
    "x_vars[Veg_015]"
   ],
   [
    "C119",
    "x_vars[Veg_016]"
   ]
  ],
  "primary_mapping": {
   "x[Oatmeal]": "x_vars[Oatmeal]",
   "x[Milk]": "x_vars[Milk]",
   "x[Egg]": "x_vars[Egg]",
   "x[Banana]": "x_vars[Banana]",
   "x[Grain_001]": "x_vars[Grain_001]",
   "x[Grain_002]": "x_vars[Grain_002]",
   "x[Grain_003]": "x_vars[Grain_003]",
   "x[Grain_004]": "x_vars[Grain_004]",
   "x[Grain_005]": "x_vars[Grain_005]",
   "x[Grain_006]": "x_vars[Grain_006]",
   "x[Grain_007]": "x_vars[Grain_007]",
   "x[Grain_008]": "x_vars[Grain_008]",
   "x[Grain_009]": "x_vars[Grain_009]",
   "x[Grain_010]": "x_vars[Grain_010]",
   "x[Grain_011]": "x_vars[Grain_011]",
   "x[Grain_012]": "x_vars[Grain_012]",
   "x[Grain_013]": "x_vars[Grain_013]",
   "x[Grain_014]": "x_vars[Grain_014]",
   "x[Grain_015]": "x_vars[Grain_015]",
   "x[Grain_016]": "x_vars[Grain_016]",
   "x[Grain_017]": "x_vars[Grain_017]",
   "x[Grain_018]": "x_vars[Grain_018]",
   "x[Grain_019]": "x_vars[Grain_019]",
   "x[Grain_020]": "x_vars[Grain_020]",
   "x[Grain_021]": "x_vars[Grain_021]",
   "x[Grain_022]": "x_vars[Grain_022]",
   "x[Grain_023]": "x_vars[Grain_023]",
   "x[Grain_024]": "x_vars[Grain_024]",
   "x[Grain_025]": "x_vars[Grain_025]",
   "x[Dairy_001]": "x_vars[Dairy_001]",
   "x[Dairy_002]": "x_vars[Dairy_002]",
   "x[Dairy_003]": "x_vars[Dairy_003]",
   "x[Dairy_004]": "x_vars[Dairy_004]",
   "x[Dairy_005]": "x_vars[Dairy_005]",
   "x[Dairy_006]": "x_vars[Dairy_006]",
   "x[Dairy_007]": "x_vars[Dairy_007]",
   "x[Dairy_008]": "x_vars[Dairy_008]",
   "x[Dairy_009]": "x_vars[Dairy_009]",
   "x[Dairy_010]": "x_vars[Dairy_010]",
   "x[Dairy_011]": "x_vars[Dairy_011]",
   "x[Dairy_012]": "x_vars[Dairy_012]",
   "x[Dairy_013]": "x_vars[Dairy_013]",
   "x[Dairy_014]": "x_vars[Dairy_014]",
   "x[Dairy_015]": "x_vars[Dairy_015]",
   "x[Dairy_016]": "x_vars[Dairy_016]",
   "x[Dairy_017]": "x_vars[Dairy_017]",
   "x[Dairy_018]": "x_vars[Dairy_018]",
   "x[Dairy_019]": "x_vars[Dairy_019]",
   "x[Dairy_020]": "x_vars[Dairy_020]",
   "x[Dairy_021]": "x_vars[Dairy_021]",
   "x[Dairy_022]": "x_vars[Dairy_022]",
   "x[Dairy_023]": "x_vars[Dairy_023]",
   "x[Dairy_024]": "x_vars[Dairy_024]",
   "x[Dairy_025]": "x_vars[Dairy_025]",
   "x[Meat_001]": "x_vars[Meat_001]",
   "x[Meat_002]": "x_vars[Meat_002]",
   "x[Meat_003]": "x_vars[Meat_003]",
   "x[Meat_004]": "x_vars[Meat_004]",
   "x[Meat_005]": "x_vars[Meat_005]",
   "x[Meat_006]": "x_vars[Meat_006]",
   "x[Meat_007]": "x_vars[Meat_007]",
   "x[Meat_008]": "x_vars[Meat_008]",
   "x[Meat_009]": "x_vars[Meat_009]",
   "x[Meat_010]": "x_vars[Meat_010]",
   "x[Meat_011]": "x_vars[Meat_011]",
   "x[Meat_012]": "x_vars[Meat_012]",
   "x[Meat_013]": "x_vars[Meat_013]",
   "x[Meat_014]": "x_vars[Meat_014]",
   "x[Meat_015]": "x_vars[Meat_015]",
   "x[Meat_016]": "x_vars[Meat_016]",
   "x[Meat_017]": "x_vars[Meat_017]",
   "x[Meat_018]": "x_vars[Meat_018]",
   "x[Meat_019]": "x_vars[Meat_019]",
   "x[Meat_020]": "x_vars[Meat_020]",
   "x[Meat_021]": "x_vars[Meat_021]",
   "x[Meat_022]": "x_vars[Meat_022]",
   "x[Meat_023]": "x_vars[Meat_023]",
   "x[Meat_024]": "x_vars[Meat_024]",
   "x[Meat_025]": "x_vars[Meat_025]",
   "x[Fruit_001]": "x_vars[Fruit_001]",
   "x[Fruit_002]": "x_vars[Fruit_002]",
   "x[Fruit_003]": "x_vars[Fruit_003]",
   "x[Fruit_004]": "x_vars[Fruit_004]",
   "x[Fruit_005]": "x_vars[Fruit_005]",
   "x[Fruit_006]": "x_vars[Fruit_006]",
   "x[Fruit_007]": "x_vars[Fruit_007]",
   "x[Fruit_008]": "x_vars[Fruit_008]",
   "x[Fruit_009]": "x_vars[Fruit_009]",
   "x[Fruit_010]": "x_vars[Fruit_010]",
   "x[Fruit_011]": "x_vars[Fruit_011]",
   "x[Fruit_012]": "x_vars[Fruit_012]",
   "x[Fruit_013]": "x_vars[Fruit_013]",
   "x[Fruit_014]": "x_vars[Fruit_014]",
   "x[Fruit_015]": "x_vars[Fruit_015]",
   "x[Fruit_016]": "x_vars[Fruit_016]",
   "x[Fruit_017]": "x_vars[Fruit_017]",
   "x[Fruit_018]": "x_vars[Fruit_018]",
   "x[Fruit_019]": "x_vars[Fruit_019]",
   "x[Fruit_020]": "x_vars[Fruit_020]",
   "x[Fruit_021]": "x_vars[Fruit_021]",
   "x[Fruit_022]": "x_vars[Fruit_022]",
   "x[Fruit_023]": "x_vars[Fruit_023]",
   "x[Fruit_024]": "x_vars[Fruit_024]",
   "x[Fruit_025]": "x_vars[Fruit_025]",
   "x[Veg_001]": "x_vars[Veg_001]",
   "x[Veg_002]": "x_vars[Veg_002]",
   "x[Veg_003]": "x_vars[Veg_003]",
   "x[Veg_004]": "x_vars[Veg_004]",
   "x[Veg_005]": "x_vars[Veg_005]",
   "x[Veg_006]": "x_vars[Veg_006]",
   "x[Veg_007]": "x_vars[Veg_007]",
   "x[Veg_008]": "x_vars[Veg_008]",
   "x[Veg_009]": "x_vars[Veg_009]",
   "x[Veg_010]": "x_vars[Veg_010]",
   "x[Veg_011]": "x_vars[Veg_011]",
   "x[Veg_012]": "x_vars[Veg_012]",
   "x[Veg_013]": "x_vars[Veg_013]",
   "x[Veg_014]": "x_vars[Veg_014]",
   "x[Veg_015]": "x_vars[Veg_015]",
   "x[Veg_016]": "x_vars[Veg_016]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-082": {
  "source_code_sha256": "5ca59fe46a035b1a52d2e94b8ff98225a8ca24350cf2162d3f6f31a2de03313e",
  "original_solution_sha256": "87752db3814dc59eb7fe379d45bb8ac8fc68b7473525457aa99fd5c4aae8a956",
  "bindings": [
   [
    "C0",
    "x_vars[Depot,A]"
   ],
   [
    "C1",
    "x_vars[Depot,B]"
   ],
   [
    "C2",
    "x_vars[Depot,C]"
   ],
   [
    "C3",
    "x_vars[A,Depot]"
   ],
   [
    "C4",
    "x_vars[A,B]"
   ],
   [
    "C5",
    "x_vars[A,C]"
   ],
   [
    "C6",
    "x_vars[B,Depot]"
   ],
   [
    "C7",
    "x_vars[B,A]"
   ],
   [
    "C8",
    "x_vars[B,C]"
   ],
   [
    "C9",
    "x_vars[C,Depot]"
   ],
   [
    "C10",
    "x_vars[C,A]"
   ],
   [
    "C11",
    "x_vars[C,B]"
   ],
   [
    "C12",
    "u_vars[A]"
   ],
   [
    "C13",
    "u_vars[B]"
   ],
   [
    "C14",
    "u_vars[C]"
   ]
  ],
  "primary_mapping": {
   "X[0,1]": "x_vars[Depot,A]",
   "X[0,2]": "x_vars[Depot,B]",
   "X[0,3]": "x_vars[Depot,C]",
   "X[1,0]": "x_vars[A,Depot]",
   "X[1,2]": "x_vars[A,B]",
   "X[1,3]": "x_vars[A,C]",
   "X[2,0]": "x_vars[B,Depot]",
   "X[2,1]": "x_vars[B,A]",
   "X[2,3]": "x_vars[B,C]",
   "X[3,0]": "x_vars[C,Depot]",
   "X[3,1]": "x_vars[C,A]",
   "X[3,2]": "x_vars[C,B]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-083": {
  "source_code_sha256": "fc060baa88cc5e3045b01bd122a5c1d4ecb79e69d0b9083f75a79cdf06a4d0cc",
  "original_solution_sha256": "bf7686c5440b2ef508aba1076bfcd3fb03f88530ae24035395df41787f0787f1",
  "bindings": [
   [
    "C0",
    "x_vars[1,A]"
   ],
   [
    "C1",
    "x_vars[1,B]"
   ],
   [
    "C2",
    "x_vars[1,C]"
   ],
   [
    "C3",
    "x_vars[1,D]"
   ],
   [
    "C4",
    "x_vars[1,E]"
   ],
   [
    "C5",
    "x_vars[1,F]"
   ],
   [
    "C6",
    "x_vars[1,G]"
   ],
   [
    "C7",
    "x_vars[1,H]"
   ],
   [
    "C8",
    "x_vars[1,I]"
   ],
   [
    "C9",
    "x_vars[1,J]"
   ],
   [
    "C10",
    "x_vars[2,A]"
   ],
   [
    "C11",
    "x_vars[2,B]"
   ],
   [
    "C12",
    "x_vars[2,C]"
   ],
   [
    "C13",
    "x_vars[2,D]"
   ],
   [
    "C14",
    "x_vars[2,E]"
   ],
   [
    "C15",
    "x_vars[2,F]"
   ],
   [
    "C16",
    "x_vars[2,G]"
   ],
   [
    "C17",
    "x_vars[2,H]"
   ],
   [
    "C18",
    "x_vars[2,I]"
   ],
   [
    "C19",
    "x_vars[2,J]"
   ],
   [
    "C20",
    "x_vars[3,A]"
   ],
   [
    "C21",
    "x_vars[3,B]"
   ],
   [
    "C22",
    "x_vars[3,C]"
   ],
   [
    "C23",
    "x_vars[3,D]"
   ],
   [
    "C24",
    "x_vars[3,E]"
   ],
   [
    "C25",
    "x_vars[3,F]"
   ],
   [
    "C26",
    "x_vars[3,G]"
   ],
   [
    "C27",
    "x_vars[3,H]"
   ],
   [
    "C28",
    "x_vars[3,I]"
   ],
   [
    "C29",
    "x_vars[3,J]"
   ],
   [
    "C30",
    "x_vars[4,A]"
   ],
   [
    "C31",
    "x_vars[4,B]"
   ],
   [
    "C32",
    "x_vars[4,C]"
   ],
   [
    "C33",
    "x_vars[4,D]"
   ],
   [
    "C34",
    "x_vars[4,E]"
   ],
   [
    "C35",
    "x_vars[4,F]"
   ],
   [
    "C36",
    "x_vars[4,G]"
   ],
   [
    "C37",
    "x_vars[4,H]"
   ],
   [
    "C38",
    "x_vars[4,I]"
   ],
   [
    "C39",
    "x_vars[4,J]"
   ],
   [
    "C40",
    "x_vars[5,A]"
   ],
   [
    "C41",
    "x_vars[5,B]"
   ],
   [
    "C42",
    "x_vars[5,C]"
   ],
   [
    "C43",
    "x_vars[5,D]"
   ],
   [
    "C44",
    "x_vars[5,E]"
   ],
   [
    "C45",
    "x_vars[5,F]"
   ],
   [
    "C46",
    "x_vars[5,G]"
   ],
   [
    "C47",
    "x_vars[5,H]"
   ],
   [
    "C48",
    "x_vars[5,I]"
   ],
   [
    "C49",
    "x_vars[5,J]"
   ],
   [
    "C50",
    "x_vars[6,A]"
   ],
   [
    "C51",
    "x_vars[6,B]"
   ],
   [
    "C52",
    "x_vars[6,C]"
   ],
   [
    "C53",
    "x_vars[6,D]"
   ],
   [
    "C54",
    "x_vars[6,E]"
   ],
   [
    "C55",
    "x_vars[6,F]"
   ],
   [
    "C56",
    "x_vars[6,G]"
   ],
   [
    "C57",
    "x_vars[6,H]"
   ],
   [
    "C58",
    "x_vars[6,I]"
   ],
   [
    "C59",
    "x_vars[6,J]"
   ],
   [
    "C60",
    "x_vars[7,A]"
   ],
   [
    "C61",
    "x_vars[7,B]"
   ],
   [
    "C62",
    "x_vars[7,C]"
   ],
   [
    "C63",
    "x_vars[7,D]"
   ],
   [
    "C64",
    "x_vars[7,E]"
   ],
   [
    "C65",
    "x_vars[7,F]"
   ],
   [
    "C66",
    "x_vars[7,G]"
   ],
   [
    "C67",
    "x_vars[7,H]"
   ],
   [
    "C68",
    "x_vars[7,I]"
   ],
   [
    "C69",
    "x_vars[7,J]"
   ],
   [
    "C70",
    "x_vars[8,A]"
   ],
   [
    "C71",
    "x_vars[8,B]"
   ],
   [
    "C72",
    "x_vars[8,C]"
   ],
   [
    "C73",
    "x_vars[8,D]"
   ],
   [
    "C74",
    "x_vars[8,E]"
   ],
   [
    "C75",
    "x_vars[8,F]"
   ],
   [
    "C76",
    "x_vars[8,G]"
   ],
   [
    "C77",
    "x_vars[8,H]"
   ],
   [
    "C78",
    "x_vars[8,I]"
   ],
   [
    "C79",
    "x_vars[8,J]"
   ],
   [
    "C80",
    "x_vars[9,A]"
   ],
   [
    "C81",
    "x_vars[9,B]"
   ],
   [
    "C82",
    "x_vars[9,C]"
   ],
   [
    "C83",
    "x_vars[9,D]"
   ],
   [
    "C84",
    "x_vars[9,E]"
   ],
   [
    "C85",
    "x_vars[9,F]"
   ],
   [
    "C86",
    "x_vars[9,G]"
   ],
   [
    "C87",
    "x_vars[9,H]"
   ],
   [
    "C88",
    "x_vars[9,I]"
   ],
   [
    "C89",
    "x_vars[9,J]"
   ],
   [
    "C90",
    "x_vars[10,A]"
   ],
   [
    "C91",
    "x_vars[10,B]"
   ],
   [
    "C92",
    "x_vars[10,C]"
   ],
   [
    "C93",
    "x_vars[10,D]"
   ],
   [
    "C94",
    "x_vars[10,E]"
   ],
   [
    "C95",
    "x_vars[10,F]"
   ],
   [
    "C96",
    "x_vars[10,G]"
   ],
   [
    "C97",
    "x_vars[10,H]"
   ],
   [
    "C98",
    "x_vars[10,I]"
   ],
   [
    "C99",
    "x_vars[10,J]"
   ],
   [
    "C100",
    "x_vars[11,A]"
   ],
   [
    "C101",
    "x_vars[11,B]"
   ],
   [
    "C102",
    "x_vars[11,C]"
   ],
   [
    "C103",
    "x_vars[11,D]"
   ],
   [
    "C104",
    "x_vars[11,E]"
   ],
   [
    "C105",
    "x_vars[11,F]"
   ],
   [
    "C106",
    "x_vars[11,G]"
   ],
   [
    "C107",
    "x_vars[11,H]"
   ],
   [
    "C108",
    "x_vars[11,I]"
   ],
   [
    "C109",
    "x_vars[11,J]"
   ],
   [
    "C110",
    "x_vars[12,A]"
   ],
   [
    "C111",
    "x_vars[12,B]"
   ],
   [
    "C112",
    "x_vars[12,C]"
   ],
   [
    "C113",
    "x_vars[12,D]"
   ],
   [
    "C114",
    "x_vars[12,E]"
   ],
   [
    "C115",
    "x_vars[12,F]"
   ],
   [
    "C116",
    "x_vars[12,G]"
   ],
   [
    "C117",
    "x_vars[12,H]"
   ],
   [
    "C118",
    "x_vars[12,I]"
   ],
   [
    "C119",
    "x_vars[12,J]"
   ],
   [
    "C120",
    "y_vars[1]"
   ],
   [
    "C121",
    "y_vars[2]"
   ],
   [
    "C122",
    "y_vars[3]"
   ],
   [
    "C123",
    "y_vars[4]"
   ],
   [
    "C124",
    "y_vars[5]"
   ],
   [
    "C125",
    "y_vars[6]"
   ],
   [
    "C126",
    "y_vars[7]"
   ],
   [
    "C127",
    "y_vars[8]"
   ],
   [
    "C128",
    "y_vars[9]"
   ],
   [
    "C129",
    "y_vars[10]"
   ],
   [
    "C130",
    "y_vars[11]"
   ],
   [
    "C131",
    "y_vars[12]"
   ]
  ],
  "primary_mapping": {
   "x[9,A]": "x_vars[9,A]",
   "x[8,A]": "x_vars[8,A]",
   "x[3,F]": "x_vars[3,F]",
   "x[10,H]": "x_vars[10,H]",
   "x[3,A]": "x_vars[3,A]",
   "x[9,G]": "x_vars[9,G]",
   "x[5,J]": "x_vars[5,J]",
   "x[12,A]": "x_vars[12,A]",
   "x[5,D]": "x_vars[5,D]",
   "x[8,H]": "x_vars[8,H]",
   "x[7,G]": "x_vars[7,G]",
   "x[3,H]": "x_vars[3,H]",
   "x[12,E]": "x_vars[12,E]",
   "x[10,D]": "x_vars[10,D]",
   "x[9,B]": "x_vars[9,B]",
   "x[11,B]": "x_vars[11,B]",
   "x[11,C]": "x_vars[11,C]",
   "x[12,J]": "x_vars[12,J]",
   "x[1,I]": "x_vars[1,I]",
   "x[11,G]": "x_vars[11,G]",
   "x[9,J]": "x_vars[9,J]",
   "x[5,A]": "x_vars[5,A]",
   "x[12,B]": "x_vars[12,B]",
   "x[3,D]": "x_vars[3,D]",
   "x[4,H]": "x_vars[4,H]",
   "x[11,I]": "x_vars[11,I]",
   "x[6,E]": "x_vars[6,E]",
   "x[11,E]": "x_vars[11,E]",
   "x[7,B]": "x_vars[7,B]",
   "x[4,E]": "x_vars[4,E]",
   "x[1,D]": "x_vars[1,D]",
   "x[9,I]": "x_vars[9,I]",
   "x[10,C]": "x_vars[10,C]",
   "x[8,B]": "x_vars[8,B]",
   "x[1,A]": "x_vars[1,A]",
   "x[3,J]": "x_vars[3,J]",
   "x[2,B]": "x_vars[2,B]",
   "x[6,D]": "x_vars[6,D]",
   "x[4,I]": "x_vars[4,I]",
   "x[5,E]": "x_vars[5,E]",
   "x[6,I]": "x_vars[6,I]",
   "x[2,A]": "x_vars[2,A]",
   "x[10,I]": "x_vars[10,I]",
   "x[10,F]": "x_vars[10,F]",
   "x[5,B]": "x_vars[5,B]",
   "x[7,I]": "x_vars[7,I]",
   "x[1,G]": "x_vars[1,G]",
   "x[6,C]": "x_vars[6,C]",
   "x[9,F]": "x_vars[9,F]",
   "x[2,E]": "x_vars[2,E]",
   "x[2,D]": "x_vars[2,D]",
   "x[9,D]": "x_vars[9,D]",
   "x[6,F]": "x_vars[6,F]",
   "x[5,G]": "x_vars[5,G]",
   "x[10,G]": "x_vars[10,G]",
   "x[7,A]": "x_vars[7,A]",
   "x[8,F]": "x_vars[8,F]",
   "x[6,B]": "x_vars[6,B]",
   "x[12,F]": "x_vars[12,F]",
   "x[2,C]": "x_vars[2,C]",
   "x[3,E]": "x_vars[3,E]",
   "x[8,D]": "x_vars[8,D]",
   "x[7,D]": "x_vars[7,D]",
   "x[1,J]": "x_vars[1,J]",
   "x[10,J]": "x_vars[10,J]",
   "x[4,J]": "x_vars[4,J]",
   "x[1,B]": "x_vars[1,B]",
   "x[10,B]": "x_vars[10,B]",
   "x[10,A]": "x_vars[10,A]",
   "x[8,G]": "x_vars[8,G]",
   "x[6,A]": "x_vars[6,A]",
   "x[2,H]": "x_vars[2,H]",
   "x[2,J]": "x_vars[2,J]",
   "x[12,G]": "x_vars[12,G]",
   "x[3,B]": "x_vars[3,B]",
   "x[8,I]": "x_vars[8,I]",
   "x[2,I]": "x_vars[2,I]",
   "x[7,H]": "x_vars[7,H]",
   "x[5,H]": "x_vars[5,H]",
   "x[11,A]": "x_vars[11,A]",
   "x[1,E]": "x_vars[1,E]",
   "x[7,F]": "x_vars[7,F]",
   "x[12,D]": "x_vars[12,D]",
   "x[2,F]": "x_vars[2,F]",
   "x[4,D]": "x_vars[4,D]",
   "x[8,C]": "x_vars[8,C]",
   "x[2,G]": "x_vars[2,G]",
   "x[12,C]": "x_vars[12,C]",
   "x[11,F]": "x_vars[11,F]",
   "x[12,H]": "x_vars[12,H]",
   "x[8,J]": "x_vars[8,J]",
   "x[4,G]": "x_vars[4,G]",
   "x[4,A]": "x_vars[4,A]",
   "x[9,H]": "x_vars[9,H]",
   "x[6,G]": "x_vars[6,G]",
   "x[1,H]": "x_vars[1,H]",
   "x[9,C]": "x_vars[9,C]",
   "x[7,E]": "x_vars[7,E]",
   "x[10,E]": "x_vars[10,E]",
   "x[4,F]": "x_vars[4,F]",
   "x[11,D]": "x_vars[11,D]",
   "x[12,I]": "x_vars[12,I]",
   "x[5,I]": "x_vars[5,I]",
   "x[5,C]": "x_vars[5,C]",
   "x[3,C]": "x_vars[3,C]",
   "x[3,I]": "x_vars[3,I]",
   "x[1,F]": "x_vars[1,F]",
   "x[6,J]": "x_vars[6,J]",
   "x[11,H]": "x_vars[11,H]",
   "x[11,J]": "x_vars[11,J]",
   "x[1,C]": "x_vars[1,C]",
   "x[4,B]": "x_vars[4,B]",
   "x[4,C]": "x_vars[4,C]",
   "x[6,H]": "x_vars[6,H]",
   "x[7,C]": "x_vars[7,C]",
   "x[5,F]": "x_vars[5,F]",
   "x[8,E]": "x_vars[8,E]",
   "x[9,E]": "x_vars[9,E]",
   "x[3,G]": "x_vars[3,G]",
   "x[7,J]": "x_vars[7,J]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-084": {
  "source_code_sha256": "e222cf8f78d36e22b296434f9194aa18e39dc7ecb1c682bca0bf0683e044ba4f",
  "original_solution_sha256": "f476b57efa4909a18430a2b6560c6abbdfecc3ed8bc7bc73385121143a3798a3",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[2,1]"
   ],
   [
    "C4",
    "x_vars[2,2]"
   ],
   [
    "C5",
    "x_vars[2,3]"
   ],
   [
    "C6",
    "x_vars[3,1]"
   ],
   [
    "C7",
    "x_vars[3,2]"
   ],
   [
    "C8",
    "x_vars[3,3]"
   ],
   [
    "C9",
    "x_vars[4,1]"
   ],
   [
    "C10",
    "x_vars[4,2]"
   ],
   [
    "C11",
    "x_vars[4,3]"
   ],
   [
    "C12",
    "x_vars[5,1]"
   ],
   [
    "C13",
    "x_vars[5,2]"
   ],
   [
    "C14",
    "x_vars[5,3]"
   ],
   [
    "C15",
    "x_vars[6,1]"
   ],
   [
    "C16",
    "x_vars[6,2]"
   ],
   [
    "C17",
    "x_vars[6,3]"
   ],
   [
    "C18",
    "x_vars[7,1]"
   ],
   [
    "C19",
    "x_vars[7,2]"
   ],
   [
    "C20",
    "x_vars[7,3]"
   ],
   [
    "C21",
    "x_vars[8,1]"
   ],
   [
    "C22",
    "x_vars[8,2]"
   ],
   [
    "C23",
    "x_vars[8,3]"
   ],
   [
    "C24",
    "x_vars[9,1]"
   ],
   [
    "C25",
    "x_vars[9,2]"
   ],
   [
    "C26",
    "x_vars[9,3]"
   ],
   [
    "C27",
    "x_vars[10,1]"
   ],
   [
    "C28",
    "x_vars[10,2]"
   ],
   [
    "C29",
    "x_vars[10,3]"
   ],
   [
    "C30",
    "x_vars[11,1]"
   ],
   [
    "C31",
    "x_vars[11,2]"
   ],
   [
    "C32",
    "x_vars[11,3]"
   ],
   [
    "C33",
    "x_vars[12,1]"
   ],
   [
    "C34",
    "x_vars[12,2]"
   ],
   [
    "C35",
    "x_vars[12,3]"
   ],
   [
    "C36",
    "x_vars[13,1]"
   ],
   [
    "C37",
    "x_vars[13,2]"
   ],
   [
    "C38",
    "x_vars[13,3]"
   ],
   [
    "C39",
    "x_vars[14,1]"
   ],
   [
    "C40",
    "x_vars[14,2]"
   ],
   [
    "C41",
    "x_vars[14,3]"
   ],
   [
    "C42",
    "x_vars[15,1]"
   ],
   [
    "C43",
    "x_vars[15,2]"
   ],
   [
    "C44",
    "x_vars[15,3]"
   ],
   [
    "C45",
    "x_vars[16,1]"
   ],
   [
    "C46",
    "x_vars[16,2]"
   ],
   [
    "C47",
    "x_vars[16,3]"
   ],
   [
    "C48",
    "x_vars[17,1]"
   ],
   [
    "C49",
    "x_vars[17,2]"
   ],
   [
    "C50",
    "x_vars[17,3]"
   ],
   [
    "C51",
    "x_vars[18,1]"
   ],
   [
    "C52",
    "x_vars[18,2]"
   ],
   [
    "C53",
    "x_vars[18,3]"
   ],
   [
    "C54",
    "x_vars[19,1]"
   ],
   [
    "C55",
    "x_vars[19,2]"
   ],
   [
    "C56",
    "x_vars[19,3]"
   ],
   [
    "C57",
    "x_vars[20,1]"
   ],
   [
    "C58",
    "x_vars[20,2]"
   ],
   [
    "C59",
    "x_vars[20,3]"
   ],
   [
    "C60",
    "x_vars[21,1]"
   ],
   [
    "C61",
    "x_vars[21,2]"
   ],
   [
    "C62",
    "x_vars[21,3]"
   ],
   [
    "C63",
    "x_vars[22,1]"
   ],
   [
    "C64",
    "x_vars[22,2]"
   ],
   [
    "C65",
    "x_vars[22,3]"
   ],
   [
    "C66",
    "x_vars[23,1]"
   ],
   [
    "C67",
    "x_vars[23,2]"
   ],
   [
    "C68",
    "x_vars[23,3]"
   ],
   [
    "C69",
    "x_vars[24,1]"
   ],
   [
    "C70",
    "x_vars[24,2]"
   ],
   [
    "C71",
    "x_vars[24,3]"
   ],
   [
    "C72",
    "x_vars[25,1]"
   ],
   [
    "C73",
    "x_vars[25,2]"
   ],
   [
    "C74",
    "x_vars[25,3]"
   ],
   [
    "C75",
    "x_vars[26,1]"
   ],
   [
    "C76",
    "x_vars[26,2]"
   ],
   [
    "C77",
    "x_vars[26,3]"
   ],
   [
    "C78",
    "x_vars[27,1]"
   ],
   [
    "C79",
    "x_vars[27,2]"
   ],
   [
    "C80",
    "x_vars[27,3]"
   ],
   [
    "C81",
    "x_vars[28,1]"
   ],
   [
    "C82",
    "x_vars[28,2]"
   ],
   [
    "C83",
    "x_vars[28,3]"
   ],
   [
    "C84",
    "x_vars[29,1]"
   ],
   [
    "C85",
    "x_vars[29,2]"
   ],
   [
    "C86",
    "x_vars[29,3]"
   ],
   [
    "C87",
    "x_vars[30,1]"
   ],
   [
    "C88",
    "x_vars[30,2]"
   ],
   [
    "C89",
    "x_vars[30,3]"
   ],
   [
    "C90",
    "x_vars[31,1]"
   ],
   [
    "C91",
    "x_vars[31,2]"
   ],
   [
    "C92",
    "x_vars[31,3]"
   ],
   [
    "C93",
    "x_vars[32,1]"
   ],
   [
    "C94",
    "x_vars[32,2]"
   ],
   [
    "C95",
    "x_vars[32,3]"
   ],
   [
    "C96",
    "x_vars[33,1]"
   ],
   [
    "C97",
    "x_vars[33,2]"
   ],
   [
    "C98",
    "x_vars[33,3]"
   ],
   [
    "C99",
    "x_vars[34,1]"
   ],
   [
    "C100",
    "x_vars[34,2]"
   ],
   [
    "C101",
    "x_vars[34,3]"
   ],
   [
    "C102",
    "x_vars[35,1]"
   ],
   [
    "C103",
    "x_vars[35,2]"
   ],
   [
    "C104",
    "x_vars[35,3]"
   ],
   [
    "C105",
    "x_vars[36,1]"
   ],
   [
    "C106",
    "x_vars[36,2]"
   ],
   [
    "C107",
    "x_vars[36,3]"
   ],
   [
    "C108",
    "x_vars[37,1]"
   ],
   [
    "C109",
    "x_vars[37,2]"
   ],
   [
    "C110",
    "x_vars[37,3]"
   ],
   [
    "C111",
    "x_vars[38,1]"
   ],
   [
    "C112",
    "x_vars[38,2]"
   ],
   [
    "C113",
    "x_vars[38,3]"
   ],
   [
    "C114",
    "x_vars[39,1]"
   ],
   [
    "C115",
    "x_vars[39,2]"
   ],
   [
    "C116",
    "x_vars[39,3]"
   ],
   [
    "C117",
    "x_vars[40,1]"
   ],
   [
    "C118",
    "x_vars[40,2]"
   ],
   [
    "C119",
    "x_vars[40,3]"
   ],
   [
    "Cmax",
    "Cmax"
   ]
  ],
  "primary_mapping": {
   "x[1,1]": "x_vars[1,1]",
   "x[1,2]": "x_vars[2,1]",
   "x[1,3]": "x_vars[3,1]",
   "x[1,4]": "x_vars[4,1]",
   "x[1,5]": "x_vars[5,1]",
   "x[1,6]": "x_vars[6,1]",
   "x[1,7]": "x_vars[7,1]",
   "x[1,8]": "x_vars[8,1]",
   "x[1,9]": "x_vars[9,1]",
   "x[1,10]": "x_vars[10,1]",
   "x[1,11]": "x_vars[11,1]",
   "x[1,12]": "x_vars[12,1]",
   "x[1,13]": "x_vars[13,1]",
   "x[1,14]": "x_vars[14,1]",
   "x[1,15]": "x_vars[15,1]",
   "x[1,16]": "x_vars[16,1]",
   "x[1,17]": "x_vars[17,1]",
   "x[1,18]": "x_vars[18,1]",
   "x[1,19]": "x_vars[19,1]",
   "x[1,20]": "x_vars[20,1]",
   "x[1,21]": "x_vars[21,1]",
   "x[1,22]": "x_vars[22,1]",
   "x[1,23]": "x_vars[23,1]",
   "x[1,24]": "x_vars[24,1]",
   "x[1,25]": "x_vars[25,1]",
   "x[1,26]": "x_vars[26,1]",
   "x[1,27]": "x_vars[27,1]",
   "x[1,28]": "x_vars[28,1]",
   "x[1,29]": "x_vars[29,1]",
   "x[1,30]": "x_vars[30,1]",
   "x[1,31]": "x_vars[31,1]",
   "x[1,32]": "x_vars[32,1]",
   "x[1,33]": "x_vars[33,1]",
   "x[1,34]": "x_vars[34,1]",
   "x[1,35]": "x_vars[35,1]",
   "x[1,36]": "x_vars[36,1]",
   "x[1,37]": "x_vars[37,1]",
   "x[1,38]": "x_vars[38,1]",
   "x[1,39]": "x_vars[39,1]",
   "x[1,40]": "x_vars[40,1]",
   "x[2,1]": "x_vars[1,2]",
   "x[2,2]": "x_vars[2,2]",
   "x[2,3]": "x_vars[3,2]",
   "x[2,4]": "x_vars[4,2]",
   "x[2,5]": "x_vars[5,2]",
   "x[2,6]": "x_vars[6,2]",
   "x[2,7]": "x_vars[7,2]",
   "x[2,8]": "x_vars[8,2]",
   "x[2,9]": "x_vars[9,2]",
   "x[2,10]": "x_vars[10,2]",
   "x[2,11]": "x_vars[11,2]",
   "x[2,12]": "x_vars[12,2]",
   "x[2,13]": "x_vars[13,2]",
   "x[2,14]": "x_vars[14,2]",
   "x[2,15]": "x_vars[15,2]",
   "x[2,16]": "x_vars[16,2]",
   "x[2,17]": "x_vars[17,2]",
   "x[2,18]": "x_vars[18,2]",
   "x[2,19]": "x_vars[19,2]",
   "x[2,20]": "x_vars[20,2]",
   "x[2,21]": "x_vars[21,2]",
   "x[2,22]": "x_vars[22,2]",
   "x[2,23]": "x_vars[23,2]",
   "x[2,24]": "x_vars[24,2]",
   "x[2,25]": "x_vars[25,2]",
   "x[2,26]": "x_vars[26,2]",
   "x[2,27]": "x_vars[27,2]",
   "x[2,28]": "x_vars[28,2]",
   "x[2,29]": "x_vars[29,2]",
   "x[2,30]": "x_vars[30,2]",
   "x[2,31]": "x_vars[31,2]",
   "x[2,32]": "x_vars[32,2]",
   "x[2,33]": "x_vars[33,2]",
   "x[2,34]": "x_vars[34,2]",
   "x[2,35]": "x_vars[35,2]",
   "x[2,36]": "x_vars[36,2]",
   "x[2,37]": "x_vars[37,2]",
   "x[2,38]": "x_vars[38,2]",
   "x[2,39]": "x_vars[39,2]",
   "x[2,40]": "x_vars[40,2]",
   "x[3,1]": "x_vars[1,3]",
   "x[3,2]": "x_vars[2,3]",
   "x[3,3]": "x_vars[3,3]",
   "x[3,4]": "x_vars[4,3]",
   "x[3,5]": "x_vars[5,3]",
   "x[3,6]": "x_vars[6,3]",
   "x[3,7]": "x_vars[7,3]",
   "x[3,8]": "x_vars[8,3]",
   "x[3,9]": "x_vars[9,3]",
   "x[3,10]": "x_vars[10,3]",
   "x[3,11]": "x_vars[11,3]",
   "x[3,12]": "x_vars[12,3]",
   "x[3,13]": "x_vars[13,3]",
   "x[3,14]": "x_vars[14,3]",
   "x[3,15]": "x_vars[15,3]",
   "x[3,16]": "x_vars[16,3]",
   "x[3,17]": "x_vars[17,3]",
   "x[3,18]": "x_vars[18,3]",
   "x[3,19]": "x_vars[19,3]",
   "x[3,20]": "x_vars[20,3]",
   "x[3,21]": "x_vars[21,3]",
   "x[3,22]": "x_vars[22,3]",
   "x[3,23]": "x_vars[23,3]",
   "x[3,24]": "x_vars[24,3]",
   "x[3,25]": "x_vars[25,3]",
   "x[3,26]": "x_vars[26,3]",
   "x[3,27]": "x_vars[27,3]",
   "x[3,28]": "x_vars[28,3]",
   "x[3,29]": "x_vars[29,3]",
   "x[3,30]": "x_vars[30,3]",
   "x[3,31]": "x_vars[31,3]",
   "x[3,32]": "x_vars[32,3]",
   "x[3,33]": "x_vars[33,3]",
   "x[3,34]": "x_vars[34,3]",
   "x[3,35]": "x_vars[35,3]",
   "x[3,36]": "x_vars[36,3]",
   "x[3,37]": "x_vars[37,3]",
   "x[3,38]": "x_vars[38,3]",
   "x[3,39]": "x_vars[39,3]",
   "x[3,40]": "x_vars[40,3]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-085": {
  "source_code_sha256": "f0cf088924ae1ddec71df22d55c22f49f70894b12ca27e50944c89e951215e63",
  "original_solution_sha256": "bc5a1d6c73e928c52c48021919d8502c2d6b4df133eba0d23cfbd3835a92e9c0",
  "bindings": [
   [
    "C0",
    "x_vars[1,2]"
   ],
   [
    "C1",
    "x_vars[1,3]"
   ],
   [
    "C2",
    "x_vars[1,4]"
   ],
   [
    "C3",
    "x_vars[1,5]"
   ],
   [
    "C4",
    "x_vars[1,6]"
   ],
   [
    "C5",
    "x_vars[1,7]"
   ],
   [
    "C6",
    "x_vars[1,8]"
   ],
   [
    "C7",
    "x_vars[1,9]"
   ],
   [
    "C8",
    "x_vars[1,10]"
   ],
   [
    "C9",
    "x_vars[1,11]"
   ],
   [
    "C10",
    "x_vars[1,12]"
   ],
   [
    "C11",
    "x_vars[1,13]"
   ],
   [
    "C12",
    "x_vars[1,14]"
   ],
   [
    "C13",
    "x_vars[1,15]"
   ],
   [
    "C14",
    "x_vars[2,1]"
   ],
   [
    "C15",
    "x_vars[2,3]"
   ],
   [
    "C16",
    "x_vars[2,4]"
   ],
   [
    "C17",
    "x_vars[2,5]"
   ],
   [
    "C18",
    "x_vars[2,6]"
   ],
   [
    "C19",
    "x_vars[2,7]"
   ],
   [
    "C20",
    "x_vars[2,8]"
   ],
   [
    "C21",
    "x_vars[2,9]"
   ],
   [
    "C22",
    "x_vars[2,10]"
   ],
   [
    "C23",
    "x_vars[2,11]"
   ],
   [
    "C24",
    "x_vars[2,12]"
   ],
   [
    "C25",
    "x_vars[2,13]"
   ],
   [
    "C26",
    "x_vars[2,14]"
   ],
   [
    "C27",
    "x_vars[2,15]"
   ],
   [
    "C28",
    "x_vars[3,1]"
   ],
   [
    "C29",
    "x_vars[3,2]"
   ],
   [
    "C30",
    "x_vars[3,4]"
   ],
   [
    "C31",
    "x_vars[3,5]"
   ],
   [
    "C32",
    "x_vars[3,6]"
   ],
   [
    "C33",
    "x_vars[3,7]"
   ],
   [
    "C34",
    "x_vars[3,8]"
   ],
   [
    "C35",
    "x_vars[3,9]"
   ],
   [
    "C36",
    "x_vars[3,10]"
   ],
   [
    "C37",
    "x_vars[3,11]"
   ],
   [
    "C38",
    "x_vars[3,12]"
   ],
   [
    "C39",
    "x_vars[3,13]"
   ],
   [
    "C40",
    "x_vars[3,14]"
   ],
   [
    "C41",
    "x_vars[3,15]"
   ],
   [
    "C42",
    "x_vars[4,1]"
   ],
   [
    "C43",
    "x_vars[4,2]"
   ],
   [
    "C44",
    "x_vars[4,3]"
   ],
   [
    "C45",
    "x_vars[4,5]"
   ],
   [
    "C46",
    "x_vars[4,6]"
   ],
   [
    "C47",
    "x_vars[4,7]"
   ],
   [
    "C48",
    "x_vars[4,8]"
   ],
   [
    "C49",
    "x_vars[4,9]"
   ],
   [
    "C50",
    "x_vars[4,10]"
   ],
   [
    "C51",
    "x_vars[4,11]"
   ],
   [
    "C52",
    "x_vars[4,12]"
   ],
   [
    "C53",
    "x_vars[4,13]"
   ],
   [
    "C54",
    "x_vars[4,14]"
   ],
   [
    "C55",
    "x_vars[4,15]"
   ],
   [
    "C56",
    "x_vars[5,1]"
   ],
   [
    "C57",
    "x_vars[5,2]"
   ],
   [
    "C58",
    "x_vars[5,3]"
   ],
   [
    "C59",
    "x_vars[5,4]"
   ],
   [
    "C60",
    "x_vars[5,6]"
   ],
   [
    "C61",
    "x_vars[5,7]"
   ],
   [
    "C62",
    "x_vars[5,8]"
   ],
   [
    "C63",
    "x_vars[5,9]"
   ],
   [
    "C64",
    "x_vars[5,10]"
   ],
   [
    "C65",
    "x_vars[5,11]"
   ],
   [
    "C66",
    "x_vars[5,12]"
   ],
   [
    "C67",
    "x_vars[5,13]"
   ],
   [
    "C68",
    "x_vars[5,14]"
   ],
   [
    "C69",
    "x_vars[5,15]"
   ],
   [
    "C70",
    "x_vars[6,1]"
   ],
   [
    "C71",
    "x_vars[6,2]"
   ],
   [
    "C72",
    "x_vars[6,3]"
   ],
   [
    "C73",
    "x_vars[6,4]"
   ],
   [
    "C74",
    "x_vars[6,5]"
   ],
   [
    "C75",
    "x_vars[6,7]"
   ],
   [
    "C76",
    "x_vars[6,8]"
   ],
   [
    "C77",
    "x_vars[6,9]"
   ],
   [
    "C78",
    "x_vars[6,10]"
   ],
   [
    "C79",
    "x_vars[6,11]"
   ],
   [
    "C80",
    "x_vars[6,12]"
   ],
   [
    "C81",
    "x_vars[6,13]"
   ],
   [
    "C82",
    "x_vars[6,14]"
   ],
   [
    "C83",
    "x_vars[6,15]"
   ],
   [
    "C84",
    "x_vars[7,1]"
   ],
   [
    "C85",
    "x_vars[7,2]"
   ],
   [
    "C86",
    "x_vars[7,3]"
   ],
   [
    "C87",
    "x_vars[7,4]"
   ],
   [
    "C88",
    "x_vars[7,5]"
   ],
   [
    "C89",
    "x_vars[7,6]"
   ],
   [
    "C90",
    "x_vars[7,8]"
   ],
   [
    "C91",
    "x_vars[7,9]"
   ],
   [
    "C92",
    "x_vars[7,10]"
   ],
   [
    "C93",
    "x_vars[7,11]"
   ],
   [
    "C94",
    "x_vars[7,12]"
   ],
   [
    "C95",
    "x_vars[7,13]"
   ],
   [
    "C96",
    "x_vars[7,14]"
   ],
   [
    "C97",
    "x_vars[7,15]"
   ],
   [
    "C98",
    "x_vars[8,1]"
   ],
   [
    "C99",
    "x_vars[8,2]"
   ],
   [
    "C100",
    "x_vars[8,3]"
   ],
   [
    "C101",
    "x_vars[8,4]"
   ],
   [
    "C102",
    "x_vars[8,5]"
   ],
   [
    "C103",
    "x_vars[8,6]"
   ],
   [
    "C104",
    "x_vars[8,7]"
   ],
   [
    "C105",
    "x_vars[8,9]"
   ],
   [
    "C106",
    "x_vars[8,10]"
   ],
   [
    "C107",
    "x_vars[8,11]"
   ],
   [
    "C108",
    "x_vars[8,12]"
   ],
   [
    "C109",
    "x_vars[8,13]"
   ],
   [
    "C110",
    "x_vars[8,14]"
   ],
   [
    "C111",
    "x_vars[8,15]"
   ],
   [
    "C112",
    "x_vars[9,1]"
   ],
   [
    "C113",
    "x_vars[9,2]"
   ],
   [
    "C114",
    "x_vars[9,3]"
   ],
   [
    "C115",
    "x_vars[9,4]"
   ],
   [
    "C116",
    "x_vars[9,5]"
   ],
   [
    "C117",
    "x_vars[9,6]"
   ],
   [
    "C118",
    "x_vars[9,7]"
   ],
   [
    "C119",
    "x_vars[9,8]"
   ],
   [
    "C120",
    "x_vars[9,10]"
   ],
   [
    "C121",
    "x_vars[9,11]"
   ],
   [
    "C122",
    "x_vars[9,12]"
   ],
   [
    "C123",
    "x_vars[9,13]"
   ],
   [
    "C124",
    "x_vars[9,14]"
   ],
   [
    "C125",
    "x_vars[9,15]"
   ],
   [
    "C126",
    "x_vars[10,1]"
   ],
   [
    "C127",
    "x_vars[10,2]"
   ],
   [
    "C128",
    "x_vars[10,3]"
   ],
   [
    "C129",
    "x_vars[10,4]"
   ],
   [
    "C130",
    "x_vars[10,5]"
   ],
   [
    "C131",
    "x_vars[10,6]"
   ],
   [
    "C132",
    "x_vars[10,7]"
   ],
   [
    "C133",
    "x_vars[10,8]"
   ],
   [
    "C134",
    "x_vars[10,9]"
   ],
   [
    "C135",
    "x_vars[10,11]"
   ],
   [
    "C136",
    "x_vars[10,12]"
   ],
   [
    "C137",
    "x_vars[10,13]"
   ],
   [
    "C138",
    "x_vars[10,14]"
   ],
   [
    "C139",
    "x_vars[10,15]"
   ],
   [
    "C140",
    "x_vars[11,1]"
   ],
   [
    "C141",
    "x_vars[11,2]"
   ],
   [
    "C142",
    "x_vars[11,3]"
   ],
   [
    "C143",
    "x_vars[11,4]"
   ],
   [
    "C144",
    "x_vars[11,5]"
   ],
   [
    "C145",
    "x_vars[11,6]"
   ],
   [
    "C146",
    "x_vars[11,7]"
   ],
   [
    "C147",
    "x_vars[11,8]"
   ],
   [
    "C148",
    "x_vars[11,9]"
   ],
   [
    "C149",
    "x_vars[11,10]"
   ],
   [
    "C150",
    "x_vars[11,12]"
   ],
   [
    "C151",
    "x_vars[11,13]"
   ],
   [
    "C152",
    "x_vars[11,14]"
   ],
   [
    "C153",
    "x_vars[11,15]"
   ],
   [
    "C154",
    "x_vars[12,1]"
   ],
   [
    "C155",
    "x_vars[12,2]"
   ],
   [
    "C156",
    "x_vars[12,3]"
   ],
   [
    "C157",
    "x_vars[12,4]"
   ],
   [
    "C158",
    "x_vars[12,5]"
   ],
   [
    "C159",
    "x_vars[12,6]"
   ],
   [
    "C160",
    "x_vars[12,7]"
   ],
   [
    "C161",
    "x_vars[12,8]"
   ],
   [
    "C162",
    "x_vars[12,9]"
   ],
   [
    "C163",
    "x_vars[12,10]"
   ],
   [
    "C164",
    "x_vars[12,11]"
   ],
   [
    "C165",
    "x_vars[12,13]"
   ],
   [
    "C166",
    "x_vars[12,14]"
   ],
   [
    "C167",
    "x_vars[12,15]"
   ],
   [
    "C168",
    "x_vars[13,1]"
   ],
   [
    "C169",
    "x_vars[13,2]"
   ],
   [
    "C170",
    "x_vars[13,3]"
   ],
   [
    "C171",
    "x_vars[13,4]"
   ],
   [
    "C172",
    "x_vars[13,5]"
   ],
   [
    "C173",
    "x_vars[13,6]"
   ],
   [
    "C174",
    "x_vars[13,7]"
   ],
   [
    "C175",
    "x_vars[13,8]"
   ],
   [
    "C176",
    "x_vars[13,9]"
   ],
   [
    "C177",
    "x_vars[13,10]"
   ],
   [
    "C178",
    "x_vars[13,11]"
   ],
   [
    "C179",
    "x_vars[13,12]"
   ],
   [
    "C180",
    "x_vars[13,14]"
   ],
   [
    "C181",
    "x_vars[13,15]"
   ],
   [
    "C182",
    "x_vars[14,1]"
   ],
   [
    "C183",
    "x_vars[14,2]"
   ],
   [
    "C184",
    "x_vars[14,3]"
   ],
   [
    "C185",
    "x_vars[14,4]"
   ],
   [
    "C186",
    "x_vars[14,5]"
   ],
   [
    "C187",
    "x_vars[14,6]"
   ],
   [
    "C188",
    "x_vars[14,7]"
   ],
   [
    "C189",
    "x_vars[14,8]"
   ],
   [
    "C190",
    "x_vars[14,9]"
   ],
   [
    "C191",
    "x_vars[14,10]"
   ],
   [
    "C192",
    "x_vars[14,11]"
   ],
   [
    "C193",
    "x_vars[14,12]"
   ],
   [
    "C194",
    "x_vars[14,13]"
   ],
   [
    "C195",
    "x_vars[14,15]"
   ],
   [
    "C196",
    "x_vars[15,1]"
   ],
   [
    "C197",
    "x_vars[15,2]"
   ],
   [
    "C198",
    "x_vars[15,3]"
   ],
   [
    "C199",
    "x_vars[15,4]"
   ],
   [
    "C200",
    "x_vars[15,5]"
   ],
   [
    "C201",
    "x_vars[15,6]"
   ],
   [
    "C202",
    "x_vars[15,7]"
   ],
   [
    "C203",
    "x_vars[15,8]"
   ],
   [
    "C204",
    "x_vars[15,9]"
   ],
   [
    "C205",
    "x_vars[15,10]"
   ],
   [
    "C206",
    "x_vars[15,11]"
   ],
   [
    "C207",
    "x_vars[15,12]"
   ],
   [
    "C208",
    "x_vars[15,13]"
   ],
   [
    "C209",
    "x_vars[15,14]"
   ],
   [
    "C210",
    "u_vars[2]"
   ],
   [
    "C211",
    "u_vars[3]"
   ],
   [
    "C212",
    "u_vars[4]"
   ],
   [
    "C213",
    "u_vars[5]"
   ],
   [
    "C214",
    "u_vars[6]"
   ],
   [
    "C215",
    "u_vars[7]"
   ],
   [
    "C216",
    "u_vars[8]"
   ],
   [
    "C217",
    "u_vars[9]"
   ],
   [
    "C218",
    "u_vars[10]"
   ],
   [
    "C219",
    "u_vars[11]"
   ],
   [
    "C220",
    "u_vars[12]"
   ],
   [
    "C221",
    "u_vars[13]"
   ],
   [
    "C222",
    "u_vars[14]"
   ],
   [
    "C223",
    "u_vars[15]"
   ]
  ],
  "primary_mapping": {
   "x[1,2]": "x_vars[1,2]",
   "x[1,3]": "x_vars[1,3]",
   "x[1,4]": "x_vars[1,4]",
   "x[1,5]": "x_vars[1,5]",
   "x[1,6]": "x_vars[1,6]",
   "x[1,7]": "x_vars[1,7]",
   "x[1,8]": "x_vars[1,8]",
   "x[1,9]": "x_vars[1,9]",
   "x[1,10]": "x_vars[1,10]",
   "x[1,11]": "x_vars[1,11]",
   "x[1,12]": "x_vars[1,12]",
   "x[1,13]": "x_vars[1,13]",
   "x[1,14]": "x_vars[1,14]",
   "x[1,15]": "x_vars[1,15]",
   "x[2,1]": "x_vars[2,1]",
   "x[2,3]": "x_vars[2,3]",
   "x[2,4]": "x_vars[2,4]",
   "x[2,5]": "x_vars[2,5]",
   "x[2,6]": "x_vars[2,6]",
   "x[2,7]": "x_vars[2,7]",
   "x[2,8]": "x_vars[2,8]",
   "x[2,9]": "x_vars[2,9]",
   "x[2,10]": "x_vars[2,10]",
   "x[2,11]": "x_vars[2,11]",
   "x[2,12]": "x_vars[2,12]",
   "x[2,13]": "x_vars[2,13]",
   "x[2,14]": "x_vars[2,14]",
   "x[2,15]": "x_vars[2,15]",
   "x[3,1]": "x_vars[3,1]",
   "x[3,2]": "x_vars[3,2]",
   "x[3,4]": "x_vars[3,4]",
   "x[3,5]": "x_vars[3,5]",
   "x[3,6]": "x_vars[3,6]",
   "x[3,7]": "x_vars[3,7]",
   "x[3,8]": "x_vars[3,8]",
   "x[3,9]": "x_vars[3,9]",
   "x[3,10]": "x_vars[3,10]",
   "x[3,11]": "x_vars[3,11]",
   "x[3,12]": "x_vars[3,12]",
   "x[3,13]": "x_vars[3,13]",
   "x[3,14]": "x_vars[3,14]",
   "x[3,15]": "x_vars[3,15]",
   "x[4,1]": "x_vars[4,1]",
   "x[4,2]": "x_vars[4,2]",
   "x[4,3]": "x_vars[4,3]",
   "x[4,5]": "x_vars[4,5]",
   "x[4,6]": "x_vars[4,6]",
   "x[4,7]": "x_vars[4,7]",
   "x[4,8]": "x_vars[4,8]",
   "x[4,9]": "x_vars[4,9]",
   "x[4,10]": "x_vars[4,10]",
   "x[4,11]": "x_vars[4,11]",
   "x[4,12]": "x_vars[4,12]",
   "x[4,13]": "x_vars[4,13]",
   "x[4,14]": "x_vars[4,14]",
   "x[4,15]": "x_vars[4,15]",
   "x[5,1]": "x_vars[5,1]",
   "x[5,2]": "x_vars[5,2]",
   "x[5,3]": "x_vars[5,3]",
   "x[5,4]": "x_vars[5,4]",
   "x[5,6]": "x_vars[5,6]",
   "x[5,7]": "x_vars[5,7]",
   "x[5,8]": "x_vars[5,8]",
   "x[5,9]": "x_vars[5,9]",
   "x[5,10]": "x_vars[5,10]",
   "x[5,11]": "x_vars[5,11]",
   "x[5,12]": "x_vars[5,12]",
   "x[5,13]": "x_vars[5,13]",
   "x[5,14]": "x_vars[5,14]",
   "x[5,15]": "x_vars[5,15]",
   "x[6,1]": "x_vars[6,1]",
   "x[6,2]": "x_vars[6,2]",
   "x[6,3]": "x_vars[6,3]",
   "x[6,4]": "x_vars[6,4]",
   "x[6,5]": "x_vars[6,5]",
   "x[6,7]": "x_vars[6,7]",
   "x[6,8]": "x_vars[6,8]",
   "x[6,9]": "x_vars[6,9]",
   "x[6,10]": "x_vars[6,10]",
   "x[6,11]": "x_vars[6,11]",
   "x[6,12]": "x_vars[6,12]",
   "x[6,13]": "x_vars[6,13]",
   "x[6,14]": "x_vars[6,14]",
   "x[6,15]": "x_vars[6,15]",
   "x[7,1]": "x_vars[7,1]",
   "x[7,2]": "x_vars[7,2]",
   "x[7,3]": "x_vars[7,3]",
   "x[7,4]": "x_vars[7,4]",
   "x[7,5]": "x_vars[7,5]",
   "x[7,6]": "x_vars[7,6]",
   "x[7,8]": "x_vars[7,8]",
   "x[7,9]": "x_vars[7,9]",
   "x[7,10]": "x_vars[7,10]",
   "x[7,11]": "x_vars[7,11]",
   "x[7,12]": "x_vars[7,12]",
   "x[7,13]": "x_vars[7,13]",
   "x[7,14]": "x_vars[7,14]",
   "x[7,15]": "x_vars[7,15]",
   "x[8,1]": "x_vars[8,1]",
   "x[8,2]": "x_vars[8,2]",
   "x[8,3]": "x_vars[8,3]",
   "x[8,4]": "x_vars[8,4]",
   "x[8,5]": "x_vars[8,5]",
   "x[8,6]": "x_vars[8,6]",
   "x[8,7]": "x_vars[8,7]",
   "x[8,9]": "x_vars[8,9]",
   "x[8,10]": "x_vars[8,10]",
   "x[8,11]": "x_vars[8,11]",
   "x[8,12]": "x_vars[8,12]",
   "x[8,13]": "x_vars[8,13]",
   "x[8,14]": "x_vars[8,14]",
   "x[8,15]": "x_vars[8,15]",
   "x[9,1]": "x_vars[9,1]",
   "x[9,2]": "x_vars[9,2]",
   "x[9,3]": "x_vars[9,3]",
   "x[9,4]": "x_vars[9,4]",
   "x[9,5]": "x_vars[9,5]",
   "x[9,6]": "x_vars[9,6]",
   "x[9,7]": "x_vars[9,7]",
   "x[9,8]": "x_vars[9,8]",
   "x[9,10]": "x_vars[9,10]",
   "x[9,11]": "x_vars[9,11]",
   "x[9,12]": "x_vars[9,12]",
   "x[9,13]": "x_vars[9,13]",
   "x[9,14]": "x_vars[9,14]",
   "x[9,15]": "x_vars[9,15]",
   "x[10,1]": "x_vars[10,1]",
   "x[10,2]": "x_vars[10,2]",
   "x[10,3]": "x_vars[10,3]",
   "x[10,4]": "x_vars[10,4]",
   "x[10,5]": "x_vars[10,5]",
   "x[10,6]": "x_vars[10,6]",
   "x[10,7]": "x_vars[10,7]",
   "x[10,8]": "x_vars[10,8]",
   "x[10,9]": "x_vars[10,9]",
   "x[10,11]": "x_vars[10,11]",
   "x[10,12]": "x_vars[10,12]",
   "x[10,13]": "x_vars[10,13]",
   "x[10,14]": "x_vars[10,14]",
   "x[10,15]": "x_vars[10,15]",
   "x[11,1]": "x_vars[11,1]",
   "x[11,2]": "x_vars[11,2]",
   "x[11,3]": "x_vars[11,3]",
   "x[11,4]": "x_vars[11,4]",
   "x[11,5]": "x_vars[11,5]",
   "x[11,6]": "x_vars[11,6]",
   "x[11,7]": "x_vars[11,7]",
   "x[11,8]": "x_vars[11,8]",
   "x[11,9]": "x_vars[11,9]",
   "x[11,10]": "x_vars[11,10]",
   "x[11,12]": "x_vars[11,12]",
   "x[11,13]": "x_vars[11,13]",
   "x[11,14]": "x_vars[11,14]",
   "x[11,15]": "x_vars[11,15]",
   "x[12,1]": "x_vars[12,1]",
   "x[12,2]": "x_vars[12,2]",
   "x[12,3]": "x_vars[12,3]",
   "x[12,4]": "x_vars[12,4]",
   "x[12,5]": "x_vars[12,5]",
   "x[12,6]": "x_vars[12,6]",
   "x[12,7]": "x_vars[12,7]",
   "x[12,8]": "x_vars[12,8]",
   "x[12,9]": "x_vars[12,9]",
   "x[12,10]": "x_vars[12,10]",
   "x[12,11]": "x_vars[12,11]",
   "x[12,13]": "x_vars[12,13]",
   "x[12,14]": "x_vars[12,14]",
   "x[12,15]": "x_vars[12,15]",
   "x[13,1]": "x_vars[13,1]",
   "x[13,2]": "x_vars[13,2]",
   "x[13,3]": "x_vars[13,3]",
   "x[13,4]": "x_vars[13,4]",
   "x[13,5]": "x_vars[13,5]",
   "x[13,6]": "x_vars[13,6]",
   "x[13,7]": "x_vars[13,7]",
   "x[13,8]": "x_vars[13,8]",
   "x[13,9]": "x_vars[13,9]",
   "x[13,10]": "x_vars[13,10]",
   "x[13,11]": "x_vars[13,11]",
   "x[13,12]": "x_vars[13,12]",
   "x[13,14]": "x_vars[13,14]",
   "x[13,15]": "x_vars[13,15]",
   "x[14,1]": "x_vars[14,1]",
   "x[14,2]": "x_vars[14,2]",
   "x[14,3]": "x_vars[14,3]",
   "x[14,4]": "x_vars[14,4]",
   "x[14,5]": "x_vars[14,5]",
   "x[14,6]": "x_vars[14,6]",
   "x[14,7]": "x_vars[14,7]",
   "x[14,8]": "x_vars[14,8]",
   "x[14,9]": "x_vars[14,9]",
   "x[14,10]": "x_vars[14,10]",
   "x[14,11]": "x_vars[14,11]",
   "x[14,12]": "x_vars[14,12]",
   "x[14,13]": "x_vars[14,13]",
   "x[14,15]": "x_vars[14,15]",
   "x[15,1]": "x_vars[15,1]",
   "x[15,2]": "x_vars[15,2]",
   "x[15,3]": "x_vars[15,3]",
   "x[15,4]": "x_vars[15,4]",
   "x[15,5]": "x_vars[15,5]",
   "x[15,6]": "x_vars[15,6]",
   "x[15,7]": "x_vars[15,7]",
   "x[15,8]": "x_vars[15,8]",
   "x[15,9]": "x_vars[15,9]",
   "x[15,10]": "x_vars[15,10]",
   "x[15,11]": "x_vars[15,11]",
   "x[15,12]": "x_vars[15,12]",
   "x[15,13]": "x_vars[15,13]",
   "x[15,14]": "x_vars[15,14]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-086": {
  "source_code_sha256": "db8ca734d30b8722b91a95108eb94ade2d0d290631ae95290519a7ddf6267551",
  "original_solution_sha256": "2a3170f3f626351be700f77678074a7fe23d39db2f413b09ac015f602e742eac",
  "bindings": [
   [
    "C0",
    "x_vars[I,Red]"
   ],
   [
    "C1",
    "x_vars[I,Yellow]"
   ],
   [
    "C2",
    "x_vars[I,Blue]"
   ],
   [
    "C3",
    "x_vars[II,Red]"
   ],
   [
    "C4",
    "x_vars[II,Yellow]"
   ],
   [
    "C5",
    "x_vars[II,Blue]"
   ],
   [
    "C6",
    "x_vars[III,Red]"
   ],
   [
    "C7",
    "x_vars[III,Yellow]"
   ],
   [
    "C8",
    "x_vars[III,Blue]"
   ]
  ],
  "primary_mapping": {
   "x[Red,I]": "x_vars[I,Red]",
   "x[Red,II]": "x_vars[II,Red]",
   "x[Red,III]": "x_vars[III,Red]",
   "x[Yellow,I]": "x_vars[I,Yellow]",
   "x[Yellow,II]": "x_vars[II,Yellow]",
   "x[Yellow,III]": "x_vars[III,Yellow]",
   "x[Blue,I]": "x_vars[I,Blue]",
   "x[Blue,II]": "x_vars[II,Blue]",
   "x[Blue,III]": "x_vars[III,Blue]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-087": {
  "source_code_sha256": "6b74106a1b78eb44cc121e6b32366cc756deb7a2e6a41ef3190bc63a7ef6d353",
  "original_solution_sha256": "76390fa4d0955f719484d36c10b60e3e7d6b05f3b0ceea21b2c6110921481ced",
  "bindings": [
   [
    "C0",
    "x_vars[A1]"
   ],
   [
    "C1",
    "x_vars[A2]"
   ],
   [
    "C2",
    "x_vars[A3]"
   ],
   [
    "C3",
    "x_vars[A4]"
   ],
   [
    "C4",
    "x_vars[A5]"
   ],
   [
    "C5",
    "x_vars[A6]"
   ],
   [
    "C6",
    "x_vars[A7]"
   ],
   [
    "C7",
    "x_vars[A8]"
   ],
   [
    "C8",
    "x_vars[A9]"
   ],
   [
    "C9",
    "x_vars[A10]"
   ],
   [
    "C10",
    "x_vars[A11]"
   ],
   [
    "C11",
    "x_vars[A12]"
   ],
   [
    "C12",
    "x_vars[A13]"
   ],
   [
    "C13",
    "x_vars[A14]"
   ],
   [
    "C14",
    "x_vars[A15]"
   ],
   [
    "C15",
    "x_vars[A16]"
   ],
   [
    "C16",
    "x_vars[A17]"
   ],
   [
    "C17",
    "x_vars[A18]"
   ],
   [
    "C18",
    "x_vars[A19]"
   ],
   [
    "C19",
    "x_vars[A20]"
   ],
   [
    "C20",
    "x_vars[A21]"
   ],
   [
    "C21",
    "x_vars[A22]"
   ],
   [
    "C22",
    "x_vars[A23]"
   ],
   [
    "C23",
    "x_vars[A24]"
   ],
   [
    "C24",
    "x_vars[A25]"
   ],
   [
    "C25",
    "x_vars[A26]"
   ],
   [
    "C26",
    "x_vars[A27]"
   ],
   [
    "C27",
    "x_vars[A28]"
   ],
   [
    "C28",
    "x_vars[A29]"
   ],
   [
    "C29",
    "x_vars[A30]"
   ],
   [
    "C30",
    "x_vars[A31]"
   ],
   [
    "C31",
    "x_vars[A32]"
   ],
   [
    "C32",
    "x_vars[A33]"
   ],
   [
    "C33",
    "x_vars[A34]"
   ],
   [
    "C34",
    "x_vars[A35]"
   ],
   [
    "C35",
    "x_vars[A36]"
   ],
   [
    "C36",
    "x_vars[A37]"
   ],
   [
    "C37",
    "x_vars[A38]"
   ],
   [
    "C38",
    "x_vars[A39]"
   ],
   [
    "C39",
    "x_vars[A40]"
   ],
   [
    "C40",
    "x_vars[A41]"
   ],
   [
    "C41",
    "x_vars[A42]"
   ],
   [
    "C42",
    "x_vars[A43]"
   ],
   [
    "C43",
    "x_vars[A44]"
   ],
   [
    "C44",
    "x_vars[A45]"
   ],
   [
    "C45",
    "x_vars[A46]"
   ],
   [
    "C46",
    "x_vars[A47]"
   ],
   [
    "C47",
    "x_vars[A48]"
   ],
   [
    "C48",
    "x_vars[A49]"
   ],
   [
    "C49",
    "x_vars[A50]"
   ],
   [
    "C50",
    "x_vars[A51]"
   ],
   [
    "C51",
    "x_vars[A52]"
   ],
   [
    "C52",
    "x_vars[A53]"
   ],
   [
    "C53",
    "x_vars[A54]"
   ],
   [
    "C54",
    "x_vars[A55]"
   ],
   [
    "C55",
    "x_vars[A56]"
   ],
   [
    "C56",
    "x_vars[A57]"
   ],
   [
    "C57",
    "x_vars[A58]"
   ],
   [
    "C58",
    "x_vars[A59]"
   ],
   [
    "C59",
    "x_vars[A60]"
   ],
   [
    "C60",
    "x_vars[A61]"
   ],
   [
    "C61",
    "x_vars[A62]"
   ],
   [
    "C62",
    "x_vars[A63]"
   ],
   [
    "C63",
    "x_vars[A64]"
   ],
   [
    "C64",
    "x_vars[A65]"
   ],
   [
    "C65",
    "x_vars[A66]"
   ],
   [
    "C66",
    "x_vars[A67]"
   ],
   [
    "C67",
    "x_vars[A68]"
   ],
   [
    "C68",
    "x_vars[A69]"
   ],
   [
    "C69",
    "x_vars[A70]"
   ],
   [
    "C70",
    "x_vars[A71]"
   ],
   [
    "C71",
    "x_vars[A72]"
   ],
   [
    "C72",
    "x_vars[A73]"
   ],
   [
    "C73",
    "x_vars[A74]"
   ],
   [
    "C74",
    "x_vars[A75]"
   ],
   [
    "C75",
    "x_vars[A76]"
   ],
   [
    "C76",
    "x_vars[A77]"
   ],
   [
    "C77",
    "x_vars[A78]"
   ],
   [
    "C78",
    "x_vars[A79]"
   ],
   [
    "C79",
    "x_vars[A80]"
   ],
   [
    "C80",
    "y_vars[A1]"
   ],
   [
    "C81",
    "y_vars[A2]"
   ],
   [
    "C82",
    "y_vars[A3]"
   ],
   [
    "C83",
    "y_vars[A4]"
   ],
   [
    "C84",
    "y_vars[A5]"
   ],
   [
    "C85",
    "y_vars[A6]"
   ],
   [
    "C86",
    "y_vars[A7]"
   ],
   [
    "C87",
    "y_vars[A8]"
   ],
   [
    "C88",
    "y_vars[A9]"
   ],
   [
    "C89",
    "y_vars[A10]"
   ],
   [
    "C90",
    "y_vars[A11]"
   ],
   [
    "C91",
    "y_vars[A12]"
   ],
   [
    "C92",
    "y_vars[A13]"
   ],
   [
    "C93",
    "y_vars[A14]"
   ],
   [
    "C94",
    "y_vars[A15]"
   ],
   [
    "C95",
    "y_vars[A16]"
   ],
   [
    "C96",
    "y_vars[A17]"
   ],
   [
    "C97",
    "y_vars[A18]"
   ],
   [
    "C98",
    "y_vars[A19]"
   ],
   [
    "C99",
    "y_vars[A20]"
   ],
   [
    "C100",
    "y_vars[A21]"
   ],
   [
    "C101",
    "y_vars[A22]"
   ],
   [
    "C102",
    "y_vars[A23]"
   ],
   [
    "C103",
    "y_vars[A24]"
   ],
   [
    "C104",
    "y_vars[A25]"
   ],
   [
    "C105",
    "y_vars[A26]"
   ],
   [
    "C106",
    "y_vars[A27]"
   ],
   [
    "C107",
    "y_vars[A28]"
   ],
   [
    "C108",
    "y_vars[A29]"
   ],
   [
    "C109",
    "y_vars[A30]"
   ],
   [
    "C110",
    "y_vars[A31]"
   ],
   [
    "C111",
    "y_vars[A32]"
   ],
   [
    "C112",
    "y_vars[A33]"
   ],
   [
    "C113",
    "y_vars[A34]"
   ],
   [
    "C114",
    "y_vars[A35]"
   ],
   [
    "C115",
    "y_vars[A36]"
   ],
   [
    "C116",
    "y_vars[A37]"
   ],
   [
    "C117",
    "y_vars[A38]"
   ],
   [
    "C118",
    "y_vars[A39]"
   ],
   [
    "C119",
    "y_vars[A40]"
   ],
   [
    "C120",
    "y_vars[A41]"
   ],
   [
    "C121",
    "y_vars[A42]"
   ],
   [
    "C122",
    "y_vars[A43]"
   ],
   [
    "C123",
    "y_vars[A44]"
   ],
   [
    "C124",
    "y_vars[A45]"
   ],
   [
    "C125",
    "y_vars[A46]"
   ],
   [
    "C126",
    "y_vars[A47]"
   ],
   [
    "C127",
    "y_vars[A48]"
   ],
   [
    "C128",
    "y_vars[A49]"
   ],
   [
    "C129",
    "y_vars[A50]"
   ],
   [
    "C130",
    "y_vars[A51]"
   ],
   [
    "C131",
    "y_vars[A52]"
   ],
   [
    "C132",
    "y_vars[A53]"
   ],
   [
    "C133",
    "y_vars[A54]"
   ],
   [
    "C134",
    "y_vars[A55]"
   ],
   [
    "C135",
    "y_vars[A56]"
   ],
   [
    "C136",
    "y_vars[A57]"
   ],
   [
    "C137",
    "y_vars[A58]"
   ],
   [
    "C138",
    "y_vars[A59]"
   ],
   [
    "C139",
    "y_vars[A60]"
   ],
   [
    "C140",
    "y_vars[A61]"
   ],
   [
    "C141",
    "y_vars[A62]"
   ],
   [
    "C142",
    "y_vars[A63]"
   ],
   [
    "C143",
    "y_vars[A64]"
   ],
   [
    "C144",
    "y_vars[A65]"
   ],
   [
    "C145",
    "y_vars[A66]"
   ],
   [
    "C146",
    "y_vars[A67]"
   ],
   [
    "C147",
    "y_vars[A68]"
   ],
   [
    "C148",
    "y_vars[A69]"
   ],
   [
    "C149",
    "y_vars[A70]"
   ],
   [
    "C150",
    "y_vars[A71]"
   ],
   [
    "C151",
    "y_vars[A72]"
   ],
   [
    "C152",
    "y_vars[A73]"
   ],
   [
    "C153",
    "y_vars[A74]"
   ],
   [
    "C154",
    "y_vars[A75]"
   ],
   [
    "C155",
    "y_vars[A76]"
   ],
   [
    "C156",
    "y_vars[A77]"
   ],
   [
    "C157",
    "y_vars[A78]"
   ],
   [
    "C158",
    "y_vars[A79]"
   ],
   [
    "C159",
    "y_vars[A80]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[A1]",
   "x[2]": "x_vars[A2]",
   "x[3]": "x_vars[A3]",
   "x[4]": "x_vars[A4]",
   "x[5]": "x_vars[A5]",
   "x[6]": "x_vars[A6]",
   "x[7]": "x_vars[A7]",
   "x[8]": "x_vars[A8]",
   "x[9]": "x_vars[A9]",
   "x[10]": "x_vars[A10]",
   "x[11]": "x_vars[A11]",
   "x[12]": "x_vars[A12]",
   "x[13]": "x_vars[A13]",
   "x[14]": "x_vars[A14]",
   "x[15]": "x_vars[A15]",
   "x[16]": "x_vars[A16]",
   "x[17]": "x_vars[A17]",
   "x[18]": "x_vars[A18]",
   "x[19]": "x_vars[A19]",
   "x[20]": "x_vars[A20]",
   "x[21]": "x_vars[A21]",
   "x[22]": "x_vars[A22]",
   "x[23]": "x_vars[A23]",
   "x[24]": "x_vars[A24]",
   "x[25]": "x_vars[A25]",
   "x[26]": "x_vars[A26]",
   "x[27]": "x_vars[A27]",
   "x[28]": "x_vars[A28]",
   "x[29]": "x_vars[A29]",
   "x[30]": "x_vars[A30]",
   "x[31]": "x_vars[A31]",
   "x[32]": "x_vars[A32]",
   "x[33]": "x_vars[A33]",
   "x[34]": "x_vars[A34]",
   "x[35]": "x_vars[A35]",
   "x[36]": "x_vars[A36]",
   "x[37]": "x_vars[A37]",
   "x[38]": "x_vars[A38]",
   "x[39]": "x_vars[A39]",
   "x[40]": "x_vars[A40]",
   "x[41]": "x_vars[A41]",
   "x[42]": "x_vars[A42]",
   "x[43]": "x_vars[A43]",
   "x[44]": "x_vars[A44]",
   "x[45]": "x_vars[A45]",
   "x[46]": "x_vars[A46]",
   "x[47]": "x_vars[A47]",
   "x[48]": "x_vars[A48]",
   "x[49]": "x_vars[A49]",
   "x[50]": "x_vars[A50]",
   "x[51]": "x_vars[A51]",
   "x[52]": "x_vars[A52]",
   "x[53]": "x_vars[A53]",
   "x[54]": "x_vars[A54]",
   "x[55]": "x_vars[A55]",
   "x[56]": "x_vars[A56]",
   "x[57]": "x_vars[A57]",
   "x[58]": "x_vars[A58]",
   "x[59]": "x_vars[A59]",
   "x[60]": "x_vars[A60]",
   "x[61]": "x_vars[A61]",
   "x[62]": "x_vars[A62]",
   "x[63]": "x_vars[A63]",
   "x[64]": "x_vars[A64]",
   "x[65]": "x_vars[A65]",
   "x[66]": "x_vars[A66]",
   "x[67]": "x_vars[A67]",
   "x[68]": "x_vars[A68]",
   "x[69]": "x_vars[A69]",
   "x[70]": "x_vars[A70]",
   "x[71]": "x_vars[A71]",
   "x[72]": "x_vars[A72]",
   "x[73]": "x_vars[A73]",
   "x[74]": "x_vars[A74]",
   "x[75]": "x_vars[A75]",
   "x[76]": "x_vars[A76]",
   "x[77]": "x_vars[A77]",
   "x[78]": "x_vars[A78]",
   "x[79]": "x_vars[A79]",
   "x[80]": "x_vars[A80]",
   "y[1]": "y_vars[A1]",
   "y[2]": "y_vars[A2]",
   "y[3]": "y_vars[A3]",
   "y[4]": "y_vars[A4]",
   "y[5]": "y_vars[A5]",
   "y[6]": "y_vars[A6]",
   "y[7]": "y_vars[A7]",
   "y[8]": "y_vars[A8]",
   "y[9]": "y_vars[A9]",
   "y[10]": "y_vars[A10]",
   "y[11]": "y_vars[A11]",
   "y[12]": "y_vars[A12]",
   "y[13]": "y_vars[A13]",
   "y[14]": "y_vars[A14]",
   "y[15]": "y_vars[A15]",
   "y[16]": "y_vars[A16]",
   "y[17]": "y_vars[A17]",
   "y[18]": "y_vars[A18]",
   "y[19]": "y_vars[A19]",
   "y[20]": "y_vars[A20]",
   "y[21]": "y_vars[A21]",
   "y[22]": "y_vars[A22]",
   "y[23]": "y_vars[A23]",
   "y[24]": "y_vars[A24]",
   "y[25]": "y_vars[A25]",
   "y[26]": "y_vars[A26]",
   "y[27]": "y_vars[A27]",
   "y[28]": "y_vars[A28]",
   "y[29]": "y_vars[A29]",
   "y[30]": "y_vars[A30]",
   "y[31]": "y_vars[A31]",
   "y[32]": "y_vars[A32]",
   "y[33]": "y_vars[A33]",
   "y[34]": "y_vars[A34]",
   "y[35]": "y_vars[A35]",
   "y[36]": "y_vars[A36]",
   "y[37]": "y_vars[A37]",
   "y[38]": "y_vars[A38]",
   "y[39]": "y_vars[A39]",
   "y[40]": "y_vars[A40]",
   "y[41]": "y_vars[A41]",
   "y[42]": "y_vars[A42]",
   "y[43]": "y_vars[A43]",
   "y[44]": "y_vars[A44]",
   "y[45]": "y_vars[A45]",
   "y[46]": "y_vars[A46]",
   "y[47]": "y_vars[A47]",
   "y[48]": "y_vars[A48]",
   "y[49]": "y_vars[A49]",
   "y[50]": "y_vars[A50]",
   "y[51]": "y_vars[A51]",
   "y[52]": "y_vars[A52]",
   "y[53]": "y_vars[A53]",
   "y[54]": "y_vars[A54]",
   "y[55]": "y_vars[A55]",
   "y[56]": "y_vars[A56]",
   "y[57]": "y_vars[A57]",
   "y[58]": "y_vars[A58]",
   "y[59]": "y_vars[A59]",
   "y[60]": "y_vars[A60]",
   "y[61]": "y_vars[A61]",
   "y[62]": "y_vars[A62]",
   "y[63]": "y_vars[A63]",
   "y[64]": "y_vars[A64]",
   "y[65]": "y_vars[A65]",
   "y[66]": "y_vars[A66]",
   "y[67]": "y_vars[A67]",
   "y[68]": "y_vars[A68]",
   "y[69]": "y_vars[A69]",
   "y[70]": "y_vars[A70]",
   "y[71]": "y_vars[A71]",
   "y[72]": "y_vars[A72]",
   "y[73]": "y_vars[A73]",
   "y[74]": "y_vars[A74]",
   "y[75]": "y_vars[A75]",
   "y[76]": "y_vars[A76]",
   "y[77]": "y_vars[A77]",
   "y[78]": "y_vars[A78]",
   "y[79]": "y_vars[A79]",
   "y[80]": "y_vars[A80]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-088": {
  "source_code_sha256": "381e621df74efc40db58b0ae91533cc876347a6bf66126c55f80b1befb54d7bf",
  "original_solution_sha256": "adec244846701a6ce834f3cba1af320e227572c5de8141ebabdea6a0aae7a9a6",
  "bindings": [
   [
    "C0",
    "x_vars[F1,C1]"
   ],
   [
    "C1",
    "x_vars[F1,C2]"
   ],
   [
    "C2",
    "x_vars[F1,C3]"
   ],
   [
    "C3",
    "x_vars[F1,C4]"
   ],
   [
    "C4",
    "x_vars[F1,C5]"
   ],
   [
    "C5",
    "x_vars[F1,C6]"
   ],
   [
    "C6",
    "x_vars[F1,C7]"
   ],
   [
    "C7",
    "x_vars[F1,C8]"
   ],
   [
    "C8",
    "x_vars[F1,C9]"
   ],
   [
    "C9",
    "x_vars[F1,C10]"
   ],
   [
    "C10",
    "x_vars[F1,C11]"
   ],
   [
    "C11",
    "x_vars[F1,C12]"
   ],
   [
    "C12",
    "x_vars[F1,C13]"
   ],
   [
    "C13",
    "x_vars[F1,C14]"
   ],
   [
    "C14",
    "x_vars[F1,C15]"
   ],
   [
    "C15",
    "x_vars[F2,C1]"
   ],
   [
    "C16",
    "x_vars[F2,C2]"
   ],
   [
    "C17",
    "x_vars[F2,C3]"
   ],
   [
    "C18",
    "x_vars[F2,C4]"
   ],
   [
    "C19",
    "x_vars[F2,C5]"
   ],
   [
    "C20",
    "x_vars[F2,C6]"
   ],
   [
    "C21",
    "x_vars[F2,C7]"
   ],
   [
    "C22",
    "x_vars[F2,C8]"
   ],
   [
    "C23",
    "x_vars[F2,C9]"
   ],
   [
    "C24",
    "x_vars[F2,C10]"
   ],
   [
    "C25",
    "x_vars[F2,C11]"
   ],
   [
    "C26",
    "x_vars[F2,C12]"
   ],
   [
    "C27",
    "x_vars[F2,C13]"
   ],
   [
    "C28",
    "x_vars[F2,C14]"
   ],
   [
    "C29",
    "x_vars[F2,C15]"
   ],
   [
    "C30",
    "x_vars[F3,C1]"
   ],
   [
    "C31",
    "x_vars[F3,C2]"
   ],
   [
    "C32",
    "x_vars[F3,C3]"
   ],
   [
    "C33",
    "x_vars[F3,C4]"
   ],
   [
    "C34",
    "x_vars[F3,C5]"
   ],
   [
    "C35",
    "x_vars[F3,C6]"
   ],
   [
    "C36",
    "x_vars[F3,C7]"
   ],
   [
    "C37",
    "x_vars[F3,C8]"
   ],
   [
    "C38",
    "x_vars[F3,C9]"
   ],
   [
    "C39",
    "x_vars[F3,C10]"
   ],
   [
    "C40",
    "x_vars[F3,C11]"
   ],
   [
    "C41",
    "x_vars[F3,C12]"
   ],
   [
    "C42",
    "x_vars[F3,C13]"
   ],
   [
    "C43",
    "x_vars[F3,C14]"
   ],
   [
    "C44",
    "x_vars[F3,C15]"
   ],
   [
    "C45",
    "x_vars[F4,C1]"
   ],
   [
    "C46",
    "x_vars[F4,C2]"
   ],
   [
    "C47",
    "x_vars[F4,C3]"
   ],
   [
    "C48",
    "x_vars[F4,C4]"
   ],
   [
    "C49",
    "x_vars[F4,C5]"
   ],
   [
    "C50",
    "x_vars[F4,C6]"
   ],
   [
    "C51",
    "x_vars[F4,C7]"
   ],
   [
    "C52",
    "x_vars[F4,C8]"
   ],
   [
    "C53",
    "x_vars[F4,C9]"
   ],
   [
    "C54",
    "x_vars[F4,C10]"
   ],
   [
    "C55",
    "x_vars[F4,C11]"
   ],
   [
    "C56",
    "x_vars[F4,C12]"
   ],
   [
    "C57",
    "x_vars[F4,C13]"
   ],
   [
    "C58",
    "x_vars[F4,C14]"
   ],
   [
    "C59",
    "x_vars[F4,C15]"
   ],
   [
    "C60",
    "x_vars[F5,C1]"
   ],
   [
    "C61",
    "x_vars[F5,C2]"
   ],
   [
    "C62",
    "x_vars[F5,C3]"
   ],
   [
    "C63",
    "x_vars[F5,C4]"
   ],
   [
    "C64",
    "x_vars[F5,C5]"
   ],
   [
    "C65",
    "x_vars[F5,C6]"
   ],
   [
    "C66",
    "x_vars[F5,C7]"
   ],
   [
    "C67",
    "x_vars[F5,C8]"
   ],
   [
    "C68",
    "x_vars[F5,C9]"
   ],
   [
    "C69",
    "x_vars[F5,C10]"
   ],
   [
    "C70",
    "x_vars[F5,C11]"
   ],
   [
    "C71",
    "x_vars[F5,C12]"
   ],
   [
    "C72",
    "x_vars[F5,C13]"
   ],
   [
    "C73",
    "x_vars[F5,C14]"
   ],
   [
    "C74",
    "x_vars[F5,C15]"
   ],
   [
    "C75",
    "x_vars[F6,C1]"
   ],
   [
    "C76",
    "x_vars[F6,C2]"
   ],
   [
    "C77",
    "x_vars[F6,C3]"
   ],
   [
    "C78",
    "x_vars[F6,C4]"
   ],
   [
    "C79",
    "x_vars[F6,C5]"
   ],
   [
    "C80",
    "x_vars[F6,C6]"
   ],
   [
    "C81",
    "x_vars[F6,C7]"
   ],
   [
    "C82",
    "x_vars[F6,C8]"
   ],
   [
    "C83",
    "x_vars[F6,C9]"
   ],
   [
    "C84",
    "x_vars[F6,C10]"
   ],
   [
    "C85",
    "x_vars[F6,C11]"
   ],
   [
    "C86",
    "x_vars[F6,C12]"
   ],
   [
    "C87",
    "x_vars[F6,C13]"
   ],
   [
    "C88",
    "x_vars[F6,C14]"
   ],
   [
    "C89",
    "x_vars[F6,C15]"
   ],
   [
    "C90",
    "x_vars[F7,C1]"
   ],
   [
    "C91",
    "x_vars[F7,C2]"
   ],
   [
    "C92",
    "x_vars[F7,C3]"
   ],
   [
    "C93",
    "x_vars[F7,C4]"
   ],
   [
    "C94",
    "x_vars[F7,C5]"
   ],
   [
    "C95",
    "x_vars[F7,C6]"
   ],
   [
    "C96",
    "x_vars[F7,C7]"
   ],
   [
    "C97",
    "x_vars[F7,C8]"
   ],
   [
    "C98",
    "x_vars[F7,C9]"
   ],
   [
    "C99",
    "x_vars[F7,C10]"
   ],
   [
    "C100",
    "x_vars[F7,C11]"
   ],
   [
    "C101",
    "x_vars[F7,C12]"
   ],
   [
    "C102",
    "x_vars[F7,C13]"
   ],
   [
    "C103",
    "x_vars[F7,C14]"
   ],
   [
    "C104",
    "x_vars[F7,C15]"
   ],
   [
    "C105",
    "x_vars[F8,C1]"
   ],
   [
    "C106",
    "x_vars[F8,C2]"
   ],
   [
    "C107",
    "x_vars[F8,C3]"
   ],
   [
    "C108",
    "x_vars[F8,C4]"
   ],
   [
    "C109",
    "x_vars[F8,C5]"
   ],
   [
    "C110",
    "x_vars[F8,C6]"
   ],
   [
    "C111",
    "x_vars[F8,C7]"
   ],
   [
    "C112",
    "x_vars[F8,C8]"
   ],
   [
    "C113",
    "x_vars[F8,C9]"
   ],
   [
    "C114",
    "x_vars[F8,C10]"
   ],
   [
    "C115",
    "x_vars[F8,C11]"
   ],
   [
    "C116",
    "x_vars[F8,C12]"
   ],
   [
    "C117",
    "x_vars[F8,C13]"
   ],
   [
    "C118",
    "x_vars[F8,C14]"
   ],
   [
    "C119",
    "x_vars[F8,C15]"
   ],
   [
    "C120",
    "x_vars[F9,C1]"
   ],
   [
    "C121",
    "x_vars[F9,C2]"
   ],
   [
    "C122",
    "x_vars[F9,C3]"
   ],
   [
    "C123",
    "x_vars[F9,C4]"
   ],
   [
    "C124",
    "x_vars[F9,C5]"
   ],
   [
    "C125",
    "x_vars[F9,C6]"
   ],
   [
    "C126",
    "x_vars[F9,C7]"
   ],
   [
    "C127",
    "x_vars[F9,C8]"
   ],
   [
    "C128",
    "x_vars[F9,C9]"
   ],
   [
    "C129",
    "x_vars[F9,C10]"
   ],
   [
    "C130",
    "x_vars[F9,C11]"
   ],
   [
    "C131",
    "x_vars[F9,C12]"
   ],
   [
    "C132",
    "x_vars[F9,C13]"
   ],
   [
    "C133",
    "x_vars[F9,C14]"
   ],
   [
    "C134",
    "x_vars[F9,C15]"
   ],
   [
    "C135",
    "x_vars[F10,C1]"
   ],
   [
    "C136",
    "x_vars[F10,C2]"
   ],
   [
    "C137",
    "x_vars[F10,C3]"
   ],
   [
    "C138",
    "x_vars[F10,C4]"
   ],
   [
    "C139",
    "x_vars[F10,C5]"
   ],
   [
    "C140",
    "x_vars[F10,C6]"
   ],
   [
    "C141",
    "x_vars[F10,C7]"
   ],
   [
    "C142",
    "x_vars[F10,C8]"
   ],
   [
    "C143",
    "x_vars[F10,C9]"
   ],
   [
    "C144",
    "x_vars[F10,C10]"
   ],
   [
    "C145",
    "x_vars[F10,C11]"
   ],
   [
    "C146",
    "x_vars[F10,C12]"
   ],
   [
    "C147",
    "x_vars[F10,C13]"
   ],
   [
    "C148",
    "x_vars[F10,C14]"
   ],
   [
    "C149",
    "x_vars[F10,C15]"
   ],
   [
    "C150",
    "x_vars[F11,C1]"
   ],
   [
    "C151",
    "x_vars[F11,C2]"
   ],
   [
    "C152",
    "x_vars[F11,C3]"
   ],
   [
    "C153",
    "x_vars[F11,C4]"
   ],
   [
    "C154",
    "x_vars[F11,C5]"
   ],
   [
    "C155",
    "x_vars[F11,C6]"
   ],
   [
    "C156",
    "x_vars[F11,C7]"
   ],
   [
    "C157",
    "x_vars[F11,C8]"
   ],
   [
    "C158",
    "x_vars[F11,C9]"
   ],
   [
    "C159",
    "x_vars[F11,C10]"
   ],
   [
    "C160",
    "x_vars[F11,C11]"
   ],
   [
    "C161",
    "x_vars[F11,C12]"
   ],
   [
    "C162",
    "x_vars[F11,C13]"
   ],
   [
    "C163",
    "x_vars[F11,C14]"
   ],
   [
    "C164",
    "x_vars[F11,C15]"
   ],
   [
    "C165",
    "x_vars[F12,C1]"
   ],
   [
    "C166",
    "x_vars[F12,C2]"
   ],
   [
    "C167",
    "x_vars[F12,C3]"
   ],
   [
    "C168",
    "x_vars[F12,C4]"
   ],
   [
    "C169",
    "x_vars[F12,C5]"
   ],
   [
    "C170",
    "x_vars[F12,C6]"
   ],
   [
    "C171",
    "x_vars[F12,C7]"
   ],
   [
    "C172",
    "x_vars[F12,C8]"
   ],
   [
    "C173",
    "x_vars[F12,C9]"
   ],
   [
    "C174",
    "x_vars[F12,C10]"
   ],
   [
    "C175",
    "x_vars[F12,C11]"
   ],
   [
    "C176",
    "x_vars[F12,C12]"
   ],
   [
    "C177",
    "x_vars[F12,C13]"
   ],
   [
    "C178",
    "x_vars[F12,C14]"
   ],
   [
    "C179",
    "x_vars[F12,C15]"
   ],
   [
    "C180",
    "x_vars[F13,C1]"
   ],
   [
    "C181",
    "x_vars[F13,C2]"
   ],
   [
    "C182",
    "x_vars[F13,C3]"
   ],
   [
    "C183",
    "x_vars[F13,C4]"
   ],
   [
    "C184",
    "x_vars[F13,C5]"
   ],
   [
    "C185",
    "x_vars[F13,C6]"
   ],
   [
    "C186",
    "x_vars[F13,C7]"
   ],
   [
    "C187",
    "x_vars[F13,C8]"
   ],
   [
    "C188",
    "x_vars[F13,C9]"
   ],
   [
    "C189",
    "x_vars[F13,C10]"
   ],
   [
    "C190",
    "x_vars[F13,C11]"
   ],
   [
    "C191",
    "x_vars[F13,C12]"
   ],
   [
    "C192",
    "x_vars[F13,C13]"
   ],
   [
    "C193",
    "x_vars[F13,C14]"
   ],
   [
    "C194",
    "x_vars[F13,C15]"
   ],
   [
    "C195",
    "x_vars[F14,C1]"
   ],
   [
    "C196",
    "x_vars[F14,C2]"
   ],
   [
    "C197",
    "x_vars[F14,C3]"
   ],
   [
    "C198",
    "x_vars[F14,C4]"
   ],
   [
    "C199",
    "x_vars[F14,C5]"
   ],
   [
    "C200",
    "x_vars[F14,C6]"
   ],
   [
    "C201",
    "x_vars[F14,C7]"
   ],
   [
    "C202",
    "x_vars[F14,C8]"
   ],
   [
    "C203",
    "x_vars[F14,C9]"
   ],
   [
    "C204",
    "x_vars[F14,C10]"
   ],
   [
    "C205",
    "x_vars[F14,C11]"
   ],
   [
    "C206",
    "x_vars[F14,C12]"
   ],
   [
    "C207",
    "x_vars[F14,C13]"
   ],
   [
    "C208",
    "x_vars[F14,C14]"
   ],
   [
    "C209",
    "x_vars[F14,C15]"
   ],
   [
    "C210",
    "x_vars[F15,C1]"
   ],
   [
    "C211",
    "x_vars[F15,C2]"
   ],
   [
    "C212",
    "x_vars[F15,C3]"
   ],
   [
    "C213",
    "x_vars[F15,C4]"
   ],
   [
    "C214",
    "x_vars[F15,C5]"
   ],
   [
    "C215",
    "x_vars[F15,C6]"
   ],
   [
    "C216",
    "x_vars[F15,C7]"
   ],
   [
    "C217",
    "x_vars[F15,C8]"
   ],
   [
    "C218",
    "x_vars[F15,C9]"
   ],
   [
    "C219",
    "x_vars[F15,C10]"
   ],
   [
    "C220",
    "x_vars[F15,C11]"
   ],
   [
    "C221",
    "x_vars[F15,C12]"
   ],
   [
    "C222",
    "x_vars[F15,C13]"
   ],
   [
    "C223",
    "x_vars[F15,C14]"
   ],
   [
    "C224",
    "x_vars[F15,C15]"
   ],
   [
    "C225",
    "y_vars[F1]"
   ],
   [
    "C226",
    "y_vars[F2]"
   ],
   [
    "C227",
    "y_vars[F3]"
   ],
   [
    "C228",
    "y_vars[F4]"
   ],
   [
    "C229",
    "y_vars[F5]"
   ],
   [
    "C230",
    "y_vars[F6]"
   ],
   [
    "C231",
    "y_vars[F7]"
   ],
   [
    "C232",
    "y_vars[F8]"
   ],
   [
    "C233",
    "y_vars[F9]"
   ],
   [
    "C234",
    "y_vars[F10]"
   ],
   [
    "C235",
    "y_vars[F11]"
   ],
   [
    "C236",
    "y_vars[F12]"
   ],
   [
    "C237",
    "y_vars[F13]"
   ],
   [
    "C238",
    "y_vars[F14]"
   ],
   [
    "C239",
    "y_vars[F15]"
   ]
  ],
  "primary_mapping": {
   "y_F1": "y_vars[F1]",
   "y_F2": "y_vars[F2]",
   "y_F3": "y_vars[F3]",
   "y_F4": "y_vars[F4]",
   "y_F5": "y_vars[F5]",
   "y_F6": "y_vars[F6]",
   "y_F7": "y_vars[F7]",
   "y_F8": "y_vars[F8]",
   "y_F9": "y_vars[F9]",
   "y_F10": "y_vars[F10]",
   "y_F11": "y_vars[F11]",
   "y_F12": "y_vars[F12]",
   "y_F13": "y_vars[F13]",
   "y_F14": "y_vars[F14]",
   "y_F15": "y_vars[F15]",
   "x_F1_C1": "x_vars[F1,C1]",
   "x_F2_C2": "x_vars[F2,C2]",
   "x_F3_C3": "x_vars[F3,C3]",
   "x_F4_C4": "x_vars[F4,C4]",
   "x_F5_C5": "x_vars[F5,C5]",
   "x_F6_C6": "x_vars[F6,C6]",
   "x_F7_C7": "x_vars[F7,C7]",
   "x_F8_C8": "x_vars[F8,C8]",
   "x_F9_C9": "x_vars[F9,C9]",
   "x_F10_C10": "x_vars[F10,C10]",
   "x_F11_C11": "x_vars[F11,C11]",
   "x_F12_C12": "x_vars[F12,C12]",
   "x_F13_C13": "x_vars[F13,C13]",
   "x_F14_C14": "x_vars[F14,C14]",
   "x_F15_C15": "x_vars[F15,C15]",
   "x_F1_C2": "x_vars[F1,C2]",
   "x_F1_C3": "x_vars[F1,C3]",
   "x_F1_C4": "x_vars[F1,C4]",
   "x_F1_C5": "x_vars[F1,C5]",
   "x_F1_C6": "x_vars[F1,C6]",
   "x_F1_C7": "x_vars[F1,C7]",
   "x_F1_C8": "x_vars[F1,C8]",
   "x_F1_C9": "x_vars[F1,C9]",
   "x_F1_C10": "x_vars[F1,C10]",
   "x_F1_C11": "x_vars[F1,C11]",
   "x_F1_C12": "x_vars[F1,C12]",
   "x_F1_C13": "x_vars[F1,C13]",
   "x_F1_C14": "x_vars[F1,C14]",
   "x_F1_C15": "x_vars[F1,C15]",
   "x_F2_C1": "x_vars[F2,C1]",
   "x_F2_C3": "x_vars[F2,C3]",
   "x_F2_C4": "x_vars[F2,C4]",
   "x_F2_C5": "x_vars[F2,C5]",
   "x_F2_C6": "x_vars[F2,C6]",
   "x_F2_C7": "x_vars[F2,C7]",
   "x_F2_C8": "x_vars[F2,C8]",
   "x_F2_C9": "x_vars[F2,C9]",
   "x_F2_C10": "x_vars[F2,C10]",
   "x_F2_C11": "x_vars[F2,C11]",
   "x_F2_C12": "x_vars[F2,C12]",
   "x_F2_C13": "x_vars[F2,C13]",
   "x_F2_C14": "x_vars[F2,C14]",
   "x_F2_C15": "x_vars[F2,C15]",
   "x_F3_C1": "x_vars[F3,C1]",
   "x_F3_C2": "x_vars[F3,C2]",
   "x_F3_C4": "x_vars[F3,C4]",
   "x_F3_C5": "x_vars[F3,C5]",
   "x_F3_C6": "x_vars[F3,C6]",
   "x_F3_C7": "x_vars[F3,C7]",
   "x_F3_C8": "x_vars[F3,C8]",
   "x_F3_C9": "x_vars[F3,C9]",
   "x_F3_C10": "x_vars[F3,C10]",
   "x_F3_C11": "x_vars[F3,C11]",
   "x_F3_C12": "x_vars[F3,C12]",
   "x_F3_C13": "x_vars[F3,C13]",
   "x_F3_C14": "x_vars[F3,C14]",
   "x_F3_C15": "x_vars[F3,C15]",
   "x_F4_C1": "x_vars[F4,C1]",
   "x_F4_C2": "x_vars[F4,C2]",
   "x_F4_C3": "x_vars[F4,C3]",
   "x_F4_C5": "x_vars[F4,C5]",
   "x_F4_C6": "x_vars[F4,C6]",
   "x_F4_C7": "x_vars[F4,C7]",
   "x_F4_C8": "x_vars[F4,C8]",
   "x_F4_C9": "x_vars[F4,C9]",
   "x_F4_C10": "x_vars[F4,C10]",
   "x_F4_C11": "x_vars[F4,C11]",
   "x_F4_C12": "x_vars[F4,C12]",
   "x_F4_C13": "x_vars[F4,C13]",
   "x_F4_C14": "x_vars[F4,C14]",
   "x_F4_C15": "x_vars[F4,C15]",
   "x_F5_C1": "x_vars[F5,C1]",
   "x_F5_C2": "x_vars[F5,C2]",
   "x_F5_C3": "x_vars[F5,C3]",
   "x_F5_C4": "x_vars[F5,C4]",
   "x_F5_C6": "x_vars[F5,C6]",
   "x_F5_C7": "x_vars[F5,C7]",
   "x_F5_C8": "x_vars[F5,C8]",
   "x_F5_C9": "x_vars[F5,C9]",
   "x_F5_C10": "x_vars[F5,C10]",
   "x_F5_C11": "x_vars[F5,C11]",
   "x_F5_C12": "x_vars[F5,C12]",
   "x_F5_C13": "x_vars[F5,C13]",
   "x_F5_C14": "x_vars[F5,C14]",
   "x_F5_C15": "x_vars[F5,C15]",
   "x_F6_C1": "x_vars[F6,C1]",
   "x_F6_C2": "x_vars[F6,C2]",
   "x_F6_C3": "x_vars[F6,C3]",
   "x_F6_C4": "x_vars[F6,C4]",
   "x_F6_C5": "x_vars[F6,C5]",
   "x_F6_C7": "x_vars[F6,C7]",
   "x_F6_C8": "x_vars[F6,C8]",
   "x_F6_C9": "x_vars[F6,C9]",
   "x_F6_C10": "x_vars[F6,C10]",
   "x_F6_C11": "x_vars[F6,C11]",
   "x_F6_C12": "x_vars[F6,C12]",
   "x_F6_C13": "x_vars[F6,C13]",
   "x_F6_C14": "x_vars[F6,C14]",
   "x_F6_C15": "x_vars[F6,C15]",
   "x_F7_C1": "x_vars[F7,C1]",
   "x_F7_C2": "x_vars[F7,C2]",
   "x_F7_C3": "x_vars[F7,C3]",
   "x_F7_C4": "x_vars[F7,C4]",
   "x_F7_C5": "x_vars[F7,C5]",
   "x_F7_C6": "x_vars[F7,C6]",
   "x_F7_C8": "x_vars[F7,C8]",
   "x_F7_C9": "x_vars[F7,C9]",
   "x_F7_C10": "x_vars[F7,C10]",
   "x_F7_C11": "x_vars[F7,C11]",
   "x_F7_C12": "x_vars[F7,C12]",
   "x_F7_C13": "x_vars[F7,C13]",
   "x_F7_C14": "x_vars[F7,C14]",
   "x_F7_C15": "x_vars[F7,C15]",
   "x_F8_C1": "x_vars[F8,C1]",
   "x_F8_C2": "x_vars[F8,C2]",
   "x_F8_C3": "x_vars[F8,C3]",
   "x_F8_C4": "x_vars[F8,C4]",
   "x_F8_C5": "x_vars[F8,C5]",
   "x_F8_C6": "x_vars[F8,C6]",
   "x_F8_C7": "x_vars[F8,C7]",
   "x_F8_C9": "x_vars[F8,C9]",
   "x_F8_C10": "x_vars[F8,C10]",
   "x_F8_C11": "x_vars[F8,C11]",
   "x_F8_C12": "x_vars[F8,C12]",
   "x_F8_C13": "x_vars[F8,C13]",
   "x_F8_C14": "x_vars[F8,C14]",
   "x_F8_C15": "x_vars[F8,C15]",
   "x_F9_C1": "x_vars[F9,C1]",
   "x_F9_C2": "x_vars[F9,C2]",
   "x_F9_C3": "x_vars[F9,C3]",
   "x_F9_C4": "x_vars[F9,C4]",
   "x_F9_C5": "x_vars[F9,C5]",
   "x_F9_C6": "x_vars[F9,C6]",
   "x_F9_C7": "x_vars[F9,C7]",
   "x_F9_C8": "x_vars[F9,C8]",
   "x_F9_C10": "x_vars[F9,C10]",
   "x_F9_C11": "x_vars[F9,C11]",
   "x_F9_C12": "x_vars[F9,C12]",
   "x_F9_C13": "x_vars[F9,C13]",
   "x_F9_C14": "x_vars[F9,C14]",
   "x_F9_C15": "x_vars[F9,C15]",
   "x_F10_C1": "x_vars[F10,C1]",
   "x_F10_C2": "x_vars[F10,C2]",
   "x_F10_C3": "x_vars[F10,C3]",
   "x_F10_C4": "x_vars[F10,C4]",
   "x_F10_C5": "x_vars[F10,C5]",
   "x_F10_C6": "x_vars[F10,C6]",
   "x_F10_C7": "x_vars[F10,C7]",
   "x_F10_C8": "x_vars[F10,C8]",
   "x_F10_C9": "x_vars[F10,C9]",
   "x_F10_C11": "x_vars[F10,C11]",
   "x_F10_C12": "x_vars[F10,C12]",
   "x_F10_C13": "x_vars[F10,C13]",
   "x_F10_C14": "x_vars[F10,C14]",
   "x_F10_C15": "x_vars[F10,C15]",
   "x_F11_C1": "x_vars[F11,C1]",
   "x_F11_C2": "x_vars[F11,C2]",
   "x_F11_C3": "x_vars[F11,C3]",
   "x_F11_C4": "x_vars[F11,C4]",
   "x_F11_C5": "x_vars[F11,C5]",
   "x_F11_C6": "x_vars[F11,C6]",
   "x_F11_C7": "x_vars[F11,C7]",
   "x_F11_C8": "x_vars[F11,C8]",
   "x_F11_C9": "x_vars[F11,C9]",
   "x_F11_C10": "x_vars[F11,C10]",
   "x_F11_C12": "x_vars[F11,C12]",
   "x_F11_C13": "x_vars[F11,C13]",
   "x_F11_C14": "x_vars[F11,C14]",
   "x_F11_C15": "x_vars[F11,C15]",
   "x_F12_C1": "x_vars[F12,C1]",
   "x_F12_C2": "x_vars[F12,C2]",
   "x_F12_C3": "x_vars[F12,C3]",
   "x_F12_C4": "x_vars[F12,C4]",
   "x_F12_C5": "x_vars[F12,C5]",
   "x_F12_C6": "x_vars[F12,C6]",
   "x_F12_C7": "x_vars[F12,C7]",
   "x_F12_C8": "x_vars[F12,C8]",
   "x_F12_C9": "x_vars[F12,C9]",
   "x_F12_C10": "x_vars[F12,C10]",
   "x_F12_C11": "x_vars[F12,C11]",
   "x_F12_C13": "x_vars[F12,C13]",
   "x_F12_C14": "x_vars[F12,C14]",
   "x_F12_C15": "x_vars[F12,C15]",
   "x_F13_C1": "x_vars[F13,C1]",
   "x_F13_C2": "x_vars[F13,C2]",
   "x_F13_C3": "x_vars[F13,C3]",
   "x_F13_C4": "x_vars[F13,C4]",
   "x_F13_C5": "x_vars[F13,C5]",
   "x_F13_C6": "x_vars[F13,C6]",
   "x_F13_C7": "x_vars[F13,C7]",
   "x_F13_C8": "x_vars[F13,C8]",
   "x_F13_C9": "x_vars[F13,C9]",
   "x_F13_C10": "x_vars[F13,C10]",
   "x_F13_C11": "x_vars[F13,C11]",
   "x_F13_C12": "x_vars[F13,C12]",
   "x_F13_C14": "x_vars[F13,C14]",
   "x_F13_C15": "x_vars[F13,C15]",
   "x_F14_C1": "x_vars[F14,C1]",
   "x_F14_C2": "x_vars[F14,C2]",
   "x_F14_C3": "x_vars[F14,C3]",
   "x_F14_C4": "x_vars[F14,C4]",
   "x_F14_C5": "x_vars[F14,C5]",
   "x_F14_C6": "x_vars[F14,C6]",
   "x_F14_C7": "x_vars[F14,C7]",
   "x_F14_C8": "x_vars[F14,C8]",
   "x_F14_C9": "x_vars[F14,C9]",
   "x_F14_C10": "x_vars[F14,C10]",
   "x_F14_C11": "x_vars[F14,C11]",
   "x_F14_C12": "x_vars[F14,C12]",
   "x_F14_C13": "x_vars[F14,C13]",
   "x_F14_C15": "x_vars[F14,C15]",
   "x_F15_C1": "x_vars[F15,C1]",
   "x_F15_C2": "x_vars[F15,C2]",
   "x_F15_C3": "x_vars[F15,C3]",
   "x_F15_C4": "x_vars[F15,C4]",
   "x_F15_C5": "x_vars[F15,C5]",
   "x_F15_C6": "x_vars[F15,C6]",
   "x_F15_C7": "x_vars[F15,C7]",
   "x_F15_C8": "x_vars[F15,C8]",
   "x_F15_C9": "x_vars[F15,C9]",
   "x_F15_C10": "x_vars[F15,C10]",
   "x_F15_C11": "x_vars[F15,C11]",
   "x_F15_C12": "x_vars[F15,C12]",
   "x_F15_C13": "x_vars[F15,C13]",
   "x_F15_C14": "x_vars[F15,C14]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-089": {
  "source_code_sha256": "6f2feb030d55e551400398f075d8a10239b2f26905f5e058eb711eed5789d1e4",
  "original_solution_sha256": "6637ba4c2c3a1adc8e06eabb9de58b44e76971b645e1e606b6bfa6f3a7488717",
  "bindings": [
   [
    "C0",
    "y_vars[SC1]"
   ],
   [
    "C1",
    "y_vars[SC2]"
   ],
   [
    "C2",
    "y_vars[SC3]"
   ],
   [
    "C3",
    "y_vars[SC4]"
   ],
   [
    "C4",
    "y_vars[SC5]"
   ],
   [
    "C5",
    "y_vars[SC6]"
   ],
   [
    "C6",
    "y_vars[SC7]"
   ],
   [
    "C7",
    "y_vars[SC8]"
   ],
   [
    "C8",
    "y_vars[SC9]"
   ],
   [
    "C9",
    "y_vars[SC10]"
   ],
   [
    "C10",
    "x_vars[SC1,C1]"
   ],
   [
    "C11",
    "x_vars[SC1,C2]"
   ],
   [
    "C12",
    "x_vars[SC1,C3]"
   ],
   [
    "C13",
    "x_vars[SC1,C4]"
   ],
   [
    "C14",
    "x_vars[SC1,C5]"
   ],
   [
    "C15",
    "x_vars[SC1,C6]"
   ],
   [
    "C16",
    "x_vars[SC1,C7]"
   ],
   [
    "C17",
    "x_vars[SC1,C8]"
   ],
   [
    "C18",
    "x_vars[SC1,C9]"
   ],
   [
    "C19",
    "x_vars[SC1,C10]"
   ],
   [
    "C20",
    "x_vars[SC1,C11]"
   ],
   [
    "C21",
    "x_vars[SC1,C12]"
   ],
   [
    "C22",
    "x_vars[SC1,C13]"
   ],
   [
    "C23",
    "x_vars[SC1,C14]"
   ],
   [
    "C24",
    "x_vars[SC1,C15]"
   ],
   [
    "C25",
    "x_vars[SC2,C1]"
   ],
   [
    "C26",
    "x_vars[SC2,C2]"
   ],
   [
    "C27",
    "x_vars[SC2,C3]"
   ],
   [
    "C28",
    "x_vars[SC2,C4]"
   ],
   [
    "C29",
    "x_vars[SC2,C5]"
   ],
   [
    "C30",
    "x_vars[SC2,C6]"
   ],
   [
    "C31",
    "x_vars[SC2,C7]"
   ],
   [
    "C32",
    "x_vars[SC2,C8]"
   ],
   [
    "C33",
    "x_vars[SC2,C9]"
   ],
   [
    "C34",
    "x_vars[SC2,C10]"
   ],
   [
    "C35",
    "x_vars[SC2,C11]"
   ],
   [
    "C36",
    "x_vars[SC2,C12]"
   ],
   [
    "C37",
    "x_vars[SC2,C13]"
   ],
   [
    "C38",
    "x_vars[SC2,C14]"
   ],
   [
    "C39",
    "x_vars[SC2,C15]"
   ],
   [
    "C40",
    "x_vars[SC3,C1]"
   ],
   [
    "C41",
    "x_vars[SC3,C2]"
   ],
   [
    "C42",
    "x_vars[SC3,C3]"
   ],
   [
    "C43",
    "x_vars[SC3,C4]"
   ],
   [
    "C44",
    "x_vars[SC3,C5]"
   ],
   [
    "C45",
    "x_vars[SC3,C6]"
   ],
   [
    "C46",
    "x_vars[SC3,C7]"
   ],
   [
    "C47",
    "x_vars[SC3,C8]"
   ],
   [
    "C48",
    "x_vars[SC3,C9]"
   ],
   [
    "C49",
    "x_vars[SC3,C10]"
   ],
   [
    "C50",
    "x_vars[SC3,C11]"
   ],
   [
    "C51",
    "x_vars[SC3,C12]"
   ],
   [
    "C52",
    "x_vars[SC3,C13]"
   ],
   [
    "C53",
    "x_vars[SC3,C14]"
   ],
   [
    "C54",
    "x_vars[SC3,C15]"
   ],
   [
    "C55",
    "x_vars[SC4,C1]"
   ],
   [
    "C56",
    "x_vars[SC4,C2]"
   ],
   [
    "C57",
    "x_vars[SC4,C3]"
   ],
   [
    "C58",
    "x_vars[SC4,C4]"
   ],
   [
    "C59",
    "x_vars[SC4,C5]"
   ],
   [
    "C60",
    "x_vars[SC4,C6]"
   ],
   [
    "C61",
    "x_vars[SC4,C7]"
   ],
   [
    "C62",
    "x_vars[SC4,C8]"
   ],
   [
    "C63",
    "x_vars[SC4,C9]"
   ],
   [
    "C64",
    "x_vars[SC4,C10]"
   ],
   [
    "C65",
    "x_vars[SC4,C11]"
   ],
   [
    "C66",
    "x_vars[SC4,C12]"
   ],
   [
    "C67",
    "x_vars[SC4,C13]"
   ],
   [
    "C68",
    "x_vars[SC4,C14]"
   ],
   [
    "C69",
    "x_vars[SC4,C15]"
   ],
   [
    "C70",
    "x_vars[SC5,C1]"
   ],
   [
    "C71",
    "x_vars[SC5,C2]"
   ],
   [
    "C72",
    "x_vars[SC5,C3]"
   ],
   [
    "C73",
    "x_vars[SC5,C4]"
   ],
   [
    "C74",
    "x_vars[SC5,C5]"
   ],
   [
    "C75",
    "x_vars[SC5,C6]"
   ],
   [
    "C76",
    "x_vars[SC5,C7]"
   ],
   [
    "C77",
    "x_vars[SC5,C8]"
   ],
   [
    "C78",
    "x_vars[SC5,C9]"
   ],
   [
    "C79",
    "x_vars[SC5,C10]"
   ],
   [
    "C80",
    "x_vars[SC5,C11]"
   ],
   [
    "C81",
    "x_vars[SC5,C12]"
   ],
   [
    "C82",
    "x_vars[SC5,C13]"
   ],
   [
    "C83",
    "x_vars[SC5,C14]"
   ],
   [
    "C84",
    "x_vars[SC5,C15]"
   ],
   [
    "C85",
    "x_vars[SC6,C1]"
   ],
   [
    "C86",
    "x_vars[SC6,C2]"
   ],
   [
    "C87",
    "x_vars[SC6,C3]"
   ],
   [
    "C88",
    "x_vars[SC6,C4]"
   ],
   [
    "C89",
    "x_vars[SC6,C5]"
   ],
   [
    "C90",
    "x_vars[SC6,C6]"
   ],
   [
    "C91",
    "x_vars[SC6,C7]"
   ],
   [
    "C92",
    "x_vars[SC6,C8]"
   ],
   [
    "C93",
    "x_vars[SC6,C9]"
   ],
   [
    "C94",
    "x_vars[SC6,C10]"
   ],
   [
    "C95",
    "x_vars[SC6,C11]"
   ],
   [
    "C96",
    "x_vars[SC6,C12]"
   ],
   [
    "C97",
    "x_vars[SC6,C13]"
   ],
   [
    "C98",
    "x_vars[SC6,C14]"
   ],
   [
    "C99",
    "x_vars[SC6,C15]"
   ],
   [
    "C100",
    "x_vars[SC7,C1]"
   ],
   [
    "C101",
    "x_vars[SC7,C2]"
   ],
   [
    "C102",
    "x_vars[SC7,C3]"
   ],
   [
    "C103",
    "x_vars[SC7,C4]"
   ],
   [
    "C104",
    "x_vars[SC7,C5]"
   ],
   [
    "C105",
    "x_vars[SC7,C6]"
   ],
   [
    "C106",
    "x_vars[SC7,C7]"
   ],
   [
    "C107",
    "x_vars[SC7,C8]"
   ],
   [
    "C108",
    "x_vars[SC7,C9]"
   ],
   [
    "C109",
    "x_vars[SC7,C10]"
   ],
   [
    "C110",
    "x_vars[SC7,C11]"
   ],
   [
    "C111",
    "x_vars[SC7,C12]"
   ],
   [
    "C112",
    "x_vars[SC7,C13]"
   ],
   [
    "C113",
    "x_vars[SC7,C14]"
   ],
   [
    "C114",
    "x_vars[SC7,C15]"
   ],
   [
    "C115",
    "x_vars[SC8,C1]"
   ],
   [
    "C116",
    "x_vars[SC8,C2]"
   ],
   [
    "C117",
    "x_vars[SC8,C3]"
   ],
   [
    "C118",
    "x_vars[SC8,C4]"
   ],
   [
    "C119",
    "x_vars[SC8,C5]"
   ],
   [
    "C120",
    "x_vars[SC8,C6]"
   ],
   [
    "C121",
    "x_vars[SC8,C7]"
   ],
   [
    "C122",
    "x_vars[SC8,C8]"
   ],
   [
    "C123",
    "x_vars[SC8,C9]"
   ],
   [
    "C124",
    "x_vars[SC8,C10]"
   ],
   [
    "C125",
    "x_vars[SC8,C11]"
   ],
   [
    "C126",
    "x_vars[SC8,C12]"
   ],
   [
    "C127",
    "x_vars[SC8,C13]"
   ],
   [
    "C128",
    "x_vars[SC8,C14]"
   ],
   [
    "C129",
    "x_vars[SC8,C15]"
   ],
   [
    "C130",
    "x_vars[SC9,C1]"
   ],
   [
    "C131",
    "x_vars[SC9,C2]"
   ],
   [
    "C132",
    "x_vars[SC9,C3]"
   ],
   [
    "C133",
    "x_vars[SC9,C4]"
   ],
   [
    "C134",
    "x_vars[SC9,C5]"
   ],
   [
    "C135",
    "x_vars[SC9,C6]"
   ],
   [
    "C136",
    "x_vars[SC9,C7]"
   ],
   [
    "C137",
    "x_vars[SC9,C8]"
   ],
   [
    "C138",
    "x_vars[SC9,C9]"
   ],
   [
    "C139",
    "x_vars[SC9,C10]"
   ],
   [
    "C140",
    "x_vars[SC9,C11]"
   ],
   [
    "C141",
    "x_vars[SC9,C12]"
   ],
   [
    "C142",
    "x_vars[SC9,C13]"
   ],
   [
    "C143",
    "x_vars[SC9,C14]"
   ],
   [
    "C144",
    "x_vars[SC9,C15]"
   ],
   [
    "C145",
    "x_vars[SC10,C1]"
   ],
   [
    "C146",
    "x_vars[SC10,C2]"
   ],
   [
    "C147",
    "x_vars[SC10,C3]"
   ],
   [
    "C148",
    "x_vars[SC10,C4]"
   ],
   [
    "C149",
    "x_vars[SC10,C5]"
   ],
   [
    "C150",
    "x_vars[SC10,C6]"
   ],
   [
    "C151",
    "x_vars[SC10,C7]"
   ],
   [
    "C152",
    "x_vars[SC10,C8]"
   ],
   [
    "C153",
    "x_vars[SC10,C9]"
   ],
   [
    "C154",
    "x_vars[SC10,C10]"
   ],
   [
    "C155",
    "x_vars[SC10,C11]"
   ],
   [
    "C156",
    "x_vars[SC10,C12]"
   ],
   [
    "C157",
    "x_vars[SC10,C13]"
   ],
   [
    "C158",
    "x_vars[SC10,C14]"
   ],
   [
    "C159",
    "x_vars[SC10,C15]"
   ]
  ],
  "primary_mapping": {
   "y_SC1": "y_vars[SC1]",
   "y_SC2": "y_vars[SC2]",
   "y_SC3": "y_vars[SC3]",
   "y_SC4": "y_vars[SC4]",
   "y_SC5": "y_vars[SC5]",
   "y_SC6": "y_vars[SC6]",
   "y_SC7": "y_vars[SC7]",
   "y_SC8": "y_vars[SC8]",
   "y_SC9": "y_vars[SC9]",
   "y_SC10": "y_vars[SC10]",
   "x_SC1_C1": "x_vars[SC1,C1]",
   "x_SC1_C11": "x_vars[SC1,C11]",
   "x_SC1_C12": "x_vars[SC1,C12]",
   "x_SC1_C13": "x_vars[SC1,C13]",
   "x_SC1_C14": "x_vars[SC1,C14]",
   "x_SC1_C15": "x_vars[SC1,C15]",
   "x_SC2_C2": "x_vars[SC2,C2]",
   "x_SC2_C11": "x_vars[SC2,C11]",
   "x_SC2_C12": "x_vars[SC2,C12]",
   "x_SC2_C13": "x_vars[SC2,C13]",
   "x_SC2_C14": "x_vars[SC2,C14]",
   "x_SC2_C15": "x_vars[SC2,C15]",
   "x_SC3_C3": "x_vars[SC3,C3]",
   "x_SC3_C11": "x_vars[SC3,C11]",
   "x_SC3_C12": "x_vars[SC3,C12]",
   "x_SC3_C13": "x_vars[SC3,C13]",
   "x_SC3_C14": "x_vars[SC3,C14]",
   "x_SC3_C15": "x_vars[SC3,C15]",
   "x_SC4_C4": "x_vars[SC4,C4]",
   "x_SC4_C11": "x_vars[SC4,C11]",
   "x_SC4_C12": "x_vars[SC4,C12]",
   "x_SC4_C13": "x_vars[SC4,C13]",
   "x_SC4_C14": "x_vars[SC4,C14]",
   "x_SC4_C15": "x_vars[SC4,C15]",
   "x_SC5_C5": "x_vars[SC5,C5]",
   "x_SC5_C11": "x_vars[SC5,C11]",
   "x_SC5_C12": "x_vars[SC5,C12]",
   "x_SC5_C13": "x_vars[SC5,C13]",
   "x_SC5_C14": "x_vars[SC5,C14]",
   "x_SC5_C15": "x_vars[SC5,C15]",
   "x_SC6_C6": "x_vars[SC6,C6]",
   "x_SC6_C11": "x_vars[SC6,C11]",
   "x_SC6_C12": "x_vars[SC6,C12]",
   "x_SC6_C13": "x_vars[SC6,C13]",
   "x_SC6_C14": "x_vars[SC6,C14]",
   "x_SC6_C15": "x_vars[SC6,C15]",
   "x_SC7_C7": "x_vars[SC7,C7]",
   "x_SC7_C11": "x_vars[SC7,C11]",
   "x_SC7_C12": "x_vars[SC7,C12]",
   "x_SC7_C13": "x_vars[SC7,C13]",
   "x_SC7_C14": "x_vars[SC7,C14]",
   "x_SC7_C15": "x_vars[SC7,C15]",
   "x_SC8_C8": "x_vars[SC8,C8]",
   "x_SC8_C11": "x_vars[SC8,C11]",
   "x_SC8_C12": "x_vars[SC8,C12]",
   "x_SC8_C13": "x_vars[SC8,C13]",
   "x_SC8_C14": "x_vars[SC8,C14]",
   "x_SC8_C15": "x_vars[SC8,C15]",
   "x_SC9_C9": "x_vars[SC9,C9]",
   "x_SC9_C11": "x_vars[SC9,C11]",
   "x_SC9_C12": "x_vars[SC9,C12]",
   "x_SC9_C13": "x_vars[SC9,C13]",
   "x_SC9_C14": "x_vars[SC9,C14]",
   "x_SC9_C15": "x_vars[SC9,C15]",
   "x_SC10_C10": "x_vars[SC10,C10]",
   "x_SC10_C11": "x_vars[SC10,C11]",
   "x_SC10_C12": "x_vars[SC10,C12]",
   "x_SC10_C13": "x_vars[SC10,C13]",
   "x_SC10_C14": "x_vars[SC10,C14]",
   "x_SC10_C15": "x_vars[SC10,C15]",
   "x_SC1_C2": "x_vars[SC1,C2]",
   "x_SC1_C3": "x_vars[SC1,C3]",
   "x_SC1_C4": "x_vars[SC1,C4]",
   "x_SC1_C5": "x_vars[SC1,C5]",
   "x_SC1_C6": "x_vars[SC1,C6]",
   "x_SC1_C7": "x_vars[SC1,C7]",
   "x_SC1_C8": "x_vars[SC1,C8]",
   "x_SC1_C9": "x_vars[SC1,C9]",
   "x_SC1_C10": "x_vars[SC1,C10]",
   "x_SC2_C1": "x_vars[SC2,C1]",
   "x_SC2_C3": "x_vars[SC2,C3]",
   "x_SC2_C4": "x_vars[SC2,C4]",
   "x_SC2_C5": "x_vars[SC2,C5]",
   "x_SC2_C6": "x_vars[SC2,C6]",
   "x_SC2_C7": "x_vars[SC2,C7]",
   "x_SC2_C8": "x_vars[SC2,C8]",
   "x_SC2_C9": "x_vars[SC2,C9]",
   "x_SC2_C10": "x_vars[SC2,C10]",
   "x_SC3_C1": "x_vars[SC3,C1]",
   "x_SC3_C2": "x_vars[SC3,C2]",
   "x_SC3_C4": "x_vars[SC3,C4]",
   "x_SC3_C5": "x_vars[SC3,C5]",
   "x_SC3_C6": "x_vars[SC3,C6]",
   "x_SC3_C7": "x_vars[SC3,C7]",
   "x_SC3_C8": "x_vars[SC3,C8]",
   "x_SC3_C9": "x_vars[SC3,C9]",
   "x_SC3_C10": "x_vars[SC3,C10]",
   "x_SC4_C1": "x_vars[SC4,C1]",
   "x_SC4_C2": "x_vars[SC4,C2]",
   "x_SC4_C3": "x_vars[SC4,C3]",
   "x_SC4_C5": "x_vars[SC4,C5]",
   "x_SC4_C6": "x_vars[SC4,C6]",
   "x_SC4_C7": "x_vars[SC4,C7]",
   "x_SC4_C8": "x_vars[SC4,C8]",
   "x_SC4_C9": "x_vars[SC4,C9]",
   "x_SC4_C10": "x_vars[SC4,C10]",
   "x_SC5_C1": "x_vars[SC5,C1]",
   "x_SC5_C2": "x_vars[SC5,C2]",
   "x_SC5_C3": "x_vars[SC5,C3]",
   "x_SC5_C4": "x_vars[SC5,C4]",
   "x_SC5_C6": "x_vars[SC5,C6]",
   "x_SC5_C7": "x_vars[SC5,C7]",
   "x_SC5_C8": "x_vars[SC5,C8]",
   "x_SC5_C9": "x_vars[SC5,C9]",
   "x_SC5_C10": "x_vars[SC5,C10]",
   "x_SC6_C1": "x_vars[SC6,C1]",
   "x_SC6_C2": "x_vars[SC6,C2]",
   "x_SC6_C3": "x_vars[SC6,C3]",
   "x_SC6_C4": "x_vars[SC6,C4]",
   "x_SC6_C5": "x_vars[SC6,C5]",
   "x_SC6_C7": "x_vars[SC6,C7]",
   "x_SC6_C8": "x_vars[SC6,C8]",
   "x_SC6_C9": "x_vars[SC6,C9]",
   "x_SC6_C10": "x_vars[SC6,C10]",
   "x_SC7_C1": "x_vars[SC7,C1]",
   "x_SC7_C2": "x_vars[SC7,C2]",
   "x_SC7_C3": "x_vars[SC7,C3]",
   "x_SC7_C4": "x_vars[SC7,C4]",
   "x_SC7_C5": "x_vars[SC7,C5]",
   "x_SC7_C6": "x_vars[SC7,C6]",
   "x_SC7_C8": "x_vars[SC7,C8]",
   "x_SC7_C9": "x_vars[SC7,C9]",
   "x_SC7_C10": "x_vars[SC7,C10]",
   "x_SC8_C1": "x_vars[SC8,C1]",
   "x_SC8_C2": "x_vars[SC8,C2]",
   "x_SC8_C3": "x_vars[SC8,C3]",
   "x_SC8_C4": "x_vars[SC8,C4]",
   "x_SC8_C5": "x_vars[SC8,C5]",
   "x_SC8_C6": "x_vars[SC8,C6]",
   "x_SC8_C7": "x_vars[SC8,C7]",
   "x_SC8_C9": "x_vars[SC8,C9]",
   "x_SC8_C10": "x_vars[SC8,C10]",
   "x_SC9_C1": "x_vars[SC9,C1]",
   "x_SC9_C2": "x_vars[SC9,C2]",
   "x_SC9_C3": "x_vars[SC9,C3]",
   "x_SC9_C4": "x_vars[SC9,C4]",
   "x_SC9_C5": "x_vars[SC9,C5]",
   "x_SC9_C6": "x_vars[SC9,C6]",
   "x_SC9_C7": "x_vars[SC9,C7]",
   "x_SC9_C8": "x_vars[SC9,C8]",
   "x_SC9_C10": "x_vars[SC9,C10]",
   "x_SC10_C1": "x_vars[SC10,C1]",
   "x_SC10_C2": "x_vars[SC10,C2]",
   "x_SC10_C3": "x_vars[SC10,C3]",
   "x_SC10_C4": "x_vars[SC10,C4]",
   "x_SC10_C5": "x_vars[SC10,C5]",
   "x_SC10_C6": "x_vars[SC10,C6]",
   "x_SC10_C7": "x_vars[SC10,C7]",
   "x_SC10_C8": "x_vars[SC10,C8]",
   "x_SC10_C9": "x_vars[SC10,C9]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-090": {
  "source_code_sha256": "45f7f3f34feea0ad655f53462943df6608902bd7776734485879849a279dc87c",
  "original_solution_sha256": "6e0684ae6efc4314940b81919f714be15af9c55276287e7cc77d5d7a5f28bf85",
  "bindings": [
   [
    "C0",
    "x_vars[P1]"
   ],
   [
    "C1",
    "x_vars[P2]"
   ],
   [
    "C2",
    "x_vars[P3]"
   ],
   [
    "C3",
    "x_vars[P4]"
   ],
   [
    "C4",
    "x_vars[P5]"
   ],
   [
    "C5",
    "x_vars[P6]"
   ],
   [
    "C6",
    "x_vars[P7]"
   ],
   [
    "C7",
    "x_vars[P8]"
   ],
   [
    "C8",
    "x_vars[P9]"
   ],
   [
    "C9",
    "x_vars[P10]"
   ],
   [
    "C10",
    "x_vars[P11]"
   ],
   [
    "C11",
    "x_vars[P12]"
   ],
   [
    "C12",
    "x_vars[P13]"
   ],
   [
    "C13",
    "x_vars[P14]"
   ],
   [
    "C14",
    "x_vars[P15]"
   ],
   [
    "C15",
    "x_vars[P16]"
   ],
   [
    "C16",
    "x_vars[P17]"
   ],
   [
    "C17",
    "x_vars[P18]"
   ],
   [
    "C18",
    "x_vars[P19]"
   ],
   [
    "C19",
    "x_vars[P20]"
   ],
   [
    "C20",
    "x_vars[P21]"
   ],
   [
    "C21",
    "x_vars[P22]"
   ],
   [
    "C22",
    "x_vars[P23]"
   ],
   [
    "C23",
    "x_vars[P24]"
   ],
   [
    "C24",
    "x_vars[P25]"
   ],
   [
    "C25",
    "x_vars[P26]"
   ],
   [
    "C26",
    "x_vars[P27]"
   ],
   [
    "C27",
    "x_vars[P28]"
   ],
   [
    "C28",
    "x_vars[P29]"
   ],
   [
    "C29",
    "x_vars[P30]"
   ],
   [
    "C30",
    "x_vars[P31]"
   ],
   [
    "C31",
    "x_vars[P32]"
   ],
   [
    "C32",
    "x_vars[P33]"
   ],
   [
    "C33",
    "x_vars[P34]"
   ],
   [
    "C34",
    "x_vars[P35]"
   ],
   [
    "C35",
    "x_vars[P36]"
   ],
   [
    "C36",
    "x_vars[P37]"
   ],
   [
    "C37",
    "x_vars[P38]"
   ],
   [
    "C38",
    "x_vars[P39]"
   ],
   [
    "C39",
    "x_vars[P40]"
   ],
   [
    "C40",
    "x_vars[P41]"
   ],
   [
    "C41",
    "x_vars[P42]"
   ],
   [
    "C42",
    "x_vars[P43]"
   ],
   [
    "C43",
    "x_vars[P44]"
   ],
   [
    "C44",
    "x_vars[P45]"
   ],
   [
    "C45",
    "x_vars[P46]"
   ],
   [
    "C46",
    "x_vars[P47]"
   ],
   [
    "C47",
    "x_vars[P48]"
   ],
   [
    "C48",
    "x_vars[P49]"
   ],
   [
    "C49",
    "x_vars[P50]"
   ],
   [
    "C50",
    "x_vars[P51]"
   ],
   [
    "C51",
    "x_vars[P52]"
   ],
   [
    "C52",
    "x_vars[P53]"
   ],
   [
    "C53",
    "x_vars[P54]"
   ],
   [
    "C54",
    "x_vars[P55]"
   ],
   [
    "C55",
    "x_vars[P56]"
   ],
   [
    "C56",
    "x_vars[P57]"
   ],
   [
    "C57",
    "x_vars[P58]"
   ],
   [
    "C58",
    "x_vars[P59]"
   ],
   [
    "C59",
    "x_vars[P60]"
   ],
   [
    "C60",
    "x_vars[P61]"
   ],
   [
    "C61",
    "x_vars[P62]"
   ],
   [
    "C62",
    "x_vars[P63]"
   ],
   [
    "C63",
    "x_vars[P64]"
   ],
   [
    "C64",
    "x_vars[P65]"
   ],
   [
    "C65",
    "x_vars[P66]"
   ],
   [
    "C66",
    "x_vars[P67]"
   ],
   [
    "C67",
    "x_vars[P68]"
   ],
   [
    "C68",
    "x_vars[P69]"
   ],
   [
    "C69",
    "x_vars[P70]"
   ],
   [
    "C70",
    "x_vars[P71]"
   ],
   [
    "C71",
    "x_vars[P72]"
   ],
   [
    "C72",
    "x_vars[P73]"
   ],
   [
    "C73",
    "x_vars[P74]"
   ],
   [
    "C74",
    "x_vars[P75]"
   ],
   [
    "C75",
    "x_vars[P76]"
   ],
   [
    "C76",
    "x_vars[P77]"
   ],
   [
    "C77",
    "x_vars[P78]"
   ],
   [
    "C78",
    "x_vars[P79]"
   ],
   [
    "C79",
    "x_vars[P80]"
   ],
   [
    "C80",
    "x_vars[P81]"
   ],
   [
    "C81",
    "x_vars[P82]"
   ],
   [
    "C82",
    "x_vars[P83]"
   ],
   [
    "C83",
    "x_vars[P84]"
   ],
   [
    "C84",
    "x_vars[P85]"
   ],
   [
    "C85",
    "x_vars[P86]"
   ],
   [
    "C86",
    "x_vars[P87]"
   ],
   [
    "C87",
    "x_vars[P88]"
   ],
   [
    "C88",
    "x_vars[P89]"
   ],
   [
    "C89",
    "x_vars[P90]"
   ],
   [
    "C90",
    "x_vars[P91]"
   ],
   [
    "C91",
    "x_vars[P92]"
   ],
   [
    "C92",
    "x_vars[P93]"
   ],
   [
    "C93",
    "x_vars[P94]"
   ],
   [
    "C94",
    "x_vars[P95]"
   ],
   [
    "C95",
    "x_vars[P96]"
   ],
   [
    "C96",
    "x_vars[P97]"
   ],
   [
    "C97",
    "x_vars[P98]"
   ],
   [
    "C98",
    "x_vars[P99]"
   ],
   [
    "C99",
    "x_vars[P100]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[P1]",
   "x[2]": "x_vars[P2]",
   "x[3]": "x_vars[P3]",
   "x[4]": "x_vars[P4]",
   "x[5]": "x_vars[P5]",
   "x[6]": "x_vars[P6]",
   "x[7]": "x_vars[P7]",
   "x[8]": "x_vars[P8]",
   "x[9]": "x_vars[P9]",
   "x[10]": "x_vars[P10]",
   "x[11]": "x_vars[P11]",
   "x[12]": "x_vars[P12]",
   "x[13]": "x_vars[P13]",
   "x[14]": "x_vars[P14]",
   "x[15]": "x_vars[P15]",
   "x[16]": "x_vars[P16]",
   "x[17]": "x_vars[P17]",
   "x[18]": "x_vars[P18]",
   "x[19]": "x_vars[P19]",
   "x[20]": "x_vars[P20]",
   "x[21]": "x_vars[P21]",
   "x[22]": "x_vars[P22]",
   "x[23]": "x_vars[P23]",
   "x[24]": "x_vars[P24]",
   "x[25]": "x_vars[P25]",
   "x[26]": "x_vars[P26]",
   "x[27]": "x_vars[P27]",
   "x[28]": "x_vars[P28]",
   "x[29]": "x_vars[P29]",
   "x[30]": "x_vars[P30]",
   "x[31]": "x_vars[P31]",
   "x[32]": "x_vars[P32]",
   "x[33]": "x_vars[P33]",
   "x[34]": "x_vars[P34]",
   "x[35]": "x_vars[P35]",
   "x[36]": "x_vars[P36]",
   "x[37]": "x_vars[P37]",
   "x[38]": "x_vars[P38]",
   "x[39]": "x_vars[P39]",
   "x[40]": "x_vars[P40]",
   "x[41]": "x_vars[P41]",
   "x[42]": "x_vars[P42]",
   "x[43]": "x_vars[P43]",
   "x[44]": "x_vars[P44]",
   "x[45]": "x_vars[P45]",
   "x[46]": "x_vars[P46]",
   "x[47]": "x_vars[P47]",
   "x[48]": "x_vars[P48]",
   "x[49]": "x_vars[P49]",
   "x[50]": "x_vars[P50]",
   "x[51]": "x_vars[P51]",
   "x[52]": "x_vars[P52]",
   "x[53]": "x_vars[P53]",
   "x[54]": "x_vars[P54]",
   "x[55]": "x_vars[P55]",
   "x[56]": "x_vars[P56]",
   "x[57]": "x_vars[P57]",
   "x[58]": "x_vars[P58]",
   "x[59]": "x_vars[P59]",
   "x[60]": "x_vars[P60]",
   "x[61]": "x_vars[P61]",
   "x[62]": "x_vars[P62]",
   "x[63]": "x_vars[P63]",
   "x[64]": "x_vars[P64]",
   "x[65]": "x_vars[P65]",
   "x[66]": "x_vars[P66]",
   "x[67]": "x_vars[P67]",
   "x[68]": "x_vars[P68]",
   "x[69]": "x_vars[P69]",
   "x[70]": "x_vars[P70]",
   "x[71]": "x_vars[P71]",
   "x[72]": "x_vars[P72]",
   "x[73]": "x_vars[P73]",
   "x[74]": "x_vars[P74]",
   "x[75]": "x_vars[P75]",
   "x[76]": "x_vars[P76]",
   "x[77]": "x_vars[P77]",
   "x[78]": "x_vars[P78]",
   "x[79]": "x_vars[P79]",
   "x[80]": "x_vars[P80]",
   "x[81]": "x_vars[P81]",
   "x[82]": "x_vars[P82]",
   "x[83]": "x_vars[P83]",
   "x[84]": "x_vars[P84]",
   "x[85]": "x_vars[P85]",
   "x[86]": "x_vars[P86]",
   "x[87]": "x_vars[P87]",
   "x[88]": "x_vars[P88]",
   "x[89]": "x_vars[P89]",
   "x[90]": "x_vars[P90]",
   "x[91]": "x_vars[P91]",
   "x[92]": "x_vars[P92]",
   "x[93]": "x_vars[P93]",
   "x[94]": "x_vars[P94]",
   "x[95]": "x_vars[P95]",
   "x[96]": "x_vars[P96]",
   "x[97]": "x_vars[P97]",
   "x[98]": "x_vars[P98]",
   "x[99]": "x_vars[P99]",
   "x[100]": "x_vars[P100]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-091": {
  "source_code_sha256": "57755fc370def47d5f7a51e13cf9392ccfcb087a1ab0e3b4e210be3f4069b294",
  "original_solution_sha256": "847ffb2527b4faca7883c5666cd04a909f8149f4045b7f83785cc75113ee682c",
  "bindings": [
   [
    "C0",
    "x_vars[C22]"
   ],
   [
    "C1",
    "x_vars[C23]"
   ],
   [
    "C2",
    "x_vars[C24]"
   ],
   [
    "C3",
    "x_vars[C25]"
   ],
   [
    "C4",
    "x_vars[C26]"
   ],
   [
    "C5",
    "x_vars[C27]"
   ],
   [
    "C6",
    "x_vars[C28]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[C22]",
   "x[2]": "x_vars[C23]",
   "x[3]": "x_vars[C24]",
   "x[4]": "x_vars[C25]",
   "x[5]": "x_vars[C26]",
   "x[6]": "x_vars[C27]",
   "x[7]": "x_vars[C28]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-092": {
  "source_code_sha256": "a55a9c8ee0f64a9c58a8e7a1bd5d1758f02223595f14b94e07106d471f3a1d64",
  "original_solution_sha256": "1d91fe4a3eed6a60d8e40db31bec568e4838ce3a256ac7a9846c95f51c78a49e",
  "bindings": [
   [
    "C0",
    "t_vars[S1,D1]"
   ],
   [
    "C1",
    "t_vars[S1,D2]"
   ],
   [
    "C2",
    "t_vars[S1,D3]"
   ],
   [
    "C3",
    "t_vars[S1,D4]"
   ],
   [
    "C4",
    "t_vars[S1,D5]"
   ],
   [
    "C5",
    "t_vars[S1,D6]"
   ],
   [
    "C6",
    "t_vars[S1,D7]"
   ],
   [
    "C7",
    "t_vars[S1,D8]"
   ],
   [
    "C8",
    "t_vars[S1,D9]"
   ],
   [
    "C9",
    "t_vars[S1,D10]"
   ],
   [
    "C10",
    "t_vars[S1,D11]"
   ],
   [
    "C11",
    "t_vars[S1,D12]"
   ],
   [
    "C12",
    "t_vars[S1,D13]"
   ],
   [
    "C13",
    "t_vars[S1,D14]"
   ],
   [
    "C14",
    "t_vars[S1,D15]"
   ],
   [
    "C15",
    "t_vars[S1,D16]"
   ],
   [
    "C16",
    "t_vars[S1,D17]"
   ],
   [
    "C17",
    "t_vars[S1,D18]"
   ],
   [
    "C18",
    "t_vars[S1,D19]"
   ],
   [
    "C19",
    "t_vars[S1,D20]"
   ],
   [
    "C20",
    "t_vars[S2,D1]"
   ],
   [
    "C21",
    "t_vars[S2,D2]"
   ],
   [
    "C22",
    "t_vars[S2,D3]"
   ],
   [
    "C23",
    "t_vars[S2,D4]"
   ],
   [
    "C24",
    "t_vars[S2,D5]"
   ],
   [
    "C25",
    "t_vars[S2,D6]"
   ],
   [
    "C26",
    "t_vars[S2,D7]"
   ],
   [
    "C27",
    "t_vars[S2,D8]"
   ],
   [
    "C28",
    "t_vars[S2,D9]"
   ],
   [
    "C29",
    "t_vars[S2,D10]"
   ],
   [
    "C30",
    "t_vars[S2,D11]"
   ],
   [
    "C31",
    "t_vars[S2,D12]"
   ],
   [
    "C32",
    "t_vars[S2,D13]"
   ],
   [
    "C33",
    "t_vars[S2,D14]"
   ],
   [
    "C34",
    "t_vars[S2,D15]"
   ],
   [
    "C35",
    "t_vars[S2,D16]"
   ],
   [
    "C36",
    "t_vars[S2,D17]"
   ],
   [
    "C37",
    "t_vars[S2,D18]"
   ],
   [
    "C38",
    "t_vars[S2,D19]"
   ],
   [
    "C39",
    "t_vars[S2,D20]"
   ],
   [
    "C40",
    "t_vars[S3,D1]"
   ],
   [
    "C41",
    "t_vars[S3,D2]"
   ],
   [
    "C42",
    "t_vars[S3,D3]"
   ],
   [
    "C43",
    "t_vars[S3,D4]"
   ],
   [
    "C44",
    "t_vars[S3,D5]"
   ],
   [
    "C45",
    "t_vars[S3,D6]"
   ],
   [
    "C46",
    "t_vars[S3,D7]"
   ],
   [
    "C47",
    "t_vars[S3,D8]"
   ],
   [
    "C48",
    "t_vars[S3,D9]"
   ],
   [
    "C49",
    "t_vars[S3,D10]"
   ],
   [
    "C50",
    "t_vars[S3,D11]"
   ],
   [
    "C51",
    "t_vars[S3,D12]"
   ],
   [
    "C52",
    "t_vars[S3,D13]"
   ],
   [
    "C53",
    "t_vars[S3,D14]"
   ],
   [
    "C54",
    "t_vars[S3,D15]"
   ],
   [
    "C55",
    "t_vars[S3,D16]"
   ],
   [
    "C56",
    "t_vars[S3,D17]"
   ],
   [
    "C57",
    "t_vars[S3,D18]"
   ],
   [
    "C58",
    "t_vars[S3,D19]"
   ],
   [
    "C59",
    "t_vars[S3,D20]"
   ],
   [
    "C60",
    "t_vars[S4,D1]"
   ],
   [
    "C61",
    "t_vars[S4,D2]"
   ],
   [
    "C62",
    "t_vars[S4,D3]"
   ],
   [
    "C63",
    "t_vars[S4,D4]"
   ],
   [
    "C64",
    "t_vars[S4,D5]"
   ],
   [
    "C65",
    "t_vars[S4,D6]"
   ],
   [
    "C66",
    "t_vars[S4,D7]"
   ],
   [
    "C67",
    "t_vars[S4,D8]"
   ],
   [
    "C68",
    "t_vars[S4,D9]"
   ],
   [
    "C69",
    "t_vars[S4,D10]"
   ],
   [
    "C70",
    "t_vars[S4,D11]"
   ],
   [
    "C71",
    "t_vars[S4,D12]"
   ],
   [
    "C72",
    "t_vars[S4,D13]"
   ],
   [
    "C73",
    "t_vars[S4,D14]"
   ],
   [
    "C74",
    "t_vars[S4,D15]"
   ],
   [
    "C75",
    "t_vars[S4,D16]"
   ],
   [
    "C76",
    "t_vars[S4,D17]"
   ],
   [
    "C77",
    "t_vars[S4,D18]"
   ],
   [
    "C78",
    "t_vars[S4,D19]"
   ],
   [
    "C79",
    "t_vars[S4,D20]"
   ],
   [
    "C80",
    "t_vars[S5,D1]"
   ],
   [
    "C81",
    "t_vars[S5,D2]"
   ],
   [
    "C82",
    "t_vars[S5,D3]"
   ],
   [
    "C83",
    "t_vars[S5,D4]"
   ],
   [
    "C84",
    "t_vars[S5,D5]"
   ],
   [
    "C85",
    "t_vars[S5,D6]"
   ],
   [
    "C86",
    "t_vars[S5,D7]"
   ],
   [
    "C87",
    "t_vars[S5,D8]"
   ],
   [
    "C88",
    "t_vars[S5,D9]"
   ],
   [
    "C89",
    "t_vars[S5,D10]"
   ],
   [
    "C90",
    "t_vars[S5,D11]"
   ],
   [
    "C91",
    "t_vars[S5,D12]"
   ],
   [
    "C92",
    "t_vars[S5,D13]"
   ],
   [
    "C93",
    "t_vars[S5,D14]"
   ],
   [
    "C94",
    "t_vars[S5,D15]"
   ],
   [
    "C95",
    "t_vars[S5,D16]"
   ],
   [
    "C96",
    "t_vars[S5,D17]"
   ],
   [
    "C97",
    "t_vars[S5,D18]"
   ],
   [
    "C98",
    "t_vars[S5,D19]"
   ],
   [
    "C99",
    "t_vars[S5,D20]"
   ],
   [
    "C100",
    "t_vars[S6,D1]"
   ],
   [
    "C101",
    "t_vars[S6,D2]"
   ],
   [
    "C102",
    "t_vars[S6,D3]"
   ],
   [
    "C103",
    "t_vars[S6,D4]"
   ],
   [
    "C104",
    "t_vars[S6,D5]"
   ],
   [
    "C105",
    "t_vars[S6,D6]"
   ],
   [
    "C106",
    "t_vars[S6,D7]"
   ],
   [
    "C107",
    "t_vars[S6,D8]"
   ],
   [
    "C108",
    "t_vars[S6,D9]"
   ],
   [
    "C109",
    "t_vars[S6,D10]"
   ],
   [
    "C110",
    "t_vars[S6,D11]"
   ],
   [
    "C111",
    "t_vars[S6,D12]"
   ],
   [
    "C112",
    "t_vars[S6,D13]"
   ],
   [
    "C113",
    "t_vars[S6,D14]"
   ],
   [
    "C114",
    "t_vars[S6,D15]"
   ],
   [
    "C115",
    "t_vars[S6,D16]"
   ],
   [
    "C116",
    "t_vars[S6,D17]"
   ],
   [
    "C117",
    "t_vars[S6,D18]"
   ],
   [
    "C118",
    "t_vars[S6,D19]"
   ],
   [
    "C119",
    "t_vars[S6,D20]"
   ],
   [
    "C120",
    "t_vars[S7,D1]"
   ],
   [
    "C121",
    "t_vars[S7,D2]"
   ],
   [
    "C122",
    "t_vars[S7,D3]"
   ],
   [
    "C123",
    "t_vars[S7,D4]"
   ],
   [
    "C124",
    "t_vars[S7,D5]"
   ],
   [
    "C125",
    "t_vars[S7,D6]"
   ],
   [
    "C126",
    "t_vars[S7,D7]"
   ],
   [
    "C127",
    "t_vars[S7,D8]"
   ],
   [
    "C128",
    "t_vars[S7,D9]"
   ],
   [
    "C129",
    "t_vars[S7,D10]"
   ],
   [
    "C130",
    "t_vars[S7,D11]"
   ],
   [
    "C131",
    "t_vars[S7,D12]"
   ],
   [
    "C132",
    "t_vars[S7,D13]"
   ],
   [
    "C133",
    "t_vars[S7,D14]"
   ],
   [
    "C134",
    "t_vars[S7,D15]"
   ],
   [
    "C135",
    "t_vars[S7,D16]"
   ],
   [
    "C136",
    "t_vars[S7,D17]"
   ],
   [
    "C137",
    "t_vars[S7,D18]"
   ],
   [
    "C138",
    "t_vars[S7,D19]"
   ],
   [
    "C139",
    "t_vars[S7,D20]"
   ],
   [
    "C140",
    "t_vars[S8,D1]"
   ],
   [
    "C141",
    "t_vars[S8,D2]"
   ],
   [
    "C142",
    "t_vars[S8,D3]"
   ],
   [
    "C143",
    "t_vars[S8,D4]"
   ],
   [
    "C144",
    "t_vars[S8,D5]"
   ],
   [
    "C145",
    "t_vars[S8,D6]"
   ],
   [
    "C146",
    "t_vars[S8,D7]"
   ],
   [
    "C147",
    "t_vars[S8,D8]"
   ],
   [
    "C148",
    "t_vars[S8,D9]"
   ],
   [
    "C149",
    "t_vars[S8,D10]"
   ],
   [
    "C150",
    "t_vars[S8,D11]"
   ],
   [
    "C151",
    "t_vars[S8,D12]"
   ],
   [
    "C152",
    "t_vars[S8,D13]"
   ],
   [
    "C153",
    "t_vars[S8,D14]"
   ],
   [
    "C154",
    "t_vars[S8,D15]"
   ],
   [
    "C155",
    "t_vars[S8,D16]"
   ],
   [
    "C156",
    "t_vars[S8,D17]"
   ],
   [
    "C157",
    "t_vars[S8,D18]"
   ],
   [
    "C158",
    "t_vars[S8,D19]"
   ],
   [
    "C159",
    "t_vars[S8,D20]"
   ],
   [
    "C160",
    "t_vars[S9,D1]"
   ],
   [
    "C161",
    "t_vars[S9,D2]"
   ],
   [
    "C162",
    "t_vars[S9,D3]"
   ],
   [
    "C163",
    "t_vars[S9,D4]"
   ],
   [
    "C164",
    "t_vars[S9,D5]"
   ],
   [
    "C165",
    "t_vars[S9,D6]"
   ],
   [
    "C166",
    "t_vars[S9,D7]"
   ],
   [
    "C167",
    "t_vars[S9,D8]"
   ],
   [
    "C168",
    "t_vars[S9,D9]"
   ],
   [
    "C169",
    "t_vars[S9,D10]"
   ],
   [
    "C170",
    "t_vars[S9,D11]"
   ],
   [
    "C171",
    "t_vars[S9,D12]"
   ],
   [
    "C172",
    "t_vars[S9,D13]"
   ],
   [
    "C173",
    "t_vars[S9,D14]"
   ],
   [
    "C174",
    "t_vars[S9,D15]"
   ],
   [
    "C175",
    "t_vars[S9,D16]"
   ],
   [
    "C176",
    "t_vars[S9,D17]"
   ],
   [
    "C177",
    "t_vars[S9,D18]"
   ],
   [
    "C178",
    "t_vars[S9,D19]"
   ],
   [
    "C179",
    "t_vars[S9,D20]"
   ],
   [
    "C180",
    "t_vars[S10,D1]"
   ],
   [
    "C181",
    "t_vars[S10,D2]"
   ],
   [
    "C182",
    "t_vars[S10,D3]"
   ],
   [
    "C183",
    "t_vars[S10,D4]"
   ],
   [
    "C184",
    "t_vars[S10,D5]"
   ],
   [
    "C185",
    "t_vars[S10,D6]"
   ],
   [
    "C186",
    "t_vars[S10,D7]"
   ],
   [
    "C187",
    "t_vars[S10,D8]"
   ],
   [
    "C188",
    "t_vars[S10,D9]"
   ],
   [
    "C189",
    "t_vars[S10,D10]"
   ],
   [
    "C190",
    "t_vars[S10,D11]"
   ],
   [
    "C191",
    "t_vars[S10,D12]"
   ],
   [
    "C192",
    "t_vars[S10,D13]"
   ],
   [
    "C193",
    "t_vars[S10,D14]"
   ],
   [
    "C194",
    "t_vars[S10,D15]"
   ],
   [
    "C195",
    "t_vars[S10,D16]"
   ],
   [
    "C196",
    "t_vars[S10,D17]"
   ],
   [
    "C197",
    "t_vars[S10,D18]"
   ],
   [
    "C198",
    "t_vars[S10,D19]"
   ],
   [
    "C199",
    "t_vars[S10,D20]"
   ],
   [
    "C200",
    "x_vars[S1,D1]"
   ],
   [
    "C201",
    "x_vars[S1,D2]"
   ],
   [
    "C202",
    "x_vars[S1,D3]"
   ],
   [
    "C203",
    "x_vars[S1,D4]"
   ],
   [
    "C204",
    "x_vars[S1,D5]"
   ],
   [
    "C205",
    "x_vars[S1,D6]"
   ],
   [
    "C206",
    "x_vars[S1,D7]"
   ],
   [
    "C207",
    "x_vars[S1,D8]"
   ],
   [
    "C208",
    "x_vars[S1,D9]"
   ],
   [
    "C209",
    "x_vars[S1,D10]"
   ],
   [
    "C210",
    "x_vars[S1,D11]"
   ],
   [
    "C211",
    "x_vars[S1,D12]"
   ],
   [
    "C212",
    "x_vars[S1,D13]"
   ],
   [
    "C213",
    "x_vars[S1,D14]"
   ],
   [
    "C214",
    "x_vars[S1,D15]"
   ],
   [
    "C215",
    "x_vars[S1,D16]"
   ],
   [
    "C216",
    "x_vars[S1,D17]"
   ],
   [
    "C217",
    "x_vars[S1,D18]"
   ],
   [
    "C218",
    "x_vars[S1,D19]"
   ],
   [
    "C219",
    "x_vars[S1,D20]"
   ],
   [
    "C220",
    "x_vars[S2,D1]"
   ],
   [
    "C221",
    "x_vars[S2,D2]"
   ],
   [
    "C222",
    "x_vars[S2,D3]"
   ],
   [
    "C223",
    "x_vars[S2,D4]"
   ],
   [
    "C224",
    "x_vars[S2,D5]"
   ],
   [
    "C225",
    "x_vars[S2,D6]"
   ],
   [
    "C226",
    "x_vars[S2,D7]"
   ],
   [
    "C227",
    "x_vars[S2,D8]"
   ],
   [
    "C228",
    "x_vars[S2,D9]"
   ],
   [
    "C229",
    "x_vars[S2,D10]"
   ],
   [
    "C230",
    "x_vars[S2,D11]"
   ],
   [
    "C231",
    "x_vars[S2,D12]"
   ],
   [
    "C232",
    "x_vars[S2,D13]"
   ],
   [
    "C233",
    "x_vars[S2,D14]"
   ],
   [
    "C234",
    "x_vars[S2,D15]"
   ],
   [
    "C235",
    "x_vars[S2,D16]"
   ],
   [
    "C236",
    "x_vars[S2,D17]"
   ],
   [
    "C237",
    "x_vars[S2,D18]"
   ],
   [
    "C238",
    "x_vars[S2,D19]"
   ],
   [
    "C239",
    "x_vars[S2,D20]"
   ],
   [
    "C240",
    "x_vars[S3,D1]"
   ],
   [
    "C241",
    "x_vars[S3,D2]"
   ],
   [
    "C242",
    "x_vars[S3,D3]"
   ],
   [
    "C243",
    "x_vars[S3,D4]"
   ],
   [
    "C244",
    "x_vars[S3,D5]"
   ],
   [
    "C245",
    "x_vars[S3,D6]"
   ],
   [
    "C246",
    "x_vars[S3,D7]"
   ],
   [
    "C247",
    "x_vars[S3,D8]"
   ],
   [
    "C248",
    "x_vars[S3,D9]"
   ],
   [
    "C249",
    "x_vars[S3,D10]"
   ],
   [
    "C250",
    "x_vars[S3,D11]"
   ],
   [
    "C251",
    "x_vars[S3,D12]"
   ],
   [
    "C252",
    "x_vars[S3,D13]"
   ],
   [
    "C253",
    "x_vars[S3,D14]"
   ],
   [
    "C254",
    "x_vars[S3,D15]"
   ],
   [
    "C255",
    "x_vars[S3,D16]"
   ],
   [
    "C256",
    "x_vars[S3,D17]"
   ],
   [
    "C257",
    "x_vars[S3,D18]"
   ],
   [
    "C258",
    "x_vars[S3,D19]"
   ],
   [
    "C259",
    "x_vars[S3,D20]"
   ],
   [
    "C260",
    "x_vars[S4,D1]"
   ],
   [
    "C261",
    "x_vars[S4,D2]"
   ],
   [
    "C262",
    "x_vars[S4,D3]"
   ],
   [
    "C263",
    "x_vars[S4,D4]"
   ],
   [
    "C264",
    "x_vars[S4,D5]"
   ],
   [
    "C265",
    "x_vars[S4,D6]"
   ],
   [
    "C266",
    "x_vars[S4,D7]"
   ],
   [
    "C267",
    "x_vars[S4,D8]"
   ],
   [
    "C268",
    "x_vars[S4,D9]"
   ],
   [
    "C269",
    "x_vars[S4,D10]"
   ],
   [
    "C270",
    "x_vars[S4,D11]"
   ],
   [
    "C271",
    "x_vars[S4,D12]"
   ],
   [
    "C272",
    "x_vars[S4,D13]"
   ],
   [
    "C273",
    "x_vars[S4,D14]"
   ],
   [
    "C274",
    "x_vars[S4,D15]"
   ],
   [
    "C275",
    "x_vars[S4,D16]"
   ],
   [
    "C276",
    "x_vars[S4,D17]"
   ],
   [
    "C277",
    "x_vars[S4,D18]"
   ],
   [
    "C278",
    "x_vars[S4,D19]"
   ],
   [
    "C279",
    "x_vars[S4,D20]"
   ],
   [
    "C280",
    "x_vars[S5,D1]"
   ],
   [
    "C281",
    "x_vars[S5,D2]"
   ],
   [
    "C282",
    "x_vars[S5,D3]"
   ],
   [
    "C283",
    "x_vars[S5,D4]"
   ],
   [
    "C284",
    "x_vars[S5,D5]"
   ],
   [
    "C285",
    "x_vars[S5,D6]"
   ],
   [
    "C286",
    "x_vars[S5,D7]"
   ],
   [
    "C287",
    "x_vars[S5,D8]"
   ],
   [
    "C288",
    "x_vars[S5,D9]"
   ],
   [
    "C289",
    "x_vars[S5,D10]"
   ],
   [
    "C290",
    "x_vars[S5,D11]"
   ],
   [
    "C291",
    "x_vars[S5,D12]"
   ],
   [
    "C292",
    "x_vars[S5,D13]"
   ],
   [
    "C293",
    "x_vars[S5,D14]"
   ],
   [
    "C294",
    "x_vars[S5,D15]"
   ],
   [
    "C295",
    "x_vars[S5,D16]"
   ],
   [
    "C296",
    "x_vars[S5,D17]"
   ],
   [
    "C297",
    "x_vars[S5,D18]"
   ],
   [
    "C298",
    "x_vars[S5,D19]"
   ],
   [
    "C299",
    "x_vars[S5,D20]"
   ],
   [
    "C300",
    "x_vars[S6,D1]"
   ],
   [
    "C301",
    "x_vars[S6,D2]"
   ],
   [
    "C302",
    "x_vars[S6,D3]"
   ],
   [
    "C303",
    "x_vars[S6,D4]"
   ],
   [
    "C304",
    "x_vars[S6,D5]"
   ],
   [
    "C305",
    "x_vars[S6,D6]"
   ],
   [
    "C306",
    "x_vars[S6,D7]"
   ],
   [
    "C307",
    "x_vars[S6,D8]"
   ],
   [
    "C308",
    "x_vars[S6,D9]"
   ],
   [
    "C309",
    "x_vars[S6,D10]"
   ],
   [
    "C310",
    "x_vars[S6,D11]"
   ],
   [
    "C311",
    "x_vars[S6,D12]"
   ],
   [
    "C312",
    "x_vars[S6,D13]"
   ],
   [
    "C313",
    "x_vars[S6,D14]"
   ],
   [
    "C314",
    "x_vars[S6,D15]"
   ],
   [
    "C315",
    "x_vars[S6,D16]"
   ],
   [
    "C316",
    "x_vars[S6,D17]"
   ],
   [
    "C317",
    "x_vars[S6,D18]"
   ],
   [
    "C318",
    "x_vars[S6,D19]"
   ],
   [
    "C319",
    "x_vars[S6,D20]"
   ],
   [
    "C320",
    "x_vars[S7,D1]"
   ],
   [
    "C321",
    "x_vars[S7,D2]"
   ],
   [
    "C322",
    "x_vars[S7,D3]"
   ],
   [
    "C323",
    "x_vars[S7,D4]"
   ],
   [
    "C324",
    "x_vars[S7,D5]"
   ],
   [
    "C325",
    "x_vars[S7,D6]"
   ],
   [
    "C326",
    "x_vars[S7,D7]"
   ],
   [
    "C327",
    "x_vars[S7,D8]"
   ],
   [
    "C328",
    "x_vars[S7,D9]"
   ],
   [
    "C329",
    "x_vars[S7,D10]"
   ],
   [
    "C330",
    "x_vars[S7,D11]"
   ],
   [
    "C331",
    "x_vars[S7,D12]"
   ],
   [
    "C332",
    "x_vars[S7,D13]"
   ],
   [
    "C333",
    "x_vars[S7,D14]"
   ],
   [
    "C334",
    "x_vars[S7,D15]"
   ],
   [
    "C335",
    "x_vars[S7,D16]"
   ],
   [
    "C336",
    "x_vars[S7,D17]"
   ],
   [
    "C337",
    "x_vars[S7,D18]"
   ],
   [
    "C338",
    "x_vars[S7,D19]"
   ],
   [
    "C339",
    "x_vars[S7,D20]"
   ],
   [
    "C340",
    "x_vars[S8,D1]"
   ],
   [
    "C341",
    "x_vars[S8,D2]"
   ],
   [
    "C342",
    "x_vars[S8,D3]"
   ],
   [
    "C343",
    "x_vars[S8,D4]"
   ],
   [
    "C344",
    "x_vars[S8,D5]"
   ],
   [
    "C345",
    "x_vars[S8,D6]"
   ],
   [
    "C346",
    "x_vars[S8,D7]"
   ],
   [
    "C347",
    "x_vars[S8,D8]"
   ],
   [
    "C348",
    "x_vars[S8,D9]"
   ],
   [
    "C349",
    "x_vars[S8,D10]"
   ],
   [
    "C350",
    "x_vars[S8,D11]"
   ],
   [
    "C351",
    "x_vars[S8,D12]"
   ],
   [
    "C352",
    "x_vars[S8,D13]"
   ],
   [
    "C353",
    "x_vars[S8,D14]"
   ],
   [
    "C354",
    "x_vars[S8,D15]"
   ],
   [
    "C355",
    "x_vars[S8,D16]"
   ],
   [
    "C356",
    "x_vars[S8,D17]"
   ],
   [
    "C357",
    "x_vars[S8,D18]"
   ],
   [
    "C358",
    "x_vars[S8,D19]"
   ],
   [
    "C359",
    "x_vars[S8,D20]"
   ],
   [
    "C360",
    "x_vars[S9,D1]"
   ],
   [
    "C361",
    "x_vars[S9,D2]"
   ],
   [
    "C362",
    "x_vars[S9,D3]"
   ],
   [
    "C363",
    "x_vars[S9,D4]"
   ],
   [
    "C364",
    "x_vars[S9,D5]"
   ],
   [
    "C365",
    "x_vars[S9,D6]"
   ],
   [
    "C366",
    "x_vars[S9,D7]"
   ],
   [
    "C367",
    "x_vars[S9,D8]"
   ],
   [
    "C368",
    "x_vars[S9,D9]"
   ],
   [
    "C369",
    "x_vars[S9,D10]"
   ],
   [
    "C370",
    "x_vars[S9,D11]"
   ],
   [
    "C371",
    "x_vars[S9,D12]"
   ],
   [
    "C372",
    "x_vars[S9,D13]"
   ],
   [
    "C373",
    "x_vars[S9,D14]"
   ],
   [
    "C374",
    "x_vars[S9,D15]"
   ],
   [
    "C375",
    "x_vars[S9,D16]"
   ],
   [
    "C376",
    "x_vars[S9,D17]"
   ],
   [
    "C377",
    "x_vars[S9,D18]"
   ],
   [
    "C378",
    "x_vars[S9,D19]"
   ],
   [
    "C379",
    "x_vars[S9,D20]"
   ],
   [
    "C380",
    "x_vars[S10,D1]"
   ],
   [
    "C381",
    "x_vars[S10,D2]"
   ],
   [
    "C382",
    "x_vars[S10,D3]"
   ],
   [
    "C383",
    "x_vars[S10,D4]"
   ],
   [
    "C384",
    "x_vars[S10,D5]"
   ],
   [
    "C385",
    "x_vars[S10,D6]"
   ],
   [
    "C386",
    "x_vars[S10,D7]"
   ],
   [
    "C387",
    "x_vars[S10,D8]"
   ],
   [
    "C388",
    "x_vars[S10,D9]"
   ],
   [
    "C389",
    "x_vars[S10,D10]"
   ],
   [
    "C390",
    "x_vars[S10,D11]"
   ],
   [
    "C391",
    "x_vars[S10,D12]"
   ],
   [
    "C392",
    "x_vars[S10,D13]"
   ],
   [
    "C393",
    "x_vars[S10,D14]"
   ],
   [
    "C394",
    "x_vars[S10,D15]"
   ],
   [
    "C395",
    "x_vars[S10,D16]"
   ],
   [
    "C396",
    "x_vars[S10,D17]"
   ],
   [
    "C397",
    "x_vars[S10,D18]"
   ],
   [
    "C398",
    "x_vars[S10,D19]"
   ],
   [
    "C399",
    "x_vars[S10,D20]"
   ]
  ],
  "primary_mapping": {
   "x[1,1]": "t_vars[S1,D1]",
   "x[1,2]": "t_vars[S1,D2]",
   "x[1,3]": "t_vars[S1,D3]",
   "x[1,4]": "t_vars[S1,D4]",
   "x[1,5]": "t_vars[S1,D5]",
   "x[1,6]": "t_vars[S1,D6]",
   "x[1,7]": "t_vars[S1,D7]",
   "x[1,8]": "t_vars[S1,D8]",
   "x[1,9]": "t_vars[S1,D9]",
   "x[1,10]": "t_vars[S1,D10]",
   "x[1,11]": "t_vars[S1,D11]",
   "x[1,12]": "t_vars[S1,D12]",
   "x[1,13]": "t_vars[S1,D13]",
   "x[1,14]": "t_vars[S1,D14]",
   "x[1,15]": "t_vars[S1,D15]",
   "x[1,16]": "t_vars[S1,D16]",
   "x[1,17]": "t_vars[S1,D17]",
   "x[1,18]": "t_vars[S1,D18]",
   "x[1,19]": "t_vars[S1,D19]",
   "x[1,20]": "t_vars[S1,D20]",
   "x[2,1]": "t_vars[S2,D1]",
   "x[2,2]": "t_vars[S2,D2]",
   "x[2,3]": "t_vars[S2,D3]",
   "x[2,4]": "t_vars[S2,D4]",
   "x[2,5]": "t_vars[S2,D5]",
   "x[2,6]": "t_vars[S2,D6]",
   "x[2,7]": "t_vars[S2,D7]",
   "x[2,8]": "t_vars[S2,D8]",
   "x[2,9]": "t_vars[S2,D9]",
   "x[2,10]": "t_vars[S2,D10]",
   "x[2,11]": "t_vars[S2,D11]",
   "x[2,12]": "t_vars[S2,D12]",
   "x[2,13]": "t_vars[S2,D13]",
   "x[2,14]": "t_vars[S2,D14]",
   "x[2,15]": "t_vars[S2,D15]",
   "x[2,16]": "t_vars[S2,D16]",
   "x[2,17]": "t_vars[S2,D17]",
   "x[2,18]": "t_vars[S2,D18]",
   "x[2,19]": "t_vars[S2,D19]",
   "x[2,20]": "t_vars[S2,D20]",
   "x[3,1]": "t_vars[S3,D1]",
   "x[3,2]": "t_vars[S3,D2]",
   "x[3,3]": "t_vars[S3,D3]",
   "x[3,4]": "t_vars[S3,D4]",
   "x[3,5]": "t_vars[S3,D5]",
   "x[3,6]": "t_vars[S3,D6]",
   "x[3,7]": "t_vars[S3,D7]",
   "x[3,8]": "t_vars[S3,D8]",
   "x[3,9]": "t_vars[S3,D9]",
   "x[3,10]": "t_vars[S3,D10]",
   "x[3,11]": "t_vars[S3,D11]",
   "x[3,12]": "t_vars[S3,D12]",
   "x[3,13]": "t_vars[S3,D13]",
   "x[3,14]": "t_vars[S3,D14]",
   "x[3,15]": "t_vars[S3,D15]",
   "x[3,16]": "t_vars[S3,D16]",
   "x[3,17]": "t_vars[S3,D17]",
   "x[3,18]": "t_vars[S3,D18]",
   "x[3,19]": "t_vars[S3,D19]",
   "x[3,20]": "t_vars[S3,D20]",
   "x[4,1]": "t_vars[S4,D1]",
   "x[4,2]": "t_vars[S4,D2]",
   "x[4,3]": "t_vars[S4,D3]",
   "x[4,4]": "t_vars[S4,D4]",
   "x[4,5]": "t_vars[S4,D5]",
   "x[4,6]": "t_vars[S4,D6]",
   "x[4,7]": "t_vars[S4,D7]",
   "x[4,8]": "t_vars[S4,D8]",
   "x[4,9]": "t_vars[S4,D9]",
   "x[4,10]": "t_vars[S4,D10]",
   "x[4,11]": "t_vars[S4,D11]",
   "x[4,12]": "t_vars[S4,D12]",
   "x[4,13]": "t_vars[S4,D13]",
   "x[4,14]": "t_vars[S4,D14]",
   "x[4,15]": "t_vars[S4,D15]",
   "x[4,16]": "t_vars[S4,D16]",
   "x[4,17]": "t_vars[S4,D17]",
   "x[4,18]": "t_vars[S4,D18]",
   "x[4,19]": "t_vars[S4,D19]",
   "x[4,20]": "t_vars[S4,D20]",
   "x[5,1]": "t_vars[S5,D1]",
   "x[5,2]": "t_vars[S5,D2]",
   "x[5,3]": "t_vars[S5,D3]",
   "x[5,4]": "t_vars[S5,D4]",
   "x[5,5]": "t_vars[S5,D5]",
   "x[5,6]": "t_vars[S5,D6]",
   "x[5,7]": "t_vars[S5,D7]",
   "x[5,8]": "t_vars[S5,D8]",
   "x[5,9]": "t_vars[S5,D9]",
   "x[5,10]": "t_vars[S5,D10]",
   "x[5,11]": "t_vars[S5,D11]",
   "x[5,12]": "t_vars[S5,D12]",
   "x[5,13]": "t_vars[S5,D13]",
   "x[5,14]": "t_vars[S5,D14]",
   "x[5,15]": "t_vars[S5,D15]",
   "x[5,16]": "t_vars[S5,D16]",
   "x[5,17]": "t_vars[S5,D17]",
   "x[5,18]": "t_vars[S5,D18]",
   "x[5,19]": "t_vars[S5,D19]",
   "x[5,20]": "t_vars[S5,D20]",
   "x[6,1]": "t_vars[S6,D1]",
   "x[6,2]": "t_vars[S6,D2]",
   "x[6,3]": "t_vars[S6,D3]",
   "x[6,4]": "t_vars[S6,D4]",
   "x[6,5]": "t_vars[S6,D5]",
   "x[6,6]": "t_vars[S6,D6]",
   "x[6,7]": "t_vars[S6,D7]",
   "x[6,8]": "t_vars[S6,D8]",
   "x[6,9]": "t_vars[S6,D9]",
   "x[6,10]": "t_vars[S6,D10]",
   "x[6,11]": "t_vars[S6,D11]",
   "x[6,12]": "t_vars[S6,D12]",
   "x[6,13]": "t_vars[S6,D13]",
   "x[6,14]": "t_vars[S6,D14]",
   "x[6,15]": "t_vars[S6,D15]",
   "x[6,16]": "t_vars[S6,D16]",
   "x[6,17]": "t_vars[S6,D17]",
   "x[6,18]": "t_vars[S6,D18]",
   "x[6,19]": "t_vars[S6,D19]",
   "x[6,20]": "t_vars[S6,D20]",
   "x[7,1]": "t_vars[S7,D1]",
   "x[7,2]": "t_vars[S7,D2]",
   "x[7,3]": "t_vars[S7,D3]",
   "x[7,4]": "t_vars[S7,D4]",
   "x[7,5]": "t_vars[S7,D5]",
   "x[7,6]": "t_vars[S7,D6]",
   "x[7,7]": "t_vars[S7,D7]",
   "x[7,8]": "t_vars[S7,D8]",
   "x[7,9]": "t_vars[S7,D9]",
   "x[7,10]": "t_vars[S7,D10]",
   "x[7,11]": "t_vars[S7,D11]",
   "x[7,12]": "t_vars[S7,D12]",
   "x[7,13]": "t_vars[S7,D13]",
   "x[7,14]": "t_vars[S7,D14]",
   "x[7,15]": "t_vars[S7,D15]",
   "x[7,16]": "t_vars[S7,D16]",
   "x[7,17]": "t_vars[S7,D17]",
   "x[7,18]": "t_vars[S7,D18]",
   "x[7,19]": "t_vars[S7,D19]",
   "x[7,20]": "t_vars[S7,D20]",
   "x[8,1]": "t_vars[S8,D1]",
   "x[8,2]": "t_vars[S8,D2]",
   "x[8,3]": "t_vars[S8,D3]",
   "x[8,4]": "t_vars[S8,D4]",
   "x[8,5]": "t_vars[S8,D5]",
   "x[8,6]": "t_vars[S8,D6]",
   "x[8,7]": "t_vars[S8,D7]",
   "x[8,8]": "t_vars[S8,D8]",
   "x[8,9]": "t_vars[S8,D9]",
   "x[8,10]": "t_vars[S8,D10]",
   "x[8,11]": "t_vars[S8,D11]",
   "x[8,12]": "t_vars[S8,D12]",
   "x[8,13]": "t_vars[S8,D13]",
   "x[8,14]": "t_vars[S8,D14]",
   "x[8,15]": "t_vars[S8,D15]",
   "x[8,16]": "t_vars[S8,D16]",
   "x[8,17]": "t_vars[S8,D17]",
   "x[8,18]": "t_vars[S8,D18]",
   "x[8,19]": "t_vars[S8,D19]",
   "x[8,20]": "t_vars[S8,D20]",
   "x[9,1]": "t_vars[S9,D1]",
   "x[9,2]": "t_vars[S9,D2]",
   "x[9,3]": "t_vars[S9,D3]",
   "x[9,4]": "t_vars[S9,D4]",
   "x[9,5]": "t_vars[S9,D5]",
   "x[9,6]": "t_vars[S9,D6]",
   "x[9,7]": "t_vars[S9,D7]",
   "x[9,8]": "t_vars[S9,D8]",
   "x[9,9]": "t_vars[S9,D9]",
   "x[9,10]": "t_vars[S9,D10]",
   "x[9,11]": "t_vars[S9,D11]",
   "x[9,12]": "t_vars[S9,D12]",
   "x[9,13]": "t_vars[S9,D13]",
   "x[9,14]": "t_vars[S9,D14]",
   "x[9,15]": "t_vars[S9,D15]",
   "x[9,16]": "t_vars[S9,D16]",
   "x[9,17]": "t_vars[S9,D17]",
   "x[9,18]": "t_vars[S9,D18]",
   "x[9,19]": "t_vars[S9,D19]",
   "x[9,20]": "t_vars[S9,D20]",
   "x[10,1]": "t_vars[S10,D1]",
   "x[10,2]": "t_vars[S10,D2]",
   "x[10,3]": "t_vars[S10,D3]",
   "x[10,4]": "t_vars[S10,D4]",
   "x[10,5]": "t_vars[S10,D5]",
   "x[10,6]": "t_vars[S10,D6]",
   "x[10,7]": "t_vars[S10,D7]",
   "x[10,8]": "t_vars[S10,D8]",
   "x[10,9]": "t_vars[S10,D9]",
   "x[10,10]": "t_vars[S10,D10]",
   "x[10,11]": "t_vars[S10,D11]",
   "x[10,12]": "t_vars[S10,D12]",
   "x[10,13]": "t_vars[S10,D13]",
   "x[10,14]": "t_vars[S10,D14]",
   "x[10,15]": "t_vars[S10,D15]",
   "x[10,16]": "t_vars[S10,D16]",
   "x[10,17]": "t_vars[S10,D17]",
   "x[10,18]": "t_vars[S10,D18]",
   "x[10,19]": "t_vars[S10,D19]",
   "x[10,20]": "t_vars[S10,D20]",
   "y[1,1]": "x_vars[S1,D1]",
   "y[1,2]": "x_vars[S1,D2]",
   "y[1,3]": "x_vars[S1,D3]",
   "y[1,4]": "x_vars[S1,D4]",
   "y[1,5]": "x_vars[S1,D5]",
   "y[1,6]": "x_vars[S1,D6]",
   "y[1,7]": "x_vars[S1,D7]",
   "y[1,8]": "x_vars[S1,D8]",
   "y[1,9]": "x_vars[S1,D9]",
   "y[1,10]": "x_vars[S1,D10]",
   "y[1,11]": "x_vars[S1,D11]",
   "y[1,12]": "x_vars[S1,D12]",
   "y[1,13]": "x_vars[S1,D13]",
   "y[1,14]": "x_vars[S1,D14]",
   "y[1,15]": "x_vars[S1,D15]",
   "y[1,16]": "x_vars[S1,D16]",
   "y[1,17]": "x_vars[S1,D17]",
   "y[1,18]": "x_vars[S1,D18]",
   "y[1,19]": "x_vars[S1,D19]",
   "y[1,20]": "x_vars[S1,D20]",
   "y[2,1]": "x_vars[S2,D1]",
   "y[2,2]": "x_vars[S2,D2]",
   "y[2,3]": "x_vars[S2,D3]",
   "y[2,4]": "x_vars[S2,D4]",
   "y[2,5]": "x_vars[S2,D5]",
   "y[2,6]": "x_vars[S2,D6]",
   "y[2,7]": "x_vars[S2,D7]",
   "y[2,8]": "x_vars[S2,D8]",
   "y[2,9]": "x_vars[S2,D9]",
   "y[2,10]": "x_vars[S2,D10]",
   "y[2,11]": "x_vars[S2,D11]",
   "y[2,12]": "x_vars[S2,D12]",
   "y[2,13]": "x_vars[S2,D13]",
   "y[2,14]": "x_vars[S2,D14]",
   "y[2,15]": "x_vars[S2,D15]",
   "y[2,16]": "x_vars[S2,D16]",
   "y[2,17]": "x_vars[S2,D17]",
   "y[2,18]": "x_vars[S2,D18]",
   "y[2,19]": "x_vars[S2,D19]",
   "y[2,20]": "x_vars[S2,D20]",
   "y[3,1]": "x_vars[S3,D1]",
   "y[3,2]": "x_vars[S3,D2]",
   "y[3,3]": "x_vars[S3,D3]",
   "y[3,4]": "x_vars[S3,D4]",
   "y[3,5]": "x_vars[S3,D5]",
   "y[3,6]": "x_vars[S3,D6]",
   "y[3,7]": "x_vars[S3,D7]",
   "y[3,8]": "x_vars[S3,D8]",
   "y[3,9]": "x_vars[S3,D9]",
   "y[3,10]": "x_vars[S3,D10]",
   "y[3,11]": "x_vars[S3,D11]",
   "y[3,12]": "x_vars[S3,D12]",
   "y[3,13]": "x_vars[S3,D13]",
   "y[3,14]": "x_vars[S3,D14]",
   "y[3,15]": "x_vars[S3,D15]",
   "y[3,16]": "x_vars[S3,D16]",
   "y[3,17]": "x_vars[S3,D17]",
   "y[3,18]": "x_vars[S3,D18]",
   "y[3,19]": "x_vars[S3,D19]",
   "y[3,20]": "x_vars[S3,D20]",
   "y[4,1]": "x_vars[S4,D1]",
   "y[4,2]": "x_vars[S4,D2]",
   "y[4,3]": "x_vars[S4,D3]",
   "y[4,4]": "x_vars[S4,D4]",
   "y[4,5]": "x_vars[S4,D5]",
   "y[4,6]": "x_vars[S4,D6]",
   "y[4,7]": "x_vars[S4,D7]",
   "y[4,8]": "x_vars[S4,D8]",
   "y[4,9]": "x_vars[S4,D9]",
   "y[4,10]": "x_vars[S4,D10]",
   "y[4,11]": "x_vars[S4,D11]",
   "y[4,12]": "x_vars[S4,D12]",
   "y[4,13]": "x_vars[S4,D13]",
   "y[4,14]": "x_vars[S4,D14]",
   "y[4,15]": "x_vars[S4,D15]",
   "y[4,16]": "x_vars[S4,D16]",
   "y[4,17]": "x_vars[S4,D17]",
   "y[4,18]": "x_vars[S4,D18]",
   "y[4,19]": "x_vars[S4,D19]",
   "y[4,20]": "x_vars[S4,D20]",
   "y[5,1]": "x_vars[S5,D1]",
   "y[5,2]": "x_vars[S5,D2]",
   "y[5,3]": "x_vars[S5,D3]",
   "y[5,4]": "x_vars[S5,D4]",
   "y[5,5]": "x_vars[S5,D5]",
   "y[5,6]": "x_vars[S5,D6]",
   "y[5,7]": "x_vars[S5,D7]",
   "y[5,8]": "x_vars[S5,D8]",
   "y[5,9]": "x_vars[S5,D9]",
   "y[5,10]": "x_vars[S5,D10]",
   "y[5,11]": "x_vars[S5,D11]",
   "y[5,12]": "x_vars[S5,D12]",
   "y[5,13]": "x_vars[S5,D13]",
   "y[5,14]": "x_vars[S5,D14]",
   "y[5,15]": "x_vars[S5,D15]",
   "y[5,16]": "x_vars[S5,D16]",
   "y[5,17]": "x_vars[S5,D17]",
   "y[5,18]": "x_vars[S5,D18]",
   "y[5,19]": "x_vars[S5,D19]",
   "y[5,20]": "x_vars[S5,D20]",
   "y[6,1]": "x_vars[S6,D1]",
   "y[6,2]": "x_vars[S6,D2]",
   "y[6,3]": "x_vars[S6,D3]",
   "y[6,4]": "x_vars[S6,D4]",
   "y[6,5]": "x_vars[S6,D5]",
   "y[6,6]": "x_vars[S6,D6]",
   "y[6,7]": "x_vars[S6,D7]",
   "y[6,8]": "x_vars[S6,D8]",
   "y[6,9]": "x_vars[S6,D9]",
   "y[6,10]": "x_vars[S6,D10]",
   "y[6,11]": "x_vars[S6,D11]",
   "y[6,12]": "x_vars[S6,D12]",
   "y[6,13]": "x_vars[S6,D13]",
   "y[6,14]": "x_vars[S6,D14]",
   "y[6,15]": "x_vars[S6,D15]",
   "y[6,16]": "x_vars[S6,D16]",
   "y[6,17]": "x_vars[S6,D17]",
   "y[6,18]": "x_vars[S6,D18]",
   "y[6,19]": "x_vars[S6,D19]",
   "y[6,20]": "x_vars[S6,D20]",
   "y[7,1]": "x_vars[S7,D1]",
   "y[7,2]": "x_vars[S7,D2]",
   "y[7,3]": "x_vars[S7,D3]",
   "y[7,4]": "x_vars[S7,D4]",
   "y[7,5]": "x_vars[S7,D5]",
   "y[7,6]": "x_vars[S7,D6]",
   "y[7,7]": "x_vars[S7,D7]",
   "y[7,8]": "x_vars[S7,D8]",
   "y[7,9]": "x_vars[S7,D9]",
   "y[7,10]": "x_vars[S7,D10]",
   "y[7,11]": "x_vars[S7,D11]",
   "y[7,12]": "x_vars[S7,D12]",
   "y[7,13]": "x_vars[S7,D13]",
   "y[7,14]": "x_vars[S7,D14]",
   "y[7,15]": "x_vars[S7,D15]",
   "y[7,16]": "x_vars[S7,D16]",
   "y[7,17]": "x_vars[S7,D17]",
   "y[7,18]": "x_vars[S7,D18]",
   "y[7,19]": "x_vars[S7,D19]",
   "y[7,20]": "x_vars[S7,D20]",
   "y[8,1]": "x_vars[S8,D1]",
   "y[8,2]": "x_vars[S8,D2]",
   "y[8,3]": "x_vars[S8,D3]",
   "y[8,4]": "x_vars[S8,D4]",
   "y[8,5]": "x_vars[S8,D5]",
   "y[8,6]": "x_vars[S8,D6]",
   "y[8,7]": "x_vars[S8,D7]",
   "y[8,8]": "x_vars[S8,D8]",
   "y[8,9]": "x_vars[S8,D9]",
   "y[8,10]": "x_vars[S8,D10]",
   "y[8,11]": "x_vars[S8,D11]",
   "y[8,12]": "x_vars[S8,D12]",
   "y[8,13]": "x_vars[S8,D13]",
   "y[8,14]": "x_vars[S8,D14]",
   "y[8,15]": "x_vars[S8,D15]",
   "y[8,16]": "x_vars[S8,D16]",
   "y[8,17]": "x_vars[S8,D17]",
   "y[8,18]": "x_vars[S8,D18]",
   "y[8,19]": "x_vars[S8,D19]",
   "y[8,20]": "x_vars[S8,D20]",
   "y[9,1]": "x_vars[S9,D1]",
   "y[9,2]": "x_vars[S9,D2]",
   "y[9,3]": "x_vars[S9,D3]",
   "y[9,4]": "x_vars[S9,D4]",
   "y[9,5]": "x_vars[S9,D5]",
   "y[9,6]": "x_vars[S9,D6]",
   "y[9,7]": "x_vars[S9,D7]",
   "y[9,8]": "x_vars[S9,D8]",
   "y[9,9]": "x_vars[S9,D9]",
   "y[9,10]": "x_vars[S9,D10]",
   "y[9,11]": "x_vars[S9,D11]",
   "y[9,12]": "x_vars[S9,D12]",
   "y[9,13]": "x_vars[S9,D13]",
   "y[9,14]": "x_vars[S9,D14]",
   "y[9,15]": "x_vars[S9,D15]",
   "y[9,16]": "x_vars[S9,D16]",
   "y[9,17]": "x_vars[S9,D17]",
   "y[9,18]": "x_vars[S9,D18]",
   "y[9,19]": "x_vars[S9,D19]",
   "y[9,20]": "x_vars[S9,D20]",
   "y[10,1]": "x_vars[S10,D1]",
   "y[10,2]": "x_vars[S10,D2]",
   "y[10,3]": "x_vars[S10,D3]",
   "y[10,4]": "x_vars[S10,D4]",
   "y[10,5]": "x_vars[S10,D5]",
   "y[10,6]": "x_vars[S10,D6]",
   "y[10,7]": "x_vars[S10,D7]",
   "y[10,8]": "x_vars[S10,D8]",
   "y[10,9]": "x_vars[S10,D9]",
   "y[10,10]": "x_vars[S10,D10]",
   "y[10,11]": "x_vars[S10,D11]",
   "y[10,12]": "x_vars[S10,D12]",
   "y[10,13]": "x_vars[S10,D13]",
   "y[10,14]": "x_vars[S10,D14]",
   "y[10,15]": "x_vars[S10,D15]",
   "y[10,16]": "x_vars[S10,D16]",
   "y[10,17]": "x_vars[S10,D17]",
   "y[10,18]": "x_vars[S10,D18]",
   "y[10,19]": "x_vars[S10,D19]",
   "y[10,20]": "x_vars[S10,D20]"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-093": {
  "source_code_sha256": "baec83dcc0910f59253f5f0c8bcf34f59ebf51a0f506c223ee1a0a2ae91567d7",
  "original_solution_sha256": "818a5456a4f6112000373feaccd8f83c1a2d753c7bf0c41e145030a3fcb182a8",
  "bindings": [
   [
    "C0",
    "x_vars[M1,A]"
   ],
   [
    "C1",
    "x_vars[M1,B]"
   ],
   [
    "C2",
    "x_vars[M1,C]"
   ],
   [
    "C3",
    "x_vars[M1,D]"
   ],
   [
    "C4",
    "x_vars[M1,E]"
   ],
   [
    "C5",
    "x_vars[M1,F]"
   ],
   [
    "C6",
    "x_vars[M1,G]"
   ],
   [
    "C7",
    "x_vars[M1,H]"
   ],
   [
    "C8",
    "x_vars[M1,I]"
   ],
   [
    "C9",
    "x_vars[M1,J]"
   ],
   [
    "C10",
    "x_vars[M1,K]"
   ],
   [
    "C11",
    "x_vars[M1,L]"
   ],
   [
    "C12",
    "x_vars[M2,A]"
   ],
   [
    "C13",
    "x_vars[M2,B]"
   ],
   [
    "C14",
    "x_vars[M2,C]"
   ],
   [
    "C15",
    "x_vars[M2,D]"
   ],
   [
    "C16",
    "x_vars[M2,E]"
   ],
   [
    "C17",
    "x_vars[M2,F]"
   ],
   [
    "C18",
    "x_vars[M2,G]"
   ],
   [
    "C19",
    "x_vars[M2,H]"
   ],
   [
    "C20",
    "x_vars[M2,I]"
   ],
   [
    "C21",
    "x_vars[M2,J]"
   ],
   [
    "C22",
    "x_vars[M2,K]"
   ],
   [
    "C23",
    "x_vars[M2,L]"
   ],
   [
    "C24",
    "x_vars[M3,A]"
   ],
   [
    "C25",
    "x_vars[M3,B]"
   ],
   [
    "C26",
    "x_vars[M3,C]"
   ],
   [
    "C27",
    "x_vars[M3,D]"
   ],
   [
    "C28",
    "x_vars[M3,E]"
   ],
   [
    "C29",
    "x_vars[M3,F]"
   ],
   [
    "C30",
    "x_vars[M3,G]"
   ],
   [
    "C31",
    "x_vars[M3,H]"
   ],
   [
    "C32",
    "x_vars[M3,I]"
   ],
   [
    "C33",
    "x_vars[M3,J]"
   ],
   [
    "C34",
    "x_vars[M3,K]"
   ],
   [
    "C35",
    "x_vars[M3,L]"
   ],
   [
    "C36",
    "x_vars[M4,A]"
   ],
   [
    "C37",
    "x_vars[M4,B]"
   ],
   [
    "C38",
    "x_vars[M4,C]"
   ],
   [
    "C39",
    "x_vars[M4,D]"
   ],
   [
    "C40",
    "x_vars[M4,E]"
   ],
   [
    "C41",
    "x_vars[M4,F]"
   ],
   [
    "C42",
    "x_vars[M4,G]"
   ],
   [
    "C43",
    "x_vars[M4,H]"
   ],
   [
    "C44",
    "x_vars[M4,I]"
   ],
   [
    "C45",
    "x_vars[M4,J]"
   ],
   [
    "C46",
    "x_vars[M4,K]"
   ],
   [
    "C47",
    "x_vars[M4,L]"
   ],
   [
    "C48",
    "x_vars[M5,A]"
   ],
   [
    "C49",
    "x_vars[M5,B]"
   ],
   [
    "C50",
    "x_vars[M5,C]"
   ],
   [
    "C51",
    "x_vars[M5,D]"
   ],
   [
    "C52",
    "x_vars[M5,E]"
   ],
   [
    "C53",
    "x_vars[M5,F]"
   ],
   [
    "C54",
    "x_vars[M5,G]"
   ],
   [
    "C55",
    "x_vars[M5,H]"
   ],
   [
    "C56",
    "x_vars[M5,I]"
   ],
   [
    "C57",
    "x_vars[M5,J]"
   ],
   [
    "C58",
    "x_vars[M5,K]"
   ],
   [
    "C59",
    "x_vars[M5,L]"
   ],
   [
    "C60",
    "x_vars[M6,A]"
   ],
   [
    "C61",
    "x_vars[M6,B]"
   ],
   [
    "C62",
    "x_vars[M6,C]"
   ],
   [
    "C63",
    "x_vars[M6,D]"
   ],
   [
    "C64",
    "x_vars[M6,E]"
   ],
   [
    "C65",
    "x_vars[M6,F]"
   ],
   [
    "C66",
    "x_vars[M6,G]"
   ],
   [
    "C67",
    "x_vars[M6,H]"
   ],
   [
    "C68",
    "x_vars[M6,I]"
   ],
   [
    "C69",
    "x_vars[M6,J]"
   ],
   [
    "C70",
    "x_vars[M6,K]"
   ],
   [
    "C71",
    "x_vars[M6,L]"
   ],
   [
    "C72",
    "x_vars[M7,A]"
   ],
   [
    "C73",
    "x_vars[M7,B]"
   ],
   [
    "C74",
    "x_vars[M7,C]"
   ],
   [
    "C75",
    "x_vars[M7,D]"
   ],
   [
    "C76",
    "x_vars[M7,E]"
   ],
   [
    "C77",
    "x_vars[M7,F]"
   ],
   [
    "C78",
    "x_vars[M7,G]"
   ],
   [
    "C79",
    "x_vars[M7,H]"
   ],
   [
    "C80",
    "x_vars[M7,I]"
   ],
   [
    "C81",
    "x_vars[M7,J]"
   ],
   [
    "C82",
    "x_vars[M7,K]"
   ],
   [
    "C83",
    "x_vars[M7,L]"
   ],
   [
    "C84",
    "x_vars[M8,A]"
   ],
   [
    "C85",
    "x_vars[M8,B]"
   ],
   [
    "C86",
    "x_vars[M8,C]"
   ],
   [
    "C87",
    "x_vars[M8,D]"
   ],
   [
    "C88",
    "x_vars[M8,E]"
   ],
   [
    "C89",
    "x_vars[M8,F]"
   ],
   [
    "C90",
    "x_vars[M8,G]"
   ],
   [
    "C91",
    "x_vars[M8,H]"
   ],
   [
    "C92",
    "x_vars[M8,I]"
   ],
   [
    "C93",
    "x_vars[M8,J]"
   ],
   [
    "C94",
    "x_vars[M8,K]"
   ],
   [
    "C95",
    "x_vars[M8,L]"
   ],
   [
    "C96",
    "x_vars[M9,A]"
   ],
   [
    "C97",
    "x_vars[M9,B]"
   ],
   [
    "C98",
    "x_vars[M9,C]"
   ],
   [
    "C99",
    "x_vars[M9,D]"
   ],
   [
    "C100",
    "x_vars[M9,E]"
   ],
   [
    "C101",
    "x_vars[M9,F]"
   ],
   [
    "C102",
    "x_vars[M9,G]"
   ],
   [
    "C103",
    "x_vars[M9,H]"
   ],
   [
    "C104",
    "x_vars[M9,I]"
   ],
   [
    "C105",
    "x_vars[M9,J]"
   ],
   [
    "C106",
    "x_vars[M9,K]"
   ],
   [
    "C107",
    "x_vars[M9,L]"
   ],
   [
    "C108",
    "x_vars[M10,A]"
   ],
   [
    "C109",
    "x_vars[M10,B]"
   ],
   [
    "C110",
    "x_vars[M10,C]"
   ],
   [
    "C111",
    "x_vars[M10,D]"
   ],
   [
    "C112",
    "x_vars[M10,E]"
   ],
   [
    "C113",
    "x_vars[M10,F]"
   ],
   [
    "C114",
    "x_vars[M10,G]"
   ],
   [
    "C115",
    "x_vars[M10,H]"
   ],
   [
    "C116",
    "x_vars[M10,I]"
   ],
   [
    "C117",
    "x_vars[M10,J]"
   ],
   [
    "C118",
    "x_vars[M10,K]"
   ],
   [
    "C119",
    "x_vars[M10,L]"
   ],
   [
    "C120",
    "x_vars[M11,A]"
   ],
   [
    "C121",
    "x_vars[M11,B]"
   ],
   [
    "C122",
    "x_vars[M11,C]"
   ],
   [
    "C123",
    "x_vars[M11,D]"
   ],
   [
    "C124",
    "x_vars[M11,E]"
   ],
   [
    "C125",
    "x_vars[M11,F]"
   ],
   [
    "C126",
    "x_vars[M11,G]"
   ],
   [
    "C127",
    "x_vars[M11,H]"
   ],
   [
    "C128",
    "x_vars[M11,I]"
   ],
   [
    "C129",
    "x_vars[M11,J]"
   ],
   [
    "C130",
    "x_vars[M11,K]"
   ],
   [
    "C131",
    "x_vars[M11,L]"
   ],
   [
    "C132",
    "x_vars[M12,A]"
   ],
   [
    "C133",
    "x_vars[M12,B]"
   ],
   [
    "C134",
    "x_vars[M12,C]"
   ],
   [
    "C135",
    "x_vars[M12,D]"
   ],
   [
    "C136",
    "x_vars[M12,E]"
   ],
   [
    "C137",
    "x_vars[M12,F]"
   ],
   [
    "C138",
    "x_vars[M12,G]"
   ],
   [
    "C139",
    "x_vars[M12,H]"
   ],
   [
    "C140",
    "x_vars[M12,I]"
   ],
   [
    "C141",
    "x_vars[M12,J]"
   ],
   [
    "C142",
    "x_vars[M12,K]"
   ],
   [
    "C143",
    "x_vars[M12,L]"
   ]
  ],
  "primary_mapping": {
   "x[1,A]": "x_vars[M1,A]",
   "x[1,B]": "x_vars[M1,B]",
   "x[1,C]": "x_vars[M1,C]",
   "x[1,D]": "x_vars[M1,D]",
   "x[1,E]": "x_vars[M1,E]",
   "x[1,F]": "x_vars[M1,F]",
   "x[1,G]": "x_vars[M1,G]",
   "x[1,H]": "x_vars[M1,H]",
   "x[1,I]": "x_vars[M1,I]",
   "x[1,J]": "x_vars[M1,J]",
   "x[1,K]": "x_vars[M1,K]",
   "x[1,L]": "x_vars[M1,L]",
   "x[2,A]": "x_vars[M2,A]",
   "x[2,B]": "x_vars[M2,B]",
   "x[2,C]": "x_vars[M2,C]",
   "x[2,D]": "x_vars[M2,D]",
   "x[2,E]": "x_vars[M2,E]",
   "x[2,F]": "x_vars[M2,F]",
   "x[2,G]": "x_vars[M2,G]",
   "x[2,H]": "x_vars[M2,H]",
   "x[2,I]": "x_vars[M2,I]",
   "x[2,J]": "x_vars[M2,J]",
   "x[2,K]": "x_vars[M2,K]",
   "x[2,L]": "x_vars[M2,L]",
   "x[3,A]": "x_vars[M3,A]",
   "x[3,B]": "x_vars[M3,B]",
   "x[3,C]": "x_vars[M3,C]",
   "x[3,D]": "x_vars[M3,D]",
   "x[3,E]": "x_vars[M3,E]",
   "x[3,F]": "x_vars[M3,F]",
   "x[3,G]": "x_vars[M3,G]",
   "x[3,H]": "x_vars[M3,H]",
   "x[3,I]": "x_vars[M3,I]",
   "x[3,J]": "x_vars[M3,J]",
   "x[3,K]": "x_vars[M3,K]",
   "x[3,L]": "x_vars[M3,L]",
   "x[4,A]": "x_vars[M4,A]",
   "x[4,B]": "x_vars[M4,B]",
   "x[4,C]": "x_vars[M4,C]",
   "x[4,D]": "x_vars[M4,D]",
   "x[4,E]": "x_vars[M4,E]",
   "x[4,F]": "x_vars[M4,F]",
   "x[4,G]": "x_vars[M4,G]",
   "x[4,H]": "x_vars[M4,H]",
   "x[4,I]": "x_vars[M4,I]",
   "x[4,J]": "x_vars[M4,J]",
   "x[4,K]": "x_vars[M4,K]",
   "x[4,L]": "x_vars[M4,L]",
   "x[5,A]": "x_vars[M5,A]",
   "x[5,B]": "x_vars[M5,B]",
   "x[5,C]": "x_vars[M5,C]",
   "x[5,D]": "x_vars[M5,D]",
   "x[5,E]": "x_vars[M5,E]",
   "x[5,F]": "x_vars[M5,F]",
   "x[5,G]": "x_vars[M5,G]",
   "x[5,H]": "x_vars[M5,H]",
   "x[5,I]": "x_vars[M5,I]",
   "x[5,J]": "x_vars[M5,J]",
   "x[5,K]": "x_vars[M5,K]",
   "x[5,L]": "x_vars[M5,L]",
   "x[6,A]": "x_vars[M6,A]",
   "x[6,B]": "x_vars[M6,B]",
   "x[6,C]": "x_vars[M6,C]",
   "x[6,D]": "x_vars[M6,D]",
   "x[6,E]": "x_vars[M6,E]",
   "x[6,F]": "x_vars[M6,F]",
   "x[6,G]": "x_vars[M6,G]",
   "x[6,H]": "x_vars[M6,H]",
   "x[6,I]": "x_vars[M6,I]",
   "x[6,J]": "x_vars[M6,J]",
   "x[6,K]": "x_vars[M6,K]",
   "x[6,L]": "x_vars[M6,L]",
   "x[7,A]": "x_vars[M7,A]",
   "x[7,B]": "x_vars[M7,B]",
   "x[7,C]": "x_vars[M7,C]",
   "x[7,D]": "x_vars[M7,D]",
   "x[7,E]": "x_vars[M7,E]",
   "x[7,F]": "x_vars[M7,F]",
   "x[7,G]": "x_vars[M7,G]",
   "x[7,H]": "x_vars[M7,H]",
   "x[7,I]": "x_vars[M7,I]",
   "x[7,J]": "x_vars[M7,J]",
   "x[7,K]": "x_vars[M7,K]",
   "x[7,L]": "x_vars[M7,L]",
   "x[8,A]": "x_vars[M8,A]",
   "x[8,B]": "x_vars[M8,B]",
   "x[8,C]": "x_vars[M8,C]",
   "x[8,D]": "x_vars[M8,D]",
   "x[8,E]": "x_vars[M8,E]",
   "x[8,F]": "x_vars[M8,F]",
   "x[8,G]": "x_vars[M8,G]",
   "x[8,H]": "x_vars[M8,H]",
   "x[8,I]": "x_vars[M8,I]",
   "x[8,J]": "x_vars[M8,J]",
   "x[8,K]": "x_vars[M8,K]",
   "x[8,L]": "x_vars[M8,L]",
   "x[9,A]": "x_vars[M9,A]",
   "x[9,B]": "x_vars[M9,B]",
   "x[9,C]": "x_vars[M9,C]",
   "x[9,D]": "x_vars[M9,D]",
   "x[9,E]": "x_vars[M9,E]",
   "x[9,F]": "x_vars[M9,F]",
   "x[9,G]": "x_vars[M9,G]",
   "x[9,H]": "x_vars[M9,H]",
   "x[9,I]": "x_vars[M9,I]",
   "x[9,J]": "x_vars[M9,J]",
   "x[9,K]": "x_vars[M9,K]",
   "x[9,L]": "x_vars[M9,L]",
   "x[10,A]": "x_vars[M10,A]",
   "x[10,B]": "x_vars[M10,B]",
   "x[10,C]": "x_vars[M10,C]",
   "x[10,D]": "x_vars[M10,D]",
   "x[10,E]": "x_vars[M10,E]",
   "x[10,F]": "x_vars[M10,F]",
   "x[10,G]": "x_vars[M10,G]",
   "x[10,H]": "x_vars[M10,H]",
   "x[10,I]": "x_vars[M10,I]",
   "x[10,J]": "x_vars[M10,J]",
   "x[10,K]": "x_vars[M10,K]",
   "x[10,L]": "x_vars[M10,L]",
   "x[11,A]": "x_vars[M11,A]",
   "x[11,B]": "x_vars[M11,B]",
   "x[11,C]": "x_vars[M11,C]",
   "x[11,D]": "x_vars[M11,D]",
   "x[11,E]": "x_vars[M11,E]",
   "x[11,F]": "x_vars[M11,F]",
   "x[11,G]": "x_vars[M11,G]",
   "x[11,H]": "x_vars[M11,H]",
   "x[11,I]": "x_vars[M11,I]",
   "x[11,J]": "x_vars[M11,J]",
   "x[11,K]": "x_vars[M11,K]",
   "x[11,L]": "x_vars[M11,L]",
   "x[12,A]": "x_vars[M12,A]",
   "x[12,B]": "x_vars[M12,B]",
   "x[12,C]": "x_vars[M12,C]",
   "x[12,D]": "x_vars[M12,D]",
   "x[12,E]": "x_vars[M12,E]",
   "x[12,F]": "x_vars[M12,F]",
   "x[12,G]": "x_vars[M12,G]",
   "x[12,H]": "x_vars[M12,H]",
   "x[12,I]": "x_vars[M12,I]",
   "x[12,J]": "x_vars[M12,J]",
   "x[12,K]": "x_vars[M12,K]",
   "x[12,L]": "x_vars[M12,L]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-094": {
  "source_code_sha256": "2b394d68a66894ccd02b71f587af89f2fff19c93d4c5bb50898c66eb56f202f4",
  "original_solution_sha256": "a0db7118366407996ddcf2e8fb576d74dd67f464ce69946a284b502f95a67ec5",
  "bindings": [
   [
    "C0",
    "x_vars[HiFi1]"
   ],
   [
    "C1",
    "x_vars[HiFi2]"
   ],
   [
    "C2",
    "x_vars[HiFi3]"
   ],
   [
    "C3",
    "x_vars[HiFi4]"
   ],
   [
    "C4",
    "x_vars[HiFi5]"
   ],
   [
    "C5",
    "x_vars[HiFi6]"
   ],
   [
    "C6",
    "x_vars[HiFi7]"
   ],
   [
    "C7",
    "x_vars[HiFi8]"
   ],
   [
    "C8",
    "x_vars[HiFi9]"
   ],
   [
    "C9",
    "x_vars[HiFi10]"
   ],
   [
    "C10",
    "x_vars[HiFi11]"
   ],
   [
    "C11",
    "x_vars[HiFi12]"
   ],
   [
    "C12",
    "x_vars[HiFi13]"
   ],
   [
    "C13",
    "x_vars[HiFi14]"
   ],
   [
    "C14",
    "x_vars[HiFi15]"
   ],
   [
    "C15",
    "x_vars[HiFi16]"
   ],
   [
    "C16",
    "x_vars[HiFi17]"
   ],
   [
    "C17",
    "x_vars[HiFi18]"
   ],
   [
    "C18",
    "x_vars[HiFi19]"
   ],
   [
    "C19",
    "x_vars[HiFi20]"
   ],
   [
    "C20",
    "x_vars[HiFi21]"
   ],
   [
    "C21",
    "x_vars[HiFi22]"
   ],
   [
    "C22",
    "x_vars[HiFi23]"
   ],
   [
    "C23",
    "x_vars[HiFi24]"
   ],
   [
    "C24",
    "x_vars[HiFi25]"
   ],
   [
    "C25",
    "x_vars[HiFi26]"
   ],
   [
    "C26",
    "x_vars[HiFi27]"
   ],
   [
    "C27",
    "x_vars[HiFi28]"
   ],
   [
    "C28",
    "x_vars[HiFi29]"
   ],
   [
    "C29",
    "x_vars[HiFi30]"
   ],
   [
    "C30",
    "x_vars[HiFi31]"
   ],
   [
    "C31",
    "x_vars[HiFi32]"
   ],
   [
    "C32",
    "x_vars[HiFi33]"
   ],
   [
    "C33",
    "x_vars[HiFi34]"
   ],
   [
    "C34",
    "x_vars[HiFi35]"
   ],
   [
    "C35",
    "x_vars[HiFi36]"
   ],
   [
    "C36",
    "x_vars[HiFi37]"
   ],
   [
    "C37",
    "x_vars[HiFi38]"
   ],
   [
    "C38",
    "x_vars[HiFi39]"
   ],
   [
    "C39",
    "x_vars[HiFi40]"
   ],
   [
    "C40",
    "x_vars[HiFi41]"
   ],
   [
    "C41",
    "x_vars[HiFi42]"
   ],
   [
    "C42",
    "x_vars[HiFi43]"
   ],
   [
    "C43",
    "x_vars[HiFi44]"
   ],
   [
    "C44",
    "x_vars[HiFi45]"
   ],
   [
    "C45",
    "x_vars[HiFi46]"
   ],
   [
    "C46",
    "x_vars[HiFi47]"
   ],
   [
    "C47",
    "x_vars[HiFi48]"
   ],
   [
    "C48",
    "x_vars[HiFi49]"
   ],
   [
    "C49",
    "x_vars[HiFi50]"
   ],
   [
    "C50",
    "x_vars[HiFi51]"
   ],
   [
    "C51",
    "x_vars[HiFi52]"
   ],
   [
    "C52",
    "x_vars[HiFi53]"
   ],
   [
    "C53",
    "x_vars[HiFi54]"
   ],
   [
    "C54",
    "x_vars[HiFi55]"
   ],
   [
    "C55",
    "x_vars[HiFi56]"
   ],
   [
    "C56",
    "x_vars[HiFi57]"
   ],
   [
    "C57",
    "x_vars[HiFi58]"
   ],
   [
    "C58",
    "x_vars[HiFi59]"
   ],
   [
    "C59",
    "x_vars[HiFi60]"
   ],
   [
    "C60",
    "x_vars[HiFi61]"
   ],
   [
    "C61",
    "x_vars[HiFi62]"
   ],
   [
    "C62",
    "x_vars[HiFi63]"
   ],
   [
    "C63",
    "x_vars[HiFi64]"
   ],
   [
    "C64",
    "x_vars[HiFi65]"
   ],
   [
    "C65",
    "x_vars[HiFi66]"
   ],
   [
    "C66",
    "x_vars[HiFi67]"
   ],
   [
    "C67",
    "x_vars[HiFi68]"
   ],
   [
    "C68",
    "x_vars[HiFi69]"
   ],
   [
    "C69",
    "x_vars[HiFi70]"
   ],
   [
    "C70",
    "x_vars[HiFi71]"
   ],
   [
    "C71",
    "x_vars[HiFi72]"
   ],
   [
    "C72",
    "x_vars[HiFi73]"
   ],
   [
    "C73",
    "x_vars[HiFi74]"
   ],
   [
    "C74",
    "x_vars[HiFi75]"
   ],
   [
    "C75",
    "x_vars[HiFi76]"
   ],
   [
    "C76",
    "x_vars[HiFi77]"
   ],
   [
    "C77",
    "x_vars[HiFi78]"
   ],
   [
    "C78",
    "x_vars[HiFi79]"
   ],
   [
    "C79",
    "x_vars[HiFi80]"
   ],
   [
    "C80",
    "x_vars[HiFi81]"
   ],
   [
    "C81",
    "x_vars[HiFi82]"
   ],
   [
    "C82",
    "x_vars[HiFi83]"
   ],
   [
    "C83",
    "x_vars[HiFi84]"
   ],
   [
    "C84",
    "x_vars[HiFi85]"
   ],
   [
    "C85",
    "x_vars[HiFi86]"
   ],
   [
    "C86",
    "x_vars[HiFi87]"
   ],
   [
    "C87",
    "x_vars[HiFi88]"
   ],
   [
    "C88",
    "x_vars[HiFi89]"
   ],
   [
    "C89",
    "x_vars[HiFi90]"
   ],
   [
    "C90",
    "x_vars[HiFi91]"
   ],
   [
    "C91",
    "x_vars[HiFi92]"
   ],
   [
    "C92",
    "x_vars[HiFi93]"
   ],
   [
    "C93",
    "x_vars[HiFi94]"
   ],
   [
    "C94",
    "x_vars[HiFi95]"
   ],
   [
    "C95",
    "x_vars[HiFi96]"
   ],
   [
    "C96",
    "x_vars[HiFi97]"
   ],
   [
    "C97",
    "x_vars[HiFi98]"
   ],
   [
    "C98",
    "x_vars[HiFi99]"
   ],
   [
    "C99",
    "x_vars[HiFi100]"
   ],
   [
    "C100",
    "x_vars[HiFi101]"
   ],
   [
    "C101",
    "y_vars[1]"
   ],
   [
    "C102",
    "y_vars[2]"
   ],
   [
    "C103",
    "y_vars[3]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x_vars[HiFi1]",
   "x[1]": "x_vars[HiFi2]",
   "x[2]": "x_vars[HiFi3]",
   "x[3]": "x_vars[HiFi4]",
   "x[4]": "x_vars[HiFi5]",
   "x[5]": "x_vars[HiFi6]",
   "x[6]": "x_vars[HiFi7]",
   "x[7]": "x_vars[HiFi8]",
   "x[8]": "x_vars[HiFi9]",
   "x[9]": "x_vars[HiFi10]",
   "x[10]": "x_vars[HiFi11]",
   "x[11]": "x_vars[HiFi12]",
   "x[12]": "x_vars[HiFi13]",
   "x[13]": "x_vars[HiFi14]",
   "x[14]": "x_vars[HiFi15]",
   "x[15]": "x_vars[HiFi16]",
   "x[16]": "x_vars[HiFi17]",
   "x[17]": "x_vars[HiFi18]",
   "x[18]": "x_vars[HiFi19]",
   "x[19]": "x_vars[HiFi20]",
   "x[20]": "x_vars[HiFi21]",
   "x[21]": "x_vars[HiFi22]",
   "x[22]": "x_vars[HiFi23]",
   "x[23]": "x_vars[HiFi24]",
   "x[24]": "x_vars[HiFi25]",
   "x[25]": "x_vars[HiFi26]",
   "x[26]": "x_vars[HiFi27]",
   "x[27]": "x_vars[HiFi28]",
   "x[28]": "x_vars[HiFi29]",
   "x[29]": "x_vars[HiFi30]",
   "x[30]": "x_vars[HiFi31]",
   "x[31]": "x_vars[HiFi32]",
   "x[32]": "x_vars[HiFi33]",
   "x[33]": "x_vars[HiFi34]",
   "x[34]": "x_vars[HiFi35]",
   "x[35]": "x_vars[HiFi36]",
   "x[36]": "x_vars[HiFi37]",
   "x[37]": "x_vars[HiFi38]",
   "x[38]": "x_vars[HiFi39]",
   "x[39]": "x_vars[HiFi40]",
   "x[40]": "x_vars[HiFi41]",
   "x[41]": "x_vars[HiFi42]",
   "x[42]": "x_vars[HiFi43]",
   "x[43]": "x_vars[HiFi44]",
   "x[44]": "x_vars[HiFi45]",
   "x[45]": "x_vars[HiFi46]",
   "x[46]": "x_vars[HiFi47]",
   "x[47]": "x_vars[HiFi48]",
   "x[48]": "x_vars[HiFi49]",
   "x[49]": "x_vars[HiFi50]",
   "x[50]": "x_vars[HiFi51]",
   "x[51]": "x_vars[HiFi52]",
   "x[52]": "x_vars[HiFi53]",
   "x[53]": "x_vars[HiFi54]",
   "x[54]": "x_vars[HiFi55]",
   "x[55]": "x_vars[HiFi56]",
   "x[56]": "x_vars[HiFi57]",
   "x[57]": "x_vars[HiFi58]",
   "x[58]": "x_vars[HiFi59]",
   "x[59]": "x_vars[HiFi60]",
   "x[60]": "x_vars[HiFi61]",
   "x[61]": "x_vars[HiFi62]",
   "x[62]": "x_vars[HiFi63]",
   "x[63]": "x_vars[HiFi64]",
   "x[64]": "x_vars[HiFi65]",
   "x[65]": "x_vars[HiFi66]",
   "x[66]": "x_vars[HiFi67]",
   "x[67]": "x_vars[HiFi68]",
   "x[68]": "x_vars[HiFi69]",
   "x[69]": "x_vars[HiFi70]",
   "x[70]": "x_vars[HiFi71]",
   "x[71]": "x_vars[HiFi72]",
   "x[72]": "x_vars[HiFi73]",
   "x[73]": "x_vars[HiFi74]",
   "x[74]": "x_vars[HiFi75]",
   "x[75]": "x_vars[HiFi76]",
   "x[76]": "x_vars[HiFi77]",
   "x[77]": "x_vars[HiFi78]",
   "x[78]": "x_vars[HiFi79]",
   "x[79]": "x_vars[HiFi80]",
   "x[80]": "x_vars[HiFi81]",
   "x[81]": "x_vars[HiFi82]",
   "x[82]": "x_vars[HiFi83]",
   "x[83]": "x_vars[HiFi84]",
   "x[84]": "x_vars[HiFi85]",
   "x[85]": "x_vars[HiFi86]",
   "x[86]": "x_vars[HiFi87]",
   "x[87]": "x_vars[HiFi88]",
   "x[88]": "x_vars[HiFi89]",
   "x[89]": "x_vars[HiFi90]",
   "x[90]": "x_vars[HiFi91]",
   "x[91]": "x_vars[HiFi92]",
   "x[92]": "x_vars[HiFi93]",
   "x[93]": "x_vars[HiFi94]",
   "x[94]": "x_vars[HiFi95]",
   "x[95]": "x_vars[HiFi96]",
   "x[96]": "x_vars[HiFi97]",
   "x[97]": "x_vars[HiFi98]",
   "x[98]": "x_vars[HiFi99]",
   "x[99]": "x_vars[HiFi100]",
   "x[100]": "x_vars[HiFi101]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-095": {
  "source_code_sha256": "74ad121cdfed4e37f43c7edb0db7378ba82462a2e413a04e3a15f692beedbd71",
  "original_solution_sha256": "d3643c67fb9bdbdb71147fdae974adff6f7fe9862f15ec3f6ed600fb55d27605",
  "bindings": [
   [
    "C0",
    "x_vars[Widget1]"
   ],
   [
    "C1",
    "x_vars[Widget2]"
   ],
   [
    "C2",
    "x_vars[Widget3]"
   ],
   [
    "C3",
    "x_vars[Widget4]"
   ],
   [
    "C4",
    "x_vars[Widget5]"
   ],
   [
    "C5",
    "x_vars[Widget6]"
   ],
   [
    "C6",
    "x_vars[Widget7]"
   ],
   [
    "C7",
    "x_vars[Widget8]"
   ],
   [
    "C8",
    "x_vars[Widget9]"
   ],
   [
    "C9",
    "x_vars[Widget10]"
   ],
   [
    "C10",
    "x_vars[Widget11]"
   ],
   [
    "C11",
    "x_vars[Widget12]"
   ],
   [
    "C12",
    "x_vars[Widget13]"
   ],
   [
    "C13",
    "x_vars[Widget14]"
   ],
   [
    "C14",
    "x_vars[Widget15]"
   ],
   [
    "C15",
    "x_vars[Widget16]"
   ],
   [
    "C16",
    "x_vars[Widget17]"
   ],
   [
    "C17",
    "x_vars[Widget18]"
   ],
   [
    "C18",
    "x_vars[Widget19]"
   ],
   [
    "C19",
    "x_vars[Widget20]"
   ],
   [
    "C20",
    "x_vars[Widget21]"
   ],
   [
    "C21",
    "x_vars[Widget22]"
   ],
   [
    "C22",
    "x_vars[Widget23]"
   ],
   [
    "C23",
    "x_vars[Widget24]"
   ],
   [
    "C24",
    "x_vars[Widget25]"
   ],
   [
    "C25",
    "x_vars[Widget26]"
   ],
   [
    "C26",
    "x_vars[Widget27]"
   ],
   [
    "C27",
    "x_vars[Widget28]"
   ],
   [
    "C28",
    "x_vars[Widget29]"
   ],
   [
    "C29",
    "x_vars[Widget30]"
   ],
   [
    "C30",
    "x_vars[Widget31]"
   ],
   [
    "C31",
    "x_vars[Widget32]"
   ],
   [
    "C32",
    "x_vars[Widget33]"
   ],
   [
    "C33",
    "x_vars[Widget34]"
   ],
   [
    "C34",
    "x_vars[Widget35]"
   ],
   [
    "C35",
    "x_vars[Widget36]"
   ],
   [
    "C36",
    "x_vars[Widget37]"
   ],
   [
    "C37",
    "x_vars[Widget38]"
   ],
   [
    "C38",
    "x_vars[Widget39]"
   ],
   [
    "C39",
    "x_vars[Widget40]"
   ],
   [
    "C40",
    "x_vars[Widget41]"
   ],
   [
    "C41",
    "x_vars[Widget42]"
   ],
   [
    "C42",
    "x_vars[Widget43]"
   ],
   [
    "C43",
    "x_vars[Widget44]"
   ],
   [
    "C44",
    "x_vars[Widget45]"
   ],
   [
    "C45",
    "x_vars[Widget46]"
   ],
   [
    "C46",
    "x_vars[Widget47]"
   ],
   [
    "C47",
    "x_vars[Widget48]"
   ],
   [
    "C48",
    "x_vars[Widget49]"
   ],
   [
    "C49",
    "x_vars[Widget50]"
   ],
   [
    "C50",
    "x_vars[Widget51]"
   ],
   [
    "C51",
    "x_vars[Widget52]"
   ],
   [
    "C52",
    "x_vars[Widget53]"
   ],
   [
    "C53",
    "x_vars[Widget54]"
   ],
   [
    "C54",
    "x_vars[Widget55]"
   ],
   [
    "C55",
    "x_vars[Widget56]"
   ],
   [
    "C56",
    "x_vars[Widget57]"
   ],
   [
    "C57",
    "x_vars[Widget58]"
   ],
   [
    "C58",
    "x_vars[Widget59]"
   ],
   [
    "C59",
    "x_vars[Widget60]"
   ],
   [
    "C60",
    "x_vars[Widget61]"
   ],
   [
    "C61",
    "x_vars[Widget62]"
   ],
   [
    "C62",
    "x_vars[Widget63]"
   ],
   [
    "C63",
    "x_vars[Widget64]"
   ],
   [
    "C64",
    "x_vars[Widget65]"
   ],
   [
    "C65",
    "x_vars[Widget66]"
   ],
   [
    "C66",
    "x_vars[Widget67]"
   ],
   [
    "C67",
    "x_vars[Widget68]"
   ],
   [
    "C68",
    "x_vars[Widget69]"
   ],
   [
    "C69",
    "x_vars[Widget70]"
   ],
   [
    "C70",
    "x_vars[Widget71]"
   ],
   [
    "C71",
    "x_vars[Widget72]"
   ],
   [
    "C72",
    "x_vars[Widget73]"
   ],
   [
    "C73",
    "x_vars[Widget74]"
   ],
   [
    "C74",
    "x_vars[Widget75]"
   ],
   [
    "C75",
    "x_vars[Widget76]"
   ],
   [
    "C76",
    "x_vars[Widget77]"
   ],
   [
    "C77",
    "x_vars[Widget78]"
   ],
   [
    "C78",
    "x_vars[Widget79]"
   ],
   [
    "C79",
    "x_vars[Widget80]"
   ],
   [
    "C80",
    "x_vars[Widget81]"
   ],
   [
    "C81",
    "x_vars[Widget82]"
   ],
   [
    "C82",
    "x_vars[Widget83]"
   ],
   [
    "C83",
    "x_vars[Widget84]"
   ],
   [
    "C84",
    "x_vars[Widget85]"
   ],
   [
    "C85",
    "x_vars[Widget86]"
   ],
   [
    "C86",
    "x_vars[Widget87]"
   ],
   [
    "C87",
    "x_vars[Widget88]"
   ],
   [
    "C88",
    "x_vars[Widget89]"
   ],
   [
    "C89",
    "x_vars[Widget90]"
   ],
   [
    "C90",
    "x_vars[Widget91]"
   ],
   [
    "C91",
    "x_vars[Widget92]"
   ],
   [
    "C92",
    "x_vars[Widget93]"
   ],
   [
    "C93",
    "x_vars[Widget94]"
   ],
   [
    "C94",
    "x_vars[Widget95]"
   ],
   [
    "C95",
    "x_vars[Widget96]"
   ],
   [
    "C96",
    "x_vars[Widget97]"
   ],
   [
    "C97",
    "x_vars[Widget98]"
   ],
   [
    "C98",
    "x_vars[Widget99]"
   ],
   [
    "C99",
    "x_vars[Widget100]"
   ],
   [
    "C100",
    "x_vars[Widget101]"
   ],
   [
    "C101",
    "x_vars[Widget102]"
   ],
   [
    "C102",
    "x_vars[Widget103]"
   ],
   [
    "C103",
    "x_vars[Widget104]"
   ],
   [
    "C104",
    "x_vars[Widget105]"
   ],
   [
    "C105",
    "x_vars[Widget106]"
   ],
   [
    "C106",
    "x_vars[Widget107]"
   ],
   [
    "C107",
    "x_vars[Widget108]"
   ],
   [
    "C108",
    "x_vars[Widget109]"
   ],
   [
    "C109",
    "x_vars[Widget110]"
   ],
   [
    "C110",
    "x_vars[Widget111]"
   ],
   [
    "C111",
    "x_vars[Widget112]"
   ],
   [
    "C112",
    "x_vars[Widget113]"
   ],
   [
    "C113",
    "x_vars[Widget114]"
   ],
   [
    "C114",
    "x_vars[Widget115]"
   ],
   [
    "C115",
    "x_vars[Widget116]"
   ],
   [
    "C116",
    "x_vars[Widget117]"
   ],
   [
    "C117",
    "x_vars[Widget118]"
   ],
   [
    "C118",
    "x_vars[Widget119]"
   ],
   [
    "C119",
    "x_vars[Widget120]"
   ],
   [
    "C120",
    "x_vars[Widget121]"
   ],
   [
    "C121",
    "x_vars[Widget122]"
   ],
   [
    "C122",
    "x_vars[Widget123]"
   ],
   [
    "C123",
    "x_vars[Widget124]"
   ],
   [
    "C124",
    "x_vars[Widget125]"
   ],
   [
    "C125",
    "x_vars[Widget126]"
   ],
   [
    "C126",
    "x_vars[Widget127]"
   ],
   [
    "C127",
    "x_vars[Widget128]"
   ],
   [
    "C128",
    "x_vars[Widget129]"
   ],
   [
    "C129",
    "x_vars[Widget130]"
   ],
   [
    "C130",
    "x_vars[Widget131]"
   ],
   [
    "C131",
    "x_vars[Widget132]"
   ],
   [
    "C132",
    "x_vars[Widget133]"
   ],
   [
    "C133",
    "x_vars[Widget134]"
   ],
   [
    "C134",
    "x_vars[Widget135]"
   ],
   [
    "C135",
    "x_vars[Widget136]"
   ],
   [
    "C136",
    "x_vars[Widget137]"
   ],
   [
    "C137",
    "x_vars[Widget138]"
   ],
   [
    "C138",
    "x_vars[Widget139]"
   ],
   [
    "C139",
    "x_vars[Widget140]"
   ],
   [
    "C140",
    "x_vars[Widget141]"
   ],
   [
    "s",
    "s"
   ],
   [
    "d",
    "d"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[Widget1]",
   "x[2]": "x_vars[Widget2]",
   "x[3]": "x_vars[Widget3]",
   "x[4]": "x_vars[Widget4]",
   "x[5]": "x_vars[Widget5]",
   "x[6]": "x_vars[Widget6]",
   "x[7]": "x_vars[Widget7]",
   "x[8]": "x_vars[Widget8]",
   "x[9]": "x_vars[Widget9]",
   "x[10]": "x_vars[Widget10]",
   "x[11]": "x_vars[Widget11]",
   "x[12]": "x_vars[Widget12]",
   "x[13]": "x_vars[Widget13]",
   "x[14]": "x_vars[Widget14]",
   "x[15]": "x_vars[Widget15]",
   "x[16]": "x_vars[Widget16]",
   "x[17]": "x_vars[Widget17]",
   "x[18]": "x_vars[Widget18]",
   "x[19]": "x_vars[Widget19]",
   "x[20]": "x_vars[Widget20]",
   "x[21]": "x_vars[Widget21]",
   "x[22]": "x_vars[Widget22]",
   "x[23]": "x_vars[Widget23]",
   "x[24]": "x_vars[Widget24]",
   "x[25]": "x_vars[Widget25]",
   "x[26]": "x_vars[Widget26]",
   "x[27]": "x_vars[Widget27]",
   "x[28]": "x_vars[Widget28]",
   "x[29]": "x_vars[Widget29]",
   "x[30]": "x_vars[Widget30]",
   "x[31]": "x_vars[Widget31]",
   "x[32]": "x_vars[Widget32]",
   "x[33]": "x_vars[Widget33]",
   "x[34]": "x_vars[Widget34]",
   "x[35]": "x_vars[Widget35]",
   "x[36]": "x_vars[Widget36]",
   "x[37]": "x_vars[Widget37]",
   "x[38]": "x_vars[Widget38]",
   "x[39]": "x_vars[Widget39]",
   "x[40]": "x_vars[Widget40]",
   "x[41]": "x_vars[Widget41]",
   "x[42]": "x_vars[Widget42]",
   "x[43]": "x_vars[Widget43]",
   "x[44]": "x_vars[Widget44]",
   "x[45]": "x_vars[Widget45]",
   "x[46]": "x_vars[Widget46]",
   "x[47]": "x_vars[Widget47]",
   "x[48]": "x_vars[Widget48]",
   "x[49]": "x_vars[Widget49]",
   "x[50]": "x_vars[Widget50]",
   "x[51]": "x_vars[Widget51]",
   "x[52]": "x_vars[Widget52]",
   "x[53]": "x_vars[Widget53]",
   "x[54]": "x_vars[Widget54]",
   "x[55]": "x_vars[Widget55]",
   "x[56]": "x_vars[Widget56]",
   "x[57]": "x_vars[Widget57]",
   "x[58]": "x_vars[Widget58]",
   "x[59]": "x_vars[Widget59]",
   "x[60]": "x_vars[Widget60]",
   "x[61]": "x_vars[Widget61]",
   "x[62]": "x_vars[Widget62]",
   "x[63]": "x_vars[Widget63]",
   "x[64]": "x_vars[Widget64]",
   "x[65]": "x_vars[Widget65]",
   "x[66]": "x_vars[Widget66]",
   "x[67]": "x_vars[Widget67]",
   "x[68]": "x_vars[Widget68]",
   "x[69]": "x_vars[Widget69]",
   "x[70]": "x_vars[Widget70]",
   "x[71]": "x_vars[Widget71]",
   "x[72]": "x_vars[Widget72]",
   "x[73]": "x_vars[Widget73]",
   "x[74]": "x_vars[Widget74]",
   "x[75]": "x_vars[Widget75]",
   "x[76]": "x_vars[Widget76]",
   "x[77]": "x_vars[Widget77]",
   "x[78]": "x_vars[Widget78]",
   "x[79]": "x_vars[Widget79]",
   "x[80]": "x_vars[Widget80]",
   "x[81]": "x_vars[Widget81]",
   "x[82]": "x_vars[Widget82]",
   "x[83]": "x_vars[Widget83]",
   "x[84]": "x_vars[Widget84]",
   "x[85]": "x_vars[Widget85]",
   "x[86]": "x_vars[Widget86]",
   "x[87]": "x_vars[Widget87]",
   "x[88]": "x_vars[Widget88]",
   "x[89]": "x_vars[Widget89]",
   "x[90]": "x_vars[Widget90]",
   "x[91]": "x_vars[Widget91]",
   "x[92]": "x_vars[Widget92]",
   "x[93]": "x_vars[Widget93]",
   "x[94]": "x_vars[Widget94]",
   "x[95]": "x_vars[Widget95]",
   "x[96]": "x_vars[Widget96]",
   "x[97]": "x_vars[Widget97]",
   "x[98]": "x_vars[Widget98]",
   "x[99]": "x_vars[Widget99]",
   "x[100]": "x_vars[Widget100]",
   "x[101]": "x_vars[Widget101]",
   "x[102]": "x_vars[Widget102]",
   "x[103]": "x_vars[Widget103]",
   "x[104]": "x_vars[Widget104]",
   "x[105]": "x_vars[Widget105]",
   "x[106]": "x_vars[Widget106]",
   "x[107]": "x_vars[Widget107]",
   "x[108]": "x_vars[Widget108]",
   "x[109]": "x_vars[Widget109]",
   "x[110]": "x_vars[Widget110]",
   "x[111]": "x_vars[Widget111]",
   "x[112]": "x_vars[Widget112]",
   "x[113]": "x_vars[Widget113]",
   "x[114]": "x_vars[Widget114]",
   "x[115]": "x_vars[Widget115]",
   "x[116]": "x_vars[Widget116]",
   "x[117]": "x_vars[Widget117]",
   "x[118]": "x_vars[Widget118]",
   "x[119]": "x_vars[Widget119]",
   "x[120]": "x_vars[Widget120]",
   "x[121]": "x_vars[Widget121]",
   "x[122]": "x_vars[Widget122]",
   "x[123]": "x_vars[Widget123]",
   "x[124]": "x_vars[Widget124]",
   "x[125]": "x_vars[Widget125]",
   "x[126]": "x_vars[Widget126]",
   "x[127]": "x_vars[Widget127]",
   "x[128]": "x_vars[Widget128]",
   "x[129]": "x_vars[Widget129]",
   "x[130]": "x_vars[Widget130]",
   "x[131]": "x_vars[Widget131]",
   "x[132]": "x_vars[Widget132]",
   "x[133]": "x_vars[Widget133]",
   "x[134]": "x_vars[Widget134]",
   "x[135]": "x_vars[Widget135]",
   "x[136]": "x_vars[Widget136]",
   "x[137]": "x_vars[Widget137]",
   "x[138]": "x_vars[Widget138]",
   "x[139]": "x_vars[Widget139]",
   "x[140]": "x_vars[Widget140]",
   "x[141]": "x_vars[Widget141]",
   "s": "s"
  },
  "mapping_basis": "frozen_automatic_name_axis_projection"
 },
 "OR-096": {
  "source_code_sha256": "fd1532b5d560ea11bcb2140e7b416ef0f9819d6a4b9fb55c40476f1bbfdac53d",
  "original_solution_sha256": "071e24647a7f09e4c073d64fb8dcc91978637bc82adf1fa0bda03fb051f444ac",
  "bindings": [
   [
    "C0",
    "x_vars[I,N01,White]"
   ],
   [
    "C1",
    "x_vars[I,N01,NonWhite]"
   ],
   [
    "C2",
    "x_vars[I,N02,White]"
   ],
   [
    "C3",
    "x_vars[I,N02,NonWhite]"
   ],
   [
    "C4",
    "x_vars[I,N03,White]"
   ],
   [
    "C5",
    "x_vars[I,N03,NonWhite]"
   ],
   [
    "C6",
    "x_vars[I,N04,White]"
   ],
   [
    "C7",
    "x_vars[I,N04,NonWhite]"
   ],
   [
    "C8",
    "x_vars[I,N05,White]"
   ],
   [
    "C9",
    "x_vars[I,N05,NonWhite]"
   ],
   [
    "C10",
    "x_vars[I,N06,White]"
   ],
   [
    "C11",
    "x_vars[I,N06,NonWhite]"
   ],
   [
    "C12",
    "x_vars[I,N07,White]"
   ],
   [
    "C13",
    "x_vars[I,N07,NonWhite]"
   ],
   [
    "C14",
    "x_vars[I,N08,White]"
   ],
   [
    "C15",
    "x_vars[I,N08,NonWhite]"
   ],
   [
    "C16",
    "x_vars[I,N09,White]"
   ],
   [
    "C17",
    "x_vars[I,N09,NonWhite]"
   ],
   [
    "C18",
    "x_vars[I,N10,White]"
   ],
   [
    "C19",
    "x_vars[I,N10,NonWhite]"
   ],
   [
    "C20",
    "x_vars[I,N11,White]"
   ],
   [
    "C21",
    "x_vars[I,N11,NonWhite]"
   ],
   [
    "C22",
    "x_vars[I,N12,White]"
   ],
   [
    "C23",
    "x_vars[I,N12,NonWhite]"
   ],
   [
    "C24",
    "x_vars[I,N13,White]"
   ],
   [
    "C25",
    "x_vars[I,N13,NonWhite]"
   ],
   [
    "C26",
    "x_vars[I,N14,White]"
   ],
   [
    "C27",
    "x_vars[I,N14,NonWhite]"
   ],
   [
    "C28",
    "x_vars[I,N15,White]"
   ],
   [
    "C29",
    "x_vars[I,N15,NonWhite]"
   ],
   [
    "C30",
    "x_vars[I,N16,White]"
   ],
   [
    "C31",
    "x_vars[I,N16,NonWhite]"
   ],
   [
    "C32",
    "x_vars[I,N17,White]"
   ],
   [
    "C33",
    "x_vars[I,N17,NonWhite]"
   ],
   [
    "C34",
    "x_vars[I,N18,White]"
   ],
   [
    "C35",
    "x_vars[I,N18,NonWhite]"
   ],
   [
    "C36",
    "x_vars[I,N19,White]"
   ],
   [
    "C37",
    "x_vars[I,N19,NonWhite]"
   ],
   [
    "C38",
    "x_vars[I,N20,White]"
   ],
   [
    "C39",
    "x_vars[I,N20,NonWhite]"
   ],
   [
    "C40",
    "x_vars[I,N21,White]"
   ],
   [
    "C41",
    "x_vars[I,N21,NonWhite]"
   ],
   [
    "C42",
    "x_vars[I,N22,White]"
   ],
   [
    "C43",
    "x_vars[I,N22,NonWhite]"
   ],
   [
    "C44",
    "x_vars[I,N23,White]"
   ],
   [
    "C45",
    "x_vars[I,N23,NonWhite]"
   ],
   [
    "C46",
    "x_vars[I,N24,White]"
   ],
   [
    "C47",
    "x_vars[I,N24,NonWhite]"
   ],
   [
    "C48",
    "x_vars[I,N25,White]"
   ],
   [
    "C49",
    "x_vars[I,N25,NonWhite]"
   ],
   [
    "C50",
    "x_vars[I,N26,White]"
   ],
   [
    "C51",
    "x_vars[I,N26,NonWhite]"
   ],
   [
    "C52",
    "x_vars[I,N27,White]"
   ],
   [
    "C53",
    "x_vars[I,N27,NonWhite]"
   ],
   [
    "C54",
    "x_vars[I,N28,White]"
   ],
   [
    "C55",
    "x_vars[I,N28,NonWhite]"
   ],
   [
    "C56",
    "x_vars[I,N29,White]"
   ],
   [
    "C57",
    "x_vars[I,N29,NonWhite]"
   ],
   [
    "C58",
    "x_vars[I,N30,White]"
   ],
   [
    "C59",
    "x_vars[I,N30,NonWhite]"
   ],
   [
    "C60",
    "x_vars[I,N31,White]"
   ],
   [
    "C61",
    "x_vars[I,N31,NonWhite]"
   ],
   [
    "C62",
    "x_vars[II,N01,White]"
   ],
   [
    "C63",
    "x_vars[II,N01,NonWhite]"
   ],
   [
    "C64",
    "x_vars[II,N02,White]"
   ],
   [
    "C65",
    "x_vars[II,N02,NonWhite]"
   ],
   [
    "C66",
    "x_vars[II,N03,White]"
   ],
   [
    "C67",
    "x_vars[II,N03,NonWhite]"
   ],
   [
    "C68",
    "x_vars[II,N04,White]"
   ],
   [
    "C69",
    "x_vars[II,N04,NonWhite]"
   ],
   [
    "C70",
    "x_vars[II,N05,White]"
   ],
   [
    "C71",
    "x_vars[II,N05,NonWhite]"
   ],
   [
    "C72",
    "x_vars[II,N06,White]"
   ],
   [
    "C73",
    "x_vars[II,N06,NonWhite]"
   ],
   [
    "C74",
    "x_vars[II,N07,White]"
   ],
   [
    "C75",
    "x_vars[II,N07,NonWhite]"
   ],
   [
    "C76",
    "x_vars[II,N08,White]"
   ],
   [
    "C77",
    "x_vars[II,N08,NonWhite]"
   ],
   [
    "C78",
    "x_vars[II,N09,White]"
   ],
   [
    "C79",
    "x_vars[II,N09,NonWhite]"
   ],
   [
    "C80",
    "x_vars[II,N10,White]"
   ],
   [
    "C81",
    "x_vars[II,N10,NonWhite]"
   ],
   [
    "C82",
    "x_vars[II,N11,White]"
   ],
   [
    "C83",
    "x_vars[II,N11,NonWhite]"
   ],
   [
    "C84",
    "x_vars[II,N12,White]"
   ],
   [
    "C85",
    "x_vars[II,N12,NonWhite]"
   ],
   [
    "C86",
    "x_vars[II,N13,White]"
   ],
   [
    "C87",
    "x_vars[II,N13,NonWhite]"
   ],
   [
    "C88",
    "x_vars[II,N14,White]"
   ],
   [
    "C89",
    "x_vars[II,N14,NonWhite]"
   ],
   [
    "C90",
    "x_vars[II,N15,White]"
   ],
   [
    "C91",
    "x_vars[II,N15,NonWhite]"
   ],
   [
    "C92",
    "x_vars[II,N16,White]"
   ],
   [
    "C93",
    "x_vars[II,N16,NonWhite]"
   ],
   [
    "C94",
    "x_vars[II,N17,White]"
   ],
   [
    "C95",
    "x_vars[II,N17,NonWhite]"
   ],
   [
    "C96",
    "x_vars[II,N18,White]"
   ],
   [
    "C97",
    "x_vars[II,N18,NonWhite]"
   ],
   [
    "C98",
    "x_vars[II,N19,White]"
   ],
   [
    "C99",
    "x_vars[II,N19,NonWhite]"
   ],
   [
    "C100",
    "x_vars[II,N20,White]"
   ],
   [
    "C101",
    "x_vars[II,N20,NonWhite]"
   ],
   [
    "C102",
    "x_vars[II,N21,White]"
   ],
   [
    "C103",
    "x_vars[II,N21,NonWhite]"
   ],
   [
    "C104",
    "x_vars[II,N22,White]"
   ],
   [
    "C105",
    "x_vars[II,N22,NonWhite]"
   ],
   [
    "C106",
    "x_vars[II,N23,White]"
   ],
   [
    "C107",
    "x_vars[II,N23,NonWhite]"
   ],
   [
    "C108",
    "x_vars[II,N24,White]"
   ],
   [
    "C109",
    "x_vars[II,N24,NonWhite]"
   ],
   [
    "C110",
    "x_vars[II,N25,White]"
   ],
   [
    "C111",
    "x_vars[II,N25,NonWhite]"
   ],
   [
    "C112",
    "x_vars[II,N26,White]"
   ],
   [
    "C113",
    "x_vars[II,N26,NonWhite]"
   ],
   [
    "C114",
    "x_vars[II,N27,White]"
   ],
   [
    "C115",
    "x_vars[II,N27,NonWhite]"
   ],
   [
    "C116",
    "x_vars[II,N28,White]"
   ],
   [
    "C117",
    "x_vars[II,N28,NonWhite]"
   ],
   [
    "C118",
    "x_vars[II,N29,White]"
   ],
   [
    "C119",
    "x_vars[II,N29,NonWhite]"
   ],
   [
    "C120",
    "x_vars[II,N30,White]"
   ],
   [
    "C121",
    "x_vars[II,N30,NonWhite]"
   ],
   [
    "C122",
    "x_vars[II,N31,White]"
   ],
   [
    "C123",
    "x_vars[II,N31,NonWhite]"
   ]
  ],
  "primary_mapping": {
   "x_w_I_N01": "x_vars[I,N01,White]",
   "x_n_I_N01": "x_vars[I,N01,NonWhite]",
   "x_w_I_N02": "x_vars[I,N02,White]",
   "x_n_I_N02": "x_vars[I,N02,NonWhite]",
   "x_w_I_N03": "x_vars[I,N03,White]",
   "x_n_I_N03": "x_vars[I,N03,NonWhite]",
   "x_w_I_N04": "x_vars[I,N04,White]",
   "x_n_I_N04": "x_vars[I,N04,NonWhite]",
   "x_w_I_N05": "x_vars[I,N05,White]",
   "x_n_I_N05": "x_vars[I,N05,NonWhite]",
   "x_w_I_N06": "x_vars[I,N06,White]",
   "x_n_I_N06": "x_vars[I,N06,NonWhite]",
   "x_w_I_N07": "x_vars[I,N07,White]",
   "x_n_I_N07": "x_vars[I,N07,NonWhite]",
   "x_w_I_N08": "x_vars[I,N08,White]",
   "x_n_I_N08": "x_vars[I,N08,NonWhite]",
   "x_w_I_N09": "x_vars[I,N09,White]",
   "x_n_I_N09": "x_vars[I,N09,NonWhite]",
   "x_w_I_N10": "x_vars[I,N10,White]",
   "x_n_I_N10": "x_vars[I,N10,NonWhite]",
   "x_w_I_N11": "x_vars[I,N11,White]",
   "x_n_I_N11": "x_vars[I,N11,NonWhite]",
   "x_w_I_N12": "x_vars[I,N12,White]",
   "x_n_I_N12": "x_vars[I,N12,NonWhite]",
   "x_w_I_N13": "x_vars[I,N13,White]",
   "x_n_I_N13": "x_vars[I,N13,NonWhite]",
   "x_w_I_N14": "x_vars[I,N14,White]",
   "x_n_I_N14": "x_vars[I,N14,NonWhite]",
   "x_w_I_N15": "x_vars[I,N15,White]",
   "x_n_I_N15": "x_vars[I,N15,NonWhite]",
   "x_w_I_N16": "x_vars[I,N16,White]",
   "x_n_I_N16": "x_vars[I,N16,NonWhite]",
   "x_w_I_N17": "x_vars[I,N17,White]",
   "x_n_I_N17": "x_vars[I,N17,NonWhite]",
   "x_w_I_N18": "x_vars[I,N18,White]",
   "x_n_I_N18": "x_vars[I,N18,NonWhite]",
   "x_w_I_N19": "x_vars[I,N19,White]",
   "x_n_I_N19": "x_vars[I,N19,NonWhite]",
   "x_w_I_N20": "x_vars[I,N20,White]",
   "x_n_I_N20": "x_vars[I,N20,NonWhite]",
   "x_w_I_N21": "x_vars[I,N21,White]",
   "x_n_I_N21": "x_vars[I,N21,NonWhite]",
   "x_w_I_N22": "x_vars[I,N22,White]",
   "x_n_I_N22": "x_vars[I,N22,NonWhite]",
   "x_w_I_N23": "x_vars[I,N23,White]",
   "x_n_I_N23": "x_vars[I,N23,NonWhite]",
   "x_w_I_N24": "x_vars[I,N24,White]",
   "x_n_I_N24": "x_vars[I,N24,NonWhite]",
   "x_w_I_N25": "x_vars[I,N25,White]",
   "x_n_I_N25": "x_vars[I,N25,NonWhite]",
   "x_w_I_N26": "x_vars[I,N26,White]",
   "x_n_I_N26": "x_vars[I,N26,NonWhite]",
   "x_w_I_N27": "x_vars[I,N27,White]",
   "x_n_I_N27": "x_vars[I,N27,NonWhite]",
   "x_w_I_N28": "x_vars[I,N28,White]",
   "x_n_I_N28": "x_vars[I,N28,NonWhite]",
   "x_w_I_N29": "x_vars[I,N29,White]",
   "x_n_I_N29": "x_vars[I,N29,NonWhite]",
   "x_w_I_N30": "x_vars[I,N30,White]",
   "x_n_I_N30": "x_vars[I,N30,NonWhite]",
   "x_w_I_N31": "x_vars[I,N31,White]",
   "x_n_I_N31": "x_vars[I,N31,NonWhite]",
   "x_w_II_N01": "x_vars[II,N01,White]",
   "x_n_II_N01": "x_vars[II,N01,NonWhite]",
   "x_w_II_N02": "x_vars[II,N02,White]",
   "x_n_II_N02": "x_vars[II,N02,NonWhite]",
   "x_w_II_N03": "x_vars[II,N03,White]",
   "x_n_II_N03": "x_vars[II,N03,NonWhite]",
   "x_w_II_N04": "x_vars[II,N04,White]",
   "x_n_II_N04": "x_vars[II,N04,NonWhite]",
   "x_w_II_N05": "x_vars[II,N05,White]",
   "x_n_II_N05": "x_vars[II,N05,NonWhite]",
   "x_w_II_N06": "x_vars[II,N06,White]",
   "x_n_II_N06": "x_vars[II,N06,NonWhite]",
   "x_w_II_N07": "x_vars[II,N07,White]",
   "x_n_II_N07": "x_vars[II,N07,NonWhite]",
   "x_w_II_N08": "x_vars[II,N08,White]",
   "x_n_II_N08": "x_vars[II,N08,NonWhite]",
   "x_w_II_N09": "x_vars[II,N09,White]",
   "x_n_II_N09": "x_vars[II,N09,NonWhite]",
   "x_w_II_N10": "x_vars[II,N10,White]",
   "x_n_II_N10": "x_vars[II,N10,NonWhite]",
   "x_w_II_N11": "x_vars[II,N11,White]",
   "x_n_II_N11": "x_vars[II,N11,NonWhite]",
   "x_w_II_N12": "x_vars[II,N12,White]",
   "x_n_II_N12": "x_vars[II,N12,NonWhite]",
   "x_w_II_N13": "x_vars[II,N13,White]",
   "x_n_II_N13": "x_vars[II,N13,NonWhite]",
   "x_w_II_N14": "x_vars[II,N14,White]",
   "x_n_II_N14": "x_vars[II,N14,NonWhite]",
   "x_w_II_N15": "x_vars[II,N15,White]",
   "x_n_II_N15": "x_vars[II,N15,NonWhite]",
   "x_w_II_N16": "x_vars[II,N16,White]",
   "x_n_II_N16": "x_vars[II,N16,NonWhite]",
   "x_w_II_N17": "x_vars[II,N17,White]",
   "x_n_II_N17": "x_vars[II,N17,NonWhite]",
   "x_w_II_N18": "x_vars[II,N18,White]",
   "x_n_II_N18": "x_vars[II,N18,NonWhite]",
   "x_w_II_N19": "x_vars[II,N19,White]",
   "x_n_II_N19": "x_vars[II,N19,NonWhite]",
   "x_w_II_N20": "x_vars[II,N20,White]",
   "x_n_II_N20": "x_vars[II,N20,NonWhite]",
   "x_w_II_N21": "x_vars[II,N21,White]",
   "x_n_II_N21": "x_vars[II,N21,NonWhite]",
   "x_w_II_N22": "x_vars[II,N22,White]",
   "x_n_II_N22": "x_vars[II,N22,NonWhite]",
   "x_w_II_N23": "x_vars[II,N23,White]",
   "x_n_II_N23": "x_vars[II,N23,NonWhite]",
   "x_w_II_N24": "x_vars[II,N24,White]",
   "x_n_II_N24": "x_vars[II,N24,NonWhite]",
   "x_w_II_N25": "x_vars[II,N25,White]",
   "x_n_II_N25": "x_vars[II,N25,NonWhite]",
   "x_w_II_N26": "x_vars[II,N26,White]",
   "x_n_II_N26": "x_vars[II,N26,NonWhite]",
   "x_w_II_N27": "x_vars[II,N27,White]",
   "x_n_II_N27": "x_vars[II,N27,NonWhite]",
   "x_w_II_N28": "x_vars[II,N28,White]",
   "x_n_II_N28": "x_vars[II,N28,NonWhite]",
   "x_w_II_N29": "x_vars[II,N29,White]",
   "x_n_II_N29": "x_vars[II,N29,NonWhite]",
   "x_w_II_N30": "x_vars[II,N30,White]",
   "x_n_II_N30": "x_vars[II,N30,NonWhite]",
   "x_w_II_N31": "x_vars[II,N31,White]",
   "x_n_II_N31": "x_vars[II,N31,NonWhite]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-097": {
  "source_code_sha256": "a7321fdc44ae8d836544aaba0f07195337023c9ccf67b36d7d3089e28a1fe966",
  "original_solution_sha256": "57685888572686b98e1ac16e74ac60d1bd481d5d715e39ce7c16ced5a9ed2e87",
  "bindings": [
   [
    "C0",
    "x_vars[Opt_1]"
   ],
   [
    "C1",
    "x_vars[Opt_2]"
   ],
   [
    "C2",
    "x_vars[Opt_3]"
   ],
   [
    "C3",
    "x_vars[Opt_4]"
   ],
   [
    "C4",
    "x_vars[Opt_5]"
   ],
   [
    "C5",
    "x_vars[Opt_6]"
   ],
   [
    "C6",
    "x_vars[Opt_7]"
   ],
   [
    "C7",
    "x_vars[Opt_8]"
   ],
   [
    "C8",
    "x_vars[Opt_9]"
   ],
   [
    "C9",
    "x_vars[Opt_10]"
   ],
   [
    "C10",
    "x_vars[Opt_11]"
   ],
   [
    "C11",
    "x_vars[Opt_12]"
   ],
   [
    "C12",
    "x_vars[Opt_13]"
   ],
   [
    "C13",
    "x_vars[Opt_14]"
   ],
   [
    "C14",
    "x_vars[Opt_15]"
   ],
   [
    "C15",
    "x_vars[Opt_16]"
   ],
   [
    "C16",
    "x_vars[Opt_17]"
   ],
   [
    "C17",
    "x_vars[Opt_18]"
   ],
   [
    "C18",
    "x_vars[Opt_19]"
   ],
   [
    "C19",
    "x_vars[Opt_20]"
   ],
   [
    "C20",
    "x_vars[Opt_21]"
   ],
   [
    "C21",
    "x_vars[Opt_22]"
   ],
   [
    "C22",
    "x_vars[Opt_23]"
   ],
   [
    "C23",
    "x_vars[Opt_24]"
   ],
   [
    "C24",
    "x_vars[Opt_25]"
   ],
   [
    "C25",
    "x_vars[Opt_26]"
   ],
   [
    "C26",
    "x_vars[Opt_27]"
   ],
   [
    "C27",
    "x_vars[Opt_28]"
   ],
   [
    "C28",
    "x_vars[Opt_29]"
   ],
   [
    "C29",
    "x_vars[Opt_30]"
   ],
   [
    "C30",
    "x_vars[Opt_31]"
   ],
   [
    "C31",
    "x_vars[Opt_32]"
   ],
   [
    "C32",
    "x_vars[Opt_33]"
   ],
   [
    "C33",
    "x_vars[Opt_34]"
   ],
   [
    "C34",
    "x_vars[Opt_35]"
   ],
   [
    "C35",
    "x_vars[Opt_36]"
   ],
   [
    "C36",
    "x_vars[Opt_37]"
   ],
   [
    "C37",
    "x_vars[Opt_38]"
   ],
   [
    "C38",
    "x_vars[Opt_39]"
   ],
   [
    "C39",
    "x_vars[Opt_40]"
   ],
   [
    "C40",
    "x_vars[Opt_41]"
   ],
   [
    "C41",
    "x_vars[Opt_42]"
   ],
   [
    "C42",
    "x_vars[Opt_43]"
   ],
   [
    "C43",
    "x_vars[Opt_44]"
   ],
   [
    "C44",
    "x_vars[Opt_45]"
   ],
   [
    "C45",
    "x_vars[Opt_46]"
   ],
   [
    "C46",
    "x_vars[Opt_47]"
   ],
   [
    "C47",
    "x_vars[Opt_48]"
   ],
   [
    "C48",
    "x_vars[Opt_49]"
   ],
   [
    "C49",
    "x_vars[Opt_50]"
   ],
   [
    "C50",
    "x_vars[Opt_51]"
   ],
   [
    "C51",
    "x_vars[Opt_52]"
   ],
   [
    "C52",
    "x_vars[Opt_53]"
   ],
   [
    "C53",
    "x_vars[Opt_54]"
   ],
   [
    "C54",
    "x_vars[Opt_55]"
   ],
   [
    "C55",
    "x_vars[Opt_56]"
   ],
   [
    "C56",
    "x_vars[Opt_57]"
   ],
   [
    "C57",
    "x_vars[Opt_58]"
   ],
   [
    "C58",
    "x_vars[Opt_59]"
   ],
   [
    "C59",
    "x_vars[Opt_60]"
   ],
   [
    "C60",
    "x_vars[Opt_61]"
   ],
   [
    "C61",
    "x_vars[Opt_62]"
   ],
   [
    "C62",
    "x_vars[Opt_63]"
   ],
   [
    "C63",
    "x_vars[Opt_64]"
   ],
   [
    "C64",
    "x_vars[Opt_65]"
   ],
   [
    "C65",
    "x_vars[Opt_66]"
   ],
   [
    "C66",
    "x_vars[Opt_67]"
   ],
   [
    "C67",
    "x_vars[Opt_68]"
   ],
   [
    "C68",
    "x_vars[Opt_69]"
   ],
   [
    "C69",
    "x_vars[Opt_70]"
   ],
   [
    "C70",
    "x_vars[Opt_71]"
   ],
   [
    "C71",
    "x_vars[Opt_72]"
   ],
   [
    "C72",
    "x_vars[Opt_73]"
   ],
   [
    "C73",
    "x_vars[Opt_74]"
   ],
   [
    "C74",
    "x_vars[Opt_75]"
   ],
   [
    "C75",
    "x_vars[Opt_76]"
   ],
   [
    "C76",
    "x_vars[Opt_77]"
   ],
   [
    "C77",
    "x_vars[Opt_78]"
   ],
   [
    "C78",
    "x_vars[Opt_79]"
   ],
   [
    "C79",
    "x_vars[Opt_80]"
   ],
   [
    "C80",
    "x_vars[Opt_81]"
   ],
   [
    "C81",
    "x_vars[Opt_82]"
   ],
   [
    "C82",
    "x_vars[Opt_83]"
   ],
   [
    "C83",
    "x_vars[Opt_84]"
   ],
   [
    "C84",
    "x_vars[Opt_85]"
   ],
   [
    "C85",
    "x_vars[Opt_86]"
   ],
   [
    "C86",
    "x_vars[Opt_87]"
   ],
   [
    "C87",
    "x_vars[Opt_88]"
   ],
   [
    "C88",
    "x_vars[Opt_89]"
   ],
   [
    "C89",
    "x_vars[Opt_90]"
   ],
   [
    "C90",
    "x_vars[Opt_91]"
   ],
   [
    "C91",
    "x_vars[Opt_92]"
   ],
   [
    "C92",
    "x_vars[Opt_93]"
   ],
   [
    "C93",
    "x_vars[Opt_94]"
   ],
   [
    "C94",
    "x_vars[Opt_95]"
   ],
   [
    "C95",
    "x_vars[Opt_96]"
   ],
   [
    "C96",
    "x_vars[Opt_97]"
   ],
   [
    "C97",
    "x_vars[Opt_98]"
   ],
   [
    "C98",
    "x_vars[Opt_99]"
   ],
   [
    "C99",
    "x_vars[Opt_100]"
   ],
   [
    "C100",
    "x_vars[Opt_101]"
   ],
   [
    "C101",
    "x_vars[Opt_102]"
   ],
   [
    "C102",
    "x_vars[Opt_103]"
   ],
   [
    "C103",
    "x_vars[Opt_104]"
   ],
   [
    "C104",
    "x_vars[Opt_105]"
   ],
   [
    "C105",
    "x_vars[Opt_106]"
   ],
   [
    "C106",
    "x_vars[Opt_107]"
   ],
   [
    "C107",
    "x_vars[Opt_108]"
   ],
   [
    "C108",
    "x_vars[Opt_109]"
   ],
   [
    "C109",
    "x_vars[Opt_110]"
   ],
   [
    "C110",
    "x_vars[Opt_111]"
   ],
   [
    "C111",
    "x_vars[Opt_112]"
   ],
   [
    "C112",
    "x_vars[Opt_113]"
   ],
   [
    "C113",
    "x_vars[Opt_114]"
   ],
   [
    "C114",
    "x_vars[Opt_115]"
   ],
   [
    "C115",
    "x_vars[Opt_116]"
   ],
   [
    "C116",
    "x_vars[Opt_117]"
   ],
   [
    "C117",
    "x_vars[Opt_118]"
   ],
   [
    "C118",
    "x_vars[Opt_119]"
   ],
   [
    "C119",
    "x_vars[Opt_120]"
   ],
   [
    "C120",
    "y_vars[Opt_1]"
   ],
   [
    "C121",
    "y_vars[Opt_2]"
   ],
   [
    "C122",
    "y_vars[Opt_3]"
   ],
   [
    "C123",
    "y_vars[Opt_4]"
   ],
   [
    "C124",
    "y_vars[Opt_5]"
   ],
   [
    "C125",
    "y_vars[Opt_6]"
   ],
   [
    "C126",
    "y_vars[Opt_7]"
   ],
   [
    "C127",
    "y_vars[Opt_8]"
   ],
   [
    "C128",
    "y_vars[Opt_9]"
   ],
   [
    "C129",
    "y_vars[Opt_10]"
   ],
   [
    "C130",
    "y_vars[Opt_11]"
   ],
   [
    "C131",
    "y_vars[Opt_12]"
   ],
   [
    "C132",
    "y_vars[Opt_13]"
   ],
   [
    "C133",
    "y_vars[Opt_14]"
   ],
   [
    "C134",
    "y_vars[Opt_15]"
   ],
   [
    "C135",
    "y_vars[Opt_16]"
   ],
   [
    "C136",
    "y_vars[Opt_17]"
   ],
   [
    "C137",
    "y_vars[Opt_18]"
   ],
   [
    "C138",
    "y_vars[Opt_19]"
   ],
   [
    "C139",
    "y_vars[Opt_20]"
   ],
   [
    "C140",
    "y_vars[Opt_21]"
   ],
   [
    "C141",
    "y_vars[Opt_22]"
   ],
   [
    "C142",
    "y_vars[Opt_23]"
   ],
   [
    "C143",
    "y_vars[Opt_24]"
   ],
   [
    "C144",
    "y_vars[Opt_25]"
   ],
   [
    "C145",
    "y_vars[Opt_26]"
   ],
   [
    "C146",
    "y_vars[Opt_27]"
   ],
   [
    "C147",
    "y_vars[Opt_28]"
   ],
   [
    "C148",
    "y_vars[Opt_29]"
   ],
   [
    "C149",
    "y_vars[Opt_30]"
   ],
   [
    "C150",
    "y_vars[Opt_31]"
   ],
   [
    "C151",
    "y_vars[Opt_32]"
   ],
   [
    "C152",
    "y_vars[Opt_33]"
   ],
   [
    "C153",
    "y_vars[Opt_34]"
   ],
   [
    "C154",
    "y_vars[Opt_35]"
   ],
   [
    "C155",
    "y_vars[Opt_36]"
   ],
   [
    "C156",
    "y_vars[Opt_37]"
   ],
   [
    "C157",
    "y_vars[Opt_38]"
   ],
   [
    "C158",
    "y_vars[Opt_39]"
   ],
   [
    "C159",
    "y_vars[Opt_40]"
   ],
   [
    "C160",
    "y_vars[Opt_41]"
   ],
   [
    "C161",
    "y_vars[Opt_42]"
   ],
   [
    "C162",
    "y_vars[Opt_43]"
   ],
   [
    "C163",
    "y_vars[Opt_44]"
   ],
   [
    "C164",
    "y_vars[Opt_45]"
   ],
   [
    "C165",
    "y_vars[Opt_46]"
   ],
   [
    "C166",
    "y_vars[Opt_47]"
   ],
   [
    "C167",
    "y_vars[Opt_48]"
   ],
   [
    "C168",
    "y_vars[Opt_49]"
   ],
   [
    "C169",
    "y_vars[Opt_50]"
   ],
   [
    "C170",
    "y_vars[Opt_51]"
   ],
   [
    "C171",
    "y_vars[Opt_52]"
   ],
   [
    "C172",
    "y_vars[Opt_53]"
   ],
   [
    "C173",
    "y_vars[Opt_54]"
   ],
   [
    "C174",
    "y_vars[Opt_55]"
   ],
   [
    "C175",
    "y_vars[Opt_56]"
   ],
   [
    "C176",
    "y_vars[Opt_57]"
   ],
   [
    "C177",
    "y_vars[Opt_58]"
   ],
   [
    "C178",
    "y_vars[Opt_59]"
   ],
   [
    "C179",
    "y_vars[Opt_60]"
   ],
   [
    "C180",
    "y_vars[Opt_61]"
   ],
   [
    "C181",
    "y_vars[Opt_62]"
   ],
   [
    "C182",
    "y_vars[Opt_63]"
   ],
   [
    "C183",
    "y_vars[Opt_64]"
   ],
   [
    "C184",
    "y_vars[Opt_65]"
   ],
   [
    "C185",
    "y_vars[Opt_66]"
   ],
   [
    "C186",
    "y_vars[Opt_67]"
   ],
   [
    "C187",
    "y_vars[Opt_68]"
   ],
   [
    "C188",
    "y_vars[Opt_69]"
   ],
   [
    "C189",
    "y_vars[Opt_70]"
   ],
   [
    "C190",
    "y_vars[Opt_71]"
   ],
   [
    "C191",
    "y_vars[Opt_72]"
   ],
   [
    "C192",
    "y_vars[Opt_73]"
   ],
   [
    "C193",
    "y_vars[Opt_74]"
   ],
   [
    "C194",
    "y_vars[Opt_75]"
   ],
   [
    "C195",
    "y_vars[Opt_76]"
   ],
   [
    "C196",
    "y_vars[Opt_77]"
   ],
   [
    "C197",
    "y_vars[Opt_78]"
   ],
   [
    "C198",
    "y_vars[Opt_79]"
   ],
   [
    "C199",
    "y_vars[Opt_80]"
   ],
   [
    "C200",
    "y_vars[Opt_81]"
   ],
   [
    "C201",
    "y_vars[Opt_82]"
   ],
   [
    "C202",
    "y_vars[Opt_83]"
   ],
   [
    "C203",
    "y_vars[Opt_84]"
   ],
   [
    "C204",
    "y_vars[Opt_85]"
   ],
   [
    "C205",
    "y_vars[Opt_86]"
   ],
   [
    "C206",
    "y_vars[Opt_87]"
   ],
   [
    "C207",
    "y_vars[Opt_88]"
   ],
   [
    "C208",
    "y_vars[Opt_89]"
   ],
   [
    "C209",
    "y_vars[Opt_90]"
   ],
   [
    "C210",
    "y_vars[Opt_91]"
   ],
   [
    "C211",
    "y_vars[Opt_92]"
   ],
   [
    "C212",
    "y_vars[Opt_93]"
   ],
   [
    "C213",
    "y_vars[Opt_94]"
   ],
   [
    "C214",
    "y_vars[Opt_95]"
   ],
   [
    "C215",
    "y_vars[Opt_96]"
   ],
   [
    "C216",
    "y_vars[Opt_97]"
   ],
   [
    "C217",
    "y_vars[Opt_98]"
   ],
   [
    "C218",
    "y_vars[Opt_99]"
   ],
   [
    "C219",
    "y_vars[Opt_100]"
   ],
   [
    "C220",
    "y_vars[Opt_101]"
   ],
   [
    "C221",
    "y_vars[Opt_102]"
   ],
   [
    "C222",
    "y_vars[Opt_103]"
   ],
   [
    "C223",
    "y_vars[Opt_104]"
   ],
   [
    "C224",
    "y_vars[Opt_105]"
   ],
   [
    "C225",
    "y_vars[Opt_106]"
   ],
   [
    "C226",
    "y_vars[Opt_107]"
   ],
   [
    "C227",
    "y_vars[Opt_108]"
   ],
   [
    "C228",
    "y_vars[Opt_109]"
   ],
   [
    "C229",
    "y_vars[Opt_110]"
   ],
   [
    "C230",
    "y_vars[Opt_111]"
   ],
   [
    "C231",
    "y_vars[Opt_112]"
   ],
   [
    "C232",
    "y_vars[Opt_113]"
   ],
   [
    "C233",
    "y_vars[Opt_114]"
   ],
   [
    "C234",
    "y_vars[Opt_115]"
   ],
   [
    "C235",
    "y_vars[Opt_116]"
   ],
   [
    "C236",
    "y_vars[Opt_117]"
   ],
   [
    "C237",
    "y_vars[Opt_118]"
   ],
   [
    "C238",
    "y_vars[Opt_119]"
   ],
   [
    "C239",
    "y_vars[Opt_120]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[Opt_1]",
   "x[2]": "x_vars[Opt_2]",
   "x[3]": "x_vars[Opt_3]",
   "x[4]": "x_vars[Opt_4]",
   "x[5]": "x_vars[Opt_5]",
   "x[6]": "x_vars[Opt_6]",
   "x[7]": "x_vars[Opt_7]",
   "x[8]": "x_vars[Opt_8]",
   "x[9]": "x_vars[Opt_9]",
   "x[10]": "x_vars[Opt_10]",
   "x[11]": "x_vars[Opt_11]",
   "x[12]": "x_vars[Opt_12]",
   "x[13]": "x_vars[Opt_13]",
   "x[14]": "x_vars[Opt_14]",
   "x[15]": "x_vars[Opt_15]",
   "x[16]": "x_vars[Opt_16]",
   "x[17]": "x_vars[Opt_17]",
   "x[18]": "x_vars[Opt_18]",
   "x[19]": "x_vars[Opt_19]",
   "x[20]": "x_vars[Opt_20]",
   "x[21]": "x_vars[Opt_21]",
   "x[22]": "x_vars[Opt_22]",
   "x[23]": "x_vars[Opt_23]",
   "x[24]": "x_vars[Opt_24]",
   "x[25]": "x_vars[Opt_25]",
   "x[26]": "x_vars[Opt_26]",
   "x[27]": "x_vars[Opt_27]",
   "x[28]": "x_vars[Opt_28]",
   "x[29]": "x_vars[Opt_29]",
   "x[30]": "x_vars[Opt_30]",
   "x[31]": "x_vars[Opt_31]",
   "x[32]": "x_vars[Opt_32]",
   "x[33]": "x_vars[Opt_33]",
   "x[34]": "x_vars[Opt_34]",
   "x[35]": "x_vars[Opt_35]",
   "x[36]": "x_vars[Opt_36]",
   "x[37]": "x_vars[Opt_37]",
   "x[38]": "x_vars[Opt_38]",
   "x[39]": "x_vars[Opt_39]",
   "x[40]": "x_vars[Opt_40]",
   "x[41]": "x_vars[Opt_41]",
   "x[42]": "x_vars[Opt_42]",
   "x[43]": "x_vars[Opt_43]",
   "x[44]": "x_vars[Opt_44]",
   "x[45]": "x_vars[Opt_45]",
   "x[46]": "x_vars[Opt_46]",
   "x[47]": "x_vars[Opt_47]",
   "x[48]": "x_vars[Opt_48]",
   "x[49]": "x_vars[Opt_49]",
   "x[50]": "x_vars[Opt_50]",
   "x[51]": "x_vars[Opt_51]",
   "x[52]": "x_vars[Opt_52]",
   "x[53]": "x_vars[Opt_53]",
   "x[54]": "x_vars[Opt_54]",
   "x[55]": "x_vars[Opt_55]",
   "x[56]": "x_vars[Opt_56]",
   "x[57]": "x_vars[Opt_57]",
   "x[58]": "x_vars[Opt_58]",
   "x[59]": "x_vars[Opt_59]",
   "x[60]": "x_vars[Opt_60]",
   "x[61]": "x_vars[Opt_61]",
   "x[62]": "x_vars[Opt_62]",
   "x[63]": "x_vars[Opt_63]",
   "x[64]": "x_vars[Opt_64]",
   "x[65]": "x_vars[Opt_65]",
   "x[66]": "x_vars[Opt_66]",
   "x[67]": "x_vars[Opt_67]",
   "x[68]": "x_vars[Opt_68]",
   "x[69]": "x_vars[Opt_69]",
   "x[70]": "x_vars[Opt_70]",
   "x[71]": "x_vars[Opt_71]",
   "x[72]": "x_vars[Opt_72]",
   "x[73]": "x_vars[Opt_73]",
   "x[74]": "x_vars[Opt_74]",
   "x[75]": "x_vars[Opt_75]",
   "x[76]": "x_vars[Opt_76]",
   "x[77]": "x_vars[Opt_77]",
   "x[78]": "x_vars[Opt_78]",
   "x[79]": "x_vars[Opt_79]",
   "x[80]": "x_vars[Opt_80]",
   "x[81]": "x_vars[Opt_81]",
   "x[82]": "x_vars[Opt_82]",
   "x[83]": "x_vars[Opt_83]",
   "x[84]": "x_vars[Opt_84]",
   "x[85]": "x_vars[Opt_85]",
   "x[86]": "x_vars[Opt_86]",
   "x[87]": "x_vars[Opt_87]",
   "x[88]": "x_vars[Opt_88]",
   "x[89]": "x_vars[Opt_89]",
   "x[90]": "x_vars[Opt_90]",
   "x[91]": "x_vars[Opt_91]",
   "x[92]": "x_vars[Opt_92]",
   "x[93]": "x_vars[Opt_93]",
   "x[94]": "x_vars[Opt_94]",
   "x[95]": "x_vars[Opt_95]",
   "x[96]": "x_vars[Opt_96]",
   "x[97]": "x_vars[Opt_97]",
   "x[98]": "x_vars[Opt_98]",
   "x[99]": "x_vars[Opt_99]",
   "x[100]": "x_vars[Opt_100]",
   "x[101]": "x_vars[Opt_101]",
   "x[102]": "x_vars[Opt_102]",
   "x[103]": "x_vars[Opt_103]",
   "x[104]": "x_vars[Opt_104]",
   "x[105]": "x_vars[Opt_105]",
   "x[106]": "x_vars[Opt_106]",
   "x[107]": "x_vars[Opt_107]",
   "x[108]": "x_vars[Opt_108]",
   "x[109]": "x_vars[Opt_109]",
   "x[110]": "x_vars[Opt_110]",
   "x[111]": "x_vars[Opt_111]",
   "x[112]": "x_vars[Opt_112]",
   "x[113]": "x_vars[Opt_113]",
   "x[114]": "x_vars[Opt_114]",
   "x[115]": "x_vars[Opt_115]",
   "x[116]": "x_vars[Opt_116]",
   "x[117]": "x_vars[Opt_117]",
   "x[118]": "x_vars[Opt_118]",
   "x[119]": "x_vars[Opt_119]",
   "x[120]": "x_vars[Opt_120]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-098": {
  "source_code_sha256": "38dc975431c131502c6c386d93d8944c799f6fd8287cb277ca46493aa0d1ca74",
  "original_solution_sha256": "3b69dc05c5ad0bd769f2ed239169ab698f91804aa1f3f977856c1325a000834f",
  "bindings": [
   [
    "C0",
    "w_vars[Carpenter]"
   ],
   [
    "C1",
    "w_vars[Electrician]"
   ],
   [
    "C2",
    "w_vars[Painter]"
   ],
   [
    "C3",
    "w_vars[Worker_004]"
   ],
   [
    "C4",
    "w_vars[Worker_005]"
   ],
   [
    "C5",
    "w_vars[Worker_006]"
   ],
   [
    "C6",
    "w_vars[Worker_007]"
   ],
   [
    "C7",
    "w_vars[Worker_008]"
   ],
   [
    "C8",
    "w_vars[Worker_009]"
   ],
   [
    "C9",
    "w_vars[Worker_010]"
   ],
   [
    "C10",
    "w_vars[Worker_011]"
   ],
   [
    "C11",
    "w_vars[Worker_012]"
   ],
   [
    "C12",
    "w_vars[Worker_013]"
   ],
   [
    "C13",
    "w_vars[Worker_014]"
   ],
   [
    "C14",
    "w_vars[Worker_015]"
   ],
   [
    "C15",
    "w_vars[Worker_016]"
   ],
   [
    "C16",
    "w_vars[Worker_017]"
   ],
   [
    "C17",
    "w_vars[Worker_018]"
   ],
   [
    "C18",
    "w_vars[Worker_019]"
   ],
   [
    "C19",
    "w_vars[Worker_020]"
   ],
   [
    "C20",
    "w_vars[Worker_021]"
   ],
   [
    "C21",
    "w_vars[Worker_022]"
   ],
   [
    "C22",
    "w_vars[Worker_023]"
   ],
   [
    "C23",
    "w_vars[Worker_024]"
   ],
   [
    "C24",
    "w_vars[Worker_025]"
   ],
   [
    "C25",
    "w_vars[Worker_026]"
   ],
   [
    "C26",
    "w_vars[Worker_027]"
   ],
   [
    "C27",
    "w_vars[Worker_028]"
   ],
   [
    "C28",
    "w_vars[Worker_029]"
   ],
   [
    "C29",
    "w_vars[Worker_030]"
   ],
   [
    "C30",
    "w_vars[Worker_031]"
   ],
   [
    "C31",
    "w_vars[Worker_032]"
   ],
   [
    "C32",
    "w_vars[Worker_033]"
   ],
   [
    "C33",
    "w_vars[Worker_034]"
   ],
   [
    "C34",
    "w_vars[Worker_035]"
   ],
   [
    "C35",
    "w_vars[Worker_036]"
   ],
   [
    "C36",
    "w_vars[Worker_037]"
   ],
   [
    "C37",
    "w_vars[Worker_038]"
   ],
   [
    "C38",
    "w_vars[Worker_039]"
   ],
   [
    "C39",
    "w_vars[Worker_040]"
   ],
   [
    "C40",
    "w_vars[Worker_041]"
   ],
   [
    "C41",
    "w_vars[Worker_042]"
   ],
   [
    "C42",
    "w_vars[Worker_043]"
   ],
   [
    "C43",
    "w_vars[Worker_044]"
   ],
   [
    "C44",
    "w_vars[Worker_045]"
   ],
   [
    "C45",
    "w_vars[Worker_046]"
   ],
   [
    "C46",
    "w_vars[Worker_047]"
   ],
   [
    "C47",
    "w_vars[Worker_048]"
   ],
   [
    "C48",
    "w_vars[Worker_049]"
   ],
   [
    "C49",
    "w_vars[Worker_050]"
   ],
   [
    "C50",
    "w_vars[Worker_051]"
   ],
   [
    "C51",
    "w_vars[Worker_052]"
   ],
   [
    "C52",
    "w_vars[Worker_053]"
   ],
   [
    "C53",
    "w_vars[Worker_054]"
   ],
   [
    "C54",
    "w_vars[Worker_055]"
   ],
   [
    "C55",
    "w_vars[Worker_056]"
   ],
   [
    "C56",
    "w_vars[Worker_057]"
   ],
   [
    "C57",
    "w_vars[Worker_058]"
   ],
   [
    "C58",
    "w_vars[Worker_059]"
   ],
   [
    "C59",
    "w_vars[Worker_060]"
   ],
   [
    "C60",
    "w_vars[Worker_061]"
   ],
   [
    "C61",
    "w_vars[Worker_062]"
   ],
   [
    "C62",
    "w_vars[Worker_063]"
   ],
   [
    "C63",
    "w_vars[Worker_064]"
   ],
   [
    "C64",
    "w_vars[Worker_065]"
   ],
   [
    "C65",
    "w_vars[Worker_066]"
   ],
   [
    "C66",
    "w_vars[Worker_067]"
   ],
   [
    "C67",
    "w_vars[Worker_068]"
   ],
   [
    "C68",
    "w_vars[Worker_069]"
   ],
   [
    "C69",
    "w_vars[Worker_070]"
   ],
   [
    "C70",
    "w_vars[Worker_071]"
   ],
   [
    "C71",
    "w_vars[Worker_072]"
   ],
   [
    "C72",
    "w_vars[Worker_073]"
   ],
   [
    "C73",
    "w_vars[Worker_074]"
   ],
   [
    "C74",
    "w_vars[Worker_075]"
   ],
   [
    "C75",
    "w_vars[Worker_076]"
   ],
   [
    "C76",
    "w_vars[Worker_077]"
   ],
   [
    "C77",
    "w_vars[Worker_078]"
   ],
   [
    "C78",
    "w_vars[Worker_079]"
   ],
   [
    "C79",
    "w_vars[Worker_080]"
   ],
   [
    "C80",
    "w_vars[Worker_081]"
   ],
   [
    "C81",
    "w_vars[Worker_082]"
   ],
   [
    "C82",
    "w_vars[Worker_083]"
   ],
   [
    "C83",
    "w_vars[Worker_084]"
   ],
   [
    "C84",
    "w_vars[Worker_085]"
   ],
   [
    "C85",
    "w_vars[Worker_086]"
   ],
   [
    "C86",
    "w_vars[Worker_087]"
   ],
   [
    "C87",
    "w_vars[Worker_088]"
   ],
   [
    "C88",
    "w_vars[Worker_089]"
   ],
   [
    "C89",
    "w_vars[Worker_090]"
   ],
   [
    "C90",
    "w_vars[Worker_091]"
   ],
   [
    "C91",
    "w_vars[Worker_092]"
   ],
   [
    "C92",
    "w_vars[Worker_093]"
   ],
   [
    "C93",
    "w_vars[Worker_094]"
   ],
   [
    "C94",
    "w_vars[Worker_095]"
   ],
   [
    "C95",
    "w_vars[Worker_096]"
   ],
   [
    "C96",
    "w_vars[Worker_097]"
   ],
   [
    "C97",
    "w_vars[Worker_098]"
   ],
   [
    "C98",
    "w_vars[Worker_099]"
   ],
   [
    "C99",
    "w_vars[Worker_100]"
   ],
   [
    "C100",
    "w_vars[Worker_101]"
   ],
   [
    "C101",
    "w_vars[Worker_102]"
   ],
   [
    "C102",
    "w_vars[Worker_103]"
   ],
   [
    "C103",
    "w_vars[Worker_104]"
   ],
   [
    "C104",
    "w_vars[Worker_105]"
   ],
   [
    "C105",
    "w_vars[Worker_106]"
   ],
   [
    "C106",
    "w_vars[Worker_107]"
   ],
   [
    "C107",
    "w_vars[Worker_108]"
   ],
   [
    "C108",
    "w_vars[Worker_109]"
   ],
   [
    "C109",
    "w_vars[Worker_110]"
   ],
   [
    "C110",
    "w_vars[Worker_111]"
   ],
   [
    "C111",
    "w_vars[Worker_112]"
   ],
   [
    "C112",
    "w_vars[Worker_113]"
   ],
   [
    "C113",
    "w_vars[Worker_114]"
   ],
   [
    "C114",
    "w_vars[Worker_115]"
   ],
   [
    "C115",
    "w_vars[Worker_116]"
   ],
   [
    "C116",
    "w_vars[Worker_117]"
   ],
   [
    "C117",
    "w_vars[Worker_118]"
   ],
   [
    "C118",
    "w_vars[Worker_119]"
   ],
   [
    "C119",
    "w_vars[Worker_120]"
   ],
   [
    "C120",
    "w_vars[Worker_121]"
   ],
   [
    "C121",
    "w_vars[Worker_122]"
   ],
   [
    "C122",
    "w_vars[Worker_123]"
   ],
   [
    "C123",
    "w_vars[Worker_124]"
   ],
   [
    "C124",
    "w_vars[Worker_125]"
   ],
   [
    "C125",
    "w_vars[Worker_126]"
   ],
   [
    "C126",
    "w_vars[Worker_127]"
   ],
   [
    "C127",
    "w_vars[Worker_128]"
   ],
   [
    "C128",
    "w_vars[Worker_129]"
   ],
   [
    "C129",
    "w_vars[Worker_130]"
   ],
   [
    "C130",
    "w_vars[Worker_131]"
   ],
   [
    "C131",
    "w_vars[Worker_132]"
   ],
   [
    "C132",
    "w_vars[Worker_133]"
   ],
   [
    "C133",
    "w_vars[Worker_134]"
   ],
   [
    "C134",
    "w_vars[Worker_135]"
   ],
   [
    "C135",
    "w_vars[Worker_136]"
   ],
   [
    "C136",
    "w_vars[Worker_137]"
   ],
   [
    "C137",
    "w_vars[Worker_138]"
   ],
   [
    "C138",
    "w_vars[Worker_139]"
   ],
   [
    "C139",
    "w_vars[Worker_140]"
   ],
   [
    "C140",
    "w_vars[Worker_141]"
   ],
   [
    "C141",
    "w_vars[Worker_142]"
   ],
   [
    "C142",
    "w_vars[Worker_143]"
   ],
   [
    "C143",
    "w_vars[Worker_144]"
   ],
   [
    "C144",
    "w_vars[Worker_145]"
   ],
   [
    "C145",
    "w_vars[Worker_146]"
   ],
   [
    "C146",
    "w_vars[Worker_147]"
   ],
   [
    "C147",
    "w_vars[Worker_148]"
   ],
   [
    "C148",
    "w_vars[Worker_149]"
   ],
   [
    "C149",
    "w_vars[Worker_150]"
   ]
  ],
  "primary_mapping": {
   "wage[0]": "w_vars[Carpenter]",
   "wage[1]": "w_vars[Electrician]",
   "wage[2]": "w_vars[Painter]",
   "wage[3]": "w_vars[Worker_004]",
   "wage[4]": "w_vars[Worker_005]",
   "wage[5]": "w_vars[Worker_006]",
   "wage[6]": "w_vars[Worker_007]",
   "wage[7]": "w_vars[Worker_008]",
   "wage[8]": "w_vars[Worker_009]",
   "wage[9]": "w_vars[Worker_010]",
   "wage[10]": "w_vars[Worker_011]",
   "wage[11]": "w_vars[Worker_012]",
   "wage[12]": "w_vars[Worker_013]",
   "wage[13]": "w_vars[Worker_014]",
   "wage[14]": "w_vars[Worker_015]",
   "wage[15]": "w_vars[Worker_016]",
   "wage[16]": "w_vars[Worker_017]",
   "wage[17]": "w_vars[Worker_018]",
   "wage[18]": "w_vars[Worker_019]",
   "wage[19]": "w_vars[Worker_020]",
   "wage[20]": "w_vars[Worker_021]",
   "wage[21]": "w_vars[Worker_022]",
   "wage[22]": "w_vars[Worker_023]",
   "wage[23]": "w_vars[Worker_024]",
   "wage[24]": "w_vars[Worker_025]",
   "wage[25]": "w_vars[Worker_026]",
   "wage[26]": "w_vars[Worker_027]",
   "wage[27]": "w_vars[Worker_028]",
   "wage[28]": "w_vars[Worker_029]",
   "wage[29]": "w_vars[Worker_030]",
   "wage[30]": "w_vars[Worker_031]",
   "wage[31]": "w_vars[Worker_032]",
   "wage[32]": "w_vars[Worker_033]",
   "wage[33]": "w_vars[Worker_034]",
   "wage[34]": "w_vars[Worker_035]",
   "wage[35]": "w_vars[Worker_036]",
   "wage[36]": "w_vars[Worker_037]",
   "wage[37]": "w_vars[Worker_038]",
   "wage[38]": "w_vars[Worker_039]",
   "wage[39]": "w_vars[Worker_040]",
   "wage[40]": "w_vars[Worker_041]",
   "wage[41]": "w_vars[Worker_042]",
   "wage[42]": "w_vars[Worker_043]",
   "wage[43]": "w_vars[Worker_044]",
   "wage[44]": "w_vars[Worker_045]",
   "wage[45]": "w_vars[Worker_046]",
   "wage[46]": "w_vars[Worker_047]",
   "wage[47]": "w_vars[Worker_048]",
   "wage[48]": "w_vars[Worker_049]",
   "wage[49]": "w_vars[Worker_050]",
   "wage[50]": "w_vars[Worker_051]",
   "wage[51]": "w_vars[Worker_052]",
   "wage[52]": "w_vars[Worker_053]",
   "wage[53]": "w_vars[Worker_054]",
   "wage[54]": "w_vars[Worker_055]",
   "wage[55]": "w_vars[Worker_056]",
   "wage[56]": "w_vars[Worker_057]",
   "wage[57]": "w_vars[Worker_058]",
   "wage[58]": "w_vars[Worker_059]",
   "wage[59]": "w_vars[Worker_060]",
   "wage[60]": "w_vars[Worker_061]",
   "wage[61]": "w_vars[Worker_062]",
   "wage[62]": "w_vars[Worker_063]",
   "wage[63]": "w_vars[Worker_064]",
   "wage[64]": "w_vars[Worker_065]",
   "wage[65]": "w_vars[Worker_066]",
   "wage[66]": "w_vars[Worker_067]",
   "wage[67]": "w_vars[Worker_068]",
   "wage[68]": "w_vars[Worker_069]",
   "wage[69]": "w_vars[Worker_070]",
   "wage[70]": "w_vars[Worker_071]",
   "wage[71]": "w_vars[Worker_072]",
   "wage[72]": "w_vars[Worker_073]",
   "wage[73]": "w_vars[Worker_074]",
   "wage[74]": "w_vars[Worker_075]",
   "wage[75]": "w_vars[Worker_076]",
   "wage[76]": "w_vars[Worker_077]",
   "wage[77]": "w_vars[Worker_078]",
   "wage[78]": "w_vars[Worker_079]",
   "wage[79]": "w_vars[Worker_080]",
   "wage[80]": "w_vars[Worker_081]",
   "wage[81]": "w_vars[Worker_082]",
   "wage[82]": "w_vars[Worker_083]",
   "wage[83]": "w_vars[Worker_084]",
   "wage[84]": "w_vars[Worker_085]",
   "wage[85]": "w_vars[Worker_086]",
   "wage[86]": "w_vars[Worker_087]",
   "wage[87]": "w_vars[Worker_088]",
   "wage[88]": "w_vars[Worker_089]",
   "wage[89]": "w_vars[Worker_090]",
   "wage[90]": "w_vars[Worker_091]",
   "wage[91]": "w_vars[Worker_092]",
   "wage[92]": "w_vars[Worker_093]",
   "wage[93]": "w_vars[Worker_094]",
   "wage[94]": "w_vars[Worker_095]",
   "wage[95]": "w_vars[Worker_096]",
   "wage[96]": "w_vars[Worker_097]",
   "wage[97]": "w_vars[Worker_098]",
   "wage[98]": "w_vars[Worker_099]",
   "wage[99]": "w_vars[Worker_100]",
   "wage[100]": "w_vars[Worker_101]",
   "wage[101]": "w_vars[Worker_102]",
   "wage[102]": "w_vars[Worker_103]",
   "wage[103]": "w_vars[Worker_104]",
   "wage[104]": "w_vars[Worker_105]",
   "wage[105]": "w_vars[Worker_106]",
   "wage[106]": "w_vars[Worker_107]",
   "wage[107]": "w_vars[Worker_108]",
   "wage[108]": "w_vars[Worker_109]",
   "wage[109]": "w_vars[Worker_110]",
   "wage[110]": "w_vars[Worker_111]",
   "wage[111]": "w_vars[Worker_112]",
   "wage[112]": "w_vars[Worker_113]",
   "wage[113]": "w_vars[Worker_114]",
   "wage[114]": "w_vars[Worker_115]",
   "wage[115]": "w_vars[Worker_116]",
   "wage[116]": "w_vars[Worker_117]",
   "wage[117]": "w_vars[Worker_118]",
   "wage[118]": "w_vars[Worker_119]",
   "wage[119]": "w_vars[Worker_120]",
   "wage[120]": "w_vars[Worker_121]",
   "wage[121]": "w_vars[Worker_122]",
   "wage[122]": "w_vars[Worker_123]",
   "wage[123]": "w_vars[Worker_124]",
   "wage[124]": "w_vars[Worker_125]",
   "wage[125]": "w_vars[Worker_126]",
   "wage[126]": "w_vars[Worker_127]",
   "wage[127]": "w_vars[Worker_128]",
   "wage[128]": "w_vars[Worker_129]",
   "wage[129]": "w_vars[Worker_130]",
   "wage[130]": "w_vars[Worker_131]",
   "wage[131]": "w_vars[Worker_132]",
   "wage[132]": "w_vars[Worker_133]",
   "wage[133]": "w_vars[Worker_134]",
   "wage[134]": "w_vars[Worker_135]",
   "wage[135]": "w_vars[Worker_136]",
   "wage[136]": "w_vars[Worker_137]",
   "wage[137]": "w_vars[Worker_138]",
   "wage[138]": "w_vars[Worker_139]",
   "wage[139]": "w_vars[Worker_140]",
   "wage[140]": "w_vars[Worker_141]",
   "wage[141]": "w_vars[Worker_142]",
   "wage[142]": "w_vars[Worker_143]",
   "wage[143]": "w_vars[Worker_144]",
   "wage[144]": "w_vars[Worker_145]",
   "wage[145]": "w_vars[Worker_146]",
   "wage[146]": "w_vars[Worker_147]",
   "wage[147]": "w_vars[Worker_148]",
   "wage[148]": "w_vars[Worker_149]",
   "wage[149]": "w_vars[Worker_150]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-099": {
  "source_code_sha256": "8e98727c740b28ee29d1701cd5688c7dddd97e4bb437717bcd71e57524381d25",
  "original_solution_sha256": "205ceda6c3138e5950628f40c255a74f6de2d6d5d0cf6111a9b1b71d2c86e21a",
  "bindings": [
   [
    "C0",
    "x_vars[1,1]"
   ],
   [
    "C1",
    "x_vars[1,2]"
   ],
   [
    "C2",
    "x_vars[1,3]"
   ],
   [
    "C3",
    "x_vars[1,4]"
   ],
   [
    "C4",
    "x_vars[1,5]"
   ],
   [
    "C5",
    "x_vars[1,6]"
   ],
   [
    "C6",
    "x_vars[1,7]"
   ],
   [
    "C7",
    "x_vars[1,8]"
   ],
   [
    "C8",
    "x_vars[1,9]"
   ],
   [
    "C9",
    "x_vars[1,10]"
   ],
   [
    "C10",
    "x_vars[1,11]"
   ],
   [
    "C11",
    "x_vars[2,1]"
   ],
   [
    "C12",
    "x_vars[2,2]"
   ],
   [
    "C13",
    "x_vars[2,3]"
   ],
   [
    "C14",
    "x_vars[2,4]"
   ],
   [
    "C15",
    "x_vars[2,5]"
   ],
   [
    "C16",
    "x_vars[2,6]"
   ],
   [
    "C17",
    "x_vars[2,7]"
   ],
   [
    "C18",
    "x_vars[2,8]"
   ],
   [
    "C19",
    "x_vars[2,9]"
   ],
   [
    "C20",
    "x_vars[2,10]"
   ],
   [
    "C21",
    "x_vars[2,11]"
   ],
   [
    "C22",
    "x_vars[3,1]"
   ],
   [
    "C23",
    "x_vars[3,2]"
   ],
   [
    "C24",
    "x_vars[3,3]"
   ],
   [
    "C25",
    "x_vars[3,4]"
   ],
   [
    "C26",
    "x_vars[3,5]"
   ],
   [
    "C27",
    "x_vars[3,6]"
   ],
   [
    "C28",
    "x_vars[3,7]"
   ],
   [
    "C29",
    "x_vars[3,8]"
   ],
   [
    "C30",
    "x_vars[3,9]"
   ],
   [
    "C31",
    "x_vars[3,10]"
   ],
   [
    "C32",
    "x_vars[3,11]"
   ],
   [
    "C33",
    "x_vars[4,1]"
   ],
   [
    "C34",
    "x_vars[4,2]"
   ],
   [
    "C35",
    "x_vars[4,3]"
   ],
   [
    "C36",
    "x_vars[4,4]"
   ],
   [
    "C37",
    "x_vars[4,5]"
   ],
   [
    "C38",
    "x_vars[4,6]"
   ],
   [
    "C39",
    "x_vars[4,7]"
   ],
   [
    "C40",
    "x_vars[4,8]"
   ],
   [
    "C41",
    "x_vars[4,9]"
   ],
   [
    "C42",
    "x_vars[4,10]"
   ],
   [
    "C43",
    "x_vars[4,11]"
   ],
   [
    "C44",
    "x_vars[5,1]"
   ],
   [
    "C45",
    "x_vars[5,2]"
   ],
   [
    "C46",
    "x_vars[5,3]"
   ],
   [
    "C47",
    "x_vars[5,4]"
   ],
   [
    "C48",
    "x_vars[5,5]"
   ],
   [
    "C49",
    "x_vars[5,6]"
   ],
   [
    "C50",
    "x_vars[5,7]"
   ],
   [
    "C51",
    "x_vars[5,8]"
   ],
   [
    "C52",
    "x_vars[5,9]"
   ],
   [
    "C53",
    "x_vars[5,10]"
   ],
   [
    "C54",
    "x_vars[5,11]"
   ],
   [
    "C55",
    "x_vars[6,1]"
   ],
   [
    "C56",
    "x_vars[6,2]"
   ],
   [
    "C57",
    "x_vars[6,3]"
   ],
   [
    "C58",
    "x_vars[6,4]"
   ],
   [
    "C59",
    "x_vars[6,5]"
   ],
   [
    "C60",
    "x_vars[6,6]"
   ],
   [
    "C61",
    "x_vars[6,7]"
   ],
   [
    "C62",
    "x_vars[6,8]"
   ],
   [
    "C63",
    "x_vars[6,9]"
   ],
   [
    "C64",
    "x_vars[6,10]"
   ],
   [
    "C65",
    "x_vars[6,11]"
   ],
   [
    "C66",
    "x_vars[7,1]"
   ],
   [
    "C67",
    "x_vars[7,2]"
   ],
   [
    "C68",
    "x_vars[7,3]"
   ],
   [
    "C69",
    "x_vars[7,4]"
   ],
   [
    "C70",
    "x_vars[7,5]"
   ],
   [
    "C71",
    "x_vars[7,6]"
   ],
   [
    "C72",
    "x_vars[7,7]"
   ],
   [
    "C73",
    "x_vars[7,8]"
   ],
   [
    "C74",
    "x_vars[7,9]"
   ],
   [
    "C75",
    "x_vars[7,10]"
   ],
   [
    "C76",
    "x_vars[7,11]"
   ],
   [
    "C77",
    "x_vars[8,1]"
   ],
   [
    "C78",
    "x_vars[8,2]"
   ],
   [
    "C79",
    "x_vars[8,3]"
   ],
   [
    "C80",
    "x_vars[8,4]"
   ],
   [
    "C81",
    "x_vars[8,5]"
   ],
   [
    "C82",
    "x_vars[8,6]"
   ],
   [
    "C83",
    "x_vars[8,7]"
   ],
   [
    "C84",
    "x_vars[8,8]"
   ],
   [
    "C85",
    "x_vars[8,9]"
   ],
   [
    "C86",
    "x_vars[8,10]"
   ],
   [
    "C87",
    "x_vars[8,11]"
   ],
   [
    "C88",
    "x_vars[9,1]"
   ],
   [
    "C89",
    "x_vars[9,2]"
   ],
   [
    "C90",
    "x_vars[9,3]"
   ],
   [
    "C91",
    "x_vars[9,4]"
   ],
   [
    "C92",
    "x_vars[9,5]"
   ],
   [
    "C93",
    "x_vars[9,6]"
   ],
   [
    "C94",
    "x_vars[9,7]"
   ],
   [
    "C95",
    "x_vars[9,8]"
   ],
   [
    "C96",
    "x_vars[9,9]"
   ],
   [
    "C97",
    "x_vars[9,10]"
   ],
   [
    "C98",
    "x_vars[9,11]"
   ],
   [
    "C99",
    "x_vars[10,1]"
   ],
   [
    "C100",
    "x_vars[10,2]"
   ],
   [
    "C101",
    "x_vars[10,3]"
   ],
   [
    "C102",
    "x_vars[10,4]"
   ],
   [
    "C103",
    "x_vars[10,5]"
   ],
   [
    "C104",
    "x_vars[10,6]"
   ],
   [
    "C105",
    "x_vars[10,7]"
   ],
   [
    "C106",
    "x_vars[10,8]"
   ],
   [
    "C107",
    "x_vars[10,9]"
   ],
   [
    "C108",
    "x_vars[10,10]"
   ],
   [
    "C109",
    "x_vars[10,11]"
   ],
   [
    "C110",
    "x_vars[11,1]"
   ],
   [
    "C111",
    "x_vars[11,2]"
   ],
   [
    "C112",
    "x_vars[11,3]"
   ],
   [
    "C113",
    "x_vars[11,4]"
   ],
   [
    "C114",
    "x_vars[11,5]"
   ],
   [
    "C115",
    "x_vars[11,6]"
   ],
   [
    "C116",
    "x_vars[11,7]"
   ],
   [
    "C117",
    "x_vars[11,8]"
   ],
   [
    "C118",
    "x_vars[11,9]"
   ],
   [
    "C119",
    "x_vars[11,10]"
   ],
   [
    "C120",
    "x_vars[11,11]"
   ],
   [
    "C121",
    "y_vars[1]"
   ],
   [
    "C122",
    "y_vars[2]"
   ],
   [
    "C123",
    "y_vars[3]"
   ],
   [
    "C124",
    "y_vars[4]"
   ],
   [
    "C125",
    "y_vars[5]"
   ],
   [
    "C126",
    "y_vars[6]"
   ],
   [
    "C127",
    "y_vars[7]"
   ],
   [
    "C128",
    "y_vars[8]"
   ],
   [
    "C129",
    "y_vars[9]"
   ],
   [
    "C130",
    "y_vars[10]"
   ],
   [
    "C131",
    "y_vars[11]"
   ]
  ],
  "primary_mapping": {
   "x_W1_S1": "x_vars[1,1]",
   "x_W1_S2": "x_vars[1,2]",
   "x_W1_S3": "x_vars[1,3]",
   "x_W1_S4": "x_vars[1,4]",
   "x_W1_S5": "x_vars[1,5]",
   "x_W1_S6": "x_vars[1,6]",
   "x_W1_S7": "x_vars[1,7]",
   "x_W1_S8": "x_vars[1,8]",
   "x_W1_S9": "x_vars[1,9]",
   "x_W1_S10": "x_vars[1,10]",
   "x_W1_S11": "x_vars[1,11]",
   "x_W2_S1": "x_vars[2,1]",
   "x_W2_S2": "x_vars[2,2]",
   "x_W2_S3": "x_vars[2,3]",
   "x_W2_S4": "x_vars[2,4]",
   "x_W2_S5": "x_vars[2,5]",
   "x_W2_S6": "x_vars[2,6]",
   "x_W2_S7": "x_vars[2,7]",
   "x_W2_S8": "x_vars[2,8]",
   "x_W2_S9": "x_vars[2,9]",
   "x_W2_S10": "x_vars[2,10]",
   "x_W2_S11": "x_vars[2,11]",
   "x_W3_S1": "x_vars[3,1]",
   "x_W3_S2": "x_vars[3,2]",
   "x_W3_S3": "x_vars[3,3]",
   "x_W3_S4": "x_vars[3,4]",
   "x_W3_S5": "x_vars[3,5]",
   "x_W3_S6": "x_vars[3,6]",
   "x_W3_S7": "x_vars[3,7]",
   "x_W3_S8": "x_vars[3,8]",
   "x_W3_S9": "x_vars[3,9]",
   "x_W3_S10": "x_vars[3,10]",
   "x_W3_S11": "x_vars[3,11]",
   "x_W4_S1": "x_vars[4,1]",
   "x_W4_S2": "x_vars[4,2]",
   "x_W4_S3": "x_vars[4,3]",
   "x_W4_S4": "x_vars[4,4]",
   "x_W4_S5": "x_vars[4,5]",
   "x_W4_S6": "x_vars[4,6]",
   "x_W4_S7": "x_vars[4,7]",
   "x_W4_S8": "x_vars[4,8]",
   "x_W4_S9": "x_vars[4,9]",
   "x_W4_S10": "x_vars[4,10]",
   "x_W4_S11": "x_vars[4,11]",
   "x_W5_S1": "x_vars[5,1]",
   "x_W5_S2": "x_vars[5,2]",
   "x_W5_S3": "x_vars[5,3]",
   "x_W5_S4": "x_vars[5,4]",
   "x_W5_S5": "x_vars[5,5]",
   "x_W5_S6": "x_vars[5,6]",
   "x_W5_S7": "x_vars[5,7]",
   "x_W5_S8": "x_vars[5,8]",
   "x_W5_S9": "x_vars[5,9]",
   "x_W5_S10": "x_vars[5,10]",
   "x_W5_S11": "x_vars[5,11]",
   "x_W6_S1": "x_vars[6,1]",
   "x_W6_S2": "x_vars[6,2]",
   "x_W6_S3": "x_vars[6,3]",
   "x_W6_S4": "x_vars[6,4]",
   "x_W6_S5": "x_vars[6,5]",
   "x_W6_S6": "x_vars[6,6]",
   "x_W6_S7": "x_vars[6,7]",
   "x_W6_S8": "x_vars[6,8]",
   "x_W6_S9": "x_vars[6,9]",
   "x_W6_S10": "x_vars[6,10]",
   "x_W6_S11": "x_vars[6,11]",
   "x_W7_S1": "x_vars[7,1]",
   "x_W7_S2": "x_vars[7,2]",
   "x_W7_S3": "x_vars[7,3]",
   "x_W7_S4": "x_vars[7,4]",
   "x_W7_S5": "x_vars[7,5]",
   "x_W7_S6": "x_vars[7,6]",
   "x_W7_S7": "x_vars[7,7]",
   "x_W7_S8": "x_vars[7,8]",
   "x_W7_S9": "x_vars[7,9]",
   "x_W7_S10": "x_vars[7,10]",
   "x_W7_S11": "x_vars[7,11]",
   "x_W8_S1": "x_vars[8,1]",
   "x_W8_S2": "x_vars[8,2]",
   "x_W8_S3": "x_vars[8,3]",
   "x_W8_S4": "x_vars[8,4]",
   "x_W8_S5": "x_vars[8,5]",
   "x_W8_S6": "x_vars[8,6]",
   "x_W8_S7": "x_vars[8,7]",
   "x_W8_S8": "x_vars[8,8]",
   "x_W8_S9": "x_vars[8,9]",
   "x_W8_S10": "x_vars[8,10]",
   "x_W8_S11": "x_vars[8,11]",
   "x_W9_S1": "x_vars[9,1]",
   "x_W9_S2": "x_vars[9,2]",
   "x_W9_S3": "x_vars[9,3]",
   "x_W9_S4": "x_vars[9,4]",
   "x_W9_S5": "x_vars[9,5]",
   "x_W9_S6": "x_vars[9,6]",
   "x_W9_S7": "x_vars[9,7]",
   "x_W9_S8": "x_vars[9,8]",
   "x_W9_S9": "x_vars[9,9]",
   "x_W9_S10": "x_vars[9,10]",
   "x_W9_S11": "x_vars[9,11]",
   "x_W10_S1": "x_vars[10,1]",
   "x_W10_S2": "x_vars[10,2]",
   "x_W10_S3": "x_vars[10,3]",
   "x_W10_S4": "x_vars[10,4]",
   "x_W10_S5": "x_vars[10,5]",
   "x_W10_S6": "x_vars[10,6]",
   "x_W10_S7": "x_vars[10,7]",
   "x_W10_S8": "x_vars[10,8]",
   "x_W10_S9": "x_vars[10,9]",
   "x_W10_S10": "x_vars[10,10]",
   "x_W10_S11": "x_vars[10,11]",
   "x_W11_S1": "x_vars[11,1]",
   "x_W11_S2": "x_vars[11,2]",
   "x_W11_S3": "x_vars[11,3]",
   "x_W11_S4": "x_vars[11,4]",
   "x_W11_S5": "x_vars[11,5]",
   "x_W11_S6": "x_vars[11,6]",
   "x_W11_S7": "x_vars[11,7]",
   "x_W11_S8": "x_vars[11,8]",
   "x_W11_S9": "x_vars[11,9]",
   "x_W11_S10": "x_vars[11,10]",
   "x_W11_S11": "x_vars[11,11]",
   "y_W1": "y_vars[1]",
   "y_W2": "y_vars[2]",
   "y_W3": "y_vars[3]",
   "y_W4": "y_vars[4]",
   "y_W5": "y_vars[5]",
   "y_W6": "y_vars[6]",
   "y_W7": "y_vars[7]",
   "y_W8": "y_vars[8]",
   "y_W9": "y_vars[9]",
   "y_W10": "y_vars[10]",
   "y_W11": "y_vars[11]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-100": {
  "source_code_sha256": "6960dab533a19cd61b6e3f987675533c6cd44fd34af96ad6fc018a3f861dda02",
  "original_solution_sha256": "9c4b9195636cb1aa0c8e47c6ba42f178080a4684d879c2ac6d9f373e40a74b27",
  "bindings": [
   [
    "C0",
    "x_vars[C1]"
   ],
   [
    "C1",
    "x_vars[C2]"
   ],
   [
    "C2",
    "x_vars[C3]"
   ],
   [
    "C3",
    "x_vars[C4]"
   ],
   [
    "C4",
    "x_vars[C5]"
   ],
   [
    "C5",
    "x_vars[C6]"
   ],
   [
    "C6",
    "x_vars[C7]"
   ],
   [
    "C7",
    "x_vars[C8]"
   ],
   [
    "C8",
    "x_vars[C9]"
   ],
   [
    "C9",
    "x_vars[C10]"
   ],
   [
    "C10",
    "x_vars[C11]"
   ],
   [
    "C11",
    "x_vars[C12]"
   ],
   [
    "C12",
    "x_vars[C13]"
   ],
   [
    "C13",
    "x_vars[C14]"
   ],
   [
    "C14",
    "x_vars[C15]"
   ],
   [
    "C15",
    "x_vars[C16]"
   ],
   [
    "C16",
    "x_vars[C17]"
   ],
   [
    "C17",
    "x_vars[C18]"
   ],
   [
    "C18",
    "x_vars[C19]"
   ],
   [
    "C19",
    "x_vars[C20]"
   ],
   [
    "C20",
    "x_vars[C21]"
   ],
   [
    "C21",
    "x_vars[C22]"
   ],
   [
    "C22",
    "x_vars[C23]"
   ],
   [
    "C23",
    "x_vars[C24]"
   ],
   [
    "C24",
    "x_vars[C25]"
   ],
   [
    "C25",
    "x_vars[C26]"
   ],
   [
    "C26",
    "x_vars[C27]"
   ],
   [
    "C27",
    "x_vars[C28]"
   ],
   [
    "C28",
    "x_vars[C29]"
   ],
   [
    "C29",
    "x_vars[C30]"
   ],
   [
    "C30",
    "x_vars[C31]"
   ],
   [
    "C31",
    "x_vars[C32]"
   ],
   [
    "C32",
    "x_vars[C33]"
   ],
   [
    "C33",
    "x_vars[C34]"
   ],
   [
    "C34",
    "x_vars[C35]"
   ],
   [
    "C35",
    "x_vars[C36]"
   ],
   [
    "C36",
    "x_vars[C37]"
   ],
   [
    "C37",
    "x_vars[C38]"
   ],
   [
    "C38",
    "x_vars[C39]"
   ],
   [
    "C39",
    "x_vars[C40]"
   ],
   [
    "C40",
    "x_vars[C41]"
   ],
   [
    "C41",
    "x_vars[C42]"
   ],
   [
    "C42",
    "x_vars[C43]"
   ],
   [
    "C43",
    "x_vars[C44]"
   ],
   [
    "C44",
    "x_vars[C45]"
   ],
   [
    "C45",
    "x_vars[C46]"
   ],
   [
    "C46",
    "x_vars[C47]"
   ],
   [
    "C47",
    "x_vars[C48]"
   ],
   [
    "C48",
    "x_vars[C49]"
   ],
   [
    "C49",
    "x_vars[C50]"
   ],
   [
    "C50",
    "x_vars[C51]"
   ],
   [
    "C51",
    "x_vars[C52]"
   ],
   [
    "C52",
    "x_vars[C53]"
   ],
   [
    "C53",
    "x_vars[C54]"
   ],
   [
    "C54",
    "x_vars[C55]"
   ],
   [
    "C55",
    "x_vars[C56]"
   ],
   [
    "C56",
    "x_vars[C57]"
   ],
   [
    "C57",
    "x_vars[C58]"
   ],
   [
    "C58",
    "x_vars[C59]"
   ],
   [
    "C59",
    "x_vars[C60]"
   ],
   [
    "C60",
    "x_vars[C61]"
   ],
   [
    "C61",
    "x_vars[C62]"
   ],
   [
    "C62",
    "x_vars[C63]"
   ],
   [
    "C63",
    "x_vars[C64]"
   ],
   [
    "C64",
    "x_vars[C65]"
   ],
   [
    "C65",
    "x_vars[C66]"
   ],
   [
    "C66",
    "x_vars[C67]"
   ],
   [
    "C67",
    "x_vars[C68]"
   ],
   [
    "C68",
    "x_vars[C69]"
   ],
   [
    "C69",
    "x_vars[C70]"
   ],
   [
    "C70",
    "x_vars[C71]"
   ],
   [
    "C71",
    "x_vars[C72]"
   ],
   [
    "C72",
    "x_vars[C73]"
   ],
   [
    "C73",
    "x_vars[C74]"
   ],
   [
    "C74",
    "x_vars[C75]"
   ],
   [
    "C75",
    "x_vars[C76]"
   ],
   [
    "C76",
    "x_vars[C77]"
   ],
   [
    "C77",
    "x_vars[C78]"
   ],
   [
    "C78",
    "x_vars[C79]"
   ],
   [
    "C79",
    "x_vars[C80]"
   ],
   [
    "C80",
    "x_vars[C81]"
   ],
   [
    "C81",
    "x_vars[C82]"
   ],
   [
    "C82",
    "x_vars[C83]"
   ],
   [
    "C83",
    "x_vars[C84]"
   ],
   [
    "C84",
    "x_vars[C85]"
   ],
   [
    "C85",
    "x_vars[C86]"
   ],
   [
    "C86",
    "x_vars[C87]"
   ],
   [
    "C87",
    "x_vars[C88]"
   ],
   [
    "C88",
    "x_vars[C89]"
   ],
   [
    "C89",
    "x_vars[C90]"
   ],
   [
    "C90",
    "x_vars[C91]"
   ],
   [
    "C91",
    "x_vars[C92]"
   ],
   [
    "C92",
    "x_vars[C93]"
   ],
   [
    "C93",
    "x_vars[C94]"
   ],
   [
    "C94",
    "x_vars[C95]"
   ],
   [
    "C95",
    "x_vars[C96]"
   ],
   [
    "C96",
    "x_vars[C97]"
   ],
   [
    "C97",
    "x_vars[C98]"
   ],
   [
    "C98",
    "x_vars[C99]"
   ],
   [
    "C99",
    "x_vars[C100]"
   ],
   [
    "C100",
    "x_vars[C101]"
   ],
   [
    "C101",
    "x_vars[C102]"
   ],
   [
    "C102",
    "x_vars[C103]"
   ],
   [
    "C103",
    "x_vars[C104]"
   ],
   [
    "C104",
    "x_vars[C105]"
   ],
   [
    "C105",
    "x_vars[C106]"
   ],
   [
    "C106",
    "x_vars[C107]"
   ],
   [
    "C107",
    "x_vars[C108]"
   ],
   [
    "C108",
    "x_vars[C109]"
   ],
   [
    "C109",
    "x_vars[C110]"
   ],
   [
    "C110",
    "x_vars[C111]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x_vars[C1]",
   "x[2]": "x_vars[C2]",
   "x[3]": "x_vars[C3]",
   "x[4]": "x_vars[C4]",
   "x[5]": "x_vars[C5]",
   "x[6]": "x_vars[C6]",
   "x[7]": "x_vars[C7]",
   "x[8]": "x_vars[C8]",
   "x[9]": "x_vars[C9]",
   "x[10]": "x_vars[C10]",
   "x[11]": "x_vars[C11]",
   "x[12]": "x_vars[C12]",
   "x[13]": "x_vars[C13]",
   "x[14]": "x_vars[C14]",
   "x[15]": "x_vars[C15]",
   "x[16]": "x_vars[C16]",
   "x[17]": "x_vars[C17]",
   "x[18]": "x_vars[C18]",
   "x[19]": "x_vars[C19]",
   "x[20]": "x_vars[C20]",
   "x[21]": "x_vars[C21]",
   "x[22]": "x_vars[C22]",
   "x[23]": "x_vars[C23]",
   "x[24]": "x_vars[C24]",
   "x[25]": "x_vars[C25]",
   "x[26]": "x_vars[C26]",
   "x[27]": "x_vars[C27]",
   "x[28]": "x_vars[C28]",
   "x[29]": "x_vars[C29]",
   "x[30]": "x_vars[C30]",
   "x[31]": "x_vars[C31]",
   "x[32]": "x_vars[C32]",
   "x[33]": "x_vars[C33]",
   "x[34]": "x_vars[C34]",
   "x[35]": "x_vars[C35]",
   "x[36]": "x_vars[C36]",
   "x[37]": "x_vars[C37]",
   "x[38]": "x_vars[C38]",
   "x[39]": "x_vars[C39]",
   "x[40]": "x_vars[C40]",
   "x[41]": "x_vars[C41]",
   "x[42]": "x_vars[C42]",
   "x[43]": "x_vars[C43]",
   "x[44]": "x_vars[C44]",
   "x[45]": "x_vars[C45]",
   "x[46]": "x_vars[C46]",
   "x[47]": "x_vars[C47]",
   "x[48]": "x_vars[C48]",
   "x[49]": "x_vars[C49]",
   "x[50]": "x_vars[C50]",
   "x[51]": "x_vars[C51]",
   "x[52]": "x_vars[C52]",
   "x[53]": "x_vars[C53]",
   "x[54]": "x_vars[C54]",
   "x[55]": "x_vars[C55]",
   "x[56]": "x_vars[C56]",
   "x[57]": "x_vars[C57]",
   "x[58]": "x_vars[C58]",
   "x[59]": "x_vars[C59]",
   "x[60]": "x_vars[C60]",
   "x[61]": "x_vars[C61]",
   "x[62]": "x_vars[C62]",
   "x[63]": "x_vars[C63]",
   "x[64]": "x_vars[C64]",
   "x[65]": "x_vars[C65]",
   "x[66]": "x_vars[C66]",
   "x[67]": "x_vars[C67]",
   "x[68]": "x_vars[C68]",
   "x[69]": "x_vars[C69]",
   "x[70]": "x_vars[C70]",
   "x[71]": "x_vars[C71]",
   "x[72]": "x_vars[C72]",
   "x[73]": "x_vars[C73]",
   "x[74]": "x_vars[C74]",
   "x[75]": "x_vars[C75]",
   "x[76]": "x_vars[C76]",
   "x[77]": "x_vars[C77]",
   "x[78]": "x_vars[C78]",
   "x[79]": "x_vars[C79]",
   "x[80]": "x_vars[C80]",
   "x[81]": "x_vars[C81]",
   "x[82]": "x_vars[C82]",
   "x[83]": "x_vars[C83]",
   "x[84]": "x_vars[C84]",
   "x[85]": "x_vars[C85]",
   "x[86]": "x_vars[C86]",
   "x[87]": "x_vars[C87]",
   "x[88]": "x_vars[C88]",
   "x[89]": "x_vars[C89]",
   "x[90]": "x_vars[C90]",
   "x[91]": "x_vars[C91]",
   "x[92]": "x_vars[C92]",
   "x[93]": "x_vars[C93]",
   "x[94]": "x_vars[C94]",
   "x[95]": "x_vars[C95]",
   "x[96]": "x_vars[C96]",
   "x[97]": "x_vars[C97]",
   "x[98]": "x_vars[C98]",
   "x[99]": "x_vars[C99]",
   "x[100]": "x_vars[C100]",
   "x[101]": "x_vars[C101]",
   "x[102]": "x_vars[C102]",
   "x[103]": "x_vars[C103]",
   "x[104]": "x_vars[C104]",
   "x[105]": "x_vars[C105]",
   "x[106]": "x_vars[C106]",
   "x[107]": "x_vars[C107]",
   "x[108]": "x_vars[C108]",
   "x[109]": "x_vars[C109]",
   "x[110]": "x_vars[C110]",
   "x[111]": "x_vars[C111]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 },
 "OR-101": {
  "source_code_sha256": "ade618355868ed653c3afaaaacb0b9ad9b04528b1a80cb46654de1776f2f2314",
  "original_solution_sha256": "5df9e97dfcc3bb4dd11f7409f24f6d74efca385c2031c324aa4040608aad7a02",
  "bindings": [
   [
    "C0",
    "x_vars[P1]"
   ],
   [
    "C1",
    "x_vars[P2]"
   ],
   [
    "C2",
    "x_vars[P3]"
   ],
   [
    "C3",
    "x_vars[P4]"
   ],
   [
    "C4",
    "x_vars[P5]"
   ],
   [
    "C5",
    "x_vars[P6]"
   ],
   [
    "C6",
    "x_vars[P7]"
   ],
   [
    "C7",
    "x_vars[P8]"
   ],
   [
    "C8",
    "x_vars[P9]"
   ],
   [
    "C9",
    "x_vars[P10]"
   ],
   [
    "C10",
    "x_vars[P11]"
   ],
   [
    "C11",
    "x_vars[P12]"
   ],
   [
    "C12",
    "x_vars[P13]"
   ],
   [
    "C13",
    "x_vars[P14]"
   ],
   [
    "C14",
    "x_vars[P15]"
   ],
   [
    "C15",
    "x_vars[P16]"
   ],
   [
    "C16",
    "x_vars[P17]"
   ],
   [
    "C17",
    "x_vars[P18]"
   ],
   [
    "C18",
    "x_vars[P19]"
   ],
   [
    "C19",
    "x_vars[P20]"
   ],
   [
    "C20",
    "x_vars[P21]"
   ],
   [
    "C21",
    "x_vars[P22]"
   ],
   [
    "C22",
    "x_vars[P23]"
   ],
   [
    "C23",
    "x_vars[P24]"
   ],
   [
    "C24",
    "x_vars[P25]"
   ],
   [
    "C25",
    "x_vars[P26]"
   ],
   [
    "C26",
    "x_vars[P27]"
   ],
   [
    "C27",
    "x_vars[P28]"
   ],
   [
    "C28",
    "x_vars[P29]"
   ],
   [
    "C29",
    "x_vars[P30]"
   ],
   [
    "C30",
    "x_vars[P31]"
   ],
   [
    "C31",
    "x_vars[P32]"
   ],
   [
    "C32",
    "x_vars[P33]"
   ],
   [
    "C33",
    "x_vars[P34]"
   ],
   [
    "C34",
    "x_vars[P35]"
   ],
   [
    "C35",
    "x_vars[P36]"
   ],
   [
    "C36",
    "x_vars[P37]"
   ],
   [
    "C37",
    "x_vars[P38]"
   ],
   [
    "C38",
    "x_vars[P39]"
   ],
   [
    "C39",
    "x_vars[P40]"
   ],
   [
    "C40",
    "x_vars[P41]"
   ],
   [
    "C41",
    "x_vars[P42]"
   ],
   [
    "C42",
    "x_vars[P43]"
   ],
   [
    "C43",
    "x_vars[P44]"
   ],
   [
    "C44",
    "x_vars[P45]"
   ],
   [
    "C45",
    "x_vars[P46]"
   ],
   [
    "C46",
    "x_vars[P47]"
   ],
   [
    "C47",
    "x_vars[P48]"
   ],
   [
    "C48",
    "x_vars[P49]"
   ],
   [
    "C49",
    "x_vars[P50]"
   ],
   [
    "C50",
    "x_vars[P51]"
   ],
   [
    "C51",
    "x_vars[P52]"
   ],
   [
    "C52",
    "x_vars[P53]"
   ],
   [
    "C53",
    "x_vars[P54]"
   ],
   [
    "C54",
    "x_vars[P55]"
   ],
   [
    "C55",
    "x_vars[P56]"
   ],
   [
    "C56",
    "x_vars[P57]"
   ],
   [
    "C57",
    "x_vars[P58]"
   ],
   [
    "C58",
    "x_vars[P59]"
   ],
   [
    "C59",
    "x_vars[P60]"
   ],
   [
    "C60",
    "x_vars[P61]"
   ],
   [
    "C61",
    "x_vars[P62]"
   ],
   [
    "C62",
    "x_vars[P63]"
   ],
   [
    "C63",
    "x_vars[P64]"
   ],
   [
    "C64",
    "x_vars[P65]"
   ],
   [
    "C65",
    "x_vars[P66]"
   ],
   [
    "C66",
    "x_vars[P67]"
   ],
   [
    "C67",
    "x_vars[P68]"
   ],
   [
    "C68",
    "x_vars[P69]"
   ],
   [
    "C69",
    "x_vars[P70]"
   ],
   [
    "C70",
    "x_vars[P71]"
   ],
   [
    "C71",
    "x_vars[P72]"
   ],
   [
    "C72",
    "x_vars[P73]"
   ],
   [
    "C73",
    "x_vars[P74]"
   ],
   [
    "C74",
    "x_vars[P75]"
   ],
   [
    "C75",
    "x_vars[P76]"
   ],
   [
    "C76",
    "x_vars[P77]"
   ],
   [
    "C77",
    "x_vars[P78]"
   ],
   [
    "C78",
    "x_vars[P79]"
   ],
   [
    "C79",
    "x_vars[P80]"
   ],
   [
    "C80",
    "x_vars[P81]"
   ],
   [
    "C81",
    "x_vars[P82]"
   ],
   [
    "C82",
    "x_vars[P83]"
   ],
   [
    "C83",
    "x_vars[P84]"
   ],
   [
    "C84",
    "x_vars[P85]"
   ],
   [
    "C85",
    "x_vars[P86]"
   ],
   [
    "C86",
    "x_vars[P87]"
   ],
   [
    "C87",
    "x_vars[P88]"
   ],
   [
    "C88",
    "x_vars[P89]"
   ],
   [
    "C89",
    "x_vars[P90]"
   ],
   [
    "C90",
    "x_vars[P91]"
   ],
   [
    "C91",
    "x_vars[P92]"
   ],
   [
    "C92",
    "x_vars[P93]"
   ],
   [
    "C93",
    "x_vars[P94]"
   ],
   [
    "C94",
    "x_vars[P95]"
   ],
   [
    "C95",
    "x_vars[P96]"
   ],
   [
    "C96",
    "x_vars[P97]"
   ],
   [
    "C97",
    "x_vars[P98]"
   ],
   [
    "C98",
    "x_vars[P99]"
   ],
   [
    "C99",
    "x_vars[P100]"
   ],
   [
    "C100",
    "x_vars[P101]"
   ],
   [
    "C101",
    "x_vars[P102]"
   ],
   [
    "C102",
    "x_vars[P103]"
   ],
   [
    "C103",
    "x_vars[P104]"
   ],
   [
    "C104",
    "x_vars[P105]"
   ],
   [
    "C105",
    "x_vars[P106]"
   ],
   [
    "C106",
    "x_vars[P107]"
   ],
   [
    "C107",
    "x_vars[P108]"
   ],
   [
    "C108",
    "x_vars[P109]"
   ],
   [
    "C109",
    "x_vars[P110]"
   ],
   [
    "C110",
    "x_vars[P111]"
   ]
  ],
  "primary_mapping": {
   "x[P1]": "x_vars[P1]",
   "x[P2]": "x_vars[P2]",
   "x[P3]": "x_vars[P3]",
   "x[P4]": "x_vars[P4]",
   "x[P5]": "x_vars[P5]",
   "x[P6]": "x_vars[P6]",
   "x[P7]": "x_vars[P7]",
   "x[P8]": "x_vars[P8]",
   "x[P9]": "x_vars[P9]",
   "x[P10]": "x_vars[P10]",
   "x[P11]": "x_vars[P11]",
   "x[P12]": "x_vars[P12]",
   "x[P13]": "x_vars[P13]",
   "x[P14]": "x_vars[P14]",
   "x[P15]": "x_vars[P15]",
   "x[P16]": "x_vars[P16]",
   "x[P17]": "x_vars[P17]",
   "x[P18]": "x_vars[P18]",
   "x[P19]": "x_vars[P19]",
   "x[P20]": "x_vars[P20]",
   "x[P21]": "x_vars[P21]",
   "x[P22]": "x_vars[P22]",
   "x[P23]": "x_vars[P23]",
   "x[P24]": "x_vars[P24]",
   "x[P25]": "x_vars[P25]",
   "x[P26]": "x_vars[P26]",
   "x[P27]": "x_vars[P27]",
   "x[P28]": "x_vars[P28]",
   "x[P29]": "x_vars[P29]",
   "x[P30]": "x_vars[P30]",
   "x[P31]": "x_vars[P31]",
   "x[P32]": "x_vars[P32]",
   "x[P33]": "x_vars[P33]",
   "x[P34]": "x_vars[P34]",
   "x[P35]": "x_vars[P35]",
   "x[P36]": "x_vars[P36]",
   "x[P37]": "x_vars[P37]",
   "x[P38]": "x_vars[P38]",
   "x[P39]": "x_vars[P39]",
   "x[P40]": "x_vars[P40]",
   "x[P41]": "x_vars[P41]",
   "x[P42]": "x_vars[P42]",
   "x[P43]": "x_vars[P43]",
   "x[P44]": "x_vars[P44]",
   "x[P45]": "x_vars[P45]",
   "x[P46]": "x_vars[P46]",
   "x[P47]": "x_vars[P47]",
   "x[P48]": "x_vars[P48]",
   "x[P49]": "x_vars[P49]",
   "x[P50]": "x_vars[P50]",
   "x[P51]": "x_vars[P51]",
   "x[P52]": "x_vars[P52]",
   "x[P53]": "x_vars[P53]",
   "x[P54]": "x_vars[P54]",
   "x[P55]": "x_vars[P55]",
   "x[P56]": "x_vars[P56]",
   "x[P57]": "x_vars[P57]",
   "x[P58]": "x_vars[P58]",
   "x[P59]": "x_vars[P59]",
   "x[P60]": "x_vars[P60]",
   "x[P61]": "x_vars[P61]",
   "x[P62]": "x_vars[P62]",
   "x[P63]": "x_vars[P63]",
   "x[P64]": "x_vars[P64]",
   "x[P65]": "x_vars[P65]",
   "x[P66]": "x_vars[P66]",
   "x[P67]": "x_vars[P67]",
   "x[P68]": "x_vars[P68]",
   "x[P69]": "x_vars[P69]",
   "x[P70]": "x_vars[P70]",
   "x[P71]": "x_vars[P71]",
   "x[P72]": "x_vars[P72]",
   "x[P73]": "x_vars[P73]",
   "x[P74]": "x_vars[P74]",
   "x[P75]": "x_vars[P75]",
   "x[P76]": "x_vars[P76]",
   "x[P77]": "x_vars[P77]",
   "x[P78]": "x_vars[P78]",
   "x[P79]": "x_vars[P79]",
   "x[P80]": "x_vars[P80]",
   "x[P81]": "x_vars[P81]",
   "x[P82]": "x_vars[P82]",
   "x[P83]": "x_vars[P83]",
   "x[P84]": "x_vars[P84]",
   "x[P85]": "x_vars[P85]",
   "x[P86]": "x_vars[P86]",
   "x[P87]": "x_vars[P87]",
   "x[P88]": "x_vars[P88]",
   "x[P89]": "x_vars[P89]",
   "x[P90]": "x_vars[P90]",
   "x[P91]": "x_vars[P91]",
   "x[P92]": "x_vars[P92]",
   "x[P93]": "x_vars[P93]",
   "x[P94]": "x_vars[P94]",
   "x[P95]": "x_vars[P95]",
   "x[P96]": "x_vars[P96]",
   "x[P97]": "x_vars[P97]",
   "x[P98]": "x_vars[P98]",
   "x[P99]": "x_vars[P99]",
   "x[P100]": "x_vars[P100]",
   "x[P101]": "x_vars[P101]",
   "x[P102]": "x_vars[P102]",
   "x[P103]": "x_vars[P103]",
   "x[P104]": "x_vars[P104]",
   "x[P105]": "x_vars[P105]",
   "x[P106]": "x_vars[P106]",
   "x[P107]": "x_vars[P107]",
   "x[P108]": "x_vars[P108]",
   "x[P109]": "x_vars[P109]",
   "x[P110]": "x_vars[P110]",
   "x[P111]": "x_vars[P111]"
  },
  "mapping_basis": "revalidated_entity_keys_container_aliases"
 }
}
""")


In [ ]:
IDENTITY_ARTIFACTS['LEAN-GPT-4.1'] = GPT41_IDENTITIES


In [ ]:
OSS20B_IDENTITIES = json.loads(r"""
{
 "OR-001": {
  "source_code_sha256": "04629d854b1c5521aaa1022fc036873ad8d79d0e72d3b81258bc6cfa9b7edbe1",
  "original_solution_sha256": "e4a0c9f56a3d9dcd9c964e9e7f378aa6ee5c7d57040e280c3d428db913b45d23",
  "bindings": [
   [
    "C0",
    "x[S2,C7]"
   ],
   [
    "C1",
    "x[S2,C14]"
   ],
   [
    "C2",
    "x[S2,C16]"
   ],
   [
    "C3",
    "x[S2,C10]"
   ],
   [
    "C4",
    "x[S2,C15]"
   ],
   [
    "C5",
    "x[S2,C6]"
   ],
   [
    "C6",
    "x[S2,C5]"
   ],
   [
    "C7",
    "x[S2,C8]"
   ],
   [
    "C8",
    "x[S2,C2]"
   ],
   [
    "C9",
    "x[S2,C4]"
   ],
   [
    "C10",
    "x[S2,C1]"
   ],
   [
    "C11",
    "x[S2,C13]"
   ],
   [
    "C12",
    "x[S2,C17]"
   ],
   [
    "C13",
    "x[S2,C9]"
   ],
   [
    "C14",
    "x[S2,C12]"
   ],
   [
    "C15",
    "x[S2,C18]"
   ],
   [
    "C16",
    "x[S2,C11]"
   ],
   [
    "C17",
    "x[S2,C3]"
   ],
   [
    "C18",
    "x[S7,C7]"
   ],
   [
    "C19",
    "x[S7,C14]"
   ],
   [
    "C20",
    "x[S7,C16]"
   ],
   [
    "C21",
    "x[S7,C10]"
   ],
   [
    "C22",
    "x[S7,C15]"
   ],
   [
    "C23",
    "x[S7,C6]"
   ],
   [
    "C24",
    "x[S7,C5]"
   ],
   [
    "C25",
    "x[S7,C8]"
   ],
   [
    "C26",
    "x[S7,C2]"
   ],
   [
    "C27",
    "x[S7,C4]"
   ],
   [
    "C28",
    "x[S7,C1]"
   ],
   [
    "C29",
    "x[S7,C13]"
   ],
   [
    "C30",
    "x[S7,C17]"
   ],
   [
    "C31",
    "x[S7,C9]"
   ],
   [
    "C32",
    "x[S7,C12]"
   ],
   [
    "C33",
    "x[S7,C18]"
   ],
   [
    "C34",
    "x[S7,C11]"
   ],
   [
    "C35",
    "x[S7,C3]"
   ],
   [
    "C36",
    "x[S3,C7]"
   ],
   [
    "C37",
    "x[S3,C14]"
   ],
   [
    "C38",
    "x[S3,C16]"
   ],
   [
    "C39",
    "x[S3,C10]"
   ],
   [
    "C40",
    "x[S3,C15]"
   ],
   [
    "C41",
    "x[S3,C6]"
   ],
   [
    "C42",
    "x[S3,C5]"
   ],
   [
    "C43",
    "x[S3,C8]"
   ],
   [
    "C44",
    "x[S3,C2]"
   ],
   [
    "C45",
    "x[S3,C4]"
   ],
   [
    "C46",
    "x[S3,C1]"
   ],
   [
    "C47",
    "x[S3,C13]"
   ],
   [
    "C48",
    "x[S3,C17]"
   ],
   [
    "C49",
    "x[S3,C9]"
   ],
   [
    "C50",
    "x[S3,C12]"
   ],
   [
    "C51",
    "x[S3,C18]"
   ],
   [
    "C52",
    "x[S3,C11]"
   ],
   [
    "C53",
    "x[S3,C3]"
   ],
   [
    "C54",
    "x[S6,C7]"
   ],
   [
    "C55",
    "x[S6,C14]"
   ],
   [
    "C56",
    "x[S6,C16]"
   ],
   [
    "C57",
    "x[S6,C10]"
   ],
   [
    "C58",
    "x[S6,C15]"
   ],
   [
    "C59",
    "x[S6,C6]"
   ],
   [
    "C60",
    "x[S6,C5]"
   ],
   [
    "C61",
    "x[S6,C8]"
   ],
   [
    "C62",
    "x[S6,C2]"
   ],
   [
    "C63",
    "x[S6,C4]"
   ],
   [
    "C64",
    "x[S6,C1]"
   ],
   [
    "C65",
    "x[S6,C13]"
   ],
   [
    "C66",
    "x[S6,C17]"
   ],
   [
    "C67",
    "x[S6,C9]"
   ],
   [
    "C68",
    "x[S6,C12]"
   ],
   [
    "C69",
    "x[S6,C18]"
   ],
   [
    "C70",
    "x[S6,C11]"
   ],
   [
    "C71",
    "x[S6,C3]"
   ],
   [
    "C72",
    "x[S18,C7]"
   ],
   [
    "C73",
    "x[S18,C14]"
   ],
   [
    "C74",
    "x[S18,C16]"
   ],
   [
    "C75",
    "x[S18,C10]"
   ],
   [
    "C76",
    "x[S18,C15]"
   ],
   [
    "C77",
    "x[S18,C6]"
   ],
   [
    "C78",
    "x[S18,C5]"
   ],
   [
    "C79",
    "x[S18,C8]"
   ],
   [
    "C80",
    "x[S18,C2]"
   ],
   [
    "C81",
    "x[S18,C4]"
   ],
   [
    "C82",
    "x[S18,C1]"
   ],
   [
    "C83",
    "x[S18,C13]"
   ],
   [
    "C84",
    "x[S18,C17]"
   ],
   [
    "C85",
    "x[S18,C9]"
   ],
   [
    "C86",
    "x[S18,C12]"
   ],
   [
    "C87",
    "x[S18,C18]"
   ],
   [
    "C88",
    "x[S18,C11]"
   ],
   [
    "C89",
    "x[S18,C3]"
   ],
   [
    "C90",
    "x[S12,C7]"
   ],
   [
    "C91",
    "x[S12,C14]"
   ],
   [
    "C92",
    "x[S12,C16]"
   ],
   [
    "C93",
    "x[S12,C10]"
   ],
   [
    "C94",
    "x[S12,C15]"
   ],
   [
    "C95",
    "x[S12,C6]"
   ],
   [
    "C96",
    "x[S12,C5]"
   ],
   [
    "C97",
    "x[S12,C8]"
   ],
   [
    "C98",
    "x[S12,C2]"
   ],
   [
    "C99",
    "x[S12,C4]"
   ],
   [
    "C100",
    "x[S12,C1]"
   ],
   [
    "C101",
    "x[S12,C13]"
   ],
   [
    "C102",
    "x[S12,C17]"
   ],
   [
    "C103",
    "x[S12,C9]"
   ],
   [
    "C104",
    "x[S12,C12]"
   ],
   [
    "C105",
    "x[S12,C18]"
   ],
   [
    "C106",
    "x[S12,C11]"
   ],
   [
    "C107",
    "x[S12,C3]"
   ],
   [
    "C108",
    "x[S4,C7]"
   ],
   [
    "C109",
    "x[S4,C14]"
   ],
   [
    "C110",
    "x[S4,C16]"
   ],
   [
    "C111",
    "x[S4,C10]"
   ],
   [
    "C112",
    "x[S4,C15]"
   ],
   [
    "C113",
    "x[S4,C6]"
   ],
   [
    "C114",
    "x[S4,C5]"
   ],
   [
    "C115",
    "x[S4,C8]"
   ],
   [
    "C116",
    "x[S4,C2]"
   ],
   [
    "C117",
    "x[S4,C4]"
   ],
   [
    "C118",
    "x[S4,C1]"
   ],
   [
    "C119",
    "x[S4,C13]"
   ],
   [
    "C120",
    "x[S4,C17]"
   ],
   [
    "C121",
    "x[S4,C9]"
   ],
   [
    "C122",
    "x[S4,C12]"
   ],
   [
    "C123",
    "x[S4,C18]"
   ],
   [
    "C124",
    "x[S4,C11]"
   ],
   [
    "C125",
    "x[S4,C3]"
   ],
   [
    "C126",
    "x[S13,C7]"
   ],
   [
    "C127",
    "x[S13,C14]"
   ],
   [
    "C128",
    "x[S13,C16]"
   ],
   [
    "C129",
    "x[S13,C10]"
   ],
   [
    "C130",
    "x[S13,C15]"
   ],
   [
    "C131",
    "x[S13,C6]"
   ],
   [
    "C132",
    "x[S13,C5]"
   ],
   [
    "C133",
    "x[S13,C8]"
   ],
   [
    "C134",
    "x[S13,C2]"
   ],
   [
    "C135",
    "x[S13,C4]"
   ],
   [
    "C136",
    "x[S13,C1]"
   ],
   [
    "C137",
    "x[S13,C13]"
   ],
   [
    "C138",
    "x[S13,C17]"
   ],
   [
    "C139",
    "x[S13,C9]"
   ],
   [
    "C140",
    "x[S13,C12]"
   ],
   [
    "C141",
    "x[S13,C18]"
   ],
   [
    "C142",
    "x[S13,C11]"
   ],
   [
    "C143",
    "x[S13,C3]"
   ],
   [
    "C144",
    "x[S9,C7]"
   ],
   [
    "C145",
    "x[S9,C14]"
   ],
   [
    "C146",
    "x[S9,C16]"
   ],
   [
    "C147",
    "x[S9,C10]"
   ],
   [
    "C148",
    "x[S9,C15]"
   ],
   [
    "C149",
    "x[S9,C6]"
   ],
   [
    "C150",
    "x[S9,C5]"
   ],
   [
    "C151",
    "x[S9,C8]"
   ],
   [
    "C152",
    "x[S9,C2]"
   ],
   [
    "C153",
    "x[S9,C4]"
   ],
   [
    "C154",
    "x[S9,C1]"
   ],
   [
    "C155",
    "x[S9,C13]"
   ],
   [
    "C156",
    "x[S9,C17]"
   ],
   [
    "C157",
    "x[S9,C9]"
   ],
   [
    "C158",
    "x[S9,C12]"
   ],
   [
    "C159",
    "x[S9,C18]"
   ],
   [
    "C160",
    "x[S9,C11]"
   ],
   [
    "C161",
    "x[S9,C3]"
   ],
   [
    "C162",
    "x[S1,C7]"
   ],
   [
    "C163",
    "x[S1,C14]"
   ],
   [
    "C164",
    "x[S1,C16]"
   ],
   [
    "C165",
    "x[S1,C10]"
   ],
   [
    "C166",
    "x[S1,C15]"
   ],
   [
    "C167",
    "x[S1,C6]"
   ],
   [
    "C168",
    "x[S1,C5]"
   ],
   [
    "C169",
    "x[S1,C8]"
   ],
   [
    "C170",
    "x[S1,C2]"
   ],
   [
    "C171",
    "x[S1,C4]"
   ],
   [
    "C172",
    "x[S1,C1]"
   ],
   [
    "C173",
    "x[S1,C13]"
   ],
   [
    "C174",
    "x[S1,C17]"
   ],
   [
    "C175",
    "x[S1,C9]"
   ],
   [
    "C176",
    "x[S1,C12]"
   ],
   [
    "C177",
    "x[S1,C18]"
   ],
   [
    "C178",
    "x[S1,C11]"
   ],
   [
    "C179",
    "x[S1,C3]"
   ],
   [
    "C180",
    "x[S16,C7]"
   ],
   [
    "C181",
    "x[S16,C14]"
   ],
   [
    "C182",
    "x[S16,C16]"
   ],
   [
    "C183",
    "x[S16,C10]"
   ],
   [
    "C184",
    "x[S16,C15]"
   ],
   [
    "C185",
    "x[S16,C6]"
   ],
   [
    "C186",
    "x[S16,C5]"
   ],
   [
    "C187",
    "x[S16,C8]"
   ],
   [
    "C188",
    "x[S16,C2]"
   ],
   [
    "C189",
    "x[S16,C4]"
   ],
   [
    "C190",
    "x[S16,C1]"
   ],
   [
    "C191",
    "x[S16,C13]"
   ],
   [
    "C192",
    "x[S16,C17]"
   ],
   [
    "C193",
    "x[S16,C9]"
   ],
   [
    "C194",
    "x[S16,C12]"
   ],
   [
    "C195",
    "x[S16,C18]"
   ],
   [
    "C196",
    "x[S16,C11]"
   ],
   [
    "C197",
    "x[S16,C3]"
   ],
   [
    "C198",
    "x[S15,C7]"
   ],
   [
    "C199",
    "x[S15,C14]"
   ],
   [
    "C200",
    "x[S15,C16]"
   ],
   [
    "C201",
    "x[S15,C10]"
   ],
   [
    "C202",
    "x[S15,C15]"
   ],
   [
    "C203",
    "x[S15,C6]"
   ],
   [
    "C204",
    "x[S15,C5]"
   ],
   [
    "C205",
    "x[S15,C8]"
   ],
   [
    "C206",
    "x[S15,C2]"
   ],
   [
    "C207",
    "x[S15,C4]"
   ],
   [
    "C208",
    "x[S15,C1]"
   ],
   [
    "C209",
    "x[S15,C13]"
   ],
   [
    "C210",
    "x[S15,C17]"
   ],
   [
    "C211",
    "x[S15,C9]"
   ],
   [
    "C212",
    "x[S15,C12]"
   ],
   [
    "C213",
    "x[S15,C18]"
   ],
   [
    "C214",
    "x[S15,C11]"
   ],
   [
    "C215",
    "x[S15,C3]"
   ],
   [
    "C216",
    "x[S8,C7]"
   ],
   [
    "C217",
    "x[S8,C14]"
   ],
   [
    "C218",
    "x[S8,C16]"
   ],
   [
    "C219",
    "x[S8,C10]"
   ],
   [
    "C220",
    "x[S8,C15]"
   ],
   [
    "C221",
    "x[S8,C6]"
   ],
   [
    "C222",
    "x[S8,C5]"
   ],
   [
    "C223",
    "x[S8,C8]"
   ],
   [
    "C224",
    "x[S8,C2]"
   ],
   [
    "C225",
    "x[S8,C4]"
   ],
   [
    "C226",
    "x[S8,C1]"
   ],
   [
    "C227",
    "x[S8,C13]"
   ],
   [
    "C228",
    "x[S8,C17]"
   ],
   [
    "C229",
    "x[S8,C9]"
   ],
   [
    "C230",
    "x[S8,C12]"
   ],
   [
    "C231",
    "x[S8,C18]"
   ],
   [
    "C232",
    "x[S8,C11]"
   ],
   [
    "C233",
    "x[S8,C3]"
   ],
   [
    "C234",
    "x[S10,C7]"
   ],
   [
    "C235",
    "x[S10,C14]"
   ],
   [
    "C236",
    "x[S10,C16]"
   ],
   [
    "C237",
    "x[S10,C10]"
   ],
   [
    "C238",
    "x[S10,C15]"
   ],
   [
    "C239",
    "x[S10,C6]"
   ],
   [
    "C240",
    "x[S10,C5]"
   ],
   [
    "C241",
    "x[S10,C8]"
   ],
   [
    "C242",
    "x[S10,C2]"
   ],
   [
    "C243",
    "x[S10,C4]"
   ],
   [
    "C244",
    "x[S10,C1]"
   ],
   [
    "C245",
    "x[S10,C13]"
   ],
   [
    "C246",
    "x[S10,C17]"
   ],
   [
    "C247",
    "x[S10,C9]"
   ],
   [
    "C248",
    "x[S10,C12]"
   ],
   [
    "C249",
    "x[S10,C18]"
   ],
   [
    "C250",
    "x[S10,C11]"
   ],
   [
    "C251",
    "x[S10,C3]"
   ],
   [
    "C252",
    "x[S17,C7]"
   ],
   [
    "C253",
    "x[S17,C14]"
   ],
   [
    "C254",
    "x[S17,C16]"
   ],
   [
    "C255",
    "x[S17,C10]"
   ],
   [
    "C256",
    "x[S17,C15]"
   ],
   [
    "C257",
    "x[S17,C6]"
   ],
   [
    "C258",
    "x[S17,C5]"
   ],
   [
    "C259",
    "x[S17,C8]"
   ],
   [
    "C260",
    "x[S17,C2]"
   ],
   [
    "C261",
    "x[S17,C4]"
   ],
   [
    "C262",
    "x[S17,C1]"
   ],
   [
    "C263",
    "x[S17,C13]"
   ],
   [
    "C264",
    "x[S17,C17]"
   ],
   [
    "C265",
    "x[S17,C9]"
   ],
   [
    "C266",
    "x[S17,C12]"
   ],
   [
    "C267",
    "x[S17,C18]"
   ],
   [
    "C268",
    "x[S17,C11]"
   ],
   [
    "C269",
    "x[S17,C3]"
   ],
   [
    "C270",
    "x[S5,C7]"
   ],
   [
    "C271",
    "x[S5,C14]"
   ],
   [
    "C272",
    "x[S5,C16]"
   ],
   [
    "C273",
    "x[S5,C10]"
   ],
   [
    "C274",
    "x[S5,C15]"
   ],
   [
    "C275",
    "x[S5,C6]"
   ],
   [
    "C276",
    "x[S5,C5]"
   ],
   [
    "C277",
    "x[S5,C8]"
   ],
   [
    "C278",
    "x[S5,C2]"
   ],
   [
    "C279",
    "x[S5,C4]"
   ],
   [
    "C280",
    "x[S5,C1]"
   ],
   [
    "C281",
    "x[S5,C13]"
   ],
   [
    "C282",
    "x[S5,C17]"
   ],
   [
    "C283",
    "x[S5,C9]"
   ],
   [
    "C284",
    "x[S5,C12]"
   ],
   [
    "C285",
    "x[S5,C18]"
   ],
   [
    "C286",
    "x[S5,C11]"
   ],
   [
    "C287",
    "x[S5,C3]"
   ],
   [
    "C288",
    "x[S14,C7]"
   ],
   [
    "C289",
    "x[S14,C14]"
   ],
   [
    "C290",
    "x[S14,C16]"
   ],
   [
    "C291",
    "x[S14,C10]"
   ],
   [
    "C292",
    "x[S14,C15]"
   ],
   [
    "C293",
    "x[S14,C6]"
   ],
   [
    "C294",
    "x[S14,C5]"
   ],
   [
    "C295",
    "x[S14,C8]"
   ],
   [
    "C296",
    "x[S14,C2]"
   ],
   [
    "C297",
    "x[S14,C4]"
   ],
   [
    "C298",
    "x[S14,C1]"
   ],
   [
    "C299",
    "x[S14,C13]"
   ],
   [
    "C300",
    "x[S14,C17]"
   ],
   [
    "C301",
    "x[S14,C9]"
   ],
   [
    "C302",
    "x[S14,C12]"
   ],
   [
    "C303",
    "x[S14,C18]"
   ],
   [
    "C304",
    "x[S14,C11]"
   ],
   [
    "C305",
    "x[S14,C3]"
   ],
   [
    "C306",
    "x[S11,C7]"
   ],
   [
    "C307",
    "x[S11,C14]"
   ],
   [
    "C308",
    "x[S11,C16]"
   ],
   [
    "C309",
    "x[S11,C10]"
   ],
   [
    "C310",
    "x[S11,C15]"
   ],
   [
    "C311",
    "x[S11,C6]"
   ],
   [
    "C312",
    "x[S11,C5]"
   ],
   [
    "C313",
    "x[S11,C8]"
   ],
   [
    "C314",
    "x[S11,C2]"
   ],
   [
    "C315",
    "x[S11,C4]"
   ],
   [
    "C316",
    "x[S11,C1]"
   ],
   [
    "C317",
    "x[S11,C13]"
   ],
   [
    "C318",
    "x[S11,C17]"
   ],
   [
    "C319",
    "x[S11,C9]"
   ],
   [
    "C320",
    "x[S11,C12]"
   ],
   [
    "C321",
    "x[S11,C18]"
   ],
   [
    "C322",
    "x[S11,C11]"
   ],
   [
    "C323",
    "x[S11,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_0_10": "x[S1,C11]",
   "x_0_11": "x[S1,C12]",
   "x_0_12": "x[S1,C13]",
   "x_0_13": "x[S1,C14]",
   "x_0_14": "x[S1,C15]",
   "x_0_15": "x[S1,C16]",
   "x_0_16": "x[S1,C17]",
   "x_0_17": "x[S1,C18]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_1_10": "x[S2,C11]",
   "x_1_11": "x[S2,C12]",
   "x_1_12": "x[S2,C13]",
   "x_1_13": "x[S2,C14]",
   "x_1_14": "x[S2,C15]",
   "x_1_15": "x[S2,C16]",
   "x_1_16": "x[S2,C17]",
   "x_1_17": "x[S2,C18]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_2_10": "x[S3,C11]",
   "x_2_11": "x[S3,C12]",
   "x_2_12": "x[S3,C13]",
   "x_2_13": "x[S3,C14]",
   "x_2_14": "x[S3,C15]",
   "x_2_15": "x[S3,C16]",
   "x_2_16": "x[S3,C17]",
   "x_2_17": "x[S3,C18]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_3_10": "x[S4,C11]",
   "x_3_11": "x[S4,C12]",
   "x_3_12": "x[S4,C13]",
   "x_3_13": "x[S4,C14]",
   "x_3_14": "x[S4,C15]",
   "x_3_15": "x[S4,C16]",
   "x_3_16": "x[S4,C17]",
   "x_3_17": "x[S4,C18]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_4_10": "x[S5,C11]",
   "x_4_11": "x[S5,C12]",
   "x_4_12": "x[S5,C13]",
   "x_4_13": "x[S5,C14]",
   "x_4_14": "x[S5,C15]",
   "x_4_15": "x[S5,C16]",
   "x_4_16": "x[S5,C17]",
   "x_4_17": "x[S5,C18]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_5_10": "x[S6,C11]",
   "x_5_11": "x[S6,C12]",
   "x_5_12": "x[S6,C13]",
   "x_5_13": "x[S6,C14]",
   "x_5_14": "x[S6,C15]",
   "x_5_15": "x[S6,C16]",
   "x_5_16": "x[S6,C17]",
   "x_5_17": "x[S6,C18]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_8": "x[S7,C9]",
   "x_6_9": "x[S7,C10]",
   "x_6_10": "x[S7,C11]",
   "x_6_11": "x[S7,C12]",
   "x_6_12": "x[S7,C13]",
   "x_6_13": "x[S7,C14]",
   "x_6_14": "x[S7,C15]",
   "x_6_15": "x[S7,C16]",
   "x_6_16": "x[S7,C17]",
   "x_6_17": "x[S7,C18]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_7_10": "x[S8,C11]",
   "x_7_11": "x[S8,C12]",
   "x_7_12": "x[S8,C13]",
   "x_7_13": "x[S8,C14]",
   "x_7_14": "x[S8,C15]",
   "x_7_15": "x[S8,C16]",
   "x_7_16": "x[S8,C17]",
   "x_7_17": "x[S8,C18]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_8_10": "x[S9,C11]",
   "x_8_11": "x[S9,C12]",
   "x_8_12": "x[S9,C13]",
   "x_8_13": "x[S9,C14]",
   "x_8_14": "x[S9,C15]",
   "x_8_15": "x[S9,C16]",
   "x_8_16": "x[S9,C17]",
   "x_8_17": "x[S9,C18]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]",
   "x_9_10": "x[S10,C11]",
   "x_9_11": "x[S10,C12]",
   "x_9_12": "x[S10,C13]",
   "x_9_13": "x[S10,C14]",
   "x_9_14": "x[S10,C15]",
   "x_9_15": "x[S10,C16]",
   "x_9_16": "x[S10,C17]",
   "x_9_17": "x[S10,C18]",
   "x_10_0": "x[S11,C1]",
   "x_10_1": "x[S11,C2]",
   "x_10_2": "x[S11,C3]",
   "x_10_3": "x[S11,C4]",
   "x_10_4": "x[S11,C5]",
   "x_10_5": "x[S11,C6]",
   "x_10_6": "x[S11,C7]",
   "x_10_7": "x[S11,C8]",
   "x_10_8": "x[S11,C9]",
   "x_10_9": "x[S11,C10]",
   "x_10_10": "x[S11,C11]",
   "x_10_11": "x[S11,C12]",
   "x_10_12": "x[S11,C13]",
   "x_10_13": "x[S11,C14]",
   "x_10_14": "x[S11,C15]",
   "x_10_15": "x[S11,C16]",
   "x_10_16": "x[S11,C17]",
   "x_10_17": "x[S11,C18]",
   "x_11_0": "x[S12,C1]",
   "x_11_1": "x[S12,C2]",
   "x_11_2": "x[S12,C3]",
   "x_11_3": "x[S12,C4]",
   "x_11_4": "x[S12,C5]",
   "x_11_5": "x[S12,C6]",
   "x_11_6": "x[S12,C7]",
   "x_11_7": "x[S12,C8]",
   "x_11_8": "x[S12,C9]",
   "x_11_9": "x[S12,C10]",
   "x_11_10": "x[S12,C11]",
   "x_11_11": "x[S12,C12]",
   "x_11_12": "x[S12,C13]",
   "x_11_13": "x[S12,C14]",
   "x_11_14": "x[S12,C15]",
   "x_11_15": "x[S12,C16]",
   "x_11_16": "x[S12,C17]",
   "x_11_17": "x[S12,C18]",
   "x_12_0": "x[S13,C1]",
   "x_12_1": "x[S13,C2]",
   "x_12_2": "x[S13,C3]",
   "x_12_3": "x[S13,C4]",
   "x_12_4": "x[S13,C5]",
   "x_12_5": "x[S13,C6]",
   "x_12_6": "x[S13,C7]",
   "x_12_7": "x[S13,C8]",
   "x_12_8": "x[S13,C9]",
   "x_12_9": "x[S13,C10]",
   "x_12_10": "x[S13,C11]",
   "x_12_11": "x[S13,C12]",
   "x_12_12": "x[S13,C13]",
   "x_12_13": "x[S13,C14]",
   "x_12_14": "x[S13,C15]",
   "x_12_15": "x[S13,C16]",
   "x_12_16": "x[S13,C17]",
   "x_12_17": "x[S13,C18]",
   "x_13_0": "x[S14,C1]",
   "x_13_1": "x[S14,C2]",
   "x_13_2": "x[S14,C3]",
   "x_13_3": "x[S14,C4]",
   "x_13_4": "x[S14,C5]",
   "x_13_5": "x[S14,C6]",
   "x_13_6": "x[S14,C7]",
   "x_13_7": "x[S14,C8]",
   "x_13_8": "x[S14,C9]",
   "x_13_9": "x[S14,C10]",
   "x_13_10": "x[S14,C11]",
   "x_13_11": "x[S14,C12]",
   "x_13_12": "x[S14,C13]",
   "x_13_13": "x[S14,C14]",
   "x_13_14": "x[S14,C15]",
   "x_13_15": "x[S14,C16]",
   "x_13_16": "x[S14,C17]",
   "x_13_17": "x[S14,C18]",
   "x_14_0": "x[S15,C1]",
   "x_14_1": "x[S15,C2]",
   "x_14_2": "x[S15,C3]",
   "x_14_3": "x[S15,C4]",
   "x_14_4": "x[S15,C5]",
   "x_14_5": "x[S15,C6]",
   "x_14_6": "x[S15,C7]",
   "x_14_7": "x[S15,C8]",
   "x_14_8": "x[S15,C9]",
   "x_14_9": "x[S15,C10]",
   "x_14_10": "x[S15,C11]",
   "x_14_11": "x[S15,C12]",
   "x_14_12": "x[S15,C13]",
   "x_14_13": "x[S15,C14]",
   "x_14_14": "x[S15,C15]",
   "x_14_15": "x[S15,C16]",
   "x_14_16": "x[S15,C17]",
   "x_14_17": "x[S15,C18]",
   "x_15_0": "x[S16,C1]",
   "x_15_1": "x[S16,C2]",
   "x_15_2": "x[S16,C3]",
   "x_15_3": "x[S16,C4]",
   "x_15_4": "x[S16,C5]",
   "x_15_5": "x[S16,C6]",
   "x_15_6": "x[S16,C7]",
   "x_15_7": "x[S16,C8]",
   "x_15_8": "x[S16,C9]",
   "x_15_9": "x[S16,C10]",
   "x_15_10": "x[S16,C11]",
   "x_15_11": "x[S16,C12]",
   "x_15_12": "x[S16,C13]",
   "x_15_13": "x[S16,C14]",
   "x_15_14": "x[S16,C15]",
   "x_15_15": "x[S16,C16]",
   "x_15_16": "x[S16,C17]",
   "x_15_17": "x[S16,C18]",
   "x_16_0": "x[S17,C1]",
   "x_16_1": "x[S17,C2]",
   "x_16_2": "x[S17,C3]",
   "x_16_3": "x[S17,C4]",
   "x_16_4": "x[S17,C5]",
   "x_16_5": "x[S17,C6]",
   "x_16_6": "x[S17,C7]",
   "x_16_7": "x[S17,C8]",
   "x_16_8": "x[S17,C9]",
   "x_16_9": "x[S17,C10]",
   "x_16_10": "x[S17,C11]",
   "x_16_11": "x[S17,C12]",
   "x_16_12": "x[S17,C13]",
   "x_16_13": "x[S17,C14]",
   "x_16_14": "x[S17,C15]",
   "x_16_15": "x[S17,C16]",
   "x_16_16": "x[S17,C17]",
   "x_16_17": "x[S17,C18]",
   "x_17_0": "x[S18,C1]",
   "x_17_1": "x[S18,C2]",
   "x_17_2": "x[S18,C3]",
   "x_17_3": "x[S18,C4]",
   "x_17_4": "x[S18,C5]",
   "x_17_5": "x[S18,C6]",
   "x_17_6": "x[S18,C7]",
   "x_17_7": "x[S18,C8]",
   "x_17_8": "x[S18,C9]",
   "x_17_9": "x[S18,C10]",
   "x_17_10": "x[S18,C11]",
   "x_17_11": "x[S18,C12]",
   "x_17_12": "x[S18,C13]",
   "x_17_13": "x[S18,C14]",
   "x_17_14": "x[S18,C15]",
   "x_17_15": "x[S18,C16]",
   "x_17_16": "x[S18,C17]",
   "x_17_17": "x[S18,C18]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 172752.82458839752,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     517.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     185.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     27.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     180.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     350.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     555.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     411.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     1456.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     556.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     333.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     205.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     1348.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     354.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     10638.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     491.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     146.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     316.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     4415.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     2155.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     81.0
    ],
    [
     "C162",
     151.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     4489.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     5430.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     0.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     0.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     0.0
    ],
    [
     "C204",
     0.0
    ],
    [
     "C205",
     0.0
    ],
    [
     "C206",
     0.0
    ],
    [
     "C207",
     0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     0.0
    ],
    [
     "C211",
     0.0
    ],
    [
     "C212",
     0.0
    ],
    [
     "C213",
     0.0
    ],
    [
     "C214",
     0.0
    ],
    [
     "C215",
     0.0
    ],
    [
     "C216",
     0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     0.0
    ],
    [
     "C219",
     0.0
    ],
    [
     "C220",
     0.0
    ],
    [
     "C221",
     0.0
    ],
    [
     "C222",
     0.0
    ],
    [
     "C223",
     0.0
    ],
    [
     "C224",
     0.0
    ],
    [
     "C225",
     0.0
    ],
    [
     "C226",
     0.0
    ],
    [
     "C227",
     788.0
    ],
    [
     "C228",
     0.0
    ],
    [
     "C229",
     0.0
    ],
    [
     "C230",
     0.0
    ],
    [
     "C231",
     0.0
    ],
    [
     "C232",
     0.0
    ],
    [
     "C233",
     0.0
    ],
    [
     "C234",
     0.0
    ],
    [
     "C235",
     0.0
    ],
    [
     "C236",
     43.0
    ],
    [
     "C237",
     0.0
    ],
    [
     "C238",
     0.0
    ],
    [
     "C239",
     0.0
    ],
    [
     "C240",
     0.0
    ],
    [
     "C241",
     0.0
    ],
    [
     "C242",
     0.0
    ],
    [
     "C243",
     0.0
    ],
    [
     "C244",
     0.0
    ],
    [
     "C245",
     0.0
    ],
    [
     "C246",
     0.0
    ],
    [
     "C247",
     0.0
    ],
    [
     "C248",
     0.0
    ],
    [
     "C249",
     0.0
    ],
    [
     "C250",
     0.0
    ],
    [
     "C251",
     0.0
    ],
    [
     "C252",
     0.0
    ],
    [
     "C253",
     0.0
    ],
    [
     "C254",
     0.0
    ],
    [
     "C255",
     0.0
    ],
    [
     "C256",
     0.0
    ],
    [
     "C257",
     0.0
    ],
    [
     "C258",
     0.0
    ],
    [
     "C259",
     0.0
    ],
    [
     "C260",
     0.0
    ],
    [
     "C261",
     0.0
    ],
    [
     "C262",
     0.0
    ],
    [
     "C263",
     0.0
    ],
    [
     "C264",
     0.0
    ],
    [
     "C265",
     1584.0
    ],
    [
     "C266",
     0.0
    ],
    [
     "C267",
     0.0
    ],
    [
     "C268",
     0.0
    ],
    [
     "C269",
     0.0
    ],
    [
     "C270",
     0.0
    ],
    [
     "C271",
     0.0
    ],
    [
     "C272",
     0.0
    ],
    [
     "C273",
     0.0
    ],
    [
     "C274",
     0.0
    ],
    [
     "C275",
     0.0
    ],
    [
     "C276",
     0.0
    ],
    [
     "C277",
     0.0
    ],
    [
     "C278",
     0.0
    ],
    [
     "C279",
     0.0
    ],
    [
     "C280",
     0.0
    ],
    [
     "C281",
     0.0
    ],
    [
     "C282",
     0.0
    ],
    [
     "C283",
     0.0
    ],
    [
     "C284",
     0.0
    ],
    [
     "C285",
     0.0
    ],
    [
     "C286",
     0.0
    ],
    [
     "C287",
     0.0
    ],
    [
     "C288",
     0.0
    ],
    [
     "C289",
     0.0
    ],
    [
     "C290",
     0.0
    ],
    [
     "C291",
     0.0
    ],
    [
     "C292",
     0.0
    ],
    [
     "C293",
     0.0
    ],
    [
     "C294",
     0.0
    ],
    [
     "C295",
     0.0
    ],
    [
     "C296",
     0.0
    ],
    [
     "C297",
     0.0
    ],
    [
     "C298",
     0.0
    ],
    [
     "C299",
     0.0
    ],
    [
     "C300",
     0.0
    ],
    [
     "C301",
     0.0
    ],
    [
     "C302",
     2038.0
    ],
    [
     "C303",
     0.0
    ],
    [
     "C304",
     76.0
    ],
    [
     "C305",
     0.0
    ],
    [
     "C306",
     0.0
    ],
    [
     "C307",
     0.0
    ],
    [
     "C308",
     0.0
    ],
    [
     "C309",
     0.0
    ],
    [
     "C310",
     0.0
    ],
    [
     "C311",
     0.0
    ],
    [
     "C312",
     0.0
    ],
    [
     "C313",
     0.0
    ],
    [
     "C314",
     0.0
    ],
    [
     "C315",
     0.0
    ],
    [
     "C316",
     0.0
    ],
    [
     "C317",
     0.0
    ],
    [
     "C318",
     0.0
    ],
    [
     "C319",
     0.0
    ],
    [
     "C320",
     0.0
    ],
    [
     "C321",
     0.0
    ],
    [
     "C322",
     0.0
    ],
    [
     "C323",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-007": {
  "source_code_sha256": "ab9b75e14bb09f2a1a2667c4ed7cb0b1652f322080ac86c38f07e042ba6abbd2",
  "original_solution_sha256": "a5ee20180faf3ea3348eed3a46c9a986e362a9c6f7804e55e4fb684461d3eb8a",
  "bindings": [
   [
    "C0",
    "x[S2,D1]"
   ],
   [
    "C1",
    "x[S2,D3]"
   ],
   [
    "C2",
    "x[S2,D5]"
   ],
   [
    "C3",
    "x[S2,D2]"
   ],
   [
    "C4",
    "x[S2,D4]"
   ],
   [
    "C5",
    "x[S3,D1]"
   ],
   [
    "C6",
    "x[S3,D3]"
   ],
   [
    "C7",
    "x[S3,D5]"
   ],
   [
    "C8",
    "x[S3,D2]"
   ],
   [
    "C9",
    "x[S3,D4]"
   ],
   [
    "C10",
    "x[S4,D1]"
   ],
   [
    "C11",
    "x[S4,D3]"
   ],
   [
    "C12",
    "x[S4,D5]"
   ],
   [
    "C13",
    "x[S4,D2]"
   ],
   [
    "C14",
    "x[S4,D4]"
   ],
   [
    "C15",
    "x[S1,D1]"
   ],
   [
    "C16",
    "x[S1,D3]"
   ],
   [
    "C17",
    "x[S1,D5]"
   ],
   [
    "C18",
    "x[S1,D2]"
   ],
   [
    "C19",
    "x[S1,D4]"
   ],
   [
    "C20",
    "x[S5,D1]"
   ],
   [
    "C21",
    "x[S5,D3]"
   ],
   [
    "C22",
    "x[S5,D5]"
   ],
   [
    "C23",
    "x[S5,D2]"
   ],
   [
    "C24",
    "x[S5,D4]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,D1]",
   "x_0_1": "x[S1,D2]",
   "x_0_2": "x[S1,D3]",
   "x_0_3": "x[S1,D4]",
   "x_0_4": "x[S1,D5]",
   "x_1_0": "x[S2,D1]",
   "x_1_1": "x[S2,D2]",
   "x_1_2": "x[S2,D3]",
   "x_1_3": "x[S2,D4]",
   "x_1_4": "x[S2,D5]",
   "x_2_0": "x[S3,D1]",
   "x_2_1": "x[S3,D2]",
   "x_2_2": "x[S3,D3]",
   "x_2_3": "x[S3,D4]",
   "x_2_4": "x[S3,D5]",
   "x_3_0": "x[S4,D1]",
   "x_3_1": "x[S4,D2]",
   "x_3_2": "x[S4,D3]",
   "x_3_3": "x[S4,D4]",
   "x_3_4": "x[S4,D5]",
   "x_4_0": "x[S5,D1]",
   "x_4_1": "x[S5,D2]",
   "x_4_2": "x[S5,D3]",
   "x_4_3": "x[S5,D4]",
   "x_4_4": "x[S5,D5]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 10291.2770066575,
   "variables": [
    [
     "C0",
     217.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     211.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     3.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     214.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     166.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     254.0
    ],
    [
     "C18",
     174.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     40.0
    ],
    [
     "C24",
     214.0
    ]
   ],
   "error": null
  }
 },
 "OR-012": {
  "source_code_sha256": "3668717fc319c18ac3d6f11f4b5d16815346c90a6ab1fb87424e961d97ffb2a9",
  "original_solution_sha256": "6b45e042c77587f8a0dd4ec1a96f1bed71e8e09c10baf187efb27f4dde100d94",
  "bindings": [
   [
    "C0",
    "x[1984 by George Orwell]"
   ],
   [
    "C1",
    "x[Adidas 3-Stripes Shorts]"
   ],
   [
    "C2",
    "x[Adidas Essential Track Pants]"
   ],
   [
    "C3",
    "x[Adidas FIFA World Cup Football]"
   ],
   [
    "C4",
    "x[Adidas Originals Superstar Sneakers]"
   ],
   [
    "C5",
    "x[Adidas Originals Trefoil Hoodie]"
   ],
   [
    "C6",
    "x[Adidas Ultraboost Running Shoes]"
   ],
   [
    "C7",
    "x[Adidas Ultraboost Shoes]"
   ],
   [
    "C8",
    "x[Amazon Echo Dot (4th Gen)]"
   ],
   [
    "C9",
    "x[Amazon Echo Show 10]"
   ],
   [
    "C10",
    "x[Amazon Fire TV Stick 4K]"
   ],
   [
    "C11",
    "x[Anastasia Beverly Hills Brow Wiz]"
   ],
   [
    "C12",
    "x[Anker PowerCore Portable Charger]"
   ],
   [
    "C13",
    "x[Anova Precision Cooker]"
   ],
   [
    "C14",
    "x[Anova Precision Oven]"
   ],
   [
    "C15",
    "x[Apple AirPods Max]"
   ],
   [
    "C16",
    "x[Apple AirPods Pro]"
   ],
   [
    "C17",
    "x[Apple MacBook Air]"
   ],
   [
    "C18",
    "x[Apple MacBook Pro 16-inch]"
   ],
   [
    "C19",
    "x[Apple TV 4K]"
   ],
   [
    "C20",
    "x[Apple Watch Series 8]"
   ],
   [
    "C21",
    "x[Apple iPad Air]"
   ],
   [
    "C22",
    "x[Atomic Habits by James Clear]"
   ],
   [
    "C23",
    "x[Babolat Pure Drive Tennis Racket]"
   ],
   [
    "C24",
    "x[Becoming by Michelle Obama]"
   ],
   [
    "C25",
    "x[Biore UV Aqua Rich Watery Essence Sunscreen]"
   ],
   [
    "C26",
    "x[Blueair Classic 480i]"
   ],
   [
    "C27",
    "x[Bose QuietComfort 35 Headphones]"
   ],
   [
    "C28",
    "x[Bose QuietComfort 35 II Wireless Headphones]"
   ],
   [
    "C29",
    "x[Bose SoundLink Color Bluetooth Speaker II]"
   ],
   [
    "C30",
    "x[Bose SoundLink Revolve+ Speaker]"
   ],
   [
    "C31",
    "x[Bose SoundSport Wireless Earbuds]"
   ],
   [
    "C32",
    "x[Bowflex SelectTech 1090 Adjustable Dumbbells]"
   ],
   [
    "C33",
    "x[Bowflex SelectTech 552 Dumbbells]"
   ],
   [
    "C34",
    "x[Breville Nespresso Creatista Plus]"
   ],
   [
    "C35",
    "x[Breville Smart Coffee Grinder Pro]"
   ],
   [
    "C36",
    "x[Breville Smart Grill]"
   ],
   [
    "C37",
    "x[Breville Smart Oven]"
   ],
   [
    "C38",
    "x[Calvin Klein Boxer Briefs]"
   ],
   [
    "C39",
    "x[Canon EOS R5 Camera]"
   ],
   [
    "C40",
    "x[Canon EOS Rebel T7i DSLR Camera]"
   ],
   [
    "C41",
    "x[Caudalie Vinoperfect Radiance Serum]"
   ],
   [
    "C42",
    "x[CeraVe Hydrating Facial Cleanser]"
   ],
   [
    "C43",
    "x[Champion Reverse Weave Hoodie]"
   ],
   [
    "C44",
    "x[Chanel No. 5 Perfume]"
   ],
   [
    "C45",
    "x[Charlotte Tilbury Magic Cream]"
   ],
   [
    "C46",
    "x[Clinique Dramatically Different Moisturizing Lotion]"
   ],
   [
    "C47",
    "x[Clinique Moisture Surge]"
   ],
   [
    "C48",
    "x[Columbia Fleece Jacket]"
   ],
   [
    "C49",
    "x[Crock-Pot 6-Quart Slow Cooker]"
   ],
   [
    "C50",
    "x[Cuisinart Coffee Center]"
   ],
   [
    "C51",
    "x[Cuisinart Custom 14-Cup Food Processor]"
   ],
   [
    "C52",
    "x[Cuisinart Griddler Deluxe]"
   ],
   [
    "C53",
    "x[De'Longhi Magnifica Espresso Machine]"
   ],
   [
    "C54",
    "x[Dr. Jart+ Cicapair Tiger Grass Color Correcting Treatment]"
   ],
   [
    "C55",
    "x[Drunk Elephant C-Firma Day Serum]"
   ],
   [
    "C56",
    "x[Dune by Frank Herbert]"
   ],
   [
    "C57",
    "x[Dyson Pure Cool Link]"
   ],
   [
    "C58",
    "x[Dyson Supersonic Hair Dryer]"
   ],
   [
    "C59",
    "x[Dyson V11 Vacuum]"
   ],
   [
    "C60",
    "x[Dyson V8 Absolute]"
   ],
   [
    "C61",
    "x[Educated by Tara Westover]"
   ],
   [
    "C62",
    "x[Estee Lauder Advanced Night Repair]"
   ],
   [
    "C63",
    "x[Eufy RoboVac 11S]"
   ],
   [
    "C64",
    "x[Fenty Beauty Killawatt Highlighter]"
   ],
   [
    "C65",
    "x[First Aid Beauty Ultra Repair Cream]"
   ],
   [
    "C66",
    "x[Fitbit Charge 5]"
   ],
   [
    "C67",
    "x[Fitbit Inspire 2]"
   ],
   [
    "C68",
    "x[Fitbit Luxe]"
   ],
   [
    "C69",
    "x[Fitbit Versa 3]"
   ],
   [
    "C70",
    "x[Forever 21 Graphic Tee]"
   ],
   [
    "C71",
    "x[Fresh Sugar Lip Treatment]"
   ],
   [
    "C72",
    "x[Gap 1969 Original Fit Jeans]"
   ],
   [
    "C73",
    "x[Gap Crewneck Sweatshirt]"
   ],
   [
    "C74",
    "x[Gap Essential Crewneck T-Shirt]"
   ],
   [
    "C75",
    "x[Gap High Rise Skinny Jeans]"
   ],
   [
    "C76",
    "x[Garmin Edge 530]"
   ],
   [
    "C77",
    "x[Garmin Fenix 6X Pro]"
   ],
   [
    "C78",
    "x[Garmin Forerunner 245]"
   ],
   [
    "C79",
    "x[Garmin Forerunner 945]"
   ],
   [
    "C80",
    "x[GlamGlow Supermud Clearing Treatment]"
   ],
   [
    "C81",
    "x[Glossier Boy Brow]"
   ],
   [
    "C82",
    "x[Glossier Cloud Paint]"
   ],
   [
    "C83",
    "x[GoPro HERO10 Black]"
   ],
   [
    "C84",
    "x[GoPro HERO9 Black]"
   ],
   [
    "C85",
    "x[Gone Girl by Gillian Flynn]"
   ],
   [
    "C86",
    "x[Google Nest Hub Max]"
   ],
   [
    "C87",
    "x[Google Nest Wifi Router]"
   ],
   [
    "C88",
    "x[Google Pixel 6 Pro]"
   ],
   [
    "C89",
    "x[Google Pixelbook Go]"
   ],
   [
    "C90",
    "x[H&M Slim Fit Jeans]"
   ],
   [
    "C91",
    "x[HP Spectre x360 Laptop]"
   ],
   [
    "C92",
    "x[Hamilton Beach FlexBrew Coffee Maker]"
   ],
   [
    "C93",
    "x[Hanes ComfortSoft T-Shirt]"
   ],
   [
    "C94",
    "x[Harry Potter and the Sorcerer's Stone]"
   ],
   [
    "C95",
    "x[Hydro Flask Standard Mouth Water Bottle]"
   ],
   [
    "C96",
    "x[Hydro Flask Wide Mouth Water Bottle]"
   ],
   [
    "C97",
    "x[Hyperice Hypervolt Massager]"
   ],
   [
    "C98",
    "x[Instant Pot Duo]"
   ],
   [
    "C99",
    "x[Instant Pot Duo Crisp]"
   ],
   [
    "C100",
    "x[Instant Pot Duo Evo Plus]"
   ],
   [
    "C101",
    "x[Instant Pot Duo Nova]"
   ],
   [
    "C102",
    "x[Instant Pot Ultra]"
   ],
   [
    "C103",
    "x[Keurig K-Elite Coffee Maker]"
   ],
   [
    "C104",
    "x[Keurig K-Mini Coffee Maker]"
   ],
   [
    "C105",
    "x[Kiehl's Midnight Recovery Concentrate]"
   ],
   [
    "C106",
    "x[Kindle Paperwhite]"
   ],
   [
    "C107",
    "x[KitchenAid Artisan Stand Mixer]"
   ],
   [
    "C108",
    "x[KitchenAid Stand Mixer]"
   ],
   [
    "C109",
    "x[L'Occitane Shea Butter Hand Cream]"
   ],
   [
    "C110",
    "x[L'Oreal Revitalift Serum]"
   ],
   [
    "C111",
    "x[LG OLED TV]"
   ],
   [
    "C112",
    "x[La Mer Cr_¨me de la Mer Moisturizer]"
   ],
   [
    "C113",
    "x[Lancome La Vie Est Belle]"
   ],
   [
    "C114",
    "x[Laneige Water Sleeping Mask]"
   ],
   [
    "C115",
    "x[Levi's 501 Jeans]"
   ],
   [
    "C116",
    "x[Levi's 511 Slim Fit Jeans]"
   ],
   [
    "C117",
    "x[Levi's Sherpa Trucker Jacket]"
   ],
   [
    "C118",
    "x[Levi's Trucker Jacket]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[1984 by George Orwell]",
   "x_2": "x[Adidas 3-Stripes Shorts]",
   "x_3": "x[Adidas Essential Track Pants]",
   "x_4": "x[Adidas FIFA World Cup Football]",
   "x_5": "x[Adidas Originals Superstar Sneakers]",
   "x_6": "x[Adidas Originals Trefoil Hoodie]",
   "x_7": "x[Adidas Ultraboost Running Shoes]",
   "x_8": "x[Adidas Ultraboost Shoes]",
   "x_9": "x[Amazon Echo Dot (4th Gen)]",
   "x_10": "x[Amazon Echo Show 10]",
   "x_11": "x[Amazon Fire TV Stick 4K]",
   "x_12": "x[Anastasia Beverly Hills Brow Wiz]",
   "x_13": "x[Anker PowerCore Portable Charger]",
   "x_14": "x[Anova Precision Cooker]",
   "x_15": "x[Anova Precision Oven]",
   "x_16": "x[Apple AirPods Max]",
   "x_17": "x[Apple AirPods Pro]",
   "x_18": "x[Apple MacBook Air]",
   "x_19": "x[Apple MacBook Pro 16-inch]",
   "x_20": "x[Apple TV 4K]",
   "x_21": "x[Apple Watch Series 8]",
   "x_22": "x[Apple iPad Air]",
   "x_23": "x[Atomic Habits by James Clear]",
   "x_24": "x[Babolat Pure Drive Tennis Racket]",
   "x_25": "x[Becoming by Michelle Obama]",
   "x_26": "x[Biore UV Aqua Rich Watery Essence Sunscreen]",
   "x_27": "x[Blueair Classic 480i]",
   "x_28": "x[Bose QuietComfort 35 Headphones]",
   "x_29": "x[Bose QuietComfort 35 II Wireless Headphones]",
   "x_30": "x[Bose SoundLink Color Bluetooth Speaker II]",
   "x_31": "x[Bose SoundLink Revolve+ Speaker]",
   "x_32": "x[Bose SoundSport Wireless Earbuds]",
   "x_33": "x[Bowflex SelectTech 1090 Adjustable Dumbbells]",
   "x_34": "x[Bowflex SelectTech 552 Dumbbells]",
   "x_35": "x[Breville Nespresso Creatista Plus]",
   "x_36": "x[Breville Smart Coffee Grinder Pro]",
   "x_37": "x[Breville Smart Grill]",
   "x_38": "x[Breville Smart Oven]",
   "x_39": "x[Calvin Klein Boxer Briefs]",
   "x_40": "x[Canon EOS R5 Camera]",
   "x_41": "x[Canon EOS Rebel T7i DSLR Camera]",
   "x_42": "x[Caudalie Vinoperfect Radiance Serum]",
   "x_43": "x[CeraVe Hydrating Facial Cleanser]",
   "x_44": "x[Champion Reverse Weave Hoodie]",
   "x_45": "x[Chanel No. 5 Perfume]",
   "x_46": "x[Charlotte Tilbury Magic Cream]",
   "x_47": "x[Clinique Dramatically Different Moisturizing Lotion]",
   "x_48": "x[Clinique Moisture Surge]",
   "x_49": "x[Columbia Fleece Jacket]",
   "x_50": "x[Crock-Pot 6-Quart Slow Cooker]",
   "x_51": "x[Cuisinart Coffee Center]",
   "x_52": "x[Cuisinart Custom 14-Cup Food Processor]",
   "x_53": "x[Cuisinart Griddler Deluxe]",
   "x_54": "x[De'Longhi Magnifica Espresso Machine]",
   "x_55": "x[Dr. Jart+ Cicapair Tiger Grass Color Correcting Treatment]",
   "x_56": "x[Drunk Elephant C-Firma Day Serum]",
   "x_57": "x[Dune by Frank Herbert]",
   "x_58": "x[Dyson Pure Cool Link]",
   "x_59": "x[Dyson Supersonic Hair Dryer]",
   "x_60": "x[Dyson V11 Vacuum]",
   "x_61": "x[Dyson V8 Absolute]",
   "x_62": "x[Educated by Tara Westover]",
   "x_63": "x[Estee Lauder Advanced Night Repair]",
   "x_64": "x[Eufy RoboVac 11S]",
   "x_65": "x[Fenty Beauty Killawatt Highlighter]",
   "x_66": "x[First Aid Beauty Ultra Repair Cream]",
   "x_67": "x[Fitbit Charge 5]",
   "x_68": "x[Fitbit Inspire 2]",
   "x_69": "x[Fitbit Luxe]",
   "x_70": "x[Fitbit Versa 3]",
   "x_71": "x[Forever 21 Graphic Tee]",
   "x_72": "x[Fresh Sugar Lip Treatment]",
   "x_73": "x[Gap 1969 Original Fit Jeans]",
   "x_74": "x[Gap Crewneck Sweatshirt]",
   "x_75": "x[Gap Essential Crewneck T-Shirt]",
   "x_76": "x[Gap High Rise Skinny Jeans]",
   "x_77": "x[Garmin Edge 530]",
   "x_78": "x[Garmin Fenix 6X Pro]",
   "x_79": "x[Garmin Forerunner 245]",
   "x_80": "x[Garmin Forerunner 945]",
   "x_81": "x[GlamGlow Supermud Clearing Treatment]",
   "x_82": "x[Glossier Boy Brow]",
   "x_83": "x[Glossier Cloud Paint]",
   "x_84": "x[GoPro HERO10 Black]",
   "x_85": "x[GoPro HERO9 Black]",
   "x_86": "x[Gone Girl by Gillian Flynn]",
   "x_87": "x[Google Nest Hub Max]",
   "x_88": "x[Google Nest Wifi Router]",
   "x_89": "x[Google Pixel 6 Pro]",
   "x_90": "x[Google Pixelbook Go]",
   "x_91": "x[H&M Slim Fit Jeans]",
   "x_92": "x[HP Spectre x360 Laptop]",
   "x_93": "x[Hamilton Beach FlexBrew Coffee Maker]",
   "x_94": "x[Hanes ComfortSoft T-Shirt]",
   "x_95": "x[Harry Potter and the Sorcerer's Stone]",
   "x_96": "x[Hydro Flask Standard Mouth Water Bottle]",
   "x_97": "x[Hydro Flask Wide Mouth Water Bottle]",
   "x_98": "x[Hyperice Hypervolt Massager]",
   "x_99": "x[Instant Pot Duo]",
   "x_100": "x[Instant Pot Duo Crisp]",
   "x_101": "x[Instant Pot Duo Evo Plus]",
   "x_102": "x[Instant Pot Duo Nova]",
   "x_103": "x[Instant Pot Ultra]",
   "x_104": "x[Keurig K-Elite Coffee Maker]",
   "x_105": "x[Keurig K-Mini Coffee Maker]",
   "x_106": "x[Kiehl's Midnight Recovery Concentrate]",
   "x_107": "x[Kindle Paperwhite]",
   "x_108": "x[KitchenAid Artisan Stand Mixer]",
   "x_109": "x[KitchenAid Stand Mixer]",
   "x_110": "x[L'Occitane Shea Butter Hand Cream]",
   "x_111": "x[L'Oreal Revitalift Serum]",
   "x_112": "x[LG OLED TV]",
   "x_113": "x[La Mer Cr_¨me de la Mer Moisturizer]",
   "x_114": "x[Lancome La Vie Est Belle]",
   "x_115": "x[Laneige Water Sleeping Mask]",
   "x_116": "x[Levi's 501 Jeans]",
   "x_117": "x[Levi's 511 Slim Fit Jeans]",
   "x_118": "x[Levi's Sherpa Trucker Jacket]",
   "x_119": "x[Levi's Trucker Jacket]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 83748.37000000001,
   "variables": [
    [
     "C0",
     6.0
    ],
    [
     "C1",
     7.0
    ],
    [
     "C2",
     5.0
    ],
    [
     "C3",
     4.0
    ],
    [
     "C4",
     6.0
    ],
    [
     "C5",
     6.0
    ],
    [
     "C6",
     3.0
    ],
    [
     "C7",
     3.0
    ],
    [
     "C8",
     5.0
    ],
    [
     "C9",
     2.0
    ],
    [
     "C10",
     5.0
    ],
    [
     "C11",
     3.0
    ],
    [
     "C12",
     6.0
    ],
    [
     "C13",
     3.0
    ],
    [
     "C14",
     2.0
    ],
    [
     "C15",
     2.0
    ],
    [
     "C16",
     3.0
    ],
    [
     "C17",
     2.0
    ],
    [
     "C18",
     2.0
    ],
    [
     "C19",
     3.0
    ],
    [
     "C20",
     4.0
    ],
    [
     "C21",
     3.0
    ],
    [
     "C22",
     6.0
    ],
    [
     "C23",
     5.0
    ],
    [
     "C24",
     6.0
    ],
    [
     "C25",
     2.0
    ],
    [
     "C26",
     3.0
    ],
    [
     "C27",
     2.0
    ],
    [
     "C28",
     2.0
    ],
    [
     "C29",
     2.0
    ],
    [
     "C30",
     5.0
    ],
    [
     "C31",
     3.0
    ],
    [
     "C32",
     2.0
    ],
    [
     "C33",
     2.0
    ],
    [
     "C34",
     2.0
    ],
    [
     "C35",
     2.0
    ],
    [
     "C36",
     3.0
    ],
    [
     "C37",
     2.0
    ],
    [
     "C38",
     7.0
    ],
    [
     "C39",
     2.0
    ],
    [
     "C40",
     2.0
    ],
    [
     "C41",
     2.0
    ],
    [
     "C42",
     3.0
    ],
    [
     "C43",
     5.0
    ],
    [
     "C44",
     2.0
    ],
    [
     "C45",
     2.0
    ],
    [
     "C46",
     2.0
    ],
    [
     "C47",
     2.0
    ],
    [
     "C48",
     6.0
    ],
    [
     "C49",
     3.0
    ],
    [
     "C50",
     3.0
    ],
    [
     "C51",
     2.0
    ],
    [
     "C52",
     2.0
    ],
    [
     "C53",
     2.0
    ],
    [
     "C54",
     2.0
    ],
    [
     "C55",
     2.0
    ],
    [
     "C56",
     6.0
    ],
    [
     "C57",
     2.0
    ],
    [
     "C58",
     5.0
    ],
    [
     "C59",
     2.0
    ],
    [
     "C60",
     2.0
    ],
    [
     "C61",
     4.0
    ],
    [
     "C62",
     2.0
    ],
    [
     "C63",
     5.0
    ],
    [
     "C64",
     2.0
    ],
    [
     "C65",
     3.0
    ],
    [
     "C66",
     3.0
    ],
    [
     "C67",
     3.0
    ],
    [
     "C68",
     3.0
    ],
    [
     "C69",
     5.0
    ],
    [
     "C70",
     7.0
    ],
    [
     "C71",
     2.0
    ],
    [
     "C72",
     5.0
    ],
    [
     "C73",
     6.0
    ],
    [
     "C74",
     8.0
    ],
    [
     "C75",
     5.0
    ],
    [
     "C76",
     3.0
    ],
    [
     "C77",
     2.0
    ],
    [
     "C78",
     2.0
    ],
    [
     "C79",
     5.0
    ],
    [
     "C80",
     2.0
    ],
    [
     "C81",
     3.0
    ],
    [
     "C82",
     2.0
    ],
    [
     "C83",
     4.0
    ],
    [
     "C84",
     2.0
    ],
    [
     "C85",
     3.0
    ],
    [
     "C86",
     3.0
    ],
    [
     "C87",
     2.0
    ],
    [
     "C88",
     2.0
    ],
    [
     "C89",
     2.0
    ],
    [
     "C90",
     5.0
    ],
    [
     "C91",
     2.0
    ],
    [
     "C92",
     2.0
    ],
    [
     "C93",
     15.0
    ],
    [
     "C94",
     4.0
    ],
    [
     "C95",
     5.0
    ],
    [
     "C96",
     6.0
    ],
    [
     "C97",
     2.0
    ],
    [
     "C98",
     4.0
    ],
    [
     "C99",
     2.0
    ],
    [
     "C100",
     3.0
    ],
    [
     "C101",
     2.0
    ],
    [
     "C102",
     2.0
    ],
    [
     "C103",
     5.0
    ],
    [
     "C104",
     3.0
    ],
    [
     "C105",
     2.0
    ],
    [
     "C106",
     3.0
    ],
    [
     "C107",
     2.0
    ],
    [
     "C108",
     2.0
    ],
    [
     "C109",
     3.0
    ],
    [
     "C110",
     3.0
    ],
    [
     "C111",
     3.0
    ],
    [
     "C112",
     2.0
    ],
    [
     "C113",
     2.0
    ],
    [
     "C114",
     2.0
    ],
    [
     "C115",
     5.0
    ],
    [
     "C116",
     5.0
    ],
    [
     "C117",
     3.0
    ],
    [
     "C118",
     3.0
    ]
   ],
   "error": null
  }
 },
 "OR-014": {
  "source_code_sha256": "f8b19beae7ed22508e91922741f34f59b6748abbab739c82c511c48129b9a935",
  "original_solution_sha256": "62176e138bbf4a4e705142dd1b3963137c511d7cb0c41dfb54ed2dce188be1cb",
  "bindings": [
   [
    "C0",
    "x[bbq_ckn_l]"
   ],
   [
    "C1",
    "x[bbq_ckn_m]"
   ],
   [
    "C2",
    "x[bbq_ckn_s]"
   ],
   [
    "C3",
    "x[big_meat_s]"
   ],
   [
    "C4",
    "x[brie_carre_s]"
   ],
   [
    "C5",
    "x[calabrese_l]"
   ],
   [
    "C6",
    "x[calabrese_m]"
   ],
   [
    "C7",
    "x[calabrese_s]"
   ],
   [
    "C8",
    "x[cali_ckn_l]"
   ],
   [
    "C9",
    "x[cali_ckn_m]"
   ],
   [
    "C10",
    "x[cali_ckn_s]"
   ],
   [
    "C11",
    "x[ckn_alfredo_l]"
   ],
   [
    "C12",
    "x[ckn_alfredo_m]"
   ],
   [
    "C13",
    "x[ckn_alfredo_s]"
   ],
   [
    "C14",
    "x[ckn_pesto_l]"
   ],
   [
    "C15",
    "x[ckn_pesto_m]"
   ],
   [
    "C16",
    "x[ckn_pesto_s]"
   ],
   [
    "C17",
    "x[classic_dlx_l]"
   ],
   [
    "C18",
    "x[classic_dlx_m]"
   ],
   [
    "C19",
    "x[classic_dlx_s]"
   ],
   [
    "C20",
    "x[five_cheese_l]"
   ],
   [
    "C21",
    "x[four_cheese_l]"
   ],
   [
    "C22",
    "x[four_cheese_m]"
   ],
   [
    "C23",
    "x[green_garden_l]"
   ],
   [
    "C24",
    "x[green_garden_m]"
   ],
   [
    "C25",
    "x[green_garden_s]"
   ],
   [
    "C26",
    "x[hawaiian_l]"
   ],
   [
    "C27",
    "x[hawaiian_m]"
   ],
   [
    "C28",
    "x[hawaiian_s]"
   ],
   [
    "C29",
    "x[ital_cpcllo_l]"
   ],
   [
    "C30",
    "x[ital_cpcllo_m]"
   ],
   [
    "C31",
    "x[ital_cpcllo_s]"
   ],
   [
    "C32",
    "x[ital_supr_l]"
   ],
   [
    "C33",
    "x[ital_supr_m]"
   ],
   [
    "C34",
    "x[ital_supr_s]"
   ],
   [
    "C35",
    "x[ital_veggie_l]"
   ],
   [
    "C36",
    "x[ital_veggie_m]"
   ],
   [
    "C37",
    "x[ital_veggie_s]"
   ],
   [
    "C38",
    "x[mediterraneo_l]"
   ],
   [
    "C39",
    "x[mediterraneo_m]"
   ],
   [
    "C40",
    "x[mediterraneo_s]"
   ],
   [
    "C41",
    "x[mexicana_l]"
   ],
   [
    "C42",
    "x[mexicana_m]"
   ],
   [
    "C43",
    "x[mexicana_s]"
   ],
   [
    "C44",
    "x[napolitana_l]"
   ],
   [
    "C45",
    "x[napolitana_m]"
   ],
   [
    "C46",
    "x[napolitana_s]"
   ],
   [
    "C47",
    "x[pep_msh_pep_l]"
   ],
   [
    "C48",
    "x[pep_msh_pep_m]"
   ],
   [
    "C49",
    "x[pep_msh_pep_s]"
   ],
   [
    "C50",
    "x[pepperoni_l]"
   ],
   [
    "C51",
    "x[pepperoni_m]"
   ],
   [
    "C52",
    "x[pepperoni_s]"
   ],
   [
    "C53",
    "x[peppr_salami_l]"
   ],
   [
    "C54",
    "x[peppr_salami_m]"
   ],
   [
    "C55",
    "x[peppr_salami_s]"
   ],
   [
    "C56",
    "x[prsc_argla_l]"
   ],
   [
    "C57",
    "x[prsc_argla_m]"
   ],
   [
    "C58",
    "x[prsc_argla_s]"
   ],
   [
    "C59",
    "x[sicilian_l]"
   ],
   [
    "C60",
    "x[sicilian_m]"
   ],
   [
    "C61",
    "x[sicilian_s]"
   ],
   [
    "C62",
    "x[soppressata_l]"
   ],
   [
    "C63",
    "x[soppressata_m]"
   ],
   [
    "C64",
    "x[soppressata_s]"
   ],
   [
    "C65",
    "x[southw_ckn_l]"
   ],
   [
    "C66",
    "x[southw_ckn_m]"
   ],
   [
    "C67",
    "x[southw_ckn_s]"
   ],
   [
    "C68",
    "x[spicy_ital_l]"
   ],
   [
    "C69",
    "x[spicy_ital_m]"
   ],
   [
    "C70",
    "x[spicy_ital_s]"
   ],
   [
    "C71",
    "x[spin_pesto_l]"
   ],
   [
    "C72",
    "x[spin_pesto_m]"
   ],
   [
    "C73",
    "x[spin_pesto_s]"
   ],
   [
    "C74",
    "x[spinach_fet_l]"
   ],
   [
    "C75",
    "x[spinach_fet_m]"
   ],
   [
    "C76",
    "x[spinach_fet_s]"
   ],
   [
    "C77",
    "x[spinach_supr_l]"
   ],
   [
    "C78",
    "x[spinach_supr_m]"
   ],
   [
    "C79",
    "x[spinach_supr_s]"
   ],
   [
    "C80",
    "x[thai_ckn_l]"
   ],
   [
    "C81",
    "x[thai_ckn_m]"
   ],
   [
    "C82",
    "x[thai_ckn_s]"
   ],
   [
    "C83",
    "x[the_greek_l]"
   ],
   [
    "C84",
    "x[the_greek_m]"
   ],
   [
    "C85",
    "x[the_greek_s]"
   ],
   [
    "C86",
    "x[the_greek_xl]"
   ],
   [
    "C87",
    "x[the_greek_xxl]"
   ],
   [
    "C88",
    "x[veggie_veg_l]"
   ],
   [
    "C89",
    "x[veggie_veg_m]"
   ],
   [
    "C90",
    "x[veggie_veg_s]"
   ],
   [
    "C91",
    "x[fried_duck_l]"
   ],
   [
    "C92",
    "x[fried_duck_m]"
   ],
   [
    "C93",
    "x[fried_duck_s]"
   ],
   [
    "C94",
    "x[mapo_Tofu_l]"
   ],
   [
    "C95",
    "x[mapo_Tofu_m]"
   ],
   [
    "C96",
    "x[mapo_Tofu_s]"
   ],
   [
    "C97",
    "x[sea_food_l]"
   ],
   [
    "C98",
    "x[sea_food_m]"
   ],
   [
    "C99",
    "x[sea_food_s]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[bbq_ckn_l]",
   "x_2": "x[bbq_ckn_m]",
   "x_3": "x[bbq_ckn_s]",
   "x_4": "x[big_meat_s]",
   "x_5": "x[brie_carre_s]",
   "x_6": "x[calabrese_l]",
   "x_7": "x[calabrese_m]",
   "x_8": "x[calabrese_s]",
   "x_9": "x[cali_ckn_l]",
   "x_10": "x[cali_ckn_m]",
   "x_11": "x[cali_ckn_s]",
   "x_12": "x[ckn_alfredo_l]",
   "x_13": "x[ckn_alfredo_m]",
   "x_14": "x[ckn_alfredo_s]",
   "x_15": "x[ckn_pesto_l]",
   "x_16": "x[ckn_pesto_m]",
   "x_17": "x[ckn_pesto_s]",
   "x_18": "x[classic_dlx_l]",
   "x_19": "x[classic_dlx_m]",
   "x_20": "x[classic_dlx_s]",
   "x_21": "x[five_cheese_l]",
   "x_22": "x[four_cheese_l]",
   "x_23": "x[four_cheese_m]",
   "x_24": "x[green_garden_l]",
   "x_25": "x[green_garden_m]",
   "x_26": "x[green_garden_s]",
   "x_27": "x[hawaiian_l]",
   "x_28": "x[hawaiian_m]",
   "x_29": "x[hawaiian_s]",
   "x_30": "x[ital_cpcllo_l]",
   "x_31": "x[ital_cpcllo_m]",
   "x_32": "x[ital_cpcllo_s]",
   "x_33": "x[ital_supr_l]",
   "x_34": "x[ital_supr_m]",
   "x_35": "x[ital_supr_s]",
   "x_36": "x[ital_veggie_l]",
   "x_37": "x[ital_veggie_m]",
   "x_38": "x[ital_veggie_s]",
   "x_39": "x[mediterraneo_l]",
   "x_40": "x[mediterraneo_m]",
   "x_41": "x[mediterraneo_s]",
   "x_42": "x[mexicana_l]",
   "x_43": "x[mexicana_m]",
   "x_44": "x[mexicana_s]",
   "x_45": "x[napolitana_l]",
   "x_46": "x[napolitana_m]",
   "x_47": "x[napolitana_s]",
   "x_48": "x[pep_msh_pep_l]",
   "x_49": "x[pep_msh_pep_m]",
   "x_50": "x[pep_msh_pep_s]",
   "x_51": "x[pepperoni_l]",
   "x_52": "x[pepperoni_m]",
   "x_53": "x[pepperoni_s]",
   "x_54": "x[peppr_salami_l]",
   "x_55": "x[peppr_salami_m]",
   "x_56": "x[peppr_salami_s]",
   "x_57": "x[prsc_argla_l]",
   "x_58": "x[prsc_argla_m]",
   "x_59": "x[prsc_argla_s]",
   "x_60": "x[sicilian_l]",
   "x_61": "x[sicilian_m]",
   "x_62": "x[sicilian_s]",
   "x_63": "x[soppressata_l]",
   "x_64": "x[soppressata_m]",
   "x_65": "x[soppressata_s]",
   "x_66": "x[southw_ckn_l]",
   "x_67": "x[southw_ckn_m]",
   "x_68": "x[southw_ckn_s]",
   "x_69": "x[spicy_ital_l]",
   "x_70": "x[spicy_ital_m]",
   "x_71": "x[spicy_ital_s]",
   "x_72": "x[spin_pesto_l]",
   "x_73": "x[spin_pesto_m]",
   "x_74": "x[spin_pesto_s]",
   "x_75": "x[spinach_fet_l]",
   "x_76": "x[spinach_fet_m]",
   "x_77": "x[spinach_fet_s]",
   "x_78": "x[spinach_supr_l]",
   "x_79": "x[spinach_supr_m]",
   "x_80": "x[spinach_supr_s]",
   "x_81": "x[thai_ckn_l]",
   "x_82": "x[thai_ckn_m]",
   "x_83": "x[thai_ckn_s]",
   "x_84": "x[the_greek_l]",
   "x_85": "x[the_greek_m]",
   "x_86": "x[the_greek_s]",
   "x_87": "x[the_greek_xl]",
   "x_88": "x[the_greek_xxl]",
   "x_89": "x[veggie_veg_l]",
   "x_90": "x[veggie_veg_m]",
   "x_91": "x[veggie_veg_s]",
   "x_92": "x[fried_duck_l]",
   "x_93": "x[fried_duck_m]",
   "x_94": "x[fried_duck_s]",
   "x_95": "x[mapo_Tofu_l]",
   "x_96": "x[mapo_Tofu_m]",
   "x_97": "x[mapo_Tofu_s]",
   "x_98": "x[sea_food_l]",
   "x_99": "x[sea_food_m]",
   "x_100": "x[sea_food_s]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1794269.0,
   "variables": [
    [
     "C0",
     1960.0
    ],
    [
     "C1",
     1884.0
    ],
    [
     "C2",
     963.0
    ],
    [
     "C3",
     3729.0
    ],
    [
     "C4",
     970.0
    ],
    [
     "C5",
     550.0
    ],
    [
     "C6",
     1116.0
    ],
    [
     "C7",
     198.0
    ],
    [
     "C8",
     1823.0
    ],
    [
     "C9",
     1858.0
    ],
    [
     "C10",
     992.0
    ],
    [
     "C11",
     375.0
    ],
    [
     "C12",
     1400.0
    ],
    [
     "C13",
     192.0
    ],
    [
     "C14",
     791.0
    ],
    [
     "C15",
     550.0
    ],
    [
     "C16",
     593.0
    ],
    [
     "C17",
     944.0
    ],
    [
     "C18",
     2340.0
    ],
    [
     "C19",
     1586.0
    ],
    [
     "C20",
     2769.0
    ],
    [
     "C21",
     2589.0
    ],
    [
     "C22",
     1163.0
    ],
    [
     "C23",
     189.0
    ],
    [
     "C24",
     602.0
    ],
    [
     "C25",
     1193.0
    ],
    [
     "C26",
     1815.0
    ],
    [
     "C27",
     956.0
    ],
    [
     "C28",
     2021.0
    ],
    [
     "C29",
     1447.0
    ],
    [
     "C30",
     803.0
    ],
    [
     "C31",
     602.0
    ],
    [
     "C32",
     1482.0
    ],
    [
     "C33",
     1861.0
    ],
    [
     "C34",
     390.0
    ],
    [
     "C35",
     380.0
    ],
    [
     "C36",
     969.0
    ],
    [
     "C37",
     607.0
    ],
    [
     "C38",
     734.0
    ],
    [
     "C39",
     546.0
    ],
    [
     "C40",
     577.0
    ],
    [
     "C41",
     1711.0
    ],
    [
     "C42",
     907.0
    ],
    [
     "C43",
     322.0
    ],
    [
     "C44",
     1123.0
    ],
    [
     "C45",
     853.0
    ],
    [
     "C46",
     939.0
    ],
    [
     "C47",
     765.0
    ],
    [
     "C48",
     788.0
    ],
    [
     "C49",
     1148.0
    ],
    [
     "C50",
     1440.0
    ],
    [
     "C51",
     1857.0
    ],
    [
     "C52",
     1490.0
    ],
    [
     "C53",
     1376.0
    ],
    [
     "C54",
     852.0
    ],
    [
     "C55",
     640.0
    ],
    [
     "C56",
     858.0
    ],
    [
     "C57",
     1183.0
    ],
    [
     "C58",
     844.0
    ],
    [
     "C59",
     1209.0
    ],
    [
     "C60",
     1135.0
    ],
    [
     "C61",
     1482.0
    ],
    [
     "C62",
     806.0
    ],
    [
     "C63",
     536.0
    ],
    [
     "C64",
     576.0
    ],
    [
     "C65",
     2009.0
    ],
    [
     "C66",
     1060.0
    ],
    [
     "C67",
     733.0
    ],
    [
     "C68",
     2198.0
    ],
    [
     "C69",
     808.0
    ],
    [
     "C70",
     807.0
    ],
    [
     "C71",
     563.0
    ],
    [
     "C72",
     563.0
    ],
    [
     "C73",
     801.0
    ],
    [
     "C74",
     882.0
    ],
    [
     "C75",
     1120.0
    ],
    [
     "C76",
     876.0
    ],
    [
     "C77",
     563.0
    ],
    [
     "C78",
     533.0
    ],
    [
     "C79",
     794.0
    ],
    [
     "C80",
     2776.0
    ],
    [
     "C81",
     955.0
    ],
    [
     "C82",
     956.0
    ],
    [
     "C83",
     510.0
    ],
    [
     "C84",
     560.0
    ],
    [
     "C85",
     604.0
    ],
    [
     "C86",
     1096.0
    ],
    [
     "C87",
     56.0
    ],
    [
     "C88",
     850.0
    ],
    [
     "C89",
     1265.0
    ],
    [
     "C90",
     921.0
    ],
    [
     "C91",
     733.0
    ],
    [
     "C92",
     2198.0
    ],
    [
     "C93",
     808.0
    ],
    [
     "C94",
     563.0
    ],
    [
     "C95",
     801.0
    ],
    [
     "C96",
     882.0
    ],
    [
     "C97",
     1120.0
    ],
    [
     "C98",
     794.0
    ],
    [
     "C99",
     2776.0
    ]
   ],
   "error": null
  }
 },
 "OR-017": {
  "source_code_sha256": "cb40373f7df978ea91b26ba11264bd9c29325ae4ee5ddd7945e430bfb1f97cf3",
  "original_solution_sha256": "1adb3ecb4c1e88130f315d3cea4a93a6a9c6e0aed6138a7e474088e6485d8aaa",
  "bindings": [
   [
    "C0",
    "x[ZZ2AO]"
   ],
   [
    "C1",
    "x[ZZDW7]"
   ],
   [
    "C2",
    "x[ZZM1A]"
   ],
   [
    "C3",
    "x[ZZNC5]"
   ],
   [
    "C4",
    "x[ZZX6K]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[ZZ2AO]",
   "x_2": "x[ZZDW7]",
   "x_3": "x[ZZM1A]",
   "x_4": "x[ZZNC5]",
   "x_5": "x[ZZX6K]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 2015.08,
   "variables": [
    [
     "C0",
     2.0
    ],
    [
     "C1",
     4.0
    ],
    [
     "C2",
     82.0
    ],
    [
     "C3",
     2.0
    ],
    [
     "C4",
     2.0
    ]
   ],
   "error": null
  }
 },
 "OR-021": {
  "source_code_sha256": "8a3d8f01a5e4ab1e5bc8c9f05a835eafb226d065f23dba2bcb6d4d46c987f48d",
  "original_solution_sha256": "adbf0a6c3c119ba3f6846ed025969f53556bb0da6ebc832920cabda62c22537e",
  "bindings": [
   [
    "C0",
    "x[ 3 Colors New Fashion Summer Ladies Casual Jumpsuit Long Suspender Overalls Bib Pants]"
   ],
   [
    "C1",
    "x[ Men's Casual Workout Athletic Gym Jersey Shorts Elastic Waist Drawstring Summer Training Running Knee Length Shorts with Zipper Pocket]"
   ],
   [
    "C2",
    "x[ New Fashion Autumn Summer Women's Long Sleeve V Neck Long Dress Floral Print Split Maxi Dress Holiday Party Beach Sundress Evening Dresses]"
   ],
   [
    "C3",
    "x[ New The New Men's Stitching Design Jogging Sports Cropped Trousers]"
   ],
   [
    "C4",
    "x[ Plus Size S-5XL Women Summer Tops Casual V-Neck Short Sleeve Shirts Ladies Cotton Loose T Shirt Candy Color Lady Pullovers Blouse]"
   ],
   [
    "C5",
    "x[ Plus Size Women Halter Striped Wide Leg Pants Casual Jumpsuit Rompers Fashion Shorts ]"
   ],
   [
    "C6",
    "x[ Spring/Summer Fashion Women \"honey\"Letter Print Sleeveless Shirt Sexy Embroidered Bodycon Vest Knitted Cotton Vest Casual Tank Top ]"
   ],
   [
    "C7",
    "x[ Summer Students Style Bodycon Rompers Womens Slim Fit Jumpsuit Chest Zipper Contrast Color Short Sleeve V-neck Shorts Bodysuit]"
   ],
   [
    "C8",
    "x[ Summer Women s Fashion Lace Up Tie Pants Plus Size Casual High Waist Short Pants(S-5XL)]"
   ],
   [
    "C9",
    "x[ Women's Fashion Graphic \\Don't Flatter Yourself....\\Tee for Women Summer Casual Tee T Shirts for Girls]"
   ],
   [
    "C10",
    "x[ XS - XXL Fashion Women's Summer Casual T - Shirt Love Gesture Printed Short Sleeve Blouse XS - XXL]"
   ],
   [
    "C11",
    "x['Let That Shit Go' Women Graphic Tee Casual Cotton Short Sleeve T-shirts Bohemia Style Mandala Namaste Printed Top Blouse]"
   ],
   [
    "C12",
    "x[(S-5XL) Women Fashion Summer Double-Layer Sports Shorts Quick-Drying Yoga Sports Leggings Fitness Shorts Plus Size]"
   ],
   [
    "C13",
    "x[(S-7XL) Women Camisole Summer Printed Vest Floral Sleeveless Vest V-neck Shirt Plus Size]"
   ],
   [
    "C14",
    "x[(US Size) Cotton Graphic Tees for Women / Girls: 4 Colors, Spring Summer Tee, Cute Funny Short Sleeve T-Shirt, Loose Blouse Tops]"
   ],
   [
    "C15",
    "x[1 Pcs Swimming Chair/Bed Swimming Pool Seat Inflatable Lazy Bed Lounge Chair Air Mattress Floating Bed/Chair with Net Foldable for Swimming Pool/Beach Water Relaxation]"
   ],
   [
    "C16",
    "x[10 Color Women Summer Shorts Lace Up Elastic Waistband Loose Panties Plus Size S-6XL]"
   ],
   [
    "C17",
    "x[10/20/30ml Slimming Firming Anti-Cellulite Massage Cream Fat Burning Weight Losing Moisturizer Gel for Shaping Waist]"
   ],
   [
    "C18",
    "x[10/30/50ml Hair Removal Spray Super Natural Painless Permanent Depilatory Cream]"
   ],
   [
    "C19",
    "x[100%Cotton ZANZEA S-5XL NEW Vintage Women Strap Dungaree Jumpsuit Casual Loose Trousers Overalls Rompers Jumpsuit]"
   ],
   [
    "C20",
    "x[100pcs Summer Disposable Sweat Pad Perspiration Absorbing Guard Underarm Armpit Sweat Pad Pure Antiperspirant Adhesive Underarm Pads]"
   ],
   [
    "C21",
    "x[11 Colors V Neck Women Sleeveless Tops Halter Neck Plus Size Chiffon CropTops(S~5XL)]"
   ],
   [
    "C22",
    "x[16 Color Fashion Women Casual Sleeveless Camisoles Loose T-shirts with Ziper Deep V-neck Blouses Ladies Chiffon Shirts Tank Tops ]"
   ],
   [
    "C23",
    "x[170cm Inflatable Spray Water Cushion Summer Kids Play Water Mat Lawn Games Pad Sprinkler Play Toys Outdoor Tub Swiming Pool ]"
   ],
   [
    "C24",
    "x[18 Styles Women Summer Sexy Printing Buttons Off Shoulder Sleeveless Dress Princess Dress Spaghetti Straps Dresses]"
   ],
   [
    "C25",
    "x[1PCS Swimwear Monokini Swimsuit Backless Bodysuit Women Swimsuit]"
   ],
   [
    "C26",
    "x[1Set Lace Bikini Diamond Swimsuit Crystal Women Swimwear Nude Bikinis Brazilian Rhinestone Beachwear Push Up Bikini]"
   ],
   [
    "C27",
    "x[1pcs Men's Running Shorts 2 in 1 Sports Jogging Fitness Shorts Summer Training Quick Dry Short Gym Shorts Sport Pants]"
   ],
   [
    "C28",
    "x[20 Pcs Fashion Comfortable Short Socks Candy Color Invisible Silicone Non-slip Socks Slippers Socks]"
   ],
   [
    "C29",
    "x[20/10pcs Women Ankle Invisible No Show Nonslip Loafer Boat Liner Cotton Socks Comfortable Socks Women Shoes Accessories  (Application Size: 33-43)]"
   ],
   [
    "C30",
    "x[2017 Women Ladies Summer Dress Sleeveless Casual Sexy Floral Print Beach Dress Fashion Spaghetti Strap Mini Short Dress]"
   ],
   [
    "C31",
    "x[2018 6 Color Summer Womenâ__s New Fashion Sexy Cute Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Slim Bodycon Off The Shoulder Short Sleeve Blouse Casual Cotton Outdoor Tops Plus Size S-XXXXL]"
   ],
   [
    "C32",
    "x[2018 Black Flora Flower Printed Short Dress Women]"
   ],
   [
    "C33",
    "x[2018 Fashion Summer Dress Women Sexy Dresses V Neck Backless Lace Stitching Dress Beach Dresses White Dress]"
   ],
   [
    "C34",
    "x[2018 Hot Selling Spring Summer Women Flared Causal Trousers Loose Pants Drawstring Elastic Waist Middle Waisted  Wide Leg Pants]"
   ],
   [
    "C35",
    "x[2018 New Fashion Women Bikini Set Push-up Padded Bra Swimsuit -Swimwear]"
   ],
   [
    "C36",
    "x[2018 New Fashion Women Casual Playsuit Ladies Jumpsuit Romper Summer Floral Playsuit Brand New (3 Colors)]"
   ],
   [
    "C37",
    "x[2018 New Fashion Women's Tops Sexy Strappy Sleeveless Lace Crop Tops]"
   ],
   [
    "C38",
    "x[2018 New Summer Women Sexy V-neck Bandage T-shirts Camouflage Printed Short Sleeve Plus size Tops Tee(S-4XL)]"
   ],
   [
    "C39",
    "x[2018 New The new cross-strait national wind body bikini Swimsuit  Swimwear LBT]"
   ],
   [
    "C40",
    "x[2018 New ZANZEA Beach Romper Women Jumpsuits Front Zipper Sleeveless Sexy Playsuits]"
   ],
   [
    "C41",
    "x[2018 Plus Size Summer Women Fashion Sexy V-neck Lace Up Plaid Blouse Tops Irregular Short Sleeve Shirtï¼_S-5XLï¼_]"
   ],
   [
    "C42",
    "x[2018 Summer Fashion Women Casual Camouflage Tank Top Sleeveless O-neck Vest]"
   ],
   [
    "C43",
    "x[2018 Summer Fashion Women Tank Tops Sexy Women Sleeveless Crop Tops Casual Style Women Cotton Print Lace Stitching Irregular Blouse Topsï¼_S-5XL)]"
   ],
   [
    "C44",
    "x[2018 Summer New Women Fashion Sexy Sleeveless Tank Top Floral Print Casual Women Cotton T-shirts Plus Size Women Vestï¼_S-5XLï¼_]"
   ],
   [
    "C45",
    "x[2018 Summer New Women Long Skirts Solid Sexy Split Pencil Skirts]"
   ],
   [
    "C46",
    "x[2018 Summer Women Print Top Fashion Women Casual Army Camo Camouflage Tank Top Sleeveless O-neck Slim T-Shirts Plus Size S-XXXXXL]"
   ],
   [
    "C47",
    "x[2018 Summer Women Sexy Off Shoulder Shirt Casual Half Sleeve Top Ladies Fashion Printing Loose T-shirts Plus Size Cotton Blouse XS-5XL]"
   ],
   [
    "C48",
    "x[2018 Summer Womenâ__s New Fashion Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Casual Sexy Cotton Tops]"
   ],
   [
    "C49",
    "x[2018 T Shirts + Shorts Brand Clothing Tshirt Men Homme Letter Printed Basketball Running Sports Set T-shirt Suit Male]"
   ],
   [
    "C50",
    "x[2018 Women Cool T-shirt Funny 3d Tshirt Print Two Cat Short Sleeve Summer Tops Tees Teen Graphic Tee Cute Shirt Funny Gifts]"
   ],
   [
    "C51",
    "x[2018 Women Fashion Stretchy Camisole  Spaghetti Strap Long Tank Top Slip Summer Fashion Floral Mini Dress 7 Color SIZE XXL]"
   ],
   [
    "C52",
    "x[2018 Women Fashion Summer Sleeveless Maxi Halter Dresses Solid Color Halter Maxi Dress with Halter Tie and Pockets]"
   ],
   [
    "C53",
    "x[2018 Women Ladies Fashion Crocheted Lace Summer Dress Maxi Dress Long Dress]"
   ],
   [
    "C54",
    "x[2018 Women Ladies Fashion Musical Note Asymmetric Tank Top]"
   ],
   [
    "C55",
    "x[2018 Women Summer Casual Solid Color Loose Sleeveless Beach Tank Top A-line Pocket Dress Sexy Deep V-neck Short Club Party Mini Halter Midi Dresses Robe Ete Femme Knee Length Pleated Skirts Ladies Fashion Swing Cotton T-Shirt Dress Plus Size S-6XL]"
   ],
   [
    "C56",
    "x[2018 Women fashion Summer Lace Patchwork Tank Tops Casual Sleeveless Tops Vest Blouse (S-5XL) Plus Size]"
   ],
   [
    "C57",
    "x[2019 Fashion Family matching swimwear beachwear mommy and me swimsuit mother daughter father son clothes dresses high waist bikini look mum]"
   ],
   [
    "C58",
    "x[2019 Fashion Women Summer V-Neck Sleeveless Collect Waist Boho Print Maxi Long Dress S-XXXXXL]"
   ],
   [
    "C59",
    "x[2019 Men's Summer Cool Jogging Shorts]"
   ],
   [
    "C60",
    "x[2019 NEW Men's Short Sleeve T-Shirt Fitness Round Neck Casual Men's Zippers Summer T-Shirt]"
   ],
   [
    "C61",
    "x[2019 New Fashion Spring Summer Women's Long Pants Short Sleeves Off Shoulder Jumpsuit]"
   ],
   [
    "C62",
    "x[2019 New Fashion Summer Women Casual Dress Round Neck Loose Big Swing Skirt Sleeveless Soild Color Beach dress]"
   ],
   [
    "C63",
    "x[2019 New Fashion Women Blouse Tops Sleeveless Spaghetti Strap Criss Cross V-neck T Shirt Tops Summer Casual Blouse Shirts]"
   ],
   [
    "C64",
    "x[2019 New Fashion Women Casual Shorts Suit Summer Tie-Dye Print Halter Bandage Sleeveless Backless Crop Top And Elastic Waist Shorts Pants Two Piece Set]"
   ],
   [
    "C65",
    "x[2019 New Fashion Women Loose Round Neck Dress Short Sleeve Solid Color Casual Dress]"
   ],
   [
    "C66",
    "x[2019 New Fashion Women Summer Funny Penguin print T-Shirts Casual Short Sleeve O-Neck Tops Loose Cotton Cute Cool Tee Plus Size S-5XL 4 Colors]"
   ],
   [
    "C67",
    "x[2019 New Fashion Women's Slim Plus Size Maxi Dress Ink Printing Spaghetti Strap Dress]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[ 3 Colors New Fashion Summer Ladies Casual Jumpsuit Long Suspender Overalls Bib Pants]",
   "x_2": "x[ Men's Casual Workout Athletic Gym Jersey Shorts Elastic Waist Drawstring Summer Training Running Knee Length Shorts with Zipper Pocket]",
   "x_3": "x[ New Fashion Autumn Summer Women's Long Sleeve V Neck Long Dress Floral Print Split Maxi Dress Holiday Party Beach Sundress Evening Dresses]",
   "x_4": "x[ New The New Men's Stitching Design Jogging Sports Cropped Trousers]",
   "x_5": "x[ Plus Size S-5XL Women Summer Tops Casual V-Neck Short Sleeve Shirts Ladies Cotton Loose T Shirt Candy Color Lady Pullovers Blouse]",
   "x_6": "x[ Plus Size Women Halter Striped Wide Leg Pants Casual Jumpsuit Rompers Fashion Shorts ]",
   "x_7": "x[ Spring/Summer Fashion Women \"honey\"Letter Print Sleeveless Shirt Sexy Embroidered Bodycon Vest Knitted Cotton Vest Casual Tank Top ]",
   "x_8": "x[ Summer Students Style Bodycon Rompers Womens Slim Fit Jumpsuit Chest Zipper Contrast Color Short Sleeve V-neck Shorts Bodysuit]",
   "x_9": "x[ Summer Women s Fashion Lace Up Tie Pants Plus Size Casual High Waist Short Pants(S-5XL)]",
   "x_10": "x[ Women's Fashion Graphic \\Don't Flatter Yourself....\\Tee for Women Summer Casual Tee T Shirts for Girls]",
   "x_11": "x[ XS - XXL Fashion Women's Summer Casual T - Shirt Love Gesture Printed Short Sleeve Blouse XS - XXL]",
   "x_12": "x['Let That Shit Go' Women Graphic Tee Casual Cotton Short Sleeve T-shirts Bohemia Style Mandala Namaste Printed Top Blouse]",
   "x_13": "x[(S-5XL) Women Fashion Summer Double-Layer Sports Shorts Quick-Drying Yoga Sports Leggings Fitness Shorts Plus Size]",
   "x_14": "x[(S-7XL) Women Camisole Summer Printed Vest Floral Sleeveless Vest V-neck Shirt Plus Size]",
   "x_15": "x[(US Size) Cotton Graphic Tees for Women / Girls: 4 Colors, Spring Summer Tee, Cute Funny Short Sleeve T-Shirt, Loose Blouse Tops]",
   "x_16": "x[1 Pcs Swimming Chair/Bed Swimming Pool Seat Inflatable Lazy Bed Lounge Chair Air Mattress Floating Bed/Chair with Net Foldable for Swimming Pool/Beach Water Relaxation]",
   "x_17": "x[10 Color Women Summer Shorts Lace Up Elastic Waistband Loose Panties Plus Size S-6XL]",
   "x_18": "x[10/20/30ml Slimming Firming Anti-Cellulite Massage Cream Fat Burning Weight Losing Moisturizer Gel for Shaping Waist]",
   "x_19": "x[10/30/50ml Hair Removal Spray Super Natural Painless Permanent Depilatory Cream]",
   "x_20": "x[100%Cotton ZANZEA S-5XL NEW Vintage Women Strap Dungaree Jumpsuit Casual Loose Trousers Overalls Rompers Jumpsuit]",
   "x_21": "x[100pcs Summer Disposable Sweat Pad Perspiration Absorbing Guard Underarm Armpit Sweat Pad Pure Antiperspirant Adhesive Underarm Pads]",
   "x_22": "x[11 Colors V Neck Women Sleeveless Tops Halter Neck Plus Size Chiffon CropTops(S~5XL)]",
   "x_23": "x[16 Color Fashion Women Casual Sleeveless Camisoles Loose T-shirts with Ziper Deep V-neck Blouses Ladies Chiffon Shirts Tank Tops ]",
   "x_24": "x[170cm Inflatable Spray Water Cushion Summer Kids Play Water Mat Lawn Games Pad Sprinkler Play Toys Outdoor Tub Swiming Pool ]",
   "x_25": "x[18 Styles Women Summer Sexy Printing Buttons Off Shoulder Sleeveless Dress Princess Dress Spaghetti Straps Dresses]",
   "x_26": "x[1PCS Swimwear Monokini Swimsuit Backless Bodysuit Women Swimsuit]",
   "x_27": "x[1Set Lace Bikini Diamond Swimsuit Crystal Women Swimwear Nude Bikinis Brazilian Rhinestone Beachwear Push Up Bikini]",
   "x_28": "x[1pcs Men's Running Shorts 2 in 1 Sports Jogging Fitness Shorts Summer Training Quick Dry Short Gym Shorts Sport Pants]",
   "x_29": "x[20 Pcs Fashion Comfortable Short Socks Candy Color Invisible Silicone Non-slip Socks Slippers Socks]",
   "x_30": "x[20/10pcs Women Ankle Invisible No Show Nonslip Loafer Boat Liner Cotton Socks Comfortable Socks Women Shoes Accessories  (Application Size: 33-43)]",
   "x_31": "x[2017 Women Ladies Summer Dress Sleeveless Casual Sexy Floral Print Beach Dress Fashion Spaghetti Strap Mini Short Dress]",
   "x_32": "x[2018 6 Color Summer Womenâ__s New Fashion Sexy Cute Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Slim Bodycon Off The Shoulder Short Sleeve Blouse Casual Cotton Outdoor Tops Plus Size S-XXXXL]",
   "x_33": "x[2018 Black Flora Flower Printed Short Dress Women]",
   "x_34": "x[2018 Fashion Summer Dress Women Sexy Dresses V Neck Backless Lace Stitching Dress Beach Dresses White Dress]",
   "x_35": "x[2018 Hot Selling Spring Summer Women Flared Causal Trousers Loose Pants Drawstring Elastic Waist Middle Waisted  Wide Leg Pants]",
   "x_36": "x[2018 New Fashion Women Bikini Set Push-up Padded Bra Swimsuit -Swimwear]",
   "x_37": "x[2018 New Fashion Women Casual Playsuit Ladies Jumpsuit Romper Summer Floral Playsuit Brand New (3 Colors)]",
   "x_38": "x[2018 New Fashion Women's Tops Sexy Strappy Sleeveless Lace Crop Tops]",
   "x_39": "x[2018 New Summer Women Sexy V-neck Bandage T-shirts Camouflage Printed Short Sleeve Plus size Tops Tee(S-4XL)]",
   "x_40": "x[2018 New The new cross-strait national wind body bikini Swimsuit  Swimwear LBT]",
   "x_41": "x[2018 New ZANZEA Beach Romper Women Jumpsuits Front Zipper Sleeveless Sexy Playsuits]",
   "x_42": "x[2018 Plus Size Summer Women Fashion Sexy V-neck Lace Up Plaid Blouse Tops Irregular Short Sleeve Shirtï¼_S-5XLï¼_]",
   "x_43": "x[2018 Summer Fashion Women Casual Camouflage Tank Top Sleeveless O-neck Vest]",
   "x_44": "x[2018 Summer Fashion Women Tank Tops Sexy Women Sleeveless Crop Tops Casual Style Women Cotton Print Lace Stitching Irregular Blouse Topsï¼_S-5XL)]",
   "x_45": "x[2018 Summer New Women Fashion Sexy Sleeveless Tank Top Floral Print Casual Women Cotton T-shirts Plus Size Women Vestï¼_S-5XLï¼_]",
   "x_46": "x[2018 Summer New Women Long Skirts Solid Sexy Split Pencil Skirts]",
   "x_47": "x[2018 Summer Women Print Top Fashion Women Casual Army Camo Camouflage Tank Top Sleeveless O-neck Slim T-Shirts Plus Size S-XXXXXL]",
   "x_48": "x[2018 Summer Women Sexy Off Shoulder Shirt Casual Half Sleeve Top Ladies Fashion Printing Loose T-shirts Plus Size Cotton Blouse XS-5XL]",
   "x_49": "x[2018 Summer Womenâ__s New Fashion Whiskey Print Lace Patchwork Spaghetti Strap T-Shirts Casual Sexy Cotton Tops]",
   "x_50": "x[2018 T Shirts + Shorts Brand Clothing Tshirt Men Homme Letter Printed Basketball Running Sports Set T-shirt Suit Male]",
   "x_51": "x[2018 Women Cool T-shirt Funny 3d Tshirt Print Two Cat Short Sleeve Summer Tops Tees Teen Graphic Tee Cute Shirt Funny Gifts]",
   "x_52": "x[2018 Women Fashion Stretchy Camisole  Spaghetti Strap Long Tank Top Slip Summer Fashion Floral Mini Dress 7 Color SIZE XXL]",
   "x_53": "x[2018 Women Fashion Summer Sleeveless Maxi Halter Dresses Solid Color Halter Maxi Dress with Halter Tie and Pockets]",
   "x_54": "x[2018 Women Ladies Fashion Crocheted Lace Summer Dress Maxi Dress Long Dress]",
   "x_55": "x[2018 Women Ladies Fashion Musical Note Asymmetric Tank Top]",
   "x_56": "x[2018 Women Summer Casual Solid Color Loose Sleeveless Beach Tank Top A-line Pocket Dress Sexy Deep V-neck Short Club Party Mini Halter Midi Dresses Robe Ete Femme Knee Length Pleated Skirts Ladies Fashion Swing Cotton T-Shirt Dress Plus Size S-6XL]",
   "x_57": "x[2018 Women fashion Summer Lace Patchwork Tank Tops Casual Sleeveless Tops Vest Blouse (S-5XL) Plus Size]",
   "x_58": "x[2019 Fashion Family matching swimwear beachwear mommy and me swimsuit mother daughter father son clothes dresses high waist bikini look mum]",
   "x_59": "x[2019 Fashion Women Summer V-Neck Sleeveless Collect Waist Boho Print Maxi Long Dress S-XXXXXL]",
   "x_60": "x[2019 Men's Summer Cool Jogging Shorts]",
   "x_61": "x[2019 NEW Men's Short Sleeve T-Shirt Fitness Round Neck Casual Men's Zippers Summer T-Shirt]",
   "x_62": "x[2019 New Fashion Spring Summer Women's Long Pants Short Sleeves Off Shoulder Jumpsuit]",
   "x_63": "x[2019 New Fashion Summer Women Casual Dress Round Neck Loose Big Swing Skirt Sleeveless Soild Color Beach dress]",
   "x_64": "x[2019 New Fashion Women Blouse Tops Sleeveless Spaghetti Strap Criss Cross V-neck T Shirt Tops Summer Casual Blouse Shirts]",
   "x_65": "x[2019 New Fashion Women Casual Shorts Suit Summer Tie-Dye Print Halter Bandage Sleeveless Backless Crop Top And Elastic Waist Shorts Pants Two Piece Set]",
   "x_66": "x[2019 New Fashion Women Loose Round Neck Dress Short Sleeve Solid Color Casual Dress]",
   "x_67": "x[2019 New Fashion Women Summer Funny Penguin print T-Shirts Casual Short Sleeve O-Neck Tops Loose Cotton Cute Cool Tee Plus Size S-5XL 4 Colors]",
   "x_68": "x[2019 New Fashion Women's Slim Plus Size Maxi Dress Ink Printing Spaghetti Strap Dress]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 6020981.48,
   "variables": [
    [
     "C0",
     1357.0
    ],
    [
     "C1",
     1363.0
    ],
    [
     "C2",
     1269.0
    ],
    [
     "C3",
     6500.0
    ],
    [
     "C4",
     6335.0
    ],
    [
     "C5",
     1283.0
    ],
    [
     "C6",
     27327.0
    ],
    [
     "C7",
     25600.0
    ],
    [
     "C8",
     264.0
    ],
    [
     "C9",
     14.0
    ],
    [
     "C10",
     1484.0
    ],
    [
     "C11",
     13124.0
    ],
    [
     "C12",
     123.0
    ],
    [
     "C13",
     1430.0
    ],
    [
     "C14",
     130.0
    ],
    [
     "C15",
     14.0
    ],
    [
     "C16",
     293.0
    ],
    [
     "C17",
     6770.0
    ],
    [
     "C18",
     1325.0
    ],
    [
     "C19",
     1377.0
    ],
    [
     "C20",
     139.0
    ],
    [
     "C21",
     1350.0
    ],
    [
     "C22",
     138.0
    ],
    [
     "C23",
     122.0
    ],
    [
     "C24",
     6577.0
    ],
    [
     "C25",
     1455.0
    ],
    [
     "C26",
     72.0
    ],
    [
     "C27",
     135.0
    ],
    [
     "C28",
     6531.0
    ],
    [
     "C29",
     26650.0
    ],
    [
     "C30",
     27115.0
    ],
    [
     "C31",
     12486.0
    ],
    [
     "C32",
     12367.0
    ],
    [
     "C33",
     6194.0
    ],
    [
     "C34",
     14307.0
    ],
    [
     "C35",
     6321.0
    ],
    [
     "C36",
     13833.0
    ],
    [
     "C37",
     144944.0
    ],
    [
     "C38",
     6319.0
    ],
    [
     "C39",
     13849.0
    ],
    [
     "C40",
     6875.0
    ],
    [
     "C41",
     29273.0
    ],
    [
     "C42",
     1207.0
    ],
    [
     "C43",
     62890.0
    ],
    [
     "C44",
     14465.0
    ],
    [
     "C45",
     25467.0
    ],
    [
     "C46",
     24105.0
    ],
    [
     "C47",
     25594.0
    ],
    [
     "C48",
     1229.0
    ],
    [
     "C49",
     1459.0
    ],
    [
     "C50",
     13071.0
    ],
    [
     "C51",
     13056.0
    ],
    [
     "C52",
     6170.0
    ],
    [
     "C53",
     24746.0
    ],
    [
     "C54",
     27118.0
    ],
    [
     "C55",
     29149.0
    ],
    [
     "C56",
     1459.0
    ],
    [
     "C57",
     136.0
    ],
    [
     "C58",
     130.0
    ],
    [
     "C59",
     6720.0
    ],
    [
     "C60",
     14932.0
    ],
    [
     "C61",
     25913.0
    ],
    [
     "C62",
     27066.0
    ],
    [
     "C63",
     7453.0
    ],
    [
     "C64",
     142.0
    ],
    [
     "C65",
     270.0
    ],
    [
     "C66",
     1418.0
    ],
    [
     "C67",
     28189.0
    ]
   ],
   "error": null
  }
 },
 "OR-025": {
  "source_code_sha256": "ec34e57d7c943c3f4318edc0dfe02957bb94bd7bed25bd47ac255865dd5c53e4",
  "original_solution_sha256": "7884e3f3dbfd1429ee23eecb0b9f194b06175907a0e15d707b187b57623a29f3",
  "bindings": [
   [
    "C0",
    "x[454]"
   ],
   [
    "C1",
    "x[455]"
   ],
   [
    "C2",
    "x[456]"
   ],
   [
    "C3",
    "x[457]"
   ],
   [
    "C4",
    "x[458]"
   ],
   [
    "C5",
    "x[459]"
   ],
   [
    "C6",
    "x[460]"
   ],
   [
    "C7",
    "x[461]"
   ],
   [
    "C8",
    "x[462]"
   ],
   [
    "C9",
    "x[463]"
   ],
   [
    "C10",
    "x[464]"
   ],
   [
    "C11",
    "x[465]"
   ],
   [
    "C12",
    "x[466]"
   ],
   [
    "C13",
    "x[467]"
   ],
   [
    "C14",
    "x[468]"
   ],
   [
    "C15",
    "x[469]"
   ],
   [
    "C16",
    "x[470]"
   ],
   [
    "C17",
    "x[471]"
   ],
   [
    "C18",
    "x[472]"
   ],
   [
    "C19",
    "x[473]"
   ],
   [
    "C20",
    "x[474]"
   ],
   [
    "C21",
    "x[475]"
   ],
   [
    "C22",
    "x[476]"
   ],
   [
    "C23",
    "x[477]"
   ],
   [
    "C24",
    "x[478]"
   ],
   [
    "C25",
    "x[479]"
   ],
   [
    "C26",
    "x[480]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[454]",
   "x_2": "x[455]",
   "x_3": "x[456]",
   "x_4": "x[457]",
   "x_5": "x[458]",
   "x_6": "x[459]",
   "x_7": "x[460]",
   "x_8": "x[461]",
   "x_9": "x[462]",
   "x_10": "x[463]",
   "x_11": "x[464]",
   "x_12": "x[465]",
   "x_13": "x[466]",
   "x_14": "x[467]",
   "x_15": "x[468]",
   "x_16": "x[469]",
   "x_17": "x[470]",
   "x_18": "x[471]",
   "x_19": "x[472]",
   "x_20": "x[473]",
   "x_21": "x[474]",
   "x_22": "x[475]",
   "x_23": "x[476]",
   "x_24": "x[477]",
   "x_25": "x[478]",
   "x_26": "x[479]",
   "x_27": "x[480]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 5486959.829999999,
   "variables": [
    [
     "C0",
     2.0
    ],
    [
     "C1",
     43.0
    ],
    [
     "C2",
     6.0
    ],
    [
     "C3",
     6.0
    ],
    [
     "C4",
     20.0
    ],
    [
     "C5",
     22.0
    ],
    [
     "C6",
     3.0
    ],
    [
     "C7",
     16.0
    ],
    [
     "C8",
     2.0
    ],
    [
     "C9",
     2.0
    ],
    [
     "C10",
     32.0
    ],
    [
     "C11",
     14.0
    ],
    [
     "C12",
     2.0
    ],
    [
     "C13",
     2.0
    ],
    [
     "C14",
     6.0
    ],
    [
     "C15",
     2.0
    ],
    [
     "C16",
     6.0
    ],
    [
     "C17",
     2.0
    ],
    [
     "C18",
     6.0
    ],
    [
     "C19",
     6.0
    ],
    [
     "C20",
     6.0
    ],
    [
     "C21",
     12.0
    ],
    [
     "C22",
     60.0
    ],
    [
     "C23",
     8.0
    ],
    [
     "C24",
     8.0
    ],
    [
     "C25",
     38.0
    ],
    [
     "C26",
     4.0
    ]
   ],
   "error": null
  }
 },
 "OR-029": {
  "source_code_sha256": "1f9423fe1343916635d39eb858a4e9ece61ae866790dc45d9f0bd287b9989ae2",
  "original_solution_sha256": "5485d4f8eb11b798c7fcbcd0cde639d4bdb7871396152f0cb1e51a40eae67523",
  "bindings": [
   [
    "C0",
    "x[FAUX FUR JEWEL SWEATER]"
   ],
   [
    "C1",
    "x[FAUX LEATHER BOMBER JACKET]"
   ],
   [
    "C2",
    "x[FAUX LEATHER BOXY FIT JACKET]"
   ],
   [
    "C3",
    "x[FAUX LEATHER JACKET]"
   ],
   [
    "C4",
    "x[FAUX LEATHER OVERSIZED JACKET LIMITED EDITION]"
   ],
   [
    "C5",
    "x[FAUX LEATHER PUFFER JACKET]"
   ],
   [
    "C6",
    "x[FAUX SHEARLING LINED SUEDE BOOTS]"
   ],
   [
    "C7",
    "x[FAUX SHEARLING PLAID JACKET]"
   ],
   [
    "C8",
    "x[FAUX SUEDE BOMBER JACKET]"
   ],
   [
    "C9",
    "x[FAUX SUEDE JACKET]"
   ],
   [
    "C10",
    "x[FAUX SUEDE OVERSHIRT]"
   ],
   [
    "C11",
    "x[FAUX SUEDE PATCH JACKET]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[FAUX FUR JEWEL SWEATER]",
   "x_2": "x[FAUX LEATHER BOMBER JACKET]",
   "x_3": "x[FAUX LEATHER BOXY FIT JACKET]",
   "x_4": "x[FAUX LEATHER JACKET]",
   "x_5": "x[FAUX LEATHER OVERSIZED JACKET LIMITED EDITION]",
   "x_6": "x[FAUX LEATHER PUFFER JACKET]",
   "x_7": "x[FAUX SHEARLING LINED SUEDE BOOTS]",
   "x_8": "x[FAUX SHEARLING PLAID JACKET]",
   "x_9": "x[FAUX SUEDE BOMBER JACKET]",
   "x_10": "x[FAUX SUEDE JACKET]",
   "x_11": "x[FAUX SUEDE OVERSHIRT]",
   "x_12": "x[FAUX SUEDE PATCH JACKET]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 5090631.58,
   "variables": [
    [
     "C0",
     3025.0
    ],
    [
     "C1",
     9585.0
    ],
    [
     "C2",
     4486.0
    ],
    [
     "C3",
     10322.0
    ],
    [
     "C4",
     4868.0
    ],
    [
     "C5",
     8482.0
    ],
    [
     "C6",
     2607.0
    ],
    [
     "C7",
     1784.0
    ],
    [
     "C8",
     6626.0
    ],
    [
     "C9",
     3256.0
    ],
    [
     "C10",
     2955.0
    ],
    [
     "C11",
     910.0
    ]
   ],
   "error": null
  }
 },
 "OR-033": {
  "source_code_sha256": "c374414ece08f09ce3217f431e5b95e873c0a8e3a7adb57fbb6d612104c4088d",
  "original_solution_sha256": "2821aefe9bfe2db49216fd701a4a17406bc1586d54ed2c33e130cc524de64cfa",
  "bindings": [
   [
    "C0",
    "x[Baby Food_255.28]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Baby Food_255.28]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 195506188.0,
   "variables": [
    [
     "C0",
     765850.0
    ]
   ],
   "error": null
  }
 },
 "OR-037": {
  "source_code_sha256": "e6f09dd830d995cd4facc94d66f385f1c0b7fcdc4763c71ad6cb8ea07b957842",
  "original_solution_sha256": "db6fd2358b41a29d98d26925c158d5d25e4e0bbed638ab55b89fc326eb5c9d58",
  "bindings": [
   [
    "C0",
    "x[Sedan]"
   ],
   [
    "C1",
    "x[SUV]"
   ],
   [
    "C2",
    "x[Truck]"
   ],
   [
    "C3",
    "x[Convertible]"
   ],
   [
    "C4",
    "x[Minivan]"
   ],
   [
    "C5",
    "x[Coupe]"
   ],
   [
    "C6",
    "x[Hatchback]"
   ],
   [
    "C7",
    "x[Station Wagon]"
   ],
   [
    "C8",
    "x[Electric Car]"
   ],
   [
    "C9",
    "x[Hybrid Car]"
   ],
   [
    "C10",
    "x[Luxury Sedan]"
   ],
   [
    "C11",
    "x[Sports Car]"
   ],
   [
    "C12",
    "x[Crossover]"
   ],
   [
    "C13",
    "x[Diesel Truck]"
   ],
   [
    "C14",
    "x[Compact SUV]"
   ],
   [
    "C15",
    "x[Luxury SUV]"
   ],
   [
    "C16",
    "x[Cargo Van]"
   ],
   [
    "C17",
    "x[Pickup Truck]"
   ],
   [
    "C18",
    "x[Roadster]"
   ],
   [
    "C19",
    "x[Muscle Car]"
   ],
   [
    "C20",
    "x[Off-road Vehicle]"
   ],
   [
    "C21",
    "x[Camper Van]"
   ],
   [
    "C22",
    "x[Compact Car]"
   ],
   [
    "C23",
    "x[Motorcycle]"
   ],
   [
    "C24",
    "x[Electric SUV]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Sedan]",
   "x_1_0": "x[SUV]",
   "x_2_0": "x[Truck]",
   "x_3_0": "x[Convertible]",
   "x_4_0": "x[Minivan]",
   "x_5_0": "x[Coupe]",
   "x_6_0": "x[Hatchback]",
   "x_7_0": "x[Station Wagon]",
   "x_8_0": "x[Electric Car]",
   "x_9_0": "x[Hybrid Car]",
   "x_10_0": "x[Luxury Sedan]",
   "x_11_0": "x[Sports Car]",
   "x_12_0": "x[Crossover]",
   "x_13_0": "x[Diesel Truck]",
   "x_14_0": "x[Compact SUV]",
   "x_15_0": "x[Luxury SUV]",
   "x_16_0": "x[Cargo Van]",
   "x_17_0": "x[Pickup Truck]",
   "x_18_0": "x[Roadster]",
   "x_19_0": "x[Muscle Car]",
   "x_20_0": "x[Off-road Vehicle]",
   "x_21_0": "x[Camper Van]",
   "x_22_0": "x[Compact Car]",
   "x_23_0": "x[Motorcycle]",
   "x_24_0": "x[Electric SUV]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 722376.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     127.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-041": {
  "source_code_sha256": "aa2921a54cfcc588ac399d2eb08e16386f66559ee44b2e1ab191089be48ba9bb",
  "original_solution_sha256": "a4bdf58012c54ed1aaf2710641de7d7d7ee530d669cb76c72c6dcc3685f5bd85",
  "bindings": [
   [
    "C0",
    "x[Queens]"
   ],
   [
    "C1",
    "x[Brooklyn]"
   ],
   [
    "C2",
    "x[Manhattan]"
   ],
   [
    "C3",
    "x[Bronx]"
   ],
   [
    "C4",
    "x[Staten Island]"
   ],
   [
    "C5",
    "x[Harlem]"
   ],
   [
    "C6",
    "x[Upper East Side]"
   ],
   [
    "C7",
    "x[Lower Manhattan]"
   ],
   [
    "C8",
    "x[Midtown]"
   ],
   [
    "C9",
    "x[Long Island City]"
   ],
   [
    "C10",
    "x[Williamsburg]"
   ],
   [
    "C11",
    "x[Bushwick]"
   ],
   [
    "C12",
    "x[Flatbush]"
   ],
   [
    "C13",
    "x[Greenpoint]"
   ],
   [
    "C14",
    "x[Park Slope]"
   ],
   [
    "C15",
    "x[Astoria]"
   ],
   [
    "C16",
    "x[Jackson Heights]"
   ],
   [
    "C17",
    "x[Flushing]"
   ],
   [
    "C18",
    "x[Sunnyside]"
   ],
   [
    "C19",
    "x[Ditmars]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Queens]",
   "x_1_0": "x[Brooklyn]",
   "x_2_0": "x[Manhattan]",
   "x_3_0": "x[Bronx]",
   "x_4_0": "x[Staten Island]",
   "x_5_0": "x[Harlem]",
   "x_6_0": "x[Upper East Side]",
   "x_7_0": "x[Lower Manhattan]",
   "x_8_0": "x[Midtown]",
   "x_9_0": "x[Long Island City]",
   "x_10_0": "x[Williamsburg]",
   "x_11_0": "x[Bushwick]",
   "x_12_0": "x[Flatbush]",
   "x_13_0": "x[Greenpoint]",
   "x_14_0": "x[Park Slope]",
   "x_15_0": "x[Astoria]",
   "x_16_0": "x[Jackson Heights]",
   "x_17_0": "x[Flushing]",
   "x_18_0": "x[Sunnyside]",
   "x_19_0": "x[Ditmars]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 3912.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     4.0
    ]
   ],
   "error": null
  }
 },
 "OR-045": {
  "source_code_sha256": "7dd8542e7f7f689dda75adefe081ae08df8dc74e71c1f581da0972ba57c20e51",
  "original_solution_sha256": "66a82d3222d1f4c64117422a0b64c6440b3775c98c5e351a4c80ae4b4888420c",
  "bindings": [
   [
    "C0",
    "x[Spinach]"
   ],
   [
    "C1",
    "x[Shiitake Mushrooms]"
   ],
   [
    "C2",
    "x[Apples]"
   ],
   [
    "C3",
    "x[Carrots]"
   ],
   [
    "C4",
    "x[Basil]"
   ],
   [
    "C5",
    "x[Potatoes]"
   ],
   [
    "C6",
    "x[Green Beans]"
   ],
   [
    "C7",
    "x[Blueberries]"
   ],
   [
    "C8",
    "x[Oranges]"
   ],
   [
    "C9",
    "x[Watermelons]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Spinach]",
   "x_1_0": "x[Shiitake Mushrooms]",
   "x_2_0": "x[Apples]",
   "x_3_0": "x[Carrots]",
   "x_4_0": "x[Basil]",
   "x_5_0": "x[Potatoes]",
   "x_6_0": "x[Green Beans]",
   "x_7_0": "x[Blueberries]",
   "x_8_0": "x[Oranges]",
   "x_9_0": "x[Watermelons]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 885.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     11.0
    ],
    [
     "C9",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-049": {
  "source_code_sha256": "4362b67ab0ae075a191f8187e738ab11967a7c50ca50f673f4f1f7d8154bf22e",
  "original_solution_sha256": "818d23acaffa9267df75706492cd632d5f60fe8b7f08eacec77592296254f75a",
  "bindings": [
   [
    "C0",
    "x[1,Smartphone]"
   ],
   [
    "C1",
    "x[1,Laptop]"
   ],
   [
    "C2",
    "x[1,Headphones]"
   ],
   [
    "C3",
    "x[1,Camera]"
   ],
   [
    "C4",
    "x[1,Smartwatch]"
   ],
   [
    "C5",
    "x[1,Tablet]"
   ],
   [
    "C6",
    "x[1,Bluetooth Speaker]"
   ],
   [
    "C7",
    "x[1,Keyboard]"
   ],
   [
    "C8",
    "x[1,Mouse]"
   ],
   [
    "C9",
    "x[1,Monitor]"
   ],
   [
    "C10",
    "x[1,Printer]"
   ],
   [
    "C11",
    "x[1,External Hard Drive]"
   ],
   [
    "C12",
    "x[1,Router]"
   ],
   [
    "C13",
    "x[1,Power Bank]"
   ],
   [
    "C14",
    "x[1,Memory Card]"
   ],
   [
    "C15",
    "x[1,USB Flash Drive]"
   ],
   [
    "C16",
    "x[1,Smart Home Hub]"
   ],
   [
    "C17",
    "x[1,Gaming Console]"
   ],
   [
    "C18",
    "x[1,Fitness Tracker]"
   ],
   [
    "C19",
    "x[1,E-Reader]"
   ],
   [
    "C20",
    "x[2,Smartphone]"
   ],
   [
    "C21",
    "x[2,Laptop]"
   ],
   [
    "C22",
    "x[2,Headphones]"
   ],
   [
    "C23",
    "x[2,Camera]"
   ],
   [
    "C24",
    "x[2,Smartwatch]"
   ],
   [
    "C25",
    "x[2,Tablet]"
   ],
   [
    "C26",
    "x[2,Bluetooth Speaker]"
   ],
   [
    "C27",
    "x[2,Keyboard]"
   ],
   [
    "C28",
    "x[2,Mouse]"
   ],
   [
    "C29",
    "x[2,Monitor]"
   ],
   [
    "C30",
    "x[2,Printer]"
   ],
   [
    "C31",
    "x[2,External Hard Drive]"
   ],
   [
    "C32",
    "x[2,Router]"
   ],
   [
    "C33",
    "x[2,Power Bank]"
   ],
   [
    "C34",
    "x[2,Memory Card]"
   ],
   [
    "C35",
    "x[2,USB Flash Drive]"
   ],
   [
    "C36",
    "x[2,Smart Home Hub]"
   ],
   [
    "C37",
    "x[2,Gaming Console]"
   ],
   [
    "C38",
    "x[2,Fitness Tracker]"
   ],
   [
    "C39",
    "x[2,E-Reader]"
   ],
   [
    "C40",
    "x[3,Smartphone]"
   ],
   [
    "C41",
    "x[3,Laptop]"
   ],
   [
    "C42",
    "x[3,Headphones]"
   ],
   [
    "C43",
    "x[3,Camera]"
   ],
   [
    "C44",
    "x[3,Smartwatch]"
   ],
   [
    "C45",
    "x[3,Tablet]"
   ],
   [
    "C46",
    "x[3,Bluetooth Speaker]"
   ],
   [
    "C47",
    "x[3,Keyboard]"
   ],
   [
    "C48",
    "x[3,Mouse]"
   ],
   [
    "C49",
    "x[3,Monitor]"
   ],
   [
    "C50",
    "x[3,Printer]"
   ],
   [
    "C51",
    "x[3,External Hard Drive]"
   ],
   [
    "C52",
    "x[3,Router]"
   ],
   [
    "C53",
    "x[3,Power Bank]"
   ],
   [
    "C54",
    "x[3,Memory Card]"
   ],
   [
    "C55",
    "x[3,USB Flash Drive]"
   ],
   [
    "C56",
    "x[3,Smart Home Hub]"
   ],
   [
    "C57",
    "x[3,Gaming Console]"
   ],
   [
    "C58",
    "x[3,Fitness Tracker]"
   ],
   [
    "C59",
    "x[3,E-Reader]"
   ],
   [
    "C60",
    "x[4,Smartphone]"
   ],
   [
    "C61",
    "x[4,Laptop]"
   ],
   [
    "C62",
    "x[4,Headphones]"
   ],
   [
    "C63",
    "x[4,Camera]"
   ],
   [
    "C64",
    "x[4,Smartwatch]"
   ],
   [
    "C65",
    "x[4,Tablet]"
   ],
   [
    "C66",
    "x[4,Bluetooth Speaker]"
   ],
   [
    "C67",
    "x[4,Keyboard]"
   ],
   [
    "C68",
    "x[4,Mouse]"
   ],
   [
    "C69",
    "x[4,Monitor]"
   ],
   [
    "C70",
    "x[4,Printer]"
   ],
   [
    "C71",
    "x[4,External Hard Drive]"
   ],
   [
    "C72",
    "x[4,Router]"
   ],
   [
    "C73",
    "x[4,Power Bank]"
   ],
   [
    "C74",
    "x[4,Memory Card]"
   ],
   [
    "C75",
    "x[4,USB Flash Drive]"
   ],
   [
    "C76",
    "x[4,Smart Home Hub]"
   ],
   [
    "C77",
    "x[4,Gaming Console]"
   ],
   [
    "C78",
    "x[4,Fitness Tracker]"
   ],
   [
    "C79",
    "x[4,E-Reader]"
   ],
   [
    "C80",
    "x[5,Smartphone]"
   ],
   [
    "C81",
    "x[5,Laptop]"
   ],
   [
    "C82",
    "x[5,Headphones]"
   ],
   [
    "C83",
    "x[5,Camera]"
   ],
   [
    "C84",
    "x[5,Smartwatch]"
   ],
   [
    "C85",
    "x[5,Tablet]"
   ],
   [
    "C86",
    "x[5,Bluetooth Speaker]"
   ],
   [
    "C87",
    "x[5,Keyboard]"
   ],
   [
    "C88",
    "x[5,Mouse]"
   ],
   [
    "C89",
    "x[5,Monitor]"
   ],
   [
    "C90",
    "x[5,Printer]"
   ],
   [
    "C91",
    "x[5,External Hard Drive]"
   ],
   [
    "C92",
    "x[5,Router]"
   ],
   [
    "C93",
    "x[5,Power Bank]"
   ],
   [
    "C94",
    "x[5,Memory Card]"
   ],
   [
    "C95",
    "x[5,USB Flash Drive]"
   ],
   [
    "C96",
    "x[5,Smart Home Hub]"
   ],
   [
    "C97",
    "x[5,Gaming Console]"
   ],
   [
    "C98",
    "x[5,Fitness Tracker]"
   ],
   [
    "C99",
    "x[5,E-Reader]"
   ],
   [
    "C100",
    "x[6,Smartphone]"
   ],
   [
    "C101",
    "x[6,Laptop]"
   ],
   [
    "C102",
    "x[6,Headphones]"
   ],
   [
    "C103",
    "x[6,Camera]"
   ],
   [
    "C104",
    "x[6,Smartwatch]"
   ],
   [
    "C105",
    "x[6,Tablet]"
   ],
   [
    "C106",
    "x[6,Bluetooth Speaker]"
   ],
   [
    "C107",
    "x[6,Keyboard]"
   ],
   [
    "C108",
    "x[6,Mouse]"
   ],
   [
    "C109",
    "x[6,Monitor]"
   ],
   [
    "C110",
    "x[6,Printer]"
   ],
   [
    "C111",
    "x[6,External Hard Drive]"
   ],
   [
    "C112",
    "x[6,Router]"
   ],
   [
    "C113",
    "x[6,Power Bank]"
   ],
   [
    "C114",
    "x[6,Memory Card]"
   ],
   [
    "C115",
    "x[6,USB Flash Drive]"
   ],
   [
    "C116",
    "x[6,Smart Home Hub]"
   ],
   [
    "C117",
    "x[6,Gaming Console]"
   ],
   [
    "C118",
    "x[6,Fitness Tracker]"
   ],
   [
    "C119",
    "x[6,E-Reader]"
   ],
   [
    "C120",
    "x[7,Smartphone]"
   ],
   [
    "C121",
    "x[7,Laptop]"
   ],
   [
    "C122",
    "x[7,Headphones]"
   ],
   [
    "C123",
    "x[7,Camera]"
   ],
   [
    "C124",
    "x[7,Smartwatch]"
   ],
   [
    "C125",
    "x[7,Tablet]"
   ],
   [
    "C126",
    "x[7,Bluetooth Speaker]"
   ],
   [
    "C127",
    "x[7,Keyboard]"
   ],
   [
    "C128",
    "x[7,Mouse]"
   ],
   [
    "C129",
    "x[7,Monitor]"
   ],
   [
    "C130",
    "x[7,Printer]"
   ],
   [
    "C131",
    "x[7,External Hard Drive]"
   ],
   [
    "C132",
    "x[7,Router]"
   ],
   [
    "C133",
    "x[7,Power Bank]"
   ],
   [
    "C134",
    "x[7,Memory Card]"
   ],
   [
    "C135",
    "x[7,USB Flash Drive]"
   ],
   [
    "C136",
    "x[7,Smart Home Hub]"
   ],
   [
    "C137",
    "x[7,Gaming Console]"
   ],
   [
    "C138",
    "x[7,Fitness Tracker]"
   ],
   [
    "C139",
    "x[7,E-Reader]"
   ],
   [
    "C140",
    "x[8,Smartphone]"
   ],
   [
    "C141",
    "x[8,Laptop]"
   ],
   [
    "C142",
    "x[8,Headphones]"
   ],
   [
    "C143",
    "x[8,Camera]"
   ],
   [
    "C144",
    "x[8,Smartwatch]"
   ],
   [
    "C145",
    "x[8,Tablet]"
   ],
   [
    "C146",
    "x[8,Bluetooth Speaker]"
   ],
   [
    "C147",
    "x[8,Keyboard]"
   ],
   [
    "C148",
    "x[8,Mouse]"
   ],
   [
    "C149",
    "x[8,Monitor]"
   ],
   [
    "C150",
    "x[8,Printer]"
   ],
   [
    "C151",
    "x[8,External Hard Drive]"
   ],
   [
    "C152",
    "x[8,Router]"
   ],
   [
    "C153",
    "x[8,Power Bank]"
   ],
   [
    "C154",
    "x[8,Memory Card]"
   ],
   [
    "C155",
    "x[8,USB Flash Drive]"
   ],
   [
    "C156",
    "x[8,Smart Home Hub]"
   ],
   [
    "C157",
    "x[8,Gaming Console]"
   ],
   [
    "C158",
    "x[8,Fitness Tracker]"
   ],
   [
    "C159",
    "x[8,E-Reader]"
   ],
   [
    "C160",
    "x[9,Smartphone]"
   ],
   [
    "C161",
    "x[9,Laptop]"
   ],
   [
    "C162",
    "x[9,Headphones]"
   ],
   [
    "C163",
    "x[9,Camera]"
   ],
   [
    "C164",
    "x[9,Smartwatch]"
   ],
   [
    "C165",
    "x[9,Tablet]"
   ],
   [
    "C166",
    "x[9,Bluetooth Speaker]"
   ],
   [
    "C167",
    "x[9,Keyboard]"
   ],
   [
    "C168",
    "x[9,Mouse]"
   ],
   [
    "C169",
    "x[9,Monitor]"
   ],
   [
    "C170",
    "x[9,Printer]"
   ],
   [
    "C171",
    "x[9,External Hard Drive]"
   ],
   [
    "C172",
    "x[9,Router]"
   ],
   [
    "C173",
    "x[9,Power Bank]"
   ],
   [
    "C174",
    "x[9,Memory Card]"
   ],
   [
    "C175",
    "x[9,USB Flash Drive]"
   ],
   [
    "C176",
    "x[9,Smart Home Hub]"
   ],
   [
    "C177",
    "x[9,Gaming Console]"
   ],
   [
    "C178",
    "x[9,Fitness Tracker]"
   ],
   [
    "C179",
    "x[9,E-Reader]"
   ],
   [
    "C180",
    "x[10,Smartphone]"
   ],
   [
    "C181",
    "x[10,Laptop]"
   ],
   [
    "C182",
    "x[10,Headphones]"
   ],
   [
    "C183",
    "x[10,Camera]"
   ],
   [
    "C184",
    "x[10,Smartwatch]"
   ],
   [
    "C185",
    "x[10,Tablet]"
   ],
   [
    "C186",
    "x[10,Bluetooth Speaker]"
   ],
   [
    "C187",
    "x[10,Keyboard]"
   ],
   [
    "C188",
    "x[10,Mouse]"
   ],
   [
    "C189",
    "x[10,Monitor]"
   ],
   [
    "C190",
    "x[10,Printer]"
   ],
   [
    "C191",
    "x[10,External Hard Drive]"
   ],
   [
    "C192",
    "x[10,Router]"
   ],
   [
    "C193",
    "x[10,Power Bank]"
   ],
   [
    "C194",
    "x[10,Memory Card]"
   ],
   [
    "C195",
    "x[10,USB Flash Drive]"
   ],
   [
    "C196",
    "x[10,Smart Home Hub]"
   ],
   [
    "C197",
    "x[10,Gaming Console]"
   ],
   [
    "C198",
    "x[10,Fitness Tracker]"
   ],
   [
    "C199",
    "x[10,E-Reader]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Smartphone]",
   "x_0_1": "x[2,Smartphone]",
   "x_0_2": "x[3,Smartphone]",
   "x_0_3": "x[4,Smartphone]",
   "x_0_4": "x[5,Smartphone]",
   "x_0_5": "x[6,Smartphone]",
   "x_0_6": "x[7,Smartphone]",
   "x_0_7": "x[8,Smartphone]",
   "x_0_8": "x[9,Smartphone]",
   "x_0_9": "x[10,Smartphone]",
   "x_1_0": "x[1,Laptop]",
   "x_1_1": "x[2,Laptop]",
   "x_1_2": "x[3,Laptop]",
   "x_1_3": "x[4,Laptop]",
   "x_1_4": "x[5,Laptop]",
   "x_1_5": "x[6,Laptop]",
   "x_1_6": "x[7,Laptop]",
   "x_1_7": "x[8,Laptop]",
   "x_1_8": "x[9,Laptop]",
   "x_1_9": "x[10,Laptop]",
   "x_2_0": "x[1,Headphones]",
   "x_2_1": "x[2,Headphones]",
   "x_2_2": "x[3,Headphones]",
   "x_2_3": "x[4,Headphones]",
   "x_2_4": "x[5,Headphones]",
   "x_2_5": "x[6,Headphones]",
   "x_2_6": "x[7,Headphones]",
   "x_2_7": "x[8,Headphones]",
   "x_2_8": "x[9,Headphones]",
   "x_2_9": "x[10,Headphones]",
   "x_3_0": "x[1,Camera]",
   "x_3_1": "x[2,Camera]",
   "x_3_2": "x[3,Camera]",
   "x_3_3": "x[4,Camera]",
   "x_3_4": "x[5,Camera]",
   "x_3_5": "x[6,Camera]",
   "x_3_6": "x[7,Camera]",
   "x_3_7": "x[8,Camera]",
   "x_3_8": "x[9,Camera]",
   "x_3_9": "x[10,Camera]",
   "x_4_0": "x[1,Smartwatch]",
   "x_4_1": "x[2,Smartwatch]",
   "x_4_2": "x[3,Smartwatch]",
   "x_4_3": "x[4,Smartwatch]",
   "x_4_4": "x[5,Smartwatch]",
   "x_4_5": "x[6,Smartwatch]",
   "x_4_6": "x[7,Smartwatch]",
   "x_4_7": "x[8,Smartwatch]",
   "x_4_8": "x[9,Smartwatch]",
   "x_4_9": "x[10,Smartwatch]",
   "x_5_0": "x[1,Tablet]",
   "x_5_1": "x[2,Tablet]",
   "x_5_2": "x[3,Tablet]",
   "x_5_3": "x[4,Tablet]",
   "x_5_4": "x[5,Tablet]",
   "x_5_5": "x[6,Tablet]",
   "x_5_6": "x[7,Tablet]",
   "x_5_7": "x[8,Tablet]",
   "x_5_8": "x[9,Tablet]",
   "x_5_9": "x[10,Tablet]",
   "x_6_0": "x[1,Bluetooth Speaker]",
   "x_6_1": "x[2,Bluetooth Speaker]",
   "x_6_2": "x[3,Bluetooth Speaker]",
   "x_6_3": "x[4,Bluetooth Speaker]",
   "x_6_4": "x[5,Bluetooth Speaker]",
   "x_6_5": "x[6,Bluetooth Speaker]",
   "x_6_6": "x[7,Bluetooth Speaker]",
   "x_6_7": "x[8,Bluetooth Speaker]",
   "x_6_8": "x[9,Bluetooth Speaker]",
   "x_6_9": "x[10,Bluetooth Speaker]",
   "x_7_0": "x[1,Keyboard]",
   "x_7_1": "x[2,Keyboard]",
   "x_7_2": "x[3,Keyboard]",
   "x_7_3": "x[4,Keyboard]",
   "x_7_4": "x[5,Keyboard]",
   "x_7_5": "x[6,Keyboard]",
   "x_7_6": "x[7,Keyboard]",
   "x_7_7": "x[8,Keyboard]",
   "x_7_8": "x[9,Keyboard]",
   "x_7_9": "x[10,Keyboard]",
   "x_8_0": "x[1,Mouse]",
   "x_8_1": "x[2,Mouse]",
   "x_8_2": "x[3,Mouse]",
   "x_8_3": "x[4,Mouse]",
   "x_8_4": "x[5,Mouse]",
   "x_8_5": "x[6,Mouse]",
   "x_8_6": "x[7,Mouse]",
   "x_8_7": "x[8,Mouse]",
   "x_8_8": "x[9,Mouse]",
   "x_8_9": "x[10,Mouse]",
   "x_9_0": "x[1,Monitor]",
   "x_9_1": "x[2,Monitor]",
   "x_9_2": "x[3,Monitor]",
   "x_9_3": "x[4,Monitor]",
   "x_9_4": "x[5,Monitor]",
   "x_9_5": "x[6,Monitor]",
   "x_9_6": "x[7,Monitor]",
   "x_9_7": "x[8,Monitor]",
   "x_9_8": "x[9,Monitor]",
   "x_9_9": "x[10,Monitor]",
   "x_10_0": "x[1,Printer]",
   "x_10_1": "x[2,Printer]",
   "x_10_2": "x[3,Printer]",
   "x_10_3": "x[4,Printer]",
   "x_10_4": "x[5,Printer]",
   "x_10_5": "x[6,Printer]",
   "x_10_6": "x[7,Printer]",
   "x_10_7": "x[8,Printer]",
   "x_10_8": "x[9,Printer]",
   "x_10_9": "x[10,Printer]",
   "x_11_0": "x[1,External Hard Drive]",
   "x_11_1": "x[2,External Hard Drive]",
   "x_11_2": "x[3,External Hard Drive]",
   "x_11_3": "x[4,External Hard Drive]",
   "x_11_4": "x[5,External Hard Drive]",
   "x_11_5": "x[6,External Hard Drive]",
   "x_11_6": "x[7,External Hard Drive]",
   "x_11_7": "x[8,External Hard Drive]",
   "x_11_8": "x[9,External Hard Drive]",
   "x_11_9": "x[10,External Hard Drive]",
   "x_12_0": "x[1,Router]",
   "x_12_1": "x[2,Router]",
   "x_12_2": "x[3,Router]",
   "x_12_3": "x[4,Router]",
   "x_12_4": "x[5,Router]",
   "x_12_5": "x[6,Router]",
   "x_12_6": "x[7,Router]",
   "x_12_7": "x[8,Router]",
   "x_12_8": "x[9,Router]",
   "x_12_9": "x[10,Router]",
   "x_13_0": "x[1,Power Bank]",
   "x_13_1": "x[2,Power Bank]",
   "x_13_2": "x[3,Power Bank]",
   "x_13_3": "x[4,Power Bank]",
   "x_13_4": "x[5,Power Bank]",
   "x_13_5": "x[6,Power Bank]",
   "x_13_6": "x[7,Power Bank]",
   "x_13_7": "x[8,Power Bank]",
   "x_13_8": "x[9,Power Bank]",
   "x_13_9": "x[10,Power Bank]",
   "x_14_0": "x[1,Memory Card]",
   "x_14_1": "x[2,Memory Card]",
   "x_14_2": "x[3,Memory Card]",
   "x_14_3": "x[4,Memory Card]",
   "x_14_4": "x[5,Memory Card]",
   "x_14_5": "x[6,Memory Card]",
   "x_14_6": "x[7,Memory Card]",
   "x_14_7": "x[8,Memory Card]",
   "x_14_8": "x[9,Memory Card]",
   "x_14_9": "x[10,Memory Card]",
   "x_15_0": "x[1,USB Flash Drive]",
   "x_15_1": "x[2,USB Flash Drive]",
   "x_15_2": "x[3,USB Flash Drive]",
   "x_15_3": "x[4,USB Flash Drive]",
   "x_15_4": "x[5,USB Flash Drive]",
   "x_15_5": "x[6,USB Flash Drive]",
   "x_15_6": "x[7,USB Flash Drive]",
   "x_15_7": "x[8,USB Flash Drive]",
   "x_15_8": "x[9,USB Flash Drive]",
   "x_15_9": "x[10,USB Flash Drive]",
   "x_16_0": "x[1,Smart Home Hub]",
   "x_16_1": "x[2,Smart Home Hub]",
   "x_16_2": "x[3,Smart Home Hub]",
   "x_16_3": "x[4,Smart Home Hub]",
   "x_16_4": "x[5,Smart Home Hub]",
   "x_16_5": "x[6,Smart Home Hub]",
   "x_16_6": "x[7,Smart Home Hub]",
   "x_16_7": "x[8,Smart Home Hub]",
   "x_16_8": "x[9,Smart Home Hub]",
   "x_16_9": "x[10,Smart Home Hub]",
   "x_17_0": "x[1,Gaming Console]",
   "x_17_1": "x[2,Gaming Console]",
   "x_17_2": "x[3,Gaming Console]",
   "x_17_3": "x[4,Gaming Console]",
   "x_17_4": "x[5,Gaming Console]",
   "x_17_5": "x[6,Gaming Console]",
   "x_17_6": "x[7,Gaming Console]",
   "x_17_7": "x[8,Gaming Console]",
   "x_17_8": "x[9,Gaming Console]",
   "x_17_9": "x[10,Gaming Console]",
   "x_18_0": "x[1,Fitness Tracker]",
   "x_18_1": "x[2,Fitness Tracker]",
   "x_18_2": "x[3,Fitness Tracker]",
   "x_18_3": "x[4,Fitness Tracker]",
   "x_18_4": "x[5,Fitness Tracker]",
   "x_18_5": "x[6,Fitness Tracker]",
   "x_18_6": "x[7,Fitness Tracker]",
   "x_18_7": "x[8,Fitness Tracker]",
   "x_18_8": "x[9,Fitness Tracker]",
   "x_18_9": "x[10,Fitness Tracker]",
   "x_19_0": "x[1,E-Reader]",
   "x_19_1": "x[2,E-Reader]",
   "x_19_2": "x[3,E-Reader]",
   "x_19_3": "x[4,E-Reader]",
   "x_19_4": "x[5,E-Reader]",
   "x_19_5": "x[6,E-Reader]",
   "x_19_6": "x[7,E-Reader]",
   "x_19_7": "x[8,E-Reader]",
   "x_19_8": "x[9,E-Reader]",
   "x_19_9": "x[10,E-Reader]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 85500.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     250.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     350.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     300.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     400.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     275.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     450.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     325.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     375.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     410.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     -0.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     285.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-053": {
  "source_code_sha256": "c85456ff3c7ec4f513a6e79a3cd0e8dd7ff3a917748b04891517261a3d2eb2fe",
  "original_solution_sha256": "85b8313391ec386654edd65ce2bbafc051bf7523976cc1e79a8eeafa9757ffe8",
  "bindings": [
   [
    "C0",
    "x[1,1]"
   ],
   [
    "C1",
    "x[1,2]"
   ],
   [
    "C2",
    "x[1,3]"
   ],
   [
    "C3",
    "x[1,4]"
   ],
   [
    "C4",
    "x[1,5]"
   ],
   [
    "C5",
    "x[1,6]"
   ],
   [
    "C6",
    "x[1,7]"
   ],
   [
    "C7",
    "x[1,8]"
   ],
   [
    "C8",
    "x[1,9]"
   ],
   [
    "C9",
    "x[1,10]"
   ],
   [
    "C10",
    "x[1,11]"
   ],
   [
    "C11",
    "x[1,12]"
   ],
   [
    "C12",
    "x[1,13]"
   ],
   [
    "C13",
    "x[1,14]"
   ],
   [
    "C14",
    "x[1,15]"
   ],
   [
    "C15",
    "x[1,16]"
   ],
   [
    "C16",
    "x[1,17]"
   ],
   [
    "C17",
    "x[1,18]"
   ],
   [
    "C18",
    "x[1,19]"
   ],
   [
    "C19",
    "x[1,20]"
   ],
   [
    "C20",
    "x[2,1]"
   ],
   [
    "C21",
    "x[2,2]"
   ],
   [
    "C22",
    "x[2,3]"
   ],
   [
    "C23",
    "x[2,4]"
   ],
   [
    "C24",
    "x[2,5]"
   ],
   [
    "C25",
    "x[2,6]"
   ],
   [
    "C26",
    "x[2,7]"
   ],
   [
    "C27",
    "x[2,8]"
   ],
   [
    "C28",
    "x[2,9]"
   ],
   [
    "C29",
    "x[2,10]"
   ],
   [
    "C30",
    "x[2,11]"
   ],
   [
    "C31",
    "x[2,12]"
   ],
   [
    "C32",
    "x[2,13]"
   ],
   [
    "C33",
    "x[2,14]"
   ],
   [
    "C34",
    "x[2,15]"
   ],
   [
    "C35",
    "x[2,16]"
   ],
   [
    "C36",
    "x[2,17]"
   ],
   [
    "C37",
    "x[2,18]"
   ],
   [
    "C38",
    "x[2,19]"
   ],
   [
    "C39",
    "x[2,20]"
   ],
   [
    "C40",
    "x[3,1]"
   ],
   [
    "C41",
    "x[3,2]"
   ],
   [
    "C42",
    "x[3,3]"
   ],
   [
    "C43",
    "x[3,4]"
   ],
   [
    "C44",
    "x[3,5]"
   ],
   [
    "C45",
    "x[3,6]"
   ],
   [
    "C46",
    "x[3,7]"
   ],
   [
    "C47",
    "x[3,8]"
   ],
   [
    "C48",
    "x[3,9]"
   ],
   [
    "C49",
    "x[3,10]"
   ],
   [
    "C50",
    "x[3,11]"
   ],
   [
    "C51",
    "x[3,12]"
   ],
   [
    "C52",
    "x[3,13]"
   ],
   [
    "C53",
    "x[3,14]"
   ],
   [
    "C54",
    "x[3,15]"
   ],
   [
    "C55",
    "x[3,16]"
   ],
   [
    "C56",
    "x[3,17]"
   ],
   [
    "C57",
    "x[3,18]"
   ],
   [
    "C58",
    "x[3,19]"
   ],
   [
    "C59",
    "x[3,20]"
   ],
   [
    "C60",
    "x[4,1]"
   ],
   [
    "C61",
    "x[4,2]"
   ],
   [
    "C62",
    "x[4,3]"
   ],
   [
    "C63",
    "x[4,4]"
   ],
   [
    "C64",
    "x[4,5]"
   ],
   [
    "C65",
    "x[4,6]"
   ],
   [
    "C66",
    "x[4,7]"
   ],
   [
    "C67",
    "x[4,8]"
   ],
   [
    "C68",
    "x[4,9]"
   ],
   [
    "C69",
    "x[4,10]"
   ],
   [
    "C70",
    "x[4,11]"
   ],
   [
    "C71",
    "x[4,12]"
   ],
   [
    "C72",
    "x[4,13]"
   ],
   [
    "C73",
    "x[4,14]"
   ],
   [
    "C74",
    "x[4,15]"
   ],
   [
    "C75",
    "x[4,16]"
   ],
   [
    "C76",
    "x[4,17]"
   ],
   [
    "C77",
    "x[4,18]"
   ],
   [
    "C78",
    "x[4,19]"
   ],
   [
    "C79",
    "x[4,20]"
   ],
   [
    "C80",
    "x[5,1]"
   ],
   [
    "C81",
    "x[5,2]"
   ],
   [
    "C82",
    "x[5,3]"
   ],
   [
    "C83",
    "x[5,4]"
   ],
   [
    "C84",
    "x[5,5]"
   ],
   [
    "C85",
    "x[5,6]"
   ],
   [
    "C86",
    "x[5,7]"
   ],
   [
    "C87",
    "x[5,8]"
   ],
   [
    "C88",
    "x[5,9]"
   ],
   [
    "C89",
    "x[5,10]"
   ],
   [
    "C90",
    "x[5,11]"
   ],
   [
    "C91",
    "x[5,12]"
   ],
   [
    "C92",
    "x[5,13]"
   ],
   [
    "C93",
    "x[5,14]"
   ],
   [
    "C94",
    "x[5,15]"
   ],
   [
    "C95",
    "x[5,16]"
   ],
   [
    "C96",
    "x[5,17]"
   ],
   [
    "C97",
    "x[5,18]"
   ],
   [
    "C98",
    "x[5,19]"
   ],
   [
    "C99",
    "x[5,20]"
   ],
   [
    "C100",
    "x[6,1]"
   ],
   [
    "C101",
    "x[6,2]"
   ],
   [
    "C102",
    "x[6,3]"
   ],
   [
    "C103",
    "x[6,4]"
   ],
   [
    "C104",
    "x[6,5]"
   ],
   [
    "C105",
    "x[6,6]"
   ],
   [
    "C106",
    "x[6,7]"
   ],
   [
    "C107",
    "x[6,8]"
   ],
   [
    "C108",
    "x[6,9]"
   ],
   [
    "C109",
    "x[6,10]"
   ],
   [
    "C110",
    "x[6,11]"
   ],
   [
    "C111",
    "x[6,12]"
   ],
   [
    "C112",
    "x[6,13]"
   ],
   [
    "C113",
    "x[6,14]"
   ],
   [
    "C114",
    "x[6,15]"
   ],
   [
    "C115",
    "x[6,16]"
   ],
   [
    "C116",
    "x[6,17]"
   ],
   [
    "C117",
    "x[6,18]"
   ],
   [
    "C118",
    "x[6,19]"
   ],
   [
    "C119",
    "x[6,20]"
   ],
   [
    "C120",
    "x[7,1]"
   ],
   [
    "C121",
    "x[7,2]"
   ],
   [
    "C122",
    "x[7,3]"
   ],
   [
    "C123",
    "x[7,4]"
   ],
   [
    "C124",
    "x[7,5]"
   ],
   [
    "C125",
    "x[7,6]"
   ],
   [
    "C126",
    "x[7,7]"
   ],
   [
    "C127",
    "x[7,8]"
   ],
   [
    "C128",
    "x[7,9]"
   ],
   [
    "C129",
    "x[7,10]"
   ],
   [
    "C130",
    "x[7,11]"
   ],
   [
    "C131",
    "x[7,12]"
   ],
   [
    "C132",
    "x[7,13]"
   ],
   [
    "C133",
    "x[7,14]"
   ],
   [
    "C134",
    "x[7,15]"
   ],
   [
    "C135",
    "x[7,16]"
   ],
   [
    "C136",
    "x[7,17]"
   ],
   [
    "C137",
    "x[7,18]"
   ],
   [
    "C138",
    "x[7,19]"
   ],
   [
    "C139",
    "x[7,20]"
   ],
   [
    "C140",
    "x[8,1]"
   ],
   [
    "C141",
    "x[8,2]"
   ],
   [
    "C142",
    "x[8,3]"
   ],
   [
    "C143",
    "x[8,4]"
   ],
   [
    "C144",
    "x[8,5]"
   ],
   [
    "C145",
    "x[8,6]"
   ],
   [
    "C146",
    "x[8,7]"
   ],
   [
    "C147",
    "x[8,8]"
   ],
   [
    "C148",
    "x[8,9]"
   ],
   [
    "C149",
    "x[8,10]"
   ],
   [
    "C150",
    "x[8,11]"
   ],
   [
    "C151",
    "x[8,12]"
   ],
   [
    "C152",
    "x[8,13]"
   ],
   [
    "C153",
    "x[8,14]"
   ],
   [
    "C154",
    "x[8,15]"
   ],
   [
    "C155",
    "x[8,16]"
   ],
   [
    "C156",
    "x[8,17]"
   ],
   [
    "C157",
    "x[8,18]"
   ],
   [
    "C158",
    "x[8,19]"
   ],
   [
    "C159",
    "x[8,20]"
   ],
   [
    "C160",
    "x[9,1]"
   ],
   [
    "C161",
    "x[9,2]"
   ],
   [
    "C162",
    "x[9,3]"
   ],
   [
    "C163",
    "x[9,4]"
   ],
   [
    "C164",
    "x[9,5]"
   ],
   [
    "C165",
    "x[9,6]"
   ],
   [
    "C166",
    "x[9,7]"
   ],
   [
    "C167",
    "x[9,8]"
   ],
   [
    "C168",
    "x[9,9]"
   ],
   [
    "C169",
    "x[9,10]"
   ],
   [
    "C170",
    "x[9,11]"
   ],
   [
    "C171",
    "x[9,12]"
   ],
   [
    "C172",
    "x[9,13]"
   ],
   [
    "C173",
    "x[9,14]"
   ],
   [
    "C174",
    "x[9,15]"
   ],
   [
    "C175",
    "x[9,16]"
   ],
   [
    "C176",
    "x[9,17]"
   ],
   [
    "C177",
    "x[9,18]"
   ],
   [
    "C178",
    "x[9,19]"
   ],
   [
    "C179",
    "x[9,20]"
   ],
   [
    "C180",
    "x[10,1]"
   ],
   [
    "C181",
    "x[10,2]"
   ],
   [
    "C182",
    "x[10,3]"
   ],
   [
    "C183",
    "x[10,4]"
   ],
   [
    "C184",
    "x[10,5]"
   ],
   [
    "C185",
    "x[10,6]"
   ],
   [
    "C186",
    "x[10,7]"
   ],
   [
    "C187",
    "x[10,8]"
   ],
   [
    "C188",
    "x[10,9]"
   ],
   [
    "C189",
    "x[10,10]"
   ],
   [
    "C190",
    "x[10,11]"
   ],
   [
    "C191",
    "x[10,12]"
   ],
   [
    "C192",
    "x[10,13]"
   ],
   [
    "C193",
    "x[10,14]"
   ],
   [
    "C194",
    "x[10,15]"
   ],
   [
    "C195",
    "x[10,16]"
   ],
   [
    "C196",
    "x[10,17]"
   ],
   [
    "C197",
    "x[10,18]"
   ],
   [
    "C198",
    "x[10,19]"
   ],
   [
    "C199",
    "x[10,20]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,1]",
   "x_0_1": "x[2,1]",
   "x_0_2": "x[3,1]",
   "x_0_3": "x[4,1]",
   "x_0_4": "x[5,1]",
   "x_0_5": "x[6,1]",
   "x_0_6": "x[7,1]",
   "x_0_7": "x[8,1]",
   "x_0_8": "x[9,1]",
   "x_0_9": "x[10,1]",
   "x_1_0": "x[1,2]",
   "x_1_1": "x[2,2]",
   "x_1_2": "x[3,2]",
   "x_1_3": "x[4,2]",
   "x_1_4": "x[5,2]",
   "x_1_5": "x[6,2]",
   "x_1_6": "x[7,2]",
   "x_1_7": "x[8,2]",
   "x_1_8": "x[9,2]",
   "x_1_9": "x[10,2]",
   "x_2_0": "x[1,3]",
   "x_2_1": "x[2,3]",
   "x_2_2": "x[3,3]",
   "x_2_3": "x[4,3]",
   "x_2_4": "x[5,3]",
   "x_2_5": "x[6,3]",
   "x_2_6": "x[7,3]",
   "x_2_7": "x[8,3]",
   "x_2_8": "x[9,3]",
   "x_2_9": "x[10,3]",
   "x_3_0": "x[1,4]",
   "x_3_1": "x[2,4]",
   "x_3_2": "x[3,4]",
   "x_3_3": "x[4,4]",
   "x_3_4": "x[5,4]",
   "x_3_5": "x[6,4]",
   "x_3_6": "x[7,4]",
   "x_3_7": "x[8,4]",
   "x_3_8": "x[9,4]",
   "x_3_9": "x[10,4]",
   "x_4_0": "x[1,5]",
   "x_4_1": "x[2,5]",
   "x_4_2": "x[3,5]",
   "x_4_3": "x[4,5]",
   "x_4_4": "x[5,5]",
   "x_4_5": "x[6,5]",
   "x_4_6": "x[7,5]",
   "x_4_7": "x[8,5]",
   "x_4_8": "x[9,5]",
   "x_4_9": "x[10,5]",
   "x_5_0": "x[1,6]",
   "x_5_1": "x[2,6]",
   "x_5_2": "x[3,6]",
   "x_5_3": "x[4,6]",
   "x_5_4": "x[5,6]",
   "x_5_5": "x[6,6]",
   "x_5_6": "x[7,6]",
   "x_5_7": "x[8,6]",
   "x_5_8": "x[9,6]",
   "x_5_9": "x[10,6]",
   "x_6_0": "x[1,7]",
   "x_6_1": "x[2,7]",
   "x_6_2": "x[3,7]",
   "x_6_3": "x[4,7]",
   "x_6_4": "x[5,7]",
   "x_6_5": "x[6,7]",
   "x_6_6": "x[7,7]",
   "x_6_7": "x[8,7]",
   "x_6_8": "x[9,7]",
   "x_6_9": "x[10,7]",
   "x_7_0": "x[1,8]",
   "x_7_1": "x[2,8]",
   "x_7_2": "x[3,8]",
   "x_7_3": "x[4,8]",
   "x_7_4": "x[5,8]",
   "x_7_5": "x[6,8]",
   "x_7_6": "x[7,8]",
   "x_7_7": "x[8,8]",
   "x_7_8": "x[9,8]",
   "x_7_9": "x[10,8]",
   "x_8_0": "x[1,9]",
   "x_8_1": "x[2,9]",
   "x_8_2": "x[3,9]",
   "x_8_3": "x[4,9]",
   "x_8_4": "x[5,9]",
   "x_8_5": "x[6,9]",
   "x_8_6": "x[7,9]",
   "x_8_7": "x[8,9]",
   "x_8_8": "x[9,9]",
   "x_8_9": "x[10,9]",
   "x_9_0": "x[1,10]",
   "x_9_1": "x[2,10]",
   "x_9_2": "x[3,10]",
   "x_9_3": "x[4,10]",
   "x_9_4": "x[5,10]",
   "x_9_5": "x[6,10]",
   "x_9_6": "x[7,10]",
   "x_9_7": "x[8,10]",
   "x_9_8": "x[9,10]",
   "x_9_9": "x[10,10]",
   "x_10_0": "x[1,11]",
   "x_10_1": "x[2,11]",
   "x_10_2": "x[3,11]",
   "x_10_3": "x[4,11]",
   "x_10_4": "x[5,11]",
   "x_10_5": "x[6,11]",
   "x_10_6": "x[7,11]",
   "x_10_7": "x[8,11]",
   "x_10_8": "x[9,11]",
   "x_10_9": "x[10,11]",
   "x_11_0": "x[1,12]",
   "x_11_1": "x[2,12]",
   "x_11_2": "x[3,12]",
   "x_11_3": "x[4,12]",
   "x_11_4": "x[5,12]",
   "x_11_5": "x[6,12]",
   "x_11_6": "x[7,12]",
   "x_11_7": "x[8,12]",
   "x_11_8": "x[9,12]",
   "x_11_9": "x[10,12]",
   "x_12_0": "x[1,13]",
   "x_12_1": "x[2,13]",
   "x_12_2": "x[3,13]",
   "x_12_3": "x[4,13]",
   "x_12_4": "x[5,13]",
   "x_12_5": "x[6,13]",
   "x_12_6": "x[7,13]",
   "x_12_7": "x[8,13]",
   "x_12_8": "x[9,13]",
   "x_12_9": "x[10,13]",
   "x_13_0": "x[1,14]",
   "x_13_1": "x[2,14]",
   "x_13_2": "x[3,14]",
   "x_13_3": "x[4,14]",
   "x_13_4": "x[5,14]",
   "x_13_5": "x[6,14]",
   "x_13_6": "x[7,14]",
   "x_13_7": "x[8,14]",
   "x_13_8": "x[9,14]",
   "x_13_9": "x[10,14]",
   "x_14_0": "x[1,15]",
   "x_14_1": "x[2,15]",
   "x_14_2": "x[3,15]",
   "x_14_3": "x[4,15]",
   "x_14_4": "x[5,15]",
   "x_14_5": "x[6,15]",
   "x_14_6": "x[7,15]",
   "x_14_7": "x[8,15]",
   "x_14_8": "x[9,15]",
   "x_14_9": "x[10,15]",
   "x_15_0": "x[1,16]",
   "x_15_1": "x[2,16]",
   "x_15_2": "x[3,16]",
   "x_15_3": "x[4,16]",
   "x_15_4": "x[5,16]",
   "x_15_5": "x[6,16]",
   "x_15_6": "x[7,16]",
   "x_15_7": "x[8,16]",
   "x_15_8": "x[9,16]",
   "x_15_9": "x[10,16]",
   "x_16_0": "x[1,17]",
   "x_16_1": "x[2,17]",
   "x_16_2": "x[3,17]",
   "x_16_3": "x[4,17]",
   "x_16_4": "x[5,17]",
   "x_16_5": "x[6,17]",
   "x_16_6": "x[7,17]",
   "x_16_7": "x[8,17]",
   "x_16_8": "x[9,17]",
   "x_16_9": "x[10,17]",
   "x_17_0": "x[1,18]",
   "x_17_1": "x[2,18]",
   "x_17_2": "x[3,18]",
   "x_17_3": "x[4,18]",
   "x_17_4": "x[5,18]",
   "x_17_5": "x[6,18]",
   "x_17_6": "x[7,18]",
   "x_17_7": "x[8,18]",
   "x_17_8": "x[9,18]",
   "x_17_9": "x[10,18]",
   "x_18_0": "x[1,19]",
   "x_18_1": "x[2,19]",
   "x_18_2": "x[3,19]",
   "x_18_3": "x[4,19]",
   "x_18_4": "x[5,19]",
   "x_18_5": "x[6,19]",
   "x_18_6": "x[7,19]",
   "x_18_7": "x[8,19]",
   "x_18_8": "x[9,19]",
   "x_18_9": "x[10,19]",
   "x_19_0": "x[1,20]",
   "x_19_1": "x[2,20]",
   "x_19_2": "x[3,20]",
   "x_19_3": "x[4,20]",
   "x_19_4": "x[5,20]",
   "x_19_5": "x[6,20]",
   "x_19_6": "x[7,20]",
   "x_19_7": "x[8,20]",
   "x_19_8": "x[9,20]",
   "x_19_9": "x[10,20]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 41040.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     100.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     140.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     120.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     160.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     110.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     180.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     130.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     150.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     164.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     114.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-057": {
  "source_code_sha256": "c1aeadaada4f35e99be8179d9b498652b428bbc9bd90b3e26321436da580e5ac",
  "original_solution_sha256": "2a181e28654bd02a714bfbb8e8ca9ce75d92efa42887dd7e48b5d52c1f84b74e",
  "bindings": [
   [
    "C0",
    "x[1,Racing]"
   ],
   [
    "C1",
    "x[1,Sports]"
   ],
   [
    "C2",
    "x[1,Action]"
   ],
   [
    "C3",
    "x[1,Adventure]"
   ],
   [
    "C4",
    "x[1,RPG]"
   ],
   [
    "C5",
    "x[1,Shooter]"
   ],
   [
    "C6",
    "x[1,Strategy]"
   ],
   [
    "C7",
    "x[1,Simulation]"
   ],
   [
    "C8",
    "x[1,Puzzle]"
   ],
   [
    "C9",
    "x[1,Fighting]"
   ],
   [
    "C10",
    "x[1,Platformer]"
   ],
   [
    "C11",
    "x[1,Survival]"
   ],
   [
    "C12",
    "x[1,Horror]"
   ],
   [
    "C13",
    "x[1,Sandbox]"
   ],
   [
    "C14",
    "x[1,MMO]"
   ],
   [
    "C15",
    "x[2,Racing]"
   ],
   [
    "C16",
    "x[2,Sports]"
   ],
   [
    "C17",
    "x[2,Action]"
   ],
   [
    "C18",
    "x[2,Adventure]"
   ],
   [
    "C19",
    "x[2,RPG]"
   ],
   [
    "C20",
    "x[2,Shooter]"
   ],
   [
    "C21",
    "x[2,Strategy]"
   ],
   [
    "C22",
    "x[2,Simulation]"
   ],
   [
    "C23",
    "x[2,Puzzle]"
   ],
   [
    "C24",
    "x[2,Fighting]"
   ],
   [
    "C25",
    "x[2,Platformer]"
   ],
   [
    "C26",
    "x[2,Survival]"
   ],
   [
    "C27",
    "x[2,Horror]"
   ],
   [
    "C28",
    "x[2,Sandbox]"
   ],
   [
    "C29",
    "x[2,MMO]"
   ],
   [
    "C30",
    "x[3,Racing]"
   ],
   [
    "C31",
    "x[3,Sports]"
   ],
   [
    "C32",
    "x[3,Action]"
   ],
   [
    "C33",
    "x[3,Adventure]"
   ],
   [
    "C34",
    "x[3,RPG]"
   ],
   [
    "C35",
    "x[3,Shooter]"
   ],
   [
    "C36",
    "x[3,Strategy]"
   ],
   [
    "C37",
    "x[3,Simulation]"
   ],
   [
    "C38",
    "x[3,Puzzle]"
   ],
   [
    "C39",
    "x[3,Fighting]"
   ],
   [
    "C40",
    "x[3,Platformer]"
   ],
   [
    "C41",
    "x[3,Survival]"
   ],
   [
    "C42",
    "x[3,Horror]"
   ],
   [
    "C43",
    "x[3,Sandbox]"
   ],
   [
    "C44",
    "x[3,MMO]"
   ],
   [
    "C45",
    "x[4,Racing]"
   ],
   [
    "C46",
    "x[4,Sports]"
   ],
   [
    "C47",
    "x[4,Action]"
   ],
   [
    "C48",
    "x[4,Adventure]"
   ],
   [
    "C49",
    "x[4,RPG]"
   ],
   [
    "C50",
    "x[4,Shooter]"
   ],
   [
    "C51",
    "x[4,Strategy]"
   ],
   [
    "C52",
    "x[4,Simulation]"
   ],
   [
    "C53",
    "x[4,Puzzle]"
   ],
   [
    "C54",
    "x[4,Fighting]"
   ],
   [
    "C55",
    "x[4,Platformer]"
   ],
   [
    "C56",
    "x[4,Survival]"
   ],
   [
    "C57",
    "x[4,Horror]"
   ],
   [
    "C58",
    "x[4,Sandbox]"
   ],
   [
    "C59",
    "x[4,MMO]"
   ],
   [
    "C60",
    "x[5,Racing]"
   ],
   [
    "C61",
    "x[5,Sports]"
   ],
   [
    "C62",
    "x[5,Action]"
   ],
   [
    "C63",
    "x[5,Adventure]"
   ],
   [
    "C64",
    "x[5,RPG]"
   ],
   [
    "C65",
    "x[5,Shooter]"
   ],
   [
    "C66",
    "x[5,Strategy]"
   ],
   [
    "C67",
    "x[5,Simulation]"
   ],
   [
    "C68",
    "x[5,Puzzle]"
   ],
   [
    "C69",
    "x[5,Fighting]"
   ],
   [
    "C70",
    "x[5,Platformer]"
   ],
   [
    "C71",
    "x[5,Survival]"
   ],
   [
    "C72",
    "x[5,Horror]"
   ],
   [
    "C73",
    "x[5,Sandbox]"
   ],
   [
    "C74",
    "x[5,MMO]"
   ],
   [
    "C75",
    "x[6,Racing]"
   ],
   [
    "C76",
    "x[6,Sports]"
   ],
   [
    "C77",
    "x[6,Action]"
   ],
   [
    "C78",
    "x[6,Adventure]"
   ],
   [
    "C79",
    "x[6,RPG]"
   ],
   [
    "C80",
    "x[6,Shooter]"
   ],
   [
    "C81",
    "x[6,Strategy]"
   ],
   [
    "C82",
    "x[6,Simulation]"
   ],
   [
    "C83",
    "x[6,Puzzle]"
   ],
   [
    "C84",
    "x[6,Fighting]"
   ],
   [
    "C85",
    "x[6,Platformer]"
   ],
   [
    "C86",
    "x[6,Survival]"
   ],
   [
    "C87",
    "x[6,Horror]"
   ],
   [
    "C88",
    "x[6,Sandbox]"
   ],
   [
    "C89",
    "x[6,MMO]"
   ],
   [
    "C90",
    "x[7,Racing]"
   ],
   [
    "C91",
    "x[7,Sports]"
   ],
   [
    "C92",
    "x[7,Action]"
   ],
   [
    "C93",
    "x[7,Adventure]"
   ],
   [
    "C94",
    "x[7,RPG]"
   ],
   [
    "C95",
    "x[7,Shooter]"
   ],
   [
    "C96",
    "x[7,Strategy]"
   ],
   [
    "C97",
    "x[7,Simulation]"
   ],
   [
    "C98",
    "x[7,Puzzle]"
   ],
   [
    "C99",
    "x[7,Fighting]"
   ],
   [
    "C100",
    "x[7,Platformer]"
   ],
   [
    "C101",
    "x[7,Survival]"
   ],
   [
    "C102",
    "x[7,Horror]"
   ],
   [
    "C103",
    "x[7,Sandbox]"
   ],
   [
    "C104",
    "x[7,MMO]"
   ],
   [
    "C105",
    "x[8,Racing]"
   ],
   [
    "C106",
    "x[8,Sports]"
   ],
   [
    "C107",
    "x[8,Action]"
   ],
   [
    "C108",
    "x[8,Adventure]"
   ],
   [
    "C109",
    "x[8,RPG]"
   ],
   [
    "C110",
    "x[8,Shooter]"
   ],
   [
    "C111",
    "x[8,Strategy]"
   ],
   [
    "C112",
    "x[8,Simulation]"
   ],
   [
    "C113",
    "x[8,Puzzle]"
   ],
   [
    "C114",
    "x[8,Fighting]"
   ],
   [
    "C115",
    "x[8,Platformer]"
   ],
   [
    "C116",
    "x[8,Survival]"
   ],
   [
    "C117",
    "x[8,Horror]"
   ],
   [
    "C118",
    "x[8,Sandbox]"
   ],
   [
    "C119",
    "x[8,MMO]"
   ],
   [
    "C120",
    "x[9,Racing]"
   ],
   [
    "C121",
    "x[9,Sports]"
   ],
   [
    "C122",
    "x[9,Action]"
   ],
   [
    "C123",
    "x[9,Adventure]"
   ],
   [
    "C124",
    "x[9,RPG]"
   ],
   [
    "C125",
    "x[9,Shooter]"
   ],
   [
    "C126",
    "x[9,Strategy]"
   ],
   [
    "C127",
    "x[9,Simulation]"
   ],
   [
    "C128",
    "x[9,Puzzle]"
   ],
   [
    "C129",
    "x[9,Fighting]"
   ],
   [
    "C130",
    "x[9,Platformer]"
   ],
   [
    "C131",
    "x[9,Survival]"
   ],
   [
    "C132",
    "x[9,Horror]"
   ],
   [
    "C133",
    "x[9,Sandbox]"
   ],
   [
    "C134",
    "x[9,MMO]"
   ],
   [
    "C135",
    "x[10,Racing]"
   ],
   [
    "C136",
    "x[10,Sports]"
   ],
   [
    "C137",
    "x[10,Action]"
   ],
   [
    "C138",
    "x[10,Adventure]"
   ],
   [
    "C139",
    "x[10,RPG]"
   ],
   [
    "C140",
    "x[10,Shooter]"
   ],
   [
    "C141",
    "x[10,Strategy]"
   ],
   [
    "C142",
    "x[10,Simulation]"
   ],
   [
    "C143",
    "x[10,Puzzle]"
   ],
   [
    "C144",
    "x[10,Fighting]"
   ],
   [
    "C145",
    "x[10,Platformer]"
   ],
   [
    "C146",
    "x[10,Survival]"
   ],
   [
    "C147",
    "x[10,Horror]"
   ],
   [
    "C148",
    "x[10,Sandbox]"
   ],
   [
    "C149",
    "x[10,MMO]"
   ],
   [
    "C150",
    "x[11,Racing]"
   ],
   [
    "C151",
    "x[11,Sports]"
   ],
   [
    "C152",
    "x[11,Action]"
   ],
   [
    "C153",
    "x[11,Adventure]"
   ],
   [
    "C154",
    "x[11,RPG]"
   ],
   [
    "C155",
    "x[11,Shooter]"
   ],
   [
    "C156",
    "x[11,Strategy]"
   ],
   [
    "C157",
    "x[11,Simulation]"
   ],
   [
    "C158",
    "x[11,Puzzle]"
   ],
   [
    "C159",
    "x[11,Fighting]"
   ],
   [
    "C160",
    "x[11,Platformer]"
   ],
   [
    "C161",
    "x[11,Survival]"
   ],
   [
    "C162",
    "x[11,Horror]"
   ],
   [
    "C163",
    "x[11,Sandbox]"
   ],
   [
    "C164",
    "x[11,MMO]"
   ],
   [
    "C165",
    "x[12,Racing]"
   ],
   [
    "C166",
    "x[12,Sports]"
   ],
   [
    "C167",
    "x[12,Action]"
   ],
   [
    "C168",
    "x[12,Adventure]"
   ],
   [
    "C169",
    "x[12,RPG]"
   ],
   [
    "C170",
    "x[12,Shooter]"
   ],
   [
    "C171",
    "x[12,Strategy]"
   ],
   [
    "C172",
    "x[12,Simulation]"
   ],
   [
    "C173",
    "x[12,Puzzle]"
   ],
   [
    "C174",
    "x[12,Fighting]"
   ],
   [
    "C175",
    "x[12,Platformer]"
   ],
   [
    "C176",
    "x[12,Survival]"
   ],
   [
    "C177",
    "x[12,Horror]"
   ],
   [
    "C178",
    "x[12,Sandbox]"
   ],
   [
    "C179",
    "x[12,MMO]"
   ],
   [
    "C180",
    "x[13,Racing]"
   ],
   [
    "C181",
    "x[13,Sports]"
   ],
   [
    "C182",
    "x[13,Action]"
   ],
   [
    "C183",
    "x[13,Adventure]"
   ],
   [
    "C184",
    "x[13,RPG]"
   ],
   [
    "C185",
    "x[13,Shooter]"
   ],
   [
    "C186",
    "x[13,Strategy]"
   ],
   [
    "C187",
    "x[13,Simulation]"
   ],
   [
    "C188",
    "x[13,Puzzle]"
   ],
   [
    "C189",
    "x[13,Fighting]"
   ],
   [
    "C190",
    "x[13,Platformer]"
   ],
   [
    "C191",
    "x[13,Survival]"
   ],
   [
    "C192",
    "x[13,Horror]"
   ],
   [
    "C193",
    "x[13,Sandbox]"
   ],
   [
    "C194",
    "x[13,MMO]"
   ],
   [
    "C195",
    "x[14,Racing]"
   ],
   [
    "C196",
    "x[14,Sports]"
   ],
   [
    "C197",
    "x[14,Action]"
   ],
   [
    "C198",
    "x[14,Adventure]"
   ],
   [
    "C199",
    "x[14,RPG]"
   ],
   [
    "C200",
    "x[14,Shooter]"
   ],
   [
    "C201",
    "x[14,Strategy]"
   ],
   [
    "C202",
    "x[14,Simulation]"
   ],
   [
    "C203",
    "x[14,Puzzle]"
   ],
   [
    "C204",
    "x[14,Fighting]"
   ],
   [
    "C205",
    "x[14,Platformer]"
   ],
   [
    "C206",
    "x[14,Survival]"
   ],
   [
    "C207",
    "x[14,Horror]"
   ],
   [
    "C208",
    "x[14,Sandbox]"
   ],
   [
    "C209",
    "x[14,MMO]"
   ],
   [
    "C210",
    "x[15,Racing]"
   ],
   [
    "C211",
    "x[15,Sports]"
   ],
   [
    "C212",
    "x[15,Action]"
   ],
   [
    "C213",
    "x[15,Adventure]"
   ],
   [
    "C214",
    "x[15,RPG]"
   ],
   [
    "C215",
    "x[15,Shooter]"
   ],
   [
    "C216",
    "x[15,Strategy]"
   ],
   [
    "C217",
    "x[15,Simulation]"
   ],
   [
    "C218",
    "x[15,Puzzle]"
   ],
   [
    "C219",
    "x[15,Fighting]"
   ],
   [
    "C220",
    "x[15,Platformer]"
   ],
   [
    "C221",
    "x[15,Survival]"
   ],
   [
    "C222",
    "x[15,Horror]"
   ],
   [
    "C223",
    "x[15,Sandbox]"
   ],
   [
    "C224",
    "x[15,MMO]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Racing]",
   "x_0_1": "x[2,Racing]",
   "x_0_2": "x[3,Racing]",
   "x_0_3": "x[4,Racing]",
   "x_0_4": "x[5,Racing]",
   "x_0_5": "x[6,Racing]",
   "x_0_6": "x[7,Racing]",
   "x_0_7": "x[8,Racing]",
   "x_0_8": "x[9,Racing]",
   "x_0_9": "x[10,Racing]",
   "x_0_10": "x[11,Racing]",
   "x_0_11": "x[12,Racing]",
   "x_0_12": "x[13,Racing]",
   "x_0_13": "x[14,Racing]",
   "x_0_14": "x[15,Racing]",
   "x_1_0": "x[1,Sports]",
   "x_1_1": "x[2,Sports]",
   "x_1_2": "x[3,Sports]",
   "x_1_3": "x[4,Sports]",
   "x_1_4": "x[5,Sports]",
   "x_1_5": "x[6,Sports]",
   "x_1_6": "x[7,Sports]",
   "x_1_7": "x[8,Sports]",
   "x_1_8": "x[9,Sports]",
   "x_1_9": "x[10,Sports]",
   "x_1_10": "x[11,Sports]",
   "x_1_11": "x[12,Sports]",
   "x_1_12": "x[13,Sports]",
   "x_1_13": "x[14,Sports]",
   "x_1_14": "x[15,Sports]",
   "x_2_0": "x[1,Action]",
   "x_2_1": "x[2,Action]",
   "x_2_2": "x[3,Action]",
   "x_2_3": "x[4,Action]",
   "x_2_4": "x[5,Action]",
   "x_2_5": "x[6,Action]",
   "x_2_6": "x[7,Action]",
   "x_2_7": "x[8,Action]",
   "x_2_8": "x[9,Action]",
   "x_2_9": "x[10,Action]",
   "x_2_10": "x[11,Action]",
   "x_2_11": "x[12,Action]",
   "x_2_12": "x[13,Action]",
   "x_2_13": "x[14,Action]",
   "x_2_14": "x[15,Action]",
   "x_3_0": "x[1,Adventure]",
   "x_3_1": "x[2,Adventure]",
   "x_3_2": "x[3,Adventure]",
   "x_3_3": "x[4,Adventure]",
   "x_3_4": "x[5,Adventure]",
   "x_3_5": "x[6,Adventure]",
   "x_3_6": "x[7,Adventure]",
   "x_3_7": "x[8,Adventure]",
   "x_3_8": "x[9,Adventure]",
   "x_3_9": "x[10,Adventure]",
   "x_3_10": "x[11,Adventure]",
   "x_3_11": "x[12,Adventure]",
   "x_3_12": "x[13,Adventure]",
   "x_3_13": "x[14,Adventure]",
   "x_3_14": "x[15,Adventure]",
   "x_4_0": "x[1,RPG]",
   "x_4_1": "x[2,RPG]",
   "x_4_2": "x[3,RPG]",
   "x_4_3": "x[4,RPG]",
   "x_4_4": "x[5,RPG]",
   "x_4_5": "x[6,RPG]",
   "x_4_6": "x[7,RPG]",
   "x_4_7": "x[8,RPG]",
   "x_4_8": "x[9,RPG]",
   "x_4_9": "x[10,RPG]",
   "x_4_10": "x[11,RPG]",
   "x_4_11": "x[12,RPG]",
   "x_4_12": "x[13,RPG]",
   "x_4_13": "x[14,RPG]",
   "x_4_14": "x[15,RPG]",
   "x_5_0": "x[1,Shooter]",
   "x_5_1": "x[2,Shooter]",
   "x_5_2": "x[3,Shooter]",
   "x_5_3": "x[4,Shooter]",
   "x_5_4": "x[5,Shooter]",
   "x_5_5": "x[6,Shooter]",
   "x_5_6": "x[7,Shooter]",
   "x_5_7": "x[8,Shooter]",
   "x_5_8": "x[9,Shooter]",
   "x_5_9": "x[10,Shooter]",
   "x_5_10": "x[11,Shooter]",
   "x_5_11": "x[12,Shooter]",
   "x_5_12": "x[13,Shooter]",
   "x_5_13": "x[14,Shooter]",
   "x_5_14": "x[15,Shooter]",
   "x_6_0": "x[1,Strategy]",
   "x_6_1": "x[2,Strategy]",
   "x_6_2": "x[3,Strategy]",
   "x_6_3": "x[4,Strategy]",
   "x_6_4": "x[5,Strategy]",
   "x_6_5": "x[6,Strategy]",
   "x_6_6": "x[7,Strategy]",
   "x_6_7": "x[8,Strategy]",
   "x_6_8": "x[9,Strategy]",
   "x_6_9": "x[10,Strategy]",
   "x_6_10": "x[11,Strategy]",
   "x_6_11": "x[12,Strategy]",
   "x_6_12": "x[13,Strategy]",
   "x_6_13": "x[14,Strategy]",
   "x_6_14": "x[15,Strategy]",
   "x_7_0": "x[1,Simulation]",
   "x_7_1": "x[2,Simulation]",
   "x_7_2": "x[3,Simulation]",
   "x_7_3": "x[4,Simulation]",
   "x_7_4": "x[5,Simulation]",
   "x_7_5": "x[6,Simulation]",
   "x_7_6": "x[7,Simulation]",
   "x_7_7": "x[8,Simulation]",
   "x_7_8": "x[9,Simulation]",
   "x_7_9": "x[10,Simulation]",
   "x_7_10": "x[11,Simulation]",
   "x_7_11": "x[12,Simulation]",
   "x_7_12": "x[13,Simulation]",
   "x_7_13": "x[14,Simulation]",
   "x_7_14": "x[15,Simulation]",
   "x_8_0": "x[1,Puzzle]",
   "x_8_1": "x[2,Puzzle]",
   "x_8_2": "x[3,Puzzle]",
   "x_8_3": "x[4,Puzzle]",
   "x_8_4": "x[5,Puzzle]",
   "x_8_5": "x[6,Puzzle]",
   "x_8_6": "x[7,Puzzle]",
   "x_8_7": "x[8,Puzzle]",
   "x_8_8": "x[9,Puzzle]",
   "x_8_9": "x[10,Puzzle]",
   "x_8_10": "x[11,Puzzle]",
   "x_8_11": "x[12,Puzzle]",
   "x_8_12": "x[13,Puzzle]",
   "x_8_13": "x[14,Puzzle]",
   "x_8_14": "x[15,Puzzle]",
   "x_9_0": "x[1,Fighting]",
   "x_9_1": "x[2,Fighting]",
   "x_9_2": "x[3,Fighting]",
   "x_9_3": "x[4,Fighting]",
   "x_9_4": "x[5,Fighting]",
   "x_9_5": "x[6,Fighting]",
   "x_9_6": "x[7,Fighting]",
   "x_9_7": "x[8,Fighting]",
   "x_9_8": "x[9,Fighting]",
   "x_9_9": "x[10,Fighting]",
   "x_9_10": "x[11,Fighting]",
   "x_9_11": "x[12,Fighting]",
   "x_9_12": "x[13,Fighting]",
   "x_9_13": "x[14,Fighting]",
   "x_9_14": "x[15,Fighting]",
   "x_10_0": "x[1,Platformer]",
   "x_10_1": "x[2,Platformer]",
   "x_10_2": "x[3,Platformer]",
   "x_10_3": "x[4,Platformer]",
   "x_10_4": "x[5,Platformer]",
   "x_10_5": "x[6,Platformer]",
   "x_10_6": "x[7,Platformer]",
   "x_10_7": "x[8,Platformer]",
   "x_10_8": "x[9,Platformer]",
   "x_10_9": "x[10,Platformer]",
   "x_10_10": "x[11,Platformer]",
   "x_10_11": "x[12,Platformer]",
   "x_10_12": "x[13,Platformer]",
   "x_10_13": "x[14,Platformer]",
   "x_10_14": "x[15,Platformer]",
   "x_11_0": "x[1,Survival]",
   "x_11_1": "x[2,Survival]",
   "x_11_2": "x[3,Survival]",
   "x_11_3": "x[4,Survival]",
   "x_11_4": "x[5,Survival]",
   "x_11_5": "x[6,Survival]",
   "x_11_6": "x[7,Survival]",
   "x_11_7": "x[8,Survival]",
   "x_11_8": "x[9,Survival]",
   "x_11_9": "x[10,Survival]",
   "x_11_10": "x[11,Survival]",
   "x_11_11": "x[12,Survival]",
   "x_11_12": "x[13,Survival]",
   "x_11_13": "x[14,Survival]",
   "x_11_14": "x[15,Survival]",
   "x_12_0": "x[1,Horror]",
   "x_12_1": "x[2,Horror]",
   "x_12_2": "x[3,Horror]",
   "x_12_3": "x[4,Horror]",
   "x_12_4": "x[5,Horror]",
   "x_12_5": "x[6,Horror]",
   "x_12_6": "x[7,Horror]",
   "x_12_7": "x[8,Horror]",
   "x_12_8": "x[9,Horror]",
   "x_12_9": "x[10,Horror]",
   "x_12_10": "x[11,Horror]",
   "x_12_11": "x[12,Horror]",
   "x_12_12": "x[13,Horror]",
   "x_12_13": "x[14,Horror]",
   "x_12_14": "x[15,Horror]",
   "x_13_0": "x[1,Sandbox]",
   "x_13_1": "x[2,Sandbox]",
   "x_13_2": "x[3,Sandbox]",
   "x_13_3": "x[4,Sandbox]",
   "x_13_4": "x[5,Sandbox]",
   "x_13_5": "x[6,Sandbox]",
   "x_13_6": "x[7,Sandbox]",
   "x_13_7": "x[8,Sandbox]",
   "x_13_8": "x[9,Sandbox]",
   "x_13_9": "x[10,Sandbox]",
   "x_13_10": "x[11,Sandbox]",
   "x_13_11": "x[12,Sandbox]",
   "x_13_12": "x[13,Sandbox]",
   "x_13_13": "x[14,Sandbox]",
   "x_13_14": "x[15,Sandbox]",
   "x_14_0": "x[1,MMO]",
   "x_14_1": "x[2,MMO]",
   "x_14_2": "x[3,MMO]",
   "x_14_3": "x[4,MMO]",
   "x_14_4": "x[5,MMO]",
   "x_14_5": "x[6,MMO]",
   "x_14_6": "x[7,MMO]",
   "x_14_7": "x[8,MMO]",
   "x_14_8": "x[9,MMO]",
   "x_14_9": "x[10,MMO]",
   "x_14_10": "x[11,MMO]",
   "x_14_11": "x[12,MMO]",
   "x_14_12": "x[13,MMO]",
   "x_14_13": "x[14,MMO]",
   "x_14_14": "x[15,MMO]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 12502.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     7.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     9.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     7.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     7.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     12.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     6.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     8.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     4.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     6.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     4.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     13.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     12.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     9.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     15.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ],
    [
     "C200",
     -0.0
    ],
    [
     "C201",
     -0.0
    ],
    [
     "C202",
     -0.0
    ],
    [
     "C203",
     -0.0
    ],
    [
     "C204",
     -0.0
    ],
    [
     "C205",
     -0.0
    ],
    [
     "C206",
     -0.0
    ],
    [
     "C207",
     -0.0
    ],
    [
     "C208",
     -0.0
    ],
    [
     "C209",
     -0.0
    ],
    [
     "C210",
     -0.0
    ],
    [
     "C211",
     -0.0
    ],
    [
     "C212",
     14.0
    ],
    [
     "C213",
     -0.0
    ],
    [
     "C214",
     -0.0
    ],
    [
     "C215",
     -0.0
    ],
    [
     "C216",
     -0.0
    ],
    [
     "C217",
     -0.0
    ],
    [
     "C218",
     -0.0
    ],
    [
     "C219",
     -0.0
    ],
    [
     "C220",
     -0.0
    ],
    [
     "C221",
     -0.0
    ],
    [
     "C222",
     -0.0
    ],
    [
     "C223",
     -0.0
    ],
    [
     "C224",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-061": {
  "source_code_sha256": "04f6c78bed04664f55a1eb7843f4817816d8fdb64d8eef8c25975aeec60dfd39",
  "original_solution_sha256": "d92f64bab01558001162fdea44c7b01f5672230ca63f47996fcd42b1ff7d96ab",
  "bindings": [
   [
    "C0",
    "y[S2]"
   ],
   [
    "C1",
    "y[S3]"
   ],
   [
    "C2",
    "y[S4]"
   ],
   [
    "C3",
    "y[S1]"
   ],
   [
    "C4",
    "y[S5]"
   ],
   [
    "C5",
    "x[S2,C5]"
   ],
   [
    "C6",
    "x[S2,C2]"
   ],
   [
    "C7",
    "x[S2,C4]"
   ],
   [
    "C8",
    "x[S2,C1]"
   ],
   [
    "C9",
    "x[S2,C3]"
   ],
   [
    "C10",
    "x[S3,C5]"
   ],
   [
    "C11",
    "x[S3,C2]"
   ],
   [
    "C12",
    "x[S3,C4]"
   ],
   [
    "C13",
    "x[S3,C1]"
   ],
   [
    "C14",
    "x[S3,C3]"
   ],
   [
    "C15",
    "x[S4,C5]"
   ],
   [
    "C16",
    "x[S4,C2]"
   ],
   [
    "C17",
    "x[S4,C4]"
   ],
   [
    "C18",
    "x[S4,C1]"
   ],
   [
    "C19",
    "x[S4,C3]"
   ],
   [
    "C20",
    "x[S1,C5]"
   ],
   [
    "C21",
    "x[S1,C2]"
   ],
   [
    "C22",
    "x[S1,C4]"
   ],
   [
    "C23",
    "x[S1,C1]"
   ],
   [
    "C24",
    "x[S1,C3]"
   ],
   [
    "C25",
    "x[S5,C5]"
   ],
   [
    "C26",
    "x[S5,C2]"
   ],
   [
    "C27",
    "x[S5,C4]"
   ],
   [
    "C28",
    "x[S5,C1]"
   ],
   [
    "C29",
    "x[S5,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]",
   "y_3": "y[S4]",
   "y_4": "y[S5]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 10199.960000000001,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     1.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     3.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     25.0
    ],
    [
     "C13",
     143.0
    ],
    [
     "C14",
     10.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     6.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-065": {
  "source_code_sha256": "0ad7b6116fdb11ddf5510760274c959f088ab9c4575f00c61bcecb9cd066981b",
  "original_solution_sha256": "b11edddfe873258140548bab8337f5b7e0dc62101ad5cfd5b581cf5bedb4453f",
  "bindings": [
   [
    "C0",
    "y[S1]"
   ],
   [
    "C1",
    "y[S2]"
   ],
   [
    "C2",
    "y[S3]"
   ],
   [
    "C3",
    "x[S1,C1]"
   ],
   [
    "C4",
    "x[S1,C2]"
   ],
   [
    "C5",
    "x[S1,C3]"
   ],
   [
    "C6",
    "x[S2,C1]"
   ],
   [
    "C7",
    "x[S2,C2]"
   ],
   [
    "C8",
    "x[S2,C3]"
   ],
   [
    "C9",
    "x[S3,C1]"
   ],
   [
    "C10",
    "x[S3,C2]"
   ],
   [
    "C11",
    "x[S3,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 317318.5,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     776.0
    ],
    [
     "C5",
     16214.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     1083.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-069": {
  "source_code_sha256": "465bc2b623017f637639a33aa7ec00129438e596c943bfe9b0099ec86ca2279f",
  "original_solution_sha256": "167914019b6a244d702ecd4d123caf4300e526dfc1991d0e0048c5d4fcc580b7",
  "bindings": [
   [
    "C0",
    "x[Manager 1,1]"
   ],
   [
    "C1",
    "x[Manager 1,2]"
   ],
   [
    "C2",
    "x[Manager 1,3]"
   ],
   [
    "C3",
    "x[Manager 1,4]"
   ],
   [
    "C4",
    "x[Manager 1,5]"
   ],
   [
    "C5",
    "x[Manager 1,6]"
   ],
   [
    "C6",
    "x[Manager 1,7]"
   ],
   [
    "C7",
    "x[Manager 1,8]"
   ],
   [
    "C8",
    "x[Manager 1,9]"
   ],
   [
    "C9",
    "x[Manager 1,10]"
   ],
   [
    "C10",
    "x[Manager 1,11]"
   ],
   [
    "C11",
    "x[Manager 2,1]"
   ],
   [
    "C12",
    "x[Manager 2,2]"
   ],
   [
    "C13",
    "x[Manager 2,3]"
   ],
   [
    "C14",
    "x[Manager 2,4]"
   ],
   [
    "C15",
    "x[Manager 2,5]"
   ],
   [
    "C16",
    "x[Manager 2,6]"
   ],
   [
    "C17",
    "x[Manager 2,7]"
   ],
   [
    "C18",
    "x[Manager 2,8]"
   ],
   [
    "C19",
    "x[Manager 2,9]"
   ],
   [
    "C20",
    "x[Manager 2,10]"
   ],
   [
    "C21",
    "x[Manager 2,11]"
   ],
   [
    "C22",
    "x[Manager 3,1]"
   ],
   [
    "C23",
    "x[Manager 3,2]"
   ],
   [
    "C24",
    "x[Manager 3,3]"
   ],
   [
    "C25",
    "x[Manager 3,4]"
   ],
   [
    "C26",
    "x[Manager 3,5]"
   ],
   [
    "C27",
    "x[Manager 3,6]"
   ],
   [
    "C28",
    "x[Manager 3,7]"
   ],
   [
    "C29",
    "x[Manager 3,8]"
   ],
   [
    "C30",
    "x[Manager 3,9]"
   ],
   [
    "C31",
    "x[Manager 3,10]"
   ],
   [
    "C32",
    "x[Manager 3,11]"
   ],
   [
    "C33",
    "x[Manager 4,1]"
   ],
   [
    "C34",
    "x[Manager 4,2]"
   ],
   [
    "C35",
    "x[Manager 4,3]"
   ],
   [
    "C36",
    "x[Manager 4,4]"
   ],
   [
    "C37",
    "x[Manager 4,5]"
   ],
   [
    "C38",
    "x[Manager 4,6]"
   ],
   [
    "C39",
    "x[Manager 4,7]"
   ],
   [
    "C40",
    "x[Manager 4,8]"
   ],
   [
    "C41",
    "x[Manager 4,9]"
   ],
   [
    "C42",
    "x[Manager 4,10]"
   ],
   [
    "C43",
    "x[Manager 4,11]"
   ],
   [
    "C44",
    "x[Manager 5,1]"
   ],
   [
    "C45",
    "x[Manager 5,2]"
   ],
   [
    "C46",
    "x[Manager 5,3]"
   ],
   [
    "C47",
    "x[Manager 5,4]"
   ],
   [
    "C48",
    "x[Manager 5,5]"
   ],
   [
    "C49",
    "x[Manager 5,6]"
   ],
   [
    "C50",
    "x[Manager 5,7]"
   ],
   [
    "C51",
    "x[Manager 5,8]"
   ],
   [
    "C52",
    "x[Manager 5,9]"
   ],
   [
    "C53",
    "x[Manager 5,10]"
   ],
   [
    "C54",
    "x[Manager 5,11]"
   ],
   [
    "C55",
    "x[Manager 6,1]"
   ],
   [
    "C56",
    "x[Manager 6,2]"
   ],
   [
    "C57",
    "x[Manager 6,3]"
   ],
   [
    "C58",
    "x[Manager 6,4]"
   ],
   [
    "C59",
    "x[Manager 6,5]"
   ],
   [
    "C60",
    "x[Manager 6,6]"
   ],
   [
    "C61",
    "x[Manager 6,7]"
   ],
   [
    "C62",
    "x[Manager 6,8]"
   ],
   [
    "C63",
    "x[Manager 6,9]"
   ],
   [
    "C64",
    "x[Manager 6,10]"
   ],
   [
    "C65",
    "x[Manager 6,11]"
   ],
   [
    "C66",
    "x[Manager 7,1]"
   ],
   [
    "C67",
    "x[Manager 7,2]"
   ],
   [
    "C68",
    "x[Manager 7,3]"
   ],
   [
    "C69",
    "x[Manager 7,4]"
   ],
   [
    "C70",
    "x[Manager 7,5]"
   ],
   [
    "C71",
    "x[Manager 7,6]"
   ],
   [
    "C72",
    "x[Manager 7,7]"
   ],
   [
    "C73",
    "x[Manager 7,8]"
   ],
   [
    "C74",
    "x[Manager 7,9]"
   ],
   [
    "C75",
    "x[Manager 7,10]"
   ],
   [
    "C76",
    "x[Manager 7,11]"
   ],
   [
    "C77",
    "x[Manager 8,1]"
   ],
   [
    "C78",
    "x[Manager 8,2]"
   ],
   [
    "C79",
    "x[Manager 8,3]"
   ],
   [
    "C80",
    "x[Manager 8,4]"
   ],
   [
    "C81",
    "x[Manager 8,5]"
   ],
   [
    "C82",
    "x[Manager 8,6]"
   ],
   [
    "C83",
    "x[Manager 8,7]"
   ],
   [
    "C84",
    "x[Manager 8,8]"
   ],
   [
    "C85",
    "x[Manager 8,9]"
   ],
   [
    "C86",
    "x[Manager 8,10]"
   ],
   [
    "C87",
    "x[Manager 8,11]"
   ],
   [
    "C88",
    "x[Manager 9,1]"
   ],
   [
    "C89",
    "x[Manager 9,2]"
   ],
   [
    "C90",
    "x[Manager 9,3]"
   ],
   [
    "C91",
    "x[Manager 9,4]"
   ],
   [
    "C92",
    "x[Manager 9,5]"
   ],
   [
    "C93",
    "x[Manager 9,6]"
   ],
   [
    "C94",
    "x[Manager 9,7]"
   ],
   [
    "C95",
    "x[Manager 9,8]"
   ],
   [
    "C96",
    "x[Manager 9,9]"
   ],
   [
    "C97",
    "x[Manager 9,10]"
   ],
   [
    "C98",
    "x[Manager 9,11]"
   ],
   [
    "C99",
    "x[Manager 10,1]"
   ],
   [
    "C100",
    "x[Manager 10,2]"
   ],
   [
    "C101",
    "x[Manager 10,3]"
   ],
   [
    "C102",
    "x[Manager 10,4]"
   ],
   [
    "C103",
    "x[Manager 10,5]"
   ],
   [
    "C104",
    "x[Manager 10,6]"
   ],
   [
    "C105",
    "x[Manager 10,7]"
   ],
   [
    "C106",
    "x[Manager 10,8]"
   ],
   [
    "C107",
    "x[Manager 10,9]"
   ],
   [
    "C108",
    "x[Manager 10,10]"
   ],
   [
    "C109",
    "x[Manager 10,11]"
   ],
   [
    "C110",
    "x[Manager 11,1]"
   ],
   [
    "C111",
    "x[Manager 11,2]"
   ],
   [
    "C112",
    "x[Manager 11,3]"
   ],
   [
    "C113",
    "x[Manager 11,4]"
   ],
   [
    "C114",
    "x[Manager 11,5]"
   ],
   [
    "C115",
    "x[Manager 11,6]"
   ],
   [
    "C116",
    "x[Manager 11,7]"
   ],
   [
    "C117",
    "x[Manager 11,8]"
   ],
   [
    "C118",
    "x[Manager 11,9]"
   ],
   [
    "C119",
    "x[Manager 11,10]"
   ],
   [
    "C120",
    "x[Manager 11,11]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Manager 1,1]",
   "x_0_1": "x[Manager 1,2]",
   "x_0_2": "x[Manager 1,3]",
   "x_0_3": "x[Manager 1,4]",
   "x_0_4": "x[Manager 1,5]",
   "x_0_5": "x[Manager 1,6]",
   "x_0_6": "x[Manager 1,7]",
   "x_0_7": "x[Manager 1,8]",
   "x_0_8": "x[Manager 1,9]",
   "x_0_9": "x[Manager 1,10]",
   "x_0_10": "x[Manager 1,11]",
   "x_1_0": "x[Manager 2,1]",
   "x_1_1": "x[Manager 2,2]",
   "x_1_2": "x[Manager 2,3]",
   "x_1_3": "x[Manager 2,4]",
   "x_1_4": "x[Manager 2,5]",
   "x_1_5": "x[Manager 2,6]",
   "x_1_6": "x[Manager 2,7]",
   "x_1_7": "x[Manager 2,8]",
   "x_1_8": "x[Manager 2,9]",
   "x_1_9": "x[Manager 2,10]",
   "x_1_10": "x[Manager 2,11]",
   "x_2_0": "x[Manager 3,1]",
   "x_2_1": "x[Manager 3,2]",
   "x_2_2": "x[Manager 3,3]",
   "x_2_3": "x[Manager 3,4]",
   "x_2_4": "x[Manager 3,5]",
   "x_2_5": "x[Manager 3,6]",
   "x_2_6": "x[Manager 3,7]",
   "x_2_7": "x[Manager 3,8]",
   "x_2_8": "x[Manager 3,9]",
   "x_2_9": "x[Manager 3,10]",
   "x_2_10": "x[Manager 3,11]",
   "x_3_0": "x[Manager 4,1]",
   "x_3_1": "x[Manager 4,2]",
   "x_3_2": "x[Manager 4,3]",
   "x_3_3": "x[Manager 4,4]",
   "x_3_4": "x[Manager 4,5]",
   "x_3_5": "x[Manager 4,6]",
   "x_3_6": "x[Manager 4,7]",
   "x_3_7": "x[Manager 4,8]",
   "x_3_8": "x[Manager 4,9]",
   "x_3_9": "x[Manager 4,10]",
   "x_3_10": "x[Manager 4,11]",
   "x_4_0": "x[Manager 5,1]",
   "x_4_1": "x[Manager 5,2]",
   "x_4_2": "x[Manager 5,3]",
   "x_4_3": "x[Manager 5,4]",
   "x_4_4": "x[Manager 5,5]",
   "x_4_5": "x[Manager 5,6]",
   "x_4_6": "x[Manager 5,7]",
   "x_4_7": "x[Manager 5,8]",
   "x_4_8": "x[Manager 5,9]",
   "x_4_9": "x[Manager 5,10]",
   "x_4_10": "x[Manager 5,11]",
   "x_5_0": "x[Manager 6,1]",
   "x_5_1": "x[Manager 6,2]",
   "x_5_2": "x[Manager 6,3]",
   "x_5_3": "x[Manager 6,4]",
   "x_5_4": "x[Manager 6,5]",
   "x_5_5": "x[Manager 6,6]",
   "x_5_6": "x[Manager 6,7]",
   "x_5_7": "x[Manager 6,8]",
   "x_5_8": "x[Manager 6,9]",
   "x_5_9": "x[Manager 6,10]",
   "x_5_10": "x[Manager 6,11]",
   "x_6_0": "x[Manager 7,1]",
   "x_6_1": "x[Manager 7,2]",
   "x_6_2": "x[Manager 7,3]",
   "x_6_3": "x[Manager 7,4]",
   "x_6_4": "x[Manager 7,5]",
   "x_6_5": "x[Manager 7,6]",
   "x_6_6": "x[Manager 7,7]",
   "x_6_7": "x[Manager 7,8]",
   "x_6_8": "x[Manager 7,9]",
   "x_6_9": "x[Manager 7,10]",
   "x_6_10": "x[Manager 7,11]",
   "x_7_0": "x[Manager 8,1]",
   "x_7_1": "x[Manager 8,2]",
   "x_7_2": "x[Manager 8,3]",
   "x_7_3": "x[Manager 8,4]",
   "x_7_4": "x[Manager 8,5]",
   "x_7_5": "x[Manager 8,6]",
   "x_7_6": "x[Manager 8,7]",
   "x_7_7": "x[Manager 8,8]",
   "x_7_8": "x[Manager 8,9]",
   "x_7_9": "x[Manager 8,10]",
   "x_7_10": "x[Manager 8,11]",
   "x_8_0": "x[Manager 9,1]",
   "x_8_1": "x[Manager 9,2]",
   "x_8_2": "x[Manager 9,3]",
   "x_8_3": "x[Manager 9,4]",
   "x_8_4": "x[Manager 9,5]",
   "x_8_5": "x[Manager 9,6]",
   "x_8_6": "x[Manager 9,7]",
   "x_8_7": "x[Manager 9,8]",
   "x_8_8": "x[Manager 9,9]",
   "x_8_9": "x[Manager 9,10]",
   "x_8_10": "x[Manager 9,11]",
   "x_9_0": "x[Manager 10,1]",
   "x_9_1": "x[Manager 10,2]",
   "x_9_2": "x[Manager 10,3]",
   "x_9_3": "x[Manager 10,4]",
   "x_9_4": "x[Manager 10,5]",
   "x_9_5": "x[Manager 10,6]",
   "x_9_6": "x[Manager 10,7]",
   "x_9_7": "x[Manager 10,8]",
   "x_9_8": "x[Manager 10,9]",
   "x_9_9": "x[Manager 10,10]",
   "x_9_10": "x[Manager 10,11]",
   "x_10_0": "x[Manager 11,1]",
   "x_10_1": "x[Manager 11,2]",
   "x_10_2": "x[Manager 11,3]",
   "x_10_3": "x[Manager 11,4]",
   "x_10_4": "x[Manager 11,5]",
   "x_10_5": "x[Manager 11,6]",
   "x_10_6": "x[Manager 11,7]",
   "x_10_7": "x[Manager 11,8]",
   "x_10_8": "x[Manager 11,9]",
   "x_10_9": "x[Manager 11,10]",
   "x_10_10": "x[Manager 11,11]"
  },
  "mapping_basis": "reviewed_oss_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 4903.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     1.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     1.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     1.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     1.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     1.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     1.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     1.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     1.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     1.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     1.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-073": {
  "source_code_sha256": "22864ceb7c010e61d5290c12c7cee0b2303a14d4d03f62e6cfdcf1e48a899532",
  "original_solution_sha256": "5b5656aba3c5900a1aed13f15d3ccf78a5004e9bcb9491807f7246100a9a2906",
  "bindings": [
   [
    "C0",
    "x[I,A1]"
   ],
   [
    "C1",
    "x[II,A1]"
   ],
   [
    "C2",
    "x[I,A2]"
   ],
   [
    "C3",
    "x[II,A2]"
   ],
   [
    "C4",
    "x[III,A2]"
   ],
   [
    "C5",
    "x[I,B1]"
   ],
   [
    "C6",
    "x[II,B1]"
   ],
   [
    "C7",
    "x[I,B2]"
   ],
   [
    "C8",
    "x[III,B2]"
   ],
   [
    "C9",
    "x[I,B3]"
   ]
  ],
  "primary_mapping": {
   "x[B1,I]": "x[I,B1]",
   "x[A2,I]": "x[I,A2]",
   "x[A1,I]": "x[I,A1]",
   "x[A2,III]": "x[III,A2]",
   "x[B2,I]": "x[I,B2]",
   "x[B3,I]": "x[I,B3]",
   "x[A2,II]": "x[II,A2]",
   "x[B2,III]": "x[III,B2]",
   "x[A1,II]": "x[II,A1]",
   "x[B1,II]": "x[II,B1]"
  },
  "mapping_basis": "explicit_physical_axes_review",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": -0.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-079": {
  "source_code_sha256": "1ea4a36d6fb8519c64e68cf94f24e8cdf2095bef46cc62979abcd90bd0580feb",
  "original_solution_sha256": "91f31399ac67c34cfef14991f8216e0d4205fcab12a6206ed327b1b2ac2a3a38",
  "bindings": [
   [
    "C0",
    "y[A1]"
   ],
   [
    "C1",
    "y[A2]"
   ],
   [
    "C2",
    "y[A3]"
   ],
   [
    "C3",
    "y[A4]"
   ],
   [
    "C4",
    "y[A5]"
   ],
   [
    "C5",
    "y[A6]"
   ],
   [
    "C6",
    "y[A7]"
   ],
   [
    "C7",
    "y[A8]"
   ],
   [
    "C8",
    "y[A9]"
   ],
   [
    "C9",
    "y[A10]"
   ],
   [
    "C10",
    "y[A11]"
   ],
   [
    "C11",
    "y[A12]"
   ],
   [
    "C12",
    "y[A13]"
   ],
   [
    "C13",
    "y[A14]"
   ],
   [
    "C14",
    "y[A15]"
   ],
   [
    "C15",
    "x[A1,B1]"
   ],
   [
    "C16",
    "x[A1,B2]"
   ],
   [
    "C17",
    "x[A1,B3]"
   ],
   [
    "C18",
    "x[A1,B4]"
   ],
   [
    "C19",
    "x[A1,B5]"
   ],
   [
    "C20",
    "x[A1,B6]"
   ],
   [
    "C21",
    "x[A1,B7]"
   ],
   [
    "C22",
    "x[A1,B8]"
   ],
   [
    "C23",
    "x[A2,B1]"
   ],
   [
    "C24",
    "x[A2,B2]"
   ],
   [
    "C25",
    "x[A2,B3]"
   ],
   [
    "C26",
    "x[A2,B4]"
   ],
   [
    "C27",
    "x[A2,B5]"
   ],
   [
    "C28",
    "x[A2,B6]"
   ],
   [
    "C29",
    "x[A2,B7]"
   ],
   [
    "C30",
    "x[A2,B8]"
   ],
   [
    "C31",
    "x[A3,B1]"
   ],
   [
    "C32",
    "x[A3,B2]"
   ],
   [
    "C33",
    "x[A3,B3]"
   ],
   [
    "C34",
    "x[A3,B4]"
   ],
   [
    "C35",
    "x[A3,B5]"
   ],
   [
    "C36",
    "x[A3,B6]"
   ],
   [
    "C37",
    "x[A3,B7]"
   ],
   [
    "C38",
    "x[A3,B8]"
   ],
   [
    "C39",
    "x[A4,B1]"
   ],
   [
    "C40",
    "x[A4,B2]"
   ],
   [
    "C41",
    "x[A4,B3]"
   ],
   [
    "C42",
    "x[A4,B4]"
   ],
   [
    "C43",
    "x[A4,B5]"
   ],
   [
    "C44",
    "x[A4,B6]"
   ],
   [
    "C45",
    "x[A4,B7]"
   ],
   [
    "C46",
    "x[A4,B8]"
   ],
   [
    "C47",
    "x[A5,B1]"
   ],
   [
    "C48",
    "x[A5,B2]"
   ],
   [
    "C49",
    "x[A5,B3]"
   ],
   [
    "C50",
    "x[A5,B4]"
   ],
   [
    "C51",
    "x[A5,B5]"
   ],
   [
    "C52",
    "x[A5,B6]"
   ],
   [
    "C53",
    "x[A5,B7]"
   ],
   [
    "C54",
    "x[A5,B8]"
   ],
   [
    "C55",
    "x[A6,B1]"
   ],
   [
    "C56",
    "x[A6,B2]"
   ],
   [
    "C57",
    "x[A6,B3]"
   ],
   [
    "C58",
    "x[A6,B4]"
   ],
   [
    "C59",
    "x[A6,B5]"
   ],
   [
    "C60",
    "x[A6,B6]"
   ],
   [
    "C61",
    "x[A6,B7]"
   ],
   [
    "C62",
    "x[A6,B8]"
   ],
   [
    "C63",
    "x[A7,B1]"
   ],
   [
    "C64",
    "x[A7,B2]"
   ],
   [
    "C65",
    "x[A7,B3]"
   ],
   [
    "C66",
    "x[A7,B4]"
   ],
   [
    "C67",
    "x[A7,B5]"
   ],
   [
    "C68",
    "x[A7,B6]"
   ],
   [
    "C69",
    "x[A7,B7]"
   ],
   [
    "C70",
    "x[A7,B8]"
   ],
   [
    "C71",
    "x[A8,B1]"
   ],
   [
    "C72",
    "x[A8,B2]"
   ],
   [
    "C73",
    "x[A8,B3]"
   ],
   [
    "C74",
    "x[A8,B4]"
   ],
   [
    "C75",
    "x[A8,B5]"
   ],
   [
    "C76",
    "x[A8,B6]"
   ],
   [
    "C77",
    "x[A8,B7]"
   ],
   [
    "C78",
    "x[A8,B8]"
   ],
   [
    "C79",
    "x[A9,B1]"
   ],
   [
    "C80",
    "x[A9,B2]"
   ],
   [
    "C81",
    "x[A9,B3]"
   ],
   [
    "C82",
    "x[A9,B4]"
   ],
   [
    "C83",
    "x[A9,B5]"
   ],
   [
    "C84",
    "x[A9,B6]"
   ],
   [
    "C85",
    "x[A9,B7]"
   ],
   [
    "C86",
    "x[A9,B8]"
   ],
   [
    "C87",
    "x[A10,B1]"
   ],
   [
    "C88",
    "x[A10,B2]"
   ],
   [
    "C89",
    "x[A10,B3]"
   ],
   [
    "C90",
    "x[A10,B4]"
   ],
   [
    "C91",
    "x[A10,B5]"
   ],
   [
    "C92",
    "x[A10,B6]"
   ],
   [
    "C93",
    "x[A10,B7]"
   ],
   [
    "C94",
    "x[A10,B8]"
   ],
   [
    "C95",
    "x[A11,B1]"
   ],
   [
    "C96",
    "x[A11,B2]"
   ],
   [
    "C97",
    "x[A11,B3]"
   ],
   [
    "C98",
    "x[A11,B4]"
   ],
   [
    "C99",
    "x[A11,B5]"
   ],
   [
    "C100",
    "x[A11,B6]"
   ],
   [
    "C101",
    "x[A11,B7]"
   ],
   [
    "C102",
    "x[A11,B8]"
   ],
   [
    "C103",
    "x[A12,B1]"
   ],
   [
    "C104",
    "x[A12,B2]"
   ],
   [
    "C105",
    "x[A12,B3]"
   ],
   [
    "C106",
    "x[A12,B4]"
   ],
   [
    "C107",
    "x[A12,B5]"
   ],
   [
    "C108",
    "x[A12,B6]"
   ],
   [
    "C109",
    "x[A12,B7]"
   ],
   [
    "C110",
    "x[A12,B8]"
   ],
   [
    "C111",
    "x[A13,B1]"
   ],
   [
    "C112",
    "x[A13,B2]"
   ],
   [
    "C113",
    "x[A13,B3]"
   ],
   [
    "C114",
    "x[A13,B4]"
   ],
   [
    "C115",
    "x[A13,B5]"
   ],
   [
    "C116",
    "x[A13,B6]"
   ],
   [
    "C117",
    "x[A13,B7]"
   ],
   [
    "C118",
    "x[A13,B8]"
   ],
   [
    "C119",
    "x[A14,B1]"
   ],
   [
    "C120",
    "x[A14,B2]"
   ],
   [
    "C121",
    "x[A14,B3]"
   ],
   [
    "C122",
    "x[A14,B4]"
   ],
   [
    "C123",
    "x[A14,B5]"
   ],
   [
    "C124",
    "x[A14,B6]"
   ],
   [
    "C125",
    "x[A14,B7]"
   ],
   [
    "C126",
    "x[A14,B8]"
   ],
   [
    "C127",
    "x[A15,B1]"
   ],
   [
    "C128",
    "x[A15,B2]"
   ],
   [
    "C129",
    "x[A15,B3]"
   ],
   [
    "C130",
    "x[A15,B4]"
   ],
   [
    "C131",
    "x[A15,B5]"
   ],
   [
    "C132",
    "x[A15,B6]"
   ],
   [
    "C133",
    "x[A15,B7]"
   ],
   [
    "C134",
    "x[A15,B8]"
   ]
  ],
  "primary_mapping": {
   "x[0,0]": "x[A1,B1]",
   "x[0,1]": "x[A1,B2]",
   "x[0,2]": "x[A1,B3]",
   "x[0,3]": "x[A1,B4]",
   "x[0,4]": "x[A1,B5]",
   "x[0,5]": "x[A1,B6]",
   "x[0,6]": "x[A1,B7]",
   "x[0,7]": "x[A1,B8]",
   "x[1,0]": "x[A2,B1]",
   "x[1,1]": "x[A2,B2]",
   "x[1,2]": "x[A2,B3]",
   "x[1,3]": "x[A2,B4]",
   "x[1,4]": "x[A2,B5]",
   "x[1,5]": "x[A2,B6]",
   "x[1,6]": "x[A2,B7]",
   "x[1,7]": "x[A2,B8]",
   "x[2,0]": "x[A3,B1]",
   "x[2,1]": "x[A3,B2]",
   "x[2,2]": "x[A3,B3]",
   "x[2,3]": "x[A3,B4]",
   "x[2,4]": "x[A3,B5]",
   "x[2,5]": "x[A3,B6]",
   "x[2,6]": "x[A3,B7]",
   "x[2,7]": "x[A3,B8]",
   "x[3,0]": "x[A4,B1]",
   "x[3,1]": "x[A4,B2]",
   "x[3,2]": "x[A4,B3]",
   "x[3,3]": "x[A4,B4]",
   "x[3,4]": "x[A4,B5]",
   "x[3,5]": "x[A4,B6]",
   "x[3,6]": "x[A4,B7]",
   "x[3,7]": "x[A4,B8]",
   "x[4,0]": "x[A5,B1]",
   "x[4,1]": "x[A5,B2]",
   "x[4,2]": "x[A5,B3]",
   "x[4,3]": "x[A5,B4]",
   "x[4,4]": "x[A5,B5]",
   "x[4,5]": "x[A5,B6]",
   "x[4,6]": "x[A5,B7]",
   "x[4,7]": "x[A5,B8]",
   "x[5,0]": "x[A6,B1]",
   "x[5,1]": "x[A6,B2]",
   "x[5,2]": "x[A6,B3]",
   "x[5,3]": "x[A6,B4]",
   "x[5,4]": "x[A6,B5]",
   "x[5,5]": "x[A6,B6]",
   "x[5,6]": "x[A6,B7]",
   "x[5,7]": "x[A6,B8]",
   "x[6,0]": "x[A7,B1]",
   "x[6,1]": "x[A7,B2]",
   "x[6,2]": "x[A7,B3]",
   "x[6,3]": "x[A7,B4]",
   "x[6,4]": "x[A7,B5]",
   "x[6,5]": "x[A7,B6]",
   "x[6,6]": "x[A7,B7]",
   "x[6,7]": "x[A7,B8]",
   "x[7,0]": "x[A8,B1]",
   "x[7,1]": "x[A8,B2]",
   "x[7,2]": "x[A8,B3]",
   "x[7,3]": "x[A8,B4]",
   "x[7,4]": "x[A8,B5]",
   "x[7,5]": "x[A8,B6]",
   "x[7,6]": "x[A8,B7]",
   "x[7,7]": "x[A8,B8]",
   "x[8,0]": "x[A9,B1]",
   "x[8,1]": "x[A9,B2]",
   "x[8,2]": "x[A9,B3]",
   "x[8,3]": "x[A9,B4]",
   "x[8,4]": "x[A9,B5]",
   "x[8,5]": "x[A9,B6]",
   "x[8,6]": "x[A9,B7]",
   "x[8,7]": "x[A9,B8]",
   "x[9,0]": "x[A10,B1]",
   "x[9,1]": "x[A10,B2]",
   "x[9,2]": "x[A10,B3]",
   "x[9,3]": "x[A10,B4]",
   "x[9,4]": "x[A10,B5]",
   "x[9,5]": "x[A10,B6]",
   "x[9,6]": "x[A10,B7]",
   "x[9,7]": "x[A10,B8]",
   "x[10,0]": "x[A11,B1]",
   "x[10,1]": "x[A11,B2]",
   "x[10,2]": "x[A11,B3]",
   "x[10,3]": "x[A11,B4]",
   "x[10,4]": "x[A11,B5]",
   "x[10,5]": "x[A11,B6]",
   "x[10,6]": "x[A11,B7]",
   "x[10,7]": "x[A11,B8]",
   "x[11,0]": "x[A12,B1]",
   "x[11,1]": "x[A12,B2]",
   "x[11,2]": "x[A12,B3]",
   "x[11,3]": "x[A12,B4]",
   "x[11,4]": "x[A12,B5]",
   "x[11,5]": "x[A12,B6]",
   "x[11,6]": "x[A12,B7]",
   "x[11,7]": "x[A12,B8]",
   "x[12,0]": "x[A13,B1]",
   "x[12,1]": "x[A13,B2]",
   "x[12,2]": "x[A13,B3]",
   "x[12,3]": "x[A13,B4]",
   "x[12,4]": "x[A13,B5]",
   "x[12,5]": "x[A13,B6]",
   "x[12,6]": "x[A13,B7]",
   "x[12,7]": "x[A13,B8]",
   "x[13,0]": "x[A14,B1]",
   "x[13,1]": "x[A14,B2]",
   "x[13,2]": "x[A14,B3]",
   "x[13,3]": "x[A14,B4]",
   "x[13,4]": "x[A14,B5]",
   "x[13,5]": "x[A14,B6]",
   "x[13,6]": "x[A14,B7]",
   "x[13,7]": "x[A14,B8]",
   "x[14,0]": "x[A15,B1]",
   "x[14,1]": "x[A15,B2]",
   "x[14,2]": "x[A15,B3]",
   "x[14,3]": "x[A15,B4]",
   "x[14,4]": "x[A15,B5]",
   "x[14,5]": "x[A15,B6]",
   "x[14,6]": "x[A15,B7]",
   "x[14,7]": "x[A15,B8]",
   "y[1]": "y[A2]",
   "y[2]": "y[A3]",
   "y[3]": "y[A4]",
   "y[4]": "y[A5]",
   "y[5]": "y[A6]",
   "y[6]": "y[A7]",
   "y[7]": "y[A8]",
   "y[8]": "y[A9]",
   "y[9]": "y[A10]",
   "y[10]": "y[A11]",
   "y[11]": "y[A12]",
   "y[12]": "y[A13]",
   "y[13]": "y[A14]",
   "y[14]": "y[A15]",
   "y[0]": "y[A1]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 2965.0,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     1.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     1.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     1.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     1.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     1.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     15.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     15.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     10.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     25.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     5.0
    ],
    [
     "C75",
     25.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     30.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     5.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     20.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     20.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     15.0
    ],
    [
     "C125",
     25.0
    ],
    [
     "C126",
     10.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-083": {
  "source_code_sha256": "0f85eb1cf3335fada68c97dec991e9c9efb2816d045a1bf97087650d0ef78cdd",
  "original_solution_sha256": "6540a37a81ff8605ccf48dee620628ef43409dd20a5aa60b241589339952e8bd",
  "bindings": [
   [
    "C0",
    "x[1,A]"
   ],
   [
    "C1",
    "x[1,B]"
   ],
   [
    "C2",
    "x[1,C]"
   ],
   [
    "C3",
    "x[1,D]"
   ],
   [
    "C4",
    "x[1,E]"
   ],
   [
    "C5",
    "x[1,F]"
   ],
   [
    "C6",
    "x[1,G]"
   ],
   [
    "C7",
    "x[1,H]"
   ],
   [
    "C8",
    "x[1,I]"
   ],
   [
    "C9",
    "x[1,J]"
   ],
   [
    "C10",
    "x[2,A]"
   ],
   [
    "C11",
    "x[2,B]"
   ],
   [
    "C12",
    "x[2,C]"
   ],
   [
    "C13",
    "x[2,D]"
   ],
   [
    "C14",
    "x[2,E]"
   ],
   [
    "C15",
    "x[2,F]"
   ],
   [
    "C16",
    "x[2,G]"
   ],
   [
    "C17",
    "x[2,H]"
   ],
   [
    "C18",
    "x[2,I]"
   ],
   [
    "C19",
    "x[2,J]"
   ],
   [
    "C20",
    "x[3,A]"
   ],
   [
    "C21",
    "x[3,B]"
   ],
   [
    "C22",
    "x[3,C]"
   ],
   [
    "C23",
    "x[3,D]"
   ],
   [
    "C24",
    "x[3,E]"
   ],
   [
    "C25",
    "x[3,F]"
   ],
   [
    "C26",
    "x[3,G]"
   ],
   [
    "C27",
    "x[3,H]"
   ],
   [
    "C28",
    "x[3,I]"
   ],
   [
    "C29",
    "x[3,J]"
   ],
   [
    "C30",
    "x[4,A]"
   ],
   [
    "C31",
    "x[4,B]"
   ],
   [
    "C32",
    "x[4,C]"
   ],
   [
    "C33",
    "x[4,D]"
   ],
   [
    "C34",
    "x[4,E]"
   ],
   [
    "C35",
    "x[4,F]"
   ],
   [
    "C36",
    "x[4,G]"
   ],
   [
    "C37",
    "x[4,H]"
   ],
   [
    "C38",
    "x[4,I]"
   ],
   [
    "C39",
    "x[4,J]"
   ],
   [
    "C40",
    "x[5,A]"
   ],
   [
    "C41",
    "x[5,B]"
   ],
   [
    "C42",
    "x[5,C]"
   ],
   [
    "C43",
    "x[5,D]"
   ],
   [
    "C44",
    "x[5,E]"
   ],
   [
    "C45",
    "x[5,F]"
   ],
   [
    "C46",
    "x[5,G]"
   ],
   [
    "C47",
    "x[5,H]"
   ],
   [
    "C48",
    "x[5,I]"
   ],
   [
    "C49",
    "x[5,J]"
   ],
   [
    "C50",
    "x[6,A]"
   ],
   [
    "C51",
    "x[6,B]"
   ],
   [
    "C52",
    "x[6,C]"
   ],
   [
    "C53",
    "x[6,D]"
   ],
   [
    "C54",
    "x[6,E]"
   ],
   [
    "C55",
    "x[6,F]"
   ],
   [
    "C56",
    "x[6,G]"
   ],
   [
    "C57",
    "x[6,H]"
   ],
   [
    "C58",
    "x[6,I]"
   ],
   [
    "C59",
    "x[6,J]"
   ],
   [
    "C60",
    "x[7,A]"
   ],
   [
    "C61",
    "x[7,B]"
   ],
   [
    "C62",
    "x[7,C]"
   ],
   [
    "C63",
    "x[7,D]"
   ],
   [
    "C64",
    "x[7,E]"
   ],
   [
    "C65",
    "x[7,F]"
   ],
   [
    "C66",
    "x[7,G]"
   ],
   [
    "C67",
    "x[7,H]"
   ],
   [
    "C68",
    "x[7,I]"
   ],
   [
    "C69",
    "x[7,J]"
   ],
   [
    "C70",
    "x[8,A]"
   ],
   [
    "C71",
    "x[8,B]"
   ],
   [
    "C72",
    "x[8,C]"
   ],
   [
    "C73",
    "x[8,D]"
   ],
   [
    "C74",
    "x[8,E]"
   ],
   [
    "C75",
    "x[8,F]"
   ],
   [
    "C76",
    "x[8,G]"
   ],
   [
    "C77",
    "x[8,H]"
   ],
   [
    "C78",
    "x[8,I]"
   ],
   [
    "C79",
    "x[8,J]"
   ],
   [
    "C80",
    "x[9,A]"
   ],
   [
    "C81",
    "x[9,B]"
   ],
   [
    "C82",
    "x[9,C]"
   ],
   [
    "C83",
    "x[9,D]"
   ],
   [
    "C84",
    "x[9,E]"
   ],
   [
    "C85",
    "x[9,F]"
   ],
   [
    "C86",
    "x[9,G]"
   ],
   [
    "C87",
    "x[9,H]"
   ],
   [
    "C88",
    "x[9,I]"
   ],
   [
    "C89",
    "x[9,J]"
   ],
   [
    "C90",
    "x[10,A]"
   ],
   [
    "C91",
    "x[10,B]"
   ],
   [
    "C92",
    "x[10,C]"
   ],
   [
    "C93",
    "x[10,D]"
   ],
   [
    "C94",
    "x[10,E]"
   ],
   [
    "C95",
    "x[10,F]"
   ],
   [
    "C96",
    "x[10,G]"
   ],
   [
    "C97",
    "x[10,H]"
   ],
   [
    "C98",
    "x[10,I]"
   ],
   [
    "C99",
    "x[10,J]"
   ],
   [
    "C100",
    "x[11,A]"
   ],
   [
    "C101",
    "x[11,B]"
   ],
   [
    "C102",
    "x[11,C]"
   ],
   [
    "C103",
    "x[11,D]"
   ],
   [
    "C104",
    "x[11,E]"
   ],
   [
    "C105",
    "x[11,F]"
   ],
   [
    "C106",
    "x[11,G]"
   ],
   [
    "C107",
    "x[11,H]"
   ],
   [
    "C108",
    "x[11,I]"
   ],
   [
    "C109",
    "x[11,J]"
   ],
   [
    "C110",
    "x[12,A]"
   ],
   [
    "C111",
    "x[12,B]"
   ],
   [
    "C112",
    "x[12,C]"
   ],
   [
    "C113",
    "x[12,D]"
   ],
   [
    "C114",
    "x[12,E]"
   ],
   [
    "C115",
    "x[12,F]"
   ],
   [
    "C116",
    "x[12,G]"
   ],
   [
    "C117",
    "x[12,H]"
   ],
   [
    "C118",
    "x[12,I]"
   ],
   [
    "C119",
    "x[12,J]"
   ]
  ],
  "primary_mapping": {
   "x[9,A]": "x[9,A]",
   "x[8,A]": "x[8,A]",
   "x[3,F]": "x[3,F]",
   "x[10,H]": "x[10,H]",
   "x[3,A]": "x[3,A]",
   "x[9,G]": "x[9,G]",
   "x[5,J]": "x[5,J]",
   "x[12,A]": "x[12,A]",
   "x[5,D]": "x[5,D]",
   "x[8,H]": "x[8,H]",
   "x[7,G]": "x[7,G]",
   "x[3,H]": "x[3,H]",
   "x[12,E]": "x[12,E]",
   "x[10,D]": "x[10,D]",
   "x[9,B]": "x[9,B]",
   "x[11,B]": "x[11,B]",
   "x[11,C]": "x[11,C]",
   "x[12,J]": "x[12,J]",
   "x[1,I]": "x[1,I]",
   "x[11,G]": "x[11,G]",
   "x[9,J]": "x[9,J]",
   "x[5,A]": "x[5,A]",
   "x[12,B]": "x[12,B]",
   "x[3,D]": "x[3,D]",
   "x[4,H]": "x[4,H]",
   "x[11,I]": "x[11,I]",
   "x[6,E]": "x[6,E]",
   "x[11,E]": "x[11,E]",
   "x[7,B]": "x[7,B]",
   "x[4,E]": "x[4,E]",
   "x[1,D]": "x[1,D]",
   "x[9,I]": "x[9,I]",
   "x[10,C]": "x[10,C]",
   "x[8,B]": "x[8,B]",
   "x[1,A]": "x[1,A]",
   "x[3,J]": "x[3,J]",
   "x[2,B]": "x[2,B]",
   "x[6,D]": "x[6,D]",
   "x[4,I]": "x[4,I]",
   "x[5,E]": "x[5,E]",
   "x[6,I]": "x[6,I]",
   "x[2,A]": "x[2,A]",
   "x[10,I]": "x[10,I]",
   "x[10,F]": "x[10,F]",
   "x[5,B]": "x[5,B]",
   "x[7,I]": "x[7,I]",
   "x[1,G]": "x[1,G]",
   "x[6,C]": "x[6,C]",
   "x[9,F]": "x[9,F]",
   "x[2,E]": "x[2,E]",
   "x[2,D]": "x[2,D]",
   "x[9,D]": "x[9,D]",
   "x[6,F]": "x[6,F]",
   "x[5,G]": "x[5,G]",
   "x[10,G]": "x[10,G]",
   "x[7,A]": "x[7,A]",
   "x[8,F]": "x[8,F]",
   "x[6,B]": "x[6,B]",
   "x[12,F]": "x[12,F]",
   "x[2,C]": "x[2,C]",
   "x[3,E]": "x[3,E]",
   "x[8,D]": "x[8,D]",
   "x[7,D]": "x[7,D]",
   "x[1,J]": "x[1,J]",
   "x[10,J]": "x[10,J]",
   "x[4,J]": "x[4,J]",
   "x[1,B]": "x[1,B]",
   "x[10,B]": "x[10,B]",
   "x[10,A]": "x[10,A]",
   "x[8,G]": "x[8,G]",
   "x[6,A]": "x[6,A]",
   "x[2,H]": "x[2,H]",
   "x[2,J]": "x[2,J]",
   "x[12,G]": "x[12,G]",
   "x[3,B]": "x[3,B]",
   "x[8,I]": "x[8,I]",
   "x[2,I]": "x[2,I]",
   "x[7,H]": "x[7,H]",
   "x[5,H]": "x[5,H]",
   "x[11,A]": "x[11,A]",
   "x[1,E]": "x[1,E]",
   "x[7,F]": "x[7,F]",
   "x[12,D]": "x[12,D]",
   "x[2,F]": "x[2,F]",
   "x[4,D]": "x[4,D]",
   "x[8,C]": "x[8,C]",
   "x[2,G]": "x[2,G]",
   "x[12,C]": "x[12,C]",
   "x[11,F]": "x[11,F]",
   "x[12,H]": "x[12,H]",
   "x[8,J]": "x[8,J]",
   "x[4,G]": "x[4,G]",
   "x[4,A]": "x[4,A]",
   "x[9,H]": "x[9,H]",
   "x[6,G]": "x[6,G]",
   "x[1,H]": "x[1,H]",
   "x[9,C]": "x[9,C]",
   "x[7,E]": "x[7,E]",
   "x[10,E]": "x[10,E]",
   "x[4,F]": "x[4,F]",
   "x[11,D]": "x[11,D]",
   "x[12,I]": "x[12,I]",
   "x[5,I]": "x[5,I]",
   "x[5,C]": "x[5,C]",
   "x[3,C]": "x[3,C]",
   "x[3,I]": "x[3,I]",
   "x[1,F]": "x[1,F]",
   "x[6,J]": "x[6,J]",
   "x[11,H]": "x[11,H]",
   "x[11,J]": "x[11,J]",
   "x[1,C]": "x[1,C]",
   "x[4,B]": "x[4,B]",
   "x[4,C]": "x[4,C]",
   "x[6,H]": "x[6,H]",
   "x[7,C]": "x[7,C]",
   "x[5,F]": "x[5,F]",
   "x[8,E]": "x[8,E]",
   "x[9,E]": "x[9,E]",
   "x[3,G]": "x[3,G]",
   "x[7,J]": "x[7,J]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 34.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     1.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     1.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     1.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     1.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     1.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     1.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     1.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     1.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     1.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-087": {
  "source_code_sha256": "34897dce2bd7df21fb7f2f13e68ea4f47e7c4e337eb1efef3e47d19999d13adb",
  "original_solution_sha256": "105b79bae3184e3cacecbd1f3a50c5630febc0d2e3142f1b0932948f93182f73",
  "bindings": [],
  "primary_mapping": null,
  "mapping_basis": "unavailable",
  "mapping_failure": "ValueError: invalid literal for int() with base 10: '71.38'",
  "frozen_replay_solution": {
   "status": "ERROR",
   "objective": null,
   "variables": [],
   "error": "ValueError: invalid literal for int() with base 10: '71.38'"
  }
 },
 "OR-089": {
  "source_code_sha256": "fb4ffac7810a0d9faa4bf5ac04ec26463a84ea6af938a4f29d52a1dc4372e8d6",
  "original_solution_sha256": "2884b675f3d59971356d346527922890e9b5661e10f851f5a6101551a572586b",
  "bindings": [
   [
    "C0",
    "y[SC1]"
   ],
   [
    "C1",
    "y[SC2]"
   ],
   [
    "C2",
    "y[SC3]"
   ],
   [
    "C3",
    "y[SC4]"
   ],
   [
    "C4",
    "y[SC5]"
   ],
   [
    "C5",
    "y[SC6]"
   ],
   [
    "C6",
    "y[SC7]"
   ],
   [
    "C7",
    "y[SC8]"
   ],
   [
    "C8",
    "y[SC9]"
   ],
   [
    "C9",
    "y[SC10]"
   ],
   [
    "C10",
    "x[SC1,C1]"
   ],
   [
    "C11",
    "x[SC1,C2]"
   ],
   [
    "C12",
    "x[SC1,C3]"
   ],
   [
    "C13",
    "x[SC1,C4]"
   ],
   [
    "C14",
    "x[SC1,C5]"
   ],
   [
    "C15",
    "x[SC1,C6]"
   ],
   [
    "C16",
    "x[SC1,C7]"
   ],
   [
    "C17",
    "x[SC1,C8]"
   ],
   [
    "C18",
    "x[SC1,C9]"
   ],
   [
    "C19",
    "x[SC1,C10]"
   ],
   [
    "C20",
    "x[SC1,C11]"
   ],
   [
    "C21",
    "x[SC1,C12]"
   ],
   [
    "C22",
    "x[SC1,C13]"
   ],
   [
    "C23",
    "x[SC1,C14]"
   ],
   [
    "C24",
    "x[SC1,C15]"
   ],
   [
    "C25",
    "x[SC2,C1]"
   ],
   [
    "C26",
    "x[SC2,C2]"
   ],
   [
    "C27",
    "x[SC2,C3]"
   ],
   [
    "C28",
    "x[SC2,C4]"
   ],
   [
    "C29",
    "x[SC2,C5]"
   ],
   [
    "C30",
    "x[SC2,C6]"
   ],
   [
    "C31",
    "x[SC2,C7]"
   ],
   [
    "C32",
    "x[SC2,C8]"
   ],
   [
    "C33",
    "x[SC2,C9]"
   ],
   [
    "C34",
    "x[SC2,C10]"
   ],
   [
    "C35",
    "x[SC2,C11]"
   ],
   [
    "C36",
    "x[SC2,C12]"
   ],
   [
    "C37",
    "x[SC2,C13]"
   ],
   [
    "C38",
    "x[SC2,C14]"
   ],
   [
    "C39",
    "x[SC2,C15]"
   ],
   [
    "C40",
    "x[SC3,C1]"
   ],
   [
    "C41",
    "x[SC3,C2]"
   ],
   [
    "C42",
    "x[SC3,C3]"
   ],
   [
    "C43",
    "x[SC3,C4]"
   ],
   [
    "C44",
    "x[SC3,C5]"
   ],
   [
    "C45",
    "x[SC3,C6]"
   ],
   [
    "C46",
    "x[SC3,C7]"
   ],
   [
    "C47",
    "x[SC3,C8]"
   ],
   [
    "C48",
    "x[SC3,C9]"
   ],
   [
    "C49",
    "x[SC3,C10]"
   ],
   [
    "C50",
    "x[SC3,C11]"
   ],
   [
    "C51",
    "x[SC3,C12]"
   ],
   [
    "C52",
    "x[SC3,C13]"
   ],
   [
    "C53",
    "x[SC3,C14]"
   ],
   [
    "C54",
    "x[SC3,C15]"
   ],
   [
    "C55",
    "x[SC4,C1]"
   ],
   [
    "C56",
    "x[SC4,C2]"
   ],
   [
    "C57",
    "x[SC4,C3]"
   ],
   [
    "C58",
    "x[SC4,C4]"
   ],
   [
    "C59",
    "x[SC4,C5]"
   ],
   [
    "C60",
    "x[SC4,C6]"
   ],
   [
    "C61",
    "x[SC4,C7]"
   ],
   [
    "C62",
    "x[SC4,C8]"
   ],
   [
    "C63",
    "x[SC4,C9]"
   ],
   [
    "C64",
    "x[SC4,C10]"
   ],
   [
    "C65",
    "x[SC4,C11]"
   ],
   [
    "C66",
    "x[SC4,C12]"
   ],
   [
    "C67",
    "x[SC4,C13]"
   ],
   [
    "C68",
    "x[SC4,C14]"
   ],
   [
    "C69",
    "x[SC4,C15]"
   ],
   [
    "C70",
    "x[SC5,C1]"
   ],
   [
    "C71",
    "x[SC5,C2]"
   ],
   [
    "C72",
    "x[SC5,C3]"
   ],
   [
    "C73",
    "x[SC5,C4]"
   ],
   [
    "C74",
    "x[SC5,C5]"
   ],
   [
    "C75",
    "x[SC5,C6]"
   ],
   [
    "C76",
    "x[SC5,C7]"
   ],
   [
    "C77",
    "x[SC5,C8]"
   ],
   [
    "C78",
    "x[SC5,C9]"
   ],
   [
    "C79",
    "x[SC5,C10]"
   ],
   [
    "C80",
    "x[SC5,C11]"
   ],
   [
    "C81",
    "x[SC5,C12]"
   ],
   [
    "C82",
    "x[SC5,C13]"
   ],
   [
    "C83",
    "x[SC5,C14]"
   ],
   [
    "C84",
    "x[SC5,C15]"
   ],
   [
    "C85",
    "x[SC6,C1]"
   ],
   [
    "C86",
    "x[SC6,C2]"
   ],
   [
    "C87",
    "x[SC6,C3]"
   ],
   [
    "C88",
    "x[SC6,C4]"
   ],
   [
    "C89",
    "x[SC6,C5]"
   ],
   [
    "C90",
    "x[SC6,C6]"
   ],
   [
    "C91",
    "x[SC6,C7]"
   ],
   [
    "C92",
    "x[SC6,C8]"
   ],
   [
    "C93",
    "x[SC6,C9]"
   ],
   [
    "C94",
    "x[SC6,C10]"
   ],
   [
    "C95",
    "x[SC6,C11]"
   ],
   [
    "C96",
    "x[SC6,C12]"
   ],
   [
    "C97",
    "x[SC6,C13]"
   ],
   [
    "C98",
    "x[SC6,C14]"
   ],
   [
    "C99",
    "x[SC6,C15]"
   ],
   [
    "C100",
    "x[SC7,C1]"
   ],
   [
    "C101",
    "x[SC7,C2]"
   ],
   [
    "C102",
    "x[SC7,C3]"
   ],
   [
    "C103",
    "x[SC7,C4]"
   ],
   [
    "C104",
    "x[SC7,C5]"
   ],
   [
    "C105",
    "x[SC7,C6]"
   ],
   [
    "C106",
    "x[SC7,C7]"
   ],
   [
    "C107",
    "x[SC7,C8]"
   ],
   [
    "C108",
    "x[SC7,C9]"
   ],
   [
    "C109",
    "x[SC7,C10]"
   ],
   [
    "C110",
    "x[SC7,C11]"
   ],
   [
    "C111",
    "x[SC7,C12]"
   ],
   [
    "C112",
    "x[SC7,C13]"
   ],
   [
    "C113",
    "x[SC7,C14]"
   ],
   [
    "C114",
    "x[SC7,C15]"
   ],
   [
    "C115",
    "x[SC8,C1]"
   ],
   [
    "C116",
    "x[SC8,C2]"
   ],
   [
    "C117",
    "x[SC8,C3]"
   ],
   [
    "C118",
    "x[SC8,C4]"
   ],
   [
    "C119",
    "x[SC8,C5]"
   ],
   [
    "C120",
    "x[SC8,C6]"
   ],
   [
    "C121",
    "x[SC8,C7]"
   ],
   [
    "C122",
    "x[SC8,C8]"
   ],
   [
    "C123",
    "x[SC8,C9]"
   ],
   [
    "C124",
    "x[SC8,C10]"
   ],
   [
    "C125",
    "x[SC8,C11]"
   ],
   [
    "C126",
    "x[SC8,C12]"
   ],
   [
    "C127",
    "x[SC8,C13]"
   ],
   [
    "C128",
    "x[SC8,C14]"
   ],
   [
    "C129",
    "x[SC8,C15]"
   ],
   [
    "C130",
    "x[SC9,C1]"
   ],
   [
    "C131",
    "x[SC9,C2]"
   ],
   [
    "C132",
    "x[SC9,C3]"
   ],
   [
    "C133",
    "x[SC9,C4]"
   ],
   [
    "C134",
    "x[SC9,C5]"
   ],
   [
    "C135",
    "x[SC9,C6]"
   ],
   [
    "C136",
    "x[SC9,C7]"
   ],
   [
    "C137",
    "x[SC9,C8]"
   ],
   [
    "C138",
    "x[SC9,C9]"
   ],
   [
    "C139",
    "x[SC9,C10]"
   ],
   [
    "C140",
    "x[SC9,C11]"
   ],
   [
    "C141",
    "x[SC9,C12]"
   ],
   [
    "C142",
    "x[SC9,C13]"
   ],
   [
    "C143",
    "x[SC9,C14]"
   ],
   [
    "C144",
    "x[SC9,C15]"
   ],
   [
    "C145",
    "x[SC10,C1]"
   ],
   [
    "C146",
    "x[SC10,C2]"
   ],
   [
    "C147",
    "x[SC10,C3]"
   ],
   [
    "C148",
    "x[SC10,C4]"
   ],
   [
    "C149",
    "x[SC10,C5]"
   ],
   [
    "C150",
    "x[SC10,C6]"
   ],
   [
    "C151",
    "x[SC10,C7]"
   ],
   [
    "C152",
    "x[SC10,C8]"
   ],
   [
    "C153",
    "x[SC10,C9]"
   ],
   [
    "C154",
    "x[SC10,C10]"
   ],
   [
    "C155",
    "x[SC10,C11]"
   ],
   [
    "C156",
    "x[SC10,C12]"
   ],
   [
    "C157",
    "x[SC10,C13]"
   ],
   [
    "C158",
    "x[SC10,C14]"
   ],
   [
    "C159",
    "x[SC10,C15]"
   ]
  ],
  "primary_mapping": {
   "y_SC1": "y[SC1]",
   "y_SC2": "y[SC2]",
   "y_SC3": "y[SC3]",
   "y_SC4": "y[SC4]",
   "y_SC5": "y[SC5]",
   "y_SC6": "y[SC6]",
   "y_SC7": "y[SC7]",
   "y_SC8": "y[SC8]",
   "y_SC9": "y[SC9]",
   "y_SC10": "y[SC10]",
   "x_SC1_C1": "x[SC1,C1]",
   "x_SC1_C11": "x[SC1,C11]",
   "x_SC1_C12": "x[SC1,C12]",
   "x_SC1_C13": "x[SC1,C13]",
   "x_SC1_C14": "x[SC1,C14]",
   "x_SC1_C15": "x[SC1,C15]",
   "x_SC2_C2": "x[SC2,C2]",
   "x_SC2_C11": "x[SC2,C11]",
   "x_SC2_C12": "x[SC2,C12]",
   "x_SC2_C13": "x[SC2,C13]",
   "x_SC2_C14": "x[SC2,C14]",
   "x_SC2_C15": "x[SC2,C15]",
   "x_SC3_C3": "x[SC3,C3]",
   "x_SC3_C11": "x[SC3,C11]",
   "x_SC3_C12": "x[SC3,C12]",
   "x_SC3_C13": "x[SC3,C13]",
   "x_SC3_C14": "x[SC3,C14]",
   "x_SC3_C15": "x[SC3,C15]",
   "x_SC4_C4": "x[SC4,C4]",
   "x_SC4_C11": "x[SC4,C11]",
   "x_SC4_C12": "x[SC4,C12]",
   "x_SC4_C13": "x[SC4,C13]",
   "x_SC4_C14": "x[SC4,C14]",
   "x_SC4_C15": "x[SC4,C15]",
   "x_SC5_C5": "x[SC5,C5]",
   "x_SC5_C11": "x[SC5,C11]",
   "x_SC5_C12": "x[SC5,C12]",
   "x_SC5_C13": "x[SC5,C13]",
   "x_SC5_C14": "x[SC5,C14]",
   "x_SC5_C15": "x[SC5,C15]",
   "x_SC6_C6": "x[SC6,C6]",
   "x_SC6_C11": "x[SC6,C11]",
   "x_SC6_C12": "x[SC6,C12]",
   "x_SC6_C13": "x[SC6,C13]",
   "x_SC6_C14": "x[SC6,C14]",
   "x_SC6_C15": "x[SC6,C15]",
   "x_SC7_C7": "x[SC7,C7]",
   "x_SC7_C11": "x[SC7,C11]",
   "x_SC7_C12": "x[SC7,C12]",
   "x_SC7_C13": "x[SC7,C13]",
   "x_SC7_C14": "x[SC7,C14]",
   "x_SC7_C15": "x[SC7,C15]",
   "x_SC8_C8": "x[SC8,C8]",
   "x_SC8_C11": "x[SC8,C11]",
   "x_SC8_C12": "x[SC8,C12]",
   "x_SC8_C13": "x[SC8,C13]",
   "x_SC8_C14": "x[SC8,C14]",
   "x_SC8_C15": "x[SC8,C15]",
   "x_SC9_C9": "x[SC9,C9]",
   "x_SC9_C11": "x[SC9,C11]",
   "x_SC9_C12": "x[SC9,C12]",
   "x_SC9_C13": "x[SC9,C13]",
   "x_SC9_C14": "x[SC9,C14]",
   "x_SC9_C15": "x[SC9,C15]",
   "x_SC10_C10": "x[SC10,C10]",
   "x_SC10_C11": "x[SC10,C11]",
   "x_SC10_C12": "x[SC10,C12]",
   "x_SC10_C13": "x[SC10,C13]",
   "x_SC10_C14": "x[SC10,C14]",
   "x_SC10_C15": "x[SC10,C15]",
   "x_SC1_C2": "x[SC1,C2]",
   "x_SC1_C3": "x[SC1,C3]",
   "x_SC1_C4": "x[SC1,C4]",
   "x_SC1_C5": "x[SC1,C5]",
   "x_SC1_C6": "x[SC1,C6]",
   "x_SC1_C7": "x[SC1,C7]",
   "x_SC1_C8": "x[SC1,C8]",
   "x_SC1_C9": "x[SC1,C9]",
   "x_SC1_C10": "x[SC1,C10]",
   "x_SC2_C1": "x[SC2,C1]",
   "x_SC2_C3": "x[SC2,C3]",
   "x_SC2_C4": "x[SC2,C4]",
   "x_SC2_C5": "x[SC2,C5]",
   "x_SC2_C6": "x[SC2,C6]",
   "x_SC2_C7": "x[SC2,C7]",
   "x_SC2_C8": "x[SC2,C8]",
   "x_SC2_C9": "x[SC2,C9]",
   "x_SC2_C10": "x[SC2,C10]",
   "x_SC3_C1": "x[SC3,C1]",
   "x_SC3_C2": "x[SC3,C2]",
   "x_SC3_C4": "x[SC3,C4]",
   "x_SC3_C5": "x[SC3,C5]",
   "x_SC3_C6": "x[SC3,C6]",
   "x_SC3_C7": "x[SC3,C7]",
   "x_SC3_C8": "x[SC3,C8]",
   "x_SC3_C9": "x[SC3,C9]",
   "x_SC3_C10": "x[SC3,C10]",
   "x_SC4_C1": "x[SC4,C1]",
   "x_SC4_C2": "x[SC4,C2]",
   "x_SC4_C3": "x[SC4,C3]",
   "x_SC4_C5": "x[SC4,C5]",
   "x_SC4_C6": "x[SC4,C6]",
   "x_SC4_C7": "x[SC4,C7]",
   "x_SC4_C8": "x[SC4,C8]",
   "x_SC4_C9": "x[SC4,C9]",
   "x_SC4_C10": "x[SC4,C10]",
   "x_SC5_C1": "x[SC5,C1]",
   "x_SC5_C2": "x[SC5,C2]",
   "x_SC5_C3": "x[SC5,C3]",
   "x_SC5_C4": "x[SC5,C4]",
   "x_SC5_C6": "x[SC5,C6]",
   "x_SC5_C7": "x[SC5,C7]",
   "x_SC5_C8": "x[SC5,C8]",
   "x_SC5_C9": "x[SC5,C9]",
   "x_SC5_C10": "x[SC5,C10]",
   "x_SC6_C1": "x[SC6,C1]",
   "x_SC6_C2": "x[SC6,C2]",
   "x_SC6_C3": "x[SC6,C3]",
   "x_SC6_C4": "x[SC6,C4]",
   "x_SC6_C5": "x[SC6,C5]",
   "x_SC6_C7": "x[SC6,C7]",
   "x_SC6_C8": "x[SC6,C8]",
   "x_SC6_C9": "x[SC6,C9]",
   "x_SC6_C10": "x[SC6,C10]",
   "x_SC7_C1": "x[SC7,C1]",
   "x_SC7_C2": "x[SC7,C2]",
   "x_SC7_C3": "x[SC7,C3]",
   "x_SC7_C4": "x[SC7,C4]",
   "x_SC7_C5": "x[SC7,C5]",
   "x_SC7_C6": "x[SC7,C6]",
   "x_SC7_C8": "x[SC7,C8]",
   "x_SC7_C9": "x[SC7,C9]",
   "x_SC7_C10": "x[SC7,C10]",
   "x_SC8_C1": "x[SC8,C1]",
   "x_SC8_C2": "x[SC8,C2]",
   "x_SC8_C3": "x[SC8,C3]",
   "x_SC8_C4": "x[SC8,C4]",
   "x_SC8_C5": "x[SC8,C5]",
   "x_SC8_C6": "x[SC8,C6]",
   "x_SC8_C7": "x[SC8,C7]",
   "x_SC8_C9": "x[SC8,C9]",
   "x_SC8_C10": "x[SC8,C10]",
   "x_SC9_C1": "x[SC9,C1]",
   "x_SC9_C2": "x[SC9,C2]",
   "x_SC9_C3": "x[SC9,C3]",
   "x_SC9_C4": "x[SC9,C4]",
   "x_SC9_C5": "x[SC9,C5]",
   "x_SC9_C6": "x[SC9,C6]",
   "x_SC9_C7": "x[SC9,C7]",
   "x_SC9_C8": "x[SC9,C8]",
   "x_SC9_C10": "x[SC9,C10]",
   "x_SC10_C1": "x[SC10,C1]",
   "x_SC10_C2": "x[SC10,C2]",
   "x_SC10_C3": "x[SC10,C3]",
   "x_SC10_C4": "x[SC10,C4]",
   "x_SC10_C5": "x[SC10,C5]",
   "x_SC10_C6": "x[SC10,C6]",
   "x_SC10_C7": "x[SC10,C7]",
   "x_SC10_C8": "x[SC10,C8]",
   "x_SC10_C9": "x[SC10,C9]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1555.4999999999993,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     1.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     1.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     1.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     1.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     1.0
    ],
    [
     "C24",
     1.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     1.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     1.0
    ],
    [
     "C75",
     1.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     1.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     1.0
    ],
    [
     "C93",
     1.0
    ],
    [
     "C94",
     1.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     1.0
    ],
    [
     "C103",
     1.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     1.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     1.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-095": {
  "source_code_sha256": "606cfd1ba50c32eddcf3458ec5b2865e1021c53fde801daf3aac134487744b89",
  "original_solution_sha256": "d9a5ccdd195fbb65c06d36be56df7594b6eecc2e97fcee6d6cd64ab0e14fc5c3",
  "bindings": [
   [
    "C0",
    "x[Widget1]"
   ],
   [
    "C1",
    "x[Widget2]"
   ],
   [
    "C2",
    "x[Widget3]"
   ],
   [
    "C3",
    "x[Widget4]"
   ],
   [
    "C4",
    "x[Widget5]"
   ],
   [
    "C5",
    "x[Widget6]"
   ],
   [
    "C6",
    "x[Widget7]"
   ],
   [
    "C7",
    "x[Widget8]"
   ],
   [
    "C8",
    "x[Widget9]"
   ],
   [
    "C9",
    "x[Widget10]"
   ],
   [
    "C10",
    "x[Widget11]"
   ],
   [
    "C11",
    "x[Widget12]"
   ],
   [
    "C12",
    "x[Widget13]"
   ],
   [
    "C13",
    "x[Widget14]"
   ],
   [
    "C14",
    "x[Widget15]"
   ],
   [
    "C15",
    "x[Widget16]"
   ],
   [
    "C16",
    "x[Widget17]"
   ],
   [
    "C17",
    "x[Widget18]"
   ],
   [
    "C18",
    "x[Widget19]"
   ],
   [
    "C19",
    "x[Widget20]"
   ],
   [
    "C20",
    "x[Widget21]"
   ],
   [
    "C21",
    "x[Widget22]"
   ],
   [
    "C22",
    "x[Widget23]"
   ],
   [
    "C23",
    "x[Widget24]"
   ],
   [
    "C24",
    "x[Widget25]"
   ],
   [
    "C25",
    "x[Widget26]"
   ],
   [
    "C26",
    "x[Widget27]"
   ],
   [
    "C27",
    "x[Widget28]"
   ],
   [
    "C28",
    "x[Widget29]"
   ],
   [
    "C29",
    "x[Widget30]"
   ],
   [
    "C30",
    "x[Widget31]"
   ],
   [
    "C31",
    "x[Widget32]"
   ],
   [
    "C32",
    "x[Widget33]"
   ],
   [
    "C33",
    "x[Widget34]"
   ],
   [
    "C34",
    "x[Widget35]"
   ],
   [
    "C35",
    "x[Widget36]"
   ],
   [
    "C36",
    "x[Widget37]"
   ],
   [
    "C37",
    "x[Widget38]"
   ],
   [
    "C38",
    "x[Widget39]"
   ],
   [
    "C39",
    "x[Widget40]"
   ],
   [
    "C40",
    "x[Widget41]"
   ],
   [
    "C41",
    "x[Widget42]"
   ],
   [
    "C42",
    "x[Widget43]"
   ],
   [
    "C43",
    "x[Widget44]"
   ],
   [
    "C44",
    "x[Widget45]"
   ],
   [
    "C45",
    "x[Widget46]"
   ],
   [
    "C46",
    "x[Widget47]"
   ],
   [
    "C47",
    "x[Widget48]"
   ],
   [
    "C48",
    "x[Widget49]"
   ],
   [
    "C49",
    "x[Widget50]"
   ],
   [
    "C50",
    "x[Widget51]"
   ],
   [
    "C51",
    "x[Widget52]"
   ],
   [
    "C52",
    "x[Widget53]"
   ],
   [
    "C53",
    "x[Widget54]"
   ],
   [
    "C54",
    "x[Widget55]"
   ],
   [
    "C55",
    "x[Widget56]"
   ],
   [
    "C56",
    "x[Widget57]"
   ],
   [
    "C57",
    "x[Widget58]"
   ],
   [
    "C58",
    "x[Widget59]"
   ],
   [
    "C59",
    "x[Widget60]"
   ],
   [
    "C60",
    "x[Widget61]"
   ],
   [
    "C61",
    "x[Widget62]"
   ],
   [
    "C62",
    "x[Widget63]"
   ],
   [
    "C63",
    "x[Widget64]"
   ],
   [
    "C64",
    "x[Widget65]"
   ],
   [
    "C65",
    "x[Widget66]"
   ],
   [
    "C66",
    "x[Widget67]"
   ],
   [
    "C67",
    "x[Widget68]"
   ],
   [
    "C68",
    "x[Widget69]"
   ],
   [
    "C69",
    "x[Widget70]"
   ],
   [
    "C70",
    "x[Widget71]"
   ],
   [
    "C71",
    "x[Widget72]"
   ],
   [
    "C72",
    "x[Widget73]"
   ],
   [
    "C73",
    "x[Widget74]"
   ],
   [
    "C74",
    "x[Widget75]"
   ],
   [
    "C75",
    "x[Widget76]"
   ],
   [
    "C76",
    "x[Widget77]"
   ],
   [
    "C77",
    "x[Widget78]"
   ],
   [
    "C78",
    "x[Widget79]"
   ],
   [
    "C79",
    "x[Widget80]"
   ],
   [
    "C80",
    "x[Widget81]"
   ],
   [
    "C81",
    "x[Widget82]"
   ],
   [
    "C82",
    "x[Widget83]"
   ],
   [
    "C83",
    "x[Widget84]"
   ],
   [
    "C84",
    "x[Widget85]"
   ],
   [
    "C85",
    "x[Widget86]"
   ],
   [
    "C86",
    "x[Widget87]"
   ],
   [
    "C87",
    "x[Widget88]"
   ],
   [
    "C88",
    "x[Widget89]"
   ],
   [
    "C89",
    "x[Widget90]"
   ],
   [
    "C90",
    "x[Widget91]"
   ],
   [
    "C91",
    "x[Widget92]"
   ],
   [
    "C92",
    "x[Widget93]"
   ],
   [
    "C93",
    "x[Widget94]"
   ],
   [
    "C94",
    "x[Widget95]"
   ],
   [
    "C95",
    "x[Widget96]"
   ],
   [
    "C96",
    "x[Widget97]"
   ],
   [
    "C97",
    "x[Widget98]"
   ],
   [
    "C98",
    "x[Widget99]"
   ],
   [
    "C99",
    "x[Widget100]"
   ],
   [
    "C100",
    "x[Widget101]"
   ],
   [
    "C101",
    "x[Widget102]"
   ],
   [
    "C102",
    "x[Widget103]"
   ],
   [
    "C103",
    "x[Widget104]"
   ],
   [
    "C104",
    "x[Widget105]"
   ],
   [
    "C105",
    "x[Widget106]"
   ],
   [
    "C106",
    "x[Widget107]"
   ],
   [
    "C107",
    "x[Widget108]"
   ],
   [
    "C108",
    "x[Widget109]"
   ],
   [
    "C109",
    "x[Widget110]"
   ],
   [
    "C110",
    "x[Widget111]"
   ],
   [
    "C111",
    "x[Widget112]"
   ],
   [
    "C112",
    "x[Widget113]"
   ],
   [
    "C113",
    "x[Widget114]"
   ],
   [
    "C114",
    "x[Widget115]"
   ],
   [
    "C115",
    "x[Widget116]"
   ],
   [
    "C116",
    "x[Widget117]"
   ],
   [
    "C117",
    "x[Widget118]"
   ],
   [
    "C118",
    "x[Widget119]"
   ],
   [
    "C119",
    "x[Widget120]"
   ],
   [
    "C120",
    "x[Widget121]"
   ],
   [
    "C121",
    "x[Widget122]"
   ],
   [
    "C122",
    "x[Widget123]"
   ],
   [
    "C123",
    "x[Widget124]"
   ],
   [
    "C124",
    "x[Widget125]"
   ],
   [
    "C125",
    "x[Widget126]"
   ],
   [
    "C126",
    "x[Widget127]"
   ],
   [
    "C127",
    "x[Widget128]"
   ],
   [
    "C128",
    "x[Widget129]"
   ],
   [
    "C129",
    "x[Widget130]"
   ],
   [
    "C130",
    "x[Widget131]"
   ],
   [
    "C131",
    "x[Widget132]"
   ],
   [
    "C132",
    "x[Widget133]"
   ],
   [
    "C133",
    "x[Widget134]"
   ],
   [
    "C134",
    "x[Widget135]"
   ],
   [
    "C135",
    "x[Widget136]"
   ],
   [
    "C136",
    "x[Widget137]"
   ],
   [
    "C137",
    "x[Widget138]"
   ],
   [
    "C138",
    "x[Widget139]"
   ],
   [
    "C139",
    "x[Widget140]"
   ],
   [
    "C140",
    "x[Widget141]"
   ],
   [
    "s",
    "s"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x[Widget1]",
   "x[2]": "x[Widget2]",
   "x[3]": "x[Widget3]",
   "x[4]": "x[Widget4]",
   "x[5]": "x[Widget5]",
   "x[6]": "x[Widget6]",
   "x[7]": "x[Widget7]",
   "x[8]": "x[Widget8]",
   "x[9]": "x[Widget9]",
   "x[10]": "x[Widget10]",
   "x[11]": "x[Widget11]",
   "x[12]": "x[Widget12]",
   "x[13]": "x[Widget13]",
   "x[14]": "x[Widget14]",
   "x[15]": "x[Widget15]",
   "x[16]": "x[Widget16]",
   "x[17]": "x[Widget17]",
   "x[18]": "x[Widget18]",
   "x[19]": "x[Widget19]",
   "x[20]": "x[Widget20]",
   "x[21]": "x[Widget21]",
   "x[22]": "x[Widget22]",
   "x[23]": "x[Widget23]",
   "x[24]": "x[Widget24]",
   "x[25]": "x[Widget25]",
   "x[26]": "x[Widget26]",
   "x[27]": "x[Widget27]",
   "x[28]": "x[Widget28]",
   "x[29]": "x[Widget29]",
   "x[30]": "x[Widget30]",
   "x[31]": "x[Widget31]",
   "x[32]": "x[Widget32]",
   "x[33]": "x[Widget33]",
   "x[34]": "x[Widget34]",
   "x[35]": "x[Widget35]",
   "x[36]": "x[Widget36]",
   "x[37]": "x[Widget37]",
   "x[38]": "x[Widget38]",
   "x[39]": "x[Widget39]",
   "x[40]": "x[Widget40]",
   "x[41]": "x[Widget41]",
   "x[42]": "x[Widget42]",
   "x[43]": "x[Widget43]",
   "x[44]": "x[Widget44]",
   "x[45]": "x[Widget45]",
   "x[46]": "x[Widget46]",
   "x[47]": "x[Widget47]",
   "x[48]": "x[Widget48]",
   "x[49]": "x[Widget49]",
   "x[50]": "x[Widget50]",
   "x[51]": "x[Widget51]",
   "x[52]": "x[Widget52]",
   "x[53]": "x[Widget53]",
   "x[54]": "x[Widget54]",
   "x[55]": "x[Widget55]",
   "x[56]": "x[Widget56]",
   "x[57]": "x[Widget57]",
   "x[58]": "x[Widget58]",
   "x[59]": "x[Widget59]",
   "x[60]": "x[Widget60]",
   "x[61]": "x[Widget61]",
   "x[62]": "x[Widget62]",
   "x[63]": "x[Widget63]",
   "x[64]": "x[Widget64]",
   "x[65]": "x[Widget65]",
   "x[66]": "x[Widget66]",
   "x[67]": "x[Widget67]",
   "x[68]": "x[Widget68]",
   "x[69]": "x[Widget69]",
   "x[70]": "x[Widget70]",
   "x[71]": "x[Widget71]",
   "x[72]": "x[Widget72]",
   "x[73]": "x[Widget73]",
   "x[74]": "x[Widget74]",
   "x[75]": "x[Widget75]",
   "x[76]": "x[Widget76]",
   "x[77]": "x[Widget77]",
   "x[78]": "x[Widget78]",
   "x[79]": "x[Widget79]",
   "x[80]": "x[Widget80]",
   "x[81]": "x[Widget81]",
   "x[82]": "x[Widget82]",
   "x[83]": "x[Widget83]",
   "x[84]": "x[Widget84]",
   "x[85]": "x[Widget85]",
   "x[86]": "x[Widget86]",
   "x[87]": "x[Widget87]",
   "x[88]": "x[Widget88]",
   "x[89]": "x[Widget89]",
   "x[90]": "x[Widget90]",
   "x[91]": "x[Widget91]",
   "x[92]": "x[Widget92]",
   "x[93]": "x[Widget93]",
   "x[94]": "x[Widget94]",
   "x[95]": "x[Widget95]",
   "x[96]": "x[Widget96]",
   "x[97]": "x[Widget97]",
   "x[98]": "x[Widget98]",
   "x[99]": "x[Widget99]",
   "x[100]": "x[Widget100]",
   "x[101]": "x[Widget101]",
   "x[102]": "x[Widget102]",
   "x[103]": "x[Widget103]",
   "x[104]": "x[Widget104]",
   "x[105]": "x[Widget105]",
   "x[106]": "x[Widget106]",
   "x[107]": "x[Widget107]",
   "x[108]": "x[Widget108]",
   "x[109]": "x[Widget109]",
   "x[110]": "x[Widget110]",
   "x[111]": "x[Widget111]",
   "x[112]": "x[Widget112]",
   "x[113]": "x[Widget113]",
   "x[114]": "x[Widget114]",
   "x[115]": "x[Widget115]",
   "x[116]": "x[Widget116]",
   "x[117]": "x[Widget117]",
   "x[118]": "x[Widget118]",
   "x[119]": "x[Widget119]",
   "x[120]": "x[Widget120]",
   "x[121]": "x[Widget121]",
   "x[122]": "x[Widget122]",
   "x[123]": "x[Widget123]",
   "x[124]": "x[Widget124]",
   "x[125]": "x[Widget125]",
   "x[126]": "x[Widget126]",
   "x[127]": "x[Widget127]",
   "x[128]": "x[Widget128]",
   "x[129]": "x[Widget129]",
   "x[130]": "x[Widget130]",
   "x[131]": "x[Widget131]",
   "x[132]": "x[Widget132]",
   "x[133]": "x[Widget133]",
   "x[134]": "x[Widget134]",
   "x[135]": "x[Widget135]",
   "x[136]": "x[Widget136]",
   "x[137]": "x[Widget137]",
   "x[138]": "x[Widget138]",
   "x[139]": "x[Widget139]",
   "x[140]": "x[Widget140]",
   "x[141]": "x[Widget141]",
   "s": "s"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1344480.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     960.0
    ],
    [
     "C2",
     300.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "s",
     1500.0
    ]
   ],
   "error": null
  }
 },
 "OR-099": {
  "source_code_sha256": "80e310f63a9aaeb3d97806acc0a14ec5ae9a29aa5f5beb88356c3a62d2146c13",
  "original_solution_sha256": "f366acaaba74fb857ddc7e972db51d635fb5b70269280e5fdc861c0b95444c80",
  "bindings": [
   [
    "C0",
    "x[1,1]"
   ],
   [
    "C1",
    "x[1,2]"
   ],
   [
    "C2",
    "x[1,3]"
   ],
   [
    "C3",
    "x[1,4]"
   ],
   [
    "C4",
    "x[1,5]"
   ],
   [
    "C5",
    "x[1,6]"
   ],
   [
    "C6",
    "x[1,7]"
   ],
   [
    "C7",
    "x[1,8]"
   ],
   [
    "C8",
    "x[1,9]"
   ],
   [
    "C9",
    "x[1,10]"
   ],
   [
    "C10",
    "x[1,11]"
   ],
   [
    "C11",
    "x[2,1]"
   ],
   [
    "C12",
    "x[2,2]"
   ],
   [
    "C13",
    "x[2,3]"
   ],
   [
    "C14",
    "x[2,4]"
   ],
   [
    "C15",
    "x[2,5]"
   ],
   [
    "C16",
    "x[2,6]"
   ],
   [
    "C17",
    "x[2,7]"
   ],
   [
    "C18",
    "x[2,8]"
   ],
   [
    "C19",
    "x[2,9]"
   ],
   [
    "C20",
    "x[2,10]"
   ],
   [
    "C21",
    "x[2,11]"
   ],
   [
    "C22",
    "x[3,1]"
   ],
   [
    "C23",
    "x[3,2]"
   ],
   [
    "C24",
    "x[3,3]"
   ],
   [
    "C25",
    "x[3,4]"
   ],
   [
    "C26",
    "x[3,5]"
   ],
   [
    "C27",
    "x[3,6]"
   ],
   [
    "C28",
    "x[3,7]"
   ],
   [
    "C29",
    "x[3,8]"
   ],
   [
    "C30",
    "x[3,9]"
   ],
   [
    "C31",
    "x[3,10]"
   ],
   [
    "C32",
    "x[3,11]"
   ],
   [
    "C33",
    "x[4,1]"
   ],
   [
    "C34",
    "x[4,2]"
   ],
   [
    "C35",
    "x[4,3]"
   ],
   [
    "C36",
    "x[4,4]"
   ],
   [
    "C37",
    "x[4,5]"
   ],
   [
    "C38",
    "x[4,6]"
   ],
   [
    "C39",
    "x[4,7]"
   ],
   [
    "C40",
    "x[4,8]"
   ],
   [
    "C41",
    "x[4,9]"
   ],
   [
    "C42",
    "x[4,10]"
   ],
   [
    "C43",
    "x[4,11]"
   ],
   [
    "C44",
    "x[5,1]"
   ],
   [
    "C45",
    "x[5,2]"
   ],
   [
    "C46",
    "x[5,3]"
   ],
   [
    "C47",
    "x[5,4]"
   ],
   [
    "C48",
    "x[5,5]"
   ],
   [
    "C49",
    "x[5,6]"
   ],
   [
    "C50",
    "x[5,7]"
   ],
   [
    "C51",
    "x[5,8]"
   ],
   [
    "C52",
    "x[5,9]"
   ],
   [
    "C53",
    "x[5,10]"
   ],
   [
    "C54",
    "x[5,11]"
   ],
   [
    "C55",
    "x[6,1]"
   ],
   [
    "C56",
    "x[6,2]"
   ],
   [
    "C57",
    "x[6,3]"
   ],
   [
    "C58",
    "x[6,4]"
   ],
   [
    "C59",
    "x[6,5]"
   ],
   [
    "C60",
    "x[6,6]"
   ],
   [
    "C61",
    "x[6,7]"
   ],
   [
    "C62",
    "x[6,8]"
   ],
   [
    "C63",
    "x[6,9]"
   ],
   [
    "C64",
    "x[6,10]"
   ],
   [
    "C65",
    "x[6,11]"
   ],
   [
    "C66",
    "x[7,1]"
   ],
   [
    "C67",
    "x[7,2]"
   ],
   [
    "C68",
    "x[7,3]"
   ],
   [
    "C69",
    "x[7,4]"
   ],
   [
    "C70",
    "x[7,5]"
   ],
   [
    "C71",
    "x[7,6]"
   ],
   [
    "C72",
    "x[7,7]"
   ],
   [
    "C73",
    "x[7,8]"
   ],
   [
    "C74",
    "x[7,9]"
   ],
   [
    "C75",
    "x[7,10]"
   ],
   [
    "C76",
    "x[7,11]"
   ],
   [
    "C77",
    "x[8,1]"
   ],
   [
    "C78",
    "x[8,2]"
   ],
   [
    "C79",
    "x[8,3]"
   ],
   [
    "C80",
    "x[8,4]"
   ],
   [
    "C81",
    "x[8,5]"
   ],
   [
    "C82",
    "x[8,6]"
   ],
   [
    "C83",
    "x[8,7]"
   ],
   [
    "C84",
    "x[8,8]"
   ],
   [
    "C85",
    "x[8,9]"
   ],
   [
    "C86",
    "x[8,10]"
   ],
   [
    "C87",
    "x[8,11]"
   ],
   [
    "C88",
    "x[9,1]"
   ],
   [
    "C89",
    "x[9,2]"
   ],
   [
    "C90",
    "x[9,3]"
   ],
   [
    "C91",
    "x[9,4]"
   ],
   [
    "C92",
    "x[9,5]"
   ],
   [
    "C93",
    "x[9,6]"
   ],
   [
    "C94",
    "x[9,7]"
   ],
   [
    "C95",
    "x[9,8]"
   ],
   [
    "C96",
    "x[9,9]"
   ],
   [
    "C97",
    "x[9,10]"
   ],
   [
    "C98",
    "x[9,11]"
   ],
   [
    "C99",
    "x[10,1]"
   ],
   [
    "C100",
    "x[10,2]"
   ],
   [
    "C101",
    "x[10,3]"
   ],
   [
    "C102",
    "x[10,4]"
   ],
   [
    "C103",
    "x[10,5]"
   ],
   [
    "C104",
    "x[10,6]"
   ],
   [
    "C105",
    "x[10,7]"
   ],
   [
    "C106",
    "x[10,8]"
   ],
   [
    "C107",
    "x[10,9]"
   ],
   [
    "C108",
    "x[10,10]"
   ],
   [
    "C109",
    "x[10,11]"
   ],
   [
    "C110",
    "x[11,1]"
   ],
   [
    "C111",
    "x[11,2]"
   ],
   [
    "C112",
    "x[11,3]"
   ],
   [
    "C113",
    "x[11,4]"
   ],
   [
    "C114",
    "x[11,5]"
   ],
   [
    "C115",
    "x[11,6]"
   ],
   [
    "C116",
    "x[11,7]"
   ],
   [
    "C117",
    "x[11,8]"
   ],
   [
    "C118",
    "x[11,9]"
   ],
   [
    "C119",
    "x[11,10]"
   ],
   [
    "C120",
    "x[11,11]"
   ],
   [
    "C121",
    "y[1]"
   ],
   [
    "C122",
    "y[2]"
   ],
   [
    "C123",
    "y[3]"
   ],
   [
    "C124",
    "y[4]"
   ],
   [
    "C125",
    "y[5]"
   ],
   [
    "C126",
    "y[6]"
   ],
   [
    "C127",
    "y[7]"
   ],
   [
    "C128",
    "y[8]"
   ],
   [
    "C129",
    "y[9]"
   ],
   [
    "C130",
    "y[10]"
   ],
   [
    "C131",
    "y[11]"
   ]
  ],
  "primary_mapping": {
   "x_W1_S1": "x[1,1]",
   "x_W1_S2": "x[1,2]",
   "x_W1_S3": "x[1,3]",
   "x_W1_S4": "x[1,4]",
   "x_W1_S5": "x[1,5]",
   "x_W1_S6": "x[1,6]",
   "x_W1_S7": "x[1,7]",
   "x_W1_S8": "x[1,8]",
   "x_W1_S9": "x[1,9]",
   "x_W1_S10": "x[1,10]",
   "x_W1_S11": "x[1,11]",
   "x_W2_S1": "x[2,1]",
   "x_W2_S2": "x[2,2]",
   "x_W2_S3": "x[2,3]",
   "x_W2_S4": "x[2,4]",
   "x_W2_S5": "x[2,5]",
   "x_W2_S6": "x[2,6]",
   "x_W2_S7": "x[2,7]",
   "x_W2_S8": "x[2,8]",
   "x_W2_S9": "x[2,9]",
   "x_W2_S10": "x[2,10]",
   "x_W2_S11": "x[2,11]",
   "x_W3_S1": "x[3,1]",
   "x_W3_S2": "x[3,2]",
   "x_W3_S3": "x[3,3]",
   "x_W3_S4": "x[3,4]",
   "x_W3_S5": "x[3,5]",
   "x_W3_S6": "x[3,6]",
   "x_W3_S7": "x[3,7]",
   "x_W3_S8": "x[3,8]",
   "x_W3_S9": "x[3,9]",
   "x_W3_S10": "x[3,10]",
   "x_W3_S11": "x[3,11]",
   "x_W4_S1": "x[4,1]",
   "x_W4_S2": "x[4,2]",
   "x_W4_S3": "x[4,3]",
   "x_W4_S4": "x[4,4]",
   "x_W4_S5": "x[4,5]",
   "x_W4_S6": "x[4,6]",
   "x_W4_S7": "x[4,7]",
   "x_W4_S8": "x[4,8]",
   "x_W4_S9": "x[4,9]",
   "x_W4_S10": "x[4,10]",
   "x_W4_S11": "x[4,11]",
   "x_W5_S1": "x[5,1]",
   "x_W5_S2": "x[5,2]",
   "x_W5_S3": "x[5,3]",
   "x_W5_S4": "x[5,4]",
   "x_W5_S5": "x[5,5]",
   "x_W5_S6": "x[5,6]",
   "x_W5_S7": "x[5,7]",
   "x_W5_S8": "x[5,8]",
   "x_W5_S9": "x[5,9]",
   "x_W5_S10": "x[5,10]",
   "x_W5_S11": "x[5,11]",
   "x_W6_S1": "x[6,1]",
   "x_W6_S2": "x[6,2]",
   "x_W6_S3": "x[6,3]",
   "x_W6_S4": "x[6,4]",
   "x_W6_S5": "x[6,5]",
   "x_W6_S6": "x[6,6]",
   "x_W6_S7": "x[6,7]",
   "x_W6_S8": "x[6,8]",
   "x_W6_S9": "x[6,9]",
   "x_W6_S10": "x[6,10]",
   "x_W6_S11": "x[6,11]",
   "x_W7_S1": "x[7,1]",
   "x_W7_S2": "x[7,2]",
   "x_W7_S3": "x[7,3]",
   "x_W7_S4": "x[7,4]",
   "x_W7_S5": "x[7,5]",
   "x_W7_S6": "x[7,6]",
   "x_W7_S7": "x[7,7]",
   "x_W7_S8": "x[7,8]",
   "x_W7_S9": "x[7,9]",
   "x_W7_S10": "x[7,10]",
   "x_W7_S11": "x[7,11]",
   "x_W8_S1": "x[8,1]",
   "x_W8_S2": "x[8,2]",
   "x_W8_S3": "x[8,3]",
   "x_W8_S4": "x[8,4]",
   "x_W8_S5": "x[8,5]",
   "x_W8_S6": "x[8,6]",
   "x_W8_S7": "x[8,7]",
   "x_W8_S8": "x[8,8]",
   "x_W8_S9": "x[8,9]",
   "x_W8_S10": "x[8,10]",
   "x_W8_S11": "x[8,11]",
   "x_W9_S1": "x[9,1]",
   "x_W9_S2": "x[9,2]",
   "x_W9_S3": "x[9,3]",
   "x_W9_S4": "x[9,4]",
   "x_W9_S5": "x[9,5]",
   "x_W9_S6": "x[9,6]",
   "x_W9_S7": "x[9,7]",
   "x_W9_S8": "x[9,8]",
   "x_W9_S9": "x[9,9]",
   "x_W9_S10": "x[9,10]",
   "x_W9_S11": "x[9,11]",
   "x_W10_S1": "x[10,1]",
   "x_W10_S2": "x[10,2]",
   "x_W10_S3": "x[10,3]",
   "x_W10_S4": "x[10,4]",
   "x_W10_S5": "x[10,5]",
   "x_W10_S6": "x[10,6]",
   "x_W10_S7": "x[10,7]",
   "x_W10_S8": "x[10,8]",
   "x_W10_S9": "x[10,9]",
   "x_W10_S10": "x[10,10]",
   "x_W10_S11": "x[10,11]",
   "x_W11_S1": "x[11,1]",
   "x_W11_S2": "x[11,2]",
   "x_W11_S3": "x[11,3]",
   "x_W11_S4": "x[11,4]",
   "x_W11_S5": "x[11,5]",
   "x_W11_S6": "x[11,6]",
   "x_W11_S7": "x[11,7]",
   "x_W11_S8": "x[11,8]",
   "x_W11_S9": "x[11,9]",
   "x_W11_S10": "x[11,10]",
   "x_W11_S11": "x[11,11]",
   "y_W1": "y[1]",
   "y_W2": "y[2]",
   "y_W3": "y[3]",
   "y_W4": "y[4]",
   "y_W5": "y[5]",
   "y_W6": "y[6]",
   "y_W7": "y[7]",
   "y_W8": "y[8]",
   "y_W9": "y[9]",
   "y_W10": "y[10]",
   "y_W11": "y[11]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 10188.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     21.0
    ],
    [
     "C56",
     40.0
    ],
    [
     "C57",
     20.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     20.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     45.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     44.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     9.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     35.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     25.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     38.0
    ],
    [
     "C107",
     32.0
    ],
    [
     "C108",
     41.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     1.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     1.0
    ],
    [
     "C131",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-003": {
  "source_code_sha256": "f0fb9bdedcfb9e092f943d7a270d7d23eccb4f3ddebe9a81638c21c369c0ceff",
  "original_solution_sha256": "8595ff8cc874b03b28a1a1da093c74144fe369f0871dd21a8de22e8489fb5031",
  "bindings": [
   [
    "C0",
    "x[S1,C1]"
   ],
   [
    "C1",
    "x[S1,C2]"
   ],
   [
    "C2",
    "x[S1,C3]"
   ],
   [
    "C3",
    "x[S1,C4]"
   ],
   [
    "C4",
    "x[S1,C5]"
   ],
   [
    "C5",
    "x[S1,C6]"
   ],
   [
    "C6",
    "x[S1,C7]"
   ],
   [
    "C7",
    "x[S1,C8]"
   ],
   [
    "C8",
    "x[S1,C9]"
   ],
   [
    "C9",
    "x[S1,C10]"
   ],
   [
    "C10",
    "x[S2,C1]"
   ],
   [
    "C11",
    "x[S2,C2]"
   ],
   [
    "C12",
    "x[S2,C3]"
   ],
   [
    "C13",
    "x[S2,C4]"
   ],
   [
    "C14",
    "x[S2,C5]"
   ],
   [
    "C15",
    "x[S2,C6]"
   ],
   [
    "C16",
    "x[S2,C7]"
   ],
   [
    "C17",
    "x[S2,C8]"
   ],
   [
    "C18",
    "x[S2,C9]"
   ],
   [
    "C19",
    "x[S2,C10]"
   ],
   [
    "C20",
    "x[S3,C1]"
   ],
   [
    "C21",
    "x[S3,C2]"
   ],
   [
    "C22",
    "x[S3,C3]"
   ],
   [
    "C23",
    "x[S3,C4]"
   ],
   [
    "C24",
    "x[S3,C5]"
   ],
   [
    "C25",
    "x[S3,C6]"
   ],
   [
    "C26",
    "x[S3,C7]"
   ],
   [
    "C27",
    "x[S3,C8]"
   ],
   [
    "C28",
    "x[S3,C9]"
   ],
   [
    "C29",
    "x[S3,C10]"
   ],
   [
    "C30",
    "x[S4,C1]"
   ],
   [
    "C31",
    "x[S4,C2]"
   ],
   [
    "C32",
    "x[S4,C3]"
   ],
   [
    "C33",
    "x[S4,C4]"
   ],
   [
    "C34",
    "x[S4,C5]"
   ],
   [
    "C35",
    "x[S4,C6]"
   ],
   [
    "C36",
    "x[S4,C7]"
   ],
   [
    "C37",
    "x[S4,C8]"
   ],
   [
    "C38",
    "x[S4,C9]"
   ],
   [
    "C39",
    "x[S4,C10]"
   ],
   [
    "C40",
    "x[S5,C1]"
   ],
   [
    "C41",
    "x[S5,C2]"
   ],
   [
    "C42",
    "x[S5,C3]"
   ],
   [
    "C43",
    "x[S5,C4]"
   ],
   [
    "C44",
    "x[S5,C5]"
   ],
   [
    "C45",
    "x[S5,C6]"
   ],
   [
    "C46",
    "x[S5,C7]"
   ],
   [
    "C47",
    "x[S5,C8]"
   ],
   [
    "C48",
    "x[S5,C9]"
   ],
   [
    "C49",
    "x[S5,C10]"
   ],
   [
    "C50",
    "x[S6,C1]"
   ],
   [
    "C51",
    "x[S6,C2]"
   ],
   [
    "C52",
    "x[S6,C3]"
   ],
   [
    "C53",
    "x[S6,C4]"
   ],
   [
    "C54",
    "x[S6,C5]"
   ],
   [
    "C55",
    "x[S6,C6]"
   ],
   [
    "C56",
    "x[S6,C7]"
   ],
   [
    "C57",
    "x[S6,C8]"
   ],
   [
    "C58",
    "x[S6,C9]"
   ],
   [
    "C59",
    "x[S6,C10]"
   ],
   [
    "C60",
    "x[S7,C1]"
   ],
   [
    "C61",
    "x[S7,C2]"
   ],
   [
    "C62",
    "x[S7,C3]"
   ],
   [
    "C63",
    "x[S7,C4]"
   ],
   [
    "C64",
    "x[S7,C5]"
   ],
   [
    "C65",
    "x[S7,C6]"
   ],
   [
    "C66",
    "x[S7,C7]"
   ],
   [
    "C67",
    "x[S7,C8]"
   ],
   [
    "C68",
    "x[S7,C9]"
   ],
   [
    "C69",
    "x[S7,C10]"
   ],
   [
    "C70",
    "x[S8,C1]"
   ],
   [
    "C71",
    "x[S8,C2]"
   ],
   [
    "C72",
    "x[S8,C3]"
   ],
   [
    "C73",
    "x[S8,C4]"
   ],
   [
    "C74",
    "x[S8,C5]"
   ],
   [
    "C75",
    "x[S8,C6]"
   ],
   [
    "C76",
    "x[S8,C7]"
   ],
   [
    "C77",
    "x[S8,C8]"
   ],
   [
    "C78",
    "x[S8,C9]"
   ],
   [
    "C79",
    "x[S8,C10]"
   ],
   [
    "C80",
    "x[S9,C1]"
   ],
   [
    "C81",
    "x[S9,C2]"
   ],
   [
    "C82",
    "x[S9,C3]"
   ],
   [
    "C83",
    "x[S9,C4]"
   ],
   [
    "C84",
    "x[S9,C5]"
   ],
   [
    "C85",
    "x[S9,C6]"
   ],
   [
    "C86",
    "x[S9,C7]"
   ],
   [
    "C87",
    "x[S9,C8]"
   ],
   [
    "C88",
    "x[S9,C9]"
   ],
   [
    "C89",
    "x[S9,C10]"
   ],
   [
    "C90",
    "x[S10,C1]"
   ],
   [
    "C91",
    "x[S10,C2]"
   ],
   [
    "C92",
    "x[S10,C3]"
   ],
   [
    "C93",
    "x[S10,C4]"
   ],
   [
    "C94",
    "x[S10,C5]"
   ],
   [
    "C95",
    "x[S10,C6]"
   ],
   [
    "C96",
    "x[S10,C7]"
   ],
   [
    "C97",
    "x[S10,C8]"
   ],
   [
    "C98",
    "x[S10,C9]"
   ],
   [
    "C99",
    "x[S10,C10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_8": "x[S7,C9]",
   "x_6_9": "x[S7,C10]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 36778.51094448115,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     120.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     168.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     168.0
    ],
    [
     "C20",
     216.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     24.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     24.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     216.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     216.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     144.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     168.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     48.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     48.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     192.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     168.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-008": {
  "source_code_sha256": "8c46d785b6af2b4dbc1d5a7f31e6cd30fbb7d40ea4959b20f2aaf98db5a178f4",
  "original_solution_sha256": "5d45a995db09f4196573b0de17bc12170a9cad541aca69ded95259fd83ef6909",
  "bindings": [
   [
    "C0",
    "x[Supplier1,Customer1]"
   ],
   [
    "C1",
    "x[Supplier1,Customer2]"
   ],
   [
    "C2",
    "x[Supplier1,Customer3]"
   ],
   [
    "C3",
    "x[Supplier1,Customer4]"
   ],
   [
    "C4",
    "x[Supplier1,Customer5]"
   ],
   [
    "C5",
    "x[Supplier1,Customer6]"
   ],
   [
    "C6",
    "x[Supplier2,Customer1]"
   ],
   [
    "C7",
    "x[Supplier2,Customer2]"
   ],
   [
    "C8",
    "x[Supplier2,Customer3]"
   ],
   [
    "C9",
    "x[Supplier2,Customer4]"
   ],
   [
    "C10",
    "x[Supplier2,Customer5]"
   ],
   [
    "C11",
    "x[Supplier2,Customer6]"
   ],
   [
    "C12",
    "x[Supplier3,Customer1]"
   ],
   [
    "C13",
    "x[Supplier3,Customer2]"
   ],
   [
    "C14",
    "x[Supplier3,Customer3]"
   ],
   [
    "C15",
    "x[Supplier3,Customer4]"
   ],
   [
    "C16",
    "x[Supplier3,Customer5]"
   ],
   [
    "C17",
    "x[Supplier3,Customer6]"
   ],
   [
    "C18",
    "x[Supplier4,Customer1]"
   ],
   [
    "C19",
    "x[Supplier4,Customer2]"
   ],
   [
    "C20",
    "x[Supplier4,Customer3]"
   ],
   [
    "C21",
    "x[Supplier4,Customer4]"
   ],
   [
    "C22",
    "x[Supplier4,Customer5]"
   ],
   [
    "C23",
    "x[Supplier4,Customer6]"
   ],
   [
    "C24",
    "x[Supplier5,Customer1]"
   ],
   [
    "C25",
    "x[Supplier5,Customer2]"
   ],
   [
    "C26",
    "x[Supplier5,Customer3]"
   ],
   [
    "C27",
    "x[Supplier5,Customer4]"
   ],
   [
    "C28",
    "x[Supplier5,Customer5]"
   ],
   [
    "C29",
    "x[Supplier5,Customer6]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Supplier1,Customer1]",
   "x_0_1": "x[Supplier1,Customer2]",
   "x_0_2": "x[Supplier1,Customer3]",
   "x_0_3": "x[Supplier1,Customer4]",
   "x_0_4": "x[Supplier1,Customer5]",
   "x_0_5": "x[Supplier1,Customer6]",
   "x_1_0": "x[Supplier2,Customer1]",
   "x_1_1": "x[Supplier2,Customer2]",
   "x_1_2": "x[Supplier2,Customer3]",
   "x_1_3": "x[Supplier2,Customer4]",
   "x_1_4": "x[Supplier2,Customer5]",
   "x_1_5": "x[Supplier2,Customer6]",
   "x_2_0": "x[Supplier3,Customer1]",
   "x_2_1": "x[Supplier3,Customer2]",
   "x_2_2": "x[Supplier3,Customer3]",
   "x_2_3": "x[Supplier3,Customer4]",
   "x_2_4": "x[Supplier3,Customer5]",
   "x_2_5": "x[Supplier3,Customer6]",
   "x_3_0": "x[Supplier4,Customer1]",
   "x_3_1": "x[Supplier4,Customer2]",
   "x_3_2": "x[Supplier4,Customer3]",
   "x_3_3": "x[Supplier4,Customer4]",
   "x_3_4": "x[Supplier4,Customer5]",
   "x_3_5": "x[Supplier4,Customer6]",
   "x_4_0": "x[Supplier5,Customer1]",
   "x_4_1": "x[Supplier5,Customer2]",
   "x_4_2": "x[Supplier5,Customer3]",
   "x_4_3": "x[Supplier5,Customer4]",
   "x_4_4": "x[Supplier5,Customer5]",
   "x_4_5": "x[Supplier5,Customer6]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 660.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     60.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     95.0
    ],
    [
     "C6",
     70.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     80.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     85.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     90.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-011": {
  "source_code_sha256": "34798306b81b4e801ef04c1a73d52b8d9b66757e96c542931fcaf6d32014eb40",
  "original_solution_sha256": "68203b0c38371b684628431f6920ddc720a25bd2b165f6fb4fdc65d32eebc533",
  "bindings": [
   [
    "C0",
    "x[id999]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[id999]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 3552260.54,
   "variables": [
    [
     "C0",
     8171.0
    ]
   ],
   "error": null
  }
 },
 "OR-016": {
  "source_code_sha256": "09b52b6445c29ac3bcece7dc5e744d4774efeb8dcb555bfba1599e6a06211a95",
  "original_solution_sha256": "d95a1f814189d1f5f3f10b537dbc1a9d093b9a77cd8e7c80b2ec2de642f6dcdf",
  "bindings": [
   [
    "C0",
    "x[Electronics - 500]"
   ],
   [
    "C1",
    "x[Home Goods - 30]"
   ],
   [
    "C2",
    "x[Furniture - 25]"
   ],
   [
    "C3",
    "x[Toys - 50]"
   ],
   [
    "C4",
    "x[Electronics - 25]"
   ],
   [
    "C5",
    "x[Sports - 500]"
   ],
   [
    "C6",
    "x[Beauty - 50]"
   ],
   [
    "C7",
    "x[Beauty - 300]"
   ],
   [
    "C8",
    "x[Electronics - 300]"
   ],
   [
    "C9",
    "x[Clothing - 30]"
   ],
   [
    "C10",
    "x[Furniture - 30]"
   ],
   [
    "C11",
    "x[Toys - 300]"
   ],
   [
    "C12",
    "x[Sports - 300]"
   ],
   [
    "C13",
    "x[Clothing - 300]"
   ],
   [
    "C14",
    "x[Clothing - 50]"
   ],
   [
    "C15",
    "x[Clothing - 500]"
   ],
   [
    "C16",
    "x[Furniture - 500]"
   ],
   [
    "C17",
    "x[Furniture - 50]"
   ],
   [
    "C18",
    "x[Beauty - 30]"
   ],
   [
    "C19",
    "x[Beauty - 500]"
   ],
   [
    "C20",
    "x[Toys - 25]"
   ],
   [
    "C21",
    "x[Electronics - 50]"
   ],
   [
    "C22",
    "x[Sports - 50]"
   ],
   [
    "C23",
    "x[Home Goods - 25]"
   ],
   [
    "C24",
    "x[Toys - 30]"
   ],
   [
    "C25",
    "x[Electronics - 30]"
   ],
   [
    "C26",
    "x[Furniture - 300]"
   ],
   [
    "C27",
    "x[Home Goods - 300]"
   ],
   [
    "C28",
    "x[Beauty - 25]"
   ],
   [
    "C29",
    "x[Home Goods - 500]"
   ],
   [
    "C30",
    "x[Clothing - 25]"
   ],
   [
    "C31",
    "x[Home Goods - 50]"
   ],
   [
    "C32",
    "x[Sports - 30]"
   ],
   [
    "C33",
    "x[Sports - 25]"
   ],
   [
    "C34",
    "x[Toys - 500]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Beauty - 25]",
   "x_2": "x[Beauty - 30]",
   "x_3": "x[Beauty - 300]",
   "x_4": "x[Beauty - 50]",
   "x_5": "x[Beauty - 500]",
   "x_6": "x[Clothing - 25]",
   "x_7": "x[Clothing - 30]",
   "x_8": "x[Clothing - 300]",
   "x_9": "x[Clothing - 50]",
   "x_10": "x[Clothing - 500]",
   "x_11": "x[Electronics - 25]",
   "x_12": "x[Electronics - 30]",
   "x_13": "x[Electronics - 300]",
   "x_14": "x[Electronics - 50]",
   "x_15": "x[Electronics - 500]",
   "x_16": "x[Home Goods - 25]",
   "x_17": "x[Home Goods - 30]",
   "x_18": "x[Home Goods - 50]",
   "x_19": "x[Home Goods - 300]",
   "x_20": "x[Home Goods - 500]",
   "x_21": "x[Sports - 25]",
   "x_22": "x[Sports - 30]",
   "x_23": "x[Sports - 50]",
   "x_24": "x[Sports - 300]",
   "x_25": "x[Sports - 500]",
   "x_26": "x[Furniture - 25]",
   "x_27": "x[Furniture - 30]",
   "x_28": "x[Furniture - 50]",
   "x_29": "x[Furniture - 300]",
   "x_30": "x[Furniture - 500]",
   "x_31": "x[Toys - 25]",
   "x_32": "x[Toys - 30]",
   "x_33": "x[Toys - 50]",
   "x_34": "x[Toys - 300]",
   "x_35": "x[Toys - 500]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1568955.0,
   "variables": [
    [
     "C0",
     262.0
    ],
    [
     "C1",
     218.0
    ],
    [
     "C2",
     242.0
    ],
    [
     "C3",
     291.0
    ],
    [
     "C4",
     273.0
    ],
    [
     "C5",
     259.0
    ],
    [
     "C6",
     263.0
    ],
    [
     "C7",
     216.0
    ],
    [
     "C8",
     286.0
    ],
    [
     "C9",
     261.0
    ],
    [
     "C10",
     213.0
    ],
    [
     "C11",
     199.0
    ],
    [
     "C12",
     208.0
    ],
    [
     "C13",
     295.0
    ],
    [
     "C14",
     290.0
    ],
    [
     "C15",
     244.0
    ],
    [
     "C16",
     251.0
    ],
    [
     "C17",
     272.0
    ],
    [
     "C18",
     202.0
    ],
    [
     "C19",
     256.0
    ],
    [
     "C20",
     257.0
    ],
    [
     "C21",
     268.0
    ],
    [
     "C22",
     285.0
    ],
    [
     "C23",
     255.0
    ],
    [
     "C24",
     235.0
    ],
    [
     "C25",
     220.0
    ],
    [
     "C26",
     224.0
    ],
    [
     "C27",
     195.0
    ],
    [
     "C28",
     240.0
    ],
    [
     "C29",
     248.0
    ],
    [
     "C30",
     281.0
    ],
    [
     "C31",
     278.0
    ],
    [
     "C32",
     227.0
    ],
    [
     "C33",
     269.0
    ],
    [
     "C34",
     264.0
    ]
   ],
   "error": null
  }
 },
 "OR-020": {
  "source_code_sha256": "97e538554be2693dcb8c0040d586640e6b16acc9fe88a56dba3b92b21f9c294f",
  "original_solution_sha256": "88c3c37d6e8288dd2e5f9fcee1cfdf0cf4449bd356c396d5d8c921abf125baa1",
  "bindings": [
   [
    "C0",
    "x[S18_3685]"
   ],
   [
    "C1",
    "x[S700_3167]"
   ],
   [
    "C2",
    "x[S32_3522]"
   ],
   [
    "C3",
    "x[S24_3420]"
   ],
   [
    "C4",
    "x[S10_4757]"
   ],
   [
    "C5",
    "x[S18_1889]"
   ],
   [
    "C6",
    "x[S18_4522]"
   ],
   [
    "C7",
    "x[S700_2466]"
   ],
   [
    "C8",
    "x[S12_3990]"
   ],
   [
    "C9",
    "x[S700_3505]"
   ],
   [
    "C10",
    "x[S10_2016]"
   ],
   [
    "C11",
    "x[S24_1444]"
   ],
   [
    "C12",
    "x[S18_4600]"
   ],
   [
    "C13",
    "x[S50_1514]"
   ],
   [
    "C14",
    "x[S700_2834]"
   ],
   [
    "C15",
    "x[S32_2509]"
   ],
   [
    "C16",
    "x[S32_1374]"
   ],
   [
    "C17",
    "x[S24_1785]"
   ],
   [
    "C18",
    "x[S24_3191]"
   ],
   [
    "C19",
    "x[S700_2824]"
   ],
   [
    "C20",
    "x[S12_3148]"
   ],
   [
    "C21",
    "x[S12_3380]"
   ],
   [
    "C22",
    "x[S18_2581]"
   ],
   [
    "C23",
    "x[S18_3856]"
   ],
   [
    "C24",
    "x[S32_2206]"
   ],
   [
    "C25",
    "x[S18_2238]"
   ],
   [
    "C26",
    "x[S18_1342]"
   ],
   [
    "C27",
    "x[S700_1138]"
   ],
   [
    "C28",
    "x[S18_2957]"
   ],
   [
    "C29",
    "x[S700_3962]"
   ],
   [
    "C30",
    "x[S24_2887]"
   ],
   [
    "C31",
    "x[S18_1749]"
   ],
   [
    "C32",
    "x[S32_4485]"
   ],
   [
    "C33",
    "x[S18_1589]"
   ],
   [
    "C34",
    "x[S18_3140]"
   ],
   [
    "C35",
    "x[S18_3029]"
   ],
   [
    "C36",
    "x[S18_4721]"
   ],
   [
    "C37",
    "x[S18_3136]"
   ],
   [
    "C38",
    "x[S24_1937]"
   ],
   [
    "C39",
    "x[S700_2610]"
   ],
   [
    "C40",
    "x[S18_2319]"
   ],
   [
    "C41",
    "x[S50_4713]"
   ],
   [
    "C42",
    "x[S32_1268]"
   ],
   [
    "C43",
    "x[S24_2841]"
   ],
   [
    "C44",
    "x[S50_1392]"
   ],
   [
    "C45",
    "x[S18_2248]"
   ],
   [
    "C46",
    "x[S12_3891]"
   ],
   [
    "C47",
    "x[S18_4027]"
   ],
   [
    "C48",
    "x[S18_3320]"
   ],
   [
    "C49",
    "x[S10_4698]"
   ],
   [
    "C50",
    "x[S50_1341]"
   ],
   [
    "C51",
    "x[S18_1129]"
   ],
   [
    "C52",
    "x[S12_1108]"
   ],
   [
    "C53",
    "x[S24_2972]"
   ],
   [
    "C54",
    "x[S32_4289]"
   ],
   [
    "C55",
    "x[S24_2300]"
   ],
   [
    "C56",
    "x[S18_4668]"
   ],
   [
    "C57",
    "x[S24_3371]"
   ],
   [
    "C58",
    "x[S24_4048]"
   ],
   [
    "C59",
    "x[S10_1949]"
   ],
   [
    "C60",
    "x[S24_3969]"
   ],
   [
    "C61",
    "x[S18_4409]"
   ],
   [
    "C62",
    "x[S18_2795]"
   ],
   [
    "C63",
    "x[S18_2625]"
   ],
   [
    "C64",
    "x[S12_2823]"
   ],
   [
    "C65",
    "x[S24_1046]"
   ],
   [
    "C66",
    "x[S18_2325]"
   ],
   [
    "C67",
    "x[S18_2432]"
   ],
   [
    "C68",
    "x[S24_4620]"
   ],
   [
    "C69",
    "x[S24_3856]"
   ],
   [
    "C70",
    "x[S700_1938]"
   ],
   [
    "C71",
    "x[S24_3151]"
   ],
   [
    "C72",
    "x[S18_3482]"
   ],
   [
    "C73",
    "x[S18_1662]"
   ],
   [
    "C74",
    "x[S24_4258]"
   ],
   [
    "C75",
    "x[S18_3782]"
   ],
   [
    "C76",
    "x[S24_1578]"
   ],
   [
    "C77",
    "x[S10_1678]"
   ],
   [
    "C78",
    "x[S24_1628]"
   ],
   [
    "C79",
    "x[S12_1666]"
   ],
   [
    "C80",
    "x[S24_4278]"
   ],
   [
    "C81",
    "x[S24_2011]"
   ],
   [
    "C82",
    "x[S18_2949]"
   ],
   [
    "C83",
    "x[S24_2360]"
   ],
   [
    "C84",
    "x[S24_2840]"
   ],
   [
    "C85",
    "x[S24_3949]"
   ],
   [
    "C86",
    "x[S24_3432]"
   ],
   [
    "C87",
    "x[S18_2870]"
   ],
   [
    "C88",
    "x[S12_4675]"
   ],
   [
    "C89",
    "x[S18_3259]"
   ],
   [
    "C90",
    "x[S18_1984]"
   ],
   [
    "C91",
    "x[S18_3232]"
   ],
   [
    "C92",
    "x[S24_3816]"
   ],
   [
    "C93",
    "x[S12_4473]"
   ],
   [
    "C94",
    "x[S24_2000]"
   ],
   [
    "C95",
    "x[S18_1367]"
   ],
   [
    "C96",
    "x[S700_1691]"
   ],
   [
    "C97",
    "x[S700_2047]"
   ],
   [
    "C98",
    "x[S72_3212]"
   ],
   [
    "C99",
    "x[S18_4933]"
   ],
   [
    "C100",
    "x[S700_4002]"
   ],
   [
    "C101",
    "x[S10_4962]"
   ],
   [
    "C102",
    "x[S24_2022]"
   ],
   [
    "C103",
    "x[S18_1097]"
   ],
   [
    "C104",
    "x[S72_1253]"
   ],
   [
    "C105",
    "x[S18_3278]"
   ],
   [
    "C106",
    "x[S24_2766]"
   ],
   [
    "C107",
    "x[S12_1099]"
   ],
   [
    "C108",
    "x[S32_3207]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[S10_1678]",
   "x_2": "x[S10_1949]",
   "x_3": "x[S10_2016]",
   "x_4": "x[S10_4698]",
   "x_5": "x[S10_4757]",
   "x_6": "x[S10_4962]",
   "x_7": "x[S12_1099]",
   "x_8": "x[S12_1108]",
   "x_9": "x[S12_1666]",
   "x_10": "x[S12_2823]",
   "x_11": "x[S12_3148]",
   "x_12": "x[S12_3380]",
   "x_13": "x[S12_3891]",
   "x_14": "x[S12_3990]",
   "x_15": "x[S12_4473]",
   "x_16": "x[S12_4675]",
   "x_17": "x[S18_1097]",
   "x_18": "x[S18_1129]",
   "x_19": "x[S18_1342]",
   "x_20": "x[S18_1367]",
   "x_21": "x[S18_1589]",
   "x_22": "x[S18_1662]",
   "x_23": "x[S18_1749]",
   "x_24": "x[S18_1889]",
   "x_25": "x[S18_1984]",
   "x_26": "x[S18_2238]",
   "x_27": "x[S18_2248]",
   "x_28": "x[S18_2319]",
   "x_29": "x[S18_2325]",
   "x_30": "x[S18_2432]",
   "x_31": "x[S18_2581]",
   "x_32": "x[S18_2625]",
   "x_33": "x[S18_2795]",
   "x_34": "x[S18_2870]",
   "x_35": "x[S18_2949]",
   "x_36": "x[S18_2957]",
   "x_37": "x[S18_3029]",
   "x_38": "x[S18_3136]",
   "x_39": "x[S18_3140]",
   "x_40": "x[S18_3232]",
   "x_41": "x[S18_3259]",
   "x_42": "x[S18_3278]",
   "x_43": "x[S18_3320]",
   "x_44": "x[S18_3482]",
   "x_45": "x[S18_3685]",
   "x_46": "x[S18_3782]",
   "x_47": "x[S18_3856]",
   "x_48": "x[S18_4027]",
   "x_49": "x[S18_4409]",
   "x_50": "x[S18_4522]",
   "x_51": "x[S18_4600]",
   "x_52": "x[S18_4668]",
   "x_53": "x[S18_4721]",
   "x_54": "x[S18_4933]",
   "x_55": "x[S24_1046]",
   "x_56": "x[S24_1444]",
   "x_57": "x[S24_1578]",
   "x_58": "x[S24_1628]",
   "x_59": "x[S24_1785]",
   "x_60": "x[S24_1937]",
   "x_61": "x[S24_2000]",
   "x_62": "x[S24_2011]",
   "x_63": "x[S24_2022]",
   "x_64": "x[S24_2300]",
   "x_65": "x[S24_2360]",
   "x_66": "x[S24_2766]",
   "x_67": "x[S24_2840]",
   "x_68": "x[S24_2841]",
   "x_69": "x[S24_2887]",
   "x_70": "x[S24_2972]",
   "x_71": "x[S24_3151]",
   "x_72": "x[S24_3191]",
   "x_73": "x[S24_3371]",
   "x_74": "x[S24_3420]",
   "x_75": "x[S24_3432]",
   "x_76": "x[S24_3816]",
   "x_77": "x[S24_3856]",
   "x_78": "x[S24_3949]",
   "x_79": "x[S24_3969]",
   "x_80": "x[S24_4048]",
   "x_81": "x[S24_4258]",
   "x_82": "x[S24_4278]",
   "x_83": "x[S24_4620]",
   "x_84": "x[S32_1268]",
   "x_85": "x[S32_1374]",
   "x_86": "x[S32_2206]",
   "x_87": "x[S32_2509]",
   "x_88": "x[S32_3207]",
   "x_89": "x[S32_3522]",
   "x_90": "x[S32_4289]",
   "x_91": "x[S32_4485]",
   "x_92": "x[S50_1341]",
   "x_93": "x[S50_1392]",
   "x_94": "x[S50_1514]",
   "x_95": "x[S50_4713]",
   "x_96": "x[S700_1138]",
   "x_97": "x[S700_1691]",
   "x_98": "x[S700_1938]",
   "x_99": "x[S700_2047]",
   "x_100": "x[S700_2466]",
   "x_101": "x[S700_2610]",
   "x_102": "x[S700_2824]",
   "x_103": "x[S700_2834]",
   "x_104": "x[S700_3167]",
   "x_105": "x[S700_3505]",
   "x_106": "x[S700_3962]",
   "x_107": "x[S700_4002]",
   "x_108": "x[S72_1253]",
   "x_109": "x[S72_3212]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 11258129.669999998,
   "variables": [
    [
     "C0",
     1312.0
    ],
    [
     "C1",
     1247.0
    ],
    [
     "C2",
     1268.0
    ],
    [
     "C3",
     1183.0
    ],
    [
     "C4",
     1292.0
    ],
    [
     "C5",
     1267.0
    ],
    [
     "C6",
     1246.0
    ],
    [
     "C7",
     1274.0
    ],
    [
     "C8",
     1091.0
    ],
    [
     "C9",
     1236.0
    ],
    [
     "C10",
     1242.0
    ],
    [
     "C11",
     1347.0
    ],
    [
     "C12",
     1394.0
    ],
    [
     "C13",
     1278.0
    ],
    [
     "C14",
     1170.0
    ],
    [
     "C15",
     1292.0
    ],
    [
     "C16",
     1194.0
    ],
    [
     "C17",
     1057.0
    ],
    [
     "C18",
     1086.0
    ],
    [
     "C19",
     1342.0
    ],
    [
     "C20",
     1232.0
    ],
    [
     "C21",
     1157.0
    ],
    [
     "C22",
     1028.0
    ],
    [
     "C23",
     1361.0
    ],
    [
     "C24",
     1160.0
    ],
    [
     "C25",
     1293.0
    ],
    [
     "C26",
     1378.0
    ],
    [
     "C27",
     1254.0
    ],
    [
     "C28",
     1267.0
    ],
    [
     "C29",
     1175.0
    ],
    [
     "C30",
     1122.0
    ],
    [
     "C31",
     1063.0
    ],
    [
     "C32",
     1093.0
    ],
    [
     "C33",
     1278.0
    ],
    [
     "C34",
     1153.0
    ],
    [
     "C35",
     1172.0
    ],
    [
     "C36",
     1211.0
    ],
    [
     "C37",
     1175.0
    ],
    [
     "C38",
     1165.0
    ],
    [
     "C39",
     1329.0
    ],
    [
     "C40",
     1363.0
    ],
    [
     "C41",
     1237.0
    ],
    [
     "C42",
     1185.0
    ],
    [
     "C43",
     1135.0
    ],
    [
     "C44",
     1330.0
    ],
    [
     "C45",
     994.0
    ],
    [
     "C46",
     1237.0
    ],
    [
     "C47",
     1241.0
    ],
    [
     "C48",
     1240.0
    ],
    [
     "C49",
     1258.0
    ],
    [
     "C50",
     1351.0
    ],
    [
     "C51",
     1287.0
    ],
    [
     "C52",
     1334.0
    ],
    [
     "C53",
     1223.0
    ],
    [
     "C54",
     1193.0
    ],
    [
     "C55",
     1358.0
    ],
    [
     "C56",
     1327.0
    ],
    [
     "C57",
     1263.0
    ],
    [
     "C58",
     1145.0
    ],
    [
     "C59",
     1316.0
    ],
    [
     "C60",
     1013.0
    ],
    [
     "C61",
     1002.0
    ],
    [
     "C62",
     1056.0
    ],
    [
     "C63",
     1200.0
    ],
    [
     "C64",
     1340.0
    ],
    [
     "C65",
     988.0
    ],
    [
     "C66",
     1134.0
    ],
    [
     "C67",
     1362.0
    ],
    [
     "C68",
     1149.0
    ],
    [
     "C69",
     1450.0
    ],
    [
     "C70",
     1144.0
    ],
    [
     "C71",
     1317.0
    ],
    [
     "C72",
     1206.0
    ],
    [
     "C73",
     1277.0
    ],
    [
     "C74",
     1247.0
    ],
    [
     "C75",
     1224.0
    ],
    [
     "C76",
     1286.0
    ],
    [
     "C77",
     1280.0
    ],
    [
     "C78",
     1193.0
    ],
    [
     "C79",
     1323.0
    ],
    [
     "C80",
     1205.0
    ],
    [
     "C81",
     1288.0
    ],
    [
     "C82",
     1335.0
    ],
    [
     "C83",
     1177.0
    ],
    [
     "C84",
     1329.0
    ],
    [
     "C85",
     1357.0
    ],
    [
     "C86",
     1119.0
    ],
    [
     "C87",
     1139.0
    ],
    [
     "C88",
     1331.0
    ],
    [
     "C89",
     1157.0
    ],
    [
     "C90",
     1233.0
    ],
    [
     "C91",
     2457.0
    ],
    [
     "C92",
     1182.0
    ],
    [
     "C93",
     1386.0
    ],
    [
     "C94",
     1263.0
    ],
    [
     "C95",
     1244.0
    ],
    [
     "C96",
     1158.0
    ],
    [
     "C97",
     1189.0
    ],
    [
     "C98",
     1291.0
    ],
    [
     "C99",
     957.0
    ],
    [
     "C100",
     1394.0
    ],
    [
     "C101",
     1262.0
    ],
    [
     "C102",
     1177.0
    ],
    [
     "C103",
     1364.0
    ],
    [
     "C104",
     1252.0
    ],
    [
     "C105",
     1139.0
    ],
    [
     "C106",
     1191.0
    ],
    [
     "C107",
     1132.0
    ],
    [
     "C108",
     1253.0
    ]
   ],
   "error": null
  }
 },
 "OR-024": {
  "source_code_sha256": "9baf38c05b51d9250b71d4c201605f7506c89d0ece00fc36d6e45c9aebcdacc0",
  "original_solution_sha256": "11c368279238fc1c9a0f65bdc7407d4e47a9ee4bfc21a22383c9d70cc64bb965",
  "bindings": [
   [
    "C0",
    "x[S700_2610]"
   ],
   [
    "C1",
    "x[S700_1691]"
   ],
   [
    "C2",
    "x[S700_2047]"
   ],
   [
    "C3",
    "x[S700_1138]"
   ],
   [
    "C4",
    "x[S700_2834]"
   ],
   [
    "C5",
    "x[S700_2824]"
   ],
   [
    "C6",
    "x[S700_3167]"
   ],
   [
    "C7",
    "x[S700_4002]"
   ],
   [
    "C8",
    "x[S700_3962]"
   ],
   [
    "C9",
    "x[S700_2466]"
   ],
   [
    "C10",
    "x[S700_3505]"
   ],
   [
    "C11",
    "x[S700_1938]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[S700_1138]",
   "x_2": "x[S700_1691]",
   "x_3": "x[S700_1938]",
   "x_4": "x[S700_2047]",
   "x_5": "x[S700_2466]",
   "x_6": "x[S700_2610]",
   "x_7": "x[S700_2824]",
   "x_8": "x[S700_2834]",
   "x_9": "x[S700_3167]",
   "x_10": "x[S700_3505]",
   "x_11": "x[S700_3962]",
   "x_12": "x[S700_4002]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1264095.5000000002,
   "variables": [
    [
     "C0",
     1340.0
    ],
    [
     "C1",
     1127.0
    ],
    [
     "C2",
     1176.0
    ],
    [
     "C3",
     1219.0
    ],
    [
     "C4",
     1158.0
    ],
    [
     "C5",
     1357.0
    ],
    [
     "C6",
     1287.0
    ],
    [
     "C7",
     1392.0
    ],
    [
     "C8",
     1135.0
    ],
    [
     "C9",
     1301.0
    ],
    [
     "C10",
     1281.0
    ],
    [
     "C11",
     1129.0
    ]
   ],
   "error": null
  }
 },
 "OR-027": {
  "source_code_sha256": "0ecbb511a9422a38ffaf43ed852daccfab91da790f8fa71ecbfa8c543acb1518",
  "original_solution_sha256": "9cf54b852201e2ca5dac95aea4adaec4ed43716172a20c3009ef0e41c5fc330c",
  "bindings": [
   [
    "C0",
    "x[Organic Fruits]"
   ],
   [
    "C1",
    "x[Organic Staples]"
   ],
   [
    "C2",
    "x[Organic Vegetables]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Organic Fruits]",
   "x_2": "x[Organic Staples]",
   "x_3": "x[Organic Vegetables]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 784297054.11,
   "variables": [
    [
     "C0",
     678906.0
    ],
    [
     "C1",
     749927.0
    ],
    [
     "C2",
     699808.0
    ]
   ],
   "error": null
  }
 },
 "OR-031": {
  "source_code_sha256": "90591edf6a4657d72e279dc8573b0babbefd2fd532eab8359fd29565cc2c87c1",
  "original_solution_sha256": "98fc3d7d6808b6c9bf727f009c193667a9a3ef014a0cc0c1fad4805e22030fb5",
  "bindings": [
   [
    "C0",
    "x[Butter_Amul]"
   ],
   [
    "C1",
    "x[Butter_Mother Dairy]"
   ],
   [
    "C2",
    "x[Butter_Parag Milk Foods]"
   ],
   [
    "C3",
    "x[Butter_Warana]"
   ],
   [
    "C4",
    "x[Buttermilk_Amul]"
   ],
   [
    "C5",
    "x[Buttermilk_Mother Dairy]"
   ],
   [
    "C6",
    "x[Buttermilk_Raj]"
   ],
   [
    "C7",
    "x[Buttermilk_Sudha]"
   ],
   [
    "C8",
    "x[Cheese_Amul]"
   ],
   [
    "C9",
    "x[Cheese_Britannia Industries]"
   ],
   [
    "C10",
    "x[Cheese_Dynamix Dairies]"
   ],
   [
    "C11",
    "x[Cheese_Passion Cheese]"
   ],
   [
    "C12",
    "x[Curd_Amul]"
   ],
   [
    "C13",
    "x[Curd_Mother Dairy]"
   ],
   [
    "C14",
    "x[Curd_Raj]"
   ],
   [
    "C15",
    "x[Curd_Sudha]"
   ],
   [
    "C16",
    "x[Ghee_Amul]"
   ],
   [
    "C17",
    "x[Ghee_Mother Dairy]"
   ],
   [
    "C18",
    "x[Ghee_Raj]"
   ],
   [
    "C19",
    "x[Ghee_Sudha]"
   ],
   [
    "C20",
    "x[Ice Cream_Amul]"
   ],
   [
    "C21",
    "x[Ice Cream_Dodla Dairy]"
   ],
   [
    "C22",
    "x[Ice Cream_Mother Dairy]"
   ],
   [
    "C23",
    "x[Ice Cream_Palle2patnam]"
   ],
   [
    "C24",
    "x[Lassi_Amul]"
   ],
   [
    "C25",
    "x[Lassi_Mother Dairy]"
   ],
   [
    "C26",
    "x[Lassi_Raj]"
   ],
   [
    "C27",
    "x[Lassi_Sudha]"
   ],
   [
    "C28",
    "x[Milk_Amul]"
   ],
   [
    "C29",
    "x[Milk_Mother Dairy]"
   ],
   [
    "C30",
    "x[Milk_Raj]"
   ],
   [
    "C31",
    "x[Milk_Sudha]"
   ],
   [
    "C32",
    "x[Paneer_Amul]"
   ],
   [
    "C33",
    "x[Paneer_Mother Dairy]"
   ],
   [
    "C34",
    "x[Paneer_Raj]"
   ],
   [
    "C35",
    "x[Paneer_Sudha]"
   ],
   [
    "C36",
    "x[Yogurt_Amul]"
   ],
   [
    "C37",
    "x[Yogurt_Dodla Dairy]"
   ],
   [
    "C38",
    "x[Yogurt_Mother Dairy]"
   ],
   [
    "C39",
    "x[Yogurt_Palle2patnam]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Butter_Amul]",
   "x_2": "x[Butter_Mother Dairy]",
   "x_3": "x[Butter_Parag Milk Foods]",
   "x_4": "x[Butter_Warana]",
   "x_5": "x[Buttermilk_Amul]",
   "x_6": "x[Buttermilk_Mother Dairy]",
   "x_7": "x[Buttermilk_Raj]",
   "x_8": "x[Buttermilk_Sudha]",
   "x_9": "x[Cheese_Amul]",
   "x_10": "x[Cheese_Britannia Industries]",
   "x_11": "x[Cheese_Dynamix Dairies]",
   "x_12": "x[Cheese_Passion Cheese]",
   "x_13": "x[Curd_Amul]",
   "x_14": "x[Curd_Mother Dairy]",
   "x_15": "x[Curd_Raj]",
   "x_16": "x[Curd_Sudha]",
   "x_17": "x[Ghee_Amul]",
   "x_18": "x[Ghee_Mother Dairy]",
   "x_19": "x[Ghee_Raj]",
   "x_20": "x[Ghee_Sudha]",
   "x_21": "x[Ice Cream_Amul]",
   "x_22": "x[Ice Cream_Dodla Dairy]",
   "x_23": "x[Ice Cream_Mother Dairy]",
   "x_24": "x[Ice Cream_Palle2patnam]",
   "x_25": "x[Lassi_Amul]",
   "x_26": "x[Lassi_Mother Dairy]",
   "x_27": "x[Lassi_Raj]",
   "x_28": "x[Lassi_Sudha]",
   "x_29": "x[Milk_Amul]",
   "x_30": "x[Milk_Mother Dairy]",
   "x_31": "x[Milk_Raj]",
   "x_32": "x[Milk_Sudha]",
   "x_33": "x[Paneer_Amul]",
   "x_34": "x[Paneer_Mother Dairy]",
   "x_35": "x[Paneer_Raj]",
   "x_36": "x[Paneer_Sudha]",
   "x_37": "x[Yogurt_Amul]",
   "x_38": "x[Yogurt_Dodla Dairy]",
   "x_39": "x[Yogurt_Mother Dairy]",
   "x_40": "x[Yogurt_Palle2patnam]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 67016511.28,
   "variables": [
    [
     "C0",
     29862.0
    ],
    [
     "C1",
     29898.0
    ],
    [
     "C2",
     25208.0
    ],
    [
     "C3",
     30816.0
    ],
    [
     "C4",
     19925.0
    ],
    [
     "C5",
     26482.0
    ],
    [
     "C6",
     30865.0
    ],
    [
     "C7",
     29649.0
    ],
    [
     "C8",
     30929.0
    ],
    [
     "C9",
     21405.0
    ],
    [
     "C10",
     25953.0
    ],
    [
     "C11",
     23825.0
    ],
    [
     "C12",
     31687.0
    ],
    [
     "C13",
     33377.0
    ],
    [
     "C14",
     34914.0
    ],
    [
     "C15",
     33547.0
    ],
    [
     "C16",
     23120.0
    ],
    [
     "C17",
     24667.0
    ],
    [
     "C18",
     25395.0
    ],
    [
     "C19",
     24676.0
    ],
    [
     "C20",
     26707.0
    ],
    [
     "C21",
     26722.0
    ],
    [
     "C22",
     25809.0
    ],
    [
     "C23",
     24391.0
    ],
    [
     "C24",
     30728.0
    ],
    [
     "C25",
     28628.0
    ],
    [
     "C26",
     30568.0
    ],
    [
     "C27",
     23461.0
    ],
    [
     "C28",
     21398.0
    ],
    [
     "C29",
     33619.0
    ],
    [
     "C30",
     26355.0
    ],
    [
     "C31",
     23815.0
    ],
    [
     "C32",
     20787.0
    ],
    [
     "C33",
     29342.0
    ],
    [
     "C34",
     23556.0
    ],
    [
     "C35",
     28753.0
    ],
    [
     "C36",
     24404.0
    ],
    [
     "C37",
     26829.0
    ],
    [
     "C38",
     25562.0
    ],
    [
     "C39",
     28695.0
    ]
   ],
   "error": null
  }
 },
 "OR-035": {
  "source_code_sha256": "e17e55027e71ded5df1df2f3cdb7568155c15254911bf198e4d3af012488ed0d",
  "original_solution_sha256": "b077b5e16efbc30347743a558640144bcbe4a1446d5008e3425b98b9088bd16d",
  "bindings": [
   [
    "C0",
    "x[Baguette]"
   ],
   [
    "C1",
    "x[Croissant]"
   ],
   [
    "C2",
    "x[Sourdough]"
   ],
   [
    "C3",
    "x[Rye Bread]"
   ],
   [
    "C4",
    "x[Brioche]"
   ],
   [
    "C5",
    "x[Focaccia]"
   ],
   [
    "C6",
    "x[Ciabatta]"
   ],
   [
    "C7",
    "x[Pita]"
   ],
   [
    "C8",
    "x[Bagel]"
   ],
   [
    "C9",
    "x[English Muffin]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Baguette]",
   "x_1_0": "x[Croissant]",
   "x_2_0": "x[Sourdough]",
   "x_3_0": "x[Rye Bread]",
   "x_4_0": "x[Brioche]",
   "x_5_0": "x[Focaccia]",
   "x_6_0": "x[Ciabatta]",
   "x_7_0": "x[Pita]",
   "x_8_0": "x[Bagel]",
   "x_9_0": "x[English Muffin]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 150660.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     180.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-039": {
  "source_code_sha256": "adff275fdf47daf83f7c78afab36f0fd6a0819d62baa8a74e6b33a479fd343bd",
  "original_solution_sha256": "2c3b16c7c15bb291e2d1947e015a04a3ad7555fd579ee2bff5f046c27b39fc81",
  "bindings": [
   [
    "C0",
    "x[Sedans,Warehouse 1]"
   ],
   [
    "C1",
    "x[Sedans,Warehouse 2]"
   ],
   [
    "C2",
    "x[Sedans,Warehouse 3]"
   ],
   [
    "C3",
    "x[Sedans,Warehouse 4]"
   ],
   [
    "C4",
    "x[Sedans,Warehouse 5]"
   ],
   [
    "C5",
    "x[Sedans,Warehouse 6]"
   ],
   [
    "C6",
    "x[Sedans,Warehouse 7]"
   ],
   [
    "C7",
    "x[Sedans,Warehouse 8]"
   ],
   [
    "C8",
    "x[Sedans,Warehouse 9]"
   ],
   [
    "C9",
    "x[Sedans,Warehouse 10]"
   ],
   [
    "C10",
    "x[SUVs,Warehouse 1]"
   ],
   [
    "C11",
    "x[SUVs,Warehouse 2]"
   ],
   [
    "C12",
    "x[SUVs,Warehouse 3]"
   ],
   [
    "C13",
    "x[SUVs,Warehouse 4]"
   ],
   [
    "C14",
    "x[SUVs,Warehouse 5]"
   ],
   [
    "C15",
    "x[SUVs,Warehouse 6]"
   ],
   [
    "C16",
    "x[SUVs,Warehouse 7]"
   ],
   [
    "C17",
    "x[SUVs,Warehouse 8]"
   ],
   [
    "C18",
    "x[SUVs,Warehouse 9]"
   ],
   [
    "C19",
    "x[SUVs,Warehouse 10]"
   ],
   [
    "C20",
    "x[Electric Vehicles,Warehouse 1]"
   ],
   [
    "C21",
    "x[Electric Vehicles,Warehouse 2]"
   ],
   [
    "C22",
    "x[Electric Vehicles,Warehouse 3]"
   ],
   [
    "C23",
    "x[Electric Vehicles,Warehouse 4]"
   ],
   [
    "C24",
    "x[Electric Vehicles,Warehouse 5]"
   ],
   [
    "C25",
    "x[Electric Vehicles,Warehouse 6]"
   ],
   [
    "C26",
    "x[Electric Vehicles,Warehouse 7]"
   ],
   [
    "C27",
    "x[Electric Vehicles,Warehouse 8]"
   ],
   [
    "C28",
    "x[Electric Vehicles,Warehouse 9]"
   ],
   [
    "C29",
    "x[Electric Vehicles,Warehouse 10]"
   ],
   [
    "C30",
    "x[Hybrid Vehicles,Warehouse 1]"
   ],
   [
    "C31",
    "x[Hybrid Vehicles,Warehouse 2]"
   ],
   [
    "C32",
    "x[Hybrid Vehicles,Warehouse 3]"
   ],
   [
    "C33",
    "x[Hybrid Vehicles,Warehouse 4]"
   ],
   [
    "C34",
    "x[Hybrid Vehicles,Warehouse 5]"
   ],
   [
    "C35",
    "x[Hybrid Vehicles,Warehouse 6]"
   ],
   [
    "C36",
    "x[Hybrid Vehicles,Warehouse 7]"
   ],
   [
    "C37",
    "x[Hybrid Vehicles,Warehouse 8]"
   ],
   [
    "C38",
    "x[Hybrid Vehicles,Warehouse 9]"
   ],
   [
    "C39",
    "x[Hybrid Vehicles,Warehouse 10]"
   ],
   [
    "C40",
    "x[Trucks,Warehouse 1]"
   ],
   [
    "C41",
    "x[Trucks,Warehouse 2]"
   ],
   [
    "C42",
    "x[Trucks,Warehouse 3]"
   ],
   [
    "C43",
    "x[Trucks,Warehouse 4]"
   ],
   [
    "C44",
    "x[Trucks,Warehouse 5]"
   ],
   [
    "C45",
    "x[Trucks,Warehouse 6]"
   ],
   [
    "C46",
    "x[Trucks,Warehouse 7]"
   ],
   [
    "C47",
    "x[Trucks,Warehouse 8]"
   ],
   [
    "C48",
    "x[Trucks,Warehouse 9]"
   ],
   [
    "C49",
    "x[Trucks,Warehouse 10]"
   ],
   [
    "C50",
    "x[Sports Cars,Warehouse 1]"
   ],
   [
    "C51",
    "x[Sports Cars,Warehouse 2]"
   ],
   [
    "C52",
    "x[Sports Cars,Warehouse 3]"
   ],
   [
    "C53",
    "x[Sports Cars,Warehouse 4]"
   ],
   [
    "C54",
    "x[Sports Cars,Warehouse 5]"
   ],
   [
    "C55",
    "x[Sports Cars,Warehouse 6]"
   ],
   [
    "C56",
    "x[Sports Cars,Warehouse 7]"
   ],
   [
    "C57",
    "x[Sports Cars,Warehouse 8]"
   ],
   [
    "C58",
    "x[Sports Cars,Warehouse 9]"
   ],
   [
    "C59",
    "x[Sports Cars,Warehouse 10]"
   ],
   [
    "C60",
    "x[Compact Cars,Warehouse 1]"
   ],
   [
    "C61",
    "x[Compact Cars,Warehouse 2]"
   ],
   [
    "C62",
    "x[Compact Cars,Warehouse 3]"
   ],
   [
    "C63",
    "x[Compact Cars,Warehouse 4]"
   ],
   [
    "C64",
    "x[Compact Cars,Warehouse 5]"
   ],
   [
    "C65",
    "x[Compact Cars,Warehouse 6]"
   ],
   [
    "C66",
    "x[Compact Cars,Warehouse 7]"
   ],
   [
    "C67",
    "x[Compact Cars,Warehouse 8]"
   ],
   [
    "C68",
    "x[Compact Cars,Warehouse 9]"
   ],
   [
    "C69",
    "x[Compact Cars,Warehouse 10]"
   ],
   [
    "C70",
    "x[Luxury Sedans,Warehouse 1]"
   ],
   [
    "C71",
    "x[Luxury Sedans,Warehouse 2]"
   ],
   [
    "C72",
    "x[Luxury Sedans,Warehouse 3]"
   ],
   [
    "C73",
    "x[Luxury Sedans,Warehouse 4]"
   ],
   [
    "C74",
    "x[Luxury Sedans,Warehouse 5]"
   ],
   [
    "C75",
    "x[Luxury Sedans,Warehouse 6]"
   ],
   [
    "C76",
    "x[Luxury Sedans,Warehouse 7]"
   ],
   [
    "C77",
    "x[Luxury Sedans,Warehouse 8]"
   ],
   [
    "C78",
    "x[Luxury Sedans,Warehouse 9]"
   ],
   [
    "C79",
    "x[Luxury Sedans,Warehouse 10]"
   ],
   [
    "C80",
    "x[Vans,Warehouse 1]"
   ],
   [
    "C81",
    "x[Vans,Warehouse 2]"
   ],
   [
    "C82",
    "x[Vans,Warehouse 3]"
   ],
   [
    "C83",
    "x[Vans,Warehouse 4]"
   ],
   [
    "C84",
    "x[Vans,Warehouse 5]"
   ],
   [
    "C85",
    "x[Vans,Warehouse 6]"
   ],
   [
    "C86",
    "x[Vans,Warehouse 7]"
   ],
   [
    "C87",
    "x[Vans,Warehouse 8]"
   ],
   [
    "C88",
    "x[Vans,Warehouse 9]"
   ],
   [
    "C89",
    "x[Vans,Warehouse 10]"
   ],
   [
    "C90",
    "x[Pickup Trucks,Warehouse 1]"
   ],
   [
    "C91",
    "x[Pickup Trucks,Warehouse 2]"
   ],
   [
    "C92",
    "x[Pickup Trucks,Warehouse 3]"
   ],
   [
    "C93",
    "x[Pickup Trucks,Warehouse 4]"
   ],
   [
    "C94",
    "x[Pickup Trucks,Warehouse 5]"
   ],
   [
    "C95",
    "x[Pickup Trucks,Warehouse 6]"
   ],
   [
    "C96",
    "x[Pickup Trucks,Warehouse 7]"
   ],
   [
    "C97",
    "x[Pickup Trucks,Warehouse 8]"
   ],
   [
    "C98",
    "x[Pickup Trucks,Warehouse 9]"
   ],
   [
    "C99",
    "x[Pickup Trucks,Warehouse 10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Sedans,Warehouse 1]",
   "x_0_1": "x[Sedans,Warehouse 2]",
   "x_0_2": "x[Sedans,Warehouse 3]",
   "x_0_3": "x[Sedans,Warehouse 4]",
   "x_0_4": "x[Sedans,Warehouse 5]",
   "x_0_5": "x[Sedans,Warehouse 6]",
   "x_0_6": "x[Sedans,Warehouse 7]",
   "x_0_7": "x[Sedans,Warehouse 8]",
   "x_0_8": "x[Sedans,Warehouse 9]",
   "x_0_9": "x[Sedans,Warehouse 10]",
   "x_1_0": "x[SUVs,Warehouse 1]",
   "x_1_1": "x[SUVs,Warehouse 2]",
   "x_1_2": "x[SUVs,Warehouse 3]",
   "x_1_3": "x[SUVs,Warehouse 4]",
   "x_1_4": "x[SUVs,Warehouse 5]",
   "x_1_5": "x[SUVs,Warehouse 6]",
   "x_1_6": "x[SUVs,Warehouse 7]",
   "x_1_7": "x[SUVs,Warehouse 8]",
   "x_1_8": "x[SUVs,Warehouse 9]",
   "x_1_9": "x[SUVs,Warehouse 10]",
   "x_2_0": "x[Electric Vehicles,Warehouse 1]",
   "x_2_1": "x[Electric Vehicles,Warehouse 2]",
   "x_2_2": "x[Electric Vehicles,Warehouse 3]",
   "x_2_3": "x[Electric Vehicles,Warehouse 4]",
   "x_2_4": "x[Electric Vehicles,Warehouse 5]",
   "x_2_5": "x[Electric Vehicles,Warehouse 6]",
   "x_2_6": "x[Electric Vehicles,Warehouse 7]",
   "x_2_7": "x[Electric Vehicles,Warehouse 8]",
   "x_2_8": "x[Electric Vehicles,Warehouse 9]",
   "x_2_9": "x[Electric Vehicles,Warehouse 10]",
   "x_3_0": "x[Hybrid Vehicles,Warehouse 1]",
   "x_3_1": "x[Hybrid Vehicles,Warehouse 2]",
   "x_3_2": "x[Hybrid Vehicles,Warehouse 3]",
   "x_3_3": "x[Hybrid Vehicles,Warehouse 4]",
   "x_3_4": "x[Hybrid Vehicles,Warehouse 5]",
   "x_3_5": "x[Hybrid Vehicles,Warehouse 6]",
   "x_3_6": "x[Hybrid Vehicles,Warehouse 7]",
   "x_3_7": "x[Hybrid Vehicles,Warehouse 8]",
   "x_3_8": "x[Hybrid Vehicles,Warehouse 9]",
   "x_3_9": "x[Hybrid Vehicles,Warehouse 10]",
   "x_4_0": "x[Trucks,Warehouse 1]",
   "x_4_1": "x[Trucks,Warehouse 2]",
   "x_4_2": "x[Trucks,Warehouse 3]",
   "x_4_3": "x[Trucks,Warehouse 4]",
   "x_4_4": "x[Trucks,Warehouse 5]",
   "x_4_5": "x[Trucks,Warehouse 6]",
   "x_4_6": "x[Trucks,Warehouse 7]",
   "x_4_7": "x[Trucks,Warehouse 8]",
   "x_4_8": "x[Trucks,Warehouse 9]",
   "x_4_9": "x[Trucks,Warehouse 10]",
   "x_5_0": "x[Sports Cars,Warehouse 1]",
   "x_5_1": "x[Sports Cars,Warehouse 2]",
   "x_5_2": "x[Sports Cars,Warehouse 3]",
   "x_5_3": "x[Sports Cars,Warehouse 4]",
   "x_5_4": "x[Sports Cars,Warehouse 5]",
   "x_5_5": "x[Sports Cars,Warehouse 6]",
   "x_5_6": "x[Sports Cars,Warehouse 7]",
   "x_5_7": "x[Sports Cars,Warehouse 8]",
   "x_5_8": "x[Sports Cars,Warehouse 9]",
   "x_5_9": "x[Sports Cars,Warehouse 10]",
   "x_6_0": "x[Compact Cars,Warehouse 1]",
   "x_6_1": "x[Compact Cars,Warehouse 2]",
   "x_6_2": "x[Compact Cars,Warehouse 3]",
   "x_6_3": "x[Compact Cars,Warehouse 4]",
   "x_6_4": "x[Compact Cars,Warehouse 5]",
   "x_6_5": "x[Compact Cars,Warehouse 6]",
   "x_6_6": "x[Compact Cars,Warehouse 7]",
   "x_6_7": "x[Compact Cars,Warehouse 8]",
   "x_6_8": "x[Compact Cars,Warehouse 9]",
   "x_6_9": "x[Compact Cars,Warehouse 10]",
   "x_7_0": "x[Luxury Sedans,Warehouse 1]",
   "x_7_1": "x[Luxury Sedans,Warehouse 2]",
   "x_7_2": "x[Luxury Sedans,Warehouse 3]",
   "x_7_3": "x[Luxury Sedans,Warehouse 4]",
   "x_7_4": "x[Luxury Sedans,Warehouse 5]",
   "x_7_5": "x[Luxury Sedans,Warehouse 6]",
   "x_7_6": "x[Luxury Sedans,Warehouse 7]",
   "x_7_7": "x[Luxury Sedans,Warehouse 8]",
   "x_7_8": "x[Luxury Sedans,Warehouse 9]",
   "x_7_9": "x[Luxury Sedans,Warehouse 10]",
   "x_8_0": "x[Vans,Warehouse 1]",
   "x_8_1": "x[Vans,Warehouse 2]",
   "x_8_2": "x[Vans,Warehouse 3]",
   "x_8_3": "x[Vans,Warehouse 4]",
   "x_8_4": "x[Vans,Warehouse 5]",
   "x_8_5": "x[Vans,Warehouse 6]",
   "x_8_6": "x[Vans,Warehouse 7]",
   "x_8_7": "x[Vans,Warehouse 8]",
   "x_8_8": "x[Vans,Warehouse 9]",
   "x_8_9": "x[Vans,Warehouse 10]",
   "x_9_0": "x[Pickup Trucks,Warehouse 1]",
   "x_9_1": "x[Pickup Trucks,Warehouse 2]",
   "x_9_2": "x[Pickup Trucks,Warehouse 3]",
   "x_9_3": "x[Pickup Trucks,Warehouse 4]",
   "x_9_4": "x[Pickup Trucks,Warehouse 5]",
   "x_9_5": "x[Pickup Trucks,Warehouse 6]",
   "x_9_6": "x[Pickup Trucks,Warehouse 7]",
   "x_9_7": "x[Pickup Trucks,Warehouse 8]",
   "x_9_8": "x[Pickup Trucks,Warehouse 9]",
   "x_9_9": "x[Pickup Trucks,Warehouse 10]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 429000.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     20.0
    ],
    [
     "C51",
     16.0
    ],
    [
     "C52",
     24.0
    ],
    [
     "C53",
     18.0
    ],
    [
     "C54",
     10.0
    ],
    [
     "C55",
     6.0
    ],
    [
     "C56",
     22.0
    ],
    [
     "C57",
     8.0
    ],
    [
     "C58",
     12.0
    ],
    [
     "C59",
     7.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-043": {
  "source_code_sha256": "a0497f3c31cc6e6b1823ced86369a46bbf5668ab8d0c42efa89f3b6c0d5385b8",
  "original_solution_sha256": "8c88ac0a4b8930b744f571f5e2f1727cc706e816f62bc63416286f852c99225d",
  "bindings": [
   [
    "C0",
    "x[NSAIDs]"
   ],
   [
    "C1",
    "x[Antirheumatic Drugs]"
   ],
   [
    "C2",
    "x[Acetic Acid Derivatives]"
   ],
   [
    "C3",
    "x[Antibiotics]"
   ],
   [
    "C4",
    "x[Antiviral Drugs]"
   ],
   [
    "C5",
    "x[Antifungal Agents]"
   ],
   [
    "C6",
    "x[Antidepressants]"
   ],
   [
    "C7",
    "x[Antipsychotics]"
   ],
   [
    "C8",
    "x[Antihistamines]"
   ],
   [
    "C9",
    "x[Corticosteroids]"
   ],
   [
    "C10",
    "x[Beta Blockers]"
   ],
   [
    "C11",
    "x[Calcium Channel Blockers]"
   ],
   [
    "C12",
    "x[ACE Inhibitors]"
   ],
   [
    "C13",
    "x[Angiotensin II Receptor Blockers]"
   ],
   [
    "C14",
    "x[Diuretics]"
   ],
   [
    "C15",
    "x[Statins]"
   ],
   [
    "C16",
    "x[Insulin]"
   ],
   [
    "C17",
    "x[Anticoagulants]"
   ],
   [
    "C18",
    "x[Antiepileptic Drugs]"
   ],
   [
    "C19",
    "x[Antiemetics]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[NSAIDs]",
   "x_1_0": "x[Antirheumatic Drugs]",
   "x_2_0": "x[Acetic Acid Derivatives]",
   "x_3_0": "x[Antibiotics]",
   "x_4_0": "x[Antiviral Drugs]",
   "x_5_0": "x[Antifungal Agents]",
   "x_6_0": "x[Antidepressants]",
   "x_7_0": "x[Antipsychotics]",
   "x_8_0": "x[Antihistamines]",
   "x_9_0": "x[Corticosteroids]",
   "x_10_0": "x[Beta Blockers]",
   "x_11_0": "x[Calcium Channel Blockers]",
   "x_12_0": "x[ACE Inhibitors]",
   "x_13_0": "x[Angiotensin II Receptor Blockers]",
   "x_14_0": "x[Diuretics]",
   "x_15_0": "x[Statins]",
   "x_16_0": "x[Insulin]",
   "x_17_0": "x[Anticoagulants]",
   "x_18_0": "x[Antiepileptic Drugs]",
   "x_19_0": "x[Antiemetics]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 13570.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     10.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-047": {
  "source_code_sha256": "70c10e85cb2c49c242c23261967e028ac808f298c0b2247c1d7fbe46b3aa4581",
  "original_solution_sha256": "4c6caf5504ac4b216a3bf536e68d9bd7090d83099c026319d8b816b8ef107d34",
  "bindings": [
   [
    "C0",
    "x[1,Racing]"
   ],
   [
    "C1",
    "x[1,Sports]"
   ],
   [
    "C2",
    "x[1,Action]"
   ],
   [
    "C3",
    "x[1,Adventure]"
   ],
   [
    "C4",
    "x[1,RPG]"
   ],
   [
    "C5",
    "x[1,Shooter]"
   ],
   [
    "C6",
    "x[1,Strategy]"
   ],
   [
    "C7",
    "x[1,Simulation]"
   ],
   [
    "C8",
    "x[1,Puzzle]"
   ],
   [
    "C9",
    "x[1,Fighting]"
   ],
   [
    "C10",
    "x[1,Platformer]"
   ],
   [
    "C11",
    "x[1,Survival]"
   ],
   [
    "C12",
    "x[1,Horror]"
   ],
   [
    "C13",
    "x[1,Sandbox]"
   ],
   [
    "C14",
    "x[1,MMO]"
   ],
   [
    "C15",
    "x[2,Racing]"
   ],
   [
    "C16",
    "x[2,Sports]"
   ],
   [
    "C17",
    "x[2,Action]"
   ],
   [
    "C18",
    "x[2,Adventure]"
   ],
   [
    "C19",
    "x[2,RPG]"
   ],
   [
    "C20",
    "x[2,Shooter]"
   ],
   [
    "C21",
    "x[2,Strategy]"
   ],
   [
    "C22",
    "x[2,Simulation]"
   ],
   [
    "C23",
    "x[2,Puzzle]"
   ],
   [
    "C24",
    "x[2,Fighting]"
   ],
   [
    "C25",
    "x[2,Platformer]"
   ],
   [
    "C26",
    "x[2,Survival]"
   ],
   [
    "C27",
    "x[2,Horror]"
   ],
   [
    "C28",
    "x[2,Sandbox]"
   ],
   [
    "C29",
    "x[2,MMO]"
   ],
   [
    "C30",
    "x[3,Racing]"
   ],
   [
    "C31",
    "x[3,Sports]"
   ],
   [
    "C32",
    "x[3,Action]"
   ],
   [
    "C33",
    "x[3,Adventure]"
   ],
   [
    "C34",
    "x[3,RPG]"
   ],
   [
    "C35",
    "x[3,Shooter]"
   ],
   [
    "C36",
    "x[3,Strategy]"
   ],
   [
    "C37",
    "x[3,Simulation]"
   ],
   [
    "C38",
    "x[3,Puzzle]"
   ],
   [
    "C39",
    "x[3,Fighting]"
   ],
   [
    "C40",
    "x[3,Platformer]"
   ],
   [
    "C41",
    "x[3,Survival]"
   ],
   [
    "C42",
    "x[3,Horror]"
   ],
   [
    "C43",
    "x[3,Sandbox]"
   ],
   [
    "C44",
    "x[3,MMO]"
   ],
   [
    "C45",
    "x[4,Racing]"
   ],
   [
    "C46",
    "x[4,Sports]"
   ],
   [
    "C47",
    "x[4,Action]"
   ],
   [
    "C48",
    "x[4,Adventure]"
   ],
   [
    "C49",
    "x[4,RPG]"
   ],
   [
    "C50",
    "x[4,Shooter]"
   ],
   [
    "C51",
    "x[4,Strategy]"
   ],
   [
    "C52",
    "x[4,Simulation]"
   ],
   [
    "C53",
    "x[4,Puzzle]"
   ],
   [
    "C54",
    "x[4,Fighting]"
   ],
   [
    "C55",
    "x[4,Platformer]"
   ],
   [
    "C56",
    "x[4,Survival]"
   ],
   [
    "C57",
    "x[4,Horror]"
   ],
   [
    "C58",
    "x[4,Sandbox]"
   ],
   [
    "C59",
    "x[4,MMO]"
   ],
   [
    "C60",
    "x[5,Racing]"
   ],
   [
    "C61",
    "x[5,Sports]"
   ],
   [
    "C62",
    "x[5,Action]"
   ],
   [
    "C63",
    "x[5,Adventure]"
   ],
   [
    "C64",
    "x[5,RPG]"
   ],
   [
    "C65",
    "x[5,Shooter]"
   ],
   [
    "C66",
    "x[5,Strategy]"
   ],
   [
    "C67",
    "x[5,Simulation]"
   ],
   [
    "C68",
    "x[5,Puzzle]"
   ],
   [
    "C69",
    "x[5,Fighting]"
   ],
   [
    "C70",
    "x[5,Platformer]"
   ],
   [
    "C71",
    "x[5,Survival]"
   ],
   [
    "C72",
    "x[5,Horror]"
   ],
   [
    "C73",
    "x[5,Sandbox]"
   ],
   [
    "C74",
    "x[5,MMO]"
   ],
   [
    "C75",
    "x[6,Racing]"
   ],
   [
    "C76",
    "x[6,Sports]"
   ],
   [
    "C77",
    "x[6,Action]"
   ],
   [
    "C78",
    "x[6,Adventure]"
   ],
   [
    "C79",
    "x[6,RPG]"
   ],
   [
    "C80",
    "x[6,Shooter]"
   ],
   [
    "C81",
    "x[6,Strategy]"
   ],
   [
    "C82",
    "x[6,Simulation]"
   ],
   [
    "C83",
    "x[6,Puzzle]"
   ],
   [
    "C84",
    "x[6,Fighting]"
   ],
   [
    "C85",
    "x[6,Platformer]"
   ],
   [
    "C86",
    "x[6,Survival]"
   ],
   [
    "C87",
    "x[6,Horror]"
   ],
   [
    "C88",
    "x[6,Sandbox]"
   ],
   [
    "C89",
    "x[6,MMO]"
   ],
   [
    "C90",
    "x[7,Racing]"
   ],
   [
    "C91",
    "x[7,Sports]"
   ],
   [
    "C92",
    "x[7,Action]"
   ],
   [
    "C93",
    "x[7,Adventure]"
   ],
   [
    "C94",
    "x[7,RPG]"
   ],
   [
    "C95",
    "x[7,Shooter]"
   ],
   [
    "C96",
    "x[7,Strategy]"
   ],
   [
    "C97",
    "x[7,Simulation]"
   ],
   [
    "C98",
    "x[7,Puzzle]"
   ],
   [
    "C99",
    "x[7,Fighting]"
   ],
   [
    "C100",
    "x[7,Platformer]"
   ],
   [
    "C101",
    "x[7,Survival]"
   ],
   [
    "C102",
    "x[7,Horror]"
   ],
   [
    "C103",
    "x[7,Sandbox]"
   ],
   [
    "C104",
    "x[7,MMO]"
   ],
   [
    "C105",
    "x[8,Racing]"
   ],
   [
    "C106",
    "x[8,Sports]"
   ],
   [
    "C107",
    "x[8,Action]"
   ],
   [
    "C108",
    "x[8,Adventure]"
   ],
   [
    "C109",
    "x[8,RPG]"
   ],
   [
    "C110",
    "x[8,Shooter]"
   ],
   [
    "C111",
    "x[8,Strategy]"
   ],
   [
    "C112",
    "x[8,Simulation]"
   ],
   [
    "C113",
    "x[8,Puzzle]"
   ],
   [
    "C114",
    "x[8,Fighting]"
   ],
   [
    "C115",
    "x[8,Platformer]"
   ],
   [
    "C116",
    "x[8,Survival]"
   ],
   [
    "C117",
    "x[8,Horror]"
   ],
   [
    "C118",
    "x[8,Sandbox]"
   ],
   [
    "C119",
    "x[8,MMO]"
   ],
   [
    "C120",
    "x[9,Racing]"
   ],
   [
    "C121",
    "x[9,Sports]"
   ],
   [
    "C122",
    "x[9,Action]"
   ],
   [
    "C123",
    "x[9,Adventure]"
   ],
   [
    "C124",
    "x[9,RPG]"
   ],
   [
    "C125",
    "x[9,Shooter]"
   ],
   [
    "C126",
    "x[9,Strategy]"
   ],
   [
    "C127",
    "x[9,Simulation]"
   ],
   [
    "C128",
    "x[9,Puzzle]"
   ],
   [
    "C129",
    "x[9,Fighting]"
   ],
   [
    "C130",
    "x[9,Platformer]"
   ],
   [
    "C131",
    "x[9,Survival]"
   ],
   [
    "C132",
    "x[9,Horror]"
   ],
   [
    "C133",
    "x[9,Sandbox]"
   ],
   [
    "C134",
    "x[9,MMO]"
   ],
   [
    "C135",
    "x[10,Racing]"
   ],
   [
    "C136",
    "x[10,Sports]"
   ],
   [
    "C137",
    "x[10,Action]"
   ],
   [
    "C138",
    "x[10,Adventure]"
   ],
   [
    "C139",
    "x[10,RPG]"
   ],
   [
    "C140",
    "x[10,Shooter]"
   ],
   [
    "C141",
    "x[10,Strategy]"
   ],
   [
    "C142",
    "x[10,Simulation]"
   ],
   [
    "C143",
    "x[10,Puzzle]"
   ],
   [
    "C144",
    "x[10,Fighting]"
   ],
   [
    "C145",
    "x[10,Platformer]"
   ],
   [
    "C146",
    "x[10,Survival]"
   ],
   [
    "C147",
    "x[10,Horror]"
   ],
   [
    "C148",
    "x[10,Sandbox]"
   ],
   [
    "C149",
    "x[10,MMO]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Racing]",
   "x_0_1": "x[2,Racing]",
   "x_0_2": "x[3,Racing]",
   "x_0_3": "x[4,Racing]",
   "x_0_4": "x[5,Racing]",
   "x_0_5": "x[6,Racing]",
   "x_0_6": "x[7,Racing]",
   "x_0_7": "x[8,Racing]",
   "x_0_8": "x[9,Racing]",
   "x_0_9": "x[10,Racing]",
   "x_1_0": "x[1,Sports]",
   "x_1_1": "x[2,Sports]",
   "x_1_2": "x[3,Sports]",
   "x_1_3": "x[4,Sports]",
   "x_1_4": "x[5,Sports]",
   "x_1_5": "x[6,Sports]",
   "x_1_6": "x[7,Sports]",
   "x_1_7": "x[8,Sports]",
   "x_1_8": "x[9,Sports]",
   "x_1_9": "x[10,Sports]",
   "x_2_0": "x[1,Action]",
   "x_2_1": "x[2,Action]",
   "x_2_2": "x[3,Action]",
   "x_2_3": "x[4,Action]",
   "x_2_4": "x[5,Action]",
   "x_2_5": "x[6,Action]",
   "x_2_6": "x[7,Action]",
   "x_2_7": "x[8,Action]",
   "x_2_8": "x[9,Action]",
   "x_2_9": "x[10,Action]",
   "x_3_0": "x[1,Adventure]",
   "x_3_1": "x[2,Adventure]",
   "x_3_2": "x[3,Adventure]",
   "x_3_3": "x[4,Adventure]",
   "x_3_4": "x[5,Adventure]",
   "x_3_5": "x[6,Adventure]",
   "x_3_6": "x[7,Adventure]",
   "x_3_7": "x[8,Adventure]",
   "x_3_8": "x[9,Adventure]",
   "x_3_9": "x[10,Adventure]",
   "x_4_0": "x[1,RPG]",
   "x_4_1": "x[2,RPG]",
   "x_4_2": "x[3,RPG]",
   "x_4_3": "x[4,RPG]",
   "x_4_4": "x[5,RPG]",
   "x_4_5": "x[6,RPG]",
   "x_4_6": "x[7,RPG]",
   "x_4_7": "x[8,RPG]",
   "x_4_8": "x[9,RPG]",
   "x_4_9": "x[10,RPG]",
   "x_5_0": "x[1,Shooter]",
   "x_5_1": "x[2,Shooter]",
   "x_5_2": "x[3,Shooter]",
   "x_5_3": "x[4,Shooter]",
   "x_5_4": "x[5,Shooter]",
   "x_5_5": "x[6,Shooter]",
   "x_5_6": "x[7,Shooter]",
   "x_5_7": "x[8,Shooter]",
   "x_5_8": "x[9,Shooter]",
   "x_5_9": "x[10,Shooter]",
   "x_6_0": "x[1,Strategy]",
   "x_6_1": "x[2,Strategy]",
   "x_6_2": "x[3,Strategy]",
   "x_6_3": "x[4,Strategy]",
   "x_6_4": "x[5,Strategy]",
   "x_6_5": "x[6,Strategy]",
   "x_6_6": "x[7,Strategy]",
   "x_6_7": "x[8,Strategy]",
   "x_6_8": "x[9,Strategy]",
   "x_6_9": "x[10,Strategy]",
   "x_7_0": "x[1,Simulation]",
   "x_7_1": "x[2,Simulation]",
   "x_7_2": "x[3,Simulation]",
   "x_7_3": "x[4,Simulation]",
   "x_7_4": "x[5,Simulation]",
   "x_7_5": "x[6,Simulation]",
   "x_7_6": "x[7,Simulation]",
   "x_7_7": "x[8,Simulation]",
   "x_7_8": "x[9,Simulation]",
   "x_7_9": "x[10,Simulation]",
   "x_8_0": "x[1,Puzzle]",
   "x_8_1": "x[2,Puzzle]",
   "x_8_2": "x[3,Puzzle]",
   "x_8_3": "x[4,Puzzle]",
   "x_8_4": "x[5,Puzzle]",
   "x_8_5": "x[6,Puzzle]",
   "x_8_6": "x[7,Puzzle]",
   "x_8_7": "x[8,Puzzle]",
   "x_8_8": "x[9,Puzzle]",
   "x_8_9": "x[10,Puzzle]",
   "x_9_0": "x[1,Fighting]",
   "x_9_1": "x[2,Fighting]",
   "x_9_2": "x[3,Fighting]",
   "x_9_3": "x[4,Fighting]",
   "x_9_4": "x[5,Fighting]",
   "x_9_5": "x[6,Fighting]",
   "x_9_6": "x[7,Fighting]",
   "x_9_7": "x[8,Fighting]",
   "x_9_8": "x[9,Fighting]",
   "x_9_9": "x[10,Fighting]",
   "x_10_0": "x[1,Platformer]",
   "x_10_1": "x[2,Platformer]",
   "x_10_2": "x[3,Platformer]",
   "x_10_3": "x[4,Platformer]",
   "x_10_4": "x[5,Platformer]",
   "x_10_5": "x[6,Platformer]",
   "x_10_6": "x[7,Platformer]",
   "x_10_7": "x[8,Platformer]",
   "x_10_8": "x[9,Platformer]",
   "x_10_9": "x[10,Platformer]",
   "x_11_0": "x[1,Survival]",
   "x_11_1": "x[2,Survival]",
   "x_11_2": "x[3,Survival]",
   "x_11_3": "x[4,Survival]",
   "x_11_4": "x[5,Survival]",
   "x_11_5": "x[6,Survival]",
   "x_11_6": "x[7,Survival]",
   "x_11_7": "x[8,Survival]",
   "x_11_8": "x[9,Survival]",
   "x_11_9": "x[10,Survival]",
   "x_12_0": "x[1,Horror]",
   "x_12_1": "x[2,Horror]",
   "x_12_2": "x[3,Horror]",
   "x_12_3": "x[4,Horror]",
   "x_12_4": "x[5,Horror]",
   "x_12_5": "x[6,Horror]",
   "x_12_6": "x[7,Horror]",
   "x_12_7": "x[8,Horror]",
   "x_12_8": "x[9,Horror]",
   "x_12_9": "x[10,Horror]",
   "x_13_0": "x[1,Sandbox]",
   "x_13_1": "x[2,Sandbox]",
   "x_13_2": "x[3,Sandbox]",
   "x_13_3": "x[4,Sandbox]",
   "x_13_4": "x[5,Sandbox]",
   "x_13_5": "x[6,Sandbox]",
   "x_13_6": "x[7,Sandbox]",
   "x_13_7": "x[8,Sandbox]",
   "x_13_8": "x[9,Sandbox]",
   "x_13_9": "x[10,Sandbox]",
   "x_14_0": "x[1,MMO]",
   "x_14_1": "x[2,MMO]",
   "x_14_2": "x[3,MMO]",
   "x_14_3": "x[4,MMO]",
   "x_14_4": "x[5,MMO]",
   "x_14_5": "x[6,MMO]",
   "x_14_6": "x[7,MMO]",
   "x_14_7": "x[8,MMO]",
   "x_14_8": "x[9,MMO]",
   "x_14_9": "x[10,MMO]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 5433.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     9.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     12.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     11.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     1.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     6.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     1.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     8.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     4.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     5.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     1.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     9.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     2.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     6.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     2.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     5.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-050": {
  "source_code_sha256": "888fb74fe439f0a97c370543879d9494a2917a07c4787c2e9ba4ca23b7d5d289",
  "original_solution_sha256": "4458cfa2d051a53ebd5570ee89439918e3b04bacb4e394fdbfa4783439747b31",
  "bindings": [
   [
    "C0",
    "x[1,Smartphone]"
   ],
   [
    "C1",
    "x[1,Laptop]"
   ],
   [
    "C2",
    "x[1,Headphones]"
   ],
   [
    "C3",
    "x[1,Camera]"
   ],
   [
    "C4",
    "x[1,Smartwatch]"
   ],
   [
    "C5",
    "x[1,Tablet]"
   ],
   [
    "C6",
    "x[1,Bluetooth Speaker]"
   ],
   [
    "C7",
    "x[1,Keyboard]"
   ],
   [
    "C8",
    "x[1,Mouse]"
   ],
   [
    "C9",
    "x[1,Monitor]"
   ],
   [
    "C10",
    "x[1,Printer]"
   ],
   [
    "C11",
    "x[1,External Hard Drive]"
   ],
   [
    "C12",
    "x[1,Router]"
   ],
   [
    "C13",
    "x[1,Power Bank]"
   ],
   [
    "C14",
    "x[1,Memory Card]"
   ],
   [
    "C15",
    "x[1,USB Flash Drive]"
   ],
   [
    "C16",
    "x[1,Smart Home Hub]"
   ],
   [
    "C17",
    "x[1,Gaming Console]"
   ],
   [
    "C18",
    "x[1,Fitness Tracker]"
   ],
   [
    "C19",
    "x[1,E-Reader]"
   ],
   [
    "C20",
    "x[2,Smartphone]"
   ],
   [
    "C21",
    "x[2,Laptop]"
   ],
   [
    "C22",
    "x[2,Headphones]"
   ],
   [
    "C23",
    "x[2,Camera]"
   ],
   [
    "C24",
    "x[2,Smartwatch]"
   ],
   [
    "C25",
    "x[2,Tablet]"
   ],
   [
    "C26",
    "x[2,Bluetooth Speaker]"
   ],
   [
    "C27",
    "x[2,Keyboard]"
   ],
   [
    "C28",
    "x[2,Mouse]"
   ],
   [
    "C29",
    "x[2,Monitor]"
   ],
   [
    "C30",
    "x[2,Printer]"
   ],
   [
    "C31",
    "x[2,External Hard Drive]"
   ],
   [
    "C32",
    "x[2,Router]"
   ],
   [
    "C33",
    "x[2,Power Bank]"
   ],
   [
    "C34",
    "x[2,Memory Card]"
   ],
   [
    "C35",
    "x[2,USB Flash Drive]"
   ],
   [
    "C36",
    "x[2,Smart Home Hub]"
   ],
   [
    "C37",
    "x[2,Gaming Console]"
   ],
   [
    "C38",
    "x[2,Fitness Tracker]"
   ],
   [
    "C39",
    "x[2,E-Reader]"
   ],
   [
    "C40",
    "x[3,Smartphone]"
   ],
   [
    "C41",
    "x[3,Laptop]"
   ],
   [
    "C42",
    "x[3,Headphones]"
   ],
   [
    "C43",
    "x[3,Camera]"
   ],
   [
    "C44",
    "x[3,Smartwatch]"
   ],
   [
    "C45",
    "x[3,Tablet]"
   ],
   [
    "C46",
    "x[3,Bluetooth Speaker]"
   ],
   [
    "C47",
    "x[3,Keyboard]"
   ],
   [
    "C48",
    "x[3,Mouse]"
   ],
   [
    "C49",
    "x[3,Monitor]"
   ],
   [
    "C50",
    "x[3,Printer]"
   ],
   [
    "C51",
    "x[3,External Hard Drive]"
   ],
   [
    "C52",
    "x[3,Router]"
   ],
   [
    "C53",
    "x[3,Power Bank]"
   ],
   [
    "C54",
    "x[3,Memory Card]"
   ],
   [
    "C55",
    "x[3,USB Flash Drive]"
   ],
   [
    "C56",
    "x[3,Smart Home Hub]"
   ],
   [
    "C57",
    "x[3,Gaming Console]"
   ],
   [
    "C58",
    "x[3,Fitness Tracker]"
   ],
   [
    "C59",
    "x[3,E-Reader]"
   ],
   [
    "C60",
    "x[4,Smartphone]"
   ],
   [
    "C61",
    "x[4,Laptop]"
   ],
   [
    "C62",
    "x[4,Headphones]"
   ],
   [
    "C63",
    "x[4,Camera]"
   ],
   [
    "C64",
    "x[4,Smartwatch]"
   ],
   [
    "C65",
    "x[4,Tablet]"
   ],
   [
    "C66",
    "x[4,Bluetooth Speaker]"
   ],
   [
    "C67",
    "x[4,Keyboard]"
   ],
   [
    "C68",
    "x[4,Mouse]"
   ],
   [
    "C69",
    "x[4,Monitor]"
   ],
   [
    "C70",
    "x[4,Printer]"
   ],
   [
    "C71",
    "x[4,External Hard Drive]"
   ],
   [
    "C72",
    "x[4,Router]"
   ],
   [
    "C73",
    "x[4,Power Bank]"
   ],
   [
    "C74",
    "x[4,Memory Card]"
   ],
   [
    "C75",
    "x[4,USB Flash Drive]"
   ],
   [
    "C76",
    "x[4,Smart Home Hub]"
   ],
   [
    "C77",
    "x[4,Gaming Console]"
   ],
   [
    "C78",
    "x[4,Fitness Tracker]"
   ],
   [
    "C79",
    "x[4,E-Reader]"
   ],
   [
    "C80",
    "x[5,Smartphone]"
   ],
   [
    "C81",
    "x[5,Laptop]"
   ],
   [
    "C82",
    "x[5,Headphones]"
   ],
   [
    "C83",
    "x[5,Camera]"
   ],
   [
    "C84",
    "x[5,Smartwatch]"
   ],
   [
    "C85",
    "x[5,Tablet]"
   ],
   [
    "C86",
    "x[5,Bluetooth Speaker]"
   ],
   [
    "C87",
    "x[5,Keyboard]"
   ],
   [
    "C88",
    "x[5,Mouse]"
   ],
   [
    "C89",
    "x[5,Monitor]"
   ],
   [
    "C90",
    "x[5,Printer]"
   ],
   [
    "C91",
    "x[5,External Hard Drive]"
   ],
   [
    "C92",
    "x[5,Router]"
   ],
   [
    "C93",
    "x[5,Power Bank]"
   ],
   [
    "C94",
    "x[5,Memory Card]"
   ],
   [
    "C95",
    "x[5,USB Flash Drive]"
   ],
   [
    "C96",
    "x[5,Smart Home Hub]"
   ],
   [
    "C97",
    "x[5,Gaming Console]"
   ],
   [
    "C98",
    "x[5,Fitness Tracker]"
   ],
   [
    "C99",
    "x[5,E-Reader]"
   ],
   [
    "C100",
    "x[6,Smartphone]"
   ],
   [
    "C101",
    "x[6,Laptop]"
   ],
   [
    "C102",
    "x[6,Headphones]"
   ],
   [
    "C103",
    "x[6,Camera]"
   ],
   [
    "C104",
    "x[6,Smartwatch]"
   ],
   [
    "C105",
    "x[6,Tablet]"
   ],
   [
    "C106",
    "x[6,Bluetooth Speaker]"
   ],
   [
    "C107",
    "x[6,Keyboard]"
   ],
   [
    "C108",
    "x[6,Mouse]"
   ],
   [
    "C109",
    "x[6,Monitor]"
   ],
   [
    "C110",
    "x[6,Printer]"
   ],
   [
    "C111",
    "x[6,External Hard Drive]"
   ],
   [
    "C112",
    "x[6,Router]"
   ],
   [
    "C113",
    "x[6,Power Bank]"
   ],
   [
    "C114",
    "x[6,Memory Card]"
   ],
   [
    "C115",
    "x[6,USB Flash Drive]"
   ],
   [
    "C116",
    "x[6,Smart Home Hub]"
   ],
   [
    "C117",
    "x[6,Gaming Console]"
   ],
   [
    "C118",
    "x[6,Fitness Tracker]"
   ],
   [
    "C119",
    "x[6,E-Reader]"
   ],
   [
    "C120",
    "x[7,Smartphone]"
   ],
   [
    "C121",
    "x[7,Laptop]"
   ],
   [
    "C122",
    "x[7,Headphones]"
   ],
   [
    "C123",
    "x[7,Camera]"
   ],
   [
    "C124",
    "x[7,Smartwatch]"
   ],
   [
    "C125",
    "x[7,Tablet]"
   ],
   [
    "C126",
    "x[7,Bluetooth Speaker]"
   ],
   [
    "C127",
    "x[7,Keyboard]"
   ],
   [
    "C128",
    "x[7,Mouse]"
   ],
   [
    "C129",
    "x[7,Monitor]"
   ],
   [
    "C130",
    "x[7,Printer]"
   ],
   [
    "C131",
    "x[7,External Hard Drive]"
   ],
   [
    "C132",
    "x[7,Router]"
   ],
   [
    "C133",
    "x[7,Power Bank]"
   ],
   [
    "C134",
    "x[7,Memory Card]"
   ],
   [
    "C135",
    "x[7,USB Flash Drive]"
   ],
   [
    "C136",
    "x[7,Smart Home Hub]"
   ],
   [
    "C137",
    "x[7,Gaming Console]"
   ],
   [
    "C138",
    "x[7,Fitness Tracker]"
   ],
   [
    "C139",
    "x[7,E-Reader]"
   ],
   [
    "C140",
    "x[8,Smartphone]"
   ],
   [
    "C141",
    "x[8,Laptop]"
   ],
   [
    "C142",
    "x[8,Headphones]"
   ],
   [
    "C143",
    "x[8,Camera]"
   ],
   [
    "C144",
    "x[8,Smartwatch]"
   ],
   [
    "C145",
    "x[8,Tablet]"
   ],
   [
    "C146",
    "x[8,Bluetooth Speaker]"
   ],
   [
    "C147",
    "x[8,Keyboard]"
   ],
   [
    "C148",
    "x[8,Mouse]"
   ],
   [
    "C149",
    "x[8,Monitor]"
   ],
   [
    "C150",
    "x[8,Printer]"
   ],
   [
    "C151",
    "x[8,External Hard Drive]"
   ],
   [
    "C152",
    "x[8,Router]"
   ],
   [
    "C153",
    "x[8,Power Bank]"
   ],
   [
    "C154",
    "x[8,Memory Card]"
   ],
   [
    "C155",
    "x[8,USB Flash Drive]"
   ],
   [
    "C156",
    "x[8,Smart Home Hub]"
   ],
   [
    "C157",
    "x[8,Gaming Console]"
   ],
   [
    "C158",
    "x[8,Fitness Tracker]"
   ],
   [
    "C159",
    "x[8,E-Reader]"
   ],
   [
    "C160",
    "x[9,Smartphone]"
   ],
   [
    "C161",
    "x[9,Laptop]"
   ],
   [
    "C162",
    "x[9,Headphones]"
   ],
   [
    "C163",
    "x[9,Camera]"
   ],
   [
    "C164",
    "x[9,Smartwatch]"
   ],
   [
    "C165",
    "x[9,Tablet]"
   ],
   [
    "C166",
    "x[9,Bluetooth Speaker]"
   ],
   [
    "C167",
    "x[9,Keyboard]"
   ],
   [
    "C168",
    "x[9,Mouse]"
   ],
   [
    "C169",
    "x[9,Monitor]"
   ],
   [
    "C170",
    "x[9,Printer]"
   ],
   [
    "C171",
    "x[9,External Hard Drive]"
   ],
   [
    "C172",
    "x[9,Router]"
   ],
   [
    "C173",
    "x[9,Power Bank]"
   ],
   [
    "C174",
    "x[9,Memory Card]"
   ],
   [
    "C175",
    "x[9,USB Flash Drive]"
   ],
   [
    "C176",
    "x[9,Smart Home Hub]"
   ],
   [
    "C177",
    "x[9,Gaming Console]"
   ],
   [
    "C178",
    "x[9,Fitness Tracker]"
   ],
   [
    "C179",
    "x[9,E-Reader]"
   ],
   [
    "C180",
    "x[10,Smartphone]"
   ],
   [
    "C181",
    "x[10,Laptop]"
   ],
   [
    "C182",
    "x[10,Headphones]"
   ],
   [
    "C183",
    "x[10,Camera]"
   ],
   [
    "C184",
    "x[10,Smartwatch]"
   ],
   [
    "C185",
    "x[10,Tablet]"
   ],
   [
    "C186",
    "x[10,Bluetooth Speaker]"
   ],
   [
    "C187",
    "x[10,Keyboard]"
   ],
   [
    "C188",
    "x[10,Mouse]"
   ],
   [
    "C189",
    "x[10,Monitor]"
   ],
   [
    "C190",
    "x[10,Printer]"
   ],
   [
    "C191",
    "x[10,External Hard Drive]"
   ],
   [
    "C192",
    "x[10,Router]"
   ],
   [
    "C193",
    "x[10,Power Bank]"
   ],
   [
    "C194",
    "x[10,Memory Card]"
   ],
   [
    "C195",
    "x[10,USB Flash Drive]"
   ],
   [
    "C196",
    "x[10,Smart Home Hub]"
   ],
   [
    "C197",
    "x[10,Gaming Console]"
   ],
   [
    "C198",
    "x[10,Fitness Tracker]"
   ],
   [
    "C199",
    "x[10,E-Reader]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Smartphone]",
   "x_1_0": "x[2,Smartphone]",
   "x_2_0": "x[3,Smartphone]",
   "x_3_0": "x[4,Smartphone]",
   "x_4_0": "x[5,Smartphone]",
   "x_5_0": "x[6,Smartphone]",
   "x_6_0": "x[7,Smartphone]",
   "x_7_0": "x[8,Smartphone]",
   "x_8_0": "x[9,Smartphone]",
   "x_9_0": "x[10,Smartphone]",
   "x_0_1": "x[1,Laptop]",
   "x_1_1": "x[2,Laptop]",
   "x_2_1": "x[3,Laptop]",
   "x_3_1": "x[4,Laptop]",
   "x_4_1": "x[5,Laptop]",
   "x_5_1": "x[6,Laptop]",
   "x_6_1": "x[7,Laptop]",
   "x_7_1": "x[8,Laptop]",
   "x_8_1": "x[9,Laptop]",
   "x_9_1": "x[10,Laptop]",
   "x_0_2": "x[1,Headphones]",
   "x_1_2": "x[2,Headphones]",
   "x_2_2": "x[3,Headphones]",
   "x_3_2": "x[4,Headphones]",
   "x_4_2": "x[5,Headphones]",
   "x_5_2": "x[6,Headphones]",
   "x_6_2": "x[7,Headphones]",
   "x_7_2": "x[8,Headphones]",
   "x_8_2": "x[9,Headphones]",
   "x_9_2": "x[10,Headphones]",
   "x_0_3": "x[1,Camera]",
   "x_1_3": "x[2,Camera]",
   "x_2_3": "x[3,Camera]",
   "x_3_3": "x[4,Camera]",
   "x_4_3": "x[5,Camera]",
   "x_5_3": "x[6,Camera]",
   "x_6_3": "x[7,Camera]",
   "x_7_3": "x[8,Camera]",
   "x_8_3": "x[9,Camera]",
   "x_9_3": "x[10,Camera]",
   "x_0_4": "x[1,Smartwatch]",
   "x_1_4": "x[2,Smartwatch]",
   "x_2_4": "x[3,Smartwatch]",
   "x_3_4": "x[4,Smartwatch]",
   "x_4_4": "x[5,Smartwatch]",
   "x_5_4": "x[6,Smartwatch]",
   "x_6_4": "x[7,Smartwatch]",
   "x_7_4": "x[8,Smartwatch]",
   "x_8_4": "x[9,Smartwatch]",
   "x_9_4": "x[10,Smartwatch]",
   "x_0_5": "x[1,Tablet]",
   "x_1_5": "x[2,Tablet]",
   "x_2_5": "x[3,Tablet]",
   "x_3_5": "x[4,Tablet]",
   "x_4_5": "x[5,Tablet]",
   "x_5_5": "x[6,Tablet]",
   "x_6_5": "x[7,Tablet]",
   "x_7_5": "x[8,Tablet]",
   "x_8_5": "x[9,Tablet]",
   "x_9_5": "x[10,Tablet]",
   "x_0_6": "x[1,Bluetooth Speaker]",
   "x_1_6": "x[2,Bluetooth Speaker]",
   "x_2_6": "x[3,Bluetooth Speaker]",
   "x_3_6": "x[4,Bluetooth Speaker]",
   "x_4_6": "x[5,Bluetooth Speaker]",
   "x_5_6": "x[6,Bluetooth Speaker]",
   "x_6_6": "x[7,Bluetooth Speaker]",
   "x_7_6": "x[8,Bluetooth Speaker]",
   "x_8_6": "x[9,Bluetooth Speaker]",
   "x_9_6": "x[10,Bluetooth Speaker]",
   "x_0_7": "x[1,Keyboard]",
   "x_1_7": "x[2,Keyboard]",
   "x_2_7": "x[3,Keyboard]",
   "x_3_7": "x[4,Keyboard]",
   "x_4_7": "x[5,Keyboard]",
   "x_5_7": "x[6,Keyboard]",
   "x_6_7": "x[7,Keyboard]",
   "x_7_7": "x[8,Keyboard]",
   "x_8_7": "x[9,Keyboard]",
   "x_9_7": "x[10,Keyboard]",
   "x_0_8": "x[1,Mouse]",
   "x_1_8": "x[2,Mouse]",
   "x_2_8": "x[3,Mouse]",
   "x_3_8": "x[4,Mouse]",
   "x_4_8": "x[5,Mouse]",
   "x_5_8": "x[6,Mouse]",
   "x_6_8": "x[7,Mouse]",
   "x_7_8": "x[8,Mouse]",
   "x_8_8": "x[9,Mouse]",
   "x_9_8": "x[10,Mouse]",
   "x_0_9": "x[1,Monitor]",
   "x_1_9": "x[2,Monitor]",
   "x_2_9": "x[3,Monitor]",
   "x_3_9": "x[4,Monitor]",
   "x_4_9": "x[5,Monitor]",
   "x_5_9": "x[6,Monitor]",
   "x_6_9": "x[7,Monitor]",
   "x_7_9": "x[8,Monitor]",
   "x_8_9": "x[9,Monitor]",
   "x_9_9": "x[10,Monitor]",
   "x_0_10": "x[1,Printer]",
   "x_1_10": "x[2,Printer]",
   "x_2_10": "x[3,Printer]",
   "x_3_10": "x[4,Printer]",
   "x_4_10": "x[5,Printer]",
   "x_5_10": "x[6,Printer]",
   "x_6_10": "x[7,Printer]",
   "x_7_10": "x[8,Printer]",
   "x_8_10": "x[9,Printer]",
   "x_9_10": "x[10,Printer]",
   "x_0_11": "x[1,External Hard Drive]",
   "x_1_11": "x[2,External Hard Drive]",
   "x_2_11": "x[3,External Hard Drive]",
   "x_3_11": "x[4,External Hard Drive]",
   "x_4_11": "x[5,External Hard Drive]",
   "x_5_11": "x[6,External Hard Drive]",
   "x_6_11": "x[7,External Hard Drive]",
   "x_7_11": "x[8,External Hard Drive]",
   "x_8_11": "x[9,External Hard Drive]",
   "x_9_11": "x[10,External Hard Drive]",
   "x_0_12": "x[1,Router]",
   "x_1_12": "x[2,Router]",
   "x_2_12": "x[3,Router]",
   "x_3_12": "x[4,Router]",
   "x_4_12": "x[5,Router]",
   "x_5_12": "x[6,Router]",
   "x_6_12": "x[7,Router]",
   "x_7_12": "x[8,Router]",
   "x_8_12": "x[9,Router]",
   "x_9_12": "x[10,Router]",
   "x_0_13": "x[1,Power Bank]",
   "x_1_13": "x[2,Power Bank]",
   "x_2_13": "x[3,Power Bank]",
   "x_3_13": "x[4,Power Bank]",
   "x_4_13": "x[5,Power Bank]",
   "x_5_13": "x[6,Power Bank]",
   "x_6_13": "x[7,Power Bank]",
   "x_7_13": "x[8,Power Bank]",
   "x_8_13": "x[9,Power Bank]",
   "x_9_13": "x[10,Power Bank]",
   "x_0_14": "x[1,Memory Card]",
   "x_1_14": "x[2,Memory Card]",
   "x_2_14": "x[3,Memory Card]",
   "x_3_14": "x[4,Memory Card]",
   "x_4_14": "x[5,Memory Card]",
   "x_5_14": "x[6,Memory Card]",
   "x_6_14": "x[7,Memory Card]",
   "x_7_14": "x[8,Memory Card]",
   "x_8_14": "x[9,Memory Card]",
   "x_9_14": "x[10,Memory Card]",
   "x_0_15": "x[1,USB Flash Drive]",
   "x_1_15": "x[2,USB Flash Drive]",
   "x_2_15": "x[3,USB Flash Drive]",
   "x_3_15": "x[4,USB Flash Drive]",
   "x_4_15": "x[5,USB Flash Drive]",
   "x_5_15": "x[6,USB Flash Drive]",
   "x_6_15": "x[7,USB Flash Drive]",
   "x_7_15": "x[8,USB Flash Drive]",
   "x_8_15": "x[9,USB Flash Drive]",
   "x_9_15": "x[10,USB Flash Drive]",
   "x_0_16": "x[1,Smart Home Hub]",
   "x_1_16": "x[2,Smart Home Hub]",
   "x_2_16": "x[3,Smart Home Hub]",
   "x_3_16": "x[4,Smart Home Hub]",
   "x_4_16": "x[5,Smart Home Hub]",
   "x_5_16": "x[6,Smart Home Hub]",
   "x_6_16": "x[7,Smart Home Hub]",
   "x_7_16": "x[8,Smart Home Hub]",
   "x_8_16": "x[9,Smart Home Hub]",
   "x_9_16": "x[10,Smart Home Hub]",
   "x_0_17": "x[1,Gaming Console]",
   "x_1_17": "x[2,Gaming Console]",
   "x_2_17": "x[3,Gaming Console]",
   "x_3_17": "x[4,Gaming Console]",
   "x_4_17": "x[5,Gaming Console]",
   "x_5_17": "x[6,Gaming Console]",
   "x_6_17": "x[7,Gaming Console]",
   "x_7_17": "x[8,Gaming Console]",
   "x_8_17": "x[9,Gaming Console]",
   "x_9_17": "x[10,Gaming Console]",
   "x_0_18": "x[1,Fitness Tracker]",
   "x_1_18": "x[2,Fitness Tracker]",
   "x_2_18": "x[3,Fitness Tracker]",
   "x_3_18": "x[4,Fitness Tracker]",
   "x_4_18": "x[5,Fitness Tracker]",
   "x_5_18": "x[6,Fitness Tracker]",
   "x_6_18": "x[7,Fitness Tracker]",
   "x_7_18": "x[8,Fitness Tracker]",
   "x_8_18": "x[9,Fitness Tracker]",
   "x_9_18": "x[10,Fitness Tracker]",
   "x_0_19": "x[1,E-Reader]",
   "x_1_19": "x[2,E-Reader]",
   "x_2_19": "x[3,E-Reader]",
   "x_3_19": "x[4,E-Reader]",
   "x_4_19": "x[5,E-Reader]",
   "x_5_19": "x[6,E-Reader]",
   "x_6_19": "x[7,E-Reader]",
   "x_7_19": "x[8,E-Reader]",
   "x_8_19": "x[9,E-Reader]",
   "x_9_19": "x[10,E-Reader]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 80250.0,
   "variables": [
    [
     "C0",
     5.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     350.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     300.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     400.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     275.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     450.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     325.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     375.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     410.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     -0.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     285.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-054": {
  "source_code_sha256": "5198bfaa351085f013fe725118e4467f16289dcff633ecd70737e6cc44a50609",
  "original_solution_sha256": "763cc8a10acb85fbf69cbc0d68bc505975588bdb7752fae71b76faf7e61c1ba9",
  "bindings": [
   [
    "C0",
    "x[1,1]"
   ],
   [
    "C1",
    "x[1,2]"
   ],
   [
    "C2",
    "x[1,3]"
   ],
   [
    "C3",
    "x[1,4]"
   ],
   [
    "C4",
    "x[1,5]"
   ],
   [
    "C5",
    "x[1,6]"
   ],
   [
    "C6",
    "x[1,7]"
   ],
   [
    "C7",
    "x[1,8]"
   ],
   [
    "C8",
    "x[1,9]"
   ],
   [
    "C9",
    "x[1,10]"
   ],
   [
    "C10",
    "x[1,11]"
   ],
   [
    "C11",
    "x[1,12]"
   ],
   [
    "C12",
    "x[1,13]"
   ],
   [
    "C13",
    "x[1,14]"
   ],
   [
    "C14",
    "x[1,15]"
   ],
   [
    "C15",
    "x[1,16]"
   ],
   [
    "C16",
    "x[1,17]"
   ],
   [
    "C17",
    "x[1,18]"
   ],
   [
    "C18",
    "x[1,19]"
   ],
   [
    "C19",
    "x[1,20]"
   ],
   [
    "C20",
    "x[2,1]"
   ],
   [
    "C21",
    "x[2,2]"
   ],
   [
    "C22",
    "x[2,3]"
   ],
   [
    "C23",
    "x[2,4]"
   ],
   [
    "C24",
    "x[2,5]"
   ],
   [
    "C25",
    "x[2,6]"
   ],
   [
    "C26",
    "x[2,7]"
   ],
   [
    "C27",
    "x[2,8]"
   ],
   [
    "C28",
    "x[2,9]"
   ],
   [
    "C29",
    "x[2,10]"
   ],
   [
    "C30",
    "x[2,11]"
   ],
   [
    "C31",
    "x[2,12]"
   ],
   [
    "C32",
    "x[2,13]"
   ],
   [
    "C33",
    "x[2,14]"
   ],
   [
    "C34",
    "x[2,15]"
   ],
   [
    "C35",
    "x[2,16]"
   ],
   [
    "C36",
    "x[2,17]"
   ],
   [
    "C37",
    "x[2,18]"
   ],
   [
    "C38",
    "x[2,19]"
   ],
   [
    "C39",
    "x[2,20]"
   ],
   [
    "C40",
    "x[3,1]"
   ],
   [
    "C41",
    "x[3,2]"
   ],
   [
    "C42",
    "x[3,3]"
   ],
   [
    "C43",
    "x[3,4]"
   ],
   [
    "C44",
    "x[3,5]"
   ],
   [
    "C45",
    "x[3,6]"
   ],
   [
    "C46",
    "x[3,7]"
   ],
   [
    "C47",
    "x[3,8]"
   ],
   [
    "C48",
    "x[3,9]"
   ],
   [
    "C49",
    "x[3,10]"
   ],
   [
    "C50",
    "x[3,11]"
   ],
   [
    "C51",
    "x[3,12]"
   ],
   [
    "C52",
    "x[3,13]"
   ],
   [
    "C53",
    "x[3,14]"
   ],
   [
    "C54",
    "x[3,15]"
   ],
   [
    "C55",
    "x[3,16]"
   ],
   [
    "C56",
    "x[3,17]"
   ],
   [
    "C57",
    "x[3,18]"
   ],
   [
    "C58",
    "x[3,19]"
   ],
   [
    "C59",
    "x[3,20]"
   ],
   [
    "C60",
    "x[4,1]"
   ],
   [
    "C61",
    "x[4,2]"
   ],
   [
    "C62",
    "x[4,3]"
   ],
   [
    "C63",
    "x[4,4]"
   ],
   [
    "C64",
    "x[4,5]"
   ],
   [
    "C65",
    "x[4,6]"
   ],
   [
    "C66",
    "x[4,7]"
   ],
   [
    "C67",
    "x[4,8]"
   ],
   [
    "C68",
    "x[4,9]"
   ],
   [
    "C69",
    "x[4,10]"
   ],
   [
    "C70",
    "x[4,11]"
   ],
   [
    "C71",
    "x[4,12]"
   ],
   [
    "C72",
    "x[4,13]"
   ],
   [
    "C73",
    "x[4,14]"
   ],
   [
    "C74",
    "x[4,15]"
   ],
   [
    "C75",
    "x[4,16]"
   ],
   [
    "C76",
    "x[4,17]"
   ],
   [
    "C77",
    "x[4,18]"
   ],
   [
    "C78",
    "x[4,19]"
   ],
   [
    "C79",
    "x[4,20]"
   ],
   [
    "C80",
    "x[5,1]"
   ],
   [
    "C81",
    "x[5,2]"
   ],
   [
    "C82",
    "x[5,3]"
   ],
   [
    "C83",
    "x[5,4]"
   ],
   [
    "C84",
    "x[5,5]"
   ],
   [
    "C85",
    "x[5,6]"
   ],
   [
    "C86",
    "x[5,7]"
   ],
   [
    "C87",
    "x[5,8]"
   ],
   [
    "C88",
    "x[5,9]"
   ],
   [
    "C89",
    "x[5,10]"
   ],
   [
    "C90",
    "x[5,11]"
   ],
   [
    "C91",
    "x[5,12]"
   ],
   [
    "C92",
    "x[5,13]"
   ],
   [
    "C93",
    "x[5,14]"
   ],
   [
    "C94",
    "x[5,15]"
   ],
   [
    "C95",
    "x[5,16]"
   ],
   [
    "C96",
    "x[5,17]"
   ],
   [
    "C97",
    "x[5,18]"
   ],
   [
    "C98",
    "x[5,19]"
   ],
   [
    "C99",
    "x[5,20]"
   ],
   [
    "C100",
    "x[6,1]"
   ],
   [
    "C101",
    "x[6,2]"
   ],
   [
    "C102",
    "x[6,3]"
   ],
   [
    "C103",
    "x[6,4]"
   ],
   [
    "C104",
    "x[6,5]"
   ],
   [
    "C105",
    "x[6,6]"
   ],
   [
    "C106",
    "x[6,7]"
   ],
   [
    "C107",
    "x[6,8]"
   ],
   [
    "C108",
    "x[6,9]"
   ],
   [
    "C109",
    "x[6,10]"
   ],
   [
    "C110",
    "x[6,11]"
   ],
   [
    "C111",
    "x[6,12]"
   ],
   [
    "C112",
    "x[6,13]"
   ],
   [
    "C113",
    "x[6,14]"
   ],
   [
    "C114",
    "x[6,15]"
   ],
   [
    "C115",
    "x[6,16]"
   ],
   [
    "C116",
    "x[6,17]"
   ],
   [
    "C117",
    "x[6,18]"
   ],
   [
    "C118",
    "x[6,19]"
   ],
   [
    "C119",
    "x[6,20]"
   ],
   [
    "C120",
    "x[7,1]"
   ],
   [
    "C121",
    "x[7,2]"
   ],
   [
    "C122",
    "x[7,3]"
   ],
   [
    "C123",
    "x[7,4]"
   ],
   [
    "C124",
    "x[7,5]"
   ],
   [
    "C125",
    "x[7,6]"
   ],
   [
    "C126",
    "x[7,7]"
   ],
   [
    "C127",
    "x[7,8]"
   ],
   [
    "C128",
    "x[7,9]"
   ],
   [
    "C129",
    "x[7,10]"
   ],
   [
    "C130",
    "x[7,11]"
   ],
   [
    "C131",
    "x[7,12]"
   ],
   [
    "C132",
    "x[7,13]"
   ],
   [
    "C133",
    "x[7,14]"
   ],
   [
    "C134",
    "x[7,15]"
   ],
   [
    "C135",
    "x[7,16]"
   ],
   [
    "C136",
    "x[7,17]"
   ],
   [
    "C137",
    "x[7,18]"
   ],
   [
    "C138",
    "x[7,19]"
   ],
   [
    "C139",
    "x[7,20]"
   ],
   [
    "C140",
    "x[8,1]"
   ],
   [
    "C141",
    "x[8,2]"
   ],
   [
    "C142",
    "x[8,3]"
   ],
   [
    "C143",
    "x[8,4]"
   ],
   [
    "C144",
    "x[8,5]"
   ],
   [
    "C145",
    "x[8,6]"
   ],
   [
    "C146",
    "x[8,7]"
   ],
   [
    "C147",
    "x[8,8]"
   ],
   [
    "C148",
    "x[8,9]"
   ],
   [
    "C149",
    "x[8,10]"
   ],
   [
    "C150",
    "x[8,11]"
   ],
   [
    "C151",
    "x[8,12]"
   ],
   [
    "C152",
    "x[8,13]"
   ],
   [
    "C153",
    "x[8,14]"
   ],
   [
    "C154",
    "x[8,15]"
   ],
   [
    "C155",
    "x[8,16]"
   ],
   [
    "C156",
    "x[8,17]"
   ],
   [
    "C157",
    "x[8,18]"
   ],
   [
    "C158",
    "x[8,19]"
   ],
   [
    "C159",
    "x[8,20]"
   ],
   [
    "C160",
    "x[9,1]"
   ],
   [
    "C161",
    "x[9,2]"
   ],
   [
    "C162",
    "x[9,3]"
   ],
   [
    "C163",
    "x[9,4]"
   ],
   [
    "C164",
    "x[9,5]"
   ],
   [
    "C165",
    "x[9,6]"
   ],
   [
    "C166",
    "x[9,7]"
   ],
   [
    "C167",
    "x[9,8]"
   ],
   [
    "C168",
    "x[9,9]"
   ],
   [
    "C169",
    "x[9,10]"
   ],
   [
    "C170",
    "x[9,11]"
   ],
   [
    "C171",
    "x[9,12]"
   ],
   [
    "C172",
    "x[9,13]"
   ],
   [
    "C173",
    "x[9,14]"
   ],
   [
    "C174",
    "x[9,15]"
   ],
   [
    "C175",
    "x[9,16]"
   ],
   [
    "C176",
    "x[9,17]"
   ],
   [
    "C177",
    "x[9,18]"
   ],
   [
    "C178",
    "x[9,19]"
   ],
   [
    "C179",
    "x[9,20]"
   ],
   [
    "C180",
    "x[10,1]"
   ],
   [
    "C181",
    "x[10,2]"
   ],
   [
    "C182",
    "x[10,3]"
   ],
   [
    "C183",
    "x[10,4]"
   ],
   [
    "C184",
    "x[10,5]"
   ],
   [
    "C185",
    "x[10,6]"
   ],
   [
    "C186",
    "x[10,7]"
   ],
   [
    "C187",
    "x[10,8]"
   ],
   [
    "C188",
    "x[10,9]"
   ],
   [
    "C189",
    "x[10,10]"
   ],
   [
    "C190",
    "x[10,11]"
   ],
   [
    "C191",
    "x[10,12]"
   ],
   [
    "C192",
    "x[10,13]"
   ],
   [
    "C193",
    "x[10,14]"
   ],
   [
    "C194",
    "x[10,15]"
   ],
   [
    "C195",
    "x[10,16]"
   ],
   [
    "C196",
    "x[10,17]"
   ],
   [
    "C197",
    "x[10,18]"
   ],
   [
    "C198",
    "x[10,19]"
   ],
   [
    "C199",
    "x[10,20]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,1]",
   "x_0_1": "x[2,1]",
   "x_0_2": "x[3,1]",
   "x_0_3": "x[4,1]",
   "x_0_4": "x[5,1]",
   "x_0_5": "x[6,1]",
   "x_0_6": "x[7,1]",
   "x_0_7": "x[8,1]",
   "x_0_8": "x[9,1]",
   "x_0_9": "x[10,1]",
   "x_1_0": "x[1,2]",
   "x_1_1": "x[2,2]",
   "x_1_2": "x[3,2]",
   "x_1_3": "x[4,2]",
   "x_1_4": "x[5,2]",
   "x_1_5": "x[6,2]",
   "x_1_6": "x[7,2]",
   "x_1_7": "x[8,2]",
   "x_1_8": "x[9,2]",
   "x_1_9": "x[10,2]",
   "x_2_0": "x[1,3]",
   "x_2_1": "x[2,3]",
   "x_2_2": "x[3,3]",
   "x_2_3": "x[4,3]",
   "x_2_4": "x[5,3]",
   "x_2_5": "x[6,3]",
   "x_2_6": "x[7,3]",
   "x_2_7": "x[8,3]",
   "x_2_8": "x[9,3]",
   "x_2_9": "x[10,3]",
   "x_3_0": "x[1,4]",
   "x_3_1": "x[2,4]",
   "x_3_2": "x[3,4]",
   "x_3_3": "x[4,4]",
   "x_3_4": "x[5,4]",
   "x_3_5": "x[6,4]",
   "x_3_6": "x[7,4]",
   "x_3_7": "x[8,4]",
   "x_3_8": "x[9,4]",
   "x_3_9": "x[10,4]",
   "x_4_0": "x[1,5]",
   "x_4_1": "x[2,5]",
   "x_4_2": "x[3,5]",
   "x_4_3": "x[4,5]",
   "x_4_4": "x[5,5]",
   "x_4_5": "x[6,5]",
   "x_4_6": "x[7,5]",
   "x_4_7": "x[8,5]",
   "x_4_8": "x[9,5]",
   "x_4_9": "x[10,5]",
   "x_5_0": "x[1,6]",
   "x_5_1": "x[2,6]",
   "x_5_2": "x[3,6]",
   "x_5_3": "x[4,6]",
   "x_5_4": "x[5,6]",
   "x_5_5": "x[6,6]",
   "x_5_6": "x[7,6]",
   "x_5_7": "x[8,6]",
   "x_5_8": "x[9,6]",
   "x_5_9": "x[10,6]",
   "x_6_0": "x[1,7]",
   "x_6_1": "x[2,7]",
   "x_6_2": "x[3,7]",
   "x_6_3": "x[4,7]",
   "x_6_4": "x[5,7]",
   "x_6_5": "x[6,7]",
   "x_6_6": "x[7,7]",
   "x_6_7": "x[8,7]",
   "x_6_8": "x[9,7]",
   "x_6_9": "x[10,7]",
   "x_7_0": "x[1,8]",
   "x_7_1": "x[2,8]",
   "x_7_2": "x[3,8]",
   "x_7_3": "x[4,8]",
   "x_7_4": "x[5,8]",
   "x_7_5": "x[6,8]",
   "x_7_6": "x[7,8]",
   "x_7_7": "x[8,8]",
   "x_7_8": "x[9,8]",
   "x_7_9": "x[10,8]",
   "x_8_0": "x[1,9]",
   "x_8_1": "x[2,9]",
   "x_8_2": "x[3,9]",
   "x_8_3": "x[4,9]",
   "x_8_4": "x[5,9]",
   "x_8_5": "x[6,9]",
   "x_8_6": "x[7,9]",
   "x_8_7": "x[8,9]",
   "x_8_8": "x[9,9]",
   "x_8_9": "x[10,9]",
   "x_9_0": "x[1,10]",
   "x_9_1": "x[2,10]",
   "x_9_2": "x[3,10]",
   "x_9_3": "x[4,10]",
   "x_9_4": "x[5,10]",
   "x_9_5": "x[6,10]",
   "x_9_6": "x[7,10]",
   "x_9_7": "x[8,10]",
   "x_9_8": "x[9,10]",
   "x_9_9": "x[10,10]",
   "x_10_0": "x[1,11]",
   "x_10_1": "x[2,11]",
   "x_10_2": "x[3,11]",
   "x_10_3": "x[4,11]",
   "x_10_4": "x[5,11]",
   "x_10_5": "x[6,11]",
   "x_10_6": "x[7,11]",
   "x_10_7": "x[8,11]",
   "x_10_8": "x[9,11]",
   "x_10_9": "x[10,11]",
   "x_11_0": "x[1,12]",
   "x_11_1": "x[2,12]",
   "x_11_2": "x[3,12]",
   "x_11_3": "x[4,12]",
   "x_11_4": "x[5,12]",
   "x_11_5": "x[6,12]",
   "x_11_6": "x[7,12]",
   "x_11_7": "x[8,12]",
   "x_11_8": "x[9,12]",
   "x_11_9": "x[10,12]",
   "x_12_0": "x[1,13]",
   "x_12_1": "x[2,13]",
   "x_12_2": "x[3,13]",
   "x_12_3": "x[4,13]",
   "x_12_4": "x[5,13]",
   "x_12_5": "x[6,13]",
   "x_12_6": "x[7,13]",
   "x_12_7": "x[8,13]",
   "x_12_8": "x[9,13]",
   "x_12_9": "x[10,13]",
   "x_13_0": "x[1,14]",
   "x_13_1": "x[2,14]",
   "x_13_2": "x[3,14]",
   "x_13_3": "x[4,14]",
   "x_13_4": "x[5,14]",
   "x_13_5": "x[6,14]",
   "x_13_6": "x[7,14]",
   "x_13_7": "x[8,14]",
   "x_13_8": "x[9,14]",
   "x_13_9": "x[10,14]",
   "x_14_0": "x[1,15]",
   "x_14_1": "x[2,15]",
   "x_14_2": "x[3,15]",
   "x_14_3": "x[4,15]",
   "x_14_4": "x[5,15]",
   "x_14_5": "x[6,15]",
   "x_14_6": "x[7,15]",
   "x_14_7": "x[8,15]",
   "x_14_8": "x[9,15]",
   "x_14_9": "x[10,15]",
   "x_15_0": "x[1,16]",
   "x_15_1": "x[2,16]",
   "x_15_2": "x[3,16]",
   "x_15_3": "x[4,16]",
   "x_15_4": "x[5,16]",
   "x_15_5": "x[6,16]",
   "x_15_6": "x[7,16]",
   "x_15_7": "x[8,16]",
   "x_15_8": "x[9,16]",
   "x_15_9": "x[10,16]",
   "x_16_0": "x[1,17]",
   "x_16_1": "x[2,17]",
   "x_16_2": "x[3,17]",
   "x_16_3": "x[4,17]",
   "x_16_4": "x[5,17]",
   "x_16_5": "x[6,17]",
   "x_16_6": "x[7,17]",
   "x_16_7": "x[8,17]",
   "x_16_8": "x[9,17]",
   "x_16_9": "x[10,17]",
   "x_17_0": "x[1,18]",
   "x_17_1": "x[2,18]",
   "x_17_2": "x[3,18]",
   "x_17_3": "x[4,18]",
   "x_17_4": "x[5,18]",
   "x_17_5": "x[6,18]",
   "x_17_6": "x[7,18]",
   "x_17_7": "x[8,18]",
   "x_17_8": "x[9,18]",
   "x_17_9": "x[10,18]",
   "x_18_0": "x[1,19]",
   "x_18_1": "x[2,19]",
   "x_18_2": "x[3,19]",
   "x_18_3": "x[4,19]",
   "x_18_4": "x[5,19]",
   "x_18_5": "x[6,19]",
   "x_18_6": "x[7,19]",
   "x_18_7": "x[8,19]",
   "x_18_8": "x[9,19]",
   "x_18_9": "x[10,19]",
   "x_19_0": "x[1,20]",
   "x_19_1": "x[2,20]",
   "x_19_2": "x[3,20]",
   "x_19_3": "x[4,20]",
   "x_19_4": "x[5,20]",
   "x_19_5": "x[6,20]",
   "x_19_6": "x[7,20]",
   "x_19_7": "x[8,20]",
   "x_19_8": "x[9,20]",
   "x_19_9": "x[10,20]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 98670.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     150.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     164.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     114.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     160.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     110.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     180.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     130.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     160.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     170.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     180.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-058": {
  "source_code_sha256": "393a248056924669197cd83e3daf8e3720f0fb983ec0ce9d3e858045e6fa612f",
  "original_solution_sha256": "f56ee48aad5c6dd2ccf5b3309993ee6fbf4904a484aaf1688bac41f00d637e25",
  "bindings": [
   [
    "C0",
    "y[S2]"
   ],
   [
    "C1",
    "y[S3]"
   ],
   [
    "C2",
    "y[S6]"
   ],
   [
    "C3",
    "y[S4]"
   ],
   [
    "C4",
    "y[S1]"
   ],
   [
    "C5",
    "y[S5]"
   ],
   [
    "C6",
    "x[S2,C6]"
   ],
   [
    "C7",
    "x[S2,C5]"
   ],
   [
    "C8",
    "x[S2,C2]"
   ],
   [
    "C9",
    "x[S2,C4]"
   ],
   [
    "C10",
    "x[S2,C1]"
   ],
   [
    "C11",
    "x[S2,C3]"
   ],
   [
    "C12",
    "x[S3,C6]"
   ],
   [
    "C13",
    "x[S3,C5]"
   ],
   [
    "C14",
    "x[S3,C2]"
   ],
   [
    "C15",
    "x[S3,C4]"
   ],
   [
    "C16",
    "x[S3,C1]"
   ],
   [
    "C17",
    "x[S3,C3]"
   ],
   [
    "C18",
    "x[S6,C6]"
   ],
   [
    "C19",
    "x[S6,C5]"
   ],
   [
    "C20",
    "x[S6,C2]"
   ],
   [
    "C21",
    "x[S6,C4]"
   ],
   [
    "C22",
    "x[S6,C1]"
   ],
   [
    "C23",
    "x[S6,C3]"
   ],
   [
    "C24",
    "x[S4,C6]"
   ],
   [
    "C25",
    "x[S4,C5]"
   ],
   [
    "C26",
    "x[S4,C2]"
   ],
   [
    "C27",
    "x[S4,C4]"
   ],
   [
    "C28",
    "x[S4,C1]"
   ],
   [
    "C29",
    "x[S4,C3]"
   ],
   [
    "C30",
    "x[S1,C6]"
   ],
   [
    "C31",
    "x[S1,C5]"
   ],
   [
    "C32",
    "x[S1,C2]"
   ],
   [
    "C33",
    "x[S1,C4]"
   ],
   [
    "C34",
    "x[S1,C1]"
   ],
   [
    "C35",
    "x[S1,C3]"
   ],
   [
    "C36",
    "x[S5,C6]"
   ],
   [
    "C37",
    "x[S5,C5]"
   ],
   [
    "C38",
    "x[S5,C2]"
   ],
   [
    "C39",
    "x[S5,C4]"
   ],
   [
    "C40",
    "x[S5,C1]"
   ],
   [
    "C41",
    "x[S5,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]",
   "y_3": "y[S4]",
   "y_4": "y[S5]",
   "y_5": "y[S6]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 35950.65,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     1.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     216.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     144.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     144.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     216.0
    ],
    [
     "C35",
     216.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     144.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-063": {
  "source_code_sha256": "154254773f7cff08222803813c1394a40a4265b3698abe0bd5e8481dd1de9b64",
  "original_solution_sha256": "b604cc4bc3c7123cf54d006ee12e8f27eae1dfdfd6249a5649a585e171d2ba3d",
  "bindings": [
   [
    "C0",
    "y[S2]"
   ],
   [
    "C1",
    "y[S7]"
   ],
   [
    "C2",
    "y[S3]"
   ],
   [
    "C3",
    "y[S6]"
   ],
   [
    "C4",
    "y[S4]"
   ],
   [
    "C5",
    "y[S1]"
   ],
   [
    "C6",
    "y[S5]"
   ],
   [
    "C7",
    "x[S2,C7]"
   ],
   [
    "C8",
    "x[S2,C6]"
   ],
   [
    "C9",
    "x[S2,C5]"
   ],
   [
    "C10",
    "x[S2,C2]"
   ],
   [
    "C11",
    "x[S2,C4]"
   ],
   [
    "C12",
    "x[S2,C1]"
   ],
   [
    "C13",
    "x[S2,C3]"
   ],
   [
    "C14",
    "x[S7,C7]"
   ],
   [
    "C15",
    "x[S7,C6]"
   ],
   [
    "C16",
    "x[S7,C5]"
   ],
   [
    "C17",
    "x[S7,C2]"
   ],
   [
    "C18",
    "x[S7,C4]"
   ],
   [
    "C19",
    "x[S7,C1]"
   ],
   [
    "C20",
    "x[S7,C3]"
   ],
   [
    "C21",
    "x[S3,C7]"
   ],
   [
    "C22",
    "x[S3,C6]"
   ],
   [
    "C23",
    "x[S3,C5]"
   ],
   [
    "C24",
    "x[S3,C2]"
   ],
   [
    "C25",
    "x[S3,C4]"
   ],
   [
    "C26",
    "x[S3,C1]"
   ],
   [
    "C27",
    "x[S3,C3]"
   ],
   [
    "C28",
    "x[S6,C7]"
   ],
   [
    "C29",
    "x[S6,C6]"
   ],
   [
    "C30",
    "x[S6,C5]"
   ],
   [
    "C31",
    "x[S6,C2]"
   ],
   [
    "C32",
    "x[S6,C4]"
   ],
   [
    "C33",
    "x[S6,C1]"
   ],
   [
    "C34",
    "x[S6,C3]"
   ],
   [
    "C35",
    "x[S4,C7]"
   ],
   [
    "C36",
    "x[S4,C6]"
   ],
   [
    "C37",
    "x[S4,C5]"
   ],
   [
    "C38",
    "x[S4,C2]"
   ],
   [
    "C39",
    "x[S4,C4]"
   ],
   [
    "C40",
    "x[S4,C1]"
   ],
   [
    "C41",
    "x[S4,C3]"
   ],
   [
    "C42",
    "x[S1,C7]"
   ],
   [
    "C43",
    "x[S1,C6]"
   ],
   [
    "C44",
    "x[S1,C5]"
   ],
   [
    "C45",
    "x[S1,C2]"
   ],
   [
    "C46",
    "x[S1,C4]"
   ],
   [
    "C47",
    "x[S1,C1]"
   ],
   [
    "C48",
    "x[S1,C3]"
   ],
   [
    "C49",
    "x[S5,C7]"
   ],
   [
    "C50",
    "x[S5,C6]"
   ],
   [
    "C51",
    "x[S5,C5]"
   ],
   [
    "C52",
    "x[S5,C2]"
   ],
   [
    "C53",
    "x[S5,C4]"
   ],
   [
    "C54",
    "x[S5,C1]"
   ],
   [
    "C55",
    "x[S5,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]",
   "y_3": "y[S4]",
   "y_4": "y[S5]",
   "y_5": "y[S6]",
   "y_6": "y[S7]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 778095.44,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     17106.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     594.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     553.0
    ],
    [
     "C33",
     1083.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     732.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     776.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     16214.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-068": {
  "source_code_sha256": "5488f0aef48869ef7fdabd3988b616cb450d60c096d587373c75526a8894691b",
  "original_solution_sha256": "42bd7e848e91f292b559ceffbe2a5b282b53743efada860871afd630b4ef5fa3",
  "bindings": [
   [
    "C0",
    "x[MA,P1]"
   ],
   [
    "C1",
    "x[MA,P2]"
   ],
   [
    "C2",
    "x[MA,P3]"
   ],
   [
    "C3",
    "x[MA,P4]"
   ],
   [
    "C4",
    "x[MA,P5]"
   ],
   [
    "C5",
    "x[MA,P6]"
   ],
   [
    "C6",
    "x[MB,P1]"
   ],
   [
    "C7",
    "x[MB,P2]"
   ],
   [
    "C8",
    "x[MB,P3]"
   ],
   [
    "C9",
    "x[MB,P4]"
   ],
   [
    "C10",
    "x[MB,P5]"
   ],
   [
    "C11",
    "x[MB,P6]"
   ],
   [
    "C12",
    "x[MC,P1]"
   ],
   [
    "C13",
    "x[MC,P2]"
   ],
   [
    "C14",
    "x[MC,P3]"
   ],
   [
    "C15",
    "x[MC,P4]"
   ],
   [
    "C16",
    "x[MC,P5]"
   ],
   [
    "C17",
    "x[MC,P6]"
   ],
   [
    "C18",
    "x[MD,P1]"
   ],
   [
    "C19",
    "x[MD,P2]"
   ],
   [
    "C20",
    "x[MD,P3]"
   ],
   [
    "C21",
    "x[MD,P4]"
   ],
   [
    "C22",
    "x[MD,P5]"
   ],
   [
    "C23",
    "x[MD,P6]"
   ],
   [
    "C24",
    "x[ME,P1]"
   ],
   [
    "C25",
    "x[ME,P2]"
   ],
   [
    "C26",
    "x[ME,P3]"
   ],
   [
    "C27",
    "x[ME,P4]"
   ],
   [
    "C28",
    "x[ME,P5]"
   ],
   [
    "C29",
    "x[ME,P6]"
   ],
   [
    "C30",
    "x[MF,P1]"
   ],
   [
    "C31",
    "x[MF,P2]"
   ],
   [
    "C32",
    "x[MF,P3]"
   ],
   [
    "C33",
    "x[MF,P4]"
   ],
   [
    "C34",
    "x[MF,P5]"
   ],
   [
    "C35",
    "x[MF,P6]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[MA,P1]",
   "x_0_1": "x[MA,P2]",
   "x_0_2": "x[MA,P3]",
   "x_0_3": "x[MA,P4]",
   "x_0_4": "x[MA,P5]",
   "x_0_5": "x[MA,P6]",
   "x_1_0": "x[MB,P1]",
   "x_1_1": "x[MB,P2]",
   "x_1_2": "x[MB,P3]",
   "x_1_3": "x[MB,P4]",
   "x_1_4": "x[MB,P5]",
   "x_1_5": "x[MB,P6]",
   "x_2_0": "x[MC,P1]",
   "x_2_1": "x[MC,P2]",
   "x_2_2": "x[MC,P3]",
   "x_2_3": "x[MC,P4]",
   "x_2_4": "x[MC,P5]",
   "x_2_5": "x[MC,P6]",
   "x_3_0": "x[MD,P1]",
   "x_3_1": "x[MD,P2]",
   "x_3_2": "x[MD,P3]",
   "x_3_3": "x[MD,P4]",
   "x_3_4": "x[MD,P5]",
   "x_3_5": "x[MD,P6]",
   "x_4_0": "x[ME,P1]",
   "x_4_1": "x[ME,P2]",
   "x_4_2": "x[ME,P3]",
   "x_4_3": "x[ME,P4]",
   "x_4_4": "x[ME,P5]",
   "x_4_5": "x[ME,P6]",
   "x_5_0": "x[MF,P1]",
   "x_5_1": "x[MF,P2]",
   "x_5_2": "x[MF,P3]",
   "x_5_3": "x[MF,P4]",
   "x_5_4": "x[MF,P5]",
   "x_5_5": "x[MF,P6]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 9089.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     1.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     1.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     1.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     1.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     1.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-072": {
  "source_code_sha256": "1d869fc88abcfa18711057075f2880d78a3685769c550aaf3f87371e37b53cc5",
  "original_solution_sha256": "6b9d488653a612575c58e5e3941364fb5f47c7b7a1cc8f9649a50a0372ff8594",
  "bindings": [
   [
    "C0",
    "x[0]"
   ],
   [
    "C1",
    "x[1]"
   ],
   [
    "C2",
    "x[2]"
   ],
   [
    "C3",
    "x[3]"
   ],
   [
    "C4",
    "x[4]"
   ],
   [
    "C5",
    "x[5]"
   ],
   [
    "C6",
    "x[6]"
   ],
   [
    "C7",
    "x[7]"
   ],
   [
    "C8",
    "x[8]"
   ],
   [
    "C9",
    "x[9]"
   ],
   [
    "C10",
    "x[10]"
   ],
   [
    "C11",
    "x[11]"
   ],
   [
    "C12",
    "x[12]"
   ],
   [
    "C13",
    "x[13]"
   ],
   [
    "C14",
    "x[14]"
   ],
   [
    "C15",
    "x[15]"
   ],
   [
    "C16",
    "x[16]"
   ],
   [
    "C17",
    "x[17]"
   ],
   [
    "C18",
    "x[18]"
   ],
   [
    "C19",
    "x[19]"
   ],
   [
    "C20",
    "x[20]"
   ],
   [
    "C21",
    "x[21]"
   ],
   [
    "C22",
    "x[22]"
   ],
   [
    "C23",
    "x[23]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x[0]",
   "x[1]": "x[1]",
   "x[2]": "x[2]",
   "x[3]": "x[3]",
   "x[4]": "x[4]",
   "x[5]": "x[5]",
   "x[6]": "x[6]",
   "x[7]": "x[7]",
   "x[8]": "x[8]",
   "x[9]": "x[9]",
   "x[10]": "x[10]",
   "x[11]": "x[11]",
   "x[12]": "x[12]",
   "x[13]": "x[13]",
   "x[14]": "x[14]",
   "x[15]": "x[15]",
   "x[16]": "x[16]",
   "x[17]": "x[17]",
   "x[18]": "x[18]",
   "x[19]": "x[19]",
   "x[20]": "x[20]",
   "x[21]": "x[21]",
   "x[22]": "x[22]",
   "x[23]": "x[23]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 320.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     15.0
    ],
    [
     "C4",
     5.0
    ],
    [
     "C5",
     10.0
    ],
    [
     "C6",
     30.0
    ],
    [
     "C7",
     25.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     40.0
    ],
    [
     "C11",
     35.0
    ],
    [
     "C12",
     5.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     20.0
    ],
    [
     "C15",
     30.0
    ],
    [
     "C16",
     10.0
    ],
    [
     "C17",
     15.0
    ],
    [
     "C18",
     30.0
    ],
    [
     "C19",
     15.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     20.0
    ],
    [
     "C23",
     15.0
    ]
   ],
   "error": null
  }
 },
 "OR-076": {
  "source_code_sha256": "fc4b593ab526d2b41ba32fe42c4d06c6d16751f6c222fe4748a09f5c101c4c15",
  "original_solution_sha256": "e6aa3a71e03ae9a888785a19494feb3d9c0eafeb3531f4229213a9738efe876f",
  "bindings": [
   [
    "C0",
    "x[W1,C1]"
   ],
   [
    "C1",
    "x[W1,C2]"
   ],
   [
    "C2",
    "x[W1,C3]"
   ],
   [
    "C3",
    "x[W1,C4]"
   ],
   [
    "C4",
    "x[W1,C5]"
   ],
   [
    "C5",
    "x[W1,C6]"
   ],
   [
    "C6",
    "x[W1,C7]"
   ],
   [
    "C7",
    "x[W1,C8]"
   ],
   [
    "C8",
    "x[W1,C9]"
   ],
   [
    "C9",
    "x[W1,C10]"
   ],
   [
    "C10",
    "x[W1,C11]"
   ],
   [
    "C11",
    "x[W1,C12]"
   ],
   [
    "C12",
    "x[W1,C13]"
   ],
   [
    "C13",
    "x[W1,C14]"
   ],
   [
    "C14",
    "x[W1,C15]"
   ],
   [
    "C15",
    "x[W1,C16]"
   ],
   [
    "C16",
    "x[W1,C17]"
   ],
   [
    "C17",
    "x[W1,C18]"
   ],
   [
    "C18",
    "x[W1,C19]"
   ],
   [
    "C19",
    "x[W1,C20]"
   ],
   [
    "C20",
    "x[W2,C1]"
   ],
   [
    "C21",
    "x[W2,C2]"
   ],
   [
    "C22",
    "x[W2,C3]"
   ],
   [
    "C23",
    "x[W2,C4]"
   ],
   [
    "C24",
    "x[W2,C5]"
   ],
   [
    "C25",
    "x[W2,C6]"
   ],
   [
    "C26",
    "x[W2,C7]"
   ],
   [
    "C27",
    "x[W2,C8]"
   ],
   [
    "C28",
    "x[W2,C9]"
   ],
   [
    "C29",
    "x[W2,C10]"
   ],
   [
    "C30",
    "x[W2,C11]"
   ],
   [
    "C31",
    "x[W2,C12]"
   ],
   [
    "C32",
    "x[W2,C13]"
   ],
   [
    "C33",
    "x[W2,C14]"
   ],
   [
    "C34",
    "x[W2,C15]"
   ],
   [
    "C35",
    "x[W2,C16]"
   ],
   [
    "C36",
    "x[W2,C17]"
   ],
   [
    "C37",
    "x[W2,C18]"
   ],
   [
    "C38",
    "x[W2,C19]"
   ],
   [
    "C39",
    "x[W2,C20]"
   ],
   [
    "C40",
    "x[W3,C1]"
   ],
   [
    "C41",
    "x[W3,C2]"
   ],
   [
    "C42",
    "x[W3,C3]"
   ],
   [
    "C43",
    "x[W3,C4]"
   ],
   [
    "C44",
    "x[W3,C5]"
   ],
   [
    "C45",
    "x[W3,C6]"
   ],
   [
    "C46",
    "x[W3,C7]"
   ],
   [
    "C47",
    "x[W3,C8]"
   ],
   [
    "C48",
    "x[W3,C9]"
   ],
   [
    "C49",
    "x[W3,C10]"
   ],
   [
    "C50",
    "x[W3,C11]"
   ],
   [
    "C51",
    "x[W3,C12]"
   ],
   [
    "C52",
    "x[W3,C13]"
   ],
   [
    "C53",
    "x[W3,C14]"
   ],
   [
    "C54",
    "x[W3,C15]"
   ],
   [
    "C55",
    "x[W3,C16]"
   ],
   [
    "C56",
    "x[W3,C17]"
   ],
   [
    "C57",
    "x[W3,C18]"
   ],
   [
    "C58",
    "x[W3,C19]"
   ],
   [
    "C59",
    "x[W3,C20]"
   ],
   [
    "C60",
    "x[W4,C1]"
   ],
   [
    "C61",
    "x[W4,C2]"
   ],
   [
    "C62",
    "x[W4,C3]"
   ],
   [
    "C63",
    "x[W4,C4]"
   ],
   [
    "C64",
    "x[W4,C5]"
   ],
   [
    "C65",
    "x[W4,C6]"
   ],
   [
    "C66",
    "x[W4,C7]"
   ],
   [
    "C67",
    "x[W4,C8]"
   ],
   [
    "C68",
    "x[W4,C9]"
   ],
   [
    "C69",
    "x[W4,C10]"
   ],
   [
    "C70",
    "x[W4,C11]"
   ],
   [
    "C71",
    "x[W4,C12]"
   ],
   [
    "C72",
    "x[W4,C13]"
   ],
   [
    "C73",
    "x[W4,C14]"
   ],
   [
    "C74",
    "x[W4,C15]"
   ],
   [
    "C75",
    "x[W4,C16]"
   ],
   [
    "C76",
    "x[W4,C17]"
   ],
   [
    "C77",
    "x[W4,C18]"
   ],
   [
    "C78",
    "x[W4,C19]"
   ],
   [
    "C79",
    "x[W4,C20]"
   ],
   [
    "C80",
    "x[W5,C1]"
   ],
   [
    "C81",
    "x[W5,C2]"
   ],
   [
    "C82",
    "x[W5,C3]"
   ],
   [
    "C83",
    "x[W5,C4]"
   ],
   [
    "C84",
    "x[W5,C5]"
   ],
   [
    "C85",
    "x[W5,C6]"
   ],
   [
    "C86",
    "x[W5,C7]"
   ],
   [
    "C87",
    "x[W5,C8]"
   ],
   [
    "C88",
    "x[W5,C9]"
   ],
   [
    "C89",
    "x[W5,C10]"
   ],
   [
    "C90",
    "x[W5,C11]"
   ],
   [
    "C91",
    "x[W5,C12]"
   ],
   [
    "C92",
    "x[W5,C13]"
   ],
   [
    "C93",
    "x[W5,C14]"
   ],
   [
    "C94",
    "x[W5,C15]"
   ],
   [
    "C95",
    "x[W5,C16]"
   ],
   [
    "C96",
    "x[W5,C17]"
   ],
   [
    "C97",
    "x[W5,C18]"
   ],
   [
    "C98",
    "x[W5,C19]"
   ],
   [
    "C99",
    "x[W5,C20]"
   ],
   [
    "C100",
    "x[W6,C1]"
   ],
   [
    "C101",
    "x[W6,C2]"
   ],
   [
    "C102",
    "x[W6,C3]"
   ],
   [
    "C103",
    "x[W6,C4]"
   ],
   [
    "C104",
    "x[W6,C5]"
   ],
   [
    "C105",
    "x[W6,C6]"
   ],
   [
    "C106",
    "x[W6,C7]"
   ],
   [
    "C107",
    "x[W6,C8]"
   ],
   [
    "C108",
    "x[W6,C9]"
   ],
   [
    "C109",
    "x[W6,C10]"
   ],
   [
    "C110",
    "x[W6,C11]"
   ],
   [
    "C111",
    "x[W6,C12]"
   ],
   [
    "C112",
    "x[W6,C13]"
   ],
   [
    "C113",
    "x[W6,C14]"
   ],
   [
    "C114",
    "x[W6,C15]"
   ],
   [
    "C115",
    "x[W6,C16]"
   ],
   [
    "C116",
    "x[W6,C17]"
   ],
   [
    "C117",
    "x[W6,C18]"
   ],
   [
    "C118",
    "x[W6,C19]"
   ],
   [
    "C119",
    "x[W6,C20]"
   ],
   [
    "C120",
    "x[W7,C1]"
   ],
   [
    "C121",
    "x[W7,C2]"
   ],
   [
    "C122",
    "x[W7,C3]"
   ],
   [
    "C123",
    "x[W7,C4]"
   ],
   [
    "C124",
    "x[W7,C5]"
   ],
   [
    "C125",
    "x[W7,C6]"
   ],
   [
    "C126",
    "x[W7,C7]"
   ],
   [
    "C127",
    "x[W7,C8]"
   ],
   [
    "C128",
    "x[W7,C9]"
   ],
   [
    "C129",
    "x[W7,C10]"
   ],
   [
    "C130",
    "x[W7,C11]"
   ],
   [
    "C131",
    "x[W7,C12]"
   ],
   [
    "C132",
    "x[W7,C13]"
   ],
   [
    "C133",
    "x[W7,C14]"
   ],
   [
    "C134",
    "x[W7,C15]"
   ],
   [
    "C135",
    "x[W7,C16]"
   ],
   [
    "C136",
    "x[W7,C17]"
   ],
   [
    "C137",
    "x[W7,C18]"
   ],
   [
    "C138",
    "x[W7,C19]"
   ],
   [
    "C139",
    "x[W7,C20]"
   ],
   [
    "C140",
    "x[W8,C1]"
   ],
   [
    "C141",
    "x[W8,C2]"
   ],
   [
    "C142",
    "x[W8,C3]"
   ],
   [
    "C143",
    "x[W8,C4]"
   ],
   [
    "C144",
    "x[W8,C5]"
   ],
   [
    "C145",
    "x[W8,C6]"
   ],
   [
    "C146",
    "x[W8,C7]"
   ],
   [
    "C147",
    "x[W8,C8]"
   ],
   [
    "C148",
    "x[W8,C9]"
   ],
   [
    "C149",
    "x[W8,C10]"
   ],
   [
    "C150",
    "x[W8,C11]"
   ],
   [
    "C151",
    "x[W8,C12]"
   ],
   [
    "C152",
    "x[W8,C13]"
   ],
   [
    "C153",
    "x[W8,C14]"
   ],
   [
    "C154",
    "x[W8,C15]"
   ],
   [
    "C155",
    "x[W8,C16]"
   ],
   [
    "C156",
    "x[W8,C17]"
   ],
   [
    "C157",
    "x[W8,C18]"
   ],
   [
    "C158",
    "x[W8,C19]"
   ],
   [
    "C159",
    "x[W8,C20]"
   ],
   [
    "C160",
    "x[W9,C1]"
   ],
   [
    "C161",
    "x[W9,C2]"
   ],
   [
    "C162",
    "x[W9,C3]"
   ],
   [
    "C163",
    "x[W9,C4]"
   ],
   [
    "C164",
    "x[W9,C5]"
   ],
   [
    "C165",
    "x[W9,C6]"
   ],
   [
    "C166",
    "x[W9,C7]"
   ],
   [
    "C167",
    "x[W9,C8]"
   ],
   [
    "C168",
    "x[W9,C9]"
   ],
   [
    "C169",
    "x[W9,C10]"
   ],
   [
    "C170",
    "x[W9,C11]"
   ],
   [
    "C171",
    "x[W9,C12]"
   ],
   [
    "C172",
    "x[W9,C13]"
   ],
   [
    "C173",
    "x[W9,C14]"
   ],
   [
    "C174",
    "x[W9,C15]"
   ],
   [
    "C175",
    "x[W9,C16]"
   ],
   [
    "C176",
    "x[W9,C17]"
   ],
   [
    "C177",
    "x[W9,C18]"
   ],
   [
    "C178",
    "x[W9,C19]"
   ],
   [
    "C179",
    "x[W9,C20]"
   ],
   [
    "C180",
    "x[W10,C1]"
   ],
   [
    "C181",
    "x[W10,C2]"
   ],
   [
    "C182",
    "x[W10,C3]"
   ],
   [
    "C183",
    "x[W10,C4]"
   ],
   [
    "C184",
    "x[W10,C5]"
   ],
   [
    "C185",
    "x[W10,C6]"
   ],
   [
    "C186",
    "x[W10,C7]"
   ],
   [
    "C187",
    "x[W10,C8]"
   ],
   [
    "C188",
    "x[W10,C9]"
   ],
   [
    "C189",
    "x[W10,C10]"
   ],
   [
    "C190",
    "x[W10,C11]"
   ],
   [
    "C191",
    "x[W10,C12]"
   ],
   [
    "C192",
    "x[W10,C13]"
   ],
   [
    "C193",
    "x[W10,C14]"
   ],
   [
    "C194",
    "x[W10,C15]"
   ],
   [
    "C195",
    "x[W10,C16]"
   ],
   [
    "C196",
    "x[W10,C17]"
   ],
   [
    "C197",
    "x[W10,C18]"
   ],
   [
    "C198",
    "x[W10,C19]"
   ],
   [
    "C199",
    "x[W10,C20]"
   ],
   [
    "C200",
    "y[W1]"
   ],
   [
    "C201",
    "y[W2]"
   ],
   [
    "C202",
    "y[W3]"
   ],
   [
    "C203",
    "y[W4]"
   ],
   [
    "C204",
    "y[W5]"
   ],
   [
    "C205",
    "y[W6]"
   ],
   [
    "C206",
    "y[W7]"
   ],
   [
    "C207",
    "y[W8]"
   ],
   [
    "C208",
    "y[W9]"
   ],
   [
    "C209",
    "y[W10]"
   ]
  ],
  "primary_mapping": {
   "y[W1]": "y[W1]",
   "y[W2]": "y[W2]",
   "y[W3]": "y[W3]",
   "y[W4]": "y[W4]",
   "y[W5]": "y[W5]",
   "y[W6]": "y[W6]",
   "y[W7]": "y[W7]",
   "y[W8]": "y[W8]",
   "y[W9]": "y[W9]",
   "y[W10]": "y[W10]",
   "s[W1,C1]": "x[W1,C1]",
   "s[W1,C11]": "x[W1,C11]",
   "s[W1,C12]": "x[W1,C12]",
   "s[W1,C13]": "x[W1,C13]",
   "s[W1,C14]": "x[W1,C14]",
   "s[W1,C15]": "x[W1,C15]",
   "s[W1,C16]": "x[W1,C16]",
   "s[W1,C17]": "x[W1,C17]",
   "s[W1,C18]": "x[W1,C18]",
   "s[W1,C19]": "x[W1,C19]",
   "s[W1,C20]": "x[W1,C20]",
   "s[W2,C2]": "x[W2,C2]",
   "s[W2,C11]": "x[W2,C11]",
   "s[W2,C12]": "x[W2,C12]",
   "s[W2,C13]": "x[W2,C13]",
   "s[W2,C14]": "x[W2,C14]",
   "s[W2,C15]": "x[W2,C15]",
   "s[W2,C16]": "x[W2,C16]",
   "s[W2,C17]": "x[W2,C17]",
   "s[W2,C18]": "x[W2,C18]",
   "s[W2,C19]": "x[W2,C19]",
   "s[W2,C20]": "x[W2,C20]",
   "s[W3,C3]": "x[W3,C3]",
   "s[W3,C11]": "x[W3,C11]",
   "s[W3,C12]": "x[W3,C12]",
   "s[W3,C13]": "x[W3,C13]",
   "s[W3,C14]": "x[W3,C14]",
   "s[W3,C15]": "x[W3,C15]",
   "s[W3,C16]": "x[W3,C16]",
   "s[W3,C17]": "x[W3,C17]",
   "s[W3,C18]": "x[W3,C18]",
   "s[W3,C19]": "x[W3,C19]",
   "s[W3,C20]": "x[W3,C20]",
   "s[W4,C4]": "x[W4,C4]",
   "s[W4,C11]": "x[W4,C11]",
   "s[W4,C12]": "x[W4,C12]",
   "s[W4,C13]": "x[W4,C13]",
   "s[W4,C14]": "x[W4,C14]",
   "s[W4,C15]": "x[W4,C15]",
   "s[W4,C16]": "x[W4,C16]",
   "s[W4,C17]": "x[W4,C17]",
   "s[W4,C18]": "x[W4,C18]",
   "s[W4,C19]": "x[W4,C19]",
   "s[W4,C20]": "x[W4,C20]",
   "s[W5,C5]": "x[W5,C5]",
   "s[W5,C11]": "x[W5,C11]",
   "s[W5,C12]": "x[W5,C12]",
   "s[W5,C13]": "x[W5,C13]",
   "s[W5,C14]": "x[W5,C14]",
   "s[W5,C15]": "x[W5,C15]",
   "s[W5,C16]": "x[W5,C16]",
   "s[W5,C17]": "x[W5,C17]",
   "s[W5,C18]": "x[W5,C18]",
   "s[W5,C19]": "x[W5,C19]",
   "s[W5,C20]": "x[W5,C20]",
   "s[W6,C6]": "x[W6,C6]",
   "s[W6,C11]": "x[W6,C11]",
   "s[W6,C12]": "x[W6,C12]",
   "s[W6,C13]": "x[W6,C13]",
   "s[W6,C14]": "x[W6,C14]",
   "s[W6,C15]": "x[W6,C15]",
   "s[W6,C16]": "x[W6,C16]",
   "s[W6,C17]": "x[W6,C17]",
   "s[W6,C18]": "x[W6,C18]",
   "s[W6,C19]": "x[W6,C19]",
   "s[W6,C20]": "x[W6,C20]",
   "s[W7,C7]": "x[W7,C7]",
   "s[W7,C11]": "x[W7,C11]",
   "s[W7,C12]": "x[W7,C12]",
   "s[W7,C13]": "x[W7,C13]",
   "s[W7,C14]": "x[W7,C14]",
   "s[W7,C15]": "x[W7,C15]",
   "s[W7,C16]": "x[W7,C16]",
   "s[W7,C17]": "x[W7,C17]",
   "s[W7,C18]": "x[W7,C18]",
   "s[W7,C19]": "x[W7,C19]",
   "s[W7,C20]": "x[W7,C20]",
   "s[W8,C8]": "x[W8,C8]",
   "s[W8,C11]": "x[W8,C11]",
   "s[W8,C12]": "x[W8,C12]",
   "s[W8,C13]": "x[W8,C13]",
   "s[W8,C14]": "x[W8,C14]",
   "s[W8,C15]": "x[W8,C15]",
   "s[W8,C16]": "x[W8,C16]",
   "s[W8,C17]": "x[W8,C17]",
   "s[W8,C18]": "x[W8,C18]",
   "s[W8,C19]": "x[W8,C19]",
   "s[W8,C20]": "x[W8,C20]",
   "s[W9,C9]": "x[W9,C9]",
   "s[W9,C11]": "x[W9,C11]",
   "s[W9,C12]": "x[W9,C12]",
   "s[W9,C13]": "x[W9,C13]",
   "s[W9,C14]": "x[W9,C14]",
   "s[W9,C15]": "x[W9,C15]",
   "s[W9,C16]": "x[W9,C16]",
   "s[W9,C17]": "x[W9,C17]",
   "s[W9,C18]": "x[W9,C18]",
   "s[W9,C19]": "x[W9,C19]",
   "s[W9,C20]": "x[W9,C20]",
   "s[W10,C10]": "x[W10,C10]",
   "s[W10,C11]": "x[W10,C11]",
   "s[W10,C12]": "x[W10,C12]",
   "s[W10,C13]": "x[W10,C13]",
   "s[W10,C14]": "x[W10,C14]",
   "s[W10,C15]": "x[W10,C15]",
   "s[W10,C16]": "x[W10,C16]",
   "s[W10,C17]": "x[W10,C17]",
   "s[W10,C18]": "x[W10,C18]",
   "s[W10,C19]": "x[W10,C19]",
   "s[W10,C20]": "x[W10,C20]",
   "s[W1,C2]": "x[W1,C2]",
   "s[W1,C3]": "x[W1,C3]",
   "s[W1,C4]": "x[W1,C4]",
   "s[W1,C5]": "x[W1,C5]",
   "s[W1,C6]": "x[W1,C6]",
   "s[W1,C7]": "x[W1,C7]",
   "s[W1,C8]": "x[W1,C8]",
   "s[W1,C9]": "x[W1,C9]",
   "s[W1,C10]": "x[W1,C10]",
   "s[W2,C1]": "x[W2,C1]",
   "s[W2,C3]": "x[W2,C3]",
   "s[W2,C4]": "x[W2,C4]",
   "s[W2,C5]": "x[W2,C5]",
   "s[W2,C6]": "x[W2,C6]",
   "s[W2,C7]": "x[W2,C7]",
   "s[W2,C8]": "x[W2,C8]",
   "s[W2,C9]": "x[W2,C9]",
   "s[W2,C10]": "x[W2,C10]",
   "s[W3,C1]": "x[W3,C1]",
   "s[W3,C2]": "x[W3,C2]",
   "s[W3,C4]": "x[W3,C4]",
   "s[W3,C5]": "x[W3,C5]",
   "s[W3,C6]": "x[W3,C6]",
   "s[W3,C7]": "x[W3,C7]",
   "s[W3,C8]": "x[W3,C8]",
   "s[W3,C9]": "x[W3,C9]",
   "s[W3,C10]": "x[W3,C10]",
   "s[W4,C1]": "x[W4,C1]",
   "s[W4,C2]": "x[W4,C2]",
   "s[W4,C3]": "x[W4,C3]",
   "s[W4,C5]": "x[W4,C5]",
   "s[W4,C6]": "x[W4,C6]",
   "s[W4,C7]": "x[W4,C7]",
   "s[W4,C8]": "x[W4,C8]",
   "s[W4,C9]": "x[W4,C9]",
   "s[W4,C10]": "x[W4,C10]",
   "s[W5,C1]": "x[W5,C1]",
   "s[W5,C2]": "x[W5,C2]",
   "s[W5,C3]": "x[W5,C3]",
   "s[W5,C4]": "x[W5,C4]",
   "s[W5,C6]": "x[W5,C6]",
   "s[W5,C7]": "x[W5,C7]",
   "s[W5,C8]": "x[W5,C8]",
   "s[W5,C9]": "x[W5,C9]",
   "s[W5,C10]": "x[W5,C10]",
   "s[W6,C1]": "x[W6,C1]",
   "s[W6,C2]": "x[W6,C2]",
   "s[W6,C3]": "x[W6,C3]",
   "s[W6,C4]": "x[W6,C4]",
   "s[W6,C5]": "x[W6,C5]",
   "s[W6,C7]": "x[W6,C7]",
   "s[W6,C8]": "x[W6,C8]",
   "s[W6,C9]": "x[W6,C9]",
   "s[W6,C10]": "x[W6,C10]",
   "s[W7,C1]": "x[W7,C1]",
   "s[W7,C2]": "x[W7,C2]",
   "s[W7,C3]": "x[W7,C3]",
   "s[W7,C4]": "x[W7,C4]",
   "s[W7,C5]": "x[W7,C5]",
   "s[W7,C6]": "x[W7,C6]",
   "s[W7,C8]": "x[W7,C8]",
   "s[W7,C9]": "x[W7,C9]",
   "s[W7,C10]": "x[W7,C10]",
   "s[W8,C1]": "x[W8,C1]",
   "s[W8,C2]": "x[W8,C2]",
   "s[W8,C3]": "x[W8,C3]",
   "s[W8,C4]": "x[W8,C4]",
   "s[W8,C5]": "x[W8,C5]",
   "s[W8,C6]": "x[W8,C6]",
   "s[W8,C7]": "x[W8,C7]",
   "s[W8,C9]": "x[W8,C9]",
   "s[W8,C10]": "x[W8,C10]",
   "s[W9,C1]": "x[W9,C1]",
   "s[W9,C2]": "x[W9,C2]",
   "s[W9,C3]": "x[W9,C3]",
   "s[W9,C4]": "x[W9,C4]",
   "s[W9,C5]": "x[W9,C5]",
   "s[W9,C6]": "x[W9,C6]",
   "s[W9,C7]": "x[W9,C7]",
   "s[W9,C8]": "x[W9,C8]",
   "s[W9,C10]": "x[W9,C10]",
   "s[W10,C1]": "x[W10,C1]",
   "s[W10,C2]": "x[W10,C2]",
   "s[W10,C3]": "x[W10,C3]",
   "s[W10,C4]": "x[W10,C4]",
   "s[W10,C5]": "x[W10,C5]",
   "s[W10,C6]": "x[W10,C6]",
   "s[W10,C7]": "x[W10,C7]",
   "s[W10,C8]": "x[W10,C8]",
   "s[W10,C9]": "x[W10,C9]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 103090.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     80.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     920.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     950.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     130.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     350.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     800.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     870.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     330.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     450.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     30.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     320.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     600.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     270.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     400.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     650.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     800.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     480.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     520.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     750.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     350.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     500.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     340.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     780.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     700.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     50.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     550.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     1.0
    ],
    [
     "C201",
     1.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     1.0
    ],
    [
     "C204",
     1.0
    ],
    [
     "C205",
     1.0
    ],
    [
     "C206",
     1.0
    ],
    [
     "C207",
     1.0
    ],
    [
     "C208",
     1.0
    ],
    [
     "C209",
     1.0
    ]
   ],
   "error": null
  }
 },
 "OR-080": {
  "source_code_sha256": "ac0425de23764418946b70bb703b263c77235de3efef5b4afd955045bcc692e1",
  "original_solution_sha256": "034567411cd0e38794044a5d1b770339825cc2e8c8f80033558542d6e126d6d4",
  "bindings": [
   [
    "C0",
    "y[1,1]"
   ],
   [
    "C1",
    "y[1,2]"
   ],
   [
    "C2",
    "y[1,3]"
   ],
   [
    "C3",
    "y[1,4]"
   ],
   [
    "C4",
    "y[2,1]"
   ],
   [
    "C5",
    "y[2,2]"
   ],
   [
    "C6",
    "y[2,3]"
   ],
   [
    "C7",
    "y[2,4]"
   ],
   [
    "C8",
    "y[3,1]"
   ],
   [
    "C9",
    "y[3,2]"
   ],
   [
    "C10",
    "y[3,3]"
   ],
   [
    "C11",
    "y[3,4]"
   ],
   [
    "C12",
    "y[4,1]"
   ],
   [
    "C13",
    "y[4,2]"
   ],
   [
    "C14",
    "y[4,3]"
   ],
   [
    "C15",
    "y[4,4]"
   ],
   [
    "C16",
    "y[5,1]"
   ],
   [
    "C17",
    "y[5,2]"
   ],
   [
    "C18",
    "y[5,3]"
   ],
   [
    "C19",
    "y[5,4]"
   ],
   [
    "C20",
    "y[6,1]"
   ],
   [
    "C21",
    "y[6,2]"
   ],
   [
    "C22",
    "y[6,3]"
   ],
   [
    "C23",
    "y[6,4]"
   ],
   [
    "C24",
    "y[7,1]"
   ],
   [
    "C25",
    "y[7,2]"
   ],
   [
    "C26",
    "y[7,3]"
   ],
   [
    "C27",
    "y[7,4]"
   ],
   [
    "C28",
    "y[8,1]"
   ],
   [
    "C29",
    "y[8,2]"
   ],
   [
    "C30",
    "y[8,3]"
   ],
   [
    "C31",
    "y[8,4]"
   ],
   [
    "C32",
    "y[9,1]"
   ],
   [
    "C33",
    "y[9,2]"
   ],
   [
    "C34",
    "y[9,3]"
   ],
   [
    "C35",
    "y[9,4]"
   ],
   [
    "C36",
    "y[10,1]"
   ],
   [
    "C37",
    "y[10,2]"
   ],
   [
    "C38",
    "y[10,3]"
   ],
   [
    "C39",
    "y[10,4]"
   ],
   [
    "C40",
    "s[1,1]"
   ],
   [
    "C41",
    "s[1,2]"
   ],
   [
    "C42",
    "s[1,3]"
   ],
   [
    "C43",
    "s[1,4]"
   ],
   [
    "C44",
    "s[2,1]"
   ],
   [
    "C45",
    "s[2,2]"
   ],
   [
    "C46",
    "s[2,3]"
   ],
   [
    "C47",
    "s[2,4]"
   ],
   [
    "C48",
    "s[3,1]"
   ],
   [
    "C49",
    "s[3,2]"
   ],
   [
    "C50",
    "s[3,3]"
   ],
   [
    "C51",
    "s[3,4]"
   ],
   [
    "C52",
    "s[4,1]"
   ],
   [
    "C53",
    "s[4,2]"
   ],
   [
    "C54",
    "s[4,3]"
   ],
   [
    "C55",
    "s[4,4]"
   ],
   [
    "C56",
    "s[5,1]"
   ],
   [
    "C57",
    "s[5,2]"
   ],
   [
    "C58",
    "s[5,3]"
   ],
   [
    "C59",
    "s[5,4]"
   ],
   [
    "C60",
    "s[6,1]"
   ],
   [
    "C61",
    "s[6,2]"
   ],
   [
    "C62",
    "s[6,3]"
   ],
   [
    "C63",
    "s[6,4]"
   ],
   [
    "C64",
    "s[7,1]"
   ],
   [
    "C65",
    "s[7,2]"
   ],
   [
    "C66",
    "s[7,3]"
   ],
   [
    "C67",
    "s[7,4]"
   ],
   [
    "C68",
    "s[8,1]"
   ],
   [
    "C69",
    "s[8,2]"
   ],
   [
    "C70",
    "s[8,3]"
   ],
   [
    "C71",
    "s[8,4]"
   ],
   [
    "C72",
    "s[9,1]"
   ],
   [
    "C73",
    "s[9,2]"
   ],
   [
    "C74",
    "s[9,3]"
   ],
   [
    "C75",
    "s[9,4]"
   ],
   [
    "C76",
    "s[10,1]"
   ],
   [
    "C77",
    "s[10,2]"
   ],
   [
    "C78",
    "s[10,3]"
   ],
   [
    "C79",
    "s[10,4]"
   ],
   [
    "C80",
    "z[1,1]"
   ],
   [
    "C81",
    "z[1,2]"
   ],
   [
    "C82",
    "z[1,3]"
   ],
   [
    "C83",
    "z[1,4]"
   ],
   [
    "C84",
    "z[2,1]"
   ],
   [
    "C85",
    "z[2,2]"
   ],
   [
    "C86",
    "z[2,3]"
   ],
   [
    "C87",
    "z[2,4]"
   ],
   [
    "C88",
    "z[3,1]"
   ],
   [
    "C89",
    "z[3,2]"
   ],
   [
    "C90",
    "z[3,3]"
   ],
   [
    "C91",
    "z[3,4]"
   ],
   [
    "C92",
    "z[4,1]"
   ],
   [
    "C93",
    "z[4,2]"
   ],
   [
    "C94",
    "z[4,3]"
   ],
   [
    "C95",
    "z[4,4]"
   ],
   [
    "C96",
    "z[5,1]"
   ],
   [
    "C97",
    "z[5,2]"
   ],
   [
    "C98",
    "z[5,3]"
   ],
   [
    "C99",
    "z[5,4]"
   ],
   [
    "C100",
    "z[6,1]"
   ],
   [
    "C101",
    "z[6,2]"
   ],
   [
    "C102",
    "z[6,3]"
   ],
   [
    "C103",
    "z[6,4]"
   ],
   [
    "C104",
    "z[7,1]"
   ],
   [
    "C105",
    "z[7,2]"
   ],
   [
    "C106",
    "z[7,3]"
   ],
   [
    "C107",
    "z[7,4]"
   ],
   [
    "C108",
    "z[8,1]"
   ],
   [
    "C109",
    "z[8,2]"
   ],
   [
    "C110",
    "z[8,3]"
   ],
   [
    "C111",
    "z[8,4]"
   ],
   [
    "C112",
    "z[9,1]"
   ],
   [
    "C113",
    "z[9,2]"
   ],
   [
    "C114",
    "z[9,3]"
   ],
   [
    "C115",
    "z[9,4]"
   ],
   [
    "C116",
    "z[10,1]"
   ],
   [
    "C117",
    "z[10,2]"
   ],
   [
    "C118",
    "z[10,3]"
   ],
   [
    "C119",
    "z[10,4]"
   ],
   [
    "C120",
    "w[1,1]"
   ],
   [
    "C121",
    "w[1,2]"
   ],
   [
    "C122",
    "w[1,3]"
   ],
   [
    "C123",
    "w[1,4]"
   ],
   [
    "C124",
    "w[2,1]"
   ],
   [
    "C125",
    "w[2,2]"
   ],
   [
    "C126",
    "w[2,3]"
   ],
   [
    "C127",
    "w[2,4]"
   ],
   [
    "C128",
    "w[3,1]"
   ],
   [
    "C129",
    "w[3,2]"
   ],
   [
    "C130",
    "w[3,3]"
   ],
   [
    "C131",
    "w[3,4]"
   ],
   [
    "C132",
    "w[4,1]"
   ],
   [
    "C133",
    "w[4,2]"
   ],
   [
    "C134",
    "w[4,3]"
   ],
   [
    "C135",
    "w[4,4]"
   ],
   [
    "C136",
    "w[5,1]"
   ],
   [
    "C137",
    "w[5,2]"
   ],
   [
    "C138",
    "w[5,3]"
   ],
   [
    "C139",
    "w[5,4]"
   ],
   [
    "C140",
    "w[6,1]"
   ],
   [
    "C141",
    "w[6,2]"
   ],
   [
    "C142",
    "w[6,3]"
   ],
   [
    "C143",
    "w[6,4]"
   ],
   [
    "C144",
    "w[7,1]"
   ],
   [
    "C145",
    "w[7,2]"
   ],
   [
    "C146",
    "w[7,3]"
   ],
   [
    "C147",
    "w[7,4]"
   ],
   [
    "C148",
    "w[8,1]"
   ],
   [
    "C149",
    "w[8,2]"
   ],
   [
    "C150",
    "w[8,3]"
   ],
   [
    "C151",
    "w[8,4]"
   ],
   [
    "C152",
    "w[9,1]"
   ],
   [
    "C153",
    "w[9,2]"
   ],
   [
    "C154",
    "w[9,3]"
   ],
   [
    "C155",
    "w[9,4]"
   ],
   [
    "C156",
    "w[10,1]"
   ],
   [
    "C157",
    "w[10,2]"
   ],
   [
    "C158",
    "w[10,3]"
   ],
   [
    "C159",
    "w[10,4]"
   ]
  ],
  "primary_mapping": {
   "w[1,1]": "w[1,1]",
   "w[1,2]": "w[1,2]",
   "w[1,3]": "w[1,3]",
   "w[1,4]": "w[1,4]",
   "w[2,1]": "w[2,1]",
   "w[2,2]": "w[2,2]",
   "w[2,3]": "w[2,3]",
   "w[2,4]": "w[2,4]",
   "w[3,1]": "w[3,1]",
   "w[3,2]": "w[3,2]",
   "w[3,3]": "w[3,3]",
   "w[3,4]": "w[3,4]",
   "w[4,1]": "w[4,1]",
   "w[4,2]": "w[4,2]",
   "w[4,3]": "w[4,3]",
   "w[4,4]": "w[4,4]",
   "w[5,1]": "w[5,1]",
   "w[5,2]": "w[5,2]",
   "w[5,3]": "w[5,3]",
   "w[5,4]": "w[5,4]",
   "w[6,1]": "w[6,1]",
   "w[6,2]": "w[6,2]",
   "w[6,3]": "w[6,3]",
   "w[6,4]": "w[6,4]",
   "w[7,1]": "w[7,1]",
   "w[7,2]": "w[7,2]",
   "w[7,3]": "w[7,3]",
   "w[7,4]": "w[7,4]",
   "w[8,1]": "w[8,1]",
   "w[8,2]": "w[8,2]",
   "w[8,3]": "w[8,3]",
   "w[8,4]": "w[8,4]",
   "w[9,1]": "w[9,1]",
   "w[9,2]": "w[9,2]",
   "w[9,3]": "w[9,3]",
   "w[9,4]": "w[9,4]",
   "w[10,1]": "w[10,1]",
   "w[10,2]": "w[10,2]",
   "w[10,3]": "w[10,3]",
   "w[10,4]": "w[10,4]",
   "y[1,1]": "y[1,1]",
   "y[1,2]": "y[1,2]",
   "y[1,3]": "y[1,3]",
   "y[1,4]": "y[1,4]",
   "y[2,1]": "y[2,1]",
   "y[2,2]": "y[2,2]",
   "y[2,3]": "y[2,3]",
   "y[2,4]": "y[2,4]",
   "y[3,1]": "y[3,1]",
   "y[3,2]": "y[3,2]",
   "y[3,3]": "y[3,3]",
   "y[3,4]": "y[3,4]",
   "y[4,1]": "y[4,1]",
   "y[4,2]": "y[4,2]",
   "y[4,3]": "y[4,3]",
   "y[4,4]": "y[4,4]",
   "y[5,1]": "y[5,1]",
   "y[5,2]": "y[5,2]",
   "y[5,3]": "y[5,3]",
   "y[5,4]": "y[5,4]",
   "y[6,1]": "y[6,1]",
   "y[6,2]": "y[6,2]",
   "y[6,3]": "y[6,3]",
   "y[6,4]": "y[6,4]",
   "y[7,1]": "y[7,1]",
   "y[7,2]": "y[7,2]",
   "y[7,3]": "y[7,3]",
   "y[7,4]": "y[7,4]",
   "y[8,1]": "y[8,1]",
   "y[8,2]": "y[8,2]",
   "y[8,3]": "y[8,3]",
   "y[8,4]": "y[8,4]",
   "y[9,1]": "y[9,1]",
   "y[9,2]": "y[9,2]",
   "y[9,3]": "y[9,3]",
   "y[9,4]": "y[9,4]",
   "y[10,1]": "y[10,1]",
   "y[10,2]": "y[10,2]",
   "y[10,3]": "y[10,3]",
   "y[10,4]": "y[10,4]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 15590.0,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     1.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     1.0
    ],
    [
     "C17",
     1.0
    ],
    [
     "C18",
     1.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     1.0
    ],
    [
     "C21",
     1.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     1.0
    ],
    [
     "C25",
     1.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     1.0
    ],
    [
     "C33",
     1.0
    ],
    [
     "C34",
     1.0
    ],
    [
     "C35",
     1.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     1.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     1.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     1.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     1.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     1.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     1.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     1.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     1.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     300.0
    ],
    [
     "C121",
     600.0
    ],
    [
     "C122",
     900.0
    ],
    [
     "C123",
     700.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     300.0
    ],
    [
     "C137",
     600.0
    ],
    [
     "C138",
     300.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     300.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     300.0
    ],
    [
     "C145",
     200.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     300.0
    ],
    [
     "C153",
     600.0
    ],
    [
     "C154",
     600.0
    ],
    [
     "C155",
     300.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-084": {
  "source_code_sha256": "02a04ba0c9e1c483f424664d13e8ddcf9ca9e4ac4d2c57e79295947117aa0918",
  "original_solution_sha256": "7be32a3397b8383c1306bbf5537fed532a1ede2397d0542ea7fd35541b136212",
  "bindings": [],
  "primary_mapping": null,
  "mapping_basis": "unavailable",
  "mapping_failure": "generated_model_has_no_incumbent",
  "frozen_replay_solution": {
   "status": "ERROR",
   "objective": null,
   "variables": [],
   "error": "generated_model_has_no_incumbent"
  }
 },
 "OR-090": {
  "source_code_sha256": "bb8783308bb39b00258ca1c0e10bbc8babcf87e3de0be89a4943fc6b979b2976",
  "original_solution_sha256": "ad7a52f7c8b976e9fd9c714094d51478601f7c744693415e971d18c03bfe9c57",
  "bindings": [
   [
    "C0",
    "x[P1]"
   ],
   [
    "C1",
    "x[P2]"
   ],
   [
    "C2",
    "x[P3]"
   ],
   [
    "C3",
    "x[P4]"
   ],
   [
    "C4",
    "x[P5]"
   ],
   [
    "C5",
    "x[P6]"
   ],
   [
    "C6",
    "x[P7]"
   ],
   [
    "C7",
    "x[P8]"
   ],
   [
    "C8",
    "x[P9]"
   ],
   [
    "C9",
    "x[P10]"
   ],
   [
    "C10",
    "x[P11]"
   ],
   [
    "C11",
    "x[P12]"
   ],
   [
    "C12",
    "x[P13]"
   ],
   [
    "C13",
    "x[P14]"
   ],
   [
    "C14",
    "x[P15]"
   ],
   [
    "C15",
    "x[P16]"
   ],
   [
    "C16",
    "x[P17]"
   ],
   [
    "C17",
    "x[P18]"
   ],
   [
    "C18",
    "x[P19]"
   ],
   [
    "C19",
    "x[P20]"
   ],
   [
    "C20",
    "x[P21]"
   ],
   [
    "C21",
    "x[P22]"
   ],
   [
    "C22",
    "x[P23]"
   ],
   [
    "C23",
    "x[P24]"
   ],
   [
    "C24",
    "x[P25]"
   ],
   [
    "C25",
    "x[P26]"
   ],
   [
    "C26",
    "x[P27]"
   ],
   [
    "C27",
    "x[P28]"
   ],
   [
    "C28",
    "x[P29]"
   ],
   [
    "C29",
    "x[P30]"
   ],
   [
    "C30",
    "x[P31]"
   ],
   [
    "C31",
    "x[P32]"
   ],
   [
    "C32",
    "x[P33]"
   ],
   [
    "C33",
    "x[P34]"
   ],
   [
    "C34",
    "x[P35]"
   ],
   [
    "C35",
    "x[P36]"
   ],
   [
    "C36",
    "x[P37]"
   ],
   [
    "C37",
    "x[P38]"
   ],
   [
    "C38",
    "x[P39]"
   ],
   [
    "C39",
    "x[P40]"
   ],
   [
    "C40",
    "x[P41]"
   ],
   [
    "C41",
    "x[P42]"
   ],
   [
    "C42",
    "x[P43]"
   ],
   [
    "C43",
    "x[P44]"
   ],
   [
    "C44",
    "x[P45]"
   ],
   [
    "C45",
    "x[P46]"
   ],
   [
    "C46",
    "x[P47]"
   ],
   [
    "C47",
    "x[P48]"
   ],
   [
    "C48",
    "x[P49]"
   ],
   [
    "C49",
    "x[P50]"
   ],
   [
    "C50",
    "x[P51]"
   ],
   [
    "C51",
    "x[P52]"
   ],
   [
    "C52",
    "x[P53]"
   ],
   [
    "C53",
    "x[P54]"
   ],
   [
    "C54",
    "x[P55]"
   ],
   [
    "C55",
    "x[P56]"
   ],
   [
    "C56",
    "x[P57]"
   ],
   [
    "C57",
    "x[P58]"
   ],
   [
    "C58",
    "x[P59]"
   ],
   [
    "C59",
    "x[P60]"
   ],
   [
    "C60",
    "x[P61]"
   ],
   [
    "C61",
    "x[P62]"
   ],
   [
    "C62",
    "x[P63]"
   ],
   [
    "C63",
    "x[P64]"
   ],
   [
    "C64",
    "x[P65]"
   ],
   [
    "C65",
    "x[P66]"
   ],
   [
    "C66",
    "x[P67]"
   ],
   [
    "C67",
    "x[P68]"
   ],
   [
    "C68",
    "x[P69]"
   ],
   [
    "C69",
    "x[P70]"
   ],
   [
    "C70",
    "x[P71]"
   ],
   [
    "C71",
    "x[P72]"
   ],
   [
    "C72",
    "x[P73]"
   ],
   [
    "C73",
    "x[P74]"
   ],
   [
    "C74",
    "x[P75]"
   ],
   [
    "C75",
    "x[P76]"
   ],
   [
    "C76",
    "x[P77]"
   ],
   [
    "C77",
    "x[P78]"
   ],
   [
    "C78",
    "x[P79]"
   ],
   [
    "C79",
    "x[P80]"
   ],
   [
    "C80",
    "x[P81]"
   ],
   [
    "C81",
    "x[P82]"
   ],
   [
    "C82",
    "x[P83]"
   ],
   [
    "C83",
    "x[P84]"
   ],
   [
    "C84",
    "x[P85]"
   ],
   [
    "C85",
    "x[P86]"
   ],
   [
    "C86",
    "x[P87]"
   ],
   [
    "C87",
    "x[P88]"
   ],
   [
    "C88",
    "x[P89]"
   ],
   [
    "C89",
    "x[P90]"
   ],
   [
    "C90",
    "x[P91]"
   ],
   [
    "C91",
    "x[P92]"
   ],
   [
    "C92",
    "x[P93]"
   ],
   [
    "C93",
    "x[P94]"
   ],
   [
    "C94",
    "x[P95]"
   ],
   [
    "C95",
    "x[P96]"
   ],
   [
    "C96",
    "x[P97]"
   ],
   [
    "C97",
    "x[P98]"
   ],
   [
    "C98",
    "x[P99]"
   ],
   [
    "C99",
    "x[P100]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x[P1]",
   "x[2]": "x[P2]",
   "x[3]": "x[P3]",
   "x[4]": "x[P4]",
   "x[5]": "x[P5]",
   "x[6]": "x[P6]",
   "x[7]": "x[P7]",
   "x[8]": "x[P8]",
   "x[9]": "x[P9]",
   "x[10]": "x[P10]",
   "x[11]": "x[P11]",
   "x[12]": "x[P12]",
   "x[13]": "x[P13]",
   "x[14]": "x[P14]",
   "x[15]": "x[P15]",
   "x[16]": "x[P16]",
   "x[17]": "x[P17]",
   "x[18]": "x[P18]",
   "x[19]": "x[P19]",
   "x[20]": "x[P20]",
   "x[21]": "x[P21]",
   "x[22]": "x[P22]",
   "x[23]": "x[P23]",
   "x[24]": "x[P24]",
   "x[25]": "x[P25]",
   "x[26]": "x[P26]",
   "x[27]": "x[P27]",
   "x[28]": "x[P28]",
   "x[29]": "x[P29]",
   "x[30]": "x[P30]",
   "x[31]": "x[P31]",
   "x[32]": "x[P32]",
   "x[33]": "x[P33]",
   "x[34]": "x[P34]",
   "x[35]": "x[P35]",
   "x[36]": "x[P36]",
   "x[37]": "x[P37]",
   "x[38]": "x[P38]",
   "x[39]": "x[P39]",
   "x[40]": "x[P40]",
   "x[41]": "x[P41]",
   "x[42]": "x[P42]",
   "x[43]": "x[P43]",
   "x[44]": "x[P44]",
   "x[45]": "x[P45]",
   "x[46]": "x[P46]",
   "x[47]": "x[P47]",
   "x[48]": "x[P48]",
   "x[49]": "x[P49]",
   "x[50]": "x[P50]",
   "x[51]": "x[P51]",
   "x[52]": "x[P52]",
   "x[53]": "x[P53]",
   "x[54]": "x[P54]",
   "x[55]": "x[P55]",
   "x[56]": "x[P56]",
   "x[57]": "x[P57]",
   "x[58]": "x[P58]",
   "x[59]": "x[P59]",
   "x[60]": "x[P60]",
   "x[61]": "x[P61]",
   "x[62]": "x[P62]",
   "x[63]": "x[P63]",
   "x[64]": "x[P64]",
   "x[65]": "x[P65]",
   "x[66]": "x[P66]",
   "x[67]": "x[P67]",
   "x[68]": "x[P68]",
   "x[69]": "x[P69]",
   "x[70]": "x[P70]",
   "x[71]": "x[P71]",
   "x[72]": "x[P72]",
   "x[73]": "x[P73]",
   "x[74]": "x[P74]",
   "x[75]": "x[P75]",
   "x[76]": "x[P76]",
   "x[77]": "x[P77]",
   "x[78]": "x[P78]",
   "x[79]": "x[P79]",
   "x[80]": "x[P80]",
   "x[81]": "x[P81]",
   "x[82]": "x[P82]",
   "x[83]": "x[P83]",
   "x[84]": "x[P84]",
   "x[85]": "x[P85]",
   "x[86]": "x[P86]",
   "x[87]": "x[P87]",
   "x[88]": "x[P88]",
   "x[89]": "x[P89]",
   "x[90]": "x[P90]",
   "x[91]": "x[P91]",
   "x[92]": "x[P92]",
   "x[93]": "x[P93]",
   "x[94]": "x[P94]",
   "x[95]": "x[P95]",
   "x[96]": "x[P96]",
   "x[97]": "x[P97]",
   "x[98]": "x[P98]",
   "x[99]": "x[P99]",
   "x[100]": "x[P100]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 108661.00000000003,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     38.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     22.0
    ],
    [
     "C6",
     44.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     27.0
    ],
    [
     "C9",
     12.0
    ],
    [
     "C10",
     30.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     60.0
    ],
    [
     "C14",
     54.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     52.0
    ],
    [
     "C23",
     60.0
    ],
    [
     "C24",
     19.0
    ],
    [
     "C25",
     18.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     15.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     44.0
    ],
    [
     "C32",
     13.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     27.0
    ],
    [
     "C37",
     5.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     21.0
    ],
    [
     "C45",
     21.0
    ],
    [
     "C46",
     36.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     7.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     46.0
    ],
    [
     "C52",
     8.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     38.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     37.0
    ],
    [
     "C59",
     26.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     45.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     22.0
    ],
    [
     "C67",
     21.0
    ],
    [
     "C68",
     13.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     4.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     39.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     54.0
    ],
    [
     "C84",
     23.0
    ],
    [
     "C85",
     55.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     13.0
    ],
    [
     "C89",
     17.0
    ],
    [
     "C90",
     10.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     18.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     25.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     57.0
    ]
   ],
   "error": null
  }
 },
 "OR-093": {
  "source_code_sha256": "a3ea8e2719aeeebdd811e35f09fc7b8033bc2eba2342eb920312391a42339e7f",
  "original_solution_sha256": "818a5456a4f6112000373feaccd8f83c1a2d753c7bf0c41e145030a3fcb182a8",
  "bindings": [
   [
    "C0",
    "x[M1,A]"
   ],
   [
    "C1",
    "x[M1,B]"
   ],
   [
    "C2",
    "x[M1,C]"
   ],
   [
    "C3",
    "x[M1,D]"
   ],
   [
    "C4",
    "x[M1,E]"
   ],
   [
    "C5",
    "x[M1,F]"
   ],
   [
    "C6",
    "x[M1,G]"
   ],
   [
    "C7",
    "x[M1,H]"
   ],
   [
    "C8",
    "x[M1,I]"
   ],
   [
    "C9",
    "x[M1,J]"
   ],
   [
    "C10",
    "x[M1,K]"
   ],
   [
    "C11",
    "x[M1,L]"
   ],
   [
    "C12",
    "x[M2,A]"
   ],
   [
    "C13",
    "x[M2,B]"
   ],
   [
    "C14",
    "x[M2,C]"
   ],
   [
    "C15",
    "x[M2,D]"
   ],
   [
    "C16",
    "x[M2,E]"
   ],
   [
    "C17",
    "x[M2,F]"
   ],
   [
    "C18",
    "x[M2,G]"
   ],
   [
    "C19",
    "x[M2,H]"
   ],
   [
    "C20",
    "x[M2,I]"
   ],
   [
    "C21",
    "x[M2,J]"
   ],
   [
    "C22",
    "x[M2,K]"
   ],
   [
    "C23",
    "x[M2,L]"
   ],
   [
    "C24",
    "x[M3,A]"
   ],
   [
    "C25",
    "x[M3,B]"
   ],
   [
    "C26",
    "x[M3,C]"
   ],
   [
    "C27",
    "x[M3,D]"
   ],
   [
    "C28",
    "x[M3,E]"
   ],
   [
    "C29",
    "x[M3,F]"
   ],
   [
    "C30",
    "x[M3,G]"
   ],
   [
    "C31",
    "x[M3,H]"
   ],
   [
    "C32",
    "x[M3,I]"
   ],
   [
    "C33",
    "x[M3,J]"
   ],
   [
    "C34",
    "x[M3,K]"
   ],
   [
    "C35",
    "x[M3,L]"
   ],
   [
    "C36",
    "x[M4,A]"
   ],
   [
    "C37",
    "x[M4,B]"
   ],
   [
    "C38",
    "x[M4,C]"
   ],
   [
    "C39",
    "x[M4,D]"
   ],
   [
    "C40",
    "x[M4,E]"
   ],
   [
    "C41",
    "x[M4,F]"
   ],
   [
    "C42",
    "x[M4,G]"
   ],
   [
    "C43",
    "x[M4,H]"
   ],
   [
    "C44",
    "x[M4,I]"
   ],
   [
    "C45",
    "x[M4,J]"
   ],
   [
    "C46",
    "x[M4,K]"
   ],
   [
    "C47",
    "x[M4,L]"
   ],
   [
    "C48",
    "x[M5,A]"
   ],
   [
    "C49",
    "x[M5,B]"
   ],
   [
    "C50",
    "x[M5,C]"
   ],
   [
    "C51",
    "x[M5,D]"
   ],
   [
    "C52",
    "x[M5,E]"
   ],
   [
    "C53",
    "x[M5,F]"
   ],
   [
    "C54",
    "x[M5,G]"
   ],
   [
    "C55",
    "x[M5,H]"
   ],
   [
    "C56",
    "x[M5,I]"
   ],
   [
    "C57",
    "x[M5,J]"
   ],
   [
    "C58",
    "x[M5,K]"
   ],
   [
    "C59",
    "x[M5,L]"
   ],
   [
    "C60",
    "x[M6,A]"
   ],
   [
    "C61",
    "x[M6,B]"
   ],
   [
    "C62",
    "x[M6,C]"
   ],
   [
    "C63",
    "x[M6,D]"
   ],
   [
    "C64",
    "x[M6,E]"
   ],
   [
    "C65",
    "x[M6,F]"
   ],
   [
    "C66",
    "x[M6,G]"
   ],
   [
    "C67",
    "x[M6,H]"
   ],
   [
    "C68",
    "x[M6,I]"
   ],
   [
    "C69",
    "x[M6,J]"
   ],
   [
    "C70",
    "x[M6,K]"
   ],
   [
    "C71",
    "x[M6,L]"
   ],
   [
    "C72",
    "x[M7,A]"
   ],
   [
    "C73",
    "x[M7,B]"
   ],
   [
    "C74",
    "x[M7,C]"
   ],
   [
    "C75",
    "x[M7,D]"
   ],
   [
    "C76",
    "x[M7,E]"
   ],
   [
    "C77",
    "x[M7,F]"
   ],
   [
    "C78",
    "x[M7,G]"
   ],
   [
    "C79",
    "x[M7,H]"
   ],
   [
    "C80",
    "x[M7,I]"
   ],
   [
    "C81",
    "x[M7,J]"
   ],
   [
    "C82",
    "x[M7,K]"
   ],
   [
    "C83",
    "x[M7,L]"
   ],
   [
    "C84",
    "x[M8,A]"
   ],
   [
    "C85",
    "x[M8,B]"
   ],
   [
    "C86",
    "x[M8,C]"
   ],
   [
    "C87",
    "x[M8,D]"
   ],
   [
    "C88",
    "x[M8,E]"
   ],
   [
    "C89",
    "x[M8,F]"
   ],
   [
    "C90",
    "x[M8,G]"
   ],
   [
    "C91",
    "x[M8,H]"
   ],
   [
    "C92",
    "x[M8,I]"
   ],
   [
    "C93",
    "x[M8,J]"
   ],
   [
    "C94",
    "x[M8,K]"
   ],
   [
    "C95",
    "x[M8,L]"
   ],
   [
    "C96",
    "x[M9,A]"
   ],
   [
    "C97",
    "x[M9,B]"
   ],
   [
    "C98",
    "x[M9,C]"
   ],
   [
    "C99",
    "x[M9,D]"
   ],
   [
    "C100",
    "x[M9,E]"
   ],
   [
    "C101",
    "x[M9,F]"
   ],
   [
    "C102",
    "x[M9,G]"
   ],
   [
    "C103",
    "x[M9,H]"
   ],
   [
    "C104",
    "x[M9,I]"
   ],
   [
    "C105",
    "x[M9,J]"
   ],
   [
    "C106",
    "x[M9,K]"
   ],
   [
    "C107",
    "x[M9,L]"
   ],
   [
    "C108",
    "x[M10,A]"
   ],
   [
    "C109",
    "x[M10,B]"
   ],
   [
    "C110",
    "x[M10,C]"
   ],
   [
    "C111",
    "x[M10,D]"
   ],
   [
    "C112",
    "x[M10,E]"
   ],
   [
    "C113",
    "x[M10,F]"
   ],
   [
    "C114",
    "x[M10,G]"
   ],
   [
    "C115",
    "x[M10,H]"
   ],
   [
    "C116",
    "x[M10,I]"
   ],
   [
    "C117",
    "x[M10,J]"
   ],
   [
    "C118",
    "x[M10,K]"
   ],
   [
    "C119",
    "x[M10,L]"
   ],
   [
    "C120",
    "x[M11,A]"
   ],
   [
    "C121",
    "x[M11,B]"
   ],
   [
    "C122",
    "x[M11,C]"
   ],
   [
    "C123",
    "x[M11,D]"
   ],
   [
    "C124",
    "x[M11,E]"
   ],
   [
    "C125",
    "x[M11,F]"
   ],
   [
    "C126",
    "x[M11,G]"
   ],
   [
    "C127",
    "x[M11,H]"
   ],
   [
    "C128",
    "x[M11,I]"
   ],
   [
    "C129",
    "x[M11,J]"
   ],
   [
    "C130",
    "x[M11,K]"
   ],
   [
    "C131",
    "x[M11,L]"
   ],
   [
    "C132",
    "x[M12,A]"
   ],
   [
    "C133",
    "x[M12,B]"
   ],
   [
    "C134",
    "x[M12,C]"
   ],
   [
    "C135",
    "x[M12,D]"
   ],
   [
    "C136",
    "x[M12,E]"
   ],
   [
    "C137",
    "x[M12,F]"
   ],
   [
    "C138",
    "x[M12,G]"
   ],
   [
    "C139",
    "x[M12,H]"
   ],
   [
    "C140",
    "x[M12,I]"
   ],
   [
    "C141",
    "x[M12,J]"
   ],
   [
    "C142",
    "x[M12,K]"
   ],
   [
    "C143",
    "x[M12,L]"
   ]
  ],
  "primary_mapping": {
   "x[1,A]": "x[M1,A]",
   "x[1,B]": "x[M1,B]",
   "x[1,C]": "x[M1,C]",
   "x[1,D]": "x[M1,D]",
   "x[1,E]": "x[M1,E]",
   "x[1,F]": "x[M1,F]",
   "x[1,G]": "x[M1,G]",
   "x[1,H]": "x[M1,H]",
   "x[1,I]": "x[M1,I]",
   "x[1,J]": "x[M1,J]",
   "x[1,K]": "x[M1,K]",
   "x[1,L]": "x[M1,L]",
   "x[2,A]": "x[M2,A]",
   "x[2,B]": "x[M2,B]",
   "x[2,C]": "x[M2,C]",
   "x[2,D]": "x[M2,D]",
   "x[2,E]": "x[M2,E]",
   "x[2,F]": "x[M2,F]",
   "x[2,G]": "x[M2,G]",
   "x[2,H]": "x[M2,H]",
   "x[2,I]": "x[M2,I]",
   "x[2,J]": "x[M2,J]",
   "x[2,K]": "x[M2,K]",
   "x[2,L]": "x[M2,L]",
   "x[3,A]": "x[M3,A]",
   "x[3,B]": "x[M3,B]",
   "x[3,C]": "x[M3,C]",
   "x[3,D]": "x[M3,D]",
   "x[3,E]": "x[M3,E]",
   "x[3,F]": "x[M3,F]",
   "x[3,G]": "x[M3,G]",
   "x[3,H]": "x[M3,H]",
   "x[3,I]": "x[M3,I]",
   "x[3,J]": "x[M3,J]",
   "x[3,K]": "x[M3,K]",
   "x[3,L]": "x[M3,L]",
   "x[4,A]": "x[M4,A]",
   "x[4,B]": "x[M4,B]",
   "x[4,C]": "x[M4,C]",
   "x[4,D]": "x[M4,D]",
   "x[4,E]": "x[M4,E]",
   "x[4,F]": "x[M4,F]",
   "x[4,G]": "x[M4,G]",
   "x[4,H]": "x[M4,H]",
   "x[4,I]": "x[M4,I]",
   "x[4,J]": "x[M4,J]",
   "x[4,K]": "x[M4,K]",
   "x[4,L]": "x[M4,L]",
   "x[5,A]": "x[M5,A]",
   "x[5,B]": "x[M5,B]",
   "x[5,C]": "x[M5,C]",
   "x[5,D]": "x[M5,D]",
   "x[5,E]": "x[M5,E]",
   "x[5,F]": "x[M5,F]",
   "x[5,G]": "x[M5,G]",
   "x[5,H]": "x[M5,H]",
   "x[5,I]": "x[M5,I]",
   "x[5,J]": "x[M5,J]",
   "x[5,K]": "x[M5,K]",
   "x[5,L]": "x[M5,L]",
   "x[6,A]": "x[M6,A]",
   "x[6,B]": "x[M6,B]",
   "x[6,C]": "x[M6,C]",
   "x[6,D]": "x[M6,D]",
   "x[6,E]": "x[M6,E]",
   "x[6,F]": "x[M6,F]",
   "x[6,G]": "x[M6,G]",
   "x[6,H]": "x[M6,H]",
   "x[6,I]": "x[M6,I]",
   "x[6,J]": "x[M6,J]",
   "x[6,K]": "x[M6,K]",
   "x[6,L]": "x[M6,L]",
   "x[7,A]": "x[M7,A]",
   "x[7,B]": "x[M7,B]",
   "x[7,C]": "x[M7,C]",
   "x[7,D]": "x[M7,D]",
   "x[7,E]": "x[M7,E]",
   "x[7,F]": "x[M7,F]",
   "x[7,G]": "x[M7,G]",
   "x[7,H]": "x[M7,H]",
   "x[7,I]": "x[M7,I]",
   "x[7,J]": "x[M7,J]",
   "x[7,K]": "x[M7,K]",
   "x[7,L]": "x[M7,L]",
   "x[8,A]": "x[M8,A]",
   "x[8,B]": "x[M8,B]",
   "x[8,C]": "x[M8,C]",
   "x[8,D]": "x[M8,D]",
   "x[8,E]": "x[M8,E]",
   "x[8,F]": "x[M8,F]",
   "x[8,G]": "x[M8,G]",
   "x[8,H]": "x[M8,H]",
   "x[8,I]": "x[M8,I]",
   "x[8,J]": "x[M8,J]",
   "x[8,K]": "x[M8,K]",
   "x[8,L]": "x[M8,L]",
   "x[9,A]": "x[M9,A]",
   "x[9,B]": "x[M9,B]",
   "x[9,C]": "x[M9,C]",
   "x[9,D]": "x[M9,D]",
   "x[9,E]": "x[M9,E]",
   "x[9,F]": "x[M9,F]",
   "x[9,G]": "x[M9,G]",
   "x[9,H]": "x[M9,H]",
   "x[9,I]": "x[M9,I]",
   "x[9,J]": "x[M9,J]",
   "x[9,K]": "x[M9,K]",
   "x[9,L]": "x[M9,L]",
   "x[10,A]": "x[M10,A]",
   "x[10,B]": "x[M10,B]",
   "x[10,C]": "x[M10,C]",
   "x[10,D]": "x[M10,D]",
   "x[10,E]": "x[M10,E]",
   "x[10,F]": "x[M10,F]",
   "x[10,G]": "x[M10,G]",
   "x[10,H]": "x[M10,H]",
   "x[10,I]": "x[M10,I]",
   "x[10,J]": "x[M10,J]",
   "x[10,K]": "x[M10,K]",
   "x[10,L]": "x[M10,L]",
   "x[11,A]": "x[M11,A]",
   "x[11,B]": "x[M11,B]",
   "x[11,C]": "x[M11,C]",
   "x[11,D]": "x[M11,D]",
   "x[11,E]": "x[M11,E]",
   "x[11,F]": "x[M11,F]",
   "x[11,G]": "x[M11,G]",
   "x[11,H]": "x[M11,H]",
   "x[11,I]": "x[M11,I]",
   "x[11,J]": "x[M11,J]",
   "x[11,K]": "x[M11,K]",
   "x[11,L]": "x[M11,L]",
   "x[12,A]": "x[M12,A]",
   "x[12,B]": "x[M12,B]",
   "x[12,C]": "x[M12,C]",
   "x[12,D]": "x[M12,D]",
   "x[12,E]": "x[M12,E]",
   "x[12,F]": "x[M12,F]",
   "x[12,G]": "x[M12,G]",
   "x[12,H]": "x[M12,H]",
   "x[12,I]": "x[M12,I]",
   "x[12,J]": "x[M12,J]",
   "x[12,K]": "x[M12,K]",
   "x[12,L]": "x[M12,L]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1577.6,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     1.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     1.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     1.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     1.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     1.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     1.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     1.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     1.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     1.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     1.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     1.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     1.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-097": {
  "source_code_sha256": "08a4a513cf240e9ac8ad9784cbc692db1ead309c175a5bfe4f21f2c026f3155f",
  "original_solution_sha256": "57685888572686b98e1ac16e74ac60d1bd481d5d715e39ce7c16ced5a9ed2e87",
  "bindings": [
   [
    "C0",
    "x[Opt_1]"
   ],
   [
    "C1",
    "x[Opt_2]"
   ],
   [
    "C2",
    "x[Opt_3]"
   ],
   [
    "C3",
    "x[Opt_4]"
   ],
   [
    "C4",
    "x[Opt_5]"
   ],
   [
    "C5",
    "x[Opt_6]"
   ],
   [
    "C6",
    "x[Opt_7]"
   ],
   [
    "C7",
    "x[Opt_8]"
   ],
   [
    "C8",
    "x[Opt_9]"
   ],
   [
    "C9",
    "x[Opt_10]"
   ],
   [
    "C10",
    "x[Opt_11]"
   ],
   [
    "C11",
    "x[Opt_12]"
   ],
   [
    "C12",
    "x[Opt_13]"
   ],
   [
    "C13",
    "x[Opt_14]"
   ],
   [
    "C14",
    "x[Opt_15]"
   ],
   [
    "C15",
    "x[Opt_16]"
   ],
   [
    "C16",
    "x[Opt_17]"
   ],
   [
    "C17",
    "x[Opt_18]"
   ],
   [
    "C18",
    "x[Opt_19]"
   ],
   [
    "C19",
    "x[Opt_20]"
   ],
   [
    "C20",
    "x[Opt_21]"
   ],
   [
    "C21",
    "x[Opt_22]"
   ],
   [
    "C22",
    "x[Opt_23]"
   ],
   [
    "C23",
    "x[Opt_24]"
   ],
   [
    "C24",
    "x[Opt_25]"
   ],
   [
    "C25",
    "x[Opt_26]"
   ],
   [
    "C26",
    "x[Opt_27]"
   ],
   [
    "C27",
    "x[Opt_28]"
   ],
   [
    "C28",
    "x[Opt_29]"
   ],
   [
    "C29",
    "x[Opt_30]"
   ],
   [
    "C30",
    "x[Opt_31]"
   ],
   [
    "C31",
    "x[Opt_32]"
   ],
   [
    "C32",
    "x[Opt_33]"
   ],
   [
    "C33",
    "x[Opt_34]"
   ],
   [
    "C34",
    "x[Opt_35]"
   ],
   [
    "C35",
    "x[Opt_36]"
   ],
   [
    "C36",
    "x[Opt_37]"
   ],
   [
    "C37",
    "x[Opt_38]"
   ],
   [
    "C38",
    "x[Opt_39]"
   ],
   [
    "C39",
    "x[Opt_40]"
   ],
   [
    "C40",
    "x[Opt_41]"
   ],
   [
    "C41",
    "x[Opt_42]"
   ],
   [
    "C42",
    "x[Opt_43]"
   ],
   [
    "C43",
    "x[Opt_44]"
   ],
   [
    "C44",
    "x[Opt_45]"
   ],
   [
    "C45",
    "x[Opt_46]"
   ],
   [
    "C46",
    "x[Opt_47]"
   ],
   [
    "C47",
    "x[Opt_48]"
   ],
   [
    "C48",
    "x[Opt_49]"
   ],
   [
    "C49",
    "x[Opt_50]"
   ],
   [
    "C50",
    "x[Opt_51]"
   ],
   [
    "C51",
    "x[Opt_52]"
   ],
   [
    "C52",
    "x[Opt_53]"
   ],
   [
    "C53",
    "x[Opt_54]"
   ],
   [
    "C54",
    "x[Opt_55]"
   ],
   [
    "C55",
    "x[Opt_56]"
   ],
   [
    "C56",
    "x[Opt_57]"
   ],
   [
    "C57",
    "x[Opt_58]"
   ],
   [
    "C58",
    "x[Opt_59]"
   ],
   [
    "C59",
    "x[Opt_60]"
   ],
   [
    "C60",
    "x[Opt_61]"
   ],
   [
    "C61",
    "x[Opt_62]"
   ],
   [
    "C62",
    "x[Opt_63]"
   ],
   [
    "C63",
    "x[Opt_64]"
   ],
   [
    "C64",
    "x[Opt_65]"
   ],
   [
    "C65",
    "x[Opt_66]"
   ],
   [
    "C66",
    "x[Opt_67]"
   ],
   [
    "C67",
    "x[Opt_68]"
   ],
   [
    "C68",
    "x[Opt_69]"
   ],
   [
    "C69",
    "x[Opt_70]"
   ],
   [
    "C70",
    "x[Opt_71]"
   ],
   [
    "C71",
    "x[Opt_72]"
   ],
   [
    "C72",
    "x[Opt_73]"
   ],
   [
    "C73",
    "x[Opt_74]"
   ],
   [
    "C74",
    "x[Opt_75]"
   ],
   [
    "C75",
    "x[Opt_76]"
   ],
   [
    "C76",
    "x[Opt_77]"
   ],
   [
    "C77",
    "x[Opt_78]"
   ],
   [
    "C78",
    "x[Opt_79]"
   ],
   [
    "C79",
    "x[Opt_80]"
   ],
   [
    "C80",
    "x[Opt_81]"
   ],
   [
    "C81",
    "x[Opt_82]"
   ],
   [
    "C82",
    "x[Opt_83]"
   ],
   [
    "C83",
    "x[Opt_84]"
   ],
   [
    "C84",
    "x[Opt_85]"
   ],
   [
    "C85",
    "x[Opt_86]"
   ],
   [
    "C86",
    "x[Opt_87]"
   ],
   [
    "C87",
    "x[Opt_88]"
   ],
   [
    "C88",
    "x[Opt_89]"
   ],
   [
    "C89",
    "x[Opt_90]"
   ],
   [
    "C90",
    "x[Opt_91]"
   ],
   [
    "C91",
    "x[Opt_92]"
   ],
   [
    "C92",
    "x[Opt_93]"
   ],
   [
    "C93",
    "x[Opt_94]"
   ],
   [
    "C94",
    "x[Opt_95]"
   ],
   [
    "C95",
    "x[Opt_96]"
   ],
   [
    "C96",
    "x[Opt_97]"
   ],
   [
    "C97",
    "x[Opt_98]"
   ],
   [
    "C98",
    "x[Opt_99]"
   ],
   [
    "C99",
    "x[Opt_100]"
   ],
   [
    "C100",
    "x[Opt_101]"
   ],
   [
    "C101",
    "x[Opt_102]"
   ],
   [
    "C102",
    "x[Opt_103]"
   ],
   [
    "C103",
    "x[Opt_104]"
   ],
   [
    "C104",
    "x[Opt_105]"
   ],
   [
    "C105",
    "x[Opt_106]"
   ],
   [
    "C106",
    "x[Opt_107]"
   ],
   [
    "C107",
    "x[Opt_108]"
   ],
   [
    "C108",
    "x[Opt_109]"
   ],
   [
    "C109",
    "x[Opt_110]"
   ],
   [
    "C110",
    "x[Opt_111]"
   ],
   [
    "C111",
    "x[Opt_112]"
   ],
   [
    "C112",
    "x[Opt_113]"
   ],
   [
    "C113",
    "x[Opt_114]"
   ],
   [
    "C114",
    "x[Opt_115]"
   ],
   [
    "C115",
    "x[Opt_116]"
   ],
   [
    "C116",
    "x[Opt_117]"
   ],
   [
    "C117",
    "x[Opt_118]"
   ],
   [
    "C118",
    "x[Opt_119]"
   ],
   [
    "C119",
    "x[Opt_120]"
   ],
   [
    "C120",
    "y[Opt_1]"
   ],
   [
    "C121",
    "y[Opt_2]"
   ],
   [
    "C122",
    "y[Opt_3]"
   ],
   [
    "C123",
    "y[Opt_4]"
   ],
   [
    "C124",
    "y[Opt_5]"
   ],
   [
    "C125",
    "y[Opt_6]"
   ],
   [
    "C126",
    "y[Opt_7]"
   ],
   [
    "C127",
    "y[Opt_8]"
   ],
   [
    "C128",
    "y[Opt_9]"
   ],
   [
    "C129",
    "y[Opt_10]"
   ],
   [
    "C130",
    "y[Opt_11]"
   ],
   [
    "C131",
    "y[Opt_12]"
   ],
   [
    "C132",
    "y[Opt_13]"
   ],
   [
    "C133",
    "y[Opt_14]"
   ],
   [
    "C134",
    "y[Opt_15]"
   ],
   [
    "C135",
    "y[Opt_16]"
   ],
   [
    "C136",
    "y[Opt_17]"
   ],
   [
    "C137",
    "y[Opt_18]"
   ],
   [
    "C138",
    "y[Opt_19]"
   ],
   [
    "C139",
    "y[Opt_20]"
   ],
   [
    "C140",
    "y[Opt_21]"
   ],
   [
    "C141",
    "y[Opt_22]"
   ],
   [
    "C142",
    "y[Opt_23]"
   ],
   [
    "C143",
    "y[Opt_24]"
   ],
   [
    "C144",
    "y[Opt_25]"
   ],
   [
    "C145",
    "y[Opt_26]"
   ],
   [
    "C146",
    "y[Opt_27]"
   ],
   [
    "C147",
    "y[Opt_28]"
   ],
   [
    "C148",
    "y[Opt_29]"
   ],
   [
    "C149",
    "y[Opt_30]"
   ],
   [
    "C150",
    "y[Opt_31]"
   ],
   [
    "C151",
    "y[Opt_32]"
   ],
   [
    "C152",
    "y[Opt_33]"
   ],
   [
    "C153",
    "y[Opt_34]"
   ],
   [
    "C154",
    "y[Opt_35]"
   ],
   [
    "C155",
    "y[Opt_36]"
   ],
   [
    "C156",
    "y[Opt_37]"
   ],
   [
    "C157",
    "y[Opt_38]"
   ],
   [
    "C158",
    "y[Opt_39]"
   ],
   [
    "C159",
    "y[Opt_40]"
   ],
   [
    "C160",
    "y[Opt_41]"
   ],
   [
    "C161",
    "y[Opt_42]"
   ],
   [
    "C162",
    "y[Opt_43]"
   ],
   [
    "C163",
    "y[Opt_44]"
   ],
   [
    "C164",
    "y[Opt_45]"
   ],
   [
    "C165",
    "y[Opt_46]"
   ],
   [
    "C166",
    "y[Opt_47]"
   ],
   [
    "C167",
    "y[Opt_48]"
   ],
   [
    "C168",
    "y[Opt_49]"
   ],
   [
    "C169",
    "y[Opt_50]"
   ],
   [
    "C170",
    "y[Opt_51]"
   ],
   [
    "C171",
    "y[Opt_52]"
   ],
   [
    "C172",
    "y[Opt_53]"
   ],
   [
    "C173",
    "y[Opt_54]"
   ],
   [
    "C174",
    "y[Opt_55]"
   ],
   [
    "C175",
    "y[Opt_56]"
   ],
   [
    "C176",
    "y[Opt_57]"
   ],
   [
    "C177",
    "y[Opt_58]"
   ],
   [
    "C178",
    "y[Opt_59]"
   ],
   [
    "C179",
    "y[Opt_60]"
   ],
   [
    "C180",
    "y[Opt_61]"
   ],
   [
    "C181",
    "y[Opt_62]"
   ],
   [
    "C182",
    "y[Opt_63]"
   ],
   [
    "C183",
    "y[Opt_64]"
   ],
   [
    "C184",
    "y[Opt_65]"
   ],
   [
    "C185",
    "y[Opt_66]"
   ],
   [
    "C186",
    "y[Opt_67]"
   ],
   [
    "C187",
    "y[Opt_68]"
   ],
   [
    "C188",
    "y[Opt_69]"
   ],
   [
    "C189",
    "y[Opt_70]"
   ],
   [
    "C190",
    "y[Opt_71]"
   ],
   [
    "C191",
    "y[Opt_72]"
   ],
   [
    "C192",
    "y[Opt_73]"
   ],
   [
    "C193",
    "y[Opt_74]"
   ],
   [
    "C194",
    "y[Opt_75]"
   ],
   [
    "C195",
    "y[Opt_76]"
   ],
   [
    "C196",
    "y[Opt_77]"
   ],
   [
    "C197",
    "y[Opt_78]"
   ],
   [
    "C198",
    "y[Opt_79]"
   ],
   [
    "C199",
    "y[Opt_80]"
   ],
   [
    "C200",
    "y[Opt_81]"
   ],
   [
    "C201",
    "y[Opt_82]"
   ],
   [
    "C202",
    "y[Opt_83]"
   ],
   [
    "C203",
    "y[Opt_84]"
   ],
   [
    "C204",
    "y[Opt_85]"
   ],
   [
    "C205",
    "y[Opt_86]"
   ],
   [
    "C206",
    "y[Opt_87]"
   ],
   [
    "C207",
    "y[Opt_88]"
   ],
   [
    "C208",
    "y[Opt_89]"
   ],
   [
    "C209",
    "y[Opt_90]"
   ],
   [
    "C210",
    "y[Opt_91]"
   ],
   [
    "C211",
    "y[Opt_92]"
   ],
   [
    "C212",
    "y[Opt_93]"
   ],
   [
    "C213",
    "y[Opt_94]"
   ],
   [
    "C214",
    "y[Opt_95]"
   ],
   [
    "C215",
    "y[Opt_96]"
   ],
   [
    "C216",
    "y[Opt_97]"
   ],
   [
    "C217",
    "y[Opt_98]"
   ],
   [
    "C218",
    "y[Opt_99]"
   ],
   [
    "C219",
    "y[Opt_100]"
   ],
   [
    "C220",
    "y[Opt_101]"
   ],
   [
    "C221",
    "y[Opt_102]"
   ],
   [
    "C222",
    "y[Opt_103]"
   ],
   [
    "C223",
    "y[Opt_104]"
   ],
   [
    "C224",
    "y[Opt_105]"
   ],
   [
    "C225",
    "y[Opt_106]"
   ],
   [
    "C226",
    "y[Opt_107]"
   ],
   [
    "C227",
    "y[Opt_108]"
   ],
   [
    "C228",
    "y[Opt_109]"
   ],
   [
    "C229",
    "y[Opt_110]"
   ],
   [
    "C230",
    "y[Opt_111]"
   ],
   [
    "C231",
    "y[Opt_112]"
   ],
   [
    "C232",
    "y[Opt_113]"
   ],
   [
    "C233",
    "y[Opt_114]"
   ],
   [
    "C234",
    "y[Opt_115]"
   ],
   [
    "C235",
    "y[Opt_116]"
   ],
   [
    "C236",
    "y[Opt_117]"
   ],
   [
    "C237",
    "y[Opt_118]"
   ],
   [
    "C238",
    "y[Opt_119]"
   ],
   [
    "C239",
    "y[Opt_120]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x[Opt_1]",
   "x[2]": "x[Opt_2]",
   "x[3]": "x[Opt_3]",
   "x[4]": "x[Opt_4]",
   "x[5]": "x[Opt_5]",
   "x[6]": "x[Opt_6]",
   "x[7]": "x[Opt_7]",
   "x[8]": "x[Opt_8]",
   "x[9]": "x[Opt_9]",
   "x[10]": "x[Opt_10]",
   "x[11]": "x[Opt_11]",
   "x[12]": "x[Opt_12]",
   "x[13]": "x[Opt_13]",
   "x[14]": "x[Opt_14]",
   "x[15]": "x[Opt_15]",
   "x[16]": "x[Opt_16]",
   "x[17]": "x[Opt_17]",
   "x[18]": "x[Opt_18]",
   "x[19]": "x[Opt_19]",
   "x[20]": "x[Opt_20]",
   "x[21]": "x[Opt_21]",
   "x[22]": "x[Opt_22]",
   "x[23]": "x[Opt_23]",
   "x[24]": "x[Opt_24]",
   "x[25]": "x[Opt_25]",
   "x[26]": "x[Opt_26]",
   "x[27]": "x[Opt_27]",
   "x[28]": "x[Opt_28]",
   "x[29]": "x[Opt_29]",
   "x[30]": "x[Opt_30]",
   "x[31]": "x[Opt_31]",
   "x[32]": "x[Opt_32]",
   "x[33]": "x[Opt_33]",
   "x[34]": "x[Opt_34]",
   "x[35]": "x[Opt_35]",
   "x[36]": "x[Opt_36]",
   "x[37]": "x[Opt_37]",
   "x[38]": "x[Opt_38]",
   "x[39]": "x[Opt_39]",
   "x[40]": "x[Opt_40]",
   "x[41]": "x[Opt_41]",
   "x[42]": "x[Opt_42]",
   "x[43]": "x[Opt_43]",
   "x[44]": "x[Opt_44]",
   "x[45]": "x[Opt_45]",
   "x[46]": "x[Opt_46]",
   "x[47]": "x[Opt_47]",
   "x[48]": "x[Opt_48]",
   "x[49]": "x[Opt_49]",
   "x[50]": "x[Opt_50]",
   "x[51]": "x[Opt_51]",
   "x[52]": "x[Opt_52]",
   "x[53]": "x[Opt_53]",
   "x[54]": "x[Opt_54]",
   "x[55]": "x[Opt_55]",
   "x[56]": "x[Opt_56]",
   "x[57]": "x[Opt_57]",
   "x[58]": "x[Opt_58]",
   "x[59]": "x[Opt_59]",
   "x[60]": "x[Opt_60]",
   "x[61]": "x[Opt_61]",
   "x[62]": "x[Opt_62]",
   "x[63]": "x[Opt_63]",
   "x[64]": "x[Opt_64]",
   "x[65]": "x[Opt_65]",
   "x[66]": "x[Opt_66]",
   "x[67]": "x[Opt_67]",
   "x[68]": "x[Opt_68]",
   "x[69]": "x[Opt_69]",
   "x[70]": "x[Opt_70]",
   "x[71]": "x[Opt_71]",
   "x[72]": "x[Opt_72]",
   "x[73]": "x[Opt_73]",
   "x[74]": "x[Opt_74]",
   "x[75]": "x[Opt_75]",
   "x[76]": "x[Opt_76]",
   "x[77]": "x[Opt_77]",
   "x[78]": "x[Opt_78]",
   "x[79]": "x[Opt_79]",
   "x[80]": "x[Opt_80]",
   "x[81]": "x[Opt_81]",
   "x[82]": "x[Opt_82]",
   "x[83]": "x[Opt_83]",
   "x[84]": "x[Opt_84]",
   "x[85]": "x[Opt_85]",
   "x[86]": "x[Opt_86]",
   "x[87]": "x[Opt_87]",
   "x[88]": "x[Opt_88]",
   "x[89]": "x[Opt_89]",
   "x[90]": "x[Opt_90]",
   "x[91]": "x[Opt_91]",
   "x[92]": "x[Opt_92]",
   "x[93]": "x[Opt_93]",
   "x[94]": "x[Opt_94]",
   "x[95]": "x[Opt_95]",
   "x[96]": "x[Opt_96]",
   "x[97]": "x[Opt_97]",
   "x[98]": "x[Opt_98]",
   "x[99]": "x[Opt_99]",
   "x[100]": "x[Opt_100]",
   "x[101]": "x[Opt_101]",
   "x[102]": "x[Opt_102]",
   "x[103]": "x[Opt_103]",
   "x[104]": "x[Opt_104]",
   "x[105]": "x[Opt_105]",
   "x[106]": "x[Opt_106]",
   "x[107]": "x[Opt_107]",
   "x[108]": "x[Opt_108]",
   "x[109]": "x[Opt_109]",
   "x[110]": "x[Opt_110]",
   "x[111]": "x[Opt_111]",
   "x[112]": "x[Opt_112]",
   "x[113]": "x[Opt_113]",
   "x[114]": "x[Opt_114]",
   "x[115]": "x[Opt_115]",
   "x[116]": "x[Opt_116]",
   "x[117]": "x[Opt_117]",
   "x[118]": "x[Opt_118]",
   "x[119]": "x[Opt_119]",
   "x[120]": "x[Opt_120]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 6.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     -1.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     -0.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     1.0
    ],
    [
     "C204",
     0.0
    ],
    [
     "C205",
     0.0
    ],
    [
     "C206",
     0.0
    ],
    [
     "C207",
     0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     -0.0
    ],
    [
     "C211",
     0.0
    ],
    [
     "C212",
     0.0
    ],
    [
     "C213",
     0.0
    ],
    [
     "C214",
     0.0
    ],
    [
     "C215",
     0.0
    ],
    [
     "C216",
     0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     0.0
    ],
    [
     "C219",
     0.0
    ],
    [
     "C220",
     0.0
    ],
    [
     "C221",
     0.0
    ],
    [
     "C222",
     0.0
    ],
    [
     "C223",
     0.0
    ],
    [
     "C224",
     0.0
    ],
    [
     "C225",
     0.0
    ],
    [
     "C226",
     0.0
    ],
    [
     "C227",
     0.0
    ],
    [
     "C228",
     0.0
    ],
    [
     "C229",
     -0.0
    ],
    [
     "C230",
     -0.0
    ],
    [
     "C231",
     0.0
    ],
    [
     "C232",
     -0.0
    ],
    [
     "C233",
     -0.0
    ],
    [
     "C234",
     -0.0
    ],
    [
     "C235",
     0.0
    ],
    [
     "C236",
     0.0
    ],
    [
     "C237",
     0.0
    ],
    [
     "C238",
     0.0
    ],
    [
     "C239",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-101": {
  "source_code_sha256": "74580356c9bb1a8320ed601d6e47df1d1e81d7d0557e75804f88355da46d3faf",
  "original_solution_sha256": "98572fbfb29973e19062af14ef7d7c48402dd8d9e14ee2b3ec010f3a5c6dd182",
  "bindings": [
   [
    "C0",
    "x[P110]"
   ],
   [
    "C1",
    "x[P77]"
   ],
   [
    "C2",
    "x[P85]"
   ],
   [
    "C3",
    "x[P11]"
   ],
   [
    "C4",
    "x[P92]"
   ],
   [
    "C5",
    "x[P75]"
   ],
   [
    "C6",
    "x[P96]"
   ],
   [
    "C7",
    "x[P87]"
   ],
   [
    "C8",
    "x[P105]"
   ],
   [
    "C9",
    "x[P9]"
   ],
   [
    "C10",
    "x[P53]"
   ],
   [
    "C11",
    "x[P17]"
   ],
   [
    "C12",
    "x[P7]"
   ],
   [
    "C13",
    "x[P58]"
   ],
   [
    "C14",
    "x[P40]"
   ],
   [
    "C15",
    "x[P43]"
   ],
   [
    "C16",
    "x[P109]"
   ],
   [
    "C17",
    "x[P95]"
   ],
   [
    "C18",
    "x[P60]"
   ],
   [
    "C19",
    "x[P66]"
   ],
   [
    "C20",
    "x[P23]"
   ],
   [
    "C21",
    "x[P51]"
   ],
   [
    "C22",
    "x[P78]"
   ],
   [
    "C23",
    "x[P35]"
   ],
   [
    "C24",
    "x[P71]"
   ],
   [
    "C25",
    "x[P20]"
   ],
   [
    "C26",
    "x[P63]"
   ],
   [
    "C27",
    "x[P106]"
   ],
   [
    "C28",
    "x[P86]"
   ],
   [
    "C29",
    "x[P44]"
   ],
   [
    "C30",
    "x[P65]"
   ],
   [
    "C31",
    "x[P89]"
   ],
   [
    "C32",
    "x[P4]"
   ],
   [
    "C33",
    "x[P3]"
   ],
   [
    "C34",
    "x[P107]"
   ],
   [
    "C35",
    "x[P25]"
   ],
   [
    "C36",
    "x[P8]"
   ],
   [
    "C37",
    "x[P54]"
   ],
   [
    "C38",
    "x[P90]"
   ],
   [
    "C39",
    "x[P1]"
   ],
   [
    "C40",
    "x[P2]"
   ],
   [
    "C41",
    "x[P41]"
   ],
   [
    "C42",
    "x[P72]"
   ],
   [
    "C43",
    "x[P26]"
   ],
   [
    "C44",
    "x[P36]"
   ],
   [
    "C45",
    "x[P46]"
   ],
   [
    "C46",
    "x[P14]"
   ],
   [
    "C47",
    "x[P33]"
   ],
   [
    "C48",
    "x[P91]"
   ],
   [
    "C49",
    "x[P94]"
   ],
   [
    "C50",
    "x[P18]"
   ],
   [
    "C51",
    "x[P98]"
   ],
   [
    "C52",
    "x[P34]"
   ],
   [
    "C53",
    "x[P47]"
   ],
   [
    "C54",
    "x[P111]"
   ],
   [
    "C55",
    "x[P12]"
   ],
   [
    "C56",
    "x[P27]"
   ],
   [
    "C57",
    "x[P103]"
   ],
   [
    "C58",
    "x[P55]"
   ],
   [
    "C59",
    "x[P30]"
   ],
   [
    "C60",
    "x[P83]"
   ],
   [
    "C61",
    "x[P6]"
   ],
   [
    "C62",
    "x[P104]"
   ],
   [
    "C63",
    "x[P38]"
   ],
   [
    "C64",
    "x[P108]"
   ],
   [
    "C65",
    "x[P37]"
   ],
   [
    "C66",
    "x[P82]"
   ],
   [
    "C67",
    "x[P68]"
   ],
   [
    "C68",
    "x[P88]"
   ],
   [
    "C69",
    "x[P81]"
   ],
   [
    "C70",
    "x[P57]"
   ],
   [
    "C71",
    "x[P48]"
   ],
   [
    "C72",
    "x[P5]"
   ],
   [
    "C73",
    "x[P16]"
   ],
   [
    "C74",
    "x[P31]"
   ],
   [
    "C75",
    "x[P64]"
   ],
   [
    "C76",
    "x[P22]"
   ],
   [
    "C77",
    "x[P100]"
   ],
   [
    "C78",
    "x[P62]"
   ],
   [
    "C79",
    "x[P10]"
   ],
   [
    "C80",
    "x[P42]"
   ],
   [
    "C81",
    "x[P21]"
   ],
   [
    "C82",
    "x[P99]"
   ],
   [
    "C83",
    "x[P84]"
   ],
   [
    "C84",
    "x[P39]"
   ],
   [
    "C85",
    "x[P59]"
   ],
   [
    "C86",
    "x[P19]"
   ],
   [
    "C87",
    "x[P76]"
   ],
   [
    "C88",
    "x[P15]"
   ],
   [
    "C89",
    "x[P79]"
   ],
   [
    "C90",
    "x[P49]"
   ],
   [
    "C91",
    "x[P24]"
   ],
   [
    "C92",
    "x[P50]"
   ],
   [
    "C93",
    "x[P45]"
   ],
   [
    "C94",
    "x[P13]"
   ],
   [
    "C95",
    "x[P67]"
   ],
   [
    "C96",
    "x[P69]"
   ],
   [
    "C97",
    "x[P80]"
   ],
   [
    "C98",
    "x[P70]"
   ],
   [
    "C99",
    "x[P32]"
   ],
   [
    "C100",
    "x[P56]"
   ],
   [
    "C101",
    "x[P101]"
   ],
   [
    "C102",
    "x[P74]"
   ],
   [
    "C103",
    "x[P29]"
   ],
   [
    "C104",
    "x[P61]"
   ],
   [
    "C105",
    "x[P52]"
   ],
   [
    "C106",
    "x[P73]"
   ],
   [
    "C107",
    "x[P102]"
   ],
   [
    "C108",
    "x[P97]"
   ],
   [
    "C109",
    "x[P93]"
   ],
   [
    "C110",
    "x[P28]"
   ]
  ],
  "primary_mapping": {
   "x[P1]": "x[P1]",
   "x[P2]": "x[P2]",
   "x[P3]": "x[P3]",
   "x[P4]": "x[P4]",
   "x[P5]": "x[P5]",
   "x[P6]": "x[P6]",
   "x[P7]": "x[P7]",
   "x[P8]": "x[P8]",
   "x[P9]": "x[P9]",
   "x[P10]": "x[P10]",
   "x[P11]": "x[P11]",
   "x[P12]": "x[P12]",
   "x[P13]": "x[P13]",
   "x[P14]": "x[P14]",
   "x[P15]": "x[P15]",
   "x[P16]": "x[P16]",
   "x[P17]": "x[P17]",
   "x[P18]": "x[P18]",
   "x[P19]": "x[P19]",
   "x[P20]": "x[P20]",
   "x[P21]": "x[P21]",
   "x[P22]": "x[P22]",
   "x[P23]": "x[P23]",
   "x[P24]": "x[P24]",
   "x[P25]": "x[P25]",
   "x[P26]": "x[P26]",
   "x[P27]": "x[P27]",
   "x[P28]": "x[P28]",
   "x[P29]": "x[P29]",
   "x[P30]": "x[P30]",
   "x[P31]": "x[P31]",
   "x[P32]": "x[P32]",
   "x[P33]": "x[P33]",
   "x[P34]": "x[P34]",
   "x[P35]": "x[P35]",
   "x[P36]": "x[P36]",
   "x[P37]": "x[P37]",
   "x[P38]": "x[P38]",
   "x[P39]": "x[P39]",
   "x[P40]": "x[P40]",
   "x[P41]": "x[P41]",
   "x[P42]": "x[P42]",
   "x[P43]": "x[P43]",
   "x[P44]": "x[P44]",
   "x[P45]": "x[P45]",
   "x[P46]": "x[P46]",
   "x[P47]": "x[P47]",
   "x[P48]": "x[P48]",
   "x[P49]": "x[P49]",
   "x[P50]": "x[P50]",
   "x[P51]": "x[P51]",
   "x[P52]": "x[P52]",
   "x[P53]": "x[P53]",
   "x[P54]": "x[P54]",
   "x[P55]": "x[P55]",
   "x[P56]": "x[P56]",
   "x[P57]": "x[P57]",
   "x[P58]": "x[P58]",
   "x[P59]": "x[P59]",
   "x[P60]": "x[P60]",
   "x[P61]": "x[P61]",
   "x[P62]": "x[P62]",
   "x[P63]": "x[P63]",
   "x[P64]": "x[P64]",
   "x[P65]": "x[P65]",
   "x[P66]": "x[P66]",
   "x[P67]": "x[P67]",
   "x[P68]": "x[P68]",
   "x[P69]": "x[P69]",
   "x[P70]": "x[P70]",
   "x[P71]": "x[P71]",
   "x[P72]": "x[P72]",
   "x[P73]": "x[P73]",
   "x[P74]": "x[P74]",
   "x[P75]": "x[P75]",
   "x[P76]": "x[P76]",
   "x[P77]": "x[P77]",
   "x[P78]": "x[P78]",
   "x[P79]": "x[P79]",
   "x[P80]": "x[P80]",
   "x[P81]": "x[P81]",
   "x[P82]": "x[P82]",
   "x[P83]": "x[P83]",
   "x[P84]": "x[P84]",
   "x[P85]": "x[P85]",
   "x[P86]": "x[P86]",
   "x[P87]": "x[P87]",
   "x[P88]": "x[P88]",
   "x[P89]": "x[P89]",
   "x[P90]": "x[P90]",
   "x[P91]": "x[P91]",
   "x[P92]": "x[P92]",
   "x[P93]": "x[P93]",
   "x[P94]": "x[P94]",
   "x[P95]": "x[P95]",
   "x[P96]": "x[P96]",
   "x[P97]": "x[P97]",
   "x[P98]": "x[P98]",
   "x[P99]": "x[P99]",
   "x[P100]": "x[P100]",
   "x[P101]": "x[P101]",
   "x[P102]": "x[P102]",
   "x[P103]": "x[P103]",
   "x[P104]": "x[P104]",
   "x[P105]": "x[P105]",
   "x[P106]": "x[P106]",
   "x[P107]": "x[P107]",
   "x[P108]": "x[P108]",
   "x[P109]": "x[P109]",
   "x[P110]": "x[P110]",
   "x[P111]": "x[P111]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 25820.15355597092,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     25.016802752471644
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     140.18461148172509
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     111.26976112384764
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     134.45522411545736
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     24.593054233545274
    ],
    [
     "C72",
     16.254369283401978
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     154.66606491755184
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-004": {
  "source_code_sha256": "9f2414e5aad98743b7cf1f3dc474fa9341ca0b9110be94d15b78f594614d82f2",
  "original_solution_sha256": "20110487759e2a0a3d08353146d463819fdb172585d2f4e100dfea0f68931a3d",
  "bindings": [
   [
    "C0",
    "x[S1,C1]"
   ],
   [
    "C1",
    "x[S1,C10]"
   ],
   [
    "C2",
    "x[S1,C11]"
   ],
   [
    "C3",
    "x[S1,C12]"
   ],
   [
    "C4",
    "x[S1,C2]"
   ],
   [
    "C5",
    "x[S1,C3]"
   ],
   [
    "C6",
    "x[S1,C4]"
   ],
   [
    "C7",
    "x[S1,C5]"
   ],
   [
    "C8",
    "x[S1,C6]"
   ],
   [
    "C9",
    "x[S1,C7]"
   ],
   [
    "C10",
    "x[S1,C8]"
   ],
   [
    "C11",
    "x[S1,C9]"
   ],
   [
    "C12",
    "x[S2,C1]"
   ],
   [
    "C13",
    "x[S2,C10]"
   ],
   [
    "C14",
    "x[S2,C11]"
   ],
   [
    "C15",
    "x[S2,C12]"
   ],
   [
    "C16",
    "x[S2,C2]"
   ],
   [
    "C17",
    "x[S2,C3]"
   ],
   [
    "C18",
    "x[S2,C4]"
   ],
   [
    "C19",
    "x[S2,C5]"
   ],
   [
    "C20",
    "x[S2,C6]"
   ],
   [
    "C21",
    "x[S2,C7]"
   ],
   [
    "C22",
    "x[S2,C8]"
   ],
   [
    "C23",
    "x[S2,C9]"
   ],
   [
    "C24",
    "x[S3,C1]"
   ],
   [
    "C25",
    "x[S3,C10]"
   ],
   [
    "C26",
    "x[S3,C11]"
   ],
   [
    "C27",
    "x[S3,C12]"
   ],
   [
    "C28",
    "x[S3,C2]"
   ],
   [
    "C29",
    "x[S3,C3]"
   ],
   [
    "C30",
    "x[S3,C4]"
   ],
   [
    "C31",
    "x[S3,C5]"
   ],
   [
    "C32",
    "x[S3,C6]"
   ],
   [
    "C33",
    "x[S3,C7]"
   ],
   [
    "C34",
    "x[S3,C8]"
   ],
   [
    "C35",
    "x[S3,C9]"
   ],
   [
    "C36",
    "x[S4,C1]"
   ],
   [
    "C37",
    "x[S4,C10]"
   ],
   [
    "C38",
    "x[S4,C11]"
   ],
   [
    "C39",
    "x[S4,C12]"
   ],
   [
    "C40",
    "x[S4,C2]"
   ],
   [
    "C41",
    "x[S4,C3]"
   ],
   [
    "C42",
    "x[S4,C4]"
   ],
   [
    "C43",
    "x[S4,C5]"
   ],
   [
    "C44",
    "x[S4,C6]"
   ],
   [
    "C45",
    "x[S4,C7]"
   ],
   [
    "C46",
    "x[S4,C8]"
   ],
   [
    "C47",
    "x[S4,C9]"
   ],
   [
    "C48",
    "x[S5,C1]"
   ],
   [
    "C49",
    "x[S5,C10]"
   ],
   [
    "C50",
    "x[S5,C11]"
   ],
   [
    "C51",
    "x[S5,C12]"
   ],
   [
    "C52",
    "x[S5,C2]"
   ],
   [
    "C53",
    "x[S5,C3]"
   ],
   [
    "C54",
    "x[S5,C4]"
   ],
   [
    "C55",
    "x[S5,C5]"
   ],
   [
    "C56",
    "x[S5,C6]"
   ],
   [
    "C57",
    "x[S5,C7]"
   ],
   [
    "C58",
    "x[S5,C8]"
   ],
   [
    "C59",
    "x[S5,C9]"
   ],
   [
    "C60",
    "x[S6,C1]"
   ],
   [
    "C61",
    "x[S6,C10]"
   ],
   [
    "C62",
    "x[S6,C11]"
   ],
   [
    "C63",
    "x[S6,C12]"
   ],
   [
    "C64",
    "x[S6,C2]"
   ],
   [
    "C65",
    "x[S6,C3]"
   ],
   [
    "C66",
    "x[S6,C4]"
   ],
   [
    "C67",
    "x[S6,C5]"
   ],
   [
    "C68",
    "x[S6,C6]"
   ],
   [
    "C69",
    "x[S6,C7]"
   ],
   [
    "C70",
    "x[S6,C8]"
   ],
   [
    "C71",
    "x[S6,C9]"
   ],
   [
    "C72",
    "x[S7,C1]"
   ],
   [
    "C73",
    "x[S7,C10]"
   ],
   [
    "C74",
    "x[S7,C11]"
   ],
   [
    "C75",
    "x[S7,C12]"
   ],
   [
    "C76",
    "x[S7,C2]"
   ],
   [
    "C77",
    "x[S7,C3]"
   ],
   [
    "C78",
    "x[S7,C4]"
   ],
   [
    "C79",
    "x[S7,C5]"
   ],
   [
    "C80",
    "x[S7,C6]"
   ],
   [
    "C81",
    "x[S7,C7]"
   ],
   [
    "C82",
    "x[S7,C8]"
   ],
   [
    "C83",
    "x[S7,C9]"
   ],
   [
    "C84",
    "x[S8,C1]"
   ],
   [
    "C85",
    "x[S8,C10]"
   ],
   [
    "C86",
    "x[S8,C11]"
   ],
   [
    "C87",
    "x[S8,C12]"
   ],
   [
    "C88",
    "x[S8,C2]"
   ],
   [
    "C89",
    "x[S8,C3]"
   ],
   [
    "C90",
    "x[S8,C4]"
   ],
   [
    "C91",
    "x[S8,C5]"
   ],
   [
    "C92",
    "x[S8,C6]"
   ],
   [
    "C93",
    "x[S8,C7]"
   ],
   [
    "C94",
    "x[S8,C8]"
   ],
   [
    "C95",
    "x[S8,C9]"
   ],
   [
    "C96",
    "x[S9,C1]"
   ],
   [
    "C97",
    "x[S9,C10]"
   ],
   [
    "C98",
    "x[S9,C11]"
   ],
   [
    "C99",
    "x[S9,C12]"
   ],
   [
    "C100",
    "x[S9,C2]"
   ],
   [
    "C101",
    "x[S9,C3]"
   ],
   [
    "C102",
    "x[S9,C4]"
   ],
   [
    "C103",
    "x[S9,C5]"
   ],
   [
    "C104",
    "x[S9,C6]"
   ],
   [
    "C105",
    "x[S9,C7]"
   ],
   [
    "C106",
    "x[S9,C8]"
   ],
   [
    "C107",
    "x[S9,C9]"
   ],
   [
    "C108",
    "x[S10,C1]"
   ],
   [
    "C109",
    "x[S10,C10]"
   ],
   [
    "C110",
    "x[S10,C11]"
   ],
   [
    "C111",
    "x[S10,C12]"
   ],
   [
    "C112",
    "x[S10,C2]"
   ],
   [
    "C113",
    "x[S10,C3]"
   ],
   [
    "C114",
    "x[S10,C4]"
   ],
   [
    "C115",
    "x[S10,C5]"
   ],
   [
    "C116",
    "x[S10,C6]"
   ],
   [
    "C117",
    "x[S10,C7]"
   ],
   [
    "C118",
    "x[S10,C8]"
   ],
   [
    "C119",
    "x[S10,C9]"
   ],
   [
    "C120",
    "x[S11,C1]"
   ],
   [
    "C121",
    "x[S11,C10]"
   ],
   [
    "C122",
    "x[S11,C11]"
   ],
   [
    "C123",
    "x[S11,C12]"
   ],
   [
    "C124",
    "x[S11,C2]"
   ],
   [
    "C125",
    "x[S11,C3]"
   ],
   [
    "C126",
    "x[S11,C4]"
   ],
   [
    "C127",
    "x[S11,C5]"
   ],
   [
    "C128",
    "x[S11,C6]"
   ],
   [
    "C129",
    "x[S11,C7]"
   ],
   [
    "C130",
    "x[S11,C8]"
   ],
   [
    "C131",
    "x[S11,C9]"
   ],
   [
    "C132",
    "x[S12,C1]"
   ],
   [
    "C133",
    "x[S12,C10]"
   ],
   [
    "C134",
    "x[S12,C11]"
   ],
   [
    "C135",
    "x[S12,C12]"
   ],
   [
    "C136",
    "x[S12,C2]"
   ],
   [
    "C137",
    "x[S12,C3]"
   ],
   [
    "C138",
    "x[S12,C4]"
   ],
   [
    "C139",
    "x[S12,C5]"
   ],
   [
    "C140",
    "x[S12,C6]"
   ],
   [
    "C141",
    "x[S12,C7]"
   ],
   [
    "C142",
    "x[S12,C8]"
   ],
   [
    "C143",
    "x[S12,C9]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_0_10": "x[S1,C11]",
   "x_0_11": "x[S1,C12]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_1_10": "x[S2,C11]",
   "x_1_11": "x[S2,C12]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_2_10": "x[S3,C11]",
   "x_2_11": "x[S3,C12]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_3_10": "x[S4,C11]",
   "x_3_11": "x[S4,C12]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_4_10": "x[S5,C11]",
   "x_4_11": "x[S5,C12]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_5_10": "x[S6,C11]",
   "x_5_11": "x[S6,C12]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_8": "x[S7,C9]",
   "x_6_9": "x[S7,C10]",
   "x_6_10": "x[S7,C11]",
   "x_6_11": "x[S7,C12]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_7_10": "x[S8,C11]",
   "x_7_11": "x[S8,C12]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_8_10": "x[S9,C11]",
   "x_8_11": "x[S9,C12]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]",
   "x_9_10": "x[S10,C11]",
   "x_9_11": "x[S10,C12]",
   "x_10_0": "x[S11,C1]",
   "x_10_1": "x[S11,C2]",
   "x_10_2": "x[S11,C3]",
   "x_10_3": "x[S11,C4]",
   "x_10_4": "x[S11,C5]",
   "x_10_5": "x[S11,C6]",
   "x_10_6": "x[S11,C7]",
   "x_10_7": "x[S11,C8]",
   "x_10_8": "x[S11,C9]",
   "x_10_9": "x[S11,C10]",
   "x_10_10": "x[S11,C11]",
   "x_10_11": "x[S11,C12]",
   "x_11_0": "x[S12,C1]",
   "x_11_1": "x[S12,C2]",
   "x_11_2": "x[S12,C3]",
   "x_11_3": "x[S12,C4]",
   "x_11_4": "x[S12,C5]",
   "x_11_5": "x[S12,C6]",
   "x_11_6": "x[S12,C7]",
   "x_11_7": "x[S12,C8]",
   "x_11_8": "x[S12,C9]",
   "x_11_9": "x[S12,C10]",
   "x_11_10": "x[S12,C11]",
   "x_11_11": "x[S12,C12]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 913520.1122932937,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     58.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     32.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     2359.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     392.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     2689.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     4.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     47.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     31.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     32.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     76.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     52.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     6.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     103.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     1426.0
    ],
    [
     "C129",
     928.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     1129.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     676.0
    ],
    [
     "C134",
     272.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-005": {
  "source_code_sha256": "1ae36420c2a2e1f96b0096a6956d03ff9b6c13908f4087119594f8a5daff16aa",
  "original_solution_sha256": "dd53c40b1b8d42e678364e1fa23f09ba5130b24aab7c6ed9de2d376ea833b8d7",
  "bindings": [
   [
    "C0",
    "x[supplier1,demand1]"
   ],
   [
    "C1",
    "x[supplier1,demand2]"
   ],
   [
    "C2",
    "x[supplier1,demand3]"
   ],
   [
    "C3",
    "x[supplier1,demand4]"
   ],
   [
    "C4",
    "x[supplier1,demand5]"
   ],
   [
    "C5",
    "x[supplier1,demand6]"
   ],
   [
    "C6",
    "x[supplier1,demand7]"
   ],
   [
    "C7",
    "x[supplier1,demand8]"
   ],
   [
    "C8",
    "x[supplier2,demand1]"
   ],
   [
    "C9",
    "x[supplier2,demand2]"
   ],
   [
    "C10",
    "x[supplier2,demand3]"
   ],
   [
    "C11",
    "x[supplier2,demand4]"
   ],
   [
    "C12",
    "x[supplier2,demand5]"
   ],
   [
    "C13",
    "x[supplier2,demand6]"
   ],
   [
    "C14",
    "x[supplier2,demand7]"
   ],
   [
    "C15",
    "x[supplier2,demand8]"
   ],
   [
    "C16",
    "x[supplier3,demand1]"
   ],
   [
    "C17",
    "x[supplier3,demand2]"
   ],
   [
    "C18",
    "x[supplier3,demand3]"
   ],
   [
    "C19",
    "x[supplier3,demand4]"
   ],
   [
    "C20",
    "x[supplier3,demand5]"
   ],
   [
    "C21",
    "x[supplier3,demand6]"
   ],
   [
    "C22",
    "x[supplier3,demand7]"
   ],
   [
    "C23",
    "x[supplier3,demand8]"
   ],
   [
    "C24",
    "x[supplier4,demand1]"
   ],
   [
    "C25",
    "x[supplier4,demand2]"
   ],
   [
    "C26",
    "x[supplier4,demand3]"
   ],
   [
    "C27",
    "x[supplier4,demand4]"
   ],
   [
    "C28",
    "x[supplier4,demand5]"
   ],
   [
    "C29",
    "x[supplier4,demand6]"
   ],
   [
    "C30",
    "x[supplier4,demand7]"
   ],
   [
    "C31",
    "x[supplier4,demand8]"
   ],
   [
    "C32",
    "x[supplier5,demand1]"
   ],
   [
    "C33",
    "x[supplier5,demand2]"
   ],
   [
    "C34",
    "x[supplier5,demand3]"
   ],
   [
    "C35",
    "x[supplier5,demand4]"
   ],
   [
    "C36",
    "x[supplier5,demand5]"
   ],
   [
    "C37",
    "x[supplier5,demand6]"
   ],
   [
    "C38",
    "x[supplier5,demand7]"
   ],
   [
    "C39",
    "x[supplier5,demand8]"
   ],
   [
    "C40",
    "x[supplier6,demand1]"
   ],
   [
    "C41",
    "x[supplier6,demand2]"
   ],
   [
    "C42",
    "x[supplier6,demand3]"
   ],
   [
    "C43",
    "x[supplier6,demand4]"
   ],
   [
    "C44",
    "x[supplier6,demand5]"
   ],
   [
    "C45",
    "x[supplier6,demand6]"
   ],
   [
    "C46",
    "x[supplier6,demand7]"
   ],
   [
    "C47",
    "x[supplier6,demand8]"
   ],
   [
    "C48",
    "x[supplier7,demand1]"
   ],
   [
    "C49",
    "x[supplier7,demand2]"
   ],
   [
    "C50",
    "x[supplier7,demand3]"
   ],
   [
    "C51",
    "x[supplier7,demand4]"
   ],
   [
    "C52",
    "x[supplier7,demand5]"
   ],
   [
    "C53",
    "x[supplier7,demand6]"
   ],
   [
    "C54",
    "x[supplier7,demand7]"
   ],
   [
    "C55",
    "x[supplier7,demand8]"
   ],
   [
    "C56",
    "x[supplier8,demand1]"
   ],
   [
    "C57",
    "x[supplier8,demand2]"
   ],
   [
    "C58",
    "x[supplier8,demand3]"
   ],
   [
    "C59",
    "x[supplier8,demand4]"
   ],
   [
    "C60",
    "x[supplier8,demand5]"
   ],
   [
    "C61",
    "x[supplier8,demand6]"
   ],
   [
    "C62",
    "x[supplier8,demand7]"
   ],
   [
    "C63",
    "x[supplier8,demand8]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[supplier1,demand1]",
   "x_0_1": "x[supplier1,demand2]",
   "x_0_2": "x[supplier1,demand3]",
   "x_0_3": "x[supplier1,demand4]",
   "x_0_4": "x[supplier1,demand5]",
   "x_0_5": "x[supplier1,demand6]",
   "x_0_6": "x[supplier1,demand7]",
   "x_0_7": "x[supplier1,demand8]",
   "x_1_0": "x[supplier2,demand1]",
   "x_1_1": "x[supplier2,demand2]",
   "x_1_2": "x[supplier2,demand3]",
   "x_1_3": "x[supplier2,demand4]",
   "x_1_4": "x[supplier2,demand5]",
   "x_1_5": "x[supplier2,demand6]",
   "x_1_6": "x[supplier2,demand7]",
   "x_1_7": "x[supplier2,demand8]",
   "x_2_0": "x[supplier3,demand1]",
   "x_2_1": "x[supplier3,demand2]",
   "x_2_2": "x[supplier3,demand3]",
   "x_2_3": "x[supplier3,demand4]",
   "x_2_4": "x[supplier3,demand5]",
   "x_2_5": "x[supplier3,demand6]",
   "x_2_6": "x[supplier3,demand7]",
   "x_2_7": "x[supplier3,demand8]",
   "x_3_0": "x[supplier4,demand1]",
   "x_3_1": "x[supplier4,demand2]",
   "x_3_2": "x[supplier4,demand3]",
   "x_3_3": "x[supplier4,demand4]",
   "x_3_4": "x[supplier4,demand5]",
   "x_3_5": "x[supplier4,demand6]",
   "x_3_6": "x[supplier4,demand7]",
   "x_3_7": "x[supplier4,demand8]",
   "x_4_0": "x[supplier5,demand1]",
   "x_4_1": "x[supplier5,demand2]",
   "x_4_2": "x[supplier5,demand3]",
   "x_4_3": "x[supplier5,demand4]",
   "x_4_4": "x[supplier5,demand5]",
   "x_4_5": "x[supplier5,demand6]",
   "x_4_6": "x[supplier5,demand7]",
   "x_4_7": "x[supplier5,demand8]",
   "x_5_0": "x[supplier6,demand1]",
   "x_5_1": "x[supplier6,demand2]",
   "x_5_2": "x[supplier6,demand3]",
   "x_5_3": "x[supplier6,demand4]",
   "x_5_4": "x[supplier6,demand5]",
   "x_5_5": "x[supplier6,demand6]",
   "x_5_6": "x[supplier6,demand7]",
   "x_5_7": "x[supplier6,demand8]",
   "x_6_0": "x[supplier7,demand1]",
   "x_6_1": "x[supplier7,demand2]",
   "x_6_2": "x[supplier7,demand3]",
   "x_6_3": "x[supplier7,demand4]",
   "x_6_4": "x[supplier7,demand5]",
   "x_6_5": "x[supplier7,demand6]",
   "x_6_6": "x[supplier7,demand7]",
   "x_6_7": "x[supplier7,demand8]",
   "x_7_0": "x[supplier8,demand1]",
   "x_7_1": "x[supplier8,demand2]",
   "x_7_2": "x[supplier8,demand3]",
   "x_7_3": "x[supplier8,demand4]",
   "x_7_4": "x[supplier8,demand5]",
   "x_7_5": "x[supplier8,demand6]",
   "x_7_6": "x[supplier8,demand7]",
   "x_7_7": "x[supplier8,demand8]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 372.932532719719,
   "variables": [
    [
     "C0",
     9.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     3.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     10.0
    ],
    [
     "C7",
     37.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     11.0
    ],
    [
     "C10",
     11.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     16.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     14.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     19.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     45.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     25.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     43.0
    ],
    [
     "C53",
     17.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     39.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-009": {
  "source_code_sha256": "0913d218d026164befbf3f6a4b3c621f94eb40e23a0911d3d41ae81d2f6cd934",
  "original_solution_sha256": "901bd3c1a1741ffe9c74e785d47475da98ebe436f3f7f303e81a718aef25e9a2",
  "bindings": [
   [
    "C0",
    "x[S2,C2]"
   ],
   [
    "C1",
    "x[S2,C4]"
   ],
   [
    "C2",
    "x[S2,C1]"
   ],
   [
    "C3",
    "x[S2,C3]"
   ],
   [
    "C4",
    "x[S4,C2]"
   ],
   [
    "C5",
    "x[S4,C4]"
   ],
   [
    "C6",
    "x[S4,C1]"
   ],
   [
    "C7",
    "x[S4,C3]"
   ],
   [
    "C8",
    "x[S3,C2]"
   ],
   [
    "C9",
    "x[S3,C4]"
   ],
   [
    "C10",
    "x[S3,C1]"
   ],
   [
    "C11",
    "x[S3,C3]"
   ],
   [
    "C12",
    "x[S1,C2]"
   ],
   [
    "C13",
    "x[S1,C4]"
   ],
   [
    "C14",
    "x[S1,C1]"
   ],
   [
    "C15",
    "x[S1,C3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 238781.51192549663,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     20.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     210.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     39.0
    ],
    [
     "C13",
     205.0
    ],
    [
     "C14",
     94.0
    ],
    [
     "C15",
     65.0
    ]
   ],
   "error": null
  }
 },
 "OR-015": {
  "source_code_sha256": "9cd84d30e7d94a3e9bb6b77c71b215916f56df0650dc9ff39b847f7e6e771779",
  "original_solution_sha256": "58ef4a44824dc79dc007715fabf283521d701f225a3ee6bda32c27164bea70dc",
  "bindings": [
   [
    "C0",
    "x[Aalopuri]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Aalopuri]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 29660.0,
   "variables": [
    [
     "C0",
     1483.0
    ]
   ],
   "error": null
  }
 },
 "OR-019": {
  "source_code_sha256": "efd6d66460442edc52e796a854234cfbd0db753322c79757e70096a7ff95ca7d",
  "original_solution_sha256": "071705e568cc90567aae65a7638d2c73c37f55cdc04aa3a33d254db5d4d1a1d8",
  "bindings": [
   [
    "C0",
    "x[27in 4K Gaming Monitor]"
   ],
   [
    "C1",
    "x[27in FHD Monitor]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[27in 4K Gaming Monitor]",
   "x_2": "x[27in FHD Monitor]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 7123134.6899999995,
   "variables": [
    [
     "C0",
     12474.0
    ],
    [
     "C1",
     15057.0
    ]
   ],
   "error": null
  }
 },
 "OR-023": {
  "source_code_sha256": "d1552c90159016a6134ad696fd33944e87405fc31c5b058b5e49be5093bbb6b6",
  "original_solution_sha256": "04d282ebf554028110e70325fd86756ad46ca03b949c5d50efbf90f233e7f997",
  "bindings": [
   [
    "C0",
    "x[ELE-ACC-10000478]"
   ],
   [
    "C1",
    "x[ELE-ACC-10002424]"
   ],
   [
    "C2",
    "x[ELE-ACC-10018567]"
   ],
   [
    "C3",
    "x[ELE-ACC-10019567]"
   ]
  ],
  "primary_mapping": null,
  "mapping_basis": "unavailable",
  "mapping_failure": "mapping_failure",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1330.0,
   "variables": [
    [
     "C0",
     140.0
    ],
    [
     "C1",
     136.0
    ],
    [
     "C2",
     268.0
    ],
    [
     "C3",
     121.0
    ]
   ],
   "error": null
  }
 },
 "OR-028": {
  "source_code_sha256": "c5b8be0c924edcadea4c254c1eb1fc3a294044a5272cb043525344b0253f8905",
  "original_solution_sha256": "c342ff4f837337896a13f2e71a8dd1634f4e6432b9786821ebf1ef1a84c3f22f",
  "bindings": [
   [
    "C0",
    "x[sku_I27]"
   ],
   [
    "C1",
    "x[sku_I499]"
   ],
   [
    "C2",
    "x[sku_I719]"
   ],
   [
    "C3",
    "x[sku_T18]"
   ],
   [
    "C4",
    "x[sku_T29]"
   ],
   [
    "C5",
    "x[sku_T39]"
   ],
   [
    "C6",
    "x[sku_T499]"
   ],
   [
    "C7",
    "x[sku_T9]"
   ],
   [
    "C8",
    "x[sku_3081]"
   ],
   [
    "C9",
    "x[sku_339]"
   ],
   [
    "C10",
    "x[sku_3799]"
   ],
   [
    "C11",
    "x[sku_439]"
   ],
   [
    "C12",
    "x[sku_539]"
   ],
   [
    "C13",
    "x[sku_61399]"
   ],
   [
    "C14",
    "x[sku_628]"
   ],
   [
    "C15",
    "x[sku_708]"
   ],
   [
    "C16",
    "x[sku_77]"
   ],
   [
    "C17",
    "x[sku_79]"
   ],
   [
    "C18",
    "x[sku_799]"
   ],
   [
    "C19",
    "x[sku_8499]"
   ],
   [
    "C20",
    "x[sku_89]"
   ],
   [
    "C21",
    "x[sku_897]"
   ],
   [
    "C22",
    "x[sku_9699]"
   ],
   [
    "C23",
    "x[sku_bobo]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[sku_I27]",
   "x_2": "x[sku_I499]",
   "x_3": "x[sku_I719]",
   "x_4": "x[sku_T18]",
   "x_5": "x[sku_T29]",
   "x_6": "x[sku_T39]",
   "x_7": "x[sku_T499]",
   "x_8": "x[sku_T9]",
   "x_9": "x[sku_3081]",
   "x_10": "x[sku_339]",
   "x_11": "x[sku_3799]",
   "x_12": "x[sku_439]",
   "x_13": "x[sku_539]",
   "x_14": "x[sku_61399]",
   "x_15": "x[sku_628]",
   "x_16": "x[sku_708]",
   "x_17": "x[sku_77]",
   "x_18": "x[sku_79]",
   "x_19": "x[sku_799]",
   "x_20": "x[sku_8499]",
   "x_21": "x[sku_89]",
   "x_22": "x[sku_897]",
   "x_23": "x[sku_9699]",
   "x_24": "x[sku_bobo]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 285816.0,
   "variables": [
    [
     "C0",
     6.0
    ],
    [
     "C1",
     4.0
    ],
    [
     "C2",
     16.0
    ],
    [
     "C3",
     14.0
    ],
    [
     "C4",
     4.0
    ],
    [
     "C5",
     32.0
    ],
    [
     "C6",
     8.0
    ],
    [
     "C7",
     2.0
    ],
    [
     "C8",
     10.0
    ],
    [
     "C9",
     8.0
    ],
    [
     "C10",
     18.0
    ],
    [
     "C11",
     2.0
    ],
    [
     "C12",
     4.0
    ],
    [
     "C13",
     8.0
    ],
    [
     "C14",
     2.0
    ],
    [
     "C15",
     198.0
    ],
    [
     "C16",
     32.0
    ],
    [
     "C17",
     18.0
    ],
    [
     "C18",
     570.0
    ],
    [
     "C19",
     6.0
    ],
    [
     "C20",
     26.0
    ],
    [
     "C21",
     6.0
    ],
    [
     "C22",
     33.0
    ],
    [
     "C23",
     33.0
    ]
   ],
   "error": null
  }
 },
 "OR-032": {
  "source_code_sha256": "a77ae86648ad399da591db5a68ec92d30868fddb9919791f2ab8919d94396faa",
  "original_solution_sha256": "1b784364c94bf18cad5b7749fd9105fc0236256f7fe15effa75af73e97f94e3d",
  "bindings": [
   [
    "C0",
    "x[Books_15.15]"
   ],
   [
    "C1",
    "x[Books_30.3]"
   ],
   [
    "C2",
    "x[Books_45.45]"
   ],
   [
    "C3",
    "x[Books_60.6]"
   ],
   [
    "C4",
    "x[Books_75.75]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Books_15.15]",
   "x_2": "x[Books_30.3]",
   "x_3": "x[Books_45.45]",
   "x_4": "x[Books_60.6]",
   "x_5": "x[Books_75.75]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1240406.25,
   "variables": [
    [
     "C0",
     1980.0
    ],
    [
     "C1",
     3024.0
    ],
    [
     "C2",
     4536.0
    ],
    [
     "C3",
     5601.0
    ],
    [
     "C4",
     7567.0
    ]
   ],
   "error": null
  }
 },
 "OR-036": {
  "source_code_sha256": "c946d644bb8a8c048218a827475f6841770d4b135a9e88679b64652f50771ca9",
  "original_solution_sha256": "c0bd95f1b2dba8a9d9976975ad9d8d84c31a63b4b6a521197d5f32239c9b2ef1",
  "bindings": [
   [
    "C0",
    "x[Sedan]"
   ],
   [
    "C1",
    "x[SUV]"
   ],
   [
    "C2",
    "x[Truck]"
   ],
   [
    "C3",
    "x[Convertible]"
   ],
   [
    "C4",
    "x[Minivan]"
   ],
   [
    "C5",
    "x[Coupe]"
   ],
   [
    "C6",
    "x[Hatchback]"
   ],
   [
    "C7",
    "x[Station Wagon]"
   ],
   [
    "C8",
    "x[Electric Car]"
   ],
   [
    "C9",
    "x[Hybrid Car]"
   ],
   [
    "C10",
    "x[Luxury Sedan]"
   ],
   [
    "C11",
    "x[Sports Car]"
   ],
   [
    "C12",
    "x[Crossover]"
   ],
   [
    "C13",
    "x[Diesel Truck]"
   ],
   [
    "C14",
    "x[Compact SUV]"
   ],
   [
    "C15",
    "x[Luxury SUV]"
   ],
   [
    "C16",
    "x[Cargo Van]"
   ],
   [
    "C17",
    "x[Pickup Truck]"
   ],
   [
    "C18",
    "x[Roadster]"
   ],
   [
    "C19",
    "x[Muscle Car]"
   ],
   [
    "C20",
    "x[Off-road Vehicle]"
   ],
   [
    "C21",
    "x[Camper Van]"
   ],
   [
    "C22",
    "x[Compact Car]"
   ],
   [
    "C23",
    "x[Motorcycle]"
   ],
   [
    "C24",
    "x[Electric SUV]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Sedan]",
   "x_1_0": "x[SUV]",
   "x_2_0": "x[Truck]",
   "x_3_0": "x[Convertible]",
   "x_4_0": "x[Minivan]",
   "x_5_0": "x[Coupe]",
   "x_6_0": "x[Hatchback]",
   "x_7_0": "x[Station Wagon]",
   "x_8_0": "x[Electric Car]",
   "x_9_0": "x[Hybrid Car]",
   "x_10_0": "x[Luxury Sedan]",
   "x_11_0": "x[Sports Car]",
   "x_12_0": "x[Crossover]",
   "x_13_0": "x[Diesel Truck]",
   "x_14_0": "x[Compact SUV]",
   "x_15_0": "x[Luxury SUV]",
   "x_16_0": "x[Cargo Van]",
   "x_17_0": "x[Pickup Truck]",
   "x_18_0": "x[Roadster]",
   "x_19_0": "x[Muscle Car]",
   "x_20_0": "x[Off-road Vehicle]",
   "x_21_0": "x[Camper Van]",
   "x_22_0": "x[Compact Car]",
   "x_23_0": "x[Motorcycle]",
   "x_24_0": "x[Electric SUV]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 509925.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     75.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-040": {
  "source_code_sha256": "3606077d740dd2ab76a89eaffd5fbca3e49a375cb978b0becdb55977de7e5bc6",
  "original_solution_sha256": "8e2d8c41e214b874c850caf35709ac7f67d1362246b12fd810ddc360af914459",
  "bindings": [
   [
    "C0",
    "x[Queens]"
   ],
   [
    "C1",
    "x[Brooklyn]"
   ],
   [
    "C2",
    "x[Manhattan]"
   ],
   [
    "C3",
    "x[Bronx]"
   ],
   [
    "C4",
    "x[Staten Island]"
   ],
   [
    "C5",
    "x[Harlem]"
   ],
   [
    "C6",
    "x[Upper East Side]"
   ],
   [
    "C7",
    "x[Lower Manhattan]"
   ],
   [
    "C8",
    "x[Midtown]"
   ],
   [
    "C9",
    "x[Long Island City]"
   ],
   [
    "C10",
    "x[Williamsburg]"
   ],
   [
    "C11",
    "x[Bushwick]"
   ],
   [
    "C12",
    "x[Flatbush]"
   ],
   [
    "C13",
    "x[Greenpoint]"
   ],
   [
    "C14",
    "x[Park Slope]"
   ],
   [
    "C15",
    "x[Astoria]"
   ],
   [
    "C16",
    "x[Jackson Heights]"
   ],
   [
    "C17",
    "x[Flushing]"
   ],
   [
    "C18",
    "x[Sunnyside]"
   ],
   [
    "C19",
    "x[Ditmars]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Queens]",
   "x_1_0": "x[Brooklyn]",
   "x_2_0": "x[Manhattan]",
   "x_3_0": "x[Bronx]",
   "x_4_0": "x[Staten Island]",
   "x_5_0": "x[Harlem]",
   "x_6_0": "x[Upper East Side]",
   "x_7_0": "x[Lower Manhattan]",
   "x_8_0": "x[Midtown]",
   "x_9_0": "x[Long Island City]",
   "x_10_0": "x[Williamsburg]",
   "x_11_0": "x[Bushwick]",
   "x_12_0": "x[Flatbush]",
   "x_13_0": "x[Greenpoint]",
   "x_14_0": "x[Park Slope]",
   "x_15_0": "x[Astoria]",
   "x_16_0": "x[Jackson Heights]",
   "x_17_0": "x[Flushing]",
   "x_18_0": "x[Sunnyside]",
   "x_19_0": "x[Ditmars]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 74260.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     79.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-044": {
  "source_code_sha256": "f38d0751640afa98743866b4e5df3056ac81ab49b5459163f701a4ae88810750",
  "original_solution_sha256": "e6dd56492f5eda5827745e102076a2166e1d9b1d6f5fceca0c0297bac094bac9",
  "bindings": [
   [
    "C0",
    "x[1,1]"
   ],
   [
    "C1",
    "x[1,2]"
   ],
   [
    "C2",
    "x[1,3]"
   ],
   [
    "C3",
    "x[1,4]"
   ],
   [
    "C4",
    "x[1,5]"
   ],
   [
    "C5",
    "x[1,6]"
   ],
   [
    "C6",
    "x[1,7]"
   ],
   [
    "C7",
    "x[1,8]"
   ],
   [
    "C8",
    "x[1,9]"
   ],
   [
    "C9",
    "x[1,10]"
   ],
   [
    "C10",
    "x[2,1]"
   ],
   [
    "C11",
    "x[2,2]"
   ],
   [
    "C12",
    "x[2,3]"
   ],
   [
    "C13",
    "x[2,4]"
   ],
   [
    "C14",
    "x[2,5]"
   ],
   [
    "C15",
    "x[2,6]"
   ],
   [
    "C16",
    "x[2,7]"
   ],
   [
    "C17",
    "x[2,8]"
   ],
   [
    "C18",
    "x[2,9]"
   ],
   [
    "C19",
    "x[2,10]"
   ],
   [
    "C20",
    "x[3,1]"
   ],
   [
    "C21",
    "x[3,2]"
   ],
   [
    "C22",
    "x[3,3]"
   ],
   [
    "C23",
    "x[3,4]"
   ],
   [
    "C24",
    "x[3,5]"
   ],
   [
    "C25",
    "x[3,6]"
   ],
   [
    "C26",
    "x[3,7]"
   ],
   [
    "C27",
    "x[3,8]"
   ],
   [
    "C28",
    "x[3,9]"
   ],
   [
    "C29",
    "x[3,10]"
   ],
   [
    "C30",
    "x[4,1]"
   ],
   [
    "C31",
    "x[4,2]"
   ],
   [
    "C32",
    "x[4,3]"
   ],
   [
    "C33",
    "x[4,4]"
   ],
   [
    "C34",
    "x[4,5]"
   ],
   [
    "C35",
    "x[4,6]"
   ],
   [
    "C36",
    "x[4,7]"
   ],
   [
    "C37",
    "x[4,8]"
   ],
   [
    "C38",
    "x[4,9]"
   ],
   [
    "C39",
    "x[4,10]"
   ],
   [
    "C40",
    "x[5,1]"
   ],
   [
    "C41",
    "x[5,2]"
   ],
   [
    "C42",
    "x[5,3]"
   ],
   [
    "C43",
    "x[5,4]"
   ],
   [
    "C44",
    "x[5,5]"
   ],
   [
    "C45",
    "x[5,6]"
   ],
   [
    "C46",
    "x[5,7]"
   ],
   [
    "C47",
    "x[5,8]"
   ],
   [
    "C48",
    "x[5,9]"
   ],
   [
    "C49",
    "x[5,10]"
   ],
   [
    "C50",
    "x[6,1]"
   ],
   [
    "C51",
    "x[6,2]"
   ],
   [
    "C52",
    "x[6,3]"
   ],
   [
    "C53",
    "x[6,4]"
   ],
   [
    "C54",
    "x[6,5]"
   ],
   [
    "C55",
    "x[6,6]"
   ],
   [
    "C56",
    "x[6,7]"
   ],
   [
    "C57",
    "x[6,8]"
   ],
   [
    "C58",
    "x[6,9]"
   ],
   [
    "C59",
    "x[6,10]"
   ],
   [
    "C60",
    "x[7,1]"
   ],
   [
    "C61",
    "x[7,2]"
   ],
   [
    "C62",
    "x[7,3]"
   ],
   [
    "C63",
    "x[7,4]"
   ],
   [
    "C64",
    "x[7,5]"
   ],
   [
    "C65",
    "x[7,6]"
   ],
   [
    "C66",
    "x[7,7]"
   ],
   [
    "C67",
    "x[7,8]"
   ],
   [
    "C68",
    "x[7,9]"
   ],
   [
    "C69",
    "x[7,10]"
   ],
   [
    "C70",
    "x[8,1]"
   ],
   [
    "C71",
    "x[8,2]"
   ],
   [
    "C72",
    "x[8,3]"
   ],
   [
    "C73",
    "x[8,4]"
   ],
   [
    "C74",
    "x[8,5]"
   ],
   [
    "C75",
    "x[8,6]"
   ],
   [
    "C76",
    "x[8,7]"
   ],
   [
    "C77",
    "x[8,8]"
   ],
   [
    "C78",
    "x[8,9]"
   ],
   [
    "C79",
    "x[8,10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,1]",
   "x_0_1": "x[2,1]",
   "x_0_2": "x[3,1]",
   "x_0_3": "x[4,1]",
   "x_0_4": "x[5,1]",
   "x_0_5": "x[6,1]",
   "x_0_6": "x[7,1]",
   "x_0_7": "x[8,1]",
   "x_1_0": "x[1,2]",
   "x_1_1": "x[2,2]",
   "x_1_2": "x[3,2]",
   "x_1_3": "x[4,2]",
   "x_1_4": "x[5,2]",
   "x_1_5": "x[6,2]",
   "x_1_6": "x[7,2]",
   "x_1_7": "x[8,2]",
   "x_2_0": "x[1,3]",
   "x_2_1": "x[2,3]",
   "x_2_2": "x[3,3]",
   "x_2_3": "x[4,3]",
   "x_2_4": "x[5,3]",
   "x_2_5": "x[6,3]",
   "x_2_6": "x[7,3]",
   "x_2_7": "x[8,3]",
   "x_3_0": "x[1,4]",
   "x_3_1": "x[2,4]",
   "x_3_2": "x[3,4]",
   "x_3_3": "x[4,4]",
   "x_3_4": "x[5,4]",
   "x_3_5": "x[6,4]",
   "x_3_6": "x[7,4]",
   "x_3_7": "x[8,4]",
   "x_4_0": "x[1,5]",
   "x_4_1": "x[2,5]",
   "x_4_2": "x[3,5]",
   "x_4_3": "x[4,5]",
   "x_4_4": "x[5,5]",
   "x_4_5": "x[6,5]",
   "x_4_6": "x[7,5]",
   "x_4_7": "x[8,5]",
   "x_5_0": "x[1,6]",
   "x_5_1": "x[2,6]",
   "x_5_2": "x[3,6]",
   "x_5_3": "x[4,6]",
   "x_5_4": "x[5,6]",
   "x_5_5": "x[6,6]",
   "x_5_6": "x[7,6]",
   "x_5_7": "x[8,6]",
   "x_6_0": "x[1,7]",
   "x_6_1": "x[2,7]",
   "x_6_2": "x[3,7]",
   "x_6_3": "x[4,7]",
   "x_6_4": "x[5,7]",
   "x_6_5": "x[6,7]",
   "x_6_6": "x[7,7]",
   "x_6_7": "x[8,7]",
   "x_7_0": "x[1,8]",
   "x_7_1": "x[2,8]",
   "x_7_2": "x[3,8]",
   "x_7_3": "x[4,8]",
   "x_7_4": "x[5,8]",
   "x_7_5": "x[6,8]",
   "x_7_6": "x[7,8]",
   "x_7_7": "x[8,8]",
   "x_8_0": "x[1,9]",
   "x_8_1": "x[2,9]",
   "x_8_2": "x[3,9]",
   "x_8_3": "x[4,9]",
   "x_8_4": "x[5,9]",
   "x_8_5": "x[6,9]",
   "x_8_6": "x[7,9]",
   "x_8_7": "x[8,9]",
   "x_9_0": "x[1,10]",
   "x_9_1": "x[2,10]",
   "x_9_2": "x[3,10]",
   "x_9_3": "x[4,10]",
   "x_9_4": "x[5,10]",
   "x_9_5": "x[6,10]",
   "x_9_6": "x[7,10]",
   "x_9_7": "x[8,10]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 8000.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     100.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     150.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     120.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     130.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     90.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     110.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     160.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     140.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-048": {
  "source_code_sha256": "e568de569bff2c61e830567acadfdc4d2157945a5b739267255e004c09045c1a",
  "original_solution_sha256": "ce3b40ea1ba2404b7a327ea54b547984f6ec09b5d8edd15a1f03969efbb848f0",
  "bindings": [
   [
    "C0",
    "x[1,Window Unit]"
   ],
   [
    "C1",
    "x[1,Portable Unit]"
   ],
   [
    "C2",
    "x[1,Split System]"
   ],
   [
    "C3",
    "x[1,Ductless System]"
   ],
   [
    "C4",
    "x[1,Central AC]"
   ],
   [
    "C5",
    "x[1,Hybrid AC]"
   ],
   [
    "C6",
    "x[1,Geothermal AC]"
   ],
   [
    "C7",
    "x[1,Smart AC]"
   ],
   [
    "C8",
    "x[1,Evaporative Cooler]"
   ],
   [
    "C9",
    "x[1,Package Unit]"
   ],
   [
    "C10",
    "x[2,Window Unit]"
   ],
   [
    "C11",
    "x[2,Portable Unit]"
   ],
   [
    "C12",
    "x[2,Split System]"
   ],
   [
    "C13",
    "x[2,Ductless System]"
   ],
   [
    "C14",
    "x[2,Central AC]"
   ],
   [
    "C15",
    "x[2,Hybrid AC]"
   ],
   [
    "C16",
    "x[2,Geothermal AC]"
   ],
   [
    "C17",
    "x[2,Smart AC]"
   ],
   [
    "C18",
    "x[2,Evaporative Cooler]"
   ],
   [
    "C19",
    "x[2,Package Unit]"
   ],
   [
    "C20",
    "x[3,Window Unit]"
   ],
   [
    "C21",
    "x[3,Portable Unit]"
   ],
   [
    "C22",
    "x[3,Split System]"
   ],
   [
    "C23",
    "x[3,Ductless System]"
   ],
   [
    "C24",
    "x[3,Central AC]"
   ],
   [
    "C25",
    "x[3,Hybrid AC]"
   ],
   [
    "C26",
    "x[3,Geothermal AC]"
   ],
   [
    "C27",
    "x[3,Smart AC]"
   ],
   [
    "C28",
    "x[3,Evaporative Cooler]"
   ],
   [
    "C29",
    "x[3,Package Unit]"
   ],
   [
    "C30",
    "x[4,Window Unit]"
   ],
   [
    "C31",
    "x[4,Portable Unit]"
   ],
   [
    "C32",
    "x[4,Split System]"
   ],
   [
    "C33",
    "x[4,Ductless System]"
   ],
   [
    "C34",
    "x[4,Central AC]"
   ],
   [
    "C35",
    "x[4,Hybrid AC]"
   ],
   [
    "C36",
    "x[4,Geothermal AC]"
   ],
   [
    "C37",
    "x[4,Smart AC]"
   ],
   [
    "C38",
    "x[4,Evaporative Cooler]"
   ],
   [
    "C39",
    "x[4,Package Unit]"
   ],
   [
    "C40",
    "x[5,Window Unit]"
   ],
   [
    "C41",
    "x[5,Portable Unit]"
   ],
   [
    "C42",
    "x[5,Split System]"
   ],
   [
    "C43",
    "x[5,Ductless System]"
   ],
   [
    "C44",
    "x[5,Central AC]"
   ],
   [
    "C45",
    "x[5,Hybrid AC]"
   ],
   [
    "C46",
    "x[5,Geothermal AC]"
   ],
   [
    "C47",
    "x[5,Smart AC]"
   ],
   [
    "C48",
    "x[5,Evaporative Cooler]"
   ],
   [
    "C49",
    "x[5,Package Unit]"
   ],
   [
    "C50",
    "x[6,Window Unit]"
   ],
   [
    "C51",
    "x[6,Portable Unit]"
   ],
   [
    "C52",
    "x[6,Split System]"
   ],
   [
    "C53",
    "x[6,Ductless System]"
   ],
   [
    "C54",
    "x[6,Central AC]"
   ],
   [
    "C55",
    "x[6,Hybrid AC]"
   ],
   [
    "C56",
    "x[6,Geothermal AC]"
   ],
   [
    "C57",
    "x[6,Smart AC]"
   ],
   [
    "C58",
    "x[6,Evaporative Cooler]"
   ],
   [
    "C59",
    "x[6,Package Unit]"
   ],
   [
    "C60",
    "x[7,Window Unit]"
   ],
   [
    "C61",
    "x[7,Portable Unit]"
   ],
   [
    "C62",
    "x[7,Split System]"
   ],
   [
    "C63",
    "x[7,Ductless System]"
   ],
   [
    "C64",
    "x[7,Central AC]"
   ],
   [
    "C65",
    "x[7,Hybrid AC]"
   ],
   [
    "C66",
    "x[7,Geothermal AC]"
   ],
   [
    "C67",
    "x[7,Smart AC]"
   ],
   [
    "C68",
    "x[7,Evaporative Cooler]"
   ],
   [
    "C69",
    "x[7,Package Unit]"
   ],
   [
    "C70",
    "x[8,Window Unit]"
   ],
   [
    "C71",
    "x[8,Portable Unit]"
   ],
   [
    "C72",
    "x[8,Split System]"
   ],
   [
    "C73",
    "x[8,Ductless System]"
   ],
   [
    "C74",
    "x[8,Central AC]"
   ],
   [
    "C75",
    "x[8,Hybrid AC]"
   ],
   [
    "C76",
    "x[8,Geothermal AC]"
   ],
   [
    "C77",
    "x[8,Smart AC]"
   ],
   [
    "C78",
    "x[8,Evaporative Cooler]"
   ],
   [
    "C79",
    "x[8,Package Unit]"
   ],
   [
    "C80",
    "x[9,Window Unit]"
   ],
   [
    "C81",
    "x[9,Portable Unit]"
   ],
   [
    "C82",
    "x[9,Split System]"
   ],
   [
    "C83",
    "x[9,Ductless System]"
   ],
   [
    "C84",
    "x[9,Central AC]"
   ],
   [
    "C85",
    "x[9,Hybrid AC]"
   ],
   [
    "C86",
    "x[9,Geothermal AC]"
   ],
   [
    "C87",
    "x[9,Smart AC]"
   ],
   [
    "C88",
    "x[9,Evaporative Cooler]"
   ],
   [
    "C89",
    "x[9,Package Unit]"
   ],
   [
    "C90",
    "x[10,Window Unit]"
   ],
   [
    "C91",
    "x[10,Portable Unit]"
   ],
   [
    "C92",
    "x[10,Split System]"
   ],
   [
    "C93",
    "x[10,Ductless System]"
   ],
   [
    "C94",
    "x[10,Central AC]"
   ],
   [
    "C95",
    "x[10,Hybrid AC]"
   ],
   [
    "C96",
    "x[10,Geothermal AC]"
   ],
   [
    "C97",
    "x[10,Smart AC]"
   ],
   [
    "C98",
    "x[10,Evaporative Cooler]"
   ],
   [
    "C99",
    "x[10,Package Unit]"
   ],
   [
    "C100",
    "x[11,Window Unit]"
   ],
   [
    "C101",
    "x[11,Portable Unit]"
   ],
   [
    "C102",
    "x[11,Split System]"
   ],
   [
    "C103",
    "x[11,Ductless System]"
   ],
   [
    "C104",
    "x[11,Central AC]"
   ],
   [
    "C105",
    "x[11,Hybrid AC]"
   ],
   [
    "C106",
    "x[11,Geothermal AC]"
   ],
   [
    "C107",
    "x[11,Smart AC]"
   ],
   [
    "C108",
    "x[11,Evaporative Cooler]"
   ],
   [
    "C109",
    "x[11,Package Unit]"
   ],
   [
    "C110",
    "x[12,Window Unit]"
   ],
   [
    "C111",
    "x[12,Portable Unit]"
   ],
   [
    "C112",
    "x[12,Split System]"
   ],
   [
    "C113",
    "x[12,Ductless System]"
   ],
   [
    "C114",
    "x[12,Central AC]"
   ],
   [
    "C115",
    "x[12,Hybrid AC]"
   ],
   [
    "C116",
    "x[12,Geothermal AC]"
   ],
   [
    "C117",
    "x[12,Smart AC]"
   ],
   [
    "C118",
    "x[12,Evaporative Cooler]"
   ],
   [
    "C119",
    "x[12,Package Unit]"
   ],
   [
    "C120",
    "x[13,Window Unit]"
   ],
   [
    "C121",
    "x[13,Portable Unit]"
   ],
   [
    "C122",
    "x[13,Split System]"
   ],
   [
    "C123",
    "x[13,Ductless System]"
   ],
   [
    "C124",
    "x[13,Central AC]"
   ],
   [
    "C125",
    "x[13,Hybrid AC]"
   ],
   [
    "C126",
    "x[13,Geothermal AC]"
   ],
   [
    "C127",
    "x[13,Smart AC]"
   ],
   [
    "C128",
    "x[13,Evaporative Cooler]"
   ],
   [
    "C129",
    "x[13,Package Unit]"
   ],
   [
    "C130",
    "x[14,Window Unit]"
   ],
   [
    "C131",
    "x[14,Portable Unit]"
   ],
   [
    "C132",
    "x[14,Split System]"
   ],
   [
    "C133",
    "x[14,Ductless System]"
   ],
   [
    "C134",
    "x[14,Central AC]"
   ],
   [
    "C135",
    "x[14,Hybrid AC]"
   ],
   [
    "C136",
    "x[14,Geothermal AC]"
   ],
   [
    "C137",
    "x[14,Smart AC]"
   ],
   [
    "C138",
    "x[14,Evaporative Cooler]"
   ],
   [
    "C139",
    "x[14,Package Unit]"
   ],
   [
    "C140",
    "x[15,Window Unit]"
   ],
   [
    "C141",
    "x[15,Portable Unit]"
   ],
   [
    "C142",
    "x[15,Split System]"
   ],
   [
    "C143",
    "x[15,Ductless System]"
   ],
   [
    "C144",
    "x[15,Central AC]"
   ],
   [
    "C145",
    "x[15,Hybrid AC]"
   ],
   [
    "C146",
    "x[15,Geothermal AC]"
   ],
   [
    "C147",
    "x[15,Smart AC]"
   ],
   [
    "C148",
    "x[15,Evaporative Cooler]"
   ],
   [
    "C149",
    "x[15,Package Unit]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Window Unit]",
   "x_0_1": "x[2,Window Unit]",
   "x_0_2": "x[3,Window Unit]",
   "x_0_3": "x[4,Window Unit]",
   "x_0_4": "x[5,Window Unit]",
   "x_0_5": "x[6,Window Unit]",
   "x_0_6": "x[7,Window Unit]",
   "x_0_7": "x[8,Window Unit]",
   "x_0_8": "x[9,Window Unit]",
   "x_0_9": "x[10,Window Unit]",
   "x_0_10": "x[11,Window Unit]",
   "x_0_11": "x[12,Window Unit]",
   "x_0_12": "x[13,Window Unit]",
   "x_0_13": "x[14,Window Unit]",
   "x_0_14": "x[15,Window Unit]",
   "x_1_0": "x[1,Portable Unit]",
   "x_1_1": "x[2,Portable Unit]",
   "x_1_2": "x[3,Portable Unit]",
   "x_1_3": "x[4,Portable Unit]",
   "x_1_4": "x[5,Portable Unit]",
   "x_1_5": "x[6,Portable Unit]",
   "x_1_6": "x[7,Portable Unit]",
   "x_1_7": "x[8,Portable Unit]",
   "x_1_8": "x[9,Portable Unit]",
   "x_1_9": "x[10,Portable Unit]",
   "x_1_10": "x[11,Portable Unit]",
   "x_1_11": "x[12,Portable Unit]",
   "x_1_12": "x[13,Portable Unit]",
   "x_1_13": "x[14,Portable Unit]",
   "x_1_14": "x[15,Portable Unit]",
   "x_2_0": "x[1,Split System]",
   "x_2_1": "x[2,Split System]",
   "x_2_2": "x[3,Split System]",
   "x_2_3": "x[4,Split System]",
   "x_2_4": "x[5,Split System]",
   "x_2_5": "x[6,Split System]",
   "x_2_6": "x[7,Split System]",
   "x_2_7": "x[8,Split System]",
   "x_2_8": "x[9,Split System]",
   "x_2_9": "x[10,Split System]",
   "x_2_10": "x[11,Split System]",
   "x_2_11": "x[12,Split System]",
   "x_2_12": "x[13,Split System]",
   "x_2_13": "x[14,Split System]",
   "x_2_14": "x[15,Split System]",
   "x_3_0": "x[1,Ductless System]",
   "x_3_1": "x[2,Ductless System]",
   "x_3_2": "x[3,Ductless System]",
   "x_3_3": "x[4,Ductless System]",
   "x_3_4": "x[5,Ductless System]",
   "x_3_5": "x[6,Ductless System]",
   "x_3_6": "x[7,Ductless System]",
   "x_3_7": "x[8,Ductless System]",
   "x_3_8": "x[9,Ductless System]",
   "x_3_9": "x[10,Ductless System]",
   "x_3_10": "x[11,Ductless System]",
   "x_3_11": "x[12,Ductless System]",
   "x_3_12": "x[13,Ductless System]",
   "x_3_13": "x[14,Ductless System]",
   "x_3_14": "x[15,Ductless System]",
   "x_4_0": "x[1,Central AC]",
   "x_4_1": "x[2,Central AC]",
   "x_4_2": "x[3,Central AC]",
   "x_4_3": "x[4,Central AC]",
   "x_4_4": "x[5,Central AC]",
   "x_4_5": "x[6,Central AC]",
   "x_4_6": "x[7,Central AC]",
   "x_4_7": "x[8,Central AC]",
   "x_4_8": "x[9,Central AC]",
   "x_4_9": "x[10,Central AC]",
   "x_4_10": "x[11,Central AC]",
   "x_4_11": "x[12,Central AC]",
   "x_4_12": "x[13,Central AC]",
   "x_4_13": "x[14,Central AC]",
   "x_4_14": "x[15,Central AC]",
   "x_5_0": "x[1,Hybrid AC]",
   "x_5_1": "x[2,Hybrid AC]",
   "x_5_2": "x[3,Hybrid AC]",
   "x_5_3": "x[4,Hybrid AC]",
   "x_5_4": "x[5,Hybrid AC]",
   "x_5_5": "x[6,Hybrid AC]",
   "x_5_6": "x[7,Hybrid AC]",
   "x_5_7": "x[8,Hybrid AC]",
   "x_5_8": "x[9,Hybrid AC]",
   "x_5_9": "x[10,Hybrid AC]",
   "x_5_10": "x[11,Hybrid AC]",
   "x_5_11": "x[12,Hybrid AC]",
   "x_5_12": "x[13,Hybrid AC]",
   "x_5_13": "x[14,Hybrid AC]",
   "x_5_14": "x[15,Hybrid AC]",
   "x_6_0": "x[1,Geothermal AC]",
   "x_6_1": "x[2,Geothermal AC]",
   "x_6_2": "x[3,Geothermal AC]",
   "x_6_3": "x[4,Geothermal AC]",
   "x_6_4": "x[5,Geothermal AC]",
   "x_6_5": "x[6,Geothermal AC]",
   "x_6_6": "x[7,Geothermal AC]",
   "x_6_7": "x[8,Geothermal AC]",
   "x_6_8": "x[9,Geothermal AC]",
   "x_6_9": "x[10,Geothermal AC]",
   "x_6_10": "x[11,Geothermal AC]",
   "x_6_11": "x[12,Geothermal AC]",
   "x_6_12": "x[13,Geothermal AC]",
   "x_6_13": "x[14,Geothermal AC]",
   "x_6_14": "x[15,Geothermal AC]",
   "x_7_0": "x[1,Smart AC]",
   "x_7_1": "x[2,Smart AC]",
   "x_7_2": "x[3,Smart AC]",
   "x_7_3": "x[4,Smart AC]",
   "x_7_4": "x[5,Smart AC]",
   "x_7_5": "x[6,Smart AC]",
   "x_7_6": "x[7,Smart AC]",
   "x_7_7": "x[8,Smart AC]",
   "x_7_8": "x[9,Smart AC]",
   "x_7_9": "x[10,Smart AC]",
   "x_7_10": "x[11,Smart AC]",
   "x_7_11": "x[12,Smart AC]",
   "x_7_12": "x[13,Smart AC]",
   "x_7_13": "x[14,Smart AC]",
   "x_7_14": "x[15,Smart AC]",
   "x_8_0": "x[1,Evaporative Cooler]",
   "x_8_1": "x[2,Evaporative Cooler]",
   "x_8_2": "x[3,Evaporative Cooler]",
   "x_8_3": "x[4,Evaporative Cooler]",
   "x_8_4": "x[5,Evaporative Cooler]",
   "x_8_5": "x[6,Evaporative Cooler]",
   "x_8_6": "x[7,Evaporative Cooler]",
   "x_8_7": "x[8,Evaporative Cooler]",
   "x_8_8": "x[9,Evaporative Cooler]",
   "x_8_9": "x[10,Evaporative Cooler]",
   "x_8_10": "x[11,Evaporative Cooler]",
   "x_8_11": "x[12,Evaporative Cooler]",
   "x_8_12": "x[13,Evaporative Cooler]",
   "x_8_13": "x[14,Evaporative Cooler]",
   "x_8_14": "x[15,Evaporative Cooler]",
   "x_9_0": "x[1,Package Unit]",
   "x_9_1": "x[2,Package Unit]",
   "x_9_2": "x[3,Package Unit]",
   "x_9_3": "x[4,Package Unit]",
   "x_9_4": "x[5,Package Unit]",
   "x_9_5": "x[6,Package Unit]",
   "x_9_6": "x[7,Package Unit]",
   "x_9_7": "x[8,Package Unit]",
   "x_9_8": "x[9,Package Unit]",
   "x_9_9": "x[10,Package Unit]",
   "x_9_10": "x[11,Package Unit]",
   "x_9_11": "x[12,Package Unit]",
   "x_9_12": "x[13,Package Unit]",
   "x_9_13": "x[14,Package Unit]",
   "x_9_14": "x[15,Package Unit]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 635052.0,
   "variables": [
    [
     "C0",
     9.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     16.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     6.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     11.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     11.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     4.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     16.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     7.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     5.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     5.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     8.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     5.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     12.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     10.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     7.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-052": {
  "source_code_sha256": "e87b7a64bc5846900f00f1c7f979ca112f57f8ebee79b19a7d9fdeb38eb38d5c",
  "original_solution_sha256": "85dfeb28f05861c79314d7ffa365df698153e8a4c2b779e8a2b4e7a4336e5a16",
  "bindings": [
   [
    "C0",
    "x[1,The Great Gatsby]"
   ],
   [
    "C1",
    "x[1,To Kill a Mockingbird]"
   ],
   [
    "C2",
    "x[1,1984]"
   ],
   [
    "C3",
    "x[1,Pride and Prejudice]"
   ],
   [
    "C4",
    "x[1,The Catcher in the Rye]"
   ],
   [
    "C5",
    "x[1,Moby Dick]"
   ],
   [
    "C6",
    "x[1,Jane Eyre]"
   ],
   [
    "C7",
    "x[1,War and Peace]"
   ],
   [
    "C8",
    "x[1,The Odyssey]"
   ],
   [
    "C9",
    "x[1,Crime and Punishment]"
   ],
   [
    "C10",
    "x[1,The Hobbit]"
   ],
   [
    "C11",
    "x[1,Brave New World]"
   ],
   [
    "C12",
    "x[1,Anna Karenina]"
   ],
   [
    "C13",
    "x[1,Wuthering Heights]"
   ],
   [
    "C14",
    "x[1,The Divine Comedy]"
   ],
   [
    "C15",
    "x[1,The Iliad]"
   ],
   [
    "C16",
    "x[1,Les Misérables]"
   ],
   [
    "C17",
    "x[1,Dracula]"
   ],
   [
    "C18",
    "x[1,Frankenstein]"
   ],
   [
    "C19",
    "x[1,The Brothers Karamazov]"
   ],
   [
    "C20",
    "x[1,Don Quixote]"
   ],
   [
    "C21",
    "x[1,One Hundred Years of Solitude]"
   ],
   [
    "C22",
    "x[1,Ulysses]"
   ],
   [
    "C23",
    "x[1,The Alchemist]"
   ],
   [
    "C24",
    "x[1,Meditations]"
   ],
   [
    "C25",
    "x[2,The Great Gatsby]"
   ],
   [
    "C26",
    "x[2,To Kill a Mockingbird]"
   ],
   [
    "C27",
    "x[2,1984]"
   ],
   [
    "C28",
    "x[2,Pride and Prejudice]"
   ],
   [
    "C29",
    "x[2,The Catcher in the Rye]"
   ],
   [
    "C30",
    "x[2,Moby Dick]"
   ],
   [
    "C31",
    "x[2,Jane Eyre]"
   ],
   [
    "C32",
    "x[2,War and Peace]"
   ],
   [
    "C33",
    "x[2,The Odyssey]"
   ],
   [
    "C34",
    "x[2,Crime and Punishment]"
   ],
   [
    "C35",
    "x[2,The Hobbit]"
   ],
   [
    "C36",
    "x[2,Brave New World]"
   ],
   [
    "C37",
    "x[2,Anna Karenina]"
   ],
   [
    "C38",
    "x[2,Wuthering Heights]"
   ],
   [
    "C39",
    "x[2,The Divine Comedy]"
   ],
   [
    "C40",
    "x[2,The Iliad]"
   ],
   [
    "C41",
    "x[2,Les Misérables]"
   ],
   [
    "C42",
    "x[2,Dracula]"
   ],
   [
    "C43",
    "x[2,Frankenstein]"
   ],
   [
    "C44",
    "x[2,The Brothers Karamazov]"
   ],
   [
    "C45",
    "x[2,Don Quixote]"
   ],
   [
    "C46",
    "x[2,One Hundred Years of Solitude]"
   ],
   [
    "C47",
    "x[2,Ulysses]"
   ],
   [
    "C48",
    "x[2,The Alchemist]"
   ],
   [
    "C49",
    "x[2,Meditations]"
   ],
   [
    "C50",
    "x[3,The Great Gatsby]"
   ],
   [
    "C51",
    "x[3,To Kill a Mockingbird]"
   ],
   [
    "C52",
    "x[3,1984]"
   ],
   [
    "C53",
    "x[3,Pride and Prejudice]"
   ],
   [
    "C54",
    "x[3,The Catcher in the Rye]"
   ],
   [
    "C55",
    "x[3,Moby Dick]"
   ],
   [
    "C56",
    "x[3,Jane Eyre]"
   ],
   [
    "C57",
    "x[3,War and Peace]"
   ],
   [
    "C58",
    "x[3,The Odyssey]"
   ],
   [
    "C59",
    "x[3,Crime and Punishment]"
   ],
   [
    "C60",
    "x[3,The Hobbit]"
   ],
   [
    "C61",
    "x[3,Brave New World]"
   ],
   [
    "C62",
    "x[3,Anna Karenina]"
   ],
   [
    "C63",
    "x[3,Wuthering Heights]"
   ],
   [
    "C64",
    "x[3,The Divine Comedy]"
   ],
   [
    "C65",
    "x[3,The Iliad]"
   ],
   [
    "C66",
    "x[3,Les Misérables]"
   ],
   [
    "C67",
    "x[3,Dracula]"
   ],
   [
    "C68",
    "x[3,Frankenstein]"
   ],
   [
    "C69",
    "x[3,The Brothers Karamazov]"
   ],
   [
    "C70",
    "x[3,Don Quixote]"
   ],
   [
    "C71",
    "x[3,One Hundred Years of Solitude]"
   ],
   [
    "C72",
    "x[3,Ulysses]"
   ],
   [
    "C73",
    "x[3,The Alchemist]"
   ],
   [
    "C74",
    "x[3,Meditations]"
   ],
   [
    "C75",
    "x[4,The Great Gatsby]"
   ],
   [
    "C76",
    "x[4,To Kill a Mockingbird]"
   ],
   [
    "C77",
    "x[4,1984]"
   ],
   [
    "C78",
    "x[4,Pride and Prejudice]"
   ],
   [
    "C79",
    "x[4,The Catcher in the Rye]"
   ],
   [
    "C80",
    "x[4,Moby Dick]"
   ],
   [
    "C81",
    "x[4,Jane Eyre]"
   ],
   [
    "C82",
    "x[4,War and Peace]"
   ],
   [
    "C83",
    "x[4,The Odyssey]"
   ],
   [
    "C84",
    "x[4,Crime and Punishment]"
   ],
   [
    "C85",
    "x[4,The Hobbit]"
   ],
   [
    "C86",
    "x[4,Brave New World]"
   ],
   [
    "C87",
    "x[4,Anna Karenina]"
   ],
   [
    "C88",
    "x[4,Wuthering Heights]"
   ],
   [
    "C89",
    "x[4,The Divine Comedy]"
   ],
   [
    "C90",
    "x[4,The Iliad]"
   ],
   [
    "C91",
    "x[4,Les Misérables]"
   ],
   [
    "C92",
    "x[4,Dracula]"
   ],
   [
    "C93",
    "x[4,Frankenstein]"
   ],
   [
    "C94",
    "x[4,The Brothers Karamazov]"
   ],
   [
    "C95",
    "x[4,Don Quixote]"
   ],
   [
    "C96",
    "x[4,One Hundred Years of Solitude]"
   ],
   [
    "C97",
    "x[4,Ulysses]"
   ],
   [
    "C98",
    "x[4,The Alchemist]"
   ],
   [
    "C99",
    "x[4,Meditations]"
   ],
   [
    "C100",
    "x[5,The Great Gatsby]"
   ],
   [
    "C101",
    "x[5,To Kill a Mockingbird]"
   ],
   [
    "C102",
    "x[5,1984]"
   ],
   [
    "C103",
    "x[5,Pride and Prejudice]"
   ],
   [
    "C104",
    "x[5,The Catcher in the Rye]"
   ],
   [
    "C105",
    "x[5,Moby Dick]"
   ],
   [
    "C106",
    "x[5,Jane Eyre]"
   ],
   [
    "C107",
    "x[5,War and Peace]"
   ],
   [
    "C108",
    "x[5,The Odyssey]"
   ],
   [
    "C109",
    "x[5,Crime and Punishment]"
   ],
   [
    "C110",
    "x[5,The Hobbit]"
   ],
   [
    "C111",
    "x[5,Brave New World]"
   ],
   [
    "C112",
    "x[5,Anna Karenina]"
   ],
   [
    "C113",
    "x[5,Wuthering Heights]"
   ],
   [
    "C114",
    "x[5,The Divine Comedy]"
   ],
   [
    "C115",
    "x[5,The Iliad]"
   ],
   [
    "C116",
    "x[5,Les Misérables]"
   ],
   [
    "C117",
    "x[5,Dracula]"
   ],
   [
    "C118",
    "x[5,Frankenstein]"
   ],
   [
    "C119",
    "x[5,The Brothers Karamazov]"
   ],
   [
    "C120",
    "x[5,Don Quixote]"
   ],
   [
    "C121",
    "x[5,One Hundred Years of Solitude]"
   ],
   [
    "C122",
    "x[5,Ulysses]"
   ],
   [
    "C123",
    "x[5,The Alchemist]"
   ],
   [
    "C124",
    "x[5,Meditations]"
   ],
   [
    "C125",
    "x[6,The Great Gatsby]"
   ],
   [
    "C126",
    "x[6,To Kill a Mockingbird]"
   ],
   [
    "C127",
    "x[6,1984]"
   ],
   [
    "C128",
    "x[6,Pride and Prejudice]"
   ],
   [
    "C129",
    "x[6,The Catcher in the Rye]"
   ],
   [
    "C130",
    "x[6,Moby Dick]"
   ],
   [
    "C131",
    "x[6,Jane Eyre]"
   ],
   [
    "C132",
    "x[6,War and Peace]"
   ],
   [
    "C133",
    "x[6,The Odyssey]"
   ],
   [
    "C134",
    "x[6,Crime and Punishment]"
   ],
   [
    "C135",
    "x[6,The Hobbit]"
   ],
   [
    "C136",
    "x[6,Brave New World]"
   ],
   [
    "C137",
    "x[6,Anna Karenina]"
   ],
   [
    "C138",
    "x[6,Wuthering Heights]"
   ],
   [
    "C139",
    "x[6,The Divine Comedy]"
   ],
   [
    "C140",
    "x[6,The Iliad]"
   ],
   [
    "C141",
    "x[6,Les Misérables]"
   ],
   [
    "C142",
    "x[6,Dracula]"
   ],
   [
    "C143",
    "x[6,Frankenstein]"
   ],
   [
    "C144",
    "x[6,The Brothers Karamazov]"
   ],
   [
    "C145",
    "x[6,Don Quixote]"
   ],
   [
    "C146",
    "x[6,One Hundred Years of Solitude]"
   ],
   [
    "C147",
    "x[6,Ulysses]"
   ],
   [
    "C148",
    "x[6,The Alchemist]"
   ],
   [
    "C149",
    "x[6,Meditations]"
   ],
   [
    "C150",
    "x[7,The Great Gatsby]"
   ],
   [
    "C151",
    "x[7,To Kill a Mockingbird]"
   ],
   [
    "C152",
    "x[7,1984]"
   ],
   [
    "C153",
    "x[7,Pride and Prejudice]"
   ],
   [
    "C154",
    "x[7,The Catcher in the Rye]"
   ],
   [
    "C155",
    "x[7,Moby Dick]"
   ],
   [
    "C156",
    "x[7,Jane Eyre]"
   ],
   [
    "C157",
    "x[7,War and Peace]"
   ],
   [
    "C158",
    "x[7,The Odyssey]"
   ],
   [
    "C159",
    "x[7,Crime and Punishment]"
   ],
   [
    "C160",
    "x[7,The Hobbit]"
   ],
   [
    "C161",
    "x[7,Brave New World]"
   ],
   [
    "C162",
    "x[7,Anna Karenina]"
   ],
   [
    "C163",
    "x[7,Wuthering Heights]"
   ],
   [
    "C164",
    "x[7,The Divine Comedy]"
   ],
   [
    "C165",
    "x[7,The Iliad]"
   ],
   [
    "C166",
    "x[7,Les Misérables]"
   ],
   [
    "C167",
    "x[7,Dracula]"
   ],
   [
    "C168",
    "x[7,Frankenstein]"
   ],
   [
    "C169",
    "x[7,The Brothers Karamazov]"
   ],
   [
    "C170",
    "x[7,Don Quixote]"
   ],
   [
    "C171",
    "x[7,One Hundred Years of Solitude]"
   ],
   [
    "C172",
    "x[7,Ulysses]"
   ],
   [
    "C173",
    "x[7,The Alchemist]"
   ],
   [
    "C174",
    "x[7,Meditations]"
   ],
   [
    "C175",
    "x[8,The Great Gatsby]"
   ],
   [
    "C176",
    "x[8,To Kill a Mockingbird]"
   ],
   [
    "C177",
    "x[8,1984]"
   ],
   [
    "C178",
    "x[8,Pride and Prejudice]"
   ],
   [
    "C179",
    "x[8,The Catcher in the Rye]"
   ],
   [
    "C180",
    "x[8,Moby Dick]"
   ],
   [
    "C181",
    "x[8,Jane Eyre]"
   ],
   [
    "C182",
    "x[8,War and Peace]"
   ],
   [
    "C183",
    "x[8,The Odyssey]"
   ],
   [
    "C184",
    "x[8,Crime and Punishment]"
   ],
   [
    "C185",
    "x[8,The Hobbit]"
   ],
   [
    "C186",
    "x[8,Brave New World]"
   ],
   [
    "C187",
    "x[8,Anna Karenina]"
   ],
   [
    "C188",
    "x[8,Wuthering Heights]"
   ],
   [
    "C189",
    "x[8,The Divine Comedy]"
   ],
   [
    "C190",
    "x[8,The Iliad]"
   ],
   [
    "C191",
    "x[8,Les Misérables]"
   ],
   [
    "C192",
    "x[8,Dracula]"
   ],
   [
    "C193",
    "x[8,Frankenstein]"
   ],
   [
    "C194",
    "x[8,The Brothers Karamazov]"
   ],
   [
    "C195",
    "x[8,Don Quixote]"
   ],
   [
    "C196",
    "x[8,One Hundred Years of Solitude]"
   ],
   [
    "C197",
    "x[8,Ulysses]"
   ],
   [
    "C198",
    "x[8,The Alchemist]"
   ],
   [
    "C199",
    "x[8,Meditations]"
   ],
   [
    "C200",
    "x[9,The Great Gatsby]"
   ],
   [
    "C201",
    "x[9,To Kill a Mockingbird]"
   ],
   [
    "C202",
    "x[9,1984]"
   ],
   [
    "C203",
    "x[9,Pride and Prejudice]"
   ],
   [
    "C204",
    "x[9,The Catcher in the Rye]"
   ],
   [
    "C205",
    "x[9,Moby Dick]"
   ],
   [
    "C206",
    "x[9,Jane Eyre]"
   ],
   [
    "C207",
    "x[9,War and Peace]"
   ],
   [
    "C208",
    "x[9,The Odyssey]"
   ],
   [
    "C209",
    "x[9,Crime and Punishment]"
   ],
   [
    "C210",
    "x[9,The Hobbit]"
   ],
   [
    "C211",
    "x[9,Brave New World]"
   ],
   [
    "C212",
    "x[9,Anna Karenina]"
   ],
   [
    "C213",
    "x[9,Wuthering Heights]"
   ],
   [
    "C214",
    "x[9,The Divine Comedy]"
   ],
   [
    "C215",
    "x[9,The Iliad]"
   ],
   [
    "C216",
    "x[9,Les Misérables]"
   ],
   [
    "C217",
    "x[9,Dracula]"
   ],
   [
    "C218",
    "x[9,Frankenstein]"
   ],
   [
    "C219",
    "x[9,The Brothers Karamazov]"
   ],
   [
    "C220",
    "x[9,Don Quixote]"
   ],
   [
    "C221",
    "x[9,One Hundred Years of Solitude]"
   ],
   [
    "C222",
    "x[9,Ulysses]"
   ],
   [
    "C223",
    "x[9,The Alchemist]"
   ],
   [
    "C224",
    "x[9,Meditations]"
   ],
   [
    "C225",
    "x[10,The Great Gatsby]"
   ],
   [
    "C226",
    "x[10,To Kill a Mockingbird]"
   ],
   [
    "C227",
    "x[10,1984]"
   ],
   [
    "C228",
    "x[10,Pride and Prejudice]"
   ],
   [
    "C229",
    "x[10,The Catcher in the Rye]"
   ],
   [
    "C230",
    "x[10,Moby Dick]"
   ],
   [
    "C231",
    "x[10,Jane Eyre]"
   ],
   [
    "C232",
    "x[10,War and Peace]"
   ],
   [
    "C233",
    "x[10,The Odyssey]"
   ],
   [
    "C234",
    "x[10,Crime and Punishment]"
   ],
   [
    "C235",
    "x[10,The Hobbit]"
   ],
   [
    "C236",
    "x[10,Brave New World]"
   ],
   [
    "C237",
    "x[10,Anna Karenina]"
   ],
   [
    "C238",
    "x[10,Wuthering Heights]"
   ],
   [
    "C239",
    "x[10,The Divine Comedy]"
   ],
   [
    "C240",
    "x[10,The Iliad]"
   ],
   [
    "C241",
    "x[10,Les Misérables]"
   ],
   [
    "C242",
    "x[10,Dracula]"
   ],
   [
    "C243",
    "x[10,Frankenstein]"
   ],
   [
    "C244",
    "x[10,The Brothers Karamazov]"
   ],
   [
    "C245",
    "x[10,Don Quixote]"
   ],
   [
    "C246",
    "x[10,One Hundred Years of Solitude]"
   ],
   [
    "C247",
    "x[10,Ulysses]"
   ],
   [
    "C248",
    "x[10,The Alchemist]"
   ],
   [
    "C249",
    "x[10,Meditations]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,The Great Gatsby]",
   "x_0_1": "x[2,The Great Gatsby]",
   "x_0_2": "x[3,The Great Gatsby]",
   "x_0_3": "x[4,The Great Gatsby]",
   "x_0_4": "x[5,The Great Gatsby]",
   "x_0_5": "x[6,The Great Gatsby]",
   "x_0_6": "x[7,The Great Gatsby]",
   "x_0_7": "x[8,The Great Gatsby]",
   "x_0_8": "x[9,The Great Gatsby]",
   "x_0_9": "x[10,The Great Gatsby]",
   "x_1_0": "x[1,To Kill a Mockingbird]",
   "x_1_1": "x[2,To Kill a Mockingbird]",
   "x_1_2": "x[3,To Kill a Mockingbird]",
   "x_1_3": "x[4,To Kill a Mockingbird]",
   "x_1_4": "x[5,To Kill a Mockingbird]",
   "x_1_5": "x[6,To Kill a Mockingbird]",
   "x_1_6": "x[7,To Kill a Mockingbird]",
   "x_1_7": "x[8,To Kill a Mockingbird]",
   "x_1_8": "x[9,To Kill a Mockingbird]",
   "x_1_9": "x[10,To Kill a Mockingbird]",
   "x_2_0": "x[1,1984]",
   "x_2_1": "x[2,1984]",
   "x_2_2": "x[3,1984]",
   "x_2_3": "x[4,1984]",
   "x_2_4": "x[5,1984]",
   "x_2_5": "x[6,1984]",
   "x_2_6": "x[7,1984]",
   "x_2_7": "x[8,1984]",
   "x_2_8": "x[9,1984]",
   "x_2_9": "x[10,1984]",
   "x_3_0": "x[1,Pride and Prejudice]",
   "x_3_1": "x[2,Pride and Prejudice]",
   "x_3_2": "x[3,Pride and Prejudice]",
   "x_3_3": "x[4,Pride and Prejudice]",
   "x_3_4": "x[5,Pride and Prejudice]",
   "x_3_5": "x[6,Pride and Prejudice]",
   "x_3_6": "x[7,Pride and Prejudice]",
   "x_3_7": "x[8,Pride and Prejudice]",
   "x_3_8": "x[9,Pride and Prejudice]",
   "x_3_9": "x[10,Pride and Prejudice]",
   "x_4_0": "x[1,The Catcher in the Rye]",
   "x_4_1": "x[2,The Catcher in the Rye]",
   "x_4_2": "x[3,The Catcher in the Rye]",
   "x_4_3": "x[4,The Catcher in the Rye]",
   "x_4_4": "x[5,The Catcher in the Rye]",
   "x_4_5": "x[6,The Catcher in the Rye]",
   "x_4_6": "x[7,The Catcher in the Rye]",
   "x_4_7": "x[8,The Catcher in the Rye]",
   "x_4_8": "x[9,The Catcher in the Rye]",
   "x_4_9": "x[10,The Catcher in the Rye]",
   "x_5_0": "x[1,Moby Dick]",
   "x_5_1": "x[2,Moby Dick]",
   "x_5_2": "x[3,Moby Dick]",
   "x_5_3": "x[4,Moby Dick]",
   "x_5_4": "x[5,Moby Dick]",
   "x_5_5": "x[6,Moby Dick]",
   "x_5_6": "x[7,Moby Dick]",
   "x_5_7": "x[8,Moby Dick]",
   "x_5_8": "x[9,Moby Dick]",
   "x_5_9": "x[10,Moby Dick]",
   "x_6_0": "x[1,Jane Eyre]",
   "x_6_1": "x[2,Jane Eyre]",
   "x_6_2": "x[3,Jane Eyre]",
   "x_6_3": "x[4,Jane Eyre]",
   "x_6_4": "x[5,Jane Eyre]",
   "x_6_5": "x[6,Jane Eyre]",
   "x_6_6": "x[7,Jane Eyre]",
   "x_6_7": "x[8,Jane Eyre]",
   "x_6_8": "x[9,Jane Eyre]",
   "x_6_9": "x[10,Jane Eyre]",
   "x_7_0": "x[1,War and Peace]",
   "x_7_1": "x[2,War and Peace]",
   "x_7_2": "x[3,War and Peace]",
   "x_7_3": "x[4,War and Peace]",
   "x_7_4": "x[5,War and Peace]",
   "x_7_5": "x[6,War and Peace]",
   "x_7_6": "x[7,War and Peace]",
   "x_7_7": "x[8,War and Peace]",
   "x_7_8": "x[9,War and Peace]",
   "x_7_9": "x[10,War and Peace]",
   "x_8_0": "x[1,The Odyssey]",
   "x_8_1": "x[2,The Odyssey]",
   "x_8_2": "x[3,The Odyssey]",
   "x_8_3": "x[4,The Odyssey]",
   "x_8_4": "x[5,The Odyssey]",
   "x_8_5": "x[6,The Odyssey]",
   "x_8_6": "x[7,The Odyssey]",
   "x_8_7": "x[8,The Odyssey]",
   "x_8_8": "x[9,The Odyssey]",
   "x_8_9": "x[10,The Odyssey]",
   "x_9_0": "x[1,Crime and Punishment]",
   "x_9_1": "x[2,Crime and Punishment]",
   "x_9_2": "x[3,Crime and Punishment]",
   "x_9_3": "x[4,Crime and Punishment]",
   "x_9_4": "x[5,Crime and Punishment]",
   "x_9_5": "x[6,Crime and Punishment]",
   "x_9_6": "x[7,Crime and Punishment]",
   "x_9_7": "x[8,Crime and Punishment]",
   "x_9_8": "x[9,Crime and Punishment]",
   "x_9_9": "x[10,Crime and Punishment]",
   "x_10_0": "x[1,The Hobbit]",
   "x_10_1": "x[2,The Hobbit]",
   "x_10_2": "x[3,The Hobbit]",
   "x_10_3": "x[4,The Hobbit]",
   "x_10_4": "x[5,The Hobbit]",
   "x_10_5": "x[6,The Hobbit]",
   "x_10_6": "x[7,The Hobbit]",
   "x_10_7": "x[8,The Hobbit]",
   "x_10_8": "x[9,The Hobbit]",
   "x_10_9": "x[10,The Hobbit]",
   "x_11_0": "x[1,Brave New World]",
   "x_11_1": "x[2,Brave New World]",
   "x_11_2": "x[3,Brave New World]",
   "x_11_3": "x[4,Brave New World]",
   "x_11_4": "x[5,Brave New World]",
   "x_11_5": "x[6,Brave New World]",
   "x_11_6": "x[7,Brave New World]",
   "x_11_7": "x[8,Brave New World]",
   "x_11_8": "x[9,Brave New World]",
   "x_11_9": "x[10,Brave New World]",
   "x_12_0": "x[1,Anna Karenina]",
   "x_12_1": "x[2,Anna Karenina]",
   "x_12_2": "x[3,Anna Karenina]",
   "x_12_3": "x[4,Anna Karenina]",
   "x_12_4": "x[5,Anna Karenina]",
   "x_12_5": "x[6,Anna Karenina]",
   "x_12_6": "x[7,Anna Karenina]",
   "x_12_7": "x[8,Anna Karenina]",
   "x_12_8": "x[9,Anna Karenina]",
   "x_12_9": "x[10,Anna Karenina]",
   "x_13_0": "x[1,Wuthering Heights]",
   "x_13_1": "x[2,Wuthering Heights]",
   "x_13_2": "x[3,Wuthering Heights]",
   "x_13_3": "x[4,Wuthering Heights]",
   "x_13_4": "x[5,Wuthering Heights]",
   "x_13_5": "x[6,Wuthering Heights]",
   "x_13_6": "x[7,Wuthering Heights]",
   "x_13_7": "x[8,Wuthering Heights]",
   "x_13_8": "x[9,Wuthering Heights]",
   "x_13_9": "x[10,Wuthering Heights]",
   "x_14_0": "x[1,The Divine Comedy]",
   "x_14_1": "x[2,The Divine Comedy]",
   "x_14_2": "x[3,The Divine Comedy]",
   "x_14_3": "x[4,The Divine Comedy]",
   "x_14_4": "x[5,The Divine Comedy]",
   "x_14_5": "x[6,The Divine Comedy]",
   "x_14_6": "x[7,The Divine Comedy]",
   "x_14_7": "x[8,The Divine Comedy]",
   "x_14_8": "x[9,The Divine Comedy]",
   "x_14_9": "x[10,The Divine Comedy]",
   "x_15_0": "x[1,The Iliad]",
   "x_15_1": "x[2,The Iliad]",
   "x_15_2": "x[3,The Iliad]",
   "x_15_3": "x[4,The Iliad]",
   "x_15_4": "x[5,The Iliad]",
   "x_15_5": "x[6,The Iliad]",
   "x_15_6": "x[7,The Iliad]",
   "x_15_7": "x[8,The Iliad]",
   "x_15_8": "x[9,The Iliad]",
   "x_15_9": "x[10,The Iliad]",
   "x_16_0": "x[1,Les Misérables]",
   "x_16_1": "x[2,Les Misérables]",
   "x_16_2": "x[3,Les Misérables]",
   "x_16_3": "x[4,Les Misérables]",
   "x_16_4": "x[5,Les Misérables]",
   "x_16_5": "x[6,Les Misérables]",
   "x_16_6": "x[7,Les Misérables]",
   "x_16_7": "x[8,Les Misérables]",
   "x_16_8": "x[9,Les Misérables]",
   "x_16_9": "x[10,Les Misérables]",
   "x_17_0": "x[1,Dracula]",
   "x_17_1": "x[2,Dracula]",
   "x_17_2": "x[3,Dracula]",
   "x_17_3": "x[4,Dracula]",
   "x_17_4": "x[5,Dracula]",
   "x_17_5": "x[6,Dracula]",
   "x_17_6": "x[7,Dracula]",
   "x_17_7": "x[8,Dracula]",
   "x_17_8": "x[9,Dracula]",
   "x_17_9": "x[10,Dracula]",
   "x_18_0": "x[1,Frankenstein]",
   "x_18_1": "x[2,Frankenstein]",
   "x_18_2": "x[3,Frankenstein]",
   "x_18_3": "x[4,Frankenstein]",
   "x_18_4": "x[5,Frankenstein]",
   "x_18_5": "x[6,Frankenstein]",
   "x_18_6": "x[7,Frankenstein]",
   "x_18_7": "x[8,Frankenstein]",
   "x_18_8": "x[9,Frankenstein]",
   "x_18_9": "x[10,Frankenstein]",
   "x_19_0": "x[1,The Brothers Karamazov]",
   "x_19_1": "x[2,The Brothers Karamazov]",
   "x_19_2": "x[3,The Brothers Karamazov]",
   "x_19_3": "x[4,The Brothers Karamazov]",
   "x_19_4": "x[5,The Brothers Karamazov]",
   "x_19_5": "x[6,The Brothers Karamazov]",
   "x_19_6": "x[7,The Brothers Karamazov]",
   "x_19_7": "x[8,The Brothers Karamazov]",
   "x_19_8": "x[9,The Brothers Karamazov]",
   "x_19_9": "x[10,The Brothers Karamazov]",
   "x_20_0": "x[1,Don Quixote]",
   "x_20_1": "x[2,Don Quixote]",
   "x_20_2": "x[3,Don Quixote]",
   "x_20_3": "x[4,Don Quixote]",
   "x_20_4": "x[5,Don Quixote]",
   "x_20_5": "x[6,Don Quixote]",
   "x_20_6": "x[7,Don Quixote]",
   "x_20_7": "x[8,Don Quixote]",
   "x_20_8": "x[9,Don Quixote]",
   "x_20_9": "x[10,Don Quixote]",
   "x_21_0": "x[1,One Hundred Years of Solitude]",
   "x_21_1": "x[2,One Hundred Years of Solitude]",
   "x_21_2": "x[3,One Hundred Years of Solitude]",
   "x_21_3": "x[4,One Hundred Years of Solitude]",
   "x_21_4": "x[5,One Hundred Years of Solitude]",
   "x_21_5": "x[6,One Hundred Years of Solitude]",
   "x_21_6": "x[7,One Hundred Years of Solitude]",
   "x_21_7": "x[8,One Hundred Years of Solitude]",
   "x_21_8": "x[9,One Hundred Years of Solitude]",
   "x_21_9": "x[10,One Hundred Years of Solitude]",
   "x_22_0": "x[1,Ulysses]",
   "x_22_1": "x[2,Ulysses]",
   "x_22_2": "x[3,Ulysses]",
   "x_22_3": "x[4,Ulysses]",
   "x_22_4": "x[5,Ulysses]",
   "x_22_5": "x[6,Ulysses]",
   "x_22_6": "x[7,Ulysses]",
   "x_22_7": "x[8,Ulysses]",
   "x_22_8": "x[9,Ulysses]",
   "x_22_9": "x[10,Ulysses]",
   "x_23_0": "x[1,The Alchemist]",
   "x_23_1": "x[2,The Alchemist]",
   "x_23_2": "x[3,The Alchemist]",
   "x_23_3": "x[4,The Alchemist]",
   "x_23_4": "x[5,The Alchemist]",
   "x_23_5": "x[6,The Alchemist]",
   "x_23_6": "x[7,The Alchemist]",
   "x_23_7": "x[8,The Alchemist]",
   "x_23_8": "x[9,The Alchemist]",
   "x_23_9": "x[10,The Alchemist]",
   "x_24_0": "x[1,Meditations]",
   "x_24_1": "x[2,Meditations]",
   "x_24_2": "x[3,Meditations]",
   "x_24_3": "x[4,Meditations]",
   "x_24_4": "x[5,Meditations]",
   "x_24_5": "x[6,Meditations]",
   "x_24_6": "x[7,Meditations]",
   "x_24_7": "x[8,Meditations]",
   "x_24_8": "x[9,Meditations]",
   "x_24_9": "x[10,Meditations]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 30240.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     40.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     40.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     60.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     80.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     110.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     120.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     130.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     150.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     -0.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ],
    [
     "C200",
     -0.0
    ],
    [
     "C201",
     -0.0
    ],
    [
     "C202",
     164.0
    ],
    [
     "C203",
     -0.0
    ],
    [
     "C204",
     -0.0
    ],
    [
     "C205",
     -0.0
    ],
    [
     "C206",
     -0.0
    ],
    [
     "C207",
     -0.0
    ],
    [
     "C208",
     -0.0
    ],
    [
     "C209",
     -0.0
    ],
    [
     "C210",
     -0.0
    ],
    [
     "C211",
     -0.0
    ],
    [
     "C212",
     -0.0
    ],
    [
     "C213",
     -0.0
    ],
    [
     "C214",
     -0.0
    ],
    [
     "C215",
     -0.0
    ],
    [
     "C216",
     -0.0
    ],
    [
     "C217",
     -0.0
    ],
    [
     "C218",
     -0.0
    ],
    [
     "C219",
     -0.0
    ],
    [
     "C220",
     -0.0
    ],
    [
     "C221",
     -0.0
    ],
    [
     "C222",
     -0.0
    ],
    [
     "C223",
     -0.0
    ],
    [
     "C224",
     -0.0
    ],
    [
     "C225",
     -0.0
    ],
    [
     "C226",
     -0.0
    ],
    [
     "C227",
     114.0
    ],
    [
     "C228",
     -0.0
    ],
    [
     "C229",
     -0.0
    ],
    [
     "C230",
     -0.0
    ],
    [
     "C231",
     -0.0
    ],
    [
     "C232",
     -0.0
    ],
    [
     "C233",
     -0.0
    ],
    [
     "C234",
     -0.0
    ],
    [
     "C235",
     -0.0
    ],
    [
     "C236",
     -0.0
    ],
    [
     "C237",
     -0.0
    ],
    [
     "C238",
     -0.0
    ],
    [
     "C239",
     -0.0
    ],
    [
     "C240",
     -0.0
    ],
    [
     "C241",
     -0.0
    ],
    [
     "C242",
     -0.0
    ],
    [
     "C243",
     -0.0
    ],
    [
     "C244",
     -0.0
    ],
    [
     "C245",
     -0.0
    ],
    [
     "C246",
     -0.0
    ],
    [
     "C247",
     -0.0
    ],
    [
     "C248",
     -0.0
    ],
    [
     "C249",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-056": {
  "source_code_sha256": "c880e92f1c1c4907a2f9f03c32637b783d515b35365c7d61a74cfc7c67261f88",
  "original_solution_sha256": "ec3f37c07a4743a464dfaab2feb60b9233d476a401f284ea59d71dd41cdbf9ac",
  "bindings": [
   [
    "C0",
    "x[1,Speedboat]"
   ],
   [
    "C1",
    "x[1,Fishing Boat]"
   ],
   [
    "C2",
    "x[1,Catamaran]"
   ],
   [
    "C3",
    "x[1,Yacht]"
   ],
   [
    "C4",
    "x[1,Sailboat]"
   ],
   [
    "C5",
    "x[1,Kayak]"
   ],
   [
    "C6",
    "x[1,Canoe]"
   ],
   [
    "C7",
    "x[1,Houseboat]"
   ],
   [
    "C8",
    "x[1,Pontoon]"
   ],
   [
    "C9",
    "x[1,Jet Ski]"
   ],
   [
    "C10",
    "x[1,Rowboat]"
   ],
   [
    "C11",
    "x[1,Hovercraft]"
   ],
   [
    "C12",
    "x[1,Cabin Cruiser]"
   ],
   [
    "C13",
    "x[1,Wakeboard Boat]"
   ],
   [
    "C14",
    "x[1,Dinghy]"
   ],
   [
    "C15",
    "x[1,Trawler]"
   ],
   [
    "C16",
    "x[1,Paddle Boat]"
   ],
   [
    "C17",
    "x[1,Submarine]"
   ],
   [
    "C18",
    "x[1,RIB]"
   ],
   [
    "C19",
    "x[1,Skiff]"
   ],
   [
    "C20",
    "x[2,Speedboat]"
   ],
   [
    "C21",
    "x[2,Fishing Boat]"
   ],
   [
    "C22",
    "x[2,Catamaran]"
   ],
   [
    "C23",
    "x[2,Yacht]"
   ],
   [
    "C24",
    "x[2,Sailboat]"
   ],
   [
    "C25",
    "x[2,Kayak]"
   ],
   [
    "C26",
    "x[2,Canoe]"
   ],
   [
    "C27",
    "x[2,Houseboat]"
   ],
   [
    "C28",
    "x[2,Pontoon]"
   ],
   [
    "C29",
    "x[2,Jet Ski]"
   ],
   [
    "C30",
    "x[2,Rowboat]"
   ],
   [
    "C31",
    "x[2,Hovercraft]"
   ],
   [
    "C32",
    "x[2,Cabin Cruiser]"
   ],
   [
    "C33",
    "x[2,Wakeboard Boat]"
   ],
   [
    "C34",
    "x[2,Dinghy]"
   ],
   [
    "C35",
    "x[2,Trawler]"
   ],
   [
    "C36",
    "x[2,Paddle Boat]"
   ],
   [
    "C37",
    "x[2,Submarine]"
   ],
   [
    "C38",
    "x[2,RIB]"
   ],
   [
    "C39",
    "x[2,Skiff]"
   ],
   [
    "C40",
    "x[3,Speedboat]"
   ],
   [
    "C41",
    "x[3,Fishing Boat]"
   ],
   [
    "C42",
    "x[3,Catamaran]"
   ],
   [
    "C43",
    "x[3,Yacht]"
   ],
   [
    "C44",
    "x[3,Sailboat]"
   ],
   [
    "C45",
    "x[3,Kayak]"
   ],
   [
    "C46",
    "x[3,Canoe]"
   ],
   [
    "C47",
    "x[3,Houseboat]"
   ],
   [
    "C48",
    "x[3,Pontoon]"
   ],
   [
    "C49",
    "x[3,Jet Ski]"
   ],
   [
    "C50",
    "x[3,Rowboat]"
   ],
   [
    "C51",
    "x[3,Hovercraft]"
   ],
   [
    "C52",
    "x[3,Cabin Cruiser]"
   ],
   [
    "C53",
    "x[3,Wakeboard Boat]"
   ],
   [
    "C54",
    "x[3,Dinghy]"
   ],
   [
    "C55",
    "x[3,Trawler]"
   ],
   [
    "C56",
    "x[3,Paddle Boat]"
   ],
   [
    "C57",
    "x[3,Submarine]"
   ],
   [
    "C58",
    "x[3,RIB]"
   ],
   [
    "C59",
    "x[3,Skiff]"
   ],
   [
    "C60",
    "x[4,Speedboat]"
   ],
   [
    "C61",
    "x[4,Fishing Boat]"
   ],
   [
    "C62",
    "x[4,Catamaran]"
   ],
   [
    "C63",
    "x[4,Yacht]"
   ],
   [
    "C64",
    "x[4,Sailboat]"
   ],
   [
    "C65",
    "x[4,Kayak]"
   ],
   [
    "C66",
    "x[4,Canoe]"
   ],
   [
    "C67",
    "x[4,Houseboat]"
   ],
   [
    "C68",
    "x[4,Pontoon]"
   ],
   [
    "C69",
    "x[4,Jet Ski]"
   ],
   [
    "C70",
    "x[4,Rowboat]"
   ],
   [
    "C71",
    "x[4,Hovercraft]"
   ],
   [
    "C72",
    "x[4,Cabin Cruiser]"
   ],
   [
    "C73",
    "x[4,Wakeboard Boat]"
   ],
   [
    "C74",
    "x[4,Dinghy]"
   ],
   [
    "C75",
    "x[4,Trawler]"
   ],
   [
    "C76",
    "x[4,Paddle Boat]"
   ],
   [
    "C77",
    "x[4,Submarine]"
   ],
   [
    "C78",
    "x[4,RIB]"
   ],
   [
    "C79",
    "x[4,Skiff]"
   ],
   [
    "C80",
    "x[5,Speedboat]"
   ],
   [
    "C81",
    "x[5,Fishing Boat]"
   ],
   [
    "C82",
    "x[5,Catamaran]"
   ],
   [
    "C83",
    "x[5,Yacht]"
   ],
   [
    "C84",
    "x[5,Sailboat]"
   ],
   [
    "C85",
    "x[5,Kayak]"
   ],
   [
    "C86",
    "x[5,Canoe]"
   ],
   [
    "C87",
    "x[5,Houseboat]"
   ],
   [
    "C88",
    "x[5,Pontoon]"
   ],
   [
    "C89",
    "x[5,Jet Ski]"
   ],
   [
    "C90",
    "x[5,Rowboat]"
   ],
   [
    "C91",
    "x[5,Hovercraft]"
   ],
   [
    "C92",
    "x[5,Cabin Cruiser]"
   ],
   [
    "C93",
    "x[5,Wakeboard Boat]"
   ],
   [
    "C94",
    "x[5,Dinghy]"
   ],
   [
    "C95",
    "x[5,Trawler]"
   ],
   [
    "C96",
    "x[5,Paddle Boat]"
   ],
   [
    "C97",
    "x[5,Submarine]"
   ],
   [
    "C98",
    "x[5,RIB]"
   ],
   [
    "C99",
    "x[5,Skiff]"
   ],
   [
    "C100",
    "x[6,Speedboat]"
   ],
   [
    "C101",
    "x[6,Fishing Boat]"
   ],
   [
    "C102",
    "x[6,Catamaran]"
   ],
   [
    "C103",
    "x[6,Yacht]"
   ],
   [
    "C104",
    "x[6,Sailboat]"
   ],
   [
    "C105",
    "x[6,Kayak]"
   ],
   [
    "C106",
    "x[6,Canoe]"
   ],
   [
    "C107",
    "x[6,Houseboat]"
   ],
   [
    "C108",
    "x[6,Pontoon]"
   ],
   [
    "C109",
    "x[6,Jet Ski]"
   ],
   [
    "C110",
    "x[6,Rowboat]"
   ],
   [
    "C111",
    "x[6,Hovercraft]"
   ],
   [
    "C112",
    "x[6,Cabin Cruiser]"
   ],
   [
    "C113",
    "x[6,Wakeboard Boat]"
   ],
   [
    "C114",
    "x[6,Dinghy]"
   ],
   [
    "C115",
    "x[6,Trawler]"
   ],
   [
    "C116",
    "x[6,Paddle Boat]"
   ],
   [
    "C117",
    "x[6,Submarine]"
   ],
   [
    "C118",
    "x[6,RIB]"
   ],
   [
    "C119",
    "x[6,Skiff]"
   ],
   [
    "C120",
    "x[7,Speedboat]"
   ],
   [
    "C121",
    "x[7,Fishing Boat]"
   ],
   [
    "C122",
    "x[7,Catamaran]"
   ],
   [
    "C123",
    "x[7,Yacht]"
   ],
   [
    "C124",
    "x[7,Sailboat]"
   ],
   [
    "C125",
    "x[7,Kayak]"
   ],
   [
    "C126",
    "x[7,Canoe]"
   ],
   [
    "C127",
    "x[7,Houseboat]"
   ],
   [
    "C128",
    "x[7,Pontoon]"
   ],
   [
    "C129",
    "x[7,Jet Ski]"
   ],
   [
    "C130",
    "x[7,Rowboat]"
   ],
   [
    "C131",
    "x[7,Hovercraft]"
   ],
   [
    "C132",
    "x[7,Cabin Cruiser]"
   ],
   [
    "C133",
    "x[7,Wakeboard Boat]"
   ],
   [
    "C134",
    "x[7,Dinghy]"
   ],
   [
    "C135",
    "x[7,Trawler]"
   ],
   [
    "C136",
    "x[7,Paddle Boat]"
   ],
   [
    "C137",
    "x[7,Submarine]"
   ],
   [
    "C138",
    "x[7,RIB]"
   ],
   [
    "C139",
    "x[7,Skiff]"
   ],
   [
    "C140",
    "x[8,Speedboat]"
   ],
   [
    "C141",
    "x[8,Fishing Boat]"
   ],
   [
    "C142",
    "x[8,Catamaran]"
   ],
   [
    "C143",
    "x[8,Yacht]"
   ],
   [
    "C144",
    "x[8,Sailboat]"
   ],
   [
    "C145",
    "x[8,Kayak]"
   ],
   [
    "C146",
    "x[8,Canoe]"
   ],
   [
    "C147",
    "x[8,Houseboat]"
   ],
   [
    "C148",
    "x[8,Pontoon]"
   ],
   [
    "C149",
    "x[8,Jet Ski]"
   ],
   [
    "C150",
    "x[8,Rowboat]"
   ],
   [
    "C151",
    "x[8,Hovercraft]"
   ],
   [
    "C152",
    "x[8,Cabin Cruiser]"
   ],
   [
    "C153",
    "x[8,Wakeboard Boat]"
   ],
   [
    "C154",
    "x[8,Dinghy]"
   ],
   [
    "C155",
    "x[8,Trawler]"
   ],
   [
    "C156",
    "x[8,Paddle Boat]"
   ],
   [
    "C157",
    "x[8,Submarine]"
   ],
   [
    "C158",
    "x[8,RIB]"
   ],
   [
    "C159",
    "x[8,Skiff]"
   ],
   [
    "C160",
    "x[9,Speedboat]"
   ],
   [
    "C161",
    "x[9,Fishing Boat]"
   ],
   [
    "C162",
    "x[9,Catamaran]"
   ],
   [
    "C163",
    "x[9,Yacht]"
   ],
   [
    "C164",
    "x[9,Sailboat]"
   ],
   [
    "C165",
    "x[9,Kayak]"
   ],
   [
    "C166",
    "x[9,Canoe]"
   ],
   [
    "C167",
    "x[9,Houseboat]"
   ],
   [
    "C168",
    "x[9,Pontoon]"
   ],
   [
    "C169",
    "x[9,Jet Ski]"
   ],
   [
    "C170",
    "x[9,Rowboat]"
   ],
   [
    "C171",
    "x[9,Hovercraft]"
   ],
   [
    "C172",
    "x[9,Cabin Cruiser]"
   ],
   [
    "C173",
    "x[9,Wakeboard Boat]"
   ],
   [
    "C174",
    "x[9,Dinghy]"
   ],
   [
    "C175",
    "x[9,Trawler]"
   ],
   [
    "C176",
    "x[9,Paddle Boat]"
   ],
   [
    "C177",
    "x[9,Submarine]"
   ],
   [
    "C178",
    "x[9,RIB]"
   ],
   [
    "C179",
    "x[9,Skiff]"
   ],
   [
    "C180",
    "x[10,Speedboat]"
   ],
   [
    "C181",
    "x[10,Fishing Boat]"
   ],
   [
    "C182",
    "x[10,Catamaran]"
   ],
   [
    "C183",
    "x[10,Yacht]"
   ],
   [
    "C184",
    "x[10,Sailboat]"
   ],
   [
    "C185",
    "x[10,Kayak]"
   ],
   [
    "C186",
    "x[10,Canoe]"
   ],
   [
    "C187",
    "x[10,Houseboat]"
   ],
   [
    "C188",
    "x[10,Pontoon]"
   ],
   [
    "C189",
    "x[10,Jet Ski]"
   ],
   [
    "C190",
    "x[10,Rowboat]"
   ],
   [
    "C191",
    "x[10,Hovercraft]"
   ],
   [
    "C192",
    "x[10,Cabin Cruiser]"
   ],
   [
    "C193",
    "x[10,Wakeboard Boat]"
   ],
   [
    "C194",
    "x[10,Dinghy]"
   ],
   [
    "C195",
    "x[10,Trawler]"
   ],
   [
    "C196",
    "x[10,Paddle Boat]"
   ],
   [
    "C197",
    "x[10,Submarine]"
   ],
   [
    "C198",
    "x[10,RIB]"
   ],
   [
    "C199",
    "x[10,Skiff]"
   ],
   [
    "C200",
    "x[11,Speedboat]"
   ],
   [
    "C201",
    "x[11,Fishing Boat]"
   ],
   [
    "C202",
    "x[11,Catamaran]"
   ],
   [
    "C203",
    "x[11,Yacht]"
   ],
   [
    "C204",
    "x[11,Sailboat]"
   ],
   [
    "C205",
    "x[11,Kayak]"
   ],
   [
    "C206",
    "x[11,Canoe]"
   ],
   [
    "C207",
    "x[11,Houseboat]"
   ],
   [
    "C208",
    "x[11,Pontoon]"
   ],
   [
    "C209",
    "x[11,Jet Ski]"
   ],
   [
    "C210",
    "x[11,Rowboat]"
   ],
   [
    "C211",
    "x[11,Hovercraft]"
   ],
   [
    "C212",
    "x[11,Cabin Cruiser]"
   ],
   [
    "C213",
    "x[11,Wakeboard Boat]"
   ],
   [
    "C214",
    "x[11,Dinghy]"
   ],
   [
    "C215",
    "x[11,Trawler]"
   ],
   [
    "C216",
    "x[11,Paddle Boat]"
   ],
   [
    "C217",
    "x[11,Submarine]"
   ],
   [
    "C218",
    "x[11,RIB]"
   ],
   [
    "C219",
    "x[11,Skiff]"
   ],
   [
    "C220",
    "x[12,Speedboat]"
   ],
   [
    "C221",
    "x[12,Fishing Boat]"
   ],
   [
    "C222",
    "x[12,Catamaran]"
   ],
   [
    "C223",
    "x[12,Yacht]"
   ],
   [
    "C224",
    "x[12,Sailboat]"
   ],
   [
    "C225",
    "x[12,Kayak]"
   ],
   [
    "C226",
    "x[12,Canoe]"
   ],
   [
    "C227",
    "x[12,Houseboat]"
   ],
   [
    "C228",
    "x[12,Pontoon]"
   ],
   [
    "C229",
    "x[12,Jet Ski]"
   ],
   [
    "C230",
    "x[12,Rowboat]"
   ],
   [
    "C231",
    "x[12,Hovercraft]"
   ],
   [
    "C232",
    "x[12,Cabin Cruiser]"
   ],
   [
    "C233",
    "x[12,Wakeboard Boat]"
   ],
   [
    "C234",
    "x[12,Dinghy]"
   ],
   [
    "C235",
    "x[12,Trawler]"
   ],
   [
    "C236",
    "x[12,Paddle Boat]"
   ],
   [
    "C237",
    "x[12,Submarine]"
   ],
   [
    "C238",
    "x[12,RIB]"
   ],
   [
    "C239",
    "x[12,Skiff]"
   ],
   [
    "C240",
    "x[13,Speedboat]"
   ],
   [
    "C241",
    "x[13,Fishing Boat]"
   ],
   [
    "C242",
    "x[13,Catamaran]"
   ],
   [
    "C243",
    "x[13,Yacht]"
   ],
   [
    "C244",
    "x[13,Sailboat]"
   ],
   [
    "C245",
    "x[13,Kayak]"
   ],
   [
    "C246",
    "x[13,Canoe]"
   ],
   [
    "C247",
    "x[13,Houseboat]"
   ],
   [
    "C248",
    "x[13,Pontoon]"
   ],
   [
    "C249",
    "x[13,Jet Ski]"
   ],
   [
    "C250",
    "x[13,Rowboat]"
   ],
   [
    "C251",
    "x[13,Hovercraft]"
   ],
   [
    "C252",
    "x[13,Cabin Cruiser]"
   ],
   [
    "C253",
    "x[13,Wakeboard Boat]"
   ],
   [
    "C254",
    "x[13,Dinghy]"
   ],
   [
    "C255",
    "x[13,Trawler]"
   ],
   [
    "C256",
    "x[13,Paddle Boat]"
   ],
   [
    "C257",
    "x[13,Submarine]"
   ],
   [
    "C258",
    "x[13,RIB]"
   ],
   [
    "C259",
    "x[13,Skiff]"
   ],
   [
    "C260",
    "x[14,Speedboat]"
   ],
   [
    "C261",
    "x[14,Fishing Boat]"
   ],
   [
    "C262",
    "x[14,Catamaran]"
   ],
   [
    "C263",
    "x[14,Yacht]"
   ],
   [
    "C264",
    "x[14,Sailboat]"
   ],
   [
    "C265",
    "x[14,Kayak]"
   ],
   [
    "C266",
    "x[14,Canoe]"
   ],
   [
    "C267",
    "x[14,Houseboat]"
   ],
   [
    "C268",
    "x[14,Pontoon]"
   ],
   [
    "C269",
    "x[14,Jet Ski]"
   ],
   [
    "C270",
    "x[14,Rowboat]"
   ],
   [
    "C271",
    "x[14,Hovercraft]"
   ],
   [
    "C272",
    "x[14,Cabin Cruiser]"
   ],
   [
    "C273",
    "x[14,Wakeboard Boat]"
   ],
   [
    "C274",
    "x[14,Dinghy]"
   ],
   [
    "C275",
    "x[14,Trawler]"
   ],
   [
    "C276",
    "x[14,Paddle Boat]"
   ],
   [
    "C277",
    "x[14,Submarine]"
   ],
   [
    "C278",
    "x[14,RIB]"
   ],
   [
    "C279",
    "x[14,Skiff]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Speedboat]",
   "x_0_1": "x[2,Speedboat]",
   "x_0_2": "x[3,Speedboat]",
   "x_0_3": "x[4,Speedboat]",
   "x_0_4": "x[5,Speedboat]",
   "x_0_5": "x[6,Speedboat]",
   "x_0_6": "x[7,Speedboat]",
   "x_0_7": "x[8,Speedboat]",
   "x_0_8": "x[9,Speedboat]",
   "x_0_9": "x[10,Speedboat]",
   "x_0_10": "x[11,Speedboat]",
   "x_0_11": "x[12,Speedboat]",
   "x_0_12": "x[13,Speedboat]",
   "x_0_13": "x[14,Speedboat]",
   "x_1_0": "x[1,Fishing Boat]",
   "x_1_1": "x[2,Fishing Boat]",
   "x_1_2": "x[3,Fishing Boat]",
   "x_1_3": "x[4,Fishing Boat]",
   "x_1_4": "x[5,Fishing Boat]",
   "x_1_5": "x[6,Fishing Boat]",
   "x_1_6": "x[7,Fishing Boat]",
   "x_1_7": "x[8,Fishing Boat]",
   "x_1_8": "x[9,Fishing Boat]",
   "x_1_9": "x[10,Fishing Boat]",
   "x_1_10": "x[11,Fishing Boat]",
   "x_1_11": "x[12,Fishing Boat]",
   "x_1_12": "x[13,Fishing Boat]",
   "x_1_13": "x[14,Fishing Boat]",
   "x_2_0": "x[1,Catamaran]",
   "x_2_1": "x[2,Catamaran]",
   "x_2_2": "x[3,Catamaran]",
   "x_2_3": "x[4,Catamaran]",
   "x_2_4": "x[5,Catamaran]",
   "x_2_5": "x[6,Catamaran]",
   "x_2_6": "x[7,Catamaran]",
   "x_2_7": "x[8,Catamaran]",
   "x_2_8": "x[9,Catamaran]",
   "x_2_9": "x[10,Catamaran]",
   "x_2_10": "x[11,Catamaran]",
   "x_2_11": "x[12,Catamaran]",
   "x_2_12": "x[13,Catamaran]",
   "x_2_13": "x[14,Catamaran]",
   "x_3_0": "x[1,Yacht]",
   "x_3_1": "x[2,Yacht]",
   "x_3_2": "x[3,Yacht]",
   "x_3_3": "x[4,Yacht]",
   "x_3_4": "x[5,Yacht]",
   "x_3_5": "x[6,Yacht]",
   "x_3_6": "x[7,Yacht]",
   "x_3_7": "x[8,Yacht]",
   "x_3_8": "x[9,Yacht]",
   "x_3_9": "x[10,Yacht]",
   "x_3_10": "x[11,Yacht]",
   "x_3_11": "x[12,Yacht]",
   "x_3_12": "x[13,Yacht]",
   "x_3_13": "x[14,Yacht]",
   "x_4_0": "x[1,Sailboat]",
   "x_4_1": "x[2,Sailboat]",
   "x_4_2": "x[3,Sailboat]",
   "x_4_3": "x[4,Sailboat]",
   "x_4_4": "x[5,Sailboat]",
   "x_4_5": "x[6,Sailboat]",
   "x_4_6": "x[7,Sailboat]",
   "x_4_7": "x[8,Sailboat]",
   "x_4_8": "x[9,Sailboat]",
   "x_4_9": "x[10,Sailboat]",
   "x_4_10": "x[11,Sailboat]",
   "x_4_11": "x[12,Sailboat]",
   "x_4_12": "x[13,Sailboat]",
   "x_4_13": "x[14,Sailboat]",
   "x_5_0": "x[1,Kayak]",
   "x_5_1": "x[2,Kayak]",
   "x_5_2": "x[3,Kayak]",
   "x_5_3": "x[4,Kayak]",
   "x_5_4": "x[5,Kayak]",
   "x_5_5": "x[6,Kayak]",
   "x_5_6": "x[7,Kayak]",
   "x_5_7": "x[8,Kayak]",
   "x_5_8": "x[9,Kayak]",
   "x_5_9": "x[10,Kayak]",
   "x_5_10": "x[11,Kayak]",
   "x_5_11": "x[12,Kayak]",
   "x_5_12": "x[13,Kayak]",
   "x_5_13": "x[14,Kayak]",
   "x_6_0": "x[1,Canoe]",
   "x_6_1": "x[2,Canoe]",
   "x_6_2": "x[3,Canoe]",
   "x_6_3": "x[4,Canoe]",
   "x_6_4": "x[5,Canoe]",
   "x_6_5": "x[6,Canoe]",
   "x_6_6": "x[7,Canoe]",
   "x_6_7": "x[8,Canoe]",
   "x_6_8": "x[9,Canoe]",
   "x_6_9": "x[10,Canoe]",
   "x_6_10": "x[11,Canoe]",
   "x_6_11": "x[12,Canoe]",
   "x_6_12": "x[13,Canoe]",
   "x_6_13": "x[14,Canoe]",
   "x_7_0": "x[1,Houseboat]",
   "x_7_1": "x[2,Houseboat]",
   "x_7_2": "x[3,Houseboat]",
   "x_7_3": "x[4,Houseboat]",
   "x_7_4": "x[5,Houseboat]",
   "x_7_5": "x[6,Houseboat]",
   "x_7_6": "x[7,Houseboat]",
   "x_7_7": "x[8,Houseboat]",
   "x_7_8": "x[9,Houseboat]",
   "x_7_9": "x[10,Houseboat]",
   "x_7_10": "x[11,Houseboat]",
   "x_7_11": "x[12,Houseboat]",
   "x_7_12": "x[13,Houseboat]",
   "x_7_13": "x[14,Houseboat]",
   "x_8_0": "x[1,Pontoon]",
   "x_8_1": "x[2,Pontoon]",
   "x_8_2": "x[3,Pontoon]",
   "x_8_3": "x[4,Pontoon]",
   "x_8_4": "x[5,Pontoon]",
   "x_8_5": "x[6,Pontoon]",
   "x_8_6": "x[7,Pontoon]",
   "x_8_7": "x[8,Pontoon]",
   "x_8_8": "x[9,Pontoon]",
   "x_8_9": "x[10,Pontoon]",
   "x_8_10": "x[11,Pontoon]",
   "x_8_11": "x[12,Pontoon]",
   "x_8_12": "x[13,Pontoon]",
   "x_8_13": "x[14,Pontoon]",
   "x_9_0": "x[1,Jet Ski]",
   "x_9_1": "x[2,Jet Ski]",
   "x_9_2": "x[3,Jet Ski]",
   "x_9_3": "x[4,Jet Ski]",
   "x_9_4": "x[5,Jet Ski]",
   "x_9_5": "x[6,Jet Ski]",
   "x_9_6": "x[7,Jet Ski]",
   "x_9_7": "x[8,Jet Ski]",
   "x_9_8": "x[9,Jet Ski]",
   "x_9_9": "x[10,Jet Ski]",
   "x_9_10": "x[11,Jet Ski]",
   "x_9_11": "x[12,Jet Ski]",
   "x_9_12": "x[13,Jet Ski]",
   "x_9_13": "x[14,Jet Ski]",
   "x_10_0": "x[1,Rowboat]",
   "x_10_1": "x[2,Rowboat]",
   "x_10_2": "x[3,Rowboat]",
   "x_10_3": "x[4,Rowboat]",
   "x_10_4": "x[5,Rowboat]",
   "x_10_5": "x[6,Rowboat]",
   "x_10_6": "x[7,Rowboat]",
   "x_10_7": "x[8,Rowboat]",
   "x_10_8": "x[9,Rowboat]",
   "x_10_9": "x[10,Rowboat]",
   "x_10_10": "x[11,Rowboat]",
   "x_10_11": "x[12,Rowboat]",
   "x_10_12": "x[13,Rowboat]",
   "x_10_13": "x[14,Rowboat]",
   "x_11_0": "x[1,Hovercraft]",
   "x_11_1": "x[2,Hovercraft]",
   "x_11_2": "x[3,Hovercraft]",
   "x_11_3": "x[4,Hovercraft]",
   "x_11_4": "x[5,Hovercraft]",
   "x_11_5": "x[6,Hovercraft]",
   "x_11_6": "x[7,Hovercraft]",
   "x_11_7": "x[8,Hovercraft]",
   "x_11_8": "x[9,Hovercraft]",
   "x_11_9": "x[10,Hovercraft]",
   "x_11_10": "x[11,Hovercraft]",
   "x_11_11": "x[12,Hovercraft]",
   "x_11_12": "x[13,Hovercraft]",
   "x_11_13": "x[14,Hovercraft]",
   "x_12_0": "x[1,Cabin Cruiser]",
   "x_12_1": "x[2,Cabin Cruiser]",
   "x_12_2": "x[3,Cabin Cruiser]",
   "x_12_3": "x[4,Cabin Cruiser]",
   "x_12_4": "x[5,Cabin Cruiser]",
   "x_12_5": "x[6,Cabin Cruiser]",
   "x_12_6": "x[7,Cabin Cruiser]",
   "x_12_7": "x[8,Cabin Cruiser]",
   "x_12_8": "x[9,Cabin Cruiser]",
   "x_12_9": "x[10,Cabin Cruiser]",
   "x_12_10": "x[11,Cabin Cruiser]",
   "x_12_11": "x[12,Cabin Cruiser]",
   "x_12_12": "x[13,Cabin Cruiser]",
   "x_12_13": "x[14,Cabin Cruiser]",
   "x_13_0": "x[1,Wakeboard Boat]",
   "x_13_1": "x[2,Wakeboard Boat]",
   "x_13_2": "x[3,Wakeboard Boat]",
   "x_13_3": "x[4,Wakeboard Boat]",
   "x_13_4": "x[5,Wakeboard Boat]",
   "x_13_5": "x[6,Wakeboard Boat]",
   "x_13_6": "x[7,Wakeboard Boat]",
   "x_13_7": "x[8,Wakeboard Boat]",
   "x_13_8": "x[9,Wakeboard Boat]",
   "x_13_9": "x[10,Wakeboard Boat]",
   "x_13_10": "x[11,Wakeboard Boat]",
   "x_13_11": "x[12,Wakeboard Boat]",
   "x_13_12": "x[13,Wakeboard Boat]",
   "x_13_13": "x[14,Wakeboard Boat]",
   "x_14_0": "x[1,Dinghy]",
   "x_14_1": "x[2,Dinghy]",
   "x_14_2": "x[3,Dinghy]",
   "x_14_3": "x[4,Dinghy]",
   "x_14_4": "x[5,Dinghy]",
   "x_14_5": "x[6,Dinghy]",
   "x_14_6": "x[7,Dinghy]",
   "x_14_7": "x[8,Dinghy]",
   "x_14_8": "x[9,Dinghy]",
   "x_14_9": "x[10,Dinghy]",
   "x_14_10": "x[11,Dinghy]",
   "x_14_11": "x[12,Dinghy]",
   "x_14_12": "x[13,Dinghy]",
   "x_14_13": "x[14,Dinghy]",
   "x_15_0": "x[1,Trawler]",
   "x_15_1": "x[2,Trawler]",
   "x_15_2": "x[3,Trawler]",
   "x_15_3": "x[4,Trawler]",
   "x_15_4": "x[5,Trawler]",
   "x_15_5": "x[6,Trawler]",
   "x_15_6": "x[7,Trawler]",
   "x_15_7": "x[8,Trawler]",
   "x_15_8": "x[9,Trawler]",
   "x_15_9": "x[10,Trawler]",
   "x_15_10": "x[11,Trawler]",
   "x_15_11": "x[12,Trawler]",
   "x_15_12": "x[13,Trawler]",
   "x_15_13": "x[14,Trawler]",
   "x_16_0": "x[1,Paddle Boat]",
   "x_16_1": "x[2,Paddle Boat]",
   "x_16_2": "x[3,Paddle Boat]",
   "x_16_3": "x[4,Paddle Boat]",
   "x_16_4": "x[5,Paddle Boat]",
   "x_16_5": "x[6,Paddle Boat]",
   "x_16_6": "x[7,Paddle Boat]",
   "x_16_7": "x[8,Paddle Boat]",
   "x_16_8": "x[9,Paddle Boat]",
   "x_16_9": "x[10,Paddle Boat]",
   "x_16_10": "x[11,Paddle Boat]",
   "x_16_11": "x[12,Paddle Boat]",
   "x_16_12": "x[13,Paddle Boat]",
   "x_16_13": "x[14,Paddle Boat]",
   "x_17_0": "x[1,Submarine]",
   "x_17_1": "x[2,Submarine]",
   "x_17_2": "x[3,Submarine]",
   "x_17_3": "x[4,Submarine]",
   "x_17_4": "x[5,Submarine]",
   "x_17_5": "x[6,Submarine]",
   "x_17_6": "x[7,Submarine]",
   "x_17_7": "x[8,Submarine]",
   "x_17_8": "x[9,Submarine]",
   "x_17_9": "x[10,Submarine]",
   "x_17_10": "x[11,Submarine]",
   "x_17_11": "x[12,Submarine]",
   "x_17_12": "x[13,Submarine]",
   "x_17_13": "x[14,Submarine]",
   "x_18_0": "x[1,RIB]",
   "x_18_1": "x[2,RIB]",
   "x_18_2": "x[3,RIB]",
   "x_18_3": "x[4,RIB]",
   "x_18_4": "x[5,RIB]",
   "x_18_5": "x[6,RIB]",
   "x_18_6": "x[7,RIB]",
   "x_18_7": "x[8,RIB]",
   "x_18_8": "x[9,RIB]",
   "x_18_9": "x[10,RIB]",
   "x_18_10": "x[11,RIB]",
   "x_18_11": "x[12,RIB]",
   "x_18_12": "x[13,RIB]",
   "x_18_13": "x[14,RIB]",
   "x_19_0": "x[1,Skiff]",
   "x_19_1": "x[2,Skiff]",
   "x_19_2": "x[3,Skiff]",
   "x_19_3": "x[4,Skiff]",
   "x_19_4": "x[5,Skiff]",
   "x_19_5": "x[6,Skiff]",
   "x_19_6": "x[7,Skiff]",
   "x_19_7": "x[8,Skiff]",
   "x_19_8": "x[9,Skiff]",
   "x_19_9": "x[10,Skiff]",
   "x_19_10": "x[11,Skiff]",
   "x_19_11": "x[12,Skiff]",
   "x_19_12": "x[13,Skiff]",
   "x_19_13": "x[14,Skiff]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 55642671.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     1.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     34.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     3.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     43.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     5.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     52.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     36.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     2.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     24.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     2.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     29.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     57.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     2.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     68.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     -0.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     3.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     49.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     -0.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     3.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     28.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     -0.0
    ],
    [
     "C199",
     -0.0
    ],
    [
     "C200",
     -0.0
    ],
    [
     "C201",
     -0.0
    ],
    [
     "C202",
     -0.0
    ],
    [
     "C203",
     -0.0
    ],
    [
     "C204",
     -0.0
    ],
    [
     "C205",
     -0.0
    ],
    [
     "C206",
     -0.0
    ],
    [
     "C207",
     -0.0
    ],
    [
     "C208",
     -0.0
    ],
    [
     "C209",
     2.0
    ],
    [
     "C210",
     -0.0
    ],
    [
     "C211",
     -0.0
    ],
    [
     "C212",
     24.0
    ],
    [
     "C213",
     -0.0
    ],
    [
     "C214",
     -0.0
    ],
    [
     "C215",
     -0.0
    ],
    [
     "C216",
     -0.0
    ],
    [
     "C217",
     -0.0
    ],
    [
     "C218",
     -0.0
    ],
    [
     "C219",
     -0.0
    ],
    [
     "C220",
     -0.0
    ],
    [
     "C221",
     -0.0
    ],
    [
     "C222",
     -0.0
    ],
    [
     "C223",
     -0.0
    ],
    [
     "C224",
     -0.0
    ],
    [
     "C225",
     -0.0
    ],
    [
     "C226",
     -0.0
    ],
    [
     "C227",
     -0.0
    ],
    [
     "C228",
     -0.0
    ],
    [
     "C229",
     3.0
    ],
    [
     "C230",
     -0.0
    ],
    [
     "C231",
     -0.0
    ],
    [
     "C232",
     66.0
    ],
    [
     "C233",
     -0.0
    ],
    [
     "C234",
     -0.0
    ],
    [
     "C235",
     -0.0
    ],
    [
     "C236",
     -0.0
    ],
    [
     "C237",
     -0.0
    ],
    [
     "C238",
     -0.0
    ],
    [
     "C239",
     -0.0
    ],
    [
     "C240",
     -0.0
    ],
    [
     "C241",
     -0.0
    ],
    [
     "C242",
     -0.0
    ],
    [
     "C243",
     -0.0
    ],
    [
     "C244",
     -0.0
    ],
    [
     "C245",
     -0.0
    ],
    [
     "C246",
     -0.0
    ],
    [
     "C247",
     -0.0
    ],
    [
     "C248",
     -0.0
    ],
    [
     "C249",
     2.0
    ],
    [
     "C250",
     -0.0
    ],
    [
     "C251",
     -0.0
    ],
    [
     "C252",
     50.0
    ],
    [
     "C253",
     -0.0
    ],
    [
     "C254",
     -0.0
    ],
    [
     "C255",
     -0.0
    ],
    [
     "C256",
     -0.0
    ],
    [
     "C257",
     -0.0
    ],
    [
     "C258",
     -0.0
    ],
    [
     "C259",
     -0.0
    ],
    [
     "C260",
     -0.0
    ],
    [
     "C261",
     -0.0
    ],
    [
     "C262",
     -0.0
    ],
    [
     "C263",
     -0.0
    ],
    [
     "C264",
     -0.0
    ],
    [
     "C265",
     -0.0
    ],
    [
     "C266",
     -0.0
    ],
    [
     "C267",
     -0.0
    ],
    [
     "C268",
     -0.0
    ],
    [
     "C269",
     1.0
    ],
    [
     "C270",
     -0.0
    ],
    [
     "C271",
     -0.0
    ],
    [
     "C272",
     67.0
    ],
    [
     "C273",
     -0.0
    ],
    [
     "C274",
     -0.0
    ],
    [
     "C275",
     -0.0
    ],
    [
     "C276",
     -0.0
    ],
    [
     "C277",
     -0.0
    ],
    [
     "C278",
     -0.0
    ],
    [
     "C279",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-060": {
  "source_code_sha256": "fd7d47b1decdc9e1dfa2a323741b5d70b6b12662c0b2ebfc3d40f584fbcd639a",
  "original_solution_sha256": "cff8298ca59b563be4bf80503b640e224ffc5c2b7c4c9da79866721f01bfd7f4",
  "bindings": [
   [
    "C0",
    "y[S1]"
   ],
   [
    "C1",
    "y[S2]"
   ],
   [
    "C2",
    "y[S3]"
   ],
   [
    "C3",
    "y[S4]"
   ],
   [
    "C4",
    "y[S5]"
   ],
   [
    "C5",
    "y[S6]"
   ],
   [
    "C6",
    "y[S7]"
   ],
   [
    "C7",
    "y[S8]"
   ],
   [
    "C8",
    "y[S9]"
   ],
   [
    "C9",
    "y[S10]"
   ],
   [
    "C10",
    "y[S11]"
   ],
   [
    "C11",
    "y[S12]"
   ],
   [
    "C12",
    "x[S1,C1]"
   ],
   [
    "C13",
    "x[S1,C2]"
   ],
   [
    "C14",
    "x[S1,C3]"
   ],
   [
    "C15",
    "x[S1,C4]"
   ],
   [
    "C16",
    "x[S1,C5]"
   ],
   [
    "C17",
    "x[S1,C6]"
   ],
   [
    "C18",
    "x[S1,C7]"
   ],
   [
    "C19",
    "x[S1,C8]"
   ],
   [
    "C20",
    "x[S1,C9]"
   ],
   [
    "C21",
    "x[S1,C10]"
   ],
   [
    "C22",
    "x[S1,C11]"
   ],
   [
    "C23",
    "x[S1,C12]"
   ],
   [
    "C24",
    "x[S2,C1]"
   ],
   [
    "C25",
    "x[S2,C2]"
   ],
   [
    "C26",
    "x[S2,C3]"
   ],
   [
    "C27",
    "x[S2,C4]"
   ],
   [
    "C28",
    "x[S2,C5]"
   ],
   [
    "C29",
    "x[S2,C6]"
   ],
   [
    "C30",
    "x[S2,C7]"
   ],
   [
    "C31",
    "x[S2,C8]"
   ],
   [
    "C32",
    "x[S2,C9]"
   ],
   [
    "C33",
    "x[S2,C10]"
   ],
   [
    "C34",
    "x[S2,C11]"
   ],
   [
    "C35",
    "x[S2,C12]"
   ],
   [
    "C36",
    "x[S3,C1]"
   ],
   [
    "C37",
    "x[S3,C2]"
   ],
   [
    "C38",
    "x[S3,C3]"
   ],
   [
    "C39",
    "x[S3,C4]"
   ],
   [
    "C40",
    "x[S3,C5]"
   ],
   [
    "C41",
    "x[S3,C6]"
   ],
   [
    "C42",
    "x[S3,C7]"
   ],
   [
    "C43",
    "x[S3,C8]"
   ],
   [
    "C44",
    "x[S3,C9]"
   ],
   [
    "C45",
    "x[S3,C10]"
   ],
   [
    "C46",
    "x[S3,C11]"
   ],
   [
    "C47",
    "x[S3,C12]"
   ],
   [
    "C48",
    "x[S4,C1]"
   ],
   [
    "C49",
    "x[S4,C2]"
   ],
   [
    "C50",
    "x[S4,C3]"
   ],
   [
    "C51",
    "x[S4,C4]"
   ],
   [
    "C52",
    "x[S4,C5]"
   ],
   [
    "C53",
    "x[S4,C6]"
   ],
   [
    "C54",
    "x[S4,C7]"
   ],
   [
    "C55",
    "x[S4,C8]"
   ],
   [
    "C56",
    "x[S4,C9]"
   ],
   [
    "C57",
    "x[S4,C10]"
   ],
   [
    "C58",
    "x[S4,C11]"
   ],
   [
    "C59",
    "x[S4,C12]"
   ],
   [
    "C60",
    "x[S5,C1]"
   ],
   [
    "C61",
    "x[S5,C2]"
   ],
   [
    "C62",
    "x[S5,C3]"
   ],
   [
    "C63",
    "x[S5,C4]"
   ],
   [
    "C64",
    "x[S5,C5]"
   ],
   [
    "C65",
    "x[S5,C6]"
   ],
   [
    "C66",
    "x[S5,C7]"
   ],
   [
    "C67",
    "x[S5,C8]"
   ],
   [
    "C68",
    "x[S5,C9]"
   ],
   [
    "C69",
    "x[S5,C10]"
   ],
   [
    "C70",
    "x[S5,C11]"
   ],
   [
    "C71",
    "x[S5,C12]"
   ],
   [
    "C72",
    "x[S6,C1]"
   ],
   [
    "C73",
    "x[S6,C2]"
   ],
   [
    "C74",
    "x[S6,C3]"
   ],
   [
    "C75",
    "x[S6,C4]"
   ],
   [
    "C76",
    "x[S6,C5]"
   ],
   [
    "C77",
    "x[S6,C6]"
   ],
   [
    "C78",
    "x[S6,C7]"
   ],
   [
    "C79",
    "x[S6,C8]"
   ],
   [
    "C80",
    "x[S6,C9]"
   ],
   [
    "C81",
    "x[S6,C10]"
   ],
   [
    "C82",
    "x[S6,C11]"
   ],
   [
    "C83",
    "x[S6,C12]"
   ],
   [
    "C84",
    "x[S7,C1]"
   ],
   [
    "C85",
    "x[S7,C2]"
   ],
   [
    "C86",
    "x[S7,C3]"
   ],
   [
    "C87",
    "x[S7,C4]"
   ],
   [
    "C88",
    "x[S7,C5]"
   ],
   [
    "C89",
    "x[S7,C6]"
   ],
   [
    "C90",
    "x[S7,C7]"
   ],
   [
    "C91",
    "x[S7,C8]"
   ],
   [
    "C92",
    "x[S7,C9]"
   ],
   [
    "C93",
    "x[S7,C10]"
   ],
   [
    "C94",
    "x[S7,C11]"
   ],
   [
    "C95",
    "x[S7,C12]"
   ],
   [
    "C96",
    "x[S8,C1]"
   ],
   [
    "C97",
    "x[S8,C2]"
   ],
   [
    "C98",
    "x[S8,C3]"
   ],
   [
    "C99",
    "x[S8,C4]"
   ],
   [
    "C100",
    "x[S8,C5]"
   ],
   [
    "C101",
    "x[S8,C6]"
   ],
   [
    "C102",
    "x[S8,C7]"
   ],
   [
    "C103",
    "x[S8,C8]"
   ],
   [
    "C104",
    "x[S8,C9]"
   ],
   [
    "C105",
    "x[S8,C10]"
   ],
   [
    "C106",
    "x[S8,C11]"
   ],
   [
    "C107",
    "x[S8,C12]"
   ],
   [
    "C108",
    "x[S9,C1]"
   ],
   [
    "C109",
    "x[S9,C2]"
   ],
   [
    "C110",
    "x[S9,C3]"
   ],
   [
    "C111",
    "x[S9,C4]"
   ],
   [
    "C112",
    "x[S9,C5]"
   ],
   [
    "C113",
    "x[S9,C6]"
   ],
   [
    "C114",
    "x[S9,C7]"
   ],
   [
    "C115",
    "x[S9,C8]"
   ],
   [
    "C116",
    "x[S9,C9]"
   ],
   [
    "C117",
    "x[S9,C10]"
   ],
   [
    "C118",
    "x[S9,C11]"
   ],
   [
    "C119",
    "x[S9,C12]"
   ],
   [
    "C120",
    "x[S10,C1]"
   ],
   [
    "C121",
    "x[S10,C2]"
   ],
   [
    "C122",
    "x[S10,C3]"
   ],
   [
    "C123",
    "x[S10,C4]"
   ],
   [
    "C124",
    "x[S10,C5]"
   ],
   [
    "C125",
    "x[S10,C6]"
   ],
   [
    "C126",
    "x[S10,C7]"
   ],
   [
    "C127",
    "x[S10,C8]"
   ],
   [
    "C128",
    "x[S10,C9]"
   ],
   [
    "C129",
    "x[S10,C10]"
   ],
   [
    "C130",
    "x[S10,C11]"
   ],
   [
    "C131",
    "x[S10,C12]"
   ],
   [
    "C132",
    "x[S11,C1]"
   ],
   [
    "C133",
    "x[S11,C2]"
   ],
   [
    "C134",
    "x[S11,C3]"
   ],
   [
    "C135",
    "x[S11,C4]"
   ],
   [
    "C136",
    "x[S11,C5]"
   ],
   [
    "C137",
    "x[S11,C6]"
   ],
   [
    "C138",
    "x[S11,C7]"
   ],
   [
    "C139",
    "x[S11,C8]"
   ],
   [
    "C140",
    "x[S11,C9]"
   ],
   [
    "C141",
    "x[S11,C10]"
   ],
   [
    "C142",
    "x[S11,C11]"
   ],
   [
    "C143",
    "x[S11,C12]"
   ],
   [
    "C144",
    "x[S12,C1]"
   ],
   [
    "C145",
    "x[S12,C2]"
   ],
   [
    "C146",
    "x[S12,C3]"
   ],
   [
    "C147",
    "x[S12,C4]"
   ],
   [
    "C148",
    "x[S12,C5]"
   ],
   [
    "C149",
    "x[S12,C6]"
   ],
   [
    "C150",
    "x[S12,C7]"
   ],
   [
    "C151",
    "x[S12,C8]"
   ],
   [
    "C152",
    "x[S12,C9]"
   ],
   [
    "C153",
    "x[S12,C10]"
   ],
   [
    "C154",
    "x[S12,C11]"
   ],
   [
    "C155",
    "x[S12,C12]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_0_10": "x[S1,C11]",
   "x_0_11": "x[S1,C12]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_1_10": "x[S2,C11]",
   "x_1_11": "x[S2,C12]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_2_10": "x[S3,C11]",
   "x_2_11": "x[S3,C12]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_3_10": "x[S4,C11]",
   "x_3_11": "x[S4,C12]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_4_10": "x[S5,C11]",
   "x_4_11": "x[S5,C12]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_5_10": "x[S6,C11]",
   "x_5_11": "x[S6,C12]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_9": "x[S7,C10]",
   "x_6_10": "x[S7,C11]",
   "x_6_11": "x[S7,C12]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_7_10": "x[S8,C11]",
   "x_7_11": "x[S8,C12]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_8_10": "x[S9,C11]",
   "x_8_11": "x[S9,C12]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]",
   "x_9_10": "x[S10,C11]",
   "x_9_11": "x[S10,C12]",
   "x_10_0": "x[S11,C1]",
   "x_10_2": "x[S11,C3]",
   "x_10_3": "x[S11,C4]",
   "x_10_4": "x[S11,C5]",
   "x_10_5": "x[S11,C6]",
   "x_10_6": "x[S11,C7]",
   "x_10_7": "x[S11,C8]",
   "x_10_8": "x[S11,C9]",
   "x_10_9": "x[S11,C10]",
   "x_10_10": "x[S11,C11]",
   "x_10_11": "x[S11,C12]",
   "x_11_0": "x[S12,C1]",
   "x_11_1": "x[S12,C2]",
   "x_11_2": "x[S12,C3]",
   "x_11_3": "x[S12,C4]",
   "x_11_4": "x[S12,C5]",
   "x_11_5": "x[S12,C6]",
   "x_11_6": "x[S12,C7]",
   "x_11_7": "x[S12,C8]",
   "x_11_8": "x[S12,C9]",
   "x_11_9": "x[S12,C10]",
   "x_11_10": "x[S12,C11]",
   "x_11_11": "x[S12,C12]",
   "x_10_1": "x[S11,C2]",
   "x_6_8": "x[S7,C9]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]",
   "y_3": "y[S4]",
   "y_4": "y[S5]",
   "y_5": "y[S6]",
   "y_6": "y[S7]",
   "y_7": "y[S8]",
   "y_8": "y[S9]",
   "y_9": "y[S10]",
   "y_10": "y[S11]",
   "y_11": "y[S12]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 8643.199999999997,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     1.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     1.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     1.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     1.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     1097.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     37.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     55.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     7.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     582.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     89.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     223.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     11.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     82.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     61.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     483.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     60.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-064": {
  "source_code_sha256": "c0bd4bd472c1116ecb3065d5d8321b6ae9dc10438d43553ae2d1d4c7be1270f3",
  "original_solution_sha256": "6e05854f72ce750aa39973d5500ef1917730e9b6e2f43545d8cb3c22cc92efc1",
  "bindings": [
   [
    "C0",
    "x[S19,C15]"
   ],
   [
    "C1",
    "x[S19,C6]"
   ],
   [
    "C2",
    "x[S19,C19]"
   ],
   [
    "C3",
    "x[S19,C2]"
   ],
   [
    "C4",
    "x[S19,C20]"
   ],
   [
    "C5",
    "x[S19,C18]"
   ],
   [
    "C6",
    "x[S19,C22]"
   ],
   [
    "C7",
    "x[S19,C21]"
   ],
   [
    "C8",
    "x[S19,C10]"
   ],
   [
    "C9",
    "x[S19,C8]"
   ],
   [
    "C10",
    "x[S19,C13]"
   ],
   [
    "C11",
    "x[S19,C4]"
   ],
   [
    "C12",
    "x[S19,C9]"
   ],
   [
    "C13",
    "x[S19,C11]"
   ],
   [
    "C14",
    "x[S19,C7]"
   ],
   [
    "C15",
    "x[S19,C24]"
   ],
   [
    "C16",
    "x[S19,C25]"
   ],
   [
    "C17",
    "x[S19,C14]"
   ],
   [
    "C18",
    "x[S19,C16]"
   ],
   [
    "C19",
    "x[S19,C5]"
   ],
   [
    "C20",
    "x[S19,C17]"
   ],
   [
    "C21",
    "x[S19,C1]"
   ],
   [
    "C22",
    "x[S19,C3]"
   ],
   [
    "C23",
    "x[S19,C23]"
   ],
   [
    "C24",
    "x[S19,C12]"
   ],
   [
    "C25",
    "x[S23,C15]"
   ],
   [
    "C26",
    "x[S23,C6]"
   ],
   [
    "C27",
    "x[S23,C19]"
   ],
   [
    "C28",
    "x[S23,C2]"
   ],
   [
    "C29",
    "x[S23,C20]"
   ],
   [
    "C30",
    "x[S23,C18]"
   ],
   [
    "C31",
    "x[S23,C22]"
   ],
   [
    "C32",
    "x[S23,C21]"
   ],
   [
    "C33",
    "x[S23,C10]"
   ],
   [
    "C34",
    "x[S23,C8]"
   ],
   [
    "C35",
    "x[S23,C13]"
   ],
   [
    "C36",
    "x[S23,C4]"
   ],
   [
    "C37",
    "x[S23,C9]"
   ],
   [
    "C38",
    "x[S23,C11]"
   ],
   [
    "C39",
    "x[S23,C7]"
   ],
   [
    "C40",
    "x[S23,C24]"
   ],
   [
    "C41",
    "x[S23,C25]"
   ],
   [
    "C42",
    "x[S23,C14]"
   ],
   [
    "C43",
    "x[S23,C16]"
   ],
   [
    "C44",
    "x[S23,C5]"
   ],
   [
    "C45",
    "x[S23,C17]"
   ],
   [
    "C46",
    "x[S23,C1]"
   ],
   [
    "C47",
    "x[S23,C3]"
   ],
   [
    "C48",
    "x[S23,C23]"
   ],
   [
    "C49",
    "x[S23,C12]"
   ],
   [
    "C50",
    "x[S24,C15]"
   ],
   [
    "C51",
    "x[S24,C6]"
   ],
   [
    "C52",
    "x[S24,C19]"
   ],
   [
    "C53",
    "x[S24,C2]"
   ],
   [
    "C54",
    "x[S24,C20]"
   ],
   [
    "C55",
    "x[S24,C18]"
   ],
   [
    "C56",
    "x[S24,C22]"
   ],
   [
    "C57",
    "x[S24,C21]"
   ],
   [
    "C58",
    "x[S24,C10]"
   ],
   [
    "C59",
    "x[S24,C8]"
   ],
   [
    "C60",
    "x[S24,C13]"
   ],
   [
    "C61",
    "x[S24,C4]"
   ],
   [
    "C62",
    "x[S24,C9]"
   ],
   [
    "C63",
    "x[S24,C11]"
   ],
   [
    "C64",
    "x[S24,C7]"
   ],
   [
    "C65",
    "x[S24,C24]"
   ],
   [
    "C66",
    "x[S24,C25]"
   ],
   [
    "C67",
    "x[S24,C14]"
   ],
   [
    "C68",
    "x[S24,C16]"
   ],
   [
    "C69",
    "x[S24,C5]"
   ],
   [
    "C70",
    "x[S24,C17]"
   ],
   [
    "C71",
    "x[S24,C1]"
   ],
   [
    "C72",
    "x[S24,C3]"
   ],
   [
    "C73",
    "x[S24,C23]"
   ],
   [
    "C74",
    "x[S24,C12]"
   ],
   [
    "C75",
    "x[S21,C15]"
   ],
   [
    "C76",
    "x[S21,C6]"
   ],
   [
    "C77",
    "x[S21,C19]"
   ],
   [
    "C78",
    "x[S21,C2]"
   ],
   [
    "C79",
    "x[S21,C20]"
   ],
   [
    "C80",
    "x[S21,C18]"
   ],
   [
    "C81",
    "x[S21,C22]"
   ],
   [
    "C82",
    "x[S21,C21]"
   ],
   [
    "C83",
    "x[S21,C10]"
   ],
   [
    "C84",
    "x[S21,C8]"
   ],
   [
    "C85",
    "x[S21,C13]"
   ],
   [
    "C86",
    "x[S21,C4]"
   ],
   [
    "C87",
    "x[S21,C9]"
   ],
   [
    "C88",
    "x[S21,C11]"
   ],
   [
    "C89",
    "x[S21,C7]"
   ],
   [
    "C90",
    "x[S21,C24]"
   ],
   [
    "C91",
    "x[S21,C25]"
   ],
   [
    "C92",
    "x[S21,C14]"
   ],
   [
    "C93",
    "x[S21,C16]"
   ],
   [
    "C94",
    "x[S21,C5]"
   ],
   [
    "C95",
    "x[S21,C17]"
   ],
   [
    "C96",
    "x[S21,C1]"
   ],
   [
    "C97",
    "x[S21,C3]"
   ],
   [
    "C98",
    "x[S21,C23]"
   ],
   [
    "C99",
    "x[S21,C12]"
   ],
   [
    "C100",
    "x[S5,C15]"
   ],
   [
    "C101",
    "x[S5,C6]"
   ],
   [
    "C102",
    "x[S5,C19]"
   ],
   [
    "C103",
    "x[S5,C2]"
   ],
   [
    "C104",
    "x[S5,C20]"
   ],
   [
    "C105",
    "x[S5,C18]"
   ],
   [
    "C106",
    "x[S5,C22]"
   ],
   [
    "C107",
    "x[S5,C21]"
   ],
   [
    "C108",
    "x[S5,C10]"
   ],
   [
    "C109",
    "x[S5,C8]"
   ],
   [
    "C110",
    "x[S5,C13]"
   ],
   [
    "C111",
    "x[S5,C4]"
   ],
   [
    "C112",
    "x[S5,C9]"
   ],
   [
    "C113",
    "x[S5,C11]"
   ],
   [
    "C114",
    "x[S5,C7]"
   ],
   [
    "C115",
    "x[S5,C24]"
   ],
   [
    "C116",
    "x[S5,C25]"
   ],
   [
    "C117",
    "x[S5,C14]"
   ],
   [
    "C118",
    "x[S5,C16]"
   ],
   [
    "C119",
    "x[S5,C5]"
   ],
   [
    "C120",
    "x[S5,C17]"
   ],
   [
    "C121",
    "x[S5,C1]"
   ],
   [
    "C122",
    "x[S5,C3]"
   ],
   [
    "C123",
    "x[S5,C23]"
   ],
   [
    "C124",
    "x[S5,C12]"
   ],
   [
    "C125",
    "x[S12,C15]"
   ],
   [
    "C126",
    "x[S12,C6]"
   ],
   [
    "C127",
    "x[S12,C19]"
   ],
   [
    "C128",
    "x[S12,C2]"
   ],
   [
    "C129",
    "x[S12,C20]"
   ],
   [
    "C130",
    "x[S12,C18]"
   ],
   [
    "C131",
    "x[S12,C22]"
   ],
   [
    "C132",
    "x[S12,C21]"
   ],
   [
    "C133",
    "x[S12,C10]"
   ],
   [
    "C134",
    "x[S12,C8]"
   ],
   [
    "C135",
    "x[S12,C13]"
   ],
   [
    "C136",
    "x[S12,C4]"
   ],
   [
    "C137",
    "x[S12,C9]"
   ],
   [
    "C138",
    "x[S12,C11]"
   ],
   [
    "C139",
    "x[S12,C7]"
   ],
   [
    "C140",
    "x[S12,C24]"
   ],
   [
    "C141",
    "x[S12,C25]"
   ],
   [
    "C142",
    "x[S12,C14]"
   ],
   [
    "C143",
    "x[S12,C16]"
   ],
   [
    "C144",
    "x[S12,C5]"
   ],
   [
    "C145",
    "x[S12,C17]"
   ],
   [
    "C146",
    "x[S12,C1]"
   ],
   [
    "C147",
    "x[S12,C3]"
   ],
   [
    "C148",
    "x[S12,C23]"
   ],
   [
    "C149",
    "x[S12,C12]"
   ],
   [
    "C150",
    "x[S16,C15]"
   ],
   [
    "C151",
    "x[S16,C6]"
   ],
   [
    "C152",
    "x[S16,C19]"
   ],
   [
    "C153",
    "x[S16,C2]"
   ],
   [
    "C154",
    "x[S16,C20]"
   ],
   [
    "C155",
    "x[S16,C18]"
   ],
   [
    "C156",
    "x[S16,C22]"
   ],
   [
    "C157",
    "x[S16,C21]"
   ],
   [
    "C158",
    "x[S16,C10]"
   ],
   [
    "C159",
    "x[S16,C8]"
   ],
   [
    "C160",
    "x[S16,C13]"
   ],
   [
    "C161",
    "x[S16,C4]"
   ],
   [
    "C162",
    "x[S16,C9]"
   ],
   [
    "C163",
    "x[S16,C11]"
   ],
   [
    "C164",
    "x[S16,C7]"
   ],
   [
    "C165",
    "x[S16,C24]"
   ],
   [
    "C166",
    "x[S16,C25]"
   ],
   [
    "C167",
    "x[S16,C14]"
   ],
   [
    "C168",
    "x[S16,C16]"
   ],
   [
    "C169",
    "x[S16,C5]"
   ],
   [
    "C170",
    "x[S16,C17]"
   ],
   [
    "C171",
    "x[S16,C1]"
   ],
   [
    "C172",
    "x[S16,C3]"
   ],
   [
    "C173",
    "x[S16,C23]"
   ],
   [
    "C174",
    "x[S16,C12]"
   ],
   [
    "C175",
    "x[S17,C15]"
   ],
   [
    "C176",
    "x[S17,C6]"
   ],
   [
    "C177",
    "x[S17,C19]"
   ],
   [
    "C178",
    "x[S17,C2]"
   ],
   [
    "C179",
    "x[S17,C20]"
   ],
   [
    "C180",
    "x[S17,C18]"
   ],
   [
    "C181",
    "x[S17,C22]"
   ],
   [
    "C182",
    "x[S17,C21]"
   ],
   [
    "C183",
    "x[S17,C10]"
   ],
   [
    "C184",
    "x[S17,C8]"
   ],
   [
    "C185",
    "x[S17,C13]"
   ],
   [
    "C186",
    "x[S17,C4]"
   ],
   [
    "C187",
    "x[S17,C9]"
   ],
   [
    "C188",
    "x[S17,C11]"
   ],
   [
    "C189",
    "x[S17,C7]"
   ],
   [
    "C190",
    "x[S17,C24]"
   ],
   [
    "C191",
    "x[S17,C25]"
   ],
   [
    "C192",
    "x[S17,C14]"
   ],
   [
    "C193",
    "x[S17,C16]"
   ],
   [
    "C194",
    "x[S17,C5]"
   ],
   [
    "C195",
    "x[S17,C17]"
   ],
   [
    "C196",
    "x[S17,C1]"
   ],
   [
    "C197",
    "x[S17,C3]"
   ],
   [
    "C198",
    "x[S17,C23]"
   ],
   [
    "C199",
    "x[S17,C12]"
   ],
   [
    "C200",
    "x[S2,C15]"
   ],
   [
    "C201",
    "x[S2,C6]"
   ],
   [
    "C202",
    "x[S2,C19]"
   ],
   [
    "C203",
    "x[S2,C2]"
   ],
   [
    "C204",
    "x[S2,C20]"
   ],
   [
    "C205",
    "x[S2,C18]"
   ],
   [
    "C206",
    "x[S2,C22]"
   ],
   [
    "C207",
    "x[S2,C21]"
   ],
   [
    "C208",
    "x[S2,C10]"
   ],
   [
    "C209",
    "x[S2,C8]"
   ],
   [
    "C210",
    "x[S2,C13]"
   ],
   [
    "C211",
    "x[S2,C4]"
   ],
   [
    "C212",
    "x[S2,C9]"
   ],
   [
    "C213",
    "x[S2,C11]"
   ],
   [
    "C214",
    "x[S2,C7]"
   ],
   [
    "C215",
    "x[S2,C24]"
   ],
   [
    "C216",
    "x[S2,C25]"
   ],
   [
    "C217",
    "x[S2,C14]"
   ],
   [
    "C218",
    "x[S2,C16]"
   ],
   [
    "C219",
    "x[S2,C5]"
   ],
   [
    "C220",
    "x[S2,C17]"
   ],
   [
    "C221",
    "x[S2,C1]"
   ],
   [
    "C222",
    "x[S2,C3]"
   ],
   [
    "C223",
    "x[S2,C23]"
   ],
   [
    "C224",
    "x[S2,C12]"
   ],
   [
    "C225",
    "x[S6,C15]"
   ],
   [
    "C226",
    "x[S6,C6]"
   ],
   [
    "C227",
    "x[S6,C19]"
   ],
   [
    "C228",
    "x[S6,C2]"
   ],
   [
    "C229",
    "x[S6,C20]"
   ],
   [
    "C230",
    "x[S6,C18]"
   ],
   [
    "C231",
    "x[S6,C22]"
   ],
   [
    "C232",
    "x[S6,C21]"
   ],
   [
    "C233",
    "x[S6,C10]"
   ],
   [
    "C234",
    "x[S6,C8]"
   ],
   [
    "C235",
    "x[S6,C13]"
   ],
   [
    "C236",
    "x[S6,C4]"
   ],
   [
    "C237",
    "x[S6,C9]"
   ],
   [
    "C238",
    "x[S6,C11]"
   ],
   [
    "C239",
    "x[S6,C7]"
   ],
   [
    "C240",
    "x[S6,C24]"
   ],
   [
    "C241",
    "x[S6,C25]"
   ],
   [
    "C242",
    "x[S6,C14]"
   ],
   [
    "C243",
    "x[S6,C16]"
   ],
   [
    "C244",
    "x[S6,C5]"
   ],
   [
    "C245",
    "x[S6,C17]"
   ],
   [
    "C246",
    "x[S6,C1]"
   ],
   [
    "C247",
    "x[S6,C3]"
   ],
   [
    "C248",
    "x[S6,C23]"
   ],
   [
    "C249",
    "x[S6,C12]"
   ],
   [
    "C250",
    "x[S18,C15]"
   ],
   [
    "C251",
    "x[S18,C6]"
   ],
   [
    "C252",
    "x[S18,C19]"
   ],
   [
    "C253",
    "x[S18,C2]"
   ],
   [
    "C254",
    "x[S18,C20]"
   ],
   [
    "C255",
    "x[S18,C18]"
   ],
   [
    "C256",
    "x[S18,C22]"
   ],
   [
    "C257",
    "x[S18,C21]"
   ],
   [
    "C258",
    "x[S18,C10]"
   ],
   [
    "C259",
    "x[S18,C8]"
   ],
   [
    "C260",
    "x[S18,C13]"
   ],
   [
    "C261",
    "x[S18,C4]"
   ],
   [
    "C262",
    "x[S18,C9]"
   ],
   [
    "C263",
    "x[S18,C11]"
   ],
   [
    "C264",
    "x[S18,C7]"
   ],
   [
    "C265",
    "x[S18,C24]"
   ],
   [
    "C266",
    "x[S18,C25]"
   ],
   [
    "C267",
    "x[S18,C14]"
   ],
   [
    "C268",
    "x[S18,C16]"
   ],
   [
    "C269",
    "x[S18,C5]"
   ],
   [
    "C270",
    "x[S18,C17]"
   ],
   [
    "C271",
    "x[S18,C1]"
   ],
   [
    "C272",
    "x[S18,C3]"
   ],
   [
    "C273",
    "x[S18,C23]"
   ],
   [
    "C274",
    "x[S18,C12]"
   ],
   [
    "C275",
    "x[S13,C15]"
   ],
   [
    "C276",
    "x[S13,C6]"
   ],
   [
    "C277",
    "x[S13,C19]"
   ],
   [
    "C278",
    "x[S13,C2]"
   ],
   [
    "C279",
    "x[S13,C20]"
   ],
   [
    "C280",
    "x[S13,C18]"
   ],
   [
    "C281",
    "x[S13,C22]"
   ],
   [
    "C282",
    "x[S13,C21]"
   ],
   [
    "C283",
    "x[S13,C10]"
   ],
   [
    "C284",
    "x[S13,C8]"
   ],
   [
    "C285",
    "x[S13,C13]"
   ],
   [
    "C286",
    "x[S13,C4]"
   ],
   [
    "C287",
    "x[S13,C9]"
   ],
   [
    "C288",
    "x[S13,C11]"
   ],
   [
    "C289",
    "x[S13,C7]"
   ],
   [
    "C290",
    "x[S13,C24]"
   ],
   [
    "C291",
    "x[S13,C25]"
   ],
   [
    "C292",
    "x[S13,C14]"
   ],
   [
    "C293",
    "x[S13,C16]"
   ],
   [
    "C294",
    "x[S13,C5]"
   ],
   [
    "C295",
    "x[S13,C17]"
   ],
   [
    "C296",
    "x[S13,C1]"
   ],
   [
    "C297",
    "x[S13,C3]"
   ],
   [
    "C298",
    "x[S13,C23]"
   ],
   [
    "C299",
    "x[S13,C12]"
   ],
   [
    "C300",
    "x[S20,C15]"
   ],
   [
    "C301",
    "x[S20,C6]"
   ],
   [
    "C302",
    "x[S20,C19]"
   ],
   [
    "C303",
    "x[S20,C2]"
   ],
   [
    "C304",
    "x[S20,C20]"
   ],
   [
    "C305",
    "x[S20,C18]"
   ],
   [
    "C306",
    "x[S20,C22]"
   ],
   [
    "C307",
    "x[S20,C21]"
   ],
   [
    "C308",
    "x[S20,C10]"
   ],
   [
    "C309",
    "x[S20,C8]"
   ],
   [
    "C310",
    "x[S20,C13]"
   ],
   [
    "C311",
    "x[S20,C4]"
   ],
   [
    "C312",
    "x[S20,C9]"
   ],
   [
    "C313",
    "x[S20,C11]"
   ],
   [
    "C314",
    "x[S20,C7]"
   ],
   [
    "C315",
    "x[S20,C24]"
   ],
   [
    "C316",
    "x[S20,C25]"
   ],
   [
    "C317",
    "x[S20,C14]"
   ],
   [
    "C318",
    "x[S20,C16]"
   ],
   [
    "C319",
    "x[S20,C5]"
   ],
   [
    "C320",
    "x[S20,C17]"
   ],
   [
    "C321",
    "x[S20,C1]"
   ],
   [
    "C322",
    "x[S20,C3]"
   ],
   [
    "C323",
    "x[S20,C23]"
   ],
   [
    "C324",
    "x[S20,C12]"
   ],
   [
    "C325",
    "x[S15,C15]"
   ],
   [
    "C326",
    "x[S15,C6]"
   ],
   [
    "C327",
    "x[S15,C19]"
   ],
   [
    "C328",
    "x[S15,C2]"
   ],
   [
    "C329",
    "x[S15,C20]"
   ],
   [
    "C330",
    "x[S15,C18]"
   ],
   [
    "C331",
    "x[S15,C22]"
   ],
   [
    "C332",
    "x[S15,C21]"
   ],
   [
    "C333",
    "x[S15,C10]"
   ],
   [
    "C334",
    "x[S15,C8]"
   ],
   [
    "C335",
    "x[S15,C13]"
   ],
   [
    "C336",
    "x[S15,C4]"
   ],
   [
    "C337",
    "x[S15,C9]"
   ],
   [
    "C338",
    "x[S15,C11]"
   ],
   [
    "C339",
    "x[S15,C7]"
   ],
   [
    "C340",
    "x[S15,C24]"
   ],
   [
    "C341",
    "x[S15,C25]"
   ],
   [
    "C342",
    "x[S15,C14]"
   ],
   [
    "C343",
    "x[S15,C16]"
   ],
   [
    "C344",
    "x[S15,C5]"
   ],
   [
    "C345",
    "x[S15,C17]"
   ],
   [
    "C346",
    "x[S15,C1]"
   ],
   [
    "C347",
    "x[S15,C3]"
   ],
   [
    "C348",
    "x[S15,C23]"
   ],
   [
    "C349",
    "x[S15,C12]"
   ],
   [
    "C350",
    "x[S8,C15]"
   ],
   [
    "C351",
    "x[S8,C6]"
   ],
   [
    "C352",
    "x[S8,C19]"
   ],
   [
    "C353",
    "x[S8,C2]"
   ],
   [
    "C354",
    "x[S8,C20]"
   ],
   [
    "C355",
    "x[S8,C18]"
   ],
   [
    "C356",
    "x[S8,C22]"
   ],
   [
    "C357",
    "x[S8,C21]"
   ],
   [
    "C358",
    "x[S8,C10]"
   ],
   [
    "C359",
    "x[S8,C8]"
   ],
   [
    "C360",
    "x[S8,C13]"
   ],
   [
    "C361",
    "x[S8,C4]"
   ],
   [
    "C362",
    "x[S8,C9]"
   ],
   [
    "C363",
    "x[S8,C11]"
   ],
   [
    "C364",
    "x[S8,C7]"
   ],
   [
    "C365",
    "x[S8,C24]"
   ],
   [
    "C366",
    "x[S8,C25]"
   ],
   [
    "C367",
    "x[S8,C14]"
   ],
   [
    "C368",
    "x[S8,C16]"
   ],
   [
    "C369",
    "x[S8,C5]"
   ],
   [
    "C370",
    "x[S8,C17]"
   ],
   [
    "C371",
    "x[S8,C1]"
   ],
   [
    "C372",
    "x[S8,C3]"
   ],
   [
    "C373",
    "x[S8,C23]"
   ],
   [
    "C374",
    "x[S8,C12]"
   ],
   [
    "C375",
    "x[S22,C15]"
   ],
   [
    "C376",
    "x[S22,C6]"
   ],
   [
    "C377",
    "x[S22,C19]"
   ],
   [
    "C378",
    "x[S22,C2]"
   ],
   [
    "C379",
    "x[S22,C20]"
   ],
   [
    "C380",
    "x[S22,C18]"
   ],
   [
    "C381",
    "x[S22,C22]"
   ],
   [
    "C382",
    "x[S22,C21]"
   ],
   [
    "C383",
    "x[S22,C10]"
   ],
   [
    "C384",
    "x[S22,C8]"
   ],
   [
    "C385",
    "x[S22,C13]"
   ],
   [
    "C386",
    "x[S22,C4]"
   ],
   [
    "C387",
    "x[S22,C9]"
   ],
   [
    "C388",
    "x[S22,C11]"
   ],
   [
    "C389",
    "x[S22,C7]"
   ],
   [
    "C390",
    "x[S22,C24]"
   ],
   [
    "C391",
    "x[S22,C25]"
   ],
   [
    "C392",
    "x[S22,C14]"
   ],
   [
    "C393",
    "x[S22,C16]"
   ],
   [
    "C394",
    "x[S22,C5]"
   ],
   [
    "C395",
    "x[S22,C17]"
   ],
   [
    "C396",
    "x[S22,C1]"
   ],
   [
    "C397",
    "x[S22,C3]"
   ],
   [
    "C398",
    "x[S22,C23]"
   ],
   [
    "C399",
    "x[S22,C12]"
   ],
   [
    "C400",
    "x[S14,C15]"
   ],
   [
    "C401",
    "x[S14,C6]"
   ],
   [
    "C402",
    "x[S14,C19]"
   ],
   [
    "C403",
    "x[S14,C2]"
   ],
   [
    "C404",
    "x[S14,C20]"
   ],
   [
    "C405",
    "x[S14,C18]"
   ],
   [
    "C406",
    "x[S14,C22]"
   ],
   [
    "C407",
    "x[S14,C21]"
   ],
   [
    "C408",
    "x[S14,C10]"
   ],
   [
    "C409",
    "x[S14,C8]"
   ],
   [
    "C410",
    "x[S14,C13]"
   ],
   [
    "C411",
    "x[S14,C4]"
   ],
   [
    "C412",
    "x[S14,C9]"
   ],
   [
    "C413",
    "x[S14,C11]"
   ],
   [
    "C414",
    "x[S14,C7]"
   ],
   [
    "C415",
    "x[S14,C24]"
   ],
   [
    "C416",
    "x[S14,C25]"
   ],
   [
    "C417",
    "x[S14,C14]"
   ],
   [
    "C418",
    "x[S14,C16]"
   ],
   [
    "C419",
    "x[S14,C5]"
   ],
   [
    "C420",
    "x[S14,C17]"
   ],
   [
    "C421",
    "x[S14,C1]"
   ],
   [
    "C422",
    "x[S14,C3]"
   ],
   [
    "C423",
    "x[S14,C23]"
   ],
   [
    "C424",
    "x[S14,C12]"
   ],
   [
    "C425",
    "x[S11,C15]"
   ],
   [
    "C426",
    "x[S11,C6]"
   ],
   [
    "C427",
    "x[S11,C19]"
   ],
   [
    "C428",
    "x[S11,C2]"
   ],
   [
    "C429",
    "x[S11,C20]"
   ],
   [
    "C430",
    "x[S11,C18]"
   ],
   [
    "C431",
    "x[S11,C22]"
   ],
   [
    "C432",
    "x[S11,C21]"
   ],
   [
    "C433",
    "x[S11,C10]"
   ],
   [
    "C434",
    "x[S11,C8]"
   ],
   [
    "C435",
    "x[S11,C13]"
   ],
   [
    "C436",
    "x[S11,C4]"
   ],
   [
    "C437",
    "x[S11,C9]"
   ],
   [
    "C438",
    "x[S11,C11]"
   ],
   [
    "C439",
    "x[S11,C7]"
   ],
   [
    "C440",
    "x[S11,C24]"
   ],
   [
    "C441",
    "x[S11,C25]"
   ],
   [
    "C442",
    "x[S11,C14]"
   ],
   [
    "C443",
    "x[S11,C16]"
   ],
   [
    "C444",
    "x[S11,C5]"
   ],
   [
    "C445",
    "x[S11,C17]"
   ],
   [
    "C446",
    "x[S11,C1]"
   ],
   [
    "C447",
    "x[S11,C3]"
   ],
   [
    "C448",
    "x[S11,C23]"
   ],
   [
    "C449",
    "x[S11,C12]"
   ],
   [
    "C450",
    "x[S7,C15]"
   ],
   [
    "C451",
    "x[S7,C6]"
   ],
   [
    "C452",
    "x[S7,C19]"
   ],
   [
    "C453",
    "x[S7,C2]"
   ],
   [
    "C454",
    "x[S7,C20]"
   ],
   [
    "C455",
    "x[S7,C18]"
   ],
   [
    "C456",
    "x[S7,C22]"
   ],
   [
    "C457",
    "x[S7,C21]"
   ],
   [
    "C458",
    "x[S7,C10]"
   ],
   [
    "C459",
    "x[S7,C8]"
   ],
   [
    "C460",
    "x[S7,C13]"
   ],
   [
    "C461",
    "x[S7,C4]"
   ],
   [
    "C462",
    "x[S7,C9]"
   ],
   [
    "C463",
    "x[S7,C11]"
   ],
   [
    "C464",
    "x[S7,C7]"
   ],
   [
    "C465",
    "x[S7,C24]"
   ],
   [
    "C466",
    "x[S7,C25]"
   ],
   [
    "C467",
    "x[S7,C14]"
   ],
   [
    "C468",
    "x[S7,C16]"
   ],
   [
    "C469",
    "x[S7,C5]"
   ],
   [
    "C470",
    "x[S7,C17]"
   ],
   [
    "C471",
    "x[S7,C1]"
   ],
   [
    "C472",
    "x[S7,C3]"
   ],
   [
    "C473",
    "x[S7,C23]"
   ],
   [
    "C474",
    "x[S7,C12]"
   ],
   [
    "C475",
    "x[S3,C15]"
   ],
   [
    "C476",
    "x[S3,C6]"
   ],
   [
    "C477",
    "x[S3,C19]"
   ],
   [
    "C478",
    "x[S3,C2]"
   ],
   [
    "C479",
    "x[S3,C20]"
   ],
   [
    "C480",
    "x[S3,C18]"
   ],
   [
    "C481",
    "x[S3,C22]"
   ],
   [
    "C482",
    "x[S3,C21]"
   ],
   [
    "C483",
    "x[S3,C10]"
   ],
   [
    "C484",
    "x[S3,C8]"
   ],
   [
    "C485",
    "x[S3,C13]"
   ],
   [
    "C486",
    "x[S3,C4]"
   ],
   [
    "C487",
    "x[S3,C9]"
   ],
   [
    "C488",
    "x[S3,C11]"
   ],
   [
    "C489",
    "x[S3,C7]"
   ],
   [
    "C490",
    "x[S3,C24]"
   ],
   [
    "C491",
    "x[S3,C25]"
   ],
   [
    "C492",
    "x[S3,C14]"
   ],
   [
    "C493",
    "x[S3,C16]"
   ],
   [
    "C494",
    "x[S3,C5]"
   ],
   [
    "C495",
    "x[S3,C17]"
   ],
   [
    "C496",
    "x[S3,C1]"
   ],
   [
    "C497",
    "x[S3,C3]"
   ],
   [
    "C498",
    "x[S3,C23]"
   ],
   [
    "C499",
    "x[S3,C12]"
   ],
   [
    "C500",
    "x[S4,C15]"
   ],
   [
    "C501",
    "x[S4,C6]"
   ],
   [
    "C502",
    "x[S4,C19]"
   ],
   [
    "C503",
    "x[S4,C2]"
   ],
   [
    "C504",
    "x[S4,C20]"
   ],
   [
    "C505",
    "x[S4,C18]"
   ],
   [
    "C506",
    "x[S4,C22]"
   ],
   [
    "C507",
    "x[S4,C21]"
   ],
   [
    "C508",
    "x[S4,C10]"
   ],
   [
    "C509",
    "x[S4,C8]"
   ],
   [
    "C510",
    "x[S4,C13]"
   ],
   [
    "C511",
    "x[S4,C4]"
   ],
   [
    "C512",
    "x[S4,C9]"
   ],
   [
    "C513",
    "x[S4,C11]"
   ],
   [
    "C514",
    "x[S4,C7]"
   ],
   [
    "C515",
    "x[S4,C24]"
   ],
   [
    "C516",
    "x[S4,C25]"
   ],
   [
    "C517",
    "x[S4,C14]"
   ],
   [
    "C518",
    "x[S4,C16]"
   ],
   [
    "C519",
    "x[S4,C5]"
   ],
   [
    "C520",
    "x[S4,C17]"
   ],
   [
    "C521",
    "x[S4,C1]"
   ],
   [
    "C522",
    "x[S4,C3]"
   ],
   [
    "C523",
    "x[S4,C23]"
   ],
   [
    "C524",
    "x[S4,C12]"
   ],
   [
    "C525",
    "x[S9,C15]"
   ],
   [
    "C526",
    "x[S9,C6]"
   ],
   [
    "C527",
    "x[S9,C19]"
   ],
   [
    "C528",
    "x[S9,C2]"
   ],
   [
    "C529",
    "x[S9,C20]"
   ],
   [
    "C530",
    "x[S9,C18]"
   ],
   [
    "C531",
    "x[S9,C22]"
   ],
   [
    "C532",
    "x[S9,C21]"
   ],
   [
    "C533",
    "x[S9,C10]"
   ],
   [
    "C534",
    "x[S9,C8]"
   ],
   [
    "C535",
    "x[S9,C13]"
   ],
   [
    "C536",
    "x[S9,C4]"
   ],
   [
    "C537",
    "x[S9,C9]"
   ],
   [
    "C538",
    "x[S9,C11]"
   ],
   [
    "C539",
    "x[S9,C7]"
   ],
   [
    "C540",
    "x[S9,C24]"
   ],
   [
    "C541",
    "x[S9,C25]"
   ],
   [
    "C542",
    "x[S9,C14]"
   ],
   [
    "C543",
    "x[S9,C16]"
   ],
   [
    "C544",
    "x[S9,C5]"
   ],
   [
    "C545",
    "x[S9,C17]"
   ],
   [
    "C546",
    "x[S9,C1]"
   ],
   [
    "C547",
    "x[S9,C3]"
   ],
   [
    "C548",
    "x[S9,C23]"
   ],
   [
    "C549",
    "x[S9,C12]"
   ],
   [
    "C550",
    "x[S1,C15]"
   ],
   [
    "C551",
    "x[S1,C6]"
   ],
   [
    "C552",
    "x[S1,C19]"
   ],
   [
    "C553",
    "x[S1,C2]"
   ],
   [
    "C554",
    "x[S1,C20]"
   ],
   [
    "C555",
    "x[S1,C18]"
   ],
   [
    "C556",
    "x[S1,C22]"
   ],
   [
    "C557",
    "x[S1,C21]"
   ],
   [
    "C558",
    "x[S1,C10]"
   ],
   [
    "C559",
    "x[S1,C8]"
   ],
   [
    "C560",
    "x[S1,C13]"
   ],
   [
    "C561",
    "x[S1,C4]"
   ],
   [
    "C562",
    "x[S1,C9]"
   ],
   [
    "C563",
    "x[S1,C11]"
   ],
   [
    "C564",
    "x[S1,C7]"
   ],
   [
    "C565",
    "x[S1,C24]"
   ],
   [
    "C566",
    "x[S1,C25]"
   ],
   [
    "C567",
    "x[S1,C14]"
   ],
   [
    "C568",
    "x[S1,C16]"
   ],
   [
    "C569",
    "x[S1,C5]"
   ],
   [
    "C570",
    "x[S1,C17]"
   ],
   [
    "C571",
    "x[S1,C1]"
   ],
   [
    "C572",
    "x[S1,C3]"
   ],
   [
    "C573",
    "x[S1,C23]"
   ],
   [
    "C574",
    "x[S1,C12]"
   ],
   [
    "C575",
    "x[S10,C15]"
   ],
   [
    "C576",
    "x[S10,C6]"
   ],
   [
    "C577",
    "x[S10,C19]"
   ],
   [
    "C578",
    "x[S10,C2]"
   ],
   [
    "C579",
    "x[S10,C20]"
   ],
   [
    "C580",
    "x[S10,C18]"
   ],
   [
    "C581",
    "x[S10,C22]"
   ],
   [
    "C582",
    "x[S10,C21]"
   ],
   [
    "C583",
    "x[S10,C10]"
   ],
   [
    "C584",
    "x[S10,C8]"
   ],
   [
    "C585",
    "x[S10,C13]"
   ],
   [
    "C586",
    "x[S10,C4]"
   ],
   [
    "C587",
    "x[S10,C9]"
   ],
   [
    "C588",
    "x[S10,C11]"
   ],
   [
    "C589",
    "x[S10,C7]"
   ],
   [
    "C590",
    "x[S10,C24]"
   ],
   [
    "C591",
    "x[S10,C25]"
   ],
   [
    "C592",
    "x[S10,C14]"
   ],
   [
    "C593",
    "x[S10,C16]"
   ],
   [
    "C594",
    "x[S10,C5]"
   ],
   [
    "C595",
    "x[S10,C17]"
   ],
   [
    "C596",
    "x[S10,C1]"
   ],
   [
    "C597",
    "x[S10,C3]"
   ],
   [
    "C598",
    "x[S10,C23]"
   ],
   [
    "C599",
    "x[S10,C12]"
   ],
   [
    "C600",
    "y[S19]"
   ],
   [
    "C601",
    "y[S23]"
   ],
   [
    "C602",
    "y[S24]"
   ],
   [
    "C603",
    "y[S21]"
   ],
   [
    "C604",
    "y[S5]"
   ],
   [
    "C605",
    "y[S12]"
   ],
   [
    "C606",
    "y[S16]"
   ],
   [
    "C607",
    "y[S17]"
   ],
   [
    "C608",
    "y[S2]"
   ],
   [
    "C609",
    "y[S6]"
   ],
   [
    "C610",
    "y[S18]"
   ],
   [
    "C611",
    "y[S13]"
   ],
   [
    "C612",
    "y[S20]"
   ],
   [
    "C613",
    "y[S15]"
   ],
   [
    "C614",
    "y[S8]"
   ],
   [
    "C615",
    "y[S22]"
   ],
   [
    "C616",
    "y[S14]"
   ],
   [
    "C617",
    "y[S11]"
   ],
   [
    "C618",
    "y[S7]"
   ],
   [
    "C619",
    "y[S3]"
   ],
   [
    "C620",
    "y[S4]"
   ],
   [
    "C621",
    "y[S9]"
   ],
   [
    "C622",
    "y[S1]"
   ],
   [
    "C623",
    "y[S10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_0_10": "x[S1,C11]",
   "x_0_11": "x[S1,C12]",
   "x_0_12": "x[S1,C13]",
   "x_0_13": "x[S1,C14]",
   "x_0_14": "x[S1,C15]",
   "x_0_15": "x[S1,C16]",
   "x_0_16": "x[S1,C17]",
   "x_0_17": "x[S1,C18]",
   "x_0_18": "x[S1,C19]",
   "x_0_19": "x[S1,C20]",
   "x_0_20": "x[S1,C21]",
   "x_0_21": "x[S1,C22]",
   "x_0_22": "x[S1,C23]",
   "x_0_23": "x[S1,C24]",
   "x_0_24": "x[S1,C25]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_1_10": "x[S2,C11]",
   "x_1_11": "x[S2,C12]",
   "x_1_12": "x[S2,C13]",
   "x_1_13": "x[S2,C14]",
   "x_1_14": "x[S2,C15]",
   "x_1_15": "x[S2,C16]",
   "x_1_16": "x[S2,C17]",
   "x_1_17": "x[S2,C18]",
   "x_1_18": "x[S2,C19]",
   "x_1_19": "x[S2,C20]",
   "x_1_21": "x[S2,C22]",
   "x_1_22": "x[S2,C23]",
   "x_1_23": "x[S2,C24]",
   "x_1_24": "x[S2,C25]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_2_10": "x[S3,C11]",
   "x_2_11": "x[S3,C12]",
   "x_2_12": "x[S3,C13]",
   "x_2_13": "x[S3,C14]",
   "x_2_14": "x[S3,C15]",
   "x_2_15": "x[S3,C16]",
   "x_2_16": "x[S3,C17]",
   "x_2_17": "x[S3,C18]",
   "x_2_18": "x[S3,C19]",
   "x_2_19": "x[S3,C20]",
   "x_2_21": "x[S3,C22]",
   "x_2_22": "x[S3,C23]",
   "x_2_23": "x[S3,C24]",
   "x_2_24": "x[S3,C25]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_3_10": "x[S4,C11]",
   "x_3_11": "x[S4,C12]",
   "x_3_12": "x[S4,C13]",
   "x_3_13": "x[S4,C14]",
   "x_3_14": "x[S4,C15]",
   "x_3_15": "x[S4,C16]",
   "x_3_16": "x[S4,C17]",
   "x_3_17": "x[S4,C18]",
   "x_3_18": "x[S4,C19]",
   "x_3_19": "x[S4,C20]",
   "x_3_20": "x[S4,C21]",
   "x_3_21": "x[S4,C22]",
   "x_3_22": "x[S4,C23]",
   "x_3_23": "x[S4,C24]",
   "x_3_24": "x[S4,C25]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_4_10": "x[S5,C11]",
   "x_4_11": "x[S5,C12]",
   "x_4_12": "x[S5,C13]",
   "x_4_13": "x[S5,C14]",
   "x_4_14": "x[S5,C15]",
   "x_4_15": "x[S5,C16]",
   "x_4_16": "x[S5,C17]",
   "x_4_17": "x[S5,C18]",
   "x_4_18": "x[S5,C19]",
   "x_4_19": "x[S5,C20]",
   "x_4_20": "x[S5,C21]",
   "x_4_21": "x[S5,C22]",
   "x_4_22": "x[S5,C23]",
   "x_4_23": "x[S5,C24]",
   "x_4_24": "x[S5,C25]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_5_10": "x[S6,C11]",
   "x_5_11": "x[S6,C12]",
   "x_5_12": "x[S6,C13]",
   "x_5_13": "x[S6,C14]",
   "x_5_14": "x[S6,C15]",
   "x_5_15": "x[S6,C16]",
   "x_5_16": "x[S6,C17]",
   "x_5_17": "x[S6,C18]",
   "x_5_18": "x[S6,C19]",
   "x_5_19": "x[S6,C20]",
   "x_5_20": "x[S6,C21]",
   "x_5_21": "x[S6,C22]",
   "x_5_22": "x[S6,C23]",
   "x_5_23": "x[S6,C24]",
   "x_5_24": "x[S6,C25]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_9": "x[S7,C10]",
   "x_6_10": "x[S7,C11]",
   "x_6_11": "x[S7,C12]",
   "x_6_12": "x[S7,C13]",
   "x_6_13": "x[S7,C14]",
   "x_6_14": "x[S7,C15]",
   "x_6_15": "x[S7,C16]",
   "x_6_16": "x[S7,C17]",
   "x_6_17": "x[S7,C18]",
   "x_6_18": "x[S7,C19]",
   "x_6_19": "x[S7,C20]",
   "x_6_20": "x[S7,C21]",
   "x_6_21": "x[S7,C22]",
   "x_6_22": "x[S7,C23]",
   "x_6_23": "x[S7,C24]",
   "x_6_24": "x[S7,C25]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_7_10": "x[S8,C11]",
   "x_7_11": "x[S8,C12]",
   "x_7_12": "x[S8,C13]",
   "x_7_13": "x[S8,C14]",
   "x_7_14": "x[S8,C15]",
   "x_7_15": "x[S8,C16]",
   "x_7_16": "x[S8,C17]",
   "x_7_17": "x[S8,C18]",
   "x_7_18": "x[S8,C19]",
   "x_7_19": "x[S8,C20]",
   "x_7_21": "x[S8,C22]",
   "x_7_22": "x[S8,C23]",
   "x_7_23": "x[S8,C24]",
   "x_7_24": "x[S8,C25]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_8_10": "x[S9,C11]",
   "x_8_11": "x[S9,C12]",
   "x_8_12": "x[S9,C13]",
   "x_8_13": "x[S9,C14]",
   "x_8_14": "x[S9,C15]",
   "x_8_15": "x[S9,C16]",
   "x_8_16": "x[S9,C17]",
   "x_8_17": "x[S9,C18]",
   "x_8_18": "x[S9,C19]",
   "x_8_19": "x[S9,C20]",
   "x_8_20": "x[S9,C21]",
   "x_8_21": "x[S9,C22]",
   "x_8_22": "x[S9,C23]",
   "x_8_23": "x[S9,C24]",
   "x_8_24": "x[S9,C25]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]",
   "x_9_10": "x[S10,C11]",
   "x_9_11": "x[S10,C12]",
   "x_9_12": "x[S10,C13]",
   "x_9_13": "x[S10,C14]",
   "x_9_14": "x[S10,C15]",
   "x_9_15": "x[S10,C16]",
   "x_9_16": "x[S10,C17]",
   "x_9_17": "x[S10,C18]",
   "x_9_18": "x[S10,C19]",
   "x_9_19": "x[S10,C20]",
   "x_9_20": "x[S10,C21]",
   "x_9_21": "x[S10,C22]",
   "x_9_22": "x[S10,C23]",
   "x_9_23": "x[S10,C24]",
   "x_9_24": "x[S10,C25]",
   "x_10_0": "x[S11,C1]",
   "x_10_2": "x[S11,C3]",
   "x_10_3": "x[S11,C4]",
   "x_10_4": "x[S11,C5]",
   "x_10_5": "x[S11,C6]",
   "x_10_6": "x[S11,C7]",
   "x_10_7": "x[S11,C8]",
   "x_10_8": "x[S11,C9]",
   "x_10_9": "x[S11,C10]",
   "x_10_10": "x[S11,C11]",
   "x_10_11": "x[S11,C12]",
   "x_10_12": "x[S11,C13]",
   "x_10_13": "x[S11,C14]",
   "x_10_14": "x[S11,C15]",
   "x_10_15": "x[S11,C16]",
   "x_10_16": "x[S11,C17]",
   "x_10_17": "x[S11,C18]",
   "x_10_18": "x[S11,C19]",
   "x_10_19": "x[S11,C20]",
   "x_10_20": "x[S11,C21]",
   "x_10_21": "x[S11,C22]",
   "x_10_22": "x[S11,C23]",
   "x_10_23": "x[S11,C24]",
   "x_10_24": "x[S11,C25]",
   "x_11_0": "x[S12,C1]",
   "x_11_1": "x[S12,C2]",
   "x_11_2": "x[S12,C3]",
   "x_11_3": "x[S12,C4]",
   "x_11_4": "x[S12,C5]",
   "x_11_5": "x[S12,C6]",
   "x_11_6": "x[S12,C7]",
   "x_11_7": "x[S12,C8]",
   "x_11_8": "x[S12,C9]",
   "x_11_9": "x[S12,C10]",
   "x_11_10": "x[S12,C11]",
   "x_11_11": "x[S12,C12]",
   "x_11_12": "x[S12,C13]",
   "x_11_13": "x[S12,C14]",
   "x_11_14": "x[S12,C15]",
   "x_11_15": "x[S12,C16]",
   "x_11_16": "x[S12,C17]",
   "x_11_17": "x[S12,C18]",
   "x_11_18": "x[S12,C19]",
   "x_11_19": "x[S12,C20]",
   "x_11_20": "x[S12,C21]",
   "x_11_21": "x[S12,C22]",
   "x_11_22": "x[S12,C23]",
   "x_11_23": "x[S12,C24]",
   "x_11_24": "x[S12,C25]",
   "x_12_0": "x[S13,C1]",
   "x_12_1": "x[S13,C2]",
   "x_12_2": "x[S13,C3]",
   "x_12_3": "x[S13,C4]",
   "x_12_4": "x[S13,C5]",
   "x_12_5": "x[S13,C6]",
   "x_12_6": "x[S13,C7]",
   "x_12_7": "x[S13,C8]",
   "x_12_8": "x[S13,C9]",
   "x_12_9": "x[S13,C10]",
   "x_12_10": "x[S13,C11]",
   "x_12_11": "x[S13,C12]",
   "x_12_12": "x[S13,C13]",
   "x_12_13": "x[S13,C14]",
   "x_12_14": "x[S13,C15]",
   "x_12_15": "x[S13,C16]",
   "x_12_16": "x[S13,C17]",
   "x_12_17": "x[S13,C18]",
   "x_12_18": "x[S13,C19]",
   "x_12_19": "x[S13,C20]",
   "x_12_20": "x[S13,C21]",
   "x_12_21": "x[S13,C22]",
   "x_12_22": "x[S13,C23]",
   "x_12_23": "x[S13,C24]",
   "x_12_24": "x[S13,C25]",
   "x_13_0": "x[S14,C1]",
   "x_13_1": "x[S14,C2]",
   "x_13_2": "x[S14,C3]",
   "x_13_3": "x[S14,C4]",
   "x_13_4": "x[S14,C5]",
   "x_13_5": "x[S14,C6]",
   "x_13_6": "x[S14,C7]",
   "x_13_7": "x[S14,C8]",
   "x_13_8": "x[S14,C9]",
   "x_13_9": "x[S14,C10]",
   "x_13_10": "x[S14,C11]",
   "x_13_11": "x[S14,C12]",
   "x_13_12": "x[S14,C13]",
   "x_13_13": "x[S14,C14]",
   "x_13_14": "x[S14,C15]",
   "x_13_15": "x[S14,C16]",
   "x_13_16": "x[S14,C17]",
   "x_13_17": "x[S14,C18]",
   "x_13_18": "x[S14,C19]",
   "x_13_19": "x[S14,C20]",
   "x_13_20": "x[S14,C21]",
   "x_13_21": "x[S14,C22]",
   "x_13_22": "x[S14,C23]",
   "x_13_23": "x[S14,C24]",
   "x_13_24": "x[S14,C25]",
   "x_14_0": "x[S15,C1]",
   "x_14_1": "x[S15,C2]",
   "x_14_2": "x[S15,C3]",
   "x_14_3": "x[S15,C4]",
   "x_14_4": "x[S15,C5]",
   "x_14_5": "x[S15,C6]",
   "x_14_6": "x[S15,C7]",
   "x_14_7": "x[S15,C8]",
   "x_14_8": "x[S15,C9]",
   "x_14_9": "x[S15,C10]",
   "x_14_10": "x[S15,C11]",
   "x_14_11": "x[S15,C12]",
   "x_14_12": "x[S15,C13]",
   "x_14_13": "x[S15,C14]",
   "x_14_14": "x[S15,C15]",
   "x_14_15": "x[S15,C16]",
   "x_14_16": "x[S15,C17]",
   "x_14_17": "x[S15,C18]",
   "x_14_18": "x[S15,C19]",
   "x_14_19": "x[S15,C20]",
   "x_14_20": "x[S15,C21]",
   "x_14_21": "x[S15,C22]",
   "x_14_22": "x[S15,C23]",
   "x_14_23": "x[S15,C24]",
   "x_14_24": "x[S15,C25]",
   "x_15_0": "x[S16,C1]",
   "x_15_1": "x[S16,C2]",
   "x_15_2": "x[S16,C3]",
   "x_15_3": "x[S16,C4]",
   "x_15_4": "x[S16,C5]",
   "x_15_5": "x[S16,C6]",
   "x_15_6": "x[S16,C7]",
   "x_15_7": "x[S16,C8]",
   "x_15_8": "x[S16,C9]",
   "x_15_9": "x[S16,C10]",
   "x_15_10": "x[S16,C11]",
   "x_15_11": "x[S16,C12]",
   "x_15_12": "x[S16,C13]",
   "x_15_13": "x[S16,C14]",
   "x_15_14": "x[S16,C15]",
   "x_15_15": "x[S16,C16]",
   "x_15_16": "x[S16,C17]",
   "x_15_17": "x[S16,C18]",
   "x_15_18": "x[S16,C19]",
   "x_15_19": "x[S16,C20]",
   "x_15_20": "x[S16,C21]",
   "x_15_21": "x[S16,C22]",
   "x_15_22": "x[S16,C23]",
   "x_15_23": "x[S16,C24]",
   "x_15_24": "x[S16,C25]",
   "x_16_0": "x[S17,C1]",
   "x_16_1": "x[S17,C2]",
   "x_16_2": "x[S17,C3]",
   "x_16_3": "x[S17,C4]",
   "x_16_4": "x[S17,C5]",
   "x_16_5": "x[S17,C6]",
   "x_16_6": "x[S17,C7]",
   "x_16_7": "x[S17,C8]",
   "x_16_8": "x[S17,C9]",
   "x_16_9": "x[S17,C10]",
   "x_16_10": "x[S17,C11]",
   "x_16_11": "x[S17,C12]",
   "x_16_12": "x[S17,C13]",
   "x_16_13": "x[S17,C14]",
   "x_16_14": "x[S17,C15]",
   "x_16_15": "x[S17,C16]",
   "x_16_16": "x[S17,C17]",
   "x_16_17": "x[S17,C18]",
   "x_16_18": "x[S17,C19]",
   "x_16_19": "x[S17,C20]",
   "x_16_20": "x[S17,C21]",
   "x_16_21": "x[S17,C22]",
   "x_16_22": "x[S17,C23]",
   "x_16_23": "x[S17,C24]",
   "x_16_24": "x[S17,C25]",
   "x_17_0": "x[S18,C1]",
   "x_17_1": "x[S18,C2]",
   "x_17_2": "x[S18,C3]",
   "x_17_3": "x[S18,C4]",
   "x_17_4": "x[S18,C5]",
   "x_17_5": "x[S18,C6]",
   "x_17_6": "x[S18,C7]",
   "x_17_7": "x[S18,C8]",
   "x_17_8": "x[S18,C9]",
   "x_17_9": "x[S18,C10]",
   "x_17_10": "x[S18,C11]",
   "x_17_11": "x[S18,C12]",
   "x_17_12": "x[S18,C13]",
   "x_17_13": "x[S18,C14]",
   "x_17_14": "x[S18,C15]",
   "x_17_15": "x[S18,C16]",
   "x_17_16": "x[S18,C17]",
   "x_17_17": "x[S18,C18]",
   "x_17_18": "x[S18,C19]",
   "x_17_19": "x[S18,C20]",
   "x_17_20": "x[S18,C21]",
   "x_17_21": "x[S18,C22]",
   "x_17_22": "x[S18,C23]",
   "x_17_23": "x[S18,C24]",
   "x_17_24": "x[S18,C25]",
   "x_18_0": "x[S19,C1]",
   "x_18_1": "x[S19,C2]",
   "x_18_2": "x[S19,C3]",
   "x_18_3": "x[S19,C4]",
   "x_18_4": "x[S19,C5]",
   "x_18_5": "x[S19,C6]",
   "x_18_6": "x[S19,C7]",
   "x_18_7": "x[S19,C8]",
   "x_18_8": "x[S19,C9]",
   "x_18_9": "x[S19,C10]",
   "x_18_10": "x[S19,C11]",
   "x_18_11": "x[S19,C12]",
   "x_18_12": "x[S19,C13]",
   "x_18_13": "x[S19,C14]",
   "x_18_14": "x[S19,C15]",
   "x_18_15": "x[S19,C16]",
   "x_18_16": "x[S19,C17]",
   "x_18_17": "x[S19,C18]",
   "x_18_18": "x[S19,C19]",
   "x_18_19": "x[S19,C20]",
   "x_18_20": "x[S19,C21]",
   "x_18_21": "x[S19,C22]",
   "x_18_22": "x[S19,C23]",
   "x_18_23": "x[S19,C24]",
   "x_18_24": "x[S19,C25]",
   "x_19_0": "x[S20,C1]",
   "x_19_1": "x[S20,C2]",
   "x_19_2": "x[S20,C3]",
   "x_19_3": "x[S20,C4]",
   "x_19_4": "x[S20,C5]",
   "x_19_5": "x[S20,C6]",
   "x_19_6": "x[S20,C7]",
   "x_19_7": "x[S20,C8]",
   "x_19_8": "x[S20,C9]",
   "x_19_9": "x[S20,C10]",
   "x_19_10": "x[S20,C11]",
   "x_19_11": "x[S20,C12]",
   "x_19_12": "x[S20,C13]",
   "x_19_13": "x[S20,C14]",
   "x_19_14": "x[S20,C15]",
   "x_19_15": "x[S20,C16]",
   "x_19_16": "x[S20,C17]",
   "x_19_17": "x[S20,C18]",
   "x_19_18": "x[S20,C19]",
   "x_19_19": "x[S20,C20]",
   "x_19_20": "x[S20,C21]",
   "x_19_21": "x[S20,C22]",
   "x_19_22": "x[S20,C23]",
   "x_19_23": "x[S20,C24]",
   "x_19_24": "x[S20,C25]",
   "x_20_0": "x[S21,C1]",
   "x_20_1": "x[S21,C2]",
   "x_20_2": "x[S21,C3]",
   "x_20_3": "x[S21,C4]",
   "x_20_4": "x[S21,C5]",
   "x_20_5": "x[S21,C6]",
   "x_20_6": "x[S21,C7]",
   "x_20_7": "x[S21,C8]",
   "x_20_8": "x[S21,C9]",
   "x_20_9": "x[S21,C10]",
   "x_20_10": "x[S21,C11]",
   "x_20_11": "x[S21,C12]",
   "x_20_12": "x[S21,C13]",
   "x_20_13": "x[S21,C14]",
   "x_20_14": "x[S21,C15]",
   "x_20_15": "x[S21,C16]",
   "x_20_16": "x[S21,C17]",
   "x_20_17": "x[S21,C18]",
   "x_20_18": "x[S21,C19]",
   "x_20_19": "x[S21,C20]",
   "x_20_20": "x[S21,C21]",
   "x_20_21": "x[S21,C22]",
   "x_20_22": "x[S21,C23]",
   "x_20_23": "x[S21,C24]",
   "x_20_24": "x[S21,C25]",
   "x_21_0": "x[S22,C1]",
   "x_21_1": "x[S22,C2]",
   "x_21_2": "x[S22,C3]",
   "x_21_3": "x[S22,C4]",
   "x_21_4": "x[S22,C5]",
   "x_21_5": "x[S22,C6]",
   "x_21_6": "x[S22,C7]",
   "x_21_7": "x[S22,C8]",
   "x_21_8": "x[S22,C9]",
   "x_21_9": "x[S22,C10]",
   "x_21_10": "x[S22,C11]",
   "x_21_11": "x[S22,C12]",
   "x_21_12": "x[S22,C13]",
   "x_21_13": "x[S22,C14]",
   "x_21_14": "x[S22,C15]",
   "x_21_15": "x[S22,C16]",
   "x_21_16": "x[S22,C17]",
   "x_21_17": "x[S22,C18]",
   "x_21_18": "x[S22,C19]",
   "x_21_19": "x[S22,C20]",
   "x_21_20": "x[S22,C21]",
   "x_21_21": "x[S22,C22]",
   "x_21_22": "x[S22,C23]",
   "x_21_23": "x[S22,C24]",
   "x_21_24": "x[S22,C25]",
   "x_22_0": "x[S23,C1]",
   "x_22_1": "x[S23,C2]",
   "x_22_2": "x[S23,C3]",
   "x_22_3": "x[S23,C4]",
   "x_22_4": "x[S23,C5]",
   "x_22_5": "x[S23,C6]",
   "x_22_6": "x[S23,C7]",
   "x_22_7": "x[S23,C8]",
   "x_22_8": "x[S23,C9]",
   "x_22_9": "x[S23,C10]",
   "x_22_10": "x[S23,C11]",
   "x_22_11": "x[S23,C12]",
   "x_22_12": "x[S23,C13]",
   "x_22_13": "x[S23,C14]",
   "x_22_14": "x[S23,C15]",
   "x_22_15": "x[S23,C16]",
   "x_22_16": "x[S23,C17]",
   "x_22_17": "x[S23,C18]",
   "x_22_18": "x[S23,C19]",
   "x_22_19": "x[S23,C20]",
   "x_22_20": "x[S23,C21]",
   "x_22_21": "x[S23,C22]",
   "x_22_22": "x[S23,C23]",
   "x_22_23": "x[S23,C24]",
   "x_22_24": "x[S23,C25]",
   "x_23_0": "x[S24,C1]",
   "x_23_1": "x[S24,C2]",
   "x_23_2": "x[S24,C3]",
   "x_23_3": "x[S24,C4]",
   "x_23_4": "x[S24,C5]",
   "x_23_5": "x[S24,C6]",
   "x_23_6": "x[S24,C7]",
   "x_23_7": "x[S24,C8]",
   "x_23_8": "x[S24,C9]",
   "x_23_9": "x[S24,C10]",
   "x_23_10": "x[S24,C11]",
   "x_23_11": "x[S24,C12]",
   "x_23_12": "x[S24,C13]",
   "x_23_13": "x[S24,C14]",
   "x_23_14": "x[S24,C15]",
   "x_23_15": "x[S24,C16]",
   "x_23_16": "x[S24,C17]",
   "x_23_17": "x[S24,C18]",
   "x_23_18": "x[S24,C19]",
   "x_23_19": "x[S24,C20]",
   "x_23_20": "x[S24,C21]",
   "x_23_21": "x[S24,C22]",
   "x_23_22": "x[S24,C23]",
   "x_23_23": "x[S24,C24]",
   "x_23_24": "x[S24,C25]",
   "x_10_1": "x[S11,C2]",
   "x_6_8": "x[S7,C9]",
   "x_1_20": "x[S2,C21]",
   "x_2_20": "x[S3,C21]",
   "x_7_20": "x[S8,C21]",
   "y_0": "y[S1]",
   "y_1": "y[S2]",
   "y_2": "y[S3]",
   "y_3": "y[S4]",
   "y_4": "y[S5]",
   "y_5": "y[S6]",
   "y_6": "y[S7]",
   "y_7": "y[S8]",
   "y_8": "y[S9]",
   "y_9": "y[S10]",
   "y_10": "y[S11]",
   "y_11": "y[S12]",
   "y_12": "y[S13]",
   "y_13": "y[S14]",
   "y_14": "y[S15]",
   "y_15": "y[S16]",
   "y_16": "y[S17]",
   "y_17": "y[S18]",
   "y_18": "y[S19]",
   "y_19": "y[S20]",
   "y_20": "y[S21]",
   "y_21": "y[S22]",
   "y_22": "y[S23]",
   "y_23": "y[S24]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 18461.110000000004,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     122.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     66.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     53.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     12.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     105.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     10.0
    ],
    [
     "C108",
     89.0
    ],
    [
     "C109",
     582.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     0.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     37.0
    ],
    [
     "C202",
     1.0
    ],
    [
     "C203",
     0.0
    ],
    [
     "C204",
     253.0
    ],
    [
     "C205",
     0.0
    ],
    [
     "C206",
     0.0
    ],
    [
     "C207",
     0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     0.0
    ],
    [
     "C211",
     0.0
    ],
    [
     "C212",
     0.0
    ],
    [
     "C213",
     0.0
    ],
    [
     "C214",
     0.0
    ],
    [
     "C215",
     0.0
    ],
    [
     "C216",
     0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     0.0
    ],
    [
     "C219",
     0.0
    ],
    [
     "C220",
     0.0
    ],
    [
     "C221",
     1097.0
    ],
    [
     "C222",
     0.0
    ],
    [
     "C223",
     0.0
    ],
    [
     "C224",
     55.0
    ],
    [
     "C225",
     0.0
    ],
    [
     "C226",
     0.0
    ],
    [
     "C227",
     0.0
    ],
    [
     "C228",
     0.0
    ],
    [
     "C229",
     0.0
    ],
    [
     "C230",
     0.0
    ],
    [
     "C231",
     0.0
    ],
    [
     "C232",
     0.0
    ],
    [
     "C233",
     0.0
    ],
    [
     "C234",
     0.0
    ],
    [
     "C235",
     0.0
    ],
    [
     "C236",
     0.0
    ],
    [
     "C237",
     0.0
    ],
    [
     "C238",
     0.0
    ],
    [
     "C239",
     0.0
    ],
    [
     "C240",
     0.0
    ],
    [
     "C241",
     0.0
    ],
    [
     "C242",
     0.0
    ],
    [
     "C243",
     0.0
    ],
    [
     "C244",
     0.0
    ],
    [
     "C245",
     0.0
    ],
    [
     "C246",
     0.0
    ],
    [
     "C247",
     0.0
    ],
    [
     "C248",
     0.0
    ],
    [
     "C249",
     0.0
    ],
    [
     "C250",
     0.0
    ],
    [
     "C251",
     0.0
    ],
    [
     "C252",
     0.0
    ],
    [
     "C253",
     0.0
    ],
    [
     "C254",
     0.0
    ],
    [
     "C255",
     0.0
    ],
    [
     "C256",
     0.0
    ],
    [
     "C257",
     0.0
    ],
    [
     "C258",
     0.0
    ],
    [
     "C259",
     0.0
    ],
    [
     "C260",
     0.0
    ],
    [
     "C261",
     0.0
    ],
    [
     "C262",
     0.0
    ],
    [
     "C263",
     0.0
    ],
    [
     "C264",
     0.0
    ],
    [
     "C265",
     0.0
    ],
    [
     "C266",
     0.0
    ],
    [
     "C267",
     0.0
    ],
    [
     "C268",
     0.0
    ],
    [
     "C269",
     0.0
    ],
    [
     "C270",
     0.0
    ],
    [
     "C271",
     0.0
    ],
    [
     "C272",
     0.0
    ],
    [
     "C273",
     0.0
    ],
    [
     "C274",
     0.0
    ],
    [
     "C275",
     0.0
    ],
    [
     "C276",
     0.0
    ],
    [
     "C277",
     0.0
    ],
    [
     "C278",
     0.0
    ],
    [
     "C279",
     0.0
    ],
    [
     "C280",
     0.0
    ],
    [
     "C281",
     0.0
    ],
    [
     "C282",
     0.0
    ],
    [
     "C283",
     0.0
    ],
    [
     "C284",
     0.0
    ],
    [
     "C285",
     0.0
    ],
    [
     "C286",
     0.0
    ],
    [
     "C287",
     0.0
    ],
    [
     "C288",
     0.0
    ],
    [
     "C289",
     0.0
    ],
    [
     "C290",
     0.0
    ],
    [
     "C291",
     0.0
    ],
    [
     "C292",
     0.0
    ],
    [
     "C293",
     0.0
    ],
    [
     "C294",
     0.0
    ],
    [
     "C295",
     0.0
    ],
    [
     "C296",
     0.0
    ],
    [
     "C297",
     0.0
    ],
    [
     "C298",
     0.0
    ],
    [
     "C299",
     0.0
    ],
    [
     "C300",
     0.0
    ],
    [
     "C301",
     0.0
    ],
    [
     "C302",
     0.0
    ],
    [
     "C303",
     0.0
    ],
    [
     "C304",
     0.0
    ],
    [
     "C305",
     0.0
    ],
    [
     "C306",
     0.0
    ],
    [
     "C307",
     0.0
    ],
    [
     "C308",
     0.0
    ],
    [
     "C309",
     0.0
    ],
    [
     "C310",
     122.0
    ],
    [
     "C311",
     7.0
    ],
    [
     "C312",
     0.0
    ],
    [
     "C313",
     0.0
    ],
    [
     "C314",
     0.0
    ],
    [
     "C315",
     0.0
    ],
    [
     "C316",
     0.0
    ],
    [
     "C317",
     0.0
    ],
    [
     "C318",
     21.0
    ],
    [
     "C319",
     0.0
    ],
    [
     "C320",
     0.0
    ],
    [
     "C321",
     0.0
    ],
    [
     "C322",
     0.0
    ],
    [
     "C323",
     0.0
    ],
    [
     "C324",
     0.0
    ],
    [
     "C325",
     0.0
    ],
    [
     "C326",
     0.0
    ],
    [
     "C327",
     0.0
    ],
    [
     "C328",
     0.0
    ],
    [
     "C329",
     0.0
    ],
    [
     "C330",
     0.0
    ],
    [
     "C331",
     0.0
    ],
    [
     "C332",
     0.0
    ],
    [
     "C333",
     0.0
    ],
    [
     "C334",
     0.0
    ],
    [
     "C335",
     0.0
    ],
    [
     "C336",
     0.0
    ],
    [
     "C337",
     0.0
    ],
    [
     "C338",
     0.0
    ],
    [
     "C339",
     0.0
    ],
    [
     "C340",
     0.0
    ],
    [
     "C341",
     0.0
    ],
    [
     "C342",
     0.0
    ],
    [
     "C343",
     0.0
    ],
    [
     "C344",
     0.0
    ],
    [
     "C345",
     0.0
    ],
    [
     "C346",
     0.0
    ],
    [
     "C347",
     0.0
    ],
    [
     "C348",
     0.0
    ],
    [
     "C349",
     0.0
    ],
    [
     "C350",
     0.0
    ],
    [
     "C351",
     0.0
    ],
    [
     "C352",
     0.0
    ],
    [
     "C353",
     0.0
    ],
    [
     "C354",
     0.0
    ],
    [
     "C355",
     0.0
    ],
    [
     "C356",
     0.0
    ],
    [
     "C357",
     0.0
    ],
    [
     "C358",
     0.0
    ],
    [
     "C359",
     0.0
    ],
    [
     "C360",
     0.0
    ],
    [
     "C361",
     0.0
    ],
    [
     "C362",
     0.0
    ],
    [
     "C363",
     0.0
    ],
    [
     "C364",
     0.0
    ],
    [
     "C365",
     0.0
    ],
    [
     "C366",
     42.0
    ],
    [
     "C367",
     0.0
    ],
    [
     "C368",
     0.0
    ],
    [
     "C369",
     82.0
    ],
    [
     "C370",
     0.0
    ],
    [
     "C371",
     0.0
    ],
    [
     "C372",
     11.0
    ],
    [
     "C373",
     0.0
    ],
    [
     "C374",
     0.0
    ],
    [
     "C375",
     0.0
    ],
    [
     "C376",
     0.0
    ],
    [
     "C377",
     0.0
    ],
    [
     "C378",
     0.0
    ],
    [
     "C379",
     0.0
    ],
    [
     "C380",
     0.0
    ],
    [
     "C381",
     0.0
    ],
    [
     "C382",
     0.0
    ],
    [
     "C383",
     0.0
    ],
    [
     "C384",
     0.0
    ],
    [
     "C385",
     0.0
    ],
    [
     "C386",
     0.0
    ],
    [
     "C387",
     0.0
    ],
    [
     "C388",
     0.0
    ],
    [
     "C389",
     0.0
    ],
    [
     "C390",
     0.0
    ],
    [
     "C391",
     0.0
    ],
    [
     "C392",
     0.0
    ],
    [
     "C393",
     0.0
    ],
    [
     "C394",
     0.0
    ],
    [
     "C395",
     0.0
    ],
    [
     "C396",
     0.0
    ],
    [
     "C397",
     0.0
    ],
    [
     "C398",
     0.0
    ],
    [
     "C399",
     0.0
    ],
    [
     "C400",
     0.0
    ],
    [
     "C401",
     0.0
    ],
    [
     "C402",
     0.0
    ],
    [
     "C403",
     0.0
    ],
    [
     "C404",
     0.0
    ],
    [
     "C405",
     0.0
    ],
    [
     "C406",
     0.0
    ],
    [
     "C407",
     0.0
    ],
    [
     "C408",
     0.0
    ],
    [
     "C409",
     0.0
    ],
    [
     "C410",
     0.0
    ],
    [
     "C411",
     0.0
    ],
    [
     "C412",
     0.0
    ],
    [
     "C413",
     0.0
    ],
    [
     "C414",
     0.0
    ],
    [
     "C415",
     0.0
    ],
    [
     "C416",
     0.0
    ],
    [
     "C417",
     0.0
    ],
    [
     "C418",
     0.0
    ],
    [
     "C419",
     0.0
    ],
    [
     "C420",
     0.0
    ],
    [
     "C421",
     0.0
    ],
    [
     "C422",
     0.0
    ],
    [
     "C423",
     0.0
    ],
    [
     "C424",
     0.0
    ],
    [
     "C425",
     0.0
    ],
    [
     "C426",
     0.0
    ],
    [
     "C427",
     0.0
    ],
    [
     "C428",
     61.0
    ],
    [
     "C429",
     0.0
    ],
    [
     "C430",
     0.0
    ],
    [
     "C431",
     53.0
    ],
    [
     "C432",
     0.0
    ],
    [
     "C433",
     0.0
    ],
    [
     "C434",
     0.0
    ],
    [
     "C435",
     0.0
    ],
    [
     "C436",
     0.0
    ],
    [
     "C437",
     0.0
    ],
    [
     "C438",
     60.0
    ],
    [
     "C439",
     483.0
    ],
    [
     "C440",
     0.0
    ],
    [
     "C441",
     0.0
    ],
    [
     "C442",
     0.0
    ],
    [
     "C443",
     0.0
    ],
    [
     "C444",
     0.0
    ],
    [
     "C445",
     0.0
    ],
    [
     "C446",
     0.0
    ],
    [
     "C447",
     0.0
    ],
    [
     "C448",
     24.0
    ],
    [
     "C449",
     0.0
    ],
    [
     "C450",
     0.0
    ],
    [
     "C451",
     0.0
    ],
    [
     "C452",
     0.0
    ],
    [
     "C453",
     0.0
    ],
    [
     "C454",
     0.0
    ],
    [
     "C455",
     0.0
    ],
    [
     "C456",
     0.0
    ],
    [
     "C457",
     0.0
    ],
    [
     "C458",
     0.0
    ],
    [
     "C459",
     0.0
    ],
    [
     "C460",
     0.0
    ],
    [
     "C461",
     0.0
    ],
    [
     "C462",
     223.0
    ],
    [
     "C463",
     0.0
    ],
    [
     "C464",
     0.0
    ],
    [
     "C465",
     0.0
    ],
    [
     "C466",
     0.0
    ],
    [
     "C467",
     0.0
    ],
    [
     "C468",
     0.0
    ],
    [
     "C469",
     0.0
    ],
    [
     "C470",
     0.0
    ],
    [
     "C471",
     0.0
    ],
    [
     "C472",
     0.0
    ],
    [
     "C473",
     0.0
    ],
    [
     "C474",
     0.0
    ],
    [
     "C475",
     0.0
    ],
    [
     "C476",
     0.0
    ],
    [
     "C477",
     0.0
    ],
    [
     "C478",
     0.0
    ],
    [
     "C479",
     0.0
    ],
    [
     "C480",
     0.0
    ],
    [
     "C481",
     0.0
    ],
    [
     "C482",
     0.0
    ],
    [
     "C483",
     0.0
    ],
    [
     "C484",
     0.0
    ],
    [
     "C485",
     0.0
    ],
    [
     "C486",
     0.0
    ],
    [
     "C487",
     0.0
    ],
    [
     "C488",
     0.0
    ],
    [
     "C489",
     0.0
    ],
    [
     "C490",
     0.0
    ],
    [
     "C491",
     0.0
    ],
    [
     "C492",
     0.0
    ],
    [
     "C493",
     0.0
    ],
    [
     "C494",
     0.0
    ],
    [
     "C495",
     0.0
    ],
    [
     "C496",
     0.0
    ],
    [
     "C497",
     0.0
    ],
    [
     "C498",
     0.0
    ],
    [
     "C499",
     0.0
    ],
    [
     "C500",
     0.0
    ],
    [
     "C501",
     0.0
    ],
    [
     "C502",
     0.0
    ],
    [
     "C503",
     0.0
    ],
    [
     "C504",
     0.0
    ],
    [
     "C505",
     0.0
    ],
    [
     "C506",
     0.0
    ],
    [
     "C507",
     0.0
    ],
    [
     "C508",
     0.0
    ],
    [
     "C509",
     0.0
    ],
    [
     "C510",
     0.0
    ],
    [
     "C511",
     0.0
    ],
    [
     "C512",
     0.0
    ],
    [
     "C513",
     0.0
    ],
    [
     "C514",
     0.0
    ],
    [
     "C515",
     0.0
    ],
    [
     "C516",
     0.0
    ],
    [
     "C517",
     0.0
    ],
    [
     "C518",
     0.0
    ],
    [
     "C519",
     0.0
    ],
    [
     "C520",
     0.0
    ],
    [
     "C521",
     0.0
    ],
    [
     "C522",
     0.0
    ],
    [
     "C523",
     0.0
    ],
    [
     "C524",
     0.0
    ],
    [
     "C525",
     0.0
    ],
    [
     "C526",
     0.0
    ],
    [
     "C527",
     0.0
    ],
    [
     "C528",
     0.0
    ],
    [
     "C529",
     0.0
    ],
    [
     "C530",
     0.0
    ],
    [
     "C531",
     0.0
    ],
    [
     "C532",
     0.0
    ],
    [
     "C533",
     0.0
    ],
    [
     "C534",
     0.0
    ],
    [
     "C535",
     0.0
    ],
    [
     "C536",
     0.0
    ],
    [
     "C537",
     0.0
    ],
    [
     "C538",
     0.0
    ],
    [
     "C539",
     0.0
    ],
    [
     "C540",
     0.0
    ],
    [
     "C541",
     0.0
    ],
    [
     "C542",
     0.0
    ],
    [
     "C543",
     0.0
    ],
    [
     "C544",
     0.0
    ],
    [
     "C545",
     0.0
    ],
    [
     "C546",
     0.0
    ],
    [
     "C547",
     0.0
    ],
    [
     "C548",
     0.0
    ],
    [
     "C549",
     0.0
    ],
    [
     "C550",
     0.0
    ],
    [
     "C551",
     0.0
    ],
    [
     "C552",
     0.0
    ],
    [
     "C553",
     0.0
    ],
    [
     "C554",
     0.0
    ],
    [
     "C555",
     0.0
    ],
    [
     "C556",
     0.0
    ],
    [
     "C557",
     0.0
    ],
    [
     "C558",
     0.0
    ],
    [
     "C559",
     0.0
    ],
    [
     "C560",
     0.0
    ],
    [
     "C561",
     0.0
    ],
    [
     "C562",
     0.0
    ],
    [
     "C563",
     0.0
    ],
    [
     "C564",
     0.0
    ],
    [
     "C565",
     0.0
    ],
    [
     "C566",
     0.0
    ],
    [
     "C567",
     0.0
    ],
    [
     "C568",
     0.0
    ],
    [
     "C569",
     0.0
    ],
    [
     "C570",
     0.0
    ],
    [
     "C571",
     0.0
    ],
    [
     "C572",
     0.0
    ],
    [
     "C573",
     0.0
    ],
    [
     "C574",
     0.0
    ],
    [
     "C575",
     0.0
    ],
    [
     "C576",
     0.0
    ],
    [
     "C577",
     0.0
    ],
    [
     "C578",
     0.0
    ],
    [
     "C579",
     0.0
    ],
    [
     "C580",
     0.0
    ],
    [
     "C581",
     0.0
    ],
    [
     "C582",
     0.0
    ],
    [
     "C583",
     0.0
    ],
    [
     "C584",
     0.0
    ],
    [
     "C585",
     0.0
    ],
    [
     "C586",
     0.0
    ],
    [
     "C587",
     0.0
    ],
    [
     "C588",
     0.0
    ],
    [
     "C589",
     0.0
    ],
    [
     "C590",
     0.0
    ],
    [
     "C591",
     0.0
    ],
    [
     "C592",
     0.0
    ],
    [
     "C593",
     0.0
    ],
    [
     "C594",
     0.0
    ],
    [
     "C595",
     0.0
    ],
    [
     "C596",
     0.0
    ],
    [
     "C597",
     0.0
    ],
    [
     "C598",
     0.0
    ],
    [
     "C599",
     0.0
    ],
    [
     "C600",
     1.0
    ],
    [
     "C601",
     0.0
    ],
    [
     "C602",
     0.0
    ],
    [
     "C603",
     0.0
    ],
    [
     "C604",
     1.0
    ],
    [
     "C605",
     -0.0
    ],
    [
     "C606",
     -0.0
    ],
    [
     "C607",
     -0.0
    ],
    [
     "C608",
     1.0
    ],
    [
     "C609",
     0.0
    ],
    [
     "C610",
     0.0
    ],
    [
     "C611",
     0.0
    ],
    [
     "C612",
     1.0
    ],
    [
     "C613",
     -0.0
    ],
    [
     "C614",
     1.0
    ],
    [
     "C615",
     0.0
    ],
    [
     "C616",
     0.0
    ],
    [
     "C617",
     1.0
    ],
    [
     "C618",
     1.0
    ],
    [
     "C619",
     -0.0
    ],
    [
     "C620",
     0.0
    ],
    [
     "C621",
     0.0
    ],
    [
     "C622",
     0.0
    ],
    [
     "C623",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-067": {
  "source_code_sha256": "c1e820244f2040d42011b31f6ef3305e336d30282b24834512c3539e268bc6fd",
  "original_solution_sha256": "80b7db5f84ce9becda4807c8f2c70bc69aab9561f86bc8bec9e138f6dbae405f",
  "bindings": [
   [
    "C0",
    "x[MA,P1]"
   ],
   [
    "C1",
    "x[MA,P2]"
   ],
   [
    "C2",
    "x[MA,P3]"
   ],
   [
    "C3",
    "x[MB,P1]"
   ],
   [
    "C4",
    "x[MB,P2]"
   ],
   [
    "C5",
    "x[MB,P3]"
   ],
   [
    "C6",
    "x[MC,P1]"
   ],
   [
    "C7",
    "x[MC,P2]"
   ],
   [
    "C8",
    "x[MC,P3]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[MA,P1]",
   "x_0_1": "x[MA,P2]",
   "x_0_2": "x[MA,P3]",
   "x_1_0": "x[MB,P1]",
   "x_1_1": "x[MB,P2]",
   "x_1_2": "x[MB,P3]",
   "x_2_0": "x[MC,P1]",
   "x_2_1": "x[MC,P2]",
   "x_2_2": "x[MC,P3]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 9000.0,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-070": {
  "source_code_sha256": "8ce92fc49d5077d903002ea5b29525915793d17c08461d54611fb4cc764d3325",
  "original_solution_sha256": "810dcd637f4b50d39b4745b5791639fbba8c1b2c3186ab8bb472fc6112e3174d",
  "bindings": [
   [
    "C0",
    "x[Manager 1,Project 1]"
   ],
   [
    "C1",
    "x[Manager 1,Project 2]"
   ],
   [
    "C2",
    "x[Manager 1,Project 3]"
   ],
   [
    "C3",
    "x[Manager 1,Project 4]"
   ],
   [
    "C4",
    "x[Manager 1,Project 5]"
   ],
   [
    "C5",
    "x[Manager 1,Project 6]"
   ],
   [
    "C6",
    "x[Manager 1,Project 7]"
   ],
   [
    "C7",
    "x[Manager 2,Project 1]"
   ],
   [
    "C8",
    "x[Manager 2,Project 2]"
   ],
   [
    "C9",
    "x[Manager 2,Project 3]"
   ],
   [
    "C10",
    "x[Manager 2,Project 4]"
   ],
   [
    "C11",
    "x[Manager 2,Project 5]"
   ],
   [
    "C12",
    "x[Manager 2,Project 6]"
   ],
   [
    "C13",
    "x[Manager 2,Project 7]"
   ],
   [
    "C14",
    "x[Manager 3,Project 1]"
   ],
   [
    "C15",
    "x[Manager 3,Project 2]"
   ],
   [
    "C16",
    "x[Manager 3,Project 3]"
   ],
   [
    "C17",
    "x[Manager 3,Project 4]"
   ],
   [
    "C18",
    "x[Manager 3,Project 5]"
   ],
   [
    "C19",
    "x[Manager 3,Project 6]"
   ],
   [
    "C20",
    "x[Manager 3,Project 7]"
   ],
   [
    "C21",
    "x[Manager 4,Project 1]"
   ],
   [
    "C22",
    "x[Manager 4,Project 2]"
   ],
   [
    "C23",
    "x[Manager 4,Project 3]"
   ],
   [
    "C24",
    "x[Manager 4,Project 4]"
   ],
   [
    "C25",
    "x[Manager 4,Project 5]"
   ],
   [
    "C26",
    "x[Manager 4,Project 6]"
   ],
   [
    "C27",
    "x[Manager 4,Project 7]"
   ],
   [
    "C28",
    "x[Manager 5,Project 1]"
   ],
   [
    "C29",
    "x[Manager 5,Project 2]"
   ],
   [
    "C30",
    "x[Manager 5,Project 3]"
   ],
   [
    "C31",
    "x[Manager 5,Project 4]"
   ],
   [
    "C32",
    "x[Manager 5,Project 5]"
   ],
   [
    "C33",
    "x[Manager 5,Project 6]"
   ],
   [
    "C34",
    "x[Manager 5,Project 7]"
   ],
   [
    "C35",
    "x[Manager 6,Project 1]"
   ],
   [
    "C36",
    "x[Manager 6,Project 2]"
   ],
   [
    "C37",
    "x[Manager 6,Project 3]"
   ],
   [
    "C38",
    "x[Manager 6,Project 4]"
   ],
   [
    "C39",
    "x[Manager 6,Project 5]"
   ],
   [
    "C40",
    "x[Manager 6,Project 6]"
   ],
   [
    "C41",
    "x[Manager 6,Project 7]"
   ],
   [
    "C42",
    "x[Manager 7,Project 1]"
   ],
   [
    "C43",
    "x[Manager 7,Project 2]"
   ],
   [
    "C44",
    "x[Manager 7,Project 3]"
   ],
   [
    "C45",
    "x[Manager 7,Project 4]"
   ],
   [
    "C46",
    "x[Manager 7,Project 5]"
   ],
   [
    "C47",
    "x[Manager 7,Project 6]"
   ],
   [
    "C48",
    "x[Manager 7,Project 7]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Manager 1,Project 1]",
   "x_0_1": "x[Manager 1,Project 2]",
   "x_0_2": "x[Manager 1,Project 3]",
   "x_0_3": "x[Manager 1,Project 4]",
   "x_0_4": "x[Manager 1,Project 5]",
   "x_0_5": "x[Manager 1,Project 6]",
   "x_0_6": "x[Manager 1,Project 7]",
   "x_1_0": "x[Manager 2,Project 1]",
   "x_1_1": "x[Manager 2,Project 2]",
   "x_1_2": "x[Manager 2,Project 3]",
   "x_1_3": "x[Manager 2,Project 4]",
   "x_1_4": "x[Manager 2,Project 5]",
   "x_1_5": "x[Manager 2,Project 6]",
   "x_1_6": "x[Manager 2,Project 7]",
   "x_2_0": "x[Manager 3,Project 1]",
   "x_2_1": "x[Manager 3,Project 2]",
   "x_2_2": "x[Manager 3,Project 3]",
   "x_2_3": "x[Manager 3,Project 4]",
   "x_2_4": "x[Manager 3,Project 5]",
   "x_2_5": "x[Manager 3,Project 6]",
   "x_2_6": "x[Manager 3,Project 7]",
   "x_3_0": "x[Manager 4,Project 1]",
   "x_3_1": "x[Manager 4,Project 2]",
   "x_3_2": "x[Manager 4,Project 3]",
   "x_3_3": "x[Manager 4,Project 4]",
   "x_3_4": "x[Manager 4,Project 5]",
   "x_3_5": "x[Manager 4,Project 6]",
   "x_3_6": "x[Manager 4,Project 7]",
   "x_4_0": "x[Manager 5,Project 1]",
   "x_4_1": "x[Manager 5,Project 2]",
   "x_4_2": "x[Manager 5,Project 3]",
   "x_4_3": "x[Manager 5,Project 4]",
   "x_4_4": "x[Manager 5,Project 5]",
   "x_4_5": "x[Manager 5,Project 6]",
   "x_4_6": "x[Manager 5,Project 7]",
   "x_5_0": "x[Manager 6,Project 1]",
   "x_5_1": "x[Manager 6,Project 2]",
   "x_5_2": "x[Manager 6,Project 3]",
   "x_5_3": "x[Manager 6,Project 4]",
   "x_5_4": "x[Manager 6,Project 5]",
   "x_5_5": "x[Manager 6,Project 6]",
   "x_5_6": "x[Manager 6,Project 7]",
   "x_6_0": "x[Manager 7,Project 1]",
   "x_6_1": "x[Manager 7,Project 2]",
   "x_6_2": "x[Manager 7,Project 3]",
   "x_6_3": "x[Manager 7,Project 4]",
   "x_6_4": "x[Manager 7,Project 5]",
   "x_6_5": "x[Manager 7,Project 6]",
   "x_6_6": "x[Manager 7,Project 7]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 8809.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     1.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     1.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     1.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     1.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     1.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     1.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-074": {
  "source_code_sha256": "9edb40566d97bb3be082a2ff33cc64b479ee3c216daac5eaa1b97a816549ac48",
  "original_solution_sha256": "bdecbccb5c8bc776e31b1be37e3606d424d7f2fa947ba920f4190f53b33bfa51",
  "bindings": [
   [
    "C0",
    "y[0]"
   ],
   [
    "C1",
    "y[1]"
   ],
   [
    "C2",
    "y[2]"
   ],
   [
    "C3",
    "y[3]"
   ],
   [
    "C4",
    "y[4]"
   ],
   [
    "C5",
    "y[5]"
   ],
   [
    "C6",
    "y[6]"
   ],
   [
    "C7",
    "y[7]"
   ],
   [
    "C8",
    "y[8]"
   ],
   [
    "C9",
    "y[9]"
   ],
   [
    "C10",
    "y[10]"
   ],
   [
    "C11",
    "y[11]"
   ],
   [
    "C12",
    "y[12]"
   ],
   [
    "C13",
    "y[13]"
   ],
   [
    "C14",
    "y[14]"
   ],
   [
    "C15",
    "y[15]"
   ],
   [
    "C16",
    "y[16]"
   ],
   [
    "C17",
    "y[17]"
   ],
   [
    "C18",
    "y[18]"
   ],
   [
    "C19",
    "y[19]"
   ],
   [
    "C20",
    "y[20]"
   ],
   [
    "C21",
    "y[21]"
   ],
   [
    "C22",
    "y[22]"
   ],
   [
    "C23",
    "y[23]"
   ],
   [
    "C24",
    "y[24]"
   ],
   [
    "C25",
    "y[25]"
   ],
   [
    "C26",
    "y[26]"
   ],
   [
    "C27",
    "y[27]"
   ],
   [
    "C28",
    "y[28]"
   ],
   [
    "C29",
    "y[29]"
   ],
   [
    "C30",
    "y[30]"
   ],
   [
    "C31",
    "y[31]"
   ],
   [
    "C32",
    "y[32]"
   ],
   [
    "C33",
    "y[33]"
   ],
   [
    "C34",
    "y[34]"
   ],
   [
    "C35",
    "y[35]"
   ],
   [
    "C36",
    "y[36]"
   ],
   [
    "C37",
    "y[37]"
   ],
   [
    "C38",
    "y[38]"
   ],
   [
    "C39",
    "y[39]"
   ],
   [
    "C40",
    "y[40]"
   ],
   [
    "C41",
    "y[41]"
   ],
   [
    "C42",
    "y[42]"
   ],
   [
    "C43",
    "y[43]"
   ],
   [
    "C44",
    "y[44]"
   ],
   [
    "C45",
    "y[45]"
   ],
   [
    "C46",
    "y[46]"
   ],
   [
    "C47",
    "y[47]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "y[0]",
   "x[1]": "y[1]",
   "x[2]": "y[2]",
   "x[3]": "y[3]",
   "x[4]": "y[4]",
   "x[5]": "y[5]",
   "x[6]": "y[6]",
   "x[7]": "y[7]",
   "x[8]": "y[8]",
   "x[9]": "y[9]",
   "x[10]": "y[10]",
   "x[11]": "y[11]",
   "x[12]": "y[12]",
   "x[13]": "y[13]",
   "x[14]": "y[14]",
   "x[15]": "y[15]",
   "x[16]": "y[16]",
   "x[17]": "y[17]",
   "x[18]": "y[18]",
   "x[19]": "y[19]",
   "x[20]": "y[20]",
   "x[21]": "y[21]",
   "x[22]": "y[22]",
   "x[23]": "y[23]",
   "x[24]": "y[24]",
   "x[25]": "y[25]",
   "x[26]": "y[26]",
   "x[27]": "y[27]",
   "x[28]": "y[28]",
   "x[29]": "y[29]",
   "x[30]": "y[30]",
   "x[31]": "y[31]",
   "x[32]": "y[32]",
   "x[33]": "y[33]",
   "x[34]": "y[34]",
   "x[35]": "y[35]",
   "x[36]": "y[36]",
   "x[37]": "y[37]",
   "x[38]": "y[38]",
   "x[39]": "y[39]",
   "x[40]": "y[40]",
   "x[41]": "y[41]",
   "x[42]": "y[42]",
   "x[43]": "y[43]",
   "x[44]": "y[44]",
   "x[45]": "y[45]",
   "x[46]": "y[46]",
   "x[47]": "y[47]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 25.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     2.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     5.0
    ],
    [
     "C6",
     2.0
    ],
    [
     "C7",
     2.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     1.0
    ],
    [
     "C19",
     1.0
    ],
    [
     "C20",
     1.0
    ],
    [
     "C21",
     4.0
    ],
    [
     "C22",
     1.0
    ],
    [
     "C23",
     1.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     1.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     1.0
    ],
    [
     "C37",
     2.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-077": {
  "source_code_sha256": "1ae2547ff3a9004dfa78f2842d2f6ddeca7fefbca3ffe1c94cd57342bb758234",
  "original_solution_sha256": "2adcbede4949f9e7827335fe535c6007bd8de9da543c3e67286e38a1281ee354",
  "bindings": [
   [
    "C0",
    "x[1]"
   ],
   [
    "C1",
    "x[2]"
   ],
   [
    "C2",
    "x[3]"
   ],
   [
    "C3",
    "x[4]"
   ],
   [
    "C4",
    "x[5]"
   ],
   [
    "C5",
    "x[6]"
   ],
   [
    "C6",
    "x[7]"
   ],
   [
    "C7",
    "x[8]"
   ],
   [
    "C8",
    "x[9]"
   ],
   [
    "C9",
    "x[10]"
   ],
   [
    "C10",
    "x[11]"
   ],
   [
    "C11",
    "x[12]"
   ],
   [
    "C12",
    "x[13]"
   ],
   [
    "C13",
    "x[14]"
   ],
   [
    "C14",
    "x[15]"
   ],
   [
    "C15",
    "x[16]"
   ],
   [
    "C16",
    "x[17]"
   ],
   [
    "C17",
    "x[18]"
   ],
   [
    "C18",
    "x[19]"
   ],
   [
    "C19",
    "x[20]"
   ],
   [
    "C20",
    "x[21]"
   ],
   [
    "C21",
    "x[22]"
   ],
   [
    "C22",
    "x[23]"
   ],
   [
    "C23",
    "x[24]"
   ],
   [
    "C24",
    "x[25]"
   ],
   [
    "C25",
    "x[26]"
   ],
   [
    "C26",
    "x[27]"
   ],
   [
    "C27",
    "x[28]"
   ],
   [
    "C28",
    "x[29]"
   ],
   [
    "C29",
    "x[30]"
   ],
   [
    "C30",
    "x[31]"
   ],
   [
    "C31",
    "x[32]"
   ],
   [
    "C32",
    "x[33]"
   ],
   [
    "C33",
    "x[34]"
   ],
   [
    "C34",
    "x[35]"
   ],
   [
    "C35",
    "x[36]"
   ],
   [
    "C36",
    "x[37]"
   ],
   [
    "C37",
    "x[38]"
   ],
   [
    "C38",
    "x[39]"
   ],
   [
    "C39",
    "x[40]"
   ],
   [
    "C40",
    "x[41]"
   ],
   [
    "C41",
    "x[42]"
   ],
   [
    "C42",
    "x[43]"
   ],
   [
    "C43",
    "x[44]"
   ],
   [
    "C44",
    "x[45]"
   ],
   [
    "C45",
    "x[46]"
   ],
   [
    "C46",
    "x[47]"
   ],
   [
    "C47",
    "x[48]"
   ],
   [
    "C48",
    "x[49]"
   ],
   [
    "C49",
    "x[50]"
   ],
   [
    "C50",
    "x[51]"
   ],
   [
    "C51",
    "x[52]"
   ],
   [
    "C52",
    "x[53]"
   ],
   [
    "C53",
    "x[54]"
   ],
   [
    "C54",
    "x[55]"
   ],
   [
    "C55",
    "x[56]"
   ],
   [
    "C56",
    "x[57]"
   ],
   [
    "C57",
    "x[58]"
   ],
   [
    "C58",
    "x[59]"
   ],
   [
    "C59",
    "x[60]"
   ],
   [
    "C60",
    "x[61]"
   ],
   [
    "C61",
    "x[62]"
   ],
   [
    "C62",
    "x[63]"
   ],
   [
    "C63",
    "x[64]"
   ],
   [
    "C64",
    "x[65]"
   ],
   [
    "C65",
    "x[66]"
   ],
   [
    "C66",
    "x[67]"
   ],
   [
    "C67",
    "x[68]"
   ],
   [
    "C68",
    "x[69]"
   ],
   [
    "C69",
    "x[70]"
   ],
   [
    "C70",
    "x[71]"
   ],
   [
    "C71",
    "x[72]"
   ],
   [
    "C72",
    "x[73]"
   ],
   [
    "C73",
    "x[74]"
   ],
   [
    "C74",
    "x[75]"
   ],
   [
    "C75",
    "x[76]"
   ],
   [
    "C76",
    "x[77]"
   ],
   [
    "C77",
    "x[78]"
   ],
   [
    "C78",
    "x[79]"
   ],
   [
    "C79",
    "x[80]"
   ],
   [
    "C80",
    "x[81]"
   ],
   [
    "C81",
    "x[82]"
   ],
   [
    "C82",
    "x[83]"
   ],
   [
    "C83",
    "x[84]"
   ],
   [
    "C84",
    "x[85]"
   ],
   [
    "C85",
    "x[86]"
   ],
   [
    "C86",
    "x[87]"
   ],
   [
    "C87",
    "x[88]"
   ],
   [
    "C88",
    "x[89]"
   ],
   [
    "C89",
    "x[90]"
   ],
   [
    "C90",
    "x[91]"
   ],
   [
    "C91",
    "x[92]"
   ],
   [
    "C92",
    "x[93]"
   ],
   [
    "C93",
    "x[94]"
   ],
   [
    "C94",
    "x[95]"
   ],
   [
    "C95",
    "x[96]"
   ],
   [
    "C96",
    "x[97]"
   ],
   [
    "C97",
    "x[98]"
   ],
   [
    "C98",
    "x[99]"
   ],
   [
    "C99",
    "x[100]"
   ],
   [
    "C100",
    "x[101]"
   ],
   [
    "C101",
    "x[102]"
   ],
   [
    "C102",
    "x[103]"
   ],
   [
    "C103",
    "x[104]"
   ],
   [
    "C104",
    "x[105]"
   ],
   [
    "C105",
    "x[106]"
   ],
   [
    "C106",
    "x[107]"
   ],
   [
    "C107",
    "x[108]"
   ],
   [
    "C108",
    "x[109]"
   ],
   [
    "C109",
    "x[110]"
   ],
   [
    "C110",
    "x[111]"
   ],
   [
    "C111",
    "x[112]"
   ],
   [
    "C112",
    "x[113]"
   ],
   [
    "C113",
    "x[114]"
   ],
   [
    "C114",
    "x[115]"
   ],
   [
    "C115",
    "x[116]"
   ],
   [
    "C116",
    "x[117]"
   ],
   [
    "C117",
    "x[118]"
   ],
   [
    "C118",
    "x[119]"
   ],
   [
    "C119",
    "x[120]"
   ],
   [
    "C120",
    "x[121]"
   ],
   [
    "C121",
    "x[122]"
   ],
   [
    "C122",
    "x[123]"
   ],
   [
    "C123",
    "x[124]"
   ],
   [
    "C124",
    "x[125]"
   ],
   [
    "C125",
    "x[126]"
   ],
   [
    "C126",
    "x[127]"
   ],
   [
    "C127",
    "x[128]"
   ],
   [
    "C128",
    "x[129]"
   ],
   [
    "C129",
    "x[130]"
   ],
   [
    "C130",
    "x[131]"
   ],
   [
    "C131",
    "x[132]"
   ],
   [
    "C132",
    "x[133]"
   ],
   [
    "C133",
    "x[134]"
   ],
   [
    "C134",
    "x[135]"
   ],
   [
    "C135",
    "x[136]"
   ],
   [
    "C136",
    "x[137]"
   ],
   [
    "C137",
    "x[138]"
   ],
   [
    "C138",
    "x[139]"
   ],
   [
    "C139",
    "x[140]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x[1]",
   "x[1]": "x[2]",
   "x[2]": "x[3]",
   "x[3]": "x[4]",
   "x[4]": "x[5]",
   "x[5]": "x[6]",
   "x[6]": "x[7]",
   "x[7]": "x[8]",
   "x[8]": "x[9]",
   "x[9]": "x[10]",
   "x[10]": "x[11]",
   "x[11]": "x[12]",
   "x[12]": "x[13]",
   "x[13]": "x[14]",
   "x[14]": "x[15]",
   "x[15]": "x[16]",
   "x[16]": "x[17]",
   "x[17]": "x[18]",
   "x[18]": "x[19]",
   "x[19]": "x[20]",
   "x[20]": "x[21]",
   "x[21]": "x[22]",
   "x[22]": "x[23]",
   "x[23]": "x[24]",
   "x[24]": "x[25]",
   "x[25]": "x[26]",
   "x[26]": "x[27]",
   "x[27]": "x[28]",
   "x[28]": "x[29]",
   "x[29]": "x[30]",
   "x[30]": "x[31]",
   "x[31]": "x[32]",
   "x[32]": "x[33]",
   "x[33]": "x[34]",
   "x[34]": "x[35]",
   "x[35]": "x[36]",
   "x[36]": "x[37]",
   "x[37]": "x[38]",
   "x[38]": "x[39]",
   "x[39]": "x[40]",
   "x[40]": "x[41]",
   "x[41]": "x[42]",
   "x[42]": "x[43]",
   "x[43]": "x[44]",
   "x[44]": "x[45]",
   "x[45]": "x[46]",
   "x[46]": "x[47]",
   "x[47]": "x[48]",
   "x[48]": "x[49]",
   "x[49]": "x[50]",
   "x[50]": "x[51]",
   "x[51]": "x[52]",
   "x[52]": "x[53]",
   "x[53]": "x[54]",
   "x[54]": "x[55]",
   "x[55]": "x[56]",
   "x[56]": "x[57]",
   "x[57]": "x[58]",
   "x[58]": "x[59]",
   "x[59]": "x[60]",
   "x[60]": "x[61]",
   "x[61]": "x[62]",
   "x[62]": "x[63]",
   "x[63]": "x[64]",
   "x[64]": "x[65]",
   "x[65]": "x[66]",
   "x[66]": "x[67]",
   "x[67]": "x[68]",
   "x[68]": "x[69]",
   "x[69]": "x[70]",
   "x[70]": "x[71]",
   "x[71]": "x[72]",
   "x[72]": "x[73]",
   "x[73]": "x[74]",
   "x[74]": "x[75]",
   "x[75]": "x[76]",
   "x[76]": "x[77]",
   "x[77]": "x[78]",
   "x[78]": "x[79]",
   "x[79]": "x[80]",
   "x[80]": "x[81]",
   "x[81]": "x[82]",
   "x[82]": "x[83]",
   "x[83]": "x[84]",
   "x[84]": "x[85]",
   "x[85]": "x[86]",
   "x[86]": "x[87]",
   "x[87]": "x[88]",
   "x[88]": "x[89]",
   "x[89]": "x[90]",
   "x[90]": "x[91]",
   "x[91]": "x[92]",
   "x[92]": "x[93]",
   "x[93]": "x[94]",
   "x[94]": "x[95]",
   "x[95]": "x[96]",
   "x[96]": "x[97]",
   "x[97]": "x[98]",
   "x[98]": "x[99]",
   "x[99]": "x[100]",
   "x[100]": "x[101]",
   "x[101]": "x[102]",
   "x[102]": "x[103]",
   "x[103]": "x[104]",
   "x[104]": "x[105]",
   "x[105]": "x[106]",
   "x[106]": "x[107]",
   "x[107]": "x[108]",
   "x[108]": "x[109]",
   "x[109]": "x[110]",
   "x[110]": "x[111]",
   "x[111]": "x[112]",
   "x[112]": "x[113]",
   "x[113]": "x[114]",
   "x[114]": "x[115]",
   "x[115]": "x[116]",
   "x[116]": "x[117]",
   "x[117]": "x[118]",
   "x[118]": "x[119]",
   "x[119]": "x[120]",
   "x[120]": "x[121]",
   "x[121]": "x[122]",
   "x[122]": "x[123]",
   "x[123]": "x[124]",
   "x[124]": "x[125]",
   "x[125]": "x[126]",
   "x[126]": "x[127]",
   "x[127]": "x[128]",
   "x[128]": "x[129]",
   "x[129]": "x[130]",
   "x[130]": "x[131]",
   "x[131]": "x[132]",
   "x[132]": "x[133]",
   "x[133]": "x[134]",
   "x[134]": "x[135]",
   "x[135]": "x[136]",
   "x[136]": "x[137]",
   "x[137]": "x[138]",
   "x[138]": "x[139]",
   "x[139]": "x[140]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 133.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     1.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     1.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     1.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-081": {
  "source_code_sha256": "c932f116420791dd1828e9d58b0f7a47fb0bc039d0b2cb499f8f3d4e37d58eba",
  "original_solution_sha256": "6df25d52e2f24f53e7a6257d6373bb319f484f03152d893005198917bacf0519",
  "bindings": [
   [
    "C0",
    "x[Fruit_010]"
   ],
   [
    "C1",
    "x[Dairy_003]"
   ],
   [
    "C2",
    "x[Fruit_025]"
   ],
   [
    "C3",
    "x[Grain_015]"
   ],
   [
    "C4",
    "x[Grain_018]"
   ],
   [
    "C5",
    "x[Grain_021]"
   ],
   [
    "C6",
    "x[Fruit_024]"
   ],
   [
    "C7",
    "x[Fruit_001]"
   ],
   [
    "C8",
    "x[Veg_010]"
   ],
   [
    "C9",
    "x[Dairy_017]"
   ],
   [
    "C10",
    "x[Dairy_001]"
   ],
   [
    "C11",
    "x[Grain_003]"
   ],
   [
    "C12",
    "x[Grain_016]"
   ],
   [
    "C13",
    "x[Veg_015]"
   ],
   [
    "C14",
    "x[Meat_002]"
   ],
   [
    "C15",
    "x[Dairy_011]"
   ],
   [
    "C16",
    "x[Meat_013]"
   ],
   [
    "C17",
    "x[Dairy_022]"
   ],
   [
    "C18",
    "x[Meat_004]"
   ],
   [
    "C19",
    "x[Veg_012]"
   ],
   [
    "C20",
    "x[Dairy_012]"
   ],
   [
    "C21",
    "x[Meat_012]"
   ],
   [
    "C22",
    "x[Dairy_002]"
   ],
   [
    "C23",
    "x[Veg_005]"
   ],
   [
    "C24",
    "x[Grain_024]"
   ],
   [
    "C25",
    "x[Dairy_019]"
   ],
   [
    "C26",
    "x[Dairy_004]"
   ],
   [
    "C27",
    "x[Veg_016]"
   ],
   [
    "C28",
    "x[Grain_005]"
   ],
   [
    "C29",
    "x[Grain_020]"
   ],
   [
    "C30",
    "x[Meat_022]"
   ],
   [
    "C31",
    "x[Fruit_015]"
   ],
   [
    "C32",
    "x[Dairy_014]"
   ],
   [
    "C33",
    "x[Grain_010]"
   ],
   [
    "C34",
    "x[Meat_008]"
   ],
   [
    "C35",
    "x[Grain_007]"
   ],
   [
    "C36",
    "x[Dairy_021]"
   ],
   [
    "C37",
    "x[Veg_013]"
   ],
   [
    "C38",
    "x[Grain_011]"
   ],
   [
    "C39",
    "x[Fruit_006]"
   ],
   [
    "C40",
    "x[Grain_012]"
   ],
   [
    "C41",
    "x[Dairy_013]"
   ],
   [
    "C42",
    "x[Grain_006]"
   ],
   [
    "C43",
    "x[Dairy_023]"
   ],
   [
    "C44",
    "x[Oatmeal]"
   ],
   [
    "C45",
    "x[Grain_001]"
   ],
   [
    "C46",
    "x[Grain_025]"
   ],
   [
    "C47",
    "x[Dairy_009]"
   ],
   [
    "C48",
    "x[Dairy_010]"
   ],
   [
    "C49",
    "x[Fruit_017]"
   ],
   [
    "C50",
    "x[Grain_009]"
   ],
   [
    "C51",
    "x[Meat_023]"
   ],
   [
    "C52",
    "x[Dairy_006]"
   ],
   [
    "C53",
    "x[Meat_016]"
   ],
   [
    "C54",
    "x[Dairy_007]"
   ],
   [
    "C55",
    "x[Grain_002]"
   ],
   [
    "C56",
    "x[Dairy_015]"
   ],
   [
    "C57",
    "x[Veg_001]"
   ],
   [
    "C58",
    "x[Fruit_022]"
   ],
   [
    "C59",
    "x[Veg_004]"
   ],
   [
    "C60",
    "x[Veg_007]"
   ],
   [
    "C61",
    "x[Grain_022]"
   ],
   [
    "C62",
    "x[Egg]"
   ],
   [
    "C63",
    "x[Fruit_020]"
   ],
   [
    "C64",
    "x[Fruit_013]"
   ],
   [
    "C65",
    "x[Veg_006]"
   ],
   [
    "C66",
    "x[Grain_019]"
   ],
   [
    "C67",
    "x[Fruit_007]"
   ],
   [
    "C68",
    "x[Fruit_016]"
   ],
   [
    "C69",
    "x[Fruit_023]"
   ],
   [
    "C70",
    "x[Fruit_011]"
   ],
   [
    "C71",
    "x[Dairy_016]"
   ],
   [
    "C72",
    "x[Dairy_020]"
   ],
   [
    "C73",
    "x[Fruit_002]"
   ],
   [
    "C74",
    "x[Fruit_018]"
   ],
   [
    "C75",
    "x[Meat_014]"
   ],
   [
    "C76",
    "x[Meat_011]"
   ],
   [
    "C77",
    "x[Fruit_014]"
   ],
   [
    "C78",
    "x[Meat_020]"
   ],
   [
    "C79",
    "x[Fruit_004]"
   ],
   [
    "C80",
    "x[Grain_017]"
   ],
   [
    "C81",
    "x[Meat_018]"
   ],
   [
    "C82",
    "x[Fruit_009]"
   ],
   [
    "C83",
    "x[Meat_005]"
   ],
   [
    "C84",
    "x[Veg_011]"
   ],
   [
    "C85",
    "x[Grain_004]"
   ],
   [
    "C86",
    "x[Grain_023]"
   ],
   [
    "C87",
    "x[Banana]"
   ],
   [
    "C88",
    "x[Meat_015]"
   ],
   [
    "C89",
    "x[Meat_024]"
   ],
   [
    "C90",
    "x[Veg_002]"
   ],
   [
    "C91",
    "x[Veg_003]"
   ],
   [
    "C92",
    "x[Fruit_005]"
   ],
   [
    "C93",
    "x[Dairy_018]"
   ],
   [
    "C94",
    "x[Meat_007]"
   ],
   [
    "C95",
    "x[Meat_025]"
   ],
   [
    "C96",
    "x[Fruit_021]"
   ],
   [
    "C97",
    "x[Grain_008]"
   ],
   [
    "C98",
    "x[Meat_019]"
   ],
   [
    "C99",
    "x[Meat_009]"
   ],
   [
    "C100",
    "x[Dairy_008]"
   ],
   [
    "C101",
    "x[Meat_021]"
   ],
   [
    "C102",
    "x[Fruit_008]"
   ],
   [
    "C103",
    "x[Veg_008]"
   ],
   [
    "C104",
    "x[Meat_017]"
   ],
   [
    "C105",
    "x[Veg_014]"
   ],
   [
    "C106",
    "x[Dairy_025]"
   ],
   [
    "C107",
    "x[Fruit_012]"
   ],
   [
    "C108",
    "x[Grain_013]"
   ],
   [
    "C109",
    "x[Dairy_005]"
   ],
   [
    "C110",
    "x[Meat_006]"
   ],
   [
    "C111",
    "x[Veg_009]"
   ],
   [
    "C112",
    "x[Meat_003]"
   ],
   [
    "C113",
    "x[Fruit_003]"
   ],
   [
    "C114",
    "x[Dairy_024]"
   ],
   [
    "C115",
    "x[Fruit_019]"
   ],
   [
    "C116",
    "x[Meat_010]"
   ],
   [
    "C117",
    "x[Grain_014]"
   ],
   [
    "C118",
    "x[Milk]"
   ],
   [
    "C119",
    "x[Meat_001]"
   ]
  ],
  "primary_mapping": {
   "x[Oatmeal]": "x[Oatmeal]",
   "x[Milk]": "x[Milk]",
   "x[Egg]": "x[Egg]",
   "x[Banana]": "x[Banana]",
   "x[Grain_001]": "x[Grain_001]",
   "x[Grain_002]": "x[Grain_002]",
   "x[Grain_003]": "x[Grain_003]",
   "x[Grain_004]": "x[Grain_004]",
   "x[Grain_005]": "x[Grain_005]",
   "x[Grain_006]": "x[Grain_006]",
   "x[Grain_007]": "x[Grain_007]",
   "x[Grain_008]": "x[Grain_008]",
   "x[Grain_009]": "x[Grain_009]",
   "x[Grain_010]": "x[Grain_010]",
   "x[Grain_011]": "x[Grain_011]",
   "x[Grain_012]": "x[Grain_012]",
   "x[Grain_013]": "x[Grain_013]",
   "x[Grain_014]": "x[Grain_014]",
   "x[Grain_015]": "x[Grain_015]",
   "x[Grain_016]": "x[Grain_016]",
   "x[Grain_017]": "x[Grain_017]",
   "x[Grain_018]": "x[Grain_018]",
   "x[Grain_019]": "x[Grain_019]",
   "x[Grain_020]": "x[Grain_020]",
   "x[Grain_021]": "x[Grain_021]",
   "x[Grain_022]": "x[Grain_022]",
   "x[Grain_023]": "x[Grain_023]",
   "x[Grain_024]": "x[Grain_024]",
   "x[Grain_025]": "x[Grain_025]",
   "x[Dairy_001]": "x[Dairy_001]",
   "x[Dairy_002]": "x[Dairy_002]",
   "x[Dairy_003]": "x[Dairy_003]",
   "x[Dairy_004]": "x[Dairy_004]",
   "x[Dairy_005]": "x[Dairy_005]",
   "x[Dairy_006]": "x[Dairy_006]",
   "x[Dairy_007]": "x[Dairy_007]",
   "x[Dairy_008]": "x[Dairy_008]",
   "x[Dairy_009]": "x[Dairy_009]",
   "x[Dairy_010]": "x[Dairy_010]",
   "x[Dairy_011]": "x[Dairy_011]",
   "x[Dairy_012]": "x[Dairy_012]",
   "x[Dairy_013]": "x[Dairy_013]",
   "x[Dairy_014]": "x[Dairy_014]",
   "x[Dairy_015]": "x[Dairy_015]",
   "x[Dairy_016]": "x[Dairy_016]",
   "x[Dairy_017]": "x[Dairy_017]",
   "x[Dairy_018]": "x[Dairy_018]",
   "x[Dairy_019]": "x[Dairy_019]",
   "x[Dairy_020]": "x[Dairy_020]",
   "x[Dairy_021]": "x[Dairy_021]",
   "x[Dairy_022]": "x[Dairy_022]",
   "x[Dairy_023]": "x[Dairy_023]",
   "x[Dairy_024]": "x[Dairy_024]",
   "x[Dairy_025]": "x[Dairy_025]",
   "x[Meat_001]": "x[Meat_001]",
   "x[Meat_002]": "x[Meat_002]",
   "x[Meat_003]": "x[Meat_003]",
   "x[Meat_004]": "x[Meat_004]",
   "x[Meat_005]": "x[Meat_005]",
   "x[Meat_006]": "x[Meat_006]",
   "x[Meat_007]": "x[Meat_007]",
   "x[Meat_008]": "x[Meat_008]",
   "x[Meat_009]": "x[Meat_009]",
   "x[Meat_010]": "x[Meat_010]",
   "x[Meat_011]": "x[Meat_011]",
   "x[Meat_012]": "x[Meat_012]",
   "x[Meat_013]": "x[Meat_013]",
   "x[Meat_014]": "x[Meat_014]",
   "x[Meat_015]": "x[Meat_015]",
   "x[Meat_016]": "x[Meat_016]",
   "x[Meat_017]": "x[Meat_017]",
   "x[Meat_018]": "x[Meat_018]",
   "x[Meat_019]": "x[Meat_019]",
   "x[Meat_020]": "x[Meat_020]",
   "x[Meat_021]": "x[Meat_021]",
   "x[Meat_022]": "x[Meat_022]",
   "x[Meat_023]": "x[Meat_023]",
   "x[Meat_024]": "x[Meat_024]",
   "x[Meat_025]": "x[Meat_025]",
   "x[Fruit_001]": "x[Fruit_001]",
   "x[Fruit_002]": "x[Fruit_002]",
   "x[Fruit_003]": "x[Fruit_003]",
   "x[Fruit_004]": "x[Fruit_004]",
   "x[Fruit_005]": "x[Fruit_005]",
   "x[Fruit_006]": "x[Fruit_006]",
   "x[Fruit_007]": "x[Fruit_007]",
   "x[Fruit_008]": "x[Fruit_008]",
   "x[Fruit_009]": "x[Fruit_009]",
   "x[Fruit_010]": "x[Fruit_010]",
   "x[Fruit_011]": "x[Fruit_011]",
   "x[Fruit_012]": "x[Fruit_012]",
   "x[Fruit_013]": "x[Fruit_013]",
   "x[Fruit_014]": "x[Fruit_014]",
   "x[Fruit_015]": "x[Fruit_015]",
   "x[Fruit_016]": "x[Fruit_016]",
   "x[Fruit_017]": "x[Fruit_017]",
   "x[Fruit_018]": "x[Fruit_018]",
   "x[Fruit_019]": "x[Fruit_019]",
   "x[Fruit_020]": "x[Fruit_020]",
   "x[Fruit_021]": "x[Fruit_021]",
   "x[Fruit_022]": "x[Fruit_022]",
   "x[Fruit_023]": "x[Fruit_023]",
   "x[Fruit_024]": "x[Fruit_024]",
   "x[Fruit_025]": "x[Fruit_025]",
   "x[Veg_001]": "x[Veg_001]",
   "x[Veg_002]": "x[Veg_002]",
   "x[Veg_003]": "x[Veg_003]",
   "x[Veg_004]": "x[Veg_004]",
   "x[Veg_005]": "x[Veg_005]",
   "x[Veg_006]": "x[Veg_006]",
   "x[Veg_007]": "x[Veg_007]",
   "x[Veg_008]": "x[Veg_008]",
   "x[Veg_009]": "x[Veg_009]",
   "x[Veg_010]": "x[Veg_010]",
   "x[Veg_011]": "x[Veg_011]",
   "x[Veg_012]": "x[Veg_012]",
   "x[Veg_013]": "x[Veg_013]",
   "x[Veg_014]": "x[Veg_014]",
   "x[Veg_015]": "x[Veg_015]",
   "x[Veg_016]": "x[Veg_016]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 3.910164341169253,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     7.56037383023496
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     2.6661428458718643
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.9202828750048574
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-086": {
  "source_code_sha256": "3bd0a092356e1b8a47ce9af719dfebb775e570d68af7f84ff90154eecceaa0e5",
  "original_solution_sha256": "1c79182d3b44963dfd68016d5e5d49fcd4be27927f51d1c50c2118f6d535a869",
  "bindings": [
   [
    "C0",
    "x[I,Red]"
   ],
   [
    "C1",
    "x[I,Yellow]"
   ],
   [
    "C2",
    "x[I,Blue]"
   ],
   [
    "C3",
    "x[II,Red]"
   ],
   [
    "C4",
    "x[II,Yellow]"
   ],
   [
    "C5",
    "x[II,Blue]"
   ],
   [
    "C6",
    "x[III,Red]"
   ],
   [
    "C7",
    "x[III,Yellow]"
   ],
   [
    "C8",
    "x[III,Blue]"
   ],
   [
    "C9",
    "y[Red]"
   ],
   [
    "C10",
    "y[Yellow]"
   ],
   [
    "C11",
    "y[Blue]"
   ]
  ],
  "primary_mapping": {
   "x[Red,I]": "x[I,Red]",
   "x[Red,II]": "x[II,Red]",
   "x[Red,III]": "x[III,Red]",
   "x[Yellow,I]": "x[I,Yellow]",
   "x[Yellow,II]": "x[II,Yellow]",
   "x[Yellow,III]": "x[III,Yellow]",
   "x[Blue,I]": "x[I,Blue]",
   "x[Blue,II]": "x[II,Blue]",
   "x[Blue,III]": "x[III,Blue]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 4500.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     2000.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     1000.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     3000.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-091": {
  "source_code_sha256": "cc042bc7bbd7c3ef36ac0a9c504efbafe51c2f8581c5ef703bf8dc31f0fe1860",
  "original_solution_sha256": "847ffb2527b4faca7883c5666cd04a909f8149f4045b7f83785cc75113ee682c",
  "bindings": [
   [
    "C0",
    "x[C22]"
   ],
   [
    "C1",
    "x[C23]"
   ],
   [
    "C2",
    "x[C24]"
   ],
   [
    "C3",
    "x[C25]"
   ],
   [
    "C4",
    "x[C26]"
   ],
   [
    "C5",
    "x[C27]"
   ],
   [
    "C6",
    "x[C28]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x[C22]",
   "x[2]": "x[C23]",
   "x[3]": "x[C24]",
   "x[4]": "x[C25]",
   "x[5]": "x[C26]",
   "x[6]": "x[C27]",
   "x[7]": "x[C28]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 421.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     1.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     1.0
    ]
   ],
   "error": null
  }
 },
 "OR-094": {
  "source_code_sha256": "f6e5c4936ce8c769698bd42bfa6a2c00785f06898af68dc8189889e48e287782",
  "original_solution_sha256": "e4a2f837ca81fa43d4f0bb1a00907524191f0195f8d4406adf8d07fe3afb35e9",
  "bindings": [
   [
    "C0",
    "x[1]"
   ],
   [
    "C1",
    "x[2]"
   ],
   [
    "C2",
    "x[3]"
   ],
   [
    "C3",
    "x[4]"
   ],
   [
    "C4",
    "x[5]"
   ],
   [
    "C5",
    "x[6]"
   ],
   [
    "C6",
    "x[7]"
   ],
   [
    "C7",
    "x[8]"
   ],
   [
    "C8",
    "x[9]"
   ],
   [
    "C9",
    "x[10]"
   ],
   [
    "C10",
    "x[11]"
   ],
   [
    "C11",
    "x[12]"
   ],
   [
    "C12",
    "x[13]"
   ],
   [
    "C13",
    "x[14]"
   ],
   [
    "C14",
    "x[15]"
   ],
   [
    "C15",
    "x[16]"
   ],
   [
    "C16",
    "x[17]"
   ],
   [
    "C17",
    "x[18]"
   ],
   [
    "C18",
    "x[19]"
   ],
   [
    "C19",
    "x[20]"
   ],
   [
    "C20",
    "x[21]"
   ],
   [
    "C21",
    "x[22]"
   ],
   [
    "C22",
    "x[23]"
   ],
   [
    "C23",
    "x[24]"
   ],
   [
    "C24",
    "x[25]"
   ],
   [
    "C25",
    "x[26]"
   ],
   [
    "C26",
    "x[27]"
   ],
   [
    "C27",
    "x[28]"
   ],
   [
    "C28",
    "x[29]"
   ],
   [
    "C29",
    "x[30]"
   ],
   [
    "C30",
    "x[31]"
   ],
   [
    "C31",
    "x[32]"
   ],
   [
    "C32",
    "x[33]"
   ],
   [
    "C33",
    "x[34]"
   ],
   [
    "C34",
    "x[35]"
   ],
   [
    "C35",
    "x[36]"
   ],
   [
    "C36",
    "x[37]"
   ],
   [
    "C37",
    "x[38]"
   ],
   [
    "C38",
    "x[39]"
   ],
   [
    "C39",
    "x[40]"
   ],
   [
    "C40",
    "x[41]"
   ],
   [
    "C41",
    "x[42]"
   ],
   [
    "C42",
    "x[43]"
   ],
   [
    "C43",
    "x[44]"
   ],
   [
    "C44",
    "x[45]"
   ],
   [
    "C45",
    "x[46]"
   ],
   [
    "C46",
    "x[47]"
   ],
   [
    "C47",
    "x[48]"
   ],
   [
    "C48",
    "x[49]"
   ],
   [
    "C49",
    "x[50]"
   ],
   [
    "C50",
    "x[51]"
   ],
   [
    "C51",
    "x[52]"
   ],
   [
    "C52",
    "x[53]"
   ],
   [
    "C53",
    "x[54]"
   ],
   [
    "C54",
    "x[55]"
   ],
   [
    "C55",
    "x[56]"
   ],
   [
    "C56",
    "x[57]"
   ],
   [
    "C57",
    "x[58]"
   ],
   [
    "C58",
    "x[59]"
   ],
   [
    "C59",
    "x[60]"
   ],
   [
    "C60",
    "x[61]"
   ],
   [
    "C61",
    "x[62]"
   ],
   [
    "C62",
    "x[63]"
   ],
   [
    "C63",
    "x[64]"
   ],
   [
    "C64",
    "x[65]"
   ],
   [
    "C65",
    "x[66]"
   ],
   [
    "C66",
    "x[67]"
   ],
   [
    "C67",
    "x[68]"
   ],
   [
    "C68",
    "x[69]"
   ],
   [
    "C69",
    "x[70]"
   ],
   [
    "C70",
    "x[71]"
   ],
   [
    "C71",
    "x[72]"
   ],
   [
    "C72",
    "x[73]"
   ],
   [
    "C73",
    "x[74]"
   ],
   [
    "C74",
    "x[75]"
   ],
   [
    "C75",
    "x[76]"
   ],
   [
    "C76",
    "x[77]"
   ],
   [
    "C77",
    "x[78]"
   ],
   [
    "C78",
    "x[79]"
   ],
   [
    "C79",
    "x[80]"
   ],
   [
    "C80",
    "x[81]"
   ],
   [
    "C81",
    "x[82]"
   ],
   [
    "C82",
    "x[83]"
   ],
   [
    "C83",
    "x[84]"
   ],
   [
    "C84",
    "x[85]"
   ],
   [
    "C85",
    "x[86]"
   ],
   [
    "C86",
    "x[87]"
   ],
   [
    "C87",
    "x[88]"
   ],
   [
    "C88",
    "x[89]"
   ],
   [
    "C89",
    "x[90]"
   ],
   [
    "C90",
    "x[91]"
   ],
   [
    "C91",
    "x[92]"
   ],
   [
    "C92",
    "x[93]"
   ],
   [
    "C93",
    "x[94]"
   ],
   [
    "C94",
    "x[95]"
   ],
   [
    "C95",
    "x[96]"
   ],
   [
    "C96",
    "x[97]"
   ],
   [
    "C97",
    "x[98]"
   ],
   [
    "C98",
    "x[99]"
   ],
   [
    "C99",
    "x[100]"
   ],
   [
    "C100",
    "x[101]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x[1]",
   "x[1]": "x[2]",
   "x[2]": "x[3]",
   "x[3]": "x[4]",
   "x[4]": "x[5]",
   "x[5]": "x[6]",
   "x[6]": "x[7]",
   "x[7]": "x[8]",
   "x[8]": "x[9]",
   "x[9]": "x[10]",
   "x[10]": "x[11]",
   "x[11]": "x[12]",
   "x[12]": "x[13]",
   "x[13]": "x[14]",
   "x[14]": "x[15]",
   "x[15]": "x[16]",
   "x[16]": "x[17]",
   "x[17]": "x[18]",
   "x[18]": "x[19]",
   "x[19]": "x[20]",
   "x[20]": "x[21]",
   "x[21]": "x[22]",
   "x[22]": "x[23]",
   "x[23]": "x[24]",
   "x[24]": "x[25]",
   "x[25]": "x[26]",
   "x[26]": "x[27]",
   "x[27]": "x[28]",
   "x[28]": "x[29]",
   "x[29]": "x[30]",
   "x[30]": "x[31]",
   "x[31]": "x[32]",
   "x[32]": "x[33]",
   "x[33]": "x[34]",
   "x[34]": "x[35]",
   "x[35]": "x[36]",
   "x[36]": "x[37]",
   "x[37]": "x[38]",
   "x[38]": "x[39]",
   "x[39]": "x[40]",
   "x[40]": "x[41]",
   "x[41]": "x[42]",
   "x[42]": "x[43]",
   "x[43]": "x[44]",
   "x[44]": "x[45]",
   "x[45]": "x[46]",
   "x[46]": "x[47]",
   "x[47]": "x[48]",
   "x[48]": "x[49]",
   "x[49]": "x[50]",
   "x[50]": "x[51]",
   "x[51]": "x[52]",
   "x[52]": "x[53]",
   "x[53]": "x[54]",
   "x[54]": "x[55]",
   "x[55]": "x[56]",
   "x[56]": "x[57]",
   "x[57]": "x[58]",
   "x[58]": "x[59]",
   "x[59]": "x[60]",
   "x[60]": "x[61]",
   "x[61]": "x[62]",
   "x[62]": "x[63]",
   "x[63]": "x[64]",
   "x[64]": "x[65]",
   "x[65]": "x[66]",
   "x[66]": "x[67]",
   "x[67]": "x[68]",
   "x[68]": "x[69]",
   "x[69]": "x[70]",
   "x[70]": "x[71]",
   "x[71]": "x[72]",
   "x[72]": "x[73]",
   "x[73]": "x[74]",
   "x[74]": "x[75]",
   "x[75]": "x[76]",
   "x[76]": "x[77]",
   "x[77]": "x[78]",
   "x[78]": "x[79]",
   "x[79]": "x[80]",
   "x[80]": "x[81]",
   "x[81]": "x[82]",
   "x[82]": "x[83]",
   "x[83]": "x[84]",
   "x[84]": "x[85]",
   "x[85]": "x[86]",
   "x[86]": "x[87]",
   "x[87]": "x[88]",
   "x[88]": "x[89]",
   "x[89]": "x[90]",
   "x[90]": "x[91]",
   "x[91]": "x[92]",
   "x[92]": "x[93]",
   "x[93]": "x[94]",
   "x[94]": "x[95]",
   "x[95]": "x[96]",
   "x[96]": "x[97]",
   "x[97]": "x[98]",
   "x[98]": "x[99]",
   "x[99]": "x[100]",
   "x[100]": "x[101]"
  },
  "mapping_basis": "reviewed_oss_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 0.6000000000003638,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     49.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     2.0
    ],
    [
     "C94",
     74.0
    ],
    [
     "C95",
     24.0
    ],
    [
     "C96",
     12.0
    ],
    [
     "C97",
     36.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     14.0
    ],
    [
     "C100",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-098": {
  "source_code_sha256": "aa31f974efb7e852cebcd0a7d695279b163bd65f808e3614e0c5fb2c17f16f97",
  "original_solution_sha256": "97e0588230297ff82261ee73c7d36cba47ae020a4b2b632fc2166f2741ad9cc3",
  "bindings": [
   [
    "C0",
    "w[Carpenter]"
   ],
   [
    "C1",
    "w[Electrician]"
   ],
   [
    "C2",
    "w[Painter]"
   ],
   [
    "C3",
    "w[Worker_004]"
   ],
   [
    "C4",
    "w[Worker_005]"
   ],
   [
    "C5",
    "w[Worker_006]"
   ],
   [
    "C6",
    "w[Worker_007]"
   ],
   [
    "C7",
    "w[Worker_008]"
   ],
   [
    "C8",
    "w[Worker_009]"
   ],
   [
    "C9",
    "w[Worker_010]"
   ],
   [
    "C10",
    "w[Worker_011]"
   ],
   [
    "C11",
    "w[Worker_012]"
   ],
   [
    "C12",
    "w[Worker_013]"
   ],
   [
    "C13",
    "w[Worker_014]"
   ],
   [
    "C14",
    "w[Worker_015]"
   ],
   [
    "C15",
    "w[Worker_016]"
   ],
   [
    "C16",
    "w[Worker_017]"
   ],
   [
    "C17",
    "w[Worker_018]"
   ],
   [
    "C18",
    "w[Worker_019]"
   ],
   [
    "C19",
    "w[Worker_020]"
   ],
   [
    "C20",
    "w[Worker_021]"
   ],
   [
    "C21",
    "w[Worker_022]"
   ],
   [
    "C22",
    "w[Worker_023]"
   ],
   [
    "C23",
    "w[Worker_024]"
   ],
   [
    "C24",
    "w[Worker_025]"
   ],
   [
    "C25",
    "w[Worker_026]"
   ],
   [
    "C26",
    "w[Worker_027]"
   ],
   [
    "C27",
    "w[Worker_028]"
   ],
   [
    "C28",
    "w[Worker_029]"
   ],
   [
    "C29",
    "w[Worker_030]"
   ],
   [
    "C30",
    "w[Worker_031]"
   ],
   [
    "C31",
    "w[Worker_032]"
   ],
   [
    "C32",
    "w[Worker_033]"
   ],
   [
    "C33",
    "w[Worker_034]"
   ],
   [
    "C34",
    "w[Worker_035]"
   ],
   [
    "C35",
    "w[Worker_036]"
   ],
   [
    "C36",
    "w[Worker_037]"
   ],
   [
    "C37",
    "w[Worker_038]"
   ],
   [
    "C38",
    "w[Worker_039]"
   ],
   [
    "C39",
    "w[Worker_040]"
   ],
   [
    "C40",
    "w[Worker_041]"
   ],
   [
    "C41",
    "w[Worker_042]"
   ],
   [
    "C42",
    "w[Worker_043]"
   ],
   [
    "C43",
    "w[Worker_044]"
   ],
   [
    "C44",
    "w[Worker_045]"
   ],
   [
    "C45",
    "w[Worker_046]"
   ],
   [
    "C46",
    "w[Worker_047]"
   ],
   [
    "C47",
    "w[Worker_048]"
   ],
   [
    "C48",
    "w[Worker_049]"
   ],
   [
    "C49",
    "w[Worker_050]"
   ],
   [
    "C50",
    "w[Worker_051]"
   ],
   [
    "C51",
    "w[Worker_052]"
   ],
   [
    "C52",
    "w[Worker_053]"
   ],
   [
    "C53",
    "w[Worker_054]"
   ],
   [
    "C54",
    "w[Worker_055]"
   ],
   [
    "C55",
    "w[Worker_056]"
   ],
   [
    "C56",
    "w[Worker_057]"
   ],
   [
    "C57",
    "w[Worker_058]"
   ],
   [
    "C58",
    "w[Worker_059]"
   ],
   [
    "C59",
    "w[Worker_060]"
   ],
   [
    "C60",
    "w[Worker_061]"
   ],
   [
    "C61",
    "w[Worker_062]"
   ],
   [
    "C62",
    "w[Worker_063]"
   ],
   [
    "C63",
    "w[Worker_064]"
   ],
   [
    "C64",
    "w[Worker_065]"
   ],
   [
    "C65",
    "w[Worker_066]"
   ],
   [
    "C66",
    "w[Worker_067]"
   ],
   [
    "C67",
    "w[Worker_068]"
   ],
   [
    "C68",
    "w[Worker_069]"
   ],
   [
    "C69",
    "w[Worker_070]"
   ],
   [
    "C70",
    "w[Worker_071]"
   ],
   [
    "C71",
    "w[Worker_072]"
   ],
   [
    "C72",
    "w[Worker_073]"
   ],
   [
    "C73",
    "w[Worker_074]"
   ],
   [
    "C74",
    "w[Worker_075]"
   ],
   [
    "C75",
    "w[Worker_076]"
   ],
   [
    "C76",
    "w[Worker_077]"
   ],
   [
    "C77",
    "w[Worker_078]"
   ],
   [
    "C78",
    "w[Worker_079]"
   ],
   [
    "C79",
    "w[Worker_080]"
   ],
   [
    "C80",
    "w[Worker_081]"
   ],
   [
    "C81",
    "w[Worker_082]"
   ],
   [
    "C82",
    "w[Worker_083]"
   ],
   [
    "C83",
    "w[Worker_084]"
   ],
   [
    "C84",
    "w[Worker_085]"
   ],
   [
    "C85",
    "w[Worker_086]"
   ],
   [
    "C86",
    "w[Worker_087]"
   ],
   [
    "C87",
    "w[Worker_088]"
   ],
   [
    "C88",
    "w[Worker_089]"
   ],
   [
    "C89",
    "w[Worker_090]"
   ],
   [
    "C90",
    "w[Worker_091]"
   ],
   [
    "C91",
    "w[Worker_092]"
   ],
   [
    "C92",
    "w[Worker_093]"
   ],
   [
    "C93",
    "w[Worker_094]"
   ],
   [
    "C94",
    "w[Worker_095]"
   ],
   [
    "C95",
    "w[Worker_096]"
   ],
   [
    "C96",
    "w[Worker_097]"
   ],
   [
    "C97",
    "w[Worker_098]"
   ],
   [
    "C98",
    "w[Worker_099]"
   ],
   [
    "C99",
    "w[Worker_100]"
   ],
   [
    "C100",
    "w[Worker_101]"
   ],
   [
    "C101",
    "w[Worker_102]"
   ],
   [
    "C102",
    "w[Worker_103]"
   ],
   [
    "C103",
    "w[Worker_104]"
   ],
   [
    "C104",
    "w[Worker_105]"
   ],
   [
    "C105",
    "w[Worker_106]"
   ],
   [
    "C106",
    "w[Worker_107]"
   ],
   [
    "C107",
    "w[Worker_108]"
   ],
   [
    "C108",
    "w[Worker_109]"
   ],
   [
    "C109",
    "w[Worker_110]"
   ],
   [
    "C110",
    "w[Worker_111]"
   ],
   [
    "C111",
    "w[Worker_112]"
   ],
   [
    "C112",
    "w[Worker_113]"
   ],
   [
    "C113",
    "w[Worker_114]"
   ],
   [
    "C114",
    "w[Worker_115]"
   ],
   [
    "C115",
    "w[Worker_116]"
   ],
   [
    "C116",
    "w[Worker_117]"
   ],
   [
    "C117",
    "w[Worker_118]"
   ],
   [
    "C118",
    "w[Worker_119]"
   ],
   [
    "C119",
    "w[Worker_120]"
   ],
   [
    "C120",
    "w[Worker_121]"
   ],
   [
    "C121",
    "w[Worker_122]"
   ],
   [
    "C122",
    "w[Worker_123]"
   ],
   [
    "C123",
    "w[Worker_124]"
   ],
   [
    "C124",
    "w[Worker_125]"
   ],
   [
    "C125",
    "w[Worker_126]"
   ],
   [
    "C126",
    "w[Worker_127]"
   ],
   [
    "C127",
    "w[Worker_128]"
   ],
   [
    "C128",
    "w[Worker_129]"
   ],
   [
    "C129",
    "w[Worker_130]"
   ],
   [
    "C130",
    "w[Worker_131]"
   ],
   [
    "C131",
    "w[Worker_132]"
   ],
   [
    "C132",
    "w[Worker_133]"
   ],
   [
    "C133",
    "w[Worker_134]"
   ],
   [
    "C134",
    "w[Worker_135]"
   ],
   [
    "C135",
    "w[Worker_136]"
   ],
   [
    "C136",
    "w[Worker_137]"
   ],
   [
    "C137",
    "w[Worker_138]"
   ],
   [
    "C138",
    "w[Worker_139]"
   ],
   [
    "C139",
    "w[Worker_140]"
   ],
   [
    "C140",
    "w[Worker_141]"
   ],
   [
    "C141",
    "w[Worker_142]"
   ],
   [
    "C142",
    "w[Worker_143]"
   ],
   [
    "C143",
    "w[Worker_144]"
   ],
   [
    "C144",
    "w[Worker_145]"
   ],
   [
    "C145",
    "w[Worker_146]"
   ],
   [
    "C146",
    "w[Worker_147]"
   ],
   [
    "C147",
    "w[Worker_148]"
   ],
   [
    "C148",
    "w[Worker_149]"
   ],
   [
    "C149",
    "w[Worker_150]"
   ]
  ],
  "primary_mapping": {
   "wage[0]": "w[Carpenter]",
   "wage[1]": "w[Electrician]",
   "wage[2]": "w[Painter]",
   "wage[3]": "w[Worker_004]",
   "wage[4]": "w[Worker_005]",
   "wage[5]": "w[Worker_006]",
   "wage[6]": "w[Worker_007]",
   "wage[7]": "w[Worker_008]",
   "wage[8]": "w[Worker_009]",
   "wage[9]": "w[Worker_010]",
   "wage[10]": "w[Worker_011]",
   "wage[11]": "w[Worker_012]",
   "wage[12]": "w[Worker_013]",
   "wage[13]": "w[Worker_014]",
   "wage[14]": "w[Worker_015]",
   "wage[15]": "w[Worker_016]",
   "wage[16]": "w[Worker_017]",
   "wage[17]": "w[Worker_018]",
   "wage[18]": "w[Worker_019]",
   "wage[19]": "w[Worker_020]",
   "wage[20]": "w[Worker_021]",
   "wage[21]": "w[Worker_022]",
   "wage[22]": "w[Worker_023]",
   "wage[23]": "w[Worker_024]",
   "wage[24]": "w[Worker_025]",
   "wage[25]": "w[Worker_026]",
   "wage[26]": "w[Worker_027]",
   "wage[27]": "w[Worker_028]",
   "wage[28]": "w[Worker_029]",
   "wage[29]": "w[Worker_030]",
   "wage[30]": "w[Worker_031]",
   "wage[31]": "w[Worker_032]",
   "wage[32]": "w[Worker_033]",
   "wage[33]": "w[Worker_034]",
   "wage[34]": "w[Worker_035]",
   "wage[35]": "w[Worker_036]",
   "wage[36]": "w[Worker_037]",
   "wage[37]": "w[Worker_038]",
   "wage[38]": "w[Worker_039]",
   "wage[39]": "w[Worker_040]",
   "wage[40]": "w[Worker_041]",
   "wage[41]": "w[Worker_042]",
   "wage[42]": "w[Worker_043]",
   "wage[43]": "w[Worker_044]",
   "wage[44]": "w[Worker_045]",
   "wage[45]": "w[Worker_046]",
   "wage[46]": "w[Worker_047]",
   "wage[47]": "w[Worker_048]",
   "wage[48]": "w[Worker_049]",
   "wage[49]": "w[Worker_050]",
   "wage[50]": "w[Worker_051]",
   "wage[51]": "w[Worker_052]",
   "wage[52]": "w[Worker_053]",
   "wage[53]": "w[Worker_054]",
   "wage[54]": "w[Worker_055]",
   "wage[55]": "w[Worker_056]",
   "wage[56]": "w[Worker_057]",
   "wage[57]": "w[Worker_058]",
   "wage[58]": "w[Worker_059]",
   "wage[59]": "w[Worker_060]",
   "wage[60]": "w[Worker_061]",
   "wage[61]": "w[Worker_062]",
   "wage[62]": "w[Worker_063]",
   "wage[63]": "w[Worker_064]",
   "wage[64]": "w[Worker_065]",
   "wage[65]": "w[Worker_066]",
   "wage[66]": "w[Worker_067]",
   "wage[67]": "w[Worker_068]",
   "wage[68]": "w[Worker_069]",
   "wage[69]": "w[Worker_070]",
   "wage[70]": "w[Worker_071]",
   "wage[71]": "w[Worker_072]",
   "wage[72]": "w[Worker_073]",
   "wage[73]": "w[Worker_074]",
   "wage[74]": "w[Worker_075]",
   "wage[75]": "w[Worker_076]",
   "wage[76]": "w[Worker_077]",
   "wage[77]": "w[Worker_078]",
   "wage[78]": "w[Worker_079]",
   "wage[79]": "w[Worker_080]",
   "wage[80]": "w[Worker_081]",
   "wage[81]": "w[Worker_082]",
   "wage[82]": "w[Worker_083]",
   "wage[83]": "w[Worker_084]",
   "wage[84]": "w[Worker_085]",
   "wage[85]": "w[Worker_086]",
   "wage[86]": "w[Worker_087]",
   "wage[87]": "w[Worker_088]",
   "wage[88]": "w[Worker_089]",
   "wage[89]": "w[Worker_090]",
   "wage[90]": "w[Worker_091]",
   "wage[91]": "w[Worker_092]",
   "wage[92]": "w[Worker_093]",
   "wage[93]": "w[Worker_094]",
   "wage[94]": "w[Worker_095]",
   "wage[95]": "w[Worker_096]",
   "wage[96]": "w[Worker_097]",
   "wage[97]": "w[Worker_098]",
   "wage[98]": "w[Worker_099]",
   "wage[99]": "w[Worker_100]",
   "wage[100]": "w[Worker_101]",
   "wage[101]": "w[Worker_102]",
   "wage[102]": "w[Worker_103]",
   "wage[103]": "w[Worker_104]",
   "wage[104]": "w[Worker_105]",
   "wage[105]": "w[Worker_106]",
   "wage[106]": "w[Worker_107]",
   "wage[107]": "w[Worker_108]",
   "wage[108]": "w[Worker_109]",
   "wage[109]": "w[Worker_110]",
   "wage[110]": "w[Worker_111]",
   "wage[111]": "w[Worker_112]",
   "wage[112]": "w[Worker_113]",
   "wage[113]": "w[Worker_114]",
   "wage[114]": "w[Worker_115]",
   "wage[115]": "w[Worker_116]",
   "wage[116]": "w[Worker_117]",
   "wage[117]": "w[Worker_118]",
   "wage[118]": "w[Worker_119]",
   "wage[119]": "w[Worker_120]",
   "wage[120]": "w[Worker_121]",
   "wage[121]": "w[Worker_122]",
   "wage[122]": "w[Worker_123]",
   "wage[123]": "w[Worker_124]",
   "wage[124]": "w[Worker_125]",
   "wage[125]": "w[Worker_126]",
   "wage[126]": "w[Worker_127]",
   "wage[127]": "w[Worker_128]",
   "wage[128]": "w[Worker_129]",
   "wage[129]": "w[Worker_130]",
   "wage[130]": "w[Worker_131]",
   "wage[131]": "w[Worker_132]",
   "wage[132]": "w[Worker_133]",
   "wage[133]": "w[Worker_134]",
   "wage[134]": "w[Worker_135]",
   "wage[135]": "w[Worker_136]",
   "wage[136]": "w[Worker_137]",
   "wage[137]": "w[Worker_138]",
   "wage[138]": "w[Worker_139]",
   "wage[139]": "w[Worker_140]",
   "wage[140]": "w[Worker_141]",
   "wage[141]": "w[Worker_142]",
   "wage[142]": "w[Worker_143]",
   "wage[143]": "w[Worker_144]",
   "wage[144]": "w[Worker_145]",
   "wage[145]": "w[Worker_146]",
   "wage[146]": "w[Worker_147]",
   "wage[147]": "w[Worker_148]",
   "wage[148]": "w[Worker_149]",
   "wage[149]": "w[Worker_150]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 0.0,
   "variables": [
    [
     "C0",
     60.0
    ],
    [
     "C1",
     34.19783533288588
    ],
    [
     "C2",
     28.534300036944465
    ],
    [
     "C3",
     39.43627035168434
    ],
    [
     "C4",
     56.75567673995285
    ],
    [
     "C5",
     31.998885828642063
    ],
    [
     "C6",
     31.955736412446925
    ],
    [
     "C7",
     73.78194847084845
    ],
    [
     "C8",
     43.86892720033491
    ],
    [
     "C9",
     28.788792072882423
    ],
    [
     "C10",
     38.058028706561856
    ],
    [
     "C11",
     64.5738697309248
    ],
    [
     "C12",
     37.01198733022927
    ],
    [
     "C13",
     53.07821941058333
    ],
    [
     "C14",
     52.96851343778717
    ],
    [
     "C15",
     28.502960334803678
    ],
    [
     "C16",
     49.5520550299615
    ],
    [
     "C17",
     23.9711927080494
    ],
    [
     "C18",
     29.825391973044056
    ],
    [
     "C19",
     22.67931116944854
    ],
    [
     "C20",
     43.506964012747225
    ],
    [
     "C21",
     30.542595414348774
    ],
    [
     "C22",
     26.647419742019224
    ],
    [
     "C23",
     34.03186495890657
    ],
    [
     "C24",
     51.86586255701261
    ],
    [
     "C25",
     54.5656852802838
    ],
    [
     "C26",
     44.08301520234885
    ],
    [
     "C27",
     51.06452022679789
    ],
    [
     "C28",
     56.31498877949078
    ],
    [
     "C29",
     42.36584098387713
    ],
    [
     "C30",
     17.605080326492374
    ],
    [
     "C31",
     44.19640727816731
    ],
    [
     "C32",
     58.03320976288379
    ],
    [
     "C33",
     27.28727556530352
    ],
    [
     "C34",
     49.193033761666264
    ],
    [
     "C35",
     50.56840432434067
    ],
    [
     "C36",
     45.796689644461196
    ],
    [
     "C37",
     62.27502057491782
    ],
    [
     "C38",
     65.7393497791183
    ],
    [
     "C39",
     43.29295108853586
    ],
    [
     "C40",
     22.732654434909755
    ],
    [
     "C41",
     45.12051943339643
    ],
    [
     "C42",
     25.011288471699984
    ],
    [
     "C43",
     44.73279017811648
    ],
    [
     "C44",
     17.209910491167065
    ],
    [
     "C45",
     34.36620727404207
    ],
    [
     "C46",
     71.70581538719192
    ],
    [
     "C47",
     52.318488223658555
    ],
    [
     "C48",
     43.09568674200238
    ],
    [
     "C49",
     21.089859672890515
    ],
    [
     "C50",
     54.69406374642706
    ],
    [
     "C51",
     32.830599506493066
    ],
    [
     "C52",
     49.80889743909683
    ],
    [
     "C53",
     72.93787319189371
    ],
    [
     "C54",
     55.57898701944371
    ],
    [
     "C55",
     38.50486458892558
    ],
    [
     "C56",
     39.420042655755914
    ],
    [
     "C57",
     38.983438915888804
    ],
    [
     "C58",
     34.06235043334586
    ],
    [
     "C59",
     41.377993400638324
    ],
    [
     "C60",
     38.60438957509227
    ],
    [
     "C61",
     57.989366184157454
    ],
    [
     "C62",
     59.577420309693665
    ],
    [
     "C63",
     32.99209842483232
    ],
    [
     "C64",
     38.45526695858649
    ],
    [
     "C65",
     23.901195376944628
    ],
    [
     "C66",
     41.252522612899064
    ],
    [
     "C67",
     32.108571654537926
    ],
    [
     "C68",
     55.56443077212319
    ],
    [
     "C69",
     39.06625659389779
    ],
    [
     "C70",
     16.508177846353757
    ],
    [
     "C71",
     42.99156349718135
    ],
    [
     "C72",
     24.940698891307086
    ],
    [
     "C73",
     43.58113578472502
    ],
    [
     "C74",
     54.03569532350585
    ],
    [
     "C75",
     33.88284809148782
    ],
    [
     "C76",
     6.21226659754808
    ],
    [
     "C77",
     30.100866033214803
    ],
    [
     "C78",
     27.444311747851255
    ],
    [
     "C79",
     22.38142729343272
    ],
    [
     "C80",
     54.19400574271635
    ],
    [
     "C81",
     33.63045176217922
    ],
    [
     "C82",
     31.485622946749253
    ],
    [
     "C83",
     38.86613163199008
    ],
    [
     "C84",
     29.061242883644077
    ],
    [
     "C85",
     65.43836845933474
    ],
    [
     "C86",
     19.872553409868694
    ],
    [
     "C87",
     23.465380131375806
    ],
    [
     "C88",
     36.648435894568635
    ],
    [
     "C89",
     69.92694340325559
    ],
    [
     "C90",
     23.03211114938591
    ],
    [
     "C91",
     44.786045267341926
    ],
    [
     "C92",
     54.58147773772418
    ],
    [
     "C93",
     43.94447141403983
    ],
    [
     "C94",
     53.02350958159588
    ],
    [
     "C95",
     52.3909100032503
    ],
    [
     "C96",
     41.73044985202626
    ],
    [
     "C97",
     40.390903828971226
    ],
    [
     "C98",
     71.99789389577745
    ],
    [
     "C99",
     34.579672714645554
    ],
    [
     "C100",
     47.83827803566071
    ],
    [
     "C101",
     22.027551286444897
    ],
    [
     "C102",
     39.93690282665093
    ],
    [
     "C103",
     83.2271558925022
    ],
    [
     "C104",
     34.62581931533061
    ],
    [
     "C105",
     31.298221756615824
    ],
    [
     "C106",
     22.572788964668078
    ],
    [
     "C107",
     36.77103375147376
    ],
    [
     "C108",
     71.55336598497823
    ],
    [
     "C109",
     28.401835483250498
    ],
    [
     "C110",
     59.42016223198024
    ],
    [
     "C111",
     37.024364433753135
    ],
    [
     "C112",
     71.5273305597843
    ],
    [
     "C113",
     36.41153585113954
    ],
    [
     "C114",
     59.265943397678114
    ],
    [
     "C115",
     26.128155367545656
    ],
    [
     "C116",
     49.243499576210645
    ],
    [
     "C117",
     58.446652603200086
    ],
    [
     "C118",
     20.8488101657815
    ],
    [
     "C119",
     39.86044777588926
    ],
    [
     "C120",
     18.254635878328862
    ],
    [
     "C121",
     44.712784239914754
    ],
    [
     "C122",
     29.290323820429727
    ],
    [
     "C123",
     34.62563535198795
    ],
    [
     "C124",
     19.223493190767595
    ],
    [
     "C125",
     35.832487128803685
    ],
    [
     "C126",
     26.110185324668922
    ],
    [
     "C127",
     54.004868717408364
    ],
    [
     "C128",
     61.29154081630062
    ],
    [
     "C129",
     70.50461650937108
    ],
    [
     "C130",
     47.98208738797067
    ],
    [
     "C131",
     44.0997829450324
    ],
    [
     "C132",
     45.006438585057886
    ],
    [
     "C133",
     41.86081440369103
    ],
    [
     "C134",
     54.66751611335187
    ],
    [
     "C135",
     40.88883071828668
    ],
    [
     "C136",
     53.86630957707768
    ],
    [
     "C137",
     30.581289009709423
    ],
    [
     "C138",
     15.902779499327913
    ],
    [
     "C139",
     34.456964031624615
    ],
    [
     "C140",
     39.2312322770895
    ],
    [
     "C141",
     40.25374546636036
    ],
    [
     "C142",
     43.40144123291909
    ],
    [
     "C143",
     64.90959354810995
    ],
    [
     "C144",
     57.307731624939734
    ],
    [
     "C145",
     46.40446041542314
    ],
    [
     "C146",
     55.700275545328964
    ],
    [
     "C147",
     47.04389117982276
    ],
    [
     "C148",
     38.09199384792918
    ],
    [
     "C149",
     27.93370893654876
    ]
   ],
   "error": null
  }
 },
 "OR-002": {
  "source_code_sha256": "9acc01cf13bbfcd0851aabf913c052629e74dda4f64d8faf2beaeaf56e706b6c",
  "original_solution_sha256": "b69dadf89947a0ef575782b3cf72fcd95b8609b6eb70a5015203d3669730a53c",
  "bindings": [
   [
    "C0",
    "x[S1,C1]"
   ],
   [
    "C1",
    "x[S1,C2]"
   ],
   [
    "C2",
    "x[S1,C3]"
   ],
   [
    "C3",
    "x[S1,C4]"
   ],
   [
    "C4",
    "x[S1,C5]"
   ],
   [
    "C5",
    "x[S1,C6]"
   ],
   [
    "C6",
    "x[S1,C7]"
   ],
   [
    "C7",
    "x[S1,C8]"
   ],
   [
    "C8",
    "x[S1,C9]"
   ],
   [
    "C9",
    "x[S1,C10]"
   ],
   [
    "C10",
    "x[S1,C11]"
   ],
   [
    "C11",
    "x[S1,C12]"
   ],
   [
    "C12",
    "x[S2,C1]"
   ],
   [
    "C13",
    "x[S2,C2]"
   ],
   [
    "C14",
    "x[S2,C3]"
   ],
   [
    "C15",
    "x[S2,C4]"
   ],
   [
    "C16",
    "x[S2,C5]"
   ],
   [
    "C17",
    "x[S2,C6]"
   ],
   [
    "C18",
    "x[S2,C7]"
   ],
   [
    "C19",
    "x[S2,C8]"
   ],
   [
    "C20",
    "x[S2,C9]"
   ],
   [
    "C21",
    "x[S2,C10]"
   ],
   [
    "C22",
    "x[S2,C11]"
   ],
   [
    "C23",
    "x[S2,C12]"
   ],
   [
    "C24",
    "x[S3,C1]"
   ],
   [
    "C25",
    "x[S3,C2]"
   ],
   [
    "C26",
    "x[S3,C3]"
   ],
   [
    "C27",
    "x[S3,C4]"
   ],
   [
    "C28",
    "x[S3,C5]"
   ],
   [
    "C29",
    "x[S3,C6]"
   ],
   [
    "C30",
    "x[S3,C7]"
   ],
   [
    "C31",
    "x[S3,C8]"
   ],
   [
    "C32",
    "x[S3,C9]"
   ],
   [
    "C33",
    "x[S3,C10]"
   ],
   [
    "C34",
    "x[S3,C11]"
   ],
   [
    "C35",
    "x[S3,C12]"
   ],
   [
    "C36",
    "x[S4,C1]"
   ],
   [
    "C37",
    "x[S4,C2]"
   ],
   [
    "C38",
    "x[S4,C3]"
   ],
   [
    "C39",
    "x[S4,C4]"
   ],
   [
    "C40",
    "x[S4,C5]"
   ],
   [
    "C41",
    "x[S4,C6]"
   ],
   [
    "C42",
    "x[S4,C7]"
   ],
   [
    "C43",
    "x[S4,C8]"
   ],
   [
    "C44",
    "x[S4,C9]"
   ],
   [
    "C45",
    "x[S4,C10]"
   ],
   [
    "C46",
    "x[S4,C11]"
   ],
   [
    "C47",
    "x[S4,C12]"
   ],
   [
    "C48",
    "x[S5,C1]"
   ],
   [
    "C49",
    "x[S5,C2]"
   ],
   [
    "C50",
    "x[S5,C3]"
   ],
   [
    "C51",
    "x[S5,C4]"
   ],
   [
    "C52",
    "x[S5,C5]"
   ],
   [
    "C53",
    "x[S5,C6]"
   ],
   [
    "C54",
    "x[S5,C7]"
   ],
   [
    "C55",
    "x[S5,C8]"
   ],
   [
    "C56",
    "x[S5,C9]"
   ],
   [
    "C57",
    "x[S5,C10]"
   ],
   [
    "C58",
    "x[S5,C11]"
   ],
   [
    "C59",
    "x[S5,C12]"
   ],
   [
    "C60",
    "x[S6,C1]"
   ],
   [
    "C61",
    "x[S6,C2]"
   ],
   [
    "C62",
    "x[S6,C3]"
   ],
   [
    "C63",
    "x[S6,C4]"
   ],
   [
    "C64",
    "x[S6,C5]"
   ],
   [
    "C65",
    "x[S6,C6]"
   ],
   [
    "C66",
    "x[S6,C7]"
   ],
   [
    "C67",
    "x[S6,C8]"
   ],
   [
    "C68",
    "x[S6,C9]"
   ],
   [
    "C69",
    "x[S6,C10]"
   ],
   [
    "C70",
    "x[S6,C11]"
   ],
   [
    "C71",
    "x[S6,C12]"
   ],
   [
    "C72",
    "x[S7,C1]"
   ],
   [
    "C73",
    "x[S7,C2]"
   ],
   [
    "C74",
    "x[S7,C3]"
   ],
   [
    "C75",
    "x[S7,C4]"
   ],
   [
    "C76",
    "x[S7,C5]"
   ],
   [
    "C77",
    "x[S7,C6]"
   ],
   [
    "C78",
    "x[S7,C7]"
   ],
   [
    "C79",
    "x[S7,C8]"
   ],
   [
    "C80",
    "x[S7,C9]"
   ],
   [
    "C81",
    "x[S7,C10]"
   ],
   [
    "C82",
    "x[S7,C11]"
   ],
   [
    "C83",
    "x[S7,C12]"
   ],
   [
    "C84",
    "x[S8,C1]"
   ],
   [
    "C85",
    "x[S8,C2]"
   ],
   [
    "C86",
    "x[S8,C3]"
   ],
   [
    "C87",
    "x[S8,C4]"
   ],
   [
    "C88",
    "x[S8,C5]"
   ],
   [
    "C89",
    "x[S8,C6]"
   ],
   [
    "C90",
    "x[S8,C7]"
   ],
   [
    "C91",
    "x[S8,C8]"
   ],
   [
    "C92",
    "x[S8,C9]"
   ],
   [
    "C93",
    "x[S8,C10]"
   ],
   [
    "C94",
    "x[S8,C11]"
   ],
   [
    "C95",
    "x[S8,C12]"
   ],
   [
    "C96",
    "x[S9,C1]"
   ],
   [
    "C97",
    "x[S9,C2]"
   ],
   [
    "C98",
    "x[S9,C3]"
   ],
   [
    "C99",
    "x[S9,C4]"
   ],
   [
    "C100",
    "x[S9,C5]"
   ],
   [
    "C101",
    "x[S9,C6]"
   ],
   [
    "C102",
    "x[S9,C7]"
   ],
   [
    "C103",
    "x[S9,C8]"
   ],
   [
    "C104",
    "x[S9,C9]"
   ],
   [
    "C105",
    "x[S9,C10]"
   ],
   [
    "C106",
    "x[S9,C11]"
   ],
   [
    "C107",
    "x[S9,C12]"
   ],
   [
    "C108",
    "x[S10,C1]"
   ],
   [
    "C109",
    "x[S10,C2]"
   ],
   [
    "C110",
    "x[S10,C3]"
   ],
   [
    "C111",
    "x[S10,C4]"
   ],
   [
    "C112",
    "x[S10,C5]"
   ],
   [
    "C113",
    "x[S10,C6]"
   ],
   [
    "C114",
    "x[S10,C7]"
   ],
   [
    "C115",
    "x[S10,C8]"
   ],
   [
    "C116",
    "x[S10,C9]"
   ],
   [
    "C117",
    "x[S10,C10]"
   ],
   [
    "C118",
    "x[S10,C11]"
   ],
   [
    "C119",
    "x[S10,C12]"
   ],
   [
    "C120",
    "x[S11,C1]"
   ],
   [
    "C121",
    "x[S11,C2]"
   ],
   [
    "C122",
    "x[S11,C3]"
   ],
   [
    "C123",
    "x[S11,C4]"
   ],
   [
    "C124",
    "x[S11,C5]"
   ],
   [
    "C125",
    "x[S11,C6]"
   ],
   [
    "C126",
    "x[S11,C7]"
   ],
   [
    "C127",
    "x[S11,C8]"
   ],
   [
    "C128",
    "x[S11,C9]"
   ],
   [
    "C129",
    "x[S11,C10]"
   ],
   [
    "C130",
    "x[S11,C11]"
   ],
   [
    "C131",
    "x[S11,C12]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_0_2": "x[S1,C3]",
   "x_0_3": "x[S1,C4]",
   "x_0_4": "x[S1,C5]",
   "x_0_5": "x[S1,C6]",
   "x_0_6": "x[S1,C7]",
   "x_0_7": "x[S1,C8]",
   "x_0_8": "x[S1,C9]",
   "x_0_9": "x[S1,C10]",
   "x_0_10": "x[S1,C11]",
   "x_0_11": "x[S1,C12]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "x_1_2": "x[S2,C3]",
   "x_1_3": "x[S2,C4]",
   "x_1_4": "x[S2,C5]",
   "x_1_5": "x[S2,C6]",
   "x_1_6": "x[S2,C7]",
   "x_1_7": "x[S2,C8]",
   "x_1_8": "x[S2,C9]",
   "x_1_9": "x[S2,C10]",
   "x_1_10": "x[S2,C11]",
   "x_1_11": "x[S2,C12]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_2_10": "x[S3,C11]",
   "x_2_11": "x[S3,C12]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_3_10": "x[S4,C11]",
   "x_3_11": "x[S4,C12]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_4_10": "x[S5,C11]",
   "x_4_11": "x[S5,C12]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_5_10": "x[S6,C11]",
   "x_5_11": "x[S6,C12]",
   "x_6_0": "x[S7,C1]",
   "x_6_1": "x[S7,C2]",
   "x_6_2": "x[S7,C3]",
   "x_6_3": "x[S7,C4]",
   "x_6_4": "x[S7,C5]",
   "x_6_5": "x[S7,C6]",
   "x_6_6": "x[S7,C7]",
   "x_6_7": "x[S7,C8]",
   "x_6_8": "x[S7,C9]",
   "x_6_9": "x[S7,C10]",
   "x_6_10": "x[S7,C11]",
   "x_6_11": "x[S7,C12]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_7_10": "x[S8,C11]",
   "x_7_11": "x[S8,C12]",
   "x_8_0": "x[S9,C1]",
   "x_8_1": "x[S9,C2]",
   "x_8_2": "x[S9,C3]",
   "x_8_3": "x[S9,C4]",
   "x_8_4": "x[S9,C5]",
   "x_8_5": "x[S9,C6]",
   "x_8_6": "x[S9,C7]",
   "x_8_7": "x[S9,C8]",
   "x_8_8": "x[S9,C9]",
   "x_8_9": "x[S9,C10]",
   "x_8_10": "x[S9,C11]",
   "x_8_11": "x[S9,C12]",
   "x_9_0": "x[S10,C1]",
   "x_9_1": "x[S10,C2]",
   "x_9_2": "x[S10,C3]",
   "x_9_3": "x[S10,C4]",
   "x_9_4": "x[S10,C5]",
   "x_9_5": "x[S10,C6]",
   "x_9_6": "x[S10,C7]",
   "x_9_7": "x[S10,C8]",
   "x_9_8": "x[S10,C9]",
   "x_9_9": "x[S10,C10]",
   "x_9_10": "x[S10,C11]",
   "x_9_11": "x[S10,C12]",
   "x_10_0": "x[S11,C1]",
   "x_10_1": "x[S11,C2]",
   "x_10_2": "x[S11,C3]",
   "x_10_3": "x[S11,C4]",
   "x_10_4": "x[S11,C5]",
   "x_10_5": "x[S11,C6]",
   "x_10_6": "x[S11,C7]",
   "x_10_7": "x[S11,C8]",
   "x_10_8": "x[S11,C9]",
   "x_10_9": "x[S11,C10]",
   "x_10_10": "x[S11,C11]",
   "x_10_11": "x[S11,C12]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 6464420.6996351,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     4.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     575.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     1148.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     315.0
    ],
    [
     "C31",
     41.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     178.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     228.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     35.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     15.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     3.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     5607.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     66.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     475.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     6.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     11.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     45.0
    ],
    [
     "C111",
     833.0
    ],
    [
     "C112",
     154.0
    ],
    [
     "C113",
     551.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     174.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     205.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     174.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-006": {
  "source_code_sha256": "5066b6ef5e293ecde1cdf7a72a7d9b63379f140a4901f19955baf911483d684b",
  "original_solution_sha256": "408badf617759d1d597d06caee6f8c313f246f59585721b7cce5056f9a2cb7d4",
  "bindings": [
   [
    "C0",
    "x[S1,C1]"
   ],
   [
    "C1",
    "x[S1,C2]"
   ],
   [
    "C2",
    "x[S1,C3]"
   ],
   [
    "C3",
    "x[S1,C4]"
   ],
   [
    "C4",
    "x[S1,C5]"
   ],
   [
    "C5",
    "x[S1,C6]"
   ],
   [
    "C6",
    "x[S1,C7]"
   ],
   [
    "C7",
    "x[S1,C8]"
   ],
   [
    "C8",
    "x[S1,C9]"
   ],
   [
    "C9",
    "x[S1,C10]"
   ],
   [
    "C10",
    "x[S2,C1]"
   ],
   [
    "C11",
    "x[S2,C2]"
   ],
   [
    "C12",
    "x[S2,C3]"
   ],
   [
    "C13",
    "x[S2,C4]"
   ],
   [
    "C14",
    "x[S2,C5]"
   ],
   [
    "C15",
    "x[S2,C6]"
   ],
   [
    "C16",
    "x[S2,C7]"
   ],
   [
    "C17",
    "x[S2,C8]"
   ],
   [
    "C18",
    "x[S2,C9]"
   ],
   [
    "C19",
    "x[S2,C10]"
   ],
   [
    "C20",
    "x[S3,C1]"
   ],
   [
    "C21",
    "x[S3,C2]"
   ],
   [
    "C22",
    "x[S3,C3]"
   ],
   [
    "C23",
    "x[S3,C4]"
   ],
   [
    "C24",
    "x[S3,C5]"
   ],
   [
    "C25",
    "x[S3,C6]"
   ],
   [
    "C26",
    "x[S3,C7]"
   ],
   [
    "C27",
    "x[S3,C8]"
   ],
   [
    "C28",
    "x[S3,C9]"
   ],
   [
    "C29",
    "x[S3,C10]"
   ],
   [
    "C30",
    "x[S4,C1]"
   ],
   [
    "C31",
    "x[S4,C2]"
   ],
   [
    "C32",
    "x[S4,C3]"
   ],
   [
    "C33",
    "x[S4,C4]"
   ],
   [
    "C34",
    "x[S4,C5]"
   ],
   [
    "C35",
    "x[S4,C6]"
   ],
   [
    "C36",
    "x[S4,C7]"
   ],
   [
    "C37",
    "x[S4,C8]"
   ],
   [
    "C38",
    "x[S4,C9]"
   ],
   [
    "C39",
    "x[S4,C10]"
   ],
   [
    "C40",
    "x[S5,C1]"
   ],
   [
    "C41",
    "x[S5,C2]"
   ],
   [
    "C42",
    "x[S5,C3]"
   ],
   [
    "C43",
    "x[S5,C4]"
   ],
   [
    "C44",
    "x[S5,C5]"
   ],
   [
    "C45",
    "x[S5,C6]"
   ],
   [
    "C46",
    "x[S5,C7]"
   ],
   [
    "C47",
    "x[S5,C8]"
   ],
   [
    "C48",
    "x[S5,C9]"
   ],
   [
    "C49",
    "x[S5,C10]"
   ],
   [
    "C50",
    "x[S6,C1]"
   ],
   [
    "C51",
    "x[S6,C2]"
   ],
   [
    "C52",
    "x[S6,C3]"
   ],
   [
    "C53",
    "x[S6,C4]"
   ],
   [
    "C54",
    "x[S6,C5]"
   ],
   [
    "C55",
    "x[S6,C6]"
   ],
   [
    "C56",
    "x[S6,C7]"
   ],
   [
    "C57",
    "x[S6,C8]"
   ],
   [
    "C58",
    "x[S6,C9]"
   ],
   [
    "C59",
    "x[S6,C10]"
   ],
   [
    "C60",
    "x[S7,C1]"
   ],
   [
    "C61",
    "x[S7,C2]"
   ],
   [
    "C62",
    "x[S7,C3]"
   ],
   [
    "C63",
    "x[S7,C4]"
   ],
   [
    "C64",
    "x[S7,C5]"
   ],
   [
    "C65",
    "x[S7,C6]"
   ],
   [
    "C66",
    "x[S7,C7]"
   ],
   [
    "C67",
    "x[S7,C8]"
   ],
   [
    "C68",
    "x[S7,C9]"
   ],
   [
    "C69",
    "x[S7,C10]"
   ],
   [
    "C70",
    "x[S8,C1]"
   ],
   [
    "C71",
    "x[S8,C2]"
   ],
   [
    "C72",
    "x[S8,C3]"
   ],
   [
    "C73",
    "x[S8,C4]"
   ],
   [
    "C74",
    "x[S8,C5]"
   ],
   [
    "C75",
    "x[S8,C6]"
   ],
   [
    "C76",
    "x[S8,C7]"
   ],
   [
    "C77",
    "x[S8,C8]"
   ],
   [
    "C78",
    "x[S8,C9]"
   ],
   [
    "C79",
    "x[S8,C10]"
   ],
   [
    "C80",
    "x[S9,C1]"
   ],
   [
    "C81",
    "x[S9,C2]"
   ],
   [
    "C82",
    "x[S9,C3]"
   ],
   [
    "C83",
    "x[S9,C4]"
   ],
   [
    "C84",
    "x[S9,C5]"
   ],
   [
    "C85",
    "x[S9,C6]"
   ],
   [
    "C86",
    "x[S9,C7]"
   ],
   [
    "C87",
    "x[S9,C8]"
   ],
   [
    "C88",
    "x[S9,C9]"
   ],
   [
    "C89",
    "x[S9,C10]"
   ],
   [
    "C90",
    "x[S10,C1]"
   ],
   [
    "C91",
    "x[S10,C2]"
   ],
   [
    "C92",
    "x[S10,C3]"
   ],
   [
    "C93",
    "x[S10,C4]"
   ],
   [
    "C94",
    "x[S10,C5]"
   ],
   [
    "C95",
    "x[S10,C6]"
   ],
   [
    "C96",
    "x[S10,C7]"
   ],
   [
    "C97",
    "x[S10,C8]"
   ],
   [
    "C98",
    "x[S10,C9]"
   ],
   [
    "C99",
    "x[S10,C10]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_2": "x[S1,C3]",
   "x_0_5": "x[S1,C6]",
   "x_0_8": "x[S1,C9]",
   "x_1_0": "x[S2,C1]",
   "x_1_2": "x[S2,C3]",
   "x_1_5": "x[S2,C6]",
   "x_1_8": "x[S2,C9]",
   "x_2_0": "x[S3,C1]",
   "x_2_1": "x[S3,C2]",
   "x_2_2": "x[S3,C3]",
   "x_2_3": "x[S3,C4]",
   "x_2_4": "x[S3,C5]",
   "x_2_5": "x[S3,C6]",
   "x_2_6": "x[S3,C7]",
   "x_2_7": "x[S3,C8]",
   "x_2_8": "x[S3,C9]",
   "x_2_9": "x[S3,C10]",
   "x_3_0": "x[S4,C1]",
   "x_3_1": "x[S4,C2]",
   "x_3_2": "x[S4,C3]",
   "x_3_3": "x[S4,C4]",
   "x_3_4": "x[S4,C5]",
   "x_3_5": "x[S4,C6]",
   "x_3_6": "x[S4,C7]",
   "x_3_7": "x[S4,C8]",
   "x_3_8": "x[S4,C9]",
   "x_3_9": "x[S4,C10]",
   "x_4_0": "x[S5,C1]",
   "x_4_1": "x[S5,C2]",
   "x_4_2": "x[S5,C3]",
   "x_4_3": "x[S5,C4]",
   "x_4_4": "x[S5,C5]",
   "x_4_5": "x[S5,C6]",
   "x_4_6": "x[S5,C7]",
   "x_4_7": "x[S5,C8]",
   "x_4_8": "x[S5,C9]",
   "x_4_9": "x[S5,C10]",
   "x_5_0": "x[S6,C1]",
   "x_5_1": "x[S6,C2]",
   "x_5_2": "x[S6,C3]",
   "x_5_3": "x[S6,C4]",
   "x_5_4": "x[S6,C5]",
   "x_5_5": "x[S6,C6]",
   "x_5_6": "x[S6,C7]",
   "x_5_7": "x[S6,C8]",
   "x_5_8": "x[S6,C9]",
   "x_5_9": "x[S6,C10]",
   "x_6_0": "x[S7,C1]",
   "x_6_2": "x[S7,C3]",
   "x_6_5": "x[S7,C6]",
   "x_6_8": "x[S7,C9]",
   "x_7_0": "x[S8,C1]",
   "x_7_1": "x[S8,C2]",
   "x_7_2": "x[S8,C3]",
   "x_7_3": "x[S8,C4]",
   "x_7_4": "x[S8,C5]",
   "x_7_5": "x[S8,C6]",
   "x_7_6": "x[S8,C7]",
   "x_7_7": "x[S8,C8]",
   "x_7_8": "x[S8,C9]",
   "x_7_9": "x[S8,C10]",
   "x_8_0": "x[S9,C1]",
   "x_8_2": "x[S9,C3]",
   "x_8_5": "x[S9,C6]",
   "x_8_8": "x[S9,C9]",
   "x_9_0": "x[S10,C1]",
   "x_9_2": "x[S10,C3]",
   "x_9_5": "x[S10,C6]",
   "x_9_8": "x[S10,C9]",
   "x_0_1": "x[S1,C2]",
   "x_1_1": "x[S2,C2]",
   "x_6_1": "x[S7,C2]",
   "x_8_1": "x[S9,C2]",
   "x_9_1": "x[S10,C2]",
   "x_0_3": "x[S1,C4]",
   "x_1_3": "x[S2,C4]",
   "x_6_3": "x[S7,C4]",
   "x_8_3": "x[S9,C4]",
   "x_9_3": "x[S10,C4]",
   "x_0_4": "x[S1,C5]",
   "x_1_4": "x[S2,C5]",
   "x_6_4": "x[S7,C5]",
   "x_8_4": "x[S9,C5]",
   "x_9_4": "x[S10,C5]",
   "x_0_6": "x[S1,C7]",
   "x_1_6": "x[S2,C7]",
   "x_6_6": "x[S7,C7]",
   "x_8_6": "x[S9,C7]",
   "x_9_6": "x[S10,C7]",
   "x_0_7": "x[S1,C8]",
   "x_1_7": "x[S2,C8]",
   "x_6_7": "x[S7,C8]",
   "x_8_7": "x[S9,C8]",
   "x_9_7": "x[S10,C8]",
   "x_0_9": "x[S1,C10]",
   "x_1_9": "x[S2,C10]",
   "x_6_9": "x[S7,C10]",
   "x_8_9": "x[S9,C10]",
   "x_9_9": "x[S10,C10]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 6340.972286592268,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     50.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     77.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     23.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     114.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     99.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     94.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     45.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     57.0
    ],
    [
     "C95",
     52.0
    ],
    [
     "C96",
     23.0
    ],
    [
     "C97",
     49.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-010": {
  "source_code_sha256": "0f7750dee89c83d1f6327ebeec6a6f05e40ed197be72923082ae4aeeba2132ac",
  "original_solution_sha256": "677952d7b571403bca398d687dde1a79617dee9b21057daafe23183b2f57d55b",
  "bindings": [
   [
    "C0",
    "x[Adams Group_service]"
   ],
   [
    "C1",
    "x[Anderson-Leach_against]"
   ],
   [
    "C2",
    "x[Anderson-Valdez_somebody]"
   ],
   [
    "C3",
    "x[Anderson-White_son]"
   ],
   [
    "C4",
    "x[Andrews LLC_matter]"
   ],
   [
    "C5",
    "x[Andrews PLC_enter]"
   ],
   [
    "C6",
    "x[Andrews-Martin_build]"
   ],
   [
    "C7",
    "x[Arias-Mendoza_life]"
   ],
   [
    "C8",
    "x[Bennett and Sons_down]"
   ],
   [
    "C9",
    "x[Bennett, Foster and Moreno_enter]"
   ],
   [
    "C10",
    "x[Fernandez, Long and Nelson_member]"
   ],
   [
    "C11",
    "x[Fernandez-Fischer_million]"
   ],
   [
    "C12",
    "x[Ferrell Inc_prove]"
   ],
   [
    "C13",
    "x[Fields, Christensen and Daniels_nor]"
   ],
   [
    "C14",
    "x[Figueroa LLC_involve]"
   ],
   [
    "C15",
    "x[Finley Group_itself]"
   ],
   [
    "C16",
    "x[Fisher Ltd_speak]"
   ],
   [
    "C17",
    "x[Fisher-Marshall_cover]"
   ],
   [
    "C18",
    "x[Fuller-Torres_behavior]"
   ],
   [
    "C19",
    "x[Fuller-Walters_radio]"
   ],
   [
    "C20",
    "x[Gallagher-Campbell_dark]"
   ],
   [
    "C21",
    "x[Gallagher-Kirby_talk]"
   ],
   [
    "C22",
    "x[Gallagher-Parker_recognize]"
   ],
   [
    "C23",
    "x[Gonzalez PLC_task]"
   ],
   [
    "C24",
    "x[Gonzalez, Coleman and Le_heavy]"
   ],
   [
    "C25",
    "x[Gonzalez, Lowe and Robinson_education]"
   ],
   [
    "C26",
    "x[Gonzalez-Horn_light]"
   ],
   [
    "C27",
    "x[Good, Davis and Smith_station]"
   ],
   [
    "C28",
    "x[Goodman, Hughes and White_realize]"
   ],
   [
    "C29",
    "x[Goodwin PLC_bad]"
   ],
   [
    "C30",
    "x[Gordon PLC_detail]"
   ],
   [
    "C31",
    "x[Graham LLC_stage]"
   ],
   [
    "C32",
    "x[Graham Ltd_marriage]"
   ],
   [
    "C33",
    "x[Graham-Swanson_message]"
   ],
   [
    "C34",
    "x[Grant, Cross and Bennett_religious]"
   ],
   [
    "C35",
    "x[Grant, Mcdonald and Watson_owner]"
   ],
   [
    "C36",
    "x[Graves, Turner and Crawford_wait]"
   ],
   [
    "C37",
    "x[Gray, Smith and Barnes_much]"
   ],
   [
    "C38",
    "x[Green Inc_direction]"
   ],
   [
    "C39",
    "x[Green-Rogers_could]"
   ],
   [
    "C40",
    "x[Greene-Baxter_them]"
   ],
   [
    "C41",
    "x[Greer and Sons_keep]"
   ],
   [
    "C42",
    "x[Griffin, Boyle and Dawson_anything]"
   ],
   [
    "C43",
    "x[Harper Inc_well]"
   ],
   [
    "C44",
    "x[Harper Ltd_off]"
   ],
   [
    "C45",
    "x[Harrington, Sosa and Mccarty_coach]"
   ],
   [
    "C46",
    "x[Harris Group_different]"
   ],
   [
    "C47",
    "x[Harris and Sons_audience]"
   ],
   [
    "C48",
    "x[Harris and Sons_fear]"
   ],
   [
    "C49",
    "x[Harris, Hamilton and Rose_contain]"
   ],
   [
    "C50",
    "x[Harris, Stevens and Hall_answer]"
   ],
   [
    "C51",
    "x[Harris-Bell_painting]"
   ],
   [
    "C52",
    "x[Harris-Melton_often]"
   ],
   [
    "C53",
    "x[Harris-Rogers_after]"
   ],
   [
    "C54",
    "x[Hart Group_cold]"
   ],
   [
    "C55",
    "x[Hutchinson, Roberts and Mcbride_among]"
   ],
   [
    "C56",
    "x[Ibarra, Jackson and Potter_upon]"
   ],
   [
    "C57",
    "x[Jackson Inc_anyone]"
   ],
   [
    "C58",
    "x[Jackson LLC_report]"
   ],
   [
    "C59",
    "x[Jackson, Collier and Barber_result]"
   ],
   [
    "C60",
    "x[Jackson, White and Brown_expect]"
   ],
   [
    "C61",
    "x[Jackson-Carroll_six]"
   ],
   [
    "C62",
    "x[Lee-Jones_soldier]"
   ],
   [
    "C63",
    "x[Lewis, Sanchez and Turner_point]"
   ],
   [
    "C64",
    "x[Lin LLC_performance]"
   ],
   [
    "C65",
    "x[Lindsey, Avila and Brown_candidate]"
   ],
   [
    "C66",
    "x[Lindsey-Strickland_modern]"
   ],
   [
    "C67",
    "x[Little-Perez_choice]"
   ],
   [
    "C68",
    "x[Lloyd, Stone and Mcguire_national]"
   ],
   [
    "C69",
    "x[Long, Hughes and Gallegos_receive]"
   ],
   [
    "C70",
    "x[Lopez PLC_analysis]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Adams Group_service]",
   "x_2": "x[Anderson-Leach_against]",
   "x_3": "x[Anderson-Valdez_somebody]",
   "x_4": "x[Anderson-White_son]",
   "x_5": "x[Andrews LLC_matter]",
   "x_6": "x[Andrews PLC_enter]",
   "x_7": "x[Andrews-Martin_build]",
   "x_8": "x[Arias-Mendoza_life]",
   "x_9": "x[Bennett and Sons_down]",
   "x_10": "x[Bennett, Foster and Moreno_enter]",
   "x_11": "x[Fernandez, Long and Nelson_member]",
   "x_12": "x[Fernandez-Fischer_million]",
   "x_13": "x[Ferrell Inc_prove]",
   "x_14": "x[Fields, Christensen and Daniels_nor]",
   "x_15": "x[Figueroa LLC_involve]",
   "x_16": "x[Finley Group_itself]",
   "x_17": "x[Fisher Ltd_speak]",
   "x_18": "x[Fisher-Marshall_cover]",
   "x_19": "x[Fuller-Torres_behavior]",
   "x_20": "x[Fuller-Walters_radio]",
   "x_21": "x[Gallagher-Campbell_dark]",
   "x_22": "x[Gallagher-Kirby_talk]",
   "x_23": "x[Gallagher-Parker_recognize]",
   "x_24": "x[Gonzalez PLC_task]",
   "x_25": "x[Gonzalez, Coleman and Le_heavy]",
   "x_26": "x[Gonzalez, Lowe and Robinson_education]",
   "x_27": "x[Gonzalez-Horn_light]",
   "x_28": "x[Good, Davis and Smith_station]",
   "x_29": "x[Goodman, Hughes and White_realize]",
   "x_30": "x[Goodwin PLC_bad]",
   "x_31": "x[Gordon PLC_detail]",
   "x_32": "x[Graham LLC_stage]",
   "x_33": "x[Graham Ltd_marriage]",
   "x_34": "x[Graham-Swanson_message]",
   "x_35": "x[Grant, Cross and Bennett_religious]",
   "x_36": "x[Grant, Mcdonald and Watson_owner]",
   "x_37": "x[Graves, Turner and Crawford_wait]",
   "x_38": "x[Gray, Smith and Barnes_much]",
   "x_39": "x[Green Inc_direction]",
   "x_40": "x[Green-Rogers_could]",
   "x_41": "x[Greene-Baxter_them]",
   "x_42": "x[Greer and Sons_keep]",
   "x_43": "x[Griffin, Boyle and Dawson_anything]",
   "x_44": "x[Harper Inc_well]",
   "x_45": "x[Harper Ltd_off]",
   "x_46": "x[Harrington, Sosa and Mccarty_coach]",
   "x_47": "x[Harris Group_different]",
   "x_48": "x[Harris and Sons_audience]",
   "x_49": "x[Harris and Sons_fear]",
   "x_50": "x[Harris, Hamilton and Rose_contain]",
   "x_51": "x[Harris, Stevens and Hall_answer]",
   "x_52": "x[Harris-Bell_painting]",
   "x_53": "x[Harris-Melton_often]",
   "x_54": "x[Harris-Rogers_after]",
   "x_55": "x[Hart Group_cold]",
   "x_56": "x[Hutchinson, Roberts and Mcbride_among]",
   "x_57": "x[Ibarra, Jackson and Potter_upon]",
   "x_58": "x[Jackson Inc_anyone]",
   "x_59": "x[Jackson LLC_report]",
   "x_60": "x[Jackson, Collier and Barber_result]",
   "x_61": "x[Jackson, White and Brown_expect]",
   "x_62": "x[Jackson-Carroll_six]",
   "x_63": "x[Lee-Jones_soldier]",
   "x_64": "x[Lewis, Sanchez and Turner_point]",
   "x_65": "x[Lin LLC_performance]",
   "x_66": "x[Lindsey, Avila and Brown_candidate]",
   "x_67": "x[Lindsey-Strickland_modern]",
   "x_68": "x[Little-Perez_choice]",
   "x_69": "x[Lloyd, Stone and Mcguire_national]",
   "x_70": "x[Long, Hughes and Gallegos_receive]",
   "x_71": "x[Lopez PLC_analysis]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 4003445.2099999995,
   "variables": [
    [
     "C0",
     10.0
    ],
    [
     "C1",
     6.0
    ],
    [
     "C2",
     75.0
    ],
    [
     "C3",
     121.0
    ],
    [
     "C4",
     130.0
    ],
    [
     "C5",
     112.0
    ],
    [
     "C6",
     125.0
    ],
    [
     "C7",
     54.0
    ],
    [
     "C8",
     75.0
    ],
    [
     "C9",
     102.0
    ],
    [
     "C10",
     48.0
    ],
    [
     "C11",
     4.0
    ],
    [
     "C12",
     125.0
    ],
    [
     "C13",
     9.0
    ],
    [
     "C14",
     132.0
    ],
    [
     "C15",
     29.0
    ],
    [
     "C16",
     43.0
    ],
    [
     "C17",
     5.0
    ],
    [
     "C18",
     40.0
    ],
    [
     "C19",
     48.0
    ],
    [
     "C20",
     87.0
    ],
    [
     "C21",
     79.0
    ],
    [
     "C22",
     20.0
    ],
    [
     "C23",
     33.0
    ],
    [
     "C24",
     60.0
    ],
    [
     "C25",
     25.0
    ],
    [
     "C26",
     42.0
    ],
    [
     "C27",
     52.0
    ],
    [
     "C28",
     116.0
    ],
    [
     "C29",
     95.0
    ],
    [
     "C30",
     66.0
    ],
    [
     "C31",
     25.0
    ],
    [
     "C32",
     78.0
    ],
    [
     "C33",
     138.0
    ],
    [
     "C34",
     32.0
    ],
    [
     "C35",
     28.0
    ],
    [
     "C36",
     84.0
    ],
    [
     "C37",
     102.0
    ],
    [
     "C38",
     112.0
    ],
    [
     "C39",
     7.0
    ],
    [
     "C40",
     62.0
    ],
    [
     "C41",
     109.0
    ],
    [
     "C42",
     81.0
    ],
    [
     "C43",
     131.0
    ],
    [
     "C44",
     26.0
    ],
    [
     "C45",
     109.0
    ],
    [
     "C46",
     65.0
    ],
    [
     "C47",
     121.0
    ],
    [
     "C48",
     90.0
    ],
    [
     "C49",
     31.0
    ],
    [
     "C50",
     24.0
    ],
    [
     "C51",
     122.0
    ],
    [
     "C52",
     101.0
    ],
    [
     "C53",
     76.0
    ],
    [
     "C54",
     100.0
    ],
    [
     "C55",
     125.0
    ],
    [
     "C56",
     45.0
    ],
    [
     "C57",
     80.0
    ],
    [
     "C58",
     64.0
    ],
    [
     "C59",
     98.0
    ],
    [
     "C60",
     102.0
    ],
    [
     "C61",
     40.0
    ],
    [
     "C62",
     110.0
    ],
    [
     "C63",
     59.0
    ],
    [
     "C64",
     97.0
    ],
    [
     "C65",
     90.0
    ],
    [
     "C66",
     43.0
    ],
    [
     "C67",
     96.0
    ],
    [
     "C68",
     21.0
    ],
    [
     "C69",
     55.0
    ],
    [
     "C70",
     30.0
    ]
   ],
   "error": null
  }
 },
 "OR-013": {
  "source_code_sha256": "afff73d746251d5968d29dfb076c4d9bf9b89d4ba94b4fbd7b8bbc6d79fb9ed0",
  "original_solution_sha256": "9b9813e507996214236de5d3ce1969f7301c4115d63f14eb27c9e87492b71516",
  "bindings": [
   [
    "C0",
    "x[MAT4US5A9FBDFFF21B5]"
   ],
   [
    "C1",
    "x[KAB4US59AD6D0B32674]"
   ],
   [
    "C2",
    "x[MAT4US5A9FB58986D2E]"
   ],
   [
    "C3",
    "x[MAT4US5AABB7579D03C]"
   ],
   [
    "C4",
    "x[ENT4US5A979B09A2BF7]"
   ],
   [
    "C5",
    "x[4U_Service_7]"
   ],
   [
    "C6",
    "x[KAB4US59AD6CF40C781]"
   ],
   [
    "C7",
    "x[4U_Service_36]"
   ],
   [
    "C8",
    "x[KAB4US59AD6CF9E69D8]"
   ],
   [
    "C9",
    "x[KAB4US59AD6D003A730]"
   ],
   [
    "C10",
    "x[4U_Service_22]"
   ],
   [
    "C11",
    "x[COM4US5A9FCF8FE2DE3]"
   ],
   [
    "C12",
    "x[KAB4US59AD6D0489179]"
   ],
   [
    "C13",
    "x[MAT4US5A8BEC8AB2742]"
   ],
   [
    "C14",
    "x[MAT4US5A9FAC1A50375]"
   ],
   [
    "C15",
    "x[WOF4US5A979D79EEF31]"
   ],
   [
    "C16",
    "x[MAT4US5A9FBC6918099]"
   ],
   [
    "C17",
    "x[COM4US5A9FCDA605F7B]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[4U_Service_22]",
   "x_2": "x[4U_Service_36]",
   "x_3": "x[4U_Service_7]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 8477.1,
   "variables": [
    [
     "C0",
     9.0
    ],
    [
     "C1",
     6.0
    ],
    [
     "C2",
     20.0
    ],
    [
     "C3",
     3.0
    ],
    [
     "C4",
     15.0
    ],
    [
     "C5",
     3.0
    ],
    [
     "C6",
     11.0
    ],
    [
     "C7",
     3.0
    ],
    [
     "C8",
     36.0
    ],
    [
     "C9",
     5.0
    ],
    [
     "C10",
     5.0
    ],
    [
     "C11",
     3.0
    ],
    [
     "C12",
     19.0
    ],
    [
     "C13",
     110.0
    ],
    [
     "C14",
     6.0
    ],
    [
     "C15",
     19.0
    ],
    [
     "C16",
     68.0
    ],
    [
     "C17",
     3.0
    ]
   ],
   "error": null
  }
 },
 "OR-018": {
  "source_code_sha256": "1449eb0e4e054636bb30ed6692a92b81b8a0364f261ad508ff8697b7cdbefd36",
  "original_solution_sha256": "055485cf17ec2c7dee00c799a4f10e17a7660386f1dec9c5add0a9ca664b7cbe",
  "bindings": [
   [
    "C0",
    "x[Baby Food_255.28]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Baby Food_255.28]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 782819438.64,
   "variables": [
    [
     "C0",
     3066513.0
    ]
   ],
   "error": null
  }
 },
 "OR-022": {
  "source_code_sha256": "6f94d578eaec9beaed015d5f29aca33e54fbb65787a3180a0be1c0032a6d9ea8",
  "original_solution_sha256": "e83759268b35c4c0697e96d5065e61abf933c0b0d55ec07a8cfeec9bd1abe122",
  "bindings": [
   [
    "C0",
    "x[27in FHD Monitor]"
   ],
   [
    "C1",
    "x[27in 4K Gaming Monitor]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[27in 4K Gaming Monitor]",
   "x_2": "x[27in FHD Monitor]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 4049812.4247,
   "variables": [
    [
     "C0",
     15057.0
    ],
    [
     "C1",
     12474.0
    ]
   ],
   "error": null
  }
 },
 "OR-026": {
  "source_code_sha256": "bcb9f6a2e1a827cee46fed0bf3c8af6ef0a42021aa4f04133f5b210704df0d44",
  "original_solution_sha256": "98299db46bc1cda46ec192ef54092dba5fd6595a39d763406f43818c2fabc865",
  "bindings": [
   [
    "C0",
    "x[Fashion accessories_10.18]"
   ],
   [
    "C1",
    "x[Fashion accessories_12.09]"
   ],
   [
    "C2",
    "x[Fashion accessories_12.19]"
   ],
   [
    "C3",
    "x[Fashion accessories_12.54]"
   ],
   [
    "C4",
    "x[Fashion accessories_12.78]"
   ],
   [
    "C5",
    "x[Fashion accessories_14.48]"
   ],
   [
    "C6",
    "x[Fashion accessories_15.43]"
   ],
   [
    "C7",
    "x[Fashion accessories_15.5]"
   ],
   [
    "C8",
    "x[Fashion accessories_15.62]"
   ],
   [
    "C9",
    "x[Fashion accessories_16.28]"
   ],
   [
    "C10",
    "x[Fashion accessories_16.45]"
   ],
   [
    "C11",
    "x[Fashion accessories_17.48]"
   ],
   [
    "C12",
    "x[Fashion accessories_17.49]"
   ],
   [
    "C13",
    "x[Fashion accessories_17.87]"
   ],
   [
    "C14",
    "x[Fashion accessories_17.94]"
   ],
   [
    "C15",
    "x[Fashion accessories_18.08]"
   ],
   [
    "C16",
    "x[Fashion accessories_19.66]"
   ],
   [
    "C17",
    "x[Fashion accessories_19.7]"
   ],
   [
    "C18",
    "x[Fashion accessories_19.77]"
   ],
   [
    "C19",
    "x[Fashion accessories_20.01]"
   ],
   [
    "C20",
    "x[Fashion accessories_21.32]"
   ],
   [
    "C21",
    "x[Fashion accessories_21.48]"
   ],
   [
    "C22",
    "x[Fashion accessories_21.94]"
   ],
   [
    "C23",
    "x[Fashion accessories_22.32]"
   ],
   [
    "C24",
    "x[Fashion accessories_22.51]"
   ],
   [
    "C25",
    "x[Fashion accessories_23.82]"
   ],
   [
    "C26",
    "x[Fashion accessories_25.42]"
   ],
   [
    "C27",
    "x[Fashion accessories_25.56]"
   ],
   [
    "C28",
    "x[Fashion accessories_27.02]"
   ],
   [
    "C29",
    "x[Fashion accessories_27.18]"
   ],
   [
    "C30",
    "x[Fashion accessories_27.38]"
   ],
   [
    "C31",
    "x[Fashion accessories_29.42]"
   ],
   [
    "C32",
    "x[Fashion accessories_29.56]"
   ],
   [
    "C33",
    "x[Fashion accessories_30.14]"
   ],
   [
    "C34",
    "x[Fashion accessories_30.37]"
   ],
   [
    "C35",
    "x[Fashion accessories_30.61]"
   ],
   [
    "C36",
    "x[Fashion accessories_30.62]"
   ],
   [
    "C37",
    "x[Fashion accessories_31.73]"
   ],
   [
    "C38",
    "x[Fashion accessories_31.9]"
   ],
   [
    "C39",
    "x[Fashion accessories_32.62]"
   ],
   [
    "C40",
    "x[Fashion accessories_33.52]"
   ],
   [
    "C41",
    "x[Fashion accessories_33.63]"
   ],
   [
    "C42",
    "x[Fashion accessories_34.7]"
   ],
   [
    "C43",
    "x[Fashion accessories_35.19]"
   ],
   [
    "C44",
    "x[Fashion accessories_36.51]"
   ],
   [
    "C45",
    "x[Fashion accessories_36.85]"
   ],
   [
    "C46",
    "x[Fashion accessories_37.15]"
   ],
   [
    "C47",
    "x[Fashion accessories_37.55]"
   ],
   [
    "C48",
    "x[Fashion accessories_37.95]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[Fashion accessories_10.18]",
   "x_2": "x[Fashion accessories_12.09]",
   "x_3": "x[Fashion accessories_12.19]",
   "x_4": "x[Fashion accessories_12.54]",
   "x_5": "x[Fashion accessories_12.78]",
   "x_6": "x[Fashion accessories_14.48]",
   "x_7": "x[Fashion accessories_15.43]",
   "x_8": "x[Fashion accessories_15.5]",
   "x_9": "x[Fashion accessories_15.62]",
   "x_10": "x[Fashion accessories_16.28]",
   "x_11": "x[Fashion accessories_16.45]",
   "x_12": "x[Fashion accessories_17.48]",
   "x_13": "x[Fashion accessories_17.49]",
   "x_14": "x[Fashion accessories_17.87]",
   "x_15": "x[Fashion accessories_17.94]",
   "x_16": "x[Fashion accessories_18.08]",
   "x_17": "x[Fashion accessories_19.66]",
   "x_18": "x[Fashion accessories_19.7]",
   "x_19": "x[Fashion accessories_19.77]",
   "x_20": "x[Fashion accessories_20.01]",
   "x_21": "x[Fashion accessories_21.32]",
   "x_22": "x[Fashion accessories_21.48]",
   "x_23": "x[Fashion accessories_21.94]",
   "x_24": "x[Fashion accessories_22.32]",
   "x_25": "x[Fashion accessories_22.51]",
   "x_26": "x[Fashion accessories_23.82]",
   "x_27": "x[Fashion accessories_25.42]",
   "x_28": "x[Fashion accessories_25.56]",
   "x_29": "x[Fashion accessories_27.02]",
   "x_30": "x[Fashion accessories_27.18]",
   "x_31": "x[Fashion accessories_27.38]",
   "x_32": "x[Fashion accessories_29.42]",
   "x_33": "x[Fashion accessories_29.56]",
   "x_34": "x[Fashion accessories_30.14]",
   "x_35": "x[Fashion accessories_30.37]",
   "x_36": "x[Fashion accessories_30.61]",
   "x_37": "x[Fashion accessories_30.62]",
   "x_38": "x[Fashion accessories_31.73]",
   "x_39": "x[Fashion accessories_31.9]",
   "x_40": "x[Fashion accessories_32.62]",
   "x_41": "x[Fashion accessories_33.52]",
   "x_42": "x[Fashion accessories_33.63]",
   "x_43": "x[Fashion accessories_34.7]",
   "x_44": "x[Fashion accessories_35.19]",
   "x_45": "x[Fashion accessories_36.51]",
   "x_46": "x[Fashion accessories_36.85]",
   "x_47": "x[Fashion accessories_37.15]",
   "x_48": "x[Fashion accessories_37.55]",
   "x_49": "x[Fashion accessories_37.95]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 8851.75,
   "variables": [
    [
     "C0",
     12.0
    ],
    [
     "C1",
     2.0
    ],
    [
     "C2",
     10.0
    ],
    [
     "C3",
     2.0
    ],
    [
     "C4",
     2.0
    ],
    [
     "C5",
     5.0
    ],
    [
     "C6",
     2.0
    ],
    [
     "C7",
     2.0
    ],
    [
     "C8",
     11.0
    ],
    [
     "C9",
     2.0
    ],
    [
     "C10",
     6.0
    ],
    [
     "C11",
     9.0
    ],
    [
     "C12",
     14.0
    ],
    [
     "C13",
     6.0
    ],
    [
     "C14",
     8.0
    ],
    [
     "C15",
     6.0
    ],
    [
     "C16",
     14.0
    ],
    [
     "C17",
     2.0
    ],
    [
     "C18",
     14.0
    ],
    [
     "C19",
     11.0
    ],
    [
     "C20",
     2.0
    ],
    [
     "C21",
     3.0
    ],
    [
     "C22",
     8.0
    ],
    [
     "C23",
     10.0
    ],
    [
     "C24",
     10.0
    ],
    [
     "C25",
     7.0
    ],
    [
     "C26",
     11.0
    ],
    [
     "C27",
     11.0
    ],
    [
     "C28",
     5.0
    ],
    [
     "C29",
     3.0
    ],
    [
     "C30",
     8.0
    ],
    [
     "C31",
     13.0
    ],
    [
     "C32",
     7.0
    ],
    [
     "C33",
     14.0
    ],
    [
     "C34",
     4.0
    ],
    [
     "C35",
     2.0
    ],
    [
     "C36",
     2.0
    ],
    [
     "C37",
     14.0
    ],
    [
     "C38",
     2.0
    ],
    [
     "C39",
     6.0
    ],
    [
     "C40",
     2.0
    ],
    [
     "C41",
     2.0
    ],
    [
     "C42",
     3.0
    ],
    [
     "C43",
     14.0
    ],
    [
     "C44",
     12.0
    ],
    [
     "C45",
     7.0
    ],
    [
     "C46",
     6.0
    ],
    [
     "C47",
     13.0
    ],
    [
     "C48",
     14.0
    ]
   ],
   "error": null
  }
 },
 "OR-030": {
  "source_code_sha256": "4c292fe40cadee8526c751b466a7202cc98f19c450961404ac97011a436dc7f9",
  "original_solution_sha256": "29694a8e68d8af1c1e0f3505b879b75c0f5ba76764205771b9a908d1f1f076bf",
  "bindings": [],
  "primary_mapping": null,
  "mapping_basis": "unavailable",
  "mapping_failure": "ValueError: Duplicate product name detected: FDK57",
  "frozen_replay_solution": {
   "status": "ERROR",
   "objective": null,
   "variables": [],
   "error": "ValueError: Duplicate product name detected: FDK57"
  }
 },
 "OR-034": {
  "source_code_sha256": "8aa672b1e05d4719abf06fe1a36a0aa3d840675a085dd9cfb6ebfe1a50035ac1",
  "original_solution_sha256": "e730af9912d37a26d827fff8899a3bba694af8b9b225479a896d006d7415d36c",
  "bindings": [
   [
    "C0",
    "x[12 MACARON]"
   ],
   [
    "C1",
    "x[ARMORICAIN]"
   ],
   [
    "C2",
    "x[ARTICLE 295]"
   ],
   [
    "C3",
    "x[BAGUETTE]"
   ],
   [
    "C4",
    "x[BAGUETTE APERO]"
   ],
   [
    "C5",
    "x[BAGUETTE GRAINE]"
   ],
   [
    "C6",
    "x[BANETTE]"
   ],
   [
    "C7",
    "x[BANETTINE]"
   ],
   [
    "C8",
    "x[BOISSON 33CL]"
   ],
   [
    "C9",
    "x[BOTTEREAU]"
   ],
   [
    "C10",
    "x[BOULE 200G]"
   ],
   [
    "C11",
    "x[BOULE 400G]"
   ],
   [
    "C12",
    "x[BOULE POLKA]"
   ],
   [
    "C13",
    "x[BRIOCHE]"
   ],
   [
    "C14",
    "x[BRIOCHE DE NOEL]"
   ],
   [
    "C15",
    "x[BRIOCHETTE]"
   ],
   [
    "C16",
    "x[BROWNIES]"
   ],
   [
    "C17",
    "x[BUCHE 4PERS]"
   ],
   [
    "C18",
    "x[BUCHE 6PERS]"
   ],
   [
    "C19",
    "x[BUCHE 8PERS]"
   ],
   [
    "C20",
    "x[CAFE OU EAU]"
   ],
   [
    "C21",
    "x[CAKE]"
   ],
   [
    "C22",
    "x[CAMPAGNE]"
   ],
   [
    "C23",
    "x[CARAMEL NOIX]"
   ],
   [
    "C24",
    "x[CEREAL BAGUETTE]"
   ],
   [
    "C25",
    "x[CHAUSSON AUX POMMES]"
   ],
   [
    "C26",
    "x[CHOCOLAT]"
   ],
   [
    "C27",
    "x[CHOU CHANTILLY]"
   ],
   [
    "C28",
    "x[COMPLET]"
   ],
   [
    "C29",
    "x[COOKIE]"
   ],
   [
    "C30",
    "x[COUPE]"
   ],
   [
    "C31",
    "x[CROISSANT]"
   ],
   [
    "C32",
    "x[CROISSANT AMANDES]"
   ],
   [
    "C33",
    "x[CRUMBLE]"
   ],
   [
    "C34",
    "x[CRUMBLECARAMEL OU PISTAE]"
   ],
   [
    "C35",
    "x[DELICETROPICAL]"
   ],
   [
    "C36",
    "x[DEMI BAGUETTE]"
   ],
   [
    "C37",
    "x[DEMI PAIN]"
   ],
   [
    "C38",
    "x[DIVERS BOISSONS]"
   ],
   [
    "C39",
    "x[DIVERS BOULANGERIE]"
   ],
   [
    "C40",
    "x[DIVERS CONFISERIE]"
   ],
   [
    "C41",
    "x[DIVERS PATISSERIE]"
   ],
   [
    "C42",
    "x[DIVERS SANDWICHS]"
   ],
   [
    "C43",
    "x[DIVERS VIENNOISERIE]"
   ],
   [
    "C44",
    "x[DOUCEUR D HIVER]"
   ],
   [
    "C45",
    "x[ECLAIR]"
   ],
   [
    "C46",
    "x[ECLAIR FRAISE PISTACHE]"
   ],
   [
    "C47",
    "x[ENTREMETS]"
   ],
   [
    "C48",
    "x[FICELLE]"
   ],
   [
    "C49",
    "x[FINANCIER]"
   ],
   [
    "C50",
    "x[FINANCIER X5]"
   ],
   [
    "C51",
    "x[FLAN]"
   ],
   [
    "C52",
    "x[FLAN ABRICOT]"
   ],
   [
    "C53",
    "x[FONDANT CHOCOLAT]"
   ],
   [
    "C54",
    "x[FORMULE PATE]"
   ],
   [
    "C55",
    "x[FORMULE PLAT PREPARE]"
   ],
   [
    "C56",
    "x[FORMULE SANDWICH]"
   ],
   [
    "C57",
    "x[FRAISIER]"
   ],
   [
    "C58",
    "x[FRAMBOISIER]"
   ],
   [
    "C59",
    "x[GACHE]"
   ],
   [
    "C60",
    "x[GAL FRANGIPANE 4P]"
   ],
   [
    "C61",
    "x[GAL FRANGIPANE 6P]"
   ],
   [
    "C62",
    "x[GAL POIRE CHOCO 4P]"
   ],
   [
    "C63",
    "x[GAL POIRE CHOCO 6P]"
   ],
   [
    "C64",
    "x[GAL POMME 4P]"
   ],
   [
    "C65",
    "x[GAL POMME 6P]"
   ],
   [
    "C66",
    "x[GALETTE 8 PERS]"
   ],
   [
    "C67",
    "x[GD FAR BRETON]"
   ],
   [
    "C68",
    "x[GD KOUIGN AMANN]"
   ],
   [
    "C69",
    "x[GD NANTAIS]"
   ],
   [
    "C70",
    "x[GD PLATEAU SALE]"
   ],
   [
    "C71",
    "x[GRAND FAR BRETON]"
   ],
   [
    "C72",
    "x[GRANDE SUCETTE]"
   ],
   [
    "C73",
    "x[GUERANDAIS]"
   ],
   [
    "C74",
    "x[KOUIGN AMANN]"
   ],
   [
    "C75",
    "x[MACARON]"
   ],
   [
    "C76",
    "x[MERINGUE]"
   ],
   [
    "C77",
    "x[MILLES FEUILLES]"
   ],
   [
    "C78",
    "x[MOISSON]"
   ],
   [
    "C79",
    "x[NANTAIS]"
   ],
   [
    "C80",
    "x[NID DE POULE]"
   ],
   [
    "C81",
    "x[NOIX JAPONAISE]"
   ],
   [
    "C82",
    "x[PAILLE]"
   ],
   [
    "C83",
    "x[PAIN]"
   ],
   [
    "C84",
    "x[PAIN AU CHOCOLAT]"
   ],
   [
    "C85",
    "x[PAIN AUX RAISINS]"
   ],
   [
    "C86",
    "x[PAIN BANETTE]"
   ],
   [
    "C87",
    "x[PAIN CHOCO AMANDES]"
   ],
   [
    "C88",
    "x[PAIN DE MIE]"
   ],
   [
    "C89",
    "x[PAIN GRAINES]"
   ],
   [
    "C90",
    "x[PAIN NOIR]"
   ],
   [
    "C91",
    "x[PAIN S/SEL]"
   ],
   [
    "C92",
    "x[PAIN SUISSE PEPITO]"
   ],
   [
    "C93",
    "x[PALET BRETON]"
   ],
   [
    "C94",
    "x[PALMIER]"
   ],
   [
    "C95",
    "x[PARIS BREST]"
   ],
   [
    "C96",
    "x[PATES]"
   ],
   [
    "C97",
    "x[PLAQUE TARTE 25P]"
   ],
   [
    "C98",
    "x[PLAT]"
   ],
   [
    "C99",
    "x[PLAT 6.50E]"
   ],
   [
    "C100",
    "x[PLAT 7.00]"
   ],
   [
    "C101",
    "x[PLAT 7.60E]"
   ],
   [
    "C102",
    "x[PLAT 8.30E]"
   ],
   [
    "C103",
    "x[PLATPREPARE5,50]"
   ],
   [
    "C104",
    "x[PLATPREPARE6,00]"
   ],
   [
    "C105",
    "x[PLATPREPARE6,50]"
   ],
   [
    "C106",
    "x[PLATPREPARE7,00]"
   ],
   [
    "C107",
    "x[PT NANTAIS]"
   ],
   [
    "C108",
    "x[PT PLATEAU SALE]"
   ],
   [
    "C109",
    "x[QUIM BREAD]"
   ],
   [
    "C110",
    "x[REDUCTION SUCREES 12]"
   ],
   [
    "C111",
    "x[REDUCTION SUCREES 24]"
   ],
   [
    "C112",
    "x[RELIGIEUSE]"
   ],
   [
    "C113",
    "x[ROYAL]"
   ],
   [
    "C114",
    "x[ROYAL 4P]"
   ],
   [
    "C115",
    "x[ROYAL 6P]"
   ],
   [
    "C116",
    "x[SABLE F  P]"
   ],
   [
    "C117",
    "x[SACHET DE CROUTON]"
   ],
   [
    "C118",
    "x[SACHET DE VIENNOISERIE]"
   ],
   [
    "C119",
    "x[SACHET VIENNOISERIE]"
   ],
   [
    "C120",
    "x[SAND JB]"
   ],
   [
    "C121",
    "x[SAND JB EMMENTAL]"
   ],
   [
    "C122",
    "x[SANDWICH COMPLET]"
   ],
   [
    "C123",
    "x[SAVARIN]"
   ],
   [
    "C124",
    "x[SEIGLE]"
   ],
   [
    "C125",
    "x[SPECIAL BREAD]"
   ],
   [
    "C126",
    "x[SPECIAL BREAD KG]"
   ],
   [
    "C127",
    "x[ST HONORE]"
   ],
   [
    "C128",
    "x[SUCETTE]"
   ],
   [
    "C129",
    "x[TARTE FINE]"
   ],
   [
    "C130",
    "x[TARTE FRAISE 4PER]"
   ],
   [
    "C131",
    "x[TARTE FRAISE 6P]"
   ],
   [
    "C132",
    "x[TARTE FRUITS 4P]"
   ],
   [
    "C133",
    "x[TARTE FRUITS 6P]"
   ],
   [
    "C134",
    "x[TARTELETTE]"
   ],
   [
    "C135",
    "x[TARTELETTE CHOC]"
   ],
   [
    "C136",
    "x[TARTELETTE COCKTAIL]"
   ],
   [
    "C137",
    "x[TARTELETTE FRAISE]"
   ],
   [
    "C138",
    "x[THE]"
   ],
   [
    "C139",
    "x[TRADITIONAL BAGUETTE]"
   ],
   [
    "C140",
    "x[TRAITEUR]"
   ],
   [
    "C141",
    "x[TRIANGLES]"
   ],
   [
    "C142",
    "x[TROIS CHOCOLAT]"
   ],
   [
    "C143",
    "x[TROPEZIENNE]"
   ],
   [
    "C144",
    "x[TROPEZIENNE FRAMBOISE]"
   ],
   [
    "C145",
    "x[TULIPE]"
   ],
   [
    "C146",
    "x[VIENNOISE]"
   ],
   [
    "C147",
    "x[VIK BREAD]"
   ]
  ],
  "primary_mapping": {
   "x_1": "x[12 MACARON]",
   "x_2": "x[ARMORICAIN]",
   "x_3": "x[ARTICLE 295]",
   "x_4": "x[BAGUETTE]",
   "x_5": "x[BAGUETTE APERO]",
   "x_6": "x[BAGUETTE GRAINE]",
   "x_7": "x[BANETTE]",
   "x_8": "x[BANETTINE]",
   "x_9": "x[BOISSON 33CL]",
   "x_10": "x[BOTTEREAU]",
   "x_11": "x[BOULE 200G]",
   "x_12": "x[BOULE 400G]",
   "x_13": "x[BOULE POLKA]",
   "x_14": "x[BRIOCHE]",
   "x_15": "x[BRIOCHE DE NOEL]",
   "x_16": "x[BRIOCHETTE]",
   "x_17": "x[BROWNIES]",
   "x_18": "x[BUCHE 4PERS]",
   "x_19": "x[BUCHE 6PERS]",
   "x_20": "x[BUCHE 8PERS]",
   "x_21": "x[CAFE OU EAU]",
   "x_22": "x[CAKE]",
   "x_23": "x[CAMPAGNE]",
   "x_24": "x[CARAMEL NOIX]",
   "x_25": "x[CEREAL BAGUETTE]",
   "x_26": "x[CHAUSSON AUX POMMES]",
   "x_27": "x[CHOCOLAT]",
   "x_28": "x[CHOU CHANTILLY]",
   "x_29": "x[COMPLET]",
   "x_30": "x[COOKIE]",
   "x_31": "x[COUPE]",
   "x_32": "x[CROISSANT]",
   "x_33": "x[CROISSANT AMANDES]",
   "x_34": "x[CRUMBLE]",
   "x_35": "x[CRUMBLECARAMEL OU PISTAE]",
   "x_36": "x[DELICETROPICAL]",
   "x_37": "x[DEMI BAGUETTE]",
   "x_38": "x[DEMI PAIN]",
   "x_39": "x[DIVERS BOISSONS]",
   "x_40": "x[DIVERS BOULANGERIE]",
   "x_41": "x[DIVERS CONFISERIE]",
   "x_42": "x[DIVERS PATISSERIE]",
   "x_43": "x[DIVERS SANDWICHS]",
   "x_44": "x[DIVERS VIENNOISERIE]",
   "x_45": "x[DOUCEUR D HIVER]",
   "x_46": "x[ECLAIR]",
   "x_47": "x[ECLAIR FRAISE PISTACHE]",
   "x_48": "x[ENTREMETS]",
   "x_49": "x[FICELLE]",
   "x_50": "x[FINANCIER]",
   "x_51": "x[FINANCIER X5]",
   "x_52": "x[FLAN]",
   "x_53": "x[FLAN ABRICOT]",
   "x_54": "x[FONDANT CHOCOLAT]",
   "x_55": "x[FORMULE PATE]",
   "x_56": "x[FORMULE PLAT PREPARE]",
   "x_57": "x[FORMULE SANDWICH]",
   "x_58": "x[FRAISIER]",
   "x_59": "x[FRAMBOISIER]",
   "x_60": "x[GACHE]",
   "x_61": "x[GAL FRANGIPANE 4P]",
   "x_62": "x[GAL FRANGIPANE 6P]",
   "x_63": "x[GAL POIRE CHOCO 4P]",
   "x_64": "x[GAL POIRE CHOCO 6P]",
   "x_65": "x[GAL POMME 4P]",
   "x_66": "x[GAL POMME 6P]",
   "x_67": "x[GALETTE 8 PERS]",
   "x_68": "x[GD FAR BRETON]",
   "x_69": "x[GD KOUIGN AMANN]",
   "x_70": "x[GD NANTAIS]",
   "x_71": "x[GD PLATEAU SALE]",
   "x_72": "x[GRAND FAR BRETON]",
   "x_73": "x[GRANDE SUCETTE]",
   "x_74": "x[GUERANDAIS]",
   "x_75": "x[KOUIGN AMANN]",
   "x_76": "x[MACARON]",
   "x_77": "x[MERINGUE]",
   "x_78": "x[MILLES FEUILLES]",
   "x_79": "x[MOISSON]",
   "x_80": "x[NANTAIS]",
   "x_81": "x[NID DE POULE]",
   "x_82": "x[NOIX JAPONAISE]",
   "x_83": "x[PAILLE]",
   "x_84": "x[PAIN]",
   "x_85": "x[PAIN AU CHOCOLAT]",
   "x_86": "x[PAIN AUX RAISINS]",
   "x_87": "x[PAIN BANETTE]",
   "x_88": "x[PAIN CHOCO AMANDES]",
   "x_89": "x[PAIN DE MIE]",
   "x_90": "x[PAIN GRAINES]",
   "x_91": "x[PAIN NOIR]",
   "x_92": "x[PAIN S/SEL]",
   "x_93": "x[PAIN SUISSE PEPITO]",
   "x_94": "x[PALET BRETON]",
   "x_95": "x[PALMIER]",
   "x_96": "x[PARIS BREST]",
   "x_97": "x[PATES]",
   "x_98": "x[PLAQUE TARTE 25P]",
   "x_99": "x[PLAT]",
   "x_100": "x[PLAT 6.50E]",
   "x_101": "x[PLAT 7.00]",
   "x_102": "x[PLAT 7.60E]",
   "x_103": "x[PLAT 8.30E]",
   "x_104": "x[PLATPREPARE5,50]",
   "x_105": "x[PLATPREPARE6,00]",
   "x_106": "x[PLATPREPARE6,50]",
   "x_107": "x[PLATPREPARE7,00]",
   "x_108": "x[PT NANTAIS]",
   "x_109": "x[PT PLATEAU SALE]",
   "x_110": "x[QUIM BREAD]",
   "x_111": "x[REDUCTION SUCREES 12]",
   "x_112": "x[REDUCTION SUCREES 24]",
   "x_113": "x[RELIGIEUSE]",
   "x_114": "x[ROYAL]",
   "x_115": "x[ROYAL 4P]",
   "x_116": "x[ROYAL 6P]",
   "x_117": "x[SABLE F  P]",
   "x_118": "x[SACHET DE CROUTON]",
   "x_119": "x[SACHET DE VIENNOISERIE]",
   "x_120": "x[SACHET VIENNOISERIE]",
   "x_121": "x[SAND JB]",
   "x_122": "x[SAND JB EMMENTAL]",
   "x_123": "x[SANDWICH COMPLET]",
   "x_124": "x[SAVARIN]",
   "x_125": "x[SEIGLE]",
   "x_126": "x[SPECIAL BREAD]",
   "x_127": "x[SPECIAL BREAD KG]",
   "x_128": "x[ST HONORE]",
   "x_129": "x[SUCETTE]",
   "x_130": "x[TARTE FINE]",
   "x_131": "x[TARTE FRAISE 4PER]",
   "x_132": "x[TARTE FRAISE 6P]",
   "x_133": "x[TARTE FRUITS 4P]",
   "x_134": "x[TARTE FRUITS 6P]",
   "x_135": "x[TARTELETTE]",
   "x_136": "x[TARTELETTE CHOC]",
   "x_137": "x[TARTELETTE COCKTAIL]",
   "x_138": "x[TARTELETTE FRAISE]",
   "x_139": "x[THE]",
   "x_140": "x[TRADITIONAL BAGUETTE]",
   "x_141": "x[TRAITEUR]",
   "x_142": "x[TRIANGLES]",
   "x_143": "x[TROIS CHOCOLAT]",
   "x_144": "x[TROPEZIENNE]",
   "x_145": "x[TROPEZIENNE FRAMBOISE]",
   "x_146": "x[TULIPE]",
   "x_147": "x[VIENNOISE]",
   "x_148": "x[VIK BREAD]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 914600.2500000002,
   "variables": [
    [
     "C0",
     135.0
    ],
    [
     "C1",
     5.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     38573.0
    ],
    [
     "C4",
     128.0
    ],
    [
     "C5",
     3390.0
    ],
    [
     "C6",
     38972.0
    ],
    [
     "C7",
     5930.0
    ],
    [
     "C8",
     3376.0
    ],
    [
     "C9",
     570.0
    ],
    [
     "C10",
     5823.0
    ],
    [
     "C11",
     9011.0
    ],
    [
     "C12",
     1034.0
    ],
    [
     "C13",
     3351.0
    ],
    [
     "C14",
     38.0
    ],
    [
     "C15",
     143.0
    ],
    [
     "C16",
     82.0
    ],
    [
     "C17",
     21.0
    ],
    [
     "C18",
     16.0
    ],
    [
     "C19",
     3.0
    ],
    [
     "C20",
     3425.0
    ],
    [
     "C21",
     2.0
    ],
    [
     "C22",
     8314.0
    ],
    [
     "C23",
     174.0
    ],
    [
     "C24",
     12788.0
    ],
    [
     "C25",
     3413.0
    ],
    [
     "C26",
     155.0
    ],
    [
     "C27",
     482.0
    ],
    [
     "C28",
     6722.0
    ],
    [
     "C29",
     6136.0
    ],
    [
     "C30",
     44306.0
    ],
    [
     "C31",
     45491.0
    ],
    [
     "C32",
     4196.0
    ],
    [
     "C33",
     17.0
    ],
    [
     "C34",
     5.0
    ],
    [
     "C35",
     105.0
    ],
    [
     "C36",
     2289.0
    ],
    [
     "C37",
     326.0
    ],
    [
     "C38",
     170.0
    ],
    [
     "C39",
     631.0
    ],
    [
     "C40",
     989.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     468.0
    ],
    [
     "C43",
     1653.0
    ],
    [
     "C44",
     2.0
    ],
    [
     "C45",
     5929.0
    ],
    [
     "C46",
     38.0
    ],
    [
     "C47",
     62.0
    ],
    [
     "C48",
     6121.0
    ],
    [
     "C49",
     36.0
    ],
    [
     "C50",
     2303.0
    ],
    [
     "C51",
     1922.0
    ],
    [
     "C52",
     1377.0
    ],
    [
     "C53",
     549.0
    ],
    [
     "C54",
     4.0
    ],
    [
     "C55",
     21.0
    ],
    [
     "C56",
     9445.0
    ],
    [
     "C57",
     998.0
    ],
    [
     "C58",
     178.0
    ],
    [
     "C59",
     256.0
    ],
    [
     "C60",
     415.0
    ],
    [
     "C61",
     251.0
    ],
    [
     "C62",
     48.0
    ],
    [
     "C63",
     30.0
    ],
    [
     "C64",
     196.0
    ],
    [
     "C65",
     123.0
    ],
    [
     "C66",
     4.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     1850.0
    ],
    [
     "C69",
     201.0
    ],
    [
     "C70",
     4.0
    ],
    [
     "C71",
     2862.0
    ],
    [
     "C72",
     564.0
    ],
    [
     "C73",
     6.0
    ],
    [
     "C74",
     3922.0
    ],
    [
     "C75",
     665.0
    ],
    [
     "C76",
     116.0
    ],
    [
     "C77",
     1839.0
    ],
    [
     "C78",
     6466.0
    ],
    [
     "C79",
     1006.0
    ],
    [
     "C80",
     53.0
    ],
    [
     "C81",
     793.0
    ],
    [
     "C82",
     238.0
    ],
    [
     "C83",
     4454.0
    ],
    [
     "C84",
     39146.0
    ],
    [
     "C85",
     4766.0
    ],
    [
     "C86",
     5758.0
    ],
    [
     "C87",
     3412.0
    ],
    [
     "C88",
     72.0
    ],
    [
     "C89",
     6.0
    ],
    [
     "C90",
     2.0
    ],
    [
     "C91",
     265.0
    ],
    [
     "C92",
     39.0
    ],
    [
     "C93",
     439.0
    ],
    [
     "C94",
     11.0
    ],
    [
     "C95",
     2227.0
    ],
    [
     "C96",
     4.0
    ],
    [
     "C97",
     3.0
    ],
    [
     "C98",
     17.0
    ],
    [
     "C99",
     1.0
    ],
    [
     "C100",
     202.0
    ],
    [
     "C101",
     567.0
    ],
    [
     "C102",
     263.0
    ],
    [
     "C103",
     6.0
    ],
    [
     "C104",
     2.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     30.0
    ],
    [
     "C107",
     1274.0
    ],
    [
     "C108",
     7.0
    ],
    [
     "C109",
     3623.0
    ],
    [
     "C110",
     16.0
    ],
    [
     "C111",
     2.0
    ],
    [
     "C112",
     205.0
    ],
    [
     "C113",
     478.0
    ],
    [
     "C114",
     140.0
    ],
    [
     "C115",
     128.0
    ],
    [
     "C116",
     215.0
    ],
    [
     "C117",
     232.0
    ],
    [
     "C118",
     2.0
    ],
    [
     "C119",
     1003.0
    ],
    [
     "C120",
     10.0
    ],
    [
     "C121",
     3305.0
    ],
    [
     "C122",
     5172.0
    ],
    [
     "C123",
     815.0
    ],
    [
     "C124",
     2761.0
    ],
    [
     "C125",
     10647.0
    ],
    [
     "C126",
     912.0
    ],
    [
     "C127",
     197.0
    ],
    [
     "C128",
     1701.0
    ],
    [
     "C129",
     125.0
    ],
    [
     "C130",
     314.0
    ],
    [
     "C131",
     242.0
    ],
    [
     "C132",
     736.0
    ],
    [
     "C133",
     555.0
    ],
    [
     "C134",
     8218.0
    ],
    [
     "C135",
     112.0
    ],
    [
     "C136",
     3.0
    ],
    [
     "C137",
     1346.0
    ],
    [
     "C138",
     13.0
    ],
    [
     "C139",
     193745.0
    ],
    [
     "C140",
     2765.0
    ],
    [
     "C141",
     147.0
    ],
    [
     "C142",
     2.0
    ],
    [
     "C143",
     881.0
    ],
    [
     "C144",
     59.0
    ],
    [
     "C145",
     4.0
    ],
    [
     "C146",
     404.0
    ],
    [
     "C147",
     6810.0
    ]
   ],
   "error": null
  }
 },
 "OR-038": {
  "source_code_sha256": "73fbdb85f077df576f6a2eda79a328a264c0d7087445a1327c2ca763ad2f9897",
  "original_solution_sha256": "56f3ba16bd12b7cb973671f35c19f522a96d8b83550add455a4eafcfa06eade3",
  "bindings": [
   [
    "C0",
    "x[Compact Cars]"
   ],
   [
    "C1",
    "x[Hybrid Vehicles]"
   ],
   [
    "C2",
    "x[Luxury Sedans]"
   ],
   [
    "C3",
    "x[Sedans]"
   ],
   [
    "C4",
    "x[Pickup Trucks]"
   ],
   [
    "C5",
    "x[SUVs]"
   ],
   [
    "C6",
    "x[Electric Vehicles]"
   ],
   [
    "C7",
    "x[Vans]"
   ],
   [
    "C8",
    "x[Trucks]"
   ],
   [
    "C9",
    "x[Sports Cars]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Sedans]",
   "x_1_0": "x[SUVs]",
   "x_2_0": "x[Electric Vehicles]",
   "x_3_0": "x[Hybrid Vehicles]",
   "x_4_0": "x[Trucks]",
   "x_5_0": "x[Sports Cars]",
   "x_6_0": "x[Compact Cars]",
   "x_7_0": "x[Luxury Sedans]",
   "x_8_0": "x[Vans]",
   "x_9_0": "x[Pickup Trucks]"
  },
  "mapping_basis": "reviewed_oss_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1314500.0,
   "variables": [
    [
     "C0",
     110.0
    ],
    [
     "C1",
     90.0
    ],
    [
     "C2",
     40.0
    ],
    [
     "C3",
     100.0
    ],
    [
     "C4",
     35.0
    ],
    [
     "C5",
     80.0
    ],
    [
     "C6",
     120.0
    ],
    [
     "C7",
     60.0
    ],
    [
     "C8",
     50.0
    ],
    [
     "C9",
     30.0
    ]
   ],
   "error": null
  }
 },
 "OR-042": {
  "source_code_sha256": "582c1445466df443005e5f8ac39de705607c9f81c39955673d8ed0a2aea9aa41",
  "original_solution_sha256": "e1410690e75d6bd907ab2e270b0c40b33929ecd14066b4710cc1031e9e6ecbc6",
  "bindings": [
   [
    "C0",
    "x[NSAIDs]"
   ],
   [
    "C1",
    "x[Antirheumatic Drugs]"
   ],
   [
    "C2",
    "x[Acetic Acid Derivatives]"
   ],
   [
    "C3",
    "x[Antibiotics]"
   ],
   [
    "C4",
    "x[Antiviral Drugs]"
   ],
   [
    "C5",
    "x[Antifungal Agents]"
   ],
   [
    "C6",
    "x[Antidepressants]"
   ],
   [
    "C7",
    "x[Antipsychotics]"
   ],
   [
    "C8",
    "x[Antihistamines]"
   ],
   [
    "C9",
    "x[Corticosteroids]"
   ],
   [
    "C10",
    "x[Beta Blockers]"
   ],
   [
    "C11",
    "x[Calcium Channel Blockers]"
   ],
   [
    "C12",
    "x[ACE Inhibitors]"
   ],
   [
    "C13",
    "x[Angiotensin II Receptor Blockers]"
   ],
   [
    "C14",
    "x[Diuretics]"
   ],
   [
    "C15",
    "x[Statins]"
   ],
   [
    "C16",
    "x[Insulin]"
   ],
   [
    "C17",
    "x[Anticoagulants]"
   ],
   [
    "C18",
    "x[Antiepileptic Drugs]"
   ],
   [
    "C19",
    "x[Antiemetics]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[NSAIDs]",
   "x_1_0": "x[Antirheumatic Drugs]",
   "x_2_0": "x[Acetic Acid Derivatives]",
   "x_3_0": "x[Antibiotics]",
   "x_4_0": "x[Antiviral Drugs]",
   "x_5_0": "x[Antifungal Agents]",
   "x_6_0": "x[Antidepressants]",
   "x_7_0": "x[Antipsychotics]",
   "x_8_0": "x[Antihistamines]",
   "x_9_0": "x[Corticosteroids]",
   "x_10_0": "x[Beta Blockers]",
   "x_11_0": "x[Calcium Channel Blockers]",
   "x_12_0": "x[ACE Inhibitors]",
   "x_13_0": "x[Angiotensin II Receptor Blockers]",
   "x_14_0": "x[Diuretics]",
   "x_15_0": "x[Statins]",
   "x_16_0": "x[Insulin]",
   "x_17_0": "x[Anticoagulants]",
   "x_18_0": "x[Antiepileptic Drugs]",
   "x_19_0": "x[Antiemetics]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 81918.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     82.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-046": {
  "source_code_sha256": "cb98f22b5e043f9dea44119b6125d2ee514d6fb04a0f7bb40ea51db72ff5fb25",
  "original_solution_sha256": "d920562de6283a4e2a31bc0cf85f74ac501f1b025af2b0aa47247e72425f650c",
  "bindings": [
   [
    "C0",
    "x[Spinach]"
   ],
   [
    "C1",
    "x[Shiitake Mushrooms]"
   ],
   [
    "C2",
    "x[Apples]"
   ],
   [
    "C3",
    "x[Carrots]"
   ],
   [
    "C4",
    "x[Basil]"
   ],
   [
    "C5",
    "x[Potatoes]"
   ],
   [
    "C6",
    "x[Green Beans]"
   ],
   [
    "C7",
    "x[Blueberries]"
   ],
   [
    "C8",
    "x[Oranges]"
   ],
   [
    "C9",
    "x[Watermelons]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[Spinach]",
   "x_1_0": "x[Shiitake Mushrooms]",
   "x_2_0": "x[Apples]",
   "x_3_0": "x[Carrots]",
   "x_4_0": "x[Basil]",
   "x_5_0": "x[Potatoes]",
   "x_6_0": "x[Green Beans]",
   "x_7_0": "x[Blueberries]",
   "x_8_0": "x[Oranges]",
   "x_9_0": "x[Watermelons]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 212.0,
   "variables": [
    [
     "C0",
     3.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     2.0
    ],
    [
     "C9",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-051": {
  "source_code_sha256": "27ee4b2fd3eac444fc0c72c518a1c16400aca1b9e0db942e547ef19baaae1430",
  "original_solution_sha256": "a5d9e9cf3740a4a52ee62291b2100d1784929af6e09424f3743684a44b840974",
  "bindings": [
   [
    "C0",
    "x[1,Espresso Beans]"
   ],
   [
    "C1",
    "x[1,Colombian Roast]"
   ],
   [
    "C2",
    "x[1,Arabica Blend]"
   ],
   [
    "C3",
    "x[1,French Roast]"
   ],
   [
    "C4",
    "x[1,Italian Roast]"
   ],
   [
    "C5",
    "x[1,House Blend]"
   ],
   [
    "C6",
    "x[1,Sumatra Coffee]"
   ],
   [
    "C7",
    "x[1,Mocha Java]"
   ],
   [
    "C8",
    "x[1,Hazelnut Flavor]"
   ],
   [
    "C9",
    "x[1,Caramel Blend]"
   ],
   [
    "C10",
    "x[1,Vanilla Flavor]"
   ],
   [
    "C11",
    "x[1,Cappuccino Mix]"
   ],
   [
    "C12",
    "x[1,Pumpkin Spice]"
   ],
   [
    "C13",
    "x[1,Decaf Roast]"
   ],
   [
    "C14",
    "x[1,Organic Roast]"
   ],
   [
    "C15",
    "x[1,Cold Brew]"
   ],
   [
    "C16",
    "x[1,Peruvian Blend]"
   ],
   [
    "C17",
    "x[1,Kenyan AA]"
   ],
   [
    "C18",
    "x[2,Espresso Beans]"
   ],
   [
    "C19",
    "x[2,Colombian Roast]"
   ],
   [
    "C20",
    "x[2,Arabica Blend]"
   ],
   [
    "C21",
    "x[2,French Roast]"
   ],
   [
    "C22",
    "x[2,Italian Roast]"
   ],
   [
    "C23",
    "x[2,House Blend]"
   ],
   [
    "C24",
    "x[2,Sumatra Coffee]"
   ],
   [
    "C25",
    "x[2,Mocha Java]"
   ],
   [
    "C26",
    "x[2,Hazelnut Flavor]"
   ],
   [
    "C27",
    "x[2,Caramel Blend]"
   ],
   [
    "C28",
    "x[2,Vanilla Flavor]"
   ],
   [
    "C29",
    "x[2,Cappuccino Mix]"
   ],
   [
    "C30",
    "x[2,Pumpkin Spice]"
   ],
   [
    "C31",
    "x[2,Decaf Roast]"
   ],
   [
    "C32",
    "x[2,Organic Roast]"
   ],
   [
    "C33",
    "x[2,Cold Brew]"
   ],
   [
    "C34",
    "x[2,Peruvian Blend]"
   ],
   [
    "C35",
    "x[2,Kenyan AA]"
   ],
   [
    "C36",
    "x[3,Espresso Beans]"
   ],
   [
    "C37",
    "x[3,Colombian Roast]"
   ],
   [
    "C38",
    "x[3,Arabica Blend]"
   ],
   [
    "C39",
    "x[3,French Roast]"
   ],
   [
    "C40",
    "x[3,Italian Roast]"
   ],
   [
    "C41",
    "x[3,House Blend]"
   ],
   [
    "C42",
    "x[3,Sumatra Coffee]"
   ],
   [
    "C43",
    "x[3,Mocha Java]"
   ],
   [
    "C44",
    "x[3,Hazelnut Flavor]"
   ],
   [
    "C45",
    "x[3,Caramel Blend]"
   ],
   [
    "C46",
    "x[3,Vanilla Flavor]"
   ],
   [
    "C47",
    "x[3,Cappuccino Mix]"
   ],
   [
    "C48",
    "x[3,Pumpkin Spice]"
   ],
   [
    "C49",
    "x[3,Decaf Roast]"
   ],
   [
    "C50",
    "x[3,Organic Roast]"
   ],
   [
    "C51",
    "x[3,Cold Brew]"
   ],
   [
    "C52",
    "x[3,Peruvian Blend]"
   ],
   [
    "C53",
    "x[3,Kenyan AA]"
   ],
   [
    "C54",
    "x[4,Espresso Beans]"
   ],
   [
    "C55",
    "x[4,Colombian Roast]"
   ],
   [
    "C56",
    "x[4,Arabica Blend]"
   ],
   [
    "C57",
    "x[4,French Roast]"
   ],
   [
    "C58",
    "x[4,Italian Roast]"
   ],
   [
    "C59",
    "x[4,House Blend]"
   ],
   [
    "C60",
    "x[4,Sumatra Coffee]"
   ],
   [
    "C61",
    "x[4,Mocha Java]"
   ],
   [
    "C62",
    "x[4,Hazelnut Flavor]"
   ],
   [
    "C63",
    "x[4,Caramel Blend]"
   ],
   [
    "C64",
    "x[4,Vanilla Flavor]"
   ],
   [
    "C65",
    "x[4,Cappuccino Mix]"
   ],
   [
    "C66",
    "x[4,Pumpkin Spice]"
   ],
   [
    "C67",
    "x[4,Decaf Roast]"
   ],
   [
    "C68",
    "x[4,Organic Roast]"
   ],
   [
    "C69",
    "x[4,Cold Brew]"
   ],
   [
    "C70",
    "x[4,Peruvian Blend]"
   ],
   [
    "C71",
    "x[4,Kenyan AA]"
   ],
   [
    "C72",
    "x[5,Espresso Beans]"
   ],
   [
    "C73",
    "x[5,Colombian Roast]"
   ],
   [
    "C74",
    "x[5,Arabica Blend]"
   ],
   [
    "C75",
    "x[5,French Roast]"
   ],
   [
    "C76",
    "x[5,Italian Roast]"
   ],
   [
    "C77",
    "x[5,House Blend]"
   ],
   [
    "C78",
    "x[5,Sumatra Coffee]"
   ],
   [
    "C79",
    "x[5,Mocha Java]"
   ],
   [
    "C80",
    "x[5,Hazelnut Flavor]"
   ],
   [
    "C81",
    "x[5,Caramel Blend]"
   ],
   [
    "C82",
    "x[5,Vanilla Flavor]"
   ],
   [
    "C83",
    "x[5,Cappuccino Mix]"
   ],
   [
    "C84",
    "x[5,Pumpkin Spice]"
   ],
   [
    "C85",
    "x[5,Decaf Roast]"
   ],
   [
    "C86",
    "x[5,Organic Roast]"
   ],
   [
    "C87",
    "x[5,Cold Brew]"
   ],
   [
    "C88",
    "x[5,Peruvian Blend]"
   ],
   [
    "C89",
    "x[5,Kenyan AA]"
   ],
   [
    "C90",
    "x[6,Espresso Beans]"
   ],
   [
    "C91",
    "x[6,Colombian Roast]"
   ],
   [
    "C92",
    "x[6,Arabica Blend]"
   ],
   [
    "C93",
    "x[6,French Roast]"
   ],
   [
    "C94",
    "x[6,Italian Roast]"
   ],
   [
    "C95",
    "x[6,House Blend]"
   ],
   [
    "C96",
    "x[6,Sumatra Coffee]"
   ],
   [
    "C97",
    "x[6,Mocha Java]"
   ],
   [
    "C98",
    "x[6,Hazelnut Flavor]"
   ],
   [
    "C99",
    "x[6,Caramel Blend]"
   ],
   [
    "C100",
    "x[6,Vanilla Flavor]"
   ],
   [
    "C101",
    "x[6,Cappuccino Mix]"
   ],
   [
    "C102",
    "x[6,Pumpkin Spice]"
   ],
   [
    "C103",
    "x[6,Decaf Roast]"
   ],
   [
    "C104",
    "x[6,Organic Roast]"
   ],
   [
    "C105",
    "x[6,Cold Brew]"
   ],
   [
    "C106",
    "x[6,Peruvian Blend]"
   ],
   [
    "C107",
    "x[6,Kenyan AA]"
   ],
   [
    "C108",
    "x[7,Espresso Beans]"
   ],
   [
    "C109",
    "x[7,Colombian Roast]"
   ],
   [
    "C110",
    "x[7,Arabica Blend]"
   ],
   [
    "C111",
    "x[7,French Roast]"
   ],
   [
    "C112",
    "x[7,Italian Roast]"
   ],
   [
    "C113",
    "x[7,House Blend]"
   ],
   [
    "C114",
    "x[7,Sumatra Coffee]"
   ],
   [
    "C115",
    "x[7,Mocha Java]"
   ],
   [
    "C116",
    "x[7,Hazelnut Flavor]"
   ],
   [
    "C117",
    "x[7,Caramel Blend]"
   ],
   [
    "C118",
    "x[7,Vanilla Flavor]"
   ],
   [
    "C119",
    "x[7,Cappuccino Mix]"
   ],
   [
    "C120",
    "x[7,Pumpkin Spice]"
   ],
   [
    "C121",
    "x[7,Decaf Roast]"
   ],
   [
    "C122",
    "x[7,Organic Roast]"
   ],
   [
    "C123",
    "x[7,Cold Brew]"
   ],
   [
    "C124",
    "x[7,Peruvian Blend]"
   ],
   [
    "C125",
    "x[7,Kenyan AA]"
   ],
   [
    "C126",
    "x[8,Espresso Beans]"
   ],
   [
    "C127",
    "x[8,Colombian Roast]"
   ],
   [
    "C128",
    "x[8,Arabica Blend]"
   ],
   [
    "C129",
    "x[8,French Roast]"
   ],
   [
    "C130",
    "x[8,Italian Roast]"
   ],
   [
    "C131",
    "x[8,House Blend]"
   ],
   [
    "C132",
    "x[8,Sumatra Coffee]"
   ],
   [
    "C133",
    "x[8,Mocha Java]"
   ],
   [
    "C134",
    "x[8,Hazelnut Flavor]"
   ],
   [
    "C135",
    "x[8,Caramel Blend]"
   ],
   [
    "C136",
    "x[8,Vanilla Flavor]"
   ],
   [
    "C137",
    "x[8,Cappuccino Mix]"
   ],
   [
    "C138",
    "x[8,Pumpkin Spice]"
   ],
   [
    "C139",
    "x[8,Decaf Roast]"
   ],
   [
    "C140",
    "x[8,Organic Roast]"
   ],
   [
    "C141",
    "x[8,Cold Brew]"
   ],
   [
    "C142",
    "x[8,Peruvian Blend]"
   ],
   [
    "C143",
    "x[8,Kenyan AA]"
   ],
   [
    "C144",
    "x[9,Espresso Beans]"
   ],
   [
    "C145",
    "x[9,Colombian Roast]"
   ],
   [
    "C146",
    "x[9,Arabica Blend]"
   ],
   [
    "C147",
    "x[9,French Roast]"
   ],
   [
    "C148",
    "x[9,Italian Roast]"
   ],
   [
    "C149",
    "x[9,House Blend]"
   ],
   [
    "C150",
    "x[9,Sumatra Coffee]"
   ],
   [
    "C151",
    "x[9,Mocha Java]"
   ],
   [
    "C152",
    "x[9,Hazelnut Flavor]"
   ],
   [
    "C153",
    "x[9,Caramel Blend]"
   ],
   [
    "C154",
    "x[9,Vanilla Flavor]"
   ],
   [
    "C155",
    "x[9,Cappuccino Mix]"
   ],
   [
    "C156",
    "x[9,Pumpkin Spice]"
   ],
   [
    "C157",
    "x[9,Decaf Roast]"
   ],
   [
    "C158",
    "x[9,Organic Roast]"
   ],
   [
    "C159",
    "x[9,Cold Brew]"
   ],
   [
    "C160",
    "x[9,Peruvian Blend]"
   ],
   [
    "C161",
    "x[9,Kenyan AA]"
   ],
   [
    "C162",
    "x[10,Espresso Beans]"
   ],
   [
    "C163",
    "x[10,Colombian Roast]"
   ],
   [
    "C164",
    "x[10,Arabica Blend]"
   ],
   [
    "C165",
    "x[10,French Roast]"
   ],
   [
    "C166",
    "x[10,Italian Roast]"
   ],
   [
    "C167",
    "x[10,House Blend]"
   ],
   [
    "C168",
    "x[10,Sumatra Coffee]"
   ],
   [
    "C169",
    "x[10,Mocha Java]"
   ],
   [
    "C170",
    "x[10,Hazelnut Flavor]"
   ],
   [
    "C171",
    "x[10,Caramel Blend]"
   ],
   [
    "C172",
    "x[10,Vanilla Flavor]"
   ],
   [
    "C173",
    "x[10,Cappuccino Mix]"
   ],
   [
    "C174",
    "x[10,Pumpkin Spice]"
   ],
   [
    "C175",
    "x[10,Decaf Roast]"
   ],
   [
    "C176",
    "x[10,Organic Roast]"
   ],
   [
    "C177",
    "x[10,Cold Brew]"
   ],
   [
    "C178",
    "x[10,Peruvian Blend]"
   ],
   [
    "C179",
    "x[10,Kenyan AA]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Espresso Beans]",
   "x_0_1": "x[2,Espresso Beans]",
   "x_0_2": "x[3,Espresso Beans]",
   "x_0_3": "x[4,Espresso Beans]",
   "x_0_4": "x[5,Espresso Beans]",
   "x_0_5": "x[6,Espresso Beans]",
   "x_0_6": "x[7,Espresso Beans]",
   "x_0_7": "x[8,Espresso Beans]",
   "x_0_8": "x[9,Espresso Beans]",
   "x_0_9": "x[10,Espresso Beans]",
   "x_1_0": "x[1,Colombian Roast]",
   "x_1_1": "x[2,Colombian Roast]",
   "x_1_2": "x[3,Colombian Roast]",
   "x_1_3": "x[4,Colombian Roast]",
   "x_1_4": "x[5,Colombian Roast]",
   "x_1_5": "x[6,Colombian Roast]",
   "x_1_6": "x[7,Colombian Roast]",
   "x_1_7": "x[8,Colombian Roast]",
   "x_1_8": "x[9,Colombian Roast]",
   "x_1_9": "x[10,Colombian Roast]",
   "x_2_0": "x[1,Arabica Blend]",
   "x_2_1": "x[2,Arabica Blend]",
   "x_2_2": "x[3,Arabica Blend]",
   "x_2_3": "x[4,Arabica Blend]",
   "x_2_4": "x[5,Arabica Blend]",
   "x_2_5": "x[6,Arabica Blend]",
   "x_2_6": "x[7,Arabica Blend]",
   "x_2_7": "x[8,Arabica Blend]",
   "x_2_8": "x[9,Arabica Blend]",
   "x_2_9": "x[10,Arabica Blend]",
   "x_3_0": "x[1,French Roast]",
   "x_3_1": "x[2,French Roast]",
   "x_3_2": "x[3,French Roast]",
   "x_3_3": "x[4,French Roast]",
   "x_3_4": "x[5,French Roast]",
   "x_3_5": "x[6,French Roast]",
   "x_3_6": "x[7,French Roast]",
   "x_3_7": "x[8,French Roast]",
   "x_3_8": "x[9,French Roast]",
   "x_3_9": "x[10,French Roast]",
   "x_4_0": "x[1,Italian Roast]",
   "x_4_1": "x[2,Italian Roast]",
   "x_4_2": "x[3,Italian Roast]",
   "x_4_3": "x[4,Italian Roast]",
   "x_4_4": "x[5,Italian Roast]",
   "x_4_5": "x[6,Italian Roast]",
   "x_4_6": "x[7,Italian Roast]",
   "x_4_7": "x[8,Italian Roast]",
   "x_4_8": "x[9,Italian Roast]",
   "x_4_9": "x[10,Italian Roast]",
   "x_5_0": "x[1,House Blend]",
   "x_5_1": "x[2,House Blend]",
   "x_5_2": "x[3,House Blend]",
   "x_5_3": "x[4,House Blend]",
   "x_5_4": "x[5,House Blend]",
   "x_5_5": "x[6,House Blend]",
   "x_5_6": "x[7,House Blend]",
   "x_5_7": "x[8,House Blend]",
   "x_5_8": "x[9,House Blend]",
   "x_5_9": "x[10,House Blend]",
   "x_6_0": "x[1,Sumatra Coffee]",
   "x_6_1": "x[2,Sumatra Coffee]",
   "x_6_2": "x[3,Sumatra Coffee]",
   "x_6_3": "x[4,Sumatra Coffee]",
   "x_6_4": "x[5,Sumatra Coffee]",
   "x_6_5": "x[6,Sumatra Coffee]",
   "x_6_6": "x[7,Sumatra Coffee]",
   "x_6_7": "x[8,Sumatra Coffee]",
   "x_6_8": "x[9,Sumatra Coffee]",
   "x_6_9": "x[10,Sumatra Coffee]",
   "x_7_0": "x[1,Mocha Java]",
   "x_7_1": "x[2,Mocha Java]",
   "x_7_2": "x[3,Mocha Java]",
   "x_7_3": "x[4,Mocha Java]",
   "x_7_4": "x[5,Mocha Java]",
   "x_7_5": "x[6,Mocha Java]",
   "x_7_6": "x[7,Mocha Java]",
   "x_7_7": "x[8,Mocha Java]",
   "x_7_8": "x[9,Mocha Java]",
   "x_7_9": "x[10,Mocha Java]",
   "x_8_0": "x[1,Hazelnut Flavor]",
   "x_8_1": "x[2,Hazelnut Flavor]",
   "x_8_2": "x[3,Hazelnut Flavor]",
   "x_8_3": "x[4,Hazelnut Flavor]",
   "x_8_4": "x[5,Hazelnut Flavor]",
   "x_8_5": "x[6,Hazelnut Flavor]",
   "x_8_6": "x[7,Hazelnut Flavor]",
   "x_8_7": "x[8,Hazelnut Flavor]",
   "x_8_8": "x[9,Hazelnut Flavor]",
   "x_8_9": "x[10,Hazelnut Flavor]",
   "x_9_0": "x[1,Caramel Blend]",
   "x_9_1": "x[2,Caramel Blend]",
   "x_9_2": "x[3,Caramel Blend]",
   "x_9_3": "x[4,Caramel Blend]",
   "x_9_4": "x[5,Caramel Blend]",
   "x_9_5": "x[6,Caramel Blend]",
   "x_9_6": "x[7,Caramel Blend]",
   "x_9_7": "x[8,Caramel Blend]",
   "x_9_8": "x[9,Caramel Blend]",
   "x_9_9": "x[10,Caramel Blend]",
   "x_10_0": "x[1,Vanilla Flavor]",
   "x_10_1": "x[2,Vanilla Flavor]",
   "x_10_2": "x[3,Vanilla Flavor]",
   "x_10_3": "x[4,Vanilla Flavor]",
   "x_10_4": "x[5,Vanilla Flavor]",
   "x_10_5": "x[6,Vanilla Flavor]",
   "x_10_6": "x[7,Vanilla Flavor]",
   "x_10_7": "x[8,Vanilla Flavor]",
   "x_10_8": "x[9,Vanilla Flavor]",
   "x_10_9": "x[10,Vanilla Flavor]",
   "x_11_0": "x[1,Cappuccino Mix]",
   "x_11_1": "x[2,Cappuccino Mix]",
   "x_11_2": "x[3,Cappuccino Mix]",
   "x_11_3": "x[4,Cappuccino Mix]",
   "x_11_4": "x[5,Cappuccino Mix]",
   "x_11_5": "x[6,Cappuccino Mix]",
   "x_11_6": "x[7,Cappuccino Mix]",
   "x_11_7": "x[8,Cappuccino Mix]",
   "x_11_8": "x[9,Cappuccino Mix]",
   "x_11_9": "x[10,Cappuccino Mix]",
   "x_12_0": "x[1,Pumpkin Spice]",
   "x_12_1": "x[2,Pumpkin Spice]",
   "x_12_2": "x[3,Pumpkin Spice]",
   "x_12_3": "x[4,Pumpkin Spice]",
   "x_12_4": "x[5,Pumpkin Spice]",
   "x_12_5": "x[6,Pumpkin Spice]",
   "x_12_6": "x[7,Pumpkin Spice]",
   "x_12_7": "x[8,Pumpkin Spice]",
   "x_12_8": "x[9,Pumpkin Spice]",
   "x_12_9": "x[10,Pumpkin Spice]",
   "x_13_0": "x[1,Decaf Roast]",
   "x_13_1": "x[2,Decaf Roast]",
   "x_13_2": "x[3,Decaf Roast]",
   "x_13_3": "x[4,Decaf Roast]",
   "x_13_4": "x[5,Decaf Roast]",
   "x_13_5": "x[6,Decaf Roast]",
   "x_13_6": "x[7,Decaf Roast]",
   "x_13_7": "x[8,Decaf Roast]",
   "x_13_8": "x[9,Decaf Roast]",
   "x_13_9": "x[10,Decaf Roast]",
   "x_14_0": "x[1,Organic Roast]",
   "x_14_1": "x[2,Organic Roast]",
   "x_14_2": "x[3,Organic Roast]",
   "x_14_3": "x[4,Organic Roast]",
   "x_14_4": "x[5,Organic Roast]",
   "x_14_5": "x[6,Organic Roast]",
   "x_14_6": "x[7,Organic Roast]",
   "x_14_7": "x[8,Organic Roast]",
   "x_14_8": "x[9,Organic Roast]",
   "x_14_9": "x[10,Organic Roast]",
   "x_15_0": "x[1,Cold Brew]",
   "x_15_1": "x[2,Cold Brew]",
   "x_15_2": "x[3,Cold Brew]",
   "x_15_3": "x[4,Cold Brew]",
   "x_15_4": "x[5,Cold Brew]",
   "x_15_5": "x[6,Cold Brew]",
   "x_15_6": "x[7,Cold Brew]",
   "x_15_7": "x[8,Cold Brew]",
   "x_15_8": "x[9,Cold Brew]",
   "x_15_9": "x[10,Cold Brew]",
   "x_16_0": "x[1,Peruvian Blend]",
   "x_16_1": "x[2,Peruvian Blend]",
   "x_16_2": "x[3,Peruvian Blend]",
   "x_16_3": "x[4,Peruvian Blend]",
   "x_16_4": "x[5,Peruvian Blend]",
   "x_16_5": "x[6,Peruvian Blend]",
   "x_16_6": "x[7,Peruvian Blend]",
   "x_16_7": "x[8,Peruvian Blend]",
   "x_16_8": "x[9,Peruvian Blend]",
   "x_16_9": "x[10,Peruvian Blend]",
   "x_17_0": "x[1,Kenyan AA]",
   "x_17_1": "x[2,Kenyan AA]",
   "x_17_2": "x[3,Kenyan AA]",
   "x_17_3": "x[4,Kenyan AA]",
   "x_17_4": "x[5,Kenyan AA]",
   "x_17_5": "x[6,Kenyan AA]",
   "x_17_6": "x[7,Kenyan AA]",
   "x_17_7": "x[8,Kenyan AA]",
   "x_17_8": "x[9,Kenyan AA]",
   "x_17_9": "x[10,Kenyan AA]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 594900.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     250.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     375.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     1.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     311.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     1.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     1.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     436.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     1.0
    ],
    [
     "C72",
     2.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     280.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     2.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     405.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     1.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     343.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     1.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     -0.0
    ],
    [
     "C138",
     -0.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     468.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     -0.0
    ],
    [
     "C158",
     300.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     325.0
    ],
    [
     "C177",
     -0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-055": {
  "source_code_sha256": "a5d9339c91a0e348c1732c72d49b44e034229ce281bc90c0e13edab3fc524a52",
  "original_solution_sha256": "78f1a854ceafe2eef30ae1d5063b9311ce898d0af26b10742e6b9875eeb2117a",
  "bindings": [
   [
    "C0",
    "x[1,Speedboat]"
   ],
   [
    "C1",
    "x[1,Fishing Boat]"
   ],
   [
    "C2",
    "x[1,Catamaran]"
   ],
   [
    "C3",
    "x[1,Yacht]"
   ],
   [
    "C4",
    "x[1,Sailboat]"
   ],
   [
    "C5",
    "x[1,Kayak]"
   ],
   [
    "C6",
    "x[1,Canoe]"
   ],
   [
    "C7",
    "x[1,Houseboat]"
   ],
   [
    "C8",
    "x[1,Pontoon]"
   ],
   [
    "C9",
    "x[1,Jet Ski]"
   ],
   [
    "C10",
    "x[1,Rowboat]"
   ],
   [
    "C11",
    "x[1,Hovercraft]"
   ],
   [
    "C12",
    "x[1,Cabin Cruiser]"
   ],
   [
    "C13",
    "x[1,Wakeboard Boat]"
   ],
   [
    "C14",
    "x[1,Dinghy]"
   ],
   [
    "C15",
    "x[1,Trawler]"
   ],
   [
    "C16",
    "x[1,Paddle Boat]"
   ],
   [
    "C17",
    "x[1,Submarine]"
   ],
   [
    "C18",
    "x[1,RIB]"
   ],
   [
    "C19",
    "x[1,Skiff]"
   ],
   [
    "C20",
    "x[2,Speedboat]"
   ],
   [
    "C21",
    "x[2,Fishing Boat]"
   ],
   [
    "C22",
    "x[2,Catamaran]"
   ],
   [
    "C23",
    "x[2,Yacht]"
   ],
   [
    "C24",
    "x[2,Sailboat]"
   ],
   [
    "C25",
    "x[2,Kayak]"
   ],
   [
    "C26",
    "x[2,Canoe]"
   ],
   [
    "C27",
    "x[2,Houseboat]"
   ],
   [
    "C28",
    "x[2,Pontoon]"
   ],
   [
    "C29",
    "x[2,Jet Ski]"
   ],
   [
    "C30",
    "x[2,Rowboat]"
   ],
   [
    "C31",
    "x[2,Hovercraft]"
   ],
   [
    "C32",
    "x[2,Cabin Cruiser]"
   ],
   [
    "C33",
    "x[2,Wakeboard Boat]"
   ],
   [
    "C34",
    "x[2,Dinghy]"
   ],
   [
    "C35",
    "x[2,Trawler]"
   ],
   [
    "C36",
    "x[2,Paddle Boat]"
   ],
   [
    "C37",
    "x[2,Submarine]"
   ],
   [
    "C38",
    "x[2,RIB]"
   ],
   [
    "C39",
    "x[2,Skiff]"
   ],
   [
    "C40",
    "x[3,Speedboat]"
   ],
   [
    "C41",
    "x[3,Fishing Boat]"
   ],
   [
    "C42",
    "x[3,Catamaran]"
   ],
   [
    "C43",
    "x[3,Yacht]"
   ],
   [
    "C44",
    "x[3,Sailboat]"
   ],
   [
    "C45",
    "x[3,Kayak]"
   ],
   [
    "C46",
    "x[3,Canoe]"
   ],
   [
    "C47",
    "x[3,Houseboat]"
   ],
   [
    "C48",
    "x[3,Pontoon]"
   ],
   [
    "C49",
    "x[3,Jet Ski]"
   ],
   [
    "C50",
    "x[3,Rowboat]"
   ],
   [
    "C51",
    "x[3,Hovercraft]"
   ],
   [
    "C52",
    "x[3,Cabin Cruiser]"
   ],
   [
    "C53",
    "x[3,Wakeboard Boat]"
   ],
   [
    "C54",
    "x[3,Dinghy]"
   ],
   [
    "C55",
    "x[3,Trawler]"
   ],
   [
    "C56",
    "x[3,Paddle Boat]"
   ],
   [
    "C57",
    "x[3,Submarine]"
   ],
   [
    "C58",
    "x[3,RIB]"
   ],
   [
    "C59",
    "x[3,Skiff]"
   ],
   [
    "C60",
    "x[4,Speedboat]"
   ],
   [
    "C61",
    "x[4,Fishing Boat]"
   ],
   [
    "C62",
    "x[4,Catamaran]"
   ],
   [
    "C63",
    "x[4,Yacht]"
   ],
   [
    "C64",
    "x[4,Sailboat]"
   ],
   [
    "C65",
    "x[4,Kayak]"
   ],
   [
    "C66",
    "x[4,Canoe]"
   ],
   [
    "C67",
    "x[4,Houseboat]"
   ],
   [
    "C68",
    "x[4,Pontoon]"
   ],
   [
    "C69",
    "x[4,Jet Ski]"
   ],
   [
    "C70",
    "x[4,Rowboat]"
   ],
   [
    "C71",
    "x[4,Hovercraft]"
   ],
   [
    "C72",
    "x[4,Cabin Cruiser]"
   ],
   [
    "C73",
    "x[4,Wakeboard Boat]"
   ],
   [
    "C74",
    "x[4,Dinghy]"
   ],
   [
    "C75",
    "x[4,Trawler]"
   ],
   [
    "C76",
    "x[4,Paddle Boat]"
   ],
   [
    "C77",
    "x[4,Submarine]"
   ],
   [
    "C78",
    "x[4,RIB]"
   ],
   [
    "C79",
    "x[4,Skiff]"
   ],
   [
    "C80",
    "x[5,Speedboat]"
   ],
   [
    "C81",
    "x[5,Fishing Boat]"
   ],
   [
    "C82",
    "x[5,Catamaran]"
   ],
   [
    "C83",
    "x[5,Yacht]"
   ],
   [
    "C84",
    "x[5,Sailboat]"
   ],
   [
    "C85",
    "x[5,Kayak]"
   ],
   [
    "C86",
    "x[5,Canoe]"
   ],
   [
    "C87",
    "x[5,Houseboat]"
   ],
   [
    "C88",
    "x[5,Pontoon]"
   ],
   [
    "C89",
    "x[5,Jet Ski]"
   ],
   [
    "C90",
    "x[5,Rowboat]"
   ],
   [
    "C91",
    "x[5,Hovercraft]"
   ],
   [
    "C92",
    "x[5,Cabin Cruiser]"
   ],
   [
    "C93",
    "x[5,Wakeboard Boat]"
   ],
   [
    "C94",
    "x[5,Dinghy]"
   ],
   [
    "C95",
    "x[5,Trawler]"
   ],
   [
    "C96",
    "x[5,Paddle Boat]"
   ],
   [
    "C97",
    "x[5,Submarine]"
   ],
   [
    "C98",
    "x[5,RIB]"
   ],
   [
    "C99",
    "x[5,Skiff]"
   ],
   [
    "C100",
    "x[6,Speedboat]"
   ],
   [
    "C101",
    "x[6,Fishing Boat]"
   ],
   [
    "C102",
    "x[6,Catamaran]"
   ],
   [
    "C103",
    "x[6,Yacht]"
   ],
   [
    "C104",
    "x[6,Sailboat]"
   ],
   [
    "C105",
    "x[6,Kayak]"
   ],
   [
    "C106",
    "x[6,Canoe]"
   ],
   [
    "C107",
    "x[6,Houseboat]"
   ],
   [
    "C108",
    "x[6,Pontoon]"
   ],
   [
    "C109",
    "x[6,Jet Ski]"
   ],
   [
    "C110",
    "x[6,Rowboat]"
   ],
   [
    "C111",
    "x[6,Hovercraft]"
   ],
   [
    "C112",
    "x[6,Cabin Cruiser]"
   ],
   [
    "C113",
    "x[6,Wakeboard Boat]"
   ],
   [
    "C114",
    "x[6,Dinghy]"
   ],
   [
    "C115",
    "x[6,Trawler]"
   ],
   [
    "C116",
    "x[6,Paddle Boat]"
   ],
   [
    "C117",
    "x[6,Submarine]"
   ],
   [
    "C118",
    "x[6,RIB]"
   ],
   [
    "C119",
    "x[6,Skiff]"
   ],
   [
    "C120",
    "x[7,Speedboat]"
   ],
   [
    "C121",
    "x[7,Fishing Boat]"
   ],
   [
    "C122",
    "x[7,Catamaran]"
   ],
   [
    "C123",
    "x[7,Yacht]"
   ],
   [
    "C124",
    "x[7,Sailboat]"
   ],
   [
    "C125",
    "x[7,Kayak]"
   ],
   [
    "C126",
    "x[7,Canoe]"
   ],
   [
    "C127",
    "x[7,Houseboat]"
   ],
   [
    "C128",
    "x[7,Pontoon]"
   ],
   [
    "C129",
    "x[7,Jet Ski]"
   ],
   [
    "C130",
    "x[7,Rowboat]"
   ],
   [
    "C131",
    "x[7,Hovercraft]"
   ],
   [
    "C132",
    "x[7,Cabin Cruiser]"
   ],
   [
    "C133",
    "x[7,Wakeboard Boat]"
   ],
   [
    "C134",
    "x[7,Dinghy]"
   ],
   [
    "C135",
    "x[7,Trawler]"
   ],
   [
    "C136",
    "x[7,Paddle Boat]"
   ],
   [
    "C137",
    "x[7,Submarine]"
   ],
   [
    "C138",
    "x[7,RIB]"
   ],
   [
    "C139",
    "x[7,Skiff]"
   ],
   [
    "C140",
    "x[8,Speedboat]"
   ],
   [
    "C141",
    "x[8,Fishing Boat]"
   ],
   [
    "C142",
    "x[8,Catamaran]"
   ],
   [
    "C143",
    "x[8,Yacht]"
   ],
   [
    "C144",
    "x[8,Sailboat]"
   ],
   [
    "C145",
    "x[8,Kayak]"
   ],
   [
    "C146",
    "x[8,Canoe]"
   ],
   [
    "C147",
    "x[8,Houseboat]"
   ],
   [
    "C148",
    "x[8,Pontoon]"
   ],
   [
    "C149",
    "x[8,Jet Ski]"
   ],
   [
    "C150",
    "x[8,Rowboat]"
   ],
   [
    "C151",
    "x[8,Hovercraft]"
   ],
   [
    "C152",
    "x[8,Cabin Cruiser]"
   ],
   [
    "C153",
    "x[8,Wakeboard Boat]"
   ],
   [
    "C154",
    "x[8,Dinghy]"
   ],
   [
    "C155",
    "x[8,Trawler]"
   ],
   [
    "C156",
    "x[8,Paddle Boat]"
   ],
   [
    "C157",
    "x[8,Submarine]"
   ],
   [
    "C158",
    "x[8,RIB]"
   ],
   [
    "C159",
    "x[8,Skiff]"
   ],
   [
    "C160",
    "x[9,Speedboat]"
   ],
   [
    "C161",
    "x[9,Fishing Boat]"
   ],
   [
    "C162",
    "x[9,Catamaran]"
   ],
   [
    "C163",
    "x[9,Yacht]"
   ],
   [
    "C164",
    "x[9,Sailboat]"
   ],
   [
    "C165",
    "x[9,Kayak]"
   ],
   [
    "C166",
    "x[9,Canoe]"
   ],
   [
    "C167",
    "x[9,Houseboat]"
   ],
   [
    "C168",
    "x[9,Pontoon]"
   ],
   [
    "C169",
    "x[9,Jet Ski]"
   ],
   [
    "C170",
    "x[9,Rowboat]"
   ],
   [
    "C171",
    "x[9,Hovercraft]"
   ],
   [
    "C172",
    "x[9,Cabin Cruiser]"
   ],
   [
    "C173",
    "x[9,Wakeboard Boat]"
   ],
   [
    "C174",
    "x[9,Dinghy]"
   ],
   [
    "C175",
    "x[9,Trawler]"
   ],
   [
    "C176",
    "x[9,Paddle Boat]"
   ],
   [
    "C177",
    "x[9,Submarine]"
   ],
   [
    "C178",
    "x[9,RIB]"
   ],
   [
    "C179",
    "x[9,Skiff]"
   ],
   [
    "C180",
    "x[10,Speedboat]"
   ],
   [
    "C181",
    "x[10,Fishing Boat]"
   ],
   [
    "C182",
    "x[10,Catamaran]"
   ],
   [
    "C183",
    "x[10,Yacht]"
   ],
   [
    "C184",
    "x[10,Sailboat]"
   ],
   [
    "C185",
    "x[10,Kayak]"
   ],
   [
    "C186",
    "x[10,Canoe]"
   ],
   [
    "C187",
    "x[10,Houseboat]"
   ],
   [
    "C188",
    "x[10,Pontoon]"
   ],
   [
    "C189",
    "x[10,Jet Ski]"
   ],
   [
    "C190",
    "x[10,Rowboat]"
   ],
   [
    "C191",
    "x[10,Hovercraft]"
   ],
   [
    "C192",
    "x[10,Cabin Cruiser]"
   ],
   [
    "C193",
    "x[10,Wakeboard Boat]"
   ],
   [
    "C194",
    "x[10,Dinghy]"
   ],
   [
    "C195",
    "x[10,Trawler]"
   ],
   [
    "C196",
    "x[10,Paddle Boat]"
   ],
   [
    "C197",
    "x[10,Submarine]"
   ],
   [
    "C198",
    "x[10,RIB]"
   ],
   [
    "C199",
    "x[10,Skiff]"
   ],
   [
    "C200",
    "x[11,Speedboat]"
   ],
   [
    "C201",
    "x[11,Fishing Boat]"
   ],
   [
    "C202",
    "x[11,Catamaran]"
   ],
   [
    "C203",
    "x[11,Yacht]"
   ],
   [
    "C204",
    "x[11,Sailboat]"
   ],
   [
    "C205",
    "x[11,Kayak]"
   ],
   [
    "C206",
    "x[11,Canoe]"
   ],
   [
    "C207",
    "x[11,Houseboat]"
   ],
   [
    "C208",
    "x[11,Pontoon]"
   ],
   [
    "C209",
    "x[11,Jet Ski]"
   ],
   [
    "C210",
    "x[11,Rowboat]"
   ],
   [
    "C211",
    "x[11,Hovercraft]"
   ],
   [
    "C212",
    "x[11,Cabin Cruiser]"
   ],
   [
    "C213",
    "x[11,Wakeboard Boat]"
   ],
   [
    "C214",
    "x[11,Dinghy]"
   ],
   [
    "C215",
    "x[11,Trawler]"
   ],
   [
    "C216",
    "x[11,Paddle Boat]"
   ],
   [
    "C217",
    "x[11,Submarine]"
   ],
   [
    "C218",
    "x[11,RIB]"
   ],
   [
    "C219",
    "x[11,Skiff]"
   ],
   [
    "C220",
    "x[12,Speedboat]"
   ],
   [
    "C221",
    "x[12,Fishing Boat]"
   ],
   [
    "C222",
    "x[12,Catamaran]"
   ],
   [
    "C223",
    "x[12,Yacht]"
   ],
   [
    "C224",
    "x[12,Sailboat]"
   ],
   [
    "C225",
    "x[12,Kayak]"
   ],
   [
    "C226",
    "x[12,Canoe]"
   ],
   [
    "C227",
    "x[12,Houseboat]"
   ],
   [
    "C228",
    "x[12,Pontoon]"
   ],
   [
    "C229",
    "x[12,Jet Ski]"
   ],
   [
    "C230",
    "x[12,Rowboat]"
   ],
   [
    "C231",
    "x[12,Hovercraft]"
   ],
   [
    "C232",
    "x[12,Cabin Cruiser]"
   ],
   [
    "C233",
    "x[12,Wakeboard Boat]"
   ],
   [
    "C234",
    "x[12,Dinghy]"
   ],
   [
    "C235",
    "x[12,Trawler]"
   ],
   [
    "C236",
    "x[12,Paddle Boat]"
   ],
   [
    "C237",
    "x[12,Submarine]"
   ],
   [
    "C238",
    "x[12,RIB]"
   ],
   [
    "C239",
    "x[12,Skiff]"
   ],
   [
    "C240",
    "x[13,Speedboat]"
   ],
   [
    "C241",
    "x[13,Fishing Boat]"
   ],
   [
    "C242",
    "x[13,Catamaran]"
   ],
   [
    "C243",
    "x[13,Yacht]"
   ],
   [
    "C244",
    "x[13,Sailboat]"
   ],
   [
    "C245",
    "x[13,Kayak]"
   ],
   [
    "C246",
    "x[13,Canoe]"
   ],
   [
    "C247",
    "x[13,Houseboat]"
   ],
   [
    "C248",
    "x[13,Pontoon]"
   ],
   [
    "C249",
    "x[13,Jet Ski]"
   ],
   [
    "C250",
    "x[13,Rowboat]"
   ],
   [
    "C251",
    "x[13,Hovercraft]"
   ],
   [
    "C252",
    "x[13,Cabin Cruiser]"
   ],
   [
    "C253",
    "x[13,Wakeboard Boat]"
   ],
   [
    "C254",
    "x[13,Dinghy]"
   ],
   [
    "C255",
    "x[13,Trawler]"
   ],
   [
    "C256",
    "x[13,Paddle Boat]"
   ],
   [
    "C257",
    "x[13,Submarine]"
   ],
   [
    "C258",
    "x[13,RIB]"
   ],
   [
    "C259",
    "x[13,Skiff]"
   ],
   [
    "C260",
    "x[14,Speedboat]"
   ],
   [
    "C261",
    "x[14,Fishing Boat]"
   ],
   [
    "C262",
    "x[14,Catamaran]"
   ],
   [
    "C263",
    "x[14,Yacht]"
   ],
   [
    "C264",
    "x[14,Sailboat]"
   ],
   [
    "C265",
    "x[14,Kayak]"
   ],
   [
    "C266",
    "x[14,Canoe]"
   ],
   [
    "C267",
    "x[14,Houseboat]"
   ],
   [
    "C268",
    "x[14,Pontoon]"
   ],
   [
    "C269",
    "x[14,Jet Ski]"
   ],
   [
    "C270",
    "x[14,Rowboat]"
   ],
   [
    "C271",
    "x[14,Hovercraft]"
   ],
   [
    "C272",
    "x[14,Cabin Cruiser]"
   ],
   [
    "C273",
    "x[14,Wakeboard Boat]"
   ],
   [
    "C274",
    "x[14,Dinghy]"
   ],
   [
    "C275",
    "x[14,Trawler]"
   ],
   [
    "C276",
    "x[14,Paddle Boat]"
   ],
   [
    "C277",
    "x[14,Submarine]"
   ],
   [
    "C278",
    "x[14,RIB]"
   ],
   [
    "C279",
    "x[14,Skiff]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[1,Speedboat]",
   "x_0_1": "x[2,Speedboat]",
   "x_0_2": "x[3,Speedboat]",
   "x_0_3": "x[4,Speedboat]",
   "x_0_4": "x[5,Speedboat]",
   "x_0_5": "x[6,Speedboat]",
   "x_0_6": "x[7,Speedboat]",
   "x_0_7": "x[8,Speedboat]",
   "x_0_8": "x[9,Speedboat]",
   "x_0_9": "x[10,Speedboat]",
   "x_0_10": "x[11,Speedboat]",
   "x_0_11": "x[12,Speedboat]",
   "x_0_12": "x[13,Speedboat]",
   "x_0_13": "x[14,Speedboat]",
   "x_1_0": "x[1,Fishing Boat]",
   "x_1_1": "x[2,Fishing Boat]",
   "x_1_2": "x[3,Fishing Boat]",
   "x_1_3": "x[4,Fishing Boat]",
   "x_1_4": "x[5,Fishing Boat]",
   "x_1_5": "x[6,Fishing Boat]",
   "x_1_6": "x[7,Fishing Boat]",
   "x_1_7": "x[8,Fishing Boat]",
   "x_1_8": "x[9,Fishing Boat]",
   "x_1_9": "x[10,Fishing Boat]",
   "x_1_10": "x[11,Fishing Boat]",
   "x_1_11": "x[12,Fishing Boat]",
   "x_1_12": "x[13,Fishing Boat]",
   "x_1_13": "x[14,Fishing Boat]",
   "x_2_0": "x[1,Catamaran]",
   "x_2_1": "x[2,Catamaran]",
   "x_2_2": "x[3,Catamaran]",
   "x_2_3": "x[4,Catamaran]",
   "x_2_4": "x[5,Catamaran]",
   "x_2_5": "x[6,Catamaran]",
   "x_2_6": "x[7,Catamaran]",
   "x_2_7": "x[8,Catamaran]",
   "x_2_8": "x[9,Catamaran]",
   "x_2_9": "x[10,Catamaran]",
   "x_2_10": "x[11,Catamaran]",
   "x_2_11": "x[12,Catamaran]",
   "x_2_12": "x[13,Catamaran]",
   "x_2_13": "x[14,Catamaran]",
   "x_3_0": "x[1,Yacht]",
   "x_3_1": "x[2,Yacht]",
   "x_3_2": "x[3,Yacht]",
   "x_3_3": "x[4,Yacht]",
   "x_3_4": "x[5,Yacht]",
   "x_3_5": "x[6,Yacht]",
   "x_3_6": "x[7,Yacht]",
   "x_3_7": "x[8,Yacht]",
   "x_3_8": "x[9,Yacht]",
   "x_3_9": "x[10,Yacht]",
   "x_3_10": "x[11,Yacht]",
   "x_3_11": "x[12,Yacht]",
   "x_3_12": "x[13,Yacht]",
   "x_3_13": "x[14,Yacht]",
   "x_4_0": "x[1,Sailboat]",
   "x_4_1": "x[2,Sailboat]",
   "x_4_2": "x[3,Sailboat]",
   "x_4_3": "x[4,Sailboat]",
   "x_4_4": "x[5,Sailboat]",
   "x_4_5": "x[6,Sailboat]",
   "x_4_6": "x[7,Sailboat]",
   "x_4_7": "x[8,Sailboat]",
   "x_4_8": "x[9,Sailboat]",
   "x_4_9": "x[10,Sailboat]",
   "x_4_10": "x[11,Sailboat]",
   "x_4_11": "x[12,Sailboat]",
   "x_4_12": "x[13,Sailboat]",
   "x_4_13": "x[14,Sailboat]",
   "x_5_0": "x[1,Kayak]",
   "x_5_1": "x[2,Kayak]",
   "x_5_2": "x[3,Kayak]",
   "x_5_3": "x[4,Kayak]",
   "x_5_4": "x[5,Kayak]",
   "x_5_5": "x[6,Kayak]",
   "x_5_6": "x[7,Kayak]",
   "x_5_7": "x[8,Kayak]",
   "x_5_8": "x[9,Kayak]",
   "x_5_9": "x[10,Kayak]",
   "x_5_10": "x[11,Kayak]",
   "x_5_11": "x[12,Kayak]",
   "x_5_12": "x[13,Kayak]",
   "x_5_13": "x[14,Kayak]",
   "x_6_0": "x[1,Canoe]",
   "x_6_1": "x[2,Canoe]",
   "x_6_2": "x[3,Canoe]",
   "x_6_3": "x[4,Canoe]",
   "x_6_4": "x[5,Canoe]",
   "x_6_5": "x[6,Canoe]",
   "x_6_6": "x[7,Canoe]",
   "x_6_7": "x[8,Canoe]",
   "x_6_8": "x[9,Canoe]",
   "x_6_9": "x[10,Canoe]",
   "x_6_10": "x[11,Canoe]",
   "x_6_11": "x[12,Canoe]",
   "x_6_12": "x[13,Canoe]",
   "x_6_13": "x[14,Canoe]",
   "x_7_0": "x[1,Houseboat]",
   "x_7_1": "x[2,Houseboat]",
   "x_7_2": "x[3,Houseboat]",
   "x_7_3": "x[4,Houseboat]",
   "x_7_4": "x[5,Houseboat]",
   "x_7_5": "x[6,Houseboat]",
   "x_7_6": "x[7,Houseboat]",
   "x_7_7": "x[8,Houseboat]",
   "x_7_8": "x[9,Houseboat]",
   "x_7_9": "x[10,Houseboat]",
   "x_7_10": "x[11,Houseboat]",
   "x_7_11": "x[12,Houseboat]",
   "x_7_12": "x[13,Houseboat]",
   "x_7_13": "x[14,Houseboat]",
   "x_8_0": "x[1,Pontoon]",
   "x_8_1": "x[2,Pontoon]",
   "x_8_2": "x[3,Pontoon]",
   "x_8_3": "x[4,Pontoon]",
   "x_8_4": "x[5,Pontoon]",
   "x_8_5": "x[6,Pontoon]",
   "x_8_6": "x[7,Pontoon]",
   "x_8_7": "x[8,Pontoon]",
   "x_8_8": "x[9,Pontoon]",
   "x_8_9": "x[10,Pontoon]",
   "x_8_10": "x[11,Pontoon]",
   "x_8_11": "x[12,Pontoon]",
   "x_8_12": "x[13,Pontoon]",
   "x_8_13": "x[14,Pontoon]",
   "x_9_0": "x[1,Jet Ski]",
   "x_9_1": "x[2,Jet Ski]",
   "x_9_2": "x[3,Jet Ski]",
   "x_9_3": "x[4,Jet Ski]",
   "x_9_4": "x[5,Jet Ski]",
   "x_9_5": "x[6,Jet Ski]",
   "x_9_6": "x[7,Jet Ski]",
   "x_9_7": "x[8,Jet Ski]",
   "x_9_8": "x[9,Jet Ski]",
   "x_9_9": "x[10,Jet Ski]",
   "x_9_10": "x[11,Jet Ski]",
   "x_9_11": "x[12,Jet Ski]",
   "x_9_12": "x[13,Jet Ski]",
   "x_9_13": "x[14,Jet Ski]",
   "x_10_0": "x[1,Rowboat]",
   "x_10_1": "x[2,Rowboat]",
   "x_10_2": "x[3,Rowboat]",
   "x_10_3": "x[4,Rowboat]",
   "x_10_4": "x[5,Rowboat]",
   "x_10_5": "x[6,Rowboat]",
   "x_10_6": "x[7,Rowboat]",
   "x_10_7": "x[8,Rowboat]",
   "x_10_8": "x[9,Rowboat]",
   "x_10_9": "x[10,Rowboat]",
   "x_10_10": "x[11,Rowboat]",
   "x_10_11": "x[12,Rowboat]",
   "x_10_12": "x[13,Rowboat]",
   "x_10_13": "x[14,Rowboat]",
   "x_11_0": "x[1,Hovercraft]",
   "x_11_1": "x[2,Hovercraft]",
   "x_11_2": "x[3,Hovercraft]",
   "x_11_3": "x[4,Hovercraft]",
   "x_11_4": "x[5,Hovercraft]",
   "x_11_5": "x[6,Hovercraft]",
   "x_11_6": "x[7,Hovercraft]",
   "x_11_7": "x[8,Hovercraft]",
   "x_11_8": "x[9,Hovercraft]",
   "x_11_9": "x[10,Hovercraft]",
   "x_11_10": "x[11,Hovercraft]",
   "x_11_11": "x[12,Hovercraft]",
   "x_11_12": "x[13,Hovercraft]",
   "x_11_13": "x[14,Hovercraft]",
   "x_12_0": "x[1,Cabin Cruiser]",
   "x_12_1": "x[2,Cabin Cruiser]",
   "x_12_2": "x[3,Cabin Cruiser]",
   "x_12_3": "x[4,Cabin Cruiser]",
   "x_12_4": "x[5,Cabin Cruiser]",
   "x_12_5": "x[6,Cabin Cruiser]",
   "x_12_6": "x[7,Cabin Cruiser]",
   "x_12_7": "x[8,Cabin Cruiser]",
   "x_12_8": "x[9,Cabin Cruiser]",
   "x_12_9": "x[10,Cabin Cruiser]",
   "x_12_10": "x[11,Cabin Cruiser]",
   "x_12_11": "x[12,Cabin Cruiser]",
   "x_12_12": "x[13,Cabin Cruiser]",
   "x_12_13": "x[14,Cabin Cruiser]",
   "x_13_0": "x[1,Wakeboard Boat]",
   "x_13_1": "x[2,Wakeboard Boat]",
   "x_13_2": "x[3,Wakeboard Boat]",
   "x_13_3": "x[4,Wakeboard Boat]",
   "x_13_4": "x[5,Wakeboard Boat]",
   "x_13_5": "x[6,Wakeboard Boat]",
   "x_13_6": "x[7,Wakeboard Boat]",
   "x_13_7": "x[8,Wakeboard Boat]",
   "x_13_8": "x[9,Wakeboard Boat]",
   "x_13_9": "x[10,Wakeboard Boat]",
   "x_13_10": "x[11,Wakeboard Boat]",
   "x_13_11": "x[12,Wakeboard Boat]",
   "x_13_12": "x[13,Wakeboard Boat]",
   "x_13_13": "x[14,Wakeboard Boat]",
   "x_14_0": "x[1,Dinghy]",
   "x_14_1": "x[2,Dinghy]",
   "x_14_2": "x[3,Dinghy]",
   "x_14_3": "x[4,Dinghy]",
   "x_14_4": "x[5,Dinghy]",
   "x_14_5": "x[6,Dinghy]",
   "x_14_6": "x[7,Dinghy]",
   "x_14_7": "x[8,Dinghy]",
   "x_14_8": "x[9,Dinghy]",
   "x_14_9": "x[10,Dinghy]",
   "x_14_10": "x[11,Dinghy]",
   "x_14_11": "x[12,Dinghy]",
   "x_14_12": "x[13,Dinghy]",
   "x_14_13": "x[14,Dinghy]",
   "x_15_0": "x[1,Trawler]",
   "x_15_1": "x[2,Trawler]",
   "x_15_2": "x[3,Trawler]",
   "x_15_3": "x[4,Trawler]",
   "x_15_4": "x[5,Trawler]",
   "x_15_5": "x[6,Trawler]",
   "x_15_6": "x[7,Trawler]",
   "x_15_7": "x[8,Trawler]",
   "x_15_8": "x[9,Trawler]",
   "x_15_9": "x[10,Trawler]",
   "x_15_10": "x[11,Trawler]",
   "x_15_11": "x[12,Trawler]",
   "x_15_12": "x[13,Trawler]",
   "x_15_13": "x[14,Trawler]",
   "x_16_0": "x[1,Paddle Boat]",
   "x_16_1": "x[2,Paddle Boat]",
   "x_16_2": "x[3,Paddle Boat]",
   "x_16_3": "x[4,Paddle Boat]",
   "x_16_4": "x[5,Paddle Boat]",
   "x_16_5": "x[6,Paddle Boat]",
   "x_16_6": "x[7,Paddle Boat]",
   "x_16_7": "x[8,Paddle Boat]",
   "x_16_8": "x[9,Paddle Boat]",
   "x_16_9": "x[10,Paddle Boat]",
   "x_16_10": "x[11,Paddle Boat]",
   "x_16_11": "x[12,Paddle Boat]",
   "x_16_12": "x[13,Paddle Boat]",
   "x_16_13": "x[14,Paddle Boat]",
   "x_17_0": "x[1,Submarine]",
   "x_17_1": "x[2,Submarine]",
   "x_17_2": "x[3,Submarine]",
   "x_17_3": "x[4,Submarine]",
   "x_17_4": "x[5,Submarine]",
   "x_17_5": "x[6,Submarine]",
   "x_17_6": "x[7,Submarine]",
   "x_17_7": "x[8,Submarine]",
   "x_17_8": "x[9,Submarine]",
   "x_17_9": "x[10,Submarine]",
   "x_17_10": "x[11,Submarine]",
   "x_17_11": "x[12,Submarine]",
   "x_17_12": "x[13,Submarine]",
   "x_17_13": "x[14,Submarine]",
   "x_18_0": "x[1,RIB]",
   "x_18_1": "x[2,RIB]",
   "x_18_2": "x[3,RIB]",
   "x_18_3": "x[4,RIB]",
   "x_18_4": "x[5,RIB]",
   "x_18_5": "x[6,RIB]",
   "x_18_6": "x[7,RIB]",
   "x_18_7": "x[8,RIB]",
   "x_18_8": "x[9,RIB]",
   "x_18_9": "x[10,RIB]",
   "x_18_10": "x[11,RIB]",
   "x_18_11": "x[12,RIB]",
   "x_18_12": "x[13,RIB]",
   "x_18_13": "x[14,RIB]",
   "x_19_0": "x[1,Skiff]",
   "x_19_1": "x[2,Skiff]",
   "x_19_2": "x[3,Skiff]",
   "x_19_3": "x[4,Skiff]",
   "x_19_4": "x[5,Skiff]",
   "x_19_5": "x[6,Skiff]",
   "x_19_6": "x[7,Skiff]",
   "x_19_7": "x[8,Skiff]",
   "x_19_8": "x[9,Skiff]",
   "x_19_9": "x[10,Skiff]",
   "x_19_10": "x[11,Skiff]",
   "x_19_11": "x[12,Skiff]",
   "x_19_12": "x[13,Skiff]",
   "x_19_13": "x[14,Skiff]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 27088640.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     25.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     34.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     21.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     20.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     12.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     32.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     -0.0
    ],
    [
     "C131",
     -0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     -0.0
    ],
    [
     "C134",
     -0.0
    ],
    [
     "C135",
     -0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     9.0
    ],
    [
     "C139",
     -0.0
    ],
    [
     "C140",
     -0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     -0.0
    ],
    [
     "C143",
     -0.0
    ],
    [
     "C144",
     -0.0
    ],
    [
     "C145",
     -0.0
    ],
    [
     "C146",
     -0.0
    ],
    [
     "C147",
     -0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     -0.0
    ],
    [
     "C150",
     -0.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     -0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     -0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     -0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     27.0
    ],
    [
     "C159",
     -0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     -0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     -0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     -0.0
    ],
    [
     "C168",
     -0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     -0.0
    ],
    [
     "C171",
     -0.0
    ],
    [
     "C172",
     -0.0
    ],
    [
     "C173",
     -0.0
    ],
    [
     "C174",
     -0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     -0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     33.0
    ],
    [
     "C179",
     -0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     -0.0
    ],
    [
     "C182",
     -0.0
    ],
    [
     "C183",
     -0.0
    ],
    [
     "C184",
     -0.0
    ],
    [
     "C185",
     -0.0
    ],
    [
     "C186",
     -0.0
    ],
    [
     "C187",
     -0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     -0.0
    ],
    [
     "C190",
     -0.0
    ],
    [
     "C191",
     -0.0
    ],
    [
     "C192",
     -0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     -0.0
    ],
    [
     "C196",
     -0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     20.0
    ],
    [
     "C199",
     -0.0
    ],
    [
     "C200",
     -0.0
    ],
    [
     "C201",
     -0.0
    ],
    [
     "C202",
     -0.0
    ],
    [
     "C203",
     -0.0
    ],
    [
     "C204",
     -0.0
    ],
    [
     "C205",
     -0.0
    ],
    [
     "C206",
     -0.0
    ],
    [
     "C207",
     -0.0
    ],
    [
     "C208",
     -0.0
    ],
    [
     "C209",
     -0.0
    ],
    [
     "C210",
     -0.0
    ],
    [
     "C211",
     -0.0
    ],
    [
     "C212",
     -0.0
    ],
    [
     "C213",
     -0.0
    ],
    [
     "C214",
     -0.0
    ],
    [
     "C215",
     -0.0
    ],
    [
     "C216",
     -0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     22.0
    ],
    [
     "C219",
     -0.0
    ],
    [
     "C220",
     -0.0
    ],
    [
     "C221",
     -0.0
    ],
    [
     "C222",
     -0.0
    ],
    [
     "C223",
     -0.0
    ],
    [
     "C224",
     -0.0
    ],
    [
     "C225",
     -0.0
    ],
    [
     "C226",
     -0.0
    ],
    [
     "C227",
     -0.0
    ],
    [
     "C228",
     -0.0
    ],
    [
     "C229",
     -0.0
    ],
    [
     "C230",
     -0.0
    ],
    [
     "C231",
     -0.0
    ],
    [
     "C232",
     -0.0
    ],
    [
     "C233",
     -0.0
    ],
    [
     "C234",
     -0.0
    ],
    [
     "C235",
     -0.0
    ],
    [
     "C236",
     -0.0
    ],
    [
     "C237",
     0.0
    ],
    [
     "C238",
     17.0
    ],
    [
     "C239",
     -0.0
    ],
    [
     "C240",
     -0.0
    ],
    [
     "C241",
     -0.0
    ],
    [
     "C242",
     -0.0
    ],
    [
     "C243",
     -0.0
    ],
    [
     "C244",
     -0.0
    ],
    [
     "C245",
     -0.0
    ],
    [
     "C246",
     -0.0
    ],
    [
     "C247",
     -0.0
    ],
    [
     "C248",
     -0.0
    ],
    [
     "C249",
     -0.0
    ],
    [
     "C250",
     -0.0
    ],
    [
     "C251",
     -0.0
    ],
    [
     "C252",
     -0.0
    ],
    [
     "C253",
     -0.0
    ],
    [
     "C254",
     -0.0
    ],
    [
     "C255",
     -0.0
    ],
    [
     "C256",
     -0.0
    ],
    [
     "C257",
     0.0
    ],
    [
     "C258",
     28.0
    ],
    [
     "C259",
     -0.0
    ],
    [
     "C260",
     -0.0
    ],
    [
     "C261",
     -0.0
    ],
    [
     "C262",
     -0.0
    ],
    [
     "C263",
     -0.0
    ],
    [
     "C264",
     -0.0
    ],
    [
     "C265",
     -0.0
    ],
    [
     "C266",
     -0.0
    ],
    [
     "C267",
     -0.0
    ],
    [
     "C268",
     -0.0
    ],
    [
     "C269",
     -0.0
    ],
    [
     "C270",
     -0.0
    ],
    [
     "C271",
     -0.0
    ],
    [
     "C272",
     -0.0
    ],
    [
     "C273",
     -0.0
    ],
    [
     "C274",
     -0.0
    ],
    [
     "C275",
     -0.0
    ],
    [
     "C276",
     -0.0
    ],
    [
     "C277",
     0.0
    ],
    [
     "C278",
     20.0
    ],
    [
     "C279",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-059": {
  "source_code_sha256": "8f5948c38dc0a27c4bb9f53f24bca6fcc0e4c440703f2cca45b19c4464df2e3e",
  "original_solution_sha256": "cc3df2aa71c38e2665a9c2719a9119dd79419b4d22ccd58fdc2ab929ca561a77",
  "bindings": [
   [
    "C0",
    "y[S1]"
   ],
   [
    "C1",
    "y[S2]"
   ],
   [
    "C2",
    "y[S3]"
   ],
   [
    "C3",
    "y[S4]"
   ],
   [
    "C4",
    "y[S5]"
   ],
   [
    "C5",
    "y[S6]"
   ],
   [
    "C6",
    "y[S7]"
   ],
   [
    "C7",
    "y[S8]"
   ],
   [
    "C8",
    "x[S1,C1]"
   ],
   [
    "C9",
    "x[S1,C2]"
   ],
   [
    "C10",
    "x[S1,C3]"
   ],
   [
    "C11",
    "x[S1,C4]"
   ],
   [
    "C12",
    "x[S1,C5]"
   ],
   [
    "C13",
    "x[S1,C6]"
   ],
   [
    "C14",
    "x[S1,C7]"
   ],
   [
    "C15",
    "x[S1,C8]"
   ],
   [
    "C16",
    "x[S1,C9]"
   ],
   [
    "C17",
    "x[S2,C1]"
   ],
   [
    "C18",
    "x[S2,C2]"
   ],
   [
    "C19",
    "x[S2,C3]"
   ],
   [
    "C20",
    "x[S2,C4]"
   ],
   [
    "C21",
    "x[S2,C5]"
   ],
   [
    "C22",
    "x[S2,C6]"
   ],
   [
    "C23",
    "x[S2,C7]"
   ],
   [
    "C24",
    "x[S2,C8]"
   ],
   [
    "C25",
    "x[S2,C9]"
   ],
   [
    "C26",
    "x[S3,C1]"
   ],
   [
    "C27",
    "x[S3,C2]"
   ],
   [
    "C28",
    "x[S3,C3]"
   ],
   [
    "C29",
    "x[S3,C4]"
   ],
   [
    "C30",
    "x[S3,C5]"
   ],
   [
    "C31",
    "x[S3,C6]"
   ],
   [
    "C32",
    "x[S3,C7]"
   ],
   [
    "C33",
    "x[S3,C8]"
   ],
   [
    "C34",
    "x[S3,C9]"
   ],
   [
    "C35",
    "x[S4,C1]"
   ],
   [
    "C36",
    "x[S4,C2]"
   ],
   [
    "C37",
    "x[S4,C3]"
   ],
   [
    "C38",
    "x[S4,C4]"
   ],
   [
    "C39",
    "x[S4,C5]"
   ],
   [
    "C40",
    "x[S4,C6]"
   ],
   [
    "C41",
    "x[S4,C7]"
   ],
   [
    "C42",
    "x[S4,C8]"
   ],
   [
    "C43",
    "x[S4,C9]"
   ],
   [
    "C44",
    "x[S5,C1]"
   ],
   [
    "C45",
    "x[S5,C2]"
   ],
   [
    "C46",
    "x[S5,C3]"
   ],
   [
    "C47",
    "x[S5,C4]"
   ],
   [
    "C48",
    "x[S5,C5]"
   ],
   [
    "C49",
    "x[S5,C6]"
   ],
   [
    "C50",
    "x[S5,C7]"
   ],
   [
    "C51",
    "x[S5,C8]"
   ],
   [
    "C52",
    "x[S5,C9]"
   ],
   [
    "C53",
    "x[S6,C1]"
   ],
   [
    "C54",
    "x[S6,C2]"
   ],
   [
    "C55",
    "x[S6,C3]"
   ],
   [
    "C56",
    "x[S6,C4]"
   ],
   [
    "C57",
    "x[S6,C5]"
   ],
   [
    "C58",
    "x[S6,C6]"
   ],
   [
    "C59",
    "x[S6,C7]"
   ],
   [
    "C60",
    "x[S6,C8]"
   ],
   [
    "C61",
    "x[S6,C9]"
   ],
   [
    "C62",
    "x[S7,C1]"
   ],
   [
    "C63",
    "x[S7,C2]"
   ],
   [
    "C64",
    "x[S7,C3]"
   ],
   [
    "C65",
    "x[S7,C4]"
   ],
   [
    "C66",
    "x[S7,C5]"
   ],
   [
    "C67",
    "x[S7,C6]"
   ],
   [
    "C68",
    "x[S7,C7]"
   ],
   [
    "C69",
    "x[S7,C8]"
   ],
   [
    "C70",
    "x[S7,C9]"
   ],
   [
    "C71",
    "x[S8,C1]"
   ],
   [
    "C72",
    "x[S8,C2]"
   ],
   [
    "C73",
    "x[S8,C3]"
   ],
   [
    "C74",
    "x[S8,C4]"
   ],
   [
    "C75",
    "x[S8,C5]"
   ],
   [
    "C76",
    "x[S8,C6]"
   ],
   [
    "C77",
    "x[S8,C7]"
   ],
   [
    "C78",
    "x[S8,C8]"
   ],
   [
    "C79",
    "x[S8,C9]"
   ]
  ],
  "primary_mapping": {
   "y[S1]": "y[S1]",
   "y[S2]": "y[S2]",
   "y[S3]": "y[S3]",
   "y[S4]": "y[S4]",
   "y[S5]": "y[S5]",
   "y[S6]": "y[S6]",
   "y[S7]": "y[S7]",
   "y[S8]": "y[S8]",
   "x[S1,C1]": "x[S1,C1]",
   "x[S1,C9]": "x[S1,C9]",
   "x[S2,C2]": "x[S2,C2]",
   "x[S2,C9]": "x[S2,C9]",
   "x[S3,C3]": "x[S3,C3]",
   "x[S3,C9]": "x[S3,C9]",
   "x[S4,C4]": "x[S4,C4]",
   "x[S4,C9]": "x[S4,C9]",
   "x[S5,C5]": "x[S5,C5]",
   "x[S5,C9]": "x[S5,C9]",
   "x[S6,C6]": "x[S6,C6]",
   "x[S6,C9]": "x[S6,C9]",
   "x[S7,C7]": "x[S7,C7]",
   "x[S7,C9]": "x[S7,C9]",
   "x[S8,C8]": "x[S8,C8]",
   "x[S8,C9]": "x[S8,C9]",
   "x[S1,C2]": "x[S1,C2]",
   "x[S1,C3]": "x[S1,C3]",
   "x[S1,C4]": "x[S1,C4]",
   "x[S1,C5]": "x[S1,C5]",
   "x[S1,C6]": "x[S1,C6]",
   "x[S1,C7]": "x[S1,C7]",
   "x[S1,C8]": "x[S1,C8]",
   "x[S2,C1]": "x[S2,C1]",
   "x[S2,C3]": "x[S2,C3]",
   "x[S2,C4]": "x[S2,C4]",
   "x[S2,C5]": "x[S2,C5]",
   "x[S2,C6]": "x[S2,C6]",
   "x[S2,C7]": "x[S2,C7]",
   "x[S2,C8]": "x[S2,C8]",
   "x[S3,C1]": "x[S3,C1]",
   "x[S3,C2]": "x[S3,C2]",
   "x[S3,C4]": "x[S3,C4]",
   "x[S3,C5]": "x[S3,C5]",
   "x[S3,C6]": "x[S3,C6]",
   "x[S3,C7]": "x[S3,C7]",
   "x[S3,C8]": "x[S3,C8]",
   "x[S4,C1]": "x[S4,C1]",
   "x[S4,C2]": "x[S4,C2]",
   "x[S4,C3]": "x[S4,C3]",
   "x[S4,C5]": "x[S4,C5]",
   "x[S4,C6]": "x[S4,C6]",
   "x[S4,C7]": "x[S4,C7]",
   "x[S4,C8]": "x[S4,C8]",
   "x[S5,C1]": "x[S5,C1]",
   "x[S5,C2]": "x[S5,C2]",
   "x[S5,C3]": "x[S5,C3]",
   "x[S5,C4]": "x[S5,C4]",
   "x[S5,C6]": "x[S5,C6]",
   "x[S5,C7]": "x[S5,C7]",
   "x[S5,C8]": "x[S5,C8]",
   "x[S6,C1]": "x[S6,C1]",
   "x[S6,C2]": "x[S6,C2]",
   "x[S6,C3]": "x[S6,C3]",
   "x[S6,C4]": "x[S6,C4]",
   "x[S6,C5]": "x[S6,C5]",
   "x[S6,C7]": "x[S6,C7]",
   "x[S6,C8]": "x[S6,C8]",
   "x[S7,C1]": "x[S7,C1]",
   "x[S7,C2]": "x[S7,C2]",
   "x[S7,C3]": "x[S7,C3]",
   "x[S7,C4]": "x[S7,C4]",
   "x[S7,C5]": "x[S7,C5]",
   "x[S7,C6]": "x[S7,C6]",
   "x[S7,C8]": "x[S7,C8]",
   "x[S8,C1]": "x[S8,C1]",
   "x[S8,C2]": "x[S8,C2]",
   "x[S8,C3]": "x[S8,C3]",
   "x[S8,C4]": "x[S8,C4]",
   "x[S8,C5]": "x[S8,C5]",
   "x[S8,C6]": "x[S8,C6]",
   "x[S8,C7]": "x[S8,C7]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1862205008776.5698,
   "variables": [
    [
     "C0",
     1.0
    ],
    [
     "C1",
     1.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     1.0
    ],
    [
     "C5",
     1.0
    ],
    [
     "C6",
     1.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     1027326000.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     9058407000.0
    ],
    [
     "C14",
     5344367000.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     677201000.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     1600594000.0
    ],
    [
     "C28",
     5086889000.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     4742532000.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     11926044000.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     3236493000.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-062": {
  "source_code_sha256": "ff5513237a2c6777b0a7c4c9c4585b33d60831afe79e1c94993b623bc8e29ae0",
  "original_solution_sha256": "9fc0b60fb12e3e839b4b5f0d76c98060f636b4173bf5294b76ff2915ee0c3f34",
  "bindings": [
   [
    "C0",
    "x[MOUNT AYR,Customer_1]"
   ],
   [
    "C1",
    "x[MOUNT AYR,Customer_2]"
   ],
   [
    "C2",
    "x[MOUNT AYR,Customer_3]"
   ],
   [
    "C3",
    "x[MOUNT AYR,Customer_4]"
   ],
   [
    "C4",
    "x[MOUNT AYR,Customer_5]"
   ],
   [
    "C5",
    "x[WAUKEE,Customer_1]"
   ],
   [
    "C6",
    "x[WAUKEE,Customer_2]"
   ],
   [
    "C7",
    "x[WAUKEE,Customer_3]"
   ],
   [
    "C8",
    "x[WAUKEE,Customer_4]"
   ],
   [
    "C9",
    "x[WAUKEE,Customer_5]"
   ],
   [
    "C10",
    "x[WAVERLY,Customer_1]"
   ],
   [
    "C11",
    "x[WAVERLY,Customer_2]"
   ],
   [
    "C12",
    "x[WAVERLY,Customer_3]"
   ],
   [
    "C13",
    "x[WAVERLY,Customer_4]"
   ],
   [
    "C14",
    "x[WAVERLY,Customer_5]"
   ],
   [
    "C15",
    "x[PELLA,Customer_1]"
   ],
   [
    "C16",
    "x[PELLA,Customer_2]"
   ],
   [
    "C17",
    "x[PELLA,Customer_3]"
   ],
   [
    "C18",
    "x[PELLA,Customer_4]"
   ],
   [
    "C19",
    "x[PELLA,Customer_5]"
   ],
   [
    "C20",
    "x[DES MOINES,Customer_1]"
   ],
   [
    "C21",
    "x[DES MOINES,Customer_2]"
   ],
   [
    "C22",
    "x[DES MOINES,Customer_3]"
   ],
   [
    "C23",
    "x[DES MOINES,Customer_4]"
   ],
   [
    "C24",
    "x[DES MOINES,Customer_5]"
   ],
   [
    "C25",
    "y[MOUNT AYR]"
   ],
   [
    "C26",
    "y[WAUKEE]"
   ],
   [
    "C27",
    "y[WAVERLY]"
   ],
   [
    "C28",
    "y[PELLA]"
   ],
   [
    "C29",
    "y[DES MOINES]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[MOUNT AYR,Customer_1]",
   "x_0_1": "x[MOUNT AYR,Customer_2]",
   "x_0_2": "x[MOUNT AYR,Customer_3]",
   "x_0_3": "x[MOUNT AYR,Customer_4]",
   "x_0_4": "x[MOUNT AYR,Customer_5]",
   "x_1_0": "x[WAUKEE,Customer_1]",
   "x_1_1": "x[WAUKEE,Customer_2]",
   "x_1_2": "x[WAUKEE,Customer_3]",
   "x_1_3": "x[WAUKEE,Customer_4]",
   "x_1_4": "x[WAUKEE,Customer_5]",
   "x_2_0": "x[WAVERLY,Customer_1]",
   "x_2_1": "x[WAVERLY,Customer_2]",
   "x_2_2": "x[WAVERLY,Customer_3]",
   "x_2_3": "x[WAVERLY,Customer_4]",
   "x_2_4": "x[WAVERLY,Customer_5]",
   "x_3_0": "x[PELLA,Customer_1]",
   "x_3_1": "x[PELLA,Customer_2]",
   "x_3_2": "x[PELLA,Customer_3]",
   "x_3_3": "x[PELLA,Customer_4]",
   "x_3_4": "x[PELLA,Customer_5]",
   "x_4_0": "x[DES MOINES,Customer_1]",
   "x_4_1": "x[DES MOINES,Customer_2]",
   "x_4_2": "x[DES MOINES,Customer_3]",
   "x_4_3": "x[DES MOINES,Customer_4]",
   "x_4_4": "x[DES MOINES,Customer_5]",
   "y_0": "y[MOUNT AYR]",
   "y_1": "y[WAUKEE]",
   "y_2": "y[WAVERLY]",
   "y_3": "y[PELLA]",
   "y_4": "y[DES MOINES]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 172612.46,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     1889.0
    ],
    [
     "C7",
     2518.0
    ],
    [
     "C8",
     3218.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     2397.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     1813.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     1.0
    ],
    [
     "C27",
     1.0
    ],
    [
     "C28",
     1.0
    ],
    [
     "C29",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-066": {
  "source_code_sha256": "6c1377b4225bf625101f3286a37e1f5afbdb1dc52dfd4fe38188a904302247cb",
  "original_solution_sha256": "feac0468c67fc04e0149f83bb4701b7245d0e0629e4616625b44ec9051a5abcb",
  "bindings": [
   [
    "C0",
    "x[S1,C1]"
   ],
   [
    "C1",
    "x[S1,C2]"
   ],
   [
    "C2",
    "x[S2,C1]"
   ],
   [
    "C3",
    "x[S2,C2]"
   ],
   [
    "C4",
    "y[S1]"
   ],
   [
    "C5",
    "y[S2]"
   ]
  ],
  "primary_mapping": {
   "x_0_0": "x[S1,C1]",
   "x_0_1": "x[S1,C2]",
   "x_1_0": "x[S2,C1]",
   "x_1_1": "x[S2,C2]",
   "y_0": "y[S1]",
   "y_1": "y[S2]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 11396.51,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     144.0
    ],
    [
     "C3",
     216.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     1.0
    ]
   ],
   "error": null
  }
 },
 "OR-071": {
  "source_code_sha256": "4010bd26936f416027b8317f25b75a5282a8f38738e0f363f484a9858d36d59a",
  "original_solution_sha256": "c25ce5b4ad2d7491ae8e3a7cc58d2762765098eccb3277ff3038df45eebe2719",
  "bindings": [
   [
    "C0",
    "x[Classic Oxford Shirt]"
   ],
   [
    "C1",
    "x[Cotton Crew Neck T-Shirt]"
   ],
   [
    "C2",
    "x[French Terry Hoodie]"
   ],
   [
    "C3",
    "x[Denim Snap Shirt]"
   ],
   [
    "C4",
    "x[Jersey V-Neck T-Shirt]"
   ],
   [
    "C5",
    "x[Unisex Jogger Pants]"
   ],
   [
    "C6",
    "x[Flannel Plaid Shirt]"
   ],
   [
    "C7",
    "x[Performance Polo Shirt]"
   ],
   [
    "C8",
    "x[Long-sleeve Henley]"
   ],
   [
    "C9",
    "x[Heavyweight Sweatshirt]"
   ],
   [
    "C10",
    "x[Linen Blend Shirt]"
   ],
   [
    "C11",
    "x[Graphic Print Tee]"
   ],
   [
    "C12",
    "x[Quilted Vest]"
   ],
   [
    "C13",
    "x[Chambray Work Shirt]"
   ],
   [
    "C14",
    "x[Microfiber Sport Shirt]"
   ],
   [
    "C15",
    "x[Wool Blend Peacoat]"
   ],
   [
    "C16",
    "x[Seersucker Short-sleeve]"
   ],
   [
    "C17",
    "x[Twill Utility Shirt]"
   ],
   [
    "C18",
    "x[Basic Scoop Neck Tee]"
   ],
   [
    "C19",
    "x[Waffle Knit Pullover]"
   ],
   [
    "C20",
    "x[Button-Down Poplin Shirt]"
   ],
   [
    "C21",
    "x[Slub Cotton T-Shirt]"
   ],
   [
    "C22",
    "x[Drawstring Cargo Pants]"
   ],
   [
    "C23",
    "x[Pima Cotton Dress Shirt]"
   ],
   [
    "C24",
    "x[Pocket T-Shirt]"
   ],
   [
    "C25",
    "x[Fleece-Lined Track Suit]"
   ],
   [
    "C26",
    "x[Printed Camp Collar Shirt]"
   ],
   [
    "C27",
    "x[Performance Dry-Fit Tee]"
   ],
   [
    "C28",
    "x[Corduroy Shacket]"
   ],
   [
    "C29",
    "x[Herringbone Casual Shirt]"
   ],
   [
    "C30",
    "x[Organic Cotton Tee]"
   ],
   [
    "C31",
    "x[Cashmere Blend Sweater]"
   ],
   [
    "C32",
    "x[Striped Rugby Shirt]"
   ],
   [
    "C33",
    "x[Raglan Sleeve Tee]"
   ],
   [
    "C34",
    "x[Waterproof Rain Jacket]"
   ],
   [
    "C35",
    "x[Micro-check Business Shirt]"
   ],
   [
    "C36",
    "x[Heather Grey T-Shirt]"
   ],
   [
    "C37",
    "x[Slim Fit Chinos]"
   ],
   [
    "C38",
    "x[Band Collar Shirt]"
   ],
   [
    "C39",
    "x[Triblend Jersey Tee]"
   ],
   [
    "C40",
    "x[Sherpa Lined Pullover]"
   ],
   [
    "C41",
    "x[Western Style Shirt]"
   ],
   [
    "C42",
    "x[Recycled Poly T-Shirt]"
   ],
   [
    "C43",
    "x[Insulated Puffer Jacket]"
   ],
   [
    "C44",
    "x[End-on-End Fabric Shirt]"
   ],
   [
    "C45",
    "x[Oversized Graphic Tee]"
   ],
   [
    "C46",
    "x[Stretch Hybrid Shorts]"
   ],
   [
    "C47",
    "x[Non-Iron Travel Shirt]"
   ],
   [
    "C48",
    "x[Sun-Protective T-Shirt]"
   ],
   [
    "C49",
    "x[Velvet Bomber Jacket]"
   ],
   [
    "C50",
    "x[Madras Plaid Shirt]"
   ],
   [
    "C51",
    "x[Tie-Dye T-Shirt]"
   ],
   [
    "C52",
    "x[Lightweight Windbreaker]"
   ],
   [
    "C53",
    "x[Pinpoint Oxford Shirt]"
   ],
   [
    "C54",
    "x[Bamboo Viscose Tee]"
   ],
   [
    "C55",
    "x[Tapered Denim Jeans]"
   ],
   [
    "C56",
    "x[Contrast Collar Shirt]"
   ],
   [
    "C57",
    "x[Mock Neck Long-sleeve]"
   ],
   [
    "C58",
    "x[Down-filled Parka]"
   ],
   [
    "C59",
    "x[Textured Grid Shirt]"
   ],
   [
    "C60",
    "x[Heavy Cotton T-Shirt]"
   ],
   [
    "C61",
    "x[Convertible Zip-Off Trousers]"
   ],
   [
    "C62",
    "x[Tencel Blend Casual Shirt]"
   ],
   [
    "C63",
    "x[Relaxed Fit Tee]"
   ],
   [
    "C64",
    "x[Faux Leather Jacket]"
   ],
   [
    "C65",
    "x[Speckled Slub Shirt]"
   ],
   [
    "C66",
    "x[Embroidered T-Shirt]"
   ],
   [
    "C67",
    "x[Knit Cardigan Sweater]"
   ],
   [
    "C68",
    "x[Two-Pocket Utility Shirt]"
   ],
   [
    "C69",
    "x[Performance Stretch Tee]"
   ],
   [
    "C70",
    "x[Technical Shell Jacket]"
   ],
   [
    "C71",
    "x[Semi-Spread Collar Shirt]"
   ],
   [
    "C72",
    "x[Washed Pigment Dye Tee]"
   ],
   [
    "C73",
    "x[Twill Blazer (Unlined)]"
   ],
   [
    "C74",
    "x[Brushed Twill Shirt]"
   ],
   [
    "C75",
    "x[Supima Cotton Tee]"
   ],
   [
    "C76",
    "x[High-Waist Trousers]"
   ],
   [
    "C77",
    "x[Printed Geometric Shirt]"
   ],
   [
    "C78",
    "x[Minimalist Plain Tee]"
   ],
   [
    "C79",
    "x[Reversible Quilted Jacket]"
   ],
   [
    "C80",
    "x[Dobby Weave Shirt]"
   ],
   [
    "C81",
    "x[Heavyweight Boxy Tee]"
   ],
   [
    "C82",
    "x[Casual Wool Overcoat]"
   ],
   [
    "C83",
    "x[Half-Zip Pullover Shirt]"
   ],
   [
    "C84",
    "x[Drop Shoulder T-Shirt]"
   ],
   [
    "C85",
    "x[Straight Leg Corduroys]"
   ],
   [
    "C86",
    "x[Camp Collar Floral Shirt]"
   ],
   [
    "C87",
    "x[Garment Dyed Tee]"
   ],
   [
    "C88",
    "x[Suede Bomber Jacket]"
   ],
   [
    "C89",
    "x[Melange Knit Shirt]"
   ],
   [
    "C90",
    "x[Longline T-Shirt]"
   ],
   [
    "C91",
    "x[Wide Leg Culottes]"
   ],
   [
    "C92",
    "x[Tropical Print Shirt]"
   ],
   [
    "C93",
    "x[Distressed Cotton Tee]"
   ],
   [
    "C94",
    "x[Patchwork Denim Jacket]"
   ],
   [
    "C95",
    "x[Micro-Dot Shirt]"
   ],
   [
    "C96",
    "x[Muscle Fit T-Shirt]"
   ],
   [
    "C97",
    "x[Woven Utility Vest]"
   ],
   [
    "C98",
    "x[Silk Blend Dress Shirt]"
   ],
   [
    "C99",
    "x[Heavyweight Jersey T-Shirt]"
   ],
   [
    "C100",
    "x[Slim Fit Ankle Grazer]"
   ],
   [
    "C101",
    "x[Lattice Weave Shirt]"
   ],
   [
    "C102",
    "x[Boxy Cropped Tee]"
   ],
   [
    "C103",
    "x[Reversible Bomber Jacket]"
   ],
   [
    "C104",
    "x[Stretch Performance Shirt]"
   ],
   [
    "C105",
    "x[Sublimation Print Tee]"
   ],
   [
    "C106",
    "x[Brushed Cotton Trousers]"
   ],
   [
    "C107",
    "x[Micro-Waffle Knit Shirt]"
   ],
   [
    "C108",
    "x[Reflective Detail Tee]"
   ],
   [
    "C109",
    "x[Shearling Lined Coat]"
   ],
   [
    "C110",
    "x[Printed Patchwork Shirt]"
   ],
   [
    "C111",
    "x[Washed Effect T-Shirt]"
   ],
   [
    "C112",
    "x[Velour Track Pants]"
   ],
   [
    "C113",
    "x[Fine Wale Corduroy Shirt]"
   ],
   [
    "C114",
    "x[Organic Hemp T-Shirt]"
   ],
   [
    "C115",
    "x[Leather Biker Jacket]"
   ],
   [
    "C116",
    "x[Supersoft Modal Shirt]"
   ],
   [
    "C117",
    "x[Deep V-Neck Tee]"
   ],
   [
    "C118",
    "x[Wool Blend Trousers]"
   ],
   [
    "C119",
    "x[Contrast Stitch Shirt]"
   ],
   [
    "C120",
    "x[Embroidered Pocket Tee]"
   ],
   [
    "C121",
    "x[Camo Print Hoodie]"
   ],
   [
    "C122",
    "x[Silk-Touch Poplin Shirt]"
   ],
   [
    "C123",
    "x[Slim Fit Pique Polo]"
   ],
   [
    "C124",
    "x[Technical Cargo Vest]"
   ],
   [
    "C125",
    "x[Cutaway Collar Shirt]"
   ],
   [
    "C126",
    "x[Lycra Blend Active Tee]"
   ],
   [
    "C127",
    "x[Heavy Knit Sweater]"
   ],
   [
    "C128",
    "x[Textured Stripe Shirt]"
   ],
   [
    "C129",
    "x[Bio-Washed T-Shirt]"
   ],
   [
    "C130",
    "x[Mohair Blend Cardigan]"
   ],
   [
    "C131",
    "x[Shantung Silk Shirt]"
   ],
   [
    "C132",
    "x[Distressed Denim Tee]"
   ],
   [
    "C133",
    "x[High-Tech Ski Jacket]"
   ],
   [
    "C134",
    "x[Fine Poplin Check Shirt]"
   ],
   [
    "C135",
    "x[Slim Fit Linen T-Shirt]"
   ],
   [
    "C136",
    "x[Ripstop Utility Pants]"
   ],
   [
    "C137",
    "x[Classic Barrel Cuff Shirt]"
   ],
   [
    "C138",
    "x[Bamboo Fiber Tee]"
   ],
   [
    "C139",
    "x[Shearling Trench Coat]"
   ],
   [
    "C140",
    "x[Slim Fit Chambray]"
   ],
   [
    "C141",
    "x[Printed Geometric Tee]"
   ],
   [
    "C142",
    "x[Wool Cargo Trousers]"
   ],
   [
    "C143",
    "x[Two-Tone Oxford Shirt]"
   ],
   [
    "C144",
    "x[Slogan Print T-Shirt]"
   ],
   [
    "C145",
    "x[Water-Resistant Shell]"
   ],
   [
    "C146",
    "x[Grandad Collar Shirt]"
   ],
   [
    "C147",
    "x[Embroidered Logo Tee]"
   ],
   [
    "C148",
    "x[Casual Knit Blazer]"
   ],
   [
    "C149",
    "x[Eco-Friendly Hemp Shirt]"
   ],
   [
    "C150",
    "x[Cotton Slub Polo]"
   ],
   [
    "C151",
    "x[Corduroy Carpenter Pants]"
   ],
   [
    "C152",
    "x[Linen Safari Shirt]"
   ],
   [
    "C153",
    "x[Ribbed Crew Neck Tee]"
   ],
   [
    "C154",
    "x[Hybrid Hiking Trousers]"
   ],
   [
    "C155",
    "x[Brushed Flannel Shirt]"
   ],
   [
    "C156",
    "x[Microfiber V-Neck]"
   ],
   [
    "C157",
    "x[Down-Alternative Vest]"
   ],
   [
    "C158",
    "x[Checked Seersucker Shirt]"
   ],
   [
    "C159",
    "x[Fitted Basic Tee]"
   ],
   [
    "C160",
    "x[Heavyweight Denim Jacket]"
   ],
   [
    "C161",
    "x[Woven Stripe Shirt]"
   ],
   [
    "C162",
    "x[Tech Fleece Pullover]"
   ],
   [
    "C163",
    "x[Pin Dot Print Shirt]"
   ],
   [
    "C164",
    "x[Supersoft Modal T-Shirt]"
   ],
   [
    "C165",
    "x[Water-Resistant Chinos]"
   ],
   [
    "C166",
    "x[Tailored Fit Dress Shirt]"
   ],
   [
    "C167",
    "x[Printed Art Graphic Tee]"
   ],
   [
    "C168",
    "x[Cashmere Cable Knit]"
   ],
   [
    "C169",
    "x[Plaid Flannel Overshirt]"
   ],
   [
    "C170",
    "x[Moisture-Wicking Polo]"
   ],
   [
    "C171",
    "x[Slim Tapered Jeans]"
   ],
   [
    "C172",
    "x[Contrast Cuff Shirt]"
   ],
   [
    "C173",
    "x[Sustainable Cotton Tee]"
   ],
   [
    "C174",
    "x[Trench Coat (Lined)]"
   ],
   [
    "C175",
    "x[Spread Collar Shirt]"
   ],
   [
    "C176",
    "x[Printed Stripe Tee]"
   ],
   [
    "C177",
    "x[Non-Iron Twill Shirt]"
   ],
   [
    "C178",
    "x[Recycled Material Tee]"
   ],
   [
    "C179",
    "x[Faux Fur Coat]"
   ],
   [
    "C180",
    "x[Seersucker Popover Shirt]"
   ],
   [
    "C181",
    "x[Graphic Back Print Tee]"
   ],
   [
    "C182",
    "x[Corduroy Field Jacket]"
   ],
   [
    "C183",
    "x[Linen/Cotton Blend Shirt]"
   ],
   [
    "C184",
    "x[Lightweight Jersey Tee]"
   ],
   [
    "C185",
    "x[High Performance Hiking Pants]"
   ],
   [
    "C186",
    "x[Cuban Collar Shirt]"
   ],
   [
    "C187",
    "x[Faded Pigment Tee]"
   ],
   [
    "C188",
    "x[Canvas Chore Coat]"
   ],
   [
    "C189",
    "x[Brushed Cotton Shirt]"
   ],
   [
    "C190",
    "x[Ribbed Knit T-Shirt]"
   ],
   [
    "C191",
    "x[Wide Leg Trousers]"
   ],
   [
    "C192",
    "x[Patchwork Denim Shirt]"
   ],
   [
    "C193",
    "x[Basic White T-Shirt]"
   ],
   [
    "C194",
    "x[Mohair Knit Sweater]"
   ],
   [
    "C195",
    "x[Contrast Stitch Polo]"
   ],
   [
    "C196",
    "x[Tailored Wool Trousers]"
   ],
   [
    "C197",
    "x[Heavyweight Denim Shirt]"
   ]
  ],
  "primary_mapping": {
   "x[Classic_Oxford_Shirt]": "x[Classic Oxford Shirt]",
   "x[Cotton_Crew_Neck_T-Shirt]": "x[Cotton Crew Neck T-Shirt]",
   "x[French_Terry_Hoodie]": "x[French Terry Hoodie]",
   "x[Denim_Snap_Shirt]": "x[Denim Snap Shirt]",
   "x[Jersey_V-Neck_T-Shirt]": "x[Jersey V-Neck T-Shirt]",
   "x[Unisex_Jogger_Pants]": "x[Unisex Jogger Pants]",
   "x[Flannel_Plaid_Shirt]": "x[Flannel Plaid Shirt]",
   "x[Performance_Polo_Shirt]": "x[Performance Polo Shirt]",
   "x[Long-sleeve_Henley]": "x[Long-sleeve Henley]",
   "x[Heavyweight_Sweatshirt]": "x[Heavyweight Sweatshirt]",
   "x[Linen_Blend_Shirt]": "x[Linen Blend Shirt]",
   "x[Graphic_Print_Tee]": "x[Graphic Print Tee]",
   "x[Quilted_Vest]": "x[Quilted Vest]",
   "x[Chambray_Work_Shirt]": "x[Chambray Work Shirt]",
   "x[Microfiber_Sport_Shirt]": "x[Microfiber Sport Shirt]",
   "x[Wool_Blend_Peacoat]": "x[Wool Blend Peacoat]",
   "x[Seersucker_Short-sleeve]": "x[Seersucker Short-sleeve]",
   "x[Twill_Utility_Shirt]": "x[Twill Utility Shirt]",
   "x[Basic_Scoop_Neck_Tee]": "x[Basic Scoop Neck Tee]",
   "x[Waffle_Knit_Pullover]": "x[Waffle Knit Pullover]",
   "x[Button-Down_Poplin_Shirt]": "x[Button-Down Poplin Shirt]",
   "x[Slub_Cotton_T-Shirt]": "x[Slub Cotton T-Shirt]",
   "x[Drawstring_Cargo_Pants]": "x[Drawstring Cargo Pants]",
   "x[Pima_Cotton_Dress_Shirt]": "x[Pima Cotton Dress Shirt]",
   "x[Pocket_T-Shirt]": "x[Pocket T-Shirt]",
   "x[Fleece-Lined_Track_Suit]": "x[Fleece-Lined Track Suit]",
   "x[Printed_Camp_Collar_Shirt]": "x[Printed Camp Collar Shirt]",
   "x[Performance_Dry-Fit_Tee]": "x[Performance Dry-Fit Tee]",
   "x[Corduroy_Shacket]": "x[Corduroy Shacket]",
   "x[Herringbone_Casual_Shirt]": "x[Herringbone Casual Shirt]",
   "x[Organic_Cotton_Tee]": "x[Organic Cotton Tee]",
   "x[Cashmere_Blend_Sweater]": "x[Cashmere Blend Sweater]",
   "x[Striped_Rugby_Shirt]": "x[Striped Rugby Shirt]",
   "x[Raglan_Sleeve_Tee]": "x[Raglan Sleeve Tee]",
   "x[Waterproof_Rain_Jacket]": "x[Waterproof Rain Jacket]",
   "x[Micro-check_Business_Shirt]": "x[Micro-check Business Shirt]",
   "x[Heather_Grey_T-Shirt]": "x[Heather Grey T-Shirt]",
   "x[Slim_Fit_Chinos]": "x[Slim Fit Chinos]",
   "x[Band_Collar_Shirt]": "x[Band Collar Shirt]",
   "x[Triblend_Jersey_Tee]": "x[Triblend Jersey Tee]",
   "x[Sherpa_Lined_Pullover]": "x[Sherpa Lined Pullover]",
   "x[Western_Style_Shirt]": "x[Western Style Shirt]",
   "x[Recycled_Poly_T-Shirt]": "x[Recycled Poly T-Shirt]",
   "x[Insulated_Puffer_Jacket]": "x[Insulated Puffer Jacket]",
   "x[End-on-End_Fabric_Shirt]": "x[End-on-End Fabric Shirt]",
   "x[Oversized_Graphic_Tee]": "x[Oversized Graphic Tee]",
   "x[Stretch_Hybrid_Shorts]": "x[Stretch Hybrid Shorts]",
   "x[Non-Iron_Travel_Shirt]": "x[Non-Iron Travel Shirt]",
   "x[Sun-Protective_T-Shirt]": "x[Sun-Protective T-Shirt]",
   "x[Velvet_Bomber_Jacket]": "x[Velvet Bomber Jacket]",
   "x[Madras_Plaid_Shirt]": "x[Madras Plaid Shirt]",
   "x[Tie-Dye_T-Shirt]": "x[Tie-Dye T-Shirt]",
   "x[Lightweight_Windbreaker]": "x[Lightweight Windbreaker]",
   "x[Pinpoint_Oxford_Shirt]": "x[Pinpoint Oxford Shirt]",
   "x[Bamboo_Viscose_Tee]": "x[Bamboo Viscose Tee]",
   "x[Tapered_Denim_Jeans]": "x[Tapered Denim Jeans]",
   "x[Contrast_Collar_Shirt]": "x[Contrast Collar Shirt]",
   "x[Mock_Neck_Long-sleeve]": "x[Mock Neck Long-sleeve]",
   "x[Down-filled_Parka]": "x[Down-filled Parka]",
   "x[Textured_Grid_Shirt]": "x[Textured Grid Shirt]",
   "x[Heavy_Cotton_T-Shirt]": "x[Heavy Cotton T-Shirt]",
   "x[Convertible_Zip-Off_Trousers]": "x[Convertible Zip-Off Trousers]",
   "x[Tencel_Blend_Casual_Shirt]": "x[Tencel Blend Casual Shirt]",
   "x[Relaxed_Fit_Tee]": "x[Relaxed Fit Tee]",
   "x[Faux_Leather_Jacket]": "x[Faux Leather Jacket]",
   "x[Speckled_Slub_Shirt]": "x[Speckled Slub Shirt]",
   "x[Embroidered_T-Shirt]": "x[Embroidered T-Shirt]",
   "x[Knit_Cardigan_Sweater]": "x[Knit Cardigan Sweater]",
   "x[Two-Pocket_Utility_Shirt]": "x[Two-Pocket Utility Shirt]",
   "x[Performance_Stretch_Tee]": "x[Performance Stretch Tee]",
   "x[Technical_Shell_Jacket]": "x[Technical Shell Jacket]",
   "x[Semi-Spread_Collar_Shirt]": "x[Semi-Spread Collar Shirt]",
   "x[Washed_Pigment_Dye_Tee]": "x[Washed Pigment Dye Tee]",
   "x[Twill_Blazer_(Unlined)]": "x[Twill Blazer (Unlined)]",
   "x[Brushed_Twill_Shirt]": "x[Brushed Twill Shirt]",
   "x[Supima_Cotton_Tee]": "x[Supima Cotton Tee]",
   "x[High-Waist_Trousers]": "x[High-Waist Trousers]",
   "x[Printed_Geometric_Shirt]": "x[Printed Geometric Shirt]",
   "x[Minimalist_Plain_Tee]": "x[Minimalist Plain Tee]",
   "x[Reversible_Quilted_Jacket]": "x[Reversible Quilted Jacket]",
   "x[Dobby_Weave_Shirt]": "x[Dobby Weave Shirt]",
   "x[Heavyweight_Boxy_Tee]": "x[Heavyweight Boxy Tee]",
   "x[Casual_Wool_Overcoat]": "x[Casual Wool Overcoat]",
   "x[Half-Zip_Pullover_Shirt]": "x[Half-Zip Pullover Shirt]",
   "x[Drop_Shoulder_T-Shirt]": "x[Drop Shoulder T-Shirt]",
   "x[Straight_Leg_Corduroys]": "x[Straight Leg Corduroys]",
   "x[Camp_Collar_Floral_Shirt]": "x[Camp Collar Floral Shirt]",
   "x[Garment_Dyed_Tee]": "x[Garment Dyed Tee]",
   "x[Suede_Bomber_Jacket]": "x[Suede Bomber Jacket]",
   "x[Melange_Knit_Shirt]": "x[Melange Knit Shirt]",
   "x[Longline_T-Shirt]": "x[Longline T-Shirt]",
   "x[Wide_Leg_Culottes]": "x[Wide Leg Culottes]",
   "x[Tropical_Print_Shirt]": "x[Tropical Print Shirt]",
   "x[Distressed_Cotton_Tee]": "x[Distressed Cotton Tee]",
   "x[Patchwork_Denim_Jacket]": "x[Patchwork Denim Jacket]",
   "x[Micro-Dot_Shirt]": "x[Micro-Dot Shirt]",
   "x[Muscle_Fit_T-Shirt]": "x[Muscle Fit T-Shirt]",
   "x[Woven_Utility_Vest]": "x[Woven Utility Vest]",
   "x[Silk_Blend_Dress_Shirt]": "x[Silk Blend Dress Shirt]",
   "x[Heavyweight_Jersey_T-Shirt]": "x[Heavyweight Jersey T-Shirt]",
   "x[Slim_Fit_Ankle_Grazer]": "x[Slim Fit Ankle Grazer]",
   "x[Lattice_Weave_Shirt]": "x[Lattice Weave Shirt]",
   "x[Boxy_Cropped_Tee]": "x[Boxy Cropped Tee]",
   "x[Reversible_Bomber_Jacket]": "x[Reversible Bomber Jacket]",
   "x[Stretch_Performance_Shirt]": "x[Stretch Performance Shirt]",
   "x[Sublimation_Print_Tee]": "x[Sublimation Print Tee]",
   "x[Brushed_Cotton_Trousers]": "x[Brushed Cotton Trousers]",
   "x[Micro-Waffle_Knit_Shirt]": "x[Micro-Waffle Knit Shirt]",
   "x[Reflective_Detail_Tee]": "x[Reflective Detail Tee]",
   "x[Shearling_Lined_Coat]": "x[Shearling Lined Coat]",
   "x[Printed_Patchwork_Shirt]": "x[Printed Patchwork Shirt]",
   "x[Washed_Effect_T-Shirt]": "x[Washed Effect T-Shirt]",
   "x[Velour_Track_Pants]": "x[Velour Track Pants]",
   "x[Fine_Wale_Corduroy_Shirt]": "x[Fine Wale Corduroy Shirt]",
   "x[Organic_Hemp_T-Shirt]": "x[Organic Hemp T-Shirt]",
   "x[Leather_Biker_Jacket]": "x[Leather Biker Jacket]",
   "x[Supersoft_Modal_Shirt]": "x[Supersoft Modal Shirt]",
   "x[Deep_V-Neck_Tee]": "x[Deep V-Neck Tee]",
   "x[Wool_Blend_Trousers]": "x[Wool Blend Trousers]",
   "x[Contrast_Stitch_Shirt]": "x[Contrast Stitch Shirt]",
   "x[Embroidered_Pocket_Tee]": "x[Embroidered Pocket Tee]",
   "x[Camo_Print_Hoodie]": "x[Camo Print Hoodie]",
   "x[Silk-Touch_Poplin_Shirt]": "x[Silk-Touch Poplin Shirt]",
   "x[Slim_Fit_Pique_Polo]": "x[Slim Fit Pique Polo]",
   "x[Technical_Cargo_Vest]": "x[Technical Cargo Vest]",
   "x[Cutaway_Collar_Shirt]": "x[Cutaway Collar Shirt]",
   "x[Lycra_Blend_Active_Tee]": "x[Lycra Blend Active Tee]",
   "x[Heavy_Knit_Sweater]": "x[Heavy Knit Sweater]",
   "x[Textured_Stripe_Shirt]": "x[Textured Stripe Shirt]",
   "x[Bio-Washed_T-Shirt]": "x[Bio-Washed T-Shirt]",
   "x[Mohair_Blend_Cardigan]": "x[Mohair Blend Cardigan]",
   "x[Shantung_Silk_Shirt]": "x[Shantung Silk Shirt]",
   "x[Distressed_Denim_Tee]": "x[Distressed Denim Tee]",
   "x[High-Tech_Ski_Jacket]": "x[High-Tech Ski Jacket]",
   "x[Fine_Poplin_Check_Shirt]": "x[Fine Poplin Check Shirt]",
   "x[Slim_Fit_Linen_T-Shirt]": "x[Slim Fit Linen T-Shirt]",
   "x[Ripstop_Utility_Pants]": "x[Ripstop Utility Pants]",
   "x[Classic_Barrel_Cuff_Shirt]": "x[Classic Barrel Cuff Shirt]",
   "x[Bamboo_Fiber_Tee]": "x[Bamboo Fiber Tee]",
   "x[Shearling_Trench_Coat]": "x[Shearling Trench Coat]",
   "x[Slim_Fit_Chambray]": "x[Slim Fit Chambray]",
   "x[Printed_Geometric_Tee]": "x[Printed Geometric Tee]",
   "x[Wool_Cargo_Trousers]": "x[Wool Cargo Trousers]",
   "x[Two-Tone_Oxford_Shirt]": "x[Two-Tone Oxford Shirt]",
   "x[Slogan_Print_T-Shirt]": "x[Slogan Print T-Shirt]",
   "x[Water-Resistant_Shell]": "x[Water-Resistant Shell]",
   "x[Grandad_Collar_Shirt]": "x[Grandad Collar Shirt]",
   "x[Embroidered_Logo_Tee]": "x[Embroidered Logo Tee]",
   "x[Casual_Knit_Blazer]": "x[Casual Knit Blazer]",
   "x[Eco-Friendly_Hemp_Shirt]": "x[Eco-Friendly Hemp Shirt]",
   "x[Cotton_Slub_Polo]": "x[Cotton Slub Polo]",
   "x[Corduroy_Carpenter_Pants]": "x[Corduroy Carpenter Pants]",
   "x[Linen_Safari_Shirt]": "x[Linen Safari Shirt]",
   "x[Ribbed_Crew_Neck_Tee]": "x[Ribbed Crew Neck Tee]",
   "x[Hybrid_Hiking_Trousers]": "x[Hybrid Hiking Trousers]",
   "x[Brushed_Flannel_Shirt]": "x[Brushed Flannel Shirt]",
   "x[Microfiber_V-Neck]": "x[Microfiber V-Neck]",
   "x[Down-Alternative_Vest]": "x[Down-Alternative Vest]",
   "x[Checked_Seersucker_Shirt]": "x[Checked Seersucker Shirt]",
   "x[Fitted_Basic_Tee]": "x[Fitted Basic Tee]",
   "x[Heavyweight_Denim_Jacket]": "x[Heavyweight Denim Jacket]",
   "x[Woven_Stripe_Shirt]": "x[Woven Stripe Shirt]",
   "x[Tech_Fleece_Pullover]": "x[Tech Fleece Pullover]",
   "x[Pin_Dot_Print_Shirt]": "x[Pin Dot Print Shirt]",
   "x[Supersoft_Modal_T-Shirt]": "x[Supersoft Modal T-Shirt]",
   "x[Water-Resistant_Chinos]": "x[Water-Resistant Chinos]",
   "x[Tailored_Fit_Dress_Shirt]": "x[Tailored Fit Dress Shirt]",
   "x[Printed_Art_Graphic_Tee]": "x[Printed Art Graphic Tee]",
   "x[Cashmere_Cable_Knit]": "x[Cashmere Cable Knit]",
   "x[Plaid_Flannel_Overshirt]": "x[Plaid Flannel Overshirt]",
   "x[Moisture-Wicking_Polo]": "x[Moisture-Wicking Polo]",
   "x[Slim_Tapered_Jeans]": "x[Slim Tapered Jeans]",
   "x[Contrast_Cuff_Shirt]": "x[Contrast Cuff Shirt]",
   "x[Sustainable_Cotton_Tee]": "x[Sustainable Cotton Tee]",
   "x[Trench_Coat_(Lined)]": "x[Trench Coat (Lined)]",
   "x[Spread_Collar_Shirt]": "x[Spread Collar Shirt]",
   "x[Printed_Stripe_Tee]": "x[Printed Stripe Tee]",
   "x[Non-Iron_Twill_Shirt]": "x[Non-Iron Twill Shirt]",
   "x[Recycled_Material_Tee]": "x[Recycled Material Tee]",
   "x[Faux_Fur_Coat]": "x[Faux Fur Coat]",
   "x[Seersucker_Popover_Shirt]": "x[Seersucker Popover Shirt]",
   "x[Graphic_Back_Print_Tee]": "x[Graphic Back Print Tee]",
   "x[Corduroy_Field_Jacket]": "x[Corduroy Field Jacket]",
   "x[Linen/Cotton_Blend_Shirt]": "x[Linen/Cotton Blend Shirt]",
   "x[Lightweight_Jersey_Tee]": "x[Lightweight Jersey Tee]",
   "x[High_Performance_Hiking_Pants]": "x[High Performance Hiking Pants]",
   "x[Cuban_Collar_Shirt]": "x[Cuban Collar Shirt]",
   "x[Faded_Pigment_Tee]": "x[Faded Pigment Tee]",
   "x[Canvas_Chore_Coat]": "x[Canvas Chore Coat]",
   "x[Brushed_Cotton_Shirt]": "x[Brushed Cotton Shirt]",
   "x[Ribbed_Knit_T-Shirt]": "x[Ribbed Knit T-Shirt]",
   "x[Wide_Leg_Trousers]": "x[Wide Leg Trousers]",
   "x[Patchwork_Denim_Shirt]": "x[Patchwork Denim Shirt]",
   "x[Basic_White_T-Shirt]": "x[Basic White T-Shirt]",
   "x[Mohair_Knit_Sweater]": "x[Mohair Knit Sweater]",
   "x[Contrast_Stitch_Polo]": "x[Contrast Stitch Polo]",
   "x[Tailored_Wool_Trousers]": "x[Tailored Wool Trousers]",
   "x[Heavyweight_Denim_Shirt]": "x[Heavyweight Denim Shirt]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 24877.394636015328,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     159.00383141762453
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     204.9808429118774
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     0.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-075": {
  "source_code_sha256": "2e26843ede0d4b97378b625441493bf94152abb27061a6bc6741c58e9bb25798",
  "original_solution_sha256": "f66d7578c64f979d4379b5f3211192585a5f4f68874a8edb433e6daef5bcc4cf",
  "bindings": [
   [
    "C0",
    "x[1]"
   ],
   [
    "C1",
    "x[2]"
   ],
   [
    "C2",
    "x[3]"
   ],
   [
    "C3",
    "x[4]"
   ],
   [
    "C4",
    "x[5]"
   ],
   [
    "C5",
    "x[6]"
   ],
   [
    "C6",
    "x[7]"
   ],
   [
    "C7",
    "x[8]"
   ],
   [
    "C8",
    "x[9]"
   ],
   [
    "C9",
    "x[10]"
   ],
   [
    "C10",
    "x[11]"
   ],
   [
    "C11",
    "x[12]"
   ],
   [
    "C12",
    "x[13]"
   ],
   [
    "C13",
    "x[14]"
   ],
   [
    "C14",
    "x[15]"
   ],
   [
    "C15",
    "x[16]"
   ],
   [
    "C16",
    "x[17]"
   ],
   [
    "C17",
    "x[18]"
   ],
   [
    "C18",
    "x[19]"
   ],
   [
    "C19",
    "x[20]"
   ],
   [
    "C20",
    "x[21]"
   ],
   [
    "C21",
    "x[22]"
   ],
   [
    "C22",
    "x[23]"
   ],
   [
    "C23",
    "x[24]"
   ],
   [
    "C24",
    "x[25]"
   ],
   [
    "C25",
    "x[26]"
   ],
   [
    "C26",
    "x[27]"
   ],
   [
    "C27",
    "x[28]"
   ],
   [
    "C28",
    "x[29]"
   ],
   [
    "C29",
    "x[30]"
   ],
   [
    "C30",
    "x[31]"
   ],
   [
    "C31",
    "x[32]"
   ],
   [
    "C32",
    "x[33]"
   ],
   [
    "C33",
    "x[34]"
   ],
   [
    "C34",
    "x[35]"
   ],
   [
    "C35",
    "x[36]"
   ],
   [
    "C36",
    "x[37]"
   ],
   [
    "C37",
    "x[38]"
   ],
   [
    "C38",
    "x[39]"
   ],
   [
    "C39",
    "x[40]"
   ],
   [
    "C40",
    "x[41]"
   ],
   [
    "C41",
    "x[42]"
   ],
   [
    "C42",
    "x[43]"
   ],
   [
    "C43",
    "x[44]"
   ],
   [
    "C44",
    "x[45]"
   ],
   [
    "C45",
    "x[46]"
   ],
   [
    "C46",
    "x[47]"
   ],
   [
    "C47",
    "x[48]"
   ],
   [
    "C48",
    "x[49]"
   ],
   [
    "C49",
    "x[50]"
   ],
   [
    "C50",
    "x[51]"
   ],
   [
    "C51",
    "x[52]"
   ],
   [
    "C52",
    "x[53]"
   ],
   [
    "C53",
    "x[54]"
   ],
   [
    "C54",
    "x[55]"
   ],
   [
    "C55",
    "x[56]"
   ],
   [
    "C56",
    "x[57]"
   ],
   [
    "C57",
    "x[58]"
   ],
   [
    "C58",
    "x[59]"
   ],
   [
    "C59",
    "x[60]"
   ],
   [
    "C60",
    "x[61]"
   ],
   [
    "C61",
    "x[62]"
   ],
   [
    "C62",
    "x[63]"
   ],
   [
    "C63",
    "x[64]"
   ],
   [
    "C64",
    "x[65]"
   ],
   [
    "C65",
    "x[66]"
   ],
   [
    "C66",
    "x[67]"
   ],
   [
    "C67",
    "x[68]"
   ],
   [
    "C68",
    "x[69]"
   ],
   [
    "C69",
    "x[70]"
   ],
   [
    "C70",
    "x[71]"
   ],
   [
    "C71",
    "x[72]"
   ],
   [
    "C72",
    "x[73]"
   ],
   [
    "C73",
    "x[74]"
   ],
   [
    "C74",
    "x[75]"
   ],
   [
    "C75",
    "x[76]"
   ],
   [
    "C76",
    "x[77]"
   ],
   [
    "C77",
    "x[78]"
   ],
   [
    "C78",
    "x[79]"
   ],
   [
    "C79",
    "x[80]"
   ],
   [
    "C80",
    "x[81]"
   ],
   [
    "C81",
    "x[82]"
   ],
   [
    "C82",
    "x[83]"
   ],
   [
    "C83",
    "x[84]"
   ],
   [
    "C84",
    "x[85]"
   ],
   [
    "C85",
    "x[86]"
   ],
   [
    "C86",
    "x[87]"
   ],
   [
    "C87",
    "x[88]"
   ],
   [
    "C88",
    "x[89]"
   ],
   [
    "C89",
    "x[90]"
   ],
   [
    "C90",
    "x[91]"
   ],
   [
    "C91",
    "x[92]"
   ],
   [
    "C92",
    "x[93]"
   ],
   [
    "C93",
    "x[94]"
   ],
   [
    "C94",
    "x[95]"
   ],
   [
    "C95",
    "x[96]"
   ],
   [
    "C96",
    "x[97]"
   ],
   [
    "C97",
    "x[98]"
   ],
   [
    "C98",
    "x[99]"
   ],
   [
    "C99",
    "x[100]"
   ],
   [
    "C100",
    "x[101]"
   ],
   [
    "C101",
    "x[102]"
   ],
   [
    "C102",
    "x[103]"
   ],
   [
    "C103",
    "x[104]"
   ],
   [
    "C104",
    "x[105]"
   ],
   [
    "C105",
    "x[106]"
   ],
   [
    "C106",
    "x[107]"
   ],
   [
    "C107",
    "x[108]"
   ],
   [
    "C108",
    "x[109]"
   ],
   [
    "C109",
    "x[110]"
   ]
  ],
  "primary_mapping": {
   "y[1]": "x[1]",
   "y[2]": "x[2]",
   "y[3]": "x[3]",
   "y[4]": "x[4]",
   "y[5]": "x[5]",
   "y[6]": "x[6]",
   "y[7]": "x[7]",
   "y[8]": "x[8]",
   "y[9]": "x[9]",
   "y[10]": "x[10]",
   "y[11]": "x[11]",
   "y[12]": "x[12]",
   "y[13]": "x[13]",
   "y[14]": "x[14]",
   "y[15]": "x[15]",
   "y[16]": "x[16]",
   "y[17]": "x[17]",
   "y[18]": "x[18]",
   "y[19]": "x[19]",
   "y[20]": "x[20]",
   "y[21]": "x[21]",
   "y[22]": "x[22]",
   "y[23]": "x[23]",
   "y[24]": "x[24]",
   "y[25]": "x[25]",
   "y[26]": "x[26]",
   "y[27]": "x[27]",
   "y[28]": "x[28]",
   "y[29]": "x[29]",
   "y[30]": "x[30]",
   "y[31]": "x[31]",
   "y[32]": "x[32]",
   "y[33]": "x[33]",
   "y[34]": "x[34]",
   "y[35]": "x[35]",
   "y[36]": "x[36]",
   "y[37]": "x[37]",
   "y[38]": "x[38]",
   "y[39]": "x[39]",
   "y[40]": "x[40]",
   "y[41]": "x[41]",
   "y[42]": "x[42]",
   "y[43]": "x[43]",
   "y[44]": "x[44]",
   "y[45]": "x[45]",
   "y[46]": "x[46]",
   "y[47]": "x[47]",
   "y[48]": "x[48]",
   "y[49]": "x[49]",
   "y[50]": "x[50]",
   "y[51]": "x[51]",
   "y[52]": "x[52]",
   "y[53]": "x[53]",
   "y[54]": "x[54]",
   "y[55]": "x[55]",
   "y[56]": "x[56]",
   "y[57]": "x[57]",
   "y[58]": "x[58]",
   "y[59]": "x[59]",
   "y[60]": "x[60]",
   "y[61]": "x[61]",
   "y[62]": "x[62]",
   "y[63]": "x[63]",
   "y[64]": "x[64]",
   "y[65]": "x[65]",
   "y[66]": "x[66]",
   "y[67]": "x[67]",
   "y[68]": "x[68]",
   "y[69]": "x[69]",
   "y[70]": "x[70]",
   "y[71]": "x[71]",
   "y[72]": "x[72]",
   "y[73]": "x[73]",
   "y[74]": "x[74]",
   "y[75]": "x[75]",
   "y[76]": "x[76]",
   "y[77]": "x[77]",
   "y[78]": "x[78]",
   "y[79]": "x[79]",
   "y[80]": "x[80]",
   "y[81]": "x[81]",
   "y[82]": "x[82]",
   "y[83]": "x[83]",
   "y[84]": "x[84]",
   "y[85]": "x[85]",
   "y[86]": "x[86]",
   "y[87]": "x[87]",
   "y[88]": "x[88]",
   "y[89]": "x[89]",
   "y[90]": "x[90]",
   "y[91]": "x[91]",
   "y[92]": "x[92]",
   "y[93]": "x[93]",
   "y[94]": "x[94]",
   "y[95]": "x[95]",
   "y[96]": "x[96]",
   "y[97]": "x[97]",
   "y[98]": "x[98]",
   "y[99]": "x[99]",
   "y[100]": "x[100]",
   "y[101]": "x[101]",
   "y[102]": "x[102]",
   "y[103]": "x[103]",
   "y[104]": "x[104]",
   "y[105]": "x[105]",
   "y[106]": "x[106]",
   "y[107]": "x[107]",
   "y[108]": "x[108]",
   "y[109]": "x[109]",
   "y[110]": "x[110]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1514.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     1.0
    ],
    [
     "C17",
     1.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     1.0
    ],
    [
     "C26",
     1.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     1.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     1.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     1.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     1.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     1.0
    ],
    [
     "C78",
     1.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     1.0
    ],
    [
     "C90",
     1.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-078": {
  "source_code_sha256": "a98b1ebb2ee8333313fdb27b7268f7013d7f9f847881eab1a7ccfa6d508549cd",
  "original_solution_sha256": "d58e31480dd49e150afb3872590cf379f82c31345c83b8ec473e9e371b0b4535",
  "bindings": [
   [
    "C0",
    "x[coal_011]"
   ],
   [
    "C1",
    "x[gas_040]"
   ],
   [
    "C2",
    "x[renewables_053]"
   ],
   [
    "C3",
    "x[gas_096]"
   ],
   [
    "C4",
    "x[gas_009]"
   ],
   [
    "C5",
    "x[gas_027]"
   ],
   [
    "C6",
    "x[gas_041]"
   ],
   [
    "C7",
    "x[gas_019]"
   ],
   [
    "C8",
    "x[gas_100]"
   ],
   [
    "C9",
    "x[gas_003]"
   ],
   [
    "C10",
    "x[gas_002]"
   ],
   [
    "C11",
    "x[renewables_030]"
   ],
   [
    "C12",
    "x[renewables_001]"
   ],
   [
    "C13",
    "x[gas_001]"
   ],
   [
    "C14",
    "x[gas_035]"
   ],
   [
    "C15",
    "x[renewables_021]"
   ],
   [
    "C16",
    "x[coal_096]"
   ],
   [
    "C17",
    "x[gas_036]"
   ],
   [
    "C18",
    "x[coal_046]"
   ],
   [
    "C19",
    "x[renewables_046]"
   ],
   [
    "C20",
    "x[renewables_038]"
   ],
   [
    "C21",
    "x[renewables_044]"
   ],
   [
    "C22",
    "x[coal_098]"
   ],
   [
    "C23",
    "x[gas_039]"
   ],
   [
    "C24",
    "x[gas_034]"
   ],
   [
    "C25",
    "x[gas_010]"
   ],
   [
    "C26",
    "x[gas_005]"
   ],
   [
    "C27",
    "x[gas_017]"
   ],
   [
    "C28",
    "x[renewables_024]"
   ],
   [
    "C29",
    "x[gas_006]"
   ],
   [
    "C30",
    "x[coal_015]"
   ],
   [
    "C31",
    "x[coal_099]"
   ],
   [
    "C32",
    "x[gas_018]"
   ],
   [
    "C33",
    "x[renewables_042]"
   ],
   [
    "C34",
    "x[gas_012]"
   ],
   [
    "C35",
    "x[coal_001]"
   ],
   [
    "C36",
    "x[gas_045]"
   ],
   [
    "C37",
    "x[gas_097]"
   ],
   [
    "C38",
    "x[gas_014]"
   ],
   [
    "C39",
    "x[gas_028]"
   ],
   [
    "C40",
    "x[gas_011]"
   ],
   [
    "C41",
    "x[renewables_022]"
   ],
   [
    "C42",
    "x[coal_097]"
   ],
   [
    "C43",
    "x[gas_043]"
   ],
   [
    "C44",
    "x[renewables_028]"
   ],
   [
    "C45",
    "x[coal_093]"
   ],
   [
    "C46",
    "x[coal_004]"
   ],
   [
    "C47",
    "x[coal_100]"
   ],
   [
    "C48",
    "x[gas_015]"
   ],
   [
    "C49",
    "x[renewables_041]"
   ],
   [
    "C50",
    "x[renewables_023]"
   ],
   [
    "C51",
    "x[gas_025]"
   ],
   [
    "C52",
    "x[gas_013]"
   ],
   [
    "C53",
    "x[coal_047]"
   ],
   [
    "C54",
    "x[gas_023]"
   ],
   [
    "C55",
    "x[coal_048]"
   ],
   [
    "C56",
    "x[gas_021]"
   ],
   [
    "C57",
    "x[renewables_037]"
   ],
   [
    "C58",
    "x[coal_002]"
   ],
   [
    "C59",
    "x[coal_045]"
   ],
   [
    "C60",
    "x[renewables_002]"
   ],
   [
    "C61",
    "x[gas_029]"
   ],
   [
    "C62",
    "x[gas_026]"
   ],
   [
    "C63",
    "x[coal_008]"
   ],
   [
    "C64",
    "x[gas_098]"
   ],
   [
    "C65",
    "x[gas_042]"
   ],
   [
    "C66",
    "x[renewables_050]"
   ],
   [
    "C67",
    "x[coal_013]"
   ],
   [
    "C68",
    "x[gas_033]"
   ],
   [
    "C69",
    "x[coal_095]"
   ],
   [
    "C70",
    "x[renewables_031]"
   ],
   [
    "C71",
    "x[coal_010]"
   ],
   [
    "C72",
    "x[gas_030]"
   ],
   [
    "C73",
    "x[renewables_033]"
   ],
   [
    "C74",
    "x[renewables_029]"
   ],
   [
    "C75",
    "x[renewables_040]"
   ],
   [
    "C76",
    "x[renewables_032]"
   ],
   [
    "C77",
    "x[coal_012]"
   ],
   [
    "C78",
    "x[gas_067]"
   ],
   [
    "C79",
    "x[coal_006]"
   ],
   [
    "C80",
    "x[gas_064]"
   ],
   [
    "C81",
    "x[renewables_005]"
   ],
   [
    "C82",
    "x[coal_051]"
   ],
   [
    "C83",
    "x[renewables_025]"
   ],
   [
    "C84",
    "x[renewables_047]"
   ],
   [
    "C85",
    "x[gas_031]"
   ],
   [
    "C86",
    "x[coal_014]"
   ],
   [
    "C87",
    "x[gas_008]"
   ],
   [
    "C88",
    "x[gas_038]"
   ],
   [
    "C89",
    "x[renewables_045]"
   ],
   [
    "C90",
    "x[gas_004]"
   ],
   [
    "C91",
    "x[gas_024]"
   ],
   [
    "C92",
    "x[renewables_048]"
   ],
   [
    "C93",
    "x[coal_005]"
   ],
   [
    "C94",
    "x[gas_063]"
   ],
   [
    "C95",
    "x[renewables_052]"
   ],
   [
    "C96",
    "x[coal_043]"
   ],
   [
    "C97",
    "x[gas_037]"
   ],
   [
    "C98",
    "x[renewables_054]"
   ],
   [
    "C99",
    "x[renewables_051]"
   ],
   [
    "C100",
    "x[coal_049]"
   ],
   [
    "C101",
    "x[gas_066]"
   ],
   [
    "C102",
    "x[renewables_027]"
   ],
   [
    "C103",
    "x[renewables_039]"
   ],
   [
    "C104",
    "x[gas_016]"
   ],
   [
    "C105",
    "x[gas_022]"
   ],
   [
    "C106",
    "x[renewables_056]"
   ],
   [
    "C107",
    "x[renewables_043]"
   ],
   [
    "C108",
    "x[renewables_055]"
   ],
   [
    "C109",
    "x[coal_091]"
   ],
   [
    "C110",
    "x[renewables_026]"
   ],
   [
    "C111",
    "x[coal_050]"
   ],
   [
    "C112",
    "x[coal_044]"
   ],
   [
    "C113",
    "x[gas_020]"
   ],
   [
    "C114",
    "x[gas_062]"
   ],
   [
    "C115",
    "x[coal_009]"
   ],
   [
    "C116",
    "x[gas_099]"
   ],
   [
    "C117",
    "x[renewables_036]"
   ],
   [
    "C118",
    "x[renewables_035]"
   ],
   [
    "C119",
    "x[coal_094]"
   ],
   [
    "C120",
    "x[gas_065]"
   ],
   [
    "C121",
    "x[renewables_004]"
   ],
   [
    "C122",
    "x[coal_003]"
   ],
   [
    "C123",
    "x[gas_007]"
   ],
   [
    "C124",
    "x[coal_092]"
   ],
   [
    "C125",
    "x[renewables_034]"
   ],
   [
    "C126",
    "x[gas_044]"
   ],
   [
    "C127",
    "x[gas_032]"
   ],
   [
    "C128",
    "x[renewables_003]"
   ],
   [
    "C129",
    "x[renewables_049]"
   ],
   [
    "C130",
    "x[coal_007]"
   ]
  ],
  "primary_mapping": {
   "x[0]": "x[coal_001]",
   "x[1]": "x[coal_002]",
   "x[2]": "x[coal_003]",
   "x[3]": "x[coal_004]",
   "x[4]": "x[coal_005]",
   "x[5]": "x[coal_006]",
   "x[6]": "x[coal_007]",
   "x[7]": "x[coal_008]",
   "x[8]": "x[coal_009]",
   "x[9]": "x[coal_010]",
   "x[10]": "x[coal_011]",
   "x[11]": "x[coal_012]",
   "x[12]": "x[coal_013]",
   "x[13]": "x[coal_014]",
   "x[14]": "x[coal_015]",
   "x[15]": "x[coal_043]",
   "x[16]": "x[coal_044]",
   "x[17]": "x[coal_045]",
   "x[18]": "x[coal_046]",
   "x[19]": "x[coal_047]",
   "x[20]": "x[coal_048]",
   "x[21]": "x[coal_049]",
   "x[22]": "x[coal_050]",
   "x[23]": "x[coal_051]",
   "x[24]": "x[coal_091]",
   "x[25]": "x[coal_092]",
   "x[26]": "x[coal_093]",
   "x[27]": "x[coal_094]",
   "x[28]": "x[coal_095]",
   "x[29]": "x[coal_096]",
   "x[30]": "x[coal_097]",
   "x[31]": "x[coal_098]",
   "x[32]": "x[coal_099]",
   "x[33]": "x[coal_100]",
   "x[34]": "x[gas_001]",
   "x[35]": "x[gas_002]",
   "x[36]": "x[gas_003]",
   "x[37]": "x[gas_004]",
   "x[38]": "x[gas_005]",
   "x[39]": "x[gas_006]",
   "x[40]": "x[gas_007]",
   "x[41]": "x[gas_008]",
   "x[42]": "x[gas_009]",
   "x[43]": "x[gas_010]",
   "x[44]": "x[gas_011]",
   "x[45]": "x[gas_012]",
   "x[46]": "x[gas_013]",
   "x[47]": "x[gas_014]",
   "x[48]": "x[gas_015]",
   "x[49]": "x[gas_016]",
   "x[50]": "x[gas_017]",
   "x[51]": "x[gas_018]",
   "x[52]": "x[gas_019]",
   "x[53]": "x[gas_020]",
   "x[54]": "x[gas_021]",
   "x[55]": "x[gas_022]",
   "x[56]": "x[gas_023]",
   "x[57]": "x[gas_024]",
   "x[58]": "x[gas_025]",
   "x[59]": "x[gas_026]",
   "x[60]": "x[gas_027]",
   "x[61]": "x[gas_028]",
   "x[62]": "x[gas_029]",
   "x[63]": "x[gas_030]",
   "x[64]": "x[gas_031]",
   "x[65]": "x[gas_032]",
   "x[66]": "x[gas_033]",
   "x[67]": "x[gas_034]",
   "x[68]": "x[gas_035]",
   "x[69]": "x[gas_036]",
   "x[70]": "x[gas_037]",
   "x[71]": "x[gas_038]",
   "x[72]": "x[gas_039]",
   "x[73]": "x[gas_040]",
   "x[74]": "x[gas_041]",
   "x[75]": "x[gas_042]",
   "x[76]": "x[gas_043]",
   "x[77]": "x[gas_044]",
   "x[78]": "x[gas_045]",
   "x[79]": "x[gas_062]",
   "x[80]": "x[gas_063]",
   "x[81]": "x[gas_064]",
   "x[82]": "x[gas_065]",
   "x[83]": "x[gas_066]",
   "x[84]": "x[gas_067]",
   "x[85]": "x[gas_096]",
   "x[86]": "x[gas_097]",
   "x[87]": "x[gas_098]",
   "x[88]": "x[gas_099]",
   "x[89]": "x[gas_100]",
   "x[90]": "x[renewables_001]",
   "x[91]": "x[renewables_002]",
   "x[92]": "x[renewables_003]",
   "x[93]": "x[renewables_004]",
   "x[94]": "x[renewables_005]",
   "x[95]": "x[renewables_021]",
   "x[96]": "x[renewables_022]",
   "x[97]": "x[renewables_023]",
   "x[98]": "x[renewables_024]",
   "x[99]": "x[renewables_025]",
   "x[100]": "x[renewables_026]",
   "x[101]": "x[renewables_027]",
   "x[102]": "x[renewables_028]",
   "x[103]": "x[renewables_029]",
   "x[104]": "x[renewables_030]",
   "x[105]": "x[renewables_031]",
   "x[106]": "x[renewables_032]",
   "x[107]": "x[renewables_033]",
   "x[108]": "x[renewables_034]",
   "x[109]": "x[renewables_035]",
   "x[110]": "x[renewables_036]",
   "x[111]": "x[renewables_037]",
   "x[112]": "x[renewables_038]",
   "x[113]": "x[renewables_039]",
   "x[114]": "x[renewables_040]",
   "x[115]": "x[renewables_041]",
   "x[116]": "x[renewables_042]",
   "x[117]": "x[renewables_043]",
   "x[118]": "x[renewables_044]",
   "x[119]": "x[renewables_045]",
   "x[120]": "x[renewables_046]",
   "x[121]": "x[renewables_047]",
   "x[122]": "x[renewables_048]",
   "x[123]": "x[renewables_049]",
   "x[124]": "x[renewables_050]",
   "x[125]": "x[renewables_051]",
   "x[126]": "x[renewables_052]",
   "x[127]": "x[renewables_053]",
   "x[128]": "x[renewables_054]",
   "x[129]": "x[renewables_055]",
   "x[130]": "x[renewables_056]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 152.76,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     4.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     -0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     -0.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     -0.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     -0.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     -0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     -0.0
    ],
    [
     "C113",
     -0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     -0.0
    ],
    [
     "C116",
     -0.0
    ],
    [
     "C117",
     -0.0
    ],
    [
     "C118",
     -0.0
    ],
    [
     "C119",
     -0.0
    ],
    [
     "C120",
     -0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     -0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     -0.0
    ],
    [
     "C125",
     -0.0
    ],
    [
     "C126",
     -0.0
    ],
    [
     "C127",
     -0.0
    ],
    [
     "C128",
     -0.0
    ],
    [
     "C129",
     -0.0
    ],
    [
     "C130",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-082": {
  "source_code_sha256": "e026f2ea0c4bf23aef83b7f7982e945699011214b833e4510e163ac04f7abbc3",
  "original_solution_sha256": "87752db3814dc59eb7fe379d45bb8ac8fc68b7473525457aa99fd5c4aae8a956",
  "bindings": [
   [
    "C0",
    "x[Depot,A]"
   ],
   [
    "C1",
    "x[Depot,B]"
   ],
   [
    "C2",
    "x[Depot,C]"
   ],
   [
    "C3",
    "x[A,Depot]"
   ],
   [
    "C4",
    "x[A,B]"
   ],
   [
    "C5",
    "x[A,C]"
   ],
   [
    "C6",
    "x[B,Depot]"
   ],
   [
    "C7",
    "x[B,A]"
   ],
   [
    "C8",
    "x[B,C]"
   ],
   [
    "C9",
    "x[C,Depot]"
   ],
   [
    "C10",
    "x[C,A]"
   ],
   [
    "C11",
    "x[C,B]"
   ],
   [
    "C12",
    "u[A]"
   ],
   [
    "C13",
    "u[B]"
   ],
   [
    "C14",
    "u[C]"
   ]
  ],
  "primary_mapping": {
   "X[0,1]": "x[Depot,A]",
   "X[0,2]": "x[Depot,B]",
   "X[0,3]": "x[Depot,C]",
   "X[1,0]": "x[A,Depot]",
   "X[1,2]": "x[A,B]",
   "X[1,3]": "x[A,C]",
   "X[2,0]": "x[B,Depot]",
   "X[2,1]": "x[B,A]",
   "X[2,3]": "x[B,C]",
   "X[3,0]": "x[C,Depot]",
   "X[3,1]": "x[C,A]",
   "X[3,2]": "x[C,B]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 199.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     1.0
    ],
    [
     "C3",
     1.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     1.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     1.0
    ],
    [
     "C12",
     3.0
    ],
    [
     "C13",
     2.0
    ],
    [
     "C14",
     1.0
    ]
   ],
   "error": null
  }
 },
 "OR-085": {
  "source_code_sha256": "4036c693c5265aa5daa800105ba6192da74d84e4724278ac89293ce75815b105",
  "original_solution_sha256": "fe74b60663b403dbaab2514ef19c7245cb72afaf3b1c22bda46e7c9a038d2f2b",
  "bindings": [
   [
    "C0",
    "x[1,2]"
   ],
   [
    "C1",
    "x[1,3]"
   ],
   [
    "C2",
    "x[1,4]"
   ],
   [
    "C3",
    "x[1,5]"
   ],
   [
    "C4",
    "x[1,6]"
   ],
   [
    "C5",
    "x[1,7]"
   ],
   [
    "C6",
    "x[1,8]"
   ],
   [
    "C7",
    "x[1,9]"
   ],
   [
    "C8",
    "x[1,10]"
   ],
   [
    "C9",
    "x[1,11]"
   ],
   [
    "C10",
    "x[1,12]"
   ],
   [
    "C11",
    "x[1,13]"
   ],
   [
    "C12",
    "x[1,14]"
   ],
   [
    "C13",
    "x[1,15]"
   ],
   [
    "C14",
    "x[2,1]"
   ],
   [
    "C15",
    "x[2,3]"
   ],
   [
    "C16",
    "x[2,4]"
   ],
   [
    "C17",
    "x[2,5]"
   ],
   [
    "C18",
    "x[2,6]"
   ],
   [
    "C19",
    "x[2,7]"
   ],
   [
    "C20",
    "x[2,8]"
   ],
   [
    "C21",
    "x[2,9]"
   ],
   [
    "C22",
    "x[2,10]"
   ],
   [
    "C23",
    "x[2,11]"
   ],
   [
    "C24",
    "x[2,12]"
   ],
   [
    "C25",
    "x[2,13]"
   ],
   [
    "C26",
    "x[2,14]"
   ],
   [
    "C27",
    "x[2,15]"
   ],
   [
    "C28",
    "x[3,1]"
   ],
   [
    "C29",
    "x[3,2]"
   ],
   [
    "C30",
    "x[3,4]"
   ],
   [
    "C31",
    "x[3,5]"
   ],
   [
    "C32",
    "x[3,6]"
   ],
   [
    "C33",
    "x[3,7]"
   ],
   [
    "C34",
    "x[3,8]"
   ],
   [
    "C35",
    "x[3,9]"
   ],
   [
    "C36",
    "x[3,10]"
   ],
   [
    "C37",
    "x[3,11]"
   ],
   [
    "C38",
    "x[3,12]"
   ],
   [
    "C39",
    "x[3,13]"
   ],
   [
    "C40",
    "x[3,14]"
   ],
   [
    "C41",
    "x[3,15]"
   ],
   [
    "C42",
    "x[4,1]"
   ],
   [
    "C43",
    "x[4,2]"
   ],
   [
    "C44",
    "x[4,3]"
   ],
   [
    "C45",
    "x[4,5]"
   ],
   [
    "C46",
    "x[4,6]"
   ],
   [
    "C47",
    "x[4,7]"
   ],
   [
    "C48",
    "x[4,8]"
   ],
   [
    "C49",
    "x[4,9]"
   ],
   [
    "C50",
    "x[4,10]"
   ],
   [
    "C51",
    "x[4,11]"
   ],
   [
    "C52",
    "x[4,12]"
   ],
   [
    "C53",
    "x[4,13]"
   ],
   [
    "C54",
    "x[4,14]"
   ],
   [
    "C55",
    "x[4,15]"
   ],
   [
    "C56",
    "x[5,1]"
   ],
   [
    "C57",
    "x[5,2]"
   ],
   [
    "C58",
    "x[5,3]"
   ],
   [
    "C59",
    "x[5,4]"
   ],
   [
    "C60",
    "x[5,6]"
   ],
   [
    "C61",
    "x[5,7]"
   ],
   [
    "C62",
    "x[5,8]"
   ],
   [
    "C63",
    "x[5,9]"
   ],
   [
    "C64",
    "x[5,10]"
   ],
   [
    "C65",
    "x[5,11]"
   ],
   [
    "C66",
    "x[5,12]"
   ],
   [
    "C67",
    "x[5,13]"
   ],
   [
    "C68",
    "x[5,14]"
   ],
   [
    "C69",
    "x[5,15]"
   ],
   [
    "C70",
    "x[6,1]"
   ],
   [
    "C71",
    "x[6,2]"
   ],
   [
    "C72",
    "x[6,3]"
   ],
   [
    "C73",
    "x[6,4]"
   ],
   [
    "C74",
    "x[6,5]"
   ],
   [
    "C75",
    "x[6,7]"
   ],
   [
    "C76",
    "x[6,8]"
   ],
   [
    "C77",
    "x[6,9]"
   ],
   [
    "C78",
    "x[6,10]"
   ],
   [
    "C79",
    "x[6,11]"
   ],
   [
    "C80",
    "x[6,12]"
   ],
   [
    "C81",
    "x[6,13]"
   ],
   [
    "C82",
    "x[6,14]"
   ],
   [
    "C83",
    "x[6,15]"
   ],
   [
    "C84",
    "x[7,1]"
   ],
   [
    "C85",
    "x[7,2]"
   ],
   [
    "C86",
    "x[7,3]"
   ],
   [
    "C87",
    "x[7,4]"
   ],
   [
    "C88",
    "x[7,5]"
   ],
   [
    "C89",
    "x[7,6]"
   ],
   [
    "C90",
    "x[7,8]"
   ],
   [
    "C91",
    "x[7,9]"
   ],
   [
    "C92",
    "x[7,10]"
   ],
   [
    "C93",
    "x[7,11]"
   ],
   [
    "C94",
    "x[7,12]"
   ],
   [
    "C95",
    "x[7,13]"
   ],
   [
    "C96",
    "x[7,14]"
   ],
   [
    "C97",
    "x[7,15]"
   ],
   [
    "C98",
    "x[8,1]"
   ],
   [
    "C99",
    "x[8,2]"
   ],
   [
    "C100",
    "x[8,3]"
   ],
   [
    "C101",
    "x[8,4]"
   ],
   [
    "C102",
    "x[8,5]"
   ],
   [
    "C103",
    "x[8,6]"
   ],
   [
    "C104",
    "x[8,7]"
   ],
   [
    "C105",
    "x[8,9]"
   ],
   [
    "C106",
    "x[8,10]"
   ],
   [
    "C107",
    "x[8,11]"
   ],
   [
    "C108",
    "x[8,12]"
   ],
   [
    "C109",
    "x[8,13]"
   ],
   [
    "C110",
    "x[8,14]"
   ],
   [
    "C111",
    "x[8,15]"
   ],
   [
    "C112",
    "x[9,1]"
   ],
   [
    "C113",
    "x[9,2]"
   ],
   [
    "C114",
    "x[9,3]"
   ],
   [
    "C115",
    "x[9,4]"
   ],
   [
    "C116",
    "x[9,5]"
   ],
   [
    "C117",
    "x[9,6]"
   ],
   [
    "C118",
    "x[9,7]"
   ],
   [
    "C119",
    "x[9,8]"
   ],
   [
    "C120",
    "x[9,10]"
   ],
   [
    "C121",
    "x[9,11]"
   ],
   [
    "C122",
    "x[9,12]"
   ],
   [
    "C123",
    "x[9,13]"
   ],
   [
    "C124",
    "x[9,14]"
   ],
   [
    "C125",
    "x[9,15]"
   ],
   [
    "C126",
    "x[10,1]"
   ],
   [
    "C127",
    "x[10,2]"
   ],
   [
    "C128",
    "x[10,3]"
   ],
   [
    "C129",
    "x[10,4]"
   ],
   [
    "C130",
    "x[10,5]"
   ],
   [
    "C131",
    "x[10,6]"
   ],
   [
    "C132",
    "x[10,7]"
   ],
   [
    "C133",
    "x[10,8]"
   ],
   [
    "C134",
    "x[10,9]"
   ],
   [
    "C135",
    "x[10,11]"
   ],
   [
    "C136",
    "x[10,12]"
   ],
   [
    "C137",
    "x[10,13]"
   ],
   [
    "C138",
    "x[10,14]"
   ],
   [
    "C139",
    "x[10,15]"
   ],
   [
    "C140",
    "x[11,1]"
   ],
   [
    "C141",
    "x[11,2]"
   ],
   [
    "C142",
    "x[11,3]"
   ],
   [
    "C143",
    "x[11,4]"
   ],
   [
    "C144",
    "x[11,5]"
   ],
   [
    "C145",
    "x[11,6]"
   ],
   [
    "C146",
    "x[11,7]"
   ],
   [
    "C147",
    "x[11,8]"
   ],
   [
    "C148",
    "x[11,9]"
   ],
   [
    "C149",
    "x[11,10]"
   ],
   [
    "C150",
    "x[11,12]"
   ],
   [
    "C151",
    "x[11,13]"
   ],
   [
    "C152",
    "x[11,14]"
   ],
   [
    "C153",
    "x[11,15]"
   ],
   [
    "C154",
    "x[12,1]"
   ],
   [
    "C155",
    "x[12,2]"
   ],
   [
    "C156",
    "x[12,3]"
   ],
   [
    "C157",
    "x[12,4]"
   ],
   [
    "C158",
    "x[12,5]"
   ],
   [
    "C159",
    "x[12,6]"
   ],
   [
    "C160",
    "x[12,7]"
   ],
   [
    "C161",
    "x[12,8]"
   ],
   [
    "C162",
    "x[12,9]"
   ],
   [
    "C163",
    "x[12,10]"
   ],
   [
    "C164",
    "x[12,11]"
   ],
   [
    "C165",
    "x[12,13]"
   ],
   [
    "C166",
    "x[12,14]"
   ],
   [
    "C167",
    "x[12,15]"
   ],
   [
    "C168",
    "x[13,1]"
   ],
   [
    "C169",
    "x[13,2]"
   ],
   [
    "C170",
    "x[13,3]"
   ],
   [
    "C171",
    "x[13,4]"
   ],
   [
    "C172",
    "x[13,5]"
   ],
   [
    "C173",
    "x[13,6]"
   ],
   [
    "C174",
    "x[13,7]"
   ],
   [
    "C175",
    "x[13,8]"
   ],
   [
    "C176",
    "x[13,9]"
   ],
   [
    "C177",
    "x[13,10]"
   ],
   [
    "C178",
    "x[13,11]"
   ],
   [
    "C179",
    "x[13,12]"
   ],
   [
    "C180",
    "x[13,14]"
   ],
   [
    "C181",
    "x[13,15]"
   ],
   [
    "C182",
    "x[14,1]"
   ],
   [
    "C183",
    "x[14,2]"
   ],
   [
    "C184",
    "x[14,3]"
   ],
   [
    "C185",
    "x[14,4]"
   ],
   [
    "C186",
    "x[14,5]"
   ],
   [
    "C187",
    "x[14,6]"
   ],
   [
    "C188",
    "x[14,7]"
   ],
   [
    "C189",
    "x[14,8]"
   ],
   [
    "C190",
    "x[14,9]"
   ],
   [
    "C191",
    "x[14,10]"
   ],
   [
    "C192",
    "x[14,11]"
   ],
   [
    "C193",
    "x[14,12]"
   ],
   [
    "C194",
    "x[14,13]"
   ],
   [
    "C195",
    "x[14,15]"
   ],
   [
    "C196",
    "x[15,1]"
   ],
   [
    "C197",
    "x[15,2]"
   ],
   [
    "C198",
    "x[15,3]"
   ],
   [
    "C199",
    "x[15,4]"
   ],
   [
    "C200",
    "x[15,5]"
   ],
   [
    "C201",
    "x[15,6]"
   ],
   [
    "C202",
    "x[15,7]"
   ],
   [
    "C203",
    "x[15,8]"
   ],
   [
    "C204",
    "x[15,9]"
   ],
   [
    "C205",
    "x[15,10]"
   ],
   [
    "C206",
    "x[15,11]"
   ],
   [
    "C207",
    "x[15,12]"
   ],
   [
    "C208",
    "x[15,13]"
   ],
   [
    "C209",
    "x[15,14]"
   ],
   [
    "C210",
    "u[2]"
   ],
   [
    "C211",
    "u[3]"
   ],
   [
    "C212",
    "u[4]"
   ],
   [
    "C213",
    "u[5]"
   ],
   [
    "C214",
    "u[6]"
   ],
   [
    "C215",
    "u[7]"
   ],
   [
    "C216",
    "u[8]"
   ],
   [
    "C217",
    "u[9]"
   ],
   [
    "C218",
    "u[10]"
   ],
   [
    "C219",
    "u[11]"
   ],
   [
    "C220",
    "u[12]"
   ],
   [
    "C221",
    "u[13]"
   ],
   [
    "C222",
    "u[14]"
   ],
   [
    "C223",
    "u[15]"
   ]
  ],
  "primary_mapping": {
   "x[1,2]": "x[1,2]",
   "x[1,3]": "x[1,3]",
   "x[1,4]": "x[1,4]",
   "x[1,5]": "x[1,5]",
   "x[1,6]": "x[1,6]",
   "x[1,7]": "x[1,7]",
   "x[1,8]": "x[1,8]",
   "x[1,9]": "x[1,9]",
   "x[1,10]": "x[1,10]",
   "x[1,11]": "x[1,11]",
   "x[1,12]": "x[1,12]",
   "x[1,13]": "x[1,13]",
   "x[1,14]": "x[1,14]",
   "x[1,15]": "x[1,15]",
   "x[2,1]": "x[2,1]",
   "x[2,3]": "x[2,3]",
   "x[2,4]": "x[2,4]",
   "x[2,5]": "x[2,5]",
   "x[2,6]": "x[2,6]",
   "x[2,7]": "x[2,7]",
   "x[2,8]": "x[2,8]",
   "x[2,9]": "x[2,9]",
   "x[2,10]": "x[2,10]",
   "x[2,11]": "x[2,11]",
   "x[2,12]": "x[2,12]",
   "x[2,13]": "x[2,13]",
   "x[2,14]": "x[2,14]",
   "x[2,15]": "x[2,15]",
   "x[3,1]": "x[3,1]",
   "x[3,2]": "x[3,2]",
   "x[3,4]": "x[3,4]",
   "x[3,5]": "x[3,5]",
   "x[3,6]": "x[3,6]",
   "x[3,7]": "x[3,7]",
   "x[3,8]": "x[3,8]",
   "x[3,9]": "x[3,9]",
   "x[3,10]": "x[3,10]",
   "x[3,11]": "x[3,11]",
   "x[3,12]": "x[3,12]",
   "x[3,13]": "x[3,13]",
   "x[3,14]": "x[3,14]",
   "x[3,15]": "x[3,15]",
   "x[4,1]": "x[4,1]",
   "x[4,2]": "x[4,2]",
   "x[4,3]": "x[4,3]",
   "x[4,5]": "x[4,5]",
   "x[4,6]": "x[4,6]",
   "x[4,7]": "x[4,7]",
   "x[4,8]": "x[4,8]",
   "x[4,9]": "x[4,9]",
   "x[4,10]": "x[4,10]",
   "x[4,11]": "x[4,11]",
   "x[4,12]": "x[4,12]",
   "x[4,13]": "x[4,13]",
   "x[4,14]": "x[4,14]",
   "x[4,15]": "x[4,15]",
   "x[5,1]": "x[5,1]",
   "x[5,2]": "x[5,2]",
   "x[5,3]": "x[5,3]",
   "x[5,4]": "x[5,4]",
   "x[5,6]": "x[5,6]",
   "x[5,7]": "x[5,7]",
   "x[5,8]": "x[5,8]",
   "x[5,9]": "x[5,9]",
   "x[5,10]": "x[5,10]",
   "x[5,11]": "x[5,11]",
   "x[5,12]": "x[5,12]",
   "x[5,13]": "x[5,13]",
   "x[5,14]": "x[5,14]",
   "x[5,15]": "x[5,15]",
   "x[6,1]": "x[6,1]",
   "x[6,2]": "x[6,2]",
   "x[6,3]": "x[6,3]",
   "x[6,4]": "x[6,4]",
   "x[6,5]": "x[6,5]",
   "x[6,7]": "x[6,7]",
   "x[6,8]": "x[6,8]",
   "x[6,9]": "x[6,9]",
   "x[6,10]": "x[6,10]",
   "x[6,11]": "x[6,11]",
   "x[6,12]": "x[6,12]",
   "x[6,13]": "x[6,13]",
   "x[6,14]": "x[6,14]",
   "x[6,15]": "x[6,15]",
   "x[7,1]": "x[7,1]",
   "x[7,2]": "x[7,2]",
   "x[7,3]": "x[7,3]",
   "x[7,4]": "x[7,4]",
   "x[7,5]": "x[7,5]",
   "x[7,6]": "x[7,6]",
   "x[7,8]": "x[7,8]",
   "x[7,9]": "x[7,9]",
   "x[7,10]": "x[7,10]",
   "x[7,11]": "x[7,11]",
   "x[7,12]": "x[7,12]",
   "x[7,13]": "x[7,13]",
   "x[7,14]": "x[7,14]",
   "x[7,15]": "x[7,15]",
   "x[8,1]": "x[8,1]",
   "x[8,2]": "x[8,2]",
   "x[8,3]": "x[8,3]",
   "x[8,4]": "x[8,4]",
   "x[8,5]": "x[8,5]",
   "x[8,6]": "x[8,6]",
   "x[8,7]": "x[8,7]",
   "x[8,9]": "x[8,9]",
   "x[8,10]": "x[8,10]",
   "x[8,11]": "x[8,11]",
   "x[8,12]": "x[8,12]",
   "x[8,13]": "x[8,13]",
   "x[8,14]": "x[8,14]",
   "x[8,15]": "x[8,15]",
   "x[9,1]": "x[9,1]",
   "x[9,2]": "x[9,2]",
   "x[9,3]": "x[9,3]",
   "x[9,4]": "x[9,4]",
   "x[9,5]": "x[9,5]",
   "x[9,6]": "x[9,6]",
   "x[9,7]": "x[9,7]",
   "x[9,8]": "x[9,8]",
   "x[9,10]": "x[9,10]",
   "x[9,11]": "x[9,11]",
   "x[9,12]": "x[9,12]",
   "x[9,13]": "x[9,13]",
   "x[9,14]": "x[9,14]",
   "x[9,15]": "x[9,15]",
   "x[10,1]": "x[10,1]",
   "x[10,2]": "x[10,2]",
   "x[10,3]": "x[10,3]",
   "x[10,4]": "x[10,4]",
   "x[10,5]": "x[10,5]",
   "x[10,6]": "x[10,6]",
   "x[10,7]": "x[10,7]",
   "x[10,8]": "x[10,8]",
   "x[10,9]": "x[10,9]",
   "x[10,11]": "x[10,11]",
   "x[10,12]": "x[10,12]",
   "x[10,13]": "x[10,13]",
   "x[10,14]": "x[10,14]",
   "x[10,15]": "x[10,15]",
   "x[11,1]": "x[11,1]",
   "x[11,2]": "x[11,2]",
   "x[11,3]": "x[11,3]",
   "x[11,4]": "x[11,4]",
   "x[11,5]": "x[11,5]",
   "x[11,6]": "x[11,6]",
   "x[11,7]": "x[11,7]",
   "x[11,8]": "x[11,8]",
   "x[11,9]": "x[11,9]",
   "x[11,10]": "x[11,10]",
   "x[11,12]": "x[11,12]",
   "x[11,13]": "x[11,13]",
   "x[11,14]": "x[11,14]",
   "x[11,15]": "x[11,15]",
   "x[12,1]": "x[12,1]",
   "x[12,2]": "x[12,2]",
   "x[12,3]": "x[12,3]",
   "x[12,4]": "x[12,4]",
   "x[12,5]": "x[12,5]",
   "x[12,6]": "x[12,6]",
   "x[12,7]": "x[12,7]",
   "x[12,8]": "x[12,8]",
   "x[12,9]": "x[12,9]",
   "x[12,10]": "x[12,10]",
   "x[12,11]": "x[12,11]",
   "x[12,13]": "x[12,13]",
   "x[12,14]": "x[12,14]",
   "x[12,15]": "x[12,15]",
   "x[13,1]": "x[13,1]",
   "x[13,2]": "x[13,2]",
   "x[13,3]": "x[13,3]",
   "x[13,4]": "x[13,4]",
   "x[13,5]": "x[13,5]",
   "x[13,6]": "x[13,6]",
   "x[13,7]": "x[13,7]",
   "x[13,8]": "x[13,8]",
   "x[13,9]": "x[13,9]",
   "x[13,10]": "x[13,10]",
   "x[13,11]": "x[13,11]",
   "x[13,12]": "x[13,12]",
   "x[13,14]": "x[13,14]",
   "x[13,15]": "x[13,15]",
   "x[14,1]": "x[14,1]",
   "x[14,2]": "x[14,2]",
   "x[14,3]": "x[14,3]",
   "x[14,4]": "x[14,4]",
   "x[14,5]": "x[14,5]",
   "x[14,6]": "x[14,6]",
   "x[14,7]": "x[14,7]",
   "x[14,8]": "x[14,8]",
   "x[14,9]": "x[14,9]",
   "x[14,10]": "x[14,10]",
   "x[14,11]": "x[14,11]",
   "x[14,12]": "x[14,12]",
   "x[14,13]": "x[14,13]",
   "x[14,15]": "x[14,15]",
   "x[15,1]": "x[15,1]",
   "x[15,2]": "x[15,2]",
   "x[15,3]": "x[15,3]",
   "x[15,4]": "x[15,4]",
   "x[15,5]": "x[15,5]",
   "x[15,6]": "x[15,6]",
   "x[15,7]": "x[15,7]",
   "x[15,8]": "x[15,8]",
   "x[15,9]": "x[15,9]",
   "x[15,10]": "x[15,10]",
   "x[15,11]": "x[15,11]",
   "x[15,12]": "x[15,12]",
   "x[15,13]": "x[15,13]",
   "x[15,14]": "x[15,14]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 419.0,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     1.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     1.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     1.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     1.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     1.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     0.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     1.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     0.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     1.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     -0.0
    ],
    [
     "C109",
     1.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     -0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     -0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     0.0
    ],
    [
     "C118",
     1.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     -0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     -0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     -0.0
    ],
    [
     "C133",
     1.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     -0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     -0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     -0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     1.0
    ],
    [
     "C151",
     -0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     -0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     -0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     0.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     -0.0
    ],
    [
     "C161",
     -0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     -0.0
    ],
    [
     "C164",
     -0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     -0.0
    ],
    [
     "C167",
     1.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     -0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     -0.0
    ],
    [
     "C176",
     1.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     -0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     -0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     1.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     -0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     0.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     -0.0
    ],
    [
     "C194",
     -0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     -0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     1.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     0.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     -0.0
    ],
    [
     "C204",
     0.0
    ],
    [
     "C205",
     0.0
    ],
    [
     "C206",
     -0.0
    ],
    [
     "C207",
     -0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     9.0
    ],
    [
     "C211",
     7.0
    ],
    [
     "C212",
     13.0
    ],
    [
     "C213",
     14.0
    ],
    [
     "C214",
     8.0
    ],
    [
     "C215",
     5.0
    ],
    [
     "C216",
     2.0
    ],
    [
     "C217",
     4.0
    ],
    [
     "C218",
     1.0
    ],
    [
     "C219",
     10.0
    ],
    [
     "C220",
     11.0
    ],
    [
     "C221",
     3.0
    ],
    [
     "C222",
     6.0
    ],
    [
     "C223",
     12.0
    ]
   ],
   "error": null
  }
 },
 "OR-088": {
  "source_code_sha256": "431b0a37d254a8db58598e1199999c4a4240202aefb5474df092f4e9b2bb221c",
  "original_solution_sha256": "86c9ad0e683506e4ca94f173550ae399aba5f6554d66044efa48cbda282eadca",
  "bindings": [
   [
    "C0",
    "x[F1,C1]"
   ],
   [
    "C1",
    "x[F1,C2]"
   ],
   [
    "C2",
    "x[F1,C3]"
   ],
   [
    "C3",
    "x[F1,C4]"
   ],
   [
    "C4",
    "x[F1,C5]"
   ],
   [
    "C5",
    "x[F1,C6]"
   ],
   [
    "C6",
    "x[F1,C7]"
   ],
   [
    "C7",
    "x[F1,C8]"
   ],
   [
    "C8",
    "x[F1,C9]"
   ],
   [
    "C9",
    "x[F1,C10]"
   ],
   [
    "C10",
    "x[F1,C11]"
   ],
   [
    "C11",
    "x[F1,C12]"
   ],
   [
    "C12",
    "x[F1,C13]"
   ],
   [
    "C13",
    "x[F1,C14]"
   ],
   [
    "C14",
    "x[F1,C15]"
   ],
   [
    "C15",
    "x[F2,C1]"
   ],
   [
    "C16",
    "x[F2,C2]"
   ],
   [
    "C17",
    "x[F2,C3]"
   ],
   [
    "C18",
    "x[F2,C4]"
   ],
   [
    "C19",
    "x[F2,C5]"
   ],
   [
    "C20",
    "x[F2,C6]"
   ],
   [
    "C21",
    "x[F2,C7]"
   ],
   [
    "C22",
    "x[F2,C8]"
   ],
   [
    "C23",
    "x[F2,C9]"
   ],
   [
    "C24",
    "x[F2,C10]"
   ],
   [
    "C25",
    "x[F2,C11]"
   ],
   [
    "C26",
    "x[F2,C12]"
   ],
   [
    "C27",
    "x[F2,C13]"
   ],
   [
    "C28",
    "x[F2,C14]"
   ],
   [
    "C29",
    "x[F2,C15]"
   ],
   [
    "C30",
    "x[F3,C1]"
   ],
   [
    "C31",
    "x[F3,C2]"
   ],
   [
    "C32",
    "x[F3,C3]"
   ],
   [
    "C33",
    "x[F3,C4]"
   ],
   [
    "C34",
    "x[F3,C5]"
   ],
   [
    "C35",
    "x[F3,C6]"
   ],
   [
    "C36",
    "x[F3,C7]"
   ],
   [
    "C37",
    "x[F3,C8]"
   ],
   [
    "C38",
    "x[F3,C9]"
   ],
   [
    "C39",
    "x[F3,C10]"
   ],
   [
    "C40",
    "x[F3,C11]"
   ],
   [
    "C41",
    "x[F3,C12]"
   ],
   [
    "C42",
    "x[F3,C13]"
   ],
   [
    "C43",
    "x[F3,C14]"
   ],
   [
    "C44",
    "x[F3,C15]"
   ],
   [
    "C45",
    "x[F4,C1]"
   ],
   [
    "C46",
    "x[F4,C2]"
   ],
   [
    "C47",
    "x[F4,C3]"
   ],
   [
    "C48",
    "x[F4,C4]"
   ],
   [
    "C49",
    "x[F4,C5]"
   ],
   [
    "C50",
    "x[F4,C6]"
   ],
   [
    "C51",
    "x[F4,C7]"
   ],
   [
    "C52",
    "x[F4,C8]"
   ],
   [
    "C53",
    "x[F4,C9]"
   ],
   [
    "C54",
    "x[F4,C10]"
   ],
   [
    "C55",
    "x[F4,C11]"
   ],
   [
    "C56",
    "x[F4,C12]"
   ],
   [
    "C57",
    "x[F4,C13]"
   ],
   [
    "C58",
    "x[F4,C14]"
   ],
   [
    "C59",
    "x[F4,C15]"
   ],
   [
    "C60",
    "x[F5,C1]"
   ],
   [
    "C61",
    "x[F5,C2]"
   ],
   [
    "C62",
    "x[F5,C3]"
   ],
   [
    "C63",
    "x[F5,C4]"
   ],
   [
    "C64",
    "x[F5,C5]"
   ],
   [
    "C65",
    "x[F5,C6]"
   ],
   [
    "C66",
    "x[F5,C7]"
   ],
   [
    "C67",
    "x[F5,C8]"
   ],
   [
    "C68",
    "x[F5,C9]"
   ],
   [
    "C69",
    "x[F5,C10]"
   ],
   [
    "C70",
    "x[F5,C11]"
   ],
   [
    "C71",
    "x[F5,C12]"
   ],
   [
    "C72",
    "x[F5,C13]"
   ],
   [
    "C73",
    "x[F5,C14]"
   ],
   [
    "C74",
    "x[F5,C15]"
   ],
   [
    "C75",
    "x[F6,C1]"
   ],
   [
    "C76",
    "x[F6,C2]"
   ],
   [
    "C77",
    "x[F6,C3]"
   ],
   [
    "C78",
    "x[F6,C4]"
   ],
   [
    "C79",
    "x[F6,C5]"
   ],
   [
    "C80",
    "x[F6,C6]"
   ],
   [
    "C81",
    "x[F6,C7]"
   ],
   [
    "C82",
    "x[F6,C8]"
   ],
   [
    "C83",
    "x[F6,C9]"
   ],
   [
    "C84",
    "x[F6,C10]"
   ],
   [
    "C85",
    "x[F6,C11]"
   ],
   [
    "C86",
    "x[F6,C12]"
   ],
   [
    "C87",
    "x[F6,C13]"
   ],
   [
    "C88",
    "x[F6,C14]"
   ],
   [
    "C89",
    "x[F6,C15]"
   ],
   [
    "C90",
    "x[F7,C1]"
   ],
   [
    "C91",
    "x[F7,C2]"
   ],
   [
    "C92",
    "x[F7,C3]"
   ],
   [
    "C93",
    "x[F7,C4]"
   ],
   [
    "C94",
    "x[F7,C5]"
   ],
   [
    "C95",
    "x[F7,C6]"
   ],
   [
    "C96",
    "x[F7,C7]"
   ],
   [
    "C97",
    "x[F7,C8]"
   ],
   [
    "C98",
    "x[F7,C9]"
   ],
   [
    "C99",
    "x[F7,C10]"
   ],
   [
    "C100",
    "x[F7,C11]"
   ],
   [
    "C101",
    "x[F7,C12]"
   ],
   [
    "C102",
    "x[F7,C13]"
   ],
   [
    "C103",
    "x[F7,C14]"
   ],
   [
    "C104",
    "x[F7,C15]"
   ],
   [
    "C105",
    "x[F8,C1]"
   ],
   [
    "C106",
    "x[F8,C2]"
   ],
   [
    "C107",
    "x[F8,C3]"
   ],
   [
    "C108",
    "x[F8,C4]"
   ],
   [
    "C109",
    "x[F8,C5]"
   ],
   [
    "C110",
    "x[F8,C6]"
   ],
   [
    "C111",
    "x[F8,C7]"
   ],
   [
    "C112",
    "x[F8,C8]"
   ],
   [
    "C113",
    "x[F8,C9]"
   ],
   [
    "C114",
    "x[F8,C10]"
   ],
   [
    "C115",
    "x[F8,C11]"
   ],
   [
    "C116",
    "x[F8,C12]"
   ],
   [
    "C117",
    "x[F8,C13]"
   ],
   [
    "C118",
    "x[F8,C14]"
   ],
   [
    "C119",
    "x[F8,C15]"
   ],
   [
    "C120",
    "x[F9,C1]"
   ],
   [
    "C121",
    "x[F9,C2]"
   ],
   [
    "C122",
    "x[F9,C3]"
   ],
   [
    "C123",
    "x[F9,C4]"
   ],
   [
    "C124",
    "x[F9,C5]"
   ],
   [
    "C125",
    "x[F9,C6]"
   ],
   [
    "C126",
    "x[F9,C7]"
   ],
   [
    "C127",
    "x[F9,C8]"
   ],
   [
    "C128",
    "x[F9,C9]"
   ],
   [
    "C129",
    "x[F9,C10]"
   ],
   [
    "C130",
    "x[F9,C11]"
   ],
   [
    "C131",
    "x[F9,C12]"
   ],
   [
    "C132",
    "x[F9,C13]"
   ],
   [
    "C133",
    "x[F9,C14]"
   ],
   [
    "C134",
    "x[F9,C15]"
   ],
   [
    "C135",
    "x[F10,C1]"
   ],
   [
    "C136",
    "x[F10,C2]"
   ],
   [
    "C137",
    "x[F10,C3]"
   ],
   [
    "C138",
    "x[F10,C4]"
   ],
   [
    "C139",
    "x[F10,C5]"
   ],
   [
    "C140",
    "x[F10,C6]"
   ],
   [
    "C141",
    "x[F10,C7]"
   ],
   [
    "C142",
    "x[F10,C8]"
   ],
   [
    "C143",
    "x[F10,C9]"
   ],
   [
    "C144",
    "x[F10,C10]"
   ],
   [
    "C145",
    "x[F10,C11]"
   ],
   [
    "C146",
    "x[F10,C12]"
   ],
   [
    "C147",
    "x[F10,C13]"
   ],
   [
    "C148",
    "x[F10,C14]"
   ],
   [
    "C149",
    "x[F10,C15]"
   ],
   [
    "C150",
    "x[F11,C1]"
   ],
   [
    "C151",
    "x[F11,C2]"
   ],
   [
    "C152",
    "x[F11,C3]"
   ],
   [
    "C153",
    "x[F11,C4]"
   ],
   [
    "C154",
    "x[F11,C5]"
   ],
   [
    "C155",
    "x[F11,C6]"
   ],
   [
    "C156",
    "x[F11,C7]"
   ],
   [
    "C157",
    "x[F11,C8]"
   ],
   [
    "C158",
    "x[F11,C9]"
   ],
   [
    "C159",
    "x[F11,C10]"
   ],
   [
    "C160",
    "x[F11,C11]"
   ],
   [
    "C161",
    "x[F11,C12]"
   ],
   [
    "C162",
    "x[F11,C13]"
   ],
   [
    "C163",
    "x[F11,C14]"
   ],
   [
    "C164",
    "x[F11,C15]"
   ],
   [
    "C165",
    "x[F12,C1]"
   ],
   [
    "C166",
    "x[F12,C2]"
   ],
   [
    "C167",
    "x[F12,C3]"
   ],
   [
    "C168",
    "x[F12,C4]"
   ],
   [
    "C169",
    "x[F12,C5]"
   ],
   [
    "C170",
    "x[F12,C6]"
   ],
   [
    "C171",
    "x[F12,C7]"
   ],
   [
    "C172",
    "x[F12,C8]"
   ],
   [
    "C173",
    "x[F12,C9]"
   ],
   [
    "C174",
    "x[F12,C10]"
   ],
   [
    "C175",
    "x[F12,C11]"
   ],
   [
    "C176",
    "x[F12,C12]"
   ],
   [
    "C177",
    "x[F12,C13]"
   ],
   [
    "C178",
    "x[F12,C14]"
   ],
   [
    "C179",
    "x[F12,C15]"
   ],
   [
    "C180",
    "x[F13,C1]"
   ],
   [
    "C181",
    "x[F13,C2]"
   ],
   [
    "C182",
    "x[F13,C3]"
   ],
   [
    "C183",
    "x[F13,C4]"
   ],
   [
    "C184",
    "x[F13,C5]"
   ],
   [
    "C185",
    "x[F13,C6]"
   ],
   [
    "C186",
    "x[F13,C7]"
   ],
   [
    "C187",
    "x[F13,C8]"
   ],
   [
    "C188",
    "x[F13,C9]"
   ],
   [
    "C189",
    "x[F13,C10]"
   ],
   [
    "C190",
    "x[F13,C11]"
   ],
   [
    "C191",
    "x[F13,C12]"
   ],
   [
    "C192",
    "x[F13,C13]"
   ],
   [
    "C193",
    "x[F13,C14]"
   ],
   [
    "C194",
    "x[F13,C15]"
   ],
   [
    "C195",
    "x[F14,C1]"
   ],
   [
    "C196",
    "x[F14,C2]"
   ],
   [
    "C197",
    "x[F14,C3]"
   ],
   [
    "C198",
    "x[F14,C4]"
   ],
   [
    "C199",
    "x[F14,C5]"
   ],
   [
    "C200",
    "x[F14,C6]"
   ],
   [
    "C201",
    "x[F14,C7]"
   ],
   [
    "C202",
    "x[F14,C8]"
   ],
   [
    "C203",
    "x[F14,C9]"
   ],
   [
    "C204",
    "x[F14,C10]"
   ],
   [
    "C205",
    "x[F14,C11]"
   ],
   [
    "C206",
    "x[F14,C12]"
   ],
   [
    "C207",
    "x[F14,C13]"
   ],
   [
    "C208",
    "x[F14,C14]"
   ],
   [
    "C209",
    "x[F14,C15]"
   ],
   [
    "C210",
    "x[F15,C1]"
   ],
   [
    "C211",
    "x[F15,C2]"
   ],
   [
    "C212",
    "x[F15,C3]"
   ],
   [
    "C213",
    "x[F15,C4]"
   ],
   [
    "C214",
    "x[F15,C5]"
   ],
   [
    "C215",
    "x[F15,C6]"
   ],
   [
    "C216",
    "x[F15,C7]"
   ],
   [
    "C217",
    "x[F15,C8]"
   ],
   [
    "C218",
    "x[F15,C9]"
   ],
   [
    "C219",
    "x[F15,C10]"
   ],
   [
    "C220",
    "x[F15,C11]"
   ],
   [
    "C221",
    "x[F15,C12]"
   ],
   [
    "C222",
    "x[F15,C13]"
   ],
   [
    "C223",
    "x[F15,C14]"
   ],
   [
    "C224",
    "x[F15,C15]"
   ],
   [
    "C225",
    "y[F1]"
   ],
   [
    "C226",
    "y[F2]"
   ],
   [
    "C227",
    "y[F3]"
   ],
   [
    "C228",
    "y[F4]"
   ],
   [
    "C229",
    "y[F5]"
   ],
   [
    "C230",
    "y[F6]"
   ],
   [
    "C231",
    "y[F7]"
   ],
   [
    "C232",
    "y[F8]"
   ],
   [
    "C233",
    "y[F9]"
   ],
   [
    "C234",
    "y[F10]"
   ],
   [
    "C235",
    "y[F11]"
   ],
   [
    "C236",
    "y[F12]"
   ],
   [
    "C237",
    "y[F13]"
   ],
   [
    "C238",
    "y[F14]"
   ],
   [
    "C239",
    "y[F15]"
   ]
  ],
  "primary_mapping": {
   "y_F1": "y[F1]",
   "y_F2": "y[F2]",
   "y_F3": "y[F3]",
   "y_F4": "y[F4]",
   "y_F5": "y[F5]",
   "y_F6": "y[F6]",
   "y_F7": "y[F7]",
   "y_F8": "y[F8]",
   "y_F9": "y[F9]",
   "y_F10": "y[F10]",
   "y_F11": "y[F11]",
   "y_F12": "y[F12]",
   "y_F13": "y[F13]",
   "y_F14": "y[F14]",
   "y_F15": "y[F15]",
   "x_F1_C1": "x[F1,C1]",
   "x_F2_C2": "x[F2,C2]",
   "x_F3_C3": "x[F3,C3]",
   "x_F4_C4": "x[F4,C4]",
   "x_F5_C5": "x[F5,C5]",
   "x_F6_C6": "x[F6,C6]",
   "x_F7_C7": "x[F7,C7]",
   "x_F8_C8": "x[F8,C8]",
   "x_F9_C9": "x[F9,C9]",
   "x_F10_C10": "x[F10,C10]",
   "x_F11_C11": "x[F11,C11]",
   "x_F12_C12": "x[F12,C12]",
   "x_F13_C13": "x[F13,C13]",
   "x_F14_C14": "x[F14,C14]",
   "x_F15_C15": "x[F15,C15]",
   "x_F1_C2": "x[F1,C2]",
   "x_F1_C3": "x[F1,C3]",
   "x_F1_C4": "x[F1,C4]",
   "x_F1_C5": "x[F1,C5]",
   "x_F1_C6": "x[F1,C6]",
   "x_F1_C7": "x[F1,C7]",
   "x_F1_C8": "x[F1,C8]",
   "x_F1_C9": "x[F1,C9]",
   "x_F1_C10": "x[F1,C10]",
   "x_F1_C11": "x[F1,C11]",
   "x_F1_C12": "x[F1,C12]",
   "x_F1_C13": "x[F1,C13]",
   "x_F1_C14": "x[F1,C14]",
   "x_F1_C15": "x[F1,C15]",
   "x_F2_C1": "x[F2,C1]",
   "x_F2_C3": "x[F2,C3]",
   "x_F2_C4": "x[F2,C4]",
   "x_F2_C5": "x[F2,C5]",
   "x_F2_C6": "x[F2,C6]",
   "x_F2_C7": "x[F2,C7]",
   "x_F2_C8": "x[F2,C8]",
   "x_F2_C9": "x[F2,C9]",
   "x_F2_C10": "x[F2,C10]",
   "x_F2_C11": "x[F2,C11]",
   "x_F2_C12": "x[F2,C12]",
   "x_F2_C13": "x[F2,C13]",
   "x_F2_C14": "x[F2,C14]",
   "x_F2_C15": "x[F2,C15]",
   "x_F3_C1": "x[F3,C1]",
   "x_F3_C2": "x[F3,C2]",
   "x_F3_C4": "x[F3,C4]",
   "x_F3_C5": "x[F3,C5]",
   "x_F3_C6": "x[F3,C6]",
   "x_F3_C7": "x[F3,C7]",
   "x_F3_C8": "x[F3,C8]",
   "x_F3_C9": "x[F3,C9]",
   "x_F3_C10": "x[F3,C10]",
   "x_F3_C11": "x[F3,C11]",
   "x_F3_C12": "x[F3,C12]",
   "x_F3_C13": "x[F3,C13]",
   "x_F3_C14": "x[F3,C14]",
   "x_F3_C15": "x[F3,C15]",
   "x_F4_C1": "x[F4,C1]",
   "x_F4_C2": "x[F4,C2]",
   "x_F4_C3": "x[F4,C3]",
   "x_F4_C5": "x[F4,C5]",
   "x_F4_C6": "x[F4,C6]",
   "x_F4_C7": "x[F4,C7]",
   "x_F4_C8": "x[F4,C8]",
   "x_F4_C9": "x[F4,C9]",
   "x_F4_C10": "x[F4,C10]",
   "x_F4_C11": "x[F4,C11]",
   "x_F4_C12": "x[F4,C12]",
   "x_F4_C13": "x[F4,C13]",
   "x_F4_C14": "x[F4,C14]",
   "x_F4_C15": "x[F4,C15]",
   "x_F5_C1": "x[F5,C1]",
   "x_F5_C2": "x[F5,C2]",
   "x_F5_C3": "x[F5,C3]",
   "x_F5_C4": "x[F5,C4]",
   "x_F5_C6": "x[F5,C6]",
   "x_F5_C7": "x[F5,C7]",
   "x_F5_C8": "x[F5,C8]",
   "x_F5_C9": "x[F5,C9]",
   "x_F5_C10": "x[F5,C10]",
   "x_F5_C11": "x[F5,C11]",
   "x_F5_C12": "x[F5,C12]",
   "x_F5_C13": "x[F5,C13]",
   "x_F5_C14": "x[F5,C14]",
   "x_F5_C15": "x[F5,C15]",
   "x_F6_C1": "x[F6,C1]",
   "x_F6_C2": "x[F6,C2]",
   "x_F6_C3": "x[F6,C3]",
   "x_F6_C4": "x[F6,C4]",
   "x_F6_C5": "x[F6,C5]",
   "x_F6_C7": "x[F6,C7]",
   "x_F6_C8": "x[F6,C8]",
   "x_F6_C9": "x[F6,C9]",
   "x_F6_C10": "x[F6,C10]",
   "x_F6_C11": "x[F6,C11]",
   "x_F6_C12": "x[F6,C12]",
   "x_F6_C13": "x[F6,C13]",
   "x_F6_C14": "x[F6,C14]",
   "x_F6_C15": "x[F6,C15]",
   "x_F7_C1": "x[F7,C1]",
   "x_F7_C2": "x[F7,C2]",
   "x_F7_C3": "x[F7,C3]",
   "x_F7_C4": "x[F7,C4]",
   "x_F7_C5": "x[F7,C5]",
   "x_F7_C6": "x[F7,C6]",
   "x_F7_C8": "x[F7,C8]",
   "x_F7_C9": "x[F7,C9]",
   "x_F7_C10": "x[F7,C10]",
   "x_F7_C11": "x[F7,C11]",
   "x_F7_C12": "x[F7,C12]",
   "x_F7_C13": "x[F7,C13]",
   "x_F7_C14": "x[F7,C14]",
   "x_F7_C15": "x[F7,C15]",
   "x_F8_C1": "x[F8,C1]",
   "x_F8_C2": "x[F8,C2]",
   "x_F8_C3": "x[F8,C3]",
   "x_F8_C4": "x[F8,C4]",
   "x_F8_C5": "x[F8,C5]",
   "x_F8_C6": "x[F8,C6]",
   "x_F8_C7": "x[F8,C7]",
   "x_F8_C9": "x[F8,C9]",
   "x_F8_C10": "x[F8,C10]",
   "x_F8_C11": "x[F8,C11]",
   "x_F8_C12": "x[F8,C12]",
   "x_F8_C13": "x[F8,C13]",
   "x_F8_C14": "x[F8,C14]",
   "x_F8_C15": "x[F8,C15]",
   "x_F9_C1": "x[F9,C1]",
   "x_F9_C2": "x[F9,C2]",
   "x_F9_C3": "x[F9,C3]",
   "x_F9_C4": "x[F9,C4]",
   "x_F9_C5": "x[F9,C5]",
   "x_F9_C6": "x[F9,C6]",
   "x_F9_C7": "x[F9,C7]",
   "x_F9_C8": "x[F9,C8]",
   "x_F9_C10": "x[F9,C10]",
   "x_F9_C11": "x[F9,C11]",
   "x_F9_C12": "x[F9,C12]",
   "x_F9_C13": "x[F9,C13]",
   "x_F9_C14": "x[F9,C14]",
   "x_F9_C15": "x[F9,C15]",
   "x_F10_C1": "x[F10,C1]",
   "x_F10_C2": "x[F10,C2]",
   "x_F10_C3": "x[F10,C3]",
   "x_F10_C4": "x[F10,C4]",
   "x_F10_C5": "x[F10,C5]",
   "x_F10_C6": "x[F10,C6]",
   "x_F10_C7": "x[F10,C7]",
   "x_F10_C8": "x[F10,C8]",
   "x_F10_C9": "x[F10,C9]",
   "x_F10_C11": "x[F10,C11]",
   "x_F10_C12": "x[F10,C12]",
   "x_F10_C13": "x[F10,C13]",
   "x_F10_C14": "x[F10,C14]",
   "x_F10_C15": "x[F10,C15]",
   "x_F11_C1": "x[F11,C1]",
   "x_F11_C2": "x[F11,C2]",
   "x_F11_C3": "x[F11,C3]",
   "x_F11_C4": "x[F11,C4]",
   "x_F11_C5": "x[F11,C5]",
   "x_F11_C6": "x[F11,C6]",
   "x_F11_C7": "x[F11,C7]",
   "x_F11_C8": "x[F11,C8]",
   "x_F11_C9": "x[F11,C9]",
   "x_F11_C10": "x[F11,C10]",
   "x_F11_C12": "x[F11,C12]",
   "x_F11_C13": "x[F11,C13]",
   "x_F11_C14": "x[F11,C14]",
   "x_F11_C15": "x[F11,C15]",
   "x_F12_C1": "x[F12,C1]",
   "x_F12_C2": "x[F12,C2]",
   "x_F12_C3": "x[F12,C3]",
   "x_F12_C4": "x[F12,C4]",
   "x_F12_C5": "x[F12,C5]",
   "x_F12_C6": "x[F12,C6]",
   "x_F12_C7": "x[F12,C7]",
   "x_F12_C8": "x[F12,C8]",
   "x_F12_C9": "x[F12,C9]",
   "x_F12_C10": "x[F12,C10]",
   "x_F12_C11": "x[F12,C11]",
   "x_F12_C13": "x[F12,C13]",
   "x_F12_C14": "x[F12,C14]",
   "x_F12_C15": "x[F12,C15]",
   "x_F13_C1": "x[F13,C1]",
   "x_F13_C2": "x[F13,C2]",
   "x_F13_C3": "x[F13,C3]",
   "x_F13_C4": "x[F13,C4]",
   "x_F13_C5": "x[F13,C5]",
   "x_F13_C6": "x[F13,C6]",
   "x_F13_C7": "x[F13,C7]",
   "x_F13_C8": "x[F13,C8]",
   "x_F13_C9": "x[F13,C9]",
   "x_F13_C10": "x[F13,C10]",
   "x_F13_C11": "x[F13,C11]",
   "x_F13_C12": "x[F13,C12]",
   "x_F13_C14": "x[F13,C14]",
   "x_F13_C15": "x[F13,C15]",
   "x_F14_C1": "x[F14,C1]",
   "x_F14_C2": "x[F14,C2]",
   "x_F14_C3": "x[F14,C3]",
   "x_F14_C4": "x[F14,C4]",
   "x_F14_C5": "x[F14,C5]",
   "x_F14_C6": "x[F14,C6]",
   "x_F14_C7": "x[F14,C7]",
   "x_F14_C8": "x[F14,C8]",
   "x_F14_C9": "x[F14,C9]",
   "x_F14_C10": "x[F14,C10]",
   "x_F14_C11": "x[F14,C11]",
   "x_F14_C12": "x[F14,C12]",
   "x_F14_C13": "x[F14,C13]",
   "x_F14_C15": "x[F14,C15]",
   "x_F15_C1": "x[F15,C1]",
   "x_F15_C2": "x[F15,C2]",
   "x_F15_C3": "x[F15,C3]",
   "x_F15_C4": "x[F15,C4]",
   "x_F15_C5": "x[F15,C5]",
   "x_F15_C6": "x[F15,C6]",
   "x_F15_C7": "x[F15,C7]",
   "x_F15_C8": "x[F15,C8]",
   "x_F15_C9": "x[F15,C9]",
   "x_F15_C10": "x[F15,C10]",
   "x_F15_C11": "x[F15,C11]",
   "x_F15_C12": "x[F15,C12]",
   "x_F15_C13": "x[F15,C13]",
   "x_F15_C14": "x[F15,C14]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 150992.5,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     0.0
    ],
    [
     "C4",
     0.0
    ],
    [
     "C5",
     0.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     0.0
    ],
    [
     "C22",
     0.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     82.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     42.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     0.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     0.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     0.0
    ],
    [
     "C63",
     0.0
    ],
    [
     "C64",
     14.0
    ],
    [
     "C65",
     55.0
    ],
    [
     "C66",
     88.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     90.0
    ],
    [
     "C80",
     0.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     43.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     67.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     51.0
    ],
    [
     "C105",
     83.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     73.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     0.0
    ],
    [
     "C115",
     0.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     6.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     91.0
    ],
    [
     "C123",
     1.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     0.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     0.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     0.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     29.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     42.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     73.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     0.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     0.0
    ],
    [
     "C156",
     0.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     66.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     41.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     22.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     107.0
    ],
    [
     "C178",
     0.0
    ],
    [
     "C179",
     0.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     76.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     38.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     0.0
    ],
    [
     "C191",
     0.0
    ],
    [
     "C192",
     0.0
    ],
    [
     "C193",
     37.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     0.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     0.0
    ],
    [
     "C204",
     0.0
    ],
    [
     "C205",
     0.0
    ],
    [
     "C206",
     0.0
    ],
    [
     "C207",
     0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     0.0
    ],
    [
     "C211",
     0.0
    ],
    [
     "C212",
     0.0
    ],
    [
     "C213",
     0.0
    ],
    [
     "C214",
     0.0
    ],
    [
     "C215",
     0.0
    ],
    [
     "C216",
     0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     0.0
    ],
    [
     "C219",
     0.0
    ],
    [
     "C220",
     0.0
    ],
    [
     "C221",
     0.0
    ],
    [
     "C222",
     0.0
    ],
    [
     "C223",
     0.0
    ],
    [
     "C224",
     0.0
    ],
    [
     "C225",
     -0.0
    ],
    [
     "C226",
     1.0
    ],
    [
     "C227",
     0.0
    ],
    [
     "C228",
     -0.0
    ],
    [
     "C229",
     1.0
    ],
    [
     "C230",
     1.0
    ],
    [
     "C231",
     1.0
    ],
    [
     "C232",
     1.0
    ],
    [
     "C233",
     1.0
    ],
    [
     "C234",
     1.0
    ],
    [
     "C235",
     1.0
    ],
    [
     "C236",
     1.0
    ],
    [
     "C237",
     1.0
    ],
    [
     "C238",
     -0.0
    ],
    [
     "C239",
     -0.0
    ]
   ],
   "error": null
  }
 },
 "OR-092": {
  "source_code_sha256": "d90a37b66891818ed55ec2cc986e26db890fa93f1c214292e81085153dc4f016",
  "original_solution_sha256": "bba2a86f5bd3f76cabe18845349d7636dc44d73c4db9ebdec9f0382e410f9e03",
  "bindings": [
   [
    "C0",
    "u[S1,D1]"
   ],
   [
    "C1",
    "u[S1,D2]"
   ],
   [
    "C2",
    "u[S1,D3]"
   ],
   [
    "C3",
    "u[S1,D4]"
   ],
   [
    "C4",
    "u[S1,D5]"
   ],
   [
    "C5",
    "u[S1,D6]"
   ],
   [
    "C6",
    "u[S1,D7]"
   ],
   [
    "C7",
    "u[S1,D8]"
   ],
   [
    "C8",
    "u[S1,D9]"
   ],
   [
    "C9",
    "u[S1,D10]"
   ],
   [
    "C10",
    "u[S1,D11]"
   ],
   [
    "C11",
    "u[S1,D12]"
   ],
   [
    "C12",
    "u[S1,D13]"
   ],
   [
    "C13",
    "u[S1,D14]"
   ],
   [
    "C14",
    "u[S1,D15]"
   ],
   [
    "C15",
    "u[S1,D16]"
   ],
   [
    "C16",
    "u[S1,D17]"
   ],
   [
    "C17",
    "u[S1,D18]"
   ],
   [
    "C18",
    "u[S1,D19]"
   ],
   [
    "C19",
    "u[S1,D20]"
   ],
   [
    "C20",
    "u[S2,D1]"
   ],
   [
    "C21",
    "u[S2,D2]"
   ],
   [
    "C22",
    "u[S2,D3]"
   ],
   [
    "C23",
    "u[S2,D4]"
   ],
   [
    "C24",
    "u[S2,D5]"
   ],
   [
    "C25",
    "u[S2,D6]"
   ],
   [
    "C26",
    "u[S2,D7]"
   ],
   [
    "C27",
    "u[S2,D8]"
   ],
   [
    "C28",
    "u[S2,D9]"
   ],
   [
    "C29",
    "u[S2,D10]"
   ],
   [
    "C30",
    "u[S2,D11]"
   ],
   [
    "C31",
    "u[S2,D12]"
   ],
   [
    "C32",
    "u[S2,D13]"
   ],
   [
    "C33",
    "u[S2,D14]"
   ],
   [
    "C34",
    "u[S2,D15]"
   ],
   [
    "C35",
    "u[S2,D16]"
   ],
   [
    "C36",
    "u[S2,D17]"
   ],
   [
    "C37",
    "u[S2,D18]"
   ],
   [
    "C38",
    "u[S2,D19]"
   ],
   [
    "C39",
    "u[S2,D20]"
   ],
   [
    "C40",
    "u[S3,D1]"
   ],
   [
    "C41",
    "u[S3,D2]"
   ],
   [
    "C42",
    "u[S3,D3]"
   ],
   [
    "C43",
    "u[S3,D4]"
   ],
   [
    "C44",
    "u[S3,D5]"
   ],
   [
    "C45",
    "u[S3,D6]"
   ],
   [
    "C46",
    "u[S3,D7]"
   ],
   [
    "C47",
    "u[S3,D8]"
   ],
   [
    "C48",
    "u[S3,D9]"
   ],
   [
    "C49",
    "u[S3,D10]"
   ],
   [
    "C50",
    "u[S3,D11]"
   ],
   [
    "C51",
    "u[S3,D12]"
   ],
   [
    "C52",
    "u[S3,D13]"
   ],
   [
    "C53",
    "u[S3,D14]"
   ],
   [
    "C54",
    "u[S3,D15]"
   ],
   [
    "C55",
    "u[S3,D16]"
   ],
   [
    "C56",
    "u[S3,D17]"
   ],
   [
    "C57",
    "u[S3,D18]"
   ],
   [
    "C58",
    "u[S3,D19]"
   ],
   [
    "C59",
    "u[S3,D20]"
   ],
   [
    "C60",
    "u[S4,D1]"
   ],
   [
    "C61",
    "u[S4,D2]"
   ],
   [
    "C62",
    "u[S4,D3]"
   ],
   [
    "C63",
    "u[S4,D4]"
   ],
   [
    "C64",
    "u[S4,D5]"
   ],
   [
    "C65",
    "u[S4,D6]"
   ],
   [
    "C66",
    "u[S4,D7]"
   ],
   [
    "C67",
    "u[S4,D8]"
   ],
   [
    "C68",
    "u[S4,D9]"
   ],
   [
    "C69",
    "u[S4,D10]"
   ],
   [
    "C70",
    "u[S4,D11]"
   ],
   [
    "C71",
    "u[S4,D12]"
   ],
   [
    "C72",
    "u[S4,D13]"
   ],
   [
    "C73",
    "u[S4,D14]"
   ],
   [
    "C74",
    "u[S4,D15]"
   ],
   [
    "C75",
    "u[S4,D16]"
   ],
   [
    "C76",
    "u[S4,D17]"
   ],
   [
    "C77",
    "u[S4,D18]"
   ],
   [
    "C78",
    "u[S4,D19]"
   ],
   [
    "C79",
    "u[S4,D20]"
   ],
   [
    "C80",
    "u[S5,D1]"
   ],
   [
    "C81",
    "u[S5,D2]"
   ],
   [
    "C82",
    "u[S5,D3]"
   ],
   [
    "C83",
    "u[S5,D4]"
   ],
   [
    "C84",
    "u[S5,D5]"
   ],
   [
    "C85",
    "u[S5,D6]"
   ],
   [
    "C86",
    "u[S5,D7]"
   ],
   [
    "C87",
    "u[S5,D8]"
   ],
   [
    "C88",
    "u[S5,D9]"
   ],
   [
    "C89",
    "u[S5,D10]"
   ],
   [
    "C90",
    "u[S5,D11]"
   ],
   [
    "C91",
    "u[S5,D12]"
   ],
   [
    "C92",
    "u[S5,D13]"
   ],
   [
    "C93",
    "u[S5,D14]"
   ],
   [
    "C94",
    "u[S5,D15]"
   ],
   [
    "C95",
    "u[S5,D16]"
   ],
   [
    "C96",
    "u[S5,D17]"
   ],
   [
    "C97",
    "u[S5,D18]"
   ],
   [
    "C98",
    "u[S5,D19]"
   ],
   [
    "C99",
    "u[S5,D20]"
   ],
   [
    "C100",
    "u[S6,D1]"
   ],
   [
    "C101",
    "u[S6,D2]"
   ],
   [
    "C102",
    "u[S6,D3]"
   ],
   [
    "C103",
    "u[S6,D4]"
   ],
   [
    "C104",
    "u[S6,D5]"
   ],
   [
    "C105",
    "u[S6,D6]"
   ],
   [
    "C106",
    "u[S6,D7]"
   ],
   [
    "C107",
    "u[S6,D8]"
   ],
   [
    "C108",
    "u[S6,D9]"
   ],
   [
    "C109",
    "u[S6,D10]"
   ],
   [
    "C110",
    "u[S6,D11]"
   ],
   [
    "C111",
    "u[S6,D12]"
   ],
   [
    "C112",
    "u[S6,D13]"
   ],
   [
    "C113",
    "u[S6,D14]"
   ],
   [
    "C114",
    "u[S6,D15]"
   ],
   [
    "C115",
    "u[S6,D16]"
   ],
   [
    "C116",
    "u[S6,D17]"
   ],
   [
    "C117",
    "u[S6,D18]"
   ],
   [
    "C118",
    "u[S6,D19]"
   ],
   [
    "C119",
    "u[S6,D20]"
   ],
   [
    "C120",
    "u[S7,D1]"
   ],
   [
    "C121",
    "u[S7,D2]"
   ],
   [
    "C122",
    "u[S7,D3]"
   ],
   [
    "C123",
    "u[S7,D4]"
   ],
   [
    "C124",
    "u[S7,D5]"
   ],
   [
    "C125",
    "u[S7,D6]"
   ],
   [
    "C126",
    "u[S7,D7]"
   ],
   [
    "C127",
    "u[S7,D8]"
   ],
   [
    "C128",
    "u[S7,D9]"
   ],
   [
    "C129",
    "u[S7,D10]"
   ],
   [
    "C130",
    "u[S7,D11]"
   ],
   [
    "C131",
    "u[S7,D12]"
   ],
   [
    "C132",
    "u[S7,D13]"
   ],
   [
    "C133",
    "u[S7,D14]"
   ],
   [
    "C134",
    "u[S7,D15]"
   ],
   [
    "C135",
    "u[S7,D16]"
   ],
   [
    "C136",
    "u[S7,D17]"
   ],
   [
    "C137",
    "u[S7,D18]"
   ],
   [
    "C138",
    "u[S7,D19]"
   ],
   [
    "C139",
    "u[S7,D20]"
   ],
   [
    "C140",
    "u[S8,D1]"
   ],
   [
    "C141",
    "u[S8,D2]"
   ],
   [
    "C142",
    "u[S8,D3]"
   ],
   [
    "C143",
    "u[S8,D4]"
   ],
   [
    "C144",
    "u[S8,D5]"
   ],
   [
    "C145",
    "u[S8,D6]"
   ],
   [
    "C146",
    "u[S8,D7]"
   ],
   [
    "C147",
    "u[S8,D8]"
   ],
   [
    "C148",
    "u[S8,D9]"
   ],
   [
    "C149",
    "u[S8,D10]"
   ],
   [
    "C150",
    "u[S8,D11]"
   ],
   [
    "C151",
    "u[S8,D12]"
   ],
   [
    "C152",
    "u[S8,D13]"
   ],
   [
    "C153",
    "u[S8,D14]"
   ],
   [
    "C154",
    "u[S8,D15]"
   ],
   [
    "C155",
    "u[S8,D16]"
   ],
   [
    "C156",
    "u[S8,D17]"
   ],
   [
    "C157",
    "u[S8,D18]"
   ],
   [
    "C158",
    "u[S8,D19]"
   ],
   [
    "C159",
    "u[S8,D20]"
   ],
   [
    "C160",
    "u[S9,D1]"
   ],
   [
    "C161",
    "u[S9,D2]"
   ],
   [
    "C162",
    "u[S9,D3]"
   ],
   [
    "C163",
    "u[S9,D4]"
   ],
   [
    "C164",
    "u[S9,D5]"
   ],
   [
    "C165",
    "u[S9,D6]"
   ],
   [
    "C166",
    "u[S9,D7]"
   ],
   [
    "C167",
    "u[S9,D8]"
   ],
   [
    "C168",
    "u[S9,D9]"
   ],
   [
    "C169",
    "u[S9,D10]"
   ],
   [
    "C170",
    "u[S9,D11]"
   ],
   [
    "C171",
    "u[S9,D12]"
   ],
   [
    "C172",
    "u[S9,D13]"
   ],
   [
    "C173",
    "u[S9,D14]"
   ],
   [
    "C174",
    "u[S9,D15]"
   ],
   [
    "C175",
    "u[S9,D16]"
   ],
   [
    "C176",
    "u[S9,D17]"
   ],
   [
    "C177",
    "u[S9,D18]"
   ],
   [
    "C178",
    "u[S9,D19]"
   ],
   [
    "C179",
    "u[S9,D20]"
   ],
   [
    "C180",
    "u[S10,D1]"
   ],
   [
    "C181",
    "u[S10,D2]"
   ],
   [
    "C182",
    "u[S10,D3]"
   ],
   [
    "C183",
    "u[S10,D4]"
   ],
   [
    "C184",
    "u[S10,D5]"
   ],
   [
    "C185",
    "u[S10,D6]"
   ],
   [
    "C186",
    "u[S10,D7]"
   ],
   [
    "C187",
    "u[S10,D8]"
   ],
   [
    "C188",
    "u[S10,D9]"
   ],
   [
    "C189",
    "u[S10,D10]"
   ],
   [
    "C190",
    "u[S10,D11]"
   ],
   [
    "C191",
    "u[S10,D12]"
   ],
   [
    "C192",
    "u[S10,D13]"
   ],
   [
    "C193",
    "u[S10,D14]"
   ],
   [
    "C194",
    "u[S10,D15]"
   ],
   [
    "C195",
    "u[S10,D16]"
   ],
   [
    "C196",
    "u[S10,D17]"
   ],
   [
    "C197",
    "u[S10,D18]"
   ],
   [
    "C198",
    "u[S10,D19]"
   ],
   [
    "C199",
    "u[S10,D20]"
   ],
   [
    "C200",
    "t[S1,D1]"
   ],
   [
    "C201",
    "t[S1,D2]"
   ],
   [
    "C202",
    "t[S1,D3]"
   ],
   [
    "C203",
    "t[S1,D4]"
   ],
   [
    "C204",
    "t[S1,D5]"
   ],
   [
    "C205",
    "t[S1,D6]"
   ],
   [
    "C206",
    "t[S1,D7]"
   ],
   [
    "C207",
    "t[S1,D8]"
   ],
   [
    "C208",
    "t[S1,D9]"
   ],
   [
    "C209",
    "t[S1,D10]"
   ],
   [
    "C210",
    "t[S1,D11]"
   ],
   [
    "C211",
    "t[S1,D12]"
   ],
   [
    "C212",
    "t[S1,D13]"
   ],
   [
    "C213",
    "t[S1,D14]"
   ],
   [
    "C214",
    "t[S1,D15]"
   ],
   [
    "C215",
    "t[S1,D16]"
   ],
   [
    "C216",
    "t[S1,D17]"
   ],
   [
    "C217",
    "t[S1,D18]"
   ],
   [
    "C218",
    "t[S1,D19]"
   ],
   [
    "C219",
    "t[S1,D20]"
   ],
   [
    "C220",
    "t[S2,D1]"
   ],
   [
    "C221",
    "t[S2,D2]"
   ],
   [
    "C222",
    "t[S2,D3]"
   ],
   [
    "C223",
    "t[S2,D4]"
   ],
   [
    "C224",
    "t[S2,D5]"
   ],
   [
    "C225",
    "t[S2,D6]"
   ],
   [
    "C226",
    "t[S2,D7]"
   ],
   [
    "C227",
    "t[S2,D8]"
   ],
   [
    "C228",
    "t[S2,D9]"
   ],
   [
    "C229",
    "t[S2,D10]"
   ],
   [
    "C230",
    "t[S2,D11]"
   ],
   [
    "C231",
    "t[S2,D12]"
   ],
   [
    "C232",
    "t[S2,D13]"
   ],
   [
    "C233",
    "t[S2,D14]"
   ],
   [
    "C234",
    "t[S2,D15]"
   ],
   [
    "C235",
    "t[S2,D16]"
   ],
   [
    "C236",
    "t[S2,D17]"
   ],
   [
    "C237",
    "t[S2,D18]"
   ],
   [
    "C238",
    "t[S2,D19]"
   ],
   [
    "C239",
    "t[S2,D20]"
   ],
   [
    "C240",
    "t[S3,D1]"
   ],
   [
    "C241",
    "t[S3,D2]"
   ],
   [
    "C242",
    "t[S3,D3]"
   ],
   [
    "C243",
    "t[S3,D4]"
   ],
   [
    "C244",
    "t[S3,D5]"
   ],
   [
    "C245",
    "t[S3,D6]"
   ],
   [
    "C246",
    "t[S3,D7]"
   ],
   [
    "C247",
    "t[S3,D8]"
   ],
   [
    "C248",
    "t[S3,D9]"
   ],
   [
    "C249",
    "t[S3,D10]"
   ],
   [
    "C250",
    "t[S3,D11]"
   ],
   [
    "C251",
    "t[S3,D12]"
   ],
   [
    "C252",
    "t[S3,D13]"
   ],
   [
    "C253",
    "t[S3,D14]"
   ],
   [
    "C254",
    "t[S3,D15]"
   ],
   [
    "C255",
    "t[S3,D16]"
   ],
   [
    "C256",
    "t[S3,D17]"
   ],
   [
    "C257",
    "t[S3,D18]"
   ],
   [
    "C258",
    "t[S3,D19]"
   ],
   [
    "C259",
    "t[S3,D20]"
   ],
   [
    "C260",
    "t[S4,D1]"
   ],
   [
    "C261",
    "t[S4,D2]"
   ],
   [
    "C262",
    "t[S4,D3]"
   ],
   [
    "C263",
    "t[S4,D4]"
   ],
   [
    "C264",
    "t[S4,D5]"
   ],
   [
    "C265",
    "t[S4,D6]"
   ],
   [
    "C266",
    "t[S4,D7]"
   ],
   [
    "C267",
    "t[S4,D8]"
   ],
   [
    "C268",
    "t[S4,D9]"
   ],
   [
    "C269",
    "t[S4,D10]"
   ],
   [
    "C270",
    "t[S4,D11]"
   ],
   [
    "C271",
    "t[S4,D12]"
   ],
   [
    "C272",
    "t[S4,D13]"
   ],
   [
    "C273",
    "t[S4,D14]"
   ],
   [
    "C274",
    "t[S4,D15]"
   ],
   [
    "C275",
    "t[S4,D16]"
   ],
   [
    "C276",
    "t[S4,D17]"
   ],
   [
    "C277",
    "t[S4,D18]"
   ],
   [
    "C278",
    "t[S4,D19]"
   ],
   [
    "C279",
    "t[S4,D20]"
   ],
   [
    "C280",
    "t[S5,D1]"
   ],
   [
    "C281",
    "t[S5,D2]"
   ],
   [
    "C282",
    "t[S5,D3]"
   ],
   [
    "C283",
    "t[S5,D4]"
   ],
   [
    "C284",
    "t[S5,D5]"
   ],
   [
    "C285",
    "t[S5,D6]"
   ],
   [
    "C286",
    "t[S5,D7]"
   ],
   [
    "C287",
    "t[S5,D8]"
   ],
   [
    "C288",
    "t[S5,D9]"
   ],
   [
    "C289",
    "t[S5,D10]"
   ],
   [
    "C290",
    "t[S5,D11]"
   ],
   [
    "C291",
    "t[S5,D12]"
   ],
   [
    "C292",
    "t[S5,D13]"
   ],
   [
    "C293",
    "t[S5,D14]"
   ],
   [
    "C294",
    "t[S5,D15]"
   ],
   [
    "C295",
    "t[S5,D16]"
   ],
   [
    "C296",
    "t[S5,D17]"
   ],
   [
    "C297",
    "t[S5,D18]"
   ],
   [
    "C298",
    "t[S5,D19]"
   ],
   [
    "C299",
    "t[S5,D20]"
   ],
   [
    "C300",
    "t[S6,D1]"
   ],
   [
    "C301",
    "t[S6,D2]"
   ],
   [
    "C302",
    "t[S6,D3]"
   ],
   [
    "C303",
    "t[S6,D4]"
   ],
   [
    "C304",
    "t[S6,D5]"
   ],
   [
    "C305",
    "t[S6,D6]"
   ],
   [
    "C306",
    "t[S6,D7]"
   ],
   [
    "C307",
    "t[S6,D8]"
   ],
   [
    "C308",
    "t[S6,D9]"
   ],
   [
    "C309",
    "t[S6,D10]"
   ],
   [
    "C310",
    "t[S6,D11]"
   ],
   [
    "C311",
    "t[S6,D12]"
   ],
   [
    "C312",
    "t[S6,D13]"
   ],
   [
    "C313",
    "t[S6,D14]"
   ],
   [
    "C314",
    "t[S6,D15]"
   ],
   [
    "C315",
    "t[S6,D16]"
   ],
   [
    "C316",
    "t[S6,D17]"
   ],
   [
    "C317",
    "t[S6,D18]"
   ],
   [
    "C318",
    "t[S6,D19]"
   ],
   [
    "C319",
    "t[S6,D20]"
   ],
   [
    "C320",
    "t[S7,D1]"
   ],
   [
    "C321",
    "t[S7,D2]"
   ],
   [
    "C322",
    "t[S7,D3]"
   ],
   [
    "C323",
    "t[S7,D4]"
   ],
   [
    "C324",
    "t[S7,D5]"
   ],
   [
    "C325",
    "t[S7,D6]"
   ],
   [
    "C326",
    "t[S7,D7]"
   ],
   [
    "C327",
    "t[S7,D8]"
   ],
   [
    "C328",
    "t[S7,D9]"
   ],
   [
    "C329",
    "t[S7,D10]"
   ],
   [
    "C330",
    "t[S7,D11]"
   ],
   [
    "C331",
    "t[S7,D12]"
   ],
   [
    "C332",
    "t[S7,D13]"
   ],
   [
    "C333",
    "t[S7,D14]"
   ],
   [
    "C334",
    "t[S7,D15]"
   ],
   [
    "C335",
    "t[S7,D16]"
   ],
   [
    "C336",
    "t[S7,D17]"
   ],
   [
    "C337",
    "t[S7,D18]"
   ],
   [
    "C338",
    "t[S7,D19]"
   ],
   [
    "C339",
    "t[S7,D20]"
   ],
   [
    "C340",
    "t[S8,D1]"
   ],
   [
    "C341",
    "t[S8,D2]"
   ],
   [
    "C342",
    "t[S8,D3]"
   ],
   [
    "C343",
    "t[S8,D4]"
   ],
   [
    "C344",
    "t[S8,D5]"
   ],
   [
    "C345",
    "t[S8,D6]"
   ],
   [
    "C346",
    "t[S8,D7]"
   ],
   [
    "C347",
    "t[S8,D8]"
   ],
   [
    "C348",
    "t[S8,D9]"
   ],
   [
    "C349",
    "t[S8,D10]"
   ],
   [
    "C350",
    "t[S8,D11]"
   ],
   [
    "C351",
    "t[S8,D12]"
   ],
   [
    "C352",
    "t[S8,D13]"
   ],
   [
    "C353",
    "t[S8,D14]"
   ],
   [
    "C354",
    "t[S8,D15]"
   ],
   [
    "C355",
    "t[S8,D16]"
   ],
   [
    "C356",
    "t[S8,D17]"
   ],
   [
    "C357",
    "t[S8,D18]"
   ],
   [
    "C358",
    "t[S8,D19]"
   ],
   [
    "C359",
    "t[S8,D20]"
   ],
   [
    "C360",
    "t[S9,D1]"
   ],
   [
    "C361",
    "t[S9,D2]"
   ],
   [
    "C362",
    "t[S9,D3]"
   ],
   [
    "C363",
    "t[S9,D4]"
   ],
   [
    "C364",
    "t[S9,D5]"
   ],
   [
    "C365",
    "t[S9,D6]"
   ],
   [
    "C366",
    "t[S9,D7]"
   ],
   [
    "C367",
    "t[S9,D8]"
   ],
   [
    "C368",
    "t[S9,D9]"
   ],
   [
    "C369",
    "t[S9,D10]"
   ],
   [
    "C370",
    "t[S9,D11]"
   ],
   [
    "C371",
    "t[S9,D12]"
   ],
   [
    "C372",
    "t[S9,D13]"
   ],
   [
    "C373",
    "t[S9,D14]"
   ],
   [
    "C374",
    "t[S9,D15]"
   ],
   [
    "C375",
    "t[S9,D16]"
   ],
   [
    "C376",
    "t[S9,D17]"
   ],
   [
    "C377",
    "t[S9,D18]"
   ],
   [
    "C378",
    "t[S9,D19]"
   ],
   [
    "C379",
    "t[S9,D20]"
   ],
   [
    "C380",
    "t[S10,D1]"
   ],
   [
    "C381",
    "t[S10,D2]"
   ],
   [
    "C382",
    "t[S10,D3]"
   ],
   [
    "C383",
    "t[S10,D4]"
   ],
   [
    "C384",
    "t[S10,D5]"
   ],
   [
    "C385",
    "t[S10,D6]"
   ],
   [
    "C386",
    "t[S10,D7]"
   ],
   [
    "C387",
    "t[S10,D8]"
   ],
   [
    "C388",
    "t[S10,D9]"
   ],
   [
    "C389",
    "t[S10,D10]"
   ],
   [
    "C390",
    "t[S10,D11]"
   ],
   [
    "C391",
    "t[S10,D12]"
   ],
   [
    "C392",
    "t[S10,D13]"
   ],
   [
    "C393",
    "t[S10,D14]"
   ],
   [
    "C394",
    "t[S10,D15]"
   ],
   [
    "C395",
    "t[S10,D16]"
   ],
   [
    "C396",
    "t[S10,D17]"
   ],
   [
    "C397",
    "t[S10,D18]"
   ],
   [
    "C398",
    "t[S10,D19]"
   ],
   [
    "C399",
    "t[S10,D20]"
   ]
  ],
  "primary_mapping": {
   "x[1,1]": "t[S1,D1]",
   "x[1,2]": "t[S1,D2]",
   "x[1,3]": "t[S1,D3]",
   "x[1,4]": "t[S1,D4]",
   "x[1,5]": "t[S1,D5]",
   "x[1,6]": "t[S1,D6]",
   "x[1,7]": "t[S1,D7]",
   "x[1,8]": "t[S1,D8]",
   "x[1,9]": "t[S1,D9]",
   "x[1,10]": "t[S1,D10]",
   "x[1,11]": "t[S1,D11]",
   "x[1,12]": "t[S1,D12]",
   "x[1,13]": "t[S1,D13]",
   "x[1,14]": "t[S1,D14]",
   "x[1,15]": "t[S1,D15]",
   "x[1,16]": "t[S1,D16]",
   "x[1,17]": "t[S1,D17]",
   "x[1,18]": "t[S1,D18]",
   "x[1,19]": "t[S1,D19]",
   "x[1,20]": "t[S1,D20]",
   "x[2,1]": "t[S2,D1]",
   "x[2,2]": "t[S2,D2]",
   "x[2,3]": "t[S2,D3]",
   "x[2,4]": "t[S2,D4]",
   "x[2,5]": "t[S2,D5]",
   "x[2,6]": "t[S2,D6]",
   "x[2,7]": "t[S2,D7]",
   "x[2,8]": "t[S2,D8]",
   "x[2,9]": "t[S2,D9]",
   "x[2,10]": "t[S2,D10]",
   "x[2,11]": "t[S2,D11]",
   "x[2,12]": "t[S2,D12]",
   "x[2,13]": "t[S2,D13]",
   "x[2,14]": "t[S2,D14]",
   "x[2,15]": "t[S2,D15]",
   "x[2,16]": "t[S2,D16]",
   "x[2,17]": "t[S2,D17]",
   "x[2,18]": "t[S2,D18]",
   "x[2,19]": "t[S2,D19]",
   "x[2,20]": "t[S2,D20]",
   "x[3,1]": "t[S3,D1]",
   "x[3,2]": "t[S3,D2]",
   "x[3,3]": "t[S3,D3]",
   "x[3,4]": "t[S3,D4]",
   "x[3,5]": "t[S3,D5]",
   "x[3,6]": "t[S3,D6]",
   "x[3,7]": "t[S3,D7]",
   "x[3,8]": "t[S3,D8]",
   "x[3,9]": "t[S3,D9]",
   "x[3,10]": "t[S3,D10]",
   "x[3,11]": "t[S3,D11]",
   "x[3,12]": "t[S3,D12]",
   "x[3,13]": "t[S3,D13]",
   "x[3,14]": "t[S3,D14]",
   "x[3,15]": "t[S3,D15]",
   "x[3,16]": "t[S3,D16]",
   "x[3,17]": "t[S3,D17]",
   "x[3,18]": "t[S3,D18]",
   "x[3,19]": "t[S3,D19]",
   "x[3,20]": "t[S3,D20]",
   "x[4,1]": "t[S4,D1]",
   "x[4,2]": "t[S4,D2]",
   "x[4,3]": "t[S4,D3]",
   "x[4,4]": "t[S4,D4]",
   "x[4,5]": "t[S4,D5]",
   "x[4,6]": "t[S4,D6]",
   "x[4,7]": "t[S4,D7]",
   "x[4,8]": "t[S4,D8]",
   "x[4,9]": "t[S4,D9]",
   "x[4,10]": "t[S4,D10]",
   "x[4,11]": "t[S4,D11]",
   "x[4,12]": "t[S4,D12]",
   "x[4,13]": "t[S4,D13]",
   "x[4,14]": "t[S4,D14]",
   "x[4,15]": "t[S4,D15]",
   "x[4,16]": "t[S4,D16]",
   "x[4,17]": "t[S4,D17]",
   "x[4,18]": "t[S4,D18]",
   "x[4,19]": "t[S4,D19]",
   "x[4,20]": "t[S4,D20]",
   "x[5,1]": "t[S5,D1]",
   "x[5,2]": "t[S5,D2]",
   "x[5,3]": "t[S5,D3]",
   "x[5,4]": "t[S5,D4]",
   "x[5,5]": "t[S5,D5]",
   "x[5,6]": "t[S5,D6]",
   "x[5,7]": "t[S5,D7]",
   "x[5,8]": "t[S5,D8]",
   "x[5,9]": "t[S5,D9]",
   "x[5,10]": "t[S5,D10]",
   "x[5,11]": "t[S5,D11]",
   "x[5,12]": "t[S5,D12]",
   "x[5,13]": "t[S5,D13]",
   "x[5,14]": "t[S5,D14]",
   "x[5,15]": "t[S5,D15]",
   "x[5,16]": "t[S5,D16]",
   "x[5,17]": "t[S5,D17]",
   "x[5,18]": "t[S5,D18]",
   "x[5,19]": "t[S5,D19]",
   "x[5,20]": "t[S5,D20]",
   "x[6,1]": "t[S6,D1]",
   "x[6,2]": "t[S6,D2]",
   "x[6,3]": "t[S6,D3]",
   "x[6,4]": "t[S6,D4]",
   "x[6,5]": "t[S6,D5]",
   "x[6,6]": "t[S6,D6]",
   "x[6,7]": "t[S6,D7]",
   "x[6,8]": "t[S6,D8]",
   "x[6,9]": "t[S6,D9]",
   "x[6,10]": "t[S6,D10]",
   "x[6,11]": "t[S6,D11]",
   "x[6,12]": "t[S6,D12]",
   "x[6,13]": "t[S6,D13]",
   "x[6,14]": "t[S6,D14]",
   "x[6,15]": "t[S6,D15]",
   "x[6,16]": "t[S6,D16]",
   "x[6,17]": "t[S6,D17]",
   "x[6,18]": "t[S6,D18]",
   "x[6,19]": "t[S6,D19]",
   "x[6,20]": "t[S6,D20]",
   "x[7,1]": "t[S7,D1]",
   "x[7,2]": "t[S7,D2]",
   "x[7,3]": "t[S7,D3]",
   "x[7,4]": "t[S7,D4]",
   "x[7,5]": "t[S7,D5]",
   "x[7,6]": "t[S7,D6]",
   "x[7,7]": "t[S7,D7]",
   "x[7,8]": "t[S7,D8]",
   "x[7,9]": "t[S7,D9]",
   "x[7,10]": "t[S7,D10]",
   "x[7,11]": "t[S7,D11]",
   "x[7,12]": "t[S7,D12]",
   "x[7,13]": "t[S7,D13]",
   "x[7,14]": "t[S7,D14]",
   "x[7,15]": "t[S7,D15]",
   "x[7,16]": "t[S7,D16]",
   "x[7,17]": "t[S7,D17]",
   "x[7,18]": "t[S7,D18]",
   "x[7,19]": "t[S7,D19]",
   "x[7,20]": "t[S7,D20]",
   "x[8,1]": "t[S8,D1]",
   "x[8,2]": "t[S8,D2]",
   "x[8,3]": "t[S8,D3]",
   "x[8,4]": "t[S8,D4]",
   "x[8,5]": "t[S8,D5]",
   "x[8,6]": "t[S8,D6]",
   "x[8,7]": "t[S8,D7]",
   "x[8,8]": "t[S8,D8]",
   "x[8,9]": "t[S8,D9]",
   "x[8,10]": "t[S8,D10]",
   "x[8,11]": "t[S8,D11]",
   "x[8,12]": "t[S8,D12]",
   "x[8,13]": "t[S8,D13]",
   "x[8,14]": "t[S8,D14]",
   "x[8,15]": "t[S8,D15]",
   "x[8,16]": "t[S8,D16]",
   "x[8,17]": "t[S8,D17]",
   "x[8,18]": "t[S8,D18]",
   "x[8,19]": "t[S8,D19]",
   "x[8,20]": "t[S8,D20]",
   "x[9,1]": "t[S9,D1]",
   "x[9,2]": "t[S9,D2]",
   "x[9,3]": "t[S9,D3]",
   "x[9,4]": "t[S9,D4]",
   "x[9,5]": "t[S9,D5]",
   "x[9,6]": "t[S9,D6]",
   "x[9,7]": "t[S9,D7]",
   "x[9,8]": "t[S9,D8]",
   "x[9,9]": "t[S9,D9]",
   "x[9,10]": "t[S9,D10]",
   "x[9,11]": "t[S9,D11]",
   "x[9,12]": "t[S9,D12]",
   "x[9,13]": "t[S9,D13]",
   "x[9,14]": "t[S9,D14]",
   "x[9,15]": "t[S9,D15]",
   "x[9,16]": "t[S9,D16]",
   "x[9,17]": "t[S9,D17]",
   "x[9,18]": "t[S9,D18]",
   "x[9,19]": "t[S9,D19]",
   "x[9,20]": "t[S9,D20]",
   "x[10,1]": "t[S10,D1]",
   "x[10,2]": "t[S10,D2]",
   "x[10,3]": "t[S10,D3]",
   "x[10,4]": "t[S10,D4]",
   "x[10,5]": "t[S10,D5]",
   "x[10,6]": "t[S10,D6]",
   "x[10,7]": "t[S10,D7]",
   "x[10,8]": "t[S10,D8]",
   "x[10,9]": "t[S10,D9]",
   "x[10,10]": "t[S10,D10]",
   "x[10,11]": "t[S10,D11]",
   "x[10,12]": "t[S10,D12]",
   "x[10,13]": "t[S10,D13]",
   "x[10,14]": "t[S10,D14]",
   "x[10,15]": "t[S10,D15]",
   "x[10,16]": "t[S10,D16]",
   "x[10,17]": "t[S10,D17]",
   "x[10,18]": "t[S10,D18]",
   "x[10,19]": "t[S10,D19]",
   "x[10,20]": "t[S10,D20]",
   "y[1,1]": "u[S1,D1]",
   "y[1,2]": "u[S1,D2]",
   "y[1,3]": "u[S1,D3]",
   "y[1,4]": "u[S1,D4]",
   "y[1,5]": "u[S1,D5]",
   "y[1,6]": "u[S1,D6]",
   "y[1,7]": "u[S1,D7]",
   "y[1,8]": "u[S1,D8]",
   "y[1,9]": "u[S1,D9]",
   "y[1,10]": "u[S1,D10]",
   "y[1,11]": "u[S1,D11]",
   "y[1,12]": "u[S1,D12]",
   "y[1,13]": "u[S1,D13]",
   "y[1,14]": "u[S1,D14]",
   "y[1,15]": "u[S1,D15]",
   "y[1,16]": "u[S1,D16]",
   "y[1,17]": "u[S1,D17]",
   "y[1,18]": "u[S1,D18]",
   "y[1,19]": "u[S1,D19]",
   "y[1,20]": "u[S1,D20]",
   "y[2,1]": "u[S2,D1]",
   "y[2,2]": "u[S2,D2]",
   "y[2,3]": "u[S2,D3]",
   "y[2,4]": "u[S2,D4]",
   "y[2,5]": "u[S2,D5]",
   "y[2,6]": "u[S2,D6]",
   "y[2,7]": "u[S2,D7]",
   "y[2,8]": "u[S2,D8]",
   "y[2,9]": "u[S2,D9]",
   "y[2,10]": "u[S2,D10]",
   "y[2,11]": "u[S2,D11]",
   "y[2,12]": "u[S2,D12]",
   "y[2,13]": "u[S2,D13]",
   "y[2,14]": "u[S2,D14]",
   "y[2,15]": "u[S2,D15]",
   "y[2,16]": "u[S2,D16]",
   "y[2,17]": "u[S2,D17]",
   "y[2,18]": "u[S2,D18]",
   "y[2,19]": "u[S2,D19]",
   "y[2,20]": "u[S2,D20]",
   "y[3,1]": "u[S3,D1]",
   "y[3,2]": "u[S3,D2]",
   "y[3,3]": "u[S3,D3]",
   "y[3,4]": "u[S3,D4]",
   "y[3,5]": "u[S3,D5]",
   "y[3,6]": "u[S3,D6]",
   "y[3,7]": "u[S3,D7]",
   "y[3,8]": "u[S3,D8]",
   "y[3,9]": "u[S3,D9]",
   "y[3,10]": "u[S3,D10]",
   "y[3,11]": "u[S3,D11]",
   "y[3,12]": "u[S3,D12]",
   "y[3,13]": "u[S3,D13]",
   "y[3,14]": "u[S3,D14]",
   "y[3,15]": "u[S3,D15]",
   "y[3,16]": "u[S3,D16]",
   "y[3,17]": "u[S3,D17]",
   "y[3,18]": "u[S3,D18]",
   "y[3,19]": "u[S3,D19]",
   "y[3,20]": "u[S3,D20]",
   "y[4,1]": "u[S4,D1]",
   "y[4,2]": "u[S4,D2]",
   "y[4,3]": "u[S4,D3]",
   "y[4,4]": "u[S4,D4]",
   "y[4,5]": "u[S4,D5]",
   "y[4,6]": "u[S4,D6]",
   "y[4,7]": "u[S4,D7]",
   "y[4,8]": "u[S4,D8]",
   "y[4,9]": "u[S4,D9]",
   "y[4,10]": "u[S4,D10]",
   "y[4,11]": "u[S4,D11]",
   "y[4,12]": "u[S4,D12]",
   "y[4,13]": "u[S4,D13]",
   "y[4,14]": "u[S4,D14]",
   "y[4,15]": "u[S4,D15]",
   "y[4,16]": "u[S4,D16]",
   "y[4,17]": "u[S4,D17]",
   "y[4,18]": "u[S4,D18]",
   "y[4,19]": "u[S4,D19]",
   "y[4,20]": "u[S4,D20]",
   "y[5,1]": "u[S5,D1]",
   "y[5,2]": "u[S5,D2]",
   "y[5,3]": "u[S5,D3]",
   "y[5,4]": "u[S5,D4]",
   "y[5,5]": "u[S5,D5]",
   "y[5,6]": "u[S5,D6]",
   "y[5,7]": "u[S5,D7]",
   "y[5,8]": "u[S5,D8]",
   "y[5,9]": "u[S5,D9]",
   "y[5,10]": "u[S5,D10]",
   "y[5,11]": "u[S5,D11]",
   "y[5,12]": "u[S5,D12]",
   "y[5,13]": "u[S5,D13]",
   "y[5,14]": "u[S5,D14]",
   "y[5,15]": "u[S5,D15]",
   "y[5,16]": "u[S5,D16]",
   "y[5,17]": "u[S5,D17]",
   "y[5,18]": "u[S5,D18]",
   "y[5,19]": "u[S5,D19]",
   "y[5,20]": "u[S5,D20]",
   "y[6,1]": "u[S6,D1]",
   "y[6,2]": "u[S6,D2]",
   "y[6,3]": "u[S6,D3]",
   "y[6,4]": "u[S6,D4]",
   "y[6,5]": "u[S6,D5]",
   "y[6,6]": "u[S6,D6]",
   "y[6,7]": "u[S6,D7]",
   "y[6,8]": "u[S6,D8]",
   "y[6,9]": "u[S6,D9]",
   "y[6,10]": "u[S6,D10]",
   "y[6,11]": "u[S6,D11]",
   "y[6,12]": "u[S6,D12]",
   "y[6,13]": "u[S6,D13]",
   "y[6,14]": "u[S6,D14]",
   "y[6,15]": "u[S6,D15]",
   "y[6,16]": "u[S6,D16]",
   "y[6,17]": "u[S6,D17]",
   "y[6,18]": "u[S6,D18]",
   "y[6,19]": "u[S6,D19]",
   "y[6,20]": "u[S6,D20]",
   "y[7,1]": "u[S7,D1]",
   "y[7,2]": "u[S7,D2]",
   "y[7,3]": "u[S7,D3]",
   "y[7,4]": "u[S7,D4]",
   "y[7,5]": "u[S7,D5]",
   "y[7,6]": "u[S7,D6]",
   "y[7,7]": "u[S7,D7]",
   "y[7,8]": "u[S7,D8]",
   "y[7,9]": "u[S7,D9]",
   "y[7,10]": "u[S7,D10]",
   "y[7,11]": "u[S7,D11]",
   "y[7,12]": "u[S7,D12]",
   "y[7,13]": "u[S7,D13]",
   "y[7,14]": "u[S7,D14]",
   "y[7,15]": "u[S7,D15]",
   "y[7,16]": "u[S7,D16]",
   "y[7,17]": "u[S7,D17]",
   "y[7,18]": "u[S7,D18]",
   "y[7,19]": "u[S7,D19]",
   "y[7,20]": "u[S7,D20]",
   "y[8,1]": "u[S8,D1]",
   "y[8,2]": "u[S8,D2]",
   "y[8,3]": "u[S8,D3]",
   "y[8,4]": "u[S8,D4]",
   "y[8,5]": "u[S8,D5]",
   "y[8,6]": "u[S8,D6]",
   "y[8,7]": "u[S8,D7]",
   "y[8,8]": "u[S8,D8]",
   "y[8,9]": "u[S8,D9]",
   "y[8,10]": "u[S8,D10]",
   "y[8,11]": "u[S8,D11]",
   "y[8,12]": "u[S8,D12]",
   "y[8,13]": "u[S8,D13]",
   "y[8,14]": "u[S8,D14]",
   "y[8,15]": "u[S8,D15]",
   "y[8,16]": "u[S8,D16]",
   "y[8,17]": "u[S8,D17]",
   "y[8,18]": "u[S8,D18]",
   "y[8,19]": "u[S8,D19]",
   "y[8,20]": "u[S8,D20]",
   "y[9,1]": "u[S9,D1]",
   "y[9,2]": "u[S9,D2]",
   "y[9,3]": "u[S9,D3]",
   "y[9,4]": "u[S9,D4]",
   "y[9,5]": "u[S9,D5]",
   "y[9,6]": "u[S9,D6]",
   "y[9,7]": "u[S9,D7]",
   "y[9,8]": "u[S9,D8]",
   "y[9,9]": "u[S9,D9]",
   "y[9,10]": "u[S9,D10]",
   "y[9,11]": "u[S9,D11]",
   "y[9,12]": "u[S9,D12]",
   "y[9,13]": "u[S9,D13]",
   "y[9,14]": "u[S9,D14]",
   "y[9,15]": "u[S9,D15]",
   "y[9,16]": "u[S9,D16]",
   "y[9,17]": "u[S9,D17]",
   "y[9,18]": "u[S9,D18]",
   "y[9,19]": "u[S9,D19]",
   "y[9,20]": "u[S9,D20]",
   "y[10,1]": "u[S10,D1]",
   "y[10,2]": "u[S10,D2]",
   "y[10,3]": "u[S10,D3]",
   "y[10,4]": "u[S10,D4]",
   "y[10,5]": "u[S10,D5]",
   "y[10,6]": "u[S10,D6]",
   "y[10,7]": "u[S10,D7]",
   "y[10,8]": "u[S10,D8]",
   "y[10,9]": "u[S10,D9]",
   "y[10,10]": "u[S10,D10]",
   "y[10,11]": "u[S10,D11]",
   "y[10,12]": "u[S10,D12]",
   "y[10,13]": "u[S10,D13]",
   "y[10,14]": "u[S10,D14]",
   "y[10,15]": "u[S10,D15]",
   "y[10,16]": "u[S10,D16]",
   "y[10,17]": "u[S10,D17]",
   "y[10,18]": "u[S10,D18]",
   "y[10,19]": "u[S10,D19]",
   "y[10,20]": "u[S10,D20]"
  },
  "mapping_basis": "frozen_name_axis_mapper",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 3148.9299999999994,
   "variables": [
    [
     "C0",
     0.0
    ],
    [
     "C1",
     0.0
    ],
    [
     "C2",
     0.0
    ],
    [
     "C3",
     3.0
    ],
    [
     "C4",
     53.0
    ],
    [
     "C5",
     47.0
    ],
    [
     "C6",
     0.0
    ],
    [
     "C7",
     0.0
    ],
    [
     "C8",
     0.0
    ],
    [
     "C9",
     0.0
    ],
    [
     "C10",
     0.0
    ],
    [
     "C11",
     0.0
    ],
    [
     "C12",
     0.0
    ],
    [
     "C13",
     0.0
    ],
    [
     "C14",
     0.0
    ],
    [
     "C15",
     0.0
    ],
    [
     "C16",
     0.0
    ],
    [
     "C17",
     0.0
    ],
    [
     "C18",
     0.0
    ],
    [
     "C19",
     0.0
    ],
    [
     "C20",
     0.0
    ],
    [
     "C21",
     46.0
    ],
    [
     "C22",
     15.0
    ],
    [
     "C23",
     0.0
    ],
    [
     "C24",
     0.0
    ],
    [
     "C25",
     0.0
    ],
    [
     "C26",
     0.0
    ],
    [
     "C27",
     0.0
    ],
    [
     "C28",
     0.0
    ],
    [
     "C29",
     0.0
    ],
    [
     "C30",
     0.0
    ],
    [
     "C31",
     26.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     8.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     0.0
    ],
    [
     "C48",
     56.0
    ],
    [
     "C49",
     0.0
    ],
    [
     "C50",
     0.0
    ],
    [
     "C51",
     0.0
    ],
    [
     "C52",
     0.0
    ],
    [
     "C53",
     31.0
    ],
    [
     "C54",
     0.0
    ],
    [
     "C55",
     0.0
    ],
    [
     "C56",
     0.0
    ],
    [
     "C57",
     0.0
    ],
    [
     "C58",
     0.0
    ],
    [
     "C59",
     0.0
    ],
    [
     "C60",
     0.0
    ],
    [
     "C61",
     0.0
    ],
    [
     "C62",
     41.0
    ],
    [
     "C63",
     15.0
    ],
    [
     "C64",
     0.0
    ],
    [
     "C65",
     0.0
    ],
    [
     "C66",
     56.0
    ],
    [
     "C67",
     0.0
    ],
    [
     "C68",
     0.0
    ],
    [
     "C69",
     0.0
    ],
    [
     "C70",
     0.0
    ],
    [
     "C71",
     0.0
    ],
    [
     "C72",
     0.0
    ],
    [
     "C73",
     0.0
    ],
    [
     "C74",
     0.0
    ],
    [
     "C75",
     0.0
    ],
    [
     "C76",
     0.0
    ],
    [
     "C77",
     0.0
    ],
    [
     "C78",
     0.0
    ],
    [
     "C79",
     0.0
    ],
    [
     "C80",
     61.0
    ],
    [
     "C81",
     0.0
    ],
    [
     "C82",
     0.0
    ],
    [
     "C83",
     36.0
    ],
    [
     "C84",
     0.0
    ],
    [
     "C85",
     0.0
    ],
    [
     "C86",
     0.0
    ],
    [
     "C87",
     0.0
    ],
    [
     "C88",
     0.0
    ],
    [
     "C89",
     0.0
    ],
    [
     "C90",
     0.0
    ],
    [
     "C91",
     0.0
    ],
    [
     "C92",
     0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     0.0
    ],
    [
     "C110",
     0.0
    ],
    [
     "C111",
     0.0
    ],
    [
     "C112",
     0.0
    ],
    [
     "C113",
     0.0
    ],
    [
     "C114",
     52.0
    ],
    [
     "C115",
     19.0
    ],
    [
     "C116",
     0.0
    ],
    [
     "C117",
     32.0
    ],
    [
     "C118",
     0.0
    ],
    [
     "C119",
     0.0
    ],
    [
     "C120",
     0.0
    ],
    [
     "C121",
     0.0
    ],
    [
     "C122",
     0.0
    ],
    [
     "C123",
     0.0
    ],
    [
     "C124",
     0.0
    ],
    [
     "C125",
     0.0
    ],
    [
     "C126",
     0.0
    ],
    [
     "C127",
     57.0
    ],
    [
     "C128",
     0.0
    ],
    [
     "C129",
     34.0
    ],
    [
     "C130",
     0.0
    ],
    [
     "C131",
     0.0
    ],
    [
     "C132",
     0.0
    ],
    [
     "C133",
     0.0
    ],
    [
     "C134",
     10.0
    ],
    [
     "C135",
     0.0
    ],
    [
     "C136",
     0.0
    ],
    [
     "C137",
     0.0
    ],
    [
     "C138",
     0.0
    ],
    [
     "C139",
     0.0
    ],
    [
     "C140",
     0.0
    ],
    [
     "C141",
     0.0
    ],
    [
     "C142",
     0.0
    ],
    [
     "C143",
     0.0
    ],
    [
     "C144",
     0.0
    ],
    [
     "C145",
     0.0
    ],
    [
     "C146",
     0.0
    ],
    [
     "C147",
     0.0
    ],
    [
     "C148",
     0.0
    ],
    [
     "C149",
     0.0
    ],
    [
     "C150",
     0.0
    ],
    [
     "C151",
     0.0
    ],
    [
     "C152",
     13.0
    ],
    [
     "C153",
     0.0
    ],
    [
     "C154",
     0.0
    ],
    [
     "C155",
     39.0
    ],
    [
     "C156",
     39.0
    ],
    [
     "C157",
     0.0
    ],
    [
     "C158",
     3.0
    ],
    [
     "C159",
     0.0
    ],
    [
     "C160",
     0.0
    ],
    [
     "C161",
     0.0
    ],
    [
     "C162",
     0.0
    ],
    [
     "C163",
     0.0
    ],
    [
     "C164",
     0.0
    ],
    [
     "C165",
     0.0
    ],
    [
     "C166",
     0.0
    ],
    [
     "C167",
     0.0
    ],
    [
     "C168",
     0.0
    ],
    [
     "C169",
     0.0
    ],
    [
     "C170",
     0.0
    ],
    [
     "C171",
     0.0
    ],
    [
     "C172",
     0.0
    ],
    [
     "C173",
     0.0
    ],
    [
     "C174",
     0.0
    ],
    [
     "C175",
     0.0
    ],
    [
     "C176",
     0.0
    ],
    [
     "C177",
     0.0
    ],
    [
     "C178",
     35.0
    ],
    [
     "C179",
     67.0
    ],
    [
     "C180",
     0.0
    ],
    [
     "C181",
     0.0
    ],
    [
     "C182",
     0.0
    ],
    [
     "C183",
     0.0
    ],
    [
     "C184",
     0.0
    ],
    [
     "C185",
     0.0
    ],
    [
     "C186",
     0.0
    ],
    [
     "C187",
     0.0
    ],
    [
     "C188",
     0.0
    ],
    [
     "C189",
     0.0
    ],
    [
     "C190",
     55.0
    ],
    [
     "C191",
     27.0
    ],
    [
     "C192",
     24.0
    ],
    [
     "C193",
     0.0
    ],
    [
     "C194",
     0.0
    ],
    [
     "C195",
     0.0
    ],
    [
     "C196",
     0.0
    ],
    [
     "C197",
     0.0
    ],
    [
     "C198",
     0.0
    ],
    [
     "C199",
     0.0
    ],
    [
     "C200",
     0.0
    ],
    [
     "C201",
     0.0
    ],
    [
     "C202",
     0.0
    ],
    [
     "C203",
     1.0
    ],
    [
     "C204",
     6.0
    ],
    [
     "C205",
     5.0
    ],
    [
     "C206",
     0.0
    ],
    [
     "C207",
     0.0
    ],
    [
     "C208",
     0.0
    ],
    [
     "C209",
     0.0
    ],
    [
     "C210",
     0.0
    ],
    [
     "C211",
     0.0
    ],
    [
     "C212",
     0.0
    ],
    [
     "C213",
     0.0
    ],
    [
     "C214",
     0.0
    ],
    [
     "C215",
     0.0
    ],
    [
     "C216",
     0.0
    ],
    [
     "C217",
     0.0
    ],
    [
     "C218",
     0.0
    ],
    [
     "C219",
     0.0
    ],
    [
     "C220",
     0.0
    ],
    [
     "C221",
     5.0
    ],
    [
     "C222",
     2.0
    ],
    [
     "C223",
     0.0
    ],
    [
     "C224",
     0.0
    ],
    [
     "C225",
     0.0
    ],
    [
     "C226",
     0.0
    ],
    [
     "C227",
     0.0
    ],
    [
     "C228",
     0.0
    ],
    [
     "C229",
     0.0
    ],
    [
     "C230",
     0.0
    ],
    [
     "C231",
     3.0
    ],
    [
     "C232",
     0.0
    ],
    [
     "C233",
     0.0
    ],
    [
     "C234",
     0.0
    ],
    [
     "C235",
     0.0
    ],
    [
     "C236",
     0.0
    ],
    [
     "C237",
     0.0
    ],
    [
     "C238",
     0.0
    ],
    [
     "C239",
     0.0
    ],
    [
     "C240",
     0.0
    ],
    [
     "C241",
     1.0
    ],
    [
     "C242",
     0.0
    ],
    [
     "C243",
     0.0
    ],
    [
     "C244",
     0.0
    ],
    [
     "C245",
     0.0
    ],
    [
     "C246",
     0.0
    ],
    [
     "C247",
     0.0
    ],
    [
     "C248",
     6.0
    ],
    [
     "C249",
     0.0
    ],
    [
     "C250",
     0.0
    ],
    [
     "C251",
     0.0
    ],
    [
     "C252",
     0.0
    ],
    [
     "C253",
     4.0
    ],
    [
     "C254",
     0.0
    ],
    [
     "C255",
     0.0
    ],
    [
     "C256",
     0.0
    ],
    [
     "C257",
     0.0
    ],
    [
     "C258",
     0.0
    ],
    [
     "C259",
     0.0
    ],
    [
     "C260",
     0.0
    ],
    [
     "C261",
     0.0
    ],
    [
     "C262",
     5.0
    ],
    [
     "C263",
     2.0
    ],
    [
     "C264",
     0.0
    ],
    [
     "C265",
     0.0
    ],
    [
     "C266",
     6.0
    ],
    [
     "C267",
     0.0
    ],
    [
     "C268",
     0.0
    ],
    [
     "C269",
     0.0
    ],
    [
     "C270",
     0.0
    ],
    [
     "C271",
     0.0
    ],
    [
     "C272",
     0.0
    ],
    [
     "C273",
     0.0
    ],
    [
     "C274",
     0.0
    ],
    [
     "C275",
     0.0
    ],
    [
     "C276",
     0.0
    ],
    [
     "C277",
     0.0
    ],
    [
     "C278",
     0.0
    ],
    [
     "C279",
     0.0
    ],
    [
     "C280",
     7.0
    ],
    [
     "C281",
     0.0
    ],
    [
     "C282",
     0.0
    ],
    [
     "C283",
     4.0
    ],
    [
     "C284",
     0.0
    ],
    [
     "C285",
     0.0
    ],
    [
     "C286",
     0.0
    ],
    [
     "C287",
     0.0
    ],
    [
     "C288",
     0.0
    ],
    [
     "C289",
     0.0
    ],
    [
     "C290",
     0.0
    ],
    [
     "C291",
     0.0
    ],
    [
     "C292",
     0.0
    ],
    [
     "C293",
     0.0
    ],
    [
     "C294",
     0.0
    ],
    [
     "C295",
     0.0
    ],
    [
     "C296",
     0.0
    ],
    [
     "C297",
     0.0
    ],
    [
     "C298",
     0.0
    ],
    [
     "C299",
     0.0
    ],
    [
     "C300",
     0.0
    ],
    [
     "C301",
     0.0
    ],
    [
     "C302",
     0.0
    ],
    [
     "C303",
     0.0
    ],
    [
     "C304",
     0.0
    ],
    [
     "C305",
     0.0
    ],
    [
     "C306",
     0.0
    ],
    [
     "C307",
     0.0
    ],
    [
     "C308",
     0.0
    ],
    [
     "C309",
     0.0
    ],
    [
     "C310",
     0.0
    ],
    [
     "C311",
     0.0
    ],
    [
     "C312",
     0.0
    ],
    [
     "C313",
     0.0
    ],
    [
     "C314",
     6.0
    ],
    [
     "C315",
     2.0
    ],
    [
     "C316",
     0.0
    ],
    [
     "C317",
     4.0
    ],
    [
     "C318",
     0.0
    ],
    [
     "C319",
     0.0
    ],
    [
     "C320",
     0.0
    ],
    [
     "C321",
     0.0
    ],
    [
     "C322",
     0.0
    ],
    [
     "C323",
     0.0
    ],
    [
     "C324",
     0.0
    ],
    [
     "C325",
     0.0
    ],
    [
     "C326",
     0.0
    ],
    [
     "C327",
     6.0
    ],
    [
     "C328",
     0.0
    ],
    [
     "C329",
     4.0
    ],
    [
     "C330",
     0.0
    ],
    [
     "C331",
     0.0
    ],
    [
     "C332",
     0.0
    ],
    [
     "C333",
     0.0
    ],
    [
     "C334",
     1.0
    ],
    [
     "C335",
     0.0
    ],
    [
     "C336",
     0.0
    ],
    [
     "C337",
     0.0
    ],
    [
     "C338",
     0.0
    ],
    [
     "C339",
     0.0
    ],
    [
     "C340",
     0.0
    ],
    [
     "C341",
     0.0
    ],
    [
     "C342",
     0.0
    ],
    [
     "C343",
     0.0
    ],
    [
     "C344",
     0.0
    ],
    [
     "C345",
     0.0
    ],
    [
     "C346",
     0.0
    ],
    [
     "C347",
     0.0
    ],
    [
     "C348",
     0.0
    ],
    [
     "C349",
     0.0
    ],
    [
     "C350",
     0.0
    ],
    [
     "C351",
     0.0
    ],
    [
     "C352",
     2.0
    ],
    [
     "C353",
     0.0
    ],
    [
     "C354",
     0.0
    ],
    [
     "C355",
     4.0
    ],
    [
     "C356",
     4.0
    ],
    [
     "C357",
     0.0
    ],
    [
     "C358",
     1.0
    ],
    [
     "C359",
     0.0
    ],
    [
     "C360",
     0.0
    ],
    [
     "C361",
     0.0
    ],
    [
     "C362",
     0.0
    ],
    [
     "C363",
     0.0
    ],
    [
     "C364",
     0.0
    ],
    [
     "C365",
     0.0
    ],
    [
     "C366",
     0.0
    ],
    [
     "C367",
     0.0
    ],
    [
     "C368",
     0.0
    ],
    [
     "C369",
     0.0
    ],
    [
     "C370",
     0.0
    ],
    [
     "C371",
     0.0
    ],
    [
     "C372",
     0.0
    ],
    [
     "C373",
     0.0
    ],
    [
     "C374",
     0.0
    ],
    [
     "C375",
     0.0
    ],
    [
     "C376",
     0.0
    ],
    [
     "C377",
     0.0
    ],
    [
     "C378",
     4.0
    ],
    [
     "C379",
     7.0
    ],
    [
     "C380",
     0.0
    ],
    [
     "C381",
     0.0
    ],
    [
     "C382",
     0.0
    ],
    [
     "C383",
     0.0
    ],
    [
     "C384",
     0.0
    ],
    [
     "C385",
     0.0
    ],
    [
     "C386",
     0.0
    ],
    [
     "C387",
     0.0
    ],
    [
     "C388",
     0.0
    ],
    [
     "C389",
     0.0
    ],
    [
     "C390",
     6.0
    ],
    [
     "C391",
     3.0
    ],
    [
     "C392",
     3.0
    ],
    [
     "C393",
     0.0
    ],
    [
     "C394",
     0.0
    ],
    [
     "C395",
     0.0
    ],
    [
     "C396",
     0.0
    ],
    [
     "C397",
     0.0
    ],
    [
     "C398",
     0.0
    ],
    [
     "C399",
     0.0
    ]
   ],
   "error": null
  }
 },
 "OR-096": {
  "source_code_sha256": "9daa832d2312538957ba2594723ba673fea195b81350486a5b285bf947398e27",
  "original_solution_sha256": "c00fdf7ed46ce7eea2059ac1078039c56132a302529a40b9e49dc340b13f9346",
  "bindings": [
   [
    "C0",
    "x_W[I,N01]"
   ],
   [
    "C1",
    "x_W[I,N02]"
   ],
   [
    "C2",
    "x_W[I,N03]"
   ],
   [
    "C3",
    "x_W[I,N04]"
   ],
   [
    "C4",
    "x_W[I,N05]"
   ],
   [
    "C5",
    "x_W[I,N06]"
   ],
   [
    "C6",
    "x_W[I,N07]"
   ],
   [
    "C7",
    "x_W[I,N08]"
   ],
   [
    "C8",
    "x_W[I,N09]"
   ],
   [
    "C9",
    "x_W[I,N10]"
   ],
   [
    "C10",
    "x_W[I,N11]"
   ],
   [
    "C11",
    "x_W[I,N12]"
   ],
   [
    "C12",
    "x_W[I,N13]"
   ],
   [
    "C13",
    "x_W[I,N14]"
   ],
   [
    "C14",
    "x_W[I,N15]"
   ],
   [
    "C15",
    "x_W[I,N16]"
   ],
   [
    "C16",
    "x_W[I,N17]"
   ],
   [
    "C17",
    "x_W[I,N18]"
   ],
   [
    "C18",
    "x_W[I,N19]"
   ],
   [
    "C19",
    "x_W[I,N20]"
   ],
   [
    "C20",
    "x_W[I,N21]"
   ],
   [
    "C21",
    "x_W[I,N22]"
   ],
   [
    "C22",
    "x_W[I,N23]"
   ],
   [
    "C23",
    "x_W[I,N24]"
   ],
   [
    "C24",
    "x_W[I,N25]"
   ],
   [
    "C25",
    "x_W[I,N26]"
   ],
   [
    "C26",
    "x_W[I,N27]"
   ],
   [
    "C27",
    "x_W[I,N28]"
   ],
   [
    "C28",
    "x_W[I,N29]"
   ],
   [
    "C29",
    "x_W[I,N30]"
   ],
   [
    "C30",
    "x_W[I,N31]"
   ],
   [
    "C31",
    "x_W[II,N01]"
   ],
   [
    "C32",
    "x_W[II,N02]"
   ],
   [
    "C33",
    "x_W[II,N03]"
   ],
   [
    "C34",
    "x_W[II,N04]"
   ],
   [
    "C35",
    "x_W[II,N05]"
   ],
   [
    "C36",
    "x_W[II,N06]"
   ],
   [
    "C37",
    "x_W[II,N07]"
   ],
   [
    "C38",
    "x_W[II,N08]"
   ],
   [
    "C39",
    "x_W[II,N09]"
   ],
   [
    "C40",
    "x_W[II,N10]"
   ],
   [
    "C41",
    "x_W[II,N11]"
   ],
   [
    "C42",
    "x_W[II,N12]"
   ],
   [
    "C43",
    "x_W[II,N13]"
   ],
   [
    "C44",
    "x_W[II,N14]"
   ],
   [
    "C45",
    "x_W[II,N15]"
   ],
   [
    "C46",
    "x_W[II,N16]"
   ],
   [
    "C47",
    "x_W[II,N17]"
   ],
   [
    "C48",
    "x_W[II,N18]"
   ],
   [
    "C49",
    "x_W[II,N19]"
   ],
   [
    "C50",
    "x_W[II,N20]"
   ],
   [
    "C51",
    "x_W[II,N21]"
   ],
   [
    "C52",
    "x_W[II,N22]"
   ],
   [
    "C53",
    "x_W[II,N23]"
   ],
   [
    "C54",
    "x_W[II,N24]"
   ],
   [
    "C55",
    "x_W[II,N25]"
   ],
   [
    "C56",
    "x_W[II,N26]"
   ],
   [
    "C57",
    "x_W[II,N27]"
   ],
   [
    "C58",
    "x_W[II,N28]"
   ],
   [
    "C59",
    "x_W[II,N29]"
   ],
   [
    "C60",
    "x_W[II,N30]"
   ],
   [
    "C61",
    "x_W[II,N31]"
   ],
   [
    "C62",
    "x_NW[I,N01]"
   ],
   [
    "C63",
    "x_NW[I,N02]"
   ],
   [
    "C64",
    "x_NW[I,N03]"
   ],
   [
    "C65",
    "x_NW[I,N04]"
   ],
   [
    "C66",
    "x_NW[I,N05]"
   ],
   [
    "C67",
    "x_NW[I,N06]"
   ],
   [
    "C68",
    "x_NW[I,N07]"
   ],
   [
    "C69",
    "x_NW[I,N08]"
   ],
   [
    "C70",
    "x_NW[I,N09]"
   ],
   [
    "C71",
    "x_NW[I,N10]"
   ],
   [
    "C72",
    "x_NW[I,N11]"
   ],
   [
    "C73",
    "x_NW[I,N12]"
   ],
   [
    "C74",
    "x_NW[I,N13]"
   ],
   [
    "C75",
    "x_NW[I,N14]"
   ],
   [
    "C76",
    "x_NW[I,N15]"
   ],
   [
    "C77",
    "x_NW[I,N16]"
   ],
   [
    "C78",
    "x_NW[I,N17]"
   ],
   [
    "C79",
    "x_NW[I,N18]"
   ],
   [
    "C80",
    "x_NW[I,N19]"
   ],
   [
    "C81",
    "x_NW[I,N20]"
   ],
   [
    "C82",
    "x_NW[I,N21]"
   ],
   [
    "C83",
    "x_NW[I,N22]"
   ],
   [
    "C84",
    "x_NW[I,N23]"
   ],
   [
    "C85",
    "x_NW[I,N24]"
   ],
   [
    "C86",
    "x_NW[I,N25]"
   ],
   [
    "C87",
    "x_NW[I,N26]"
   ],
   [
    "C88",
    "x_NW[I,N27]"
   ],
   [
    "C89",
    "x_NW[I,N28]"
   ],
   [
    "C90",
    "x_NW[I,N29]"
   ],
   [
    "C91",
    "x_NW[I,N30]"
   ],
   [
    "C92",
    "x_NW[I,N31]"
   ],
   [
    "C93",
    "x_NW[II,N01]"
   ],
   [
    "C94",
    "x_NW[II,N02]"
   ],
   [
    "C95",
    "x_NW[II,N03]"
   ],
   [
    "C96",
    "x_NW[II,N04]"
   ],
   [
    "C97",
    "x_NW[II,N05]"
   ],
   [
    "C98",
    "x_NW[II,N06]"
   ],
   [
    "C99",
    "x_NW[II,N07]"
   ],
   [
    "C100",
    "x_NW[II,N08]"
   ],
   [
    "C101",
    "x_NW[II,N09]"
   ],
   [
    "C102",
    "x_NW[II,N10]"
   ],
   [
    "C103",
    "x_NW[II,N11]"
   ],
   [
    "C104",
    "x_NW[II,N12]"
   ],
   [
    "C105",
    "x_NW[II,N13]"
   ],
   [
    "C106",
    "x_NW[II,N14]"
   ],
   [
    "C107",
    "x_NW[II,N15]"
   ],
   [
    "C108",
    "x_NW[II,N16]"
   ],
   [
    "C109",
    "x_NW[II,N17]"
   ],
   [
    "C110",
    "x_NW[II,N18]"
   ],
   [
    "C111",
    "x_NW[II,N19]"
   ],
   [
    "C112",
    "x_NW[II,N20]"
   ],
   [
    "C113",
    "x_NW[II,N21]"
   ],
   [
    "C114",
    "x_NW[II,N22]"
   ],
   [
    "C115",
    "x_NW[II,N23]"
   ],
   [
    "C116",
    "x_NW[II,N24]"
   ],
   [
    "C117",
    "x_NW[II,N25]"
   ],
   [
    "C118",
    "x_NW[II,N26]"
   ],
   [
    "C119",
    "x_NW[II,N27]"
   ],
   [
    "C120",
    "x_NW[II,N28]"
   ],
   [
    "C121",
    "x_NW[II,N29]"
   ],
   [
    "C122",
    "x_NW[II,N30]"
   ],
   [
    "C123",
    "x_NW[II,N31]"
   ]
  ],
  "primary_mapping": {
   "x_n_I_N28": "x_NW[I,N28]",
   "x_n_I_N10": "x_NW[I,N10]",
   "x_w_II_N13": "x_W[II,N13]",
   "x_n_I_N30": "x_NW[I,N30]",
   "x_w_I_N13": "x_W[I,N13]",
   "x_n_I_N08": "x_NW[I,N08]",
   "x_n_II_N18": "x_NW[II,N18]",
   "x_n_I_N13": "x_NW[I,N13]",
   "x_w_II_N07": "x_W[II,N07]",
   "x_w_II_N01": "x_W[II,N01]",
   "x_n_I_N12": "x_NW[I,N12]",
   "x_n_I_N20": "x_NW[I,N20]",
   "x_n_II_N19": "x_NW[II,N19]",
   "x_n_I_N22": "x_NW[I,N22]",
   "x_n_II_N22": "x_NW[II,N22]",
   "x_n_I_N06": "x_NW[I,N06]",
   "x_n_II_N07": "x_NW[II,N07]",
   "x_n_I_N09": "x_NW[I,N09]",
   "x_w_II_N09": "x_W[II,N09]",
   "x_n_II_N11": "x_NW[II,N11]",
   "x_n_I_N07": "x_NW[I,N07]",
   "x_w_I_N05": "x_W[I,N05]",
   "x_w_I_N23": "x_W[I,N23]",
   "x_w_II_N16": "x_W[II,N16]",
   "x_w_II_N30": "x_W[II,N30]",
   "x_n_II_N06": "x_NW[II,N06]",
   "x_n_II_N08": "x_NW[II,N08]",
   "x_n_II_N27": "x_NW[II,N27]",
   "x_n_II_N01": "x_NW[II,N01]",
   "x_w_I_N28": "x_W[I,N28]",
   "x_w_II_N22": "x_W[II,N22]",
   "x_w_I_N09": "x_W[I,N09]",
   "x_w_II_N12": "x_W[II,N12]",
   "x_n_II_N15": "x_NW[II,N15]",
   "x_n_I_N25": "x_NW[I,N25]",
   "x_w_I_N08": "x_W[I,N08]",
   "x_n_I_N17": "x_NW[I,N17]",
   "x_n_I_N29": "x_NW[I,N29]",
   "x_n_II_N16": "x_NW[II,N16]",
   "x_w_II_N17": "x_W[II,N17]",
   "x_n_II_N12": "x_NW[II,N12]",
   "x_n_II_N29": "x_NW[II,N29]",
   "x_w_I_N15": "x_W[I,N15]",
   "x_w_I_N26": "x_W[I,N26]",
   "x_w_I_N14": "x_W[I,N14]",
   "x_w_II_N08": "x_W[II,N08]",
   "x_n_I_N05": "x_NW[I,N05]",
   "x_n_I_N18": "x_NW[I,N18]",
   "x_n_II_N04": "x_NW[II,N04]",
   "x_w_II_N31": "x_W[II,N31]",
   "x_n_I_N21": "x_NW[I,N21]",
   "x_w_I_N19": "x_W[I,N19]",
   "x_w_I_N11": "x_W[I,N11]",
   "x_n_II_N02": "x_NW[II,N02]",
   "x_n_I_N27": "x_NW[I,N27]",
   "x_n_I_N01": "x_NW[I,N01]",
   "x_w_II_N24": "x_W[II,N24]",
   "x_n_II_N24": "x_NW[II,N24]",
   "x_w_I_N27": "x_W[I,N27]",
   "x_w_I_N04": "x_W[I,N04]",
   "x_w_I_N10": "x_W[I,N10]",
   "x_w_II_N21": "x_W[II,N21]",
   "x_n_I_N31": "x_NW[I,N31]",
   "x_n_II_N20": "x_NW[II,N20]",
   "x_w_I_N01": "x_W[I,N01]",
   "x_n_II_N03": "x_NW[II,N03]",
   "x_w_I_N25": "x_W[I,N25]",
   "x_w_II_N03": "x_W[II,N03]",
   "x_n_II_N30": "x_NW[II,N30]",
   "x_w_II_N05": "x_W[II,N05]",
   "x_n_I_N15": "x_NW[I,N15]",
   "x_w_II_N06": "x_W[II,N06]",
   "x_w_II_N27": "x_W[II,N27]",
   "x_n_I_N26": "x_NW[I,N26]",
   "x_w_I_N07": "x_W[I,N07]",
   "x_w_I_N29": "x_W[I,N29]",
   "x_w_I_N21": "x_W[I,N21]",
   "x_n_II_N21": "x_NW[II,N21]",
   "x_w_II_N29": "x_W[II,N29]",
   "x_w_II_N11": "x_W[II,N11]",
   "x_w_I_N03": "x_W[I,N03]",
   "x_n_II_N28": "x_NW[II,N28]",
   "x_n_I_N14": "x_NW[I,N14]",
   "x_w_I_N22": "x_W[I,N22]",
   "x_n_II_N17": "x_NW[II,N17]",
   "x_n_II_N25": "x_NW[II,N25]",
   "x_n_II_N09": "x_NW[II,N09]",
   "x_w_II_N14": "x_W[II,N14]",
   "x_w_II_N28": "x_W[II,N28]",
   "x_w_I_N02": "x_W[I,N02]",
   "x_w_II_N20": "x_W[II,N20]",
   "x_w_II_N25": "x_W[II,N25]",
   "x_n_II_N05": "x_NW[II,N05]",
   "x_w_I_N16": "x_W[I,N16]",
   "x_w_I_N18": "x_W[I,N18]",
   "x_w_II_N26": "x_W[II,N26]",
   "x_n_I_N16": "x_NW[I,N16]",
   "x_n_I_N19": "x_NW[I,N19]",
   "x_w_II_N04": "x_W[II,N04]",
   "x_w_I_N24": "x_W[I,N24]",
   "x_w_I_N20": "x_W[I,N20]",
   "x_w_I_N12": "x_W[I,N12]",
   "x_w_II_N15": "x_W[II,N15]",
   "x_n_II_N23": "x_NW[II,N23]",
   "x_n_II_N26": "x_NW[II,N26]",
   "x_n_I_N02": "x_NW[I,N02]",
   "x_n_I_N11": "x_NW[I,N11]",
   "x_w_II_N23": "x_W[II,N23]",
   "x_n_II_N13": "x_NW[II,N13]",
   "x_w_I_N30": "x_W[I,N30]",
   "x_w_I_N06": "x_W[I,N06]",
   "x_n_I_N24": "x_NW[I,N24]",
   "x_n_II_N14": "x_NW[II,N14]",
   "x_w_I_N17": "x_W[I,N17]",
   "x_n_II_N31": "x_NW[II,N31]",
   "x_n_II_N10": "x_NW[II,N10]",
   "x_w_I_N31": "x_W[I,N31]",
   "x_n_I_N23": "x_NW[I,N23]",
   "x_n_I_N03": "x_NW[I,N03]",
   "x_n_I_N04": "x_NW[I,N04]",
   "x_w_II_N10": "x_W[II,N10]",
   "x_w_II_N19": "x_W[II,N19]",
   "x_w_II_N02": "x_W[II,N02]",
   "x_w_II_N18": "x_W[II,N18]"
  },
  "mapping_basis": "explicit_physical_axes_review",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 5044.0,
   "variables": [
    [
     "C0",
     78.0
    ],
    [
     "C1",
     57.0
    ],
    [
     "C2",
     47.0
    ],
    [
     "C3",
     78.0
    ],
    [
     "C4",
     57.0
    ],
    [
     "C5",
     47.0
    ],
    [
     "C6",
     78.0
    ],
    [
     "C7",
     57.0
    ],
    [
     "C8",
     46.0
    ],
    [
     "C9",
     77.0
    ],
    [
     "C10",
     56.0
    ],
    [
     "C11",
     46.0
    ],
    [
     "C12",
     77.0
    ],
    [
     "C13",
     56.0
    ],
    [
     "C14",
     46.0
    ],
    [
     "C15",
     77.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     0.0
    ],
    [
     "C32",
     0.0
    ],
    [
     "C33",
     0.0
    ],
    [
     "C34",
     0.0
    ],
    [
     "C35",
     0.0
    ],
    [
     "C36",
     0.0
    ],
    [
     "C37",
     0.0
    ],
    [
     "C38",
     0.0
    ],
    [
     "C39",
     0.0
    ],
    [
     "C40",
     0.0
    ],
    [
     "C41",
     0.0
    ],
    [
     "C42",
     0.0
    ],
    [
     "C43",
     0.0
    ],
    [
     "C44",
     0.0
    ],
    [
     "C45",
     0.0
    ],
    [
     "C46",
     0.0
    ],
    [
     "C47",
     56.0
    ],
    [
     "C48",
     46.0
    ],
    [
     "C49",
     77.0
    ],
    [
     "C50",
     56.0
    ],
    [
     "C51",
     46.0
    ],
    [
     "C52",
     77.0
    ],
    [
     "C53",
     56.0
    ],
    [
     "C54",
     46.0
    ],
    [
     "C55",
     77.0
    ],
    [
     "C56",
     56.0
    ],
    [
     "C57",
     46.0
    ],
    [
     "C58",
     77.0
    ],
    [
     "C59",
     56.0
    ],
    [
     "C60",
     46.0
    ],
    [
     "C61",
     74.0
    ],
    [
     "C62",
     22.0
    ],
    [
     "C63",
     33.0
    ],
    [
     "C64",
     63.0
    ],
    [
     "C65",
     22.0
    ],
    [
     "C66",
     33.0
    ],
    [
     "C67",
     63.0
    ],
    [
     "C68",
     22.0
    ],
    [
     "C69",
     33.0
    ],
    [
     "C70",
     64.0
    ],
    [
     "C71",
     23.0
    ],
    [
     "C72",
     34.0
    ],
    [
     "C73",
     64.0
    ],
    [
     "C74",
     23.0
    ],
    [
     "C75",
     34.0
    ],
    [
     "C76",
     64.0
    ],
    [
     "C77",
     23.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     -0.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     0.0
    ],
    [
     "C94",
     0.0
    ],
    [
     "C95",
     0.0
    ],
    [
     "C96",
     0.0
    ],
    [
     "C97",
     0.0
    ],
    [
     "C98",
     0.0
    ],
    [
     "C99",
     0.0
    ],
    [
     "C100",
     0.0
    ],
    [
     "C101",
     0.0
    ],
    [
     "C102",
     0.0
    ],
    [
     "C103",
     0.0
    ],
    [
     "C104",
     0.0
    ],
    [
     "C105",
     0.0
    ],
    [
     "C106",
     0.0
    ],
    [
     "C107",
     0.0
    ],
    [
     "C108",
     0.0
    ],
    [
     "C109",
     34.0
    ],
    [
     "C110",
     64.0
    ],
    [
     "C111",
     23.0
    ],
    [
     "C112",
     34.0
    ],
    [
     "C113",
     64.0
    ],
    [
     "C114",
     23.0
    ],
    [
     "C115",
     34.0
    ],
    [
     "C116",
     64.0
    ],
    [
     "C117",
     23.0
    ],
    [
     "C118",
     34.0
    ],
    [
     "C119",
     64.0
    ],
    [
     "C120",
     23.0
    ],
    [
     "C121",
     34.0
    ],
    [
     "C122",
     64.0
    ],
    [
     "C123",
     46.0
    ]
   ],
   "error": null
  }
 },
 "OR-100": {
  "source_code_sha256": "14f00e913c52614dce84dd123609c5dfa4aa5db47af3db00a0e1f3b93751ea81",
  "original_solution_sha256": "81163616ff6295622495c8c68e6ac989f44cc2b0f7ede221b3faaf6c2991efe8",
  "bindings": [
   [
    "C0",
    "x[C58]"
   ],
   [
    "C1",
    "x[C60]"
   ],
   [
    "C2",
    "x[C100]"
   ],
   [
    "C3",
    "x[C101]"
   ],
   [
    "C4",
    "x[C15]"
   ],
   [
    "C5",
    "x[C42]"
   ],
   [
    "C6",
    "x[C30]"
   ],
   [
    "C7",
    "x[C43]"
   ],
   [
    "C8",
    "x[C52]"
   ],
   [
    "C9",
    "x[C81]"
   ],
   [
    "C10",
    "x[C18]"
   ],
   [
    "C11",
    "x[C22]"
   ],
   [
    "C12",
    "x[C56]"
   ],
   [
    "C13",
    "x[C88]"
   ],
   [
    "C14",
    "x[C4]"
   ],
   [
    "C15",
    "x[C44]"
   ],
   [
    "C16",
    "x[C37]"
   ],
   [
    "C17",
    "x[C9]"
   ],
   [
    "C18",
    "x[C61]"
   ],
   [
    "C19",
    "x[C16]"
   ],
   [
    "C20",
    "x[C105]"
   ],
   [
    "C21",
    "x[C104]"
   ],
   [
    "C22",
    "x[C74]"
   ],
   [
    "C23",
    "x[C5]"
   ],
   [
    "C24",
    "x[C53]"
   ],
   [
    "C25",
    "x[C12]"
   ],
   [
    "C26",
    "x[C103]"
   ],
   [
    "C27",
    "x[C45]"
   ],
   [
    "C28",
    "x[C63]"
   ],
   [
    "C29",
    "x[C6]"
   ],
   [
    "C30",
    "x[C2]"
   ],
   [
    "C31",
    "x[C79]"
   ],
   [
    "C32",
    "x[C54]"
   ],
   [
    "C33",
    "x[C33]"
   ],
   [
    "C34",
    "x[C96]"
   ],
   [
    "C35",
    "x[C31]"
   ],
   [
    "C36",
    "x[C62]"
   ],
   [
    "C37",
    "x[C98]"
   ],
   [
    "C38",
    "x[C38]"
   ],
   [
    "C39",
    "x[C91]"
   ],
   [
    "C40",
    "x[C109]"
   ],
   [
    "C41",
    "x[C67]"
   ],
   [
    "C42",
    "x[C92]"
   ],
   [
    "C43",
    "x[C86]"
   ],
   [
    "C44",
    "x[C97]"
   ],
   [
    "C45",
    "x[C3]"
   ],
   [
    "C46",
    "x[C49]"
   ],
   [
    "C47",
    "x[C78]"
   ],
   [
    "C48",
    "x[C108]"
   ],
   [
    "C49",
    "x[C110]"
   ],
   [
    "C50",
    "x[C82]"
   ],
   [
    "C51",
    "x[C71]"
   ],
   [
    "C52",
    "x[C20]"
   ],
   [
    "C53",
    "x[C26]"
   ],
   [
    "C54",
    "x[C99]"
   ],
   [
    "C55",
    "x[C21]"
   ],
   [
    "C56",
    "x[C10]"
   ],
   [
    "C57",
    "x[C13]"
   ],
   [
    "C58",
    "x[C77]"
   ],
   [
    "C59",
    "x[C40]"
   ],
   [
    "C60",
    "x[C93]"
   ],
   [
    "C61",
    "x[C11]"
   ],
   [
    "C62",
    "x[C24]"
   ],
   [
    "C63",
    "x[C25]"
   ],
   [
    "C64",
    "x[C14]"
   ],
   [
    "C65",
    "x[C69]"
   ],
   [
    "C66",
    "x[C83]"
   ],
   [
    "C67",
    "x[C47]"
   ],
   [
    "C68",
    "x[C39]"
   ],
   [
    "C69",
    "x[C85]"
   ],
   [
    "C70",
    "x[C111]"
   ],
   [
    "C71",
    "x[C90]"
   ],
   [
    "C72",
    "x[C107]"
   ],
   [
    "C73",
    "x[C73]"
   ],
   [
    "C74",
    "x[C28]"
   ],
   [
    "C75",
    "x[C87]"
   ],
   [
    "C76",
    "x[C80]"
   ],
   [
    "C77",
    "x[C75]"
   ],
   [
    "C78",
    "x[C68]"
   ],
   [
    "C79",
    "x[C19]"
   ],
   [
    "C80",
    "x[C65]"
   ],
   [
    "C81",
    "x[C51]"
   ],
   [
    "C82",
    "x[C8]"
   ],
   [
    "C83",
    "x[C36]"
   ],
   [
    "C84",
    "x[C7]"
   ],
   [
    "C85",
    "x[C27]"
   ],
   [
    "C86",
    "x[C48]"
   ],
   [
    "C87",
    "x[C89]"
   ],
   [
    "C88",
    "x[C57]"
   ],
   [
    "C89",
    "x[C17]"
   ],
   [
    "C90",
    "x[C102]"
   ],
   [
    "C91",
    "x[C106]"
   ],
   [
    "C92",
    "x[C1]"
   ],
   [
    "C93",
    "x[C95]"
   ],
   [
    "C94",
    "x[C41]"
   ],
   [
    "C95",
    "x[C34]"
   ],
   [
    "C96",
    "x[C59]"
   ],
   [
    "C97",
    "x[C94]"
   ],
   [
    "C98",
    "x[C32]"
   ],
   [
    "C99",
    "x[C55]"
   ],
   [
    "C100",
    "x[C23]"
   ],
   [
    "C101",
    "x[C64]"
   ],
   [
    "C102",
    "x[C76]"
   ],
   [
    "C103",
    "x[C66]"
   ],
   [
    "C104",
    "x[C50]"
   ],
   [
    "C105",
    "x[C29]"
   ],
   [
    "C106",
    "x[C46]"
   ],
   [
    "C107",
    "x[C72]"
   ],
   [
    "C108",
    "x[C84]"
   ],
   [
    "C109",
    "x[C35]"
   ],
   [
    "C110",
    "x[C70]"
   ]
  ],
  "primary_mapping": {
   "x[1]": "x[C1]",
   "x[2]": "x[C2]",
   "x[3]": "x[C3]",
   "x[4]": "x[C4]",
   "x[5]": "x[C5]",
   "x[6]": "x[C6]",
   "x[7]": "x[C7]",
   "x[8]": "x[C8]",
   "x[9]": "x[C9]",
   "x[10]": "x[C10]",
   "x[11]": "x[C11]",
   "x[12]": "x[C12]",
   "x[13]": "x[C13]",
   "x[14]": "x[C14]",
   "x[15]": "x[C15]",
   "x[16]": "x[C16]",
   "x[17]": "x[C17]",
   "x[18]": "x[C18]",
   "x[19]": "x[C19]",
   "x[20]": "x[C20]",
   "x[21]": "x[C21]",
   "x[22]": "x[C22]",
   "x[23]": "x[C23]",
   "x[24]": "x[C24]",
   "x[25]": "x[C25]",
   "x[26]": "x[C26]",
   "x[27]": "x[C27]",
   "x[28]": "x[C28]",
   "x[29]": "x[C29]",
   "x[30]": "x[C30]",
   "x[31]": "x[C31]",
   "x[32]": "x[C32]",
   "x[33]": "x[C33]",
   "x[34]": "x[C34]",
   "x[35]": "x[C35]",
   "x[36]": "x[C36]",
   "x[37]": "x[C37]",
   "x[38]": "x[C38]",
   "x[39]": "x[C39]",
   "x[40]": "x[C40]",
   "x[41]": "x[C41]",
   "x[42]": "x[C42]",
   "x[43]": "x[C43]",
   "x[44]": "x[C44]",
   "x[45]": "x[C45]",
   "x[46]": "x[C46]",
   "x[47]": "x[C47]",
   "x[48]": "x[C48]",
   "x[49]": "x[C49]",
   "x[50]": "x[C50]",
   "x[51]": "x[C51]",
   "x[52]": "x[C52]",
   "x[53]": "x[C53]",
   "x[54]": "x[C54]",
   "x[55]": "x[C55]",
   "x[56]": "x[C56]",
   "x[57]": "x[C57]",
   "x[58]": "x[C58]",
   "x[59]": "x[C59]",
   "x[60]": "x[C60]",
   "x[61]": "x[C61]",
   "x[62]": "x[C62]",
   "x[63]": "x[C63]",
   "x[64]": "x[C64]",
   "x[65]": "x[C65]",
   "x[66]": "x[C66]",
   "x[67]": "x[C67]",
   "x[68]": "x[C68]",
   "x[69]": "x[C69]",
   "x[70]": "x[C70]",
   "x[71]": "x[C71]",
   "x[72]": "x[C72]",
   "x[73]": "x[C73]",
   "x[74]": "x[C74]",
   "x[75]": "x[C75]",
   "x[76]": "x[C76]",
   "x[77]": "x[C77]",
   "x[78]": "x[C78]",
   "x[79]": "x[C79]",
   "x[80]": "x[C80]",
   "x[81]": "x[C81]",
   "x[82]": "x[C82]",
   "x[83]": "x[C83]",
   "x[84]": "x[C84]",
   "x[85]": "x[C85]",
   "x[86]": "x[C86]",
   "x[87]": "x[C87]",
   "x[88]": "x[C88]",
   "x[89]": "x[C89]",
   "x[90]": "x[C90]",
   "x[91]": "x[C91]",
   "x[92]": "x[C92]",
   "x[93]": "x[C93]",
   "x[94]": "x[C94]",
   "x[95]": "x[C95]",
   "x[96]": "x[C96]",
   "x[97]": "x[C97]",
   "x[98]": "x[C98]",
   "x[99]": "x[C99]",
   "x[100]": "x[C100]",
   "x[101]": "x[C101]",
   "x[102]": "x[C102]",
   "x[103]": "x[C103]",
   "x[104]": "x[C104]",
   "x[105]": "x[C105]",
   "x[106]": "x[C106]",
   "x[107]": "x[C107]",
   "x[108]": "x[C108]",
   "x[109]": "x[C109]",
   "x[110]": "x[C110]",
   "x[111]": "x[C111]"
  },
  "mapping_basis": "reviewed_gpt_physical_projection",
  "frozen_replay_solution": {
   "status": "OPTIMAL",
   "objective": 1382045.0,
   "variables": [
    [
     "C0",
     -0.0
    ],
    [
     "C1",
     -0.0
    ],
    [
     "C2",
     -0.0
    ],
    [
     "C3",
     -0.0
    ],
    [
     "C4",
     -0.0
    ],
    [
     "C5",
     -0.0
    ],
    [
     "C6",
     -0.0
    ],
    [
     "C7",
     -0.0
    ],
    [
     "C8",
     -0.0
    ],
    [
     "C9",
     -0.0
    ],
    [
     "C10",
     -0.0
    ],
    [
     "C11",
     -0.0
    ],
    [
     "C12",
     -0.0
    ],
    [
     "C13",
     -0.0
    ],
    [
     "C14",
     -0.0
    ],
    [
     "C15",
     -0.0
    ],
    [
     "C16",
     -0.0
    ],
    [
     "C17",
     -0.0
    ],
    [
     "C18",
     -0.0
    ],
    [
     "C19",
     -0.0
    ],
    [
     "C20",
     -0.0
    ],
    [
     "C21",
     -0.0
    ],
    [
     "C22",
     -0.0
    ],
    [
     "C23",
     -0.0
    ],
    [
     "C24",
     -0.0
    ],
    [
     "C25",
     -0.0
    ],
    [
     "C26",
     -0.0
    ],
    [
     "C27",
     -0.0
    ],
    [
     "C28",
     -0.0
    ],
    [
     "C29",
     -0.0
    ],
    [
     "C30",
     -0.0
    ],
    [
     "C31",
     -0.0
    ],
    [
     "C32",
     -0.0
    ],
    [
     "C33",
     -0.0
    ],
    [
     "C34",
     -0.0
    ],
    [
     "C35",
     -0.0
    ],
    [
     "C36",
     -0.0
    ],
    [
     "C37",
     -0.0
    ],
    [
     "C38",
     -0.0
    ],
    [
     "C39",
     1290.0
    ],
    [
     "C40",
     -0.0
    ],
    [
     "C41",
     -0.0
    ],
    [
     "C42",
     -0.0
    ],
    [
     "C43",
     -0.0
    ],
    [
     "C44",
     -0.0
    ],
    [
     "C45",
     -0.0
    ],
    [
     "C46",
     -0.0
    ],
    [
     "C47",
     4882.0
    ],
    [
     "C48",
     -0.0
    ],
    [
     "C49",
     105.0
    ],
    [
     "C50",
     -0.0
    ],
    [
     "C51",
     -0.0
    ],
    [
     "C52",
     -0.0
    ],
    [
     "C53",
     -0.0
    ],
    [
     "C54",
     -0.0
    ],
    [
     "C55",
     -0.0
    ],
    [
     "C56",
     -0.0
    ],
    [
     "C57",
     -0.0
    ],
    [
     "C58",
     -0.0
    ],
    [
     "C59",
     -0.0
    ],
    [
     "C60",
     -0.0
    ],
    [
     "C61",
     -0.0
    ],
    [
     "C62",
     -0.0
    ],
    [
     "C63",
     -0.0
    ],
    [
     "C64",
     -0.0
    ],
    [
     "C65",
     -0.0
    ],
    [
     "C66",
     -0.0
    ],
    [
     "C67",
     1.0
    ],
    [
     "C68",
     -0.0
    ],
    [
     "C69",
     -0.0
    ],
    [
     "C70",
     -0.0
    ],
    [
     "C71",
     -0.0
    ],
    [
     "C72",
     -0.0
    ],
    [
     "C73",
     -0.0
    ],
    [
     "C74",
     -0.0
    ],
    [
     "C75",
     -0.0
    ],
    [
     "C76",
     -0.0
    ],
    [
     "C77",
     -0.0
    ],
    [
     "C78",
     -0.0
    ],
    [
     "C79",
     -0.0
    ],
    [
     "C80",
     -0.0
    ],
    [
     "C81",
     -0.0
    ],
    [
     "C82",
     -0.0
    ],
    [
     "C83",
     -0.0
    ],
    [
     "C84",
     -0.0
    ],
    [
     "C85",
     -0.0
    ],
    [
     "C86",
     -0.0
    ],
    [
     "C87",
     -0.0
    ],
    [
     "C88",
     1.0
    ],
    [
     "C89",
     -0.0
    ],
    [
     "C90",
     -0.0
    ],
    [
     "C91",
     -0.0
    ],
    [
     "C92",
     -0.0
    ],
    [
     "C93",
     -0.0
    ],
    [
     "C94",
     -0.0
    ],
    [
     "C95",
     -0.0
    ],
    [
     "C96",
     -0.0
    ],
    [
     "C97",
     -0.0
    ],
    [
     "C98",
     -0.0
    ],
    [
     "C99",
     -0.0
    ],
    [
     "C100",
     -0.0
    ],
    [
     "C101",
     -0.0
    ],
    [
     "C102",
     -0.0
    ],
    [
     "C103",
     -0.0
    ],
    [
     "C104",
     -0.0
    ],
    [
     "C105",
     -0.0
    ],
    [
     "C106",
     -0.0
    ],
    [
     "C107",
     -0.0
    ],
    [
     "C108",
     3025.0
    ],
    [
     "C109",
     -0.0
    ],
    [
     "C110",
     -0.0
    ]
   ],
   "error": null
  }
 }
}
""")


In [ ]:
IDENTITY_ARTIFACTS['LEAN-gpt-oss-20b'] = OSS20B_IDENTITIES


In [ ]:
EMBEDDED_INPUT_SHA256 = json.loads(r"""
{
 "COHORTS": "fbd2a3c96f24a4df30ffe153c726e2a34e384d1b0e543d5da8538db4656f7892",
 "DATA_SHA256": "4b938580df768f05fbfe37d2d320e819728d5b7fb610b392cb2911033b118668",
 "REFERENCE_CERTIFICATES": "ae20c417b98d56b5634b00fb7a72034dab48c2ba4faffe203066e11e3f6862d5",
 "IDENTITY_ARTIFACTS": "344c449b8642bf41c34bbc8468c1abc8b56f4518471632a849e249121631440c"
}
""")


In [ ]:
for name, expected in EMBEDDED_INPUT_SHA256.items():
    value = globals()[name]
    actual = hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        separators=(',', ':'), allow_nan=False).encode()).hexdigest()
    assert actual == expected, f'Embedded input annotations changed: {name}'
assert set(METHODS) <= set(COHORTS)
assert all(set(IDENTITY_ARTIFACTS[method]) == EXPECTED_IDS for method in METHODS)
display(pd.DataFrame.from_dict(COHORTS, orient='index')[['relative_directory', 'source_commit']])


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
import math
import re
import tempfile
from typing import Any, Callable

import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd


MARKDOWN_MODEL_IDS = {73, 80, 83, 84, 85, 87, 90, 91, 92, 93, 95, 97, 100}





_LP_AUXILIARY_VARIABLES: dict[int, set[str]] = {
    82: {"U[1]", "U[2]", "U[3]"},
}

_SECTION_RE = re.compile(
    r"(?im)^\s*(Bounds|Binaries|Binary|Generals|General|Integers|Integer|End)\s*$"
)
_INTEGER_SECTIONS = {"generals", "general", "integers", "integer"}
_BINARY_SECTIONS = {"binaries", "binary"}
_NUMBER_RE = re.compile(
    r"Optimal\s+objective\s*[:=]?\s*"
    r"([-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?)",
    re.I,
)


@dataclass
class GroundTruthBundle:
    benchmark_index: int
    model: gp.Model
    source_kind: str
    published_objective: float | None
    optimum: float | None = None
    solver_status: int | None = None
    auxiliary_variables: set[str] = field(default_factory=set)
    metadata: dict[str, Any] = field(default_factory=dict)


def parse_published_objective(value: Any) -> float | None:
    if value is None:
        return None
    if isinstance(value, (int, float, np.number)):
        number = float(value)
        return None if math.isnan(number) else number
    text = str(value).strip()
    if not text:
        return None
    match = _NUMBER_RE.search(text)
    if match:
        return float(match.group(1).replace(",", ""))
    cleaned = re.sub(r"[,\s$¥€£]", "", text)
    try:
        return float(cleaned)
    except ValueError:
        return None


def _segment_declared_names(payload: str, candidates: list[str]) -> tuple[list[str], str]:
    compact = re.sub(r"\s+", "", payload)
    names = sorted(set(candidates), key=lambda value: (-len(value), value))
    found: list[str] = []
    unknown: list[str] = []
    cursor = 0
    while cursor < len(compact):
        match = next((name for name in names if compact.startswith(name, cursor)), None)
        if match is not None:
            found.append(match)
            cursor += len(match)
        else:
            unknown.append(compact[cursor])
            cursor += 1
    return found, "".join(unknown)


def repair_lp_variable_types(model: gp.Model, text: str) -> dict[str, Any]:


    matches = list(_SECTION_RE.finditer(text))
    first_type_offset = min(
        (
            match.start()
            for match in matches
            if match.group(1).strip().casefold()
            in _INTEGER_SECTIONS | _BINARY_SECTIONS
        ),
        default=len(text),
    )
    body = text[:first_type_offset]
    variables = list(model.getVars())
    candidates = [var.VarName for var in variables if var.VarName in body]
    assignments: dict[str, str] = {}
    unknown_fragments: list[str] = []

    for index, match in enumerate(matches):
        section = match.group(1).strip().casefold()
        if section not in _INTEGER_SECTIONS | _BINARY_SECTIONS:
            continue
        stop = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        names, unknown = _segment_declared_names(text[match.end() : stop], candidates)
        if unknown:


            unknown_fragments.append(unknown[:500])
        target_type = GRB.INTEGER if section in _INTEGER_SECTIONS else GRB.BINARY
        for name in names:
            assignments[name] = target_type

    for name, variable_type in assignments.items():
        variable = model.getVarByName(name)
        if variable is not None:
            variable.VType = variable_type

    ghosts = [
        variable
        for variable in variables
        if variable.VarName not in body
        and model.getCol(variable).size() == 0
        and abs(variable.Obj) == 0.0
    ]
    ghost_names = [variable.VarName for variable in ghosts]
    if ghosts:
        model.remove(ghosts)
    model.update()
    return {
        "repaired_variable_count": len(assignments),
        "repaired_variables": sorted(assignments),
        "removed_ghost_count": len(ghost_names),
        "removed_ghost_variables": ghost_names,
        "unparsed_type_fragments": unknown_fragments,
    }


def _quiet_model(name: str) -> gp.Model:
    model = gp.Model(name)
    model.Params.OutputFlag = 0
    return model


def _csv(repo_root: Path, relative: str, **kwargs: Any) -> pd.DataFrame:
    return pd.read_csv(repo_root / relative, **kwargs)


def _build_73(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture3/43.csv"
    data = _csv(repo_root, relative).set_index("Equipment / Cost")
    equipment_products = {
        "A1": ["I", "II"],
        "A2": ["I", "II", "III"],
        "B1": ["I", "II"],
        "B2": ["I", "III"],
        "B3": ["I"],
    }
    column = {"I": "Product I", "II": "Product II", "III": "Product III"}
    price = {p: float(data.loc["Unit Price (yuan/unit)", column[p]]) for p in column}
    raw = {p: float(data.loc["Raw Material Cost (yuan/unit)", column[p]]) for p in column}
    model = _quiet_model("gt_073_mixture")
    x = {
        (equipment, product): model.addVar(lb=0.0, name=f"x[{equipment},{product}]")
        for equipment, products in equipment_products.items()
        for product in products
    }
    production = {
        "I": x["A1", "I"] + x["A2", "I"],
        "II": x["A1", "II"] + x["A2", "II"],
        "III": x["A2", "III"],
    }
    revenue_minus_raw = gp.quicksum((price[p] - raw[p]) * production[p] for p in column)
    equipment_cost = gp.quicksum(
        float(data.loc[equipment, "Equipment Cost at Full Load (yuan)"])
        / float(data.loc[equipment, "Available Equipment Operating Time"])
        * gp.quicksum(
            float(data.loc[equipment, column[product]]) * x[equipment, product]
            for product in products
        )
        for equipment, products in equipment_products.items()
    )
    model.setObjective(revenue_minus_raw - equipment_cost, GRB.MAXIMIZE)
    for equipment, products in equipment_products.items():
        model.addConstr(
            gp.quicksum(
                float(data.loc[equipment, column[product]]) * x[equipment, product]
                for product in products
            )
            <= float(data.loc[equipment, "Available Equipment Operating Time"]),
            name=f"capacity[{equipment}]",
        )
    model.addConstr(
        production["I"] == x["B1", "I"] + x["B2", "I"] + x["B3", "I"],
        name="flow[I]",
    )
    model.addConstr(production["II"] == x["B1", "II"], name="flow[II]")
    model.addConstr(production["III"] == x["B2", "III"], name="flow[III]")
    return model, set()


def _build_80(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture6/parameters.csv"
    data = _csv(repo_root, relative)
    trucks = [int(value) for value in data["truck_id"]]
    periods = [1, 2, 3, 4]
    q = dict(zip(trucks, data["Q"].astype(float)))
    startup = dict(zip(trucks, data["S"].astype(float)))
    cost = dict(zip(trucks, data["C"].astype(float)))
    demand = {t: float(data[f"d{t}"].iloc[0]) for t in periods}
    model = _quiet_model("gt_080_truck_schedule")
    w = model.addVars(trucks, periods, lb=0.0, name="w")
    y = model.addVars(trucks, periods, vtype=GRB.BINARY, name="y")
    u = model.addVars(trucks, [2, 3, 4], vtype=GRB.BINARY, name="u")
    model.setObjective(
        gp.quicksum(cost[i] * w[i, t] for i in trucks for t in periods)
        + gp.quicksum(startup[i] * y[i, 1] for i in trucks)
        + gp.quicksum(startup[i] * u[i, t] for i in trucks for t in [2, 3, 4]),
        GRB.MINIMIZE,
    )
    for t in periods:
        model.addConstr(gp.quicksum(w[i, t] for i in trucks) >= demand[t], name=f"demand[{t}]")
        model.addConstr(
            gp.quicksum(w[i, t] for i in trucks)
            <= 0.9 * gp.quicksum(q[i] * y[i, t] for i in trucks),
            name=f"reserve[{t}]",
        )
    for i in trucks:
        for t in periods:
            model.addConstr(w[i, t] <= q[i] * y[i, t], name=f"link[{i},{t}]")
        for t in [2, 3, 4]:
            model.addConstr(u[i, t] >= y[i, t] - y[i, t - 1], name=f"startup_lb[{i},{t}]")
            model.addConstr(u[i, t] <= y[i, t], name=f"startup_on[{i},{t}]")
            model.addConstr(u[i, t] <= 1 - y[i, t - 1], name=f"startup_off[{i},{t}]")
            model.addConstr(w[i, t] - w[i, t - 1] <= 300, name=f"ramp_up[{i},{t}]")
            model.addConstr(w[i, t - 1] - w[i, t] <= 300, name=f"ramp_down[{i},{t}]")
        model.addConstr(y[i, 1] <= y[i, 2], name=f"min_runtime_initial[{i}]")
        for t in [2, 3]:
            model.addConstr(y[i, t] + y[i, t + 1] >= 2 * u[i, t], name=f"min_runtime[{i},{t}]")
            model.addConstr(
                y[i, t - 1] - y[i, t] <= 1 - y[i, t + 1],
                name=f"cooldown[{i},{t}]",
            )
        model.addConstr(u[i, 4] == 0, name=f"no_start_4[{i}]")
    return model, {f"u[{i},{t}]" for i in trucks for t in [2, 3, 4]}


def _build_83(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture7/15.csv"
    frame = _csv(repo_root, relative)
    task_columns = list(frame.columns[1:])
    numeric = frame[task_columns].apply(pd.to_numeric, errors="coerce")
    keep = numeric.notna().all(axis=1)
    hours = numeric.loc[keep].to_numpy(dtype=float)
    workers = list(range(1, hours.shape[0] + 1))
    model = _quiet_model("gt_083_assignment")
    x = model.addVars(workers, task_columns, vtype=GRB.BINARY, name="x")
    model.setObjective(
        gp.quicksum(hours[i - 1, j] * x[i, task] for i in workers for j, task in enumerate(task_columns)),
        GRB.MINIMIZE,
    )
    for task in task_columns:
        model.addConstr(gp.quicksum(x[i, task] for i in workers) == 1, name=f"task[{task}]")
    for i in workers:
        model.addConstr(gp.quicksum(x[i, task] for task in task_columns) <= 1, name=f"worker[{i}]")
    model.addConstr(gp.quicksum(x.values()) == 10, name="ten_assignments")
    return model, set()


def _build_84(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Others_example/Others6/18.csv"
    frame = _csv(repo_root, relative)
    instructions = frame.iloc[0, 1:].astype(float).to_numpy()
    processors = [1, 2, 3]
    speeds = {1: 1.33, 2: 2.0, 3: 2.66}
    tasks = list(range(1, len(instructions) + 1))
    model = _quiet_model("gt_084_parallel_machine")
    x = model.addVars(processors, tasks, vtype=GRB.BINARY, name="x")
    cmax = model.addVar(lb=0.0, name="Cmax")
    model.setObjective(cmax, GRB.MINIMIZE)
    for task in tasks:
        model.addConstr(gp.quicksum(x[m, task] for m in processors) == 1, name=f"assign[{task}]")
    for machine in processors:
        model.addConstr(
            gp.quicksum(instructions[task - 1] / speeds[machine] * x[machine, task] for task in tasks)
            <= cmax,
            name=f"load[{machine}]",
        )
    return model, {"Cmax"}


def _build_85(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Others_example/Others7/20.csv"
    raw = _csv(repo_root, relative, index_col=0).apply(pd.to_numeric, errors="coerce").to_numpy()
    n = raw.shape[0]
    distance = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            value = raw[i, j] if not np.isnan(raw[i, j]) else raw[j, i]
            distance[i, j] = float(value)
    cities = list(range(1, n + 1))
    model = _quiet_model("gt_085_tsp")
    arcs = [(i, j) for i in cities for j in cities if i != j]
    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    order = model.addVars(cities[1:], lb=2.0, ub=float(n), name="u")
    model.setObjective(
        gp.quicksum(distance[i - 1, j - 1] * x[i, j] for i, j in arcs),
        GRB.MINIMIZE,
    )
    for i in cities:
        model.addConstr(gp.quicksum(x[i, j] for j in cities if j != i) == 1, name=f"out[{i}]")
    for j in cities:
        model.addConstr(gp.quicksum(x[i, j] for i in cities if i != j) == 1, name=f"in[{j}]")
    for i in cities[1:]:
        for j in cities[1:]:
            if i != j:
                model.addConstr(order[i] - order[j] + n * x[i, j] <= n - 1, name=f"mtz[{i},{j}]")
    return model, {f"u[{city}]" for city in cities[1:]}


def _build_87(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture9"
    attributes = _csv(repo_root, f"{base}/36-1.csv", index_col=0)
    activation = _csv(repo_root, f"{base}/36-2.csv", index_col=0)
    batches = _csv(repo_root, f"{base}/36-3.csv", index_col=0)
    products = list(range(1, 81))
    demand = attributes.loc["Maximum Demand (100 kg units)"].astype(float).to_numpy()
    selling = attributes.loc["Selling Price ($/100 kg)"].astype(float).to_numpy()
    production = attributes.loc["Production Cost ($/100 kg)"].astype(float).to_numpy()
    quota = attributes.loc["Production Quota (max per day)"].astype(float).to_numpy()
    fixed = activation.iloc[0].astype(float).to_numpy()
    minimum = batches.iloc[0].astype(float).to_numpy()
    model = _quiet_model("gt_087_fixed_charge_production")
    x = model.addVars(products, lb=0.0, vtype=GRB.INTEGER, name="x")
    y = model.addVars(products, vtype=GRB.BINARY, name="y")
    model.setObjective(
        gp.quicksum((selling[i - 1] - production[i - 1]) * x[i] - fixed[i - 1] * y[i] for i in products),
        GRB.MAXIMIZE,
    )
    for i in products:
        model.addConstr(x[i] <= demand[i - 1], name=f"demand[{i}]")
        model.addConstr(x[i] <= 22.0 * quota[i - 1] * y[i], name=f"activation_ub[{i}]")
        model.addConstr(x[i] >= minimum[i - 1] * y[i], name=f"activation_lb[{i}]")
    return model, set()


def _build_90(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture10"
    products_frame = _csv(repo_root, f"{base}/factory_products_100.csv")
    capacities_frame = _csv(repo_root, f"{base}/resources_capacities.csv")
    products = list(range(1, len(products_frame) + 1))
    batch = products_frame["batch_size_units"].astype(float).to_numpy()
    profit = products_frame["profit_per_unit"].astype(float).to_numpy()
    demand = products_frame["upper_demand_units"].astype(float).to_numpy()
    capacities = dict(zip(capacities_frame["resource"], capacities_frame["capacity"].astype(float)))
    model = _quiet_model("gt_090_batch_production")
    x = model.addVars(products, lb=0.0, vtype=GRB.INTEGER, name="x")
    model.setObjective(gp.quicksum(batch[i - 1] * profit[i - 1] * x[i] for i in products), GRB.MAXIMIZE)
    for resource, column in [("R1", "r1_per_unit"), ("R2", "r2_per_unit"), ("R3", "r3_per_unit")]:
        usage = products_frame[column].astype(float).to_numpy()
        model.addConstr(
            gp.quicksum(batch[i - 1] * usage[i - 1] * x[i] for i in products) <= capacities[resource],
            name=f"resource[{resource}]",
        )
    for i in products:
        model.addConstr(batch[i - 1] * x[i] <= demand[i - 1], name=f"demand[{i}]")
    return model, set()


def _build_91(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture11/courses_42.csv"
    frame = _csv(repo_root, relative)
    frame = frame.loc[frame["discipline"].astype(str).str.casefold() == "operations research"].reset_index(drop=True)
    courses = list(range(1, len(frame) + 1))
    model = _quiet_model("gt_091_course_selection")
    x = model.addVars(courses, vtype=GRB.BINARY, name="x")
    model.setObjective(gp.quicksum(float(frame.loc[i - 1, "interest_points"]) * x[i] for i in courses), GRB.MAXIMIZE)
    model.addConstr(
        gp.quicksum(float(frame.loc[i - 1, "credits"]) * x[i] for i in courses) <= 20,
        name="credit_limit",
    )
    return model, set()


def _build_92(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture12"
    sources = _csv(repo_root, f"{base}/expanded_sources.csv")
    destinations = _csv(repo_root, f"{base}/expanded_destinations.csv")
    costs = _csv(repo_root, f"{base}/expanded_cost_matrix.csv", index_col=0)
    origins = list(range(1, len(sources) + 1))
    sinks = list(range(1, len(destinations) + 1))
    supply = sources["supply_units"].astype(float).to_numpy()
    demand = destinations["demand_units"].astype(float).to_numpy()
    cost = costs.astype(float).to_numpy()
    model = _quiet_model("gt_092_partial_truck_transport")
    cargo = model.addVars(origins, sinks, lb=0.0, name="y")
    trucks = model.addVars(origins, sinks, lb=0.0, vtype=GRB.INTEGER, name="x")
    model.setObjective(gp.quicksum(cost[i - 1, j - 1] * cargo[i, j] for i in origins for j in sinks), GRB.MINIMIZE)
    for j in sinks:
        model.addConstr(gp.quicksum(cargo[i, j] for i in origins) == demand[j - 1], name=f"demand[{j}]")
    for i in origins:
        model.addConstr(gp.quicksum(cargo[i, j] for j in sinks) <= supply[i - 1], name=f"supply[{i}]")
    for i in origins:
        for j in sinks:
            model.addConstr(cargo[i, j] <= 10.0 * trucks[i, j], name=f"truck_capacity[{i},{j}]")
    return model, set()


def _build_93(repo_root: Path) -> tuple[gp.Model, set[str]]:
    relative = "Test_Dataset/Large-scale-or/AP_testing/AP1/cost_12x12.csv"
    frame = _csv(repo_root, relative, index_col=0)
    machines = list(range(1, len(frame) + 1))
    tasks = list(frame.columns)
    costs = frame.astype(float).to_numpy()
    model = _quiet_model("gt_093_assignment")
    x = model.addVars(machines, tasks, vtype=GRB.BINARY, name="x")
    model.setObjective(
        gp.quicksum(costs[i - 1, j] * x[i, task] for i in machines for j, task in enumerate(tasks)),
        GRB.MINIMIZE,
    )
    for i in machines:
        model.addConstr(gp.quicksum(x[i, task] for task in tasks) == 1, name=f"machine[{i}]")
    for task in tasks:
        model.addConstr(gp.quicksum(x[i, task] for i in machines) == 1, name=f"task[{task}]")
    return model, set()


def _build_95(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture14"
    products_frame = _csv(repo_root, f"{base}/product_resources.csv")
    limits_frame = _csv(repo_root, f"{base}/resource_limits.csv")
    limits = dict(zip(limits_frame["Resource"], limits_frame["MonthlyLimit"].astype(float)))
    products = list(range(1, len(products_frame) + 1))
    model = _quiet_model("gt_095_byproduct_production")
    x = model.addVars(products, lb=0.0, name="x")
    sold = model.addVar(lb=0.0, name="s")
    profit = products_frame["Profit"].astype(float).to_numpy()


    model.setObjective(
        gp.quicksum(profit[i - 1] * x[i] for i in products) - 1000.0 * x[3] + 500.0 * sold,
        GRB.MAXIMIZE,
    )
    for column, key in [("LaborHours", "LaborHours"), ("MaterialA", "MaterialA"), ("MaterialB", "MaterialB")]:
        coefficients = products_frame[column].astype(float).to_numpy()
        model.addConstr(
            gp.quicksum(coefficients[i - 1] * x[i] for i in products) <= limits[key],
            name=f"resource[{key}]",
        )
    model.addConstr(sold <= 1500.0, name="sales_cap")
    model.addConstr(sold <= 5.0 * x[3], name="byproduct_balance")
    return model, set()


def _build_97(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Mixture_testing/Mixture16"
    options = _csv(repo_root, f"{base}/OptionCharacteristics.csv")
    references = _csv(repo_root, f"{base}/Option_AssetReferenceMatrix.csv", index_col=0)
    indices = list(range(1, len(options) + 1))
    multiplicity = references.astype(float).sum(axis=1).to_numpy()
    model = _quiet_model("gt_097_option_hedging")
    x = {
        i: model.addVar(
            lb=float(options.loc[i - 1, "MaxShort"]),
            ub=float(options.loc[i - 1, "MaxLong"]),
            vtype=GRB.INTEGER,
            name=f"x[{i}]",
        )
        for i in indices
    }
    z = model.addVars(indices, lb=0.0, name="z")
    model.setObjective(gp.quicksum(float(options.loc[i - 1, "Cost"]) * z[i] for i in indices), GRB.MINIMIZE)
    for i in indices:
        model.addConstr(z[i] >= x[i], name=f"abs_pos[{i}]")
        model.addConstr(z[i] >= -x[i], name=f"abs_neg[{i}]")
    for greek, initial, tolerance in [("Delta", 0.25, 0.06), ("Gamma", 0.08, 0.05), ("Vega", 0.17, 0.07)]:
        exposure = gp.quicksum(
            float(options.loc[i - 1, greek]) * multiplicity[i - 1] * x[i]
            for i in indices
        )
        model.addConstr(exposure <= tolerance - initial, name=f"{greek}_upper")
        model.addConstr(exposure >= -tolerance - initial, name=f"{greek}_lower")
    return model, {f"z[{i}]" for i in indices}


def _build_100(repo_root: Path) -> tuple[gp.Model, set[str]]:
    base = "Test_Dataset/Large-scale-or/Others_example/Others8"
    times = _csv(repo_root, f"{base}/processing_time_unit.csv", index_col=0)
    prices = _csv(repo_root, f"{base}/unit_price.csv", index_col=0)
    hours = _csv(repo_root, f"{base}/total_working_hours.csv")
    components = list(range(1, times.shape[1] + 1))
    price = prices.iloc[:, 0].astype(float).to_numpy()
    capacities = dict(zip(hours["workshop"], hours["total_hours"].astype(float)))
    model = _quiet_model("gt_100_flexible_flow_shop")
    x = model.addVars(components, lb=0.0, name="x")
    model.setObjective(gp.quicksum(price[j - 1] * x[j] for j in components), GRB.MAXIMIZE)
    for workshop, row in times.iterrows():
        coefficients = row.astype(float).to_numpy()
        model.addConstr(
            gp.quicksum(coefficients[j - 1] * x[j] for j in components) <= capacities[workshop],
            name=f"workshop[{workshop}]",
        )
    return model, set()


_BUILDERS: dict[int, Callable[[Path], tuple[gp.Model, set[str]]]] = {
    73: _build_73,
    80: _build_80,
    83: _build_83,
    84: _build_84,
    85: _build_85,
    87: _build_87,
    90: _build_90,
    91: _build_91,
    92: _build_92,
    93: _build_93,
    95: _build_95,
    97: _build_97,
    100: _build_100,
}


def _read_lp_model(text: str) -> tuple[gp.Model, dict[str, Any]]:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".lp", encoding="utf-8") as handle:
        handle.write(text.rstrip() + "\n")
        handle.flush()
        model = gp.read(handle.name)
    model.Params.OutputFlag = 0
    metadata = repair_lp_variable_types(model, text)
    return model, metadata


def load_ground_truth(
    repo_root: str | Path,
    row: pd.Series,
    benchmark_index: int,
    *,
    solve: bool = True,
    time_limit_s: float = 300.0,
) -> GroundTruthBundle:


    root = Path(repo_root).resolve()
    published = parse_published_objective(row.get("Label-objective"))
    if benchmark_index in _BUILDERS:
        model, auxiliary = _BUILDERS[benchmark_index](root)
        source_kind = "deterministic_markdown_reconstruction"
        metadata: dict[str, Any] = {"constructor_id": benchmark_index}
    else:
        model, metadata = _read_lp_model(str(row["Label-model"]))
        auxiliary = set(_LP_AUXILIARY_VARIABLES.get(benchmark_index, set()))
        missing_auxiliary = auxiliary.difference(
            variable.VarName for variable in model.getVars()
        )
        if missing_auxiliary:
            raise ValueError(
                f"declared LP auxiliaries missing for row {benchmark_index}: "
                f"{sorted(missing_auxiliary)}"
            )
        if auxiliary:
            metadata["declared_lp_auxiliary_variables"] = sorted(auxiliary)
        source_kind = "repaired_label_lp"

    model.update()
    bundle = GroundTruthBundle(
        benchmark_index=benchmark_index,
        model=model,
        source_kind=source_kind,
        published_objective=published,
        auxiliary_variables=auxiliary,
        metadata=metadata,
    )
    if solve:
        model.Params.TimeLimit = float(time_limit_s)
        model.optimize()
        bundle.solver_status = int(model.Status)
        bundle.optimum = float(model.ObjVal) if model.SolCount > 0 else None
        bundle.metadata.update(
            num_variables=int(model.NumVars),
            num_constraints=int(model.NumConstrs),
            num_integer=int(sum(v.VType in {GRB.INTEGER, GRB.BINARY, GRB.SEMIINT} for v in model.getVars())),
            model_sense="min" if model.ModelSense == GRB.MINIMIZE else "max",
            label_relative_error=(
                abs(bundle.optimum - published) / max(1.0, abs(published))
                if bundle.optimum is not None and published is not None
                else None
            ),
        )
    return bundle


In [ ]:
def _derive_auxiliary_values(
    bundle: GroundTruthBundle,
    primary_values: dict[str, float],
    auxiliary_names: set[str],
) -> tuple[dict[str, float], dict[str, Any]]:












    if not auxiliary_names:
        return {}, {"auxiliary_completion": "not_needed"}
    completion = bundle.model.copy()
    completion.Params.OutputFlag = 0
    completion.Params.TimeLimit = 30.0
    completion.update()
    for name, value in primary_values.items():
        variable = completion.getVarByName(name)
        if variable is None:
            continue
        variable.VType = GRB.CONTINUOUS
        variable.LB = min(variable.LB, value)
        variable.UB = max(variable.UB, value)
        completion.addConstr(variable == value, name=f"__transfer_fix__{name}")
    completion.optimize()
    relaxed = False
    if completion.SolCount == 0:
        relaxed = True
        completion.feasRelaxS(0, False, False, True)
        completion.optimize()
    values: dict[str, float] = {}
    if completion.SolCount > 0:
        for name in auxiliary_names:
            variable = completion.getVarByName(name)
            if variable is not None:
                values[name] = float(variable.X)
    details = {
        "auxiliary_completion": "least_row_relaxation" if relaxed else "ground_truth_objective_completion",
        "auxiliary_completion_status": int(completion.Status),
        "auxiliary_completion_count": len(values),
    }
    completion.dispose()
    return values, details


In [ ]:
def binary_fraction(value):
    value = float(value)
    assert math.isfinite(value)
    return Fraction.from_float(value)


def residual(lhs, rhs, sense):
    if sense == "<":
        return max(Fraction(0), lhs - rhs)
    if sense == ">":
        return max(Fraction(0), rhs - lhs)
    assert sense == "="
    return abs(lhs - rhs)


def check_model(model, values):
    model.update()
    assert not any((model.NumQNZs, model.NumQConstrs, model.NumGenConstrs, model.NumSOS))
    variables = {v.VarName: v for v in model.getVars()}
    assert set(values) == set(variables)
    x = {name: binary_fraction(value) for name, value in values.items()}
    checks = []
    for row in model.getConstrs():
        expr = model.getRow(row)
        lhs = sum((binary_fraction(expr.getCoeff(k)) * x[expr.getVar(k).VarName]
                   for k in range(expr.size())), Fraction(0))
        rhs = binary_fraction(row.RHS)
        checks.append(dict(kind="constraint", name=row.ConstrName, sense=row.Sense,
                           lhs=lhs, rhs=rhs, violation=residual(lhs, rhs, row.Sense),
                           historical_tolerance=binary_fraction(1e-6 + 1e-6 * max(1, abs(float(rhs))))))
    for name, var in variables.items():
        assert var.VType in {"C", "I", "B"}
        for side, bound in (("lower", float(var.LB)), ("upper", float(var.UB))):
            if abs(bound) >= gp.GRB.INFINITY / 2:
                continue
            b = binary_fraction(bound)
            violation = max(Fraction(0), b - x[name] if side == "lower" else x[name] - b)
            checks.append(dict(kind="bound", name=name, side=side, value=x[name], bound=b,
                               violation=violation,
                               historical_tolerance=binary_fraction(1e-6 + 1e-6 * max(1, abs(bound)))))
        if var.VType in {"I", "B"}:
            checks.append(dict(kind="integrality", name=name, value=x[name],
                               violation=abs(x[name] - round(x[name])),
                               historical_tolerance=binary_fraction(1e-5)))
        if var.VType == "B":
            checks.append(dict(kind="binary_domain", name=name, value=x[name],
                               violation=max(Fraction(0), -x[name], x[name] - 1),
                               historical_tolerance=binary_fraction(2e-6)))
    objective = binary_fraction(model.ObjCon) + sum(
        (binary_fraction(v.Obj) * x[name] for name, v in variables.items()), Fraction(0))
    return objective, checks


In [ ]:
REPLAY_WORKER_SOURCE = 'from __future__ import annotations\nimport ast\nimport contextlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nimport traceback\nimport gurobipy as gp\nimport numpy as np\nimport pandas as pd\n\n_BLOCKED_IMPORT_ROOTS = {\n    "cffi",\n    "ctypes",\n    "http",\n    "openai",\n    "requests",\n    "shutil",\n    "socket",\n    "subprocess",\n    "urllib",\n}\n\n\n_BLOCKED_CALLS = {\n    "compile",\n    "eval",\n    "exec",\n    "__import__",\n    "os.remove",\n    "os.unlink",\n    "os.rmdir",\n    "os.removedirs",\n    "os.rename",\n    "os.replace",\n    "os.system",\n    "os.popen",\n    "pathlib.Path.unlink",\n    "pathlib.Path.rmdir",\n    "pathlib.Path.rename",\n    "pathlib.Path.replace",\n    "pathlib.Path.write_text",\n    "pathlib.Path.write_bytes",\n}\n\n\n_FORBIDDEN_READ_FRAGMENTS = {\n    "large-scale-or-101.csv",\n    "results/benchmark_test/",\n    "results/large-scale-or-results-",\n    "/runs/",\n}\n\n\ndef _call_name(node: ast.AST) -> str:\n    parts: list[str] = []\n    while isinstance(node, ast.Attribute):\n        parts.append(node.attr)\n        node = node.value\n    if isinstance(node, ast.Name):\n        parts.append(node.id)\n    return ".".join(reversed(parts))\n\n\ndef scan_code(code: str) -> list[str]:\n    try:\n        tree = ast.parse(code)\n    except SyntaxError as exc:\n        return [f"SyntaxError: {exc}"]\n    issues: list[str] = []\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Constant) and isinstance(node.value, str):\n            normalized = node.value.replace("\\\\", "/").casefold()\n            for fragment in _FORBIDDEN_READ_FRAGMENTS:\n                if fragment in normalized:\n                    issues.append(f"blocked evaluation-artifact path: {fragment}")\n        if isinstance(node, ast.Import):\n            for alias in node.names:\n                if alias.name.split(".")[0] in _BLOCKED_IMPORT_ROOTS:\n                    issues.append(f"blocked import: {alias.name}")\n        elif isinstance(node, ast.ImportFrom):\n            root = (node.module or "").split(".")[0]\n            if root in _BLOCKED_IMPORT_ROOTS:\n                issues.append(f"blocked import: {node.module}")\n        elif isinstance(node, ast.Call):\n            name = _call_name(node.func)\n            if name in _BLOCKED_CALLS or name.startswith(("subprocess.", "shutil.")):\n                issues.append(f"blocked call: {name}")\n            if name == "open":\n                mode: str | None = None\n                if len(node.args) >= 2 and isinstance(node.args[1], ast.Constant):\n                    mode = str(node.args[1].value)\n                for keyword in node.keywords:\n                    if keyword.arg == "mode" and isinstance(keyword.value, ast.Constant):\n                        mode = str(keyword.value.value)\n                if mode and any(flag in mode for flag in "wax+"):\n                    issues.append(f"blocked write-mode open: {mode}")\n    return sorted(set(issues))\n\n\ndef flatten(value, keys=(), depth=0):\n    if depth > 5:\n        return []\n    if isinstance(value, gp.Var):\n        return [(value, keys)]\n    result = []\n    if isinstance(value, dict):\n        for key, item in value.items():\n            result.extend(flatten(item, keys + (tuple(key) if isinstance(key, tuple) else (key,)), depth + 1))\n    elif isinstance(value, (list, tuple)):\n        for position, item in enumerate(value):\n            result.extend(flatten(item, keys + (position,), depth + 1))\n    return result\n\n\ndef replay(code, workdir):\n    issues = scan_code(code)\n    if issues:\n        return dict(ok=False, error=\'unsafe_source_rejected\', scan_issues=issues)\n    tree = ast.parse(code)\n    constructors = {target.id for node in ast.walk(tree) if isinstance(node, ast.Assign)\n        and isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Attribute)\n        and node.value.func.attr in {\'addVars\', \'addVar\'}\n        for target in node.targets if isinstance(target, ast.Name)}\n    models, snapshots = [], []\n    optimize = gp.Model.optimize\n    write = gp.Model.write\n\n    def observe_and_solve(model, *args, **kwargs):\n        assert not models, \'Only one optimization per saved program is allowed\'\n        models.append(model)\n        model.update()\n        frame = sys._getframe().f_back\n        while frame is not None:\n            if frame.f_code.co_filename == \'<fixed-code-replay>\':\n                for symbol, value in list(frame.f_locals.items()):\n                    if symbol.startswith(\'_\') or symbol in {\'m\', \'model\', \'CSVQA_DATA\'}:\n                        continue\n                    items = flatten(value)\n                    if items:\n                        snapshots.append((frame.f_code.co_name, symbol, items))\n            frame = frame.f_back\n        model.Params.Seed = 0\n        model.Params.Threads = 1\n        model.Params.TimeLimit = 180\n        model.Params.MIPGap = 1e-6\n        optimize(model, *args, **kwargs)\n\n    def prohibit_write(*args, **kwargs):\n        raise PermissionError(\'Generated model writes are disabled\')\n\n    gp.Model.optimize, gp.Model.write = observe_and_solve, prohibit_write\n    namespace = {\'__name__\': \'__main__\'}\n    output = io.StringIO()\n    os.chdir(workdir)\n    try:\n        with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):\n            exec(compile(code, \'<fixed-code-replay>\', \'exec\'), namespace)\n        assert len(models) == 1\n        model = models[0]\n        if model.SolCount <= 0:\n            return dict(ok=False, error=\'generated_model_has_no_incumbent\',\n                solver_status=int(model.Status), real_generated_optimizations=len(models),\n                code_output_tail=output.getvalue()[-1200:])\n        assert model.Status == gp.GRB.OPTIMAL, (\'not_optimal\', model.Status)\n        assert not any((model.NumQNZs, model.NumQConstrs, model.NumGenConstrs, model.NumSOS))\n        bindings = {}\n        for scope, symbol, items in snapshots:\n            family = re.sub(r\'[^A-Za-z0-9_]\', \'_\', symbol).strip(\'_\')\n            for variable, keys in items:\n                labels = [str(key) for key in keys]\n                semantic = family + \'[\' + \',\'.join(labels) + \']\' if labels else family\n                identity = dict(scope=scope, family=family, keys=labels, semantic_name=semantic)\n                rank = (0 if len(items) > 1 else 1, 0 if symbol in constructors else 1,\n                        0 if labels else 1, -len(items), symbol, scope)\n                bindings.setdefault(variable.VarName, []).append((rank, identity))\n        identities, variables = [], []\n        for variable in model.getVars():\n            raw = variable.VarName\n            options = sorted(bindings.get(raw, []), key=lambda pair: pair[0])\n            binding = options[0][1] if options else None\n            explicit = not re.fullmatch(r\'C\\d+\', raw)\n            assert explicit or binding is not None\n            semantic = raw if explicit else binding[\'semantic_name\']\n            identities.append(dict(original_name=raw, semantic_name=semantic, binding=binding,\n                                   preserved_explicit_name=bool(explicit)))\n            variables.append(dict(name=semantic, original_name=raw, value=float(variable.X),\n                vtype=variable.VType, lb=float(variable.LB), ub=float(variable.UB),\n                objective_coefficient=float(variable.Obj)))\n        assert len({record[\'semantic_name\'] for record in identities}) == len(identities)\n        constraints = []\n        for constraint in model.getConstrs():\n            expression = model.getRow(constraint)\n            constraints.append(dict(name=constraint.ConstrName, sense=constraint.Sense,\n                rhs=float(constraint.RHS), terms=[[expression.getVar(i).VarName, float(expression.getCoeff(i))]\n                                               for i in range(expression.size())]))\n        return dict(ok=True, objective=float(model.ObjVal), objective_constant=float(model.ObjCon),\n            variables=variables, variable_identity=identities, generated_constraints=constraints,\n            num_variables=model.NumVars, num_constraints=model.NumConstrs, num_nonzeros=model.NumNZs,\n            solver_status=int(model.Status), runtime_seconds=float(model.Runtime),\n            mip_gap=float(model.MIPGap) if model.IsMIP else 0.0,\n            objective_bound=float(model.ObjBound), model_solution_count=int(model.SolCount),\n            real_generated_optimizations=len(models), metadata_consults_ground_truth=False,\n            semantic_identity_complete=True, llm_calls=0, code_output_tail=output.getvalue()[-1200:])\n    except BaseException as exc:\n        return dict(ok=False, error=f\'{type(exc).__name__}: {exc}\',\n            traceback_tail=traceback.format_exc()[-2000:],\n            real_generated_optimizations=len(models), code_output_tail=output.getvalue()[-1200:])\n    finally:\n        gp.Model.optimize, gp.Model.write = optimize, write\n\n\ndef main():\n    code, output, workdir = map(Path, sys.argv[1:])\n    for key in list(os.environ):\n        if any(part in key.upper() for part in [\'OPENAI\', \'OPENROUTER\', \'DEEPSEEK\', \'API_KEY\', \'TOKEN\']):\n            os.environ.pop(key, None)\n    gp.setParam(\'OutputFlag\', 0)\n    gp.setParam(\'Threads\', 1)\n    def audit(event, args):\n        if event in {\'socket.connect\', \'socket.getaddrinfo\', \'subprocess.Popen\', \'os.system\'}:\n            raise PermissionError(\'Network/process calls prohibited\')\n        if event == \'open\' and len(args) >= 3:\n            frame = sys._getframe()\n            while frame is not None:\n                if frame.f_code.co_filename == \'<fixed-code-replay>\':\n                    mode, flags = args[1:3]\n                    if (isinstance(mode, str) and any(c in mode for c in \'wax+\')) or (\n                        isinstance(flags, int) and flags & (os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC)):\n                        raise PermissionError(\'Generated file writes prohibited\')\n                    break\n                frame = frame.f_back\n    sys.addaudithook(audit)\n    result = replay(code.read_text(), workdir)\n    result[\'llm_calls\'] = 0\n    output.write_text(json.dumps(result, ensure_ascii=False) + \'\\n\')\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:
def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def safe_path(root, relative):
    root = Path(root).resolve()
    path = (root / relative).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f'Path is outside the selected repository: {relative}')
    return path


def read_records(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        records = list(csv.DictReader(stream))
    if len(records) != 101 or {r['problem_id'] for r in records} != EXPECTED_IDS:
        raise ValueError('The source cohort must contain exactly the 101 unique benchmark IDs.')
    return sorted(records, key=lambda r: r['problem_id'])


def verify_inputs():
    for relative, expected in DATA_SHA256.items():
        path = safe_path(PROJECT_ROOT, relative)
        if not path.is_file() or file_sha256(path) != expected:
            raise ValueError(f'Missing or changed benchmark input: {relative}')
    source = {}
    for method in METHODS:
        directory = safe_path(PROJECT_ROOT, COHORTS[method]['relative_directory'])
        if not (directory / 'results.csv').is_file():
            raise FileNotFoundError(f'Required public cohort is missing: {COHORTS[method]["relative_directory"]}')
        if file_sha256(directory / 'results.csv') != COHORTS[method]['results_sha256']:
            raise ValueError(f'The source results.csv changed for {method}; use the documented source commit.')
        records = read_records(directory / 'results.csv')
        for row in records:
            fixture = IDENTITY_ARTIFACTS[method][row['problem_id']]
            for field in ['solve_code', 'solution']:
                path = safe_path(directory, row[field + '_path'])
                if file_sha256(path) != row[field + '_sha256']:
                    raise ValueError(f'Artifact hash mismatch: {method}/{row["problem_id"]}/{field}')
            if row['solve_code_sha256'] != fixture['source_code_sha256']:
                raise ValueError('Runtime identity metadata belongs to a different generated program.')
            if row['solution_sha256'] != fixture['original_solution_sha256']:
                raise ValueError('The original saved source solution has changed.')
        source[method] = records
    return source


def map_primary_values(bundle, solution, fixture, identities=None):

    pairs = solution.get('variables') or []
    raw = dict(pairs)
    if len(raw) != len(pairs) or not all(math.isfinite(float(v)) for v in raw.values()):
        raise ValueError('Duplicate variable names or non-finite incumbent values.')
    bindings = fixture['bindings'] if identities is None else identities
    original_to_semantic = dict(bindings)
    if set(original_to_semantic) != set(raw) or len(original_to_semantic) != len(bindings):
        raise ValueError('Saved-variable identities do not match the incumbent.')
    semantic_to_original = {semantic: original for original, semantic in bindings}
    if len(semantic_to_original) != len(bindings):
        raise ValueError('Variable identity is ambiguous.')
    mapping = fixture.get('primary_mapping')
    primary = {v.VarName for v in bundle.model.getVars()} - bundle.auxiliary_variables
    if mapping is None or set(mapping) != primary:
        return None
    if len(set(mapping.values())) != len(mapping) or not set(mapping.values()) <= set(semantic_to_original):
        return None
    values = {name: float(raw[semantic_to_original[semantic]]) for name, semantic in mapping.items()}
    assert all(values[name] == float(raw[semantic_to_original[semantic]]) for name, semantic in mapping.items())
    return values


def decimal_crosscheck(model, values, reference, exact_objective, checks):

    d = lambda value: Decimal.from_float(float(value))
    with localcontext() as context:
        context.prec = 120
        x = {name: d(value) for name, value in values.items()}
        independent = []
        for row in model.getConstrs():
            expression = model.getRow(row)
            lhs = sum((d(expression.getCoeff(i)) * x[expression.getVar(i).VarName]
                       for i in range(expression.size())), Decimal(0))
            rhs = d(row.RHS)
            residual = abs(lhs - rhs) if row.Sense == '=' else max(Decimal(0), lhs - rhs if row.Sense == '<' else rhs - lhs)
            independent.append(('constraint', residual))
        for var in model.getVars():
            value = x[var.VarName]
            for side, bound in [('lower', float(var.LB)), ('upper', float(var.UB))]:
                if abs(bound) < gp.GRB.INFINITY / 2:
                    independent.append(('bound', max(Decimal(0), d(bound) - value if side == 'lower' else value - d(bound))))
            if var.VType in {'I', 'B'}:
                independent.append(('integrality', abs(value - value.to_integral_value())))
            if var.VType == 'B':
                independent.append(('binary_domain', max(Decimal(0), -value, value - 1)))
        assert len(independent) == len(checks)
        for (kind, value), exact in zip(independent, checks):
            assert kind == exact['kind']
            for tolerance in TOLERANCES.values():
                threshold = exact['historical_tolerance'] if tolerance is None else tolerance
                decimal_threshold = Decimal(threshold.numerator) / Decimal(threshold.denominator)
                assert (value > decimal_threshold) == (exact['violation'] > threshold)
        objective = d(model.ObjCon) + sum((d(v.Obj) * x[v.VarName] for v in model.getVars()), Decimal(0))
        assert abs(float(objective) - float(exact_objective)) <= max(1e-9, abs(float(exact_objective)) * 1e-14)
        gap = 100 * abs(objective - d(reference)) / max(abs(d(reference)), Decimal(1))
        fraction_gap = 100 * abs(exact_objective - binary_fraction(reference)) / max(abs(binary_fraction(reference)), Fraction(1))
        assert abs(float(gap) - float(fraction_gap)) < 1e-10
        return {'linear_rows': model.NumConstrs, 'checks': len(checks), 'gap_percent': float(gap)}


def normalized_data_paths(row):
    result = []
    for address in row['dataset_address'].replace('\\n', '\n').splitlines():
        address = address.strip()
        if not address:
            continue
        if address.count('Test_Dataset/') != 1:
            raise ValueError('A dataset address must point into Test_Dataset/.')
        relative = 'Test_Dataset/' + address.split('Test_Dataset/', 1)[1]
        result.append((address, relative, safe_path(PROJECT_ROOT, relative)))
    return result


def replay_fixed_program(directory, row):

    if sys.version_info < (3, 12):
        raise RuntimeError('Fixed-code replay requires Python 3.12 or newer.')
    original = safe_path(directory, row['solve_code_path']).read_text(encoding='utf-8')
    rewrites = []
    with tempfile.TemporaryDirectory(prefix='lean_transfer_') as temporary:
        temporary = Path(temporary)
        inputs = temporary / 'inputs'
        inputs.mkdir()
        source = original
        for address, relative, path in normalized_data_paths(row):
            shutil.copyfile(path, inputs / path.name)
            for old in [address, relative]:
                if old in source:
                    source = source.replace(old, str(inputs / path.name))
                    rewrites.append(old)
            for old in [str(Path(address).parent), str(Path(relative).parent)]:
                if old in source:
                    source = source.replace(old, str(inputs))
                    rewrites.append(old)
        program, worker, captured = temporary / 'solve.py', temporary / 'worker.py', temporary / 'capture.json'
        program.write_text(source, encoding='utf-8')
        worker.write_text(REPLAY_WORKER_SOURCE, encoding='utf-8')
        environment = os.environ.copy()
        for key in list(environment):
            if any(part in key.upper() for part in ['OPENAI', 'OPENROUTER', 'DEEPSEEK', 'API_KEY', 'TOKEN']):
                environment.pop(key, None)
        environment.update(PYTHONHASHSEED='0', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
        environment.pop('LEAN_CSV_READ_ADAPTER', None)
        try:
            process = subprocess.run([sys.executable, str(worker), str(program), str(captured), str(inputs)],
                env=environment, capture_output=True, text=True, timeout=210)
            if not captured.exists():
                raise RuntimeError(f'Replay worker failed with exit status {process.returncode}.')
            capture = json.loads(captured.read_text())
        except subprocess.TimeoutExpired:
            return {'status': 'ERROR', 'objective': None, 'variables': [], 'error': 'replay_wall_timeout_210s'}, None
    if not capture['ok']:
        return {'status': 'ERROR', 'objective': None, 'variables': [], 'error': capture['error']}, None
    saved = {'status': 'OPTIMAL', 'objective': capture['objective'],
             'variables': [[v['original_name'], v['value']] for v in capture['variables']]}
    identities = [[v['original_name'], v['semantic_name']] for v in capture['variable_identity']]
    return saved, identities


def evaluate_cohorts(source_records, benchmark):
    rows, violations, audit = [], [], []
    gp.setParam('OutputFlag', 0)
    gp.setParam('Threads', 1)
    gp.setParam('Seed', 0)
    for method in METHODS:
        directory = safe_path(PROJECT_ROOT, COHORTS[method]['relative_directory'])
        for source in source_records[method]:
            pid, index = source['problem_id'], int(source['problem_id'][-3:])
            fixture = IDENTITY_ARTIFACTS[method][pid]
            if source['query'] != benchmark.iloc[index - 1]['Query']:
                raise ValueError(f'Query mismatch for {method}/{pid}')
            published = parse_published_objective(benchmark.iloc[index - 1]['Label-objective'])
            if not math.isclose(float(source['label_objective']), published, rel_tol=1e-12, abs_tol=1e-12):
                raise ValueError(f'Reference objective mismatch for {method}/{pid}')
            if method in REPLAY_METHODS:
                solution, live_bindings = replay_fixed_program(directory, source)
            elif COHORTS[method]['incumbent_protocol'] == 'published_saved_solution':
                solution = json.loads(safe_path(directory, source['solution_path']).read_text())
                live_bindings = None
            else:
                solution = fixture['frozen_replay_solution']
                live_bindings = None
            solved = bool(solution.get('status') == 'OPTIMAL' and solution.get('variables'))
            protocol = 'fresh_fixed_code_replay' if method in REPLAY_METHODS else COHORTS[method]['incumbent_protocol']
            basic = {'method': method, 'problem_id': pid, 'protocol': protocol, 'solved': solved,
                'predicted_label': source['predicted_label'], 'assigned_route': source['assigned_route'],
                'source_code_sha256': source['solve_code_sha256'],
                'original_solution_sha256': source['solution_sha256']}
            if not solved:
                failure = solution.get('error') or solution.get('status') or 'no_valid_incumbent'
                for profile in TOLERANCES:
                    rows.append({**basic, 'profile': profile, 'mapping_success': False, 'evaluable': False,
                        'transfer_feasible': False, 'feasible_gap_le_1': False,
                        **{kind + '_violation_count': None for kind in CHECK_KINDS},
                        'gap_percent': None, 'failure_reason': failure})
                continue
            with contextlib.redirect_stdout(io.StringIO()):
                bundle = load_ground_truth(PROJECT_ROOT, benchmark.iloc[index - 1], index, solve=False)
            primary = map_primary_values(bundle, solution, fixture, live_bindings)
            if primary is None:
                for profile in TOLERANCES:
                    rows.append({**basic, 'profile': profile, 'mapping_success': False, 'evaluable': False,
                        'transfer_feasible': False, 'feasible_gap_le_1': False,
                        **{kind + '_violation_count': None for kind in CHECK_KINDS},
                        'gap_percent': None, 'failure_reason': fixture.get('mapping_failure', 'incomplete_or_ambiguous_mapping')})
                bundle.model.dispose()
                continue
            auxiliary, completion = _derive_auxiliary_values(bundle, primary, bundle.auxiliary_variables)
            if set(auxiliary) != bundle.auxiliary_variables:
                raise RuntimeError(f'Incomplete reference auxiliary completion: {method}/{pid}')
            transferred = {**primary, **auxiliary}
            assert all(transferred[name] == value for name, value in primary.items())
            objective, checks = check_model(bundle.model, transferred)
            certificate = REFERENCE_CERTIFICATES[pid]
            assert certificate['status'] == 2 and certificate['solution_count'] > 0
            optimum = certificate['historical_reference_objective']
            if optimum is None:
                optimum = certificate['objective']
            gap = float(100 * abs(objective - binary_fraction(optimum)) / max(abs(binary_fraction(optimum)), Fraction(1)))
            verified = decimal_crosscheck(bundle.model, transferred, optimum, objective, checks)
            for profile, tolerance in TOLERANCES.items():
                failed = [check for check in checks if check['violation'] > (check['historical_tolerance'] if tolerance is None else tolerance)]
                feasible = not failed
                rows.append({**basic, 'profile': profile, 'mapping_success': True, 'evaluable': True,
                    'transfer_feasible': feasible, 'feasible_gap_le_1': feasible and gap <= 1,
                    **{kind + '_violation_count': sum(c['kind'] == kind for c in failed) for kind in CHECK_KINDS},
                    'transferred_objective': float(objective), 'ground_truth_optimum': optimum,
                    'gap_percent': gap if feasible else None, 'failure_reason': ''})
                violations.extend({'method': method, 'problem_id': pid, 'profile': profile,
                    'kind': c['kind'], 'name': c['name'], 'violation': float(c['violation']),
                    'exact_encoded_residual': str(c['violation'])} for c in failed)
            audit.append({'method': method, 'problem_id': pid, 'protocol': protocol,
                'primary_values_unchanged': True, 'auxiliary_completion': completion,
                'absolute_gap_percent': gap, 'linear_rows_checked': verified['linear_rows'],
                'total_checks': verified['checks'], 'independent_decimal_crosscheck_passed': True,
                'mapping': fixture['primary_mapping'], 'transferred_values': transferred})
            bundle.model.dispose()
        print(f'{method}: 101 instances evaluated.')
    return rows, violations, audit


def summarize(rows):
    summary = []
    for method in METHODS:
        for profile in TOLERANCES:
            selected = [r for r in rows if r['method'] == method and r['profile'] == profile]
            assert len(selected) == 101 and {r['problem_id'] for r in selected} == EXPECTED_IDS
            feasible = [r for r in selected if r['transfer_feasible']]
            evaluated = [r for r in selected if r['evaluable']]
            summary.append({'method': method, 'profile': profile, 'denominator': len(selected),
                'solved': sum(r['solved'] for r in selected), 'mapped': len(evaluated),
                'transfer_feasible': len(feasible), 'feasible_gap_le_1': sum(r['feasible_gap_le_1'] for r in selected),
                **{kind + '_violation_count': sum(r[kind + '_violation_count'] for r in evaluated) for kind in CHECK_KINDS},
                'mean_feasible_gap_percent': statistics.mean(r['gap_percent'] for r in feasible) if feasible else None,
                'median_feasible_gap_percent': statistics.median(r['gap_percent'] for r in feasible) if feasible else None,
                'max_feasible_gap_percent': max((r['gap_percent'] for r in feasible), default=None)})
    return summary


def save_results(rows, violations, audit, summary):
    if OUTPUT_DIR.resolve() == PROJECT_ROOT.resolve():
        raise ValueError('Use a dedicated output directory, not the repository root.')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    def write_table(name, records):
        records = list(records)
        if not records:
            return
        fields = list(dict.fromkeys(key for row in records for key in row))
        with (OUTPUT_DIR / name).open('w', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
    write_table('solution_transfer_results.csv', (r for r in rows if r['profile'] == 'diagnostic_1e-12'))
    write_table('tolerance_sensitivity.csv', rows)
    write_table('violations.csv', violations)
    (OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False) + '\n')
    (OUTPUT_DIR / 'transfer_audit.jsonl').write_text(''.join(json.dumps(r, allow_nan=False) + '\n' for r in audit))
    report = {'source_commits': {name: COHORTS[name]['source_commit'] for name in METHODS},
        'reference_data_sha256_verified': len(DATA_SHA256), 'llm_calls': 0,
        'fresh_generated_code_execution_attempts': 101 * len(REPLAY_METHODS),
        'saved_vector_reproduction': not bool(REPLAY_METHODS),
        'primary_decisions_never_repaired': True, 'mapping_uses_no_optimum_or_feasibility_feedback': True,
        'decimal_crosscheck_passed_for_every_evaluated_instance': True,
        'missing_violation_counts_are_NA': True,
        'embedded_input_sha256': EMBEDDED_INPUT_SHA256,
        'absolute_gap_formula': '100*abs(z_transfer-z_reference)/max(abs(z_reference),1)',
        'primary_profile': 'diagnostic_1e-12',
        'arithmetic': 'Exact Fraction evaluation of binary64 model and incumbent; independent Decimal120 crosscheck',
        'versions': {'python': sys.version.split()[0], 'gurobi': '.'.join(map(str, gp.gurobi.version())),
                     'numpy': np.__version__, 'pandas': pd.__version__}}
    (OUTPUT_DIR / 'validation.json').write_text(json.dumps(report, indent=2) + '\n')


In [ ]:
assert residual(Fraction(3), Fraction(5), '<') == 0
assert residual(Fraction(7), Fraction(5), '<') == 2
assert residual(Fraction(3), Fraction(5), '>') == 2
assert residual(Fraction(3), Fraction(5), '=') == 2

for directory in ['Test_Dataset', *(COHORTS[m]['relative_directory'] for m in METHODS)]:
    protected = (PROJECT_ROOT / directory).resolve()
    assert not OUTPUT_DIR.resolve().is_relative_to(protected)
    assert not protected.is_relative_to(OUTPUT_DIR.resolve())

source_records = verify_inputs()
benchmark = pd.read_csv(PROJECT_ROOT / 'Test_Dataset/Large-scale-or/Large-scale-or-101.csv')
assert len(benchmark) == 101
print('Verified benchmark + referenced CSV inputs:', len(DATA_SHA256))
print('Verified source cohort rows:', {method: len(records) for method, records in source_records.items()})


In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    rows, violations, audit = evaluate_cohorts(source_records, benchmark)
summary = summarize(rows)

assert len(rows) == len({(r['method'], r['profile'], r['problem_id']) for r in rows}) == 101 * len(METHODS) * len(TOLERANCES)
assert all(r['solved'] or not r['mapping_success'] for r in rows)
assert all(not r['feasible_gap_le_1'] or r['transfer_feasible'] for r in rows)
assert all(not r['transfer_feasible'] or r['mapping_success'] for r in rows)
assert all(r['primary_values_unchanged'] and r['independent_decimal_crosscheck_passed'] for r in audit)
assert all(all(r[k + '_violation_count'] is None for k in CHECK_KINDS) for r in rows if not r['evaluable'])
verify_inputs()
save_results(rows, violations, audit, summary)
print('Transfer recomputed and cross-checked for all selected cohorts; LLM calls: 0.')
print('Results written to:', OUTPUT_DIR.relative_to(PROJECT_ROOT) if OUTPUT_DIR.is_relative_to(PROJECT_ROOT) else OUTPUT_DIR.name)


In [ ]:
def rate(count):
    return f'{count}/101 ({100 * count / 101:.2f}%)'

main_table = pd.DataFrame([{
    'Method': r['method'],
    'Solved': rate(r['solved']),
    'Transfer feasible': rate(r['transfer_feasible']),
    'Feasible and gap ≤ 1%': rate(r['feasible_gap_le_1']),
    'Violated constraints': r['constraint_violation_count'],
} for r in summary if r['profile'] == 'diagnostic_1e-12'])
display(main_table.set_index('Method'))


In [ ]:
sensitivity = pd.DataFrame(summary)[[
    'method', 'profile', 'solved', 'mapped', 'transfer_feasible', 'feasible_gap_le_1',
    'constraint_violation_count', 'bound_violation_count', 'integrality_violation_count',
    'binary_domain_violation_count', 'mean_feasible_gap_percent', 'max_feasible_gap_percent']]
display(sensitivity.set_index(['method', 'profile']))

unavailable = [r for r in rows if r['profile'] == 'diagnostic_1e-12' and not r['evaluable']]
if unavailable:
    display(pd.DataFrame(unavailable)[['method', 'problem_id', 'solved', 'failure_reason']].set_index(['method', 'problem_id']))
